# Block-pointer matmul with a transposed operand

Hand-built pointer tiles (`base + rows[:, None] * s0 + cols[None, :] * s1`) plus hand-built masks are where most matmul bugs live. **Block pointers** describe a tile once, as a window into a 2-D tensor with a shape, strides, an offset, a block shape and a memory order. `tl.load` then bounds-checks it for you, and `tl.advance` slides the window.

They also make strided operands trivial. Here $A$ arrives **transposed**: you get `a_t` with shape `(K, M)` and must compute $C = a_t^{\top} B$ without materialising the transpose.

Write `matmul_at_b(a_t, b)`.

## Inputs
- `a_t`: contiguous `float32` CUDA `(K, M)`; `b`: contiguous `float32` CUDA `(K, N)`; all sizes $\ge 1$.

## Output
- `float32` `(M, N)`, $C = a_t^{\top} b$, i.e. $C_{mn} = \sum_k (a_t)_{km}\, b_{kn}$.

## Requirements
- 2-D grid over `(BM, BN)` output tiles with `BM = BN = BK = 32` (constexpr).
- Build every operand with `tl.make_block_ptr`:
  - $A$ as a logical `(M, K)` tensor whose strides are `a_t`'s strides swapped (`(1, M)` for contiguous `a_t`); its fastest-varying dimension is 0, so `order=(0, 1)`.
  - $B$ as `(K, N)` and $C$ as `(M, N)`, both row-major: `order=(1, 0)`.
- Load with `boundary_check=(0, 1), padding_option="zero"`, step with `tl.advance` (A along dim 1, B along dim 0), store $C$ with `boundary_check=(0, 1)`. No hand-written masks or offset arithmetic beyond the tile origin.
- Accumulate in float32 with `tl.dot(..., input_precision="ieee")`.

This problem runs in **Colab with a GPU runtime**. Expected outputs come from a float64 PyTorch reference.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
import triton
import triton.language as tl

def matmul_at_b(a_t, b):
    """a_t: (K, M), b: (K, N) float32 CUDA. Return a_t.T @ b (M, N) using block pointers."""
    raise NotImplementedError


@triton.jit
def _matmul_bp_kernel(a_ptr, b_ptr, c_ptr, M, N, K, stride_am, stride_ak, stride_bk, stride_bn,
                      stride_cm, stride_cn, BM: tl.constexpr, BN: tl.constexpr, BK: tl.constexpr):
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, 2.0], [3.0, 4.0]], "dtype": "float32"}, {"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float32"}], "e": {"__nd__": [[1.0, 3.0], [2.0, 4.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[1.0, 0.0, 2.0], [0.0, 1.0, -1.0]], "dtype": "float32"}, {"__nd__": [[2.0], [5.0]], "dtype": "float32"}], "e": {"__nd__": [[2.0], [5.0], [-1.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[-0.5667637586593628]], "dtype": "float32"}, {"__nd__": [[-1.0565742254257202]], "dtype": "float32"}], "e": {"__nd__": [[0.5988279581069946]], "dtype": "float32"}}, {"a": [{"__nd__": [[1.062079906463623, -1.1293522119522095, -1.7794053554534912, 0.6549562215805054, 0.8412511944770813, -0.949110209941864, -0.5653303265571594, 2.7797272205352783, -1.0174384117126465, -0.5057891011238098, 1.1787562370300293, 0.7454060912132263, 1.6118038892745972, 0.8177558183670044, -0.4076789915561676, 0.11844004690647125, -0.5501276850700378, 0.8057569265365601, 0.5083563923835754, -1.2307981252670288, 0.4845283329486847, -0.8193381428718567, 0.07219039648771286, -0.38102027773857117, 0.28305163979530334, -2.595606803894043, 0.444396436214447, -1.4232256412506104, 1.1720445156097412, -1.1824777126312256, -0.5179637670516968, -0.7100846171379089], [0.25725021958351135, 1.0592385530471802, 0.21323181688785553, -0.022286226972937584, 0.16465023159980774, -0.10464085638523102, 0.05473983660340309, -0.7373988032341003, -0.13434216380119324, 1.4720549583435059, 1.1176034212112427, -1.7048850059509277, -0.21112701296806335, 0.205278679728508, -0.5214464664459229, -0.6513322591781616, -0.03925357386469841, 1.0155750513076782, 1.0835871696472168, 0.06190035492181778, 0.858604907989502, -0.07819980382919312, 0.4139052629470825, -0.189886212348938, -1.5125874280929565, -0.6628860235214233, 0.39923182129859924, 1.1914005279541016, 0.35632291436195374, 2.1037588119506836, 0.9529293775558472, 0.07806254178285599], [-0.41850796341896057, 2.3035948276519775, -0.19698457419872284, -1.1008868217468262, 0.4240606427192688, -0.5360121726989746, 0.6286531090736389, -1.5325065851211548, 0.932472288608551, 1.0127750635147095, -0.34254005551338196, -0.957282543182373, 0.15243510901927948, 1.1884896755218506, -0.5825433731079102, -0.17130433022975922, 1.1884204149246216, 0.6206063032150269, -0.10920681059360504, 0.3603389263153076, 0.196267768740654, -0.4048452377319336, -0.13486114144325256, -2.663576602935791, 0.13029299676418304, 0.8937222361564636, 0.6832270622253418, -0.42692291736602783, 1.6850537061691284, 1.382283091545105, -0.8226597309112549, -0.01894349791109562], [-0.43496304750442505, -0.8234632611274719, 1.054262399673462, 1.0301522016525269, -0.67210853099823, -0.14206130802631378, 0.4213707447052002, -0.29953455924987793, 0.9848403930664062, 0.35335180163383484, 0.36339810490608215, -1.513570785522461, 1.2181483507156372, -2.2636935710906982, -0.06876127421855927, 1.1086928844451904, 0.10089243948459625, -0.787555456161499, -0.44443199038505554, -1.262221097946167, 1.2413873672485352, 0.4186970293521881, -0.9245229959487915, 1.8010272979736328, -0.5432034730911255, 0.3884659707546234, -1.8446612358093262, -0.06702657043933868, 0.6382720470428467, 0.11125891655683517, -0.8069060444831848, 1.051361322402954], [-1.0815694332122803, 1.3633941411972046, -0.17884860932826996, -0.07401799410581589, 0.3690797984600067, 0.4188652038574219, 2.000436544418335, 0.18845723569393158, -1.5597255229949951, 0.3568294048309326, -0.9714114665985107, -0.46613210439682007, -0.45383399724960327, 0.09706144779920578, 1.2228292226791382, 0.0800313875079155, -0.44283977150917053, -0.8519112467765808, -0.8099374175071716, -0.47756433486938477, 0.20130307972431183, 0.09794282168149948, 0.7757894992828369, -2.159285545349121, 0.7371731996536255, 0.6576628088951111, -2.341902017593384, 1.121353030204773, -0.008792261593043804, -1.44153892993927, 0.30152779817581177, -1.2299855947494507], [-0.21809421479701996, 0.5544072389602661, 1.1002355813980103, 1.005007028579712, -0.20611177384853363, -2.208400249481201, -0.7059615850448608, 0.2664274573326111, 0.7529094219207764, 0.11904478073120117, -1.0897504091262817, -0.40565934777259827, -1.1231166124343872, -0.07144622504711151, -1.842127799987793, -0.11156686395406723, 0.7656123638153076, -1.1334340572357178, 1.1243962049484253, -2.5127930641174316, 0.6183862090110779, -1.248807668685913, -0.3544003665447235, 0.2775155007839203, 0.6223472356796265, -1.820715069770813, 0.19981098175048828, -1.8050596714019775, 0.11012789607048035, -0.3142622113227844, 0.3115191161632538, -1.0734326839447021], [-0.3592701256275177, -1.2169311046600342, -0.010555564425885677, 0.7222630381584167, 0.5383386015892029, -0.2327880561351776, 1.0999341011047363, 0.5043041706085205, 1.2059051990509033, 0.35288140177726746, -0.3223401606082916, 0.46849820017814636, 1.1432780027389526, -0.2252640724182129, 0.3290991187095642, -0.8067342042922974, 1.0436227321624756, -0.6611064672470093, -1.9393669366836548, 0.8362221717834473, -1.383833646774292, 1.3410886526107788, -0.6843841671943665, -1.312221646308899, -1.0080028772354126, 0.6213821768760681, -0.5475226640701294, 0.6714887619018555, -1.0713274478912354, 0.8579623103141785, 0.12128520756959915, 0.09864996373653412], [3.3595077991485596, -1.3475337028503418, 0.8153982758522034, -1.5965765714645386, -0.4145280420780182, -1.1044670343399048, -0.6070932149887085, 0.04121388494968414, -1.2722073793411255, 0.5615665316581726, -0.9382362961769104, 0.8440154194831848, -0.6857907176017761, -0.209314227104187, 0.007303725462406874, -0.20356950163841248, 0.994574785232544, 3.0188095569610596, -0.005254310555756092, -0.6055663824081421, 0.03455417603254318, 0.43684691190719604, -0.5293230414390564, 2.6057186126708984, -1.1184368133544922, 0.3313504755496979, -1.2481743097305298, 0.8758131861686707, 1.5836604833602905, -0.49512892961502075, 0.24642804265022278, -0.6573264598846436], [1.5866836309432983, 0.6415498852729797, 0.06037897244095802, -0.1829547882080078, 0.8426260352134705, -0.5308070778846741, 0.1400223970413208, -1.5757960081100464, -0.06448652595281601, 0.5309867858886719, -1.0935847759246826, 0.8213834166526794, -1.0541452169418335, -0.92988520860672, -0.6610170006752014, 0.5249036550521851, -0.051078081130981445, 1.36409592628479, -2.837730884552002, -0.39325302839279175, 1.4241478443145752, -0.7181280851364136, 1.4974850416183472, -0.25015246868133545, 2.7334201335906982, -2.1537067890167236, 1.3180629014968872, 1.5857433080673218, 1.2899518013000488, -1.5207605361938477, 1.1333216428756714, -2.2533681392669678], [0.18106529116630554, 1.3457218408584595, 0.9966495633125305, 0.5897036790847778, 0.6687995195388794, 0.7077021598815918, -1.4211512804031372, -1.8443924188613892, 0.08611337840557098, -0.05741961672902107, -0.4648098051548004, -0.13473699986934662, 0.48527225852012634, -0.2027885764837265, -0.9641335606575012, 0.2038564234972, -0.05223659425973892, -0.3931408226490021, 0.2939595878124237, -0.8351733088493347, 0.15883566439151764, 0.33359259366989136, 0.25289493799209595, -0.12262345105409622, -0.0007598690572194755, 0.4935910999774933, 1.8814424276351929, 1.2563704252243042, -0.6705008745193481, 0.045930687338113785, -2.521317720413208, -0.5792743563652039], [-0.2872174084186554, 0.5474686026573181, 0.2211528867483139, 1.2661240100860596, 0.12320660799741745, 0.5198653936386108, -1.2984251976013184, -0.24829334020614624, -0.6896228790283203, 0.9479512572288513, -1.255516529083252, -1.4906052350997925, -1.1759103536605835, -0.1347000002861023, -0.5384586453437805, 0.5762940645217896, 2.256972312927246, -0.2619418799877167, -1.2410567998886108, 0.03914129361510277, 0.7193304300308228, 1.8127652406692505, -0.5557599067687988, 2.1295225620269775, 0.2646985948085785, -0.2780354917049408, 0.2364061176776886, 0.05432615056633949, 0.04648873582482338, -1.4315212965011597, -0.6431742906570435, -1.6172032356262207], [-0.143041729927063, -0.6451120376586914, 0.67289799451828, -0.12936460971832275, -1.0841292142868042, -0.6468349695205688, -0.2534097135066986, -0.5647029876708984, 1.2470664978027344, -0.9855298399925232, -1.0465476512908936, -0.07285678386688232, -1.184900164604187, 0.14431831240653992, -0.4165077209472656, -0.007252564188092947, -0.2242584079504013, 0.33594992756843567, -0.5311209559440613, 0.3026086390018463, -0.7244753241539001, -1.008270263671875, 0.1931850165128708, 0.9657096266746521, -0.7274405360221863, -0.7108874320983887, -0.8984264135360718, -0.061495356261730194, -0.895704448223114, -0.019662383943796158, -0.3943900763988495, 0.9817719459533691], [0.4729725420475006, -0.4119536280632019, 0.3264937698841095, 0.9122138023376465, 1.3325791358947754, 1.5090420246124268, -0.731002926826477, 0.39148759841918945, 0.009193471632897854, 1.8855700492858887, 1.3742866516113281, -0.6764547824859619, -0.4293668866157532, -0.49471157789230347, -1.0371105670928955, -0.061944786459207535, -0.843216061592102, -1.4006068706512451, -0.6010572910308838, -0.6743959784507751, -2.1621954441070557, -1.3302671909332275, -0.460935115814209, -1.0721572637557983, 1.2381296157836914, 1.0035135746002197, -0.8022541403770447, 0.6421301364898682, 0.025620946660637856, -2.1944637298583984, -0.6092824339866638, 0.9002020359039307], [-0.13170726597309113, -0.32445403933525085, 0.7891705632209778, 0.6418176889419556, 0.6160814762115479, -0.9775369763374329, -0.22682254016399384, -0.3977823257446289, -0.7284435033798218, -1.8940916061401367, -0.8300178647041321, -0.10293823480606079, -2.8071484565734863, 2.00555157661438, 0.6581276059150696, 0.9841212630271912, 1.514835000038147, 0.19687126576900482, -0.9743987917900085, 1.9898163080215454, -0.6363010406494141, 1.9147149324417114, 1.3185296058654785, -0.7433643937110901, 0.9655458927154541, -0.747001588344574, 0.49358242750167847, -0.5166568756103516, 0.49322745203971863, -0.9699884653091431, -0.9438047409057617, 0.7691196203231812], [-2.312052011489868, 0.0675349086523056, -1.727978229522705, -0.8159225583076477, -2.566218137741089, -0.6038421988487244, 0.2272329181432724, -1.0036816596984863, 0.017732279375195503, 0.6900856494903564, -0.7084251642227173, -0.7227290272712708, -0.5335671901702881, -0.7211620211601257, -0.5441758632659912, -0.9927384853363037, 0.9170172214508057, -0.5256836414337158, -0.5180709362030029, -0.4321618974208832, 0.18863798677921295, 1.8475844860076904, -0.49412205815315247, -0.28221559524536133, -0.7201268672943115, -0.3952846825122833, 0.4842472970485687, 0.1750585436820984, -0.38744133710861206, -0.13339954614639282, 0.07610827684402466, -0.6065033078193665], [-1.152763843536377, -0.2126939296722412, 0.4446299076080322, -1.4443937540054321, 1.1833478212356567, 0.3178035318851471, -0.3087521195411682, 1.0401326417922974, 0.14793401956558228, 0.03948061913251877, -0.19249224662780762, -1.4229934215545654, -0.4858081638813019, -1.6509743928909302, 1.144303560256958, 0.7791352868080139, 1.0519963502883911, 1.4010754823684692, 0.45554235577583313, -1.1898415088653564, 1.1364282369613647, 0.28074175119400024, 0.36722320318222046, 1.509736180305481, 1.1918559074401855, -0.037493910640478134, -0.6244887709617615, -0.26280802488327026, -2.349062204360962, -0.8802189230918884, 1.0957218408584595, -1.365097165107727], [-0.015332872048020363, 0.6491456627845764, 0.44771674275398254, 0.5682019591331482, 0.22797875106334686, 0.7470857501029968, 0.8790934681892395, -0.3075357675552368, 0.58867347240448, 0.20924903452396393, 0.6164904236793518, 0.16561341285705566, 0.6143879890441895, -0.05866718292236328, -0.6800382733345032, 0.1076260507106781, -0.3320368826389313, 1.2161401510238647, 0.07428757846355438, 1.907057285308838, -0.2722356915473938, 1.3485740423202515, -1.9450139999389648, 0.7418251037597656, 0.28369736671447754, -0.21429860591888428, 1.2521758079528809, -0.7358847260475159, 1.715596079826355, -0.06735369563102722, 0.026820514351129532, -1.1013463735580444], [0.3816840350627899, -0.46432414650917053, 0.09991856664419174, -0.0646267682313919, 0.27564775943756104, -1.1427342891693115, -0.5697979927062988, 0.007651076186448336, 0.16785353422164917, -0.564260721206665, -0.5406275987625122, -0.5882918238639832, -0.9158657193183899, 0.5736588835716248, 1.2972760200500488, -0.34849482774734497, -0.14587034285068512, -2.7304916381835938, 1.388788104057312, 0.41966262459754944, -2.3490467071533203, 1.5172494649887085, 0.12887553870677948, -0.12783940136432648, -1.658360242843628, -0.2651028335094452, -2.2331976890563965, -0.19872182607650757, 1.0575412511825562, -0.4357316792011261, -1.8596441745758057, 0.8420261740684509], [-1.6087568998336792, 0.03980128467082977, -0.04534559324383736, 1.2910873889923096, 0.012839648872613907, 0.4014602601528168, -1.8669875860214233, -0.12740181386470795, -0.48737725615501404, -1.8815521001815796, 0.27588996291160583, 0.48938819766044617, -1.4259650707244873, -1.2220686674118042, 0.0034192281309515238, -0.12232302129268646, 0.672244131565094, -1.7944577932357788, -1.145361304283142, 0.41737455129623413, -1.686977744102478, -1.8881382942199707, 0.4066704511642456, -0.6130165457725525, 1.506197452545166, -0.7495930790901184, 0.321210116147995, 0.007990265265107155, 0.2950458824634552, 0.6166986227035522, 0.7675505876541138, -1.1744568347930908], [0.4932946562767029, -1.6097596883773804, -1.6484792232513428, -0.6722596883773804, -0.8874754905700684, 1.1109267473220825, 0.9620845913887024, -0.9655353426933289, -1.3755568265914917, -0.6979666948318481, -0.04267911612987518, -0.19028910994529724, -0.33723732829093933, -0.4268753230571747, -0.8004330992698669, -0.9364813566207886, 0.8655208945274353, -0.008015166968107224, -1.2927876710891724, 0.8540742993354797, -0.15174689888954163, -2.167954921722412, -0.3569020926952362, 0.8721391558647156, 0.11648516356945038, -0.7859987616539001, -0.794873833656311, -1.2601608037948608, 1.3591609001159668, 0.4052714705467224, 0.4509289860725403, 0.9615164995193481], [-0.8716264963150024, -1.4592992067337036, -0.8316017985343933, 0.07277686148881912, -0.7334768176078796, 0.3617231249809265, -0.6326248049736023, -0.2924449145793915, 1.076640248298645, -0.283511221408844, 0.26914384961128235, 0.6021312475204468, -0.8917419910430908, -0.8211752772331238, 0.3094368875026703, 2.463549852371216, -1.8940061330795288, -0.2702198624610901, 0.40279117226600647, 0.7003364562988281, 0.9872080683708191, 0.056289192289114, 1.7783831357955933, -0.8286057710647583, -0.3396422266960144, -1.2389917373657227, -0.9227331876754761, 0.7531219720840454, 1.9686033725738525, -0.7394295334815979, 0.6526057124137878, 0.3744230568408966], [-0.1014808863401413, -0.16703283786773682, 1.5552027225494385, 0.7267358899116516, 0.9874078035354614, 0.8803759813308716, 0.49852776527404785, -0.5389261841773987, 0.1955738663673401, 0.6872026324272156, 2.8502233028411865, 0.5144485235214233, -0.7752201557159424, 0.6437568664550781, 1.3433644771575928, -0.4320298433303833, -1.2295137643814087, 1.2164299488067627, -1.5551968812942505, -0.9745355844497681, 0.08749483525753021, 1.7896257638931274, -0.32406750321388245, -0.39622366428375244, -0.4746072590351105, -0.1539275199174881, 0.128587007522583, -1.4890152215957642, -0.9671362638473511, 0.7596856355667114, 0.1061784029006958, 1.2823809385299683], [0.6833597421646118, 0.07854463160037994, -0.18495261669158936, -0.16146069765090942, 0.6140320301055908, 0.3002951443195343, -0.36693429946899414, 1.2108943462371826, -0.22702820599079132, 0.6595902442932129, -0.3446066677570343, 1.1341642141342163, -0.2730146646499634, 0.14305926859378815, -0.4206429421901703, -2.2186100482940674, 0.6429783701896667, -1.2101978063583374, 1.1809989213943481, 0.15483443439006805, 0.9224805235862732, 0.948737621307373, 0.06711088120937347, -0.5720142722129822, -0.27017995715141296, -0.31686195731163025, 0.3494656980037689, -0.38167324662208557, 1.3396713733673096, 0.2583934962749481, -0.0251744594424963, 1.3278093338012695], [0.9289272427558899, 0.05131682753562927, -0.305829793214798, 0.06525921821594238, 0.7744067311286926, 0.7216528058052063, 0.8332889676094055, 1.6045629978179932, 1.1918059587478638, -2.214012384414673, 1.2688432931900024, 0.028337424620985985, 0.5703970789909363, -0.9826217889785767, 1.54109525680542, -2.7150380611419678, -0.6720967888832092, -0.3542287349700928, 0.062100451439619064, -1.9208904504776, 0.28695428371429443, 0.5435783267021179, 0.37720391154289246, -2.8539934158325195, -0.5143648982048035, 0.16885140538215637, 0.8322480916976929, 0.22200706601142883, 1.5193490982055664, -0.13007238507270813, -1.096251368522644, 0.2415028214454651], [-0.44101500511169434, 0.6969330310821533, -1.0495543479919434, -0.7018816471099854, -0.9680126905441284, -0.2834516763687134, -1.3272212743759155, 1.237321376800537, -0.0010426294757053256, -0.8589547872543335, -0.3450329303741455, -0.07491253316402435, 2.0309829711914062, 0.4730406403541565, -0.1739674061536789, -1.7681347131729126, -0.11907944083213806, 0.049937985837459564, 0.9131983518600464, -1.0271703004837036, -0.14548158645629883, -0.5374516248703003, -0.13366879522800446, 0.32667312026023865, -0.7324159145355225, -0.19033151865005493, -0.9833791851997375, -1.0802700519561768, 1.028057336807251, 2.534308671951294, 1.2072598934173584, -1.2666394710540771], [1.141064167022705, 0.0634729191660881, -0.06200789287686348, 0.763615608215332, 0.6680075526237488, -0.20444738864898682, 1.0621174573898315, -0.1406513750553131, 1.0014331340789795, 1.2340060472488403, -2.8798611164093018, 0.6673644781112671, 1.1766541004180908, -0.03549889102578163, -0.1361139863729477, -0.23586557805538177, 0.03526376932859421, 0.7080811262130737, -1.2149070501327515, 0.9473744034767151, -0.0667915791273117, 1.184619426727295, -0.2087012380361557, 1.7816143035888672, 0.5234881043434143, -0.12459087371826172, -0.4799937307834625, -1.1785582304000854, -1.801210880279541, -1.7608160972595215, 1.9698594808578491, 2.1495418548583984], [2.038818120956421, -1.0596933364868164, 0.5776998996734619, -1.3287005424499512, -1.7584290504455566, -1.398425579071045, -2.3818938732147217, 1.1588315963745117, -0.6138187050819397, 0.2107284963130951, -0.30098259449005127, 0.25077033042907715, -0.39273887872695923, -0.20676305890083313, 0.1262994110584259, -0.136983722448349, 0.08243903517723083, -0.5638187527656555, -0.9350297451019287, 0.04055585339665413, -1.605591893196106, 1.1185330152511597, -0.9922125935554504, 1.55544912815094, -1.131925344467163, 0.6442556381225586, -0.525745689868927, 0.5658895373344421, -2.604153871536255, -0.9192445278167725, 0.20539936423301697, -1.5441303253173828], [1.1347647905349731, -1.6409718990325928, 0.5792561173439026, -1.5609632730484009, 0.38072022795677185, 2.548741340637207, -0.03948672488331795, 0.39400672912597656, 1.4614354372024536, -0.41008391976356506, 0.39587926864624023, -0.7529802322387695, -0.1855766326189041, -0.2212258279323578, -1.0711222887039185, -0.7062222361564636, 0.4608944356441498, 0.2528762221336365, -0.8413698673248291, 0.9916360378265381, -0.3054427206516266, -0.11303028464317322, -1.6015214920043945, -2.0119218826293945, -2.0838470458984375, -0.2234848439693451, -1.9656352996826172, 0.3891220986843109, 0.38729214668273926, 0.420579195022583, 1.0497171878814697, -0.11935528367757797], [0.06763994693756104, 0.4742501378059387, 1.0166687965393066, -1.3187875747680664, -0.4106341600418091, 1.5190106630325317, -1.4137842655181885, 1.6196982860565186, -0.24166825413703918, 1.0866483449935913, 0.4860173463821411, 0.7785782814025879, -0.14162638783454895, -0.05597065016627312, 0.0782516822218895, -1.8592193126678467, 0.7134523391723633, 0.04569334536790848, -0.029703715816140175, 1.7226580381393433, 0.43067196011543274, -2.1992764472961426, -0.08387988805770874, -1.1747276782989502, -0.13198429346084595, -0.2623783349990845, 0.3869430422782898, 0.4264375567436218, 0.7049683928489685, 0.7800485491752625, -0.9163790941238403, 0.2915242314338684], [1.4607603549957275, 0.20754386484622955, 0.8246243596076965, -1.9684178829193115, -0.11606395989656448, -0.7225621938705444, 0.6338872909545898, 1.3989585638046265, 0.58791184425354, -1.3482221364974976, -0.671733021736145, 0.01775532029569149, 0.008856242522597313, 1.2691541910171509, 0.3003518879413605, 0.6231352686882019, 0.6772747039794922, 1.9086600542068481, -1.2515323162078857, 0.6104336380958557, 0.8551994562149048, -1.0350019931793213, 1.3618437051773071, -0.6162368655204773, -2.9936044216156006, -1.736688256263733, 0.815022349357605, 0.8715629577636719, 0.4027722477912903, 1.5723369121551514, 1.3475866317749023, 0.4587467908859253], [-0.6687247157096863, -0.46575409173965454, 0.7490395903587341, -0.9366777539253235, 1.2237664461135864, -0.3103436827659607, 0.9593636989593506, -0.5586949586868286, -0.35539329051971436, -0.9148093461990356, -0.15854354202747345, 1.052166223526001, 0.17741960287094116, 0.3038073778152466, -0.2735661566257477, 0.3444500267505646, -0.18651911616325378, -0.24929079413414001, 0.2155471295118332, -1.5462660789489746, 0.5145476460456848, -0.08930359780788422, 0.8195546865463257, 1.6807985305786133, 0.1952020227909088, -1.3191931247711182, 1.1954444646835327, 1.6044511795043945, -0.024912994354963303, 0.2184392362833023, -0.9598696827888489, 0.4099704921245575], [2.3121767044067383, 1.0464136600494385, -1.295419454574585, 1.6026188135147095, -0.2253706306219101, 0.02309732884168625, -1.0849950313568115, -0.5227312445640564, 1.199146032333374, 1.7916561365127563, 1.28045654296875, 0.45159807801246643, -2.830838441848755, 0.8770661354064941, -1.0677074193954468, 0.7418134212493896, -0.4984191358089447, -0.13675710558891296, -1.3387956619262695, 1.2482037544250488, 0.7299785614013672, 0.11341030150651932, -0.8395370244979858, -0.9502428770065308, 0.39479130506515503, 0.6373819708824158, 0.6068839430809021, -2.207974433898926, -0.6388959884643555, 1.0145610570907593, 0.5958765149116516, -0.9996033310890198]], "dtype": "float32"}, {"__nd__": [[0.532985270023346, -1.5516332387924194, 0.7042592763900757, 0.4641467332839966, 1.0706425905227661, -0.12313778698444366, -0.4484756588935852, -0.6993123888969421, -0.09749900549650192, -2.1200363636016846, -1.447239875793457, 0.015686240047216415, -0.022724732756614685, 1.0935041904449463, -0.29746347665786743, 2.0197627544403076, -1.6586583852767944, -0.39294594526290894, -0.9340791702270508, 0.00978577509522438, -0.46189525723457336, -0.716882050037384, 0.5403283834457397, 0.4236285090446472, 0.9668829441070557, 0.3313187062740326, 1.0287524461746216, -0.753984808921814, 0.7230554819107056, -1.387832522392273, 0.396123468875885, 0.2823820114135742], [0.8422399163246155, -1.5389790534973145, -0.03466133773326874, -0.34073662757873535, -0.07799281924962997, -1.9227653741836548, 0.49170517921447754, 0.3334798812866211, 0.054748326539993286, -1.1573418378829956, -0.25314590334892273, 0.8717934489250183, 1.3604613542556763, -0.8154370784759521, 0.14985056221485138, -0.412798136472702, -0.23225241899490356, -0.965029239654541, 1.0599968433380127, -0.2473660409450531, -1.9096333980560303, 0.7716284990310669, 0.9560927152633667, -1.570735216140747, -0.48816269636154175, 0.29336705803871155, 0.02763102576136589, 1.8033194541931152, 0.11530354619026184, 0.5720802545547485, 1.9670792818069458, -0.859208345413208], [2.0642058849334717, -0.8496920466423035, -0.41471847891807556, -1.2981899976730347, -2.344977617263794, 0.5371314883232117, 0.14394378662109375, 2.143230676651001, 0.5406020879745483, 1.6429228782653809, 0.2306399792432785, 1.049108862876892, -1.122025966644287, -0.29340121150016785, 0.14196011424064636, 1.4658854007720947, 0.4698461592197418, 2.5423550605773926, -0.7881056070327759, -1.3533549308776855, 1.6891825199127197, 3.9145562648773193, 1.5878504514694214, -0.7232027649879456, -1.2947566509246826, 0.7334433197975159, -0.4916488230228424, 0.5192122459411621, 1.12128484249115, 0.44258591532707214, 1.6634323596954346, -1.2834807634353638], [1.2101256847381592, -0.1526997685432434, -0.27049723267555237, 0.21020369231700897, -0.3540915250778198, 1.0311073064804077, -1.232835054397583, -0.335038423538208, 1.7465405464172363, 1.1372723579406738, -1.4950553178787231, -0.6420350670814514, -0.9466757774353027, -1.1660743951797485, -4.340343475341797, -1.154490351676941, -1.7575141191482544, 0.5125803351402283, 0.711580753326416, 0.8458338379859924, -0.7165018320083618, -0.1594904661178589, 1.3928775787353516, -0.7328793406486511, 0.13109908998012543, -1.6016980409622192, -0.5287588238716125, 0.2460700273513794, 0.16458088159561157, -2.0678210258483887, 0.2815953195095062, 0.5930783748626709], [0.4680730104446411, -0.4647449254989624, -0.08571334928274155, 0.048628006130456924, -0.5043686032295227, 0.32569876313209534, -1.4905030727386475, -1.5296911001205444, -0.28174957633018494, -0.6049579381942749, -1.674000859260559, -1.076361894607544, -0.20840100944042206, -0.29878225922584534, 1.1848329305648804, 1.17107093334198, 1.0302329063415527, -1.1834744215011597, -0.027062498033046722, -1.5727880001068115, -2.145071506500244, 0.6131384968757629, -0.7422275543212891, 1.5217260122299194, 0.37137070298194885, -0.6486104130744934, 0.7823224663734436, -1.0281739234924316, 0.37413790822029114, -0.4798617362976074, 1.4011057615280151, 0.7166483998298645], [0.37651002407073975, 0.7918022274971008, 0.0413966178894043, -1.19354248046875, -0.9129601120948792, -0.6518461108207703, 0.5724815130233765, -0.3269459307193756, -1.0643378496170044, 0.6511316895484924, 1.1490803956985474, 0.66603022813797, 1.4525865316390991, 0.21674931049346924, 0.3881312310695648, 0.18567177653312683, 1.4146677255630493, 0.6163712739944458, 0.18712939321994781, 0.7437073588371277, 0.08468592911958694, -0.612342894077301, 0.37322738766670227, -0.04012750834226608, 0.2521413266658783, -0.10583115369081497, -0.20344917476177216, -1.6529000997543335, -0.05361541733145714, 1.5334352254867554, -0.510607898235321, -0.18818022310733795], [0.16576308012008667, 0.3003236949443817, -0.2926511764526367, 0.682685375213623, -0.8207361698150635, 0.5234125256538391, 0.44160035252571106, 0.5753074288368225, -0.22921490669250488, 0.9258748292922974, -0.33838358521461487, -1.903375506401062, 0.6268598437309265, -0.25033530592918396, 0.12212231010198593, -0.9675038456916809, -0.3174193799495697, -0.6810387969017029, -1.0096794366836548, 0.966107189655304, -0.5876514911651611, 0.4378911256790161, 0.1944054812192917, 1.2484800815582275, 0.19832772016525269, 1.868960976600647, -0.33618536591529846, 1.1328586339950562, 1.3692580461502075, -1.3383748531341553, 0.6904333829879761, 0.39337339997291565], [-0.21631862223148346, -0.6185483336448669, 0.6398777961730957, -0.6707035899162292, -0.25761523842811584, -0.6690564155578613, -0.1016574576497078, 1.4560524225234985, 0.3812352418899536, -1.0906246900558472, 0.6492493748664856, -0.2916693389415741, -0.7227415442466736, -0.8437613844871521, 0.04984736070036888, 1.281947374343872, -0.28488394618034363, -0.5124700665473938, 1.308686375617981, 0.4344533681869507, 0.8913595676422119, 0.24190931022167206, -0.23129978775978088, -1.5132054090499878, 1.0676209926605225, -0.9838197231292725, -2.083317995071411, 1.9397883415222168, 1.3231384754180908, 1.1882586479187012, 1.2283393144607544, -0.8053411841392517], [-0.6204490065574646, -1.6310443878173828, -0.5493906736373901, -0.04219723120331764, -1.8878366947174072, 0.24551787972450256, -0.592014491558075, -1.1564581394195557, 0.20020592212677002, 0.2727949023246765, 2.218872547149658, -0.29580578207969666, 0.5294348001480103, 0.4281884431838989, -0.6227466464042664, -0.608319878578186, -0.004770462401211262, 0.4199509024620056, -0.09983387589454651, 2.0064857006073, 1.589869499206543, 0.7213531732559204, -0.9077731966972351, -0.03731335699558258, -1.7329158782958984, -0.46081992983818054, 0.012765339575707912, -0.37362781167030334, -0.14583559334278107, 0.8050060868263245, 0.1228535994887352, -1.1335666179656982], [1.2218042612075806, -0.46463271975517273, 1.9884982109069824, -0.03075566329061985, 1.3520773649215698, 2.0100913047790527, -0.9648690819740295, 0.11045219004154205, 0.3525546193122864, 0.0068892063573002815, -0.5995966196060181, -1.5727835893630981, 0.801791787147522, -1.3557525873184204, 1.7164032459259033, 1.1595638990402222, 1.028662919998169, 1.4178307056427002, -0.4337369203567505, 0.21355903148651123, -0.7242283821105957, -0.3769536316394806, 0.005898190662264824, -0.3814142942428589, 1.042641520500183, -0.7877259254455566, -0.27011221647262573, 1.150987148284912, 0.5391225218772888, 2.395314931869507, -0.5908951759338379, 1.3459975719451904], [0.14275076985359192, -0.20446230471134186, -0.3870398998260498, 1.2830544710159302, 0.6265947818756104, -0.20489323139190674, 1.3500686883926392, 1.5855045318603516, -1.3026849031448364, -0.14952607452869415, -0.07866242527961731, -0.4544537365436554, 0.02025487646460533, 1.3150547742843628, 0.3363575339317322, -1.26203191280365, -0.42334073781967163, 0.7344223260879517, -1.3201628923416138, -0.4399082362651825, -0.22724203765392303, -0.8197583556175232, -1.0918329954147339, -1.0086027383804321, 0.03695504367351532, -0.039474181830883026, 0.40437403321266174, -0.6066356897354126, -0.7159108519554138, 1.2503999471664429, -1.5148472785949707, -1.2709492444992065], [-0.8230700492858887, 0.2607966959476471, -0.6131086945533752, 1.5921521186828613, -0.8199197053909302, -1.9890410900115967, 1.5835224390029907, 0.7218965291976929, -0.6107096672058105, -0.1582825928926468, 0.6832665801048279, -0.8494370579719543, 0.20466351509094238, -0.37292277812957764, 0.09387949854135513, -0.8742759823799133, 0.2607628405094147, -0.08450061827898026, -0.8520892262458801, 0.25542500615119934, 0.5140702724456787, 0.43369239568710327, 0.5054205060005188, -1.0257328748703003, 0.17855580151081085, 0.11260869354009628, 0.6065087914466858, 1.5056771039962769, -0.057517893612384796, -0.4133727252483368, 1.0724616050720215, 0.7645923495292664], [-0.4785282611846924, 0.7221960425376892, -0.9856847524642944, -1.0659077167510986, -0.9275733828544617, -0.906690239906311, -0.11849597841501236, -0.4862149655818939, -0.7532597184181213, -0.6327300667762756, 1.7318065166473389, -0.9588086009025574, 1.9596916437149048, -0.263593465089798, -0.11326908320188522, -0.4870866537094116, -0.5856509208679199, 0.5038577914237976, -1.7000200748443604, -0.33340173959732056, -0.07549065351486206, -0.17603282630443573, 1.0464794635772705, -0.0808749869465828, 0.5089319348335266, 1.530154824256897, 0.7613077759742737, -1.529282569885254, -0.8171274662017822, 0.3827311098575592, -1.0498287677764893, 1.1887961626052856], [1.055689811706543, 0.7147088646888733, -1.0334720611572266, -0.9374141693115234, 0.0029685182962566614, 2.5786538124084473, 0.19803060591220856, -0.9074723124504089, -0.6003896594047546, 1.3524750471115112, -0.18447813391685486, -1.473174810409546, 1.7509751319885254, 0.6415054798126221, -0.3565007150173187, -0.059291478246450424, -0.5323552489280701, 0.47225409746170044, -0.7422216534614563, 0.8270266056060791, -1.6768414974212646, -1.1096574068069458, -1.8666964769363403, -1.0957684516906738, 0.79938805103302, 1.0241881608963013, 0.6211854815483093, 0.45187950134277344, -2.348236083984375, 0.33115172386169434, 0.4775577783584595, 0.7506691813468933], [0.08144083619117737, 0.4042523503303528, -1.2941757440567017, 1.1921179294586182, -0.6405112147331238, -2.3719468116760254, -0.6062820553779602, 0.5005435347557068, -1.2389442920684814, 0.1563039869070053, -0.6712813377380371, 1.271289587020874, -0.815747082233429, 1.3484498262405396, 2.227555513381958, -0.4925375282764435, 0.28633904457092285, -0.738084614276886, -0.2794036865234375, 0.9080060124397278, -1.802710771560669, 1.8960555791854858, -0.0208414439111948, -0.5117122530937195, 0.8142926096916199, 0.012080882675945759, -0.5888054966926575, -0.4413314163684845, -0.10451626777648926, 1.6613966226577759, -0.18575456738471985, 1.0365333557128906], [-1.0373109579086304, 0.2754729986190796, 0.25395655632019043, 0.4903014004230499, -0.6547458171844482, 2.0263259410858154, -0.36546072363853455, 0.45009204745292664, 0.6469670534133911, 0.037122007459402084, 2.9425113201141357, -0.15324516594409943, 1.2067234516143799, -0.30400094389915466, -1.4746649265289307, 0.6404529809951782, 0.3957899510860443, -0.46086058020591736, -2.192380428314209, 1.0930243730545044, -0.9856479167938232, 0.2915787696838379, -0.4476819336414337, -1.1407166719436646, -0.9842994809150696, -0.2109118551015854, -0.2657773792743683, -0.39442095160484314, 1.7924044132232666, 1.2385897636413574, -1.2363609075546265, 1.4909515380859375], [0.4152331054210663, -0.16643062233924866, 0.8298836946487427, 0.7566864490509033, 0.06667201220989227, 0.6099730730056763, 1.4118674993515015, -1.0852203369140625, 0.1733940690755844, 1.4099398851394653, -1.2966077327728271, -0.8861716389656067, 0.28674331307411194, 0.6664829254150391, -0.686217725276947, -0.15207308530807495, -0.34093865752220154, -0.17467504739761353, 0.529147744178772, 0.44096794724464417, 0.1076851412653923, 1.6771684885025024, -0.40760746598243713, -1.6366230249404907, 0.28644293546676636, 1.783045768737793, 0.154524028301239, 0.1874178797006607, -0.23260022699832916, 0.728735625743866, -0.3726932108402252, 0.39126986265182495], [-0.696322500705719, 0.3607338070869446, 1.3798086643218994, 0.7140512466430664, 0.6277682781219482, 3.9183852672576904, -2.184296131134033, 1.9042521715164185, -1.107942819595337, -0.03674442693591118, -1.1090985536575317, 0.45795297622680664, -0.41998863220214844, 0.1181425154209137, 1.1049058437347412, 0.0033568472135812044, 0.4319967031478882, 1.0563571453094482, -1.6342402696609497, 0.47326311469078064, -1.1865580081939697, -0.06532219797372818, -0.8015745282173157, -0.3807624578475952, 0.6491849422454834, -0.0786268338561058, -0.019386624917387962, 1.5227694511413574, -1.9904972314834595, 0.44097524881362915, 0.11725419759750366, 0.2943066954612732], [-0.13049711287021637, 0.9720045924186707, -0.17410731315612793, 0.5188553929328918, -0.40267229080200195, -0.5618288516998291, -0.025621159002184868, 2.070199489593506, -0.8023906946182251, -0.2556931972503662, 0.43415367603302, -0.9513689279556274, -0.2677660882472992, -0.7325557470321655, 1.9675570726394653, -1.0632634162902832, 1.7366143465042114, -0.12109849601984024, -1.3012219667434692, 0.40438729524612427, 1.9555039405822754, 1.209994912147522, -1.739603877067566, 0.32753801345825195, -0.3837308883666992, -1.291617512702942, 0.06496061384677887, 1.6353340148925781, -1.418740153312683, 0.6856797933578491, -1.2947800159454346, -0.37598320841789246], [-0.39308464527130127, -0.1293514370918274, -0.20274324715137482, 0.3076935410499573, -0.14556345343589783, -0.47161149978637695, 1.1719067096710205, 1.5763317346572876, -0.4613734483718872, -1.1150246858596802, 1.0823707580566406, 0.2551233768463135, 0.5957826375961304, 0.9420640468597412, 1.6027218103408813, -0.9006729125976562, 0.1348053514957428, 0.367713063955307, 1.6599702835083008, -1.6484824419021606, 0.5215966701507568, 0.4950372874736786, 1.7138150930404663, -0.8099446892738342, -1.840652346611023, 0.5624649524688721, -1.2391146421432495, 0.6375381946563721, 1.1508578062057495, 1.1392241716384888, 0.3748527467250824, -2.001831531524658], [-0.47346705198287964, -1.4026142358779907, -2.3996777534484863, 1.6063408851623535, 1.5145246982574463, 0.9246509075164795, -0.08031997829675674, -0.46837836503982544, -0.19254852831363678, -3.336482286453247, -0.37661653757095337, 0.4082975685596466, 0.4636572599411011, -0.3800789415836334, 0.6226357221603394, -0.20848006010055542, -0.17414641380310059, -0.6506198644638062, -0.45448535680770874, -0.9921586513519287, 1.295856237411499, -0.17960423231124878, 0.07337164133787155, 0.3612506687641144, 0.06614634394645691, 1.5026875734329224, 1.381570816040039, -2.0727739334106445, -1.1032795906066895, 0.12621912360191345, 1.5317039489746094, -1.6558046340942383], [2.1779348850250244, -1.2270902395248413, 1.2799962759017944, 0.6300312280654907, -1.1883864402770996, -2.1153745651245117, 1.0411169528961182, -0.3795880973339081, -0.6987963318824768, -0.0961059182882309, -1.341455340385437, 0.6871215105056763, 1.202108383178711, 1.7988232374191284, -1.5754456520080566, 0.7097818851470947, 0.8681690692901611, 0.0866553783416748, -0.5954645872116089, 1.4018465280532837, 0.33558914065361023, 0.3030542731285095, -0.2963635325431824, 0.8994580507278442, -0.5284715890884399, 0.12134911864995956, -0.5965205430984497, -0.059252504259347916, 2.018650770187378, -0.667104184627533, 1.9345393180847168, 1.0297075510025024], [-2.028998613357544, 1.4298113584518433, -1.321546196937561, -0.9337649345397949, -1.0631524324417114, -0.38863107562065125, -0.37784847617149353, 1.7808117866516113, 1.3274693489074707, -1.786556363105774, 0.3171009123325348, 1.3678041696548462, 0.7095343470573425, 0.2463235855102539, 0.626122236251831, 1.204127550125122, 0.05121701583266258, 0.398975670337677, -0.14890295267105103, -2.3444933891296387, -0.593183696269989, 0.9886918067932129, -0.17596504092216492, -0.4335673749446869, -0.46919170022010803, 0.23853720724582672, 0.9800254702568054, -0.6894792914390564, -1.38188636302948, -0.49294304847717285, -0.2302212417125702, 1.1385009288787842], [-0.5042470693588257, -1.141292691230774, -0.27536019682884216, 0.6986468434333801, 0.08375092595815659, -0.0070684244856238365, -1.9876707792282104, -1.1986114978790283, -1.6590042114257812, 0.5071556568145752, 0.039880093187093735, 1.151222586631775, -0.3484480082988739, 1.0434104204177856, 0.3037513196468353, -1.5921238660812378, -0.996810257434845, 3.7791013717651367, -0.13156786561012268, 0.9054674506187439, -0.7250639200210571, -0.226526141166687, -0.5021630525588989, -0.12033899128437042, 2.554295539855957, 1.1034150123596191, -0.4946218729019165, -1.542275309562683, -0.030500901862978935, -0.7926139235496521, -0.8890292644500732, 0.5315788984298706], [-0.12963420152664185, -0.586844265460968, 0.1785040944814682, 0.38825926184654236, -1.0379160642623901, 0.2855883538722992, -1.7427494525909424, 1.2338911294937134, -1.3646577596664429, 0.4086097180843353, -0.5463712811470032, 0.2439349740743637, 2.1041624546051025, 0.36647698283195496, -1.3875876665115356, -1.308565378189087, -0.16121909022331238, 1.3690290451049805, -1.172012209892273, -1.380151629447937, -1.5580639839172363, 1.386671543121338, 0.6217940449714661, -0.20281970500946045, -0.44431984424591064, 0.6265116333961487, 0.6491190195083618, -0.12021445482969284, 0.19938085973262787, -0.29533255100250244, 0.14884178340435028, 0.29595524072647095], [-1.3368918895721436, -0.04436704143881798, -0.10005167126655579, 0.8767361640930176, 0.4808497726917267, 2.3534209728240967, 0.5586480498313904, -0.37515363097190857, -0.1480378657579422, 0.9777110815048218, 0.8320550322532654, -0.21565048396587372, 0.08802139759063721, 0.5030754208564758, 0.21895696222782135, -0.2985013425350189, -0.5154038071632385, -1.3048661947250366, 0.23726516962051392, -0.3719552159309387, 0.18124078214168549, -0.09560684114694595, -1.4862748384475708, -0.16729988157749176, 0.2695419490337372, -0.06470882147550583, -1.4265730381011963, 0.040298331528902054, 0.10906168073415756, -0.9311914443969727, 0.5454691052436829, 1.1187942028045654], [0.3519634008407593, -0.22648468613624573, 0.13629789650440216, 0.27931177616119385, 2.7935121059417725, 0.12326394021511078, 0.5286679863929749, 1.8827788829803467, 1.7317733764648438, -0.5861753225326538, 1.7148702144622803, 0.07008775323629379, -0.11713958531618118, -0.3667769730091095, -1.1406352519989014, -0.8464364409446716, -0.9047549962997437, 0.6350249648094177, -2.2058446407318115, -0.16263090074062347, -0.06730356812477112, -0.25606676936149597, 0.48480087518692017, 0.9937636256217957, -0.40998575091362, 0.22555328905582428, 0.11597155034542084, -0.47756823897361755, 0.3749046325683594, 0.9622117280960083, -0.1452471762895584, -0.1602616161108017], [0.27710598707199097, -0.9400190711021423, -1.5149353742599487, 0.0568363331258297, -0.4477923512458801, 0.5889755487442017, 1.4835588932037354, -1.7940354347229004, 0.9855930805206299, -0.3409336507320404, -1.0269560813903809, -0.767096757888794, 0.430210143327713, -0.0014526566956192255, 1.1852060556411743, 1.020851731300354, 0.6197958588600159, -0.3509250283241272, -0.47830212116241455, -0.8342782855033875, -1.533584713935852, 0.0751042366027832, 0.10573092103004456, 0.13510267436504364, 0.19104282557964325, 0.3850458562374115, 0.8590694069862366, 0.02702130377292633, -1.8008657693862915, -1.0702251195907593, 0.6481061577796936, -0.1404515951871872], [0.648683488368988, 2.4392075538635254, 0.9051742553710938, -0.0957377552986145, 0.7939082384109497, 1.3330525159835815, 0.0355178602039814, -0.400622695684433, 2.31402325630188, -0.6412191390991211, -0.32242780923843384, -1.0626850128173828, -0.14364609122276306, -0.40589040517807007, -0.7676260471343994, 0.26055389642715454, 0.7361380457878113, -0.3209870159626007, 0.5635318160057068, -0.4228055775165558, 0.23362316191196442, -0.7518231868743896, 0.1958501785993576, -0.39726123213768005, -0.28724566102027893, 0.5610564351081848, 0.17373217642307281, -0.32668906450271606, -0.16390347480773926, -1.163529872894287, 0.9954453706741333, 1.3926596641540527], [0.33598974347114563, -1.3218203783035278, 2.3151423931121826, -0.46527212858200073, 0.8314163684844971, -0.1243162602186203, -0.19456878304481506, -0.5837920904159546, -1.3174736499786377, -1.573002815246582, -0.6395111083984375, -0.6513095498085022, -0.23729582130908966, -0.7301196455955505, -0.902413547039032, -0.9200472831726074, 0.1879943311214447, 0.33332473039627075, -0.6566119194030762, -0.4238174259662628, 0.3380375802516937, 0.023445528000593185, -0.781657874584198, 0.8011723160743713, 0.29999834299087524, 1.2849986553192139, -0.4942289888858795, 0.45806989073753357, 0.23854999244213104, 0.6817111968994141, 0.38358670473098755, 0.7291676998138428], [0.5751123428344727, -1.2961173057556152, 0.5469423532485962, 0.11949153244495392, 0.0024393931962549686, 0.022511880844831467, 1.04781174659729, -0.8554612398147583, 0.9129825830459595, 0.3373655378818512, -0.5940229892730713, -0.9443503022193909, -0.054994259029626846, -2.3160693645477295, 0.488635390996933, -1.5001904964447021, -0.17129744589328766, -1.1685783863067627, 0.7965609431266785, -0.9537001848220825, 0.8447625637054443, 0.004683538340032101, 0.677280068397522, -0.11415068060159683, 0.8737913370132446, 0.46687138080596924, -0.3756011128425598, -0.014174978248775005, -0.6716160178184509, -1.143654465675354, 1.037997841835022, -1.254683494567871], [0.7345977425575256, 0.5213126540184021, -0.3669646978378296, -0.008197448216378689, -0.22253340482711792, -0.6287828683853149, 0.5870711803436279, 0.23296591639518738, -0.16619767248630524, -0.09451071172952652, 0.8277806639671326, 0.7872534394264221, 1.583095908164978, 1.2473440170288086, 1.8459737300872803, 0.10788926482200623, 0.05095871537923813, 0.6222185492515564, -2.1806204319000244, -0.7011190056800842, -0.7146785259246826, -0.4811771810054779, 0.3313964307308197, -1.2696858644485474, -0.04104907438158989, 0.9160153865814209, -0.2194770723581314, 0.786510169506073, -0.4566713869571686, 0.2524583339691162, -0.8310792446136475, 0.2179274559020996]], "dtype": "float32"}], "e": {"__nd__": [[-2.533595323562622, -8.370561599731445, 7.51695442199707, -7.001518726348877, 7.321900844573975, 2.744844913482666, 3.2918074131011963, 0.6447142958641052, 6.070894718170166, -9.035823822021484, 10.1813325881958, 1.0503132343292236, 3.5615782737731934, 2.2122550010681152, -3.0984673500061035, 5.503978729248047, -9.067561149597168, 7.466788291931152, -1.4554486274719238, -1.5706981420516968, 3.7257285118103027, -8.865129470825195, 0.5777410268783569, -5.088596343994141, 2.860205888748169, 3.921900987625122, -7.758244514465332, 6.989621162414551, 2.3612477779388428, 0.9053287506103516, 3.0593454837799072, -2.6774516105651855], [8.159156799316406, 1.730741262435913, 7.060600757598877, -7.315849304199219, -9.016390800476074, -0.7071248888969421, -5.896608352661133, 0.5663083791732788, -3.099763870239258, 12.05325984954834, -1.0255513191223145, 4.631255626678467, 1.0312777757644653, 0.6845782399177551, 4.093409538269043, 4.3932785987854, 8.256312370300293, 9.271121978759766, -1.2024874687194824, -2.9231932163238525, -1.7498390674591064, 11.037867546081543, -2.2503864765167236, -3.4739298820495605, -3.5190141201019287, -2.3786561489105225, 0.31299009919166565, 0.49265432357788086, 2.308835983276367, 8.561492919921875, -2.487664222717285, 3.707555055618286], [7.064801216125488, 1.8387967348098755, 9.684911727905273, -5.071104049682617, -0.00224484340287745, 6.3705735206604, 5.459278106689453, -5.234143257141113, 7.763073921203613, 7.495669364929199, 0.06920336186885834, -10.611557960510254, 1.6437350511550903, -10.469758033752441, -14.984874725341797, -0.21230493485927582, 4.402924060821533, 0.11157897114753723, 1.1784334182739258, 8.815872192382812, 3.063627243041992, -5.682861328125, -4.425264358520508, -0.6354502439498901, 2.3232264518737793, -3.762887716293335, -2.7328786849975586, 4.487757205963135, 0.0801776871085167, -0.2677222192287445, 4.095078468322754, 6.923137664794922], [2.4188039302825928, 5.458098888397217, -4.180490493774414, 2.943507432937622, -1.4363088607788086, -1.5764150619506836, 1.2508115768432617, -2.992187023162842, -9.053911209106445, 7.481521129608154, -3.8997349739074707, -1.9119271039962769, 7.039436340332031, 8.465299606323242, 2.9221296310424805, -3.1962714195251465, -2.1510260105133057, 1.4898027181625366, -3.9448602199554443, 6.916049480438232, -0.28032925724983215, -8.414616584777832, -4.894721031188965, -0.2495301514863968, 3.9536595344543457, -1.9064631462097168, 3.906320333480835, -2.0889267921447754, -6.210511207580566, -2.893402099609375, -10.071479797363281, 2.2894370555877686], [0.7668153643608093, -5.007785320281982, 5.087230205535889, -6.052855491638184, -6.973423004150391, 12.034687042236328, -0.5535931587219238, -10.903952598571777, 0.3181890845298767, 3.5560903549194336, 1.2468302249908447, -6.291055679321289, 7.084911823272705, -2.1358485221862793, -2.208242654800415, 6.957992076873779, 0.23896950483322144, 2.0548861026763916, -2.7382917404174805, 2.3045735359191895, 1.5342929363250732, -3.5791261196136475, -5.819503307342529, 1.148437261581421, 2.3675217628479004, 3.231626510620117, 1.7106904983520508, -2.2199172973632812, -0.45993512868881226, -6.279051780700684, -0.7923315763473511, 3.8447930812835693], [-0.14475098252296448, 2.1762778759002686, -5.0373358726501465, 3.1581718921661377, -1.8299379348754883, -0.8248749375343323, 5.266324520111084, -9.722375869750977, 5.443317890167236, -4.10330867767334, -4.711835861206055, -5.400039196014404, 3.3474338054656982, 1.9087032079696655, 4.768039226531982, 1.234521508216858, 3.352199077606201, -0.8240389227867126, 2.64349365234375, -7.748023509979248, -2.319521188735962, -0.12237881124019623, 1.8252825736999512, 0.16502223908901215, -2.334091901779175, 4.278342247009277, 4.5148115158081055, -4.257050037384033, -2.674182891845703, -6.823601245880127, -1.9872053861618042, 3.3970916271209717], [0.47596681118011475, -8.918539047241211, -0.3606007397174835, -0.31027284264564514, -11.278912544250488, -2.3687798976898193, 0.6604864001274109, -16.744741439819336, -3.4016432762145996, 9.147802352905273, -10.443790435791016, 0.4215390384197235, -7.697350978851318, 1.1349059343338013, -2.021692991256714, 0.4145582914352417, -2.4303715229034424, -7.197060585021973, 17.212276458740234, -0.22672198712825775, -2.341170072555542, 6.858561038970947, 1.2860666513442993, 4.6482086181640625, 1.0676946640014648, 3.3691141605377197, -6.028263568878174, -1.7804759740829468, 8.706343650817871, -12.010602951049805, 10.933984756469727, -1.3805227279663086], [-7.694650173187256, 3.855933666229248, 4.132513523101807, 0.07572189718484879, 9.850186347961426, 3.7106823921203613, -7.69245719909668, -3.3632616996765137, 1.536834955215454, -12.107254028320312, -2.3235366344451904, 0.4822477698326111, 1.429222583770752, 2.976257801055908, -11.469932556152344, 1.8681812286376953, -7.6633195877075195, 1.1674418449401855, -8.486919403076172, -4.381716251373291, -7.9920759201049805, -9.939382553100586, -1.8333462476730347, 7.9447407722473145, 8.05064582824707, 6.688459396362305, 6.8318095207214355, -11.787493705749512, 0.26439210772514343, -13.690082550048828, -4.569216251373291, 11.17680835723877], [0.6598744988441467, -1.9493451118469238, -5.067356586456299, 4.4125566482543945, -5.089040279388428, 3.157860279083252, 2.829092264175415, -5.786045074462891, -1.4566686153411865, 7.526754856109619, -0.3795855641365051, 1.7475383281707764, 2.305532932281494, 0.2863500714302063, -3.6306138038635254, -5.025920391082764, -0.36158180236816406, 7.189815998077393, -6.780050754547119, 4.390522480010986, 0.7319801449775696, 3.696129560470581, 2.482769250869751, -1.6313872337341309, 2.987485408782959, 8.239859580993652, 0.4122450649738312, -1.044610619544983, -2.515717029571533, -6.681038856506348, 0.7785688042640686, 4.999817848205566], [3.1703224182128906, 4.686346530914307, -3.4053866863250732, -4.44061803817749, -5.795544624328613, -8.12321662902832, 6.456226825714111, 3.23378849029541, 10.265300750732422, -0.5048468708992004, 6.486653804779053, 4.893438816070557, 3.8428280353546143, 3.1362454891204834, -1.8689466714859009, 9.130328178405762, -0.9246162176132202, -8.537833213806152, -0.5134164094924927, -3.7985341548919678, -2.020059823989868, 4.2025957107543945, 8.560830116271973, -5.050073623657227, -8.479190826416016, 0.09136758744716644, -2.6625876426696777, -2.153233766555786, 8.40046501159668, 2.4556360244750977, 3.5141282081604004, 2.368541955947876], [11.577983856201172, -4.065044403076172, 2.291221857070923, -1.3294413089752197, -0.7622255682945251, -16.50035858154297, 0.9193022847175598, -7.433854579925537, 1.9700855016708374, -6.78851842880249, -8.872211456298828, 6.436498641967773, 5.544105052947998, 5.251031398773193, -6.779487609863281, 3.2702131271362305, -2.487191677093506, 5.471226692199707, -1.8712949752807617, 2.9795329570770264, -0.9919090270996094, -1.9615235328674316, 10.183073997497559, 2.193100929260254, 0.9662812948226929, 6.430445671081543, 4.9799933433532715, -3.4826881885528564, 4.551700115203857, -8.883622169494629, 1.3550292253494263, 1.7331700325012207], [-5.0627336502075195, 1.574866771697998, 2.3024790287017822, 0.04115138575434685, 3.670825481414795, -0.9853007197380066, 1.8694679737091064, -2.966203451156616, 3.817939519882202, -6.000995635986328, 0.38888072967529297, -1.396831750869751, -1.1581755876541138, 0.16808827221393585, 4.679808139801025, 2.3888943195343018, 0.3560407757759094, -4.751713275909424, 3.9739608764648438, -1.537315845489502, 12.418024063110352, -3.8111557960510254, -4.87558650970459, 6.684357166290283, 1.8514457941055298, 2.7011656761169434, -0.6417508125305176, -1.2876455783843994, -0.679366946220398, -6.4816436767578125, 2.8749313354492188, -0.17949101328849792], [-1.9513291120529175, -7.300896644592285, 9.000636100769043, 1.739854097366333, 3.3437135219573975, 3.382322072982788, -9.115102767944336, -5.189011096954346, 5.722046375274658, 4.411027431488037, -12.547587394714355, -0.18555748462677002, -9.441442489624023, -5.72893762588501, -16.567020416259766, 0.12526345252990723, -10.021798133850098, -0.768402636051178, 12.284677505493164, -3.79264235496521, -1.526448130607605, 4.356308937072754, 10.334222793579102, 8.817408561706543, 2.865525960922241, -1.1543879508972168, -2.035813570022583, -4.048768997192383, 14.52706241607666, -15.760061264038086, 1.9624109268188477, 3.4221038818359375], [7.508859634399414, -1.282950520515442, 6.654325008392334, -7.1595892906188965, 0.9125134348869324, 1.0480624437332153, 4.854494571685791, -0.14061695337295532, -5.361798286437988, 0.9381013512611389, -8.07870101928711, 1.0730476379394531, 4.118419170379639, 4.518449306488037, 6.024967193603516, 7.6236796379089355, 1.4742567539215088, 1.4583827257156372, -2.272491693496704, -7.4222636222839355, -2.898115873336792, -0.6412292718887329, -2.273930072784424, 0.9311953783035278, 1.5025687217712402, 8.498736381530762, 2.6075782775878906, 5.208393573760986, -3.7741434574127197, -0.34788742661476135, 8.138691902160645, 0.06551932543516159], [-1.406252384185791, -1.7243337631225586, 5.09182071685791, 4.908995151519775, 4.128349304199219, 9.908679962158203, -11.23190689086914, -2.052394390106201, -2.764599084854126, 0.08289187401533127, -5.9485764503479, 0.46232840418815613, -5.891660213470459, 1.082992434501648, -8.179744720458984, -0.945802628993988, -1.4519176483154297, 0.885837733745575, -2.5256054401397705, 5.733715534210205, -3.840315580368042, -3.3655288219451904, -9.047632217407227, 7.3718953132629395, 4.363434791564941, -1.4189212322235107, -0.7968438267707825, -0.5308981537818909, 3.6046414375305176, -7.801179885864258, 3.5458033084869385, 5.8999810218811035], [5.0811872482299805, -6.861325263977051, -2.133878707885742, 1.789038062095642, 7.68359899520874, 7.840307712554932, 6.339593887329102, -7.5916032791137695, 1.5247066020965576, -2.585554838180542, 1.7281606197357178, -7.374966144561768, -1.58769953250885, -6.618334770202637, -6.590620040893555, 1.7970656156539917, -2.254805564880371, -11.645221710205078, -3.390948534011841, 5.764313220977783, 9.516157150268555, -8.580402374267578, -3.191549062728882, 0.5745881795883179, -3.366013765335083, -3.9124162197113037, 3.153111457824707, -0.8889878988265991, -2.4274942874908447, 5.341311454772949, -0.13743042945861816, -8.569811820983887], [0.879542350769043, 7.960021018981934, 0.14867618680000305, -4.018211364746094, -5.1156907081604, 5.167675971984863, 5.136295318603516, 13.471631050109863, 0.541578471660614, 7.965190887451172, 6.328644752502441, -4.810907363891602, -1.3426406383514404, 0.39243772625923157, 2.752990961074829, -1.4623152017593384, 2.7845687866210938, 2.381842613220215, -2.1520423889160156, -0.09869423508644104, -3.0696916580200195, 4.519235134124756, -4.154654502868652, -8.091102600097656, -3.732816696166992, -2.0042002201080322, -5.6115546226501465, 8.32912826538086, -0.48885437846183777, 9.22901725769043, -4.646823406219482, -1.211910605430603], [5.166608810424805, -16.086698532104492, 8.526859283447266, -1.758105993270874, -3.2446365356445312, -9.272904396057129, 10.823782920837402, -8.848783493041992, 3.7998852729797363, -2.5049965381622314, 4.601747035980225, -0.6855384111404419, -0.20969192683696747, 1.418192744255066, -14.138358116149902, 7.7279534339904785, -4.716106414794922, -6.882861137390137, 10.264616966247559, 6.033698081970215, 7.273849964141846, 3.8896644115448, -0.7106621265411377, -8.09261703491211, -4.143062591552734, 2.18501353263855, -9.933013916015625, 6.874525547027588, 19.326562881469727, 3.075403928756714, 12.20291805267334, -3.3590073585510254], [-5.371365547180176, 5.413360118865967, 3.522225856781006, -5.688272476196289, 6.049983024597168, 4.414454460144043, -10.228175163269043, 3.917107105255127, 3.0475759506225586, -4.3661017417907715, -6.878306865692139, 11.879059791564941, -2.8760366439819336, -7.088993072509766, -0.5020820498466492, 9.344593048095703, 0.044459450989961624, 1.4629756212234497, 8.440589904785156, -7.668574810028076, -6.634181022644043, -2.1389079093933105, 7.215208053588867, -4.21260404586792, 6.327175617218018, -4.142203330993652, 5.267539024353027, -3.5245370864868164, -4.062597751617432, 0.05013220012187958, -2.2362587451934814, 1.2467973232269287], [-0.8692923188209534, 10.517797470092773, -4.946080207824707, 2.058734893798828, 6.136961936950684, 11.980156898498535, 12.957731246948242, 2.442856550216675, 5.917314529418945, 0.18185558915138245, -3.352327346801758, -7.906561374664307, -0.7090020775794983, 3.916409969329834, 8.714839935302734, 0.7672398686408997, 1.5135029554367065, -8.52306079864502, -0.39918678998947144, -7.404376029968262, 2.04582142829895, 1.8086602687835693, -7.282853603363037, -4.858778476715088, -7.371872425079346, 10.772773742675781, 3.1059651374816895, 12.11624526977539, -12.3577880859375, -1.2209383249282837, 4.530637264251709, -1.0297526121139526], [2.554324150085449, -9.379992485046387, -1.2276489734649658, -1.1597648859024048, -3.972283363342285, -5.8126983642578125, 0.005903014447540045, -10.758452415466309, 6.069732666015625, -5.446812152862549, -0.6880941987037659, 7.639486312866211, -0.6662507057189941, 0.7616754174232483, -8.9649019241333, 4.345546245574951, -2.267723321914673, -1.948448657989502, 11.103263854980469, -1.8328851461410522, 0.4535585641860962, -0.3110460937023163, 2.509793519973755, -5.466958522796631, -4.186084270477295, -3.8307535648345947, -1.426833987236023, -8.550689697265625, 7.877692222595215, -1.1229318380355835, 4.179304122924805, -3.9700398445129395], [2.710768461227417, -4.091970920562744, -2.3406763076782227, 8.066943168640137, 4.233515739440918, 10.7535400390625, -1.6547678709030151, 1.400099754333496, -3.556098699569702, 11.242460250854492, -10.844551086425781, 2.9717249870300293, -0.8801081776618958, 10.182319641113281, -3.2019495964050293, 0.7761869430541992, -7.771986484527588, 0.24292944371700287, -8.209988594055176, 9.3344087600708, -16.83338165283203, 0.09324721992015839, -12.293729782104492, -2.752941846847534, 10.243857383728027, 2.5811848640441895, -2.315556526184082, 0.9956254959106445, -2.6832895278930664, 1.2953603267669678, 0.42264071106910706, 7.927245616912842], [-3.369673013687134, -6.203936576843262, -0.4642447531223297, -0.9574182033538818, 0.9512149095535278, 5.218790531158447, -8.807796478271484, -4.773809909820557, -5.636295318603516, -9.33514404296875, 2.8288607597351074, -0.46432390809059143, 1.0007485151290894, -5.604063510894775, 2.5176758766174316, -1.8774082660675049, 2.5806994438171387, -0.9103543758392334, 0.7460722923278809, 0.4198005795478821, 5.178545951843262, -3.4560883045196533, -7.368722915649414, 4.722656726837158, -0.3072090446949005, -0.7137930393218994, 3.7613792419433594, -3.494201183319092, -4.114655017852783, 1.7806316614151, 5.123913288116455, -3.34375262260437], [-8.586712837219238, 1.1196905374526978, 8.964815139770508, 7.168790340423584, 11.698602676391602, 0.28084149956703186, 11.728802680969238, 13.667677879333496, 7.021554946899414, 1.9156298637390137, 11.716320991516113, -3.7903926372528076, -2.966505527496338, -6.749140739440918, -17.57996368408203, -9.58495044708252, -8.714153289794922, -16.352249145507812, 8.612810134887695, 6.294216156005859, 7.650296688079834, -10.448542594909668, -0.15147222578525543, -12.133456230163574, -4.234432697296143, -14.987152099609375, -12.070006370544434, 9.882338523864746, 8.351726531982422, 8.784576416015625, -6.430741786956787, -6.140964984893799], [-4.090749263763428, 7.592512607574463, -8.157830238342285, -3.6526219844818115, -10.630337715148926, 3.171823740005493, 0.048768144100904465, -5.806209087371826, 0.9183510541915894, 10.023158073425293, 17.528057098388672, -3.3768138885498047, 5.464462757110596, 4.363640308380127, 2.385725736618042, 2.2054665088653564, 2.758831024169922, -1.7871066331863403, -0.6411654949188232, 6.998543739318848, 12.043171882629395, -1.1436166763305664, -6.469069004058838, -1.460681676864624, -8.326128005981445, -6.902095794677734, 2.9281210899353027, -10.642675399780273, -0.871487557888031, 4.7617716789245605, -12.7179594039917, -2.3257863521575928], [1.8729108572006226, 10.112015724182129, -1.303288459777832, -3.9234745502471924, -0.6056604385375977, 1.8993308544158936, -3.320180892944336, 6.005823135375977, 6.222311973571777, 12.1419677734375, 0.6697127819061279, 0.18715864419937134, -5.117480754852295, -2.2133615016937256, -1.3847362995147705, 2.8548898696899414, -0.30042749643325806, 4.44716215133667, -1.142271637916565, -3.2832508087158203, -4.917055606842041, 2.3240790367126465, 4.334143161773682, 1.3403961658477783, 0.9080225229263306, -3.17925763130188, -3.5208327770233154, 3.668236017227173, 5.515639781951904, -1.793349027633667, -4.9459757804870605, 6.00379753112793], [6.387165546417236, -1.9721299409866333, 8.254911422729492, -4.881232738494873, 0.9198626279830933, -8.904115676879883, 6.131321907043457, -4.597545623779297, -0.9563657641410828, 6.167486190795898, 4.069693088531494, 2.624690294265747, 1.1932957172393799, 2.027714252471924, 5.450107574462891, -1.2744462490081787, 1.2730118036270142, 7.119935512542725, 4.120538711547852, 6.479176998138428, 11.422595024108887, 1.559857964515686, -3.3945562839508057, -3.878081798553467, 0.7512096762657166, 4.73308801651001, -2.1437342166900635, -0.7815443277359009, 2.2708396911621094, 9.96528434753418, -7.826229572296143, -0.9745193719863892], [-1.6237748861312866, -7.60405969619751, 0.19970647990703583, -0.5087507367134094, 4.826601028442383, 1.5136523246765137, -5.274016857147217, -5.863996982574463, 7.178750991821289, -6.125880718231201, -0.6940791606903076, -10.633038520812988, -7.939729690551758, -18.082965850830078, 1.2318545579910278, -4.96449613571167, 0.17658773064613342, -5.7836809158325195, 5.79568338394165, 1.680527687072754, 2.084397077560425, 0.31174084544181824, -1.8607293367385864, 4.924979209899902, 3.6795730590820312, -2.8372678756713867, 1.6107991933822632, 2.3529114723205566, -3.151639461517334, 2.15838623046875, 5.31604528427124, -4.244705677032471], [1.1854926347732544, -8.171762466430664, -5.248400688171387, -4.209686279296875, -8.438749313354492, 2.1583478450775146, -8.7852783203125, -0.8699337244033813, -4.445110321044922, -9.348916053771973, -13.549808502197266, 5.2955002784729, -3.1136972904205322, 1.9344912767410278, 3.8658690452575684, 2.712700843811035, -2.72422194480896, 12.81924819946289, 13.06329345703125, -8.976563453674316, 6.422300338745117, 10.80787467956543, 3.8899717330932617, -8.809673309326172, 2.25411057472229, 6.841041564941406, 3.7110118865966797, -2.372356414794922, -11.134721755981445, -1.7561954259872437, 8.251879692077637, -14.04746150970459], [10.630960464477539, -1.9394477605819702, 8.195157051086426, -3.501291513442993, -6.952609539031982, -12.204654693603516, 0.9981362819671631, 9.452773094177246, 0.29573264718055725, 2.1888797283172607, -11.44509506225586, 6.742023944854736, 1.8329299688339233, -4.43934965133667, -1.7322416305541992, -3.984924077987671, 6.3029913902282715, 5.974475860595703, 5.4627604484558105, -7.771562576293945, -1.8443671464920044, 12.855623245239258, 10.739666938781738, -2.4009900093078613, -6.361306190490723, 4.062020778656006, -2.9566845893859863, 15.662925720214844, 6.051766872406006, -3.923391103744507, 9.319189071655273, -3.202737331390381], [-8.437012672424316, -5.389392375946045, -6.979830265045166, 1.8885122537612915, -5.532896041870117, -10.642857551574707, 7.609303951263428, -2.7097902297973633, -2.6384177207946777, -6.060884952545166, 10.756690979003906, 4.32171106338501, 5.776353359222412, 4.272089004516602, -2.205569267272949, -3.366486072540283, 1.8768831491470337, -13.8309965133667, 0.5199494957923889, -2.948758125305176, 2.612473726272583, 5.262454032897949, -4.011954307556152, 3.0548624992370605, -11.507224082946777, 1.4972301721572876, -2.056215286254883, -1.8873752355575562, 5.071372985839844, -0.5496343374252319, 2.874387264251709, -4.153058052062988], [-3.411555528640747, 5.012208938598633, -1.6764076948165894, -1.2896132469177246, 0.06082262098789215, 3.6577370166778564, 2.828338623046875, -2.6399056911468506, 1.693360686302185, -0.8978038430213928, -8.158909797668457, 2.1761586666107178, -2.459916591644287, -1.8056552410125732, -4.39533805847168, -0.7321692705154419, -4.31813383102417, -3.0183210372924805, 11.046651840209961, -5.878939628601074, 1.0615267753601074, -4.669523239135742, 3.901679039001465, 0.8131616115570068, 3.825807809829712, 3.9782674312591553, -3.531358242034912, 3.1362013816833496, -3.7503621578216553, -16.378568649291992, 10.790699005126953, 6.810008525848389]], "dtype": "float32"}}, {"a": [{"__nd__": [[1.0237752199172974, 0.5945010185241699, -0.3072544038295746, -0.30930858850479126, -0.8363120555877686, -0.8083183765411377, -0.3257502615451813, -0.547157883644104, -0.11642728000879288, -0.589547336101532, -0.1102934181690216, -0.4164901673793793, 0.8339800238609314, 0.8978884220123291, 0.06286247819662094, 0.5030118823051453, -0.19270803034305573, -1.1198511123657227, -1.1229859590530396, 0.7248769998550415, -2.8197827339172363, 0.44419100880622864, -0.19625209271907806, 0.8939945697784424, -1.314199447631836, -0.7387915253639221, 0.8654393553733826, 0.2287784069776535, 0.2870989441871643, -1.0603680610656738, -1.081944465637207, -0.7856231331825256, 0.7991149425506592, 1.6607530117034912, 1.2603893280029297, -1.6003211736679077, -1.4020899534225464], [-0.0016882854979485273, -1.1641441583633423, -0.10025809705257416, -0.6490828394889832, -0.5426715016365051, 1.6303601264953613, -1.6632086038589478, 0.4525182545185089, -1.4852497577667236, 0.11354725062847137, 0.36517539620399475, 1.1128849983215332, 2.3341569900512695, -0.5997141003608704, -0.6692060828208923, 0.18756555020809174, 0.06034012883901596, 0.4989599287509918, 0.8403404355049133, -0.9702734351158142, -0.3715229034423828, -0.9816334843635559, 0.24104665219783783, 0.659694492816925, -0.0892735943198204, -0.46862590312957764, -0.2827285826206207, 0.20763476192951202, 0.3180661201477051, 0.5803442597389221, -0.5726941823959351, -1.2718437910079956, -0.12083149701356888, 1.0009145736694336, 0.15217755734920502, -0.17498913407325745, -1.5795775651931763], [0.04138745367527008, -0.9720706343650818, 0.27780845761299133, -1.4179702997207642, 0.4021984338760376, 0.6489606499671936, -0.6109597682952881, 1.4459573030471802, -0.25847411155700684, -0.042523615062236786, -0.6779012680053711, -1.3213839530944824, 0.4617949426174164, -0.2215210646390915, 1.5015225410461426, -0.4689289331436157, 1.014155626296997, 1.411275029182434, 0.9076058864593506, -1.8103768825531006, -0.08894668519496918, -0.8029366731643677, -1.8090319633483887, -0.32868504524230957, -0.4044038653373718, 0.04692377150058746, -1.1231223344802856, 0.8577334880828857, -0.6683942675590515, 0.939924955368042, -0.5921103358268738, 0.6999707221984863, -0.2963358163833618, 0.23915156722068787, 1.2808408737182617, 0.591244101524353, -0.18110325932502747], [-0.13140186667442322, 2.116349458694458, 1.2291793823242188, 1.1852829456329346, 1.4121789932250977, -0.8598136305809021, -0.7463093400001526, -0.6298437118530273, -0.42724910378456116, 1.3351467847824097, -0.28664126992225647, 0.8251820206642151, 0.1010335311293602, 2.8886594772338867, -0.8569555282592773, 0.9672772884368896, -2.3267574310302734, 0.1403270810842514, -0.7989546656608582, -2.729013681411743, 0.6285947561264038, -0.12044422328472137, -0.7381158471107483, -1.6689624786376953, 0.621667206287384, 1.297038197517395, 0.16317707300186157, -1.0559366941452026, -1.0392190217971802, -2.4964747428894043, 1.5237807035446167, 0.476166307926178, 1.5544123649597168, 1.345981478691101, -1.8337782621383667, -0.3324134647846222, -1.5040969848632812], [-0.5403783321380615, 2.722627878189087, -0.7618060111999512, -1.87326979637146, 0.5704711079597473, -1.948478102684021, 0.03565540909767151, -0.21430076658725739, -0.4259907603263855, -0.5055411458015442, -0.45917847752571106, -0.35218021273612976, 1.1691267490386963, 0.15795426070690155, 1.272132396697998, -1.7077538967132568, 0.8576015830039978, 0.31802722811698914, -0.7590683102607727, -1.458323359489441, 0.4898281693458557, 0.4511004090309143, 0.029753144830465317, 2.199838876724243, 0.5175981521606445, 1.0780996084213257, -1.032141923904419, -0.42258378863334656, 0.27793681621551514, 0.5730780363082886, 0.5040512084960938, 0.18954762816429138, 1.4087474346160889, -0.013245789334177971, 0.18427211046218872, 0.8313836455345154, -0.812201201915741], [-1.010465383529663, -2.8567559719085693, 0.8760930895805359, -0.7759642004966736, -1.7425940036773682, -1.4447479248046875, -2.961090326309204, 1.0777047872543335, 1.0142871141433716, 0.7549474835395813, 0.5130964517593384, -0.36198464035987854, -0.5586236715316772, -0.21082386374473572, 0.5819976329803467, 0.232695534825325, 0.37837567925453186, 0.7131022214889526, 0.16819536685943604, 0.9253333806991577, -0.8521913886070251, 0.5448697209358215, -1.1920416355133057, -0.0252251997590065, -1.3726547956466675, 0.8603202104568481, -0.445247620344162, 0.29566967487335205, -1.6671526432037354, 0.8226589560508728, -0.5567352175712585, 0.21852444112300873, -0.19027508795261383, 0.21519337594509125, -0.6298646330833435, -1.4495905637741089, 0.2586781084537506], [-0.3634839951992035, 0.10286661237478256, 1.431322455406189, 0.32515180110931396, 0.3548676669597626, 1.2149488925933838, -1.4278048276901245, 0.9531749486923218, 1.425706386566162, -0.06894051283597946, -1.4209693670272827, 0.7003417611122131, 0.5485111474990845, 0.3369942903518677, -0.4945686161518097, -0.10588124394416809, -0.16132168471813202, 0.1152266412973404, 0.23816032707691193, 0.7226079702377319, 0.06895096600055695, 1.6747206449508667, -0.5748392343521118, 1.3672887086868286, -0.3375833034515381, 1.7907631397247314, -1.4918410778045654, -0.6688299775123596, 1.8990037441253662, -0.7870087623596191, 0.06031924858689308, 0.6606612801551819, -1.2654310464859009, 1.2142356634140015, -1.4573969841003418, 1.9851465225219727, -0.5066586136817932], [0.7421615719795227, -0.7390286922454834, -0.38455167412757874, -0.3267020881175995, -0.71059650182724, -0.10045608878135681, -0.7159728407859802, -0.7409875988960266, 0.7369668483734131, -1.681382656097412, 0.2745152711868286, -0.4458812475204468, 0.688025176525116, 0.7421532869338989, -0.8122292160987854, -0.8176742196083069, 0.05128759518265724, -1.0884100198745728, -1.1159437894821167, 0.6655688881874084, -1.1330054998397827, 0.8617369532585144, 2.6553332805633545, 0.8555194139480591, 0.21148543059825897, -0.4897020161151886, 1.1061266660690308, 1.9796468019485474, -0.8116727471351624, 0.39175939559936523, 0.04204297438263893, -0.3331628143787384, -0.08414710313081741, -1.6042628288269043, -2.4836905002593994, 0.11197496950626373, -1.1355136632919312], [0.2366786152124405, 2.6709651947021484, 0.20562683045864105, 1.0171735286712646, -0.1792122721672058, 0.9856081604957581, -0.43428710103034973, -0.7223528623580933, 0.7035990953445435, -0.17362821102142334, 0.44448456168174744, -0.2884550988674164, -1.1963611841201782, 0.7478294968605042, 2.1725921630859375, 0.02093997597694397, -0.7916016578674316, 0.472099244594574, 1.676024317741394, 0.8531702160835266, -0.18036270141601562, -0.7835910320281982, -0.40144938230514526, -1.45656156539917, -1.9199975728988647, 0.863944411277771, 0.8925241231918335, 1.0154534578323364, -0.5878879427909851, 0.689245879650116, -0.19052405655384064, 0.7536835074424744, 0.4254568815231323, 1.2020432949066162, 1.4986828565597534, -0.2512489855289459, -1.1575042009353638], [-0.07366524636745453, -0.8493350148200989, -1.4260344505310059, -0.047900427132844925, 0.17250871658325195, 1.1636464595794678, -0.5568313002586365, -1.7624365091323853, -0.2793312966823578, -1.1050505638122559, 2.1468448638916016, 1.5248945951461792, 0.9497634172439575, 2.352452278137207, 0.15913793444633484, -0.03431005775928497, -1.9577642679214478, -0.26333653926849365, 1.4546105861663818, -3.0258171558380127, -0.6024826169013977, 0.7393698692321777, -0.38104531168937683, -0.2590882480144501, -1.5296838283538818, -0.6073002815246582, -0.28554391860961914, 0.6815552115440369, -0.4318970739841461, -0.24911044538021088, -0.29079216718673706, 1.1466153860092163, -1.3155585527420044, -0.9782600998878479, 0.774873673915863, -2.9669406414031982, -0.8195570111274719], [-2.3059346675872803, 1.3470392227172852, -0.3623751103878021, 0.6484211683273315, 1.1523919105529785, -0.14312027394771576, -0.6084389686584473, 1.041504144668579, 0.5542235970497131, 0.14008140563964844, -1.2056987285614014, 1.0075947046279907, -0.9905200600624084, 0.2687966525554657, -0.1921117752790451, 1.6689040660858154, -0.3507483899593353, 1.3059322834014893, 0.8429322838783264, -1.257264494895935, 0.33122143149375916, 1.3190500736236572, 0.4727958142757416, -1.6478127241134644, 1.0818780660629272, -1.999804973602295, -0.23687459528446198, 0.8331093192100525, -0.748036801815033, -0.17270727455615997, 0.819098174571991, 0.8604378700256348, -0.9173883199691772, -0.607940673828125, -0.5219482779502869, -0.775118887424469, -0.585032045841217], [0.8608348965644836, -0.02913603000342846, -0.937484622001648, -0.4100996255874634, -0.21906858682632446, -0.528373658657074, -0.925130307674408, 0.26955023407936096, -0.9840216040611267, -0.1972929835319519, -0.22978255152702332, 1.1028547286987305, -1.652767300605774, 0.5660891532897949, -0.8643879890441895, -1.1954611539840698, -0.6570746302604675, 0.1982412487268448, -0.6081705689430237, 0.5462245941162109, 0.3288438320159912, -0.6280807852745056, -1.1356594562530518, -0.03637584298849106, -0.6173104047775269, 0.10934843868017197, -0.35910671949386597, -0.7419194579124451, 0.18806076049804688, 1.5727545022964478, 0.7911815643310547, -0.3847307562828064, 1.3977528810501099, -0.38163188099861145, -1.1326929330825806, -1.3063143491744995, 0.001453456119634211], [0.6679595708847046, -0.6132826209068298, 0.44579949975013733, 0.7336581349372864, -0.3791017532348633, -0.04782513529062271, 1.0863971710205078, 1.4997167587280273, -0.9682043194770813, 0.8230229020118713, -0.8225904107093811, 1.2136731147766113, 1.7458505630493164, -1.1059707403182983, -0.42415332794189453, 1.643235206604004, -0.08409242331981659, 1.0331462621688843, -0.6883236765861511, 0.8630988597869873, 1.788603663444519, -0.338972806930542, 0.28027647733688354, 0.678024411201477, -0.12382065504789352, -1.3714776039123535, 0.840811014175415, -0.9795968532562256, -0.6490815877914429, 1.3709465265274048, 0.04517758637666702, 0.2859985828399658, -1.4571027755737305, 0.5233405828475952, -0.5884236097335815, -1.1047883033752441, -0.5267527103424072], [0.9949674606323242, 0.13664308190345764, -0.4196086525917053, -0.8938271999359131, 0.3088594675064087, 0.15223170816898346, 0.02397911436855793, 0.7628807425498962, 0.8739233016967773, -0.0933605283498764, 0.4834915101528168, -1.2827975749969482, -0.4937779903411865, 0.7316951155662537, 0.9367944598197937, 1.3918006420135498, -0.2926598787307739, -1.2326488494873047, 1.7847281694412231, 0.4589765667915344, -0.5719600915908813, -1.1308938264846802, -0.16048863530158997, -1.4294087886810303, -0.5867310762405396, 0.08061428368091583, 0.06634187698364258, 1.919094204902649, -1.303678274154663, -0.5244680047035217, 0.2045537680387497, 2.8201239109039307, 0.9755683541297913, -0.33107879757881165, 1.0324398279190063, -1.65321683883667, 0.3611906170845032], [-1.131998896598816, 0.6040505170822144, -0.45427292585372925, -1.042114496231079, 0.4444129168987274, 1.1074689626693726, 0.7007666230201721, 0.43848374485969543, 0.3317723870277405, 0.5522021055221558, 0.10361117869615555, 1.1067856550216675, -0.7787473201751709, 0.7856126427650452, 0.0229493398219347, -0.16653665900230408, -0.49889567494392395, 1.5986770391464233, 0.7082956433296204, -0.7114042639732361, 0.42009323835372925, 2.4167439937591553, -1.7621641159057617, -0.8899449110031128, 1.391877293586731, -0.4584091007709503, -0.1330864578485489, 0.32471776008605957, -0.0024578005541116, -0.24993190169334412, -0.3423718214035034, 1.5332454442977905, 1.452185034751892, 1.6971631050109863, -1.0172760486602783, 0.47397586703300476, 1.359468936920166], [0.18109898269176483, 0.5105213522911072, 1.5080116987228394, 0.7295655012130737, -0.8092129230499268, -0.3955286145210266, -0.029263658449053764, 0.7750540375709534, 0.1832043081521988, -0.6949602961540222, 1.2862628698349, -1.0990084409713745, 1.1321678161621094, 0.7361858487129211, -0.3041207194328308, -0.13395722210407257, -0.2527775764465332, -0.7223576307296753, 1.1428605318069458, -0.3432477116584778, -0.979460597038269, 0.6225562691688538, -0.6115768551826477, -1.3210198879241943, -0.8383532166481018, -0.16868045926094055, 0.2620082199573517, -1.2075198888778687, -1.1186933517456055, -0.6534400582313538, -0.4502195119857788, 0.02201707847416401, -0.26304033398628235, -1.1786543130874634, 1.104722023010254, 0.1825285702943802, -1.676153540611267], [-1.2454440593719482, 1.2044968605041504, 1.187757968902588, 1.1589972972869873, 0.03141119331121445, 0.7283297777175903, 0.3683262765407562, 0.304063618183136, -0.02483234740793705, -1.3084367513656616, -0.9985212683677673, -1.180527925491333, -1.9261367321014404, -1.2494187355041504, 0.587428867816925, 0.22283819317817688, 0.743296205997467, 0.2961464524269104, 1.3329123258590698, 0.053287118673324585, -0.2598840296268463, -1.241312861442566, 2.400752067565918, 2.7180778980255127, -1.3328851461410522, 1.378624677658081, 1.2084678411483765, 0.4348710775375366, 1.0995076894760132, -0.4200840890407562, 0.4245118796825409, 1.27826726436615, 1.1976337432861328, 1.2296119928359985, 1.2836463451385498, 1.8827959299087524, 0.010786143131554127], [-0.37694284319877625, 0.45467835664749146, -0.27697429060935974, 2.7993762493133545, 0.20777243375778198, -1.224755048751831, 1.006759524345398, -0.9925451278686523, -0.7107205986976624, 0.9693577289581299, -0.11684436351060867, -2.256528854370117, 0.7383409738540649, 0.5887042284011841, 0.4479561448097229, -0.2429095059633255, -1.1543997526168823, -0.38534706830978394, 0.24431729316711426, -0.050587575882673264, -0.848055899143219, 0.5099051594734192, -0.5938544869422913, 0.26880568265914917, 0.6090711951255798, -2.891965627670288, 0.009783850982785225, -0.6214701533317566, -0.6423771381378174, 0.21553091704845428, -0.29714325070381165, 1.7284613847732544, -1.4095312356948853, 0.6583845019340515, 1.3372063636779785, -0.47616705298423767, -0.7351449728012085], [1.805565595626831, 0.6337560415267944, -0.1141677051782608, 0.6327128410339355, 0.978813111782074, 0.5077072381973267, -0.6674744486808777, -0.21162889897823334, -0.0630151778459549, -0.013282053172588348, 0.7871347069740295, 0.4643421471118927, -0.6139124631881714, 2.581857681274414, -1.2570666074752808, 0.03175345063209534, 0.28077107667922974, 0.017679227516055107, -0.6274768710136414, 0.37754884362220764, 2.1579182147979736, 0.15039004385471344, -0.5345127582550049, 1.6386405229568481, -0.5921757817268372, 0.9544874429702759, -0.26679345965385437, 1.7195971012115479, -0.9554867744445801, -1.324080467224121, 0.18107719719409943, -0.0028808307833969593, -0.5324548482894897, -0.6793315410614014, 1.0903804302215576, 0.6829552054405212, -0.446806401014328], [1.8793561458587646, -0.6539785861968994, 1.085961937904358, 0.3546767830848694, 0.2529897093772888, 2.366553783416748, 0.3284016251564026, 0.9592840075492859, -0.9197303652763367, -1.8478463888168335, -0.3571718633174896, -0.10990083962678909, -0.7069055438041687, -1.433793067932129, -0.12252101302146912, -2.116328001022339, -0.9957646727561951, -0.04756217449903488, 0.1263357251882553, 0.3494085669517517, -0.374063104391098, 1.181265115737915, -0.2834167182445526, 0.03872155770659447, 0.7123575806617737, -0.35018131136894226, 1.437495470046997, 0.6687464714050293, 0.6944688558578491, 0.8601076006889343, 0.040295928716659546, 0.8122403621673584, 0.7109807729721069, 0.22823607921600342, 1.706993579864502, 0.7222982048988342, -1.0755343437194824], [0.6297320127487183, 1.119620442390442, 0.7000522613525391, 0.6011181473731995, -0.11441469937562943, 0.025537364184856415, -0.37765833735466003, -0.022148167714476585, -0.4176609218120575, 0.762273371219635, 0.922782301902771, 0.8719603419303894, -0.6171621084213257, 0.752974808216095, -1.2417609691619873, 1.5755128860473633, -1.0733306407928467, -0.965216875076294, 0.38136082887649536, -0.7597061991691589, -0.33464911580085754, -2.720878839492798, 1.3795253038406372, -0.3680135905742645, -1.2678593397140503, 0.03622133284807205, 0.09203549474477768, -1.1105878353118896, 0.5588307976722717, 0.11753208935260773, -0.5350754261016846, -0.12197257578372955, 0.8314908146858215, 0.8709355592727661, 1.3723198175430298, 1.0022156238555908, -0.9625720977783203], [0.30674582719802856, 1.1228506565093994, -1.3792352676391602, -0.687140166759491, -0.4056462347507477, 0.5887314677238464, -1.1139987707138062, 0.30268150568008423, 0.8811928033828735, 1.2557162046432495, 0.6228813529014587, -1.5680114030838013, -1.4803121089935303, 0.7873981595039368, -0.1709185391664505, 0.570935070514679, 0.5571669936180115, 0.9171921014785767, 2.244739294052124, 0.004834097344428301, -1.1977897882461548, 0.9999694228172302, -0.7311750650405884, -0.8143358826637268, 0.9538437128067017, 1.7784677743911743, 0.4658837616443634, -0.7193801999092102, 0.451722651720047, 0.04491342604160309, 1.1658604145050049, 0.33438488841056824, -0.7022624015808105, 1.663081169128418, -0.3266095519065857, 1.3336868286132812, 0.5323076844215393], [1.7976323366165161, -0.4195083975791931, -0.6644594669342041, 0.8698704242706299, 1.192662239074707, 0.8533955216407776, -0.6139461994171143, 1.1502481698989868, -0.2047649323940277, 1.9571144580841064, -2.6535050868988037, 0.6100184321403503, -0.07751025259494781, 1.7254117727279663, 1.1028136014938354, 1.1153961420059204, 0.4413842558860779, 1.0491975545883179, -1.0980465412139893, -1.2577059268951416, 1.018409013748169, 0.7518269419670105, -0.25128334760665894, 0.49180981516838074, 1.6358747482299805, 1.0644439458847046, 0.32483407855033875, -0.2526038587093353, 1.2807382345199585, 0.23682619631290436, -1.3375805616378784, 0.26391929388046265, -0.5369424819946289, 0.5434032678604126, -0.21660549938678741, -0.020225385203957558, 1.0290149450302124], [0.11699329316616058, -0.6249147057533264, -0.40028709173202515, -0.10229307413101196, 0.7581589221954346, -0.7469770908355713, 1.721697211265564, 1.8480284214019775, -0.9401216506958008, 0.9042399525642395, 0.8545551300048828, -0.24956251680850983, 0.513437032699585, -1.0206059217453003, 1.9651525020599365, 0.711950957775116, 0.5838162302970886, -0.006980011239647865, -1.5798859596252441, -0.4071248769760132, 0.12659816443920135, -0.4914264678955078, 0.3937738239765167, 1.5410664081573486, -1.2849253416061401, -1.8277291059494019, 1.4178659915924072, 0.7213431000709534, 0.08807848393917084, 1.7418783903121948, -2.241215944290161, 0.0803697407245636, 2.189838171005249, 2.598243236541748, 0.47378766536712646, -0.11496933549642563, -0.7752625346183777], [0.21338719129562378, -0.21567516028881073, 1.3031147718429565, -0.06732634454965591, -1.1833654642105103, -0.4049902856349945, 0.11831199377775192, 1.7403900623321533, -2.278053045272827, 0.6272673010826111, -0.6297882795333862, 0.2866513133049011, 0.6766613721847534, 1.810882568359375, 2.013949394226074, 0.2755080461502075, -1.6095573902130127, -0.41468682885169983, 0.997401237487793, 0.6764881610870361, 1.4360064268112183, 0.30862805247306824, -0.30358919501304626, -0.9485564827919006, -1.0490411520004272, 0.5310683846473694, 0.2249436378479004, 0.11335841566324234, 0.15488839149475098, 2.769193649291992, 0.14502350986003876, 1.8199079036712646, -1.012515902519226, -0.9128590226173401, 1.76678466796875, -2.533426284790039, -0.07531605660915375], [-0.4927426278591156, 0.5460764765739441, -0.6469336152076721, -0.2257506549358368, 0.8706017136573792, -1.4858160018920898, 0.394505113363266, 0.7228341102600098, -0.6288231015205383, 1.2040717601776123, 1.3496322631835938, 0.29149287939071655, -0.8566321730613708, -0.9147460460662842, -1.8954362869262695, -0.4571307599544525, -0.2286836802959442, 1.8956953287124634, 0.683140218257904, 0.4771006107330322, -0.020088838413357735, 1.4143320322036743, -0.24446116387844086, 0.15789616107940674, -0.06837961077690125, 0.6517813205718994, -1.0938607454299927, -0.5445457100868225, 0.2712338864803314, -0.5228633284568787, 0.1611768752336502, 0.7868377566337585, -0.2404548078775406, 0.693559467792511, 0.3298951983451843, -0.38181835412979126, 2.5652639865875244], [-0.012445548549294472, -1.5641155242919922, -0.3613802492618561, -0.6849517226219177, -0.6499080061912537, 0.397026926279068, 1.2108960151672363, 0.4221962094306946, -0.4703759551048279, 1.7052603960037231, -0.01988755352795124, 0.8819356560707092, 1.6142065525054932, -0.7639737129211426, 1.4423227310180664, 0.08480533212423325, -0.3062417507171631, -2.2868781089782715, 0.4519321322441101, 0.6924376487731934, -2.2673416137695312, -0.589606761932373, -0.670742392539978, -1.2208141088485718, 1.0571670532226562, -0.3670714497566223, 1.122249722480774, -0.5278133749961853, -1.034151315689087, -1.8625110387802124, 1.591351866722107, 0.4048460125923157, 0.19060032069683075, 1.5459648370742798, -0.16463369131088257, 0.39735862612724304, -0.18876990675926208], [0.7391631603240967, 0.5622862577438354, 0.6692925095558167, 1.0842258930206299, 0.28708574175834656, -0.23088718950748444, 0.8905068039894104, -0.34715181589126587, 0.6983751058578491, 0.24568834900856018, -0.10128731280565262, -0.09911417216062546, 0.007336058188229799, 1.2576022148132324, -0.9824644923210144, -0.5888016223907471, 0.08316847681999207, 0.6096616387367249, -0.48615479469299316, -1.2546508312225342, -0.7092323303222656, -1.1071069240570068, -0.6152461171150208, -1.2070674896240234, 0.47150924801826477, 2.3509294986724854, -1.3193026781082153, 0.6269053220748901, -0.6470820307731628, -0.7335518002510071, 1.0588637590408325, -1.5624938011169434, -0.38319408893585205, -0.6970551013946533, -0.005446535535156727, 1.4865070581436157, -1.6801044940948486], [0.757565975189209, 0.5912303924560547, 0.260343074798584, -0.9622591137886047, 0.011322100646793842, -0.6078981161117554, 0.20422214269638062, -1.21355140209198, 0.5609431266784668, -0.6161521077156067, 0.49408581852912903, 0.06979120522737503, -1.3664698600769043, 0.5004349946975708, -1.0618610382080078, 0.20014090836048126, -0.22544999420642853, 0.62594074010849, -1.0722627639770508, -0.47568070888519287, -0.7404499053955078, 1.8599404096603394, -0.7338951826095581, -0.13840466737747192, -1.1436991691589355, 0.20808041095733643, -0.14418375492095947, 0.6438924670219421, -0.2662014663219452, 1.0377100706100464, -0.026142243295907974, -1.1566364765167236, 0.022863663733005524, -0.6126899719238281, -1.0570646524429321, -0.15074661374092102, 0.42846354842185974], [0.18191121518611908, 1.4333785772323608, 1.9325004816055298, -0.7531524896621704, -0.5203310251235962, -1.1957954168319702, 0.2051943689584732, 1.1308670043945312, 1.4332237243652344, 0.25176775455474854, -1.9705603122711182, -2.167741298675537, 1.1097838878631592, 0.9188266396522522, -0.9656334519386292, -1.2176660299301147, -0.4170703887939453, 1.1108053922653198, 0.569476306438446, 0.5402045845985413, 0.19373254477977753, 0.4462640881538391, -2.379340171813965, -0.4313008487224579, -0.42567405104637146, -0.8984030485153198, -0.24657149612903595, 0.898686408996582, -0.17071464657783508, 1.2867939472198486, 0.3180314898490906, -1.6125859022140503, -0.5711058378219604, 0.6534625291824341, -0.4720967411994934, 0.46795931458473206, 1.0136951208114624], [-0.6675060391426086, 1.6616839170455933, 1.1989238262176514, 0.6908076405525208, 0.3606243133544922, -0.28384000062942505, 1.791890263557434, -2.5286386013031006, -0.8256253004074097, -0.0017021788517013192, 0.5218746066093445, -0.651091992855072, -1.272870421409607, 0.09574229270219803, 0.3497103750705719, 0.02354517951607704, -0.09515328705310822, 0.4985456168651581, -1.1362504959106445, -0.3959105908870697, -1.2279796600341797, 1.1089962720870972, 0.1907443404197693, -0.8426365256309509, 1.2282856702804565, 0.32354751229286194, 0.19271984696388245, 0.6060880422592163, -0.9836263060569763, -0.32493847608566284, 0.13974088430404663, -1.858052134513855, -1.2555105686187744, 1.022214651107788, 1.8639785051345825, 1.7584716081619263, 0.3271031081676483], [0.192986398935318, 0.7455633878707886, -0.393903523683548, 0.885671079158783, -0.32276660203933716, 2.0009610652923584, 0.9529698491096497, 0.5551455616950989, -0.699184775352478, 0.9535753130912781, -0.4958152174949646, -0.3862353563308716, -1.4495095014572144, -0.05482269823551178, -0.23886825144290924, -0.2573010325431824, 0.38243088126182556, 0.9606069922447205, -0.24245978891849518, 0.6110130548477173, -0.42087146639823914, -0.36274099349975586, 0.22246333956718445, -0.2850096523761749, 0.9666215181350708, 1.4960108995437622, -0.24208790063858032, -0.24691469967365265, 1.0678666830062866, 1.660430908203125, -0.9098235368728638, 0.088478684425354, 1.3650202751159668, -0.007091337349265814, -0.19675223529338837, -1.4607077836990356, -1.9504724740982056], [-0.5765148997306824, -1.4525668621063232, 0.15859557688236237, 1.8199390172958374, -0.7705143690109253, -0.7122613787651062, -3.263760566711426, -0.7802462577819824, 0.3980439007282257, 0.8183029294013977, -0.6873772144317627, -0.25600114464759827, 1.854313611984253, 0.032186109572649, 1.9303747415542603, -0.4224715232849121, -1.1212892532348633, 0.06528773158788681, -1.2555824518203735, -0.2605493366718292, 0.6675234436988831, 1.3528468608856201, 0.20262211561203003, 0.04015801101922989, -0.6134787797927856, 3.380746841430664, -0.03982286527752876, -1.3758703470230103, -0.9780980944633484, 2.2845563888549805, -0.26329076290130615, -0.8804749846458435, 0.41468021273612976, -0.6697034239768982, 1.3361684083938599, -0.053621385246515274, 0.988601803779602], [0.8738568425178528, 1.5789116621017456, -0.8332694172859192, 1.2712132930755615, 1.224773645401001, 1.1277025938034058, 2.045393705368042, 0.06350056827068329, -0.6795294284820557, 0.1849306970834732, 0.4926827847957611, 0.7234156727790833, -0.023665910586714745, 1.2792950868606567, -0.3861251771450043, 1.0042526721954346, 1.6859718561172485, -1.0377575159072876, 0.8430442214012146, -1.0531597137451172, -0.06324362754821777, -0.8969862461090088, -0.20356979966163635, 0.7164198160171509, 0.9139777421951294, 0.14804890751838684, 0.22002625465393066, 1.3645888566970825, 0.5980792045593262, 0.3070283532142639, 0.6947177052497864, 0.9563145041465759, -0.03321535885334015, -0.05531651899218559, -0.3506911098957062, 0.33268460631370544, 1.3546348810195923], [0.13617798686027527, 0.5118318796157837, -0.9866927862167358, 1.5111349821090698, 0.4288042187690735, 0.5411419868469238, -0.48198917508125305, 0.3435163199901581, 0.6638971567153931, -1.3204721212387085, 1.3000379800796509, 1.5824717283248901, 1.3877878189086914, 0.7876940965652466, -0.6506161093711853, 0.24298687279224396, -1.4036003351211548, 0.39991748332977295, -1.9584702253341675, 1.844871997833252, -0.5567843317985535, -1.0090172290802002, 1.2522265911102295, -0.2911595106124878, -0.2769332826137543, -1.1871328353881836, 0.06935573369264603, -0.18067169189453125, -0.16946792602539062, 0.4700540006160736, -0.5932797789573669, -1.8524746894836426, -1.5242111682891846, -0.5011163353919983, -1.1263008117675781, -0.7551520466804504, -0.5574275851249695], [1.4362229108810425, -0.2533419132232666, -0.606910765171051, -0.3952732980251312, 0.25431913137435913, 0.3586827218532562, -1.1479319334030151, 0.23552080988883972, -1.353583574295044, 0.47256553173065186, 0.8839806914329529, -0.35956618189811707, -0.3160347640514374, -0.6411428451538086, 0.8117786049842834, -0.9898849725723267, 0.06005746126174927, 0.31556621193885803, -0.215526282787323, -0.04282557591795921, 0.535537600517273, -0.8241816759109497, -0.37353742122650146, -1.458094596862793, 0.4079040586948395, -0.15705174207687378, -0.1508631408214569, -0.09036146849393845, -0.12839683890342712, -1.563994288444519, -0.6283499598503113, 1.6979154348373413, -0.532509982585907, 0.33213114738464355, 1.1372474431991577, 1.1817597150802612, -0.5593642592430115], [0.5367059111595154, 0.1459047496318817, -0.11555010080337524, -0.7007867693901062, -1.3253298997879028, 0.8800355195999146, 0.4558079242706299, 0.5360304117202759, 0.3857353925704956, 0.09064879268407822, -0.23175451159477234, 0.5520824193954468, 1.8758955001831055, 0.2620772421360016, -1.2717911005020142, -0.2865917384624481, -0.012667289935052395, -1.2055026292800903, 0.3324206471443176, 0.35587993264198303, 0.9600600004196167, -0.7346619963645935, -0.2519378960132599, -0.45100072026252747, 1.3316245079040527, 0.08544924110174179, -0.6828774213790894, 1.4961711168289185, -0.8893918991088867, -0.19623853266239166, 0.12861613929271698, -1.3653390407562256, -0.5049263834953308, 1.1759557723999023, 1.0053291320800781, 1.2735416889190674, 0.9158247709274292], [-1.5017367601394653, 1.277026653289795, -0.04775967821478844, 0.6307440996170044, -0.10342935472726822, 0.7040310502052307, -0.3228279650211334, 0.5075606107711792, 0.15156255662441254, -1.2424744367599487, -0.974050760269165, 0.1592790186405182, -0.6877161860466003, 0.4288938641548157, 0.6849735379219055, -2.6198244094848633, -0.5083994269371033, -1.050971508026123, 0.025263875722885132, 0.46605178713798523, -1.2769742012023926, -0.7345126867294312, 0.9382331371307373, 1.446213960647583, -0.6421062350273132, 0.5342812538146973, -0.004704208578914404, 1.444510579109192, -0.313652366399765, 0.06009023264050484, -0.14535054564476013, 1.7576478719711304, 0.6269163489341736, -0.024505825713276863, 0.037785451859235764, 1.0511486530303955, -0.2014976143836975], [-1.0463060140609741, 0.23650269210338593, -0.06966769695281982, 0.7464245557785034, -0.6085360050201416, 0.7466007471084595, 0.5421610474586487, -0.5442229509353638, -0.7104884386062622, -0.4635113477706909, -0.3661772906780243, -0.046699974685907364, -0.8855472803115845, 1.1519641876220703, -1.9135812520980835, 0.12712030112743378, 0.09360585361719131, -1.572355031967163, -0.8511507511138916, -0.6547785997390747, -0.46065306663513184, -1.5811567306518555, 0.9169180989265442, -0.6128881573677063, 0.2107490748167038, -0.34329941868782043, -0.05055786669254303, 1.6798409223556519, 2.535464286804199, 0.7573909163475037, 1.7075275182724, -1.1045122146606445, 0.36043697595596313, -0.9268812537193298, -0.018454667180776596, 0.561539888381958, -0.942367672920227], [-0.12009255588054657, -0.6382889151573181, -0.10032635182142258, -0.033899400383234024, -0.5649291276931763, 0.5742642283439636, -0.3642091751098633, -0.10737848281860352, 0.5080615878105164, 1.443676233291626, 0.33573439717292786, -1.564887285232544, 0.5455866456031799, -1.3950272798538208, 0.24545301496982574, 2.3483259677886963, 0.10380087047815323, -1.032772421836853, -0.1761620193719864, -0.2255966067314148, 1.9479100704193115, 0.6573247313499451, 0.176214337348938, -0.13468952476978302, 1.0857110023498535, 1.3675432205200195, -0.636781632900238, -0.30789217352867126, -0.5656314492225647, 0.6386560797691345, -0.01877444051206112, -0.25207746028900146, -0.8508001565933228, -0.11051622778177261, 2.2582144737243652, -1.1639870405197144, 2.222794532775879], [0.13789111375808716, 0.18140295147895813, -0.7551682591438293, -1.3002091646194458, -0.25790292024612427, -0.6686044931411743, -0.05721267685294151, 0.17081357538700104, -0.5240177512168884, -0.9422586560249329, -0.2824912965297699, 1.5611984729766846, -0.9759026169776917, 1.124189853668213, -0.7626190185546875, -1.5564813613891602, -1.024722695350647, -1.2862484455108643, -1.912387490272522, 1.114061713218689, -0.7379765510559082, -0.8447265028953552, -1.9056777954101562, -0.10770836472511292, 0.5856332778930664, 3.2279434204101562, 1.9095686674118042, -0.532016932964325, -0.6145792603492737, 1.5084582567214966, 0.5654658675193787, 1.6590559482574463, -0.7032788991928101, -2.0980584621429443, 1.5760104656219482, -0.8386052250862122, 2.03045392036438], [-0.23672915995121002, -1.1839678287506104, -1.6459040641784668, 1.0049715042114258, 0.4605066180229187, 0.10440543293952942, 0.34817054867744446, -1.038321614265442, -0.42513319849967957, 0.18991784751415253, 0.3361598253250122, -0.25087615847587585, 0.6563818454742432, -0.33544856309890747, -0.48528918623924255, -0.0869019404053688, 1.3057819604873657, -0.3544391095638275, -0.1307625025510788, -0.34827369451522827, 0.4669901430606842, 1.4340301752090454, 0.6663355231285095, 0.6576889157295227, 0.6391777992248535, -0.35530468821525574, -0.7613756060600281, -0.49721911549568176, -0.832231879234314, -0.07065247744321823, -0.9685829877853394, -1.1200898885726929, -1.3868975639343262, 0.0008144907187670469, -0.22545118629932404, -0.6706907749176025, 0.056219153106212616], [0.7461956739425659, -1.7261152267456055, 1.9842772483825684, 1.7429953813552856, 1.8361324071884155, -0.5538815855979919, 0.3611677289009094, -0.8198105096817017, -1.0651869773864746, -0.08425848931074142, 0.36038845777511597, 1.0092724561691284, 1.503037929534912, 0.11117324233055115, 1.1996990442276, 0.4516321122646332, 0.1678798347711563, -0.3922777473926544, 0.22041019797325134, -0.5712133049964905, 1.0729801654815674, -1.1228936910629272, -0.5272328853607178, 0.35805726051330566, 0.33533087372779846, -0.3935084640979767, 0.3219093382358551, 1.704047441482544, 1.470765471458435, -0.2608892023563385, -0.8897759318351746, -0.39130353927612305, 0.1811007857322693, 0.7593558430671692, 2.8976380825042725, 1.5983729362487793, -0.2854478657245636], [-1.3342164754867554, 1.8834002017974854, -0.35873594880104065, -0.045988988131284714, -0.8555544018745422, -1.3134219646453857, 1.9660781621932983, -0.17192769050598145, 0.4984700083732605, -0.9713854789733887, 0.007370845880359411, -0.05947698652744293, -0.1127413883805275, 1.072128176689148, -0.16838383674621582, -0.06775406002998352, -0.3643610179424286, -0.48631012439727783, -0.08553851395845413, -0.7581266164779663, 0.047280699014663696, 0.9668375253677368, 0.4220260977745056, 0.4931286871433258, 1.249881386756897, 1.052248239517212, -0.9690502882003784, -0.2587886452674866, 1.238166332244873, 0.9245860576629639, -0.6677625179290771, 1.2578243017196655, -0.8446193933486938, 0.6266825199127197, 0.9935144186019897, 1.2080867290496826, 0.7489625215530396], [-0.28357937932014465, 3.103609085083008, -0.8430378437042236, -0.16522912681102753, 3.077083110809326, -1.9084019660949707, -1.158326268196106, -0.8276580572128296, -1.2402855157852173, 0.7840576767921448, 2.1504549980163574, 0.2566947340965271, 0.17788898944854736, -0.3155028522014618, -1.5914461612701416, 0.5825938582420349, -1.9171696901321411, 0.7147464752197266, 0.7781094908714294, -0.10289350897073746, -0.9163088202476501, 0.7015210390090942, 0.1307554990053177, -1.7489149570465088, -1.6254581212997437, -0.704010009765625, 0.8583337664604187, 1.468698501586914, 1.075490117073059, -0.5767156481742859, -0.8312571048736572, -0.9251327514648438, -0.24889211356639862, 0.18135671317577362, 0.45326265692710876, -0.5043323040008545, -0.07668371498584747], [1.1606507301330566, -0.1427578181028366, 0.8648639917373657, 1.515707015991211, 1.574828028678894, -0.9709230661392212, -0.941278874874115, 0.5661839246749878, 1.3918673992156982, -0.865003764629364, 0.11102476716041565, -0.5058125853538513, 0.6277177929878235, -0.4004139006137848, -0.3836764097213745, 1.068875789642334, 0.4535734951496124, -2.013239860534668, 0.7054531574249268, 0.3139997720718384, -0.26471006870269775, 0.6044983863830566, -0.3949272632598877, 0.17334379255771637, -0.5139117240905762, 0.14285962283611298, -1.5589889287948608, -0.6620786786079407, 0.524028480052948, -0.6682656407356262, -0.7093133926391602, 0.022120948880910873, 0.790799081325531, 1.1773515939712524, 0.8274093270301819, -0.15145383775234222, -1.53170907497406], [1.3020740747451782, -0.8602861166000366, -0.6373084187507629, -2.274822235107422, 2.049135208129883, -0.46078744530677795, 0.4830527901649475, -0.5853559970855713, 1.605605125427246, -0.7172552943229675, -0.09436991810798645, 0.01528302114456892, 1.7488924264907837, 0.5903245806694031, -0.5786389112472534, -2.4482178688049316, 0.061885472387075424, 0.2519543468952179, -0.26357752084732056, -2.5291080474853516, 0.6612607836723328, 0.9002247452735901, 0.896039605140686, 0.913360059261322, 1.3619751930236816, 0.2267395704984665, -0.2607465386390686, 0.6797225475311279, -0.45100128650665283, 0.05746092274785042, 0.7348899245262146, 0.40742653608322144, -1.7503101825714111, 1.2654500007629395, 0.331724613904953, 0.37241291999816895, -0.7749120593070984], [-0.41751375794410706, -0.628095269203186, 0.06722095608711243, -0.6510145664215088, -0.8776452541351318, -0.9416573643684387, 0.4456782639026642, -1.0525926351547241, -2.0002551078796387, -0.05043619126081467, -0.6621754169464111, 0.5543155074119568, -1.0988564491271973, 1.5756542682647705, 1.1320139169692993, 2.2535758018493652, 0.7710844874382019, -1.7484444379806519, 1.9992644786834717, 1.2524757385253906, -1.1285791397094727, -0.277156263589859, 1.5775318145751953, 0.25000518560409546, 0.4825797975063324, -1.3432011604309082, -0.2129584550857544, -0.021814335137605667, -0.431552916765213, -0.07964207977056503, -0.6973838806152344, 0.8964042663574219, -0.8388912677764893, -0.009378583170473576, -0.24724864959716797, 0.6477094888687134, -0.5978322625160217], [0.24130743741989136, -1.4654463529586792, 1.057179570198059, -0.02435147948563099, -0.4471867084503174, -0.2014191895723343, 0.4616442918777466, 1.3440138101577759, -0.2959973216056824, -0.6543526649475098, 1.5068708658218384, -0.14452768862247467, -2.1065499782562256, 0.5795941352844238, 1.2411025762557983, -0.6598091125488281, 1.1929820775985718, -0.35204535722732544, -0.22864457964897156, 1.3687183856964111, -0.7415825128555298, 0.04479081183671951, -0.3803854286670685, 2.784557342529297, 0.4740432798862457, 1.231053352355957, -0.7807404398918152, -0.6824831962585449, 0.4167158305644989, 1.245206356048584, -0.15109986066818237, 0.7069319486618042, 0.1492399126291275, -0.34255269169807434, -0.18399322032928467, 0.9305426478385925, -0.7893570065498352], [0.6177605986595154, 1.009535551071167, -0.6132186055183411, 0.08974802494049072, 1.8870794773101807, -0.4571587145328522, -0.1669553518295288, 0.781225323677063, -2.700061559677124, 0.6199861168861389, -1.2766846418380737, -0.6591789722442627, -1.3960567712783813, -1.0210310220718384, -0.09310352802276611, 0.6721634864807129, 1.791473150253296, -1.01283597946167, 0.4855448603630066, 1.7254818677902222, -1.2131264209747314, -0.4482356309890747, -0.04939485713839531, -0.147359237074852, -0.9394493103027344, 1.8321346044540405, -1.3728471994400024, 0.13621336221694946, 0.8187188506126404, 2.0757641792297363, -0.145679771900177, -0.4546964466571808, 0.8349618911743164, -0.8453773856163025, -1.945650577545166, 0.266924113035202, -0.779960036277771]], "dtype": "float32"}, {"__nd__": [[0.3148569166660309, -1.4400386810302734, 0.20159631967544556, 0.6372988224029541, 0.8895930647850037, 0.5935527682304382, -1.213453769683838, -0.49296504259109497, 1.995828628540039, -0.9703734517097473, -0.36630895733833313, 0.8552225232124329, 1.9069112539291382, 0.11526008695363998, 0.8804201483726501, 0.9243025183677673, -0.8758776187896729, -0.11708007752895355, -0.2067277431488037, -1.968078851699829, -0.8997029066085815, -0.08221882581710815, 0.32835957407951355], [0.019657138735055923, 0.9348294138908386, -0.6788322329521179, 0.3235118091106415, -0.879938006401062, 0.4554367959499359, 0.4421027898788452, 0.7908647060394287, 0.43098965287208557, -0.11648638546466827, -0.4876626133918762, 1.6349576711654663, -0.7552106380462646, -2.561495304107666, 0.4031869173049927, 0.7895610928535461, 0.8150587677955627, 0.9885738492012024, -1.9449690580368042, -1.3833459615707397, -0.44088202714920044, 1.294201135635376, -1.6050159931182861], [-0.8500012159347534, -0.30144238471984863, 1.32396399974823, 1.108886957168579, -0.7964398860931396, 0.12038702517747879, -0.9585502743721008, -0.16673807799816132, -0.20578661561012268, 0.5571412444114685, -0.9552465081214905, 1.9547971487045288, -0.5733412504196167, 2.5795233249664307, -1.121840000152588, -2.6647303104400635, 0.21779762208461761, -0.38521328568458557, -0.3757835924625397, 1.6003153324127197, -1.683310866355896, 0.2465572953224182, -0.7642322182655334], [-1.492275595664978, 0.9564327597618103, 0.4179345667362213, -1.0510252714157104, -0.4479069411754608, -0.7838009595870972, 0.3134200870990753, -1.4517775774002075, -0.3827506899833679, -0.027466073632240295, -0.2737032473087311, -0.4026549458503723, 0.72556471824646, -0.06797058880329132, -0.10893857479095459, -1.2488716840744019, 0.09765813499689102, 0.5968074798583984, -0.1544785350561142, -0.6267808675765991, 1.0269527435302734, 0.6286144256591797, 0.8375734686851501], [-0.027691876515746117, -0.8552097678184509, 0.002680383389815688, -0.8301336765289307, -0.9717414975166321, -0.9703120589256287, -0.7809525728225708, -0.9145475625991821, -1.8527227640151978, -0.6895700693130493, -0.039237845689058304, 0.23526781797409058, -1.528220295906067, 1.7522776126861572, 0.044012948870658875, 0.7846887707710266, -0.9585237503051758, -0.7090591788291931, 0.13825306296348572, 1.355358600616455, -0.7274612188339233, 0.23298463225364685, -1.7886385917663574], [0.061524540185928345, -0.997859537601471, 0.0950833261013031, -1.8666648864746094, -0.5352470874786377, 1.691005825996399, -0.7092728018760681, -0.9051162004470825, 0.013012527488172054, -0.7360266447067261, 1.9239672422409058, -1.617885947227478, -0.3812032639980316, 0.1687803566455841, -0.47633281350135803, -0.2213248908519745, -0.01918119005858898, 0.3754148781299591, 0.9079872965812683, 0.5658336877822876, -0.4315981864929199, -0.5683704018592834, -0.5001523494720459], [-0.669562816619873, -0.45063328742980957, -2.3690438270568848, -0.7779809236526489, -1.0504695177078247, 1.5947935581207275, -1.3473384380340576, -0.8745747804641724, -0.6121416687965393, -2.5185627937316895, 0.4809413254261017, 0.19106167554855347, -0.7230198979377747, -0.3595122694969177, -1.5759929418563843, -0.5290861129760742, 0.46852773427963257, -0.17951878905296326, -0.8800258040428162, -0.14831103384494781, -1.2519937753677368, -0.4323951303958893, 1.9609674215316772], [0.10995624214410782, 0.9189646244049072, 1.0556843280792236, 0.12424851953983307, -0.732001006603241, -0.17101998627185822, -0.6959190368652344, 1.5865708589553833, -0.6445765495300293, 0.0616036020219326, 0.2098945677280426, -0.016970844939351082, -0.6255917549133301, 1.3969495296478271, 0.028268272057175636, -0.37396612763404846, -0.9269225001335144, -0.6616500020027161, -2.2533786296844482, -0.8709034323692322, 0.25078338384628296, -1.173048973083496, 1.6437430381774902], [1.7016663551330566, -1.1964964866638184, 0.2103673666715622, -0.46011030673980713, -0.08105072379112244, 0.5176918506622314, -0.44678860902786255, -1.0875669717788696, -1.0137876272201538, -0.33963295817375183, -1.2218120098114014, -0.2720985412597656, -2.3316457271575928, -0.5522984862327576, -0.6901451945304871, 0.9392398595809937, -0.8132178783416748, 1.0454964637756348, -1.2476773262023926, 1.1907371282577515, 0.10114254057407379, 0.25578585267066956, 0.9773874282836914], [0.6487525701522827, 0.7690184712409973, -0.2718871831893921, 0.23711350560188293, -0.2577962875366211, -0.027720218524336815, 0.24660435318946838, 0.4324398338794708, 2.2342658042907715, -0.16131336987018585, -0.47164276242256165, -0.6845689415931702, 0.31629127264022827, -0.5663414001464844, -0.2765941619873047, 0.28722643852233887, -1.6752032041549683, 0.5275899767875671, -0.7159226536750793, -1.7912448644638062, 0.07745994627475739, -0.2295667678117752, 1.5790321826934814], [0.7212588787078857, 0.7675338387489319, -0.6623085737228394, 1.4414427280426025, 0.7300195097923279, -0.6987751722335815, 1.9074621200561523, -0.25278204679489136, 1.0169981718063354, -0.3325728476047516, -1.046785831451416, 0.8907695412635803, 0.5022252202033997, 0.4261852204799652, -0.36553308367729187, 0.8715631365776062, -0.5272536277770996, -0.13736200332641602, 0.30152761936187744, 0.044496726244688034, -0.2564261257648468, -0.6603937149047852, 1.0720489025115967], [1.2931722402572632, -0.2434065192937851, 0.13543212413787842, -0.7992148399353027, -0.26369354128837585, 0.11599793285131454, 0.41173255443573, -0.7171007394790649, 0.8997870683670044, -1.7399591207504272, -0.5101311206817627, -2.057406425476074, 0.16278597712516785, -0.5051581263542175, 0.11225061863660812, -3.130821466445923, -0.9607771635055542, -2.3033504486083984, 0.023416662588715553, 0.31034210324287415, 1.8601412773132324, 2.496347427368164, 2.00899076461792], [0.2577730715274811, -0.19340761005878448, -0.6946276426315308, 0.26498135924339294, 0.20672231912612915, -0.3008716404438019, 0.10521155595779419, -0.2732594907283783, -0.5660893321037292, -0.07313916832208633, 0.5223429203033447, 0.6654646992683411, 0.3981059789657593, 1.0338494777679443, 0.6022493839263916, -0.8392086625099182, -1.4613364934921265, 1.6614315509796143, -0.34157559275627136, -0.8455583453178406, 1.148797869682312, -0.42476004362106323, -0.2160409539937973], [0.2771894633769989, 0.8736554384231567, 0.07921303063631058, 0.6733824610710144, 0.7373782396316528, -0.14189551770687103, 0.3821369409561157, -0.3795185685157776, 0.45052534341812134, -1.7559748888015747, 1.068259835243225, 0.3648242950439453, -1.8617674112319946, 0.49383842945098877, -0.7809755802154541, -1.6040624380111694, 0.8296037316322327, -0.15918084979057312, 0.805336058139801, 0.37371158599853516, 0.8508683443069458, -0.28995636105537415, 0.6327817440032959], [1.7064132690429688, 0.12327472120523453, -0.8118355870246887, 0.9970895051956177, 0.07058970630168915, 0.7292032241821289, -0.8497557640075684, -1.7943929433822632, 0.3380049169063568, 0.6817556619644165, -0.4606828987598419, -0.611121654510498, 0.360499769449234, 0.04685061424970627, -1.2791649103164673, 0.3075455129146576, 0.37016505002975464, -1.2138065099716187, 0.3566458821296692, 0.7210075259208679, 0.953712522983551, -0.9559399485588074, 0.13411526381969452], [-1.75263512134552, 0.16701066493988037, -1.1181046962738037, -0.21999190747737885, 0.6360346078872681, -0.44954392313957214, -0.20375017821788788, 0.760549783706665, 0.7610089182853699, -1.5691133737564087, 0.011995181441307068, -0.7631621360778809, 3.397265672683716, 0.31374433636665344, -1.2237145900726318, 0.747617781162262, 1.9711326360702515, 0.8439751267433167, 0.366984486579895, 0.8390795588493347, 0.0028562480583786964, -0.44555479288101196, 0.6175345182418823], [0.7942613959312439, -0.4610740542411804, 0.04822798818349838, -0.060193486511707306, -0.330260694026947, 0.45058244466781616, -0.09372013807296753, -0.4063410460948944, 0.2896032929420471, 0.9572526812553406, 1.8135817050933838, -0.946719229221344, -0.5625584721565247, 0.633089005947113, -0.6443237662315369, -1.625053882598877, -0.28049325942993164, -0.26696717739105225, 0.949446976184845, 0.49978408217430115, -0.15753960609436035, 0.7491357326507568, -0.32202020287513733], [1.0943481922149658, 0.11454581469297409, 1.4015339612960815, 0.666271448135376, 0.2626975476741791, -0.5969444513320923, 0.1461869180202484, -0.14578501880168915, 2.5642011165618896, 2.2884013652801514, 1.1282973289489746, -0.1217670738697052, 1.4093098640441895, -0.14023461937904358, 1.3247884511947632, -0.6156345009803772, -1.3165665864944458, 0.17191891372203827, 0.8297020196914673, 1.1329039335250854, -0.08871979266405106, 0.08821889758110046, 1.1913517713546753], [0.911452054977417, 0.42633843421936035, 0.6333264112472534, -1.8633999824523926, 0.435063898563385, -0.10862325876951218, -1.33709716796875, -0.3714834153652191, -0.9970365166664124, -0.3520473539829254, 0.010985580272972584, -1.398812174797058, -1.5206016302108765, 0.789633572101593, 1.001022458076477, -0.922009289264679, 1.0371789932250977, -0.15524014830589294, 0.46488112211227417, 1.3391385078430176, -1.5378243923187256, -0.025899212807416916, -0.9846948981285095], [1.492828607559204, 0.6952278017997742, 0.2635681927204132, 0.6588805913925171, 1.4210773706436157, 0.7773553729057312, 0.15371261537075043, -0.9275937080383301, -0.6207067966461182, 0.5650628209114075, 0.47993624210357666, 0.6264650225639343, 0.210970938205719, -0.24930821359157562, -2.277580738067627, -0.650781512260437, 0.8067793846130371, 0.10078473389148712, 0.5816056132316589, 0.9237636923789978, 0.2560805678367615, -0.33019331097602844, 0.9288507699966431], [1.1832847595214844, -1.193130612373352, -1.2438162565231323, 1.3585336208343506, -0.4734608232975006, 0.3507799804210663, 0.477065771818161, 0.9410555958747864, -0.6696544885635376, 1.0192817449569702, 0.9597888588905334, 0.8639135360717773, 0.006641100626438856, -0.04137851670384407, -0.11047405749559402, 0.9291371703147888, -0.04224073514342308, -0.022339235991239548, -0.1133846715092659, 0.018214238807559013, 1.5305134057998657, 1.0533764362335205, 0.14461714029312134], [0.002247877884656191, 0.07483325898647308, 1.6779028177261353, -1.4893945455551147, 1.65215003490448, -0.24250353872776031, 3.40836238861084, -1.3204519748687744, -0.7557954788208008, 0.19289135932922363, -0.5369020700454712, -0.4887676239013672, 0.7400614023208618, -0.39748260378837585, 2.9229376316070557, -1.1860084533691406, 0.37623661756515503, 0.8870632648468018, -0.46333402395248413, 0.3915586471557617, 0.1860070377588272, 0.5311123728752136, -0.3220521807670593], [-0.981300413608551, -1.1085501909255981, -0.9684260487556458, -1.3315420150756836, 0.13927988708019257, 0.5712798237800598, -0.3488902747631073, -1.2009012699127197, -0.17222033441066742, 0.11722669750452042, -2.1175525188446045, 1.3221052885055542, 0.6162363886833191, 0.5185446739196777, -0.5958458781242371, -0.2651771605014801, 2.261101484298706, 1.8672388792037964, -0.6841577291488647, -0.3114609122276306, -1.0879689455032349, -0.782686710357666, -0.35471537709236145], [0.6540930271148682, -0.9608787894248962, 0.028528369963169098, -0.10885769128799438, 0.5057435631752014, 1.1766072511672974, -0.36481326818466187, 1.3381109237670898, 0.0644918829202652, -0.2961319386959076, -0.4049830436706543, -0.7658893465995789, -1.2659709453582764, -0.0846475288271904, 0.8826279044151306, 0.9775100946426392, -0.2082587629556656, -1.2344383001327515, -0.4149640202522278, -0.6091122627258301, -1.0532877445220947, 0.12448550760746002, -0.5745624303817749], [-0.0875738337635994, -0.9413615465164185, 0.3333933651447296, -0.2788695991039276, -0.41843199729919434, 2.669816493988037, -1.3973121643066406, -1.2293250560760498, 0.8257349133491516, 0.20200654864311218, 0.4691981375217438, -0.7234438061714172, -0.5406072735786438, -0.6452236175537109, 1.154134750366211, 1.5358296632766724, -0.6912914514541626, 0.17462407052516937, -0.7633012533187866, 0.22888877987861633, -0.7213367223739624, -0.3361296057701111, 0.5016894936561584], [0.8135901093482971, -0.06313329935073853, -0.8846269249916077, 1.6439863443374634, 1.4734582901000977, -0.5336631536483765, 0.2177373170852661, 0.478083997964859, 0.6335235238075256, 0.4640602171421051, -1.3805660009384155, 1.5432183742523193, 0.01049406174570322, -0.0880846455693245, -0.27847912907600403, 0.7719003558158875, -0.1158699318766594, -0.20959630608558655, 0.015158429741859436, -0.6471432447433472, -0.7155804634094238, -1.453325629234314, 0.8135743141174316], [0.7237593531608582, -0.035223137587308884, 0.2548806071281433, -1.3622539043426514, 0.46388113498687744, 0.4170760214328766, -0.6177063584327698, 0.6909592747688293, 1.104561686515808, -0.35315364599227905, 0.9313555955886841, 0.7714686989784241, -0.6446114182472229, 0.3540114760398865, 0.20171095430850983, -0.8520413637161255, -0.36056390404701233, -0.9136086702346802, 1.090336799621582, 0.4388043284416199, 2.117626428604126, -0.4726986587047577, 0.027819408103823662], [-0.20419888198375702, -0.49260246753692627, -0.8616452217102051, 0.6071149110794067, -0.3465445339679718, -0.16262869536876678, 0.9430757164955139, 0.6686159372329712, 0.7156853079795837, 1.5340099334716797, 0.7555825710296631, -0.3382725417613983, 0.49548864364624023, 0.29881757497787476, 2.007972240447998, 1.6196550130844116, 2.7536699771881104, -2.4423961639404297, -0.024183478206396103, 0.6115090847015381, -0.28085044026374817, -0.16117388010025024, -0.19830629229545593], [-0.5741610527038574, 0.6573903560638428, -0.6034263372421265, 1.8767460584640503, 0.9510436654090881, -2.2285196781158447, 0.5765025615692139, -0.5535410642623901, 1.480571985244751, 1.372184157371521, -1.0378342866897583, -1.6712995767593384, 1.4722974300384521, -0.7914126515388489, -0.9903170466423035, -0.38392800092697144, -0.6591317057609558, -1.3267096281051636, -0.43776753544807434, -0.4217402935028076, 0.11324260383844376, 0.022699877619743347, 0.29814302921295166], [-0.804804265499115, 0.05007033422589302, 0.5033921599388123, 0.7166165113449097, -0.8672120571136475, -0.31317880749702454, 0.49356216192245483, 1.558414101600647, 0.2672303020954132, 0.2033262401819229, -0.4635775685310364, -1.3215863704681396, -0.840700089931488, 0.5083533525466919, 0.8818987011909485, 2.1686408519744873, -0.15934894979000092, 0.529534101486206, -1.2430188655853271, 0.18461105227470398, -0.6234018206596375, 0.43068403005599976, -0.13416023552417755], [1.1177221536636353, -1.1190476417541504, 0.6911548376083374, -0.011688905768096447, -0.06893507391214371, 0.4427920877933502, -0.5185050368309021, -0.7116959691047668, -1.159938931465149, -0.14481107890605927, -1.5494509935379028, -0.25146812200546265, 1.7347710132598877, 0.3878161311149597, -0.012493180111050606, 0.2413301318883896, 0.06459684669971466, 0.7021416425704956, -0.5846726894378662, 2.1396327018737793, -1.1506123542785645, -0.3642827272415161, -1.8842130899429321], [-0.40985187888145447, 1.201454520225525, -0.41706204414367676, -0.750206708908081, -1.6477928161621094, -1.0704357624053955, -1.0189939737319946, 1.77838933467865, -0.5133165121078491, -1.8559234142303467, -0.8501023054122925, 0.01040272694081068, 0.34363216161727905, -0.8270743489265442, -1.9510340690612793, 1.812869906425476, -0.41453954577445984, -0.18767257034778595, 1.107707142829895, 0.4188227951526642, 0.16678139567375183, 0.18529166281223297, 0.1380128562450409], [-1.5328041315078735, 0.32072094082832336, 1.1444978713989258, 1.0733379125595093, 0.2670194208621979, 0.18295453488826752, -0.972890317440033, 0.2992667853832245, 0.6446036100387573, 0.310226708650589, 0.03303440287709236, 0.5177901387214661, 0.9749183058738708, 0.08759946376085281, -1.1543687582015991, -0.9749866127967834, 0.7861896753311157, 0.6330831050872803, -1.549767017364502, 0.556229293346405, 0.071644626557827, -1.4030492305755615, 0.5778211355209351], [-0.9984735250473022, 0.23726524412631989, -1.703428030014038, 0.25675782561302185, -0.034136511385440826, -0.11726381629705429, 0.3231157064437866, -1.695046067237854, 0.5740125775337219, 0.6320028305053711, 0.012011697515845299, -1.0181421041488647, -0.2394801527261734, -0.7021820545196533, -0.3533577024936676, -1.1871058940887451, 1.5039594173431396, 1.1910064220428467, -0.2581171989440918, 0.29567667841911316, 1.2570327520370483, -0.3156864643096924, 0.146027073264122], [-0.3239581882953644, -0.7106638550758362, -0.2975448966026306, -0.2014061063528061, 0.512419581413269, -0.021601157262921333, -1.5099624395370483, -0.3478255271911621, -0.24030368030071259, -0.4396832585334778, -0.2911450266838074, -1.1807445287704468, -2.414856433868408, -1.1626375913619995, -0.5403862595558167, -0.03308652341365814, 1.6530423164367676, -0.33711013197898865, -0.9421890377998352, -1.0794641971588135, 0.6530840396881104, 0.8822870850563049, 0.5626777410507202], [-0.7198320031166077, -0.1811191886663437, -1.3815929889678955, 0.8844806551933289, 0.879875659942627, 2.465848684310913, -0.055032700300216675, 0.8968539237976074, 1.6315456628799438, 0.6819972991943359, 0.9571958780288696, 0.6861228346824646, 0.33128130435943604, 1.8798985481262207, -2.0850558280944824, -1.0260977745056152, 1.3191992044448853, 0.5020256042480469, 0.05729612335562706, -0.9927403926849365, -0.5612014532089233, 0.2825701832771301, 0.0028161872178316116], [-1.9194865226745605, -1.9488859176635742, -0.3519049286842346, -0.5347087979316711, -0.9734363555908203, -1.4604671001434326, 1.2950183153152466, 0.34596794843673706, -0.6900832653045654, -0.8749212026596069, -0.772972583770752, -0.47156357765197754, 1.0883134603500366, 0.9696968793869019, 1.0186693668365479, 0.8657045364379883, 1.878799557685852, 0.8223583102226257, -2.120300769805908, 0.8941729068756104, -0.551417887210846, 1.0037721395492554, 0.5832337141036987], [-1.1323020458221436, -0.24521301686763763, -1.0869531631469727, -1.0780861377716064, -1.1897555589675903, 0.9617319703102112, -0.7928851246833801, -1.5768470764160156, 0.737389862537384, 0.6502183675765991, 1.3248493671417236, -1.2385783195495605, -0.18565167486667633, -0.11768946796655655, -0.5883793234825134, -2.0003116130828857, 1.5823396444320679, -1.737368106842041, -1.3165802955627441, -0.5614307522773743, -0.13046684861183167, -1.1710288524627686, -1.196104884147644], [-1.3571906089782715, -0.23452356457710266, 0.5656564235687256, -1.7880223989486694, 1.09321129322052, -0.49118733406066895, -0.8747321963310242, -0.883260190486908, 0.49262863397598267, -1.0239120721817017, 0.862714946269989, 0.9768792390823364, 1.1990169286727905, -1.3910527229309082, 1.1919516324996948, -0.4929516017436981, -0.5178186893463135, -0.21013571321964264, -1.2725296020507812, 2.045240640640259, 0.12301003932952881, 1.1189380884170532, -1.065151572227478], [0.5767770409584045, 0.04433456435799599, -0.8685915470123291, 1.6908226013183594, 0.17964734137058258, 0.42037853598594666, 1.102821946144104, -0.2485503852367401, 0.49726590514183044, -0.9846423864364624, 0.31075048446655273, -1.1080713272094727, -0.7276484370231628, 0.5972707271575928, -0.431273877620697, -3.017453193664551, -1.153564691543579, -0.05440354719758034, 0.15770520269870758, 0.5080100893974304, -0.10337183624505997, -0.35724911093711853, -0.1752413511276245], [-0.6465268731117249, 0.25164708495140076, 0.6398923397064209, 0.3852975070476532, -1.121187448501587, 0.18220683932304382, -0.368541955947876, -0.49121254682540894, 1.2646315097808838, 0.625006914138794, -1.286352515220642, -0.6124128699302673, -0.6322277188301086, -0.6524614691734314, -2.014004945755005, 0.3194864094257355, 0.33443692326545715, -1.0431420803070068, -0.22822833061218262, 0.3367730975151062, -1.8921247720718384, -0.21673329174518585, 1.0900604724884033], [1.451529622077942, -0.7870097160339355, -0.3903431296348572, -1.5206918716430664, 0.8542069792747498, 2.061486005783081, -1.2232587337493896, 0.2019708752632141, -0.4379616379737854, -0.530541181564331, -0.04994872584939003, -0.5090048313140869, 0.19958023726940155, -0.07114039361476898, 2.493152618408203, -1.6901963949203491, 0.39770689606666565, -0.2372061163187027, 0.9417480230331421, -1.7428234815597534, 0.7523199915885925, 1.6813750267028809, -0.42993518710136414], [1.2992615699768066, 1.2601466178894043, -0.32569652795791626, -1.313148856163025, -0.563804566860199, 0.39216184616088867, 0.3938995599746704, 0.6471651792526245, -1.0427892208099365, 0.46345028281211853, -0.1151626855134964, 1.0333833694458008, -0.6596710085868835, -1.0116426944732666, -0.011926645413041115, 0.16853538155555725, -0.9040300250053406, 0.6129491925239563, 0.2820427119731903, -0.6022329926490784, -0.5192807912826538, 0.8131763339042664, -0.29028770327568054], [1.4081281423568726, 1.1752452850341797, -0.734912633895874, -0.7144315838813782, -1.6490384340286255, -0.9941415786743164, 0.3587174713611603, 3.1839871406555176, -0.4092792570590973, 1.976206660270691, 0.24015802145004272, 0.3845341205596924, -0.06590275466442108, 1.24299156665802, -0.02159736678004265, -0.6683408617973328, 0.13007280230522156, 0.9647932052612305, 0.8749029040336609, -0.697866678237915, 1.9980123043060303, 1.2762595415115356, 0.410288542509079], [0.6982061862945557, 0.279482364654541, -0.029137179255485535, -0.5599872469902039, 0.7693358659744263, -1.5593063831329346, 0.20489947497844696, 2.621568441390991, 0.7295962572097778, 0.5755950212478638, -0.07515361905097961, 1.1169703006744385, 0.1911904215812683, -0.5078265070915222, -0.6502421498298645, 1.8979933261871338, 0.20829084515571594, -0.7209469676017761, 1.2147902250289917, 0.976908266544342, -2.5831689834594727, -0.7107520699501038, -0.31898048520088196], [0.3374698758125305, -0.7274348139762878, -0.9975837469100952, -1.0309916734695435, -1.6041576862335205, 0.13265195488929749, -0.6958638429641724, -1.2959951162338257, -0.8532160520553589, 0.2413340061903, 0.3727099597454071, 0.2478468269109726, 2.051210403442383, -1.6918126344680786, -0.4848218262195587, -0.7938127517700195, 0.7671682238578796, 0.5087741017341614, -2.025576591491699, 0.10451333969831467, -0.8030412197113037, 0.770959734916687, 1.0968385934829712], [0.9430696964263916, -1.4692068099975586, -0.181949645280838, 0.715820848941803, 0.36306968331336975, -0.0505836047232151, 1.24208664894104, -0.2941327393054962, -0.8080574870109558, -0.5582893490791321, 0.20693890750408173, 0.6006360054016113, -0.07282260805368423, -0.04754740372300148, 0.37904444336891174, 1.5916177034378052, -0.6214622259140015, 0.7897534966468811, -0.40439119935035706, -1.026229739189148, 2.975044012069702, -0.8203883767127991, -1.1304781436920166], [-0.1944994181394577, -0.8630690574645996, 1.7401617765426636, -0.45404699444770813, -0.5512668490409851, -0.21104441583156586, -0.3267599940299988, 0.47759386897087097, -0.4663454294204712, 1.4471746683120728, 0.14488938450813293, 0.2052849382162094, 1.557031273841858, -1.2012392282485962, -0.29778406023979187, 2.2224392890930176, 0.16775813698768616, 1.2485010623931885, -1.1698179244995117, -0.33757537603378296, -0.1877414733171463, 0.39441925287246704, -0.04688939079642296], [0.5119039416313171, 0.4900115132331848, 0.9252874851226807, -1.2233145236968994, 0.9211491942405701, -0.9288760423660278, -0.27960729598999023, 0.0557231679558754, 1.540137767791748, -0.36273840069770813, -1.0787289142608643, 1.2265881299972534, -0.09818260371685028, -0.6591292023658752, 0.8517300486564636, -0.22732987999916077, 0.9638190269470215, 1.0477758646011353, 0.4399504065513611, -1.643728256225586, -1.3065552711486816, -0.8282108902931213, -0.2816435694694519], [-0.9003939628601074, -0.41402772068977356, 1.7868949174880981, -0.3875162601470947, -0.7487585544586182, 0.1638077050447464, 0.06919285655021667, -1.6960117816925049, 0.46624845266342163, -0.7430310249328613, 0.9174227714538574, -0.22204771637916565, 2.0784928798675537, -0.1389142870903015, 0.5091016292572021, 0.3537628650665283, -1.1632353067398071, 0.2651475667953491, -1.1522716283798218, -1.3175324201583862, 0.8260800242424011, -0.7925469279289246, 1.9727190732955933]], "dtype": "float32"}], "e": {"__nd__": [[-0.4309714138507843, -4.0524678230285645, -1.2815970182418823, -0.42900386452674866, 5.393111705780029, 3.3278298377990723, 0.7276850342750549, -3.7806968688964844, -3.3194375038146973, -4.327322483062744, -4.133991718292236, 1.4754571914672852, 0.38181912899017334, 0.006760000251233578, -0.4002799689769745, -1.7816495895385742, 9.408930778503418, 6.391693592071533, -6.2173590660095215, -6.3874006271362305, 1.2837421894073486, 2.339092969894409, 5.915892601013184], [0.6311429142951965, -2.104837656021118, -5.525599002838135, 2.4704108238220215, -4.722598075866699, -21.466571807861328, 7.403384685516357, -3.0350587368011475, -5.901614665985107, 7.332257270812988, -8.491312026977539, -8.349340438842773, 1.3434083461761475, 4.847219944000244, -2.1777546405792236, 14.346571922302246, 2.0678212642669678, -5.418859004974365, -0.16457343101501465, 17.888370513916016, -4.099078178405762, 3.7959136962890625, 2.0991368293762207], [-4.419800281524658, 0.7125593423843384, -2.540350914001465, -0.7118138670921326, -8.352286338806152, 3.971648693084717, -4.883565425872803, -1.3097050189971924, -7.100846767425537, 1.3363195657730103, 6.437398433685303, 0.20200398564338684, 5.550706386566162, -1.1766157150268555, -6.855544567108154, 7.244414329528809, 3.1357228755950928, 6.928189754486084, -2.9277729988098145, 9.445454597473145, -9.549633026123047, -1.3803250789642334, -1.8452266454696655], [1.2685927152633667, 4.247896671295166, -5.720080852508545, -8.7399263381958, 0.34153205156326294, 3.1138150691986084, -8.299307823181152, -3.8961093425750732, 3.5559680461883545, 7.711096286773682, 5.723852157592773, -2.4279723167419434, 10.954983711242676, -11.808504104614258, 3.091923475265503, -10.2206392288208, 8.072649002075195, 7.9776201248168945, -1.0338689088821411, 2.8696911334991455, -1.3274060487747192, 3.7351174354553223, 10.403502464294434], [8.967723846435547, 2.4612231254577637, -6.271673679351807, -0.3651835322380066, 4.095138072967529, -5.551064968109131, 4.0985331535339355, -5.85552453994751, -5.082815170288086, 3.4599661827087402, -6.041458606719971, 9.477679252624512, -0.5849486589431763, -3.6870718002319336, -4.824145793914795, 2.3735029697418213, 1.6907771825790405, -1.8935455083847046, 4.232842445373535, -2.785862922668457, -4.447351455688477, -4.930755138397217, 0.7628583312034607], [-2.0419609546661377, 4.906799793243408, -10.209203720092773, -0.35835427045822144, 1.4155943393707275, 9.846929550170898, -1.027541160583496, -12.013490676879883, -3.979353427886963, -11.119619369506836, -2.268056631088257, 1.172363042831421, -9.425243377685547, -5.946183681488037, -11.74390983581543, -13.826926231384277, 11.570530891418457, 3.840721607208252, -4.679205894470215, 2.084348201751709, 6.706945419311523, 2.1706228256225586, 0.24953535199165344], [11.777472496032715, 0.06152692437171936, -2.8961377143859863, 3.46121883392334, 0.654364287853241, -10.960210800170898, 3.3227322101593018, 8.286751747131348, -2.362423896789551, 11.857046127319336, -4.165431022644043, -0.7356687784194946, 2.430617094039917, 3.226553440093994, 11.966987609863281, 11.221019744873047, -4.510899066925049, -1.6702219247817993, 12.635159492492676, 3.174704074859619, 14.437759399414062, 3.4376590251922607, -8.434453964233398], [-8.14052963256836, -2.961320638656616, -7.450282573699951, -1.9184894561767578, -2.1142971515655518, 10.582345008850098, -1.7853283882141113, -5.66261625289917, 1.238205075263977, -10.611837387084961, 2.7073352336883545, 3.716115713119507, -10.558792114257812, 5.497230529785156, -5.371596813201904, -3.365431785583496, 10.995879173278809, 0.8410671353340149, -0.44172370433807373, -2.4401283264160156, -5.139970302581787, -4.525417327880859, 4.555258274078369], [-1.6688694953918457, -0.6271919012069702, -10.25072193145752, 4.577322959899902, -1.3114120960235596, -7.866330623626709, 8.006146430969238, -0.20194445550441742, -9.538415908813477, -6.419619560241699, -0.024390943348407745, -8.828699111938477, -10.531760215759277, 7.200143814086914, -0.583376407623291, -5.606791019439697, 8.300110816955566, -2.9556052684783936, -1.3957527875900269, 3.749661922454834, 6.234133243560791, -3.8402364253997803, 0.791606605052948], [-2.8974342346191406, -3.6939709186553955, -0.17203666269779205, -1.1201742887496948, 5.347341537475586, 3.8139498233795166, 7.231598377227783, -0.1803293526172638, 2.4384539127349854, -1.5926119089126587, -3.9066452980041504, 5.904451847076416, 3.710217237472534, 2.8289575576782227, 10.0822172164917, 1.4171996116638184, -1.452300786972046, 6.459694862365723, 7.100374698638916, 4.9442243576049805, -0.3275073170661926, -4.32138729095459, -3.6981136798858643], [13.988205909729004, 5.695688247680664, -2.802344560623169, 1.8982957601547241, 14.169272422790527, -5.626474380493164, 2.811185359954834, 14.745664596557617, 7.812658309936523, 1.3943742513656616, -1.273293137550354, -1.8191322088241577, -4.022003650665283, -7.879605770111084, 1.9017117023468018, 2.712547540664673, 0.4925900101661682, -2.8864526748657227, 9.286284446716309, -5.0782294273376465, -2.246717929840088, 1.0406494140625, -1.286330223083496], [2.4253451824188232, 1.9193471670150757, -14.133770942687988, -5.1100616455078125, -3.6263084411621094, 1.112027645111084, -5.596185684204102, -3.50993275642395, -1.0476783514022827, -1.650952935218811, -8.781441688537598, 3.1045496463775635, -8.157722473144531, -13.799772262573242, -9.355557441711426, 7.806896209716797, 3.765024185180664, -2.92611026763916, -4.568971157073975, -13.378499031066895, 6.927179336547852, 2.354292631149292, 5.503133773803711], [-9.640314102172852, -4.434683322906494, -9.40629768371582, 3.9622678756713867, -6.826314449310303, 4.417692184448242, -5.03264856338501, 15.079936027526855, -3.803403615951538, -4.658341407775879, 4.1314568519592285, 10.153081893920898, -5.592823505401611, 3.9228320121765137, 8.850385665893555, 8.92361831665039, 1.093592882156372, 10.455323219299316, -14.071758270263672, -9.981846809387207, 5.099761486053467, 0.553587019443512, -5.2678399085998535], [-7.459101676940918, 0.12745623290538788, 2.935195207595825, -13.401785850524902, -5.611873626708984, -5.41593074798584, -4.518161773681641, -8.941010475158691, 6.291303634643555, 3.215613842010498, -9.771970748901367, -10.976834297180176, 2.854261875152588, -5.133092880249023, 8.009852409362793, 8.446673393249512, 10.773641586303711, 4.915731430053711, -11.593249320983887, 0.5915648341178894, 0.19843217730522156, 3.26057505607605, 7.297694683074951], [4.5614519119262695, -5.521496772766113, 10.982989311218262, -4.282362461090088, -2.995908498764038, 20.328079223632812, -11.75278091430664, -8.169042587280273, 0.8415447473526001, 1.9556410312652588, 1.8478820323944092, 4.5317559242248535, -13.023451805114746, 8.303288459777832, -7.230953693389893, -6.2962470054626465, -4.861331939697266, 8.259068489074707, 2.2210264205932617, 1.3121329545974731, -5.056447505950928, -8.245583534240723, -5.817912578582764], [1.8168879747390747, -0.1500665694475174, -2.4541573524475098, 1.8123602867126465, 4.064484119415283, -3.0687296390533447, 7.219728469848633, -0.6784653067588806, -1.0999195575714111, -1.084749460220337, 0.08796310424804688, 4.583249092102051, 8.233582496643066, -7.482999801635742, 7.434179306030273, -1.143531322479248, -6.465971946716309, 16.740856170654297, 1.2508618831634521, -2.8251030445098877, -0.21148593723773956, 4.372143745422363, 4.901387691497803], [-3.909348249435425, -5.200860500335693, 2.392117500305176, -8.02676773071289, -1.5663050413131714, 1.8476494550704956, 0.9261818528175354, -5.497115612030029, -11.595741271972656, -5.009239196777344, 1.2969672679901123, 3.4430627822875977, 2.9284770488739014, 6.0179123878479, 11.072063446044922, -6.704859733581543, 4.15512752532959, 5.205246448516846, 1.4838693141937256, -2.7260847091674805, -1.1660022735595703, 1.135421872138977, -13.615011215209961], [6.002859115600586, 4.013299942016602, -4.139535903930664, 12.459268569946289, 7.303165435791016, -3.269463062286377, 7.207677364349365, 1.8159501552581787, -5.613205909729004, 3.3581223487854004, -16.18562126159668, 3.1629528999328613, -9.83378791809082, 8.969358444213867, -3.976062536239624, 8.52165412902832, -0.59541916847229, -0.6991641521453857, 9.48673152923584, 3.232222080230713, -8.907313346862793, -5.507643699645996, -8.399198532104492], [7.359237194061279, 2.437469244003296, 1.1000103950500488, -0.030026057735085487, 0.23576603829860687, 1.8075945377349854, 15.493294715881348, -2.3286521434783936, 1.1359028816223145, 2.806663990020752, 7.654226303100586, 4.557905197143555, 3.049283742904663, -4.604982376098633, 2.383716106414795, 4.285789489746094, -1.6140282154083252, 12.603541374206543, 2.4722087383270264, 5.090866565704346, 3.5730223655700684, 1.0075750350952148, 5.64539098739624], [-0.365129292011261, -4.901648044586182, 10.63396167755127, -8.941953659057617, 0.6475043296813965, 6.808754920959473, -13.832030296325684, -0.9064828753471375, 0.26753324270248413, -8.48099422454834, 7.16170597076416, -8.851855278015137, -6.162722110748291, -7.702672004699707, -2.0830070972442627, 2.9250917434692383, 0.6040279269218445, -2.814770221710205, -1.6371268033981323, -2.068821430206299, -8.99106216430664, -2.919420003890991, 11.743069648742676], [0.7266281247138977, 1.9481239318847656, -10.592363357543945, 2.61604905128479, -2.7023792266845703, 4.506618976593018, 3.1362035274505615, -2.706526517868042, -16.99032974243164, -2.624424457550049, -4.190945625305176, -3.2854435443878174, -16.95162010192871, 9.491046905517578, 1.1713008880615234, -9.470833778381348, -1.2999056577682495, 9.166180610656738, 1.549299955368042, 4.363120079040527, 0.49862515926361084, 4.2583394050598145, -5.898882865905762], [9.037296295166016, 4.418542861938477, -1.5095442533493042, 6.16375732421875, 9.314553260803223, -1.5170789957046509, -0.1600409746170044, -4.372533798217773, 2.01631498336792, -0.5891746878623962, -12.98968505859375, -3.7669637203216553, 16.077171325683594, 4.084062576293945, -3.90487003326416, 1.6157480478286743, -5.581706523895264, 2.9259867668151855, 2.5683023929595947, -2.770961046218872, -5.182705402374268, -16.914142608642578, 7.418256759643555], [4.116058826446533, -3.2411606311798096, -0.3846345841884613, -1.025991678237915, 0.9495813250541687, -1.0526330471038818, -2.5206096172332764, 10.775432586669922, -10.859061241149902, 2.7817280292510986, 12.289340019226074, 5.845080852508545, -2.5094799995422363, -2.30825138092041, 4.758871078491211, 2.8512041568756104, -4.471549987792969, 3.1565349102020264, -5.268222332000732, -10.703116416931152, 11.98354721069336, 0.8040716052055359, -5.922702789306641], [5.8833818435668945, -4.357937812805176, -0.4370745122432709, -11.020011901855469, -7.42735481262207, 2.480607032775879, -13.12750244140625, -3.7407031059265137, -7.152161598205566, 0.07368588447570801, 4.415953159332275, -1.541910171508789, -12.2866849899292, 1.0473068952560425, 7.314147472381592, -10.783853530883789, -3.5536866188049316, 1.3310492038726807, -2.1987335681915283, -14.784496307373047, -6.462701320648193, -0.3990996479988098, -11.99513053894043], [-0.5572050213813782, 4.475056171417236, -2.198352575302124, -1.783874750137329, -0.13103337585926056, -6.231899738311768, 11.9446439743042, -1.5092082023620605, -9.124658584594727, 4.124875068664551, -10.109589576721191, 9.228429794311523, 6.227411270141602, 6.558211326599121, 3.6029391288757324, -4.445210933685303, 10.450240135192871, 7.997035980224609, 4.3485870361328125, 4.0874247550964355, 11.86468505859375, -0.7034434080123901, -8.164785385131836], [-15.281824111938477, -0.25502076745033264, -0.144840270280838, -10.071475982666016, -12.88859748840332, 3.1798808574676514, -2.3655104637145996, -14.212474822998047, -7.040544033050537, -7.636778354644775, -3.270779848098755, -6.382169723510742, -2.0855777263641357, 2.787484884262085, -11.637191772460938, -13.722633361816406, 17.031553268432617, -2.6885199546813965, -5.535336971282959, 7.590419769287109, -6.867084503173828, -8.038735389709473, 3.6176655292510986], [5.0706586837768555, 2.733053684234619, 7.247247219085693, 1.2743443250656128, 9.39788818359375, 3.2114369869232178, -0.05029537156224251, 2.578026294708252, 3.4595351219177246, 4.729400157928467, -0.8205103278160095, -2.111737012863159, -7.285490989685059, -1.9277867078781128, -6.269702434539795, 3.3206984996795654, -5.547125816345215, -0.4843142330646515, 4.986360549926758, 2.5040879249572754, -0.20380879938602448, -3.5070011615753174, -1.020969033241272], [3.0893940925598145, 1.5037716627120972, 1.8384716510772705, -2.644974708557129, -0.3475123941898346, -2.545531988143921, 0.04735604301095009, 1.2510275840759277, -1.5282312631607056, 0.42273426055908203, 1.3860623836517334, -2.792145013809204, -13.806758880615234, 2.3614585399627686, 1.187386155128479, 4.639554977416992, 2.6632096767425537, -6.054389476776123, -8.524014472961426, 8.40578556060791, -9.179852485656738, -3.1496026515960693, -6.644895076751709], [0.989217221736908, 3.0074121952056885, -8.42766284942627, -9.339762687683105, -2.6559293270111084, -1.5598241090774536, -1.5731185674667358, 1.9630616903305054, -4.485517501831055, -0.397419273853302, -0.186574786901474, 12.725152015686035, 1.5917413234710693, -13.078333854675293, -2.317256450653076, 5.310949325561523, -3.0444607734680176, 2.2405858039855957, -0.33227893710136414, -2.326478958129883, -4.296026229858398, 7.669970512390137, -0.15698128938674927], [-1.5503252744674683, -2.229564666748047, 12.282089233398438, 4.679710388183594, -8.534966468811035, 2.617337226867676, -5.503462314605713, -1.0859630107879639, 3.130894184112549, -0.9665253758430481, -5.816783905029297, -7.2513017654418945, -5.03762149810791, -6.6268229484558105, -0.97459477186203, 3.882296085357666, -12.272364616394043, 0.8165837526321411, -13.045964241027832, 1.2484779357910156, -0.9859685301780701, -0.4295448064804077, 4.996843338012695], [-5.898126125335693, 4.340001106262207, 5.036340713500977, -1.4131193161010742, 4.1095290184021, -8.866012573242188, 11.951887130737305, -13.24697494506836, 0.38322868943214417, -2.5518646240234375, 4.783020496368408, -4.432531833648682, -0.8746041655540466, 1.0744935274124146, 8.311568260192871, -8.35023307800293, -3.039506673812866, -5.819027423858643, 0.7371002435684204, 11.895553588867188, 13.378145217895508, -0.02658451534807682, 1.2599999904632568], [8.906271934509277, 7.450989246368408, -3.236514091491699, 2.929058790206909, -1.6978243589401245, 12.789002418518066, -1.0951640605926514, -12.908830642700195, 13.05926513671875, 7.840479850769043, 10.426656723022461, 3.0555992126464844, -10.002317428588867, 8.310155868530273, -20.285003662109375, -15.86991024017334, -3.447427749633789, 3.308948278427124, 12.388511657714844, -0.6832876205444336, 4.904515266418457, -11.408722877502441, 12.539875984191895], [-0.6898248195648193, 4.217427730560303, 2.1814584732055664, -3.4490535259246826, -4.140094757080078, -3.303518533706665, -5.5831685066223145, -5.9688639640808105, -3.89703369140625, -6.198483943939209, 4.045876979827881, -1.811363697052002, -2.366132974624634, -2.231233835220337, -10.067562103271484, -4.511131286621094, -2.5056140422821045, -15.628265380859375, 4.372692584991455, 5.080524444580078, 2.7470507621765137, 3.6049561500549316, 0.5784765481948853], [14.836141586303711, -13.77717399597168, -7.345345973968506, -4.5163774490356445, 0.9549201130867004, 7.610487461090088, 5.450531482696533, -2.3447232246398926, -9.497823715209961, 1.3780497312545776, -1.2794877290725708, 7.697885036468506, -1.3173941373825073, 0.0871146023273468, 7.120912551879883, 1.503350853919983, -0.5119377374649048, 8.237720489501953, -0.9960911273956299, -1.3375043869018555, 5.159041404724121, 6.631533622741699, -8.776052474975586], [15.333016395568848, -6.1601128578186035, -1.724926233291626, 8.052573204040527, 1.7703430652618408, 10.943258285522461, -0.7003572583198547, 7.278365135192871, 4.07845401763916, 10.083935737609863, -0.23066310584545135, 9.683566093444824, -1.3839495182037354, 3.4627304077148438, -8.055444717407227, -1.8311984539031982, 2.344944715499878, 9.619816780090332, 5.688424110412598, 17.465591430664062, -17.425668716430664, 3.6504921913146973, -10.710896492004395], [-3.3394978046417236, -4.618653297424316, -3.04209566116333, -8.123044967651367, -4.0711350440979, -3.2915596961975098, 6.834372043609619, 2.114799976348877, -20.327821731567383, 13.812909126281738, 4.330190181732178, 8.234762191772461, 4.135457992553711, 10.323721885681152, 1.940342664718628, -2.489888906478882, 21.630321502685547, 1.5562691688537598, -4.872169017791748, 13.405426025390625, -3.5221407413482666, 3.1300294399261475, -15.650434494018555], [-1.2685449123382568, 0.9121260643005371, -2.1286368370056152, 13.794808387756348, 5.390346050262451, -0.8110786080360413, 9.219932556152344, -3.933626413345337, 2.6103615760803223, 8.59675407409668, -9.897672653198242, -7.751413822174072, -10.364197731018066, 8.570684432983398, -4.037446022033691, -14.731271743774414, 1.0523983240127563, -0.15331682562828064, 6.386302471160889, 6.514033317565918, -3.3171253204345703, -8.790511131286621, -1.9737893342971802]], "dtype": "float32"}}, {"a": [{"__nd__": [[-0.37848278880119324, 0.35877013206481934, -0.21380172669887543, 0.3659714162349701, -0.7454580664634705, -0.40423884987831116, -0.7241525053977966, -1.1421908140182495, 1.1333047151565552, -0.19900104403495789, 0.5781188607215881, 0.32769015431404114, 0.29634514451026917, -0.14984272420406342, -1.5217112302780151, -1.7034811973571777, -1.8501148223876953, -0.6513007283210754, -1.0373704433441162, 0.5625361204147339, -0.6604148745536804, 0.3114281892776489, 0.5974972248077393, 0.5375422835350037, 1.144538402557373, -0.3381427526473999, -0.277109295129776, 0.28675898909568787, 0.38964375853538513, 0.3557474911212921, -0.6796082854270935, 0.6349384784698486, -0.9671207666397095, 0.2466849535703659, 1.3309434652328491, -1.0264315605163574, -1.0002355575561523, 1.2442679405212402, -0.5640214681625366, -0.22182978689670563, -0.27683576941490173, -0.6255545020103455, 0.8849020004272461, 2.14170503616333, -0.529712438583374, -0.11745951324701309, 0.22994935512542725, -1.953227162361145, 1.1531331539154053, -0.313516229391098, -2.348815441131592, -1.2163794040679932, -1.0589097738265991, -0.2834521234035492, 1.1322001218795776, -1.9320769309997559, -0.4969478249549866, 1.0089393854141235, -1.137247085571289, 1.4374823570251465, 1.5660001039505005, -0.6918378472328186, 1.3781986236572266, -0.4083169102668762, -0.7397563457489014, 0.6983063817024231, -0.010923776775598526, 1.3686834573745728, -1.428655982017517, -2.6443355083465576, -0.24811354279518127, 1.5497121810913086, -0.03077269159257412, -0.7217203974723816, 1.9755233526229858, 0.7986875176429749, 0.5108265280723572, 1.8253748416900635, 1.3705427646636963, 1.1748361587524414, 0.6888577938079834, 0.4489785432815552, 0.9181240200996399, 0.3119005560874939, 1.0572643280029297, -0.3884642422199249, -0.27321740984916687, -0.1548280417919159, 1.7496567964553833, 1.1097018718719482, -0.5247269868850708, 0.2359965592622757, 1.5812931060791016, 0.24855923652648926, 1.8807847499847412, -0.9089154601097107, 0.12293876707553864, 0.5372487306594849, -0.44758573174476624, -0.20502568781375885, -0.15961897373199463, -1.5511984825134277, 1.1318979263305664, -0.1936158537864685, 1.130326271057129, 1.8551678657531738, -0.3412875533103943, 1.2727104425430298, -0.7898711562156677, -0.24132806062698364, 0.4236929714679718, 2.659836530685425, -0.6888571381568909, -0.9834270477294922, -0.8259102702140808, 0.5066719055175781, 2.015083074569702, -2.1553092002868652, -0.22102396190166473, 0.6171557307243347, -0.4174460470676422, 1.2209168672561646, 1.3599773645401, 0.14989925920963287, 0.5106721520423889, -0.9703629016876221, -0.31264162063598633, -0.10115350782871246, 0.15296900272369385, 0.6694508790969849], [-0.3554542362689972, 1.0380865335464478, -0.11682933568954468, -1.3336900472640991, 1.0019704103469849, -0.7829434275627136, -0.029831543564796448, 0.17834818363189697, 1.8229373693466187, -1.0967650413513184, 1.5458089113235474, 0.35370171070098877, 1.7151086330413818, -0.22970183193683624, 1.1257853507995605, -0.31217968463897705, -0.43052053451538086, -0.7308512330055237, -1.0456302165985107, 0.33470824360847473, 2.272742748260498, 1.0289570093154907, -0.8085303902626038, -0.5929310917854309, -0.25661465525627136, 0.31440654397010803, 1.6871232986450195, -0.617821991443634, -1.8021681308746338, -0.06168770790100098, 0.6768441200256348, 0.2575618326663971, -0.5252869725227356, -1.128472089767456, -1.2790136337280273, -0.37617236375808716, -0.37590107321739197, 1.1687668561935425, 1.2107696533203125, 1.0653536319732666, 0.9172306060791016, -1.6364197731018066, -1.0783677101135254, 1.1533702611923218, 0.6458162665367126, 0.19996318221092224, -0.4182250499725342, -1.3335199356079102, -0.30059051513671875, -0.0662320926785469, 0.1789737194776535, -1.7428239583969116, -0.3076392412185669, 0.2547418177127838, -0.7238537669181824, -2.222438097000122, 0.5251030921936035, 2.332019805908203, 0.1378464698791504, -0.5554017424583435, 1.2333885431289673, 1.3070248365402222, -0.6894415020942688, 0.754996657371521, 0.046194590628147125, 2.2558882236480713, -0.32404789328575134, -1.7509994506835938, -1.8656840324401855, 1.065547227859497, -0.6973511576652527, 0.5450095534324646, -0.28455907106399536, -0.2161695659160614, -0.6823331117630005, -0.8752738237380981, -0.37256520986557007, 1.1612472534179688, 0.34895652532577515, 1.6601465940475464, 0.24005471169948578, 0.6338828802108765, 1.6830766201019287, 1.0847448110580444, -1.6615569591522217, 0.8683544397354126, 0.4596114158630371, 0.27653396129608154, 0.6253520250320435, 0.9940037727355957, -0.16921833157539368, -1.2112212181091309, 0.07973659783601761, -0.8347129225730896, -0.8994110226631165, -1.4848917722702026, -0.7865249514579773, -1.0742453336715698, -0.8237001895904541, -0.391265869140625, 0.3136110007762909, -1.2020771503448486, 1.0898023843765259, 0.5076873302459717, -0.6291904449462891, -0.40970566868782043, -0.5195990800857544, -0.1636619120836258, 1.9027243852615356, -0.7660152912139893, 0.07589972764253616, -0.8997259736061096, -0.023005042225122452, 0.4062004089355469, 0.5673352479934692, 2.445730447769165, -0.7316396832466125, 0.7291167378425598, 0.6971726417541504, 0.6432750821113586, 0.8452722430229187, 1.3556793928146362, -0.07792661339044571, 0.13773301243782043, -1.4151692390441895, 1.2789843082427979, 0.7923469543457031, 1.2856992483139038, 0.3146001100540161, 0.5265938639640808], [0.7313348054885864, -1.1086442470550537, -0.8493873476982117, -0.40665116906166077, -1.0805094242095947, -0.6068484783172607, 0.6070205569267273, 0.34004703164100647, -0.06904801726341248, -1.368666172027588, -0.6056720018386841, -1.6397254467010498, 0.3058894872665405, -0.554527759552002, 0.2144170105457306, -1.3895108699798584, 1.7803025245666504, 0.5410903096199036, 1.4157408475875854, 0.46834999322891235, 0.06024571880698204, -0.4511064887046814, 0.7467462420463562, 0.5865061283111572, 0.8435489535331726, 0.8937814831733704, 0.5104559659957886, -2.1911728382110596, 1.9897980690002441, 1.602553129196167, 1.2444955110549927, 0.403555303812027, -0.12523892521858215, 0.1387365311384201, 0.7875176072120667, 0.17767050862312317, -0.05134041979908943, 0.950934112071991, -0.32645854353904724, -1.2748558521270752, 1.01854407787323, -0.31338298320770264, 0.08062155544757843, -0.1487167775630951, 1.762202501296997, -0.5957223773002625, -0.17321385443210602, 0.24968324601650238, -0.556168794631958, 1.4170416593551636, 1.5604267120361328, -0.8123112320899963, -1.5440759658813477, -0.19051142036914825, 0.694915235042572, -0.5585925579071045, 0.01765250600874424, -0.5415347814559937, -0.21243666112422943, 0.5194101333618164, 1.3044131994247437, 1.2258089780807495, -0.9456723928451538, 0.8335492610931396, -0.7146729230880737, -0.19096624851226807, 0.4580257534980774, -1.5193884372711182, -0.8182381391525269, 0.27006950974464417, -0.9296465516090393, -0.7330566048622131, -0.959256112575531, 1.1100714206695557, 0.10977765172719955, 1.0600987672805786, -1.1766607761383057, 0.3093011677265167, -0.0725645050406456, 0.7686294317245483, 1.0675804615020752, 0.8195585608482361, 1.6344705820083618, 0.49854356050491333, -1.3217841386795044, 0.7217164635658264, 0.4357426166534424, 0.40755021572113037, 1.6707020998001099, 0.7709708213806152, 0.3981269896030426, -0.6022084355354309, -0.17941641807556152, 1.3693690299987793, -0.616850733757019, -1.717197299003601, 1.0041756629943848, -0.49868354201316833, 0.16319751739501953, -0.8118710517883301, 0.5625407695770264, -0.7323703169822693, 0.014990516006946564, -1.023108720779419, -1.382846474647522, 0.42680513858795166, 0.18427756428718567, 0.24731479585170746, -0.38736945390701294, 0.6740463376045227, -1.0278156995773315, -0.46133941411972046, -0.06187974289059639, -0.5035901069641113, -0.6455796957015991, -0.9396596550941467, -0.8530191779136658, -1.7245538234710693, 0.2749099135398865, -0.39893069863319397, 1.4701197147369385, 0.24974149465560913, -0.5136609077453613, -0.23751817643642426, 0.21277198195457458, 1.6096689701080322, -1.881850004196167, 0.5545007586479187, -0.5194132328033447, 0.3321399688720703], [1.861285924911499, -0.8769339323043823, -0.20662301778793335, -0.9762836694717407, 0.29847580194473267, 1.2254228591918945, 1.6687692403793335, 1.0519553422927856, 0.649509847164154, 0.23091866075992584, 0.9643974304199219, 0.2570813000202179, -0.4616119861602783, -0.37768644094467163, 1.3142858743667603, -0.2617437541484833, 0.7888892889022827, -2.3893113136291504, -0.35895535349845886, -1.2978459596633911, 0.24755613505840302, 0.8799670338630676, 0.6036013960838318, -0.0070521957240998745, 1.2845145463943481, -0.706688404083252, -1.0781210660934448, 0.3357410132884979, 0.6268723011016846, -0.4719044268131256, 1.3038471937179565, 0.669704794883728, -0.05895060673356056, -1.305625081062317, -0.10170283913612366, 0.40681910514831543, 0.40424972772598267, -0.04593741148710251, -0.6276535987854004, 0.5075743198394775, 0.3546467125415802, 1.2761600017547607, -0.9617400765419006, 1.1527491807937622, 1.278290867805481, -0.599016547203064, -0.1594851166009903, 1.2006969451904297, -1.0965328216552734, -0.49410319328308105, 0.40084272623062134, 0.11666227132081985, -0.39795982837677, 0.1767527461051941, -0.3887174129486084, -0.8666569590568542, 0.7595940232276917, 1.448498249053955, -1.1249513626098633, -0.48818811774253845, 0.043075378984212875, 1.1272188425064087, 1.2183082103729248, 1.307060718536377, -0.5396357178688049, 1.1215709447860718, -1.0979210138320923, 0.4981619715690613, 0.35613811016082764, -0.3311462998390198, 1.1180169582366943, 0.1757771223783493, -0.025258051231503487, 2.397465705871582, -0.9876226186752319, 1.5051885843276978, 0.7136590480804443, -0.09667705744504929, 0.6787788271903992, 0.07403773069381714, 0.7541494965553284, 1.0388383865356445, -1.0074489116668701, -2.5953264236450195, -0.7695564031600952, 0.9048706889152527, 0.22993877530097961, -2.2674574851989746, -0.08687955141067505, -2.653421401977539, 0.462486207485199, -0.21601739525794983, 1.3473082780838013, -2.568500280380249, -0.8524900078773499, -0.2414463311433792, 0.5987251400947571, -1.0787875652313232, -0.9159884452819824, 0.09293422847986221, 0.11116331815719604, 0.6286918520927429, -0.6480729579925537, -0.10627296566963196, -0.8065345883369446, 0.42674747109413147, 0.14093957841396332, 1.9016854763031006, 0.3982805907726288, -0.11661386489868164, 0.3215479254722595, 0.7084643244743347, -0.1081877127289772, 0.052056241780519485, 1.1408586502075195, -0.39234253764152527, -0.9260473251342773, 2.4094035625457764, 0.9463398456573486, -1.165963888168335, -0.8263022899627686, 2.971296787261963, -0.38543060421943665, -0.8571003675460815, 1.0079954862594604, -1.0201915502548218, -0.44641587138175964, 0.17268036305904388, -0.811463475227356, 0.027017876505851746], [-0.3325212895870209, 1.0326491594314575, 1.235241174697876, -2.2905945777893066, 0.7332708239555359, -0.4302840530872345, -0.06330138444900513, -0.8751125335693359, 0.5649458169937134, -1.6235121488571167, -1.5116140842437744, 0.7714672088623047, 0.8739103078842163, 0.4470185935497284, -0.34162843227386475, -1.1808147430419922, 1.437885046005249, -1.1397298574447632, -2.8441998958587646, -0.8263559937477112, 0.553810179233551, 0.2756020128726959, -1.2390559911727905, 0.1940673440694809, -0.6057876944541931, 0.233321875333786, -1.3973031044006348, 0.3331281542778015, -0.3414522111415863, -1.642632007598877, -0.5088359713554382, 0.5097521543502808, 0.623176097869873, 1.2160453796386719, -0.28134316205978394, -0.7022241353988647, -1.4309666156768799, -0.006612242199480534, -1.0120230913162231, 0.29757189750671387, -0.48355385661125183, -0.47736766934394836, -0.7877868413925171, 0.32875069975852966, 0.05907890200614929, -0.41142839193344116, 0.7323839068412781, -1.170117974281311, -0.058085616677999496, 1.2663898468017578, 1.93248450756073, 0.5389607548713684, 0.953111469745636, 0.28484004735946655, 0.8753032684326172, 0.1367422342300415, -0.31387245655059814, 1.2242838144302368, -0.930316686630249, -0.2565736472606659, -1.756272315979004, 0.08133047819137573, -1.0811359882354736, 1.945478081703186, 0.1877327859401703, -0.501654863357544, 0.4089067876338959, 1.6652591228485107, 0.6558029651641846, 0.028357641771435738, 0.36800968647003174, 0.7075574994087219, 0.3236269950866699, 1.005077600479126, -1.009900450706482, 1.022379994392395, -1.5581918954849243, -0.5903169512748718, -1.6562376022338867, -0.23885931074619293, -0.40424343943595886, -0.3750992715358734, -0.8712974786758423, -1.2986798286437988, 0.358550101518631, 0.8144316673278809, -1.1314594745635986, 0.10186703503131866, 0.2426842302083969, 0.03348061069846153, -1.262847900390625, -0.48377490043640137, -0.2411186397075653, -1.1032041311264038, 0.11141681671142578, -0.025642728433012962, -1.3116531372070312, 1.6427040100097656, -0.37744206190109253, -0.15898437798023224, -0.08771687000989914, -0.46512213349342346, 0.5391592383384705, 0.9200239181518555, -0.6679974794387817, 0.589078426361084, -1.5239901542663574, -1.3373712301254272, 0.9383327960968018, 0.6544927358627319, -0.6670957803726196, 1.0620619058609009, 0.6551462411880493, 0.351441353559494, -0.3745144009590149, -1.3092643022537231, 0.03466472774744034, -0.34899574518203735, 1.7357145547866821, 0.20599178969860077, -0.10094118118286133, -1.0555751323699951, -0.23301328718662262, 1.189833402633667, -1.2284029722213745, -0.8582446575164795, 0.19890457391738892, -0.8594738841056824, 0.5678325891494751, 0.4306294918060303], [-1.3115570545196533, 0.4158283472061157, -0.6433470249176025, -0.7209029793739319, 0.8782284259796143, -0.49946123361587524, 0.7310221195220947, 1.4643827676773071, -0.763148844242096, 0.06143125146627426, -2.2921226024627686, 1.7973673343658447, -0.32712507247924805, -0.27978217601776123, 1.0394798517227173, 0.004113425966352224, -1.2040048837661743, -0.7590042948722839, -0.3131369948387146, -1.1168490648269653, 1.0307717323303223, 0.5338672399520874, 0.19662891328334808, -0.602612316608429, -1.0695186853408813, 0.2872104346752167, -0.31387484073638916, -2.0688350200653076, 0.14078330993652344, 0.36614862084388733, -2.2447144985198975, -2.5407228469848633, -0.07943283021450043, 0.4114840030670166, 0.2860918343067169, -0.8765102028846741, -0.3109491467475891, -1.720932960510254, 0.29148927330970764, 0.07425438612699509, -0.1810591071844101, -0.4072917699813843, 0.913015604019165, 1.0283454656600952, -0.7040346264839172, 0.21102549135684967, -0.859127402305603, 0.27874988317489624, -0.8872634172439575, 0.7355971932411194, -1.6649144887924194, 0.12373453378677368, -1.6686420440673828, 1.5706372261047363, -0.5591870546340942, 0.2010004073381424, 0.537950873374939, 1.5283149480819702, -0.2812396287918091, -0.7585537433624268, -1.520881175994873, 1.5552468299865723, 0.8562097549438477, -0.2323872596025467, 1.1768816709518433, -1.0920865535736084, 0.028672942891716957, -0.13594432175159454, 0.7635984420776367, -0.33971378207206726, -1.3232237100601196, -0.04092885181307793, -0.9112222194671631, -1.2400336265563965, 0.3421562612056732, 0.01783607341349125, 0.017903964966535568, 1.2766534090042114, -0.13078472018241882, -1.9614418745040894, -0.09767605364322662, -2.3896586894989014, 1.7532405853271484, 3.7887232303619385, 0.4087657034397125, -0.3692808151245117, 0.5591932535171509, -0.7044963240623474, -1.729021430015564, -1.250977635383606, -0.9698431491851807, 0.6935157179832458, -0.561674177646637, 0.13826651871204376, 0.5406787991523743, 2.148751974105835, 1.448112964630127, 0.04006025195121765, -0.12638501822948456, -1.0379788875579834, -0.06498280167579651, -0.16964292526245117, 0.4852465093135834, 1.0852094888687134, -1.1372628211975098, -0.3000415861606598, -0.007125270087271929, 0.8323032855987549, -0.7887194156646729, -2.144134283065796, -0.18215630948543549, 0.29470160603523254, 1.2084200382232666, -0.8181787729263306, 1.3401910066604614, 0.8016383051872253, -0.3354058861732483, 1.1838569641113281, 0.8067042827606201, -0.590713381767273, -2.0296690464019775, -0.11257626116275787, -0.530551552772522, -0.06920378655195236, 0.2343205213546753, 0.5749005079269409, -0.8656452894210815, -1.6476205587387085, 0.02513718418776989, 0.380950003862381], [1.8487162590026855, -1.1398569345474243, -1.9412987232208252, 1.5543545484542847, -1.0421510934829712, -0.6406004428863525, -1.0371226072311401, -0.5052761435508728, -0.40247634053230286, -0.09386920928955078, -1.0015501976013184, -0.09352434426546097, 1.3686951398849487, 1.3082104921340942, 0.5570582151412964, 0.07976648211479187, 0.24619175493717194, -0.01901943050324917, 1.7698824405670166, 2.1187984943389893, 1.0424035787582397, -0.41938522458076477, 1.7132624387741089, 0.7960376143455505, 0.6534450054168701, 1.388527274131775, 0.5418898463249207, -0.9555646181106567, 1.5818450450897217, -0.12420061230659485, -0.4743827283382416, -0.8594372272491455, -0.09918878227472305, -0.1194574162364006, -0.6114040017127991, 0.38348865509033203, -0.27914392948150635, 0.3587673008441925, 0.6994047164916992, -0.3544071316719055, -0.1024412214756012, 0.4419398307800293, 0.7102419137954712, -0.33254656195640564, 1.6413856744766235, 0.032016098499298096, 1.5678207874298096, -0.48577484488487244, -0.41667574644088745, 1.408409595489502, 1.9211028814315796, -0.4159438908100128, 0.8623048663139343, 0.19358478486537933, -1.2791088819503784, -0.348317950963974, 1.7822827100753784, 0.44950929284095764, 0.1925097554922104, 0.6288728713989258, -1.342828631401062, -0.8272436261177063, -0.5503342747688293, -1.5460329055786133, -0.3264405429363251, -1.3433440923690796, -0.4821542799472809, 0.6957504153251648, -0.6587414741516113, 0.4528718590736389, 1.4511007070541382, -1.5486476421356201, 0.2784704566001892, 1.1613763570785522, -1.3809236288070679, 1.1090720891952515, -0.4043816030025482, -0.5951130390167236, -0.1073090061545372, -1.1344964504241943, 0.23208054900169373, -1.1403812170028687, -0.12387168407440186, -1.0837650299072266, 0.5828599333763123, 1.3159396648406982, 0.6435085535049438, 1.063727617263794, 0.7748727202415466, -1.0490732192993164, -4.096689224243164, 1.8163930177688599, -1.2932196855545044, 0.24987509846687317, -1.9405773878097534, -1.3121262788772583, 2.2695910930633545, -0.23245608806610107, 0.11765018105506897, -0.17851772904396057, -1.045546531677246, 2.4405276775360107, 1.664061188697815, 0.4598759710788727, 0.9870719313621521, -0.32114505767822266, -0.09873468428850174, 0.0012085000053048134, -1.597115159034729, 0.3722281754016876, 1.2988486289978027, -0.7106826901435852, 0.703762412071228, 0.2054096758365631, 0.061349544674158096, -0.2859906554222107, -0.592904806137085, -0.15053018927574158, 0.029079128056764603, -0.6286792159080505, 0.3329448997974396, -0.9774748086929321, -1.2357758283615112, 0.12592972815036774, 0.22989045083522797, 0.4797544777393341, -0.9933339357376099, -0.667604923248291, -1.6883295774459839, 1.483791708946228]], "dtype": "float32"}, {"__nd__": [[-1.7985543012619019, -1.3231346607208252, 0.1204332485795021, -1.1566505432128906, -0.5989471077919006, -0.37070024013519287, -0.4588681161403656, -1.004054069519043, 1.9515659809112549, 0.32871001958847046, -1.4039740562438965, 0.3970869481563568, -1.5168938636779785, -0.5426534414291382, -1.5397926568984985, 0.007351747248321772, 1.6005945205688477, 0.041499365121126175, -0.7096234560012817, 0.28725388646125793, -0.8888938426971436, 0.5297281742095947, 1.7088960409164429, -0.184479758143425, -0.24674654006958008, 0.7005047798156738, -0.42369920015335083, 0.5108510255813599, -1.5595730543136597, 1.9656026363372803, 0.45505666732788086, -1.0552482604980469, -1.3998671770095825, 0.11190124601125717, 0.05053703114390373, -0.12513476610183716, -0.9390674829483032, 1.9072881937026978, 0.7629911303520203, 0.5592089295387268, -0.4875528812408447, 3.942065954208374, 0.48466333746910095, 0.4465973377227783, -0.7271105051040649, 0.13014787435531616, 0.5049954056739807, -0.6758873462677002, -0.32039758563041687, 0.7285432815551758, -0.464549720287323, 1.235845685005188, -0.4290406107902527, 0.9380866289138794, -0.9604654312133789, 0.21078073978424072, -0.7367426753044128, -0.9315478801727295, -0.4892798960208893, 0.5335972309112549, 0.5875012278556824, -1.4446768760681152, 1.6437469720840454, 0.5762895345687866, 0.17233943939208984, -0.4824153482913971, 2.3581113815307617, -0.7562970519065857, -0.594516396522522, 0.028936410322785378], [2.3314034938812256, 0.7268617749214172, -0.06193811073899269, -0.9816762208938599, 1.6583659648895264, 0.08687581866979599, 0.7024234533309937, 1.3315937519073486, -0.43868938088417053, 0.8982930183410645, -1.3439123630523682, -0.32570624351501465, 0.49146226048469543, -0.6664103269577026, 0.23254971206188202, -0.14005820453166962, 0.3143772780895233, -2.013219118118286, 1.9148578643798828, 0.07753046602010727, 1.5282745361328125, 0.5256595611572266, -0.565190851688385, 0.6940172910690308, -0.6191236972808838, 1.5379010438919067, -0.6026937961578369, 0.9160334467887878, -0.6464903354644775, -0.7444907426834106, -0.49995285272598267, -0.1390680968761444, 0.9665500521659851, 0.709047257900238, -0.7627048492431641, 0.8742847442626953, -1.7523106336593628, 0.5199307203292847, 1.0926436185836792, 1.9672086238861084, 0.33520886301994324, -0.24554122984409332, -0.9471971988677979, -0.26014986634254456, 0.07850250601768494, -0.2518870532512665, -0.7685503363609314, 0.744915246963501, -0.2334350347518921, -0.6081417798995972, -1.1639368534088135, 1.5374614000320435, -0.1601104885339737, 0.8922132849693298, -0.06532429158687592, 1.2044252157211304, -0.7597507238388062, 0.5872464776039124, 0.4636865556240082, 0.5471985340118408, -0.7383351922035217, -1.4986019134521484, 1.2497578859329224, 0.2506844103336334, 1.9100239276885986, 0.06136450543999672, -0.8711749911308289, -0.08854540437459946, 0.7340213656425476, -0.1963537186384201], [0.3512982130050659, 1.6469067335128784, -0.9897242188453674, 0.1564311385154724, -0.39682477712631226, -0.5748998522758484, 0.367160439491272, 1.1080039739608765, -0.7332590222358704, -1.0799838304519653, -0.2668585777282715, 1.4351726770401, -0.49586525559425354, -0.3564963936805725, 0.42858678102493286, -2.1603825092315674, 1.3305459022521973, -1.0125923156738281, -0.9359242916107178, 0.16808733344078064, 1.3165887594223022, 0.005127917975187302, -0.34418362379074097, -0.45424944162368774, -0.5748592615127563, 1.2475234270095825, -0.8705190420150757, 1.3994485139846802, 0.10594166070222855, 0.18775328993797302, 0.9851841926574707, -0.39192673563957214, -2.7682127952575684, -2.502164602279663, 1.3546936511993408, -0.038227736949920654, -1.8354893922805786, -2.6845450401306152, 0.39821672439575195, -0.805679202079773, 0.03706710413098335, -1.3210176229476929, 1.709035873413086, -0.7857188582420349, -0.4494663178920746, -1.4111703634262085, -0.5884712338447571, -0.9240479469299316, -1.2077456712722778, 0.15565048158168793, -1.3565089702606201, 0.6874275207519531, -1.9898380041122437, -1.5102633237838745, 0.9893826842308044, 1.091951608657837, -0.33579620718955994, -2.132108688354492, -0.7342912554740906, -0.3780064582824707, 0.054271943867206573, 0.4036557376384735, -0.20863863825798035, -1.8989070653915405, -1.7052202224731445, -2.4749643802642822, 1.2947229146957397, -2.182642936706543, -1.1385194063186646, 1.2471556663513184], [0.8751564621925354, 0.4219054579734802, -0.22787393629550934, 0.3112485408782959, 1.3659040927886963, 1.67489755153656, 1.227434515953064, -0.3837357759475708, -0.8897018432617188, -0.21381619572639465, 0.28795546293258667, 0.45628929138183594, -0.429699182510376, -0.5054570436477661, -1.0933985710144043, -3.497394561767578, 0.03481514751911163, 0.25233912467956543, 0.15956367552280426, -1.1265716552734375, -0.8476259112358093, -0.05225876718759537, -0.39613044261932373, -0.14416004717350006, -0.8686413764953613, 1.0557523965835571, 0.5210545063018799, -1.931731104850769, -0.0939706489443779, 0.7227106094360352, -0.9253447651863098, -0.8599650263786316, 0.9357032775878906, 1.0420172214508057, -0.08988073468208313, -1.5324114561080933, 0.32902899384498596, 0.14881408214569092, 0.26288560032844543, 0.8465619087219238, -0.030351726338267326, 1.561821699142456, 0.025136098265647888, -0.0010658594546839595, 0.5900961756706238, -0.29257193207740784, -0.2950544059276581, 1.1152206659317017, -0.2857986092567444, 0.07204140722751617, -1.3366693258285522, 0.459688663482666, 1.9364105463027954, 1.3421432971954346, -0.8488068580627441, 3.2976303100585938, -0.9497658014297485, -0.78438401222229, -0.28270459175109863, 0.9394869208335876, -0.7949514389038086, 0.06309279054403305, -0.9618950486183167, -1.111276388168335, 0.4500729739665985, 0.1554282009601593, 1.7050397396087646, -2.7598118782043457, -2.0665900707244873, 0.36747342348098755], [0.7731391787528992, -0.28751814365386963, 0.8796846866607666, 0.7402811050415039, -0.023986592888832092, 0.8158201575279236, -1.1738418340682983, 1.5135248899459839, 0.09539738297462463, -0.28296399116516113, 0.5216929316520691, 0.0018811995396390557, -0.8852485418319702, -0.023962097242474556, 0.4375118613243103, -0.2322169542312622, -1.7905759811401367, 0.030720509588718414, -1.0989453792572021, -0.16758275032043457, -0.5041335821151733, 0.8424755930900574, 0.1893860250711441, -0.3304150104522705, 0.2073536515235901, 0.7271979451179504, -0.22067859768867493, 0.7053835988044739, -0.3716124892234802, 0.4858321249485016, 0.6799869537353516, -0.8944390416145325, 0.32495731115341187, -0.4609774649143219, 1.2779474258422852, -1.5343221426010132, -0.6530290246009827, -0.1040220707654953, 0.574815034866333, -0.7978223562240601, 0.7915789484977722, -1.899801254272461, 0.4138932228088379, 0.3915693163871765, -1.0065914392471313, 0.39032360911369324, -0.9262029528617859, -0.165045827627182, 0.14449267089366913, 0.5939439535140991, -1.6935105323791504, 0.12938596308231354, 0.0026507368311285973, 0.8087984919548035, -1.3066085577011108, 2.1838390827178955, 1.708807110786438, 1.5515328645706177, -2.5122106075286865, -0.969017744064331, -1.3027291297912598, 0.9093619585037231, -0.31399109959602356, -0.15272779762744904, -0.5451608300209045, -1.3992339372634888, 1.4830279350280762, 0.7086696624755859, 0.7965219616889954, 2.256540298461914], [1.0333648920059204, -0.8012827038764954, 0.12926679849624634, 0.6682159900665283, 0.45934706926345825, 0.13079681992530823, -0.3682915270328522, 0.621865451335907, 0.27466174960136414, -0.6363319754600525, 0.7544572949409485, -0.4629250466823578, -0.8537757396697998, 1.0190112590789795, 0.5312598347663879, 0.5386170148849487, -0.2537875175476074, -1.1058424711227417, -1.6106593608856201, 0.5528172254562378, -0.6805547475814819, -0.06713298708200455, -0.39464861154556274, -0.32398536801338196, 1.346367359161377, 1.1576112508773804, -2.3483664989471436, -0.47535330057144165, -0.1272997111082077, -0.5062943696975708, -0.935515284538269, -1.689886212348938, -0.4284214377403259, -0.23903506994247437, 1.1087965965270996, 0.04487725347280502, 0.004916924983263016, 0.05581939220428467, 0.4238074719905853, -0.6019240617752075, 0.468624085187912, -0.630616307258606, -2.299739360809326, -0.16770613193511963, -0.7340688705444336, 1.9779056310653687, 0.5784158110618591, -0.5831166505813599, -0.5788723230361938, 0.961264967918396, 1.0457708835601807, -0.11890140920877457, -0.8919278383255005, 0.16726207733154297, 0.18792100250720978, -1.5121428966522217, 1.3480621576309204, -1.841557264328003, -0.0011962049175053835, 0.19260497391223907, 0.6227724552154541, 1.1509066820144653, 0.07798448204994202, -0.5569815635681152, -0.06715219467878342, 0.22685153782367706, 1.0218244791030884, -0.12544293701648712, 0.7710090279579163, 0.5973790884017944], [-1.0409691333770752, 1.1401395797729492, 2.1858785152435303, -0.5881944894790649, 1.7861591577529907, -0.11777884513139725, -1.167724609375, 1.3425707817077637, -0.13769936561584473, 0.976686954498291, 0.7054409980773926, 0.3322785794734955, 1.2300423383712769, -0.14998754858970642, 0.9351469278335571, -0.13291913270950317, -0.33141347765922546, -1.6130093336105347, 0.6671874523162842, 1.7742489576339722, -0.3346404433250427, -0.04589267820119858, -0.985902726650238, -0.05284816026687622, 0.11629095673561096, -0.36811646819114685, 0.48162102699279785, -0.6630655527114868, -1.2162247896194458, 0.23954008519649506, 0.0943341925740242, 1.7468560934066772, 0.9382359981536865, -1.4725005626678467, 0.13545861840248108, 0.6785922646522522, 1.7814027070999146, -1.3847401142120361, 0.8882258534431458, -0.10143177211284637, 1.2968199253082275, -1.6753429174423218, -0.8668050169944763, -0.8599655032157898, 0.1145971417427063, 0.25606080889701843, -0.12635576725006104, -0.4270201623439789, 0.031771305948495865, -0.31089338660240173, 0.4714500308036804, 0.059499990195035934, 0.06355701386928558, 0.6624093651771545, 0.24709565937519073, -0.320329874753952, -1.3859978914260864, 0.09383200854063034, 0.8052107691764832, -0.7379748821258545, 0.45017823576927185, 0.18463821709156036, 0.6038942933082581, -2.832130193710327, -0.040054090321063995, -1.1972460746765137, -0.9638874530792236, 0.4123924672603607, 0.22943128645420074, -2.1396632194519043]], "dtype": "float32"}], "e": {"__nd__": [[-1.79901123046875, 5.486473083496094, 2.4074904918670654, -0.72952800989151, 4.596961975097656, 2.1458775997161865, 1.191689372062683, 1.1659176349639893, -3.421470880508423, 1.1027783155441284, 1.4910563230514526, 3.0851755142211914, 2.925128221511841, -2.365062952041626, -0.3350009620189667, -8.91755199432373, 0.6359032392501831, -1.1128045320510864, 2.911790132522583, 0.5005333423614502, -0.38004133105278015, -0.7577940821647644, -2.802938938140869, -0.3403064012527466, -3.343543291091919, -0.3749929666519165, 4.751561164855957, -3.927917003631592, -1.2352827787399292, 1.948493242263794, 0.17891623079776764, 4.304798603057861, 2.0917584896087646, -2.4402573108673096, -0.553350031375885, -1.4377597570419312, 3.752352237701416, -5.191615104675293, 0.9984628558158875, 0.9428010582923889, 1.5556021928787231, -1.1022549867630005, 2.726045846939087, -2.1532466411590576, 2.526266098022461, -3.786865711212158, -1.5817368030548096, 1.421217441558838, -0.44106560945510864, -1.8446587324142456, -2.8273181915283203, 0.5670292377471924, 3.6547040939331055, 1.4577041864395142, 0.1752529740333557, 7.093410015106201, -6.363056182861328, -0.8025541305541992, 1.2826964855194092, -0.21895644068717957, -0.9512203335762024, 0.021592630073428154, -1.8907512426376343, -8.21886920928955, -0.9582231640815735, -3.4055795669555664, -0.07767320424318314, -5.724048137664795, -5.566980361938477, -3.8345704078674316], [3.032651662826538, -3.845669984817505, -0.2534371614456177, -0.16763603687286377, -1.1209532022476196, 0.15686942636966705, -0.953154981136322, 0.42140448093414307, 2.207581043243408, 0.7651669383049011, -1.8071192502975464, -2.7561845779418945, -1.7787275314331055, 0.521954357624054, -0.22056140005588531, 5.455007076263428, -2.1818411350250244, 0.23679572343826294, 0.0657956525683403, -0.9804458022117615, 0.1291111707687378, 1.6702501773834229, 1.9105925559997559, 0.8685982823371887, 1.3092525005340576, 1.1908178329467773, -2.0228734016418457, 2.563258409500122, -0.3160445988178253, -0.8914461731910706, -0.4308394491672516, -2.951826810836792, 1.8375076055526733, 3.739443778991699, -0.5703393816947937, -0.09036016464233398, -3.112440824508667, 5.563915252685547, 0.49334609508514404, 1.4350429773330688, -0.3073200583457947, 0.9399215579032898, -2.2670021057128906, 2.0770394802093506, -1.6738770008087158, 2.539928674697876, -0.2773895263671875, 0.651094913482666, 1.1045958995819092, 0.7617701292037964, -0.550203800201416, 0.8905196189880371, -0.2528146505355835, 1.9098312854766846, -2.3177034854888916, -0.7849923968315125, 4.057152271270752, 4.056466579437256, -2.144765853881836, 0.27530592679977417, -1.5181608200073242, -1.3696538209915161, 1.981740951538086, 6.385616302490234, 2.995171070098877, 2.5122830867767334, 0.06611552089452744, 4.6862688064575195, 4.504775047302246, 3.1191790103912354], [1.9439750909805298, -3.3410704135894775, -2.370753288269043, 1.791192889213562, -3.803476333618164, 1.3635778427124023, 0.504429280757904, -1.9395745992660522, 0.6491090059280396, -1.0499094724655151, -0.5860844254493713, -1.705049991607666, -2.15523624420166, 0.20711290836334229, -1.4528244733810425, 2.1971089839935303, -2.9214279651641846, 4.914995193481445, -0.9264599680900574, -3.9874744415283203, -0.4669117033481598, 1.0047147274017334, 2.476625919342041, 0.27686789631843567, -0.04295901954174042, -0.7390713095664978, 1.095999002456665, 1.458570122718811, 2.322321891784668, -0.18125084042549133, 0.5741897821426392, -2.6563868522644043, 1.199947714805603, 4.2461676597595215, -0.451520174741745, -2.967766284942627, -2.3714942932128906, 4.304736137390137, -1.9702719449996948, -0.24132056534290314, -1.8013473749160767, 1.2965466976165771, 2.223727226257324, 2.8635354042053223, -0.5874598026275635, -0.026742126792669296, -0.7282864451408386, 1.612170934677124, 1.669891595840454, 0.4869660437107086, -2.016221046447754, -1.0019105672836304, 1.8541817665100098, 0.3061988055706024, -2.666566848754883, 2.4976229667663574, 4.661895751953125, 5.022738456726074, -3.933023452758789, 0.060688674449920654, -2.8049662113189697, 0.15246959030628204, -1.7318446636199951, 7.357711315155029, 0.5429577827453613, 2.6159369945526123, 1.1912916898727417, 2.7516965866088867, 1.477878212928772, 5.438281536102295], [-8.898785591125488, 0.4731619954109192, 2.0410637855529785, -2.5731945037841797, -1.1029608249664307, -3.798997640609741, -1.3131381273269653, -4.047669887542725, 1.8355202674865723, 2.195183277130127, 0.46356821060180664, 0.39652490615844727, 3.9657113552093506, 0.4157736599445343, 0.08791330456733704, 4.4194769859313965, 3.3607382774353027, 1.085263729095459, 2.126699924468994, 3.776374101638794, -0.9461935758590698, -2.4109745025634766, 0.22414924204349518, 0.24060796201229095, 0.5524185299873352, -6.405163764953613, 3.441084861755371, -2.021620750427246, -0.6073409914970398, 0.5548285245895386, 0.5995626449584961, 6.780505180358887, -0.5663481950759888, -1.9650472402572632, -2.9434752464294434, 4.836723804473877, 6.679757118225098, -1.003369927406311, -1.8381755352020264, -0.8140907883644104, -0.7462376356124878, 2.9847824573516846, 0.08362407237291336, -1.281762719154358, 2.248884916305542, -0.6788849830627441, 3.2453532218933105, -1.8191651105880737, 1.0999459028244019, -1.5926318168640137, 7.096960544586182, -2.4447128772735596, -0.2890830636024475, -2.486375570297241, 3.4034740924835205, -9.602744102478027, -5.232900619506836, -1.5718193054199219, 6.784025192260742, -0.3642981946468353, 5.188549518585205, -1.3814618587493896, 1.5603731870651245, -1.9170833826065063, -0.9953832626342773, 1.77690851688385, -5.798099040985107, 2.5314040184020996, -0.7396886348724365, -9.518717765808105], [6.117673873901367, -2.0416698455810547, -0.6698893904685974, 1.545159101486206, 1.4689611196517944, 2.320319652557373, 1.0482710599899292, 1.0277551412582397, -0.9129541516304016, -0.026052210479974747, 0.3842862546443939, -2.7883448600769043, -0.6500833630561829, 1.0047966241836548, 0.4042314887046814, 1.5858840942382812, -3.4959328174591064, -0.14634576439857483, 0.5908648371696472, -2.1407461166381836, -0.10026650130748749, 0.7172982096672058, -0.7668173313140869, 0.8089581727981567, 1.1387397050857544, 1.9194031953811646, -1.918046474456787, -0.7608889937400818, 1.2555123567581177, -2.5364222526550293, -2.602149486541748, -3.1463570594787598, 4.166607856750488, 4.628275394439697, -0.5227774977684021, -1.2396489381790161, -1.305276870727539, 3.460099697113037, 0.04223606735467911, 1.6695008277893066, 0.2907274067401886, -1.4920654296875, -3.9623453617095947, 1.2911324501037598, -0.21974222362041473, 2.844465970993042, -0.6382300853729248, 2.3934242725372314, 0.7890841960906982, 0.3046121299266815, -0.5678690075874329, -0.05790190398693085, 2.0398075580596924, 2.276750087738037, -1.7224233150482178, 1.4612398147583008, 3.7486560344696045, 2.775076150894165, -1.143966555595398, 1.0670360326766968, -2.351133108139038, 0.6432157158851624, -0.8258917927742004, 3.892036199569702, 3.34517502784729, 3.562641143798828, -0.5314391255378723, 1.9894009828567505, 2.8140947818756104, 2.9529497623443604], [-0.42100945115089417, -0.7230892777442932, -1.5221718549728394, 1.2471596002578735, -0.5049858093261719, 2.14225435256958, 2.3539276123046875, -3.6012086868286133, -1.1807342767715454, -0.6289067268371582, 1.081354022026062, -0.19974757730960846, 0.022122707217931747, -0.06450015306472778, -2.2122457027435303, -2.9520275592803955, -0.5484216809272766, 4.055574417114258, 0.4010491967201233, -2.999938726425171, -1.903694987297058, -0.9924240112304688, 0.22234192490577698, -0.031953830271959305, -0.9672924876213074, -1.605843424797058, 2.7692818641662598, -3.7814836502075195, 1.9597554206848145, 0.4503886103630066, -1.4100773334503174, -0.1706671416759491, 2.1087660789489746, 3.4559922218322754, -1.5459661483764648, -2.2855112552642822, 2.405996084213257, 1.537340521812439, -2.111427068710327, 0.4689596891403198, -1.530456304550171, 3.5199062824249268, 1.0651732683181763, 0.9648277759552002, 1.9546867609024048, -0.6774251461029053, 0.5837167501449585, 2.2531774044036865, 0.9015767574310303, -0.3610607087612152, 0.18865346908569336, -1.591575264930725, 4.282872200012207, 0.6275367140769958, -0.8910860419273376, 2.370929002761841, -0.588131844997406, 0.44152966141700745, 0.4996597170829773, 1.5300363302230835, -0.7053941488265991, 0.5052759647369385, -2.985772132873535, 1.5195059776306152, 0.3150056302547455, 3.095080614089966, 0.5015162825584412, -2.1888065338134766, -3.0507020950317383, -0.06313934922218323], [4.6926422119140625, 0.8902173638343811, -3.294628620147705, 2.5328848361968994, 0.9076005816459656, 2.8780171871185303, 3.598665237426758, -0.3140418231487274, -2.992398738861084, -2.7374207973480225, 1.1621971130371094, 0.6716445088386536, -1.7780640125274658, 0.25494417548179626, -1.0655559301376343, -6.602592468261719, -0.03114767000079155, 0.6989854574203491, -1.6449159383773804, -3.413667917251587, -0.1357167363166809, -0.5381895303726196, -1.3686009645462036, -0.5645354390144348, -0.7508713006973267, 3.1479179859161377, -1.5363397598266602, -2.4758453369140625, 2.2479867935180664, -0.7304781079292297, -2.085538387298584, -3.8951103687286377, -0.4408271312713623, 1.4994395971298218, 1.2476646900177002, -3.0897631645202637, -1.63540518283844, -1.2943803071975708, -0.5524691343307495, 0.17569585144519806, -0.7375814318656921, 0.35390469431877136, -0.051724836230278015, -0.049864187836647034, 0.644342839717865, -0.2759588956832886, -0.5798473954200745, 1.7944021224975586, -1.436343789100647, 0.692811906337738, -2.3001627922058105, 0.08677268773317337, 1.6209137439727783, 0.0011071707122027874, -0.15459893643856049, 5.065825462341309, 1.0821465253829956, -3.4878740310668945, -1.2539681196212769, 1.9031087160110474, -1.6262261867523193, 2.03346848487854, -3.5088624954223633, -0.8921710252761841, -0.4388540983200073, 0.6006291508674622, 3.602365493774414, -5.944345474243164, -3.455887794494629, 3.868129014968872], [4.872812271118164, 1.1468989849090576, -2.40986967086792, 2.1545443534851074, 2.0729568004608154, 1.542441964149475, 3.1433963775634766, 0.2651847004890442, -3.10426664352417, -1.985115885734558, 1.8679255247116089, -0.3910541832447052, 0.10251834243535995, 1.4369940757751465, 0.7183332443237305, -3.3879923820495605, 0.07972516119480133, -1.3165791034698486, -0.7324024438858032, -1.3825126886367798, 0.4575556814670563, -1.3769112825393677, -2.8319339752197266, -0.130213662981987, 0.7935412526130676, 2.2537992000579834, -2.8605759143829346, -2.954690933227539, 2.356518507003784, -2.8413679599761963, -3.260021924972534, -2.431976079940796, 0.428477942943573, 1.0413343906402588, 0.609269917011261, -0.26062723994255066, 0.16061778366565704, -1.969636082649231, -0.5958649516105652, 0.1966894567012787, -0.06438958644866943, -1.7670377492904663, -3.4068403244018555, -0.9785323739051819, 1.0604345798492432, 1.4442381858825684, 0.4970358908176422, 1.2700799703598022, -1.377206563949585, 0.2330954670906067, 1.2308337688446045, -0.7374469041824341, 0.48131245374679565, -0.811588704586029, 1.8226803541183472, -0.14927595853805542, 0.7716949582099915, -4.483335971832275, 1.8843239545822144, 1.8508083820343018, 0.2040298730134964, 2.382736921310425, -2.653554916381836, -1.6792267560958862, 0.43638530373573303, 2.045478343963623, 0.07066187262535095, -3.8095908164978027, -1.4350682497024536, 0.7237727046012878], [2.8229947090148926, -0.02398856170475483, -0.5375295877456665, -2.7640092372894287, 2.1758878231048584, 1.274290919303894, 1.6202014684677124, 0.8038978576660156, 0.7844829559326172, 1.8784188032150269, -4.400500774383545, 0.27415621280670166, -1.4116758108139038, -2.864328145980835, -2.5955312252044678, -2.8581457138061523, 1.6332780122756958, -1.8786507844924927, 3.194509267807007, -1.507096290588379, 1.5063532590866089, 2.0699474811553955, 1.4778380393981934, 1.0756624937057495, -2.8899004459381104, 3.872521162033081, 0.2933249771595001, 1.925650954246521, -2.6376233100891113, 1.8913480043411255, -0.004584510810673237, -1.8996673822402954, 1.107287049293518, 2.7835755348205566, -1.663731336593628, -0.7149004936218262, -5.007808685302734, 3.847318410873413, 2.643594264984131, 4.874788284301758, -0.3961244225502014, 5.20784330368042, 1.0586498975753784, 0.780769944190979, -0.321213036775589, -1.7962501049041748, -1.893530011177063, 1.90372633934021, -0.3802730441093445, -0.5198134779930115, -5.367343902587891, 4.59428071975708, 1.2735979557037354, 3.7282803058624268, -2.808230400085449, 7.017582893371582, -2.319185495376587, 1.8966878652572632, -1.584578037261963, 1.8411321640014648, -2.5926225185394287, -4.794894218444824, 3.0507874488830566, 1.998066782951355, 3.8466176986694336, -0.6447660326957703, 2.5483715534210205, -2.3302321434020996, -0.8300949335098267, 1.5075148344039917], [-3.571812629699707, -2.3799896240234375, -0.2794775068759918, 0.0590212456882, -0.9416179060935974, -0.15330466628074646, 1.0945744514465332, -5.410787105560303, 0.76583331823349, 0.706757128238678, 1.318236231803894, -1.6433870792388916, 1.61159348487854, 1.3256703615188599, -1.5531617403030396, 2.7239487171173096, 0.44618919491767883, 3.677544116973877, 0.9814587831497192, -0.4929128587245941, -2.6888983249664307, -2.068613052368164, 0.4202396869659424, 0.3854551315307617, 1.0494927167892456, -4.364713668823242, 2.225903034210205, -4.57994270324707, 1.5623695850372314, -0.5070518851280212, -2.274585723876953, 1.8847002983093262, 2.5813651084899902, 3.7372632026672363, -3.067791223526001, 1.1945241689682007, 5.590183258056641, 3.061110258102417, -2.8250911235809326, 0.29716265201568604, -1.7064443826675415, 4.856385707855225, -2.1227612495422363, 0.7062949538230896, 2.3883919715881348, 1.577996015548706, 3.0308070182800293, 1.1119600534439087, 1.6336584091186523, -0.5504302978515625, 5.686380386352539, -2.98982572555542, 3.3664987087249756, -0.15325520932674408, 0.8222835063934326, -5.704262733459473, -1.3411985635757446, -0.36254584789276123, 4.5314764976501465, 1.6822922229766846, 2.5460140705108643, -0.02978464774787426, -1.176488995552063, 2.4323203563690186, 1.1933741569519043, 5.8499836921691895, -3.1465728282928467, 1.4006839990615845, -0.8730363845825195, -4.8384623527526855], [0.7006490230560303, 0.8974071741104126, -3.4617388248443604, -4.042290210723877, 0.9693249464035034, 0.468406081199646, 5.569987773895264, -4.621143341064453, -0.5996541976928711, 2.934617280960083, -5.6742048263549805, 0.022327790036797523, 1.831856369972229, -3.764660120010376, -4.6604180335998535, -3.0270767211914062, 4.259301662445068, 1.8723922967910767, 7.955291748046875, -3.6931581497192383, 2.890794515609741, -0.008346089161932468, 1.5464386940002441, 2.3972697257995605, -5.105184078216553, -0.33909139037132263, 5.087111949920654, -0.31182432174682617, 0.015871327370405197, 0.7549615502357483, -0.9768958687782288, 2.058908224105835, 2.814924955368042, 6.400654315948486, -6.665890216827393, 1.3612287044525146, -2.6309173107147217, 5.092005729675293, -0.5874646902084351, 7.355889797210693, -3.3849523067474365, 10.200899124145508, 3.3189196586608887, 0.984704315662384, 3.6316888332366943, -5.1216559410095215, -0.6234006285667419, 4.409681797027588, 0.986409068107605, -3.33345890045166, -2.8445515632629395, 3.1354169845581055, 4.553867340087891, 1.8611879348754883, -0.777199387550354, 4.988250732421875, -6.597757816314697, 2.6859047412872314, 3.5997846126556396, 4.051759243011475, -1.51032292842865, -7.532931804656982, 1.771938443183899, 5.14313268661499, 5.53712797164917, 4.25910758972168, -2.741774082183838, -3.1104297637939453, -3.713557243347168, -3.325077533721924], [2.4353504180908203, -4.537137985229492, 2.2884058952331543, 0.9244111776351929, 2.032191038131714, 2.1579997539520264, -1.6467349529266357, 0.3862871825695038, 2.0381038188934326, 0.6879878640174866, 1.2687201499938965, -3.0827383995056152, -1.9531526565551758, 1.8681590557098389, -0.20124252140522003, 3.3975658416748047, -3.343609571456909, -0.7862846851348877, -1.7847135066986084, 0.2547109127044678, -3.7083137035369873, 0.8712419867515564, 0.3515908420085907, 0.06052394583821297, 2.9884655475616455, 1.6754149198532104, -3.2268171310424805, -2.5481112003326416, -1.3393393754959106, -0.29888370633125305, -3.046736478805542, -3.864161968231201, 4.055723667144775, 4.010658264160156, 0.4684987962245941, -1.22952401638031, 1.5052099227905273, 5.3986592292785645, 1.1732293367385864, 0.7298941612243652, 1.2218974828720093, 1.330151081085205, -6.705197334289551, 1.423500418663025, -1.4284577369689941, 6.024469375610352, 1.1196305751800537, 0.7084180116653442, 0.787394106388092, 2.001962661743164, 1.8458093404769897, -0.1796950250864029, 1.9563580751419067, 4.307077884674072, -2.871720790863037, -1.450821042060852, 3.6671741008758545, 1.0750997066497803, -0.8805124163627625, 0.8973860144615173, -0.28975483775138855, 1.1037477254867554, 0.9169601798057556, 2.251457929611206, 3.106326103210449, 3.402085781097412, 1.8507858514785767, 2.872968912124634, 3.379204511642456, 1.0041762590408325], [2.0819385051727295, 2.734915256500244, 3.4501826763153076, -2.4989843368530273, 4.188361644744873, -0.4008989632129669, -1.8891528844833374, 5.45917272567749, -0.18261046707630157, 2.704087734222412, -1.7609249353408813, 0.39529529213905334, 1.6292779445648193, -1.7390743494033813, 2.0668528079986572, 0.1545068770647049, -0.5309380292892456, -5.685937881469727, 3.193633556365967, 2.8906679153442383, 2.475780725479126, 1.7796334028244019, -1.440158724784851, 0.8081388473510742, -1.0099098682403564, 2.492497682571411, -0.4315055310726166, 2.906679391860962, -3.4429426193237305, -0.05251318961381912, 1.0352809429168701, 1.8879073858261108, 1.6724854707717896, -2.3372020721435547, 0.10222911834716797, 1.73134183883667, -2.131131887435913, -1.4373681545257568, 3.6799795627593994, 2.2633135318756104, 2.7692062854766846, -4.124958038330078, -1.042129397392273, -1.333662748336792, -0.9734075665473938, -0.6454979777336121, -2.383876085281372, -0.2580847442150116, -0.3737008273601532, -1.0336906909942627, -3.1086673736572266, 3.2346339225769043, -1.5232131481170654, 2.2854552268981934, -0.567340075969696, 2.9046802520751953, -2.030327796936035, 2.527770757675171, -0.5368024110794067, -1.3725758790969849, -1.4346977472305298, -2.233114719390869, 3.5374252796173096, -3.29473614692688, 2.0883262157440186, -3.802206516265869, -1.5438686609268188, 1.4551135301589966, 2.4463398456573486, -1.2682815790176392], [-2.0966830253601074, 0.5458983778953552, 3.847733736038208, -0.4310089647769928, 1.6104176044464111, -0.10418432950973511, -2.7090933322906494, 1.6340456008911133, 0.33663979172706604, 1.7532984018325806, 1.5032861232757568, -0.3878132104873657, 2.0039713382720947, 0.13095179200172424, 1.6229161024093628, 2.121593713760376, -2.2259984016418457, -0.8646008372306824, 0.9574229121208191, 2.3629400730133057, -1.1005315780639648, 0.15212056040763855, -0.8804613351821899, 0.04837428778409958, 0.6941708922386169, -2.0291357040405273, 1.6763051748275757, -0.7525206208229065, -1.3626450300216675, 0.1716042160987854, 0.5389476418495178, 3.090421438217163, 2.661926746368408, -1.2512015104293823, -0.1113932803273201, 0.6072102189064026, 3.4739394187927246, -0.8464212417602539, 0.6149441599845886, -0.7295562028884888, 1.90621817111969, -3.256139039993286, -1.1177711486816406, -0.47411292791366577, 0.022621754556894302, 0.8874713778495789, -0.202531099319458, -0.4478902518749237, 1.1474131345748901, -0.49315202236175537, 1.1611686944961548, -0.9242130517959595, 0.8070070743560791, 1.1663870811462402, -0.38253557682037354, -1.1790019273757935, -0.596638560295105, 2.814809799194336, 0.4114789664745331, -1.8033474683761597, 0.18405435979366302, 0.6390824913978577, 0.5734577775001526, -2.2886857986450195, 0.033737462013959885, -0.8832782506942749, -2.399080514907837, 3.2777256965637207, 1.7728315591812134, -2.7471556663513184], [6.817225456237793, 3.639779567718506, 0.286807656288147, 1.2115799188613892, 5.969162464141846, 2.531571865081787, 2.54866361618042, 3.637449026107788, -4.613927364349365, -0.022213349118828773, 1.9437127113342285, -0.36025241017341614, 2.2906343936920166, 0.31864890456199646, 2.1834774017333984, -4.663492202758789, -1.587384581565857, -4.2736101150512695, 2.3174405097961426, -0.1741778701543808, 1.5198142528533936, -0.6650588512420654, -4.855291366577148, 0.521836519241333, -0.1929614245891571, 3.070250988006592, -1.6329941749572754, -3.089357376098633, 0.8617417812347412, -3.3979296684265137, -3.4124343395233154, -0.2429896742105484, 3.8208532333374023, 0.5496978163719177, 0.028242550790309906, 0.10128499567508698, 0.7156912684440613, -3.3748602867126465, 1.23888099193573, 1.8939406871795654, 2.0264484882354736, -5.445431709289551, -4.4191670417785645, -1.929486632347107, 1.5186787843704224, 0.8965678811073303, -1.3003625869750977, 2.347078561782837, -1.0432178974151611, -1.0420907735824585, -0.7228225469589233, 0.46715715527534485, 1.6983261108398438, 1.283635139465332, 1.2639368772506714, 3.106992721557617, -1.009045958518982, -1.801444411277771, 2.0431034564971924, 1.0779086351394653, -1.4151865243911743, 1.6692842245101929, -1.8785539865493774, -4.566878795623779, 2.2080490589141846, 0.5236726999282837, -2.032020330429077, -3.186765432357788, -0.5720348954200745, -0.856560230255127], [0.6270692944526672, 0.05535217747092247, 0.38520410656929016, 1.059657335281372, 0.8691512942314148, -0.007393606472760439, 1.0223736763000488, -1.8219947814941406, -2.0582613945007324, 1.1256515979766846, 2.54996919631958, -2.6659910678863525, 4.371975421905518, 1.7806179523468018, 1.8012330532073975, 4.214313983917236, -2.5957865715026855, 1.729264497756958, 3.2140073776245117, -0.11053735762834549, -0.004640055820345879, -2.058673620223999, -2.456594705581665, 1.1511279344558716, 1.4097086191177368, -4.566469669342041, 2.272463321685791, -3.482895612716675, 3.077181577682495, -4.122655391693115, -2.545088291168213, 3.7992429733276367, 5.3738203048706055, 3.217961072921753, -3.200476884841919, 2.2605113983154297, 5.5242533683776855, 0.2925165295600891, -2.869133472442627, 0.26270681619644165, -0.14700667560100555, -3.1047277450561523, -3.478555202484131, -0.11916987597942352, 2.878916025161743, 1.4620062112808228, 1.3605659008026123, 2.06931471824646, 2.2011806964874268, -2.0085272789001465, 5.431087970733643, -3.809236526489258, 3.037174940109253, -1.0308291912078857, 2.0672852993011475, -5.7259521484375, 0.0845981314778328, 2.7392804622650146, 4.813706874847412, 0.28569918870925903, 0.9391106963157654, 1.297088861465454, -2.229308843612671, 1.8216077089309692, 2.002033233642578, 5.758605480194092, -7.81421422958374, 4.266720771789551, 1.9874305725097656, -4.649887561798096], [3.2508645057678223, 6.231874465942383, -0.49053266644477844, 3.2016985416412354, 0.6174308657646179, 1.932828664779663, 0.6366086006164551, 4.7122697830200195, -5.656479358673096, -2.4865307807922363, 2.9436089992523193, 2.9624433517456055, 1.4309587478637695, -1.0408215522766113, 2.8687973022460938, -7.573617458343506, -3.0510623455047607, 0.16480325162410736, -0.5285103917121887, -1.6240789890289307, 2.673964738845825, 0.04245078191161156, -3.3388285636901855, -0.9779373407363892, -2.2798826694488525, 0.656958818435669, 2.533336639404297, 1.05136239528656, 2.59770131111145, -1.0445512533187866, 2.5245964527130127, 1.814629077911377, -0.8022358417510986, -4.882413864135742, 3.1116130352020264, -3.514991283416748, -1.0227046012878418, -8.972137451171875, -0.4307672679424286, -3.0954625606536865, 1.6930012702941895, -10.692156791687012, 5.72416353225708, -1.5606803894042969, 0.4414481222629547, -4.632590293884277, -3.943141460418701, 0.5240981578826904, -0.7697944641113281, -1.1320174932479858, -5.687030792236328, -1.0180442333221436, -0.058850694447755814, -2.624952554702759, 0.8527028560638428, 8.518845558166504, 0.8358414173126221, 1.5273287296295166, -4.237279415130615, -2.9615278244018555, -3.8117501735687256, 4.053716659545898, -5.106138706207275, -5.67767333984375, -4.534790515899658, -5.997284412384033, 0.32720884680747986, -3.3540468215942383, -2.5998148918151855, 4.539839744567871], [-4.079145908355713, 1.1277776956558228, -1.1665302515029907, -0.5279518365859985, -4.655492305755615, -5.161830425262451, -1.308955192565918, -1.0254043340682983, 0.4639979302883148, -0.15719778835773468, -0.11644292622804642, 0.008658942766487598, 3.020714521408081, 1.1120039224624634, 2.757622718811035, 7.143355846405029, 1.604231834411621, 1.128519058227539, 0.6373438835144043, 2.276592254638672, 3.297116279602051, -1.509921669960022, 0.16275210678577423, 0.3350769579410553, 1.1171598434448242, -5.128154277801514, 1.0252183675765991, 3.939992666244507, 2.3133792877197266, -2.535269260406494, 2.7462873458862305, 4.90040397644043, -3.5912439823150635, -3.6998653411865234, -0.8283989429473877, 4.784996032714844, 0.8196389675140381, -3.3278276920318604, -2.7018399238586426, -2.89250111579895, -1.1174064874649048, -4.158719539642334, 2.53155517578125, -0.8259740471839905, 0.4652927815914154, -1.9161710739135742, 1.2383570671081543, -2.6300017833709717, 0.6827235221862793, -1.518574833869934, 4.740384578704834, -2.713291645050049, -4.634171009063721, -6.3484015464782715, 4.578551769256592, -9.640929222106934, 0.17829333245754242, 0.5256391763687134, 3.1067686080932617, -2.2244489192962646, 3.089235782623291, 0.1903582513332367, 0.4886024296283722, 1.7198395729064941, -2.8331639766693115, 0.0041418843902647495, -6.719936847686768, 5.250009536743164, 2.675050973892212, -3.0630900859832764], [-4.753753662109375, 5.879283428192139, -0.05329527705907822, -1.0196986198425293, 0.9208686351776123, -3.6911768913269043, 1.207982063293457, -0.7676920890808105, -2.8855791091918945, 0.00018318994261790067, 1.9090110063552856, 2.5243895053863525, 5.474122047424316, 0.4200839698314667, 2.597795248031616, -1.4077513217926025, 4.467742443084717, -2.058046340942383, 2.1624274253845215, 3.7070436477661133, 2.546999931335449, -3.5295233726501465, -3.686863899230957, -0.1779809594154358, -0.4042375981807709, -4.029867172241211, 1.8656902313232422, -1.8440617322921753, 1.421789526939392, -2.053513288497925, 0.30351248383522034, 7.158778667449951, -2.9429423809051514, -5.994128704071045, -1.0469573736190796, 5.262474060058594, 5.098424434661865, -8.548704147338867, -1.6601414680480957, -1.8034617900848389, 0.11570727080106735, -3.627756357192993, 0.9069671630859375, -3.8864822387695312, 3.11968731880188, -3.040778398513794, 1.7820888757705688, -1.8900439739227295, -1.204276442527771, -2.4659245014190674, 5.581921100616455, -2.3068935871124268, -2.515444755554199, -5.706353187561035, 6.864798069000244, -7.420536041259766, -6.311153888702393, -6.054775238037109, 7.655364036560059, -0.6084586977958679, 4.831727981567383, 0.994462251663208, -1.0246044397354126, -7.55314826965332, -3.250981330871582, -1.333740472793579, -6.558315753936768, -2.4686927795410156, -3.1216487884521484, -8.583026885986328], [-5.201303958892822, 3.2709662914276123, 3.6393566131591797, -3.9142186641693115, 1.550864815711975, -3.6922554969787598, -2.5369200706481934, 1.797239065170288, 1.0849223136901855, 3.271186113357544, -1.5173413753509521, 1.4138232469558716, 3.927914619445801, -1.4753488302230835, 1.857949137687683, 2.7932324409484863, 2.6444904804229736, -3.6602096557617188, 3.7169198989868164, 5.008726596832275, 2.195838451385498, -0.17428536713123322, -0.6796069741249084, 0.625775158405304, -0.9165422320365906, -2.550884962081909, 2.3015592098236084, 2.2995948791503906, -3.0498011112213135, 0.6780237555503845, 2.4338066577911377, 6.620089530944824, -0.7769700884819031, -4.696028709411621, -1.4831210374832153, 4.848742961883545, 1.9071238040924072, -3.1138687133789062, 1.5738874673843384, 0.6135994791984558, 1.4648749828338623, -1.7858314514160156, 1.113263487815857, -2.1608221530914307, 0.5353432893753052, -2.2813310623168945, -0.014189512468874454, -2.128174066543579, 0.141334667801857, -2.037435293197632, 1.678946614265442, 1.0871050357818604, -2.611421585083008, -1.0745383501052856, 2.3962271213531494, -4.041205406188965, -5.447674751281738, 0.665397584438324, 3.686359405517578, -1.8910062313079834, 2.475318670272827, -2.8527445793151855, 3.9455487728118896, -4.291439056396484, -0.20588812232017517, -3.2455251216888428, -4.9805779457092285, 2.532752513885498, 1.026943325996399, -7.007665157318115], [7.13251256942749, 2.932767629623413, 2.5626468658447266, -0.8951351642608643, 6.800920486450195, 1.286116361618042, -0.02150043472647667, 6.539926052093506, -2.3578908443450928, 1.911988615989685, -0.27000734210014343, -0.9328282475471497, 1.8943910598754883, -0.42205920815467834, 3.065277338027954, -1.031095027923584, -1.8524821996688843, -7.405736923217773, 3.2303900718688965, 2.0442397594451904, 2.6003761291503906, 1.1817545890808105, -3.8615312576293945, 1.0640703439712524, 0.13002996146678925, 4.5813822746276855, -3.054205894470215, 0.56011962890625, -2.0610382556915283, -2.803044080734253, -2.0958967208862305, -0.27197322249412537, 3.9024593830108643, -0.3918306529521942, 0.28441566228866577, 1.591906189918518, -1.8911497592926025, -1.6463546752929688, 3.749553918838501, 3.094660758972168, 3.3517887592315674, -6.302932262420654, -5.408478260040283, -1.786236047744751, -0.4170474410057068, 1.7059820890426636, -2.23715877532959, 1.2221850156784058, -0.946000337600708, -0.84037846326828, -2.1196491718292236, 2.844414710998535, -0.572708010673523, 2.960341453552246, 0.06298301368951797, 2.7971251010894775, -0.6043828129768372, 0.6860790848731995, 0.7095785737037659, -0.00633935397490859, -1.8698269128799438, -0.5295094847679138, 2.0401248931884766, -3.811279058456421, 3.82297682762146, -1.4416602849960327, -2.167369842529297, 0.17657046020030975, 2.955697536468506, -0.664201557636261], [3.651754856109619, -1.0209962129592896, -0.38554686307907104, -0.35955318808555603, 2.390349864959717, 2.0512051582336426, 1.4639270305633545, 0.4060291051864624, -0.065076544880867, 0.49840065836906433, -1.1955840587615967, -0.8433446288108826, -1.336783766746521, -0.5383575558662415, -1.3837295770645142, -1.9655588865280151, -0.23761601746082306, -1.285188913345337, 0.8693636059761047, -1.3930857181549072, -0.4060252606868744, 0.8731487393379211, 0.012301287613809109, 0.49285605549812317, -0.4917907416820526, 3.139662981033325, -1.4174104928970337, -1.0107980966567993, -0.9416993856430054, 0.16050264239311218, -1.9830060005187988, -2.9329590797424316, 2.09808349609375, 3.17299747467041, -0.5719093084335327, -1.154091238975525, -1.9024136066436768, 3.0528078079223633, 1.425761103630066, 2.808030128479004, 0.07412475347518921, 2.7876498699188232, -2.322686195373535, 0.6039462089538574, -0.14101894199848175, 1.216610074043274, -0.5211848020553589, 1.7764889001846313, -0.32918980717658997, 0.4015803933143616, -2.012761116027832, 2.0084948539733887, 1.801148772239685, 3.1069247722625732, -1.9229782819747925, 3.6430928707122803, 0.07642792165279388, -0.009176170453429222, -0.6234915852546692, 1.8717176914215088, -1.5161192417144775, -1.330863356590271, 0.7473710179328918, 1.164446234703064, 3.0149893760681152, 1.4037320613861084, 2.112785816192627, -1.8151911497116089, -0.19989095628261566, 1.4059078693389893], [-4.707292079925537, 2.2582790851593018, 1.925842046737671, -1.386286735534668, 2.009626626968384, -0.8969823718070984, -0.44562599062919617, -0.5336684584617615, 0.13605384528636932, 0.4333805441856384, 0.9328050017356873, 2.32365345954895, 1.103026270866394, -0.3836384117603302, -0.2834652364253998, -3.4407436847686768, 3.317685604095459, -1.9703060388565063, -0.38678404688835144, 2.910564422607422, -1.3777323961257935, -1.2719157934188843, -1.019457459449768, -0.8424294590950012, -0.3933883011341095, -0.5601522922515869, 0.535409152507782, -2.6598546504974365, -2.035037040710449, 2.0616884231567383, -0.011606661602854729, 2.4389894008636475, -1.9996984004974365, -3.744548797607422, 0.47087526321411133, 1.3373762369155884, 3.5457804203033447, -3.4281883239746094, 0.9213632345199585, -0.6506624817848206, 0.7801486253738403, 1.8698145151138306, -0.10328558832406998, -2.1016950607299805, 0.8218463659286499, -0.6049816608428955, 1.3504657745361328, -1.6647621393203735, -1.3155102729797363, 0.007160115987062454, 1.9554284811019897, 0.20437592267990112, -0.5137537121772766, -0.3129182755947113, 1.7846696376800537, -1.5940475463867188, -4.876772403717041, -5.2207841873168945, 3.10585355758667, 0.1353912353515625, 3.016573190689087, 0.1038709357380867, 0.674281656742096, -6.7195868492126465, -1.8493846654891968, -2.36507248878479, 0.8213037252426147, -3.87221097946167, -3.4885387420654297, -5.015158653259277], [-3.4506218433380127, 1.1553846597671509, 1.355452537536621, -0.6773651242256165, -0.4072422683238983, -0.6140245199203491, -1.3918840885162354, 0.31101125478744507, 0.6287603378295898, 0.11818738281726837, 0.0917629674077034, 1.7889277935028076, -0.07273761928081512, -0.8402012586593628, -0.19733481109142303, -1.6308680772781372, 0.9957365393638611, 0.008676769211888313, -0.7784419059753418, 1.2616820335388184, -0.5599260926246643, 0.1438673585653305, 0.5444089770317078, -0.6870278716087341, -0.7750966548919678, -0.6605836153030396, 1.3710768222808838, 0.4613879323005676, -1.3557802438735962, 2.1931142807006836, 1.8962019681930542, 1.5267424583435059, -1.8876510858535767, -2.9527289867401123, 0.9622307419776917, -0.381890207529068, 0.7437337636947632, -2.014716625213623, 0.5572026371955872, -1.2171745300292969, 0.4646565020084381, 0.15650875866413116, 2.6004974842071533, -0.5740178227424622, -0.36693939566612244, -1.5186123847961426, -0.24479761719703674, -1.3753868341445923, -0.33798304200172424, 0.13150309026241302, -0.9293127059936523, 0.29677456617355347, -0.7278062105178833, -0.3365335762500763, -0.06140914186835289, 1.0963963270187378, -1.7198407649993896, -0.6083654761314392, -0.8124545812606812, -1.1575250625610352, 0.5212768912315369, -0.021799733862280846, 0.3997715711593628, -2.8932199478149414, -2.1403846740722656, -3.1096835136413574, 1.4362225532531738, -1.073307991027832, -1.5353810787200928, -0.7644678950309753], [-3.4900755882263184, 2.0064685344696045, -0.21665751934051514, -2.0876333713531494, 0.9991108775138855, 0.5088343620300293, 1.5228807926177979, -1.7538187503814697, 0.1433160901069641, 0.4502331018447876, -1.7792342901229858, 3.0459041595458984, -0.5793343782424927, -2.5734102725982666, -3.087144374847412, -6.792729377746582, 4.057948112487793, 0.1441774219274521, 0.9362201690673828, -0.32678478956222534, -0.5731366872787476, -0.059948910027742386, 0.9648854732513428, -0.44546011090278625, -3.2138266563415527, 0.8964247703552246, 2.56471586227417, -1.3034026622772217, -2.0839004516601562, 3.9311728477478027, 0.94182288646698, 0.88334721326828, -2.1089959144592285, -1.2533799409866333, -0.5906686186790466, -1.0433284044265747, -0.1964259296655655, -0.9253846406936646, 1.045400857925415, 1.6038126945495605, -0.785092294216156, 6.197282791137695, 3.9141969680786133, -0.7060385346412659, 0.996253490447998, -3.5371456146240234, -0.2403087615966797, 0.1329049915075302, -1.1403658390045166, -0.37730562686920166, -2.8787615299224854, 2.2779600620269775, 1.352713704109192, 1.0587407350540161, -0.5862326622009277, 5.17414665222168, -5.5341410636901855, -2.9319846630096436, 0.3877694010734558, 1.2570194005966187, 0.30381691455841064, -2.508523464202881, 0.6504780650138855, -3.5964341163635254, -0.7773033976554871, -2.6333115100860596, 3.583707809448242, -6.254737854003906, -5.640980243682861, -1.7964768409729004], [0.06846696138381958, 3.1356513500213623, 2.493769645690918, -0.44975462555885315, 2.010450839996338, -1.4804258346557617, -2.164320707321167, 4.415611267089844, -0.9145213961601257, 0.46448376774787903, 0.9281352162361145, 1.0524600744247437, 1.78410005569458, 0.09135930240154266, 3.3026814460754395, 0.41009044647216797, -0.2286209613084793, -4.28051233291626, 0.10012607276439667, 3.456878900527954, 1.779077410697937, 0.14122441411018372, -2.221346855163574, -0.2670655846595764, 0.5853832960128784, 0.6065903902053833, -1.2497175931930542, 1.8385698795318604, -1.3268342018127441, -0.9410976767539978, 1.2443588972091675, 2.3020431995391846, -1.1026445627212524, -5.008496284484863, 1.8221486806869507, 1.963106632232666, 0.21612600982189178, -4.91701602935791, 1.7448395490646362, -1.388814926147461, 2.4447896480560303, -6.645247459411621, -0.7194703817367554, -2.085205078125, -0.8347607254981995, -0.16303065419197083, -0.9552757143974304, -1.9501729011535645, -0.9309730529785156, -0.3663671910762787, 0.08316535502672195, 0.43370521068573, -3.2194838523864746, -1.1784887313842773, 1.8805814981460571, -1.4165782928466797, -0.7573026418685913, -0.8883134722709656, 0.38628053665161133, -2.205641984939575, 0.6794898509979248, 1.132631540298462, 1.1180542707443237, -5.156028747558594, -1.5020078420639038, -4.063224792480469, -1.8178963661193848, 0.9293560981750488, 1.6005196571350098, -1.4894311428070068], [1.6988112926483154, 3.2498490810394287, -0.4826596975326538, -3.1542725563049316, 2.1459107398986816, -3.094733715057373, 1.2993600368499756, 1.9215809106826782, -0.9901366233825684, 2.2280497550964355, -2.908458948135376, -0.09615541249513626, 3.6311399936676025, -0.9786103367805481, 1.9452824592590332, 2.5128934383392334, 3.1305489540100098, -4.766887664794922, 5.18011474609375, 2.373680591583252, 5.147310256958008, -0.38197243213653564, -1.8507217168807983, 1.6803042888641357, -0.9824118614196777, 0.32014960050582886, -0.5990956425666809, 3.005154848098755, -0.6029948592185974, -2.874201536178589, -0.07445013523101807, 3.511698007583618, -0.214419424533844, -1.3142050504684448, -2.572667360305786, 5.639863014221191, -2.111549139022827, -1.8046681880950928, 1.0969494581222534, 3.0887651443481445, 0.2018565982580185, -1.9201141595840454, -1.213272213935852, -1.9230958223342896, 1.1673227548599243, -1.8934005498886108, -0.374697208404541, -0.04772334173321724, -0.6164140701293945, -2.526219606399536, 1.2072596549987793, 1.995495319366455, -2.9439539909362793, -1.796277642250061, 3.4767396450042725, -3.7746920585632324, -3.7869839668273926, -0.5328854918479919, 4.794877529144287, 0.4631468951702118, 1.3450746536254883, -3.5218074321746826, 3.3250441551208496, -0.6544507145881653, 2.580143451690674, 0.0414559543132782, -6.215859889984131, 1.1940765380859375, 1.819344401359558, -4.598888874053955], [-3.317667007446289, -3.8230340480804443, 0.10182013362646103, -0.5372098684310913, -2.5333211421966553, 1.7757822275161743, 0.5287618637084961, -5.732525825500488, 1.7337791919708252, 2.1228349208831787, -0.952043354511261, -2.0355911254882812, 0.4996747076511383, -1.1052711009979248, -3.7383649349212646, 2.5835368633270264, -2.393771171569824, 7.398571014404297, 3.046369791030884, -3.6069958209991455, -2.8087713718414307, 0.26175153255462646, 3.282043695449829, 1.0759598016738892, -1.5477311611175537, -4.929250717163086, 6.657903671264648, -2.2824368476867676, 0.9902569651603699, 1.8352484703063965, 0.04179558902978897, 1.8822702169418335, 5.479247093200684, 7.1745758056640625, -4.510473251342773, -2.25917387008667, 3.0157129764556885, 7.331047534942627, -2.774622678756714, 2.0710105895996094, -2.383324146270752, 6.973731517791748, 2.7117760181427, 3.309231758117676, 1.9998103380203247, -1.0197831392288208, 0.4255695343017578, 3.304568290710449, 3.818135976791382, -1.225990653038025, -0.06872156262397766, -1.7151904106140137, 6.771501541137695, 2.768057346343994, -3.748112678527832, 2.19278621673584, -0.22022058069705963, 8.015600204467773, -0.5165818929672241, 0.9425519704818726, -1.9137496948242188, -2.6062371730804443, -1.0095492601394653, 7.605818748474121, 2.752497911453247, 5.507612228393555, -1.7489603757858276, 3.7953271865844727, -0.3721067011356354, -0.9193235039710999], [-5.419914245605469, 3.5049049854278564, 1.221899151802063, 0.7357138991355896, -0.25709834694862366, -0.8414510488510132, -1.4428582191467285, 0.8676624298095703, -0.6774842143058777, -2.2217793464660645, 2.5684030055999756, 4.343237400054932, -0.6049750447273254, -0.12228629738092422, 0.5529762506484985, -6.291009902954102, 2.7778656482696533, -0.9300450682640076, -4.285799980163574, 2.5420827865600586, -1.4651895761489868, -1.1331892013549805, -0.9285210371017456, -2.333240270614624, -0.366060346364975, -0.02210150845348835, 0.022131552919745445, -1.2347687482833862, -1.1056116819381714, 3.075969696044922, 2.2438876628875732, 1.3512659072875977, -5.896081447601318, -7.765329360961914, 3.9674534797668457, -1.0574144124984741, 2.3876359462738037, -7.589322090148926, 0.5537626147270203, -4.372572898864746, 1.1077109575271606, -1.7612093687057495, 3.4760072231292725, -2.438886880874634, -0.527587890625, -1.9364674091339111, 0.42373496294021606, -3.446603298187256, -2.3670566082000732, 1.1754612922668457, -0.14925800263881683, -0.6000150442123413, -2.650057077407837, -2.610954761505127, 2.043531894683838, 0.6862355470657349, -2.7675423622131348, -6.796070098876953, -0.5332468748092651, -1.7508221864700317, 2.413785457611084, 3.124161720275879, -1.5564731359481812, -9.208553314208984, -6.372605800628662, -6.509954452514648, 4.246658802032471, -5.81547737121582, -4.915895462036133, -0.9939311742782593], [-1.3959952592849731, 1.9619081020355225, -3.101041078567505, -1.1453975439071655, -1.6101353168487549, -3.126507043838501, 1.740963101387024, -0.9078282117843628, -0.07293928414583206, -1.4577980041503906, -1.648422122001648, 2.032111644744873, -0.17306506633758545, -0.053616248071193695, 0.00039312741137109697, -1.205267071723938, 5.5553507804870605, -1.8578871488571167, -0.8131800889968872, 1.1557374000549316, 2.719888687133789, -1.2138564586639404, -0.0549778938293457, -0.3376823961734772, -0.4229925274848938, 0.9304010272026062, -2.3116672039031982, 2.0291266441345215, 0.4140564203262329, -0.30815500020980835, 0.7369881868362427, 0.04443885385990143, -6.242574691772461, -3.652935743331909, 0.5683745741844177, 3.0159175395965576, -2.4694790840148926, -3.3626186847686768, -0.18121516704559326, -0.5103226900100708, -1.4101516008377075, 1.6613491773605347, 1.5435441732406616, -1.6815307140350342, 0.10817494988441467, -2.0103249549865723, 1.1721322536468506, -2.1828725337982178, -2.3534350395202637, -0.07291853427886963, 1.4696084260940552, 0.9660574793815613, -4.584201335906982, -4.124544143676758, 3.8328351974487305, -3.9066972732543945, -2.4463679790496826, -6.648819923400879, 2.7801969051361084, 0.8608655333518982, 3.0286824703216553, -0.8996651768684387, 1.096553921699524, -1.9304440021514893, -2.125727415084839, -1.684826135635376, 0.2206653356552124, -3.7202560901641846, -2.1606667041778564, -1.3745607137680054], [2.1593761444091797, 5.394941806793213, -3.4273152351379395, -0.875480592250824, 0.9503673911094666, 1.1262346506118774, 4.822548866271973, -0.34072068333625793, -4.295554161071777, -0.12916593253612518, -2.2057676315307617, 2.7712316513061523, 1.9695910215377808, -3.3890061378479004, -1.5471605062484741, -8.376275062561035, 2.464266061782837, 0.9098589420318604, 4.679770469665527, -3.3997552394866943, 4.114737510681152, -0.32219064235687256, -1.231552243232727, 0.7622933983802795, -5.282230377197266, 0.7000243663787842, 4.631267070770264, 0.518413245677948, 1.6834434270858765, 0.11186207830905914, 1.0811110734939575, 2.4337704181671143, -0.2681983709335327, 0.11822030693292618, -2.185314893722534, -1.0107382535934448, -2.92691707611084, -3.5066418647766113, -0.6058164834976196, 2.857799768447876, -1.5051151514053345, 0.7241170406341553, 6.552026748657227, -0.8736448287963867, 2.8629226684570312, -7.156517028808594, -2.7475955486297607, 2.863116502761841, -0.6051177978515625, -2.9316070079803467, -5.612459659576416, 1.8284326791763306, 2.202272891998291, -1.264445185661316, 0.8588963150978088, 8.7655668258667, -4.907806396484375, 0.6542276740074158, 0.26294660568237305, 1.173061490058899, -2.8165903091430664, -2.5816566944122314, -2.0867867469787598, -1.3626011610031128, 0.08748853951692581, -1.73732328414917, -0.94892418384552, -6.135252952575684, -5.455376148223877, 0.40450742840766907], [-1.1503628492355347, 1.2036687135696411, -2.2501373291015625, -1.5305465459823608, -1.9129434823989868, 0.8614582419395447, 2.200697183609009, -2.06671404838562, -0.29647985100746155, 0.4941502511501312, -3.409639835357666, 1.9445375204086304, -0.663627028465271, -3.4709014892578125, -3.4075348377227783, -4.618075370788574, 1.6743981838226318, 3.4797561168670654, 2.730433940887451, -3.499117136001587, 1.5526098012924194, 1.078268051147461, 2.4818389415740967, 0.48190656304359436, -4.544844627380371, -0.20273645222187042, 5.013530254364014, 1.9685345888137817, 0.002348425565287471, 2.9441938400268555, 2.58046817779541, 0.8963539600372314, -0.6825722455978394, 1.5796154737472534, -1.9599820375442505, -2.3753130435943604, -3.444326877593994, 1.3564741611480713, -0.4445045590400696, 2.313356399536133, -2.130267381668091, 5.026228904724121, 7.569241523742676, 1.2635468244552612, 1.0258188247680664, -5.79407262802124, -2.145521640777588, 1.9010814428329468, 0.574754536151886, -1.4553442001342773, -5.962812900543213, 2.08286190032959, 2.3930304050445557, 0.532813310623169, -2.151705741882324, 8.323604583740234, -2.7978501319885254, 3.563188314437866, -2.646484136581421, 0.6072996258735657, -2.9608819484710693, -3.71738338470459, -0.24001865088939667, 2.6912546157836914, 0.141781747341156, -1.4458644390106201, 1.925444483757019, -2.906548261642456, -3.7819595336914062, 2.1886024475097656], [0.9221473336219788, 0.4380762279033661, 0.37456291913986206, 2.0629324913024902, -0.551288902759552, 0.795833170413971, -0.6299643516540527, 0.9160569906234741, -1.461390733718872, -0.8645704388618469, 2.275404214859009, -0.4145943522453308, 0.6904391646385193, 0.8683138489723206, 1.5154823064804077, 0.36888638138771057, -2.9446074962615967, 1.396300196647644, -0.8348252773284912, -0.5975049138069153, -0.2849481403827667, -0.25109949707984924, -1.042204737663269, -0.29568570852279663, 0.6977903246879578, -1.3060548305511475, 0.8059062361717224, -0.49351826310157776, 1.7393276691436768, -1.2568063735961914, 0.2423955500125885, 0.5969539880752563, 1.2811260223388672, -0.3509618043899536, 0.8822771310806274, -1.270132064819336, 1.455098032951355, -1.7221612930297852, -1.1407808065414429, -1.9624849557876587, 0.6200276017189026, -4.57774543762207, 0.3398832678794861, 0.14584213495254517, 0.10313016921281815, 0.2611563801765442, -0.6041886210441589, 0.2981758415699005, 0.7334628105163574, -0.08426795154809952, 0.12417536228895187, -2.031838893890381, 0.7002867460250854, -0.8408532738685608, 0.03564508259296417, 0.3451317846775055, 2.3049330711364746, 2.0095605850219727, -1.3070707321166992, -1.357500433921814, -1.046231746673584, 2.5870590209960938, -2.4251151084899902, -0.15571348369121552, -1.3133800029754639, -0.03611600771546364, -1.1469964981079102, 1.624672293663025, 0.8661839962005615, 1.468302845954895], [-2.6787514686584473, -2.284553050994873, 1.1216198205947876, 1.6832345724105835, -3.9111084938049316, -1.3960663080215454, -3.896989583969116, 0.8404068350791931, 2.2818336486816406, -1.5258902311325073, 1.6178226470947266, -0.15901784598827362, -2.011320114135742, 1.6367243528366089, 1.4836887121200562, 4.381551742553711, -2.063045024871826, 1.5871620178222656, -4.752923011779785, 1.289310336112976, -1.5076626539230347, 0.6087722182273865, 1.714491844177246, -1.2322908639907837, 2.484429359436035, -1.3633956909179688, -1.5176753997802734, 2.5499584674835205, 0.12321913242340088, 0.7613278031349182, 2.45194935798645, -1.0266765356063843, -2.934985876083374, -2.963188648223877, 3.1725664138793945, -0.9504255056381226, 0.05665042996406555, -0.6210771799087524, -0.5654962062835693, -4.504814624786377, 0.5467371940612793, -3.3424930572509766, 1.0532853603363037, 0.8060113191604614, -2.6405649185180664, 1.7605007886886597, 0.422248512506485, -2.9812421798706055, 0.3236926794052124, 1.9484734535217285, 1.0704785585403442, -1.8336219787597656, -3.100830078125, -1.7640589475631714, -0.45881491899490356, -3.389448404312134, 4.667325973510742, 0.9535740613937378, -3.528364896774292, -2.775887966156006, 0.6418624520301819, 2.86572527885437, -0.1997784525156021, 0.9701401591300964, -3.6228857040405273, -2.1997179985046387, 1.8573272228240967, 3.974714517593384, 2.823714256286621, 3.1674273014068604], [-4.473450660705566, -2.282059669494629, -2.063706159591675, 0.15020078420639038, -4.323561191558838, -1.3476771116256714, -0.4059862792491913, -3.1966285705566406, 2.8074734210968018, -2.2397849559783936, -0.7514097094535828, 1.692776083946228, -3.7415404319763184, 0.2907414436340332, -2.440944194793701, -0.8560293316841125, 3.406282901763916, 2.468255043029785, -4.706419944763184, -0.3493747413158417, -1.862974762916565, -0.18610627949237823, 3.2031643390655518, -1.4436756372451782, 0.35483989119529724, 0.19206833839416504, -1.4358350038528442, 0.877798855304718, -0.3441096842288971, 3.214689016342163, 1.5984320640563965, -2.747647285461426, -6.162188529968262, -1.8728220462799072, 1.9936139583587646, -1.1294077634811401, -1.3915847539901733, 0.6361039876937866, -0.6786788105964661, -2.3781232833862305, -1.9268808364868164, 5.7399444580078125, 2.9554624557495117, 0.6761136651039124, -1.4790045022964478, -0.28668850660324097, 1.7249984741210938, -2.55275821685791, -1.2756067514419556, 2.1607117652893066, 0.42546772956848145, 0.06621917337179184, -2.4249942302703857, -1.803168773651123, -0.058999404311180115, -1.5865552425384521, 0.5756301879882812, -4.6109700202941895, -1.5796387195587158, 0.396008163690567, 2.1193039417266846, 0.2659520208835602, 0.2642243504524231, 0.67917400598526, -3.443582534790039, -1.494872808456421, 5.563387870788574, -2.8189327716827393, -2.5603089332580566, 2.0786774158477783], [-0.460342675447464, 2.890393018722534, -0.26164522767066956, 0.37980982661247253, 0.7753119468688965, 0.19435551762580872, 1.4706439971923828, -0.5226118564605713, -2.6908926963806152, 0.1768268495798111, 1.2592525482177734, 0.6874167323112488, 2.9508960247039795, -0.3951522409915924, 0.7100769877433777, -2.121511220932007, -0.15785883367061615, 0.966608464717865, 2.346014976501465, -0.4389229118824005, 1.0487797260284424, -1.3121803998947144, -1.9289207458496094, 0.28466713428497314, -1.2504669427871704, -2.3128702640533447, 3.1169536113739014, -1.7391279935836792, 1.730706810951233, -1.2156590223312378, -0.10176429897546768, 3.495168447494507, 1.4692363739013672, -0.4336923658847809, -1.3781706094741821, 0.4676894247531891, 2.5682129859924316, -3.076620101928711, -1.4509782791137695, -0.0638023242354393, -0.10072248429059982, -2.3088691234588623, 1.5653960704803467, -0.9583333730697632, 2.2712225914001465, -2.3181347846984863, -0.3588648736476898, 1.1663020849227905, 0.5039340853691101, -1.8409358263015747, 0.5832659602165222, -1.501537799835205, 1.7391390800476074, -1.48136568069458, 1.6884742975234985, 0.5351443886756897, -2.317101001739502, 0.5979524254798889, 2.156287431716919, -0.2098589539527893, -0.09746774286031723, 0.5674289464950562, -2.201979875564575, -1.9659311771392822, -0.588937520980835, 0.42019355297088623, -3.4757487773895264, -0.9304850697517395, -1.8560477495193481, -2.513502836227417], [0.12126678228378296, 1.4785444736480713, -2.0476536750793457, 0.5408192276954651, -0.05885889381170273, -0.13048198819160461, 2.7924859523773193, -2.4422099590301514, -2.292616128921509, -0.36730238795280457, 0.8615468144416809, -0.11547403782606125, 2.373145341873169, 0.36655381321907043, -0.06357046961784363, -1.0556955337524414, 0.9602984189987183, 1.6194195747375488, 1.9896937608718872, -1.2078748941421509, 0.930805504322052, -1.9207066297531128, -1.5123884677886963, 0.476994127035141, -0.5899304747581482, -2.2138259410858154, 1.8172458410263062, -2.384538412094116, 2.6703805923461914, -2.008338451385498, -1.4003549814224243, 2.098010540008545, 0.9635600447654724, 1.316248893737793, -2.081024169921875, 1.1711896657943726, 2.260894775390625, -1.3871593475341797, -2.2903366088867188, 0.4419088363647461, -1.2929511070251465, 0.23079584538936615, 0.15849155187606812, -0.5771240592002869, 2.596062421798706, -1.326361894607544, 0.8755002617835999, 1.4309955835342407, 0.3190605938434601, -1.541016697883606, 2.398052930831909, -1.8283177614212036, 1.630089521408081, -2.0478742122650146, 2.3336260318756104, -1.951953411102295, -1.8217271566390991, -1.1703566312789917, 3.609002113342285, 1.1925218105316162, 0.9106029272079468, 0.3024439513683319, -2.235565662384033, 0.15991295874118805, 0.191297248005867, 2.915281295776367, -3.5792365074157715, -1.3040289878845215, -1.9018146991729736, -2.688126564025879], [-1.3760868310928345, 2.53981614112854, -0.29728901386260986, -3.817950963973999, 0.6033607721328735, -1.256087303161621, 0.7653999328613281, 0.7797508835792542, 0.7364276647567749, 1.889086127281189, -4.633363723754883, 2.373063087463379, 0.1516297310590744, -3.577172040939331, -1.7679879665374756, -3.021343469619751, 3.952355146408081, -1.9516651630401611, 3.476196765899658, 0.34591683745384216, 3.0255658626556396, 1.3742700815200806, 1.480851411819458, 0.6970449090003967, -3.814037799835205, 1.6778295040130615, 2.13228440284729, 3.7012879848480225, -2.805873155593872, 2.6749672889709473, 2.6005442142486572, 1.7320683002471924, -2.2157657146453857, -1.5731887817382812, -1.4042057991027832, 1.0765480995178223, -4.3420729637146, -0.1709793210029602, 2.1785340309143066, 3.194733142852783, -0.5246708989143372, 3.786811113357544, 4.764003276824951, -0.5179913640022278, 0.04356333985924721, -4.775506019592285, -1.8505728244781494, -0.048896655440330505, -0.8002057671546936, -1.429307222366333, -4.7863054275512695, 4.1923508644104, -1.1443991661071777, 0.6566609740257263, -0.5177054405212402, 5.029781818389893, -4.908841609954834, 0.7283980846405029, -0.4445764720439911, 0.3110544681549072, -0.945432186126709, -5.088533878326416, 3.4362502098083496, -0.8011740446090698, 0.9093809127807617, -3.699878215789795, 0.9546841382980347, -2.634150266647339, -2.119370222091675, -0.8350163102149963], [1.9639577865600586, 1.6787099838256836, 0.9994425177574158, -1.748422384262085, 3.0253682136535645, -1.4191830158233643, 0.4829080402851105, 1.6462424993515015, -0.946864664554596, 2.172980546951294, -0.7435751557350159, -1.2776811122894287, 3.389512777328491, 0.1492110639810562, 2.062530517578125, 3.0257484912872314, 0.527984619140625, -3.77034068107605, 4.033386707305908, 2.155733346939087, 2.5317187309265137, -0.5354699492454529, -2.283419370651245, 1.3861088752746582, 0.38636717200279236, -0.2589395344257355, -0.6579450368881226, 0.26039454340934753, -0.39038217067718506, -2.996671676635742, -1.4976927042007446, 2.728879928588867, 2.478695869445801, 0.3251751959323883, -2.2131707668304443, 4.1439008712768555, 0.708939790725708, -0.5101964473724365, 0.7606326341629028, 2.3591222763061523, 0.9303051829338074, -2.5026471614837646, -3.689370632171631, -1.3563240766525269, 1.1663808822631836, 0.6265509724617004, 0.17950470745563507, 0.5832248330116272, 0.17899249494075775, -1.7815910577774048, 2.4830029010772705, 0.5275408029556274, -0.7358821630477905, -0.10467434674501419, 2.2223095893859863, -3.961759090423584, -2.104377031326294, 0.38344502449035645, 4.359772682189941, 0.4159696698188782, 1.070699691772461, -1.6266916990280151, 2.020773410797119, -0.6927028298377991, 2.9937169551849365, 1.7016314268112183, -5.754820346832275, 2.2987818717956543, 2.471940517425537, -4.49788236618042], [3.5548176765441895, -1.3666744232177734, 0.550068199634552, -0.35233303904533386, 2.492746114730835, 2.0520544052124023, 1.0422579050064087, 0.054769616574048996, -0.3194809854030609, 1.6747807264328003, -0.6726840138435364, -2.184691905975342, 0.5113683342933655, -0.2699691951274872, -0.6738309860229492, 0.8461471796035767, -2.132934093475342, -0.23631608486175537, 2.7885055541992188, -1.4048553705215454, -0.3653092682361603, 0.6714181303977966, -0.3670307695865631, 1.1825777292251587, -0.19242535531520844, 0.8612903952598572, 0.4154330790042877, -1.4924142360687256, -0.2145337611436844, -1.0796234607696533, -2.259774684906006, -0.861656904220581, 5.076629638671875, 4.816304683685303, -2.181814432144165, -0.46363356709480286, 0.023173896595835686, 4.092711448669434, 0.5082875490188599, 3.182384729385376, 0.21335557103157043, 1.3223917484283447, -3.0230295658111572, 1.0337620973587036, 0.7227949500083923, 1.5255850553512573, -0.5182276368141174, 2.746542453765869, 1.2057735919952393, -0.6130622029304504, -0.6794323325157166, 0.7293333411216736, 3.356262683868408, 3.367365598678589, -2.011115074157715, 2.1691858768463135, 0.39984333515167236, 3.443962335586548, 0.36212438344955444, 1.4108482599258423, -1.8905575275421143, -1.4680238962173462, 0.4428843557834625, 2.9129281044006348, 4.245960712432861, 3.4312877655029297, -1.3775684833526611, 1.5105931758880615, 1.5293351411819458, -0.14487582445144653], [2.8502063751220703, 3.0273780822753906, -1.8517481088638306, -0.7292029857635498, 1.5125974416732788, -0.2153652161359787, 2.3345115184783936, 1.5097970962524414, -2.086782693862915, -0.29090815782546997, -1.4748263359069824, 1.2638006210327148, 0.669904887676239, -1.160941481590271, 0.2847881317138672, -3.542900800704956, 2.158562183380127, -2.4493467807769775, 1.810800313949585, -0.43755295872688293, 3.089531183242798, -0.06833303719758987, -1.4016729593276978, 0.39769330620765686, -1.7491029500961304, 2.3382368087768555, -0.6548205614089966, 1.2520112991333008, 0.24067972600460052, -0.9472724199295044, -0.07835543900728226, 0.019923189654946327, -1.389298915863037, -1.1426037549972534, -0.19821807742118835, 0.9184473156929016, -2.967747926712036, -2.550612211227417, 0.8441365361213684, 1.634350299835205, -0.1310412883758545, -0.9036833047866821, 1.05171799659729, -1.2338030338287354, 0.6326810717582703, -2.381258487701416, -1.1926770210266113, 0.5538262128829956, -1.4252305030822754, -1.0048083066940308, -2.213442802429199, 1.884150505065918, -1.214381456375122, -0.9928536415100098, 1.4851551055908203, 2.57867693901062, -2.1001603603363037, -2.079730987548828, 0.8451083302497864, 0.8116565942764282, -0.5954492688179016, -1.2081307172775269, 0.2134757936000824, -1.7930076122283936, 0.4068739414215088, -1.5177241563796997, -0.3318515419960022, -3.4359359741210938, -1.6029540300369263, 0.2323640137910843], [-2.9333107471466064, 0.6280289888381958, 0.5388227701187134, 1.792670726776123, 0.142098531126976, 1.912559151649475, 0.783220648765564, -2.7703466415405273, -1.6268038749694824, -0.7841380834579468, 3.2840120792388916, 0.75162672996521, 1.0656216144561768, 0.4267805516719818, -0.9589411616325378, -3.7288765907287598, -1.0765892267227173, 3.6307482719421387, -0.7172055840492249, -1.1579748392105103, -3.069199323654175, -1.6549807786941528, -0.9071619510650635, -0.7955884337425232, -0.3568292558193207, -2.979810953140259, 3.463724374771118, -5.158496856689453, 1.5721513032913208, 0.9323147535324097, -0.8580440282821655, 1.800317645072937, 1.7896417379379272, 0.5502733588218689, -0.3245384693145752, -2.2819859981536865, 5.54706335067749, -1.5977861881256104, -2.109095335006714, -1.6549736261367798, -0.28952592611312866, 0.7662979960441589, 1.0993338823318481, -0.10745608806610107, 2.050435781478882, -0.47910255193710327, 0.9003627300262451, 1.0441601276397705, 0.7770208716392517, -0.22985126078128815, 1.5054391622543335, -2.9048545360565186, 4.015244960784912, -0.022252993658185005, -0.02915220335125923, 1.1951273679733276, -1.3799880743026733, -0.6602131128311157, 0.9722135066986084, 0.14614659547805786, 0.37640151381492615, 2.488827705383301, -3.850518226623535, -2.5456764698028564, -1.8547652959823608, 1.2217655181884766, 0.17052963376045227, -2.3249073028564453, -3.702636241912842, -1.884775996208191], [-5.3239359855651855, -1.922961711883545, 1.2902472019195557, -0.6424953937530518, -1.9570826292037964, -2.685807228088379, -2.555281639099121, -1.5370662212371826, 3.074376344680786, -0.22363010048866272, 0.687281608581543, 0.1913466900587082, -0.7074937224388123, 1.538540005683899, 0.2773418426513672, 3.9272494316101074, 2.0946390628814697, -0.2960875332355499, -3.0527405738830566, 3.164501905441284, -1.9751676321029663, -0.8050035834312439, 1.265169382095337, -0.8826722502708435, 2.3868563175201416, -1.7307384014129639, -1.9244884252548218, -0.025752704590559006, -1.2712875604629517, 1.187422275543213, 0.5883570909500122, 0.41406556963920593, -3.3849005699157715, -2.770392656326294, 1.1646628379821777, 2.1488242149353027, 2.3783957958221436, -0.08305530250072479, -0.1588556468486786, -2.4987499713897705, -0.03541456535458565, 1.875533938407898, -1.4774816036224365, -0.4589637219905853, -1.127677083015442, 2.2346343994140625, 2.679983377456665, -3.254098415374756, -0.4740816056728363, 1.4326927661895752, 4.644009590148926, -1.1192539930343628, -3.001039743423462, -1.5585490465164185, 1.493024468421936, -7.524242877960205, -0.046066764742136, -3.712120771408081, 1.8295880556106567, -0.6368275880813599, 3.9995992183685303, 0.7750611305236816, 1.7625988721847534, -1.2444175481796265, -2.1378238201141357, -0.3829980790615082, 0.5707604885101318, 1.5245758295059204, 0.8175050616264343, -2.767453670501709], [1.4565874338150024, -3.051661729812622, -0.23377647995948792, -2.1477859020233154, 2.1340019702911377, 1.764376163482666, 0.8114059567451477, -0.5310993194580078, 3.116752862930298, 0.5820120573043823, -3.4725425243377686, 0.20140954852104187, -4.681541919708252, -1.3705737590789795, -3.9745471477508545, -3.3343825340270996, 2.8934295177459717, -2.3823225498199463, -1.2276036739349365, -0.6956437826156616, -2.06827712059021, 1.9029877185821533, 2.5869081020355225, -0.11748676002025604, -0.7443453073501587, 5.857440948486328, -3.5201034545898438, -0.3207368552684784, -4.058492660522461, 3.715663433074951, -1.585094690322876, -5.966210842132568, -1.0387440919876099, 2.723069429397583, 0.43878448009490967, -1.7043523788452148, -4.182039260864258, 5.7389984130859375, 3.4675590991973877, 3.7147324085235596, -0.3871883749961853, 9.440465927124023, -2.2202556133270264, 1.014298915863037, -1.8435406684875488, 1.9379593133926392, 0.2748605012893677, 0.32270270586013794, -1.6636203527450562, 2.2059683799743652, -3.3145954608917236, 4.748230457305908, 0.48709866404533386, 5.027527332305908, -3.5764474868774414, 4.748985290527344, -1.0901086330413818, -3.3197989463806152, -1.8246681690216064, 3.0380492210388184, -0.45532339811325073, -3.388732671737671, 3.6602063179016113, 0.8435868620872498, 3.109523296356201, -0.24375425279140472, 7.6774001121521, -4.611828804016113, -1.6611992120742798, 2.1413254737854004], [1.8056577444076538, 7.0303544998168945, 1.4096529483795166, -0.7399269938468933, 5.041969299316406, 1.1431783437728882, 1.1859883069992065, 4.709105491638184, -4.160283088684082, 0.26394346356391907, 0.4311717748641968, 3.563070774078369, 2.265578269958496, -2.3822906017303467, 1.5101661682128906, -8.983160972595215, 1.2732828855514526, -4.651186466217041, 2.3313708305358887, 1.2671499252319336, 2.5945026874542236, 0.022819314152002335, -3.7123358249664307, -0.3169969618320465, -3.1372909545898438, 2.793825626373291, 1.3980573415756226, -0.39422523975372314, -1.4534448385238647, 0.5110141634941101, 0.8429542779922485, 2.6833865642547607, -0.45548099279403687, -4.9545674324035645, 1.2702219486236572, -0.4037294089794159, -0.5662172436714172, -7.533355712890625, 2.5327694416046143, 0.8467701077461243, 2.3466873168945312, -4.9963202476501465, 2.3961381912231445, -3.060866117477417, 1.0435612201690674, -4.041533470153809, -2.847357749938965, 0.33619990944862366, -2.006432056427002, -1.5642589330673218, -4.6671857833862305, 2.326294183731079, -0.17490914463996887, 0.1508432924747467, 1.3211369514465332, 7.473589897155762, -5.029300689697266, -2.344974994659424, 0.07737033069133759, -0.7986001968383789, -1.4850776195526123, 0.13592438399791718, -0.7430750131607056, -9.175671577453613, -1.3380591869354248, -6.075053691864014, 0.43546557426452637, -6.223586559295654, -3.9782135486602783, -1.273931860923767], [-0.18941007554531097, -0.9473623037338257, 0.43490439653396606, -0.5224658250808716, -0.015849027782678604, -0.9117173552513123, -0.3917750418186188, -0.4944870173931122, 0.6671160459518433, 0.925872266292572, -0.15018343925476074, -1.3278849124908447, 1.052672028541565, 0.6657263040542603, 0.5890541076660156, 3.5580623149871826, -0.26610440015792847, -0.2530181109905243, 1.061830759048462, 0.7988755702972412, 0.18651482462882996, -0.2911144495010376, -0.06418134272098541, 0.5832885503768921, 0.970497727394104, -1.2170381546020508, -0.2536337971687317, 0.0348694771528244, 0.1341802477836609, -1.2235709428787231, -0.6601856350898743, 0.9120693206787109, 1.252220869064331, 1.0871223211288452, -1.1990928649902344, 1.7926957607269287, 0.9829989075660706, 1.4002822637557983, -0.38445502519607544, 0.4985194206237793, -0.0648706704378128, -0.06581152230501175, -1.9628435373306274, 0.1402055323123932, 0.2782854437828064, 1.2152622938156128, 0.8133909106254578, 0.041965972632169724, 0.7010465264320374, -0.3945263624191284, 2.363147735595703, -0.5990225672721863, -0.14344748854637146, -0.11230411380529404, 0.6039482951164246, -3.6376023292541504, 0.24063147604465485, 0.9428946375846863, 1.8160912990570068, 0.12486107647418976, 0.909024178981781, -0.5335898399353027, 0.9222894906997681, 1.633956789970398, 1.3167697191238403, 2.035447359085083, -2.6692166328430176, 2.7197153568267822, 1.97510826587677, -1.8765897750854492], [-3.5426623821258545, 1.304564356803894, 4.221650123596191, -0.8862412571907043, 1.4077688455581665, 0.011349763721227646, -3.0327186584472656, 1.7606210708618164, 0.5191472172737122, 1.7917886972427368, 1.0794235467910767, 0.8262078762054443, 1.6137170791625977, -0.831873893737793, 0.9789607524871826, 0.15105096995830536, -1.6123963594436646, -0.5696786642074585, 0.7975900769233704, 2.3682141304016113, -1.245623230934143, 0.5121527910232544, -0.3158312737941742, -0.277505099773407, -0.38221243023872375, -1.9056613445281982, 2.8333377838134766, -0.31452709436416626, -2.161226749420166, 1.7819136381149292, 1.7403013706207275, 3.556058883666992, 1.6811760663986206, -2.4444541931152344, 0.3060115873813629, -0.24176111817359924, 3.0928072929382324, -1.6327660083770752, 1.057034969329834, -0.9158150553703308, 1.956428050994873, -2.4873433113098145, 1.1274549961090088, -0.5696484446525574, -0.1431729793548584, -0.5855807065963745, -0.7868337035179138, -0.7741617560386658, 0.9316929578781128, -0.494861364364624, -0.5714914798736572, -0.2610114514827728, 0.8720104694366455, 1.3773038387298584, -0.9605263471603394, 1.2259961366653442, -1.7216793298721313, 2.900160074234009, -0.7106070518493652, -2.2226874828338623, -0.22207187116146088, 0.18127544224262238, 0.6946829557418823, -3.539785861968994, -0.9399750232696533, -2.829423666000366, -0.8925334215164185, 1.9546819925308228, 0.3637749254703522, -2.4010367393493652], [1.4315859079360962, 2.0921051502227783, -2.7285094261169434, 3.586838483810425, -0.21218904852867126, 1.6147804260253906, 3.363152027130127, -2.2485063076019287, -4.44636869430542, -2.6870455741882324, 4.070708274841309, 0.2722843885421753, 1.8680475950241089, 1.637632966041565, 0.673503041267395, -4.0798821449279785, -0.9861013889312744, 3.0931248664855957, -0.6967206001281738, -2.4868593215942383, -0.4280335307121277, -2.779339075088501, -2.9984238147735596, -0.529680073261261, 0.19723835587501526, -2.1893439292907715, 1.4092131853103638, -4.825148105621338, 4.812087059020996, -2.757812976837158, -2.189471483230591, 0.8431395888328552, 0.9222322702407837, 0.6503718495368958, -0.10337811708450317, -1.2927616834640503, 4.007857322692871, -4.100360870361328, -3.518225908279419, -2.0852413177490234, -0.9474686980247498, -2.9658308029174805, 0.06906330585479736, -0.8100323677062988, 2.8293704986572266, -0.6517193913459778, 0.8436846137046814, 1.673140048980713, -0.053479600697755814, -0.7626859545707703, 2.559968948364258, -3.9539811611175537, 2.5971364974975586, -3.0092077255249023, 2.6522271633148193, -0.6069818735122681, 0.2774941325187683, -2.81213116645813, 2.3626549243927, 0.8077637553215027, 0.3753756284713745, 4.1638007164001465, -5.988438129425049, -1.8691198825836182, -2.1303956508636475, 2.7111966609954834, -2.055943250656128, -3.3278942108154297, -3.4117820262908936, -0.4765641391277313], [-4.457818031311035, -2.870248556137085, -0.11877473443746567, -1.8577828407287598, -3.6166298389434814, -2.084782600402832, -1.4088858366012573, -2.952620029449463, 3.573821783065796, 1.118206262588501, -2.375981330871582, -0.47056135535240173, -0.8535301089286804, -0.5131560564041138, -1.7713409662246704, 4.6781005859375, 1.4402908086776733, 2.5909740924835205, 0.1665978580713272, 0.22972805798053741, -0.5146429538726807, 0.5370416045188904, 3.516228437423706, 0.31804412603378296, -0.08129873126745224, -2.421964645385742, 1.5011427402496338, 2.3106794357299805, -0.9186409711837769, 1.9146760702133179, 1.8930084705352783, 0.8093737959861755, -1.4208977222442627, 1.017342448234558, -1.4818106889724731, 1.061052918434143, -0.6047893762588501, 3.9064550399780273, -0.7378658652305603, 0.19600017368793488, -1.6524336338043213, 5.009603023529053, 2.2431349754333496, 1.5157275199890137, -0.5971019268035889, -0.5528334975242615, 1.0574133396148682, -1.007358193397522, 1.1777862310409546, 0.09949224442243576, 1.00838303565979, 0.1497480869293213, -0.6985268592834473, -0.28958991169929504, -0.9012268781661987, -2.9939568042755127, -0.11081144213676453, 2.2999227046966553, -0.17372705042362213, -0.17622600495815277, 1.0764379501342773, -2.6600358486175537, 2.3879990577697754, 4.546989917755127, 0.18740299344062805, 1.0101927518844604, -0.19977471232414246, 3.292950391769409, 1.1671433448791504, -0.7737455368041992], [0.7479738593101501, 3.144205331802368, 2.964181661605835, 1.0961312055587769, 1.6638895273208618, -0.5682834386825562, -3.3909482955932617, 6.251330375671387, -1.053332805633545, -1.0381380319595337, 2.217942476272583, 1.8351647853851318, -0.0640270859003067, 0.46683865785598755, 3.8768484592437744, -1.4113833904266357, -1.575416088104248, -4.485566139221191, -2.9462478160858154, 3.392930507659912, 0.8515830039978027, 0.7850778102874756, -2.229346752166748, -1.2917780876159668, 1.149723768234253, 2.178656816482544, -2.6468594074249268, 2.326474666595459, -1.5488654375076294, -0.07778382301330566, 2.0495355129241943, 0.2940879762172699, -2.5923447608947754, -6.976074695587158, 4.623527526855469, -0.2699773609638214, -0.6674948930740356, -6.550996780395508, 2.413496255874634, -3.461568593978882, 3.3717963695526123, -9.092615127563477, -0.06820791214704514, -2.0743234157562256, -2.3590357303619385, 0.4306352436542511, -1.7209361791610718, -2.9372549057006836, -1.652387022972107, 1.01823890209198, -1.750427007675171, 0.4178827106952667, -4.194582939147949, -1.0762264728546143, 0.9584129452705383, 0.9742243885993958, 1.4783397912979126, -1.6382043361663818, -2.8264012336730957, -3.328233242034912, -0.22321395576000214, 3.351264476776123, 0.09177087992429733, -6.930951118469238, -3.6154823303222656, -6.728061676025391, 1.582798719406128, -0.10029406100511551, 1.4445466995239258, 1.8732025623321533], [3.314513683319092, 8.945651054382324, 3.754361867904663, 2.0980026721954346, 4.252184867858887, 1.7930653095245361, -1.630129337310791, 8.64052677154541, -6.700678825378418, 0.0066803074441850185, 3.8634438514709473, 2.8441126346588135, 4.778631687164307, -1.6345869302749634, 5.646302700042725, -6.416216850280762, -5.287466526031494, -3.134972333908081, 2.452653408050537, 1.3141497373580933, 3.5920004844665527, 0.48857688903808594, -5.681873798370361, -0.4097294211387634, -2.3939309120178223, -0.22946761548519135, 4.146419525146484, 1.254215955734253, 0.8324060440063477, -1.925432562828064, 3.06091046333313, 5.938326835632324, 2.66020131111145, -6.944377899169922, 2.7064578533172607, -1.9596401453018188, 1.3119239807128906, -11.470382690429688, 1.2417840957641602, -2.613753318786621, 4.491659164428711, -16.578357696533203, 4.332482814788818, -2.938182830810547, 0.7541671991348267, -4.716907978057861, -5.355855941772461, 0.5575150847434998, 0.015655873343348503, -2.598191261291504, -5.877797603607178, -0.8083624839782715, 0.26248699426651, -1.3053064346313477, 1.0847088098526, 8.868654251098633, -0.9150044322013855, 4.896331310272217, -3.3328356742858887, -4.979660511016846, -4.43556547164917, 3.9761102199554443, -3.9247958660125732, -9.525915145874023, -3.562072277069092, -8.037330627441406, -3.677961587905884, -0.3809567093849182, -0.3808104991912842, 1.245916485786438], [-1.0812278985977173, -1.6742801666259766, 0.31974199414253235, 3.7533745765686035, -2.37903094291687, 1.466766357421875, -1.0136131048202515, -1.709991693496704, -0.9747706055641174, -1.7505537271499634, 4.381433963775635, -1.2224100828170776, 0.24686911702156067, 2.2276852130889893, 0.9045429229736328, 1.5788198709487915, -4.4301934242248535, 4.860833644866943, -2.7643070220947266, -1.512405276298523, -2.967371702194214, -1.1059012413024902, -0.39694684743881226, -0.8291657567024231, 1.9747650623321533, -3.734297752380371, 1.7238495349884033, -2.9828670024871826, 3.216576337814331, -1.0620402097702026, -0.37892773747444153, 0.3262355327606201, 2.1079297065734863, 1.1166934967041016, 0.9264862537384033, -2.622877597808838, 4.633293151855469, -0.5012695789337158, -3.1323890686035156, -3.8177683353424072, -0.07960103452205658, -3.516928195953369, -0.025015855208039284, 1.0962743759155273, 0.500565767288208, 1.7414604425430298, 0.7937218546867371, 0.42110955715179443, 1.737318992614746, 0.6240352392196655, 2.5601398944854736, -4.657285690307617, 2.507824659347534, -1.1315770149230957, -0.40430763363838196, -1.734647512435913, 4.046518802642822, 2.3193984031677246, -1.3385354280471802, -1.3775403499603271, -0.3769729733467102, 4.604259014129639, -4.531031131744385, 1.3017491102218628, -2.386258602142334, 2.280350923538208, -0.8762056827545166, 2.7201740741729736, 0.5568904876708984, 1.5168869495391846], [-1.5884937047958374, 0.5127835869789124, 4.018052101135254, 0.2447388619184494, 0.9440711736679077, 1.044711947441101, -2.296776294708252, 0.6580137014389038, -0.9314276576042175, 2.762558937072754, 2.0441946983337402, -1.6571050882339478, 4.033294200897217, -0.3212944567203522, 1.6692150831222534, 3.5282154083251953, -5.428836345672607, 2.5221292972564697, 3.759495496749878, 0.3085276782512665, -0.8579461574554443, 0.16565173864364624, -0.9577176570892334, 0.9207280874252319, -0.2636359632015228, -5.117274761199951, 5.894416332244873, -1.3211174011230469, 0.5336204767227173, -0.9154435992240906, 0.8094803094863892, 5.581229209899902, 6.920590877532959, 1.8019838333129883, -2.3902089595794678, -0.4197038412094116, 5.142159938812256, 0.5199669003486633, -1.256972312927246, -0.13369305431842804, 1.4587466716766357, -4.883650779724121, 0.6137738823890686, 0.7322580814361572, 1.569291591644287, -0.4725305140018463, -1.2291450500488281, 1.9170100688934326, 3.5207180976867676, -2.159376382827759, 0.5238945484161377, -2.652980089187622, 4.351059436798096, 1.5929867029190063, -1.4985930919647217, 0.7363384366035461, 0.09443438053131104, 9.042651176452637, -0.07632190734148026, -2.4049084186553955, -2.0550506114959717, 0.44791415333747864, -1.3287605047225952, 1.0286177396774292, 1.2417035102844238, 1.5070840120315552, -6.029440402984619, 6.536919593811035, 2.6545815467834473, -2.733297109603882], [2.833216667175293, -0.7986816167831421, 0.975114643573761, 1.2495143413543701, 1.9696635007858276, 0.947786808013916, -0.6828553080558777, 2.012638807296753, -0.2505806088447571, -0.5873585343360901, 1.6274856328964233, -1.0505199432373047, -0.7813284397125244, 1.5272657871246338, 1.3608533143997192, 0.5097370147705078, -1.593729853630066, -2.3274831771850586, -1.8181816339492798, 0.8711889982223511, -1.0366544723510742, 0.09918618202209473, -1.3895750045776367, -0.32306408882141113, 2.064436197280884, 2.0962085723876953, -3.433546781539917, -1.1935449838638306, -0.3006508946418762, -1.2652842998504639, -1.8649977445602417, -2.3844542503356934, 0.9370747208595276, 0.017979752272367477, 1.6491615772247314, -0.24057388305664062, 0.3941986858844757, -0.08045925945043564, 1.03399658203125, -0.5465471744537354, 1.4237195253372192, -2.508141279220581, -4.361780643463135, -0.36170709133148193, -1.0014606714248657, 3.3833959102630615, 0.34123605489730835, -0.2910382151603699, -0.6509653329849243, 1.2404474020004272, 1.1087572574615479, -0.14673730731010437, -0.5856579542160034, 1.1076499223709106, -0.11209727078676224, -1.193086862564087, 2.2471354007720947, -1.7511208057403564, -0.21485060453414917, 0.10985329747200012, 0.1887652426958084, 2.0644137859344482, -0.1278746873140335, -1.4007246494293213, 0.5736193060874939, 0.37733009457588196, 1.0051219463348389, 0.20449692010879517, 1.6893996000289917, 0.9359455704689026], [-2.38958477973938, -2.3056976795196533, -2.516277313232422, 0.4154263436794281, -5.247801303863525, -0.7415586113929749, -0.5778390169143677, -1.9217756986618042, 2.4694466590881348, -2.086590051651001, -1.7817400693893433, 1.3407891988754272, -4.121525764465332, -0.5822051167488098, -2.2991037368774414, -0.3665091097354889, 1.4942450523376465, 3.410984516143799, -3.816582202911377, -1.9014389514923096, -0.5009192824363708, 1.0767983198165894, 3.906261682510376, -1.0113087892532349, -0.6131565570831299, 0.596488356590271, -0.34696030616760254, 3.3700759410858154, 0.11395882070064545, 3.015866279602051, 2.919084310531616, -3.1045217514038086, -5.248070240020752, -0.9167357087135315, 1.9109407663345337, -2.441507339477539, -4.0511603355407715, 1.5086692571640015, -0.6225046515464783, -1.9117931127548218, -1.9850436449050903, 3.948517084121704, 5.16923713684082, 1.68486487865448, -2.0389537811279297, -1.629130482673645, -0.1386992484331131, -1.6522786617279053, -0.5124334692955017, 1.725243330001831, -2.776668071746826, 0.6889753937721252, -2.0855700969696045, -1.387830138206482, -1.5874947309494019, 2.0106146335601807, 2.36639666557312, -0.3886948823928833, -4.518211364746094, -0.4317666292190552, -0.5180147886276245, -0.3786904513835907, 0.09443585574626923, 3.3837859630584717, -2.9357869625091553, -2.1911160945892334, 5.497026443481445, -1.0731985569000244, -1.219698190689087, 5.2766900062561035], [-1.985131859779358, -0.9421095848083496, 0.04020043462514877, 4.499743461608887, -4.023612976074219, -0.4284098148345947, -1.771182894706726, -1.441514015197754, -1.4987411499023438, -2.3497111797332764, 5.576120376586914, -1.4489930868148804, 1.7667936086654663, 3.4204864501953125, 3.0072479248046875, 4.657688140869141, -4.744983196258545, 5.084775924682617, -3.2065014839172363, -0.3746463656425476, -1.7690869569778442, -2.031601905822754, -1.220069169998169, -0.8991986513137817, 3.185091733932495, -5.922811031341553, 1.5228030681610107, -1.898537278175354, 4.819492340087891, -2.993098497390747, 0.3556353747844696, 2.2416763305664062, 0.9234408140182495, -0.8955792784690857, 1.2690321207046509, -0.7889971137046814, 5.740085124969482, -2.9906198978424072, -4.498361110687256, -5.930819034576416, -0.046661898493766785, -7.489312648773193, 0.0885130763053894, 0.4745034873485565, 0.6449161171913147, 1.7119194269180298, 1.3504148721694946, -0.7910457253456116, 1.9524929523468018, 0.17729702591896057, 5.214904308319092, -6.613977432250977, 0.41702669858932495, -4.201412677764893, 1.956870675086975, -6.445617198944092, 5.110032081604004, 2.174798011779785, -0.054245829582214355, -2.6868691444396973, 0.9546752572059631, 6.132986068725586, -5.240786075592041, 1.2068884372711182, -4.089510917663574, 2.3147637844085693, -4.076908111572266, 5.197076320648193, 2.1283037662506104, 0.5392722487449646], [1.2469109296798706, 3.0799994468688965, 3.40634822845459, -0.6227176785469055, 5.637061595916748, 1.096314787864685, -0.37519004940986633, 3.178582191467285, -2.0165469646453857, 1.614098310470581, 1.7054390907287598, 0.3461611270904541, 2.687584161758423, -0.18212005496025085, 1.8795007467269897, -2.646198034286499, -0.7455838918685913, -4.383339881896973, 2.130415678024292, 2.5573928356170654, -0.18067018687725067, -0.40916457772254944, -3.481891632080078, 0.1738152801990509, -0.005985857453197241, 1.0218104124069214, -0.06115297973155975, -2.8743693828582764, -1.7534555196762085, -0.8133780360221863, -1.722721815109253, 2.2762908935546875, 3.2048251628875732, -1.5442588329315186, -0.03318092226982117, 1.07175612449646, 3.146627426147461, -3.014476776123047, 2.031938076019287, 1.12973952293396, 2.710848569869995, -3.6537845134735107, -3.600914716720581, -2.1190426349639893, 0.9681500792503357, 0.9537870287895203, -0.5123668909072876, 0.534885823726654, -0.5018989443778992, -0.8473244905471802, 0.5147752165794373, 0.5559585094451904, 1.1975277662277222, 2.0118680000305176, 0.7673107385635376, 0.9820123910903931, -3.0415852069854736, -1.1725704669952393, 2.481910467147827, -0.17839832603931427, 0.2637147903442383, 0.6488440632820129, 0.3218778669834137, -6.331738471984863, 1.2926857471466064, -1.2262840270996094, -1.9450321197509766, -1.360435128211975, -0.33531656861305237, -4.016729831695557], [6.757592678070068, -0.9847474098205566, 2.4400808811187744, -1.4269888401031494, 6.932000637054443, 3.711747884750366, 0.2293178141117096, 4.343298435211182, 0.5289991497993469, 1.8217079639434814, -1.880087971687317, -1.031008243560791, -2.573962688446045, -1.1800720691680908, -1.0591992139816284, -3.7361199855804443, -1.0510841608047485, -6.116657257080078, 0.9803737998008728, 0.18501350283622742, -1.0813653469085693, 2.5900380611419678, -0.7957281470298767, 0.5460802912712097, -0.2758733034133911, 7.640880107879639, -4.249541759490967, -1.0652872323989868, -4.470848083496094, 1.1208715438842773, -3.1355080604553223, -5.314904689788818, 3.860903739929199, 3.0391921997070312, 0.7285908460617065, -1.791223406791687, -3.5545272827148438, 4.141662120819092, 5.233733177185059, 4.872038841247559, 2.4940199851989746, 2.339613199234009, -6.006619453430176, 0.10438690334558487, -1.7551156282424927, 3.5001449584960938, -1.6982048749923706, 1.8858286142349243, -1.3210970163345337, 1.3934412002563477, -4.647741794586182, 5.129308223724365, 1.7448650598526, 7.332671165466309, -4.088054656982422, 7.425307273864746, -0.17966170608997345, -0.42475733160972595, -2.1397175788879395, 2.156275749206543, -2.7506935596466064, -2.124302864074707, 3.298809766769409, -1.726609706878662, 5.415393829345703, -0.6827399134635925, 5.060244083404541, -2.923882484436035, 0.9916622042655945, 2.14203143119812], [0.09735515713691711, 1.4927632808685303, -0.11283686012029648, -0.19315217435359955, -0.3055432140827179, -2.146925210952759, 0.13069498538970947, 0.19722378253936768, -1.3157460689544678, 0.850192666053772, 0.5824478268623352, -1.122260332107544, 3.682032346725464, 0.8764481544494629, 2.5457425117492676, 4.404642581939697, -0.42538022994995117, -0.42156079411506653, 2.6940853595733643, 1.2576302289962769, 2.491386651992798, -1.2459957599639893, -1.7575958967208862, 0.952473521232605, 0.745398998260498, -3.1102991104125977, 0.9560121297836304, 0.7709289789199829, 1.9150938987731934, -3.4543750286102295, -0.10608897358179092, 3.8752477169036865, 1.2594749927520752, -0.4575764536857605, -1.823940634727478, 3.540255308151245, 1.7952638864517212, -1.8800013065338135, -1.5803810358047485, -0.25398537516593933, 0.00838097371160984, -4.371039867401123, -0.978230893611908, -0.8583073616027832, 1.4343383312225342, -0.4239143133163452, 0.4513521194458008, 0.04840235412120819, 0.9447662234306335, -1.9092246294021606, 3.5318710803985596, -1.9321662187576294, -1.0291838645935059, -2.8048176765441895, 3.0382442474365234, -5.683382511138916, -0.3627731204032898, 1.5682474374771118, 3.5868678092956543, -0.8027192950248718, 1.2363075017929077, 0.1455194652080536, -0.18422971665859222, 0.7862195372581482, 0.441585898399353, 1.915457844734192, -6.847598075866699, 3.871600389480591, 2.4302825927734375, -3.417515754699707], [-5.761896133422852, -0.25764378905296326, 0.8555737137794495, -2.254849910736084, -1.8739862442016602, -2.0800023078918457, -1.6120586395263672, -1.4357783794403076, 2.783052682876587, 0.6865343451499939, -1.8134784698486328, 1.834023356437683, -0.8529440760612488, -1.209488868713379, -1.5133414268493652, 0.24104003608226776, 3.243832588195801, 0.34524139761924744, -0.7243040204048157, 1.7465797662734985, -0.5937866568565369, 0.3036045432090759, 2.415792942047119, -0.5189080238342285, -0.8867200016975403, -1.010807752609253, 1.1600089073181152, 1.6581228971481323, -2.3548355102539062, 3.3937528133392334, 2.4897637367248535, 1.3865013122558594, -3.6121084690093994, -2.667720079421997, 0.1599908173084259, 0.8491640090942383, -0.20656026899814606, 0.0994153544306755, 0.6580483317375183, -0.522993266582489, -0.5959938764572144, 4.266629219055176, 3.1913609504699707, -0.13518774509429932, -0.7231816649436951, -1.7026270627975464, 0.7105783820152283, -2.193570375442505, -0.3966946303844452, 0.3536338806152344, -0.13564901053905487, 1.1496539115905762, -1.790822982788086, -0.5045406818389893, -0.06801316887140274, -1.0233601331710815, -2.6804683208465576, -1.3319071531295776, -0.052399203181266785, -0.5534343123435974, 1.8158149719238281, -2.055738687515259, 2.431135416030884, -1.073987603187561, -1.7529010772705078, -2.654933452606201, 1.951930284500122, -0.6516932249069214, -1.4896925687789917, -1.7586491107940674], [-0.9766991138458252, 1.183498501777649, -5.865446090698242, -4.331212997436523, -2.4063193798065186, -2.624696969985962, 4.86934232711792, -3.907998561859131, 1.1199160814285278, 0.35797280073165894, -7.202840805053711, 2.3663856983184814, -1.2331217527389526, -3.4648330211639404, -4.444650650024414, -3.362765073776245, 8.607112884521484, 0.06583397090435028, 3.520285129547119, -2.212768077850342, 4.543615818023682, 0.16644544899463654, 3.104508876800537, 1.1123669147491455, -4.505289554595947, 2.1231536865234375, 0.7924830913543701, 4.046560764312744, -0.6260794997215271, 2.0310251712799072, 1.4431010484695435, -0.5770553350448608, -5.749673366546631, 0.9812586903572083, -3.211030960083008, 2.481699228286743, -7.264623165130615, 2.0900094509124756, 0.22643370926380157, 4.7404351234436035, -4.147378444671631, 10.759910583496094, 5.755784034729004, 0.07570269703865051, 1.1276936531066895, -5.997734546661377, -0.020780518651008606, 0.45315995812416077, -1.7934049367904663, -1.4906610250473022, -3.23940110206604, 4.621812343597412, -2.1150007247924805, -1.9070631265640259, 1.3464897871017456, 2.276531457901001, -5.759950637817383, -3.5995640754699707, 2.168381690979004, 3.457817316055298, 0.782194972038269, -7.1768879890441895, 3.423882246017456, 3.6052324771881104, 1.5341277122497559, -0.18133938312530518, 1.5162655115127563, -5.867125988006592, -4.479407787322998, -0.552671492099762], [8.239779472351074, 2.1470441818237305, -3.1700186729431152, 1.6457676887512207, 2.869999647140503, 1.9204761981964111, 3.3669424057006836, 3.340332508087158, -3.276432752609253, -2.438819169998169, -0.15550798177719116, 0.5784744620323181, -1.8177775144577026, 0.2045975774526596, 0.7503300309181213, -5.84993839263916, 0.7076154947280884, -3.999849319458008, -1.1199672222137451, -1.7828471660614014, 2.448286771774292, 0.2700178921222687, -2.5722179412841797, -0.17162677645683289, -0.30772122740745544, 6.4087653160095215, -5.0430073738098145, 0.24841946363449097, 1.0358397960662842, -2.234201192855835, -2.2813854217529297, -5.047511577606201, -1.5227936506271362, -0.2344028800725937, 2.243773937225342, -1.161291241645813, -5.038820743560791, -2.539094924926758, 1.6558095216751099, 1.2338414192199707, 0.5070775747299194, -2.656358242034912, -2.2759745121002197, -1.1309149265289307, -0.598469614982605, 0.4171702265739441, -1.479055643081665, 0.9985547661781311, -2.8008856773376465, 0.7736085653305054, -3.270747184753418, 2.291701078414917, -1.6083742380142212, -0.04332417994737625, 0.8166971206665039, 4.574910640716553, 1.4165829420089722, -4.901215553283691, -1.1465036869049072, 1.7729055881500244, -1.7108290195465088, 1.3178584575653076, -1.247578740119934, -2.1872007846832275, 0.6786323189735413, -1.2152498960494995, 3.2461321353912354, -5.857518672943115, -1.2803181409835815, 4.548959255218506], [-2.730344533920288, -4.370772361755371, -1.1763253211975098, -0.59052973985672, -0.4932103753089905, 1.308208703994751, 1.6278754472732544, -5.659900665283203, 2.809399127960205, -0.1818913221359253, -0.7115025520324707, -0.610741138458252, -2.9348669052124023, 0.4138299822807312, -4.552651882171631, -1.3258191347122192, 2.6742963790893555, 2.617846727371216, -1.7768380641937256, -1.5109496116638184, -4.409955978393555, -0.6439117789268494, 2.5876622200012207, -0.36988601088523865, 0.43673592805862427, 0.4191705584526062, -0.7475520372390747, -4.40907096862793, -0.9562569260597229, 2.8346328735351562, -2.675240993499756, -3.4767720699310303, -0.07236241549253464, 4.405184268951416, -1.3019284009933472, -1.2822446823120117, 1.7803806066513062, 5.912486553192139, -0.5054736733436584, 1.6107101440429688, -2.1433310508728027, 11.190268516540527, -2.204066753387451, 1.4429292678833008, 0.48441988229751587, 2.4616832733154297, 2.989022731781006, 0.7016173005104065, -0.15602795779705048, 1.7159353494644165, 2.283421754837036, 0.27877911925315857, 2.9584426879882812, 2.645343065261841, -1.8108789920806885, -1.0344314575195312, -1.2615951299667358, -3.933891534805298, 1.62785005569458, 3.478879690170288, 1.992815613746643, -1.362052083015442, 0.5030884742736816, 2.3101208209991455, 1.6355152130126953, 4.188573837280273, 4.505451679229736, -3.380044460296631, -3.093770742416382, -1.3070653676986694], [6.804646015167236, 0.8774247169494629, -2.916839838027954, 2.4626152515411377, 0.03629494830965996, 3.6657943725585938, 2.235311985015869, 2.5616884231567383, -2.5675148963928223, -2.5483150482177734, -0.5384665727615356, 0.9821645617485046, -3.4100561141967773, -1.290920615196228, -0.9856486916542053, -6.852281093597412, -2.173788070678711, 0.759373664855957, -1.6312882900238037, -4.588687419891357, 1.2010722160339355, 1.8421134948730469, 0.05524682626128197, -0.453578382730484, -2.0704922676086426, 5.009740829467773, -0.954795777797699, 1.6325280666351318, 1.3011267185211182, 0.42894670367240906, 0.4429054856300354, -5.172937393188477, -0.5018759369850159, 1.2012009620666504, 2.4343745708465576, -5.36818790435791, -6.065165042877197, -0.5039141774177551, 0.8355169892311096, 0.4363864064216614, -0.1304456889629364, -1.8140716552734375, 3.2241523265838623, 1.095203161239624, -1.212084412574768, -1.8981572389602661, -3.4035987854003906, 2.0004093647003174, -1.0591801404953003, 0.8800538778305054, -7.833498477935791, 2.017367124557495, 1.0408482551574707, 1.2964887619018555, -2.909554958343506, 11.138924598693848, 3.3598835468292236, 1.3226410150527954, -6.563772201538086, 0.3191074728965759, -5.166284561157227, 1.0936105251312256, -2.72139835357666, 1.129492163658142, -0.44448795914649963, -2.540456771850586, 5.825155735015869, -4.414344310760498, -1.8375009298324585, 8.91903018951416], [2.4159674644470215, -1.7614798545837402, 0.3420635759830475, 1.6479302644729614, 0.01921105943620205, 0.13080821931362152, -0.8254798650741577, 0.7972156405448914, -0.07368887960910797, -0.43529027700424194, 1.7674040794372559, -2.2336266040802, 0.15857838094234467, 2.141906976699829, 1.8356468677520752, 4.053061485290527, -2.665860891342163, -0.3053250014781952, -1.123482584953308, 0.31884732842445374, -0.541692852973938, -0.24891729652881622, -0.9375927448272705, 0.14489321410655975, 2.6189985275268555, -0.2893945574760437, -2.3358490467071533, -0.5038658380508423, 1.2762805223464966, -2.5954787731170654, -1.5685909986495972, -1.208586573600769, 1.8041633367538452, 1.2887213230133057, 0.5083379149436951, 0.5304733514785767, 1.0496222972869873, 0.9495554566383362, -0.6236805319786072, -1.028901219367981, 0.6428290009498596, -3.378146171569824, -3.9831135272979736, 0.37658417224884033, -0.5459961295127869, 3.3759427070617676, 0.7188059091567993, 0.015135033056139946, 0.579093337059021, 0.6271334290504456, 2.73958158493042, -1.717618703842163, -0.38282716274261475, -0.165215864777565, 0.3536597192287445, -3.925262212753296, 3.622180461883545, 0.7566360831260681, 0.3248177766799927, -0.3206222653388977, 0.2628907263278961, 2.1418771743774414, -0.6543623805046082, 1.7823970317840576, 0.7682433724403381, 2.4397449493408203, -1.8344438076019287, 3.4553415775299072, 3.384603977203369, 0.7050573825836182], [4.799914360046387, 0.3621640205383301, -3.641054630279541, -3.0140018463134766, 2.0415380001068115, 1.5315386056900024, 5.130423545837402, -1.581121563911438, -0.6475125551223755, 1.7472726106643677, -5.671489238739014, -0.1615312248468399, -0.6137012243270874, -3.2804555892944336, -3.914696455001831, -4.1140055656433105, 3.2324750423431396, -0.6771396994590759, 5.5958709716796875, -3.8232202529907227, 3.070462703704834, 1.208483338356018, 1.200155258178711, 1.952430248260498, -4.164031028747559, 3.7698590755462646, 1.1234989166259766, 1.0453717708587646, -0.7138438820838928, 0.4552341103553772, -1.4822238683700562, -1.992702841758728, 1.82546865940094, 5.794571399688721, -4.0787529945373535, -0.01739429496228695, -5.960039138793945, 5.035746097564697, 1.2720961570739746, 7.125481128692627, -2.2763407230377197, 8.095113754272461, 1.3717964887619019, 1.0157896280288696, 1.5696994066238403, -3.2358298301696777, -1.5969750881195068, 3.9289751052856445, -0.3232303559780121, -1.7421804666519165, -5.116065502166748, 4.700652599334717, 2.778374195098877, 2.9832775592803955, -1.8406932353973389, 7.340409278869629, -3.6970486640930176, 1.3084323406219482, 0.7074032425880432, 4.000046253204346, -2.788649559020996, -6.356935501098633, 2.1892740726470947, 4.5736083984375, 5.6601996421813965, 2.5110206604003906, 0.7814290523529053, -4.178887367248535, -2.4094858169555664, 0.8411471843719482], [-0.6881092190742493, -0.620244026184082, -0.874891459941864, 0.6661344766616821, -3.0700998306274414, -1.7321943044662476, -1.329591989517212, 0.49766889214515686, 0.8750850558280945, -1.15945303440094, -0.09272689372301102, 0.08486945182085037, -0.8775649666786194, 0.7052810192108154, 1.0814814567565918, 2.8802335262298584, -0.1278209090232849, 0.6696577072143555, -2.033855438232422, 0.3774769902229309, 0.9838249683380127, 0.2482956498861313, 0.9832389950752258, -0.3915788233280182, 0.9610428810119629, -0.585703432559967, -1.1606539487838745, 3.0539586544036865, 0.8090296983718872, -0.41905665397644043, 1.8299765586853027, -0.43519720435142517, -2.9249391555786133, -2.006474256515503, 1.454809308052063, 0.4297201335430145, -1.7496564388275146, -0.9555705785751343, -0.6497020125389099, -2.2366466522216797, -0.34114477038383484, -2.2704615592956543, 1.5780651569366455, 0.2906560003757477, -1.3591454029083252, -0.15207220613956451, -0.003280704142525792, -1.759985327720642, -0.13308285176753998, 0.6016333699226379, 0.338672935962677, -0.6807441711425781, -3.035987138748169, -2.4485385417938232, 0.7687146663665771, -2.5089120864868164, 2.5488686561584473, 0.2408989816904068, -1.5863789319992065, -1.4226641654968262, 0.3985966444015503, 0.9328339695930481, 0.12025831639766693, 1.549918293952942, -2.101534366607666, -1.3072550296783447, 0.07802177965641022, 2.154649496078491, 1.7533067464828491, 2.20257568359375], [-6.218984127044678, -4.952410697937012, 4.631701946258545, 0.7858816981315613, -1.2998629808425903, 2.307201862335205, -4.521514892578125, -2.2105369567871094, 4.135828971862793, 0.7062169313430786, 2.237504720687866, -0.5422333478927612, -2.8996453285217285, 0.43123456835746765, -2.4035847187042236, 1.2430933713912964, -3.5418927669525146, 4.32541036605835, -3.9695167541503906, 0.32101210951805115, -7.295108795166016, 1.1509126424789429, 3.33728289604187, -1.392303705215454, 1.4302538633346558, -2.3061463832855225, 2.344480514526367, -3.2154617309570312, -2.658051013946533, 5.113171577453613, 0.8655622005462646, -1.0780720710754395, 2.3158762454986572, 1.472825050354004, 1.3732082843780518, -4.496462345123291, 4.887020587921143, 4.708829879760742, 0.17457395792007446, -2.350660562515259, 0.8310447931289673, 4.367008686065674, 0.13652099668979645, 2.3365955352783203, -1.6524766683578491, 3.1768124103546143, 1.0751292705535889, -0.7625458836555481, 2.0042996406555176, 2.503483772277832, 0.1631285697221756, -1.543060302734375, 3.8510019779205322, 4.469952583312988, -5.155762195587158, 1.782555103302002, 2.0570600032806396, 3.4448256492614746, -4.129836559295654, -1.3387588262557983, -0.3223792612552643, 1.5511925220489502, -0.21405087411403656, 0.532322883605957, -1.2200443744659424, -0.12377315014600754, 5.295223236083984, 2.545454978942871, -0.01739105023443699, 0.8594098687171936], [0.22591638565063477, -2.214566230773926, -0.09214630722999573, 4.849998474121094, -2.268740653991699, 2.146899461746216, -0.8000311851501465, -1.5101368427276611, -1.323535442352295, -2.6528635025024414, 5.2875447273254395, -1.5425875186920166, -0.5398605465888977, 2.991466522216797, 1.1024534702301025, 1.1195110082626343, -5.099273204803467, 4.853435039520264, -4.126168727874756, -1.9503271579742432, -3.5903501510620117, -1.2288544178009033, -0.7741081118583679, -1.1401841640472412, 2.756082057952881, -2.9114530086517334, 0.37242141366004944, -3.735502243041992, 3.774355411529541, -1.5412180423736572, -1.1835936307907104, -1.246229887008667, 2.062859535217285, 1.4209020137786865, 1.8058191537857056, -3.3857970237731934, 4.631921768188477, -0.5586982369422913, -3.245319128036499, -4.424401760101318, 0.05270376428961754, -4.16043758392334, -1.2283375263214111, 1.185076117515564, 0.17410673201084137, 2.932105779647827, 1.0063434839248657, 0.45688924193382263, 1.4114973545074463, 1.3203892707824707, 2.8465046882629395, -5.077918529510498, 2.508254051208496, -1.0692707300186157, -0.49394768476486206, -1.7787619829177856, 5.469553470611572, 1.249947190284729, -1.844791054725647, -1.1416171789169312, -0.46469345688819885, 5.905611038208008, -5.396031379699707, 1.2071449756622314, -2.6365606784820557, 2.69946026802063, 0.19203564524650574, 2.14603328704834, 0.6354628205299377, 2.780883312225342], [6.24472188949585, 5.3587751388549805, 0.39465633034706116, 1.4793418645858765, 3.4435744285583496, 0.2882895767688751, 1.2175617218017578, 4.9399189949035645, -5.684410095214844, 0.5175483822822571, 2.191119432449341, -0.8527936339378357, 5.365215301513672, 0.381190687417984, 5.052776336669922, 0.1562606245279312, -3.664363384246826, -2.965895175933838, 4.429400444030762, 0.35242757201194763, 4.68058967590332, -0.7960606217384338, -5.389957904815674, 1.229152798652649, -0.27366405725479126, -0.7656983733177185, 1.0801854133605957, 0.5240527391433716, 2.976811408996582, -5.885366439819336, -0.7837415933609009, 4.160988807678223, 4.253815174102783, -1.1599223613739014, -0.8297982811927795, 2.008178472518921, 0.7979317307472229, -5.912814617156982, -0.5582689046859741, 0.25541436672210693, 2.1170382499694824, -12.158099174499512, -1.4372224807739258, -1.9913790225982666, 1.9623035192489624, -1.4416725635528564, -2.495509624481201, 1.962183952331543, 0.5821052193641663, -3.006843328475952, -0.1253032684326172, -1.5253164768218994, 0.11715080589056015, -2.1161413192749023, 3.0294790267944336, 0.3594576418399811, 0.32529065012931824, 3.485086441040039, 1.9770302772521973, -1.6682575941085815, -2.1070125102996826, 2.0299272537231445, -2.5146710872650146, -2.4993374347686768, 0.9591431021690369, -0.03776970133185387, -8.120458602905273, 2.479461908340454, 2.595667600631714, -1.178551435470581], [-3.121105909347534, 1.3709901571273804, 4.003251552581787, -0.2911888659000397, 2.8634047508239746, 2.394657611846924, -0.9841603636741638, -0.4562141001224518, -1.0194668769836426, 2.2121171951293945, 2.072885274887085, 0.3999601900577545, 2.6030802726745605, -1.2091773748397827, -0.5859724283218384, -2.7969565391540527, -2.618419885635376, 1.751039981842041, 2.5842087268829346, 0.24031542241573334, -2.787416696548462, -0.2289191335439682, -0.991515040397644, 0.05333740636706352, -1.4802592992782593, -3.02400541305542, 5.64232063293457, -4.299834728240967, -1.098942756652832, 1.861265778541565, -0.08966104686260223, 4.203497886657715, 5.340852737426758, 0.978808581829071, -1.6408650875091553, -1.8956818580627441, 5.8673553466796875, -0.29528364539146423, -0.08790440112352371, 0.5405710339546204, 1.3718476295471191, -0.1284189671278, 0.9171457886695862, -0.08202438056468964, 1.9704384803771973, -0.9738471508026123, -0.6617324352264404, 1.8453088998794556, 1.9107856750488281, -1.3253586292266846, -0.6293069124221802, -1.2126165628433228, 5.506312847137451, 3.087155342102051, -1.9558436870574951, 4.119218826293945, -3.203226327896118, 4.0706963539123535, 0.4101143479347229, -0.7945507764816284, -1.2203413248062134, 0.1784529685974121, -1.503242015838623, -3.223814010620117, 0.5438345670700073, -0.0009169633849523962, -1.4799673557281494, 0.21817414462566376, -2.010586977005005, -3.6837286949157715], [0.49654704332351685, -4.72376012802124, -1.929669976234436, -0.9801127314567566, -2.2952842712402344, 0.9429965019226074, 0.6112266182899475, -2.7436604499816895, 3.4359121322631836, 0.06637657433748245, -3.4161932468414307, -1.0283044576644897, -4.291253566741943, -0.8580560684204102, -3.9262585639953613, 0.9234738349914551, 0.9392481446266174, 2.318711757659912, -1.0868459939956665, -2.722714424133301, -1.46934175491333, 1.7643898725509644, 4.1999077796936035, 0.2613219618797302, -0.5395817160606384, 2.232060670852661, -1.0612446069717407, 1.4708999395370483, -1.2376352548599243, 2.623279094696045, -0.07879342883825302, -4.843948841094971, -0.6544165015220642, 4.541243076324463, -0.6971678137779236, -2.0971274375915527, -4.227978706359863, 7.301784992218018, 0.5460315942764282, 2.2953879833221436, -2.072791814804077, 8.494266510009766, 0.7158099412918091, 2.741807699203491, -1.5104650259017944, 0.8461261987686157, 0.25991398096084595, 0.8003554344177246, 0.2880876064300537, 1.5585222244262695, -2.5660014152526855, 2.3342857360839844, 0.9868430495262146, 2.8226425647735596, -3.7133779525756836, 2.8654184341430664, 1.823727011680603, 1.3293622732162476, -3.0414175987243652, 2.016735076904297, -1.315873146057129, -3.0300168991088867, 2.0517513751983643, 6.527082443237305, 2.3162388801574707, 1.9822380542755127, 5.03042459487915, -0.23750366270542145, 0.12677961587905884, 3.973952531814575], [-1.9484608173370361, -0.8019874095916748, 1.7446751594543457, -0.3760940432548523, -0.03625275567173958, 0.6078962087631226, -0.9384321570396423, -1.1041606664657593, 0.5328835248947144, 1.5358924865722656, 0.3521391749382019, -0.7927937507629395, 1.2273635864257812, -0.41699737310409546, -0.4843922555446625, 1.597360610961914, -1.8564379215240479, 2.104999303817749, 1.6685198545455933, -0.22758309543132782, -1.2852692604064941, 0.1515597701072693, 0.5947667956352234, 0.42114558815956116, -0.37020325660705566, -2.60455060005188, 3.2090137004852295, -1.0932341814041138, -0.21024289727210999, 0.6382868885993958, 0.3053849935531616, 2.2065701484680176, 3.156651258468628, 1.827269434928894, -1.6408090591430664, -0.5180323123931885, 2.5601770877838135, 1.8946266174316406, -0.675845205783844, 0.43651875853538513, 0.07510804384946823, 0.669579803943634, 0.6027222275733948, 0.8540825247764587, 0.7913367748260498, -0.17595158517360687, -0.08689093589782715, 1.0260798931121826, 1.825132131576538, -0.7707796096801758, 0.3107949197292328, -0.9797713756561279, 2.7499189376831055, 1.4258720874786377, -1.4047702550888062, 0.5154756903648376, -0.47635823488235474, 4.132925510406494, 0.006922435015439987, -0.5278655886650085, -0.7036818861961365, -0.6209268569946289, -0.186289444565773, 1.4299808740615845, 0.9491649270057678, 1.3746426105499268, -1.8292770385742188, 2.6703808307647705, 0.5728574395179749, -1.5605403184890747], [1.5688992738723755, 5.666266441345215, 1.5439692735671997, 1.199158787727356, 4.38867712020874, 4.14707612991333, 1.4503623247146606, 3.056072235107422, -4.665282249450684, -0.5039231181144714, 2.1059927940368652, 3.4327380657196045, 1.0054084062576294, -2.5337347984313965, -0.21758319437503815, -11.813760757446289, -1.5325299501419067, -0.58498215675354, 1.1094235181808472, -1.5317795276641846, -0.31090623140335083, 0.26115837693214417, -2.908224582672119, -0.8584672808647156, -3.7348318099975586, 1.9458459615707397, 3.968545913696289, -3.116130828857422, -0.47049957513809204, 2.0777313709259033, 0.6078609824180603, 1.5201456546783447, 1.9192860126495361, -2.3904507160186768, 1.4835304021835327, -4.6246819496154785, 1.2142888307571411, -5.8941545486450195, 1.3692083358764648, 0.13313080370426178, 1.9683794975280762, -3.5871667861938477, 4.0734405517578125, -1.5380661487579346, 1.4552541971206665, -4.07039213180542, -3.3538637161254883, 2.035987138748169, -0.844231367111206, -1.0049481391906738, -6.574927806854248, 0.9874805808067322, 3.96037220954895, 2.046138048171997, -1.4886915683746338, 12.403627395629883, -3.517655372619629, 0.250023752450943, -2.8283324241638184, -0.7404634356498718, -3.6688053607940674, 1.6672052145004272, -3.7051398754119873, -7.99427604675293, -1.8623274564743042, -5.117945194244385, 3.1154589653015137, -7.127703666687012, -5.837069988250732, 1.3292721509933472], [-4.959362030029297, -4.903979778289795, -3.466110944747925, -1.6121076345443726, -5.992508888244629, -3.1253836154937744, 0.11426111310720444, -5.561224460601807, 5.140678405761719, -1.1516387462615967, -3.413149833679199, 0.09445908665657043, -4.058760643005371, 0.4227365255355835, -3.625094413757324, 3.9293925762176514, 5.238306999206543, 2.9133403301239014, -3.331387519836426, -0.44605550169944763, -1.0787748098373413, -0.07042184472084045, 5.150219917297363, -0.4496738314628601, 0.8204507827758789, -0.4012048840522766, -2.2816851139068604, 2.4862382411956787, -0.5241648554801941, 2.7032699584960938, 1.1250722408294678, -3.2706663608551025, -6.423382759094238, 0.8506203889846802, -0.24053679406642914, 1.2932711839675903, -2.9847466945648193, 5.007822513580322, -1.1162261962890625, -0.42225193977355957, -3.587733507156372, 10.284036636352539, 1.7586870193481445, 1.70928955078125, -1.5149813890457153, 0.4919811189174652, 3.056607961654663, -2.489523410797119, -0.7118565440177917, 1.9585453271865845, 2.4647319316864014, 0.7603316903114319, -3.2648367881774902, -1.921211838722229, 0.13668668270111084, -5.82285737991333, 0.6135780811309814, -4.026943683624268, 0.34036287665367126, 1.7750062942504883, 3.3625333309173584, -2.628993034362793, 2.8314454555511475, 5.731107711791992, -1.0116159915924072, 1.723917007446289, 3.894099473953247, -0.2757072150707245, -0.6167303919792175, 0.8453553915023804], [-2.133049726486206, 1.6442092657089233, 2.084177255630493, 0.686170220375061, 1.67002272605896, 2.245258092880249, -1.2463303804397583, 1.6770613193511963, -0.22411859035491943, -1.2078691720962524, 1.5346955060958862, 3.1726441383361816, -2.37021541595459, -1.1615188121795654, -1.1308602094650269, -7.801220417022705, 0.26338502764701843, -0.6756263375282288, -3.407099723815918, 0.4503489136695862, -2.8264248371124268, 0.6990025639533997, -0.008400656282901764, -1.8555349111557007, -1.207064151763916, 2.4808566570281982, 0.3172173798084259, -1.840531587600708, -2.4399709701538086, 4.261730194091797, 1.2357604503631592, -1.4381897449493408, -2.125061511993408, -3.7240123748779297, 3.485318422317505, -4.027527809143066, 0.6413286924362183, -3.194775104522705, 2.051215648651123, -1.7939859628677368, 1.5667273998260498, 0.5021791458129883, 2.4865214824676514, -0.8065721392631531, -1.152833342552185, -0.8936112523078918, -1.0686769485473633, -1.1455284357070923, -1.5894502401351929, 1.6671881675720215, -3.992126226425171, 1.258164644241333, 0.6600002646446228, 1.9520000219345093, -1.9971579313278198, 7.085752010345459, -1.4750683307647705, -3.0414323806762695, -3.675997734069824, -0.8451266884803772, -0.8450391292572021, 1.835741400718689, -1.0999162197113037, -6.7519683837890625, -3.267387866973877, -5.583102703094482, 7.050273895263672, -5.814732551574707, -4.352297782897949, 2.014855146408081], [-2.3413937091827393, -2.6108386516571045, -1.165790319442749, -1.0907317399978638, -0.15877962112426758, 0.428802490234375, 2.2425291538238525, -5.476743698120117, 1.3001586198806763, 0.9859889149665833, -0.7816454172134399, -1.1844738721847534, 0.18552975356578827, 0.14606495201587677, -3.2081940174102783, 0.5272583961486816, 2.079279661178589, 2.7274200916290283, 1.5529677867889404, -1.3303680419921875, -2.268872022628784, -1.2639544010162354, 1.3023205995559692, 0.6092327237129211, -0.1848975569009781, -1.8931251764297485, 1.5113226175308228, -3.945101737976074, 0.3210514485836029, 0.7133504152297974, -2.5153348445892334, 0.01726165972650051, 1.956406593322754, 4.790303707122803, -3.3743984699249268, 0.6788731217384338, 2.864978313446045, 4.7686357498168945, -1.5455466508865356, 2.3783135414123535, -2.1886701583862305, 8.40063190460205, -1.728124737739563, 0.9834319353103638, 2.0583012104034424, 0.935666561126709, 2.5308167934417725, 1.6797996759414673, 0.8920872211456299, -0.3155517578125, 3.305454969406128, -0.6501075625419617, 3.51798677444458, 1.3565574884414673, -0.2968297004699707, -2.5729057788848877, -2.4540233612060547, -1.2341620922088623, 3.8284387588500977, 2.9957547187805176, 1.803013563156128, -2.080610513687134, 0.17953592538833618, 3.0155529975891113, 2.568551540374756, 5.422304153442383, -0.6802463531494141, -1.027935266494751, -2.0324628353118896, -3.757479190826416], [0.9306954741477966, -2.634294271469116, -1.7912909984588623, -2.46687388420105, 0.11530007421970367, -1.1600425243377686, 0.8906656503677368, -0.8051941990852356, 3.2884082794189453, 0.1032140776515007, -3.9983725547790527, -0.04345373436808586, -3.609433174133301, -0.42148277163505554, -2.438936233520508, 0.42449724674224854, 4.6251654624938965, -3.069674253463745, -1.1811935901641846, 0.524085521697998, 0.26922351121902466, 1.0282896757125854, 2.3659861087799072, 0.15550123155117035, 0.2640462815761566, 4.61602258682251, -4.947311878204346, 1.9871786832809448, -2.775045156478882, 1.635915756225586, -1.007617712020874, -4.794774055480957, -3.6766812801361084, 0.9962414503097534, 0.21479958295822144, 1.4823579788208008, -5.016909122467041, 4.1973090171813965, 2.4324655532836914, 2.7370238304138184, -1.1270735263824463, 7.664453983306885, -2.3535125255584717, 0.3367125988006592, -1.843299388885498, 1.6791740655899048, 1.236224889755249, -1.1552256345748901, -2.045071840286255, 1.7264434099197388, -0.435711532831192, 3.9458513259887695, -2.949824571609497, 1.4934390783309937, -0.5768163800239563, -1.226681113243103, -0.7020358443260193, -4.924884796142578, 0.4478313624858856, 2.658798933029175, 1.604840874671936, -3.4359560012817383, 4.405721187591553, 1.9276747703552246, 2.2215800285339355, 0.23823386430740356, 4.531089782714844, -2.7155392169952393, -0.007615175563842058, 0.8789385557174683], [-2.386836290359497, -0.9342492818832397, -1.6478493213653564, -2.978239059448242, 0.5017381310462952, -0.6548018455505371, 2.5403659343719482, -3.984461545944214, 1.7917729616165161, 1.1442817449569702, -3.2167699337005615, 0.6579155921936035, -0.717309296131134, -1.3709917068481445, -3.696931838989258, -1.927561640739441, 5.264842987060547, -0.13405130803585052, 1.831025242805481, -0.34127774834632874, -0.39597269892692566, -0.5080319046974182, 1.7447205781936646, 0.5197432637214661, -1.634115219116211, 0.8065205812454224, 0.246782124042511, -1.427930474281311, -1.6718921661376953, 2.1469414234161377, -1.2643728256225586, -0.5351161956787109, -1.3281409740447998, 2.242422580718994, -2.632333755493164, 1.5587040185928345, -0.6522154808044434, 3.4048478603363037, 0.4737694263458252, 3.4969725608825684, -2.0860228538513184, 9.881854057312012, -0.06495106965303421, 0.04327519237995148, 1.214880347251892, -0.9383412599563599, 1.7382758855819702, 0.5530927181243896, -0.7939477562904358, -0.25217926502227783, 0.7657771110534668, 2.287302255630493, 0.9203360080718994, 1.1851211786270142, 0.12586095929145813, -0.5165161490440369, -4.7529377937316895, -3.7884390354156494, 3.427184820175171, 3.2463252544403076, 2.031885862350464, -4.165857315063477, 2.496203899383545, 0.890501081943512, 2.2479488849639893, 2.0616090297698975, 1.50483238697052, -3.9839282035827637, -3.323495864868164, -3.4558162689208984], [1.0616967678070068, 1.2961901426315308, -3.682481050491333, -3.6655049324035645, -1.0760612487792969, -0.9269617199897766, 3.327662706375122, -1.2501388788223267, 0.5297054648399353, 1.239209532737732, -6.469082355499268, 1.5932729244232178, -0.8885502815246582, -3.8781516551971436, -3.3819258213043213, -2.9935553073883057, 4.729095458984375, -0.06144833564758301, 4.302474498748779, -2.5451929569244385, 4.277001857757568, 1.4775974750518799, 2.6228418350219727, 1.3499714136123657, -4.646170139312744, 2.386509895324707, 1.9836320877075195, 4.569688320159912, -1.1127822399139404, 1.8763818740844727, 1.9588689804077148, -0.28907689452171326, -2.4001729488372803, 1.7120081186294556, -2.8059933185577393, 0.67019122838974, -7.27345085144043, 2.5378527641296387, 1.0596200227737427, 4.852522850036621, -2.569547653198242, 6.915280818939209, 5.7077202796936035, 0.6998298764228821, 0.5245634317398071, -5.6348748207092285, -1.9267274141311646, 1.48255455493927, -0.6485532522201538, -1.7033411264419556, -5.801258563995361, 4.701549530029297, -0.4792136549949646, 0.24907436966896057, -0.8760421276092529, 6.138373851776123, -3.935171127319336, 1.318681001663208, -0.7014623284339905, 2.005234956741333, -1.9737569093704224, -6.554342269897461, 3.011254072189331, 3.89339542388916, 2.4034032821655273, -1.1081724166870117, 1.1805583238601685, -3.3085784912109375, -2.4708166122436523, 1.4105473756790161], [-0.2993059754371643, 1.7985179424285889, -1.021301507949829, -1.1317239999771118, 0.9713246822357178, 0.04496665298938751, 1.409652590751648, 0.16050483286380768, -0.3120881915092468, -0.4689342975616455, -1.4783467054367065, 2.1931934356689453, -1.053661584854126, -1.4202203750610352, -1.3836318254470825, -4.962083339691162, 3.296473264694214, -1.6241703033447266, -0.15159307420253754, -0.028152167797088623, 0.7134788632392883, 0.11249923706054688, 0.10850515961647034, -0.40119558572769165, -1.775731086730957, 2.4872868061065674, -0.5425822138786316, 0.2164110243320465, -1.3068898916244507, 1.8294267654418945, 0.3857586979866028, -0.8952115774154663, -2.8536832332611084, -1.7701747417449951, 0.6367154717445374, -0.2994581162929535, -2.1019985675811768, -1.5998508930206299, 1.343648910522461, 0.9935276508331299, -0.30350103974342346, 2.8648977279663086, 1.8061219453811646, -0.9359197616577148, -0.011646864004433155, -1.9895472526550293, -0.39879024028778076, -0.4076501429080963, -1.776140809059143, 0.17023199796676636, -2.3637900352478027, 2.274074077606201, -0.8971536159515381, 0.07068070769309998, 0.3059915006160736, 3.277533531188965, -2.9087581634521484, -3.7940175533294678, -0.020315395668148994, 1.0055280923843384, 0.2561587691307068, -1.3135757446289062, 0.7436307072639465, -2.949284553527832, -0.6861687898635864, -2.5769765377044678, 3.1603379249572754, -5.132208347320557, -3.3513622283935547, 0.11425089836120605], [0.29508838057518005, 2.3771605491638184, -4.164658069610596, -1.893758773803711, -1.2495598793029785, 0.6731559038162231, 4.467297077178955, -2.682100296020508, -1.4621731042861938, 0.122726209461689, -4.20485782623291, 2.3486456871032715, -0.2527303993701935, -3.7383816242218018, -3.828601121902466, -6.537695407867432, 3.7005856037139893, 2.645256519317627, 3.7941977977752686, -4.135956764221191, 2.965139389038086, 0.4177054762840271, 1.7117406129837036, 0.7934751510620117, -5.304497241973877, 0.7892612814903259, 4.400920391082764, 1.577683448791504, 0.7097454071044922, 2.069716215133667, 1.6064542531967163, 0.6051645278930664, -1.480555772781372, 1.954847812652588, -2.7273733615875244, -1.4308184385299683, -4.493141174316406, 0.6251195073127747, -0.6066516637802124, 3.570518970489502, -2.9032108783721924, 6.284189224243164, 7.372787952423096, 0.625129759311676, 1.9690163135528564, -6.7266411781311035, -1.9399439096450806, 2.5122768878936768, -0.2856622040271759, -2.0213325023651123, -5.848128318786621, 2.738116502761841, 2.144637107849121, -0.3152179718017578, -0.7842947840690613, 8.338371276855469, -4.3560471534729, 1.103474497795105, -0.7942926287651062, 1.9974005222320557, -2.498525381088257, -4.5041375160217285, -0.39728599786758423, 2.3249361515045166, 0.7687706351280212, -0.6965388655662537, 1.4399851560592651, -5.487831115722656, -5.284440517425537, 1.458462119102478], [3.2321956157684326, 0.9797762036323547, -2.1923773288726807, -2.172670841217041, 0.8215630650520325, -3.2880749702453613, 0.6461797952651978, 3.1221649646759033, 1.166756272315979, -0.7261814475059509, -3.496391534805298, 0.8480278849601746, -1.821032166481018, 0.13272570073604584, 1.2141191959381104, 0.926505982875824, 5.2945556640625, -7.0253400802612305, -1.0681734085083008, 2.699378490447998, 4.049480438232422, 0.5860486030578613, -0.28055793046951294, 0.12789736688137054, 0.8323787450790405, 5.648543834686279, -7.335795879364014, 4.878384113311768, -2.0008912086486816, -1.2102171182632446, -0.12550796568393707, -3.3769736289978027, -6.276190757751465, -3.6784098148345947, 1.8812010288238525, 4.169416427612305, -6.786048889160156, -1.5514930486679077, 3.0577287673950195, 1.30704665184021, 0.17898185551166534, 0.2306182086467743, -2.66642689704895, -1.8396581411361694, -2.2887372970581055, 0.7797192931175232, 0.3422849178314209, -2.8263001441955566, -3.517885208129883, 1.0335097312927246, -0.0054170070216059685, 4.054225444793701, -7.4405035972595215, -1.9511858224868774, 2.9177815914154053, -3.831012487411499, -0.5008644461631775, -7.153700351715088, 1.5028760433197021, 1.1199597120285034, 2.357473373413086, -2.0498335361480713, 4.57614803314209, -1.5257577896118164, 0.4946126639842987, -2.776299238204956, 1.7159745693206787, -2.518976926803589, 1.5400185585021973, 0.71059250831604], [3.9110677242279053, -3.7962422370910645, -2.9532885551452637, 0.05232996493577957, -2.294996500015259, -5.091193675994873, -0.9890896677970886, 1.615042805671692, 3.14229416847229, -2.008457899093628, -1.3596957921981812, -2.8146300315856934, -2.49015212059021, 4.296380043029785, 3.254519462585449, 10.336501121520996, 3.1362524032592773, -5.812113285064697, -4.423157215118408, 3.570554256439209, 2.6757302284240723, -0.42510345578193665, 0.1037357822060585, 0.1018589660525322, 5.924959182739258, 3.6090638637542725, -11.704902648925781, 4.865713596343994, 0.4273964762687683, -4.7853312492370605, -2.0373897552490234, -5.577591419219971, -6.258702278137207, -1.8589065074920654, 2.491539716720581, 6.294582366943359, -5.026631832122803, 1.2816011905670166, 0.8360316753387451, -1.4249134063720703, -0.34916847944259644, -1.8551579713821411, -8.400686264038086, -0.7437658309936523, -3.4953417778015137, 6.532471656799316, 3.3274497985839844, -4.289942741394043, -2.6287920475006104, 2.665741205215454, 7.03587007522583, 0.5198851823806763, -9.776726722717285, -4.11037540435791, 4.466804504394531, -14.859841346740723, 5.633954048156738, -7.77454137802124, 3.103375196456909, 0.9212342500686646, 5.036005020141602, 0.9406898021697998, 4.309525966644287, 3.54656982421875, 0.6044059991836548, 2.253002643585205, -0.9991060495376587, 3.899991273880005, 7.04473352432251, 1.1157523393630981], [-6.820255756378174, -4.874245643615723, 3.35611891746521, 0.1576707363128662, -2.6950929164886475, -0.7879828214645386, -4.334194183349609, -2.8639025688171387, 4.512331962585449, 0.654721736907959, 1.7863693237304688, -1.282004952430725, -1.3837034702301025, 1.7142630815505981, -0.8203393816947937, 5.846922874450684, -1.5545094013214111, 3.1520192623138428, -3.4811408519744873, 2.0196917057037354, -5.2210540771484375, -0.032033488154411316, 2.8375751972198486, -0.918541669845581, 2.888618230819702, -3.7567408084869385, 0.5447670221328735, -1.6729971170425415, -1.5365828275680542, 2.617697238922119, 0.6381059288978577, 0.3019310534000397, 0.3411453664302826, 0.3243628144264221, 0.5896638631820679, -0.4914313852787018, 4.897810459136963, 3.7648918628692627, -0.8404189348220825, -2.855166435241699, 0.13316430151462555, 3.204538106918335, -1.488959789276123, 1.5144014358520508, -1.353380799293518, 3.744236707687378, 2.6464953422546387, -2.1355721950531006, 1.6991407871246338, 1.9442322254180908, 4.359506130218506, -2.4778826236724854, 0.6257882714271545, 1.2171871662139893, -1.8091181516647339, -5.781210899353027, 2.014077663421631, 1.3194071054458618, -0.5315253734588623, -1.2672405242919922, 2.437819480895996, 1.2846291065216064, 0.9486204981803894, 1.6247283220291138, -1.3300915956497192, 1.4329636096954346, 1.3048007488250732, 4.7995147705078125, 1.981536626815796, -1.7682689428329468], [2.6468143463134766, 4.277623176574707, 2.5241265296936035, -0.42646634578704834, 4.783604621887207, 1.7812329530715942, -0.1928044706583023, 4.768529891967773, -2.6782584190368652, 0.9692179560661316, 0.520962119102478, 1.6213279962539673, 1.4822962284088135, -1.6757328510284424, 1.5107601881027222, -5.411288738250732, -1.1577057838439941, -3.9560158252716064, 1.9850993156433105, 1.0518208742141724, 1.2559661865234375, 0.8576295375823975, -2.7588911056518555, -0.0029725737404078245, -1.8179404735565186, 2.5993499755859375, 0.8057224750518799, -0.26349613070487976, -1.8202378749847412, 0.27727827429771423, 0.2862056493759155, 1.422489047050476, 1.88945734500885, -2.515613317489624, 1.0240432024002075, -0.9796107411384583, -0.3732515573501587, -4.019818305969238, 2.658179521560669, 1.1145879030227661, 2.6579344272613525, -4.803755760192871, 0.2910812497138977, -1.7182457447052002, 0.1622784584760666, -1.628039836883545, -2.689445734024048, 0.7706208825111389, -0.8352470993995667, -0.9139388203620911, -4.163808345794678, 1.9946485757827759, 0.7589044570922852, 2.0034637451171875, -0.5460008382797241, 6.651453971862793, -2.4052960872650146, 0.6904135942459106, -1.1790225505828857, -0.9862749576568604, -2.2480735778808594, 0.16688166558742523, -0.06410670280456543, -6.027834892272949, 0.2962953746318817, -4.20374870300293, 0.3667997121810913, -2.689446449279785, -1.157434344291687, -0.14760713279247284], [0.9504464864730835, 2.121150016784668, -0.061444927006959915, -0.8378850817680359, 2.500419855117798, -0.6498920321464539, 1.261202335357666, 0.7801162004470825, -1.3018776178359985, 0.3961329758167267, 0.0014134980738162994, 0.42492616176605225, 1.6411893367767334, 0.07082582265138626, 0.8667444586753845, -1.3335378170013428, 1.9657424688339233, -3.010967969894409, 1.4749228954315186, 1.411840558052063, 1.2985665798187256, -0.9332133531570435, -2.0371339321136475, 0.29697003960609436, -0.07426436245441437, 0.8894516229629517, -1.174329161643982, -1.0435501337051392, -0.27984774112701416, -1.309891700744629, -1.36940598487854, 1.0470417737960815, -0.16785182058811188, -1.1150429248809814, -0.5334701538085938, 2.2648003101348877, 0.6150164604187012, -2.15987229347229, 0.6858879327774048, 1.0957942008972168, 0.4973730742931366, -0.6875806450843811, -2.129380226135254, -1.6744223833084106, 0.9767495393753052, -0.004336068872362375, 0.47462183237075806, -0.03330221399664879, -1.0784776210784912, -0.7287241220474243, 1.497832179069519, 0.599626898765564, -0.8390283584594727, -0.5910302996635437, 2.210801601409912, -1.7925989627838135, -2.5841281414031982, -3.309891939163208, 3.321784257888794, 0.8862361311912537, 1.4529244899749756, -0.37017685174942017, 0.6007007360458374, -3.086350202560425, 0.7447359561920166, 0.05689186975359917, -1.8153167963027954, -2.0263209342956543, -0.7939420938491821, -3.066242218017578], [-2.674581527709961, 1.8684148788452148, 2.401279926300049, -1.7553983926773071, -1.133592128753662, -4.08496618270874, -3.4704864025115967, 2.9895644187927246, 0.9647966027259827, 1.7005841732025146, -0.6439198851585388, 0.07851386070251465, 2.9627325534820557, 0.020668525248765945, 3.6216554641723633, 6.46536111831665, -0.05370048061013222, -2.4816102981567383, 1.6286158561706543, 4.0807108879089355, 3.090904712677002, 0.2682284712791443, -0.4143539369106293, 0.5006013512611389, 0.798631489276886, -2.7016522884368896, 0.5069353580474854, 4.8261027336120605, -0.9229646921157837, -1.4114258289337158, 3.119663715362549, 4.872713088989258, -1.4328792095184326, -4.6486358642578125, 0.030296754091978073, 4.254166603088379, -0.008351603522896767, -3.1059470176696777, 0.45504629611968994, -1.5555933713912964, 1.3820664882659912, -6.289356708526611, 1.042826771736145, -1.215621829032898, -0.8504018783569336, -1.0828213691711426, -0.49777546525001526, -2.6549129486083984, 0.5972014665603638, -1.328296422958374, 1.820292353630066, -0.36810898780822754, -4.4832987785339355, -2.8880982398986816, 2.4558515548706055, -5.784765243530273, -0.3292866349220276, 2.7714715003967285, 1.14719557762146, -3.2350049018859863, 1.4369217157363892, -0.6910588145256042, 2.742581605911255, -0.9098126292228699, -1.2648128271102905, -2.8453338146209717, -5.538565158843994, 6.060067176818848, 4.298949718475342, -2.850733518600464], [-3.583723306655884, 3.0534703731536865, 0.22200514376163483, -3.8348071575164795, -0.20853057503700256, -1.819698452949524, -0.40974992513656616, 1.2928595542907715, 1.4340307712554932, 1.1394928693771362, -4.398988723754883, 3.9075260162353516, -0.9233308434486389, -3.8018124103546143, -1.8254109621047974, -4.4708571434021, 4.9644646644592285, -2.2304203510284424, 1.4135024547576904, 1.3080940246582031, 2.468764066696167, 1.5536412000656128, 2.060293436050415, -0.19612154364585876, -3.8913092613220215, 2.0695905685424805, 1.7621448040008545, 4.451837062835693, -3.760340452194214, 4.40336275100708, 4.065534591674805, 1.5450061559677124, -5.004380226135254, -4.47128438949585, 0.3805295526981354, 0.4729340076446533, -4.620619773864746, -2.0305192470550537, 2.7557098865509033, 1.5575467348098755, -0.1921486109495163, 3.7321152687072754, 6.513849258422852, -0.9752713441848755, -0.9115586280822754, -5.388731956481934, -1.877349615097046, -1.7201544046401978, -1.6389505863189697, -0.6106250286102295, -5.544700622558594, 4.515397548675537, -2.751405954360962, -0.020162014290690422, -0.44517290592193604, 5.5561089515686035, -5.232758522033691, -1.1232924461364746, -1.7519950866699219, -0.5773746967315674, -0.31816452741622925, -4.422118663787842, 3.6494314670562744, -3.179457902908325, -1.439276099205017, -6.613635063171387, 3.4423186779022217, -4.076979160308838, -3.265768051147461, -0.16368290781974792], [-1.9045318365097046, -0.7988870143890381, -2.5117287635803223, -3.1586759090423584, -5.395738124847412, -5.225203037261963, -1.1383740901947021, -0.0538611002266407, 3.329085111618042, 0.7543323636054993, -5.530055999755859, 0.2432306855916977, -0.6888980269432068, -1.3164703845977783, 0.12313596159219742, 6.94127893447876, 3.6273176670074463, -0.32875901460647583, 1.2491225004196167, 0.9562016725540161, 4.982406139373779, 1.4132992029190063, 3.654630184173584, 0.9671201109886169, -0.8268853425979614, -0.5711385607719421, -0.6978718638420105, 8.995941162109375, -0.6195371747016907, 0.06663662940263748, 4.317011833190918, 0.9219238758087158, -5.647160053253174, -2.03667950630188, -0.9055290222167969, 3.9474375247955322, -6.968886375427246, 1.5481442213058472, 0.09950929880142212, 0.541218638420105, -2.018920660018921, 1.450649619102478, 4.647356033325195, 0.7591351866722107, -1.8767616748809814, -3.147479772567749, -0.4963766932487488, -2.5092337131500244, -0.06471002846956253, -0.7236690521240234, -1.0310724973678589, 2.3005566596984863, -6.2582831382751465, -3.674858808517456, 1.3462121486663818, -4.176210403442383, 0.5133095979690552, 2.244749069213867, -0.8253703117370605, -1.1474335193634033, 0.7742660045623779, -4.551992893218994, 4.716193199157715, 6.0361433029174805, -0.31131866574287415, -1.8695170879364014, -1.9925843477249146, 4.460934162139893, 3.4971158504486084, 1.396299123764038], [3.3798108100891113, -2.1084976196289062, -10.743279457092285, 1.805993914604187, -7.225159168243408, 0.05094346031546593, 7.459138870239258, -7.449380874633789, -1.475949764251709, -3.8800487518310547, -3.289447069168091, -0.28549134731292725, -2.7764949798583984, -0.3217522203922272, -4.465186595916748, -2.1423516273498535, 3.5178115367889404, 7.67415714263916, -0.03386303037405014, -8.211013793945312, 3.0075619220733643, -1.1998432874679565, 3.0612215995788574, 0.6798214316368103, -2.4403891563415527, -0.17585411667823792, 0.8018956184387207, 1.5272916555404663, 6.501711845397949, -1.600257158279419, -0.5277850031852722, -4.364369869232178, -3.936891555786133, 6.153386116027832, -2.643826961517334, -1.6921120882034302, -6.267254829406738, 3.661325216293335, -5.0808515548706055, 1.4512754678726196, -6.566975116729736, 8.043548583984375, 5.856738567352295, 2.6875391006469727, 1.9758517742156982, -4.182982921600342, 0.6206443905830383, 2.8998172283172607, -0.15660539269447327, -0.5927942991256714, -1.524504542350769, -0.7141971588134766, 0.9568870663642883, -4.521064758300781, 0.9718945622444153, 1.666407585144043, 2.1548397541046143, -1.3799102306365967, -0.3698123097419739, 3.971595525741577, -1.332468032836914, -1.8194525241851807, -3.755005121231079, 10.720643043518066, 0.033287595957517624, 5.781040191650391, 1.2989747524261475, -4.196236610412598, -3.9148483276367188, 6.021024703979492], [-5.197081565856934, -0.6212322115898132, 4.383181095123291, -0.20848122239112854, 1.3684502840042114, -0.7262036204338074, -3.2539234161376953, -0.2964593470096588, 1.519893765449524, 1.1557382345199585, 2.9471473693847656, -0.19303342700004578, 1.508579969406128, 1.4488348960876465, 1.1884137392044067, 2.4723217487335205, -0.7235792875289917, -0.7081104516983032, -1.3311330080032349, 3.803213119506836, -3.506540060043335, -1.040960669517517, -0.7753959894180298, -0.740286648273468, 2.2701356410980225, -2.894374370574951, 0.3946237862110138, -3.289731740951538, -1.7461605072021484, 0.9453762173652649, -0.4868575632572174, 2.7749180793762207, 1.213725209236145, -2.168078899383545, 0.5361137986183167, 1.2715448141098022, 6.490146636962891, -1.0213258266448975, 0.1892361044883728, -2.164155960083008, 1.760748267173767, -0.8754813075065613, -3.1425764560699463, -0.9738823175430298, 0.06255071610212326, 2.8968117237091064, 2.087883234024048, -1.8463938236236572, 0.5825310349464417, 0.6138443350791931, 4.806673049926758, -2.1208043098449707, 0.36827215552330017, 0.6882027983665466, 0.651251494884491, -5.466036319732666, -1.2555499076843262, -1.3350188732147217, 2.5032641887664795, -1.2502496242523193, 3.051806688308716, 1.9111065864562988, 0.5105092525482178, -4.240690231323242, -1.1987055540084839, -0.07173516601324081, -1.295918345451355, 2.158571481704712, 0.6687862277030945, -5.149606704711914], [-0.9626973271369934, -2.716416835784912, -3.0554773807525635, -1.3091481924057007, -1.4655036926269531, 1.662630558013916, 2.918281316757202, -4.6477952003479, 1.9846773147583008, -0.3403274416923523, -3.3532490730285645, 0.7890592217445374, -3.7471585273742676, -1.9008843898773193, -5.579598903656006, -4.398644924163818, 3.3671391010284424, 3.12644100189209, -0.2797147333621979, -3.6521718502044678, -1.7254036664962769, 0.7021607160568237, 3.6362297534942627, -0.019118495285511017, -2.4633402824401855, 2.079442262649536, 0.8895418047904968, -1.0184727907180786, -0.9293566942214966, 3.8463101387023926, -0.5042670369148254, -3.862302780151367, -1.4302525520324707, 4.236010551452637, -1.4511500597000122, -2.7187557220458984, -3.0010931491851807, 5.524091720581055, 0.05106613412499428, 2.9878923892974854, -2.9229307174682617, 11.5341157913208, 2.7309815883636475, 1.9368984699249268, 0.23898400366306305, -1.4914758205413818, 0.5071656703948975, 1.5785044431686401, -0.44441476464271545, 0.8916098475456238, -3.1736574172973633, 2.531472682952881, 2.6928863525390625, 2.4881746768951416, -2.955153703689575, 5.41337776184082, -1.821763277053833, -1.5615906715393066, -1.4207701683044434, 3.300835132598877, -0.8285049200057983, -3.495844602584839, 0.6913004517555237, 3.78696870803833, 1.5581144094467163, 1.6537787914276123, 6.039320468902588, -5.163432598114014, -4.383458137512207, 2.188868522644043], [-5.130045413970947, 0.7272536754608154, -1.0947699546813965, -0.924572765827179, -5.048566818237305, -6.165243625640869, -2.3979861736297607, -0.10644238442182541, 2.0306828022003174, -1.1295889616012573, -0.6271767616271973, 1.1808184385299683, 0.8033102750778198, 1.3613232374191284, 2.642906665802002, 6.440876007080078, 3.7254819869995117, -0.9338134527206421, -2.309830904006958, 3.8351118564605713, 2.861855983734131, -1.1160296201705933, 0.9328300952911377, -0.570405900478363, 1.8858329057693481, -2.8471412658691406, -2.0935304164886475, 5.23077392578125, 0.6268832683563232, -1.0352946519851685, 3.4003114700317383, 2.715508222579956, -7.532089710235596, -6.559284687042236, 1.5124555826187134, 4.991209983825684, -0.9630786180496216, -4.241816997528076, -1.205905795097351, -4.009134769439697, -0.7567071318626404, -3.0456578731536865, 2.1956653594970703, -1.4150980710983276, -1.3397908210754395, -1.0314843654632568, 1.7892465591430664, -4.924843788146973, -1.036067008972168, 0.11679936200380325, 4.562445640563965, -1.3598382472991943, -7.781856060028076, -6.730598449707031, 4.879967212677002, -10.625978469848633, 0.38559678196907043, -3.569516897201538, 2.1844687461853027, -2.3435723781585693, 4.514251232147217, 0.4845708906650543, 2.0583808422088623, -0.42819902300834656, -4.460443019866943, -2.683642864227295, -3.028780221939087, 3.2895541191101074, 2.273721933364868, -2.0064234733581543], [-3.7774038314819336, -7.195644855499268, -2.986973762512207, -0.06910431385040283, -6.758174419403076, -1.4583762884140015, -0.8315115571022034, -5.542902946472168, 5.702159881591797, -1.6121516227722168, -2.4156291484832764, -1.1293870210647583, -5.570030689239502, 1.0689150094985962, -3.9161767959594727, 4.97721004486084, 2.183607339859009, 4.833950996398926, -4.903613567352295, -1.8356051445007324, -2.9106459617614746, 0.7115344405174255, 5.993354797363281, -0.677504301071167, 1.7132633924484253, -0.31398412585258484, -2.3909530639648438, 2.0287489891052246, -0.08705564588308334, 2.9501004219055176, 0.8735482096672058, -5.287994384765625, -4.6084303855896, 2.904789447784424, 0.5010327696800232, -1.1552925109863281, -2.865457057952881, 7.354454040527344, -1.4479445219039917, -1.1597814559936523, -3.390471935272217, 9.81695556640625, 1.1726152896881104, 3.1812968254089355, -2.3953731060028076, 2.2072207927703857, 2.71030592918396, -1.8269004821777344, 0.23745465278625488, 2.94899845123291, 1.6112557649612427, -0.039706211537122726, -1.6915539503097534, -0.3555919826030731, -2.1578588485717773, -4.124250411987305, 3.923370361328125, -1.3012570142745972, -2.4864470958709717, 1.3719654083251953, 1.7312088012695312, -1.3067641258239746, 1.7514700889587402, 8.154905319213867, -0.744899332523346, 2.721775531768799, 5.554675579071045, 1.567137598991394, 0.7460846304893494, 3.8750383853912354], [0.9248027205467224, -6.017016887664795, -0.8758650422096252, 4.3538408279418945, -2.9224960803985596, 1.205423355102539, -0.781861424446106, -3.338880777359009, 1.1199760437011719, -2.3680672645568848, 4.342517375946045, -3.8826746940612793, -1.821615219116211, 4.604008197784424, 0.485554963350296, 6.093241214752197, -4.279391765594482, 4.369090557098389, -4.93789005279541, -1.5287206172943115, -4.527904987335205, -1.3638521432876587, 0.4134511649608612, -0.6663691997528076, 5.075584411621094, -2.36568021774292, -3.0232579708099365, -3.930729389190674, 3.55008864402771, -2.5926496982574463, -3.2909719944000244, -3.854022741317749, 2.2047958374023438, 4.320852279663086, 0.9540453553199768, -1.503466010093689, 4.217864513397217, 4.0079216957092285, -3.3327884674072266, -3.3900904655456543, -0.8258605599403381, -0.43511125445365906, -5.789678573608398, 2.087850332260132, -0.5282260775566101, 6.653665542602539, 3.196394205093384, 0.13727618753910065, 1.4915364980697632, 2.4143848419189453, 6.474610805511475, -5.034560680389404, 1.5771191120147705, -0.43861690163612366, -0.41096267104148865, -7.536226272583008, 7.275166988372803, -0.29461604356765747, 0.09068203717470169, 0.5317754149436951, 1.4419997930526733, 5.037386417388916, -3.3760271072387695, 5.156287670135498, -0.25104767084121704, 6.654095649719238, -0.06233305484056473, 4.4044365882873535, 3.2396953105926514, 2.068189859390259], [-3.0583114624023438, 2.9764564037323, 2.927633047103882, -0.36494845151901245, 3.7918386459350586, -0.8363749384880066, -1.1492085456848145, 1.674507975578308, -0.7239527702331543, -0.21227924525737762, 2.798172950744629, 2.100651979446411, 1.3882285356521606, 0.9634833335876465, 1.721386432647705, -3.3694241046905518, 2.5353846549987793, -4.57974100112915, -1.8143601417541504, 4.51578950881958, -1.580482840538025, -1.6808691024780273, -2.985668420791626, -1.2667213678359985, 1.3009332418441772, 0.6484063267707825, -2.1584043502807617, -3.527435064315796, -2.0903804302215576, 0.6216949820518494, -1.1480984687805176, 1.7619003057479858, -2.0690784454345703, -5.516138553619385, 2.149510622024536, 1.958701491355896, 4.523355960845947, -5.706634998321533, 1.667358160018921, -1.836092233657837, 2.279054880142212, -1.937341570854187, -3.304659605026245, -3.2383499145507812, 0.2682008445262909, 1.5552974939346313, 1.6646751165390015, -2.526280164718628, -2.1953792572021484, 0.6746892929077148, 3.501574993133545, -0.2986447811126709, -1.9164247512817383, -0.6146658658981323, 2.965366840362549, -3.631722927093506, -3.8337228298187256, -7.675945281982422, 3.7894434928894043, -0.3068482577800751, 3.863780975341797, 2.3371262550354004, 0.3290647268295288, -9.73253059387207, -2.3970534801483154, -3.053255319595337, 0.6429522037506104, -4.042682647705078, -2.4385452270507812, -5.320512771606445], [-3.036649703979492, -3.537551164627075, 1.8127453327178955, 1.3989324569702148, -3.815110683441162, -0.43988052010536194, -4.179923057556152, 0.09063763171434402, 3.044921636581421, -0.736510157585144, 1.2350702285766602, -0.7374081611633301, -2.4064009189605713, 1.1837327480316162, 0.41135281324386597, 4.675710678100586, -3.0533945560455322, 2.7988476753234863, -4.168533802032471, 0.5369713306427002, -2.6390678882598877, 1.1656440496444702, 2.648709774017334, -1.0060796737670898, 2.123802900314331, -1.7102736234664917, -0.27672597765922546, 1.9702823162078857, -0.4276765286922455, 1.7046160697937012, 2.3461177349090576, -1.2374337911605835, -1.1208043098449707, -1.0024324655532837, 2.3801016807556152, -2.010596990585327, 0.4516403377056122, 1.7976045608520508, -0.4912778437137604, -3.635432720184326, 0.4098740220069885, -1.401092767715454, 1.187792420387268, 1.7487919330596924, -2.596994638442993, 2.020759344100952, 0.2397480458021164, -2.1008379459381104, 1.19601571559906, 1.9758163690567017, 0.2695918083190918, -1.6324208974838257, -1.2013304233551025, 0.03212185204029083, -2.219822883605957, -1.6813002824783325, 4.795647621154785, 3.2312357425689697, -4.403863430023193, -2.53869891166687, -0.2803899943828583, 2.0613529682159424, 0.029230203479528427, 2.5712456703186035, -2.483330249786377, -1.2696799039840698, 2.4188873767852783, 4.817744731903076, 2.9752542972564697, 3.436253070831299], [-2.404559850692749, 0.21973977982997894, -0.04687479883432388, 0.633669376373291, -2.252688407897949, -1.8719608783721924, -1.085378885269165, -0.6070214509963989, 0.09622552245855331, -0.5653148293495178, 1.218799352645874, 0.0037041231989860535, 1.1735471487045288, 1.0592302083969116, 1.4468642473220825, 2.9670166969299316, -0.12118159979581833, 1.1817179918289185, -0.8617042899131775, 1.0690511465072632, 0.36721837520599365, -0.9362797737121582, -0.1302448958158493, -0.27173328399658203, 1.0875194072723389, -2.807858467102051, 0.5034887790679932, 0.7304696440696716, 1.347184181213379, -0.9890957474708557, 1.089197039604187, 2.0673141479492188, -1.4365686178207397, -1.9659878015518188, 0.3024831712245941, 1.3865749835968018, 1.7181992530822754, -1.987072467803955, -1.5833492279052734, -2.412337303161621, -0.22946995496749878, -2.608696699142456, 0.8516162633895874, -0.340630441904068, 0.13309632241725922, -0.18025650084018707, 0.8428747653961182, -1.3976473808288574, 0.4227340817451477, -0.24799199402332306, 2.732154130935669, -2.155245304107666, -1.655347466468811, -2.8791308403015137, 1.9211498498916626, -4.599662780761719, 0.7923222780227661, -0.03806481137871742, 1.0192739963531494, -1.3572241067886353, 1.5481922626495361, 1.4221327304840088, -0.7384024262428284, 0.03843103349208832, -2.131431818008423, -0.02229931764304638, -2.490673065185547, 2.380086898803711, 0.9975603818893433, -1.1632283926010132], [-1.7570221424102783, -0.6370957493782043, 0.11764311790466309, -0.18312087655067444, -0.8687928318977356, 0.41997000575065613, 0.41259005665779114, -2.3761515617370605, 0.008473030291497707, 0.9692014455795288, 0.06510796397924423, -0.6558558344841003, 1.2887121438980103, -0.4126707911491394, -1.012797474861145, 0.9837912321090698, -0.9209014177322388, 3.0556485652923584, 1.8983920812606812, -1.1942962408065796, -0.7170992493629456, -0.3793655335903168, 0.668919563293457, 0.5199275016784668, -0.7724104523658752, -2.91153621673584, 3.464517116546631, -1.2792152166366577, 0.8862907886505127, 0.2085488736629486, 0.06257493793964386, 2.0934627056121826, 2.4687588214874268, 2.4121744632720947, -2.1983935832977295, -0.35158857703208923, 2.179616928100586, 1.8046592473983765, -1.572670340538025, 0.6181615591049194, -0.9078853130340576, 1.7611733675003052, 1.3620752096176147, 0.913368284702301, 1.439632534980774, -0.9704214930534363, 0.21693773567676544, 1.408697485923767, 1.6832115650177002, -1.0677984952926636, 0.627332866191864, -1.2780929803848267, 2.860095500946045, 0.3889930248260498, -0.6910937428474426, 0.18503354489803314, -0.7908353209495544, 3.2674002647399902, 0.7456665635108948, 0.15658213198184967, -0.46918338537216187, -0.811457633972168, -0.8848356604576111, 2.330155611038208, 0.8071098923683167, 2.2994132041931152, -2.1596434116363525, 1.6491690874099731, -0.40091556310653687, -1.5043386220932007], [2.2665553092956543, 0.29772263765335083, -2.9917404651641846, 0.5059842467308044, -1.3509646654129028, -0.007718903012573719, 1.9843289852142334, -0.4183838665485382, -0.8427202105522156, -1.3570575714111328, -1.147832989692688, 0.37504252791404724, -1.0833927392959595, -0.2864052355289459, -0.6123775839805603, -1.5248408317565918, 1.1155271530151367, 0.5760769844055176, -0.29148241877555847, -1.9284977912902832, 1.705905556678772, 0.05581945553421974, 0.3521654009819031, 0.08083044737577438, -0.8019881248474121, 1.4355031251907349, -0.8847578167915344, 1.4405274391174316, 1.407829999923706, -0.6214357018470764, 0.12443077564239502, -1.8293954133987427, -1.9082841873168945, 0.5083044767379761, 0.17904111742973328, -0.4755235016345978, -3.2011897563934326, -0.18169933557510376, -0.5325290560722351, 0.3837059736251831, -1.2553472518920898, 0.683530867099762, 1.6092066764831543, 0.2806960940361023, -0.0303859394043684, -1.3566240072250366, -0.5097035765647888, 0.444494366645813, -0.7415188550949097, -0.0009522727923467755, -1.614883303642273, 0.6568741202354431, -0.8945609331130981, -1.344710350036621, 0.4390839636325836, 1.5665427446365356, 0.7964869737625122, -1.0682649612426758, -0.8424239158630371, 0.8222979307174683, -0.7800461053848267, -0.35289716720581055, -0.703655481338501, 1.8055998086929321, -0.24366647005081177, 0.08102823048830032, 1.0795645713806152, -1.9268511533737183, -0.9049511551856995, 2.6765894889831543], [-2.795109987258911, 3.290008544921875, 5.372827529907227, 1.1621748208999634, 4.377359867095947, 1.255544900894165, -1.8711973428726196, 1.3712000846862793, -2.9492838382720947, 1.6900051832199097, 5.520812511444092, -0.10005710273981094, 5.413774490356445, 1.0583789348602295, 3.0963172912597656, -0.7673828601837158, -3.746243953704834, -0.5073773264884949, 1.9973896741867065, 2.9441146850585938, -2.4221434593200684, -1.9826748371124268, -4.395669460296631, -0.2263832837343216, 0.9608542323112488, -4.618218898773193, 4.023286819458008, -5.998631477355957, 0.2858819365501404, -1.3927249908447266, -1.3355258703231812, 6.516403675079346, 5.836555004119873, -1.8770086765289307, -0.6621181964874268, 0.5698906779289246, 9.76467514038086, -4.864522933959961, -0.7948760390281677, -1.8842484951019287, 3.0243608951568604, -6.968490123748779, -2.7668871879577637, -2.057720899581909, 2.606086254119873, 1.0583018064498901, 0.41028785705566406, 0.6643987894058228, 1.5909786224365234, -1.6658560037612915, 4.033732891082764, -3.874436378479004, 3.8378851413726807, 0.6342623233795166, 1.4890618324279785, -2.0422959327697754, -2.701129198074341, 1.6272118091583252, 3.695449113845825, -2.001012086868286, 1.0356298685073853, 3.618863344192505, -2.8973824977874756, -7.249595642089844, -0.8643137812614441, 0.27528271079063416, -5.702479839324951, 1.84115469455719, -0.3669107258319855, -6.864021301269531], [-0.8708643913269043, 0.39916911721229553, 4.376112461090088, -2.8338229656219482, 3.4204320907592773, -1.1116567850112915, -3.298617362976074, 3.931894540786743, 2.252061367034912, 2.637333393096924, -1.4231003522872925, 0.14962507784366608, 0.24495495855808258, -0.7862910628318787, 1.2753942012786865, 2.0048418045043945, 0.5116541385650635, -5.529940605163574, 0.902323842048645, 4.272613048553467, 0.06953264772891998, 1.5516918897628784, -0.1600836217403412, 0.21084459125995636, 0.5589431524276733, 2.1446385383605957, -1.9442028999328613, 1.8956767320632935, -4.693338871002197, 1.362831711769104, 0.6543307900428772, 0.8100737929344177, 0.3495367169380188, -2.6282920837402344, 0.7570303082466125, 2.127450704574585, -0.5986881852149963, 0.2555016279220581, 3.883624315261841, 1.3251011371612549, 2.64585018157959, -0.9557251930236816, -2.809541702270508, -1.0903927087783813, -1.8348464965820312, 1.6375800371170044, -0.5125297904014587, -1.7723503112792969, -0.6000636219978333, 0.38686269521713257, -0.5694530010223389, 2.897848129272461, -2.270495891571045, 2.761232376098633, -0.7955190539360046, -0.658941924571991, -1.782335638999939, 0.1609984040260315, 0.10857558995485306, -1.07123601436615, 0.725295901298523, -1.9472308158874512, 4.71626091003418, -3.4482359886169434, 1.5662107467651367, -3.2536213397979736, 0.325608491897583, 1.8107538223266602, 2.6346096992492676, -2.454707622528076], [2.4334380626678467, -1.7143638134002686, 2.9368948936462402, 0.6681720614433289, 2.5165646076202393, 1.3644202947616577, -2.0772769451141357, 2.462355613708496, 0.566692590713501, 1.018339991569519, 1.4550973176956177, -2.106901168823242, -0.0791151151061058, 1.200008511543274, 1.5029997825622559, 2.8191962242126465, -3.590482473373413, -1.9345369338989258, -0.4019969403743744, 1.1931736469268799, -1.6651966571807861, 0.8457536101341248, -0.9310041069984436, 0.188243567943573, 2.1192498207092285, 1.0125962495803833, -1.9186967611312866, -1.0321635007858276, -1.1640119552612305, -1.0197365283966064, -1.5977810621261597, -1.2273643016815186, 3.759998083114624, 1.4268755912780762, 0.6678655743598938, -0.3807871639728546, 1.3588886260986328, 1.9535117149353027, 1.3688746690750122, 0.19091056287288666, 2.063084363937378, -2.9050028324127197, -4.839449405670166, 0.3682253658771515, -1.0922297239303589, 3.9450998306274414, -0.1370665580034256, 0.3549041152000427, 0.7288996577262878, 0.8299372792243958, 0.8225550055503845, -0.19352735579013824, 0.8955540657043457, 2.9041213989257812, -1.6537879705429077, -0.6760928630828857, 2.5991058349609375, 2.2153608798980713, -0.830851137638092, -0.5604871511459351, -0.775314450263977, 1.2697227001190186, 0.7053808569908142, 0.029193997383117676, 2.0402672290802, 1.0484651327133179, -0.3746480643749237, 3.333362579345703, 3.547248363494873, 0.32005563378334045], [-7.410659313201904, -2.341878652572632, 3.1505064964294434, -2.9921176433563232, -1.0166425704956055, -1.8395105600357056, -2.4079842567443848, -3.5884792804718018, 3.7014784812927246, 3.349010467529297, -1.1147782802581787, -0.8456738591194153, 1.7849159240722656, -0.5843535661697388, -1.5709737539291382, 5.3160505294799805, 0.9009649157524109, 2.155301570892334, 2.3830275535583496, 2.1866469383239746, -2.3229010105133057, -0.22864001989364624, 2.431821584701538, 0.6362439393997192, 0.0512763187289238, -4.918099880218506, 3.977353811264038, -0.9612318873405457, -2.2342631816864014, 2.33537220954895, 0.9157039523124695, 4.373891830444336, 2.079149007797241, 1.4263591766357422, -3.2447829246520996, 2.240976572036743, 4.502913951873779, 4.060179710388184, -0.5769636034965515, 0.9430575966835022, -0.5704494118690491, 5.509969234466553, 0.24351730942726135, 0.8361835479736328, 0.894690215587616, 0.23561254143714905, 1.9422792196273804, -0.502406895160675, 2.278524398803711, -0.8640540838241577, 3.568458318710327, -0.784285306930542, 1.8809635639190674, 1.4282985925674438, -0.8251132965087891, -4.744499683380127, -3.1670174598693848, 3.3091022968292236, 2.872941732406616, -0.276332288980484, 2.3010497093200684, -3.0332143306732178, 2.853095054626465, 1.9557867050170898, 1.3890888690948486, 2.208167791366577, -3.0561745166778564, 4.521326541900635, 0.9248877763748169, -6.163495063781738], [-3.2887134552001953, -2.164583683013916, -0.493429958820343, -1.1195056438446045, -2.1026294231414795, 0.22524505853652954, -0.6645268797874451, -1.8252674341201782, 3.1255874633789062, -0.599831759929657, -2.190610885620117, 1.7106612920761108, -4.070793151855469, -1.3732316493988037, -3.4375219345092773, -2.5992605686187744, 2.5510993003845215, 1.4452265501022339, -2.8107268810272217, -0.7422603964805603, -2.0602967739105225, 1.278421401977539, 3.632495880126953, -0.9624390006065369, -1.1393060684204102, 1.8517200946807861, -0.2683489918708801, 1.116433024406433, -2.4134135246276855, 4.701269626617432, 1.7255973815917969, -3.0158026218414307, -3.7565083503723145, -0.43310871720314026, 1.3226972818374634, -2.395841598510742, -2.625429391860962, 2.609726667404175, 1.1761629581451416, -0.007953333668410778, -1.1297334432601929, 7.124587535858154, 3.2395591735839844, 1.15645170211792, -1.7305998802185059, -0.828258752822876, 0.19607795774936676, -1.2626816034317017, -0.8875874280929565, 1.8592071533203125, -2.9971206188201904, 2.2451488971710205, -0.5044930577278137, 1.5164620876312256, -2.6004459857940674, 3.613910436630249, -0.5568838119506836, -1.7771201133728027, -3.26983380317688, 0.6136906743049622, -0.022506235167384148, -1.7358553409576416, 1.635557770729065, 0.6683829426765442, -1.2866891622543335, -2.417935609817505, 6.888411045074463, -3.153424024581909, -2.607300043106079, 2.6604228019714355], [-1.4923300743103027, 0.7681566476821899, -1.7807936668395996, -0.09734465181827545, -0.6809616684913635, -1.0211119651794434, 1.9391233921051025, -2.6427133083343506, -0.8323659896850586, -0.46875330805778503, 0.2987806797027588, 0.3301192820072174, 1.3441407680511475, 0.4386001229286194, -0.43332532048225403, -0.4575839936733246, 2.3038339614868164, 1.0011917352676392, 0.7176300287246704, -0.18984916806221008, 0.43861645460128784, -1.7392572164535522, -0.5972757339477539, 0.10939971357584, -0.1595320999622345, -1.7396225929260254, 0.6762745380401611, -1.6708307266235352, 1.5617815256118774, -0.9079885482788086, -0.8833421468734741, 1.4416582584381104, -0.9875187277793884, 0.1287730485200882, -1.3528292179107666, 1.6363788843154907, 1.758413314819336, -1.0999641418457031, -1.6844334602355957, -0.011988240294158459, -1.3429642915725708, 1.8240797519683838, 0.11643365770578384, -0.6728294491767883, 1.735656499862671, -0.8490440845489502, 1.4968410730361938, 0.12835687398910522, -0.2514181435108185, -0.7751345038414001, 2.8518552780151367, -1.2313868999481201, 0.13189411163330078, -2.1720924377441406, 2.3899505138397217, -3.3175134658813477, -2.0264968872070312, -2.851321220397949, 3.4999868869781494, 1.1445837020874023, 2.0175530910491943, -0.057290010154247284, -0.9660415649414062, -0.31713083386421204, -0.46681177616119385, 1.9475951194763184, -2.0454680919647217, -1.6068851947784424, -1.9216017723083496, -2.8581788539886475], [-1.0946062803268433, -0.8743048906326294, -1.5809385776519775, -1.1154167652130127, 1.8822354078292847, 1.5746006965637207, 2.9879531860351562, -3.4564616680145264, 0.7831500172615051, -0.5523837804794312, -1.154209852218628, 1.3939322233200073, -2.4759957790374756, -0.7509704828262329, -4.112890243530273, -6.394268035888672, 4.563949108123779, -0.3516858220100403, -1.0146152973175049, -1.0615360736846924, -2.560349464416504, -0.6925845146179199, 0.8460546135902405, -0.5626935958862305, -1.1633461713790894, 2.9465889930725098, -1.323860764503479, -4.166994094848633, -1.6420186758041382, 2.9734513759613037, -2.542998790740967, -3.2607674598693848, -1.635050892829895, 1.804909348487854, -0.4328157305717468, -1.1358364820480347, 0.14297892153263092, 2.145303964614868, 0.9757219552993774, 2.3662707805633545, -1.3909571170806885, 9.714531898498535, -1.226330041885376, -0.24967628717422485, 0.8081256151199341, 0.42600101232528687, 1.7817988395690918, 0.645026683807373, -1.8867605924606323, 1.2076138257980347, -0.142333984375, 2.0935168266296387, 1.9246642589569092, 2.285064697265625, -0.6770514249801636, 2.4327034950256348, -3.867508888244629, -6.908250331878662, 1.9419199228286743, 3.7380316257476807, 1.6313430070877075, -1.6316065788269043, 0.4922141134738922, -2.153250217437744, 1.0140540599822998, 1.1181198358535767, 5.572368621826172, -7.787812232971191, -5.511610507965088, -1.4469950199127197], [7.642094612121582, 0.4994642436504364, -2.687964677810669, 0.21612267196178436, 1.0887300968170166, 2.1983392238616943, 3.099616527557373, 1.5301604270935059, -2.353694200515747, 0.4592508375644684, -2.4622802734375, -1.4713916778564453, 0.03242264315485954, -1.4892377853393555, -0.4448055922985077, -1.2588045597076416, -2.1183202266693115, 0.1065559983253479, 3.803663969039917, -4.0201239585876465, 3.3605785369873047, 1.4757356643676758, -0.38608062267303467, 1.614683985710144, -2.159473180770874, 2.6673741340637207, 0.6085740327835083, 2.123775005340576, 1.617462158203125, -2.2813985347747803, -0.8356351852416992, -1.918150782585144, 3.5341379642486572, 4.752745628356934, -1.9434319734573364, -1.3920485973358154, -5.2121052742004395, 2.651909589767456, 0.21328385174274445, 3.8387420177459717, -0.7015523314476013, -1.0567072629928589, 0.749512255191803, 1.329350471496582, 0.5842574238777161, -1.7546708583831787, -2.8742690086364746, 3.74039888381958, 0.7043336629867554, -1.4685018062591553, -5.021471977233887, 1.986161708831787, 2.1807379722595215, 1.6453031301498413, -1.855860948562622, 6.769021034240723, 1.6419246196746826, 5.125145435333252, -2.2017784118652344, 1.257761001586914, -4.639108180999756, -2.1908998489379883, -0.543312132358551, 5.133998394012451, 3.943321704864502, 1.9387059211730957, -1.2175817489624023, 0.28047841787338257, 1.257041335105896, 4.330294132232666], [-3.3142504692077637, 2.7776849269866943, 0.49006232619285583, -0.06692159920930862, -1.8882946968078613, -0.3502514362335205, -0.7362465262413025, 0.17085085809230804, -1.103154182434082, 0.07227537781000137, 0.14119261503219604, 2.185319185256958, 1.4145517349243164, -1.7963234186172485, 0.10519526898860931, -2.299168348312378, -0.48542240262031555, 2.610954761505127, 1.0375421047210693, -0.5186100006103516, 1.034806489944458, 0.15730276703834534, 0.4378894865512848, -0.31773948669433594, -2.4601752758026123, -2.7724854946136475, 4.986444473266602, 1.5776594877243042, 0.5309749841690063, 1.6309128999710083, 3.5311522483825684, 3.8854644298553467, -0.8970714211463928, -2.7155237197875977, 0.005075233522802591, -1.3344181776046753, 0.5184888243675232, -3.3886237144470215, -0.9852176308631897, -1.5529693365097046, -0.1145879253745079, -2.3506898880004883, 6.636833667755127, -0.14222396910190582, 0.7013552784919739, -4.64565372467041, -1.7888251543045044, -0.17709703743457794, 0.8229628205299377, -1.4015299081802368, -2.929949998855591, -0.7044408917427063, 0.5969369411468506, -1.6670478582382202, -0.1184186190366745, 3.930309534072876, -1.6437408924102783, 3.428238868713379, -2.041029691696167, -2.2341666221618652, -1.4672858715057373, -0.04248190298676491, -1.5304054021835327, -1.4413787126541138, -2.934309720993042, -3.4647889137268066, -0.8069570660591125, -0.012737570330500603, -1.9916481971740723, 0.34080031514167786], [-2.7208001613616943, -0.2438584417104721, 3.2190518379211426, -2.0048084259033203, 2.971445083618164, 0.2579491436481476, -0.7903456091880798, -0.9657029509544373, 0.9686205387115479, 2.8219690322875977, 0.10082916170358658, -0.6702028512954712, 2.1097989082336426, -0.441060334444046, -0.6008574962615967, 0.9725362062454224, 0.15591931343078613, -0.9274346828460693, 2.7510087490081787, 1.9081578254699707, -1.860759973526001, -0.36712560057640076, -0.4274574816226959, 0.6058359742164612, -0.07252626121044159, -1.7033283710479736, 2.037550926208496, -3.018740177154541, -2.15755558013916, 0.894970178604126, -1.315953016281128, 2.8420536518096924, 3.706221580505371, 1.446553349494934, -2.3363006114959717, 1.456640362739563, 4.209977149963379, 1.9152926206588745, 0.7744572758674622, 1.9966671466827393, 0.8419628143310547, 2.7177555561065674, -2.5980780124664307, -0.3709242343902588, 1.3036519289016724, 1.1042847633361816, 1.0139830112457275, 0.7402032017707825, 1.0462950468063354, -0.8494157791137695, 2.230851173400879, 0.09420246630907059, 2.7171547412872314, 2.739370822906494, -0.5433937907218933, -1.4787003993988037, -3.5157737731933594, 1.011383056640625, 3.2136521339416504, 0.6110504269599915, 1.221800684928894, -1.6968985795974731, 1.676071047782898, -1.6175720691680908, 2.439244031906128, 1.7311055660247803, -2.276888608932495, 1.1145195960998535, -0.06430411338806152, -5.559572219848633], [-4.558080673217773, -5.985955238342285, 0.09012702107429504, -0.6437528133392334, -3.0939083099365234, 1.3763636350631714, -1.6776419878005981, -3.815138101577759, 5.573627948760986, -0.7693125009536743, -1.9230048656463623, 0.6398242712020874, -6.618503093719482, -0.6559666395187378, -5.320550441741943, -1.3289753198623657, 1.644334077835083, 3.420720338821411, -5.18147611618042, -1.4573841094970703, -5.445405960083008, 1.804243803024292, 5.7175421714782715, -1.4165223836898804, 0.08489308506250381, 2.0270628929138184, -1.0826795101165771, -0.39927932620048523, -3.2498362064361572, 6.519957065582275, 0.9295675754547119, -5.799530029296875, -3.1009697914123535, 2.038719892501831, 1.7197500467300415, -4.2860612869262695, -1.7993979454040527, 6.839278697967529, 1.1530230045318604, -0.26373279094696045, -1.5798280239105225, 11.40919017791748, 1.7485722303390503, 2.761277675628662, -2.7460432052612305, 1.8320149183273315, 1.3737133741378784, -1.2952340841293335, -0.3271888494491577, 3.599236249923706, -2.335047721862793, 1.972478985786438, 0.9875399470329285, 3.7775378227233887, -5.061622619628906, 3.410876989364624, 1.4031329154968262, -1.5397790670394897, -4.8208489418029785, 1.319005012512207, 0.11874323338270187, -1.462043046951294, 1.9227705001831055, 3.082425832748413, -0.7248761057853699, -0.6548007130622864, 10.22788143157959, -2.457646369934082, -2.2704756259918213, 4.031866073608398], [2.0915565490722656, 0.3929089903831482, 2.2752304077148438, 1.6545261144638062, 2.0476207733154297, 0.7173829078674316, -1.8914674520492554, 3.3218746185302734, -0.8951310515403748, -0.4241216480731964, 2.733363628387451, -0.7285465598106384, 0.3647598326206207, 1.5760250091552734, 2.7338593006134033, 0.9154127836227417, -2.908921003341675, -2.398285150527954, -1.711351752281189, 1.718713402748108, -0.8007891774177551, 0.0668589174747467, -2.146695613861084, -0.49035361409187317, 2.1584300994873047, 0.9068920612335205, -2.3402140140533447, -0.8295223712921143, -0.16041894257068634, -1.5516448020935059, -0.8814352750778198, -0.5513045191764832, 1.3676143884658813, -1.6784487962722778, 2.1810944080352783, -0.23916645348072052, 1.596974492073059, -2.151015520095825, 0.9100145697593689, -1.793654441833496, 2.3266806602478027, -5.982837677001953, -3.5384628772735596, -0.8042572736740112, -1.0028380155563354, 2.861182689666748, -0.25860700011253357, -0.7283200621604919, -0.250764399766922, 0.8266394138336182, 1.0613586902618408, -0.996003270149231, -0.8184919357299805, 0.4797011613845825, 0.23869843780994415, -1.219244360923767, 2.4216511249542236, -0.2978695034980774, -0.6782340407371521, -1.3798701763153076, -0.0891621932387352, 3.114332675933838, -0.7305558323860168, -2.931291341781616, -0.5723279118537903, -1.0179094076156616, -0.3408963084220886, 1.5595775842666626, 2.3016879558563232, 0.5620884895324707], [1.7968218326568604, 1.577797293663025, 0.9953513145446777, 0.2687738239765167, 1.5162270069122314, 0.9320532083511353, 0.26451507210731506, 1.2492473125457764, -1.993951678276062, 1.1961723566055298, 0.6951534748077393, -0.7741243243217468, 2.558821678161621, -0.4567837715148926, 1.247167706489563, 0.2921659052371979, -2.6043195724487305, 0.2487308382987976, 3.023946762084961, -0.5410434603691101, 1.0987063646316528, 0.028852097690105438, -1.5705116987228394, 0.8226845264434814, -0.769365131855011, -1.4046614170074463, 2.580129384994507, -0.4349737763404846, 0.9366083145141602, -1.6581836938858032, -0.17112363874912262, 2.560981273651123, 3.86948561668396, 1.2233784198760986, -1.4766454696655273, -0.018877185881137848, 1.2855703830718994, -0.6714926362037659, -0.4556533098220825, 0.8901993036270142, 0.7565426826477051, -3.7257628440856934, 0.12827898561954498, -0.05106211081147194, 1.274393916130066, -0.96339350938797, -1.3525309562683105, 1.8220419883728027, 1.3445253372192383, -1.6797410249710083, -0.7563551068305969, -0.7581226229667664, 2.2035014629364014, 0.5537585020065308, -0.18660950660705566, 1.8426125049591064, -0.2515237033367157, 4.258795738220215, 0.3080681264400482, -0.7129405736923218, -1.8212907314300537, 0.027875399217009544, -1.1039663553237915, 0.25380000472068787, 1.3436533212661743, 0.8305254578590393, -3.748983383178711, 2.0996878147125244, 1.0448263883590698, -0.8523706197738647], [4.611262321472168, 0.02704460918903351, 0.22225745022296906, 1.2346575260162354, 3.9841878414154053, 2.499948501586914, 1.8151917457580566, 0.7805731892585754, -2.078425407409668, 0.004516886081546545, 1.7569165229797363, -1.519426941871643, 0.6243149042129517, 1.0890389680862427, 0.4850700795650482, -1.880201816558838, -1.6527037620544434, -1.8273553848266602, 0.752616286277771, -0.6745407581329346, -0.9595918655395508, -0.6105188727378845, -2.6220953464508057, 0.36119288206100464, 0.9665271639823914, 1.9495301246643066, -1.8706181049346924, -3.9514260292053223, 0.6396430730819702, -2.1882729530334473, -3.8538260459899902, -1.7580575942993164, 3.9208223819732666, 2.875941038131714, -0.4358524978160858, -0.4478220045566559, 1.7022148370742798, 0.6514029502868652, 0.43976640701293945, 1.6260279417037964, 0.9454413056373596, -0.996881902217865, -5.302586078643799, -0.43457844853401184, 1.0086688995361328, 2.8471436500549316, 0.30449527502059937, 2.1038215160369873, -0.24214091897010803, 0.08173756301403046, 1.1388179063796997, -0.27194827795028687, 2.5648353099823, 2.1994948387145996, -0.09454356133937836, 0.7023236155509949, 0.3923422396183014, -1.4592379331588745, 1.8073397874832153, 1.761360764503479, -0.4959278702735901, 1.3675448894500732, -1.3520922660827637, -1.238667607307434, 2.6673285961151123, 2.962961435317993, -0.5775835514068604, -1.5733110904693604, 0.03383716940879822, -0.6969725489616394], [4.2310943603515625, -1.1977213621139526, -0.7443106174468994, -3.5214037895202637, 3.4782345294952393, -1.0218607187271118, 2.2344532012939453, -0.009653009474277496, 1.0886292457580566, 3.0432803630828857, -4.140404224395752, -2.5915825366973877, 1.190774917602539, -0.48036348819732666, -0.5995384454727173, 3.8372042179107666, 2.5516111850738525, -4.515677452087402, 5.097361087799072, 0.7743701338768005, 2.5929737091064453, 0.42599010467529297, -0.31998342275619507, 2.275304079055786, 0.016308587044477463, 2.6109225749969482, -2.8064963817596436, 0.8271353244781494, -1.7016937732696533, -2.395336151123047, -3.2220773696899414, -0.8523140549659729, 2.8514883518218994, 4.566234111785889, -3.9005086421966553, 4.562751293182373, -2.816370725631714, 5.279114246368408, 1.9146997928619385, 6.110562801361084, -0.4817184507846832, 4.486292362213135, -5.82435417175293, -0.07241692394018173, 0.711077868938446, 1.8919998407363892, 0.7573863863945007, 1.780914545059204, -0.14856640994548798, -1.2108720541000366, 1.6377915143966675, 3.278348445892334, -0.23558640480041504, 2.435678720474243, 0.5476071238517761, -3.2471702098846436, -2.30350399017334, -0.25900334119796753, 4.744955062866211, 3.2294094562530518, 0.8289058208465576, -5.121995449066162, 4.763757705688477, 3.688849687576294, 6.8558502197265625, 4.526518821716309, -3.6683080196380615, 1.3876442909240723, 2.884222984313965, -3.6452462673187256], [-6.142684459686279, -5.410780429840088, 0.03439447656273842, -1.883901834487915, -4.560568809509277, -1.8169455528259277, -2.1132497787475586, -4.5394182205200195, 5.695746421813965, 0.7489402294158936, -2.538114070892334, -0.6499890685081482, -3.0689597129821777, -0.08742879331111908, -3.3434813022613525, 5.088990688323975, 2.047646999359131, 3.5149903297424316, -2.073786973953247, 0.1788049340248108, -2.8382785320281982, 0.8058037161827087, 5.240993022918701, -0.22998332977294922, 0.7371885180473328, -1.9002556800842285, 0.3416646718978882, 1.531341791152954, -1.9220995903015137, 3.7207610607147217, 1.580712080001831, -1.3939028978347778, -2.434506893157959, 1.813407063484192, -0.8203956484794617, 0.08929659426212311, -0.42971622943878174, 6.413792610168457, -0.4939170181751251, -0.17476005852222443, -2.1298537254333496, 8.94274616241455, 1.4881528615951538, 2.3411879539489746, -1.5423046350479126, 1.1195497512817383, 2.203927516937256, -1.7084534168243408, 1.0003844499588013, 1.5960344076156616, 1.621435523033142, 0.3624691665172577, -0.5816806554794312, 0.8621296286582947, -2.200378179550171, -3.6688730716705322, 0.6660869121551514, 0.8541262745857239, -1.0011259317398071, 0.4666852653026581, 1.8929736614227295, -2.681431293487549, 3.071554660797119, 5.487463474273682, 0.01499931700527668, 1.5355160236358643, 2.9859819412231445, 2.7804653644561768, 0.7828793525695801, -0.05570153892040253], [8.189309120178223, 0.5382056832313538, 0.3700478971004486, 2.8785905838012695, 6.7587199211120605, 5.777122497558594, 3.974787712097168, 0.30543598532676697, -5.092595100402832, 0.4922158122062683, 3.8050527572631836, -3.0676820278167725, 2.560577392578125, 1.317946195602417, 0.4502038061618805, -4.080206394195557, -5.056929588317871, -0.2801330089569092, 3.3004231452941895, -3.120969533920288, -1.8620349168777466, -1.2598047256469727, -4.841058731079102, 1.0793848037719727, 0.4829052686691284, 1.175887107849121, 0.45484352111816406, -8.210005760192871, 2.6429450511932373, -4.166782855987549, -6.632873058319092, -1.174497365951538, 9.9884614944458, 7.20110559463501, -2.3715572357177734, -2.2326605319976807, 4.6700758934021, 1.5673205852508545, -0.733751118183136, 3.23931622505188, 1.2414954900741577, -2.465521812438965, -7.35850715637207, -0.005533924791961908, 3.2863073348999023, 3.431342601776123, -0.3178161382675171, 5.712016582489014, 1.1740608215332031, -1.130977988243103, 1.0295029878616333, -1.8154449462890625, 7.838745594024658, 4.282990455627441, -1.0876139402389526, 3.9819695949554443, 0.5328717231750488, 1.487240195274353, 2.7319326400756836, 2.8416874408721924, -2.689378023147583, 2.4943113327026367, -4.478348731994629, -0.6418138146400452, 5.163126468658447, 6.664302825927734, -3.005120277404785, -1.7778279781341553, -0.5990122556686401, -1.2291547060012817], [5.092983245849609, 0.5389161705970764, 1.1371936798095703, 1.7156662940979004, 2.852930784225464, 3.0875980854034424, -0.5148756504058838, 4.259477138519287, -1.3975763320922852, -0.9217037558555603, 1.1071622371673584, 0.15099270641803741, -2.0545706748962402, -0.14491428434848785, 0.800713062286377, -3.9753270149230957, -3.0581648349761963, -2.337977409362793, -1.8018431663513184, -0.8226708173751831, -0.6120303273200989, 1.6081541776657104, -1.2595499753952026, -0.5730810761451721, 0.09224504232406616, 4.444764614105225, -2.3362209796905518, -0.09603504091501236, -0.9488893151283264, 0.2238629013299942, -0.6256633996963501, -3.6502153873443604, 1.353447437286377, -0.26794689893722534, 2.861016035079956, -3.430715799331665, -2.28503155708313, -0.8320441842079163, 2.316805601119995, -0.045784614980220795, 2.112632989883423, -3.7825734615325928, -1.4358789920806885, 0.02226760797202587, -1.6856920719146729, 1.4113296270370483, -2.23311448097229, 0.7007771134376526, -0.9096696972846985, 1.3232934474945068, -4.428763389587402, 1.5531158447265625, 0.5556069612503052, 2.8276610374450684, -2.473642587661743, 6.775326251983643, 2.655230760574341, 0.49702951312065125, -4.376017093658447, -0.4993123412132263, -3.127655267715454, 1.9574034214019775, -0.9241746664047241, -2.3230440616607666, 0.24908503890037537, -2.6643807888031006, 4.211315155029297, -1.9654768705368042, 0.38563787937164307, 4.883720874786377], [-0.5675252676010132, -1.8006162643432617, -0.5743612051010132, -1.6430777311325073, -2.136352062225342, -1.7315856218338013, -0.6990803480148315, -0.6572882533073425, 2.196075439453125, 1.1644341945648193, -2.7419631481170654, -1.004058599472046, -0.37250393629074097, -0.5446103811264038, -0.5084174871444702, 4.571680545806885, 0.6080828309059143, 0.5139232873916626, 1.2867730855941772, -0.0028701776172965765, 1.4061354398727417, 0.966005802154541, 2.18221378326416, 0.8384349942207336, -0.13412818312644958, -0.6096230745315552, 0.12952814996242523, 3.441538095474243, -0.5478038191795349, 0.06517397612333298, 1.2785191535949707, 0.13410739600658417, -0.4986863136291504, 1.2803828716278076, -1.3719652891159058, 1.5179803371429443, -2.615525484085083, 3.225144386291504, 0.01802806928753853, 1.199912667274475, -0.9937102198600769, 2.015308141708374, 0.9673974514007568, 1.1433310508728027, -0.7527367472648621, -0.426575630903244, -0.056977368891239166, -0.29070284962654114, 0.8188775181770325, -0.3377015292644501, -0.19876082241535187, 1.0009862184524536, -1.344294548034668, -0.15815719962120056, -0.5752971768379211, -1.8311889171600342, 0.7249683141708374, 2.9164233207702637, -0.40413662791252136, -0.11273223906755447, -0.12638817727565765, -2.6988115310668945, 2.5327513217926025, 4.64821720123291, 1.4430772066116333, 0.8783132433891296, -1.3017405271530151, 3.5256705284118652, 2.533421039581299, 0.4226701557636261], [-0.007238996680825949, 5.274226665496826, -0.9927374124526978, -2.0009758472442627, -0.3954395055770874, -2.388000249862671, 0.7880405187606812, 2.522722005844116, -2.141249656677246, 0.8563437461853027, -2.529226541519165, 2.3417952060699463, 2.9064974784851074, -2.558976173400879, 1.561807632446289, -1.5216008424758911, 2.1103715896606445, -1.7118322849273682, 4.0092082023620605, 0.6092257499694824, 5.619583606719971, 0.30985021591186523, -0.9161417484283447, 0.7883026599884033, -3.262568473815918, -0.5763741731643677, 2.906151056289673, 4.887438774108887, 0.22892379760742188, -0.7126648426055908, 3.46195125579834, 4.559173107147217, -2.2922780513763428, -3.9454452991485596, -0.934346616268158, 2.2910056114196777, -3.4103894233703613, -4.990111827850342, 0.35079583525657654, 0.8138949275016785, -0.03284258022904396, -4.171826362609863, 5.826089859008789, -1.546006441116333, 0.8511962294578552, -6.068718910217285, -2.6043317317962646, -0.3101482689380646, -0.43203005194664, -2.7633895874023438, -3.474313735961914, 1.6625138521194458, -2.6503844261169434, -3.1673033237457275, 2.334350109100342, 2.552595853805542, -3.4135828018188477, 2.0113165378570557, 0.27439695596694946, -1.6310467720031738, -1.1153262853622437, -2.488640785217285, 0.9327777624130249, -1.6990822553634644, -1.1582362651824951, -4.23148250579834, -3.7708303928375244, -0.36706164479255676, -0.666412353515625, -0.8008773922920227], [3.7379159927368164, 0.3141060769557953, -3.9409523010253906, -2.060835599899292, 3.704033374786377, 3.737464666366577, 6.552749156951904, -3.264106273651123, -1.0357376337051392, 0.12973552942276, -4.072265148162842, 1.482784628868103, -2.7581121921539307, -3.099681854248047, -6.14218807220459, -10.797820091247559, 5.058732509613037, -0.5129845142364502, 2.6590964794158936, -4.469202518463135, -0.2672887146472931, 0.36850446462631226, 0.865436315536499, 0.6107519268989563, -4.3492655754089355, 5.850564002990723, 0.022982534021139145, -3.567633867263794, -1.437874436378479, 2.8948440551757812, -3.330291748046875, -4.703113079071045, 0.47823625802993774, 5.521949291229248, -2.5072219371795654, -2.579092264175415, -4.055358409881592, 4.262295722961426, 1.7706960439682007, 6.672897815704346, -2.377688407897949, 12.50476360321045, 0.4784226417541504, 0.4393283426761627, 1.8929390907287598, -2.2892982959747314, -0.4129398465156555, 3.92480731010437, -1.976869821548462, -0.11330334097146988, -5.246451377868652, 4.949369430541992, 4.5513105392456055, 4.445545673370361, -2.4196362495422363, 10.139244079589844, -5.436134338378906, -4.726500511169434, 0.872746467590332, 5.812934398651123, -1.7671571969985962, -4.777121067047119, 0.5233385562896729, 0.25953593850135803, 4.333398342132568, 1.9596599340438843, 6.349244117736816, -10.925766944885254, -7.307394981384277, 0.8147437572479248], [-2.587432384490967, -3.7814815044403076, -2.209991455078125, -1.496978998184204, -3.7118237018585205, -0.9751102328300476, 0.571489691734314, -4.232211589813232, 3.410050392150879, 0.2107691913843155, -3.1721651554107666, -0.5131040811538696, -2.5417232513427734, -0.6578319668769836, -3.4503626823425293, 2.4112231731414795, 2.4168338775634766, 3.209057569503784, -0.4089306592941284, -1.7143305540084839, -0.7854682207107544, 0.5929861664772034, 4.081184387207031, 0.2981136739253998, -0.5635827779769897, -0.5436058640480042, 0.41924071311950684, 1.5563048124313354, -0.43169328570365906, 2.2155864238739014, 0.7297491431236267, -1.9452369213104248, -1.925702691078186, 3.0344831943511963, -1.5864951610565186, -0.13291630148887634, -2.3764162063598633, 5.580792427062988, -0.8098071813583374, 1.3253705501556396, -2.73218035697937, 8.304437637329102, 2.0402543544769287, 2.092099189758301, -0.5091449022293091, -0.42250916361808777, 1.227756142616272, -0.056900572031736374, 0.5471752882003784, 0.6662713289260864, -0.07192801684141159, 0.9900363087654114, 0.09879527986049652, 0.36891812086105347, -1.5827713012695312, -0.9498381018638611, 0.19519709050655365, 0.5844182968139648, -0.6244503259658813, 1.5506776571273804, 0.5518664717674255, -3.1302871704101562, 1.9014939069747925, 5.9988932609558105, 1.000124454498291, 2.235741138458252, 2.2560958862304688, 0.5550073385238647, -0.3625745475292206, 1.073803186416626], [-0.06472226977348328, -0.994079053401947, 1.752909779548645, 0.14798018336296082, -0.7732318639755249, -0.3958015739917755, -2.62951922416687, 2.025503158569336, 1.2460013628005981, 0.44312623143196106, 0.07837362587451935, -0.6411845684051514, -0.5129331946372986, 0.22685350477695465, 1.2381349802017212, 3.1622474193573, -2.217299699783325, -0.33688193559646606, -0.869175374507904, 0.9651769399642944, -0.10384314507246017, 1.196651816368103, 0.7280847430229187, -0.07798511534929276, 0.9269785284996033, -0.14559385180473328, -0.4257594645023346, 2.314722776412964, -0.8539488911628723, 0.17133307456970215, 1.444158673286438, -0.07447494566440582, 0.31324127316474915, -0.9017465114593506, 1.1186681985855103, -0.3190680146217346, -0.7811697721481323, 0.5655805468559265, 0.7114209532737732, -1.099843978881836, 1.0630205869674683, -2.8955600261688232, 0.05717724189162254, 0.5878623723983765, -1.6296406984329224, 0.9305452108383179, -0.7954601645469666, -0.9448911547660828, 0.6676238775253296, 0.5277508497238159, -0.790090024471283, 0.009404675103724003, -1.2005584239959717, 0.5060465335845947, -1.1969887018203735, -0.22556187212467194, 2.444088935852051, 3.1052799224853516, -2.480393409729004, -1.8193297386169434, -0.8816025853157043, 0.45267820358276367, 0.9895797371864319, 1.0245884656906128, -0.34087738394737244, -1.4405534267425537, 0.037039145827293396, 3.6621015071868896, 2.9769394397735596, 1.740228295326233], [-4.207798480987549, -0.5010876655578613, -0.8389315009117126, 0.25758296251296997, -0.812621533870697, 0.2551325559616089, 1.1741944551467896, -3.953083038330078, 0.4801143705844879, -1.1256704330444336, 1.1164805889129639, 1.394616723060608, -0.8386190533638, 0.314344197511673, -2.324350357055664, -3.402158260345459, 2.754570722579956, 2.241469621658325, -1.984614610671997, -0.3195625841617584, -2.8080897331237793, -1.5861457586288452, 0.6482297778129578, -1.0005077123641968, -0.16024047136306763, -1.1956968307495117, 0.8080727458000183, -3.815190553665161, 0.1933446228504181, 2.1654317378997803, -0.816042959690094, -0.18797080218791962, -2.0123934745788574, -0.25657743215560913, 0.023925796151161194, -0.8026779294013977, 3.1542420387268066, -0.36046090722084045, -1.2095235586166382, -1.000770092010498, -1.3105076551437378, 5.454629421234131, 0.7303464412689209, -0.29003217816352844, 1.1076083183288574, -0.12938787043094635, 2.1671292781829834, -0.503873348236084, -0.6841591596603394, 0.7625759243965149, 2.2076921463012695, -1.108147382736206, 1.3383536338806152, -0.5541013479232788, 0.662761926651001, -1.15109384059906, -2.431718587875366, -4.866928577423096, 1.9235788583755493, 1.4305063486099243, 2.4048261642456055, 0.7275676727294922, -1.4003709554672241, -2.1786422729492188, -1.87941575050354, 0.7936142683029175, 2.6273250579833984, -4.31231689453125, -4.412768363952637, -2.1954286098480225], [3.830855369567871, 4.767206192016602, -1.1887192726135254, -0.7322820425033569, 1.8115628957748413, -2.844771385192871, 0.9179369807243347, 4.555028438568115, -2.717475175857544, -0.344753623008728, -0.7553660273551941, 0.9344205856323242, 2.5997486114501953, 0.15050247311592102, 3.9755141735076904, 0.349427193403244, 2.1869990825653076, -5.938481330871582, 1.8056329488754272, 2.553149461746216, 5.682075023651123, -0.5638121962547302, -3.3934268951416016, 0.5544925928115845, 0.0602898932993412, 2.0830295085906982, -3.222137928009033, 3.7024848461151123, 0.6151604056358337, -3.8877336978912354, 0.37268123030662537, 1.7267577648162842, -2.890990972518921, -4.74066686630249, 0.8534201383590698, 4.409623146057129, -3.202223062515259, -6.201794624328613, 1.206344723701477, 0.10287922620773315, 1.2046594619750977, -7.394836902618408, -1.0496078729629517, -2.874797821044922, -0.022674933075904846, -1.4965358972549438, -1.0524039268493652, -1.415008306503296, -2.0817208290100098, -1.413975477218628, 0.4230940341949463, 1.2538745403289795, -5.4514970779418945, -3.849618911743164, 4.654946327209473, -3.1679065227508545, -1.184890866279602, -3.3220393657684326, 2.7159810066223145, -0.7965102791786194, 1.076021671295166, 0.04034070298075676, 1.2529032230377197, -3.709343910217285, -0.5182824730873108, -2.8389196395874023, -4.205617427825928, -0.5596434473991394, 1.6610902547836304, -1.266340732574463], [1.651092529296875, -2.7940845489501953, -0.2307230532169342, -0.6964564919471741, -0.5383946895599365, 0.6849470138549805, 0.7064050436019897, -2.115687131881714, 0.7373223304748535, 2.261188507080078, -1.5023280382156372, -3.21565318107605, 1.3297655582427979, -0.1997361034154892, -0.9545348882675171, 5.133127689361572, -2.5780060291290283, 2.7503907680511475, 3.942061424255371, -2.116070508956909, 0.21085423231124878, 0.5358412265777588, 1.2010676860809326, 1.793994426727295, -0.1835920661687851, -2.311187505722046, 2.5710506439208984, 0.005343136377632618, 1.0623247623443604, -1.583414912223816, -1.127914547920227, 0.8908963203430176, 5.498662948608398, 6.348265647888184, -3.9695234298706055, 0.46979066729545593, 0.3087109923362732, 6.107645034790039, -1.3743689060211182, 2.985246419906616, -1.1745630502700806, 2.193913221359253, -1.1952943801879883, 2.210618734359741, 1.1933205127716064, 0.6570667028427124, -0.08712498843669891, 2.9387266635894775, 2.8138680458068848, -1.4398589134216309, 0.6620316505432129, -0.5974631905555725, 3.5968921184539795, 2.0146830081939697, -1.9024388790130615, -0.5770303606987, 1.2339634895324707, 6.928549766540527, 0.7299018502235413, 0.9322817921638489, -1.7613438367843628, -2.5223398208618164, 0.568540096282959, 7.353027820587158, 4.457052230834961, 5.50214958190918, -4.257242202758789, 5.345624923706055, 3.095661163330078, -0.6185149550437927], [1.853201150894165, 2.8605926036834717, -3.1083202362060547, -2.3491904735565186, 0.2799081802368164, -0.7184165716171265, 3.760335683822632, -0.8600221872329712, -1.7642581462860107, 1.1255035400390625, -3.8464958667755127, 0.9549466967582703, 1.782517671585083, -2.6450886726379395, -1.3720656633377075, -2.581796169281006, 3.1644415855407715, -0.23804302513599396, 5.195147514343262, -1.9820054769515991, 4.41649055480957, 0.03323770686984062, -0.013127974234521389, 1.4872527122497559, -3.713958740234375, 0.49392473697662354, 2.6125998497009277, 2.188101291656494, 0.7101724743843079, -0.6704086661338806, 0.5916421413421631, 1.9489481449127197, -0.1888916939496994, 1.4658719301223755, -3.2657344341278076, 1.6426479816436768, -3.755030870437622, -0.0654449537396431, -0.1914578676223755, 3.917273998260498, -1.8226114511489868, 2.613057851791382, 3.697216749191284, -0.30162546038627625, 2.025249481201172, -4.935290813446045, -1.489071249961853, 2.093985080718994, -0.17840202152729034, -2.643558979034424, -3.0147316455841064, 2.357239246368408, 0.49340784549713135, -0.966406524181366, 1.0636193752288818, 3.530470132827759, -4.016963005065918, 1.1695910692214966, 1.8132588863372803, 1.6103647947311401, -1.3228557109832764, -4.346990585327148, 0.896966814994812, 1.9588738679885864, 2.176389694213867, 0.41029122471809387, -2.6609554290771484, -2.4018990993499756, -2.092390775680542, -0.9956164360046387], [1.7881988286972046, -3.2798492908477783, -2.4897961616516113, 0.610632598400116, -3.4898624420166016, -0.4244954586029053, 0.259764164686203, -1.3904352188110352, 1.5568982362747192, -0.7582938075065613, -1.608432650566101, -1.728999137878418, -2.072030782699585, 0.567903995513916, -0.814789354801178, 4.02329158782959, -0.8392016291618347, 2.407116651535034, -0.9404212832450867, -2.1815826892852783, 0.6103986501693726, 0.8403266668319702, 2.34596586227417, 0.4365026354789734, 0.7261878848075867, 0.14982236921787262, -1.22255277633667, 2.715029716491699, 1.4184486865997314, -0.7587935328483582, 0.3548254370689392, -2.80340838432312, -0.6418017745018005, 2.9125819206237793, -0.3380938768386841, -0.4965371787548065, -3.3868253231048584, 4.009191036224365, -1.122267246246338, 0.13904210925102234, -1.691942811012268, 1.6784694194793701, 0.5087229013442993, 2.0654854774475098, -1.1154167652130127, 0.7500993013381958, 0.08248994499444962, 0.3185369670391083, 0.7506406307220459, 0.6671319007873535, -0.37929508090019226, -0.08732277899980545, -0.7819994688034058, -0.5353596210479736, -1.1469851732254028, -1.0890589952468872, 3.937624931335449, 2.462494373321533, -2.104170083999634, 0.3883013129234314, -1.009652853012085, -0.719747006893158, 0.3376220166683197, 6.7359418869018555, 0.904128909111023, 2.337433338165283, 0.5257295966148376, 2.932631731033325, 2.492621421813965, 3.9054832458496094], [-0.6540004014968872, 1.3180570602416992, 3.384575128555298, -1.530313491821289, 3.192369222640991, -0.12172837555408478, -2.1606380939483643, 3.2674524784088135, 0.7492800951004028, 1.4135448932647705, -0.16964775323867798, 0.9008124470710754, 0.1856754720211029, -0.6909472346305847, 0.9828053712844849, -0.9729113578796387, 0.3204329013824463, -4.163283348083496, 0.12989579141139984, 3.0295658111572266, -0.34878891706466675, 0.9008526802062988, -0.8102843165397644, -0.25692853331565857, 0.06912879645824432, 1.929616093635559, -1.1510926485061646, 0.37581074237823486, -3.364993095397949, 1.3774850368499756, 0.4199943542480469, 0.6299583315849304, 0.046559467911720276, -2.829078435897827, 1.253426194190979, 0.6857831478118896, 0.21172480285167694, -1.4151924848556519, 2.9524548053741455, 0.4421845078468323, 2.3052279949188232, -1.4310503005981445, -1.5900208950042725, -1.270291805267334, -1.1218416690826416, 0.7793835997581482, -0.6360592842102051, -1.2638088464736938, -0.8574348092079163, 0.3817870020866394, -1.0419355630874634, 1.976405143737793, -1.2244548797607422, 2.0273685455322266, -0.49614039063453674, 1.1161978244781494, -1.8376102447509766, -0.9379143118858337, -0.2224225252866745, -0.8937153220176697, 0.34527072310447693, -0.51651930809021, 2.453784704208374, -4.623163223266602, 0.24719201028347015, -3.4010705947875977, 1.193671464920044, -0.4831473231315613, 0.5316989421844482, -1.6353787183761597]], "dtype": "float32"}}, {"a": [{"__nd__": [[-0.17162849009037018, 1.2784976959228516, -0.6607664227485657, -0.5751718282699585, -0.72486412525177], [1.157288670539856, 0.3844483196735382, -0.1941821575164795, 0.27456194162368774, -0.6327259540557861], [-0.6746059656143188, 0.6001091599464417, 1.3734056949615479, 1.0353906154632568, 0.7337484359741211], [-1.6955722570419312, 0.13741277158260345, 1.138066291809082, 1.3250458240509033, -0.04342273995280266], [0.42303115129470825, 0.20746980607509613, 0.39633655548095703, -0.75607830286026, 1.856050968170166], [0.6316961646080017, -0.67677241563797, 0.7646202445030212, 2.561624526977539, 0.2373804897069931], [0.22171905636787415, 1.3828219175338745, -1.9707797765731812, -0.2599867582321167, 0.20283381640911102], [-0.07164660841226578, 1.3552367687225342, -0.237032949924469, 0.3379349410533905, 0.31756362318992615], [-0.11775229871273041, -0.1328263282775879, -0.6333496570587158, -1.2007006406784058, 0.6026635766029358], [1.4407504796981812, -0.45094066858291626, 1.8026058673858643, 1.2882921695709229, 1.2960927486419678], [1.0279691219329834, -0.5045681595802307, 0.29151901602745056, -0.3737872838973999, 1.421331524848938], [0.027719099074602127, -0.32505249977111816, 0.3221772015094757, 0.7293627262115479, -0.3040401339530945], [0.8677602410316467, 1.603669285774231, -0.6033235788345337, -0.7755388021469116, 0.08511590212583542], [-0.5971119403839111, -0.13602544367313385, 1.2233885526657104, -0.026711970567703247, -0.1293352097272873], [-1.0502597093582153, 0.5494943261146545, -1.3047298192977905, -1.1540194749832153, -0.8805739879608154], [2.9550766944885254, 0.5658479928970337, 1.8363304138183594, -0.26471462845802307, -0.6002237796783447], [0.3027026057243347, 0.7949895858764648, -0.3317551910877228, -0.7688575387001038, -1.1579025983810425], [-1.4121030569076538, -0.711121678352356, -0.5040743947029114, 0.45100969076156616, -0.9222480654716492], [-1.7877520322799683, -1.8716447353363037, 0.6335983276367188, -0.35261228680610657, 1.6720939874649048], [0.8357957601547241, 2.499330520629883, -1.8810750246047974, 0.7077200412750244, -0.9040243029594421], [-0.023751720786094666, 1.4561524391174316, -1.1281830072402954, 0.19181013107299805, -0.22899024188518524], [0.5815374851226807, -2.779764175415039, -0.7210475206375122, -0.1265549510717392, -0.4882861375808716], [-1.5956854820251465, -0.8228452205657959, 0.14392633736133575, -0.3469680845737457, 0.4006751477718353], [0.8259192705154419, 0.4381815493106842, 1.070218563079834, -0.24592112004756927, 0.8690578937530518], [0.2601248621940613, 0.24273023009300232, -1.0581631660461426, -1.4820795059204102, -0.07779188454151154], [0.028099166229367256, -2.22739577293396, 0.1484454721212387, -0.3261137902736664, 0.8454635739326477], [1.0798178911209106, 0.6330601572990417, -1.3788654804229736, 0.38756412267684937, 0.7170735001564026], [-0.5059215426445007, -2.838160753250122, -0.9323052763938904, -0.5285984873771667, 1.643548846244812], [1.5579794645309448, 0.45654258131980896, 0.6659035086631775, -0.716901957988739, -0.3424127399921417], [0.5941520929336548, -0.6686295866966248, 0.6215258836746216, 0.15684981644153595, 1.148863434791565], [-0.07905993610620499, 0.3093400001525879, -0.4508725702762604, 2.076767683029175, 0.07032565772533417], [-0.7620398998260498, 0.9870074391365051, 0.980392575263977, -1.131699800491333, -1.7755684852600098], [0.10640763491392136, -0.38943779468536377, 2.1705687046051025, -1.5127605199813843, -2.9218363761901855], [-0.07622087001800537, 1.1677442789077759, 0.6510889530181885, 0.0612027570605278, 1.1769708395004272], [0.1341663897037506, 0.139614537358284, 2.0894546508789062, 0.05420253798365593, -0.42913922667503357], [0.37599992752075195, 1.6013234853744507, 0.0016915348824113607, -1.7012659311294556, -1.1956596374511719], [-0.12684805691242218, 0.8847345113754272, -0.5987668633460999, 1.0889842510223389, 1.5560452938079834], [-1.613531231880188, 1.0525540113449097, 0.7370023727416992, -0.3605855405330658, -0.3671831488609314], [-0.5821479558944702, -0.8684001564979553, -2.3127975463867188, -0.5900253653526306, 0.5418069362640381], [-0.5383180379867554, 0.117904894053936, -0.8456311225891113, -0.28406721353530884, 0.5057129859924316], [-0.5154343843460083, -0.14173932373523712, 0.29141101241111755, 0.035638228058815, -0.3265448808670044], [-0.3016679883003235, 0.625947892665863, 1.0636225938796997, 0.334006667137146, 1.0482268333435059], [-0.8932836055755615, 0.3730278015136719, 1.031123399734497, -0.5079004168510437, -0.5979512333869934], [0.16282778978347778, 0.8878944516181946, 2.4004364013671875, -0.049541763961315155, 1.4825215339660645], [-0.7494799494743347, 1.545656681060791, -0.13348568975925446, 0.5189496874809265, 0.7750687599182129], [0.1347118616104126, 1.2848379611968994, -0.5805422067642212, -0.9081106185913086, -1.347807765007019], [-1.8941129446029663, 0.6589617729187012, -1.2991721630096436, -0.5598427653312683, 0.9218122959136963], [-0.2056610882282257, -0.45399510860443115, 0.1884147822856903, 1.3767658472061157, -0.30954980850219727], [0.7139313220977783, -0.5614569187164307, 0.9527668952941895, 0.2746204733848572, -0.6789901256561279], [-0.2466614544391632, 0.7916702032089233, 0.2923731207847595, -0.7919594645500183, -1.0464959144592285], [0.07830387353897095, -0.2706845700740814, 0.33071503043174744, 0.6066237688064575, -0.5494917035102844], [-0.12007304280996323, -0.47222909331321716, 1.834817886352539, 0.2662340998649597, 1.239549160003662], [-0.6972723007202148, 0.4024598002433777, -0.5947337746620178, 0.40998974442481995, 1.119057536125183], [2.2335712909698486, -0.6359376311302185, 0.08524475991725922, 0.5760494470596313, -0.49856311082839966], [-0.2692430317401886, -1.5100758075714111, 0.4704952836036682, -0.12216924130916595, -0.6283802390098572], [0.8525404334068298, 0.6871466636657715, 1.6590927839279175, -0.8726388216018677, -1.634438157081604], [-0.7799005508422852, -0.0852317288517952, -1.4353104829788208, -0.5281007289886475, -2.544560194015503], [0.08449999243021011, 1.0559234619140625, 0.9647386074066162, 0.49472156167030334, 0.5277290940284729], [-0.09138689190149307, 1.196657657623291, -1.2425466775894165, 0.46622875332832336, 0.8356673121452332], [-1.117456316947937, 0.7551992535591125, 0.3751835823059082, 1.1813645362854004, -0.7814258933067322], [1.1963268518447876, -1.4442734718322754, 0.6244468688964844, 1.753121256828308, 0.013994796201586723], [0.9752752780914307, 0.8174716830253601, -0.2863039970397949, 0.2891013026237488, -0.14160390198230743], [0.7715225219726562, 0.5478342771530151, -0.384809672832489, 0.34428954124450684, -0.5106582045555115], [-0.9386147856712341, -0.7134591937065125, 0.007199686020612717, 0.4538368582725525, -0.654136061668396], [0.174038827419281, -0.70760577917099, -0.9588468074798584, -1.8345801830291748, 0.21313916146755219], [-0.4216783046722412, 2.543365955352783, -1.656944990158081, 0.6343509554862976, -1.363812804222107], [0.015073275193572044, -1.1120237112045288, 0.42139849066734314, -0.5015987753868103, 0.11051902920007706], [-0.7281412482261658, -0.9599704742431641, -0.12459871917963028, -0.36900565028190613, 0.21782952547073364], [-0.3157136142253876, -1.1951465606689453, 1.1469591856002808, 0.4240380525588989, 1.2177338600158691], [-0.8361260294914246, 0.3064369857311249, 1.529281735420227, 0.06046006828546524, -1.471706748008728], [1.4628715515136719, 0.6765002012252808, 0.9348184466362, -0.9332426190376282, 2.098400115966797], [-1.143651008605957, 0.4683045744895935, -0.3816016614437103, 0.07729741185903549, -0.7549511790275574], [0.8637759685516357, -0.9732505679130554, -0.751512885093689, -0.5486015677452087, -2.212235927581787], [1.1059406995773315, -0.19064272940158844, -0.2476022094488144, -0.11849963665008545, -0.9942542910575867], [1.565057635307312, 0.9936922788619995, 0.9498422741889954, 0.08168929815292358, 1.9896491765975952], [0.6759583950042725, -0.6359211802482605, -0.04950829967856407, -1.6767487525939941, -0.9967584609985352], [-2.0043299198150635, -1.1983840465545654, -0.17147238552570343, -0.8659492135047913, -1.3313274383544922], [-1.1608424186706543, 1.821290373802185, 1.977940320968628, -0.5278594493865967, 0.6868223547935486], [0.6531656980514526, -0.03755997493863106, -0.23817384243011475, -0.6461231708526611, 0.5716117024421692], [1.0021693706512451, 0.643673300743103, 0.4872702956199646, 1.4919028282165527, 2.560652494430542], [1.8847085237503052, 2.6719703674316406, 0.12388354539871216, -1.0924596786499023, 1.2515532970428467], [2.9834978580474854, 0.5562422275543213, 0.07731498032808304, -0.27402251958847046, 0.9813594222068787], [-0.40652021765708923, -0.4663005471229553, -0.5161243677139282, -0.36026039719581604, 0.708785355091095], [-1.0873584747314453, -1.1944122314453125, 0.13837213814258575, 1.0063649415969849, 0.5891053676605225], [-0.2500411868095398, -1.830833911895752, -0.4289683997631073, -0.9307256937026978, -0.6596512198448181], [0.012785682454705238, -1.097693920135498, -0.2493404746055603, -0.44673091173171997, 0.5243804454803467], [0.6041952967643738, -1.8518611192703247, 0.2161685675382614, 0.94570392370224, 1.3228212594985962], [0.6674779057502747, 0.502980649471283, 0.940382719039917, 0.39098060131073, -0.4235709011554718], [1.7502621412277222, -1.2759971618652344, 1.9326214790344238, -0.5428720116615295, -2.0901384353637695], [0.23040100932121277, -0.8758494257926941, 1.5225424766540527, -0.7873325347900391, -0.4941488206386566], [2.049973964691162, -0.9129617214202881, -1.5162731409072876, 0.7092010974884033, 0.4139823019504547], [-0.5683770775794983, -0.7550241947174072, 0.7046694159507751, 1.6251457929611206, -0.6894885897636414], [0.2345127910375595, 1.178589105606079, 0.06901099532842636, 0.00563439866527915, -0.5605970621109009], [0.06359805166721344, 0.21420370042324066, 0.7929838299751282, 0.8207124471664429, 0.14733447134494781], [0.7786684632301331, 0.2508655786514282, -0.8293875455856323, 0.041781309992074966, 0.02188635990023613], [1.7360939979553223, 1.5700592994689941, 1.8127484321594238, -0.5691061615943909, 0.04140078276395798], [0.3831256926059723, 0.14481998980045319, 0.8181056976318359, 0.49086833000183105, -0.3664219081401825], [-0.5376947522163391, -0.23967674374580383, -0.025419214740395546, 0.9702243804931641, 0.30035457015037537], [-0.44280102849006653, 0.30022889375686646, 0.5463424324989319, -0.1846698522567749, 1.4052371978759766], [-0.3980731964111328, 0.4177245497703552, 1.2263745069503784, 0.7044009566307068, 0.36917412281036377], [-1.3250391483306885, 0.2909969985485077, 1.6705360412597656, 0.07908221334218979, -0.9995347857475281], [1.2446327209472656, -0.43275323510169983, -0.46979770064353943, 0.6685886383056641, -1.275672435760498], [-0.7636069655418396, -1.6408745050430298, 0.4747658371925354, -1.3797593116760254, -1.2814265489578247], [0.8135299682617188, -0.487957239151001, 1.0649135112762451, 0.010951980948448181, 1.4740766286849976], [0.028832964599132538, -0.8981053829193115, 1.2091758251190186, 0.02711670845746994, 0.7720834016799927], [-0.9572664499282837, -1.5169497728347778, 0.3265705108642578, -2.192899703979492, 0.32853370904922485], [-0.6431291699409485, 0.6587575674057007, 0.7750431299209595, -0.6163629293441772, 0.24524343013763428], [0.15925879776477814, -0.9422639012336731, 0.6261498332023621, -0.3066960573196411, -1.5487455129623413], [2.175462007522583, 1.7868492603302002, 0.6258957386016846, 0.09195205569267273, -0.47678446769714355], [0.3913798928260803, 1.62698495388031, 1.769342064857483, -0.18058636784553528, -0.38280540704727173], [0.4796786904335022, -0.4475210905075073, 2.1316487789154053, -1.1115717887878418, -0.07017320394515991], [-0.9008559584617615, -0.7779624462127686, 1.6831344366073608, 0.16340205073356628, -0.9368159770965576], [0.34459981322288513, -1.2339295148849487, 0.19518034160137177, 0.02284115180373192, -0.5624023675918579], [0.051286730915308, 0.868741512298584, -1.4275693893432617, 1.7863388061523438, 0.26276952028274536], [0.5847827792167664, -0.9176508784294128, 0.5905128121376038, -0.840420126914978, 0.8611655831336975], [0.7217962741851807, -0.5136440396308899, 0.5162264108657837, -0.6372079849243164, 0.156077578663826], [-0.20105209946632385, 0.9893044233322144, 0.5845733284950256, 1.123634934425354, -0.6392644047737122], [-1.2469958066940308, 0.9710592031478882, 0.6062716841697693, 0.4712819457054138, -0.8830004334449768], [0.4509023129940033, -0.3175852596759796, -0.8305392861366272, 1.1580023765563965, -0.7883557677268982], [0.5918852090835571, -1.136749029159546, 0.29150155186653137, 0.5053035616874695, -1.1583620309829712], [1.141120195388794, -0.8655511736869812, 1.3098094463348389, 0.8151583075523376, -2.2390899658203125], [-1.1519649028778076, 0.9959818720817566, 0.6620765924453735, -1.2835668325424194, 2.206580400466919], [-0.5144052505493164, 0.23064136505126953, 0.7172099351882935, 0.04924989864230156, -0.800034761428833], [-1.0221095085144043, 0.03280061110854149, -1.0891457796096802, -0.5894609093666077, -0.7191561460494995], [-1.0122016668319702, 0.5755589604377747, 1.228587031364441, 1.614669680595398, 0.21322938799858093], [0.10297093540430069, 1.0854073762893677, 0.06885324418544769, 0.46506959199905396, 0.6306895613670349], [1.256087303161621, 0.5562450885772705, 0.49856919050216675, 0.32233941555023193, -1.0167251825332642], [-0.6789695620536804, -1.6248953342437744, 0.026398329064249992, 2.678589105606079, -1.005996823310852], [1.8145616054534912, -0.01083536259829998, 1.3397115468978882, -0.85224848985672, -0.6236382722854614], [0.8063470125198364, -0.05528324469923973, 0.5209538340568542, 0.6050150394439697, 0.38600969314575195], [-0.08898422122001648, -0.7664053440093994, -0.7010195851325989, 0.8367534875869751, 0.6454334855079651], [-0.2522052228450775, 0.6229559779167175, -0.041253674775362015, -0.6660207509994507, -0.20158381760120392], [1.3355897665023804, -1.255943775177002, -1.179411768913269, 0.5869283080101013, -0.9409881234169006], [-0.10293083637952805, -0.6202769875526428, -0.9654014110565186, -1.3903225660324097, 0.7640752196311951], [-0.9698899984359741, 1.0811798572540283, -0.8174850940704346, 0.5145548582077026, 0.2662140727043152], [0.05961250886321068, -2.048953056335449, -0.559214174747467, 1.35404634475708, 0.5004688501358032], [-0.4466150999069214, 1.1980841159820557, -0.535853385925293, 0.0170835442841053, 0.8709389567375183], [-0.18136900663375854, 1.363881230354309, 0.29710084199905396, 0.6785154938697815, -0.5295345783233643], [1.1888116598129272, 0.12589341402053833, 0.5194966793060303, -0.4869343340396881, 0.41111600399017334], [-1.0089445114135742, 1.7454127073287964, -0.06249494478106499, 0.3739672005176544, -0.5298413038253784], [0.3348644971847534, 0.07851542532444, -1.0269972085952759, -0.12253458052873611, -1.3541042804718018], [-1.741170883178711, 0.12662963569164276, 2.055429458618164, -0.6387712955474854, 0.3849145174026489], [0.13839305937290192, -0.5359675288200378, -0.9068666696548462, 0.36246687173843384, 0.46876904368400574], [0.520064651966095, 0.3325245678424835, 0.20248498022556305, 0.4607333540916443, -1.442003607749939], [-0.30480971932411194, 0.0699729397892952, -0.879141628742218, 0.22588735818862915, 1.513506293296814], [-1.4160372018814087, 0.41770854592323303, 0.7893017530441284, -0.2863849699497223, -0.367540180683136], [-0.9215466976165771, 0.0932888388633728, 0.12196679413318634, 1.3942131996154785, 0.6849267482757568], [0.023149490356445312, 0.6876637935638428, -0.7023837566375732, 0.6202259063720703, 0.05115177854895592], [-1.396415114402771, -0.49149417877197266, -0.5752555131912231, -1.5401215553283691, 0.9064483642578125], [-1.454354166984558, -0.8672552108764648, -0.8621423840522766, -0.07756543904542923, 0.24754072725772858], [0.1708187460899353, 1.2542002201080322, 0.39573371410369873, -2.255821466445923, -1.1421704292297363], [0.5263218879699707, -1.4006696939468384, -1.823232650756836, -1.1793391704559326, 0.5949997305870056], [1.0442166328430176, 2.0068862438201904, 0.055974844843149185, -0.846743106842041, 1.278908371925354], [0.23911255598068237, -0.15068240463733673, 0.6974781155586243, -0.8042997717857361, 0.26934370398521423], [0.4434935450553894, -0.28648045659065247, 1.174795150756836, -0.6282979249954224, -0.5095837712287903], [-0.20688146352767944, -0.6226661205291748, 0.3134383261203766, -0.8430365324020386, 0.5021853446960449], [0.37487277388572693, -0.9064611196517944, 0.06809768080711365, -0.30568835139274597, 0.0750441625714302], [-0.5878873467445374, 0.5888523459434509, -0.4244215488433838, 0.03663161024451256, 0.0008917080704122782], [-0.7064895033836365, -1.0352367162704468, 0.5866292119026184, 0.8019323945045471, -0.2634483277797699], [2.506788730621338, 0.7866722941398621, 0.6029161214828491, -0.7585570216178894, -0.95402991771698], [-0.8429686427116394, 0.278299480676651, -0.10518047213554382, -0.3338828384876251, -1.0897184610366821], [-1.4650981426239014, -0.25013667345046997, -0.8055180907249451, -0.1343764066696167, -0.09943517297506332], [0.00301776803098619, -0.8204025626182556, -0.9140759110450745, 0.7699959874153137, -1.9356684684753418], [0.06403030455112457, 1.4452283382415771, -0.5881057381629944, -0.5169155597686768, -0.847507655620575], [-0.7426352500915527, -0.7308349609375, -0.7894580364227295, 0.45195433497428894, 0.02054452709853649], [-0.7013583779335022, 0.2852894365787506, 1.098875880241394, 1.2493723630905151, 0.48146283626556396], [-0.6198412179946899, 0.803645670413971, 0.20863164961338043, -0.3541204035282135, 0.8935890197753906], [-0.16574764251708984, -0.2415117472410202, 1.3906692266464233, 0.8401854038238525, -0.3624667823314667], [0.030839046463370323, 0.017457209527492523, -0.36546432971954346, -0.2775923013687134, 0.41971516609191895], [1.9283151626586914, -0.49200382828712463, -0.39887771010398865, 1.0537968873977661, -0.30616286396980286], [1.3615225553512573, -0.32114627957344055, 0.4744814932346344, 0.12897810339927673, 0.3157082498073578], [1.0836708545684814, -0.9274153709411621, 0.5427866578102112, -0.3174571096897125, -1.2512606382369995], [-0.9873526692390442, 0.17509524524211884, -0.22363175451755524, 0.5757052302360535, 0.8494753837585449], [1.2123836278915405, -0.42503419518470764, -0.20810845494270325, -0.18985159695148468, 0.9908354878425598], [0.9004926085472107, 0.24118536710739136, -0.019294122233986855, -0.10581032931804657, -1.244515061378479], [-0.1628241240978241, 0.6476773023605347, 0.9862952828407288, 0.6907892823219299, 0.149681955575943], [1.263219952583313, -0.78850919008255, -0.9387016296386719, 0.4278436303138733, 1.455680012702942], [0.10846859216690063, 0.6277642846107483, -0.25967758893966675, 2.0498361587524414, -0.7922046780586243], [-1.2892470359802246, 1.995673656463623, -1.137631893157959, -0.27446117997169495, 2.1624698638916016], [1.3001840114593506, 0.9809030294418335, -1.362516164779663, 0.30570462346076965, 0.2890551686286926], [0.03315075859427452, -0.7145821452140808, 0.7069200873374939, -0.23925675451755524, -0.39522430300712585], [0.4056594669818878, 0.4134513735771179, -0.6369115114212036, -0.0457276813685894, 1.6191413402557373], [-1.3224421739578247, -1.8287005424499512, -0.5894514918327332, -0.8217947483062744, -0.1353272646665573], [-0.7174299359321594, 1.04021155834198, 0.0703546404838562, -1.3675469160079956, -0.8626819252967834], [0.1553627997636795, -2.0285849571228027, -0.7671658992767334, -0.4359357953071594, 0.21192650496959686], [2.324463129043579, -0.982081413269043, 0.9135600924491882, 0.6521519422531128, 0.3082154095172882], [-0.10614252835512161, -0.21846871078014374, -0.6249950528144836, 1.475048542022705, 0.48345696926116943], [0.21780544519424438, -0.5618629455566406, 0.7988858819007874, 0.6633786559104919, 0.4653269946575165], [0.3646814525127411, -0.7811547517776489, 1.5605114698410034, -0.8121236562728882, -0.9175571799278259], [-1.1774187088012695, 0.37913987040519714, 0.8306897282600403, 0.5003707408905029, 1.0839321613311768], [-0.5982825756072998, 0.017340799793601036, -0.45065799355506897, -0.8215250968933105, -0.2036503106355667], [-1.75957190990448, -0.24458864331245422, 0.6414627432823181, 0.9837482571601868, -1.0987091064453125], [0.32113736867904663, 0.22396235167980194, 0.6528681516647339, 0.1644914746284485, 1.9854844808578491], [0.31423497200012207, -0.024122120812535286, 2.274125337600708, 0.7160163521766663, -0.7948650121688843], [0.5147392153739929, -0.11067509651184082, 0.5203830003738403, -0.17510893940925598, -0.03354368731379509], [1.305949091911316, 1.976636528968811, 0.18932802975177765, 0.06818985939025879, 0.3208589255809784], [-1.0656347274780273, 1.001451015472412, 0.16636331379413605, 0.8419996500015259, -0.01373651809990406], [0.5808772444725037, 1.2234874963760376, 0.6597742438316345, -0.7287744283676147, 0.9761713743209839], [-0.0480734184384346, 0.21167050302028656, -0.7265540957450867, -0.8145537972450256, -0.25280338525772095], [1.6759963035583496, -2.7410356998443604, -0.6469911932945251, -0.9856457114219666, -0.2841363549232483], [0.20340920984745026, -1.1243001222610474, -1.5979126691818237, -0.9637130498886108, 0.7964500188827515], [1.2967230081558228, -2.1674232482910156, 2.5227010250091553, 0.5630172491073608, 0.2783026099205017], [0.9562310576438904, -0.23153893649578094, 1.059910535812378, -0.2607562243938446, 1.1003201007843018], [0.797677218914032, 1.5363523960113525, -1.6319559812545776, 0.15677489340305328, -1.0563184022903442], [-0.021365733817219734, -0.006067408714443445, 0.13916732370853424, 0.025018593296408653, 0.5623485445976257], [-0.6122426390647888, -0.7611029148101807, 0.07916226238012314, 0.0328635647892952, -0.44287288188934326], [0.5970597267150879, 0.7953922152519226, -0.2303909808397293, -0.5539200901985168, 0.1884717047214508], [0.24491667747497559, -0.8225634098052979, -0.8720148801803589, 1.4744573831558228, -0.12741681933403015], [-0.41017770767211914, 0.026976479217410088, 1.5339076519012451, -0.7795928120613098, 0.6677549481391907], [-0.639849841594696, -1.7505130767822266, 0.5179279446601868, -1.3106024265289307, 0.8805666565895081], [0.2738381624221802, -1.9927515983581543, 0.10685588419437408, 2.084176540374756, 0.02329726703464985], [0.39709144830703735, -1.8748161792755127, 0.9692773222923279, -0.26608583331108093, 2.42215895652771], [0.0009249607101082802, -0.8644571304321289, 0.6423763632774353, 0.6795375943183899, 0.2904941439628601], [2.421675682067871, 0.8175122141838074, -1.6431223154067993, 0.058457694947719574, 0.775877833366394], [-0.5785048007965088, -1.004239559173584, -1.3196414709091187, -0.3795555830001831, -1.7479990720748901], [0.05415736883878708, -0.8408272862434387, -0.3266289532184601, -0.6298422813415527, 1.6482731103897095], [-1.257912516593933, -1.3439222574234009, 0.7937746047973633, 1.121828556060791, 0.8047963380813599], [-1.3237924575805664, 1.6639313697814941, -0.4731328785419464, 0.5250673890113831, 0.120782271027565], [1.529627799987793, 0.24454955756664276, -1.5672720670700073, 0.5329959988594055, -0.3637707531452179], [1.0946756601333618, -0.18092375993728638, 0.9215230345726013, 0.24940353631973267, 0.3024260103702545], [-0.5781614780426025, -0.7958775758743286, 0.29851648211479187, -1.14766526222229, -0.84965580701828], [0.7739413976669312, 0.12653782963752747, -0.7938956618309021, -1.0816965103149414, -0.9142987728118896], [0.6109596490859985, 2.0487725734710693, -0.2821229100227356, -0.685681164264679, -0.4472410976886749], [0.16794714331626892, 1.0064488649368286, -0.4734496772289276, -0.12936259806156158, 0.30592480301856995], [-0.29458504915237427, -0.3819390535354614, 0.31791332364082336, -0.4449227750301361, -1.3325450420379639], [-0.7822510600090027, -1.6269508600234985, -0.9826334714889526, 1.3250476121902466, -0.1468190997838974], [1.535546898841858, 0.5451502799987793, -0.5757272839546204, -1.4542001485824585, -1.9637874364852905], [0.18001306056976318, 0.826239287853241, 0.30610817670822144, 0.6735277771949768, 2.1918983459472656], [-0.10827843099832535, -0.8442324995994568, 0.49999821186065674, 0.34998995065689087, -1.133997917175293], [-0.844835102558136, -1.5277138948440552, 0.7052788138389587, 1.3818837404251099, 0.407806932926178], [0.11369170248508453, 0.6784409880638123, 0.11702025681734085, 0.46264082193374634, -1.0637046098709106], [-0.5458843111991882, -0.6139209270477295, 0.22220481932163239, 1.31122887134552, 0.5572198629379272], [0.21586357057094574, 1.042919635772705, -0.6769289374351501, -0.941002607345581, -0.7951803207397461], [-1.1314806938171387, 0.11886654794216156, 1.9308936595916748, -1.845170021057129, 1.1910821199417114], [0.4673663079738617, 1.2489053010940552, -0.09454800188541412, -0.9758688807487488, -0.04763216897845268], [1.0249837636947632, 0.6114140152931213, 0.6655533313751221, -0.8238375186920166, 1.7090470790863037], [-2.2535560131073, 1.7347527742385864, 1.1577668190002441, 0.41863366961479187, -0.3975672125816345], [-0.39037707448005676, 0.7484282851219177, -0.3824805021286011, 1.5834819078445435, 0.7724348306655884], [1.2403355836868286, -1.0202031135559082, 1.3514643907546997, -0.725418746471405, 0.07406718283891678], [0.24905139207839966, -0.1234012097120285, 0.33799055218696594, -1.9427101612091064, -0.17895640432834625], [0.5403128266334534, 1.6927424669265747, 0.03205126151442528, -2.3831393718719482, 1.0513137578964233], [0.33230888843536377, 0.7647866010665894, 0.028768358752131462, -1.036913275718689, 1.029330849647522], [1.433996319770813, -1.2004035711288452, -0.9136975407600403, -0.22363731265068054, -0.45275023579597473], [0.7034947872161865, 0.10449289530515671, 0.40101367235183716, 0.3462008535861969, -0.557059109210968], [-0.24424131214618683, 1.2502461671829224, -0.6469113230705261, 0.9130620360374451, -1.7846617698669434], [2.7896244525909424, -1.4064559936523438, 1.0472034215927124, -0.6336106061935425, -1.740188717842102], [0.08407046645879745, 0.26273325085639954, -0.9039376974105835, 0.7254772782325745, 0.13830876350402832], [2.115020275115967, -0.5512458682060242, -0.5752403140068054, -0.6931821703910828, -0.06745809316635132], [-1.1851701736450195, 1.6418029069900513, 0.11520282179117203, -0.4841630458831787, -0.1400359719991684], [-0.8011086583137512, -0.13951532542705536, -0.9779167771339417, 0.9285526275634766, -1.5684254169464111], [-1.0060522556304932, 0.2423953115940094, -0.8924428224563599, -0.4218083620071411, 0.9947627782821655], [-0.7771812677383423, 0.6979913711547852, -0.8481202125549316, 0.563107430934906, 1.719827651977539], [-1.9578630924224854, 0.46589818596839905, -0.04511692374944687, -0.15544280409812927, -0.2412070631980896], [1.6879856586456299, 0.706822395324707, 0.2812787890434265, -0.30966171622276306, -0.6778947114944458], [-0.40216195583343506, -0.8603569269180298, -0.6599081754684448, -1.374875783920288, 1.8229180574417114], [0.03282465785741806, 0.8109237551689148, 1.572879433631897, 0.5138691663742065, -1.2026337385177612], [0.25724130868911743, -0.90613853931427, -0.7595270872116089, -1.1956372261047363, -0.46571534872055054], [2.382824420928955, 1.4290746450424194, 1.4385688304901123, -0.3751072883605957, 0.1257655918598175], [1.1243312358856201, -1.037234902381897, -0.10270500928163528, -1.7481805086135864, 0.6293731927871704], [-0.4398854374885559, 2.0942394733428955, 0.02374664694070816, 0.3366074562072754, 1.0650979280471802], [0.7820420265197754, 0.8377234935760498, -0.5766457319259644, 0.23233404755592346, -1.3326163291931152], [1.2433345317840576, -1.0619120597839355, 1.6523587703704834, -0.9141504764556885, 2.1842331886291504], [-1.2297987937927246, -1.6140986680984497, -0.015710750594735146, 0.7147035002708435, -1.7518712282180786], [0.11955533176660538, 1.0695172548294067, -0.8792767524719238, 0.44678613543510437, -0.5476720929145813], [-1.0584235191345215, -0.6022536158561707, 0.061381880193948746, -0.5070268511772156, 0.9058035016059875], [0.11414865404367447, 1.1561503410339355, -1.34553861618042, -0.008658679202198982, -1.1503496170043945], [1.5310804843902588, -0.24709416925907135, 0.0390520915389061, 1.446365475654602, 2.0808887481689453], [0.4275786280632019, -0.3045494258403778, 0.5413780808448792, 0.590460479259491, -0.6939994096755981], [0.23037146031856537, 0.13400836288928986, -1.7585959434509277, 0.3214133679866791, 0.506263017654419], [-2.282855272293091, 0.23239369690418243, 0.9333877563476562, 0.5453585982322693, 0.08297976106405258], [0.1414763182401657, -1.313088059425354, -0.20457762479782104, -0.12457893043756485, 0.691376268863678], [-2.5160961151123047, 0.8600751757621765, -1.0155384540557861, 1.5571484565734863, 1.2996639013290405], [-0.6101250648498535, -0.25783661007881165, 0.6403716206550598, -0.2886674404144287, 0.11557690799236298], [0.7884537577629089, -0.39584195613861084, 2.1059536933898926, -0.3547638952732086, -0.2488861232995987], [-0.3049336373806, -2.2457025051116943, 0.09679345786571503, -1.0218393802642822, 0.9970501661300659], [-2.1941325664520264, 0.24479101598262787, 0.17563508450984955, 0.04430912062525749, 0.6203107833862305], [-0.41212788224220276, 0.31533950567245483, -0.697767972946167, 0.053222157061100006, -0.03764813765883446], [0.2720431387424469, 0.2605486512184143, 0.6520373821258545, 0.22676755487918854, -0.8558917045593262], [0.6149486899375916, -1.4069675207138062, 1.1087816953659058, -1.4467356204986572, -0.08139414340257645], [-0.6798980236053467, 0.3643087148666382, -1.0999746322631836, 0.8480032682418823, 0.902816891670227], [0.44814109802246094, -0.5983178615570068, 1.2652970552444458, -2.290222644805908, -0.5182193517684937], [0.6731166839599609, -1.4095995426177979, 0.8245953321456909, 0.3669334948062897, 0.10903122276067734], [-1.2501764297485352, 0.5162225961685181, 0.4923362731933594, -1.6848284006118774, 0.42618972063064575], [-0.3064320683479309, 0.6296362280845642, -1.1604030132293701, 1.5279189348220825, 0.9951921105384827], [-0.4218784272670746, -0.11913231015205383, 0.9104896783828735, 0.7726881504058838, -0.6875438690185547], [0.2829291522502899, -1.3694950342178345, 0.8398795127868652, 0.0789419636130333, 1.0905671119689941], [-1.0705167055130005, -0.6796948909759521, 0.168301522731781, 0.521261990070343, 1.221272349357605], [0.8841172456741333, -0.20986071228981018, -0.042510028928518295, 0.9072263836860657, -2.0569584369659424], [-1.6813567876815796, -1.2519665956497192, -0.2636403441429138, 2.1564924716949463, 0.8800659775733948], [-1.183814287185669, 0.5829406976699829, 0.4502897560596466, 0.6113258004188538, -0.7111102342605591], [0.19370773434638977, -1.4590370655059814, 0.4672453999519348, -2.0267412662506104, -1.6675927639007568], [1.52262282371521, -0.5972530245780945, 0.08722609281539917, -0.7483882904052734, 1.1245743036270142], [-1.2683031558990479, -1.7351199388504028, 1.5989809036254883, -0.14191505312919617, -0.14509014785289764], [0.14633314311504364, -1.494223952293396, 0.17674915492534637, 0.4639672040939331, 0.159116730093956], [-1.3829067945480347, 0.9636760950088501, -0.31509995460510254, -0.6193835735321045, 0.496990442276001], [0.3660947382450104, -0.4816123843193054, 0.3211826682090759, 0.5380368828773499, -0.25981763005256653], [-0.8126224279403687, -0.3180932402610779, -1.2705439329147339, 1.0442705154418945, -0.16977472603321075], [1.2147235870361328, -0.45293059945106506, -1.5486966371536255, -0.07727169245481491, 0.9582291841506958], [-0.6294345259666443, 0.22214996814727783, -0.4525434076786041, -0.7551459670066833, 0.428694486618042], [1.2541836500167847, 0.5544618368148804, -0.5627328157424927, -0.5424679517745972, -1.3190163373947144]], "dtype": "float32"}, {"__nd__": [[0.5119665265083313, -0.3410091698169708, -0.5872074365615845, -0.08246198296546936, 0.13641862571239471, 0.8404362797737122, 0.7524388432502747, -0.5547254681587219, -1.3413878679275513, 0.09046822041273117, -0.8078956007957458, 0.2506684958934784, 0.8660669922828674, -0.20235037803649902, -1.3062411546707153, -1.3207294940948486, 1.65104079246521, 0.4969461262226105, 0.7701725959777832, 1.1997216939926147, 0.44140246510505676, 0.6398769617080688, 0.1803923398256302, 0.48345062136650085, -0.13738492131233215, -0.0276914332062006, -0.32744359970092773, -0.7736148834228516, 0.6202375292778015, 1.8198068141937256, -0.32955271005630493, 0.7392014265060425, 0.6945914030075073, 0.3256077468395233, 1.053986668586731, 0.6555535197257996, 0.1150631234049797, 0.9526247382164001, 2.9161012172698975, -1.3671337366104126, 0.477701336145401, -1.1426633596420288, -1.103137493133545, -0.512480616569519, 2.2648212909698486, 0.16872185468673706, -0.4999912679195404, 1.1208248138427734, -0.8060333728790283, 0.3821083903312683, 2.2244527339935303, 0.18492743372917175, -0.9870989918708801, -1.8771486282348633, 1.4516392946243286, 1.983285903930664, 0.5293824076652527, 0.07601228356361389, -0.14461363852024078, -0.5805380344390869, 1.2530827522277832, -1.6813944578170776, -1.106030821800232, -0.3118398189544678, 0.5096181631088257, -1.538102149963379, 0.5717968940734863, 0.35164764523506165, -0.5615503191947937, -0.9824197888374329, -0.2943500876426697, 0.42856302857398987, 0.7580634355545044, -1.1947143077850342, 1.3006207942962646, -0.19246962666511536, 1.1552397012710571, -0.4782801866531372, -1.4012144804000854, -0.27353692054748535, 0.45054692029953003, -1.2514629364013672, -1.1864088773727417, -0.07893742620944977, -0.41847243905067444, 0.1209290400147438, -0.5478143692016602, 1.8092749118804932, 1.1011557579040527, -0.23506027460098267, 0.4503209888935089, -0.4919900894165039, 0.5903893113136292, -0.7478792667388916, 0.7787787914276123, 0.512850284576416, 0.9179629683494568, 1.1350136995315552, -0.7362634539604187, 0.7647776007652283, 1.2043858766555786, -1.2891230583190918, 0.6531704068183899, 1.6064269542694092, 0.5283418893814087, 0.5312126874923706, 0.6601857542991638, 1.0089153051376343, -1.111224889755249, 0.5574202537536621, -1.2029072046279907, 0.11303368210792542, 0.6054698824882507, 1.6413817405700684, -0.40742698311805725, 1.572371244430542, 2.706833600997925, -0.4529956877231598, 0.16593332588672638, -0.22161482274532318, -0.41134852170944214, 1.146325707435608, 1.08100163936615, -1.1037156581878662, -0.10742627829313278, 2.3355095386505127, -0.557586669921875, 1.6827176809310913, -0.33124393224716187], [-0.6260356903076172, -0.2165108174085617, 0.6955517530441284, 2.3872756958007812, 1.5477664470672607, 1.1277931928634644, -0.8731723427772522, 0.545139729976654, -1.1091222763061523, -0.6262305974960327, -1.4908509254455566, -0.3322528004646301, 0.808828592300415, 0.9806056618690491, -1.656571388244629, -0.37055152654647827, 0.526995837688446, -1.4225237369537354, -1.57577383518219, -1.4090989828109741, -2.208921432495117, -0.32825180888175964, -1.3435182571411133, 0.03619585558772087, 0.3702099919319153, -0.4582313001155853, 0.03164796531200409, 1.105339765548706, 1.2825851440429688, 0.09459300339221954, -0.289406418800354, -1.0711458921432495, 1.2281718254089355, -0.3272903263568878, 1.0633829832077026, -0.6606231331825256, 1.1023627519607544, -0.4045476019382477, -0.29388076066970825, 1.1667826175689697, 1.6069369316101074, -1.0078344345092773, 0.3918784558773041, -0.6601285934448242, 0.30229923129081726, -0.45460885763168335, -1.7168076038360596, 0.08984342217445374, 0.15298043191432953, -1.4031920433044434, 1.4648634195327759, -1.7430874109268188, 0.019691472873091698, 0.20280545949935913, 0.5384746789932251, -0.11330167204141617, -1.2141278982162476, -1.4317659139633179, -0.1945686638355255, 1.1181650161743164, 1.0270768404006958, -0.1754566878080368, 0.3051924705505371, 0.5645555257797241, 1.6719874143600464, 1.3305104970932007, 0.10915374755859375, -1.4860103130340576, -0.836154580116272, 0.7018895745277405, -0.8794474601745605, 0.20585140585899353, 2.6858456134796143, -0.940981924533844, -1.3247194290161133, -0.46441295742988586, 0.26693853735923767, 1.343919038772583, 0.7809852361679077, 1.6635475158691406, -0.29366692900657654, -0.5776112675666809, 1.7105906009674072, 0.44298413395881653, 0.9843975305557251, 2.066657781600952, 2.1310136318206787, 1.1622635126113892, 1.1292638778686523, -0.7815852761268616, 0.7542884945869446, -2.0112392902374268, 2.1557013988494873, 1.1253620386123657, -0.3087822496891022, -0.9156357645988464, -0.5996026396751404, 0.0044768089428544044, 0.9555538892745972, 0.3966011703014374, -1.2694470882415771, 0.3037010431289673, -1.6710996627807617, 1.5908058881759644, -0.7983797788619995, -0.5081560015678406, 1.3603259325027466, 0.18422842025756836, -1.4162641763687134, -0.6796189546585083, 0.37208065390586853, -1.0433462858200073, -0.6876648664474487, -1.7454394102096558, 1.4641860723495483, -1.5760259628295898, 1.3845646381378174, 1.2094804048538208, -0.5858719944953918, -0.5678020119667053, -0.5133853554725647, 0.22018490731716156, 1.3108710050582886, 0.2291332483291626, -1.4535444974899292, -0.322079062461853, -0.6861879229545593, -0.8476234078407288, -0.48650065064430237], [0.4827735424041748, -1.3899402618408203, 1.213526725769043, 0.40075433254241943, 0.40159809589385986, -2.1320993900299072, -0.4531399607658386, 1.1488093137741089, 0.6624518632888794, 1.2453052997589111, 0.06836558133363724, -0.625501811504364, 1.0777918100357056, 1.163831353187561, 0.24947574734687805, -0.7047275304794312, 0.023497523739933968, -1.0389609336853027, 0.2772652804851532, 0.2582826614379883, 1.5547012090682983, 0.7420247197151184, -0.49960553646087646, -0.06814126670360565, 1.126029372215271, -0.6228066682815552, -0.13172172009944916, -0.0848972424864769, 0.13581861555576324, -1.56365966796875, -0.8261201977729797, 0.9158092737197876, -0.11086023598909378, -0.16390001773834229, 0.14968447387218475, -0.5058875679969788, -0.9630491733551025, 1.0615659952163696, -0.48885780572891235, -1.0562678575515747, 0.3629053831100464, -0.20037542283535004, 0.27726683020591736, -0.3421129882335663, 1.786101222038269, 0.9180881381034851, 0.0952954813838005, 0.18859589099884033, -1.2238986492156982, -0.7173619866371155, -0.15756411850452423, 0.701627790927887, -0.1012076586484909, 0.43879827857017517, 0.6483054161071777, 0.7668353915214539, -0.5784310102462769, 1.107059359550476, 0.9471214413642883, 0.5740694999694824, -1.6354877948760986, -0.3204743564128876, -1.714671015739441, -1.1830627918243408, 2.11002779006958, -0.08213461190462112, 1.5985825061798096, 0.5059781670570374, 0.3622380793094635, 0.6308767795562744, -0.1709098070859909, 0.330931693315506, -0.2835645377635956, -0.025154469534754753, 0.49522724747657776, -0.3475077450275421, 0.6308171153068542, -1.2319095134735107, -0.14553987979888916, -0.1126219630241394, -1.8540949821472168, 0.6558861136436462, -0.6512674689292908, -2.1543960571289062, 0.5813392400741577, 0.09352139383554459, -0.15124844014644623, 0.97112637758255, 1.0791115760803223, -1.4314312934875488, -0.7165842652320862, -0.05765945836901665, -0.3729611039161682, 0.5601179003715515, 1.1265006065368652, -1.415343165397644, 0.9198865294456482, 0.61420738697052, -0.36565902829170227, 1.1646732091903687, 1.262531042098999, 0.5155186057090759, -0.20216569304466248, -0.4124729037284851, 0.04495885968208313, 0.09029857069253922, 0.27492278814315796, 0.10103709250688553, -0.3630546033382416, -2.1750617027282715, 0.4427652060985565, 0.8580757975578308, 0.698253870010376, 1.3230704069137573, 0.7322523593902588, -2.175715684890747, -0.44490575790405273, -1.3820627927780151, -0.5932180881500244, 1.236637830734253, 0.8869841694831848, 0.8062052130699158, 0.31221428513526917, 0.1780674010515213, -0.2983843684196472, 0.5868769288063049, 2.5632708072662354, -2.621044158935547, -0.9203536510467529], [3.000507354736328, -1.053802251815796, 0.8225401639938354, -0.925727903842926, -0.8112219572067261, 1.4918359518051147, -0.2710873782634735, 0.5845131278038025, -0.1476263701915741, 1.2493090629577637, 0.6826428174972534, 1.3999367952346802, 0.22201816737651825, -1.8958772420883179, 0.9227407574653625, 1.278764009475708, 0.8715513348579407, -0.5492548942565918, -0.5856596827507019, 0.7426435351371765, -0.9648696184158325, -0.10756248235702515, -0.20396628975868225, 0.8173761367797852, 0.7344834208488464, -1.0383806228637695, -2.149099111557007, -0.10372675955295563, -0.5274129509925842, -2.0953147411346436, -1.6642711162567139, -1.6609880924224854, -0.0033242760691791773, -0.10066790878772736, -0.16223874688148499, 0.2530299723148346, -0.5778188705444336, 1.7367316484451294, -0.5339086055755615, 0.7484485507011414, -1.615462064743042, 0.7390338778495789, 1.8420754671096802, 0.9093125462532043, 1.3230955600738525, -0.17091217637062073, 1.1292191743850708, 0.6178029179573059, -0.15520203113555908, 2.330089807510376, -1.1626189947128296, -2.052666425704956, 1.474124550819397, 0.1271767020225525, 1.8246960639953613, -0.8997067213058472, 0.05964334309101105, 1.7310621738433838, -0.6238707304000854, 1.971086859703064, -2.660860300064087, -0.9687142968177795, 0.5984373688697815, -1.5285634994506836, -0.8146645426750183, 0.5369752645492554, -1.1641472578048706, 1.4735435247421265, 0.9294756054878235, -0.39488476514816284, 1.9507395029067993, -0.9078414440155029, 0.7511876821517944, 0.5074775815010071, -1.0998579263687134, -0.6611744165420532, 0.3216755986213684, -1.7551072835922241, -1.3773058652877808, -0.44217151403427124, -0.1409919261932373, 1.64348304271698, -1.1712299585342407, -1.177044153213501, -1.0121921300888062, 0.5594750046730042, -1.0837275981903076, -0.5617343187332153, -1.3791183233261108, 1.4840130805969238, 0.33969035744667053, -0.38631296157836914, -0.8688581585884094, 0.7286471128463745, -1.5820808410644531, 0.967357873916626, 0.5312114953994751, 1.5317654609680176, 0.10098429769277573, 0.7215375304222107, -1.1363433599472046, 0.5674366354942322, -1.9367270469665527, -1.0469825267791748, -0.8894463777542114, 1.5710886716842651, 2.603482246398926, 0.0584886372089386, 0.2049296498298645, -0.028604093939065933, 0.7975162267684937, 0.044923607259988785, 1.9516431093215942, 1.2836347818374634, 0.5364305973052979, -1.65299391746521, -0.8135010600090027, -0.45578959584236145, -0.9453971982002258, 0.7628412842750549, 0.10866422206163406, 1.0827052593231201, 0.2659534811973572, -0.2111130952835083, -0.7777950763702393, 0.9051910638809204, 0.8454826474189758, 0.8934845924377441, 0.4122229516506195], [0.4070097506046295, 1.5977556705474854, -0.40062466263771057, 0.5469838976860046, 0.5926687717437744, -0.2538911998271942, -0.8942570090293884, 0.7562117576599121, 0.5634002685546875, 0.12820179760456085, 0.6017341613769531, 2.606684684753418, 1.4600119590759277, 1.0171488523483276, 0.021688684821128845, -1.5375659465789795, -0.6451261639595032, -0.298488587141037, 1.6874945163726807, -0.456612765789032, 0.6096629500389099, -0.3316734731197357, -0.08886075019836426, -2.2858684062957764, 1.4506288766860962, 0.5423719882965088, -1.2283103466033936, -1.5189687013626099, -0.4318125247955322, -0.11211404949426651, -0.7120524048805237, -2.1467185020446777, -0.9455457329750061, 0.0818147286772728, -2.2325611114501953, -0.3148772716522217, -1.178026795387268, 0.032413139939308167, -1.1390680074691772, -1.6413099765777588, 0.19519126415252686, 1.3726377487182617, -0.8198365569114685, 0.8725457191467285, -0.9507761597633362, 0.5183660984039307, -0.2787562310695648, -0.10164954513311386, -0.7788172960281372, 0.37536999583244324, 0.7186170220375061, -0.6950765252113342, -0.7872619032859802, -2.298687219619751, -0.4753953516483307, 2.810988664627075, -1.3836150169372559, -0.35089540481567383, -1.0793849229812622, -0.4181406497955322, -1.4588727951049805, 0.47488731145858765, -1.2903944253921509, 1.6414064168930054, -0.4353378117084503, 1.3685181140899658, -0.7553597092628479, 0.5175815224647522, -0.29651108384132385, -0.6383114457130432, -0.19939503073692322, 0.21013692021369934, -0.016233334317803383, -1.7405896186828613, -0.2148873656988144, -2.1845736503601074, -1.197826862335205, 0.7188484072685242, 1.5448013544082642, 1.1286823749542236, -0.46278372406959534, 0.08852647989988327, -0.7780972123146057, 0.414861798286438, 0.2391570508480072, 0.06086636707186699, 1.4444833993911743, 0.09163236618041992, 1.5134084224700928, -2.3501317501068115, -0.4492420554161072, 3.0772411823272705, 1.0512866973876953, 0.9954615235328674, -1.4842702150344849, -0.7499669194221497, 0.9920496344566345, 0.5621527433395386, -0.04559868574142456, 0.44144466519355774, -0.8214824199676514, 0.47743335366249084, -0.9794548749923706, 0.43251919746398926, -0.4492342472076416, -0.6057223677635193, 1.2065093517303467, -0.7907512784004211, 1.885404109954834, 0.5300158262252808, -0.5040633678436279, 0.19945169985294342, 0.09374281764030457, -0.23879362642765045, -1.764232873916626, 0.479970246553421, -0.6029679179191589, -0.37609967589378357, -1.1062175035476685, -0.7892804145812988, 0.11808662116527557, -0.2981053590774536, -0.20493243634700775, 0.44884389638900757, -1.4955979585647583, -0.318021297454834, 0.5196241140365601, -0.4665968418121338, -1.0454217195510864], [-0.09097976982593536, 1.6830434799194336, 1.5307155847549438, -0.38678714632987976, 0.3415156304836273, 0.8324804306030273, 0.2816509008407593, -1.273903489112854, -1.1032880544662476, 0.18663017451763153, -0.9122896790504456, 2.4348549842834473, -1.3356095552444458, -1.0757735967636108, -1.6127967834472656, -0.6029538512229919, 0.7402480244636536, -0.5194109082221985, -1.5311334133148193, -1.5231627225875854, -0.14844638109207153, -0.46600914001464844, 2.530299186706543, -0.08901575952768326, -1.2580301761627197, -1.282995343208313, -0.44186094403266907, -0.8227037787437439, 0.5469246506690979, 0.6649038195610046, -2.5904314517974854, 0.3918735682964325, 0.8252003788948059, 0.5358167886734009, 1.3521502017974854, -0.9398733377456665, -0.07162576168775558, -0.48992541432380676, 1.6521010398864746, -0.6575464606285095, 0.9297587871551514, 0.3622087240219116, -1.587607979774475, -0.8042507767677307, 1.4808430671691895, 0.8798192143440247, -0.7578907608985901, -1.1622309684753418, 0.32286933064460754, 0.5205927491188049, 1.3337767124176025, 1.0920372009277344, 0.11275431513786316, 1.3119778633117676, 1.2354300022125244, 0.2829707860946655, 0.7547033429145813, 1.4913368225097656, -1.0270483493804932, 1.0410577058792114, 0.7827314734458923, 1.5787090063095093, 0.7816330790519714, -1.146161675453186, -0.9601203203201294, -1.7257269620895386, 1.2980693578720093, 0.01630682870745659, -1.5340681076049805, 0.44203752279281616, -0.11336886882781982, -0.23470064997673035, -0.48348551988601685, 1.9802883863449097, 0.8554654717445374, -0.6652917265892029, -0.8637492060661316, 1.2979466915130615, -1.1748863458633423, 1.169407606124878, 0.6982232928276062, 0.012816349975764751, 0.41349127888679504, -0.04184127226471901, -0.04555518180131912, 1.7103831768035889, 1.1939142942428589, -1.5749930143356323, 0.22174155712127686, -1.2379732131958008, -0.3041495978832245, 0.3432597815990448, 1.7210116386413574, 1.4820584058761597, -0.028641005977988243, 0.7608689069747925, -0.7793343663215637, 0.5459195375442505, -0.46347934007644653, 0.3674260973930359, -1.8779102563858032, 0.4841054081916809, 0.2166644036769867, 1.489730954170227, 0.997279167175293, 0.07698923349380493, -1.6162548065185547, -0.10492563992738724, 0.7575992941856384, 0.7064961791038513, -0.3328438401222229, -0.1332525759935379, 1.75726318359375, -0.7828774452209473, -0.26501643657684326, 0.4230960011482239, 0.5972732901573181, 0.19064369797706604, -0.9865936636924744, -0.3936363756656647, -0.16303199529647827, 1.686464548110962, -0.7359548211097717, 0.029422620311379433, -0.5609044432640076, 0.20644117891788483, 0.8244916796684265, 3.0141289234161377, -0.18022102117538452], [1.1271811723709106, -0.6388222575187683, -1.1941035985946655, -1.92143714427948, 1.2922438383102417, -0.6769401431083679, 0.04409699887037277, 1.170150876045227, -0.5663034915924072, 0.11175365000963211, -0.08941593021154404, 0.903836190700531, -1.5328552722930908, -2.113210678100586, -0.4211205542087555, 1.1869703531265259, 0.6803336143493652, -1.476720929145813, -0.7044722437858582, 0.18303290009498596, -0.17494632303714752, -0.5704624056816101, 0.23333978652954102, 0.06398461014032364, -1.7554997205734253, 0.8364248871803284, 0.7117527723312378, -0.2147877961397171, 1.8464558124542236, 1.2793052196502686, 0.38870784640312195, 0.09628625214099884, 0.7840737700462341, -0.4108388125896454, -0.2915356457233429, 0.6552501320838928, -0.4631749391555786, -0.45374664664268494, 0.2525395452976227, 0.9893544912338257, -1.4445234537124634, 2.3430159091949463, 0.8021520376205444, -0.35263434052467346, -1.2428088188171387, -2.0913984775543213, 0.46251463890075684, 0.576910674571991, -1.6442558765411377, 0.7401313781738281, -0.13673387467861176, 0.07291463017463684, -0.14927878975868225, -2.5056793689727783, 0.20961394906044006, -0.2886035442352295, -0.815482497215271, 0.9433607459068298, 0.8822686672210693, 0.08402641117572784, 0.6577305197715759, 0.6523729562759399, 0.9182493090629578, 0.5338952541351318, 2.068540573120117, -1.0941205024719238, -1.3262754678726196, 2.04547119140625, -0.63809734582901, -0.7231377959251404, -0.05730317160487175, 0.6906355619430542, 0.268253892660141, 0.3763575255870819, 1.7084015607833862, 1.0602149963378906, -1.0662667751312256, -0.9732702970504761, -2.80631947517395, 1.1936166286468506, 0.709959089756012, -2.128462314605713, 0.44363611936569214, 0.3176705241203308, -0.1262674629688263, 0.4169357120990753, -0.6866921186447144, -0.21667435765266418, -1.40505051612854, 1.1871768236160278, -0.08484050631523132, 1.4129421710968018, 0.3728221654891968, -0.30395498871803284, -0.9269679188728333, 0.5984229445457458, -1.3240389823913574, -1.489916443824768, -0.0731491893529892, 0.6038513779640198, -0.44025272130966187, 0.17567789554595947, 1.387271523475647, 0.937203586101532, -0.7680180668830872, -0.21761053800582886, -0.2528786361217499, -0.207303985953331, -0.015320412814617157, 0.006370313931256533, 0.6582921743392944, -0.2550448775291443, -0.6270353198051453, -0.10616011172533035, 0.6755943894386292, 0.012060957960784435, 0.3114534616470337, 0.006113057490438223, 0.3950909376144409, 0.13797776401042938, -1.4711675643920898, -0.25021588802337646, 1.2210794687271118, -0.04270123317837715, -0.8101130127906799, 1.5001718997955322, -1.0337111949920654, 0.3668682277202606, 0.03639323264360428], [1.9752414226531982, -1.262022852897644, 0.714642345905304, -1.94912850856781, 0.635572075843811, -1.6536616086959839, -0.8073720932006836, -0.17139273881912231, -1.004711389541626, 0.6882961392402649, -0.8442255258560181, 1.0433743000030518, 1.5270944833755493, 1.1163208484649658, 0.14840057492256165, -0.5319594740867615, 0.3062623143196106, 0.42373719811439514, 0.5364599823951721, 1.1884050369262695, 0.5883685946464539, -0.6872385740280151, -1.4073734283447266, -1.4003968238830566, -0.5042949914932251, 0.5031269192695618, -0.21637959778308868, 1.198053240776062, 0.43596628308296204, -0.7110251188278198, -0.18264201283454895, -0.2501718997955322, 1.2125173807144165, 2.053152084350586, -1.4718785285949707, -0.3436168432235718, -2.1984524726867676, 0.01484012696892023, -0.6425639390945435, -0.14428181946277618, 0.8273510932922363, -1.6235154867172241, -0.9532662034034729, -0.8777974247932434, 0.1965198516845703, -0.22229290008544922, -0.26247531175613403, 0.883567214012146, 0.7446737289428711, 1.1719855070114136, 0.006337564904242754, 0.34118908643722534, 1.232940435409546, 0.6451037526130676, -1.0658013820648193, 0.17141996324062347, 1.4159648418426514, 0.21485117077827454, 1.349062204360962, 0.5962492227554321, 0.975572407245636, 1.065678596496582, 1.1854408979415894, -0.06639564782381058, -1.1230355501174927, 1.3397001028060913, -0.5714750289916992, 0.6919997930526733, 1.498295783996582, -1.027336835861206, 1.1086734533309937, 0.18934181332588196, -0.28279611468315125, -0.1908571720123291, -0.8158742189407349, 0.037048693746328354, -1.2908320426940918, -0.03931033983826637, -0.5717867016792297, -0.155984565615654, 0.06882227957248688, 0.5149037837982178, -0.5447073578834534, -0.38853344321250916, -0.8097712397575378, -2.720449209213257, -0.5904365181922913, 1.6479835510253906, -0.10432886332273483, 0.287610799074173, -1.0513869524002075, 0.9909980297088623, 0.6081662774085999, 1.0355722904205322, -0.9356645941734314, -1.163379192352295, -0.13861557841300964, -0.8558570742607117, -1.5297447443008423, 2.3131842613220215, -2.818906784057617, -0.8897181153297424, 0.9697282910346985, 0.34223029017448425, 2.042464256286621, 0.9220032095909119, -0.8146061301231384, 0.38976508378982544, 0.8530066013336182, 0.5423375964164734, 0.17514175176620483, 1.1533633470535278, -0.469584196805954, 0.06439454108476639, -0.8179301023483276, -1.6499922275543213, 1.0593169927597046, -0.1609053909778595, -2.8142249584198, -0.3947817385196686, 0.26925674080848694, -0.255722314119339, -0.42765507102012634, -0.06252186745405197, -0.38295483589172363, -1.0919952392578125, 0.6440650224685669, 0.8132004737854004, -1.1871553659439087], [-1.6574971675872803, 0.13976624608039856, -1.4615718126296997, 0.7358225584030151, 0.22338946163654327, -0.0012031723745167255, -0.7270553112030029, 0.5746878385543823, -0.5571920871734619, -1.3609408140182495, 1.142352819442749, -1.4542406797409058, -1.578291893005371, -0.28976693749427795, -1.3126600980758667, 1.0765150785446167, -0.12429489195346832, 0.5774263143539429, 0.1017470508813858, 0.8098930716514587, 0.20837606489658356, -0.018590258434414864, -0.3321520686149597, -2.1398723125457764, 0.48866328597068787, 0.7813488841056824, -1.2342339754104614, -1.172638177871704, -0.06130053102970123, 2.3079097270965576, 0.3482621908187866, -0.43688562512397766, 0.533064603805542, 0.5887671113014221, -1.9017407894134521, 2.2090864181518555, -0.2843945324420929, -0.3039819598197937, 0.06202634423971176, -1.5319249629974365, -0.11088064312934875, 0.602033257484436, 1.038011908531189, -0.4719221889972687, 1.4238500595092773, 0.9211717844009399, -1.030492901802063, -0.1984112560749054, 0.23964138329029083, 0.014248008839786053, 0.35010799765586853, 0.801212728023529, -0.04317035153508186, 0.7132102251052856, 0.35469335317611694, 0.7657029032707214, 0.5957592129707336, 0.4358879625797272, -0.02226918190717697, -2.3902974128723145, 0.6306682229042053, -0.13863275945186615, 1.323798418045044, -2.138517379760742, -0.23362383246421814, -0.2825476825237274, -1.0864983797073364, 0.7622537016868591, 1.004042387008667, 0.8110374808311462, -0.12487770617008209, -0.829145073890686, -0.3998832106590271, 1.5232102870941162, -0.43092185258865356, 0.231485515832901, -0.29599475860595703, -1.3864542245864868, -0.5622119307518005, 0.8445307016372681, -0.7129372358322144, -0.14331510663032532, -1.4922964572906494, -0.566414475440979, -1.4433382749557495, 0.20798827707767487, -0.8077267408370972, -1.4555811882019043, -0.12522783875465393, -0.7078381180763245, 0.5662662982940674, -0.6422751545906067, -2.1629819869995117, -0.2879611849784851, 1.0330517292022705, 0.082854263484478, 0.016234902665019035, -0.8455173373222351, 1.7210540771484375, -0.24618855118751526, 1.2733961343765259, 0.1619412750005722, 0.24665004014968872, 0.08876202255487442, -0.3021589517593384, 1.2030634880065918, -0.9353222846984863, -1.0251083374023438, 0.5423631072044373, 1.9930126667022705, -1.383527398109436, 0.870400607585907, -0.06303244084119797, -0.9504647254943848, 1.619207739830017, 1.852577567100525, -0.16493764519691467, 2.538367986679077, 0.16841726005077362, 0.896725058555603, -1.030572772026062, -0.5111403465270996, 1.9610564708709717, -0.07494452595710754, 0.31374651193618774, 0.13390018045902252, -1.4621323347091675, 0.6349800825119019, 0.28171390295028687], [0.5713431239128113, 0.8735357522964478, 0.8353070616722107, -0.32454004883766174, 2.318964958190918, -0.461578905582428, 1.018535852432251, 0.7333428859710693, -1.294046401977539, -0.1346186101436615, -1.10732102394104, -1.1121374368667603, -0.8309732675552368, -0.3618152439594269, 0.8053330779075623, 1.0061850547790527, -0.5396401882171631, -0.07538051158189774, 2.0248727798461914, 0.19504866003990173, 1.4048917293548584, -0.11057095229625702, -1.4324687719345093, 0.09737668931484222, -0.4250737130641937, -2.0276777744293213, 0.2299174964427948, -0.09170100837945938, -0.4081699550151825, 0.899654746055603, -0.12352815270423889, -1.5511376857757568, -0.9085249900817871, -0.37614521384239197, 0.7229072451591492, 0.9606084823608398, -1.4924262762069702, 0.9406832456588745, -1.4386076927185059, 0.3688361346721649, 0.9751121997833252, -0.4641876518726349, 0.3294614553451538, -0.3814014792442322, 0.387771874666214, 0.5226668119430542, -0.2945263385772705, -0.33932363986968994, 0.7878212332725525, -2.047680377960205, 0.6655160784721375, 0.7176295518875122, -0.4420282244682312, 1.9100651741027832, 1.342183232307434, -0.3641749620437622, -0.0036811232566833496, 1.5072249174118042, 0.19687026739120483, 1.1102313995361328, -0.6549464464187622, -0.4167477786540985, 0.6553802490234375, 2.2982540130615234, -1.291229486465454, 0.1606789231300354, -0.5524691939353943, 1.7518723011016846, -1.8820654153823853, 0.6521478891372681, 0.5619552731513977, -0.8302265405654907, -0.7954087257385254, 0.4336183965206146, 1.1925896406173706, 0.6550911068916321, -0.5560703873634338, -0.08997392654418945, 2.025083541870117, -0.587911069393158, -0.6943009495735168, -0.19883137941360474, -2.2259933948516846, -1.3506946563720703, -1.786120057106018, 1.1585124731063843, -0.9514970183372498, 0.43836817145347595, 0.5656527280807495, 0.4967845678329468, 0.4598098695278168, -1.8837934732437134, -1.5266777276992798, -0.3760710656642914, -1.076988935470581, 0.11957335472106934, 1.9410133361816406, -1.1701092720031738, -2.0031161308288574, -0.6612981557846069, -1.0131666660308838, 0.4454427659511566, 0.7694387435913086, -0.6941543221473694, -1.5317201614379883, -0.5516027808189392, 0.1764775961637497, -0.6677939891815186, -1.1455707550048828, 1.0447741746902466, 0.9341850280761719, 0.4279092252254486, -0.6767828464508057, 1.2545901536941528, 1.3063064813613892, 0.7965812683105469, -0.4295298159122467, 0.4834982752799988, 0.5027896165847778, -2.566549777984619, -1.5698904991149902, 0.11671818047761917, 0.8250453472137451, 0.8091989755630493, 0.12508748471736908, 0.30886736512184143, 0.8674909472465515, -0.9855745434761047, 1.6232362985610962], [-0.5510329604148865, 0.35854431986808777, 0.40803220868110657, 0.9252477884292603, 0.35023757815361023, 0.15691687166690826, -0.5493399500846863, -0.8078000545501709, 0.6170021891593933, 0.6870907545089722, 0.6482323408126831, 0.16467368602752686, 0.9676951169967651, -0.45804667472839355, -0.41877463459968567, 0.7492649555206299, 0.6597285866737366, -0.1998044103384018, 1.4179686307907104, 0.12360317260026932, -0.6736463904380798, -0.6130515336990356, 0.5912838578224182, -1.3103768825531006, 1.655383825302124, 0.0820784792304039, -0.13360057771205902, 0.9700402617454529, 1.4847146272659302, -2.267946720123291, 1.0883833169937134, 0.6245678663253784, 0.3687913715839386, -0.5053975582122803, -1.1875802278518677, -2.262730836868286, 0.5769190192222595, 0.06747722625732422, -0.8232565522193909, -2.022488594055176, 0.7276518940925598, 0.5146355628967285, 0.3213854730129242, -1.0473157167434692, -0.7786331176757812, -0.13038860261440277, -1.4240878820419312, 0.2115785926580429, -0.09925144165754318, -0.10999809950590134, 0.866930365562439, 0.19516050815582275, 0.2171112596988678, 0.2743793725967407, -0.8882128596305847, 0.7391076683998108, 0.08629738539457321, -1.1036661863327026, -0.2231380045413971, 0.5127739310264587, -0.11029703170061111, 0.5291402339935303, -0.7054452300071716, 1.1106126308441162, 1.0235134363174438, -0.3865227699279785, 0.5053455233573914, 0.5188624858856201, -0.23979651927947998, -0.8445622324943542, -0.26483675837516785, -0.7998021245002747, -0.19304902851581573, 0.12871497869491577, 1.0609241724014282, -0.41572368144989014, 0.023344801738858223, 0.18902483582496643, 1.0203686952590942, -1.0117967128753662, 0.72927325963974, -0.5113233327865601, -0.09372548013925552, 1.031733751296997, -1.0618771314620972, -0.2674590051174164, 0.558066725730896, 1.5317424535751343, -1.6937286853790283, -0.46141958236694336, 0.7152523994445801, 0.11984443664550781, -1.9845421314239502, -0.45729732513427734, 1.7810101509094238, 0.8346683979034424, 0.23278464376926422, 2.103532075881958, -0.13739314675331116, -0.9880914092063904, -1.6783134937286377, -2.301771879196167, 1.3465232849121094, -0.11252657324075699, 1.1239912509918213, 0.44920024275779724, 0.49770572781562805, 1.0893442630767822, 1.936130166053772, 0.724280059337616, 0.9131993055343628, -0.0937415361404419, -2.813817024230957, 0.7888627052307129, -0.054530661553144455, -0.6481882929801941, 0.9731588959693909, -1.656720757484436, 0.25727832317352295, -0.8295077085494995, -1.3460222482681274, -0.5908911228179932, 0.20748111605644226, 3.231898069381714, -0.016267940402030945, 2.210541248321533, 0.46398481726646423, 0.9738987684249878, -0.6621740460395813], [1.1830600500106812, 1.0656391382217407, 0.14132113754749298, 1.8276747465133667, 0.26243120431900024, -3.115521192550659, 0.6520598530769348, 0.7992932796478271, 0.08228335529565811, 1.6190524101257324, 0.6073881387710571, 0.4949071705341339, 1.0209110975265503, 0.28607797622680664, 0.4012310206890106, -2.6422204971313477, -0.5932387113571167, -0.15924635529518127, -1.1098929643630981, 0.36979246139526367, -1.1589514017105103, 0.05290455371141434, 0.8061602711677551, 0.47161000967025757, -1.8901236057281494, -0.7601190805435181, -1.3446506261825562, -0.010451647453010082, -0.4297136962413788, 1.4775733947753906, 0.3617917001247406, -1.3056577444076538, 0.9587869048118591, 0.760983407497406, -0.11263778805732727, 0.5408878922462463, -0.5948359966278076, -1.2347912788391113, 1.612118124961853, 1.8414627313613892, -0.62513267993927, -2.8836328983306885, -0.07392049580812454, 1.6804920434951782, -0.33095094561576843, -2.473104953765869, -0.6538702845573425, 0.850204586982727, -2.6007354259490967, -0.16852249205112457, 0.11117852479219437, -0.9024484753608704, 0.15231512486934662, -1.0176037549972534, -0.8531923890113831, 1.128050684928894, 1.0971217155456543, 1.575609803199768, 0.11200761795043945, 0.5134915709495544, 0.2904863655567169, -0.21176044642925262, -1.0511305332183838, 0.1655408889055252, -0.3676989674568176, 0.6979227662086487, 0.48074671626091003, 0.31818684935569763, 1.1656508445739746, -2.1673364639282227, 0.011157692410051823, -1.6507588624954224, -0.0920070931315422, 0.5917826890945435, 0.4487183392047882, 1.6839559078216553, -1.393635869026184, 1.2032235860824585, -1.2772451639175415, -1.397996187210083, -0.3788001835346222, -1.0214265584945679, -1.3082414865493774, -0.978810727596283, -0.11437845975160599, 0.4910973906517029, 0.2528415322303772, 0.0814264789223671, -0.7230911254882812, 0.0755648985505104, 0.9233526587486267, 0.0909276157617569, -0.23190781474113464, 0.34602946043014526, -0.7683941721916199, 0.7710925936698914, -2.16961932182312, -0.9940323829650879, -0.6613122820854187, 1.3259458541870117, 0.838555097579956, 0.30571258068084717, -0.4240777790546417, -0.6569855809211731, 2.254025936126709, -0.06346200406551361, -0.6630913019180298, 0.04019901901483536, 1.0523426532745361, -1.0925124883651733, 0.14780011773109436, -0.409284383058548, 1.5830134153366089, 1.2143927812576294, 0.10560652613639832, 0.0074700540862977505, 0.6539742350578308, 0.8785983324050903, -0.15225307643413544, -0.21847592294216156, 1.1732341051101685, -0.2348756343126297, 0.5188558101654053, 0.8210399746894836, -2.1142892837524414, 1.5427218675613403, 1.4226770401000977, 0.6304202675819397, 0.8603007197380066], [0.18371456861495972, -1.2742072343826294, -1.0378235578536987, 0.164134219288826, -1.1314492225646973, -0.44224974513053894, -0.9116817116737366, 1.5980141162872314, -1.556541085243225, -0.20970936119556427, -0.9016669392585754, -0.6412529945373535, -0.37298673391342163, 0.6905486583709717, -1.1965711116790771, 0.0747230276465416, -1.0771191120147705, -0.04873950034379959, 0.6985619068145752, 1.2421836853027344, -0.7925047874450684, 0.9144870042800903, 0.9216455817222595, -0.6230852007865906, -0.42083659768104553, 0.08109608292579651, 0.8082602620124817, 0.9124152064323425, 0.853854775428772, 0.2166450023651123, 1.2093740701675415, -1.7773032188415527, 0.6176012754440308, -0.8445165157318115, 1.441670536994934, -1.1393084526062012, 0.32350999116897583, 1.6814860105514526, 2.3337535858154297, 0.6875145435333252, -0.1523447185754776, 0.1899762600660324, -0.23481908440589905, -1.498848557472229, -0.9763172268867493, 0.024570221081376076, -1.0755447149276733, 0.7925658226013184, -0.29606375098228455, -0.808957576751709, 1.5641688108444214, 0.2544596493244171, -0.7838963270187378, -0.66080242395401, -1.7215789556503296, 0.6155652403831482, 0.6239758729934692, 1.048664927482605, 2.425168037414551, -2.516242027282715, -1.4086190462112427, 0.4791853129863739, -1.1021811962127686, 0.235599547624588, 1.9373289346694946, -0.9901144504547119, 0.3493814170360565, -1.0599333047866821, -0.15810716152191162, -0.7825022339820862, 1.231429100036621, 1.5692458152770996, 0.5194679498672485, 0.09808181971311569, -0.40196746587753296, 0.2447679042816162, -0.2250427007675171, 0.2597186267375946, -0.18097013235092163, 2.2704501152038574, -0.26709526777267456, -0.27177608013153076, 0.8801657557487488, -1.2790356874465942, 0.7338409423828125, 0.21886727213859558, 0.5402235984802246, -0.08458825945854187, 0.08075882494449615, 1.5526623725891113, -0.7380093932151794, -0.5634359121322632, -1.0754191875457764, 2.362578868865967, -2.0893843173980713, 0.4465939402580261, -0.0573078878223896, -0.5770443081855774, 3.0570068359375, 0.47646135091781616, 0.5953778624534607, -0.4392937123775482, -0.6710630059242249, 0.3527749478816986, -0.7223508954048157, 0.8690871000289917, -0.2098824381828308, 1.5640276670455933, 0.5552412867546082, 0.19969169795513153, 0.35415253043174744, 0.6215009689331055, -0.6372499465942383, -1.0506305694580078, 1.3689907789230347, 0.8129442930221558, 0.4813535511493683, 0.9452430009841919, 1.7346962690353394, -1.1278904676437378, 0.24571798741817474, 0.2237725406885147, 0.7234673500061035, -0.5330080389976501, -0.9007011651992798, 0.9902960658073425, 0.2162981778383255, 0.3776509761810303, -0.32888686656951904], [-0.5763843655586243, -0.8119872808456421, 0.40324050188064575, -0.7464355230331421, -0.2658180296421051, 0.33170244097709656, -0.8393760323524475, -0.5267823338508606, -0.5248103737831116, 0.06505870819091797, -0.7783603668212891, 1.6120156049728394, 0.9929764270782471, 0.5562172532081604, 0.8772451877593994, 0.7419366240501404, -0.25047290325164795, 0.04437495023012161, -2.491379976272583, -2.0905070304870605, -1.5639865398406982, -0.6674454212188721, -0.7286175489425659, -1.4710112810134888, 1.4759397506713867, 1.2293500900268555, 0.3789375126361847, 1.096556544303894, 1.2248201370239258, 0.06894954293966293, -0.31359294056892395, 0.1480565071105957, -0.1708110123872757, -3.1510984897613525, 0.7620100378990173, -0.9324705600738525, 0.536088228225708, 0.6307669281959534, 0.4127254784107208, 0.31819507479667664, 0.8905851244926453, -0.4714314341545105, 1.5913063287734985, 0.5719612836837769, -1.5615782737731934, 1.3297652006149292, -0.8664933443069458, -1.805764079093933, 1.6076492071151733, 1.590915560722351, 2.1043221950531006, -0.2620691955089569, 0.44705942273139954, 1.9664585590362549, -0.7163776159286499, 0.9781586527824402, -1.3797645568847656, 0.7435895800590515, -0.9354809522628784, -0.7316541075706482, 0.31791672110557556, -0.7018937468528748, 0.03920987248420715, -0.8543592095375061, -0.052947141230106354, -1.0473288297653198, -0.20405207574367523, 0.06316941976547241, 0.8034617304801941, 0.5334306359291077, -0.36170724034309387, -0.15571710467338562, 0.6990460157394409, 0.4084450602531433, -0.3045169413089752, 0.9148136377334595, 1.7908287048339844, 0.49646493792533875, 1.671505331993103, 0.8361671566963196, 0.4213293790817261, 0.1089877113699913, -0.08048571646213531, 1.0196129083633423, 0.09384968876838684, 0.9246315956115723, 0.917483925819397, -1.774692177772522, 1.034907341003418, -1.4961740970611572, -0.6289069056510925, 0.6957727074623108, -0.47199875116348267, -0.9092060923576355, 2.843571901321411, 1.0133110284805298, 0.6031399369239807, -0.1663304567337036, 0.3241162598133087, 0.3339531719684601, 0.5549911260604858, 1.2527365684509277, -0.032859690487384796, -0.07106318324804306, -0.4486515522003174, 0.2584739029407501, -0.34073030948638916, 0.04157138988375664, 0.2090190201997757, -1.6302999258041382, 1.6846781969070435, -0.2387135922908783, 0.6313377022743225, 0.11225686967372894, 0.8262276649475098, 0.8802261352539062, -1.3828082084655762, -0.4322780966758728, -0.5317566990852356, -1.3869478702545166, 0.42474260926246643, -0.7266735434532166, -1.0457046031951904, -1.3213157653808594, 0.4237707555294037, -0.2945350408554077, 1.124251127243042, -0.33780011534690857, 1.2033698558807373], [-2.1006782054901123, 1.0355327129364014, 0.5451257824897766, -0.7402223944664001, 0.17139095067977905, 0.5560970306396484, 0.9397600889205933, -0.7647675275802612, 0.2682417929172516, -1.115634560585022, 0.5763900876045227, 0.20658519864082336, -0.6907444000244141, 2.210756778717041, -0.3526931703090668, -0.090107262134552, -0.46722695231437683, -0.26202404499053955, 0.8817844390869141, -1.7188847064971924, 0.07941454648971558, -0.5256628394126892, 0.8428842425346375, -1.4651060104370117, -1.4111233949661255, -1.1729257106781006, 1.781243920326233, -0.5356135964393616, -0.48455944657325745, 1.4500302076339722, -0.3723234236240387, -0.7899200320243835, -0.9853849411010742, -0.823716402053833, 1.1977431774139404, 2.294788122177124, 1.3905502557754517, 1.0172337293624878, -0.6109442114830017, 1.0576461553573608, 1.0671684741973877, 0.41909927129745483, 0.23422321677207947, 0.44011425971984863, -0.08900769054889679, -1.6657676696777344, 0.7440488338470459, 0.17321127653121948, 0.7143757939338684, 0.9275020360946655, 1.7443817853927612, 1.9476245641708374, 1.5114728212356567, -0.6326456665992737, -0.3683948516845703, -0.5210089087486267, -0.9868555665016174, -0.4989625811576843, 0.09033247083425522, -2.120914936065674, -0.13209810853004456, -0.524626612663269, 0.8201084733009338, 0.42973726987838745, -1.4505211114883423, 0.971060574054718, -1.3966941833496094, 1.0829870700836182, 0.6263531446456909, 0.7669613361358643, -0.21715570986270905, 0.12093113362789154, 1.7633917331695557, 1.111240029335022, -0.6865363717079163, 0.24319127202033997, -2.21694016456604, -1.8572335243225098, 0.4462026059627533, -0.2856130003929138, -0.8813212513923645, -1.3521658182144165, 0.23263594508171082, -0.8827307820320129, 1.0601935386657715, -1.133301854133606, -0.16020798683166504, 0.37457698583602905, -0.6439591646194458, 0.0687168762087822, -0.1411707103252411, -0.413239449262619, 0.4683047831058502, -0.9719882607460022, 0.23852957785129547, 0.14224088191986084, -0.40813711285591125, 0.283255934715271, -1.7763535976409912, -2.2078161239624023, 0.3033877909183502, -0.8657718896865845, -2.07124400138855, 0.7726315855979919, -0.3070167005062103, 0.18478147685527802, 1.2090009450912476, 0.055276595056056976, -0.49458202719688416, 0.6188051700592041, -1.035902976989746, -0.5190213918685913, -0.1266852617263794, -0.25700077414512634, 1.527705430984497, -0.8770127892494202, -0.9859040975570679, -0.4708513021469116, 0.16355332732200623, 2.211519956588745, -0.13958416879177094, -0.6591734886169434, 1.2377406358718872, -1.4249309301376343, -0.6511427164077759, 0.9720214009284973, -0.4282466769218445, -2.1749448776245117, 0.10260661691427231], [0.1672874391078949, -0.9182078838348389, 0.1461724191904068, 0.5684130787849426, 0.6897590756416321, 0.0840635895729065, 0.07905308157205582, -0.06097632274031639, -0.8583961129188538, 1.9650534391403198, -0.9698778390884399, 1.6408189535140991, -2.2644083499908447, 1.176198959350586, -0.7170286774635315, 1.3400061130523682, -0.14182034134864807, 0.7975306510925293, -2.0097131729125977, 1.8404854536056519, 1.2415695190429688, 0.9243375062942505, 0.29134413599967957, -1.0072710514068604, -0.8420075178146362, -0.9639951586723328, -0.5856955051422119, 1.1688625812530518, 1.1135588884353638, -1.0130609273910522, 1.490328311920166, 0.7445521950721741, 0.2753501534461975, 0.29155978560447693, -1.9836238622665405, -0.4635801613330841, 0.7884218096733093, -0.312371164560318, -2.529825448989868, -2.948664665222168, 1.6201685667037964, -1.9967116117477417, 0.29670631885528564, -0.6828747987747192, -0.2991095781326294, 2.0505011081695557, 0.9310728311538696, 1.8811602592468262, 0.7692403793334961, 0.8504799008369446, 0.5368487238883972, 1.2427445650100708, -0.1182432770729065, -0.23554939031600952, 0.7779181003570557, -1.424012303352356, -0.7056289911270142, 0.5532585382461548, 0.962380051612854, 0.9904161095619202, -0.5591309666633606, 0.040534839034080505, -0.03179020807147026, -0.8783993124961853, 1.9258781671524048, 0.45210346579551697, 0.04528506100177765, -0.20893435180187225, -1.8037234544754028, -0.3978930711746216, -1.2745434045791626, 0.7501814365386963, -1.0391454696655273, -0.47868454456329346, 1.1934860944747925, 0.08285333216190338, 1.043410062789917, -0.34034937620162964, 0.632935106754303, -0.8126769065856934, -0.06288540363311768, -1.4080361127853394, -0.14703358709812164, 0.410316526889801, -0.3570800721645355, 0.9127769470214844, -0.27884969115257263, 0.8323063254356384, 0.7980605363845825, 1.0309370756149292, 2.289027452468872, 0.8499523997306824, 0.5919808149337769, 1.574202060699463, -0.8600147366523743, 0.48321351408958435, -0.14338260889053345, 0.9662426114082336, -1.097947359085083, -0.6352387070655823, 2.747640609741211, 0.9467934966087341, -0.6506958603858948, -0.342043399810791, -0.18328550457954407, -1.4161491394042969, 0.04940978065133095, 0.5237346887588501, 0.9509177207946777, -0.27575793862342834, 0.8612273931503296, -2.122338056564331, 0.4364118278026581, 0.34849897027015686, 0.22047968208789825, -0.2376956045627594, 0.46357250213623047, -0.5258848667144775, -1.3797950744628906, 0.37691351771354675, -0.1609794944524765, -0.6427263021469116, 0.6231576800346375, -0.5681613683700562, -0.7008669972419739, 0.7238366007804871, 0.6374760866165161, -1.2209402322769165, -0.7867705821990967], [1.267347812652588, -0.7945492267608643, -1.353281021118164, -0.7527301907539368, -1.6685736179351807, -0.0564146414399147, -0.6076126098632812, 0.6513552069664001, -0.017815319821238518, 0.5003877878189087, -0.8065716028213501, 0.5575712323188782, -0.5012690424919128, -1.4265251159667969, -0.8073014616966248, 0.891633152961731, -0.9727075099945068, -0.47366753220558167, -0.8504149913787842, 0.3557986915111542, -2.2097935676574707, 0.25080204010009766, -1.5668389797210693, -1.3069549798965454, 0.5434668660163879, 1.8660780191421509, -0.875599205493927, -1.3697291612625122, 0.48084136843681335, -0.38034334778785706, -1.2187000513076782, 1.482875943183899, -0.05704319477081299, 0.5615586638450623, -1.3271915912628174, 0.5273405909538269, -2.517843246459961, 1.3150800466537476, 0.9086625576019287, 0.5771331787109375, 0.1776464581489563, 0.5935913324356079, -0.033966388553380966, -0.7201018333435059, -1.4680546522140503, 0.838399350643158, -0.5275331139564514, -1.1906917095184326, -0.4372568726539612, -1.0113706588745117, 0.4106511175632477, -0.008330069482326508, 0.7808740139007568, 0.4612121284008026, -0.6579842567443848, -0.06216081976890564, 0.07538328319787979, -0.330024391412735, 0.5780601501464844, 1.0102473497390747, -0.3566027581691742, 0.9532185792922974, -1.134913682937622, -2.0248825550079346, 0.4347775876522064, 0.6316066384315491, 0.4938378930091858, -0.5944741368293762, 0.12067101895809174, 0.13792595267295837, -0.5923739075660706, -1.2644524574279785, -1.1988227367401123, -0.45867931842803955, -0.6470798254013062, 0.6225394010543823, -0.05768585577607155, -1.017723560333252, -1.0105112791061401, -1.2340350151062012, -0.46836361289024353, -0.6440517902374268, -0.08591087907552719, -0.43451619148254395, 0.5961794853210449, -1.4773201942443848, -0.13138265907764435, 0.05323689058423042, -0.45806998014450073, -0.7886178493499756, -1.5795320272445679, -0.08936991542577744, -0.23140501976013184, -0.5111487507820129, -0.28900280594825745, -0.19559229910373688, -0.6517109274864197, -0.9981436729431152, 0.2466171234846115, 0.7919418215751648, 0.14981025457382202, 0.09295639395713806, 0.2536951005458832, 0.1851518452167511, -0.6296818256378174, 0.15229569375514984, -1.7182024717330933, 0.10138792544603348, -0.12567856907844543, -0.8515244722366333, -1.029881238937378, -1.2537729740142822, -0.21781161427497864, 0.6917120814323425, -2.0662646293640137, -0.7856259942054749, 0.03586903214454651, 0.07170803844928741, 2.0677735805511475, 0.9571065902709961, 1.951806664466858, 0.6560645699501038, -0.11204925924539566, 0.9319040775299072, 1.4994235038757324, 0.4765995740890503, 2.1338438987731934, -0.4945108890533447, 0.7483214139938354], [0.33492201566696167, 0.3950417637825012, -0.619960367679596, 0.0488487146794796, -1.4990965127944946, 0.23200666904449463, -0.9545156359672546, 0.11066773533821106, 0.6569967865943909, 1.6978814601898193, -0.952715277671814, -2.513960599899292, 1.1422784328460693, 0.19366581737995148, 0.5076369047164917, 0.004777798894792795, -1.1907109022140503, -0.36554908752441406, -0.075628362596035, -0.6066704988479614, -0.7824898958206177, 0.8363281488418579, -1.0472763776779175, -0.06033369153738022, -0.044845469295978546, -0.8173458576202393, -1.9807299375534058, -0.682612419128418, 1.8140202760696411, -0.13546453416347504, -1.0006892681121826, 1.2455127239227295, 0.02271987311542034, 0.721515953540802, 0.9876527786254883, 0.6520048379898071, -0.6118907928466797, -0.8677733540534973, 0.6351628303527832, -0.18683983385562897, 0.8567151427268982, -0.20680922269821167, 0.26828762888908386, 0.18352298438549042, 0.6058158874511719, 0.3489629924297333, -0.6250985860824585, 2.304529905319214, 0.43744754791259766, 0.8825333714485168, 1.3989615440368652, 1.2175273895263672, 0.24008142948150635, -0.01537618599832058, 0.13630326092243195, 1.921210527420044, -0.9272988438606262, -0.7085976004600525, -1.511093020439148, 0.8737525939941406, 0.257805198431015, 1.396554946899414, -0.9860548377037048, 0.40859073400497437, 0.28303688764572144, 0.5892619490623474, -0.6465756893157959, -1.694881558418274, 0.884726345539093, 0.24460192024707794, 0.4620184302330017, -1.9914226531982422, 0.514168918132782, -1.0324510335922241, 0.5248867273330688, 0.5624523758888245, 0.599061906337738, -0.779835045337677, -1.0147794485092163, 0.2215525209903717, -0.11747469007968903, 1.2908539772033691, -0.04659128189086914, -0.177467480301857, 0.4397106468677521, -1.0198744535446167, -0.3169015347957611, -0.1323869824409485, 1.4640319347381592, 0.10656995326280594, 0.7544528841972351, -0.43643537163734436, 0.5610314607620239, -0.7536991238594055, 1.1076931953430176, -0.8501060009002686, 0.901894748210907, -0.5220120549201965, 0.02485513500869274, 0.6612902283668518, 0.8385360240936279, -0.24319276213645935, 1.0365495681762695, 0.1348896473646164, -0.24650348722934723, 0.8652183413505554, 1.1599498987197876, 0.29781439900398254, -1.6525208950042725, -0.2109123021364212, -1.8027573823928833, 1.3609905242919922, -0.171657532453537, 0.3297393321990967, 0.24734202027320862, 1.0546361207962036, 0.07913628965616226, -0.5985296964645386, 0.40488943457603455, 1.0234910249710083, 0.19901609420776367, 0.004034723620861769, -0.05400450527667999, 0.15340960025787354, 0.12961441278457642, 0.8241385817527771, 1.6408069133758545, -0.7132188677787781, -0.00413900800049305], [1.1105540990829468, -1.6579368114471436, 1.7062667608261108, 0.31276634335517883, 0.649120569229126, -0.23972713947296143, 0.9604072570800781, 0.31676894426345825, 0.32114359736442566, -0.04774957522749901, 0.10920614004135132, -1.176469087600708, 0.15162071585655212, -1.3269531726837158, 0.06104010343551636, 1.367527723312378, -0.025714198127388954, 0.7265175580978394, -0.37137705087661743, -1.5634278059005737, 1.81144380569458, -0.21406738460063934, -0.500856876373291, -0.9187840223312378, 0.37640711665153503, -2.1270639896392822, -0.7086620926856995, -0.7669352889060974, 1.9196103811264038, -0.6060258150100708, 0.3096165060997009, 0.8219363689422607, 0.6263077855110168, -0.09805658459663391, -0.6651313304901123, 0.26346495747566223, 0.7770789265632629, -0.494157999753952, -0.7425267696380615, -1.862460970878601, -1.427234172821045, 1.6156649589538574, -0.09979097545146942, -1.6786150932312012, -2.6387836933135986, 1.0650848150253296, -0.05117354914546013, 0.4406343400478363, -0.9458887577056885, -0.8611024618148804, 0.2434866726398468, 0.16114725172519684, -0.6897989511489868, 0.893759548664093, -2.7253620624542236, -1.1644175052642822, -0.09329038858413696, 0.3145701587200165, -1.1579746007919312, 1.3473368883132935, 0.3546273112297058, -0.07854237407445908, -1.0158660411834717, -0.011781728826463223, -0.9504647254943848, -0.8694424033164978, 0.0028645286802202463, 0.05096679553389549, 0.04862949252128601, 0.8807674646377563, -1.3491201400756836, -0.19275301694869995, -0.29637840390205383, 0.5116562843322754, 1.1938529014587402, 0.14535482227802277, -1.128774881362915, 1.288415789604187, 1.3776553869247437, 0.5477112531661987, -0.8073118329048157, 0.38906198740005493, -0.31965094804763794, 0.7530121207237244, -1.382812738418579, 0.40048593282699585, -0.5773100852966309, 1.5841617584228516, -0.9517303705215454, -0.5791067481040955, 0.48191845417022705, 0.5331690907478333, -0.5909342169761658, 0.39793726801872253, 0.35279911756515503, 1.5025861263275146, -1.5263186693191528, 1.7979457378387451, -0.7402628660202026, 0.15383411943912506, -0.013941860757768154, -0.5912305116653442, -1.0619735717773438, 0.064898282289505, -0.3390110731124878, 0.8965176939964294, 0.44640830159187317, 0.5199178457260132, -0.8594598174095154, 0.6185643672943115, -0.7011789083480835, -0.016106899827718735, -0.3326210081577301, -0.8050456643104553, 0.7693578004837036, 0.156757190823555, -0.7479633688926697, 0.7605509161949158, 1.8646576404571533, 0.3221748471260071, -0.6284210085868835, 1.5793460607528687, -0.21422706544399261, 0.4838535487651825, -0.4472949206829071, 0.18123313784599304, 0.28487980365753174, -0.6330478191375732, 1.0034157037734985], [-1.160680890083313, -0.9848334193229675, -0.6720942258834839, 0.8017789721488953, -0.9545570015907288, -1.5491948127746582, -0.19506651163101196, 0.4510888159275055, -0.3252376914024353, -0.8077137470245361, -0.03413591906428337, -0.1492447853088379, -0.2593025863170624, -0.3682580888271332, -1.4044123888015747, -0.9974476099014282, 1.8045117855072021, 0.39894893765449524, 1.0164952278137207, 2.4517557621002197, 1.0572309494018555, -0.1746533215045929, -0.3999168574810028, 0.7595928907394409, -0.2737162113189697, 0.4398118257522583, -0.49549445509910583, -1.3056674003601074, 0.5231224894523621, 1.4181355237960815, -0.22802066802978516, 0.8992737531661987, 0.5957815051078796, -0.032795898616313934, 0.543221652507782, -0.731609582901001, 0.7306416630744934, 1.0468614101409912, -0.2919389307498932, 0.35757002234458923, 0.10423069447278976, -0.647487998008728, 0.26487550139427185, 0.826731264591217, -1.2613554000854492, -0.12636475265026093, 0.9664445519447327, -0.042605236172676086, -0.0645066574215889, 0.5391862392425537, -0.561251163482666, 0.6905689239501953, 0.6660923361778259, -0.4617268145084381, -0.34766656160354614, 0.47536802291870117, -0.8014094829559326, 0.11427396535873413, -0.5704710483551025, -1.2048300504684448, -1.2326149940490723, 0.5672582983970642, -0.04056814685463905, 0.6517552733421326, 0.2502635717391968, 0.42673197388648987, 0.6009832620620728, 0.10673439502716064, -1.9470906257629395, 0.7070484757423401, -0.4869197905063629, 0.0983426421880722, -0.9904476404190063, -2.062448024749756, -0.4994135797023773, -0.931807816028595, 0.039123643189668655, -0.5325700044631958, 0.2652982771396637, 0.3685491383075714, -1.8132905960083008, 0.6670216917991638, 1.077457070350647, 1.3082984685897827, -0.10992984473705292, -2.0523834228515625, 1.1162413358688354, 1.7256429195404053, -0.15409785509109497, -0.08319273591041565, -0.15622247755527496, -0.2392248511314392, 0.019747238606214523, -0.0378178209066391, 0.4770621955394745, -1.1414698362350464, -0.7549694776535034, 1.1431317329406738, 0.7652161121368408, 0.9273380637168884, -0.43965235352516174, -0.6930572986602783, 1.219078540802002, 1.352603793144226, -0.24838900566101074, -0.9044093489646912, 0.42706313729286194, -0.8337897658348083, -0.39540448784828186, -1.3248059749603271, -0.5685157179832458, 1.6593047380447388, 0.7042867541313171, 1.4157840013504028, 0.3495613634586334, 0.877639889717102, -1.6860277652740479, -0.4343397319316864, -0.35221216082572937, -1.4913380146026611, -0.16798359155654907, 0.6753144860267639, 1.5107358694076538, 1.2209967374801636, 0.011340069584548473, -0.32134297490119934, 0.5360918641090393, -0.7948428392410278, 1.386965274810791], [0.6692714691162109, -0.3242131173610687, 0.20513218641281128, -0.3279508948326111, -1.6838873624801636, 0.057726915925741196, -1.9516233205795288, -1.088447093963623, 1.8719578981399536, -0.9224661588668823, 0.36481285095214844, 0.8101280927658081, 0.38364776968955994, -0.06414700299501419, -0.8675851225852966, 0.6153038144111633, 0.6167445778846741, -0.05020046606659889, 1.3851840496063232, -0.3648250699043274, 0.9199574589729309, -0.6108545660972595, -0.6852962970733643, 1.0652879476547241, -1.0294550657272339, -0.02315955050289631, -1.7525614500045776, 0.7437291145324707, 0.8449780344963074, 0.5933288335800171, -1.1138585805892944, 1.3658747673034668, 0.36594390869140625, 0.11368536204099655, 0.5384340286254883, -0.7749344110488892, 2.7213845252990723, -0.7684020400047302, -1.7311657667160034, 0.005444738082587719, -0.57930588722229, 0.9137087464332581, -0.5296972990036011, -0.5270035862922668, -0.3878040611743927, 0.7454398274421692, -2.218250274658203, 0.04100776091217995, 1.0800764560699463, 0.7926453351974487, 1.5695621967315674, -0.66017085313797, -0.5695919394493103, -2.5844128131866455, -0.3037504553794861, -1.228292465209961, 0.3721916079521179, 1.2870383262634277, 0.403082013130188, 1.1257247924804688, -1.0252532958984375, 1.6334643363952637, -0.07491426169872284, 0.5944029688835144, -2.5772039890289307, 0.5322965979576111, -1.1155831813812256, -0.5283209681510925, -1.4959665536880493, -0.48826470971107483, -0.03993840515613556, -0.502081036567688, -1.3489807844161987, 0.20937518775463104, 0.131750226020813, -0.08272391557693481, 2.175525426864624, 0.6745614409446716, 0.1328246295452118, -1.096206545829773, 0.3073423504829407, -0.5101473927497864, -0.176773339509964, 1.1123310327529907, -0.18485738337039948, 0.9487352967262268, 0.875606894493103, 0.13854476809501648, 1.421832799911499, -2.326017141342163, 0.4905858635902405, -0.40889906883239746, 0.5275125503540039, 0.5630652904510498, 1.4709731340408325, -0.2018873244524002, 0.3343261480331421, 1.4761191606521606, 0.5892006158828735, 0.030468786135315895, -0.1953865885734558, -1.1366827487945557, 0.8921458721160889, -1.7572193145751953, -0.03414594754576683, 0.8144640922546387, -0.14818991720676422, -0.0036223968490958214, -2.6159417629241943, 0.806221604347229, -0.45021122694015503, 0.24600663781166077, 0.7040901184082031, 1.0156172513961792, -0.40302392840385437, -0.34012192487716675, 0.3365830183029175, -0.2511720657348633, -0.6262425780296326, -0.7919566631317139, 2.0522351264953613, -0.1349789798259735, 1.7724798917770386, -1.1215816736221313, -0.15280887484550476, 0.3822915554046631, 1.477057933807373, 1.5192573070526123, -1.7407078742980957], [-0.827875554561615, -1.0300548076629639, 0.9516847133636475, 0.5010849237442017, 0.95270174741745, -0.030853012576699257, -0.3708381950855255, -0.33614879846572876, 0.4142642021179199, 1.3563017845153809, 0.8922345042228699, 0.14780057966709137, -0.21359924972057343, 0.618706464767456, 0.5196560621261597, 0.6162858605384827, -1.5252916812896729, 0.6563467383384705, 0.9698220491409302, -1.0740244388580322, 0.06855536997318268, -0.12634588778018951, -1.0508909225463867, 0.1848335713148117, 0.3133464455604553, -0.3417573571205139, 1.470526099205017, -0.6377695798873901, 0.17387335002422333, 0.055146731436252594, -1.399880051612854, 0.22388599812984467, 0.5703902840614319, -0.5766692757606506, -1.3813115358352661, -0.5787039399147034, -0.22422289848327637, -0.7425392270088196, 0.48588383197784424, -2.057035207748413, -1.3254351615905762, -0.5706570744514465, -0.4132287800312042, -0.06358177214860916, 0.9455651640892029, -0.3736264705657959, 1.1385776996612549, -1.2271490097045898, 1.1209193468093872, 0.07891464978456497, -0.5410585403442383, 0.535662055015564, -0.5243312120437622, 1.4681943655014038, -0.8230220079421997, 0.9343242049217224, -1.5807206630706787, 1.289716124534607, 1.8853609561920166, -0.7641071081161499, 1.5504447221755981, 1.0301733016967773, -0.3228878080844879, 0.6290431618690491, 0.6190651059150696, 0.9645186066627502, 2.257413864135742, -0.4011322259902954, -1.7925808429718018, 0.9878020286560059, -0.5732483863830566, 0.1603117138147354, 0.30767393112182617, 0.05211513862013817, 0.17791922390460968, -0.40661194920539856, 0.033309418708086014, -1.269696593284607, -1.4710744619369507, 0.49035850167274475, -1.809747338294983, 0.6954678297042847, 0.791958212852478, -0.2625954747200012, 0.5407546758651733, 0.27972841262817383, -1.029713749885559, -1.0550264120101929, 0.6134015321731567, 1.0854828357696533, 0.4724254608154297, -0.48637688159942627, 0.4552544951438904, 0.7635160684585571, -0.2860955595970154, -0.4362384080886841, -0.2274596095085144, -1.4067517518997192, -0.8710035085678101, 0.41044944524765015, 0.89728844165802, -0.7073608040809631, -0.705705463886261, 1.1695696115493774, 2.0804338455200195, -1.3446907997131348, -0.7503310441970825, 1.0034594535827637, 1.5030933618545532, 0.1305871456861496, 0.748549222946167, -0.48538461327552795, 1.979837417602539, -0.1936754733324051, 1.5574932098388672, -0.6937536001205444, -0.3310759365558624, 0.6131726503372192, 0.7690576314926147, -0.10380514711141586, -0.3952641189098358, 1.2173006534576416, -0.02463497780263424, -0.7758702635765076, 1.0386364459991455, -1.4979721307754517, 0.6809138655662537, 0.35422465205192566, -0.17583727836608887], [0.8121356964111328, -0.3350954055786133, -0.5211664438247681, 0.7958483099937439, -0.3585728704929352, -0.17416003346443176, -0.616263210773468, -0.7833003401756287, -0.5431350469589233, 1.790530800819397, 0.40987300872802734, 0.533564567565918, -0.4959242343902588, 1.236812710762024, -0.6739679574966431, -0.15805448591709137, 1.0656715631484985, -0.9619928002357483, 1.3818753957748413, -0.7281749248504639, 0.5383962988853455, -0.7500052452087402, -0.2585190534591675, 2.1073503494262695, -1.2144757509231567, 0.4375951886177063, 1.3611260652542114, -0.07354927062988281, 0.8415346741676331, -0.5920682549476624, 0.5917122960090637, -0.45895546674728394, -1.8656518459320068, -0.8386276960372925, 1.6022015810012817, -1.4896650314331055, 0.5381020307540894, 0.5468969941139221, 0.4255405366420746, 0.1730581670999527, 1.0273882150650024, 1.4541231393814087, 1.1227730512619019, -0.0952439084649086, -0.6298124194145203, -0.8768530488014221, 1.0802067518234253, -0.4350157082080841, 0.9283572435379028, -1.629325032234192, 0.06105279549956322, -1.0714186429977417, 0.27415817975997925, -1.897996187210083, 0.08868744969367981, 0.5709041953086853, 0.3155061900615692, 0.850893497467041, 0.019098740071058273, 0.7296935319900513, 2.374518394470215, -0.5910376906394958, -1.578305959701538, 0.5572794079780579, 0.7458235025405884, 0.9698984026908875, -0.1789257675409317, 0.3227025866508484, -1.5570085048675537, 1.848512887954712, 0.7123649716377258, 0.4121363162994385, 0.9129884243011475, -0.38198983669281006, -0.723361611366272, -0.4574545621871948, -0.5013859272003174, -0.19413134455680847, 1.4200983047485352, 0.8269625306129456, 0.4087563753128052, 0.3258741796016693, 1.7894474267959595, -0.06714481115341187, 0.1948239952325821, -0.42465779185295105, -0.27703791856765747, -0.41851377487182617, 0.8725090026855469, -1.2581709623336792, 1.5448665618896484, 1.7171721458435059, 0.9512036442756653, 0.6685527563095093, 0.2077377736568451, 1.1647834777832031, 0.461524099111557, 0.3379749357700348, -0.43076249957084656, 0.28037890791893005, -1.256911039352417, 0.7832846641540527, -0.10672986507415771, -0.48439788818359375, 2.1082065105438232, 1.8026947975158691, 0.5568943619728088, -0.47036436200141907, 0.7621957063674927, -0.832997739315033, -0.7988090515136719, -0.07482659816741943, 0.6440503001213074, 0.21730086207389832, 0.4877384305000305, -1.145735740661621, 1.3264585733413696, -0.188015878200531, -1.6347390413284302, 0.8151875138282776, 0.4576263725757599, -0.16861753165721893, 1.12104070186615, 1.0787022113800049, -0.9331529140472412, -0.061958737671375275, -0.3858802020549774, -1.7879778146743774, 0.32687655091285706], [-1.4262701272964478, -0.9537222981452942, -1.221687912940979, -0.11021742224693298, -1.7099500894546509, -0.8366647958755493, 0.8481073379516602, -0.24022549390792847, 0.543423593044281, -0.6200419664382935, -0.12724557518959045, 0.21175560355186462, -0.5535242557525635, -0.08677996695041656, 0.9231885075569153, -0.9536182284355164, -0.21201197803020477, 0.07335101068019867, -0.286533921957016, -0.2757839560508728, -2.3596994876861572, -1.9094189405441284, -0.13440483808517456, 0.6416926383972168, 0.022914579138159752, -0.6822530031204224, 0.48267820477485657, -0.025042636319994926, -0.6240215301513672, 0.04993251711130142, -0.4456884562969208, -1.1159634590148926, -1.1338527202606201, -0.5487679243087769, 0.40139254927635193, -0.7182361483573914, 0.2706682085990906, -1.4607930183410645, 0.7678999900817871, -0.5016164779663086, -0.396615207195282, -0.4622344970703125, -0.22843383252620697, -1.7267217636108398, 0.4593867361545563, 0.8760767579078674, 1.230238676071167, 0.3247334659099579, 0.46267229318618774, 0.5357909202575684, -0.5079178214073181, 0.09181267023086548, 0.9258037209510803, 0.574046790599823, -0.4777832627296448, -1.753118872642517, 0.12967386841773987, -0.8548115491867065, -0.44761642813682556, 0.3769457936286926, 0.5685301423072815, 0.5843230485916138, 0.17260406911373138, 0.15954060852527618, 1.051443338394165, 0.7627100944519043, 0.16482983529567719, 0.9434633255004883, 0.8527100682258606, -0.3159409761428833, 0.20969873666763306, 0.4517727792263031, -0.32234519720077515, 0.7728214859962463, -0.1924951821565628, 0.7235246300697327, -1.315690279006958, -0.2274908423423767, -1.4847224950790405, 0.6728740930557251, 1.4382127523422241, 1.1557849645614624, 0.2701821029186249, -0.8744539618492126, 1.6006746292114258, -0.03863077610731125, 0.8449996113777161, 0.7612819671630859, -0.5607679486274719, -0.04406098276376724, -0.8416128158569336, -1.4957867860794067, 0.9654001593589783, -0.6119627356529236, -1.3951929807662964, -0.11887084692716599, -0.6615772247314453, -0.44409555196762085, -0.9398176074028015, -1.0874900817871094, 1.459991216659546, 0.10182252526283264, -0.6016201972961426, 1.0765990018844604, -0.4340917468070984, 0.2843307852745056, 1.2794841527938843, -0.10610393434762955, -0.28308314085006714, -0.8513948917388916, 0.7977198362350464, -0.05010053887963295, -0.4415402412414551, 0.7356488108634949, 0.05786176025867462, -1.2231837511062622, 0.672271192073822, -0.34021836519241333, 0.13153481483459473, 0.5394524335861206, 0.472403347492218, -0.06580837070941925, 0.948210597038269, -0.7066438794136047, 1.6971484422683716, -0.0616607628762722, 0.011775554157793522, -0.26912033557891846, -1.0437647104263306], [0.21868711709976196, -0.5580759644508362, -1.026597499847412, -0.4672209322452545, -0.20111839473247528, 1.6509945392608643, 0.9742655754089355, -0.015455782413482666, -1.2557238340377808, -0.7763787508010864, 0.3176984488964081, 0.4606925845146179, -1.5738945007324219, -1.8959239721298218, -1.521737813949585, -2.137481451034546, -0.24469012022018433, 0.6220282912254333, -0.8497419953346252, -1.5437037944793701, 1.5555027723312378, 0.6962210536003113, 0.8423675894737244, -0.06748124957084656, 0.30939480662345886, -0.1860344558954239, 1.3894481658935547, -0.7327224016189575, -1.412192463874817, 2.0582523345947266, -0.0510869175195694, 0.8391620516777039, 1.376685380935669, -1.48695969581604, 0.2002909779548645, -0.8448408842086792, -1.9791133403778076, -0.4190548062324524, 2.155081272125244, 0.8938406109809875, 0.5506767630577087, -0.10090654343366623, 1.146475076675415, 1.3885029554367065, 0.8736723065376282, 0.196415975689888, 0.46337997913360596, -0.8175243139266968, 0.534128725528717, -1.0873593091964722, -0.7919020652770996, -1.8839443922042847, -0.366197407245636, 1.2145997285842896, -0.38807567954063416, 0.31816598773002625, 0.27307847142219543, 1.9855217933654785, 0.12103481590747833, 1.8199312686920166, -0.1438267081975937, -0.27723798155784607, 1.4235001802444458, 0.12765711545944214, 1.0413814783096313, -0.8853104710578918, 0.016877101734280586, -1.892199158668518, 0.7045631408691406, 1.1523844003677368, 0.18703876435756683, -2.456787109375, 0.16894537210464478, -0.03972194343805313, -0.3143676817417145, -0.24781109392642975, 1.3427600860595703, 0.15739931166172028, 0.8877536654472351, -0.6836720108985901, 0.8926748037338257, -1.1845940351486206, -0.6810857057571411, -0.12179151177406311, -2.5332720279693604, 1.8524199724197388, 0.47650131583213806, 0.833747923374176, 0.09889817237854004, 0.47984862327575684, 0.8027451634407043, 1.1014361381530762, 0.21423041820526123, 0.2666776478290558, 0.5040569305419922, -0.5097563862800598, -0.704274594783783, 0.609815776348114, -0.1355333924293518, 0.6935954689979553, -1.542457103729248, -0.4506351053714752, -2.290208339691162, 1.0714856386184692, -0.8949594497680664, 0.6976051330566406, 0.9559302926063538, -2.0945258140563965, 0.5126309394836426, -1.2066375017166138, 0.34174543619155884, 0.06317020207643509, 0.6613649129867554, -1.390604019165039, -0.8051638007164001, -0.5117011070251465, 0.6076866984367371, 0.45274826884269714, 0.3251500725746155, 2.4431397914886475, 1.1707375049591064, 1.7553541660308838, -0.7617870569229126, -0.677575945854187, 0.6319712996482849, -0.5963281989097595, -1.4444057941436768, -0.5278791785240173, -1.6527519226074219], [-0.9932822585105896, 0.45840010046958923, -0.733726978302002, 0.09220986813306808, 2.5215468406677246, 0.719708263874054, 2.053605318069458, -0.7167983651161194, 0.06292806565761566, -0.8741962313652039, -2.196021318435669, -0.6138390302658081, -0.014993362128734589, 0.38120704889297485, 0.027591831982135773, 0.07905363291501999, -0.46497416496276855, 0.34174472093582153, 0.8240537047386169, -0.05385880544781685, 2.2432265281677246, 1.0181680917739868, -2.1619064807891846, 0.19876402616500854, 0.4557911157608032, 2.5586094856262207, 0.9216116666793823, 0.4392439126968384, 0.11730660498142242, -1.4048441648483276, -0.9506046772003174, 0.6272832155227661, 0.6184343695640564, 0.6874663233757019, 1.0210620164871216, -0.4198004901409149, -0.8465017676353455, 0.2782518267631531, 1.389314889907837, 1.640462040901184, -0.2172681987285614, 0.20264284312725067, -0.7461237907409668, -0.06646990031003952, 1.180368185043335, -2.013566255569458, 0.6301286816596985, 0.37530434131622314, -0.9542088508605957, -0.615227222442627, 1.3919486999511719, 0.7077653408050537, 1.6180663108825684, -0.7575240135192871, 0.18555022776126862, 3.2941980361938477, -0.3267896771430969, 0.09320222586393356, -0.920448899269104, -0.15691472589969635, 0.006224971730262041, 0.03422257676720619, 0.08448657393455505, 1.9595357179641724, -0.8075878024101257, -0.7663629055023193, -0.03091057576239109, -1.1394580602645874, 0.5197612047195435, 0.9528907537460327, -1.280104637145996, 0.36463093757629395, -0.22255660593509674, -0.6200456023216248, 0.20693080127239227, 1.149208903312683, -2.4600398540496826, -1.0387965440750122, 0.3091111183166504, 0.13323529064655304, 1.0868961811065674, -0.4975999891757965, 1.2201505899429321, 0.3289586007595062, -1.44781494140625, -0.2201939970254898, -0.6451384425163269, -0.7169228196144104, -0.9309942126274109, -0.2327008843421936, 0.8558962345123291, -0.8437065482139587, -1.3177502155303955, -0.7623840570449829, 0.821815550327301, -0.8389551639556885, -0.5126729607582092, -1.1238499879837036, -1.0644235610961914, 0.4547913372516632, 0.3048584461212158, 0.7829802632331848, -1.5524064302444458, 0.3469752073287964, -0.5505660176277161, -0.2611013948917389, 1.8061858415603638, 1.704679250717163, 0.46660274267196655, -1.444946527481079, -0.69675213098526, 0.0577387735247612, -2.3189475536346436, 0.1528933346271515, 0.9371213316917419, 0.47394677996635437, 0.2711505591869354, -0.04144885390996933, -0.8107128143310547, 0.350365549325943, -0.12237521260976791, -1.3065627813339233, 0.6811899542808533, -1.2406249046325684, 0.573724627494812, 1.1223278045654297, -0.5342344045639038, 0.2997736632823944, 0.09286985546350479], [-1.4677990674972534, -0.7906555533409119, 2.014221668243408, 0.24025917053222656, 0.5151017308235168, 0.9767831563949585, 1.0076545476913452, -2.484513998031616, 2.276320695877075, -0.4183626174926758, 0.9943708181381226, -1.0702084302902222, 0.914739191532135, 2.049621105194092, 0.8467899560928345, 0.847802460193634, -0.22063153982162476, 0.9149264097213745, -2.0334701538085938, -0.4539574384689331, -1.5062857866287231, 0.9445486068725586, 0.7756063938140869, 0.4244741201400757, 0.05958526208996773, -0.5002278089523315, 1.2435451745986938, -0.025302324444055557, -0.13120776414871216, -1.137056589126587, 1.3818144798278809, 0.1812206208705902, -0.42071533203125, 0.6133341789245605, -1.325594186782837, 0.19419623911380768, 0.04167301580309868, -0.48648789525032043, -1.1095082759857178, 1.3516603708267212, -1.1780376434326172, -0.007041714154183865, -0.2807196378707886, 0.5927861928939819, 0.7526810169219971, -0.33419784903526306, -1.6854208707809448, 0.09670588374137878, 0.2897246479988098, -1.0335217714309692, 0.07320461422204971, -0.6248828768730164, -0.2955457866191864, 1.323740839958191, 0.31168684363365173, -1.3525633811950684, 0.031212281435728073, -2.271921396255493, -0.9085140824317932, 0.3525483012199402, -0.018037056550383568, -0.7861028909683228, 0.8729102611541748, 0.15511780977249146, -0.35595834255218506, 0.5223054885864258, 0.16904397308826447, 0.9122000336647034, 0.2972632050514221, -1.0289912223815918, 0.8323822617530823, -0.22428151965141296, -0.5947454571723938, 0.3123851716518402, -0.6229081153869629, 0.4825419783592224, 0.26840102672576904, -0.19771303236484528, -1.4083024263381958, 1.2622179985046387, 0.32455480098724365, 0.5052194595336914, -0.8718385696411133, -0.7129841446876526, 0.3265554904937744, 0.27345776557922363, -0.4573546350002289, 1.6154701709747314, -0.3124126195907593, -0.7332185506820679, -1.2328299283981323, -0.6069902181625366, 0.8007461428642273, -0.9867803454399109, -0.4278343617916107, -1.3141765594482422, 0.23341010510921478, 0.682758629322052, 0.48476818203926086, -1.4691400527954102, 0.04191659390926361, 0.7692009806632996, 2.5298924446105957, -0.8387271761894226, 0.7658008933067322, -1.1952683925628662, -1.6142953634262085, 0.31981202960014343, -0.9361177086830139, -0.7755168676376343, 0.2557322680950165, 0.04476145654916763, -0.9906238913536072, -1.3856154680252075, -0.40262970328330994, -0.760888934135437, 1.7623931169509888, 0.35051149129867554, 1.1569526195526123, 0.04533793777227402, -0.14558301866054535, -2.338921546936035, -0.8828995227813721, -0.534554123878479, -0.6928931474685669, 0.9392658472061157, -0.09514668583869934, -0.22571051120758057, -2.004965305328369], [-2.458165407180786, -1.005780577659607, 1.624733328819275, -1.8431470394134521, 0.5307279825210571, 1.0365025997161865, 1.333433985710144, 1.202187418937683, -1.4938011169433594, -0.602766215801239, 0.9137215614318848, 0.6973361968994141, 0.2842029631137848, -1.075819969177246, 1.2752110958099365, 0.4382317066192627, 0.9153748154640198, -0.3621032238006592, -2.308734178543091, -0.3924207091331482, 0.29504337906837463, 0.4931926131248474, -0.6791015863418579, -0.5404134392738342, 0.44069936871528625, -1.2643108367919922, 1.298280119895935, -0.6452592015266418, 1.6230835914611816, 0.3530249297618866, -1.097943902015686, 0.4946649968624115, -0.1313733160495758, -0.8652499914169312, -0.5972616672515869, -0.4659571349620819, -1.275998592376709, -0.3124352693557739, -0.0981130599975586, 0.49305257201194763, 0.49965664744377136, 1.3570424318313599, -0.21382883191108704, 0.11777074635028839, -0.4723714292049408, -0.28407663106918335, 1.2754898071289062, 0.3019132912158966, -1.0217247009277344, 0.08105634152889252, 0.7215320467948914, -0.39730602502822876, -1.845024824142456, -0.6324771046638489, 1.0486369132995605, -1.116303563117981, -0.815163791179657, -1.7114160060882568, 1.3706797361373901, 0.47794368863105774, -2.2017550468444824, -0.490493506193161, -0.7689445614814758, 0.11704886704683304, 2.6482529640197754, 1.037575125694275, -0.8438297510147095, 0.6368922591209412, -2.542282819747925, -0.12715037167072296, -1.4044355154037476, -1.023268699645996, -1.605122447013855, 1.280836582183838, 2.254565954208374, -0.18935047090053558, -1.9157605171203613, 0.7285712957382202, -0.6947125792503357, -0.14946454763412476, 0.7790867686271667, -0.87283855676651, -0.08840622007846832, -0.3253205120563507, -0.15402068197727203, 0.013770803809165955, 1.8209339380264282, 0.9932966232299805, 2.2341911792755127, 1.4992388486862183, 0.22532472014427185, 0.7028878331184387, -0.06391027569770813, 0.5236087441444397, -0.004189241211861372, -1.7014942169189453, 0.8850939869880676, -1.5616800785064697, 1.7735847234725952, -0.09220419079065323, -0.3062632083892822, 1.0329924821853638, 0.2098267376422882, -0.6903635859489441, 0.10235555469989777, -0.5589765906333923, -0.14203661680221558, -2.3459768295288086, -0.0505082942545414, 1.282317042350769, 1.9019445180892944, -0.3899725079536438, -0.2560495138168335, -0.6141830086708069, -1.1893357038497925, 1.4264408349990845, 0.8802021741867065, 0.34512391686439514, 0.031596589833498, 1.8716232776641846, -0.20653651654720306, 1.7602901458740234, 0.47482016682624817, -2.036539316177368, -0.34961116313934326, 0.02494174800813198, 0.8934093713760376, 1.119726300239563, 0.2597236931324005], [2.2179884910583496, -1.7501182556152344, -0.8932766914367676, -0.46095967292785645, 2.0411202907562256, 0.026897549629211426, -0.456362783908844, -0.8964980840682983, 0.8873379826545715, -0.3973068296909332, 0.33914339542388916, 1.5816506147384644, -1.188108205795288, 0.7572157979011536, -1.188469648361206, -1.3176711797714233, 0.964631199836731, -1.3119410276412964, 0.5547955632209778, -1.4948194026947021, -2.3527841567993164, -0.2579305171966553, 0.09298140555620193, -1.1023764610290527, 0.05748388543725014, 0.8065316677093506, -0.38592997193336487, 0.32777345180511475, -0.932394802570343, 1.0549753904342651, -1.286206841468811, -1.2508779764175415, 1.549267292022705, 0.5020983815193176, 0.7464231252670288, 0.9439193606376648, -0.2372274100780487, -0.8014895915985107, 0.538623034954071, -0.8117971420288086, -0.7793672680854797, -0.12757731974124908, 0.025878440588712692, -0.7120644450187683, -1.915320873260498, -0.6844388842582703, 0.252498596906662, 0.9111186861991882, 1.2675873041152954, 2.27832293510437, -0.3187383711338043, 0.3984775245189667, 0.8288866877555847, -0.34777575731277466, -1.5101076364517212, 0.39070314168930054, 0.5116729140281677, 0.6124463081359863, 0.03341040015220642, -1.5813615322113037, 2.086082935333252, 1.0306142568588257, -1.0471248626708984, 2.085904359817505, 0.7265279293060303, 0.581186056137085, -0.46138787269592285, 0.9725872874259949, 0.783053994178772, 0.8790596127510071, -0.26081275939941406, 1.3334825038909912, -0.3583138585090637, -1.2685890197753906, 0.1396622359752655, -0.4747098684310913, 1.1746512651443481, 0.9726579189300537, -0.023423712700605392, 1.8520183563232422, 0.3340907692909241, -0.7911232113838196, -0.6715468168258667, -0.40860694646835327, -1.8952748775482178, -0.3612559139728546, 0.4500628709793091, -2.53725266456604, -0.8903719782829285, -0.757952868938446, -1.6512669324874878, -0.7932839393615723, 0.8658614754676819, 0.8880059123039246, 0.4475492537021637, -0.815369188785553, -0.3916603922843933, -0.9452980160713196, -1.3150640726089478, 1.2361838817596436, -0.13809247314929962, 1.0005849599838257, -0.3161841630935669, -0.3107976019382477, -0.19315019249916077, 0.17700117826461792, 0.10040045529603958, 0.6459820866584778, -1.0772098302841187, -0.020417114719748497, 1.8443877696990967, -1.2004330158233643, -0.4258425235748291, 1.6466150283813477, -1.2565841674804688, -0.6600404381752014, -1.428456425666809, 1.9831957817077637, -0.30920910835266113, 0.7339311242103577, 0.9670597910881042, 1.5831334590911865, -1.3505808115005493, -0.042862147092819214, 0.1303250789642334, 1.120396375656128, -0.7582446336746216, 1.5962215662002563, 0.48271867632865906], [-0.4953538179397583, 0.7098287343978882, -0.3290693461894989, 0.4607628881931305, 0.6171109676361084, 0.42573750019073486, -1.2319846153259277, -0.24521629512310028, -0.8064253330230713, 0.7827050685882568, -1.9837279319763184, -0.023453570902347565, -1.9261081218719482, 1.3607782125473022, -0.1958862543106079, -0.09649930894374847, 1.036374807357788, -1.2550042867660522, 2.3360328674316406, -0.50083327293396, -1.702892541885376, 1.2512335777282715, 0.3259510099887848, -0.34999898076057434, -1.1422159671783447, -0.37232547998428345, 0.3525271713733673, -1.3404606580734253, -0.17254549264907837, 0.9613950848579407, -0.6047828793525696, -1.0447677373886108, -0.49535125494003296, -1.257300615310669, 0.6413748860359192, 0.4058345854282379, 1.4237170219421387, -0.10115204006433487, 1.4234713315963745, 0.5224438905715942, -0.27219849824905396, -1.595409631729126, 1.934989094734192, 0.7525262236595154, -0.12188518792390823, 1.4147642850875854, -0.42409181594848633, -1.4386439323425293, 0.05598173663020134, 1.4850717782974243, 0.5984299778938293, -0.3209514915943146, -0.5135475993156433, -0.39992383122444153, 0.2239551842212677, 0.6816694736480713, 1.5232837200164795, -0.19626310467720032, -1.7002555131912231, -0.4317198097705841, 0.1788734793663025, -0.07441305369138718, 1.1151689291000366, 2.1864750385284424, -1.4958202838897705, -1.1017429828643799, 0.2546994090080261, 1.7571005821228027, -1.4808462858200073, -0.6407659649848938, -1.1025348901748657, -0.5175135135650635, -0.10491995513439178, -0.05367476865649223, 0.9008201956748962, 0.08018171787261963, -0.515815794467926, 0.13033276796340942, -1.236164927482605, 1.6745535135269165, -0.2999897599220276, -0.3855213522911072, -0.7609537839889526, -0.3289450705051422, 0.5776561498641968, -0.43654438853263855, 0.8299514055252075, 1.1790167093276978, 0.4492708146572113, 0.09248453378677368, 1.9231200218200684, -0.24708221852779388, -0.6438992619514465, -0.5410796999931335, -1.7255134582519531, 0.4701094925403595, 0.25354111194610596, -1.1979973316192627, -0.6248344779014587, 0.28832390904426575, -0.9960689544677734, 0.1112918108701706, -0.15222331881523132, 1.2957050800323486, -0.9911419749259949, -0.1889316439628601, 0.3034895062446594, -0.3592039942741394, 0.05451245978474617, -0.7658474445343018, 0.6447274684906006, 2.1104941368103027, 1.3742259740829468, -2.672710418701172, -0.15607033669948578, -2.6540472507476807, -1.2287726402282715, -0.33596599102020264, -0.9254904985427856, -0.14060688018798828, -1.8097615242004395, 0.8690813183784485, 0.413222998380661, 1.8007664680480957, 0.5357300639152527, -0.8358694911003113, -0.9357477426528931, 0.08461236953735352, -0.21538381278514862], [-0.06107436120510101, -1.169601559638977, -0.02547209896147251, -0.3743184804916382, -0.4410359859466553, 1.1734052896499634, 0.14738763868808746, -0.7516590356826782, -0.17983689904212952, 0.5805197358131409, -0.058039046823978424, -0.8810824751853943, 0.2502448558807373, 0.4935826361179352, -0.004075637087225914, -1.9567967653274536, -1.6308355331420898, 0.7429297566413879, 0.3474181294441223, -0.17035001516342163, 0.18092487752437592, -0.8298899531364441, 0.04510727524757385, 0.49584707617759705, -0.6067100167274475, -0.7045190334320068, 1.3902052640914917, 1.1743125915527344, -1.742688775062561, 0.9086859822273254, -0.0034877806901931763, -0.7404757142066956, 0.6142242550849915, -0.07553209364414215, 0.03744497895240784, -1.2845888137817383, 1.0052393674850464, -1.0014431476593018, -0.09386125952005386, -0.6826226115226746, -0.7132374048233032, -0.2861027121543884, 1.9978580474853516, 0.8888280391693115, -0.6081358194351196, 0.5672038793563843, 0.12040340155363083, 0.085084468126297, -0.010494392365217209, -0.5329185724258423, 0.030836714431643486, 0.4285416305065155, 0.9681253433227539, 0.2463231086730957, -0.9310672283172607, 1.2998459339141846, 0.039460282772779465, -0.30556419491767883, 0.6538065075874329, -0.04851504787802696, -2.1977720260620117, 0.9855347275733948, -0.7340089082717896, -0.816648006439209, 0.6996934413909912, -0.9547042846679688, 2.2645199298858643, 0.08905696123838425, -0.6392318606376648, -0.10459353029727936, -0.5765035152435303, 0.19185298681259155, -0.6689380407333374, -1.236366629600525, 0.3662012815475464, 0.5413417220115662, -0.7315017580986023, 0.02501514181494713, 0.6266189217567444, 0.5826969146728516, 0.009326200000941753, 1.111107587814331, -0.4199331998825073, 0.2701435983181, -0.5618702173233032, -0.4295961856842041, 0.06830903887748718, -0.42038536071777344, -1.8734136819839478, 0.10900089889764786, 0.17900331318378448, -1.9015889167785645, 0.9553682804107666, -0.7224282622337341, -1.2126797437667847, 1.2676422595977783, 0.686277449131012, 0.6539483666419983, 0.13182656466960907, 0.8303428292274475, -0.07321334630250931, -0.35914191603660583, 1.164560317993164, -0.8964406251907349, -1.5936336517333984, 0.31980547308921814, -0.8848098516464233, -1.172181248664856, -0.08527740836143494, -0.763803243637085, -0.09815637767314911, -0.6620224714279175, -0.40090033411979675, -1.4428764581680298, -0.4120161831378937, 0.9488445520401001, -0.3814009726047516, 0.8424201011657715, -0.6314915418624878, -0.3303029537200928, -0.5994859933853149, 1.2746162414550781, 0.34549829363822937, 2.2650396823883057, -0.434435099363327, -0.7273476123809814, -0.5969667434692383, 1.5201512575149536, -1.101830244064331], [-0.6277527809143066, 1.7006839513778687, 1.166902780532837, 0.29335451126098633, 0.29767337441444397, 0.12171497195959091, -0.09089292585849762, -0.7567006349563599, 1.2626060247421265, -0.18398143351078033, -0.8018192648887634, 0.7187982201576233, -0.2886577248573303, -0.11010576039552689, -0.9793796539306641, 0.20941311120986938, -0.8399320840835571, 1.1583166122436523, -0.05262938141822815, -0.8714985847473145, -1.2436720132827759, -1.171397089958191, -0.6024885773658752, -2.140223503112793, -0.792852520942688, -0.7803406715393066, 0.8100379109382629, -0.029197629541158676, 1.1427537202835083, -0.991788387298584, -0.5112637281417847, 0.7560891509056091, -1.3473244905471802, -0.49349161982536316, 0.4371180236339569, 0.061230674386024475, 0.48582831025123596, -1.0107845067977905, -0.23752543330192566, 0.4698580801486969, 0.7643252611160278, -0.314777672290802, 0.12119599431753159, -0.2870618999004364, 1.3083713054656982, -0.12210869789123535, 1.441866159439087, -0.20898796617984772, -1.0276538133621216, -1.740254282951355, -0.030957989394664764, 0.1329302191734314, 2.1940345764160156, 0.23381249606609344, -1.809743046760559, 0.999855101108551, -0.24226698279380798, -1.2655491828918457, 0.4520516097545624, 0.4578184485435486, 0.6037726402282715, -0.4276879131793976, 0.6041716933250427, -0.24019846320152283, -0.27939826250076294, 1.2652380466461182, -0.4829994738101959, -0.006151203531771898, -0.45566415786743164, 0.9765576720237732, 0.04010558873414993, -0.36299189925193787, 0.018824150785803795, 2.142843246459961, -0.21166415512561798, 0.6288080215454102, 0.281619668006897, -1.6828241348266602, 0.3254817724227905, -0.9487227201461792, 0.42272189259529114, 0.6605568528175354, -0.6201596260070801, 0.797909140586853, 0.8549550175666809, 0.3105396628379822, -0.9249370098114014, 0.8226640820503235, 1.2394089698791504, 0.6909123659133911, -1.3147372007369995, -0.17673422396183014, 1.0708450078964233, -1.2847044467926025, 0.20604930818080902, -1.905935287475586, 0.9686489701271057, -0.848924458026886, 0.28211095929145813, 0.406766802072525, -1.8123944997787476, -0.06808852404356003, 0.7476056814193726, 0.3530547618865967, 1.7438039779663086, -1.4872928857803345, 0.20869800448417664, 0.10845805704593658, -0.7775352597236633, -1.262508511543274, 0.6383832097053528, -1.243544340133667, -0.5711052417755127, -1.1693869829177856, 2.0019009113311768, 0.4027289152145386, -0.4060881733894348, -0.8439165353775024, 0.10636039823293686, -0.25038689374923706, 1.6004136800765991, -0.5753637552261353, 2.012092113494873, -1.5402251482009888, -1.8031009435653687, -1.1069061756134033, 0.2885272204875946, -0.5199791193008423, -0.44961023330688477], [3.4876434803009033, 0.3044998049736023, 0.5293325185775757, 1.8663206100463867, -1.0564652681350708, 0.38602927327156067, 0.9508005380630493, 0.7398187518119812, 0.08382660150527954, -1.513992190361023, 0.42123234272003174, -0.1439538449048996, -0.7840494513511658, 0.909178614616394, 1.6612944602966309, -0.7444068789482117, 0.15993179380893707, 0.2870975136756897, -2.782172441482544, 2.2160918712615967, 0.9357694983482361, 0.27959364652633667, 0.3632875382900238, 0.6618473529815674, 0.2108931541442871, -1.4897717237472534, -0.03946884348988533, -0.01795821264386177, -0.7554551959037781, 2.0183324813842773, -0.42404183745384216, 1.5559309720993042, 0.5420429706573486, 0.6482548117637634, 0.4163212478160858, 2.064974308013916, -0.7303569316864014, 0.38127103447914124, 0.10998499393463135, -0.732861340045929, 1.1830377578735352, -0.31502801179885864, 1.864801049232483, 1.52687668800354, -2.210150957107544, -1.3674609661102295, -0.8968291282653809, 1.0867619514465332, -0.9705818295478821, 0.4733641743659973, -0.26946189999580383, -0.6526774168014526, -0.04833967238664627, -1.5797926187515259, 0.15587489306926727, -1.1571823358535767, -2.2017412185668945, -0.558434247970581, 0.631157398223877, 1.1868854761123657, 0.3504863679409027, -0.006989805027842522, 0.8102595210075378, 0.5861232280731201, 0.1303866058588028, 1.9383299350738525, 2.1785757541656494, 1.091557502746582, -1.615972876548767, -1.6118803024291992, 1.631174921989441, -1.4682918787002563, 1.3827546834945679, -0.7020280957221985, -2.1658313274383545, 0.2613539695739746, 0.04341410845518112, -2.4407076835632324, 2.2410988807678223, 0.8492701053619385, -0.6186326146125793, 0.8670712113380432, -0.4343506097793579, 0.5464971661567688, -1.2400116920471191, -1.135669469833374, 0.8269492387771606, 0.10847146064043045, -1.2169214487075806, -1.3650099039077759, -0.4031452536582947, 0.47434642910957336, -0.07357257604598999, 0.5672560930252075, 1.593874454498291, -1.3974707126617432, 0.5892711877822876, -0.5560811758041382, 0.16613365709781647, 0.6584794521331787, -2.901512861251831, -1.3312705755233765, -0.9860469102859497, -1.0312063694000244, 0.8035239577293396, 0.5508759617805481, -0.23764142394065857, 0.23306380212306976, 0.4264974594116211, -1.0788464546203613, 0.3011784553527832, -0.8093262314796448, 0.45090094208717346, -1.0491924285888672, -0.8730854988098145, -1.4259568452835083, -0.03918209299445152, -1.3289767503738403, -0.4341832399368286, 1.9256600141525269, 2.0240249633789062, 1.5259217023849487, 1.0644546747207642, 1.7357001304626465, -0.9524044394493103, 1.4779963493347168, -1.0210065841674805, -0.04151589423418045, -1.2487609386444092], [0.4427790343761444, 0.22422641515731812, 1.0872317552566528, 0.10033926367759705, 1.6256294250488281, 1.4120404720306396, -0.06440777331590652, -0.4085864722728729, -0.267431378364563, 0.3121296167373657, -0.32325172424316406, 0.5364853143692017, 0.778144359588623, -0.7720112800598145, -0.21219690144062042, 1.1183751821517944, 0.7618963718414307, 0.0485013872385025, -0.26393741369247437, -0.5644359588623047, 1.0141047239303589, -0.11362915486097336, 0.48649439215660095, -1.8394862413406372, -1.3520890474319458, -0.25069043040275574, -0.7398514747619629, -0.061093978583812714, 0.8742761015892029, -1.1955618858337402, -0.09862309694290161, 0.004362835083156824, -0.1041402667760849, -0.8645787239074707, -0.766486644744873, -0.1390010267496109, 0.04647648334503174, -1.3678877353668213, 0.10473015904426575, 0.4867951273918152, -0.15901875495910645, -0.7242565155029297, -1.2756178379058838, 2.330359697341919, -0.4336663782596588, 0.3575666844844818, 0.029780205339193344, 0.47685539722442627, -0.3507983684539795, 0.10520952194929123, 1.1431694030761719, 1.981175422668457, -1.153069257736206, -0.4037229120731354, -0.7855920791625977, -0.2118685394525528, 0.6240485310554504, -0.3451080620288849, 0.5771176815032959, 0.5629816651344299, 0.04579712450504303, 0.223274827003479, 0.27384936809539795, 0.36320760846138, -0.3732624053955078, -0.22535090148448944, -0.7902570366859436, -0.7578095197677612, 0.4606391489505768, 0.42885082960128784, -1.731763482093811, -1.157418131828308, -1.4806334972381592, 0.007809376809746027, 0.6955400109291077, -0.6052951812744141, 1.0864980220794678, -1.1679022312164307, 0.6279580593109131, 2.046766757965088, 1.421286940574646, 0.2252959907054901, -0.13829044997692108, -0.11272142827510834, 0.5908225178718567, 2.51692795753479, 0.8114356994628906, 0.36692193150520325, 0.16200348734855652, 1.27101469039917, -0.3870544731616974, -0.6506898403167725, -1.6952043771743774, 0.6802747249603271, -0.7645047903060913, 0.12090317159891129, 0.521492600440979, 1.2720918655395508, 0.3106071949005127, -0.7161096334457397, -1.3305186033248901, -0.2025814652442932, -0.48202040791511536, -1.2245430946350098, 1.0471798181533813, -1.2120357751846313, -0.11970174312591553, -1.0926858186721802, 0.016755957156419754, 1.1921690702438354, 0.0444246344268322, -1.113115668296814, 1.105804681777954, 0.04792492464184761, -1.098678469657898, -0.5312623977661133, 0.8221302628517151, 1.2058964967727661, 1.45052170753479, 0.9893917441368103, -0.2729407548904419, -0.3581399917602539, -0.9840327501296997, 1.7970246076583862, -1.753944993019104, -0.4414236843585968, -0.0735902413725853, 0.3032241463661194, 1.407012701034546], [-0.009102582931518555, -0.6051759719848633, 0.29079169034957886, -0.48983287811279297, 0.21478243172168732, 0.6717215776443481, 0.7301791310310364, -0.9440422058105469, 0.4662480056285858, 0.7100505232810974, -1.1291580200195312, 0.8217512369155884, 0.8211362361907959, 0.8340924978256226, -1.5123289823532104, 0.7361177802085876, 0.5949966907501221, 0.3841690123081207, -0.548790454864502, 1.0589282512664795, 0.3916769027709961, 0.009334164671599865, 0.6267643570899963, 0.14235271513462067, 0.6193074584007263, -0.32652127742767334, -1.4416898488998413, 0.5446037650108337, -1.4105294942855835, 0.49549993872642517, -0.3913431465625763, 1.2578706741333008, 1.1402167081832886, 0.30295056104660034, 0.37420332431793213, 0.08988707512617111, -0.38566845655441284, -0.5438640117645264, -1.7227280139923096, 0.14898940920829773, -0.266973614692688, -0.7179344296455383, 0.16221171617507935, 0.29794904589653015, -2.17940092086792, 0.8200094103813171, -0.4959976077079773, -1.2014029026031494, -0.3654257357120514, -1.6363015174865723, 1.1701332330703735, 0.2762148380279541, -0.23430988192558289, -0.18839283287525177, -0.6562830805778503, 0.4885251224040985, -1.336035966873169, 0.06933824717998505, 0.5446314811706543, -0.11063496023416519, -1.7981077432632446, 0.7397153377532959, -1.5404633283615112, -1.7394996881484985, 0.8483193516731262, -0.4684912860393524, 0.4668687582015991, 0.1276932954788208, 2.105464458465576, 0.30420824885368347, 0.09516794234514236, -0.6969249248504639, -1.5823506116867065, -0.28385472297668457, -0.6289708614349365, -0.07180749624967575, 0.7109952569007874, -0.2993292510509491, 0.17594178020954132, -0.40542852878570557, -1.3633952140808105, 0.3716374337673187, -0.6019037961959839, 0.22777730226516724, -0.16592809557914734, 1.7453947067260742, -1.5613133907318115, 0.891044557094574, 2.3164565563201904, 0.5192919969558716, 0.12243861705064774, -1.4918723106384277, -1.4529924392700195, 2.7686545848846436, -0.8510847687721252, -0.943773090839386, -1.3043028116226196, 2.212925910949707, -0.7724226117134094, -0.16130205988883972, -0.4953934848308563, 0.3354507386684418, 0.4091024696826935, -1.8616060018539429, -1.0388007164001465, 1.3048696517944336, -1.208018183708191, 0.8483651280403137, -0.3948642909526825, 0.06437785923480988, 0.4313337206840515, 0.28096553683280945, 1.1412878036499023, -2.0604634284973145, 0.44435253739356995, -1.466752052307129, -0.5120573043823242, -1.9463950395584106, 0.13006673753261566, 0.478048175573349, -0.4339491128921509, -1.2357527017593384, -0.37643668055534363, 0.9505078196525574, -0.29670649766921997, 0.07456310093402863, -1.6658499240875244, -0.4259060323238373, 0.4116036295890808], [1.5917174816131592, 1.6616697311401367, 1.444250226020813, -0.7281818389892578, 0.7299039959907532, -0.7180583477020264, -0.7408443689346313, -0.2616397440433502, 0.9172488451004028, 0.09570898860692978, -1.1407554149627686, 0.4080178141593933, -1.1355969905853271, -0.9023385643959045, -0.33655306696891785, -0.2156492918729782, -0.1295253485441208, 0.8598423004150391, -0.4519384205341339, 2.485076665878296, 0.15604880452156067, 0.7744714021682739, 1.354948878288269, 0.624322235584259, -0.8248188495635986, -0.7704663276672363, 0.09434645622968674, -0.308095782995224, -1.1447973251342773, -0.019330745562911034, 2.372403144836426, 2.559126138687134, -0.5546855926513672, 1.2384191751480103, -0.6472213864326477, -0.4426007866859436, -0.4233107268810272, 0.8428954482078552, 0.36449626088142395, -0.8268697261810303, -0.5068241953849792, -1.631689190864563, -0.6068750619888306, -1.9161598682403564, -0.5924453735351562, -0.2623625099658966, -1.6024399995803833, 1.0558831691741943, 0.8526583909988403, -0.202460378408432, -0.3332765996456146, -1.724127173423767, -0.44393306970596313, -1.2969883680343628, -1.493542194366455, 1.156780481338501, -0.7951468825340271, -0.9172586798667908, -1.1013801097869873, -0.8314590454101562, -1.4414474964141846, -1.0501880645751953, -0.0627409815788269, 0.4529007077217102, -1.620996356010437, 1.2951585054397583, -0.022481441497802734, 0.8519797325134277, 1.1964155435562134, -0.07467606663703918, 0.9102911949157715, 0.43869081139564514, -0.5465573668479919, -0.045301035046577454, 1.1703425645828247, -0.9682188630104065, -2.1416354179382324, 0.39827629923820496, -0.32113364338874817, 0.6695947051048279, 0.22384478151798248, -0.8565633296966553, 0.313146710395813, 0.18782643973827362, -1.1786487102508545, -0.15931864082813263, 1.310193419456482, 0.6576521992683411, 1.7387059926986694, -1.794305443763733, 0.38285139203071594, -0.04745185002684593, -1.6417120695114136, 0.41775983572006226, -0.09182269126176834, 0.15492422878742218, -0.4285728633403778, 0.21580252051353455, -0.8261159658432007, -2.0249733924865723, 0.46215689182281494, 1.059970498085022, 0.34954389929771423, -0.523581862449646, -1.325838327407837, 2.434434175491333, 1.3139574527740479, 1.2772098779678345, -0.7405251264572144, -0.5530007481575012, -0.7668395042419434, 0.43561026453971863, 0.900066614151001, -0.3658944368362427, -3.223784923553467, 0.49101802706718445, 0.6941155791282654, 0.26340121030807495, -0.1805533766746521, -0.744864821434021, 0.8611474633216858, 0.6821698546409607, -0.4530611038208008, -2.013578176498413, -1.1337782144546509, 0.6607602834701538, -0.07082457095384598, 0.24471665918827057, -0.9081860780715942], [-0.5131481289863586, -0.5660167336463928, -0.11200044304132462, -0.09562980383634567, 1.730445146560669, 0.7842190265655518, -0.23638498783111572, -0.5945695638656616, -1.2659058570861816, 0.5180111527442932, -1.126556634902954, 1.363350749015808, 0.646386981010437, -0.9447432160377502, -0.7895399332046509, -0.617262065410614, -0.9351831674575806, -0.609046995639801, 0.3846912682056427, -0.7021616101264954, -0.4777614176273346, -1.8469903469085693, -0.5593202114105225, -1.740394115447998, -0.21013158559799194, -0.13907086849212646, -0.1521116942167282, -0.08078277111053467, -0.16558846831321716, -0.24915504455566406, 0.656995952129364, -0.15507152676582336, 1.5927034616470337, 0.7559621930122375, -0.7522016167640686, -0.5510818958282471, -1.398687720298767, 0.6685054302215576, 2.083744764328003, 0.3682198226451874, 0.9424444437026978, 1.6620079278945923, -0.9504045248031616, -0.6482222080230713, 2.0063071250915527, 0.31894201040267944, -0.11036279797554016, -0.18530097603797913, 0.639178454875946, 0.06509220600128174, -0.6162096858024597, -0.4400714933872223, 0.41805312037467957, 0.4158608615398407, 0.29903122782707214, 0.01677211932837963, 1.8066853284835815, -0.25345441699028015, -0.7514904737472534, 0.5676109790802002, 1.5325783491134644, -0.27674299478530884, -0.2655111253261566, -0.4755968749523163, -0.4228641986846924, -0.5090458989143372, 0.04178675264120102, -0.7226402759552002, -1.1576341390609741, -2.967514991760254, -0.12814590334892273, 1.204753041267395, 0.003023821162059903, 1.181241512298584, -0.5137966275215149, 0.19850091636180878, -0.6656895279884338, -0.960163414478302, -0.9527170658111572, -1.3535900115966797, 0.3089388608932495, 0.314240962266922, 1.503129482269287, -1.8249799013137817, -0.3317473232746124, 0.14042246341705322, -0.5064426064491272, -1.8116439580917358, 0.5261590480804443, -0.43289414048194885, 0.48564913868904114, -1.6231237649917603, -1.2564609050750732, -0.008394161239266396, 0.17630676925182343, 1.2794580459594727, 1.1082651615142822, 1.6547704935073853, 1.0420254468917847, 0.5530043244361877, 0.21213378012180328, 0.5974831581115723, 0.952832043170929, 0.019226273521780968, 0.06946929544210434, 1.0806492567062378, -0.38711270689964294, -2.1282958984375, 2.2002029418945312, -1.464173674583435, -1.4037349224090576, 0.7727940678596497, -1.0063551664352417, 0.4855177402496338, 0.15201205015182495, -0.02052580751478672, 1.2210159301757812, 0.19063608348369598, 2.0004334449768066, -0.659277081489563, 0.3793025314807892, 0.8057203888893127, 1.5015344619750977, -0.18280404806137085, 0.4620738625526428, -1.8717955350875854, -0.24126151204109192, 1.8102846145629883, -0.959581196308136], [0.21330755949020386, 0.06868400424718857, -1.1754491329193115, 0.2794899642467499, -0.4009762704372406, -0.7716410756111145, 0.6376647353172302, -0.23005974292755127, 0.4324743151664734, -0.5451602339744568, 0.9681881666183472, 2.0064895153045654, 0.3756687343120575, -0.038154494017362595, -0.992300808429718, 1.1158323287963867, 0.7936264276504517, 1.0076820850372314, 1.8778767585754395, 0.19407403469085693, 0.10803068429231644, 0.18044638633728027, -1.4647876024246216, 0.9068660736083984, 0.9286348223686218, 0.480791836977005, -0.48373672366142273, -1.0309414863586426, 0.6965541243553162, 0.17619237303733826, -0.6529483199119568, 0.06699775159358978, 2.5865590572357178, -0.1670488864183426, 0.7164036631584167, 1.0375878810882568, 0.18111975491046906, -0.16146430373191833, 0.4461604952812195, -0.08203105628490448, 0.7089672088623047, 0.4676859676837921, 0.2829950749874115, 1.1772539615631104, -0.49252697825431824, 0.05294199287891388, -0.003589572384953499, -0.5723088979721069, -0.19663584232330322, -0.38401758670806885, 0.3318842649459839, 0.7378230094909668, 0.028343914076685905, 0.8234317898750305, -0.5099290609359741, -0.6874939203262329, 0.8472171425819397, 0.41993051767349243, 0.49936604499816895, 0.3111675977706909, 0.7109133005142212, -0.5319889187812805, 1.6690140962600708, 0.051675766706466675, -0.7926367521286011, 2.248056411743164, 1.100501298904419, 1.9736768007278442, -1.5127387046813965, -0.4357335567474365, 0.4437233507633209, -0.44579577445983887, 0.6343865394592285, 0.4936217665672302, 0.23504683375358582, -0.40084895491600037, 0.9935064911842346, -1.0935138463974, -0.1433553397655487, -2.3505587577819824, -1.772348403930664, 1.2680559158325195, 0.4898780584335327, -1.7350482940673828, -1.332369327545166, 0.18970943987369537, 0.45245251059532166, -1.0185998678207397, -2.643792152404785, -0.32455483078956604, 0.5397683382034302, -1.0620455741882324, -0.46994513273239136, 0.3382948637008667, -1.0030652284622192, -0.655137300491333, 1.4782899618148804, -0.9555001854896545, 1.8015905618667603, -0.20733915269374847, -1.0910539627075195, 0.19981838762760162, 0.6225752234458923, -2.2055535316467285, -0.1408250331878662, 1.1829475164413452, 0.3339977264404297, -1.1551512479782104, -0.7920512557029724, 0.03106921911239624, 0.49464288353919983, -1.1289759874343872, 0.977949321269989, 0.20695741474628448, -0.7050553560256958, -0.32558733224868774, -0.11228460818529129, 1.459661841392517, -0.521984338760376, -1.8249092102050781, 1.6224212646484375, 0.23863673210144043, -0.6317378878593445, 1.1910719871520996, -0.6227589249610901, -0.1635856181383133, -0.7850605845451355, -0.08695830404758453, -1.3481826782226562], [1.7199735641479492, -1.9572899341583252, -0.49572259187698364, -0.18832091987133026, -0.8057264089584351, -0.00450383685529232, 0.3254987895488739, -1.305050015449524, 0.27337220311164856, 2.684950113296509, 0.008319055661559105, -0.45915013551712036, 0.0884113684296608, 1.8163944482803345, -0.16294577717781067, -1.2494064569473267, -1.5505605936050415, 0.821849524974823, 0.04822627454996109, -1.1201205253601074, 0.7004261016845703, -1.247115135192871, 1.1766126155853271, -0.9059361219406128, 0.4839242100715637, -0.9410396814346313, -0.9853081107139587, 0.520232617855072, 1.4397854804992676, 0.5925951600074768, -0.3477761745452881, -0.5162695646286011, -0.5160597562789917, -0.540490448474884, 0.7330003976821899, -1.4259427785873413, -0.14440199732780457, -0.31257960200309753, -0.07099424302577972, 0.35991448163986206, -0.49844005703926086, -1.3075076341629028, -0.22640053927898407, 1.0780047178268433, 2.128159761428833, 0.22249799966812134, 1.8869003057479858, -0.10289251059293747, 0.5038987994194031, -0.04362141340970993, -0.35367462038993835, 1.0409319400787354, -1.3061286211013794, 0.9733574390411377, 0.4322667419910431, -0.22154612839221954, -1.5846441984176636, -0.2976408898830414, -0.42706289887428284, -1.5120142698287964, -0.5312597751617432, -0.3493214249610901, -0.6556200981140137, -1.0609170198440552, -0.767113208770752, 2.339921712875366, -0.5002812743186951, 1.5366699695587158, -1.3374656438827515, 1.5171810388565063, -0.24366573989391327, 2.4761998653411865, -0.7739088535308838, 0.2696320414543152, 0.688137948513031, -0.9674422144889832, 0.2871067523956299, 0.015691254287958145, -1.2108373641967773, -1.480661392211914, 0.9727254509925842, -1.0577726364135742, -0.455985963344574, 0.5956050753593445, -1.7844449281692505, 0.06554775685071945, -1.3196824789047241, -0.9937700629234314, 0.3489319682121277, -1.318826675415039, -0.9111613035202026, -0.7980533838272095, -0.7833670377731323, 0.13112053275108337, -1.3384857177734375, -0.4755343794822693, 0.5905440449714661, -0.00875906739383936, -0.8244457840919495, -0.49302634596824646, 1.2095121145248413, -0.4478813111782074, -2.322911500930786, 0.5566189289093018, 0.04264282435178757, 0.7097046375274658, 1.191667079925537, 0.6552681922912598, -0.02581334300339222, 0.3579177260398865, 0.14800553023815155, -0.5980545878410339, 1.036534309387207, 0.3138837218284607, 1.8166015148162842, -0.4827478528022766, -0.7569736242294312, -0.19898496568202972, 0.016437310725450516, -0.4919516444206238, 1.2383211851119995, -0.15421375632286072, -0.03165457770228386, -1.5258479118347168, 0.8548880815505981, 1.7536941766738892, -1.6498583555221558, -1.6206612586975098, 1.009372591972351], [-0.02338465116918087, 0.04514273628592491, -1.04923415184021, -0.38909587264060974, 0.8745290040969849, -1.71799635887146, -0.09538910537958145, 0.10952327400445938, 0.2989848852157593, -0.33344826102256775, 0.9780011773109436, 0.9330270290374756, 1.721384882926941, -0.5414413809776306, -0.9123668074607849, 1.2893128395080566, -0.5866637229919434, 0.7115676999092102, -0.6029810905456543, -1.146457314491272, 0.8727753162384033, -0.741202175617218, -0.2855772376060486, -1.7376195192337036, -0.49385809898376465, -1.1781057119369507, -0.7820536494255066, 1.470445990562439, -2.097970724105835, -1.0721101760864258, 1.4536070823669434, -1.8608932495117188, -0.11141349375247955, -1.0582751035690308, 1.584770917892456, -0.38029780983924866, 1.3354765176773071, -1.8182910680770874, -0.06252363324165344, 0.4497072100639343, -0.6455207467079163, -0.6420933604240417, 0.9363411068916321, 0.1430869996547699, 1.3920619487762451, 1.5807218551635742, -1.1899585723876953, 0.5704957842826843, 0.05858242139220238, 0.7985945343971252, -0.9475680589675903, 2.079507350921631, 2.17905592918396, 0.29794222116470337, -0.5038337707519531, -0.5767893195152283, -0.12751144170761108, 0.2956867218017578, 0.23301962018013, 1.535494327545166, -1.2690621614456177, -0.2381540834903717, 0.14959104359149933, -1.7267212867736816, 0.1702815145254135, -0.8112837076187134, -0.3970334231853485, 1.0378527641296387, -0.6283705234527588, 0.4932246804237366, 0.8330482244491577, 0.7466239929199219, 0.21153031289577484, -1.096481442451477, 0.6123819947242737, -0.17509445548057556, -0.15071645379066467, -0.2024983912706375, 0.7772640585899353, 0.20599877834320068, 0.10685472935438156, 0.32001450657844543, -1.243470311164856, -1.6216061115264893, 1.729100227355957, 0.18837173283100128, 0.23384849727153778, 0.7460751533508301, -0.3817204535007477, -0.4843474328517914, 0.8095381259918213, -0.9185997247695923, 0.791592001914978, -2.4547579288482666, -0.4597223103046417, 0.7485297322273254, -1.2573050260543823, -0.7959248423576355, 0.8727633953094482, -0.7633461356163025, 2.632399082183838, 0.9151004552841187, 1.2665181159973145, 0.7018104195594788, 0.7993016242980957, 0.909191906452179, 1.0796869993209839, 1.0086017847061157, -0.5521432161331177, 1.042848825454712, 0.6621835827827454, 0.4708242118358612, -0.6888219118118286, -0.3046541213989258, -0.7608143091201782, 1.0903630256652832, -0.6900330185890198, 0.590347945690155, 0.07190583646297455, -1.6158345937728882, 0.7030622959136963, -0.1385515183210373, 0.045511484146118164, -0.4183512032032013, -1.584834098815918, 0.10996328294277191, -1.609537959098816, -0.4927409291267395, 1.061895489692688], [1.1355116367340088, -0.6294277310371399, 1.0715937614440918, -1.1898014545440674, -0.045088306069374084, 0.5097636580467224, -0.2774518132209778, 0.36125046014785767, -0.09122403711080551, 1.670676827430725, -0.1557680070400238, 1.5367581844329834, 0.684576153755188, -0.6239129304885864, 1.729766607284546, 2.4123737812042236, -0.9149934649467468, -0.3067004978656769, -0.8141940236091614, 0.05035312846302986, 1.544977068901062, -1.255535364151001, -0.1206614077091217, -1.1386202573776245, 0.06909595429897308, 0.07680848240852356, 1.1958284378051758, 1.2174038887023926, 0.11494097858667374, 0.7555552124977112, 0.18176856637001038, 0.6731881499290466, 1.9480878114700317, -0.4678840637207031, 0.16958729922771454, 0.33824828267097473, 0.13647159934043884, 1.1916279792785645, 0.4043259620666504, 0.057484664022922516, -0.7779749631881714, 0.46509242057800293, 0.8197595477104187, -2.790872812271118, -0.8739000558853149, 0.02057858556509018, -0.629987895488739, 0.6533588767051697, 3.0289952754974365, 0.10737944394350052, -0.7869634628295898, 0.2314196228981018, 0.5637302994728088, 1.646337866783142, 0.1990964114665985, 0.08670870214700699, -0.3906848132610321, -0.7781170010566711, 0.4779722988605499, 0.06738661229610443, 0.6544269323348999, -1.2102043628692627, 1.7332807779312134, -0.20983567833900452, -0.4284851551055908, 0.026156621053814888, 1.5065158605575562, -0.1646590530872345, -1.2233246564865112, 0.09760674834251404, -2.57722806930542, 1.01607084274292, 0.682397186756134, -0.08217893540859222, 1.8504570722579956, 0.2855508625507355, -0.3310532569885254, -0.3185924291610718, 0.8481183648109436, -1.7927727699279785, 0.5256608724594116, 1.4575763940811157, 0.20995506644248962, 0.4648090600967407, -0.11564317345619202, 1.777686357498169, -0.27899131178855896, 2.649502992630005, 1.4093940258026123, -0.3079337477684021, -0.06207483634352684, 0.1294703185558319, 0.3313805162906647, 0.477324903011322, -0.30214452743530273, 0.5664918422698975, 0.2101023644208908, 0.24020566046237946, -0.4656783938407898, -1.0045262575149536, -0.5648108720779419, 1.2185331583023071, -0.5725159645080566, -0.31503432989120483, -0.08384989202022552, -0.36565569043159485, -0.4439743161201477, -0.9211380481719971, 0.05339282006025314, -0.6204681992530823, -0.372450053691864, 0.46913790702819824, -1.8862451314926147, 0.1759330928325653, 0.09241172671318054, -0.7523854374885559, -0.9155452847480774, -0.49928781390190125, 0.1719028204679489, -0.8198308944702148, 0.8231082558631897, -1.0931577682495117, -1.3349168300628662, -0.33778083324432373, 0.7619233727455139, 0.4634263217449188, 0.48219960927963257, 0.35862281918525696, 0.654299259185791], [1.5010337829589844, -0.687788188457489, -0.011056414805352688, -0.8442546725273132, 0.5001617074012756, -0.5573302507400513, 2.0803592205047607, -0.5475292801856995, -0.6017862558364868, 0.7480540871620178, 0.09900180250406265, -0.43212422728538513, -0.45753946900367737, 0.11289161443710327, -1.911075472831726, 0.19364941120147705, 0.20996730029582977, 0.985166072845459, 0.5166753530502319, -0.29071590304374695, 0.7666687965393066, 1.2542369365692139, -0.04455955699086189, 1.579939842224121, 0.37010204792022705, -0.18067044019699097, -0.36982181668281555, -0.6529579162597656, -1.2648178339004517, 0.005593054927885532, -0.32347556948661804, -0.19336572289466858, 0.07698894292116165, -0.15833574533462524, 0.2822898328304291, -1.3588719367980957, 0.3109692931175232, -0.8172722458839417, -1.1106590032577515, -1.1413272619247437, -0.9147516489028931, -0.9706352949142456, -0.7295566201210022, 0.09694170206785202, 0.664293110370636, -0.3629484176635742, -0.0060561830177903175, 0.5333080887794495, -1.2270845174789429, -1.0465068817138672, 1.446340799331665, 0.5467424392700195, -1.2922943830490112, -0.2904898524284363, -1.1356277465820312, 0.371741384267807, -0.41171884536743164, -0.812995433807373, 0.8418569564819336, -0.13860881328582764, 1.1998848915100098, 0.7673991918563843, -1.794663906097412, 0.1337829828262329, 1.4517264366149902, -0.78699791431427, -0.0415053553879261, 1.3210777044296265, -0.373497873544693, -0.3798964023590088, -0.5200414061546326, -1.7059706449508667, 1.677811622619629, -0.46342960000038147, -0.1243043914437294, -0.6133643388748169, 0.2528803050518036, -0.6041105389595032, 1.2544556856155396, -0.07452482730150223, 2.480807304382324, -1.2082191705703735, -0.8860256671905518, -0.9012565016746521, 2.048656463623047, -0.4028230309486389, -0.13034281134605408, 1.1128872632980347, 0.5484786033630371, 0.47861161828041077, -0.51179039478302, -2.541818141937256, 0.49325069785118103, 1.0663875341415405, 0.3723563849925995, -0.889424741268158, -0.8441200256347656, -0.3715182840824127, -1.8004897832870483, -0.07473625987768173, 0.5001029968261719, -0.5183965563774109, 1.0166850090026855, -1.3391821384429932, -0.8073632121086121, -0.3506418466567993, -0.042639635503292084, -1.692746639251709, 0.027763064950704575, -0.7223529815673828, -0.5622563362121582, -0.11813639104366302, -0.1341414451599121, -0.5374410152435303, 0.0785079076886177, 0.007534876000136137, 0.2017369568347931, 1.2606408596038818, 1.8376911878585815, 0.2645934224128723, 1.088026762008667, -0.7390773892402649, 0.9383910894393921, 0.98819500207901, 1.2509092092514038, -0.059696074575185776, -0.30688169598579407, 0.474589467048645, 0.06155304238200188], [-1.5071840286254883, -0.7087275385856628, 1.347509741783142, 2.2249157428741455, 0.7450960278511047, -1.6783417463302612, -0.6449247002601624, 0.023817958310246468, 2.6090705394744873, -1.3570120334625244, -0.3905270993709564, 2.2518503665924072, -0.16682302951812744, 0.8615469932556152, -0.3274633586406708, 0.19094014167785645, 2.7454822063446045, -0.6005129814147949, -0.6991387605667114, 1.546789526939392, 0.15155741572380066, -1.2114207744598389, -0.7258622646331787, 0.159202441573143, 0.2572641372680664, -0.5842614769935608, 1.8947718143463135, 0.9946645498275757, 0.8288605809211731, 1.6352699995040894, -1.1465473175048828, -0.7941237092018127, -0.2676495909690857, 2.860311269760132, 0.2843720316886902, 0.30950453877449036, -0.6035308241844177, -0.5517833232879639, 0.8156012892723083, 0.30551084876060486, -0.5337536334991455, -0.5485885739326477, -0.9117289781570435, 1.8244903087615967, -2.325248956680298, 0.8345189094543457, -0.3184007704257965, -1.4595882892608643, 1.1708847284317017, -0.2222781479358673, 1.7589607238769531, 0.4012034237384796, 1.754469394683838, -0.7871480584144592, -1.3823039531707764, -1.4014534950256348, 1.4627940654754639, -1.49094820022583, -1.132330060005188, 0.6065090298652649, -0.24780204892158508, 1.7981168031692505, -0.7978366017341614, 0.4273225963115692, 0.3063594102859497, -0.08690308779478073, -0.8270440101623535, 0.2413959950208664, 0.5338944792747498, -1.2501471042633057, -0.7331752181053162, 0.20153026282787323, -0.6712141036987305, -0.5838528871536255, -1.6777225732803345, 0.7616955041885376, 0.7723913192749023, 0.0581582672894001, -0.7755963802337646, -0.3925883173942566, -2.8894643783569336, -0.15254521369934082, 0.3888740539550781, -0.39665496349334717, 0.7163926959037781, -0.21261656284332275, -0.8107984662055969, 1.024977207183838, 0.6716307997703552, 0.9163960814476013, -0.6336822509765625, -0.5707671642303467, -0.13738180696964264, 0.4362865388393402, -0.8666102886199951, 0.4245837926864624, 0.21450774371623993, -2.3791139125823975, 0.8573574423789978, 0.009776600636541843, 0.44207432866096497, -1.010364294052124, -0.16636748611927032, -1.115074634552002, 0.5470548868179321, -3.0945799350738525, -0.26370227336883545, -0.09728103131055832, -0.31738734245300293, 0.9759489893913269, 0.3808153569698334, -1.044411063194275, 0.8458428382873535, -0.28337204456329346, -0.22168144583702087, 0.4136715531349182, 0.2735899090766907, 0.5506708025932312, 0.15496671199798584, -0.10464486479759216, -0.2816276550292969, -0.5837042331695557, 1.1084200143814087, -1.2590992450714111, 0.571837306022644, 1.0946195125579834, -0.15706956386566162, 0.35268840193748474, 1.3474009037017822], [-0.7263787388801575, -0.08093877136707306, 0.6018375754356384, 0.38382071256637573, -1.524402379989624, 2.5710113048553467, 1.4860364198684692, -1.9950546026229858, 2.204735517501831, 1.1002405881881714, -0.03085894137620926, 0.22770151495933533, -0.35384801030158997, 0.12413794547319412, -0.6348181962966919, -1.5856108665466309, -0.6439580321311951, -0.637609601020813, 0.28114697337150574, -2.494992971420288, -0.21281662583351135, 1.0163943767547607, -0.18882139027118683, 1.6667386293411255, 0.3868383765220642, 2.416816473007202, 1.0125961303710938, 0.7531722784042358, 0.7196366190910339, 0.48763081431388855, 1.3583351373672485, 0.46044790744781494, 0.35648420453071594, 0.05980624631047249, -1.0854723453521729, 1.2221705913543701, -2.1774938106536865, -1.0657532215118408, -1.661527156829834, -1.1226036548614502, -0.449232280254364, -0.9565849304199219, 0.06065287068486214, 0.9528827667236328, -0.04852091521024704, -0.514818549156189, -1.1805073022842407, 0.40872305631637573, 0.14650876820087433, 1.8175724744796753, -0.3142774999141693, 0.7476657032966614, 2.539334297180176, 0.45477786660194397, 0.15633048117160797, 1.0591752529144287, 1.5869728326797485, 0.25791317224502563, -1.550642967224121, 1.0738266706466675, -0.6446492075920105, 2.173576831817627, 0.3980405032634735, 0.1823299080133438, -0.4468916356563568, -1.051426649093628, -1.944730520248413, -0.31686022877693176, 0.041586458683013916, 0.8311926126480103, 0.49666398763656616, -0.7917094230651855, -1.8869496583938599, -0.2937397360801697, -1.161714792251587, 0.812265157699585, -1.026986002922058, -0.31029653549194336, -1.3932390213012695, -0.9035459160804749, -0.9880416393280029, -0.8601623177528381, 1.3300200700759888, 0.16897375881671906, -0.17890004813671112, -0.11734861135482788, 0.3091394901275635, 0.8222850561141968, -0.6417438983917236, 1.9603644609451294, 0.44867217540740967, 0.39135968685150146, -0.6795966029167175, 0.49500611424446106, 0.08987286686897278, -2.0645251274108887, 2.7220747470855713, -0.3254046142101288, -1.9458271265029907, -0.5525607466697693, -0.749727725982666, 0.28844672441482544, 0.5477419495582581, -0.03673424944281578, 0.1466779261827469, 0.16532568633556366, -0.5427332520484924, -0.5749789476394653, 0.8223150968551636, -0.12947726249694824, 1.9424293041229248, 0.8443688750267029, -0.23806114494800568, 0.34052610397338867, -0.0197286494076252, 0.39194974303245544, -0.024119462817907333, 0.7534053921699524, 1.2908343076705933, 1.14707612991333, 1.908280372619629, -0.31672096252441406, 1.512587070465088, -1.1671842336654663, 1.4747512340545654, -0.7170929908752441, 0.06276986747980118, -1.9069968461990356, 0.01669919118285179], [-0.8387503027915955, 0.7356197237968445, -1.073860764503479, 1.2132562398910522, -0.13019278645515442, -1.8549060821533203, 0.12183991074562073, -0.5099862217903137, -0.3509206473827362, -0.23224394023418427, -0.8179081678390503, 2.281578540802002, -1.8990880250930786, -0.9466974139213562, -0.08297965675592422, -2.553048610687256, 1.3690694570541382, -1.2239298820495605, -1.1232986450195312, 0.09826401621103287, -0.8944711089134216, 0.5348385572433472, 0.14383763074874878, -1.2230972051620483, -2.9588735103607178, 0.6370301842689514, 0.43570828437805176, -0.219554603099823, 0.7106318473815918, 2.1199021339416504, 1.1580619812011719, -0.802645742893219, 0.6698728799819946, -0.5076525807380676, 0.6322896480560303, 0.23275898396968842, 1.0977038145065308, -0.7077558040618896, -0.3875220715999603, 0.6484465599060059, -1.0441370010375977, -1.579639196395874, -2.3938000202178955, -2.119600772857666, 0.7137532830238342, 0.5056262016296387, -0.47707921266555786, -0.4897008240222931, -0.4680018424987793, 1.696385145187378, -1.7863185405731201, -1.0398285388946533, 1.267034649848938, -0.3925919532775879, 0.12479174882173538, -0.3094143271446228, 0.13499963283538818, 2.220221757888794, 0.6529490947723389, -0.14986056089401245, -0.8490275144577026, -0.8314886093139648, 2.9315574169158936, 0.6831551790237427, 1.3964519500732422, 0.013464822433888912, -0.35497069358825684, 1.704471468925476, -0.21720559895038605, 0.35204243659973145, -0.753559947013855, 0.7245926856994629, -1.0880171060562134, -0.3216458559036255, -0.9186569452285767, -0.8656551241874695, 0.10677608102560043, -0.10914399474859238, 0.3594554662704468, -0.5461098551750183, -0.6911827921867371, 0.12766212224960327, 0.4929375648498535, -2.0213866233825684, -0.3498687744140625, 0.4065287709236145, -1.5147818326950073, -0.24847395718097687, -0.9209694266319275, -0.09187383949756622, 1.005934476852417, 0.7370920777320862, 0.8957812190055847, -0.8810020089149475, -0.07373849302530289, 0.25546130537986755, -0.4039068818092346, 0.213398739695549, -2.0378453731536865, 0.026653781533241272, -1.8703848123550415, -0.0691114217042923, 0.6685142517089844, -0.05384564772248268, -0.35273119807243347, 0.1650010645389557, 0.40816158056259155, 0.34943363070487976, -0.2780837416648865, -0.34295010566711426, 1.3176558017730713, 1.7276620864868164, -0.928625762462616, -0.023354193195700645, 0.7561480402946472, -1.121077537536621, 0.13241930305957794, -1.9135162830352783, -0.5992138981819153, 0.6292087435722351, 0.95451819896698, 2.836948871612549, -0.16005831956863403, 0.5428348779678345, -0.7215253710746765, 0.4577110707759857, -0.8175676465034485, -0.5159653425216675, 0.1452181488275528], [-1.2618159055709839, 0.368757039308548, -0.28154969215393066, -0.09074987471103668, 0.154650017619133, 0.20958317816257477, 0.6346455812454224, -0.05295506864786148, 0.38319510221481323, 1.0712426900863647, 0.7127864360809326, -0.20187905430793762, -0.5429232120513916, 1.4000816345214844, 1.4490317106246948, -0.0539417564868927, 0.2803579568862915, 0.45019081234931946, 1.2604734897613525, -0.7724350690841675, -2.0552170276641846, -0.8621459007263184, -0.17498049139976501, -1.3553541898727417, 1.652419924736023, 2.1003258228302, 0.9489650726318359, 0.6567277312278748, 0.10941381007432938, -0.13685360550880432, 0.3314080536365509, -0.7184079885482788, -1.0916749238967896, 0.31662726402282715, -0.6896483302116394, 0.6187306642532349, -1.1193580627441406, 0.12051178514957428, -1.503909707069397, -0.533298909664154, -1.524902105331421, -1.162790298461914, 0.41672953963279724, 1.6988064050674438, -0.7586503624916077, 0.9477130174636841, -0.4857896864414215, 1.5995419025421143, 0.04782995954155922, 1.4394898414611816, -0.06475710868835449, -0.6831269264221191, 0.10616277903318405, 0.06166602671146393, -0.880976140499115, -0.06020626053214073, -0.3966144323348999, -0.652416467666626, -2.479978561401367, 0.3518179655075073, 0.669506311416626, 2.097648859024048, -0.6117043495178223, -0.5915877223014832, 0.7625558376312256, -0.6995266079902649, 0.4821525514125824, 0.6756603121757507, 0.7906457781791687, 0.5747926235198975, -0.04235873743891716, 0.549210786819458, 0.9119696021080017, -0.2513650357723236, 0.4543934166431427, -1.4755276441574097, 0.5267991423606873, 0.5761027932167053, 2.14123272895813, -0.7742217779159546, -0.5502382516860962, 0.15380269289016724, 0.021976234391331673, -0.4141343832015991, 0.252223402261734, -0.6598578095436096, 0.07737751305103302, 0.026345044374465942, -0.6966869831085205, 0.5283876061439514, -1.983915090560913, 1.1244412660598755, 1.2579857110977173, 1.5239468812942505, 0.5933774709701538, -0.6732578873634338, 0.10418694466352463, -0.184430792927742, 1.1820446252822876, -1.1690741777420044, -1.0055791139602661, -0.3391367793083191, 1.6796495914459229, 1.8202481269836426, -0.5305732488632202, 0.3003999590873718, 1.739498496055603, 0.24314475059509277, -0.13203024864196777, -0.02995583601295948, -0.5604156255722046, 0.21967501938343048, -0.9982339143753052, 0.1927805095911026, 0.6252976059913635, -0.35880282521247864, -0.5442772507667542, 0.8747433423995972, -0.6117157340049744, 0.6645322442054749, 1.9459072351455688, 0.42567265033721924, -0.34950265288352966, -0.2190258800983429, 0.33504530787467957, -1.3656141757965088, 1.26976478099823, -1.681573748588562, -1.063076138496399], [-2.2461068630218506, 0.6321627497673035, 1.3955951929092407, -0.8213213086128235, 0.15065115690231323, 1.3056172132492065, -0.5914172530174255, 0.611278772354126, 0.20448951423168182, 1.2149244546890259, 0.7530018091201782, 0.04429428279399872, -0.39401760697364807, 0.4569011926651001, -0.881546139717102, 0.6321719884872437, 1.1345311403274536, -0.12284576147794724, 2.2303669452667236, 0.049747176468372345, -0.6924117207527161, 1.510721206665039, 1.6044600009918213, -1.1315689086914062, -0.014644059352576733, 0.9203147292137146, 0.4680388271808624, -0.1927156001329422, 0.13891816139221191, -0.5391834378242493, -0.27487194538116455, 1.0547587871551514, 1.4233367443084717, 0.29234790802001953, -0.6182092428207397, 0.00795171782374382, 0.8857215046882629, 0.03186589106917381, -0.08338459581136703, -0.434756338596344, -0.44593262672424316, -1.2207920551300049, 1.874912977218628, 0.4801606237888336, -0.041478391736745834, 0.25304800271987915, 0.9990973472595215, 0.9985125660896301, -0.346767783164978, -0.22990718483924866, 2.2368152141571045, 1.7839454412460327, 0.3271673321723938, 1.5924514532089233, 2.06260085105896, -0.546549916267395, 0.06945211440324783, 1.1456105709075928, -0.5366684198379517, -1.6430619955062866, -0.12844128906726837, 0.2650587260723114, 1.508405089378357, -0.8491629362106323, -0.4225364029407501, 0.04318386688828468, -0.7666495442390442, 0.14915123581886292, 0.8186854124069214, 0.5244316458702087, -0.5594172477722168, -1.1208423376083374, 0.8864578008651733, 0.34226906299591064, 0.743901789188385, -0.02400132641196251, 0.6335296630859375, -0.981222927570343, 0.08612259477376938, 0.7155865430831909, -0.6823643445968628, 0.672873318195343, 1.7587031126022339, -0.27137237787246704, -2.240407943725586, -1.6399037837982178, -0.48565948009490967, 0.04397473484277725, -1.1712857484817505, 1.5900623798370361, -0.49797922372817993, 1.7556723356246948, 0.5900049805641174, -1.5900968313217163, -1.13071870803833, -1.539103388786316, -0.24156568944454193, 0.6989011764526367, -0.30241405963897705, -0.36877965927124023, -0.8125509023666382, -0.9160263538360596, 0.04115084558725357, -0.9660753607749939, -0.5907726883888245, 0.3498761057853699, -0.5903591513633728, -1.0847201347351074, 1.239000678062439, 1.801031470298767, 0.39287588000297546, 0.9915478229522705, -0.4765053689479828, -1.0628957748413086, 0.2707160711288452, -1.672741174697876, 0.793541669845581, -0.06746872514486313, -0.09540118277072906, -1.2907456159591675, 0.231117382645607, -0.031180407851934433, -0.16780607402324677, 0.5136628746986389, -1.8882358074188232, 2.807347297668457, 0.8284868001937866, -0.10223924368619919, 0.5351769924163818], [-1.8923768997192383, 2.078986406326294, -1.1128649711608887, -0.27460959553718567, -1.7042843103408813, -0.1884668916463852, 1.9284125566482544, -0.25511792302131653, -0.19866332411766052, 0.825456440448761, -0.4251680076122284, 2.3691186904907227, -1.1425422430038452, 0.3178674280643463, -0.691777765750885, -0.0031984117813408375, 1.268887996673584, -0.2123318314552307, -0.13412074744701385, -0.2878224849700928, 1.1431013345718384, -0.9654459357261658, -0.7326794266700745, 0.4324474334716797, 0.3349992632865906, 1.619645357131958, 0.5643426775932312, -0.27113276720046997, 1.0664688348770142, 1.487571358680725, 0.06524493545293808, -1.2411137819290161, -2.4470691680908203, -0.7111070156097412, 0.11367549747228622, 0.5534546971321106, 1.1811753511428833, 1.3453062772750854, -1.2373193502426147, 1.3394346237182617, 1.1686939001083374, 0.7090117931365967, -1.253556728363037, -1.2082676887512207, -0.0978509783744812, 0.912510335445404, 1.0335596799850464, -1.684553861618042, -0.3086967468261719, 0.33904409408569336, 0.9522116184234619, -1.0179954767227173, 0.7376348376274109, 0.3003719449043274, 1.3946046829223633, 1.6641660928726196, -2.277859687805176, 0.8042912483215332, -1.6241023540496826, 0.8541837334632874, 0.439675509929657, 0.7632747888565063, -1.3683536052703857, 0.8008387684822083, 0.33249038457870483, 0.4448457360267639, 0.8225309252738953, -0.3679560422897339, -0.42157575488090515, -1.1290372610092163, 1.0681226253509521, 2.4583029747009277, -0.29420021176338196, -0.2559754252433777, 0.05933065339922905, -0.24699611961841583, -0.16517087817192078, 0.3991127908229828, -0.05424965173006058, 0.42991313338279724, 2.3712613582611084, 1.3873530626296997, -0.7687077522277832, -0.21811051666736603, -0.6196432709693909, -1.3861404657363892, 0.7738824486732483, 2.3538074493408203, 0.8677825331687927, -0.2459040880203247, 0.8197579979896545, 0.5721467733383179, 1.3435665369033813, 0.3249015510082245, -0.3040595054626465, 0.03315242752432823, 1.209442138671875, 0.15133634209632874, 0.008891185745596886, -0.3529302179813385, -1.104684829711914, 0.3932715356349945, -0.8105790615081787, 1.7220025062561035, -1.2070651054382324, -0.7573928833007812, -0.01606469415128231, -0.12897805869579315, -0.22248178720474243, 1.676029086112976, 0.20764851570129395, 0.7316374182701111, -0.8479342460632324, -1.4200512170791626, -0.7243437170982361, -0.1974712312221527, -1.0159200429916382, 0.06999015808105469, 0.8121473789215088, 1.1171066761016846, -0.20880208909511566, 0.20862413942813873, 1.324652075767517, -0.28707417845726013, 0.7920974493026733, 0.15952540934085846, -1.3462417125701904, 0.911424994468689, -1.099084734916687], [-0.5079508423805237, 0.12954656779766083, 0.09559155255556107, -0.68523770570755, -0.01771010272204876, -0.032790619879961014, 0.47205713391304016, 0.6937092542648315, -2.4969358444213867, 0.7269050478935242, 3.049811601638794, -1.186957597732544, 0.841493546962738, 0.9752413034439087, 1.1107152700424194, -1.6715707778930664, 0.6334804892539978, 1.4957257509231567, 0.5417989492416382, 1.1338038444519043, 0.3508661687374115, -0.5286474227905273, 2.3084585666656494, 1.2391209602355957, -0.00635770196095109, 0.5616454482078552, 0.6899241209030151, -0.01363574992865324, 0.18435169756412506, -2.1194305419921875, -0.5886475443840027, -2.0269370079040527, -0.5904943943023682, 0.0879417136311531, 0.7345725893974304, -0.8981532454490662, -0.3075491487979889, -0.0988888218998909, 0.049838192760944366, 0.8770128488540649, -0.02158067747950554, -0.1106804609298706, -0.3222649395465851, -0.6084209084510803, -0.3369562327861786, 0.27110692858695984, -0.06891521066427231, -1.7111793756484985, 0.5712716579437256, 0.34462881088256836, -0.4529277980327606, -1.7818080186843872, -2.1797258853912354, -0.3458448648452759, -0.713916003704071, -0.5136896967887878, -0.22695428133010864, 0.12633386254310608, 2.526336193084717, -2.3550302982330322, -1.0136412382125854, -1.6425678730010986, -0.6321834325790405, 0.8929750919342041, 0.9751114845275879, -0.546288251876831, 0.7560331225395203, -0.9622086882591248, 0.32871559262275696, 0.4197050631046295, -0.7765136957168579, 0.7520177960395813, 0.5831465721130371, -0.7112919092178345, 0.8739844560623169, 0.4297638237476349, -1.0415542125701904, -0.12258359044790268, 1.8150984048843384, -0.09023590385913849, -0.795949399471283, -0.19959232211112976, 0.43835872411727905, 0.2534431517124176, -0.9530578851699829, -0.3866554796695709, -0.4617556035518646, 0.6876866221427917, -1.1437500715255737, -0.05980898067355156, 1.0343531370162964, 0.22388511896133423, 0.6163979768753052, -1.571547031402588, 0.04759996011853218, -0.06904064863920212, -2.5005688667297363, 1.378866195678711, -0.49140092730522156, -0.6285297274589539, 0.37172260880470276, 1.0980379581451416, -1.0857086181640625, 0.135220468044281, -1.1499226093292236, 0.46442756056785583, 0.21444547176361084, -0.03979157656431198, -2.199526786804199, 0.7996789813041687, -0.5059734582901001, -0.7841418385505676, 1.810013771057129, -0.953316330909729, -1.2899047136306763, 0.27276504039764404, -0.007968747988343239, 1.5399879217147827, -0.9451708793640137, 0.06183606758713722, 0.43311744928359985, 0.4496581256389618, 0.898427426815033, -0.42555198073387146, 1.5708398818969727, 1.3558300733566284, 0.44179055094718933, -1.164875864982605, 1.4204814434051514], [1.1091214418411255, 0.3623206913471222, -0.12871086597442627, -2.209136724472046, 0.5930795669555664, -1.0218602418899536, -1.6145210266113281, -1.0279815196990967, -1.5104646682739258, -0.3013310730457306, 1.8388053178787231, 0.0526900477707386, 1.133254051208496, 0.023660019040107727, -0.07176535576581955, -1.1737648248672485, 0.1759650856256485, -0.07296741753816605, -0.7278668880462646, 1.1715143918991089, -0.8278719186782837, 0.06825772672891617, 0.7553052306175232, 0.6812726259231567, 0.5393503904342651, -1.391772985458374, -1.2238752841949463, 0.2143586128950119, 0.1500358283519745, -2.422562837600708, 1.6369229555130005, 0.3166457712650299, -0.7704302668571472, 0.5330503582954407, 2.328402280807495, 1.5893378257751465, 0.4088205099105835, 1.0389890670776367, -1.4375556707382202, -2.132248640060425, -1.5265194177627563, -0.2605645954608917, 0.15797947347164154, -0.4343445897102356, 0.9488881826400757, -0.72666335105896, 0.45563197135925293, 1.446216344833374, -0.19016900658607483, -1.3262964487075806, -0.9541189670562744, 0.40749087929725647, -0.5101091861724854, -0.1519317477941513, -0.2072838693857193, 0.9222359657287598, 1.154587984085083, 0.21260850131511688, 1.1525787115097046, -1.2783284187316895, 1.2723585367202759, -0.6937472820281982, -0.3940325379371643, 0.0917714536190033, 0.6724368929862976, 0.133114755153656, -0.26424673199653625, 1.5795347690582275, -0.20598085224628448, 2.1328225135803223, 0.9247869849205017, -0.66776043176651, -1.3482271432876587, -1.203405737876892, 1.057557463645935, 0.027742160484194756, -0.726875901222229, -1.012548565864563, 0.7585177421569824, -3.161677598953247, 0.8569620847702026, -0.7935267090797424, 0.10163702070713043, 1.0528563261032104, 1.0504480600357056, -0.24079254269599915, -0.20239131152629852, -1.053398847579956, -0.11194898933172226, -0.6525728106498718, 0.7866587042808533, 1.9870730638504028, 0.40551963448524475, -0.531798243522644, -1.262308955192566, -1.1790024042129517, -1.364027738571167, -1.4774651527404785, -0.9750518202781677, 0.00853956863284111, 0.5527570247650146, -0.5319724082946777, -0.9304120540618896, 0.26081377267837524, 0.10028661042451859, -1.204930305480957, -0.45422083139419556, 0.09837259352207184, -1.0619239807128906, 0.2513852119445801, 0.9409251809120178, -0.016214687377214432, 0.12248709797859192, -1.3978404998779297, 2.1565489768981934, 0.29355037212371826, -2.145554542541504, -2.4416048526763916, 0.17484058439731598, 0.04419499263167381, 0.9246385097503662, -1.4751063585281372, 1.0740151405334473, 0.8035568594932556, 0.5090426802635193, 0.5783146619796753, -0.1762588769197464, 0.022402826696634293, -0.3297947347164154], [0.24485249817371368, 0.15668781101703644, -0.19543883204460144, -0.912003219127655, -1.002753734588623, 1.4958699941635132, 0.5581064224243164, -0.7007192373275757, -1.9555401802062988, -0.777730405330658, -0.12170489877462387, 0.2927732467651367, 1.47312331199646, -0.6541416049003601, 0.15695402026176453, 0.6388018131256104, -0.1899787336587906, -0.2407001256942749, 0.797683596611023, -0.5249534249305725, -0.13051043450832367, -0.3210189938545227, 0.6193968057632446, 0.4138948321342468, -1.9705965518951416, 0.5909932255744934, -1.2781052589416504, -1.9715142250061035, 0.8667809367179871, 0.5956758260726929, -0.23985403776168823, -0.4872797429561615, 0.8491644263267517, 0.8010951280593872, -0.167199045419693, -1.1326944828033447, 0.1307702362537384, 2.479464292526245, 0.024730384349822998, 0.0006984358769841492, -0.644115149974823, -0.09371116012334824, -0.7188465595245361, 0.9870641231536865, 0.8817280530929565, 0.5597041249275208, -1.1824743747711182, 1.0238938331604004, -0.29765763878822327, 0.06752058863639832, 0.5204214453697205, -0.31543728709220886, -1.5240713357925415, -0.7697784900665283, -1.8694026470184326, 0.4421541690826416, 1.2166653871536255, 0.734954297542572, 1.4329673051834106, -0.2779272496700287, -0.17013335227966309, -0.9419167041778564, -0.5090245008468628, 0.40184199810028076, 1.267925500869751, -0.686900794506073, 1.7407162189483643, -0.11458516120910645, -0.9156337976455688, -0.8182283043861389, 1.0210416316986084, 2.0650405883789062, 0.8532034158706665, -1.3571642637252808, -0.05260142311453819, -2.25972580909729, -1.4442247152328491, 1.374439001083374, -0.1800655573606491, 0.9033958315849304, -0.11675049364566803, -0.5227004289627075, 0.5778262615203857, 0.01643921434879303, 1.9495354890823364, -0.8063595294952393, 0.7420835494995117, -0.8599200248718262, 1.6765908002853394, 0.34814462065696716, -0.5661473870277405, 0.07630995661020279, -0.1203722283244133, -1.7258387804031372, -2.6040732860565186, 0.20503512024879456, -0.012197805568575859, 0.45931899547576904, -0.8915666937828064, -0.2790434956550598, 1.1934651136398315, -0.43533948063850403, -1.077589988708496, 0.8934846520423889, 0.39144185185432434, -0.8461810946464539, -1.5679348707199097, -0.18437008559703827, 1.764633297920227, -0.9279801845550537, 0.698600172996521, 0.6384348273277283, -1.3949655294418335, 1.5505043268203735, -0.23176422715187073, 0.6853393912315369, 0.3465651869773865, -0.10627246648073196, 0.33031436800956726, -1.7664318084716797, 1.0718779563903809, 0.8517381548881531, -0.013751983642578125, 0.024142485111951828, -0.32242611050605774, -0.8699585795402527, 1.7276662588119507, -0.14486435055732727, 0.08864450454711914], [0.19557631015777588, -0.6141278743743896, -0.5749995112419128, 1.1071923971176147, 0.8460599780082703, -0.9846189022064209, 0.911483883857727, -2.6091468334198, -0.7386066317558289, 0.9305562973022461, -1.3005998134613037, 0.02982034720480442, 1.2200701236724854, -0.6006912589073181, -1.252816915512085, 0.3442932963371277, -1.0970807075500488, -0.9653734564781189, 0.4509310722351074, -0.10959646850824356, -1.7417258024215698, 0.8447862267494202, 0.6447086930274963, -0.20199674367904663, 0.3213891088962555, -0.43783655762672424, 0.22431069612503052, -1.7080729007720947, -0.7338060140609741, 0.6319977045059204, 1.0189987421035767, -2.0320541858673096, -0.7874278426170349, 1.3190524578094482, 1.0570707321166992, -0.9480620622634888, -1.1471259593963623, -0.9142962098121643, -0.8917495608329773, -1.3610360622406006, 0.43900808691978455, -1.6373692750930786, -0.014763169921934605, -1.4904628992080688, -0.4514480531215668, 0.3808845281600952, -0.14600306749343872, 0.5252930521965027, 0.3626753091812134, -0.5215185284614563, -0.4273008108139038, -0.6797506809234619, -0.8780346512794495, -1.5662856101989746, -0.6734158396720886, -1.4235632419586182, 0.08044421672821045, -0.9728729128837585, 0.7563830614089966, 0.08915159106254578, 0.40976354479789734, -1.0265741348266602, 0.6377828121185303, -0.8839586973190308, 0.1797131448984146, 0.36739975214004517, 2.0373916625976562, 1.4928808212280273, 1.1440091133117676, -0.12047034502029419, -0.7042176723480225, -0.2206079065799713, -0.7642890214920044, -1.5324037075042725, 0.981543242931366, 1.202417254447937, -1.3100160360336304, 0.26447999477386475, 0.9736124873161316, -1.565024971961975, 1.4123327732086182, 1.4379725456237793, -0.33668428659439087, 0.9523283839225769, 0.9586905837059021, 2.500418186187744, 0.3637979328632355, -0.5983583927154541, -1.977400779724121, -2.2787563800811768, -0.5516071319580078, 1.1853021383285522, 0.4238325357437134, -0.6406749486923218, -2.0333728790283203, 1.172305703163147, 1.7602936029434204, 0.9600101113319397, 0.01201685518026352, 1.5799446105957031, 0.5760300159454346, -1.0125676393508911, -0.5276720523834229, -0.5123546719551086, 1.3648260831832886, -2.3120994567871094, 0.05159565433859825, 1.1081931591033936, 0.07773984223604202, -0.09070107340812683, 1.770482063293457, 1.0022395849227905, -0.9442817568778992, -0.1519230604171753, -2.4230220317840576, -0.5931048393249512, 0.979962944984436, 1.063699722290039, 0.2082240879535675, -1.6467783451080322, 0.8579951524734497, 1.0341322422027588, -0.972815752029419, -1.415584921836853, -0.004660307429730892, 2.008269786834717, -1.4470971822738647, -0.1424502432346344, 0.27136242389678955], [-0.7436663508415222, -1.0702507495880127, 0.9037371873855591, -0.4200734794139862, -0.9384137392044067, -0.3928873836994171, 0.1630600392818451, 0.8481348156929016, -0.4474378526210785, -0.7250086665153503, 1.470917820930481, -0.6411037445068359, 0.20604358613491058, -0.9780429005622864, -0.2733525037765503, -1.6885398626327515, 0.7561262845993042, -0.33409157395362854, 0.18656513094902039, 1.8031251430511475, -1.8273667097091675, 1.5191113948822021, -0.25861185789108276, -1.6342638731002808, -0.5570195913314819, -1.0425466299057007, 1.60676109790802, 0.9866572618484497, 0.3690302073955536, 0.6042246222496033, -1.2475115060806274, 0.4368520677089691, -0.3777689039707184, -0.9716957211494446, -0.11675097793340683, 0.7588028311729431, 0.09594814479351044, 0.5542559623718262, 0.05487451329827309, 0.48777705430984497, -0.010878434404730797, -0.41141512989997864, -0.6369985938072205, 0.4601987600326538, -1.0186866521835327, 2.2270147800445557, 0.31220337748527527, 2.5687496662139893, 1.2752891778945923, 1.1084210872650146, 0.18970243632793427, 0.672832190990448, -1.5280545949935913, 1.2464454174041748, -0.3269011080265045, -0.479280948638916, 0.23047617077827454, -0.03266370669007301, 0.5729107856750488, -1.5231404304504395, -0.4770682752132416, -0.21290552616119385, -0.32775819301605225, -0.46923795342445374, 0.8490253686904907, -0.44324636459350586, -0.1541132926940918, -0.2881060242652893, 0.27810177206993103, -0.7785764336585999, -0.2889914810657501, -2.3126027584075928, 1.213165283203125, 1.0980452299118042, 0.8273910880088806, -0.9312124848365784, -0.6036518216133118, -0.2694634795188904, 1.0479729175567627, -0.38387003540992737, 1.10873544216156, -0.8166012167930603, 0.2850635051727295, 0.061142124235630035, 1.4180595874786377, 0.010495850816369057, -2.1610617637634277, 0.6136729717254639, -0.7489035129547119, -1.7883588075637817, -0.4569069445133209, -1.3777097463607788, 0.4836670756340027, 0.7368385791778564, -1.3824855089187622, -0.5354581475257874, 0.5107061266899109, -0.9831904768943787, -1.6730014085769653, -0.22838546335697174, 0.21945083141326904, -0.5103683471679688, -0.15668877959251404, 0.6856704950332642, 0.8394908905029297, 1.6340134143829346, 1.031795859336853, 0.8068240284919739, 1.1222572326660156, 0.23707610368728638, 0.10615561157464981, -0.5805896520614624, 0.7222791910171509, -0.7001913189888, -0.3715475797653198, -0.903223991394043, 0.08349039405584335, -0.2090226709842682, 0.1265653818845749, -1.2522385120391846, 0.39408329129219055, 0.03972487151622772, 1.1120909452438354, 1.2928228378295898, 0.5418673753738403, 1.2892378568649292, 0.3250654637813568, -0.46584123373031616, 1.54483163356781], [0.6988193988800049, 0.877109944820404, 0.1607246696949005, 0.3431825637817383, -1.8691704273223877, 0.09058906883001328, 0.6792857646942139, -0.1533193737268448, -0.029549486935138702, 1.4154388904571533, -0.5099489092826843, 1.1251221895217896, 0.7243286967277527, -1.0025867223739624, 0.06011681631207466, -1.1769628524780273, 0.11797536909580231, 0.500049352645874, 0.6080343723297119, -0.4004080593585968, -1.4391878843307495, -0.9724620580673218, -1.2907283306121826, 1.9366772174835205, -1.6656444072723389, 1.3318383693695068, 0.3747524321079254, -0.13550224900245667, 2.2252919673919678, 1.0876468420028687, 1.0563982725143433, 0.42506876587867737, -1.5476421117782593, -0.7463611960411072, -0.288259357213974, -0.17974953353405, -1.2093682289123535, 2.951662063598633, 0.17854785919189453, -0.07156144827604294, -0.9892557263374329, 0.09446538984775543, -0.3376467525959015, -2.7833688259124756, 1.71017587184906, -0.6317663788795471, -1.277624487876892, -0.6682546138763428, 1.128332495689392, -0.05724843591451645, 0.03825421631336212, -2.0283894538879395, 0.1300307959318161, 1.3660192489624023, -0.6678197979927063, 0.49252021312713623, 0.5481268167495728, 1.1221048831939697, 0.16250169277191162, 0.41828301548957825, 0.03199519217014313, -0.7143336534500122, -0.20101875066757202, 0.13410022854804993, 1.2586039304733276, -2.1757681369781494, 0.12007372081279755, 0.8370124697685242, -1.5955091714859009, -2.2902302742004395, -0.6112155914306641, 0.2904437780380249, -0.6875492334365845, 0.32150426506996155, -0.3662072718143463, -0.9328674077987671, -0.6159390807151794, -0.0006458907155320048, 0.2143101692199707, 1.1647998094558716, 1.4085241556167603, -1.1515544652938843, 0.21691618859767914, -0.11740925163030624, 0.023641306906938553, -0.49828287959098816, 0.36266788840293884, 0.7120228409767151, 0.20329198241233826, -0.18308594822883606, 0.6155245304107666, 0.9052255153656006, 0.15028899908065796, -2.6385653018951416, -0.5593875646591187, -0.18979987502098083, 1.2126290798187256, -1.6607452630996704, -0.5440548658370972, -1.08607017993927, 1.2109274864196777, -0.3701096773147583, -0.16280651092529297, 2.207369804382324, -0.2207600325345993, 1.4179390668869019, 0.2484169602394104, -0.3692307472229004, 1.0534393787384033, 0.05980198457837105, 0.7982360124588013, -0.6171435713768005, 0.10659493505954742, 1.2698512077331543, -0.40396925806999207, 1.4360283613204956, 1.174062728881836, 0.8052741885185242, -0.20101238787174225, -0.43609538674354553, 1.3997137546539307, -0.6766678094863892, -0.07740961015224457, -0.21034081280231476, 0.24643953144550323, 1.1304590702056885, -0.38443779945373535, 0.2834121882915497, -0.6917349696159363], [1.3558213710784912, 0.5690363645553589, 0.8700815439224243, -1.0214084386825562, -0.8191573023796082, 0.4862689673900604, 0.7135689854621887, -0.2317790985107422, -0.9386172890663147, -1.5961438417434692, 0.8309975862503052, -0.3352232277393341, 1.5088419914245605, 0.01655679941177368, 1.8548989295959473, 0.05823148414492607, -0.7689736485481262, -0.3384533226490021, 0.9208992719650269, 0.8128883242607117, 0.41553229093551636, 0.17319032549858093, 1.930354118347168, -0.7125000953674316, -0.9572364687919617, -0.23140797019004822, -0.10270298272371292, -0.9688581228256226, -1.0614328384399414, 0.5452893376350403, -1.3228157758712769, 0.8009746670722961, -0.5779927968978882, -0.49625468254089355, -0.758884072303772, -0.8723031878471375, 0.9237284064292908, -0.40453454852104187, -0.27259141206741333, 0.5228793025016785, 0.47455593943595886, -0.6061840653419495, -0.5653223991394043, 1.5132650136947632, 0.2252608984708786, -0.815549373626709, -1.0992927551269531, 1.0520968437194824, -0.5658340454101562, -0.4585578143596649, -1.280912160873413, -0.21601974964141846, 0.07216835021972656, -0.6790321469306946, 1.808617115020752, 0.25832727551460266, -1.468454122543335, 2.0285985469818115, -2.518528699874878, 0.7980133295059204, -0.3516751825809479, -0.1552840620279312, -0.3368612825870514, -0.5543851852416992, 0.343811571598053, 0.003465034766122699, 1.0212231874465942, -1.1290491819381714, 0.654066801071167, 0.7763747572898865, 1.4575610160827637, 0.2047148197889328, 0.28082340955734253, -1.1063779592514038, -0.29697173833847046, -0.023824278265237808, 0.14674720168113708, 1.446352243423462, 0.04819832742214203, 0.564527690410614, 0.4859233796596527, -0.6321713328361511, 2.079986333847046, -0.6355416178703308, 1.0192360877990723, -1.6311615705490112, -0.6971468925476074, -0.3589484393596649, -0.5620588660240173, -0.3545815646648407, 0.068616583943367, 0.2779080867767334, -0.974891722202301, 1.2728890180587769, 1.486647367477417, -1.0712347030639648, -0.4877815246582031, -0.6696507930755615, -1.0092830657958984, 1.9510334730148315, 1.2572370767593384, -1.546662449836731, 0.31819671392440796, -1.4208511114120483, 0.21719156205654144, -0.3660755455493927, -0.9200258255004883, 1.0401251316070557, 0.10470130294561386, -1.3169668912887573, 0.23790188133716583, 0.5010718107223511, 0.5509328246116638, -1.2012996673583984, -2.34790301322937, 0.05326419696211815, 0.5819762349128723, -0.4619164764881134, -0.7541519999504089, 0.6095514893531799, -0.147993803024292, -0.9593303203582764, 1.3410958051681519, -0.31737038493156433, -0.6760143637657166, -0.7564295530319214, 0.1333848088979721, -1.9221097230911255, -0.655207097530365], [-0.03715740889310837, 0.5365321040153503, -1.1222862005233765, -0.11198914051055908, 1.600329875946045, 1.3571823835372925, 1.0020196437835693, -0.7593380212783813, -0.8901891708374023, 0.7995664477348328, -0.941836416721344, -0.21274656057357788, -1.5421476364135742, 0.2746559977531433, 2.238631248474121, -1.159635066986084, -2.355252981185913, -0.49748778343200684, 0.9604279398918152, 1.2063779830932617, 1.1476633548736572, 1.0504488945007324, 0.22235210239887238, 0.6847471594810486, 1.5411831140518188, -0.2650578022003174, 0.19092369079589844, 0.07848525047302246, 0.6444991827011108, 0.28382474184036255, -0.8055845499038696, -0.5854008793830872, 0.6327676773071289, -1.0390350818634033, 0.4235309064388275, -1.4412752389907837, 2.9310462474823, -1.034864068031311, 0.5265253186225891, 0.15212681889533997, 0.2410718947649002, -2.0211424827575684, -1.5347099304199219, 0.10306629538536072, -0.6021825075149536, 0.588129460811615, 0.19390350580215454, -0.5213848948478699, -1.1608268022537231, 0.37141603231430054, -2.3641819953918457, -1.2023807764053345, 1.2382354736328125, 0.6680787801742554, -0.4815775454044342, 1.2950658798217773, 1.3108572959899902, -0.1143103837966919, -0.693496584892273, -0.928142786026001, 0.26447364687919617, 0.3836826980113983, 2.015925645828247, -0.31363874673843384, -1.377756118774414, -0.3046925365924835, 0.732406497001648, -2.4016916751861572, -2.1192026138305664, -1.647939920425415, -0.18557721376419067, -1.5939371585845947, -0.37939950823783875, 0.9631989598274231, -0.8636032342910767, -0.15978975594043732, -0.16390565037727356, 2.136380910873413, 0.5366423726081848, 1.0021494626998901, 0.3638555407524109, -0.8067077994346619, -1.1948015689849854, 0.5874738693237305, 2.2512171268463135, -1.323232650756836, -0.2759351432323456, 1.4113998413085938, -0.09744661301374435, -0.13712836802005768, 0.7442325949668884, 0.2723778784275055, 0.8466414213180542, -1.5711393356323242, -0.1419907808303833, -0.5996036529541016, -1.2860063314437866, 0.338725745677948, 1.1472560167312622, -1.4463245868682861, -0.13952019810676575, -0.5330764055252075, -1.2810415029525757, 0.23712053894996643, 0.5971405506134033, -0.1521567702293396, -0.8947150111198425, -0.5045477151870728, 2.187896966934204, -2.0858349800109863, -0.697331964969635, -1.331214189529419, 0.4933519661426544, -0.049078647047281265, -0.42256343364715576, -0.8459736704826355, -1.323972463607788, -0.30378302931785583, -0.39984631538391113, 1.9563853740692139, -1.942298173904419, -1.5698556900024414, -0.3551419973373413, -0.7106484174728394, -0.24703308939933777, -0.245780810713768, 1.3553568124771118, 2.028357982635498, -0.055515993386507034], [-0.05875217914581299, 1.2422016859054565, -0.5178842544555664, 1.634170413017273, 0.25912654399871826, 0.7605819702148438, -0.9523808360099792, 1.134068250656128, -0.011414075270295143, -0.8852694630622864, -0.3863343596458435, 0.2413274347782135, -0.16001442074775696, 0.9508948922157288, -1.0038930177688599, 1.176295518875122, 0.013867155648767948, 0.48848703503608704, -0.1962478905916214, -0.6127285957336426, -1.306979775428772, 1.7903391122817993, -0.6848437190055847, 2.0163633823394775, 0.02365298569202423, -1.9998383522033691, 1.5281587839126587, 1.4885203838348389, 1.5326919555664062, -0.0396541990339756, 0.20466341078281403, 0.04235683009028435, 0.6728794574737549, 0.40137770771980286, 0.9164157509803772, -0.45775115489959717, 2.1009864807128906, 0.8051283955574036, 0.09790007025003433, -0.2288970649242401, -0.8526633381843567, -1.2818254232406616, 0.22095640003681183, -0.6584051251411438, -0.46712788939476013, 0.923858642578125, -1.1378239393234253, -0.4369775056838989, -0.7039077281951904, -0.3560042381286621, 0.3542109429836273, -0.21116229891777039, -0.7141979336738586, -0.10965409129858017, 0.5666300654411316, 2.271289110183716, -0.2404957413673401, -1.8569931983947754, -0.28574806451797485, 1.7796111106872559, -0.5391877293586731, -1.6840590238571167, 0.5695555806159973, -1.0054737329483032, -1.069703459739685, 0.10294453799724579, -0.03039260022342205, -0.42925435304641724, 2.3308308124542236, 0.02546854317188263, 1.4253404140472412, -1.0046281814575195, -1.31058931350708, 0.9299805760383606, 0.6757729649543762, 0.5541565418243408, -1.24803626537323, -0.8349300026893616, 0.6652066707611084, 0.8902944326400757, -0.042171087116003036, -1.7400141954421997, 1.0118329524993896, 0.3684157729148865, -1.7252452373504639, 0.011583776213228703, 0.8894233703613281, 0.7146716713905334, 0.369690865278244, -0.7820265293121338, 1.1011948585510254, 0.6529481410980225, 0.14842158555984497, 1.580623984336853, 0.8663167953491211, 0.22441989183425903, 0.13743384182453156, 0.2185208797454834, -0.21506357192993164, 0.0035392988938838243, 0.020004404708743095, 0.49111831188201904, -0.5191987156867981, -1.4122276306152344, 1.5440324544906616, -1.3866878747940063, 0.00728784641250968, -0.16881366074085236, 0.2160131186246872, -0.17076189815998077, -1.474453330039978, -0.48396262526512146, 0.07906807959079742, 1.0818607807159424, -0.7581338286399841, 1.1859838962554932, -0.49636104702949524, -2.383648633956909, -1.2996612787246704, 0.7400893568992615, -1.2864704132080078, -0.1928226202726364, 1.4570063352584839, -0.5380982160568237, -1.2845194339752197, 0.8626517653465271, 0.3088275194168091, 0.35680821537971497, 0.9171653985977173], [-0.8040316104888916, 1.519992709159851, -1.1608591079711914, 1.149261474609375, 1.0270484685897827, 2.17814564704895, 3.1079483032226562, 0.1417119950056076, 0.9442086219787598, -0.17927402257919312, 1.4555599689483643, -1.70517897605896, 0.40831607580184937, 0.49959099292755127, 0.11915160715579987, -0.6334869861602783, -0.41502535343170166, 0.6202250123023987, -0.0877208262681961, 1.8514065742492676, -0.8803002834320068, 0.07036453485488892, 1.0517771244049072, 0.13787902891635895, -0.9868475794792175, -1.1646302938461304, 1.7837132215499878, 0.25759533047676086, 0.8055551648139954, 0.5589025020599365, -1.918229103088379, 0.387999951839447, -0.025932611897587776, 0.8046079277992249, -0.1275956630706787, -0.8257617354393005, 0.9280864596366882, 0.004852441139519215, 0.3532398045063019, 1.8085829019546509, -0.22911980748176575, 0.18733318150043488, -1.0052553415298462, 0.7123441696166992, 1.4777944087982178, -1.849145770072937, 0.3280477821826935, 0.6445904970169067, -1.1688752174377441, 0.6699751019477844, -0.012344873510301113, 0.22531767189502716, 1.2643795013427734, -0.8583652973175049, -1.0379115343093872, 1.0655643939971924, 0.5638550519943237, 0.4079650044441223, 0.3959166705608368, 0.8101760149002075, -1.0421432256698608, -0.9285051226615906, 0.7960891723632812, -1.0861181020736694, -1.1002888679504395, -1.3908666372299194, 1.6673448085784912, -1.0039691925048828, 1.1402997970581055, 1.2577099800109863, -0.3425007462501526, -0.407511442899704, -0.3312515318393707, 0.28318989276885986, -0.7294094562530518, -1.3920351266860962, -0.12677046656608582, -0.3274100124835968, 0.4148820638656616, 1.8443526029586792, -2.334810972213745, 0.6553019881248474, 0.7984901666641235, -0.1113511323928833, 0.5892977118492126, 0.40062400698661804, -0.09306532144546509, 0.49585679173469543, 0.7699498534202576, 1.4141466617584229, 1.1971614360809326, -1.6362059116363525, -0.17923137545585632, 0.8947418332099915, -0.7050076127052307, 0.8544521331787109, 0.19480350613594055, -1.6022688150405884, 0.7893011569976807, 0.44023874402046204, 1.096484899520874, -2.0411033630371094, 0.2424459457397461, 1.8818888664245605, 0.5093131065368652, 0.9910082221031189, 0.06984297186136246, -1.0776822566986084, 0.08748328685760498, -0.14255055785179138, 0.024499909952282906, -1.1382420063018799, -0.1449258029460907, -0.926518976688385, 0.7270668148994446, -0.917891800403595, 1.7505831718444824, -0.4342622458934784, -0.24880501627922058, 0.012614450417459011, -1.3377848863601685, -0.450918585062027, 1.208335518836975, -1.5547460317611694, -0.803722083568573, -1.1762136220932007, 0.21427221596240997, -1.884092092514038, -1.4498443603515625], [0.10253933072090149, 0.6811447739601135, 0.7495591640472412, -1.0062230825424194, 1.7283374071121216, 0.06998007744550705, 0.3302765190601349, 0.023749085143208504, -0.457660436630249, 0.041011497378349304, 0.34735241532325745, -0.5256065726280212, -0.06645359843969345, 1.3673045635223389, -0.8987117409706116, 1.0899946689605713, 0.994558572769165, 0.738300621509552, -0.6739393472671509, 0.1689460277557373, -0.1156640499830246, -0.2330952137708664, -0.032141949981451035, 0.8196761012077332, -1.97171950340271, 0.5551366209983826, 0.7065513730049133, -0.06388536840677261, -1.6087260246276855, -0.43991783261299133, -0.7995920181274414, -0.6510220766067505, -0.16518327593803406, 1.0409927368164062, 1.0658141374588013, -1.1134394407272339, -1.8628789186477661, 0.05096304789185524, -0.9095177054405212, 1.0254534482955933, 0.11183561384677887, -1.3280879259109497, 0.2960105240345001, 0.6433572769165039, 0.4524749219417572, 0.7625051736831665, 0.3320939540863037, -0.25377774238586426, 0.32521453499794006, -1.4620635509490967, -0.7202674746513367, 1.6440283060073853, 0.4493964612483978, -0.20750850439071655, -0.13536030054092407, -0.5779527425765991, 0.7291483283042908, 2.1484086513519287, 0.86177659034729, -0.3269558250904083, -0.8421677350997925, -1.942793846130371, -0.9833545684814453, -0.04227948561310768, 0.39237743616104126, -0.057494863867759705, 0.19733524322509766, 1.273354411125183, -0.40651607513427734, 0.27330997586250305, 0.26164478063583374, 0.336235910654068, -0.9507148861885071, -1.8018242120742798, -0.600718080997467, 0.22971519827842712, -1.3598709106445312, -0.8673124313354492, -0.18603333830833435, 0.8059786558151245, 0.18840597569942474, -1.7843157052993774, 1.2599127292633057, -0.8629598617553711, 0.505037248134613, 1.3978757858276367, 0.4608544409275055, -0.5195610523223877, 0.35195043683052063, 0.053895045071840286, -0.7791953682899475, 0.1875295639038086, 0.9230116605758667, 1.7135677337646484, -0.5472008585929871, 0.0005192155367694795, -0.009474708698689938, -0.010368261486291885, -0.2035582810640335, -1.6723179817199707, -0.26256847381591797, 0.8364897966384888, -0.57582026720047, -0.2512509524822235, 0.4338781237602234, 0.8502956628799438, 0.08607025444507599, 1.3285365104675293, 1.4888123273849487, 0.8870694041252136, 0.5410339832305908, -0.3668503165245056, -0.15070849657058716, -1.502062201499939, 1.016793131828308, -1.1337286233901978, -0.05928043648600578, 0.993811845779419, -0.9728712439537048, -1.3139050006866455, 0.44447943568229675, -0.09759611636400223, -0.277981698513031, 0.2530031204223633, -0.3828338384628296, -1.0590558052062988, -2.1080710887908936, 0.727567732334137, 0.1564134955406189], [-0.9083629846572876, -0.19561754167079926, -1.139486312866211, -1.1901135444641113, 0.45862409472465515, -0.17643028497695923, 1.3705391883850098, -0.7279812097549438, -1.0684422254562378, 0.18087469041347504, 0.22830256819725037, 1.2063584327697754, -0.36170557141304016, 2.3563733100891113, 0.2942366302013397, 2.261789321899414, -0.3255174160003662, -0.6838351488113403, -0.4291161894798279, -0.019244642928242683, 0.6745604276657104, -0.08041710406541824, -1.716208815574646, 0.721942126750946, -0.8283374309539795, -2.4937245845794678, 0.7802914381027222, 0.3594401776790619, -0.9773285984992981, 1.178552269935608, 0.06100073456764221, -0.9712516069412231, -0.38764771819114685, 0.42650851607322693, 0.06386491656303406, -0.9321767091751099, 0.252522736787796, 0.25367864966392517, -0.9900394678115845, 0.5142202377319336, 1.7234992980957031, -0.05970761179924011, -0.7262514233589172, 0.40197452902793884, 0.5483701229095459, 1.8932493925094604, 0.3520200550556183, 0.13188228011131287, -0.31117624044418335, -0.9875479340553284, 0.21076035499572754, -0.7668723464012146, 0.5959267020225525, 1.6131213903427124, -0.29839879274368286, -0.32441505789756775, 1.159393310546875, 0.5125437378883362, 0.3240939974784851, 1.8636751174926758, 0.2439100295305252, -1.149166226387024, -1.5159832239151, -0.5185770392417908, 0.19629085063934326, 1.4055147171020508, -0.166510671377182, 0.6788498759269714, 1.2528351545333862, 0.8869682550430298, 0.6063039302825928, -0.20976220071315765, 1.121077060699463, 0.13885672390460968, 0.032906219363212585, 2.028742551803589, 0.043692294508218765, 0.7991084456443787, -1.1935917139053345, -0.37950363755226135, 2.452216625213623, -0.2258400171995163, 0.6259807348251343, 0.3398467004299164, 0.589871883392334, 0.23047973215579987, -0.9467495679855347, -0.5352916717529297, -0.8481860756874084, 1.7977994680404663, 1.5354427099227905, 0.5921110510826111, -0.1610964685678482, -0.4020521938800812, 0.5669423937797546, -0.9436392784118652, 0.6145484447479248, -1.1654247045516968, 0.43241652846336365, 0.8289528489112854, -1.4798766374588013, 0.3617522716522217, -0.24975021183490753, -0.9626812934875488, -0.9264981746673584, 0.7044958472251892, 0.7027077674865723, -0.04753986746072769, 0.4256839156150818, -1.74152410030365, 0.2854612469673157, -0.1699187308549881, 0.09491926431655884, -0.5457907915115356, -0.15557564795017242, -0.5182390809059143, 1.0100646018981934, 0.22198732197284698, 0.7307167053222656, 1.8836512565612793, 0.8342154622077942, -0.9143185019493103, 0.10370057821273804, -0.7436132431030273, 0.9823511838912964, -0.8458858728408813, -0.49719491600990295, -0.4677063822746277, 0.6633990406990051], [-2.829428195953369, -0.8363320827484131, 1.171447992324829, 0.14913906157016754, 0.6282578706741333, -0.2256443053483963, -0.7533105611801147, 1.241957187652588, -1.0255553722381592, 1.1029918193817139, 0.0011572513030841947, -0.35760796070098877, -0.716422438621521, -1.1223764419555664, -1.576786756515503, 1.2488949298858643, 0.13830459117889404, 1.147826075553894, -0.17103788256645203, -1.4943674802780151, 0.26354649662971497, -0.3326377868652344, -1.6692821979522705, -0.6694695949554443, -0.43533003330230713, -0.4634237587451935, -0.07861223071813583, -2.2934253215789795, 0.4484911262989044, 0.40678513050079346, -0.6191616058349609, 1.5517343282699585, -0.05027170106768608, 0.07778946310281754, 0.18506577610969543, -0.41914471983909607, -0.806175947189331, -0.501729428768158, 0.5600436329841614, 0.001160251908004284, 2.0795764923095703, -0.3798747658729553, 1.3712010383605957, -0.5174580812454224, -0.02752985619008541, -1.147769808769226, 1.1179171800613403, -0.16405177116394043, -1.0123882293701172, -0.04075870290398598, -0.22154824435710907, -0.2954851984977722, 1.2562296390533447, -0.5263699889183044, -0.1708921194076538, -0.7176383137702942, -0.04501382261514664, 0.5654994249343872, -0.885982871055603, -1.1544684171676636, -1.834862470626831, -0.8867221474647522, -0.6944208741188049, -1.3968151807785034, 0.014961072243750095, 0.27456361055374146, 0.3706296682357788, -1.1434661149978638, -0.08950255066156387, 0.1547941118478775, 0.262519896030426, -0.8227484822273254, 0.21836808323860168, 0.018075328320264816, -0.8357912302017212, -0.3470228314399719, -1.0986510515213013, -1.4108266830444336, 0.05167301371693611, -0.5899728536605835, 1.169404149055481, 0.2938980162143707, -0.9434730410575867, -0.8103120923042297, -0.31022217869758606, 0.6270309686660767, -0.2901720106601715, 0.3264395296573639, 0.27941644191741943, -1.2094571590423584, -0.6265676617622375, -2.2356374263763428, 0.8301280736923218, 0.36754053831100464, -1.4041801691055298, 0.15943554043769836, -0.01960408128798008, -0.7201178073883057, 0.5320368409156799, -0.10574622452259064, 1.1406344175338745, 0.869783341884613, 0.6542155742645264, 0.6398847103118896, 0.23778468370437622, -0.18777580559253693, -0.35069817304611206, 0.2501639127731323, -1.1020355224609375, -1.3770532608032227, 0.42838427424430847, -1.6050424575805664, -0.7048121094703674, 0.7650061845779419, -2.6196484565734863, -0.6759452223777771, -0.4730834662914276, 0.03783703222870827, 0.16665156185626984, 0.5029964447021484, 0.5461600422859192, 0.353054940700531, -1.3849189281463623, 1.332119107246399, -0.1394507735967636, -0.9318329095840454, 0.5589682459831238, -0.6904457807540894, -0.024052388966083527], [-0.25845280289649963, 0.17735189199447632, 0.28909239172935486, -3.0708730220794678, 0.03840210661292076, 0.4745115637779236, -1.3996968269348145, 1.384482502937317, 1.270048975944519, -0.05180284380912781, -1.110588788986206, 2.3980295658111572, -0.11778612434864044, -0.8266044855117798, 0.2327234297990799, 0.9313422441482544, 2.0960569381713867, -0.4691072106361389, -0.9773442149162292, -0.21299083530902863, -0.7760142683982849, -0.05691102147102356, -0.27480119466781616, -0.6666791439056396, -0.017641758546233177, 0.6217621564865112, -0.16842611134052277, 1.1584631204605103, -1.1211202144622803, 1.5156060457229614, 1.3971636295318604, -0.7648944854736328, -0.5837521553039551, -1.7976113557815552, -0.38967928290367126, -0.6095080971717834, -0.9401093125343323, 0.18947026133537292, -1.3882439136505127, -0.20164287090301514, 1.7933253049850464, 0.8843806982040405, -0.6412513256072998, 0.5705884099006653, -0.982818067073822, 0.45048046112060547, 0.2852752208709717, 0.6829749941825867, -0.14919814467430115, 0.13552650809288025, 0.04639948159456253, -0.09316930919885635, 0.21672937273979187, -0.03270378336310387, -0.28747063875198364, 0.9951386451721191, -0.3907106816768646, -0.6590014100074768, 0.8185814023017883, -0.2164301574230194, -0.12967374920845032, -0.5034369230270386, -0.3027757406234741, -0.796690046787262, 0.49646902084350586, 0.5807744264602661, 3.260798692703247, 0.5401224493980408, 0.055390018969774246, 0.6203228235244751, 0.07003504037857056, -0.2430870234966278, 2.0968832969665527, -0.638694703578949, -3.0696346759796143, 1.091224193572998, -1.3012924194335938, -0.4669576585292816, 1.0731263160705566, 0.4347838759422302, -0.33272886276245117, -0.8620569109916687, 1.3076225519180298, -0.8742709159851074, 0.10512173920869827, -1.1196175813674927, -1.0682436227798462, -0.5052867531776428, -0.9323559403419495, -0.5709806680679321, 0.1830417960882187, -0.9538807272911072, -0.031189890578389168, 1.007391095161438, 0.523892879486084, -1.2442288398742676, 0.8186275959014893, -0.16997528076171875, -0.0957060232758522, -0.0026253696996718645, -1.7728129625320435, 0.6132956147193909, -2.1023499965667725, 2.9464688301086426, -0.5215774774551392, 0.7408061623573303, -0.27767130732536316, 1.383636236190796, 0.4820455312728882, 0.7009087204933167, -0.5190591812133789, 0.9611027836799622, 0.19266995787620544, 0.36551401019096375, 1.3693015575408936, -1.3079530000686646, 1.09649658203125, -1.4823050498962402, 0.7282249331474304, 0.4856576919555664, -0.9087311029434204, -0.3431325852870941, -1.0990923643112183, 0.7247541546821594, -1.296141266822815, 0.3608754575252533, 0.4113706648349762, 0.09608970582485199, -1.9984793663024902], [1.3284156322479248, -0.33835431933403015, -0.07194297760725021, 0.4885898530483246, -0.28029584884643555, -2.0280370712280273, -0.7588826417922974, 0.6282875537872314, 1.159561038017273, -0.6555151343345642, 0.6520562171936035, -0.6638133525848389, -0.4356885254383087, -0.3739387094974518, -0.24636892974376678, -0.8015493750572205, 1.0795791149139404, -2.161240816116333, 0.5298897624015808, -0.4680282771587372, 0.282735139131546, 1.3452318906784058, 0.517026960849762, -1.635339617729187, 1.13222336769104, 0.15291567146778107, -0.833736002445221, 0.3991335928440094, 1.4725888967514038, 0.666306734085083, -0.09942858666181564, 0.6863148212432861, 0.5607948899269104, -0.9636158347129822, -0.2376052886247635, -1.4877820014953613, -0.18587638437747955, 0.24614623188972473, 1.207923173904419, 1.2118884325027466, -0.4250587224960327, -0.6975859999656677, 0.8827309012413025, -0.5210543274879456, 0.2846255302429199, 0.29616451263427734, -0.2090522199869156, -0.5632879137992859, 0.5471373200416565, 2.1879234313964844, -1.1507911682128906, -1.6153148412704468, 0.6546978950500488, 0.14131581783294678, -1.9561610221862793, -0.8329424858093262, -1.5809478759765625, -0.5874724388122559, -0.8652029633522034, 0.285905659198761, 0.8255244493484497, -0.43958958983421326, -0.6584716439247131, -0.7717723250389099, -0.595605194568634, -1.2458250522613525, 0.802603542804718, 0.4003826379776001, -0.3126142919063568, -1.4287725687026978, 1.7286829948425293, 0.23975035548210144, -2.1444249153137207, 1.3853745460510254, -0.1561983972787857, -0.1316455900669098, 1.4766207933425903, 0.5471876263618469, -0.3111615478992462, 1.293716311454773, 0.3829905092716217, 0.473736047744751, -0.5067773461341858, -1.2379165887832642, -0.6320947408676147, 0.6475493311882019, 0.5683266520500183, -0.9724719524383545, -1.6865178346633911, 0.37231212854385376, -1.1640794277191162, -0.16665133833885193, -0.08165320008993149, -0.5667225122451782, -0.14160992205142975, -0.9229192733764648, 0.09995127469301224, 0.9141054749488831, 0.056166112422943115, -1.048610806465149, 2.9624438285827637, 0.2547949254512787, -0.743205726146698, 1.5388416051864624, -0.7064905762672424, -1.54038667678833, -0.54656982421875, -0.42532116174697876, 1.0134795904159546, 1.3046518564224243, -1.2440013885498047, 0.897066593170166, 0.2161814421415329, -0.02323298342525959, 1.004002571105957, 0.13223136961460114, -0.7894979119300842, 0.0001365485804853961, 1.2678816318511963, 2.062906265258789, -1.2811908721923828, -0.8861937522888184, 1.0204743146896362, -0.010988069698214531, 2.1701550483703613, 1.713783860206604, -0.3451320230960846, 1.3328756093978882, 0.008480585180222988], [-0.9737719297409058, -0.15850844979286194, 0.16878682374954224, 1.1492729187011719, -0.23197869956493378, 0.44928407669067383, 0.6031321883201599, -0.7236024141311646, 0.13599073886871338, 0.21928583085536957, -0.6191747188568115, 0.04650375247001648, -1.5301320552825928, 1.933591365814209, -1.2967228889465332, -0.811219334602356, 0.2026986926794052, 1.6682758331298828, 0.7293566465377808, 1.1732150316238403, -1.2282569408416748, 0.6583133935928345, -0.42157280445098877, 0.5761801600456238, 0.0593033991754055, 1.4316071271896362, -0.08571840822696686, -2.476689100265503, 0.07013356685638428, 0.6295205950737, 0.3811950981616974, 0.1369723230600357, 0.017115138471126556, 0.204189270734787, -0.10071436315774918, 0.12548816204071045, -0.5048995614051819, 0.3081390857696533, -1.7625800371170044, -1.0332907438278198, -0.3914406895637512, -0.6984541416168213, -1.1102464199066162, 0.08690255880355835, -0.5139275193214417, -0.1924399882555008, -0.13548196852207184, 1.442763328552246, -0.9692329168319702, 0.1705331802368164, 0.21465478837490082, 0.9881694912910461, -0.11673580855131149, 0.24785178899765015, -1.2775758504867554, 1.4062644243240356, -2.0920872688293457, 1.240960955619812, -1.1177221536636353, -1.4569523334503174, 0.5356142520904541, 1.1298863887786865, 0.2301018387079239, -1.027254343032837, -1.5970187187194824, -0.04886859282851219, 2.316175699234009, -0.7263001799583435, 0.5260576009750366, 0.32544153928756714, -2.653665542602539, -0.19171789288520813, -0.6185555458068848, -0.9653412699699402, 1.4764074087142944, 0.25028836727142334, -0.42551058530807495, 0.8216333389282227, -0.79692542552948, -0.9239258766174316, -0.9505664110183716, -0.24613702297210693, -0.1510872095823288, 0.3651028573513031, -0.2707526981830597, -0.5910879373550415, 1.1667039394378662, -0.25825148820877075, -0.6922810673713684, -0.049836017191410065, 0.938380777835846, 0.26247283816337585, -0.7784848213195801, -0.15297338366508484, -1.2581121921539307, 0.2775469422340393, -0.09191786497831345, 0.8757408857345581, -0.04887773469090462, 1.0136982202529907, 0.37362387776374817, -0.3703868091106415, -1.5904016494750977, 0.15715964138507843, -0.37873032689094543, 0.4637128412723541, -1.0453153848648071, -0.4467523396015167, -0.7047529816627502, -0.38708633184432983, -0.10097189992666245, 0.5236956477165222, 1.4629544019699097, -1.402208924293518, 1.4854607582092285, 0.41633957624435425, 1.067034125328064, -0.6884129047393799, -1.2999520301818848, -0.8723495006561279, 0.13106273114681244, -0.34697428345680237, -0.7489137053489685, -0.7406792640686035, -1.5928224325180054, -0.31848853826522827, -0.2526710331439972, -0.602673351764679, 0.32959961891174316], [-0.02376081794500351, 0.8486085534095764, 2.268413543701172, -1.643557071685791, -0.9550105333328247, 1.0903582572937012, 0.2432784140110016, -1.0960445404052734, -0.8613528609275818, 0.3180707097053528, -0.8024107217788696, -0.2132013589143753, -1.5433189868927002, -0.1441250592470169, 0.43796613812446594, 0.9935515522956848, 0.8345699310302734, 0.9519842267036438, -0.2487628161907196, -0.3159661293029785, -0.8471722602844238, -0.43564507365226746, -1.133064866065979, -0.21518085896968842, 0.13466912508010864, 1.0097795724868774, -1.2853559255599976, 0.836384654045105, 0.7358831763267517, -0.7770033478736877, -0.7392594814300537, -0.05807375907897949, -2.3435215950012207, 0.9948322176933289, 0.03689982369542122, -0.5136770606040955, -0.12096115201711655, -1.1169487237930298, -0.30560630559921265, -0.8485236167907715, -1.0281068086624146, -0.4246712327003479, 0.3409777283668518, -0.38390088081359863, -0.7789840698242188, 1.0982614755630493, 0.8524785041809082, 1.0476164817810059, -0.6131033897399902, 0.8125630021095276, -1.5557883977890015, 1.902301549911499, -0.2663966119289398, -0.2532992959022522, -0.0571698360145092, -1.0404201745986938, -0.09104589372873306, 0.004570977762341499, 1.2671658992767334, -0.30852195620536804, -2.266066551208496, 0.2447357177734375, -2.2360429763793945, 0.19652174413204193, -0.6494913101196289, 1.6466490030288696, 1.3670473098754883, -0.21372567117214203, -0.09479733556509018, -0.4657069146633148, 1.653803825378418, 1.7976710796356201, 1.4675166606903076, -0.8903022408485413, 0.8400899171829224, 0.39249518513679504, -0.001101463451050222, 0.08404502272605896, 1.3527660369873047, 0.3283591866493225, -0.20173661410808563, 1.821926474571228, 1.2553032636642456, -0.6620701551437378, 0.11032175272703171, -1.4013099670410156, -0.968668520450592, 0.18615511059761047, -0.6751450300216675, 0.9470805525779724, -1.1913046836853027, 0.6724292635917664, 1.4156616926193237, 1.4403774738311768, 1.3069051504135132, 0.05929010733962059, 0.874530553817749, -0.09366508573293686, -1.7098333835601807, -1.3194177150726318, -0.5949008464813232, -1.270472526550293, 0.49786534905433655, -1.603140115737915, 0.6253541707992554, 0.23097820580005646, 0.6572432518005371, -0.5686506032943726, -0.15162190794944763, -0.6752396821975708, -0.7390108704566956, -0.5520259141921997, -2.447995662689209, -0.662428081035614, -0.11610481888055801, 0.18750064074993134, -0.395294725894928, -0.8281757235527039, 0.8180021643638611, 1.020757794380188, 1.3789440393447876, -0.6935267448425293, -0.24947014451026917, -0.8620426058769226, 0.7843547463417053, -0.42088013887405396, -0.45072758197784424, 0.8366899490356445, -0.5705210566520691], [0.7638146281242371, -0.4813365340232849, 1.196714162826538, -0.9645722508430481, 0.5129532814025879, -0.11326563358306885, 1.5433063507080078, -0.03115023672580719, 0.7775412797927856, 0.7086233496665955, -0.29830077290534973, 1.2311797142028809, -1.631583333015442, -0.22165891528129578, 2.7690353393554688, 0.025020696222782135, 1.6995021104812622, -0.5002928972244263, -1.202555537223816, 0.12486537545919418, -1.5789501667022705, 0.8374406695365906, -0.5015532374382019, -1.0390760898590088, -0.7589328289031982, 0.22667667269706726, 0.8610799312591553, -0.26222801208496094, 0.5591612458229065, -0.6101051568984985, 1.0524210929870605, 1.0319736003875732, 1.2460476160049438, -0.056719716638326645, 1.386328935623169, -0.6571522355079651, 0.3665666878223419, -0.28636032342910767, 0.4383663535118103, 1.6768232583999634, -0.13721464574337006, -0.10037094354629517, -1.6847401857376099, 2.298415184020996, 0.12076243758201599, -1.1860089302062988, 1.2984044551849365, 0.5337979197502136, 0.014437641017138958, 0.8929848670959473, 0.2949116826057434, 1.1362923383712769, 2.318056583404541, 0.17951276898384094, -1.484315037727356, -0.27775824069976807, 1.0789257287979126, -1.4026328325271606, -2.2177555561065674, 0.5786516070365906, -0.33508530259132385, -0.9623562693595886, -0.5555528998374939, 1.3766164779663086, 0.4971636235713959, -0.05045159533619881, 0.44162583351135254, -0.22229568660259247, -0.796667754650116, 1.167403221130371, 0.5614586472511292, 0.2275160849094391, 0.47707700729370117, 1.0550165176391602, -0.6139931678771973, 0.0803380012512207, 0.055206041783094406, -3.3620009422302246, -0.1992136836051941, -2.012150526046753, 0.8497992157936096, 0.2522464096546173, -0.24319234490394592, -0.8827370405197144, 0.02905110828578472, 1.2956205606460571, -0.10900554060935974, 1.4245325326919556, 0.2946057617664337, 0.27188727259635925, 0.803176999092102, 1.3621318340301514, -0.27409109473228455, -0.08996190875768661, -1.289607286453247, -0.33754995465278625, -0.7673580646514893, 0.970454216003418, -0.8660097718238831, 0.10450932383537292, 0.30635467171669006, -0.8948065042495728, 0.15011198818683624, -0.4989137053489685, 2.0725033283233643, 0.418719619512558, 0.4517139494419098, -1.498366117477417, -0.5458614826202393, -0.7528539299964905, -0.2879566252231598, -1.2630101442337036, -1.2876510620117188, -0.40411248803138733, 1.160460114479065, 1.1140824556350708, -1.4567193984985352, 1.368381142616272, -1.0733238458633423, 0.21645186841487885, -1.0079816579818726, 1.3572360277175903, 0.33736279606819153, 0.547396183013916, -0.5257800817489624, 0.622261106967926, 0.20032429695129395, -1.0557582378387451, -0.634183943271637], [1.18390953540802, -1.8964396715164185, 0.8130945563316345, -0.08062324672937393, -1.6899421215057373, 0.4046914577484131, 1.314531683921814, 1.2032755613327026, -2.0455286502838135, 0.987922191619873, 0.473591148853302, -0.2902805507183075, 0.45722052454948425, 0.3345576524734497, 1.246425986289978, -0.7424514889717102, -0.3033919036388397, -1.495784044265747, 1.6483416557312012, -0.4948939383029938, 0.875133216381073, -1.2712420225143433, 3.0255396366119385, -1.2495803833007812, 0.5269227027893066, -0.07620001584291458, 1.4201257228851318, -0.679214596748352, 1.850766897201538, -0.3045273423194885, 1.0367647409439087, 0.6801360249519348, -0.6201523542404175, -0.3246871829032898, -0.4272672235965729, -0.5432990789413452, -0.3719445466995239, 0.11165273934602737, -0.42222121357917786, 0.1514226347208023, 0.4527199864387512, 0.7675317525863647, -0.18175935745239258, -0.009729133918881416, 1.227690577507019, -1.0536786317825317, 1.3267273902893066, -0.4549376964569092, 0.2919783294200897, -1.9829450845718384, 0.2199752926826477, -0.36711153388023376, -0.3176634609699249, 0.44621381163597107, 1.305628776550293, 0.5367918014526367, -1.7681609392166138, -0.7510692477226257, 0.4796532094478607, 1.6014835834503174, 0.3941726088523865, 0.15967153012752533, -0.3934474587440491, -0.5253540277481079, -0.5269305109977722, -1.093680739402771, 1.1660983562469482, 1.0773277282714844, -1.1417738199234009, -0.19330470263957977, -0.7725828886032104, 0.19291025400161743, -1.3779070377349854, 0.7425529956817627, -0.7307008504867554, -1.1281152963638306, -1.5409369468688965, -0.4046250581741333, 0.43534496426582336, -0.951265275478363, 0.1301151067018509, -0.14763209223747253, 0.6842885613441467, 1.1652120351791382, -0.9349215626716614, 0.1650828719139099, -0.5534809231758118, -0.36220142245292664, 0.07859957963228226, 1.46673583984375, 0.18551942706108093, 0.07340070605278015, 0.6360334753990173, -0.9332900643348694, 0.18873590230941772, -0.9225298762321472, 3.0967154502868652, 0.029046257957816124, 0.13043518364429474, -0.4991694688796997, -1.2202937602996826, 0.1965213418006897, -1.0130435228347778, -0.37239620089530945, 0.609742283821106, -0.9408767819404602, -0.2722735106945038, -0.24208296835422516, 1.7917500734329224, -1.7250449657440186, 0.43087080121040344, 0.1564827412366867, 1.2624092102050781, 0.16802555322647095, 0.11997488141059875, 0.015740767121315002, -0.9376405477523804, 0.3929857611656189, -1.5243754386901855, -0.9724656939506531, -1.262000560760498, -0.4236074686050415, -1.2273681163787842, -1.217024326324463, -0.88885897397995, -0.5167860388755798, 2.4667184352874756, -0.22621393203735352, -0.5644509196281433], [2.1756482124328613, -0.7093021869659424, -1.979527235031128, -1.1200109720230103, -0.9213590025901794, -0.026939231902360916, 2.120448112487793, -2.520780563354492, -0.855504035949707, 0.4905543029308319, 0.4570103883743286, -0.6030164957046509, 2.4852280616760254, 0.7279641032218933, -0.04063532501459122, -0.42999619245529175, -0.04277844727039337, -0.0005512235220521688, -2.4381823539733887, 0.24646209180355072, 0.9070688486099243, 1.7856582403182983, -0.5781275033950806, 0.21465055644512177, 0.7133657336235046, -1.2271606922149658, -0.31087198853492737, -1.5356359481811523, 0.4848966598510742, -1.8612737655639648, 1.1456329822540283, 0.037686266005039215, 1.4078832864761353, -0.9411913752555847, 1.113156795501709, -0.8791446089744568, -1.2528148889541626, -1.3610066175460815, -1.9850378036499023, -1.4144176244735718, 0.5429486036300659, 0.2930389642715454, -0.4539683163166046, -0.43439623713493347, -0.6468045115470886, 0.6534615159034729, 1.8472883701324463, 0.4298468828201294, 0.5207343697547913, -0.361195832490921, -2.076760768890381, 0.05468888580799103, 0.8469943404197693, -0.1854715347290039, -0.11277027428150177, -0.6746922731399536, 0.0019568002317100763, -0.47533130645751953, -0.8976393938064575, 1.184341549873352, 0.5298531651496887, -0.761634349822998, 1.0929019451141357, 0.14142157137393951, -0.08254308998584747, 0.22920756042003632, -1.4116853475570679, -0.4954572916030884, -2.1694869995117188, -0.22945836186408997, -0.8065984845161438, -0.13606059551239014, 0.21680188179016113, -0.41095784306526184, -0.05412780120968819, -1.383198857307434, 0.25161153078079224, 0.5943487882614136, -1.2401024103164673, -0.30674320459365845, 0.4446505904197693, 0.6060362458229065, 0.17204627394676208, -1.2483985424041748, 0.22537869215011597, 1.9309115409851074, 1.4263509511947632, -1.4670383930206299, 1.0490614175796509, 1.990413784980774, -0.4905862808227539, -0.045131053775548935, 1.5548251867294312, 0.8005449175834656, 1.6475238800048828, 0.021394474431872368, 0.49665069580078125, 0.9436700344085693, -1.1561477184295654, -1.4850331544876099, -1.7049661874771118, 0.3732735812664032, -0.32460644841194153, -0.5783965587615967, 1.4797002077102661, -1.604849100112915, -0.013797455467283726, -0.8340655565261841, -0.4569213092327118, -1.9753648042678833, -1.7626873254776, 0.45202842354774475, -0.8119120597839355, -0.3515462279319763, 0.2653445899486542, -0.43383046984672546, 0.32667720317840576, -1.126135230064392, -1.332696795463562, 0.32878056168556213, 0.3767028748989105, -1.375964641571045, 0.13494770228862762, 1.3957359790802002, 1.703667402267456, -0.28743240237236023, -1.0582892894744873, 0.387583464384079, -1.4732391834259033], [-0.2994285821914673, -0.08158539980649948, 0.5203696489334106, -0.8620033860206604, 1.4758234024047852, 0.37558114528656006, -0.3526307940483093, -0.06507737934589386, -0.47907450795173645, -0.500201940536499, -1.0089137554168701, -0.39053353667259216, -0.5152512788772583, -0.08940600603818893, 0.1588938981294632, 0.638897180557251, -0.16498199105262756, 0.4778590500354767, 0.8456026315689087, 1.0572792291641235, -1.0621259212493896, 0.32548579573631287, 0.9768009185791016, -0.8090158700942993, -1.7025954723358154, 0.2986389100551605, -1.9892524480819702, 0.0574738048017025, -0.43398475646972656, 0.4640607535839081, -0.5875418186187744, 1.234657883644104, -1.7866625785827637, -0.19575567543506622, 0.4859917461872101, 1.1049494743347168, -0.19170749187469482, 0.1586330384016037, -0.7869908809661865, 0.5900783538818359, 0.006726202089339495, 2.223644733428955, 0.13425573706626892, -0.1302614063024521, -1.8466334342956543, -0.4902203381061554, -0.8998542428016663, 0.20636217296123505, 2.393291473388672, 1.1662001609802246, 0.088901087641716, 1.1364091634750366, 0.5218166708946228, -0.28540828824043274, 0.2685837149620056, -0.8886013627052307, 0.6886254549026489, -0.18563786149024963, 0.6146701574325562, 0.13232742249965668, 0.7299671173095703, 0.9419330954551697, -0.6814804673194885, 0.20361879467964172, -0.03744051605463028, -0.5539482831954956, 0.21565303206443787, -0.6096818447113037, -0.06247292831540108, -0.8962939381599426, 0.09132417291402817, -1.3393350839614868, -0.649350643157959, 1.1305592060089111, 0.4314766824245453, 0.39733025431632996, 0.5582814812660217, 1.2433314323425293, 0.07439740747213364, 0.01499826181679964, 0.20462028682231903, -0.6025509834289551, 0.17795725166797638, -1.1292312145233154, -0.9511613845825195, -0.6508280634880066, 0.49159419536590576, 1.721907138824463, 0.14718593657016754, 0.5269640684127808, -1.5931240320205688, -0.8011152148246765, -0.15771080553531647, -1.4854227304458618, 1.0019466876983643, -2.034421920776367, 0.3055076599121094, -1.3275421857833862, 0.5447174906730652, -2.092538356781006, -0.7930728197097778, -1.168717622756958, 1.9727411270141602, -0.8324037790298462, 0.053181104362010956, -0.8930434584617615, -1.2779759168624878, 0.5760074257850647, -1.6578084230422974, -0.06558572500944138, 1.2635539770126343, -2.1518173217773438, -1.4132088422775269, -0.504410445690155, -1.7607102394104004, -0.0925905853509903, -1.0825868844985962, 0.41760560870170593, 1.0265358686447144, -0.6394947171211243, 0.897570013999939, 0.48474910855293274, -0.2012193351984024, 2.05145525932312, -0.3943244516849518, 0.798412561416626, -0.7159296274185181, 1.2549794912338257, -0.9188180565834045], [0.8086672425270081, -1.0166586637496948, 0.6922920346260071, -1.0839958190917969, -0.017214199528098106, -1.0392131805419922, -1.062487244606018, 0.8269826173782349, 0.6979665160179138, 0.7376768589019775, 1.470857858657837, -0.17573300004005432, 0.7536222338676453, 0.35666316747665405, 0.8363246321678162, -1.2983425855636597, -0.888352632522583, -1.153165578842163, -0.2428537905216217, 0.7777538895606995, -0.9328120946884155, -0.02702440693974495, 2.465470314025879, -1.7103819847106934, 0.39804786443710327, -1.5771241188049316, 1.376143455505371, -0.3575517237186432, 1.4303101301193237, 0.15925011038780212, 2.8323981761932373, 0.5968571305274963, 1.0529474020004272, 0.547454297542572, -1.124943733215332, -0.7076161503791809, 0.18992842733860016, -0.6043305397033691, 0.25828298926353455, 2.2781665325164795, -0.8327084183692932, -1.8195009231567383, -0.20040170848369598, 0.7430539727210999, -0.41329053044319153, -1.8522100448608398, 1.512898564338684, 0.09278397262096405, 1.0038564205169678, -1.2598843574523926, -0.2099066823720932, 0.6302400827407837, -1.025559425354004, 0.7495953440666199, 0.1627071499824524, -0.7211161255836487, 0.2925303876399994, -0.31187906861305237, 3.566696882247925, -0.5738747119903564, 1.0755008459091187, 0.03676208108663559, -0.6496471762657166, -1.394601583480835, 0.9316415190696716, 1.1247080564498901, 0.1905873566865921, -0.95767742395401, 0.86822110414505, -0.9212387800216675, -0.23637808859348297, 0.275918573141098, 0.1659880429506302, 0.00490029389038682, -0.8798604607582092, 1.4862439632415771, 0.050577837973833084, 1.7333444356918335, -0.8673496246337891, -0.19763748347759247, 0.5797945857048035, 0.017901182174682617, -0.6040557026863098, 0.9914702773094177, 0.37224283814430237, -0.09732998162508011, 0.541365385055542, 1.4851874113082886, -0.02516375109553337, -1.4229176044464111, 0.6422585844993591, -0.2507391571998596, 1.1639803647994995, 0.5956729650497437, 0.4547120928764343, -0.5631006360054016, 0.9694945812225342, -0.18964675068855286, 0.5895559191703796, -1.9429925680160522, -0.46820583939552307, -1.182196021080017, 1.2588311433792114, -0.08257276564836502, -0.5694077610969543, -0.32936763763427734, 1.5597236156463623, 0.20938919484615326, -0.3189171254634857, 2.0986766815185547, -0.5776154398918152, 0.8503084182739258, 0.13797935843467712, -1.31172776222229, 0.20527103543281555, 2.011763334274292, 1.6785926818847656, -0.569006085395813, 1.8962494134902954, -0.34429600834846497, 0.16999241709709167, 0.46267983317375183, -0.555778443813324, -1.2381409406661987, -0.49217694997787476, 0.12421390414237976, 2.2201664447784424, -0.31596285104751587, 2.3596768379211426], [-1.1331710815429688, -0.5851767659187317, 0.9093437790870667, -0.03560386598110199, 1.582275629043579, 0.3081003427505493, 1.1672592163085938, 1.1579726934432983, 0.0921458750963211, 1.6002205610275269, 0.8646955490112305, 0.8681936860084534, -0.4585748314857483, -1.30618417263031, 0.02797366864979267, -0.3805283010005951, 1.2815526723861694, 1.2109497785568237, -0.3600477874279022, -0.59527188539505, -1.7335197925567627, -0.3022212088108063, -0.3409731686115265, -1.4388625621795654, 0.3048524260520935, 0.4877268970012665, 0.41855600476264954, 0.22389619052410126, 1.1156806945800781, -0.7844412326812744, 0.04415666684508324, -0.2900286018848419, -1.3254523277282715, 1.5770280361175537, -2.8351998329162598, -2.6664769649505615, 0.831149697303772, -0.14726987481117249, -0.7376498579978943, -1.0554249286651611, 0.19478757679462433, 0.23903580009937286, -0.35202765464782715, -1.2673678398132324, -1.1567368507385254, -1.0291482210159302, 1.1019847393035889, -1.993284821510315, 0.9570998549461365, -0.4535759687423706, 0.24612955749034882, -1.1607098579406738, 0.5074301362037659, -0.5284023284912109, -0.16834492981433868, 0.7941745519638062, -0.144069641828537, 0.1929987370967865, 0.6229926347732544, 1.0469080209732056, 0.6668368577957153, -1.1040533781051636, 1.8146954774856567, 1.2244335412979126, -2.031348943710327, 2.1784186363220215, 0.787839412689209, 0.59527587890625, -1.8068370819091797, -0.93398118019104, 0.6961393356323242, -1.6983567476272583, 0.19549071788787842, 1.3944673538208008, -1.1227911710739136, -0.7887821793556213, -0.29683852195739746, -0.7505322098731995, 0.38120147585868835, -0.660528838634491, 0.11920666694641113, 0.4811707139015198, 0.1470337063074112, 0.5608357787132263, -0.5583468675613403, 0.6701717972755432, -0.55296790599823, 1.2545610666275024, 0.17420648038387299, 0.22942934930324554, -1.5041999816894531, 1.059809923171997, 0.0014705103822052479, -0.28602027893066406, -0.08589266240596771, 0.2919800877571106, 0.7087437510490417, 0.19896318018436432, -0.3315902054309845, -0.1484960913658142, 1.1254817247390747, 0.19298693537712097, 1.168100118637085, 0.6647030115127563, 0.12373651564121246, -0.20213401317596436, -0.48943522572517395, 0.6069508194923401, 0.8398755788803101, 1.1075812578201294, -0.25035396218299866, 0.0775013417005539, -0.9705970883369446, -1.367004156112671, -0.605018675327301, -0.5556749105453491, 0.22739088535308838, 0.26759588718414307, -1.3186776638031006, -0.725933313369751, -0.921491801738739, 1.6417728662490845, -0.5776394605636597, 0.5934991836547852, -0.3530024588108063, 0.7027620673179626, 0.37094396352767944, -0.20044004917144775, -1.4126983880996704], [-0.8406813144683838, -2.5378358364105225, -0.46746382117271423, 1.1770488023757935, 1.5525351762771606, -1.4181562662124634, 1.8664672374725342, -0.4528236389160156, 0.10482759028673172, -0.017422879114747047, -0.3836669623851776, 1.2593107223510742, -1.7147465944290161, -0.16783487796783447, 0.3638227880001068, -0.4881334602832794, -0.24832279980182648, 0.4065961241722107, 0.005752502009272575, -1.133338212966919, 1.3966262340545654, 0.26125866174697876, -1.4582246541976929, 0.7853308916091919, 0.33056995272636414, 0.5193783640861511, -1.38601553440094, 0.40251991152763367, 0.055664531886577606, -1.1057538986206055, -0.72127366065979, -0.4295557737350464, 1.084636926651001, -0.3570694625377655, -0.8955020904541016, 1.2378815412521362, 0.6452726721763611, -1.3970509767532349, 1.2998405694961548, -0.5307924151420593, -0.6104456186294556, -0.8040205240249634, -0.20040437579154968, -0.9886823892593384, 0.7839357852935791, -0.00795072689652443, -0.4320424497127533, 0.35428348183631897, -1.357872724533081, -0.29085367918014526, 0.4661182165145874, -0.4235790967941284, -0.9196602702140808, 0.667118489742279, -0.6993511319160461, 0.7550055384635925, 1.464568853378296, 2.1733548641204834, 0.18355782330036163, 0.6897122859954834, -0.2655157148838043, 1.1879029273986816, -0.7181258201599121, -1.0754398107528687, -0.39332273602485657, -0.17638446390628815, -0.30181026458740234, -0.3672983944416046, 1.2359517812728882, -0.41667142510414124, 1.0922223329544067, 2.450259208679199, 0.5364519953727722, -0.20089948177337646, -0.4953397810459137, -1.2649370431900024, 0.7943808436393738, -0.7987061142921448, -0.44852861762046814, 1.2003227472305298, -0.19859685003757477, 0.8042611479759216, 1.7749614715576172, -0.5433176159858704, -0.44702425599098206, 0.04286764934659004, -1.3309409618377686, -1.0839978456497192, 0.35895514488220215, 0.6561610698699951, 0.4534637928009033, -1.1198322772979736, 0.4171026051044464, -0.14212262630462646, 1.6958431005477905, -0.3540489375591278, -0.2364598661661148, 1.35062575340271, 0.8182324171066284, -1.7222861051559448, -0.47292235493659973, 1.8617076873779297, 0.6237752437591553, -0.7965987920761108, 0.6077949404716492, 0.7662340402603149, 0.48645099997520447, 0.6238567233085632, 1.7699246406555176, 0.17448711395263672, -0.22651289403438568, 0.2715685963630676, -0.9226052761077881, -0.43741852045059204, 0.5617513656616211, -0.9458246231079102, 0.20997034013271332, 0.5858612656593323, -0.31117087602615356, 0.22741365432739258, -1.4063996076583862, 1.8110010623931885, 0.26241689920425415, -0.4436083734035492, 1.9176230430603027, 1.3446307182312012, -1.10293447971344, 0.5187076926231384, -0.3313673138618469], [0.9358789920806885, 0.3737378418445587, 1.5946624279022217, 0.9726446866989136, -0.015500284731388092, -0.9575772881507874, 0.683428168296814, -0.343050092458725, 0.9512293338775635, -0.3001963496208191, 0.09486833214759827, -0.42453765869140625, -0.6442131400108337, -0.6604671478271484, 0.5367234945297241, -1.7795883417129517, -0.8750789761543274, -0.27554380893707275, -0.30955564975738525, -0.9566114544868469, -1.6528611183166504, -0.4889223873615265, -0.3624223470687866, 0.7975871562957764, -0.7480847239494324, 1.4267328977584839, -0.9331346750259399, 1.3808590173721313, 0.4221414029598236, -0.7676851153373718, -1.9310979843139648, -1.1720393896102905, 1.05946946144104, 1.759222149848938, -0.6802073121070862, -0.6163110733032227, -0.6871658563613892, 0.022502562031149864, 0.8207417726516724, 0.32608118653297424, 0.25263655185699463, -0.8070027828216553, 0.40917089581489563, 2.3127281665802, 0.6376296877861023, 2.334569215774536, -0.06122050806879997, 0.12740270793437958, -0.1637110412120819, 1.3693859577178955, 0.9660168290138245, 1.0526864528656006, -0.2719053030014038, -0.24393503367900848, 2.2181789875030518, -0.8375663161277771, -0.44325751066207886, -0.0035858878400176764, 0.6009633541107178, 1.0800358057022095, -0.3703345060348511, 0.6345932483673096, 0.5651831030845642, 0.7544111609458923, -0.9745344519615173, 1.04978609085083, 1.0785486698150635, -1.4777730703353882, 0.06309063732624054, -0.4482671618461609, -0.33556950092315674, 0.10078038275241852, -0.3073981702327728, 1.1441618204116821, 1.07175612449646, 0.08024784177541733, 0.21651510894298553, -2.05241060256958, -0.8797623515129089, -1.0997438430786133, -0.6679133772850037, -0.9590430855751038, 0.09764254093170166, -1.073099970817566, -0.6567656397819519, -0.5698866844177246, -0.3130035400390625, -0.6816082000732422, -1.8818438053131104, 0.3548991084098816, 0.35153117775917053, -1.7517274618148804, -0.04783470556139946, 0.41090497374534607, -0.3992856442928314, -0.8624928593635559, -0.051895078271627426, -0.012222855351865292, -0.762146532535553, -1.099543809890747, -0.332821249961853, -0.0422920286655426, -1.6289008855819702, -0.6559566855430603, 2.176989793777466, -0.5905112624168396, -0.8389053344726562, 0.18065032362937927, 0.4563782215118408, 0.6760491132736206, 0.26474595069885254, -0.7389060258865356, -0.2938455045223236, -0.03381551429629326, 1.0783889293670654, -0.9665743112564087, 1.2824281454086304, -0.5246779918670654, -0.5575810074806213, -0.04450685530900955, -1.4712536334991455, -0.3920815587043762, 0.7675970792770386, 0.277284175157547, -0.2791435122489929, 0.4126527011394501, -1.301958441734314, -0.13113565742969513, 1.4265356063842773], [-0.3673570454120636, -0.06105922535061836, 0.9334097504615784, -0.6535941958427429, 1.5957692861557007, 1.3050589561462402, 0.9637355804443359, 0.8006077408790588, 0.03355500102043152, 1.1239622831344604, -0.07408645749092102, -0.9250423312187195, -2.6229805946350098, 0.05643686652183533, 0.6440922617912292, 0.7102201581001282, -1.7353754043579102, -0.16732874512672424, -0.03627874329686165, 0.4090774357318878, -0.8635680079460144, -0.05837810039520264, 0.0040388163179159164, -0.857789158821106, -0.7506674528121948, 0.4736325740814209, -0.8472979068756104, 0.7643327116966248, 1.3785983324050903, 0.974055826663971, -0.4696682095527649, 0.6850259900093079, -1.2337695360183716, -0.03716558590531349, 0.9762129783630371, -0.4744141399860382, -1.710448145866394, 0.6259351968765259, -0.5447261333465576, -0.5895534753799438, -0.2352605015039444, 0.23119167983531952, 0.843295156955719, 0.6290947794914246, 0.032975807785987854, -0.7878467440605164, 2.731264591217041, 0.7792240977287292, -1.020511507987976, 0.6260595321655273, 0.5218948721885681, -0.9839490056037903, 1.1019285917282104, -0.06231888011097908, -0.3044474124908447, 1.3135464191436768, -0.23884670436382294, -0.7450666427612305, 1.9784430265426636, -2.34728741645813, -0.1288759708404541, -0.12906083464622498, -0.0368289090692997, 0.9730508327484131, 0.6182751059532166, 1.1552308797836304, -0.3480185866355896, 1.5650275945663452, -0.5732852816581726, -0.738865852355957, -1.0245351791381836, 0.12156011909246445, 1.26036536693573, -0.35526031255722046, 0.4193691909313202, -1.3175100088119507, 0.07564438879489899, -1.224586009979248, 0.10184966772794724, 1.1311689615249634, 1.4568039178848267, 0.9365880489349365, -0.7776796221733093, 0.643339991569519, -1.6227747201919556, -1.0446853637695312, -0.8650075197219849, 0.8564831614494324, -1.1780308485031128, -1.5788785219192505, -1.2460355758666992, -0.22674991190433502, 0.31291720271110535, -0.4217665195465088, 2.454153537750244, 0.2876443564891815, 0.18728463351726532, 0.2378387153148651, -0.3771277368068695, -0.4605041742324829, 0.49722591042518616, -1.3972647190093994, -0.16702236235141754, 0.001302490709349513, -0.40143245458602905, 0.31795793771743774, -0.09601801633834839, -0.6009359359741211, 0.1464150995016098, -0.8777919411659241, 0.7145957946777344, 0.8028610944747925, -0.3798023462295532, 0.2916708290576935, -0.24693065881729126, -0.35159987211227417, 1.0195255279541016, -0.061762191355228424, 0.6317508816719055, 1.1661616563796997, 1.569474458694458, 0.10136741399765015, 0.9042330384254456, -0.4387264847755432, 0.9453441500663757, 1.1503630876541138, -0.2666287124156952, -0.25800204277038574, -0.8447830080986023], [0.2784757614135742, 1.076843023300171, -0.426664263010025, -0.5343838930130005, 0.6811153888702393, -0.9058067202568054, 1.2002326250076294, -0.3288203775882721, 0.34027299284935, 0.8496570587158203, 1.7431727647781372, 0.061411041766405106, 0.7457313537597656, -0.3494439125061035, 0.06905386596918106, 0.5108591914176941, 0.444021612405777, -0.9452778100967407, -0.889375627040863, 1.6349925994873047, 1.550622820854187, 0.4992445707321167, -1.4287186861038208, 0.12608380615711212, 0.3059345781803131, -0.34549906849861145, 0.818730354309082, -0.3792126178741455, 1.1843596696853638, 0.1479150354862213, 0.4336645007133484, -2.941368341445923, -0.4979722499847412, 0.5257323384284973, 0.19525869190692902, 0.36840128898620605, -0.3783175051212311, 0.01796690747141838, 0.7395474910736084, -0.41399675607681274, -0.8301278352737427, -0.11218252778053284, -0.5326817035675049, -0.15822336077690125, 0.18364298343658447, -1.6310776472091675, -1.3776180744171143, -0.14628921449184418, -1.1664395332336426, -0.5872621536254883, -1.138623833656311, 0.025589607656002045, -0.5351023077964783, -0.7042722105979919, -0.22591832280158997, -0.9281198382377625, 0.44354838132858276, 0.37571007013320923, -0.4400920867919922, -0.7608514428138733, 0.28169894218444824, 0.7487582564353943, 0.555949866771698, 0.12168248742818832, -1.5604631900787354, 0.09647428244352341, -0.2840747535228729, 0.24708950519561768, 1.7329518795013428, 0.16339975595474243, 1.0072818994522095, 0.44041377305984497, 0.5807203054428101, 0.4662933349609375, 0.32502779364585876, 0.11023291945457458, -2.1736702919006348, -0.534325122833252, 1.9328502416610718, -0.5079871416091919, 1.1651396751403809, 1.3982714414596558, -1.379713535308838, 0.37533220648765564, 0.1355987936258316, -1.5508651733398438, -0.419888436794281, 0.03397185355424881, 0.8710536956787109, 0.7015323042869568, -0.9313682913780212, -1.6339324712753296, 0.05424785241484642, -0.19908128678798676, 0.23856541514396667, 0.9404440522193909, 0.4728300869464874, 0.5479950308799744, 2.014937162399292, 0.9818630218505859, -1.0368645191192627, 1.2558530569076538, -0.07586443424224854, -0.8032869100570679, -0.8638279438018799, -0.4640367925167084, -0.08744523674249649, 0.8245376348495483, -0.3783556818962097, 0.13613715767860413, -2.032700777053833, 0.5171802639961243, -0.011145533993840218, -2.20806884765625, 0.21347548067569733, 0.294384628534317, 1.4127037525177002, 0.9536441564559937, -1.3445279598236084, 0.3566579222679138, 2.8075389862060547, -0.34116071462631226, 0.788724422454834, 0.37434136867523193, -0.6507902145385742, 1.4636061191558838, 0.7536424398422241, 0.1970650851726532, 0.41846024990081787], [-0.5310351848602295, -1.7147247791290283, -0.3827342987060547, -1.5837494134902954, -0.1308964490890503, 1.2912400960922241, -0.18749186396598816, 0.8081966638565063, -0.1674945205450058, -1.1700273752212524, 0.44151103496551514, 0.5212265849113464, -0.7488957643508911, 1.3133127689361572, -0.5539076328277588, 0.5455559492111206, 0.6284748315811157, -1.5596569776535034, 0.8444172739982605, 1.1937510967254639, 0.20364515483379364, -0.17087076604366302, -0.22459208965301514, -1.1494499444961548, 0.388301819562912, -0.5899199843406677, 0.8855583071708679, 1.259379506111145, 2.240992784500122, 1.051915168762207, -1.1335082054138184, 1.9448261260986328, -0.43021437525749207, 1.6443060636520386, -0.9088209867477417, 2.23384952545166, 0.37109610438346863, -0.38387253880500793, 0.7940029501914978, -0.5534598231315613, 0.7645421028137207, -0.40091338753700256, -1.051423192024231, -2.583090305328369, -2.5004053115844727, -0.17016823589801788, 1.209128499031067, 1.140441656112671, -0.39620015025138855, -0.08003181964159012, -0.2162317931652069, -0.7691165208816528, 0.14946213364601135, -0.49549439549446106, -1.9654651880264282, -0.473346471786499, 0.29373055696487427, -0.3539593517780304, 0.8279305696487427, 0.4874742031097412, 0.829879105091095, -0.8789274096488953, -1.3251032829284668, -0.32260337471961975, 0.19843940436840057, -1.6037925481796265, -0.6357434988021851, 0.06957003474235535, 1.1251705884933472, -1.143688678741455, -0.35303637385368347, -0.4107896685600281, -0.5340123176574707, -0.5492678880691528, -1.0011457204818726, 1.3136646747589111, -0.04046041518449783, -0.874673068523407, -0.2681485116481781, 0.986107587814331, 0.5349791049957275, 0.022103991359472275, 0.9401652812957764, -0.2968473434448242, -0.37816956639289856, 0.017485275864601135, 0.12683410942554474, -0.556557834148407, -0.38334211707115173, 0.4327565133571625, -1.722604513168335, -0.030693326145410538, 1.280701756477356, 1.0051637887954712, 0.6092104911804199, -1.18574059009552, 0.8883172273635864, -0.4964499771595001, -0.427821546792984, 0.23910358548164368, -0.46479532122612, -1.1646322011947632, 0.4983716607093811, 1.9978049993515015, -2.0755391120910645, 0.0062639713287353516, -0.46017327904701233, -0.46153226494789124, -0.2828977704048157, 0.054576847702264786, -0.8844792246818542, 1.6900473833084106, 1.7262911796569824, 1.1501871347427368, 0.06255070120096207, -1.714536190032959, -0.7744203209877014, 0.06075739488005638, 0.23527798056602478, 1.3365209102630615, -2.3609848022460938, -0.5862774848937988, -0.10290350019931793, -0.41565248370170593, 0.12815414369106293, 0.16027367115020752, 0.3510567545890808, 1.898669958114624, -2.435464859008789], [-2.3316617012023926, -0.40679633617401123, 0.44423845410346985, -0.21716783940792084, 1.2723791599273682, 1.3948686122894287, 0.41829216480255127, 0.8871724605560303, -0.021636154502630234, 0.6710293292999268, -0.1747172325849533, 1.3231807947158813, 0.4987194240093231, 0.6341861486434937, -1.006578803062439, 0.5978440046310425, -1.0490742921829224, -1.8978898525238037, -0.16227439045906067, -1.8806508779525757, -0.6051763296127319, -0.5708531141281128, 0.5895607471466064, 0.9189373850822449, -1.544226050376892, 0.7191348671913147, 0.16838714480400085, -0.08165794610977173, -1.815058708190918, 0.24687440693378448, -0.666222333908081, -0.6969867944717407, 0.07543163746595383, 0.6043865084648132, -0.08322788774967194, -0.8848561644554138, 0.07006070762872696, -0.17899030447006226, 0.7810991406440735, -2.6050937175750732, 1.2669514417648315, -0.45705902576446533, -1.1749141216278076, -0.538745641708374, 0.48458099365234375, -1.4908887147903442, -0.07119140774011612, -1.4080883264541626, 0.3751063048839569, -1.486998200416565, 0.20833784341812134, -0.8145949840545654, 0.629386842250824, 0.6230536103248596, -0.5280959606170654, 0.6418400406837463, 1.3128455877304077, -0.5197004675865173, -0.1204715445637703, 2.143954038619995, -1.065895915031433, 0.03920218348503113, 2.8160905838012695, -0.5615088939666748, -1.8627021312713623, 0.11935970187187195, 0.39352142810821533, 1.3194708824157715, -0.07901770621538162, 1.1044059991836548, 0.15162400901317596, -0.12661726772785187, -0.7210574150085449, 0.319324791431427, -0.4402681887149811, -1.1894652843475342, 0.721624493598938, 1.6786863803863525, 0.8078331351280212, 1.5627373456954956, 0.24228297173976898, -0.6706655621528625, -1.035976529121399, -0.4626273214817047, 0.28068870306015015, 2.0923666954040527, -1.3393504619598389, -0.015149242244660854, -0.14622971415519714, 0.7286813855171204, -0.6326896548271179, -0.26180389523506165, -0.7647396922111511, 0.6474859118461609, 1.3228774070739746, -1.2876008749008179, -0.5879354476928711, 1.5164787769317627, -0.0338725708425045, -1.0383241176605225, -1.4760411977767944, -0.6315311193466187, -0.9503500461578369, -2.4969735145568848, 0.9130820631980896, -0.054121579974889755, -0.014265866950154305, 0.7526165246963501, 1.9056153297424316, 1.6418462991714478, -0.3577178716659546, 0.85308438539505, 0.48843106627464294, -0.2171340137720108, 0.6637887954711914, 0.6647246479988098, 0.2724437117576599, -0.20815089344978333, -0.03861483186483383, -2.3382680416107178, -0.8190628886222839, -0.7728524804115295, -0.39777377247810364, 0.40274107456207275, -1.7549960613250732, -1.4239223003387451, 0.44919273257255554, -0.1533958911895752, -0.8251615166664124], [-1.3408567905426025, -0.5676809549331665, -0.967820405960083, -0.37881186604499817, 1.2729361057281494, 0.23020653426647186, 1.2721208333969116, 0.9616186618804932, -1.6928398609161377, 1.4287446737289429, 0.14034974575042725, -0.932864248752594, 0.4217146337032318, -0.4817751348018646, -0.5697762966156006, -2.3964335918426514, 1.3399544954299927, -1.0486112833023071, -0.3245547115802765, -0.41059431433677673, 0.29506146907806396, 0.44217824935913086, 1.27025306224823, 1.4541596174240112, -0.4919111728668213, 0.6183607578277588, -1.9545375108718872, 0.08446856588125229, 0.10802450776100159, 1.6620138883590698, 1.1030923128128052, -1.160595178604126, 0.8038954138755798, -0.0901453047990799, -2.355710744857788, -1.6234114170074463, 1.8621952533721924, 2.0224854946136475, 0.29970866441726685, -0.49582991003990173, -0.628473162651062, -0.7867944836616516, 0.32439252734184265, 0.10721684992313385, -2.086337089538574, -0.9408559799194336, 1.406412124633789, -0.12890850007534027, -1.5456008911132812, -0.2919105887413025, -0.4368094801902771, -1.1253843307495117, -1.149770975112915, -1.22788405418396, 0.17986324429512024, -0.12620483338832855, -0.9664347171783447, 1.2210649251937866, -0.976104736328125, 0.2652217447757721, 1.847851276397705, -0.16802187263965607, -0.7770958542823792, 0.14674408733844757, -0.012364733032882214, -0.11961022019386292, 2.095607042312622, 1.675571322441101, 1.4730619192123413, 1.204107642173767, -1.5905934572219849, 1.0925190448760986, -0.019501445814967155, -2.040949583053589, 0.38866880536079407, -0.5942303538322449, 2.5654244422912598, 0.03763463348150253, -0.17871013283729553, 1.4755529165267944, 1.5804835557937622, 0.573980450630188, 0.81693035364151, -1.4758700132369995, 0.1230676993727684, -0.37987467646598816, -0.7403916716575623, -0.28360503911972046, 0.7798058390617371, -0.18060339987277985, 0.3743390738964081, -0.7131539583206177, -0.2076258510351181, 0.03822961449623108, -0.85676509141922, 0.5942379236221313, -1.7104592323303223, 1.0352140665054321, -1.9620611667633057, -1.9169172048568726, 0.271904855966568, -0.2729538679122925, -0.16487832367420197, -0.7947269678115845, -0.07552821189165115, -0.3577285408973694, 1.1568660736083984, 1.4322395324707031, 1.462499737739563, -0.6534021496772766, -1.2229866981506348, 0.036327846348285675, -1.7900315523147583, 1.2621718645095825, -0.5351102948188782, -0.3912825286388397, -0.45180901885032654, -0.15218867361545563, -0.26663342118263245, 0.817290186882019, 0.5560100078582764, 1.056151270866394, 0.48758557438850403, -0.805868923664093, -0.5655125975608826, 0.750831127166748, -0.078654445707798, 1.3155031204223633, 2.030858278274536], [0.9185131788253784, 0.7308258414268494, 0.8137634992599487, -0.6233667135238647, -0.5982025861740112, 1.6669280529022217, -0.22089099884033203, 0.5698674917221069, -2.158679962158203, -2.4477527141571045, -1.3710126876831055, -0.8972858786582947, -0.5183588862419128, -0.989289402961731, -0.7199550867080688, 1.0533959865570068, 0.6187482476234436, 0.6657587289810181, -0.733863890171051, -0.32547032833099365, 0.9748021364212036, 1.0977466106414795, -0.4645532965660095, -1.173376202583313, -1.7701597213745117, -0.20206397771835327, -0.08843210339546204, 1.2616021633148193, -0.2855898439884186, -1.5994081497192383, 2.0766561031341553, -0.4825071692466736, 0.33723101019859314, 0.29699909687042236, 0.5154399275779724, 1.4643300771713257, 1.7831041812896729, -0.1872982382774353, 0.2607872784137726, -0.22053125500679016, 0.3344627916812897, -0.5226683020591736, -0.5038701295852661, 0.8974172472953796, 0.7554941177368164, -0.8740847706794739, -0.0006439128774218261, 0.7660609483718872, 0.7861830592155457, 0.8659968972206116, 0.6917674541473389, -0.1748867630958557, 1.1807492971420288, -0.6355554461479187, -0.8252639770507812, -0.7648239135742188, -2.203248977661133, -0.1438479721546173, -0.4431260824203491, 0.45548349618911743, 0.7553401589393616, -1.9051042795181274, -0.13068580627441406, -0.470851331949234, -0.14338622987270355, -0.16414283215999603, 2.3698008060455322, 3.253140687942505, 1.7357748746871948, 0.29069146513938904, -0.8247842192649841, -0.21032218635082245, 0.44827982783317566, -0.5578136444091797, 1.4376929998397827, -0.5392023324966431, -0.9720028042793274, 0.38201841711997986, 0.43025416135787964, 0.5017356276512146, -0.36147695779800415, -0.5907795429229736, 1.620288610458374, 0.7351575493812561, -0.2562318444252014, 0.39979225397109985, 0.6909431219100952, -1.548153042793274, -0.741541862487793, -0.19699563086032867, 0.9058265089988708, 1.5070523023605347, -1.4770041704177856, -0.7567064166069031, -0.2703174948692322, 0.04185701161623001, -0.22865845263004303, -1.8958419561386108, 2.236330270767212, -0.26008862257003784, -0.0038218246772885323, -0.222024604678154, -0.1318066269159317, -0.7975965738296509, 1.45340096950531, -0.05550483614206314, 0.3700747489929199, -0.8458536863327026, 0.00589569890871644, 0.18917767703533173, 0.0479043647646904, -1.8698971271514893, 0.7983594536781311, -0.9110457897186279, 0.19166308641433716, 0.12121449410915375, -0.6832332015037537, 0.27042943239212036, 0.9893364310264587, -2.972478151321411, 1.0462205410003662, -0.510313868522644, 0.011145848780870438, 2.48083233833313, -0.38775748014450073, -0.08280807733535767, 1.6624170541763306, 1.1280570030212402, -0.4224019944667816], [-0.17115715146064758, 0.9013333320617676, 1.1818652153015137, -0.757232666015625, -0.47361406683921814, 0.061752017587423325, -0.4224846363067627, -0.45529571175575256, 0.13126197457313538, -1.3933656215667725, -1.2516080141067505, 1.0828711986541748, 0.22828711569309235, -0.5217080116271973, -0.24143783748149872, -1.1957319974899292, 0.23642562329769135, 0.09454015642404556, 0.11930924654006958, -0.10585316270589828, -1.0423247814178467, -0.6909322142601013, 1.388921856880188, 1.2067797183990479, 0.6942516565322876, 0.46341201663017273, -0.9334819316864014, 0.9403379559516907, -1.3980103731155396, -1.7020771503448486, 0.9872259497642517, -0.04915813356637955, 1.1096997261047363, -1.4439277648925781, 0.5585700869560242, -0.0285678468644619, -0.2523382902145386, 1.0661927461624146, 0.8127506971359253, 0.605003833770752, 0.6406790018081665, -0.039347462356090546, 0.1679619699716568, 0.09646289795637131, 1.2208865880966187, 0.5425218939781189, -0.2813267111778259, 0.9639366269111633, -0.2683592736721039, 1.1255974769592285, -1.5021412372589111, -1.620876669883728, 0.45449280738830566, -1.4364469051361084, 2.125387191772461, 0.23810668289661407, -0.9747722744941711, -1.4678740501403809, 0.7940965294837952, -1.6320120096206665, -0.037536974996328354, -1.1694964170455933, -0.49640563130378723, -1.1505446434020996, -0.2423630803823471, -0.004459282383322716, 0.03731027990579605, 0.6851952075958252, -2.2245967388153076, -2.130913257598877, 0.02876829355955124, -0.385884165763855, -1.6439051628112793, 0.2113896906375885, -0.6504110097885132, -0.760652482509613, 0.41457805037498474, 0.40669262409210205, 1.7123435735702515, -1.0330373048782349, -0.9365351796150208, -0.4402155578136444, 0.07444719225168228, -0.02454422228038311, 1.7118849754333496, -1.249092936515808, 0.15690162777900696, -1.4397047758102417, 1.2042145729064941, -1.201098918914795, -1.179810643196106, -1.3160960674285889, -1.2270545959472656, 1.1782528162002563, 0.24466373026371002, -1.415870189666748, -0.11102266609668732, 0.10527016967535019, -0.3270743191242218, -0.5925262570381165, -1.02879798412323, -0.32121092081069946, 0.19837036728858948, -0.1654299795627594, 0.004831244703382254, 0.9263951182365417, 0.11308379471302032, -0.6815778017044067, 0.3100138008594513, 1.3200087547302246, 0.33020296692848206, 0.22737590968608856, 0.3558564782142639, -0.35743001103401184, -1.1369965076446533, -0.057281166315078735, -0.07183583080768585, 0.26873552799224854, 0.0016137172933667898, 0.49895286560058594, 1.6130156517028809, 1.3327986001968384, 1.4774978160858154, -1.3484878540039062, -0.1699773371219635, -0.8158217072486877, 1.1741623878479004, 0.8819083571434021, 0.3752398192882538], [0.6862704157829285, -0.205719456076622, -0.5343012809753418, 0.10094252228736877, -1.7762805223464966, -1.94379460811615, -0.8204997777938843, -2.3089497089385986, 0.8888121843338013, -0.8564250469207764, -0.55330890417099, 0.5372595191001892, 1.0502341985702515, 1.786339521408081, 0.03519851341843605, 0.6877965331077576, -0.5352347493171692, 0.9106990098953247, -0.4993654787540436, -0.6330627202987671, 1.4368042945861816, 1.7906172275543213, 1.0257011651992798, 0.8466719388961792, -1.7083847522735596, 0.7361943125724792, 0.24927593767642975, -0.4382396936416626, 1.731829285621643, 1.1158324480056763, 0.2431505173444748, 0.03807765245437622, 0.19700117409229279, -0.8097749352455139, -0.9945024251937866, -0.3049456775188446, -1.5321404933929443, 0.07610543817281723, 1.7020684480667114, 0.03745120391249657, -0.23121504485607147, -0.3310382068157196, 0.7235006093978882, 1.0571048259735107, -0.0808187797665596, -0.5262816548347473, -0.31307899951934814, -0.3722527325153351, 0.11172250658273697, 0.08583856374025345, 0.05992419645190239, -0.04016262665390968, -0.09767431020736694, -0.2933885157108307, 0.04378041625022888, 0.5200635194778442, -1.114276647567749, 0.17579355835914612, -0.8944769501686096, 0.44978567957878113, 1.4208301305770874, 1.0313093662261963, -0.12242751568555832, 2.005394220352173, 0.9572155475616455, 0.9148246049880981, 1.0800256729125977, -0.8832321166992188, 0.9384444952011108, 0.13962942361831665, 1.5724883079528809, -0.03401336446404457, -0.14893633127212524, 1.4206271171569824, -0.4765901565551758, -1.1594184637069702, -1.0887905359268188, -0.41777050495147705, -0.7157090902328491, -0.8128530979156494, 0.647042989730835, -1.4395296573638916, 0.5213116407394409, 1.4842742681503296, 0.5958665609359741, -0.14593294262886047, -0.0801088809967041, 0.6296668648719788, 0.05137696489691734, 0.6514175534248352, -1.2068085670471191, 0.3026054799556732, -0.03086717240512371, 1.9828875064849854, 0.29005932807922363, 0.6118593215942383, 0.04196956381201744, 1.215561866760254, -0.6515595316886902, 0.1724259853363037, -0.30445027351379395, -0.3848576843738556, 0.4345066249370575, 1.2939183712005615, -2.101564407348633, 0.07208403944969177, 0.02138889767229557, 0.8963176608085632, 1.237563133239746, -1.547573208808899, -1.827579379081726, -2.832836866378784, 0.9004194140434265, -0.12141872197389603, 0.9218716621398926, 1.1386553049087524, 0.1380186825990677, -1.0638659000396729, 2.545606851577759, -0.43728873133659363, 0.11282200366258621, 1.1319384574890137, -0.7245282530784607, 0.16220733523368835, 0.5064557790756226, 0.510124146938324, 0.5813661217689514, 0.19105778634548187, 0.8263953924179077], [0.32460445165634155, 0.6258752942085266, -1.5224543809890747, -1.4197282791137695, 1.6756006479263306, 0.7066486477851868, -0.8484070301055908, 0.5103758573532104, 0.4648059010505676, -0.08491196483373642, -1.8099817037582397, 0.19468587636947632, 2.2883853912353516, -1.5252580642700195, 2.3038110733032227, 1.1184849739074707, -1.511919379234314, 1.3765708208084106, -0.10895606875419617, 1.7109270095825195, 0.804415225982666, -0.4725777208805084, 0.03881281986832619, -0.5532193183898926, 0.3738349378108978, -0.8551592230796814, 0.2950756251811981, 0.9686678647994995, -0.15604975819587708, 0.3659609854221344, 0.1978469341993332, 1.1778197288513184, 0.3862461447715759, -1.1953363418579102, -0.9928746223449707, -1.198113203048706, -0.5656801462173462, -2.257261037826538, 0.29567408561706543, 0.5192278623580933, 0.4633983075618744, -3.0843141078948975, -1.1699568033218384, 0.7613397836685181, -1.2987010478973389, -0.7857323288917542, 1.4303151369094849, 2.527773141860962, 0.6144739389419556, 0.28492531180381775, -0.48933660984039307, -0.7460766434669495, -0.5563889741897583, -1.916242241859436, 0.4315803050994873, -0.36554306745529175, 0.24981807172298431, 0.5538318157196045, 0.50869220495224, 1.225811243057251, -0.01385979913175106, 0.6389954090118408, -0.6175876259803772, -0.21605439484119415, -0.9350081086158752, 0.7997204661369324, -1.6523194313049316, 0.19823236763477325, 1.3037587404251099, 0.8853052854537964, 0.3520033061504364, -0.40227293968200684, 0.10746710002422333, 1.359350562095642, -0.030112281441688538, -0.3646474778652191, -0.2568841278553009, 0.6862941980361938, -1.5189251899719238, 0.71441650390625, 0.902573823928833, 0.7823206186294556, -0.0617949403822422, -0.5320916175842285, -1.1648998260498047, -0.12597395479679108, -1.8408987522125244, 0.5413411855697632, 1.8836884498596191, 0.8337134122848511, -1.1261786222457886, -0.41677653789520264, 0.19072367250919342, 1.2349494695663452, -0.18705137073993683, 0.06118674948811531, -0.24275094270706177, 0.7072697877883911, -0.14019781351089478, 0.3659469187259674, -0.612578272819519, -1.0995222330093384, 1.7434940338134766, -0.5915789604187012, 1.461310625076294, -0.20957006514072418, 0.17698487639427185, -1.2310494184494019, 2.4593303203582764, -0.5610122084617615, 0.294676810503006, 0.049878451973199844, 1.3498800992965698, -0.8861107230186462, -0.6916504502296448, 1.2645819187164307, -1.109598994255066, -0.6139770746231079, -0.32241421937942505, -1.1493675708770752, 1.0646421909332275, 0.5901405811309814, 0.12374213337898254, 0.4348331689834595, -0.1775125414133072, 0.10723859071731567, -0.2976820766925812, 0.32643061876296997, 0.09218328446149826], [-0.4116074740886688, -1.4762483835220337, -0.19285954535007477, -0.5372588038444519, -0.9130226373672485, 0.356417715549469, -0.45992085337638855, -0.6202152371406555, -0.5955291986465454, 0.058073144406080246, 0.8107423186302185, 1.0803792476654053, 1.1889472007751465, 0.8812832236289978, 0.6637763977050781, -0.7886631488800049, 0.8441839814186096, -0.01981113851070404, 0.496010959148407, -0.9193335175514221, 0.8608976602554321, 1.0387860536575317, 0.025568218901753426, 0.46588945388793945, -0.3146864175796509, 1.3991656303405762, -2.041152238845825, -0.1513337641954422, 0.6195155382156372, -0.7185752987861633, 0.0400317944586277, -1.2420415878295898, 0.7518571615219116, -1.0834170579910278, 0.5627578496932983, 1.241881251335144, -0.6475827097892761, -0.34325963258743286, -0.04949017986655235, -0.500938355922699, -0.5176675915718079, -1.4539188146591187, -0.4076317846775055, 1.387945294380188, -1.710370421409607, -0.5712617039680481, 1.4038599729537964, -0.2541685402393341, -0.274460107088089, 0.28359535336494446, -0.15219800174236298, -0.10334384441375732, 0.0920490100979805, 0.3344540297985077, -0.001699331565760076, 0.2511598467826843, 1.5779658555984497, 0.32137399911880493, -0.10436074435710907, -0.25057488679885864, 0.6115304827690125, 0.8934049606323242, -0.6849634647369385, -0.6934985518455505, -0.17270539700984955, 0.6811853647232056, -0.5772603750228882, -0.7550885677337646, 0.5511139035224915, 0.7115433216094971, 0.8688274025917053, 0.24828369915485382, 0.07872617244720459, 0.27487900853157043, 1.7803996801376343, 0.6274181008338928, -0.3821115791797638, 0.4177806079387665, 0.01483287662267685, -0.1839672476053238, -0.7031278014183044, -2.5151140689849854, 0.12460620701313019, 0.6201935410499573, 1.3788772821426392, 0.7196289896965027, 1.793639898300171, -0.4115443527698517, 0.9832792282104492, 2.3735921382904053, -0.9753080606460571, 0.31281083822250366, -0.17127183079719543, -0.31463029980659485, 1.0427212715148926, 0.03837895020842552, -1.0849120616912842, 0.13528969883918762, 0.2632635235786438, -0.0019410124514251947, 0.8178558349609375, -0.6398659348487854, 0.4326353073120117, 0.014193695038557053, -1.0130521059036255, 0.65143221616745, 1.2673425674438477, -1.1236019134521484, 0.45264729857444763, 2.799384832382202, 0.1835002601146698, 0.6255959868431091, 1.5129590034484863, -0.944340169429779, 0.015059860423207283, -0.46348580718040466, 0.6132091283798218, -0.5885314345359802, -1.392900824546814, 0.042539551854133606, 0.5559902787208557, 1.7084858417510986, 0.6093761920928955, 1.390251636505127, -0.029317764565348625, -0.41641223430633545, -0.825725793838501, 0.54600989818573, -0.5025623440742493], [0.034793056547641754, 0.14703640341758728, -0.0033170455135405064, -0.5434638261795044, -0.8240730166435242, -1.6898406744003296, 0.545055091381073, -0.7064424753189087, -0.20111031830310822, -0.20638975501060486, 0.48752549290657043, -0.6905246376991272, 1.2272464036941528, -0.16853907704353333, -0.0858900249004364, 1.088323712348938, 0.8511025905609131, -0.08225410431623459, 1.472127079963684, 2.328244209289551, -1.1799224615097046, -0.1325189620256424, -2.9658150672912598, -0.06628891080617905, -0.23377390205860138, -1.196014165878296, 0.10152241587638855, 0.14970853924751282, -1.3087362051010132, -1.2620172500610352, -1.3395555019378662, -1.1931188106536865, -0.44632336497306824, -1.726204514503479, -0.5722909569740295, -0.35164523124694824, -0.5967479944229126, 0.8446811437606812, 1.2386351823806763, -0.8430510759353638, -1.994511604309082, -0.24362795054912567, 1.8512979745864868, -1.2886253595352173, -2.0980565547943115, -0.6295748949050903, 0.1943957358598709, -0.09530551731586456, 1.3749178647994995, 1.9913532733917236, -0.03014945611357689, 1.3026323318481445, 2.914228916168213, 0.5645941495895386, -1.220063328742981, 0.21207666397094727, 0.5000544190406799, 1.0968317985534668, 0.5029805302619934, 0.15307526290416718, -0.9592748880386353, 0.9838059544563293, 0.6837978959083557, 0.1736682653427124, -0.8482121229171753, -0.4440920054912567, 1.0415067672729492, -1.3792203664779663, 0.2389432191848755, 1.5488015413284302, -0.42364200949668884, -0.2499806433916092, -0.191963329911232, 0.16995933651924133, -1.2132283449172974, 0.5958158373832703, 0.41417455673217773, -0.22785615921020508, -0.0351986400783062, 0.4728321135044098, 0.05336407944560051, 0.07493430376052856, -1.6357229948043823, -1.250075340270996, -0.33512625098228455, -0.11335619539022446, 0.9252049922943115, -0.5297281742095947, 0.14936202764511108, -0.7257032990455627, -1.3526172637939453, -0.4652208983898163, 1.1917725801467896, 0.034739408642053604, -2.321509838104248, -0.5325490832328796, 1.3497458696365356, 1.0788003206253052, -1.442223310470581, 0.514762282371521, -0.8366185426712036, -0.014877449721097946, 0.2501175105571747, 0.5730389952659607, 0.189304918050766, -0.1994502991437912, 1.4614111185073853, -0.290098637342453, -0.8071225881576538, 2.1563098430633545, 0.1227901354432106, -1.217661738395691, -0.9947023391723633, 0.7271189093589783, 1.4524972438812256, 0.6448231339454651, 0.7585657835006714, 0.7231754660606384, 0.39463427662849426, 0.2869061231613159, -0.3737276792526245, 0.04743259772658348, -2.0248069763183594, -0.9626120924949646, 0.5284368395805359, 1.8869370222091675, 0.7280678153038025, -1.6905696392059326, 0.27665287256240845], [1.7271158695220947, 1.241320252418518, -0.2673663794994354, -1.2883797883987427, -0.3011987805366516, 0.3418444097042084, -2.03562331199646, -1.3381520509719849, 0.5012047290802002, 0.44087764620780945, -1.1815744638442993, 1.2949302196502686, -0.5335143208503723, -0.3781268000602722, 0.6996609568595886, -1.3842530250549316, -2.0962252616882324, -0.36861613392829895, 0.06473667174577713, -0.37197455763816833, 2.43192982673645, -0.44119447469711304, 0.7915291786193848, 1.388551115989685, -0.8964125514030457, 0.2760259211063385, 1.5489262342453003, -0.8986977338790894, 1.2365256547927856, 0.3869291841983795, -0.5692475438117981, 0.9249457716941833, 0.5383318066596985, 1.4266941547393799, 0.8731837868690491, 1.1452233791351318, -0.450888991355896, -1.8199889659881592, -0.5448352694511414, 0.4905937910079956, -1.5506999492645264, -0.16540613770484924, 0.7016749978065491, 0.3451167047023773, 2.0008351802825928, -0.046621743589639664, 0.4098796546459198, -0.9060757756233215, 0.4037269949913025, 1.6289002895355225, 1.0878082513809204, 1.0054291486740112, -0.6845254302024841, -0.7844957113265991, 1.7713818550109863, 1.3281278610229492, -0.20284907519817352, -0.4644981622695923, 0.10978268086910248, -0.41588112711906433, 0.9858832359313965, 0.0009033880778588355, -0.8091496229171753, 0.9415262937545776, -0.24800367653369904, -2.1596693992614746, -0.3661942481994629, 0.7616488933563232, -0.33273768424987793, -1.2067278623580933, 0.45091626048088074, -1.02994704246521, -1.0943931341171265, -2.011014938354492, 1.2999988794326782, -0.9496695399284363, -1.1188019514083862, 0.3311369717121124, 0.38359901309013367, 1.1879628896713257, 0.17213870584964752, -0.7279176712036133, 0.16565920412540436, -0.6980922818183899, -0.873594343662262, 0.07684153318405151, -1.9262171983718872, -0.7443106770515442, 0.19283327460289001, -0.3327103853225708, -1.40071439743042, 0.9504957795143127, -0.5333536863327026, 0.4031135141849518, -0.2657529413700104, 0.9661664366722107, -0.05902563035488129, -1.2255650758743286, -0.9066658020019531, -0.5444071888923645, 1.078180193901062, -0.6784417629241943, 0.5025930404663086, -0.5545297265052795, 0.23869411647319794, 0.2697591185569763, 1.1519612073898315, 1.372031569480896, -0.46952149271965027, -1.7453219890594482, 0.9103295803070068, 0.18758733570575714, -0.9425458312034607, -1.2769949436187744, -1.1065791845321655, -0.8103949427604675, -1.934156060218811, -0.11499350517988205, -1.34408700466156, -1.4456977844238281, -2.514038562774658, 1.4863587617874146, 0.6665509343147278, 0.9186709523200989, 1.7367976903915405, -0.3030707538127899, -0.13330630958080292, 0.9652470946311951, -0.9237169027328491], [-2.211772918701172, -0.3821607232093811, 0.10508980602025986, -0.10506369918584824, -0.5066015124320984, 0.6060272455215454, 0.5480590462684631, 0.41272759437561035, -0.7094695568084717, 0.1472059041261673, -0.7594558596611023, 0.026489708572626114, -0.006810181308537722, -0.2912769019603729, 1.0387250185012817, 0.09885375946760178, 1.0924053192138672, 0.5958340167999268, -0.09440163522958755, 0.7567838430404663, 1.205011010169983, -0.3856428265571594, -1.118076205253601, -0.09079183638095856, 1.112831711769104, -0.4797864854335785, 1.1077280044555664, 0.16252261400222778, 1.2963131666183472, 1.2396721839904785, 0.8872238993644714, 0.4849703907966614, 1.2396196126937866, 0.21175284683704376, 0.9814900755882263, 0.03196315839886665, -0.7640188932418823, -0.09329677373170853, 0.11036210507154465, 0.23336239159107208, -0.7483193874359131, 0.08939192444086075, -0.28785601258277893, -1.1619608402252197, 0.9305452108383179, -0.17523068189620972, 0.024398349225521088, -0.5954989790916443, 0.4109416604042053, -0.05506008118391037, -0.6760799884796143, 0.8513917326927185, 0.8696986436843872, -0.9933722019195557, -1.2961422204971313, -0.27537670731544495, -0.5381909012794495, -0.009539840742945671, 1.0997865200042725, -0.14328178763389587, 0.7499169111251831, 0.9032723307609558, -1.0728402137756348, -0.3425799608230591, 0.9112758040428162, 0.09171906858682632, -0.23583808541297913, 1.348077654838562, 0.6421471238136292, -1.0995380878448486, -0.5425828099250793, 1.7164374589920044, -1.8754428625106812, 1.2660489082336426, 0.5506406426429749, 0.1845627725124359, 0.8436930179595947, 0.45676031708717346, -0.9086102247238159, 0.7179350256919861, -0.44660577178001404, -0.36939966678619385, -0.10403405874967575, -1.0932360887527466, -0.8665662407875061, 0.22427058219909668, 0.006870895624160767, 0.9871745705604553, -0.0018941363086923957, -0.11989710479974747, 1.1749532222747803, -0.579858124256134, -0.21884816884994507, -0.7280007600784302, -0.7619346380233765, -2.6580727100372314, -1.5786712169647217, -0.7915480732917786, 1.1540093421936035, 0.9993323087692261, 1.6075032949447632, 3.036695957183838, 0.03045336902141571, 0.5738099217414856, -1.6359280347824097, 0.4429170787334442, -1.1186838150024414, -0.3405643701553345, 1.8072911500930786, 0.5387774705886841, -1.721482276916504, 0.5984049439430237, -1.2030668258666992, 0.4033484160900116, -1.245934247970581, 0.14547401666641235, 0.8008716702461243, 2.194183826446533, -0.251958429813385, 0.8257675170898438, 0.2285490781068802, 0.7791976928710938, -1.063559889793396, 1.0358166694641113, -1.2229737043380737, 0.7658383846282959, 1.3752244710922241, -0.5175463557243347, -0.2057090550661087], [-0.39201322197914124, 0.31077295541763306, 0.192073255777359, 0.7730464935302734, 1.8345096111297607, -0.24392205476760864, 0.30537155270576477, -0.8952414989471436, 0.7757689356803894, 1.621753454208374, -0.443930447101593, -1.4509894847869873, 0.4929367005825043, 0.7600094676017761, -1.6996499300003052, -0.9791674613952637, -0.19186019897460938, -0.14787577092647552, -0.08684073388576508, -1.135985255241394, -1.3168718814849854, -0.29599565267562866, -0.13883723318576813, 0.0217538271099329, 0.024742789566516876, 0.3065339922904968, -0.08634539693593979, 0.6140995621681213, -0.9042664170265198, 1.019921064376831, -1.1514110565185547, -0.3363671600818634, 2.049959421157837, -0.41684240102767944, -0.8466373682022095, -0.35329991579055786, 1.7736682891845703, -0.4549815058708191, -0.9157495498657227, -0.021866323426365852, -0.1244250163435936, 1.1052422523498535, -0.02537882886826992, 0.2384280264377594, 0.1980799436569214, -0.32483363151550293, -0.2596398591995239, -1.1711173057556152, 0.9045349955558777, -0.9510800242424011, 0.6963187456130981, 0.1407441645860672, -0.5634551644325256, -0.04676508903503418, 0.07663888484239578, 1.9665759801864624, 0.5224744081497192, -0.5895378589630127, -0.7691984176635742, -0.725333034992218, -1.4629487991333008, 0.18618817627429962, -0.382494181394577, 0.05171574279665947, 0.4542028605937958, -0.2111845761537552, -0.01901361718773842, -0.440572053194046, 0.3371060788631439, 0.25668373703956604, 1.311816930770874, 0.9018768668174744, -0.5935126543045044, -0.32758429646492004, -1.3797132968902588, 0.04386096075177193, -0.4412837624549866, -0.20719827711582184, -1.1052603721618652, 0.5179411768913269, -1.8335802555084229, -2.326918125152588, -0.9549765586853027, 0.9189546704292297, -2.0939934253692627, 1.6704529523849487, -0.8805957436561584, -0.05991725996136665, 0.6696553230285645, -1.2742663621902466, 1.126258373260498, 0.031320273876190186, 1.2992274761199951, -1.087536334991455, 1.3485766649246216, -1.0540577173233032, -0.3802941143512726, -0.11322640627622604, 1.0560835599899292, 1.1838809251785278, 0.9576792120933533, 0.2854439318180084, -2.2896316051483154, -0.4214200973510742, 1.4382272958755493, 1.7674970626831055, 2.3484809398651123, 0.1561128795146942, 0.09410650283098221, -0.677180290222168, -1.2247936725616455, -0.4170774519443512, 1.3307123184204102, 0.5078298449516296, 1.0426467657089233, 0.516247570514679, -0.2675665020942688, -0.2879685163497925, -1.4571136236190796, -0.7680854201316833, -0.8599694967269897, 0.7845048308372498, -1.4646947383880615, -1.8992234468460083, 1.3608595132827759, -0.3076123297214508, -0.33273714780807495, 0.27600157260894775, 0.08624338358640671], [1.7587944269180298, -0.3815368711948395, -0.30005723237991333, -0.8239342570304871, -1.478806972503662, 1.2989144325256348, -1.4031199216842651, 1.3146253824234009, 0.5236164927482605, -1.074120044708252, -0.3883037269115448, -0.024436399340629578, -0.7292059659957886, 1.0340744256973267, 0.7196163535118103, 0.41946524381637573, -0.2819870710372925, 1.1130961179733276, -1.4334287643432617, 1.6143375635147095, 0.5706138610839844, -1.5456602573394775, -1.3697806596755981, 0.008250120095908642, -0.17486858367919922, -0.6337292790412903, 1.2481275796890259, 0.07823622226715088, 1.0428131818771362, -0.8308210372924805, 0.20726467669010162, -0.34756237268447876, -0.21651515364646912, 1.602927327156067, -0.9998391270637512, -1.5431522130966187, 1.629574179649353, 1.1392467021942139, -0.31927260756492615, 0.16965971887111664, -1.3621721267700195, -1.2440431118011475, 1.4610179662704468, -0.9236518740653992, -0.4713871479034424, -0.5224736332893372, -0.26100388169288635, -0.21896369755268097, -0.6443915963172913, -1.496808648109436, -0.6614047288894653, 0.6528971195220947, 0.8001630902290344, -0.9960541725158691, -0.5210952758789062, -0.8503988981246948, 0.4955365061759949, 1.408481478691101, 1.0885248184204102, 0.7334506511688232, -0.41414186358451843, 0.6317660212516785, 0.8255910873413086, -0.6533169746398926, -0.16147224605083466, -1.234968662261963, -0.21900664269924164, -0.5507310628890991, 1.1604983806610107, -0.08327293395996094, 0.81646329164505, 0.011571470648050308, 0.932935893535614, 0.5357219576835632, 1.1289223432540894, -0.7145044207572937, 0.224643275141716, 0.08527602255344391, 1.9947599172592163, -0.051110170781612396, 0.5329360961914062, 0.5006161332130432, 0.013190400786697865, -0.19802866876125336, -0.30351772904396057, -0.01865917257964611, -0.1460123360157013, -0.37387266755104065, -0.2705272436141968, -0.8338823914527893, 1.0134985446929932, 0.12120013684034348, -0.10502152889966965, 2.4415078163146973, 0.8441082835197449, -0.01601681485772133, 0.7534688115119934, -0.6997688412666321, 0.08293652534484863, 0.6008256077766418, 0.44805043935775757, -0.8985382914543152, 1.2062339782714844, -0.5975921154022217, 0.23358498513698578, 0.40636658668518066, -0.5101789236068726, 1.0075435638427734, -1.711529016494751, 0.36788156628608704, 0.6943992972373962, -0.05160311236977577, 1.4026857614517212, -0.45072677731513977, 0.5683335661888123, 0.9335219860076904, 0.23238606750965118, -0.18931280076503754, 0.4176413416862488, 0.3916856348514557, -1.5155760049819946, 0.6842923760414124, 0.3765789866447449, 0.3190343379974365, 0.1288262903690338, 0.6882991790771484, 1.1497465372085571, 0.12377465516328812, -0.7460503578186035], [1.3972041606903076, 1.2703546285629272, -0.01176530122756958, -1.1802866458892822, 0.9750764966011047, 0.43048298358917236, 0.22743655741214752, -1.052446961402893, -2.411597490310669, -2.422010898590088, -1.6110928058624268, 0.9931363463401794, -0.4458322525024414, -0.041167229413986206, 0.4436558187007904, -0.10346728563308716, 0.7197076678276062, 1.1218291521072388, 2.8111820220947266, -0.4371333420276642, 1.969420313835144, -0.9679951071739197, -0.2782081961631775, -0.1766405552625656, -0.4297351539134979, 0.41107815504074097, -0.18200042843818665, -0.48774492740631104, -0.9508533477783203, -0.6234588623046875, -0.4824027717113495, -0.9831516146659851, 0.41181710362434387, -0.5110325813293457, -0.706345796585083, -0.35313934087753296, -0.0821802094578743, -1.5708484649658203, -2.3262436389923096, 0.40059229731559753, 0.44933032989501953, -0.7425505518913269, 0.7796710729598999, 0.2624861001968384, 0.5291211009025574, 1.6838209629058838, -1.4047367572784424, -0.4090907871723175, -0.0036723020020872355, -1.0969724655151367, -1.0029560327529907, 0.6235002279281616, -0.28978511691093445, -1.9205011129379272, 1.212730884552002, -0.6483675837516785, -0.7444472908973694, -1.4328794479370117, 0.7021213173866272, -0.6882601976394653, 1.236365795135498, 0.3121419847011566, 0.0048199002631008625, -1.6249580383300781, -0.23426905274391174, 0.10537199676036835, -0.2333468496799469, -0.9329982995986938, -0.7212792634963989, -1.207018256187439, -0.7789093852043152, -1.4834996461868286, 0.060810111463069916, -1.2266958951950073, -0.018175773322582245, -0.2667144536972046, -0.18448346853256226, 0.7868028283119202, 0.7994039058685303, 0.21560029685497284, -0.8072441220283508, 0.6347975730895996, -0.18069423735141754, 1.5729161500930786, -1.7173950672149658, 0.09276416897773743, 0.5345171689987183, 0.10558289289474487, 1.0326387882232666, -1.7550064325332642, -1.1489702463150024, 0.6336463093757629, -0.44269034266471863, -0.7567406296730042, 1.2359333038330078, 0.5778499841690063, 0.1828462928533554, 0.36232203245162964, -0.5371078252792358, 1.8628038167953491, 2.6481761932373047, -0.7434391975402832, -0.9013797044754028, -1.0404826402664185, 0.1728888899087906, 0.19165365397930145, -1.051838994026184, 0.8731400370597839, 1.623095154762268, 0.7072977423667908, 0.014019113965332508, 0.5373687148094177, -0.4332306981086731, 0.08712517470121384, 0.5813463926315308, 0.4156596064567566, -2.030710220336914, -0.7559137940406799, -1.9359757900238037, -0.20675358176231384, -1.3591703176498413, -0.06045426428318024, -0.35316532850265503, 0.6025813817977905, 1.9037816524505615, -1.159576177597046, 1.2625658512115479, -1.698194146156311, 0.24324685335159302], [-0.44049644470214844, 1.221681833267212, -0.29530587792396545, 0.9467130899429321, 1.6683144569396973, 0.9829239845275879, -0.5465290546417236, -0.9180644750595093, -0.4472287893295288, 0.9170685410499573, -0.49029162526130676, -0.4266991913318634, 1.3858935832977295, -1.069478988647461, 1.8204984664916992, 0.014923174865543842, -1.5656224489212036, 1.2537329196929932, -0.8140568733215332, 0.36273062229156494, -0.5228040814399719, -0.4743911921977997, 0.15103012323379517, 0.1587560474872589, 0.9689843058586121, 1.4431613683700562, 0.3292238712310791, -1.4293004274368286, 0.4731760621070862, 2.0987296104431152, 0.6114541292190552, -0.10344383120536804, 1.9157813787460327, 0.08038415759801865, -0.260458767414093, -2.0032589435577393, -0.45128515362739563, -1.7238175868988037, -0.7578983902931213, 2.144706964492798, -0.48732975125312805, -1.1043905019760132, 0.4056416153907776, -1.1145800352096558, -0.2624712288379669, -0.3084089756011963, -1.9581512212753296, 1.9687719345092773, -0.6081129312515259, -0.026787476614117622, 1.0468066930770874, -0.42474988102912903, 0.4763186573982239, 0.8670763373374939, -0.4072408974170685, -0.9587690830230713, 0.8929548859596252, 1.8194797039031982, 0.07421872019767761, -0.04363805055618286, -1.676328420639038, -1.797264575958252, 1.1458706855773926, -1.2028728723526, -0.9476400017738342, 1.0946370363235474, 0.5421016812324524, -0.2102189064025879, -0.5209283828735352, 1.4425280094146729, 0.982408881187439, -0.7668260931968689, -1.6850799322128296, 0.159013032913208, -1.46870756149292, -0.16256669163703918, -0.18120364844799042, 0.4591846764087677, 0.972495973110199, -0.7336907982826233, -0.7125327587127686, 0.10961299389600754, -0.4487035572528839, 0.2573356330394745, -0.0615963488817215, -0.14358462393283844, -1.6072431802749634, -0.6514276266098022, -0.2861279845237732, -0.24372552335262299, -0.26215729117393494, -1.3182588815689087, 0.3327586054801941, -2.01308274269104, -0.6748554110527039, 0.2107078582048416, -1.6080968379974365, -2.0591301918029785, -0.9418755173683167, 0.12042098492383957, -0.7570663094520569, -0.030461551621556282, -1.4819265604019165, 0.9192723631858826, 1.032686710357666, -1.213715672492981, 0.590806245803833, -1.4203641414642334, 0.15496860444545746, -0.2541307806968689, -0.8325729370117188, -0.4171678423881531, -1.416915774345398, 0.7581044435501099, 1.6763864755630493, 1.7886826992034912, 1.075581431388855, 0.08229276537895203, 1.1984872817993164, -1.3910712003707886, 0.4159535765647888, -1.0288450717926025, 1.524756669998169, 1.5535942316055298, 1.3113731145858765, -0.6286860704421997, -0.25843480229377747, 1.0610295534133911, -0.585582435131073], [1.9353739023208618, -0.4633612632751465, -2.8489062786102295, -0.997554361820221, -0.09465644508600235, 0.7268245220184326, 1.5303843021392822, 0.006080371327698231, -1.9372167587280273, -1.9724884033203125, 1.186916708946228, -0.31754234433174133, 0.7801898717880249, -1.9603500366210938, -0.2593447268009186, -0.8840268850326538, -1.2015886306762695, -1.7404972314834595, -1.061954140663147, -0.5383923053741455, -0.43204057216644287, 1.571031093597412, 0.06401430815458298, -0.6870126724243164, 0.36396074295043945, -1.1746015548706055, -1.956796646118164, 0.8923177123069763, 1.1246248483657837, -0.3288610279560089, -0.19145303964614868, -0.8969926238059998, -0.2181963473558426, 1.326992392539978, -0.03942373767495155, 0.5444648265838623, -0.5736979246139526, -0.12032300978899002, 0.8786101341247559, 0.13091382384300232, 0.2070804387331009, 0.15230432152748108, 0.3155265152454376, -0.14034077525138855, -1.3837765455245972, -0.12698963284492493, -0.6457470059394836, -0.19082607328891754, -0.9026163816452026, 0.07461696863174438, 0.06444043666124344, -0.09072710573673248, 1.7297793626785278, -0.557365894317627, 1.5701957941055298, -2.476750373840332, -0.5733200907707214, -0.9318492412567139, 1.7350655794143677, -0.0738854855298996, 0.8824067115783691, -1.0029675960540771, 0.5879379510879517, 0.22511635720729828, 0.7705758213996887, -0.31015563011169434, 0.23140735924243927, -1.0252513885498047, -2.0961520671844482, 0.8899663686752319, 0.9015242457389832, 0.03502563014626503, -0.9571408629417419, -0.23635129630565643, 0.019931523129343987, -1.1217856407165527, -0.5592548847198486, 0.8526118993759155, -1.539223313331604, -0.9309458136558533, 0.8020842671394348, -0.3239053189754486, -1.0193595886230469, -0.5399240851402283, -0.34818530082702637, -0.905450701713562, 0.5522559285163879, -1.8789921998977661, -1.00734543800354, -1.0471183061599731, -0.14275941252708435, -0.27030590176582336, -1.7443382740020752, -1.2906419038772583, -0.6596853137016296, 0.18851467967033386, 1.1897876262664795, 0.5643200874328613, -2.1911306381225586, -1.4059277772903442, -0.6524524092674255, -0.7105137705802917, -1.1198508739471436, -0.5885566473007202, -0.8034849762916565, -1.272758960723877, 1.5158189535140991, 0.8833461403846741, 0.5285599827766418, 1.3660459518432617, -0.21016451716423035, 1.1901171207427979, -0.3634530007839203, -0.239303857088089, 1.5367345809936523, 0.9080367088317871, 0.3295114040374756, -1.8988864421844482, 0.26781946420669556, 0.18746095895767212, -0.12507741153240204, 0.675839364528656, -0.38562947511672974, 0.7162278294563293, 2.5249881744384766, -0.37651926279067993, -1.025549054145813, 0.3083506226539612, -0.0457349456846714], [-0.6841161251068115, -1.2231987714767456, -0.8072590231895447, 0.7073771357536316, -0.8477351665496826, -0.45797044038772583, 0.6682052612304688, -0.44335266947746277, -0.5320687294006348, 0.140625461935997, 0.250832200050354, -1.5150781869888306, 1.6401808261871338, -0.34690290689468384, 1.0431933403015137, 0.4631669521331787, -0.08725094795227051, 0.07805536687374115, 1.0641252994537354, -0.5528963804244995, 0.06408874690532684, -0.7426192760467529, -0.2400992065668106, 0.31887510418891907, 0.042676255106925964, -1.361053228378296, 2.3475332260131836, 0.966440737247467, 0.6758737564086914, 0.0051267570815980434, -1.3628249168395996, -2.0549285411834717, 1.3130860328674316, -1.3558708429336548, -0.2255851775407791, -0.8658937215805054, 1.7555991411209106, -0.10295718908309937, 0.10351387411355972, 1.0580388307571411, -0.31731870770454407, -1.2009093761444092, 2.1507463455200195, 0.49246737360954285, 0.2816055715084076, 0.07004941999912262, 0.8304588794708252, 0.5481369495391846, -0.9893592596054077, 0.24912045896053314, 0.6731388568878174, -0.48982328176498413, -0.5893107056617737, 0.5817760825157166, -1.7689456939697266, -0.1136857122182846, 1.108022928237915, -0.23312421143054962, -0.8781126737594604, -0.04492196440696716, -0.8225634098052979, -1.9623481035232544, -0.5276829600334167, 1.26030695438385, 0.6014670133590698, -0.9819312691688538, -0.4914570450782776, 0.7812997102737427, 2.071988344192505, 0.2467900961637497, -2.8338623046875, 0.2622196674346924, 0.153679758310318, -1.2365756034851074, 0.17382118105888367, -0.05452694743871689, -0.9119091033935547, 0.3814675509929657, -0.6310918927192688, -0.3472253680229187, 0.21469925343990326, 0.6273716688156128, 0.12368419021368027, 0.4233037233352661, -0.40736427903175354, 0.9816317558288574, 0.030590297654271126, -1.0383650064468384, -0.2189687192440033, 1.365225911140442, -0.46374183893203735, -1.3422454595565796, -1.427955985069275, 0.5435581803321838, -1.9157662391662598, 0.384602814912796, -1.3425272703170776, 0.09365318715572357, 0.34261205792427063, -0.8553922176361084, -0.13589689135551453, -1.4696400165557861, -1.183235764503479, 0.9176046848297119, 0.840334415435791, -0.9776199460029602, 0.7169193029403687, -0.6778233051300049, 0.0575747936964035, 1.5144381523132324, -0.555054783821106, 1.496728539466858, -1.2011537551879883, -2.525097131729126, 1.2067111730575562, -1.3769985437393188, 0.43365275859832764, -1.6377781629562378, -0.6108441948890686, 0.8243290781974792, -0.9853952527046204, -1.36148202419281, 0.44650864601135254, 0.8532342314720154, -0.22679021954536438, -0.7722434997558594, -1.3996713161468506, 0.4875737428665161, -1.0694501399993896], [0.8313027620315552, -1.3075454235076904, 0.25782865285873413, 0.38172563910484314, 1.1365857124328613, 0.3045336902141571, 0.8119568228721619, 1.105427861213684, 0.647578239440918, -0.06768535077571869, -0.5606312155723572, 1.887549638748169, 1.1297351121902466, 0.033390142023563385, 0.4232540428638458, -0.36727356910705566, 0.5323227643966675, 0.7164078950881958, -1.5511609315872192, 0.9721962213516235, -1.4791940450668335, 0.2596101760864258, 0.8330949544906616, -0.4119538366794586, 0.5971672534942627, -0.7804537415504456, -0.3683798611164093, -0.3236233592033386, 1.0980658531188965, -2.0797832012176514, 0.13883288204669952, 0.09675614535808563, 0.6086790561676025, 2.111243963241577, -1.9585071802139282, 1.5163334608078003, 0.4760929346084595, -1.4573255777359009, -0.4325881004333496, 1.0355303287506104, -1.0493141412734985, 0.944833517074585, 1.6587448120117188, -1.0089107751846313, -1.4596446752548218, 3.0759057998657227, 0.5387513041496277, 1.0653873682022095, -0.5671088099479675, -1.3262898921966553, 0.43728262186050415, -0.19909237325191498, 1.635197401046753, 0.9245697855949402, 1.3368322849273682, 0.1935722380876541, 0.45889487862586975, -1.6032682657241821, 0.6068242788314819, 0.12796524167060852, -0.5585451126098633, 0.04797456040978432, 0.19399400055408478, 1.5830610990524292, 0.7697352766990662, 0.5024710893630981, -1.305169701576233, -0.05342741683125496, -0.704704999923706, 2.221367359161377, -0.14419637620449066, -0.9932811856269836, -2.268320322036743, -3.7426247596740723, 0.8508436679840088, -2.204739809036255, 0.8768661618232727, 1.4867652654647827, -0.23914176225662231, -1.858162760734558, -1.8674519062042236, 0.734638512134552, -0.5018215179443359, 0.03328721970319748, 0.273591011762619, 0.11643275618553162, 1.4901556968688965, 0.29845690727233887, -0.1160082072019577, 0.13499198853969574, -1.921886682510376, 0.6761245727539062, 1.0097545385360718, -0.6166872978210449, 0.1597207486629486, 0.45322683453559875, 0.19598662853240967, -0.211471289396286, -0.639448881149292, 1.041002631187439, -0.03207802772521973, 0.6304803490638733, -0.5151689648628235, -0.9973871111869812, 0.634382426738739, -0.2328803837299347, 0.4503031373023987, 0.5633499622344971, 0.42803072929382324, 0.18167032301425934, -0.07666915655136108, 0.5941464304924011, -0.44968903064727783, -1.7015467882156372, -0.699233889579773, 0.9401237368583679, 0.8906960487365723, 0.34656772017478943, 1.1577739715576172, -0.6895821690559387, 0.6876001954078674, -1.7332532405853271, 0.9549545645713806, 1.923026442527771, -0.3752051591873169, 0.3936338722705841, -0.37493017315864563, -0.2994970381259918, 2.6086843013763428], [-0.6651086807250977, 0.5089908838272095, 1.0758769512176514, -2.6176719665527344, -1.4955006837844849, 2.3320271968841553, 0.16994304955005646, 0.15425215661525726, -1.139699101448059, 1.1197669506072998, 0.6379850506782532, -0.7242158055305481, -0.3932366967201233, 2.6646432876586914, 0.05577673390507698, -1.26699960231781, 0.7209413647651672, 0.8171289563179016, 1.4679172039031982, 0.6271584630012512, 0.874221682548523, -0.5518273115158081, -0.6283787488937378, 1.9709092378616333, 1.9159855842590332, -0.7159200310707092, 0.7426021695137024, -0.7422627210617065, 1.5476102828979492, 6.795691297156736e-05, -1.2824068069458008, 0.36159637570381165, -0.7530662417411804, -1.815524697303772, -0.15805357694625854, -2.502455949783325, -1.5413093566894531, 0.0224210973829031, 0.1895531862974167, 0.40757593512535095, 0.6718366742134094, 1.2673161029815674, -1.8223378658294678, -0.5032115578651428, -0.37613263726234436, 0.9892997145652771, 1.7312904596328735, 1.083430528640747, -0.9216243028640747, -0.23703081905841827, 2.704564094543457, 0.7789406776428223, 0.8058995008468628, -0.726070761680603, 0.3896385729312897, 0.8112751245498657, -1.444453477859497, -1.5884559154510498, -1.5605722665786743, -0.1637900322675705, -0.4194769859313965, -0.7855217456817627, 0.7738921046257019, 0.1687948852777481, 0.5962735414505005, -2.2902872562408447, -1.1322959661483765, -0.4197327196598053, 0.7078620791435242, 0.40723323822021484, -0.9796910285949707, -0.8645815849304199, 0.6426151394844055, 1.0213356018066406, 1.2900432348251343, -0.18300136923789978, 0.5430206060409546, -1.5995758771896362, 1.2726582288742065, -0.33293840289115906, 1.9486483335494995, 0.6034408807754517, 0.05508707836270332, -0.33149629831314087, 0.49019595980644226, -0.3350374698638916, 0.1356111317873001, -1.4311866760253906, 0.6434295773506165, -0.5076950788497925, -1.13860023021698, -0.5867643356323242, -0.6979573369026184, 0.44716864824295044, -0.7685616612434387, 0.5415136218070984, 0.23782707750797272, -1.1308590173721313, -1.3905800580978394, 0.5285106897354126, -2.7900283336639404, -0.16862982511520386, 1.2329994440078735, 0.3486696779727936, -1.5800809860229492, -0.6130932569503784, 1.5237505435943604, 1.2246214151382446, 1.1028399467468262, -0.9367824196815491, -1.5248820781707764, -0.8224636316299438, 0.5984869003295898, 0.6814573407173157, 2.7179136276245117, -0.9448150992393494, 1.0231915712356567, -0.6735572814941406, -2.5091447830200195, -0.11121083796024323, 0.4649096131324768, -0.9978749752044678, 1.3424056768417358, 0.8743633031845093, -0.9652349948883057, -0.31625133752822876, -0.6420782804489136, -1.2644426822662354, 0.6241835951805115], [0.07569362968206406, -0.11254590004682541, -0.3550095856189728, 0.7212486267089844, -1.4137978553771973, -0.8363182544708252, -0.6692084074020386, -1.9244232177734375, -0.690791130065918, 0.3536403775215149, 0.8723009824752808, -1.079114556312561, 2.7477755546569824, 0.8470794558525085, -1.723932147026062, 0.288481205701828, -0.9600777626037598, -1.6181257963180542, 0.565359354019165, 0.29041367769241333, 0.6795797944068909, -0.8665029406547546, 2.0279853343963623, -0.025702305138111115, 0.37341687083244324, -0.8325486779212952, 0.35304364562034607, 0.3291015028953552, -0.32684215903282166, 1.3929492235183716, -0.0050529916770756245, -0.6657865643501282, -0.9014092087745667, -1.651824712753296, -0.2521265149116516, 0.15572991967201233, -2.596668004989624, -0.6839523315429688, 1.0108736753463745, -0.2403651475906372, 2.125905752182007, -0.10366548597812653, -0.5953007936477661, -0.5880086421966553, -0.40616777539253235, 1.5586705207824707, -0.2576306462287903, 1.782732367515564, 1.0142807960510254, 0.7927348017692566, -0.16804051399230957, -0.9503429532051086, 0.8396073579788208, -1.754644751548767, 0.011286887340247631, 0.8318434953689575, -1.1063549518585205, 0.5261291265487671, -0.21740445494651794, 1.0680381059646606, -0.9788042902946472, -0.3890701234340668, -0.7393248677253723, -1.4247151613235474, 0.019007951021194458, 2.543499708175659, -0.39924588799476624, -0.7878215909004211, 0.1835945099592209, 2.448420524597168, 2.100862979888916, -1.6123660802841187, 0.5178020000457764, -1.867750644683838, -0.5408058166503906, 0.8794139623641968, -0.9803214073181152, -0.15113770961761475, -0.023506775498390198, 0.5734521150588989, -0.1540144681930542, 1.6068440675735474, 0.16066458821296692, -1.366320252418518, 0.6105318069458008, -0.008070816285908222, 0.24504567682743073, 1.3054962158203125, -0.9134411811828613, 0.6266037821769714, 0.0477772057056427, 0.9290251731872559, 1.7679039239883423, 0.1010051742196083, -0.9721345901489258, -0.2934657335281372, 1.4660327434539795, -2.3185622692108154, -0.3911866843700409, -0.5265771746635437, -1.9783401489257812, -1.1464403867721558, 0.998328685760498, 0.7514534592628479, -1.2882298231124878, 0.6283262968063354, 0.20762397348880768, 0.9811316728591919, 0.67286616563797, 1.089689016342163, -1.0315712690353394, -1.1651285886764526, -0.9863969683647156, 0.26293888688087463, 1.3723844289779663, 0.8816026449203491, -1.4529372453689575, 1.0345686674118042, -2.607799530029297, -0.7837916612625122, 1.3408327102661133, -0.8731964230537415, 0.26369708776474, -2.0836663246154785, 1.1394134759902954, 0.09811301529407501, -0.7926504015922546, 1.0173075199127197, 0.6788961887359619], [0.1431538611650467, 0.2944676876068115, -1.4580821990966797, 0.2877906262874603, -2.340296506881714, 2.1345555782318115, 1.4324378967285156, -0.3444809019565582, 0.36151713132858276, -1.9767122268676758, 0.44689276814460754, -0.04158412292599678, -0.5825569033622742, -1.1024430990219116, -0.5068235993385315, -0.4633798599243164, -0.24210144579410553, 0.8764380216598511, -0.43369704484939575, -1.3779817819595337, -2.1342995166778564, -0.12464018911123276, 1.0686277151107788, 0.5891865491867065, 0.2841764986515045, -0.4896203577518463, 0.23168912529945374, 0.8993962407112122, -0.38338717818260193, 0.0201959740370512, -0.6665916442871094, -1.0442641973495483, -1.172380805015564, -1.616225004196167, 1.900754451751709, -0.6046708226203918, 1.6389213800430298, -2.4460089206695557, -0.8925805687904358, 0.8221239447593689, 0.2850622236728668, 0.10647986084222794, 0.6614241600036621, -1.93868887424469, -0.668075680732727, 0.685823917388916, 0.2499643862247467, 0.7339159250259399, 1.0059822797775269, 0.5020179748535156, 0.33637359738349915, 1.28960382938385, -0.8715482950210571, -0.44391930103302, -0.9794974327087402, 0.7635977864265442, -1.1781871318817139, -0.37976351380348206, 2.696582078933716, 0.7397081255912781, 1.2494641542434692, 1.1040467023849487, 0.42500394582748413, 0.11166023463010788, -0.1997741311788559, -0.7442821860313416, -0.44584938883781433, -0.6794726848602295, 0.9992138147354126, 1.736603856086731, 0.17259155213832855, 0.6843097805976868, 0.31133076548576355, -0.8352678418159485, 0.23388084769248962, 2.1679842472076416, -1.0041576623916626, 0.22146672010421753, 0.2896402180194855, -0.8287182450294495, -0.6776053309440613, -0.8800840377807617, 0.37842750549316406, 0.3574170470237732, -0.04295758530497551, -1.0742027759552002, -1.1693305969238281, 1.548957109451294, -0.805679976940155, -1.0238194465637207, 0.2888144254684448, 0.43976378440856934, -1.7353546619415283, -0.27749037742614746, 0.2730022072792053, -1.2939965724945068, 1.2049514055252075, -3.3777129650115967, 0.32950934767723083, -1.0387842655181885, -1.3841618299484253, -1.0440481901168823, 0.28281137347221375, 0.041241809725761414, -0.7852451205253601, -0.08153847604990005, -1.0054680109024048, -0.25191617012023926, 0.27363064885139465, 1.6441714763641357, 1.011492371559143, -0.7403791546821594, -1.8029587268829346, 0.29018694162368774, 1.0933343172073364, 1.1176230907440186, 1.226158618927002, 0.8055378198623657, -1.619299054145813, 0.015411815606057644, 0.8138936758041382, -1.217854619026184, 0.3427737057209015, -0.23496472835540771, 0.11407189071178436, 0.3078821301460266, -0.22228412330150604, 1.3419760465621948, 0.765364408493042], [-0.7177293300628662, 1.5536961555480957, 2.1821305751800537, 0.762535810470581, -0.4451669454574585, 0.08735285699367523, 0.8836655616760254, 1.1771327257156372, 0.6955990791320801, -1.2335771322250366, -0.5366390347480774, -0.06806882470846176, -1.199646234512329, 0.03158776834607124, -0.0038779403548687696, 0.14966656267642975, 0.9284758567810059, 1.7406442165374756, -0.43404334783554077, -0.03624309226870537, 1.5619834661483765, -2.713472604751587, -0.39751753211021423, 1.0275557041168213, 0.5880982279777527, -0.8439329266548157, -1.0850398540496826, -0.2154625803232193, 0.7063642740249634, 0.48774784803390503, -0.3750154674053192, -1.5350794792175293, -0.955852746963501, 0.5489267110824585, 0.14408473670482635, -0.3319874405860901, -0.6270909905433655, -0.06834524124860764, -1.1279066801071167, 1.7019470930099487, -0.2768189311027527, -0.13916850090026855, 0.5199337601661682, -0.1382792592048645, -1.0311156511306763, 0.5744426250457764, -1.8636362552642822, 1.9368395805358887, 1.0586779117584229, -1.4961893558502197, 0.04016570374369621, -1.5182243585586548, -1.1112271547317505, -1.6335132122039795, -0.7271416783332825, -1.1697347164154053, 1.6858012676239014, 0.21589075028896332, -0.12851864099502563, 1.6237581968307495, -0.8856682777404785, 1.7509031295776367, -0.7460338473320007, -1.0597671270370483, -1.104830026626587, 0.4834883511066437, -0.988216757774353, 0.28091326355934143, -0.13551554083824158, -0.3086546063423157, 0.14137032628059387, -0.3494561016559601, -1.7158877849578857, 0.05221712961792946, 0.7681573033332825, -2.379997730255127, -0.22247791290283203, -0.27723485231399536, 0.7920467257499695, 0.07221504300832748, 1.0158253908157349, -0.2719098925590515, 0.18594834208488464, 0.042486779391765594, -0.4568224251270294, -0.2676788568496704, 0.14349694550037384, -0.8111743330955505, 0.8258814811706543, 0.780142605304718, 0.8662880659103394, -1.3431411981582642, 0.3257215917110443, -0.9417824745178223, -1.2768086194992065, 0.40896502137184143, 0.12425565719604492, 0.08201032876968384, -0.6078781485557556, 0.2720697224140167, -0.6733553409576416, -0.2726643681526184, 0.42456477880477905, 1.094681739807129, -0.2957039773464203, -0.867493212223053, -1.2778009176254272, -0.32969552278518677, 0.841837465763092, 0.09878689795732498, -0.014179940335452557, 1.1648353338241577, 0.17730721831321716, 0.03765631467103958, -1.797540307044983, -0.9996380805969238, -0.7435126304626465, 0.5873294472694397, 0.5382113456726074, -0.22673280537128448, -0.3383389413356781, -1.44297456741333, 1.093631386756897, -0.41411077976226807, 0.3549351394176483, -0.38919559121131897, -1.631540060043335, -0.7252098917961121, 0.6178988814353943], [-1.5659911632537842, -1.38966703414917, -0.025557948276400566, 1.2396667003631592, 1.2484992742538452, 0.5919505953788757, -1.8138664960861206, -1.377874493598938, 0.017970407381653786, 0.2716365158557892, 0.17011940479278564, 0.31862711906433105, -0.2666628062725067, -0.05542122945189476, 0.09859266877174377, 0.5245600938796997, 1.2445584535598755, 1.5628095865249634, 0.5157748460769653, 0.17766772210597992, -0.195030078291893, 2.7288665771484375, 1.3240023851394653, 0.6613816618919373, 0.20633891224861145, -0.005212866235524416, 0.5339123606681824, -1.6646440029144287, 0.578795850276947, 0.039447810500860214, 0.07308237999677658, -0.31833505630493164, -0.5789244771003723, 0.43732187151908875, -2.277082920074463, -0.755237340927124, 0.2301897406578064, 0.3162252902984619, -0.4153888523578644, 0.6890647411346436, 1.0817289352416992, 1.5649069547653198, -1.5241851806640625, 0.4338289499282837, 0.6968122124671936, -2.2263400554656982, -1.2383077144622803, 1.4657167196273804, 0.614449143409729, 0.4514375925064087, 1.0410054922103882, -0.2574237287044525, 1.9880610704421997, 0.4371282160282135, 1.50490140914917, 0.9862799048423767, 0.6619355082511902, 0.7612008452415466, -0.8302897214889526, -0.12771037220954895, 0.6160308718681335, -1.0524903535842896, 0.13251937925815582, -0.049924034625291824, -1.5319061279296875, -0.7934784293174744, 1.010489821434021, 0.7211440801620483, -2.1668930053710938, -2.5265634059906006, 0.21388062834739685, 1.1773569583892822, 0.8186056017875671, 0.7794066667556763, 0.7696316242218018, -1.780318260192871, 1.245486855506897, -0.19098791480064392, 0.7338204979896545, 0.6193279027938843, -0.5703344941139221, 0.12743447721004486, 0.9036845564842224, 0.02261710725724697, 1.5463931560516357, -1.6307506561279297, 0.4467707574367523, 0.08744153380393982, -0.08297540992498398, -0.36189112067222595, 0.05493047460913658, -0.960205614566803, -0.45249003171920776, 0.5699011087417603, -0.24218225479125977, 1.5979702472686768, -1.68869948387146, -1.8385865688323975, 0.08345595747232437, 0.7711697220802307, 0.8786749839782715, 0.45532339811325073, -0.14787442982196808, -1.4895031452178955, -0.08358223736286163, -0.5744710564613342, 1.3207950592041016, -1.4512337446212769, 0.2544856369495392, -0.10497256368398666, -1.3247193098068237, -1.8038331270217896, 0.6760938167572021, 0.9689149856567383, -0.9258442521095276, 0.3356306254863739, -1.7033772468566895, -1.2676366567611694, 0.7219558358192444, 1.1401176452636719, -0.5219241380691528, -0.9823625683784485, 0.08942292630672455, -0.3538196384906769, -0.87583327293396, 2.1101791858673096, -0.38183268904685974, 0.40985727310180664, -0.5036870241165161], [-0.5834397673606873, -0.7979774475097656, -1.7261167764663696, -0.3122493028640747, 0.17847709357738495, -0.19592760503292084, -1.221910834312439, -1.5226155519485474, 2.293826103210449, 2.0159006118774414, -0.2020540088415146, -0.29251614212989807, 0.6958569884300232, 1.573651909828186, 1.250145435333252, -0.15917019546031952, 0.6492292881011963, -1.3675334453582764, 0.05148882046341896, -0.9945492744445801, 1.300947666168213, 0.8213658928871155, -0.3591966927051544, -0.3233816623687744, -0.06835075467824936, 0.19393013417720795, -1.5812206268310547, 0.1934366226196289, -0.13717782497406006, 0.3143167197704315, -1.0985453128814697, 0.5909003019332886, -0.0052184127271175385, -1.2466570138931274, 0.5120357871055603, -0.06660982221364975, -1.4243533611297607, 0.7410731315612793, -0.7903492450714111, -0.18840491771697998, -0.2057952880859375, 0.39266425371170044, 0.7013945579528809, -0.784633994102478, 0.6111829280853271, 0.5352213978767395, -0.12862513959407806, 0.3298088312149048, 0.18611390888690948, -0.1389736384153366, -0.7072049379348755, 0.3211665153503418, 0.669721245765686, -0.6910127401351929, -0.5120641589164734, 0.5816358923912048, -0.01982615701854229, 1.7807037830352783, 1.5875592231750488, -1.9002920389175415, 0.250022828578949, -0.1567697823047638, -1.770896553993225, -1.7591030597686768, -1.692460060119629, 0.9768977761268616, 1.1937624216079712, 0.26963043212890625, 0.9127048254013062, 0.23681235313415527, 0.685701310634613, 0.049552060663700104, -2.061373710632324, -0.6503428220748901, 0.6220674514770508, 0.8859820365905762, 1.3182016611099243, 0.38030245900154114, 0.9360858201980591, -0.331066370010376, -0.21056829392910004, -0.046476155519485474, -0.39861005544662476, -1.1008422374725342, 0.4252224564552307, 0.054128363728523254, -0.8811991810798645, -0.12848395109176636, -0.7056849002838135, 0.3975428640842438, 0.33435046672821045, -1.769389033317566, -1.6276233196258545, 0.022957852110266685, 1.994013786315918, 0.8151301145553589, -0.036006320267915726, 0.18651169538497925, 0.5393857359886169, -0.44190412759780884, 0.638451337814331, 0.28749901056289673, 1.4092247486114502, -1.7319350242614746, -0.7477250099182129, 0.28271597623825073, -2.8330554962158203, -1.1884706020355225, -0.18885117769241333, -0.6219239830970764, -0.08103974163532257, -0.6935287714004517, -1.3415720462799072, -0.33936166763305664, -0.318895548582077, 1.8082051277160645, 0.14401960372924805, -1.5805003643035889, -0.9087547659873962, 0.49577441811561584, -1.4224166870117188, -1.4785583019256592, -0.4584867060184479, -0.9601654410362244, -1.1462855339050293, -1.5253266096115112, 0.7978584170341492, -0.6760242581367493, 0.4644854664802551], [-0.9608129858970642, 1.4277291297912598, -0.7622668147087097, 0.3674687147140503, 1.7299575805664062, 0.5603947043418884, 1.1666208505630493, -1.131328821182251, 2.20566725730896, 1.25991690158844, -0.03065224550664425, 0.17030267417430878, 1.4989827871322632, 0.36596864461898804, -1.1415305137634277, 0.39267563819885254, 0.5138601660728455, -0.0885862410068512, -1.1903338432312012, -0.9452376961708069, -0.8829764127731323, -1.7872889041900635, -0.27688711881637573, -0.020587965846061707, 1.4086296558380127, 0.05583516135811806, -0.9245414137840271, -0.7269254922866821, -0.4510281980037689, -1.4612689018249512, 1.1947647333145142, 2.0181829929351807, -0.3596467077732086, 0.6828702092170715, 0.6911545395851135, -0.16371077299118042, 0.28927525877952576, 2.1252031326293945, -0.06542468070983887, 0.6789873838424683, -0.904688835144043, 0.391379177570343, -0.3763689696788788, -1.2567657232284546, -0.1654341220855713, -0.11932557076215744, -0.7208880186080933, -1.5310595035552979, 0.055088818073272705, -1.9375183582305908, -0.39765307307243347, -1.6168510913848877, -1.342633843421936, -0.37997907400131226, -0.06826478987932205, 0.2919104993343353, -0.5568883419036865, -3.277391195297241, 0.21078115701675415, 0.7425157427787781, 0.055314674973487854, 0.9596684575080872, 0.014856128953397274, 1.3689296245574951, -0.3675345778465271, -0.42157191038131714, 0.2438994199037552, 2.218459129333496, 0.46725496649742126, -0.5937572717666626, -0.08739940822124481, 1.2312039136886597, -1.0741857290267944, -1.7563101053237915, 0.7995589375495911, 1.950195550918579, 0.5835902094841003, 0.3519216775894165, 0.14173391461372375, 1.2153081893920898, 0.9130550026893616, -1.1252669095993042, -0.772090494632721, -1.1104367971420288, 0.009515737183392048, -0.8851484656333923, -0.4933171570301056, -0.29182958602905273, -0.5903768539428711, 1.663611650466919, 0.44112247228622437, 0.022112423554062843, -1.4934555292129517, 0.5772679448127747, 0.44853878021240234, 0.9397723078727722, 1.8876503705978394, 0.43896621465682983, -0.359162837266922, 0.18414530158042908, -0.22873087227344513, 0.1636429876089096, -0.37200677394866943, -0.3031446933746338, 0.22121992707252502, -0.22394715249538422, 2.589287281036377, 0.11608104407787323, 2.1902542114257812, -0.21602270007133484, -0.38058310747146606, 0.3067851960659027, 0.8703327178955078, 1.4046332836151123, 0.8459711074829102, -0.8636247515678406, 0.9349797368049622, 0.5781359076499939, 0.1147761270403862, -0.2867670953273773, 0.6783761978149414, -0.40175896883010864, -1.1840734481811523, 0.46401217579841614, -0.42785394191741943, -0.8070603609085083, 0.6584380865097046, -1.3353525400161743, 0.27720990777015686], [-0.11963226646184921, -0.8950806260108948, -0.035918738692998886, -1.2218751907348633, 0.30651357769966125, -0.5910382866859436, 2.540260076522827, 0.6183067560195923, 1.5270243883132935, 0.9416208863258362, -0.2908919155597687, -0.7197757959365845, -0.3880647122859955, -0.43074867129325867, 0.7102102637290955, 0.6382830142974854, 0.8691040277481079, -1.3037357330322266, -0.5958796143531799, 0.3573479652404785, 0.9590023159980774, 0.5106133222579956, 1.0687755346298218, 0.22603100538253784, -1.403632402420044, 0.2469913363456726, 0.4116487205028534, 0.26520296931266785, -1.925464153289795, -1.3043659925460815, 0.05273976922035217, 0.31638166308403015, -0.5527734160423279, 0.6352051496505737, -1.0219109058380127, -0.3615921139717102, -0.08731178194284439, 0.6971449851989746, -1.7185938358306885, -1.9256231784820557, 0.40168070793151855, 0.10564959794282913, 0.8241044878959656, -1.139884352684021, -1.3044755458831787, 0.37186214327812195, -0.1860738843679428, 0.17717838287353516, -0.24326226115226746, -0.1278071254491806, -0.6730250120162964, 0.3396661579608917, -1.467113971710205, 0.6973576545715332, -1.5283581018447876, -0.5643144845962524, 0.7392246127128601, -1.004982829093933, 0.633712649345398, -0.4418843984603882, -0.04417405650019646, 0.8514977693557739, 0.15994980931282043, 1.4915401935577393, -0.11755038052797318, 0.07769337296485901, -0.4235062897205353, 1.0017485618591309, -1.2110700607299805, -0.46780848503112793, 1.693867802619934, 0.5627309083938599, 1.0576236248016357, -1.1245040893554688, 0.10222700238227844, 0.014982392080128193, 0.8298369646072388, -0.2965344488620758, 0.9995898604393005, -1.3892502784729004, 0.6434466242790222, 0.6733448505401611, 0.3566577136516571, -0.33271682262420654, -1.3224915266036987, 1.8463586568832397, -1.0260003805160522, 0.6376410722732544, -0.5766512751579285, 1.3575310707092285, 0.523840606212616, -0.14873188734054565, -0.6118386387825012, 0.9175999760627747, 1.0329501628875732, 1.1353508234024048, -0.43435564637184143, 1.1643295288085938, 1.329206109046936, -0.7905864119529724, -1.2970514297485352, 0.3154210150241852, 1.3318649530410767, 1.3000179529190063, 1.8325183391571045, -0.0949564129114151, 0.4186999797821045, 2.101543426513672, 1.6894283294677734, 0.37191811203956604, -1.0021613836288452, -0.2774161696434021, -1.7426501512527466, 0.12575729191303253, 0.2935545742511749, -0.917451024055481, 0.7211383581161499, 0.4706127643585205, 0.06654223799705505, 0.07131019979715347, 1.725725531578064, 2.8534491062164307, 0.40804535150527954, -0.9528635740280151, 0.6002494692802429, 0.6391056180000305, 1.4356516599655151, 1.1652452945709229, 0.7237836718559265], [0.01015754695981741, 0.7013509273529053, 0.4857338070869446, -0.149021178483963, -1.377272367477417, -0.8095827102661133, 0.1393345594406128, 0.18332374095916748, -0.03721501678228378, -0.2685695290565491, -0.48410263657569885, 0.01849406212568283, -0.4672355055809021, 1.213504672050476, 0.5453237295150757, 0.4876392185688019, -0.7225176692008972, 2.049283742904663, 1.2654105424880981, -1.8164888620376587, 1.0844389200210571, -1.0050127506256104, -0.047845762223005295, -0.03574201464653015, 1.0106347799301147, 0.7439075112342834, 1.457112431526184, 1.0847331285476685, -1.9701347351074219, -0.014149267226457596, -0.6315312385559082, 0.4253905713558197, -1.6563186645507812, 1.1466350555419922, 0.5613564252853394, 1.4023994207382202, -0.1805083453655243, -0.40229344367980957, -1.3729640245437622, 0.5399768948554993, -0.9144325852394104, -0.458981990814209, 0.4337750971317291, -0.14841489493846893, 0.28211501240730286, -0.035060055553913116, 0.6042262315750122, 2.0192947387695312, 0.3971787393093109, -0.3117237091064453, -0.3339828550815582, 1.2086279392242432, -0.9904291033744812, 0.11595220863819122, 0.7985741496086121, 2.2306153774261475, 0.8777193427085876, -0.5495724678039551, -2.1089422702789307, 0.26758137345314026, 0.35019269585609436, 0.29280826449394226, -0.9408483505249023, -0.6165095567703247, -0.7091529369354248, -0.5675852298736572, 0.1337575614452362, 0.5627312660217285, 0.7648611664772034, 0.28391310572624207, -1.2549933195114136, -0.2514244019985199, -2.0957202911376953, 1.062406063079834, 0.7861113548278809, 0.06596764177083969, -0.12320754677057266, -0.40971994400024414, 0.4866413176059723, 1.628282904624939, -1.296282410621643, -0.9448865056037903, -0.8333598971366882, -0.4208690822124481, 1.8647792339324951, -0.4723767936229706, 0.07534965127706528, 0.23458079993724823, -1.524019479751587, -0.7943365573883057, 0.26910296082496643, -0.6811317205429077, 0.3139890730381012, -0.8371481895446777, -0.7996455430984497, 0.034208815544843674, -0.8204302191734314, -0.674830436706543, 0.3228561282157898, 0.7898874878883362, -0.8430140614509583, -1.2757970094680786, 0.9960178136825562, -0.11327824741601944, 2.247861623764038, 0.8239215612411499, -1.0622855424880981, -1.6904155015945435, 1.852661371231079, 1.4419523477554321, 1.6844671964645386, 1.3197883367538452, -1.1420156955718994, 0.9604189395904541, 0.8909798860549927, 0.056299831718206406, 1.518317461013794, -0.24904191493988037, -0.9990724921226501, 1.1167060136795044, -0.42184433341026306, -0.36857685446739197, 1.1594492197036743, -0.8995387554168701, -0.8733786344528198, -1.3899508714675903, -0.0045152991078794, -0.16028429567813873, 0.20030832290649414], [0.07867689430713654, 0.5058395862579346, -0.11677569150924683, 0.030873538926243782, -1.0941240787506104, -1.482193946838379, -0.4611198604106903, 0.4774606227874756, 1.7623324394226074, -0.8183019161224365, 0.052918124943971634, 1.402642011642456, -1.3133794069290161, 0.20214959979057312, 0.7961783409118652, 0.2504044771194458, 0.040197741240262985, -0.6431307196617126, -0.4332330524921417, -1.6708430051803589, -0.6733363270759583, 1.0427696704864502, -2.1203951835632324, 0.8820272088050842, -0.9966521263122559, 0.6344020962715149, 0.5653632879257202, -0.8446568846702576, -0.023073485121130943, -2.0722334384918213, 0.9458181262016296, -0.38388386368751526, 1.090846061706543, -1.5542409420013428, -0.10840310156345367, -0.92122483253479, -1.0004311800003052, 0.6675997972488403, -1.9803568124771118, -1.411154866218567, -0.21208690106868744, -0.6961956024169922, 0.7443119287490845, -0.5276721119880676, -0.4926159977912903, -0.06370139867067337, -1.213789701461792, -1.160650610923767, -1.6222634315490723, -0.21650511026382446, 1.056878685951233, -0.25164371728897095, -1.0260249376296997, -1.4424973726272583, 0.8990500569343567, 0.13815942406654358, 1.2160121202468872, -2.378910541534424, 1.7977148294448853, 1.0908191204071045, -0.643062174320221, -0.38517144322395325, 1.0333495140075684, -2.248349905014038, -0.8906250596046448, -0.13159997761249542, -2.086012840270996, 0.6767705678939819, -1.8343453407287598, -0.6163234710693359, 0.960449755191803, -1.249002456665039, -0.6306107640266418, -1.4890793561935425, -1.1018109321594238, 0.3922038674354553, 0.7842402458190918, -1.9744930267333984, 0.035589009523391724, -0.5638969540596008, -0.5451555252075195, 1.49927818775177, -0.1530904918909073, 1.1287776231765747, -0.5264933109283447, -0.8180553317070007, -2.0526678562164307, -1.1680128574371338, -0.8630279898643494, -0.12987375259399414, -1.4591704607009888, -1.2228398323059082, -0.23611271381378174, 1.109365463256836, 1.1598104238510132, 1.016142725944519, 0.36040905117988586, 0.02028011530637741, -0.8611045479774475, -1.0594322681427002, 0.8039847016334534, 0.5723203420639038, -1.8027606010437012, -0.8422831296920776, 0.7242440581321716, -0.4779343008995056, -0.9855355024337769, 1.052317500114441, -1.8284292221069336, 1.2953648567199707, 0.6117205023765564, -0.24094828963279724, -1.0493100881576538, 0.5925987362861633, -0.37764865159988403, -0.0858864039182663, 0.03153437748551369, -0.3413406014442444, 0.23941707611083984, 0.3950031101703644, 0.9762324094772339, 1.1448075771331787, 0.8126009106636047, 0.38450607657432556, -0.6069034337997437, -0.2725520133972168, -1.392012357711792, 0.3410617709159851, -2.0854992866516113], [0.3462686538696289, -0.09349536895751953, -1.604493498802185, 0.7308154106140137, -1.309761881828308, 0.9997386336326599, 0.05114217847585678, -0.0178715530782938, 1.0768119096755981, -0.46628710627555847, 0.09597798436880112, 1.2187026739120483, -1.2717654705047607, 1.3403196334838867, 0.049510657787323, -2.046299457550049, 0.10942751914262772, -1.4683177471160889, 0.0034338899422436953, -1.1995540857315063, 1.181948184967041, -0.5829995274543762, 0.07685048878192902, -0.392273873090744, 0.3289562165737152, -0.907813310623169, 0.19893668591976166, 0.07044631242752075, -0.3695966601371765, -1.7790443897247314, 1.2246931791305542, -0.18572604656219482, -0.10681978613138199, 0.8125540018081665, 0.9846383929252625, 1.6494332551956177, 0.5901424884796143, 0.9807498455047607, -1.1801514625549316, -0.4998543858528137, -1.1578537225723267, -2.4962682723999023, 0.6395127177238464, 0.13863223791122437, 0.45723918080329895, -0.4653603732585907, -0.8228355646133423, 0.06293721497058868, -0.36335277557373047, -0.9153470396995544, -0.7579747438430786, -0.6996548771858215, -1.7741813659667969, 0.3399786353111267, -1.159885048866272, 0.030394934117794037, 0.8134982585906982, -0.5889817476272583, 0.9838846921920776, 0.2274467498064041, -0.052049919962882996, 0.05689716339111328, 1.5393052101135254, -0.13676518201828003, 0.8522435426712036, 0.8369839191436768, -1.3888988494873047, -3.0318634510040283, 0.7284059524536133, -0.9221106171607971, 1.4009236097335815, 1.0278087854385376, 0.20007632672786713, -0.2710613012313843, 0.4045950174331665, 0.41648560762405396, -0.08434263616800308, 1.420556902885437, 1.07199227809906, 2.5779001712799072, -1.1751863956451416, 0.6049803495407104, 0.2271665781736374, 0.35305988788604736, -0.5052158832550049, -0.7885689735412598, -0.04888836294412613, -0.026278305798768997, 0.6952626705169678, -0.16752439737319946, -1.2381088733673096, -0.14339375495910645, -0.6002459526062012, 0.3350430727005005, 1.8325189352035522, -0.6504724621772766, -0.9765211343765259, -0.31271815299987793, -0.553459644317627, -1.526862621307373, -1.8556790351867676, -0.9960902333259583, -1.133252501487732, -1.132677674293518, 0.9207771420478821, -0.1037764847278595, -0.47385722398757935, 1.060655951499939, -1.7692151069641113, 1.0010358095169067, 0.1412705034017563, -1.085960865020752, 0.8816860914230347, -0.052990056574344635, -0.10443151742219925, -0.06720215082168579, -0.40621864795684814, -0.9979579448699951, 0.9051800966262817, 1.3872418403625488, -0.13152268528938293, 0.28403887152671814, -0.12303207069635391, 0.2993064820766449, 0.13352996110916138, -0.9850499033927917, -0.3603474795818329, 0.4405635595321655, -0.07197286933660507], [0.6880592107772827, -0.9911962151527405, 0.24227577447891235, 0.6398687958717346, -1.4581141471862793, 1.1304359436035156, -1.413409948348999, 0.13047905266284943, -0.561926007270813, -0.9925079941749573, 1.4537044763565063, -0.5921092629432678, -0.7527250647544861, -1.2305642366409302, -1.7960654497146606, -0.7624815106391907, -0.09434536099433899, 1.0305113792419434, -0.6747967004776001, -0.8822791576385498, -1.219069004058838, 0.6031723022460938, 0.14329609274864197, 0.9059332013130188, 0.47183170914649963, 1.0649391412734985, -0.7450790405273438, -0.39688873291015625, -0.12699675559997559, -0.31465983390808105, 0.8185437321662903, -1.9987798929214478, -1.2085821628570557, -1.295342206954956, -0.2870512306690216, -0.31059929728507996, -0.16907899081707, 0.9649826288223267, 0.6779161691665649, -0.8894292712211609, 0.010131802409887314, -0.19679556787014008, 1.0014756917953491, -2.3657288551330566, -0.7537832260131836, 1.296048641204834, 1.4264748096466064, -2.6777684688568115, -0.12153059989213943, 0.19846178591251373, -0.6579949855804443, -0.9517157077789307, 0.2927466332912445, 0.1914384961128235, 0.15522430837154388, -0.08610783517360687, 0.5027593970298767, 1.0755982398986816, -1.157652735710144, -0.4836857318878174, 0.8680174946784973, -0.3097893297672272, -0.3699217736721039, 0.908934473991394, -0.9852018356323242, -0.5054103136062622, 0.6147979497909546, -1.069006085395813, -0.4128536581993103, 0.8744426965713501, 1.1351914405822754, 0.10826077312231064, 0.9250596761703491, -1.140269160270691, -1.2024670839309692, 0.047249048948287964, -1.9926445484161377, -0.5483034253120422, -0.3282487690448761, 0.30877619981765747, -1.0971381664276123, 1.0461852550506592, 1.1311348676681519, 0.16251125931739807, 0.5667777061462402, -0.45287784934043884, 1.2270008325576782, 0.6594220995903015, 0.30230712890625, -1.2852636575698853, -0.015077724121510983, 1.6407853364944458, -0.5591340661048889, 0.789023756980896, -0.4768318831920624, -1.064305305480957, 0.09261420369148254, -0.14657363295555115, 1.026919960975647, -1.3351997137069702, -1.9330782890319824, -1.9855060577392578, -0.4556884169578552, -0.7209326028823853, -3.1340525150299072, 1.3203818798065186, -0.6981507539749146, 0.1407674103975296, -1.77573823928833, -0.15196233987808228, 0.17108020186424255, -0.8095072507858276, -1.8097590208053589, 0.6951709985733032, -0.6874273419380188, 1.4365607500076294, 0.43035849928855896, -1.1900312900543213, -0.7982388138771057, 0.9827038645744324, -0.34599241614341736, 1.7856485843658447, 0.11952092498540878, 1.6495180130004883, 1.1726865768432617, -0.37204331159591675, -0.14274552464485168, 0.4867324233055115, -0.25603872537612915], [-0.1261155903339386, 1.253656268119812, -0.5666949152946472, 0.7170116901397705, 0.6229939460754395, 0.19804488122463226, 0.4547368884086609, 0.3472883105278015, -1.302054524421692, 1.0321720838546753, -0.2435566484928131, -0.10590992122888565, 0.1311156004667282, -1.0926729440689087, -0.5451233983039856, -0.09734439849853516, -0.7461792230606079, 0.5118647813796997, -0.23539060354232788, -0.4063081443309784, 1.3325791358947754, 0.05497737601399422, -0.27331113815307617, 0.19128364324569702, 1.2952203750610352, 0.36931735277175903, 2.012481451034546, -0.8570894002914429, -0.926256537437439, 1.1619889736175537, -0.338908314704895, -1.3421266078948975, 1.9228276014328003, 0.5643651485443115, 0.5684137940406799, 0.08998166769742966, 1.2845066785812378, -0.44819653034210205, -0.1756068915128708, 0.5506314039230347, 0.2839035093784332, 0.9561766386032104, -0.561431348323822, -1.2495590448379517, 0.029858743771910667, 0.1272324025630951, 0.6627382636070251, 0.12008480727672577, -0.21523594856262207, -0.36912813782691956, 1.681191086769104, 0.35305777192115784, -0.18969960510730743, 0.33956149220466614, 0.5901374220848083, -1.0274492502212524, 0.1919046938419342, 0.3253723680973053, -0.607155442237854, 0.6160076260566711, -2.422612190246582, 0.18623362481594086, -0.7604798674583435, 0.6063386797904968, -0.22875180840492249, -0.9501533508300781, 0.7809018492698669, 0.9082682132720947, 0.4247152805328369, -1.6828912496566772, 0.5728631019592285, -0.24030712246894836, -0.6879358291625977, -0.15750591456890106, -0.014870709739625454, 1.1631653308868408, -1.4721819162368774, 1.2206023931503296, -0.6766117215156555, 0.053020279854536057, -0.992470920085907, 1.2237684726715088, 1.844868540763855, -0.4365771412849426, 0.701644241809845, -0.030363894999027252, 1.3952724933624268, 2.6536855697631836, 2.260601758956909, 0.09294117987155914, -0.06371380388736725, 1.3424465656280518, -1.78153657913208, 0.8670822978019714, 0.6219770312309265, 0.05060600861907005, -0.31668102741241455, 0.6302215456962585, -0.7410778999328613, -0.13968585431575775, -0.08214545249938965, 0.7421695590019226, -0.1967954784631729, -0.9609090089797974, -1.4032979011535645, 1.8122953176498413, 0.3505951762199402, 1.3247839212417603, -2.4192278385162354, 1.1455625295639038, -0.3546926975250244, 0.026157846674323082, 0.6373379230499268, 0.09044644981622696, 1.288451075553894, -1.19041109085083, -0.4836047887802124, 0.23660466074943542, -0.16254015266895294, 0.5950424671173096, -0.36178526282310486, 0.9873510599136353, 1.5602277517318726, 0.7379999756813049, -3.265674114227295, 2.6744019985198975, -0.7182629704475403, 0.4725666046142578, -1.2070343494415283], [-0.05681832879781723, -0.3423972725868225, 1.1587635278701782, 0.2457006424665451, -0.10637211054563522, -0.6868345737457275, 2.5225746631622314, 1.2872076034545898, -1.6838349103927612, -1.6111481189727783, -0.25277549028396606, 0.2592431604862213, -1.5577632188796997, -1.0171645879745483, -1.5018810033798218, -0.8613204956054688, -1.305169701576233, -1.526826024055481, 1.1497697830200195, 2.1995184421539307, 0.7848113775253296, -0.9458795785903931, -0.44332098960876465, -0.4347562789916992, 0.38702860474586487, -0.3661012351512909, -0.24265164136886597, -0.5512830018997192, -1.5130751132965088, 0.3768521249294281, -0.09412214905023575, 0.6953856348991394, -0.30685263872146606, -0.29783201217651367, 0.2642780840396881, 1.4002498388290405, -1.0641354322433472, -0.6322909593582153, 0.25620704889297485, 1.0136035680770874, -0.4144020974636078, 0.6803494095802307, -1.4960250854492188, -1.0739178657531738, -0.6997947692871094, -1.5786280632019043, 0.7339134216308594, -0.3717547357082367, 1.8138855695724487, 0.14081625640392303, -1.4393104314804077, -0.48991039395332336, 0.9314160346984863, 0.7752512097358704, 1.1609867811203003, 0.4367712438106537, 1.209625005722046, 1.3699203729629517, 0.4593721926212311, -0.036085765808820724, 1.0738887786865234, 0.36471953988075256, -0.4537210166454315, -0.8893922567367554, -0.027551783248782158, 0.32322949171066284, 0.5031160116195679, 0.6031662821769714, 2.8321869373321533, 0.5677888989448547, 0.641817569732666, -0.15829890966415405, -0.038216814398765564, -1.2471027374267578, -2.611644983291626, 0.16395705938339233, -0.03573447838425636, -0.2443554699420929, -1.101773977279663, 1.315946340560913, 0.8295888900756836, -1.0621650218963623, 0.10319231450557709, 0.15558910369873047, 1.6423557996749878, 2.42081356048584, 0.2847920358181, 0.8327994346618652, 2.225118398666382, -1.3444331884384155, -0.648184597492218, -1.0909068584442139, -0.8920162916183472, 0.9246771931648254, -0.5077990889549255, -2.0235843658447266, 0.45299869775772095, 0.6357401013374329, -1.181634545326233, -0.8816561698913574, 0.36915650963783264, 2.1898975372314453, -0.7526093125343323, 2.1418769359588623, 1.0425090789794922, -0.4662487506866455, -1.9270336627960205, -0.9971275329589844, 0.578655481338501, 0.7308883666992188, 1.7668389081954956, -0.25178954005241394, -1.0426924228668213, -0.37626537680625916, -1.3920738697052002, 1.1760435104370117, 0.9505392909049988, -0.5647463798522949, 0.8738266229629517, -0.020163612440228462, -1.6998248100280762, -1.8118444681167603, 0.5686959028244019, -0.3364121615886688, -0.04491111636161804, -0.34704098105430603, 0.06378306448459625, -0.7226372957229614, -1.3598408699035645], [-0.3415336310863495, 0.3996717631816864, -2.3812503814697266, 2.385329246520996, -1.1525629758834839, 1.210792064666748, 0.010525822639465332, -0.05883214622735977, -1.7370233535766602, 0.010906091891229153, 0.5039017200469971, 0.6530631184577942, -0.546023964881897, 0.0394187793135643, -1.6726069450378418, -0.5055590867996216, 0.7757214307785034, -0.06996013969182968, 1.707336664199829, -0.09639065712690353, -1.6925746202468872, 0.35699954628944397, 0.5680702328681946, -0.727038562297821, -0.3607376217842102, 0.7642518281936646, 0.005860263481736183, -0.07901062071323395, 0.5948718190193176, 1.1333123445510864, 1.335236668586731, 0.3500358760356903, 0.7898867130279541, 0.8251091241836548, 1.2514485120773315, -0.19083310663700104, 0.8147903084754944, -0.052292048931121826, -1.5040276050567627, -2.6331279277801514, 0.1569770872592926, 0.9179224967956543, 0.19673660397529602, -0.10694120824337006, 0.2560364305973053, 0.20391146838665009, -0.7739001512527466, -0.7773400545120239, 0.8108137845993042, 0.00411463575437665, 0.6273886561393738, 1.5912235975265503, 0.9732009768486023, -0.8169254064559937, -0.7834754586219788, 1.316101312637329, -0.3140515089035034, 1.1721973419189453, -0.3271210789680481, 0.10740520805120468, -0.28290796279907227, -1.518149971961975, 0.5160788893699646, 0.5754370093345642, -0.33561792969703674, 2.0901894569396973, -0.477545827627182, 0.2383417934179306, -2.087728500366211, -1.1588366031646729, 1.9873944520950317, 0.4929007887840271, 0.6987430453300476, -1.0629057884216309, 1.0454035997390747, -1.7052438259124756, 0.18412569165229797, 1.2202157974243164, 0.7176159620285034, -0.19359928369522095, 0.8843567371368408, -0.6785292029380798, 0.26384732127189636, -0.7704299688339233, -1.0424258708953857, 0.5732988715171814, -0.6523110866546631, 0.0041314177215099335, 0.6957678198814392, -0.8847057223320007, -1.4176024198532104, 0.06084255501627922, -0.20168031752109528, -1.0507582426071167, 0.37063995003700256, 0.716923177242279, 1.3324297666549683, -0.6851199269294739, 1.681968092918396, 0.07490164041519165, -0.42466601729393005, -1.5905804634094238, -0.13373760879039764, -1.117091178894043, 0.46556103229522705, 1.679114580154419, 0.9033367037773132, 0.6230536699295044, 0.32448410987854004, 0.16401353478431702, -0.3281652629375458, -0.5277572870254517, -1.0502002239227295, -0.40889105200767517, 1.0666310787200928, -0.18885108828544617, -0.361367404460907, -0.7172771096229553, -3.010335683822632, 1.2224708795547485, -0.37217170000076294, -1.3822786808013916, 0.9507633447647095, 0.7132216691970825, -0.16078805923461914, -0.11347181349992752, 0.17168079316616058, -1.3606483936309814, -0.35912442207336426], [0.35945385694503784, -0.7675147652626038, 0.986288845539093, -1.8980436325073242, -1.5555942058563232, -0.4740752577781677, -0.16657286882400513, 0.6473584175109863, 0.6134545207023621, -0.008000911213457584, 1.6693155765533447, 0.45986244082450867, -1.1147700548171997, -1.2245405912399292, -1.0213488340377808, -0.29545220732688904, -0.0387897752225399, 0.6844145059585571, -0.6233982443809509, -0.7276299595832825, -0.38514167070388794, -1.3639708757400513, -1.447766661643982, 0.2832759618759155, -0.22603219747543335, 0.983022153377533, -0.5207127332687378, 1.4457587003707886, -0.3930051624774933, -1.3682581186294556, 0.41148310899734497, 0.6517717838287354, 0.17431163787841797, -0.4139661192893982, 0.4045940935611725, 0.860908567905426, 0.3434280455112457, -0.040956780314445496, 1.5175321102142334, 0.1480826437473297, -0.019422585144639015, -0.23869013786315918, 0.8623484969139099, 1.201123833656311, -0.8244260549545288, -1.0279521942138672, 1.857677698135376, 0.7712975144386292, -0.6403972506523132, 1.8667948246002197, 0.6164237856864929, 0.9349634051322937, 0.40557122230529785, -0.6419365406036377, -0.17907370626926422, 0.6450726389884949, -0.6999261379241943, -0.4041171669960022, 0.14084181189537048, -1.093643307685852, 1.0261012315750122, 2.1540110111236572, 1.0251569747924805, -0.08918417990207672, -0.847295343875885, -0.5156349539756775, -0.341856449842453, -1.0790634155273438, 0.36760351061820984, -0.5130639672279358, -0.7220491170883179, 0.3898564577102661, 0.08882149308919907, 1.9335744380950928, 0.30519819259643555, 0.32844191789627075, 0.24391843378543854, 0.7198504209518433, 0.8057384490966797, 2.1873135566711426, -0.3005170524120331, 1.4840000867843628, 1.9073091745376587, 0.5062543153762817, 0.8040837049484253, -0.06720175594091415, 2.1028568744659424, 0.11772361397743225, -0.9868499636650085, -0.3856605589389801, 2.894915819168091, -1.4929536581039429, -0.24873419106006622, -2.4423584938049316, 0.3693503439426422, 0.30002760887145996, -0.08394256234169006, 1.381041407585144, -2.725403070449829, -1.3327327966690063, -1.9573757648468018, 0.4349871277809143, -0.8730495572090149, -2.516756772994995, 1.013452172279358, 0.23402182757854462, 0.7047340273857117, -0.8667150735855103, -0.5525307059288025, -1.712158203125, 0.4503132700920105, -0.6598125100135803, 0.759384036064148, -0.5443056225776672, 0.37854254245758057, 0.43275541067123413, -0.46206215023994446, -1.0105485916137695, 0.3868035078048706, 0.6394315958023071, -0.4613582491874695, 0.4401727616786957, 1.0898196697235107, 0.8768696188926697, 0.7458456754684448, -1.2181141376495361, -0.06682503968477249, -0.8988417387008667, 1.1647913455963135], [-0.047168612480163574, -0.671019971370697, 0.07721107453107834, -0.19563670456409454, -0.1665264368057251, 0.3401983678340912, -0.363943874835968, -0.7460418343544006, 0.301668256521225, -0.012338901869952679, 1.3763517141342163, 0.547581136226654, 1.4789761304855347, -0.500272810459137, 2.015204906463623, 0.4758765399456024, -0.1744089126586914, -0.31448253989219666, 1.0632429122924805, -0.7702611088752747, 2.2197725772857666, -0.8252674341201782, -0.30396097898483276, -1.1186691522598267, 1.2002956867218018, -0.9056580662727356, -0.3391612768173218, 0.08813242614269257, 0.337765097618103, -2.237938404083252, -0.6122382879257202, -0.11774015426635742, 0.7463706731796265, -0.984855592250824, 0.6219321489334106, 2.2828052043914795, 0.19481538236141205, -1.7175827026367188, 2.365631580352783, -0.14542727172374725, 1.6872742176055908, -0.4653536081314087, -1.6812584400177002, 2.238323450088501, -1.388543725013733, -0.5752232074737549, 0.41321173310279846, 0.6363399028778076, -0.7726088166236877, 1.1434448957443237, -0.1506030112504959, -0.24455797672271729, 0.9210849404335022, 0.3899029791355133, 1.701007604598999, 1.2130687236785889, -1.3550255298614502, 0.4260713458061218, -0.478298544883728, -1.5071203708648682, 1.1804085969924927, 0.20368820428848267, 0.5995620489120483, -0.6642278432846069, 0.32505354285240173, 0.5193410515785217, -0.6271075010299683, 0.5169033408164978, 2.3904316425323486, 1.4607809782028198, 0.6934252977371216, 0.3003399074077606, -0.853104829788208, -0.2095380276441574, -0.4915727376937866, 0.061190683394670486, 1.6879953145980835, -1.7091326713562012, -0.38067546486854553, -0.8042056560516357, 0.7134068012237549, -1.1163127422332764, -0.1600225865840912, -0.3084540069103241, 0.8173414468765259, 0.9837371706962585, -0.47816771268844604, -0.34272274374961853, 0.15092673897743225, -0.7173574566841125, -0.49639153480529785, 0.06207374110817909, -0.8092231154441833, -1.6505459547042847, 0.7085334062576294, 0.9695383906364441, -1.0354795455932617, 0.7735798954963684, -0.8990221619606018, -1.0718393325805664, 0.27373820543289185, -0.9551753997802734, -0.15263144671916962, 0.009254717268049717, -0.3724624514579773, -0.8850146532058716, 0.7132806777954102, -0.16638518869876862, 0.6882557272911072, 0.20381304621696472, 1.5781686305999756, -0.8617464900016785, 0.9184077382087708, 0.03506889566779137, -1.0524461269378662, -1.2922943830490112, 0.5196435451507568, 1.4305462837219238, 0.8756021857261658, -1.6073336601257324, 1.8522683382034302, 1.0808099508285522, 2.440720319747925, -0.5240747928619385, -1.3249647617340088, -1.9042582511901855, -0.6801513433456421, -0.2861984372138977, -0.7695687413215637], [0.38569554686546326, -0.3497055768966675, -1.5415682792663574, 1.299370288848877, -0.14692698419094086, -0.581921398639679, -0.9840676784515381, 0.05523950606584549, 0.1526680737733841, 1.4985911846160889, -0.5818300843238831, -0.31977787613868713, -1.3211524486541748, -1.42903733253479, 1.5237350463867188, -0.09402148425579071, 0.8587899208068848, 0.9616276621818542, -0.39322900772094727, 1.2294702529907227, 1.1834726333618164, -0.6165047287940979, 0.10875820368528366, -0.10685290396213531, -0.31852298974990845, 0.21659515798091888, 0.618473470211029, -1.0617704391479492, 0.06948472559452057, 1.4108322858810425, -0.24099528789520264, -0.44132837653160095, 1.0405205488204956, -0.15776550769805908, -1.1422282457351685, 0.6052824258804321, -0.8392153978347778, 0.2771994173526764, -0.15294447541236877, -1.8811506032943726, 0.029426908120512962, -2.4032320976257324, -0.26304447650909424, -1.0502159595489502, -1.0889204740524292, -0.7971228957176208, -0.1578022986650467, -0.9266932010650635, -1.8196995258331299, -0.5746661424636841, 0.3669920861721039, -0.8307707905769348, -0.09959951043128967, -1.2217391729354858, 0.5750147104263306, -1.1837162971496582, 0.9121083617210388, 0.43844765424728394, -0.1711711585521698, -0.7308378219604492, 2.0975730419158936, 0.5844665169715881, -1.1348682641983032, -0.11051011830568314, 1.5749248266220093, 1.024771809577942, -0.42492491006851196, 1.186018943786621, -0.543132483959198, -0.9548560976982117, 0.2943098545074463, -0.7934250235557556, 0.3785676062107086, 0.3456421494483948, 1.3418480157852173, -1.2923836708068848, 1.395842432975769, -1.2254372835159302, 0.900432288646698, 0.10412539541721344, -0.5478215217590332, 0.02636924386024475, 0.2953348457813263, 0.19060681760311127, 1.4328135251998901, 0.8559160232543945, 0.16353607177734375, -0.7674426436424255, -1.2686183452606201, -0.11487489193677902, -0.22522643208503723, 2.1840341091156006, 0.5291417837142944, 1.1200894117355347, -0.312461256980896, 0.32297343015670776, -0.3453052043914795, 1.2682892084121704, -0.6643529534339905, 0.239668071269989, -0.13840103149414062, -0.5658516883850098, -0.297387957572937, 1.955080270767212, 1.2201865911483765, -0.45503130555152893, 0.7308099269866943, 0.386121928691864, -0.030703769996762276, 0.30421414971351624, -0.8969243168830872, 0.2590767443180084, -0.818478524684906, 0.9363296627998352, 1.2605668306350708, -0.1909782588481903, 1.7942936420440674, -0.5926192402839661, -0.2796487510204315, 1.2015328407287598, -0.11192119866609573, 1.241679310798645, -0.23826922476291656, 0.027755066752433777, 1.2959167957305908, 0.43051740527153015, 0.6930627822875977, 1.070443868637085, 0.7046130895614624], [1.4720567464828491, -0.25523021817207336, 0.3836697041988373, 0.5473630428314209, 0.2511914372444153, 0.905508279800415, -0.13668979704380035, 0.7505595684051514, 0.22070692479610443, -1.876452922821045, 0.7964043617248535, -0.4531399607658386, 0.3676770031452179, 0.293109655380249, -1.060647964477539, 0.3217432498931885, -0.9147302508354187, -0.49282306432724, 0.33864158391952515, 1.0798165798187256, 0.7045876979827881, -0.33565109968185425, 1.7291687726974487, 1.0807747840881348, 0.44681477546691895, 2.1590163707733154, 0.719167172908783, -0.6615515947341919, 0.4643237292766571, 1.648328185081482, 0.24675138294696808, -0.4968855082988739, 0.18889638781547546, -0.2437555342912674, -2.4979941844940186, -0.9670869708061218, 0.03426707163453102, 1.2906116247177124, -1.4254231452941895, 0.8344613313674927, 0.43709880113601685, 1.0593830347061157, -1.171478271484375, -0.6551117897033691, -0.6765902638435364, 1.0424679517745972, 1.683837890625, 1.006994605064392, 0.017281079664826393, 1.195085883140564, 0.11482927203178406, 0.05096912756562233, 0.08610402047634125, 1.5606001615524292, 0.15521010756492615, -0.7859537601470947, 0.2385808676481247, 0.1376689076423645, 1.4736900329589844, -0.22461043298244476, -1.8537492752075195, -0.27600669860839844, -1.4244284629821777, 0.4183364510536194, 0.28584930300712585, -0.5583285093307495, 0.4471433162689209, -0.0648360624909401, 1.512814998626709, -1.5479209423065186, 1.0475672483444214, 1.614978313446045, 0.6360586881637573, -0.8712755441665649, -0.8522176742553711, -0.4651266038417816, 0.8216946125030518, -0.2716313302516937, -0.9364843964576721, 1.4379398822784424, 0.48336997628211975, 0.22690580785274506, 1.0941846370697021, -1.0256537199020386, 1.115089774131775, -1.112852692604065, 0.9202376008033752, -1.9021790027618408, 0.8454013466835022, 1.4638625383377075, -0.975420355796814, -0.24235191941261292, 1.1965254545211792, 1.605686068534851, 0.3417482078075409, -0.8349976539611816, 1.1692309379577637, -2.1245405673980713, -0.27441611886024475, -0.906578004360199, 0.24797669053077698, 0.3817175030708313, 1.3515559434890747, 1.8533282279968262, 0.4789526164531708, -0.21840597689151764, 0.12065114080905914, -0.07625328004360199, -0.2655220329761505, -1.9843411445617676, 0.2877791225910187, -1.0321508646011353, -1.7415413856506348, -0.1443471759557724, 0.5234986543655396, 0.16381824016571045, -1.0981876850128174, 0.1738332360982895, 2.0786678791046143, -0.4064548909664154, -0.1285984218120575, -1.0334973335266113, -0.5276384949684143, -1.1771044731140137, 0.8837285041809082, -1.8180707693099976, 0.25675201416015625, 0.9980553984642029, -0.49762392044067383], [-1.2068355083465576, -0.33281031250953674, 0.20931203663349152, -0.8685038089752197, 0.3274809420108795, 0.18346448242664337, -0.7319318652153015, -1.0991299152374268, 1.2547094821929932, -0.3747222423553467, -0.40287086367607117, 1.0105966329574585, -0.02774990163743496, 0.786526083946228, -0.5677028298377991, 0.9613106846809387, 1.387647032737732, 0.028018387034535408, -1.5429658889770508, 0.6160544157028198, 0.07993905246257782, 1.6376090049743652, -0.7329450249671936, -0.006486891303211451, 0.8833221793174744, 2.0678675174713135, -2.401876211166382, 0.0688401460647583, -0.8692320585250854, -1.6178231239318848, 0.19548963010311127, 0.5649741888046265, -0.4988466203212738, 1.7431200742721558, 0.6726705431938171, 1.1187607049942017, -0.9013481736183167, -0.40617263317108154, 0.8907813429832458, -2.01261305809021, -0.5628179907798767, 1.6821881532669067, -0.8961449861526489, -0.8067769408226013, -0.5478251576423645, 0.33409571647644043, 1.2506104707717896, -2.2354886531829834, -0.9976162910461426, -0.38553586602211, -1.1825231313705444, -0.8052792549133301, -1.6095030307769775, 0.6788269877433777, 1.0850343704223633, 0.05112927779555321, -0.0020304699428379536, 0.15113261342048645, -1.0450429916381836, -0.4376390278339386, -0.4309373199939728, 0.3930183947086334, 0.3575465977191925, 0.7092387676239014, -0.1029990166425705, -0.2181803286075592, -0.06639490276575089, -0.6322386860847473, 1.05487859249115, 0.14340537786483765, -0.055569641292095184, 0.9466434121131897, 0.6410602331161499, -0.6028702855110168, 0.3956044912338257, -0.5359815359115601, -0.6333311200141907, 0.4876759350299835, -0.42431363463401794, -0.4223623275756836, -0.14601539075374603, -2.3987152576446533, 1.1516640186309814, 0.14000669121742249, -0.6020652651786804, 0.4070294499397278, 0.9257800579071045, 0.20664340257644653, 1.3346699476242065, -0.9803763031959534, 0.2093745768070221, 0.8640952110290527, 0.4381401240825653, 1.1248703002929688, -1.786746621131897, 0.84748375415802, 0.9338785409927368, -1.1989110708236694, -0.6998310685157776, -0.4953417181968689, 0.7315497994422913, 0.4375937581062317, 0.7277605533599854, 1.4830505847930908, 0.3386313319206238, -0.5613934397697449, 1.5517771244049072, 1.157252550125122, -0.9979432225227356, 0.02806808240711689, -1.222666621208191, -1.400028109550476, -0.6550474762916565, 0.4477831721305847, 2.100377082824707, 0.898445188999176, 0.8903182148933411, -1.383151650428772, -0.3812015950679779, -2.489180564880371, 0.6357906460762024, 1.0017759799957275, 0.3108973801136017, 0.9106374979019165, -0.8151199817657471, -1.034956932067871, 2.0035696029663086, -0.42050719261169434, -0.2021612972021103], [1.3695011138916016, -0.2428416758775711, -0.2991795837879181, -0.27649593353271484, -0.30603456497192383, 1.415271282196045, -2.03289532661438, -0.4520584046840668, 0.6418601870536804, 0.7622354626655579, 0.2936099171638489, 1.4655842781066895, -0.8619368076324463, -0.7961555123329163, 1.0566109418869019, 0.7739812135696411, 0.42878982424736023, 0.578359067440033, 1.3927195072174072, 0.1239418163895607, -1.189863681793213, 1.7028884887695312, 0.8892116546630859, -0.7544369697570801, 0.12498743832111359, -1.5192164182662964, -0.005338321439921856, 0.5794225931167603, -0.030284158885478973, 0.9732858538627625, -0.7156410217285156, -1.5213677883148193, -0.6795929670333862, -0.9613659977912903, 0.1852293163537979, -0.6119934916496277, 0.43594425916671753, 0.44835394620895386, 1.9038478136062622, -1.1056829690933228, 0.010061461478471756, -0.21935966610908508, -0.9968986511230469, 0.18858186900615692, -1.7140648365020752, -1.2030155658721924, 2.2152020931243896, -0.3855505585670471, 1.7589852809906006, 0.7026571035385132, 1.8667019605636597, -0.49214968085289, -0.8966493010520935, -0.48468485474586487, 1.3435591459274292, 2.918874979019165, 0.9987600445747375, -0.16505657136440277, -0.17058250308036804, -0.20913414657115936, -0.1797439008951187, 0.43701282143592834, -0.49610283970832825, -1.6246154308319092, 0.26678937673568726, -0.44611117243766785, 0.7013230323791504, 1.0306299924850464, -0.5517757534980774, 0.5678061842918396, 0.14144809544086456, 0.6151780486106873, -0.022520704194903374, 0.38970693945884705, -0.972808837890625, -0.042305342853069305, 2.065600872039795, 0.13327565789222717, -0.5927645564079285, 0.0010241515701636672, 1.2737988233566284, 1.1786444187164307, -0.5117472410202026, 0.6529565453529358, 1.0916110277175903, 1.5130361318588257, -0.7727251052856445, -0.869236946105957, 0.08716961741447449, 0.4442225694656372, -0.566725492477417, 0.13229618966579437, -0.4095451235771179, -0.2580905854701996, 1.6182397603988647, -1.4106446504592896, 1.2319945096969604, -0.26379045844078064, -0.803884744644165, 0.17482255399227142, -0.44244834780693054, -0.6029338836669922, -0.3500955104827881, 0.7546120285987854, 1.0909453630447388, -0.04182949289679527, -1.3847492933273315, -1.3771998882293701, 1.524195909500122, 0.04378665238618851, 0.9615171551704407, -0.16227127611637115, -1.0932095050811768, -0.5947710871696472, 0.1201910674571991, 1.431066870689392, 1.0384668111801147, 0.7678932547569275, -0.8622139692306519, 0.6108422875404358, -0.8136947154998779, -1.3952476978302002, -0.8134451508522034, 0.4483889639377594, 0.8239908218383789, 0.6615992188453674, -0.3352069854736328, 0.1116737350821495, -0.9493551850318909], [0.49662530422210693, 0.20918028056621552, 0.7968270778656006, 0.6822392344474792, 0.3182412087917328, -0.5993071794509888, 0.3160133957862854, 0.5004475116729736, -0.7900726199150085, -0.3525139093399048, 1.8696292638778687, 1.2855024337768555, 1.5098929405212402, -0.19856396317481995, -0.30232810974121094, -0.8830061554908752, -1.981699824333191, 0.3727332651615143, 1.0739582777023315, 0.19627411663532257, 0.6377862691879272, 1.077999472618103, -0.8968638777732849, 0.16873744130134583, -0.6448037028312683, 0.809779942035675, 0.48917120695114136, -2.446894645690918, -0.8341481685638428, 0.16302822530269623, -1.2877113819122314, -2.3765816688537598, -1.0618836879730225, 0.005486499983817339, 0.8581029772758484, -0.00914757139980793, 0.1388205587863922, -0.4064784646034241, -1.2564610242843628, -0.3180473744869232, -2.15216326713562, 0.22311802208423615, 1.0167566537857056, -0.024004850536584854, -0.32018235325813293, 0.32690533995628357, -0.2855554223060608, -0.7965520024299622, 1.1174025535583496, -0.4356634318828583, -0.15211094915866852, 0.23466217517852783, 0.09541440010070801, 0.05674686282873154, -0.9026639461517334, -1.49803626537323, -0.5572199821472168, -0.9727407097816467, -0.9967706799507141, -0.655198872089386, 0.28463834524154663, 1.1455174684524536, -0.6861939430236816, 1.6425275802612305, -1.2483937740325928, 0.33476755023002625, -0.18656238913536072, -0.7600881457328796, 0.7260271906852722, 1.409633755683899, 1.4801042079925537, -1.0971852540969849, -0.6081730127334595, 0.2350481152534485, -0.09118394553661346, -0.6856999397277832, 0.708968460559845, 1.33500075340271, 1.033334732055664, 1.699285864830017, -1.8201286792755127, -0.48142316937446594, -1.1415096521377563, 0.3540486693382263, 0.7466611266136169, 0.6580056548118591, -0.6184122562408447, -0.6605105400085449, -1.1163012981414795, -1.3126956224441528, -1.1084026098251343, 1.3185911178588867, -0.6975997090339661, -0.21940816938877106, -0.18326729536056519, -0.28632766008377075, -0.9924395680427551, -0.8288435935974121, -1.7165359258651733, -2.2259109020233154, 0.9272255897521973, -0.24592477083206177, -0.41957423090934753, -1.0739694833755493, -0.8368513584136963, 0.7893348932266235, 0.7269809246063232, -0.5950074791908264, -2.2110936641693115, 0.22286832332611084, 0.14736966788768768, -0.8318749666213989, -2.2211103439331055, -1.2130656242370605, -0.09833720326423645, 1.316794991493225, -0.8588665127754211, 0.8537268042564392, -0.17676004767417908, -1.803219199180603, 0.8358173966407776, -1.4000606536865234, 0.3803284168243408, -0.002895224839448929, -0.16905514895915985, -0.6364750862121582, -0.1789940446615219, -0.37971287965774536, -0.07045739889144897], [-0.6936984658241272, 0.044556163251399994, 1.4122270345687866, 0.6721794605255127, 1.1827410459518433, -0.04043715447187424, 0.38356730341911316, 1.2927740812301636, -0.3881327211856842, 0.2778857946395874, 7.850307156331837e-06, -0.41799506545066833, 0.2232287973165512, -0.3813669979572296, -1.1525895595550537, 1.4933308362960815, -1.4258068799972534, -2.035928726196289, 0.47146105766296387, -2.2685964107513428, -2.187203884124756, -0.6490402817726135, -0.5013430714607239, 0.7859782576560974, 0.6200756430625916, 0.6672706604003906, -1.2782049179077148, 1.1839252710342407, 0.32396644353866577, -0.06835729628801346, 0.841421902179718, 0.46900099515914917, -0.9082856774330139, -0.6139541864395142, -1.1736308336257935, 0.7502217888832092, 0.42200368642807007, -0.4053986072540283, 0.316710889339447, -0.3445947766304016, -0.5331506133079529, 0.08716987073421478, -0.1270236074924469, -0.703454852104187, 0.6282867789268494, -1.2600544691085815, 1.0738970041275024, 0.6574096083641052, 0.7669070959091187, -0.8581680655479431, -0.7473012208938599, -1.6959527730941772, -1.9687739610671997, 0.09082333743572235, 0.5797211527824402, -1.6649671792984009, -2.266436815261841, -0.3237980008125305, -0.4516831934452057, 0.8035497069358826, -1.0292165279388428, -0.7632297873497009, -1.6467890739440918, 0.1980627179145813, 0.9590426683425903, 1.008763313293457, 0.5037899613380432, 0.9293562769889832, -1.544823408126831, -0.4556271731853485, 0.14789408445358276, -0.9560912251472473, 0.002316545695066452, 0.9905513525009155, -0.69957435131073, 1.0260385274887085, 0.26270633935928345, -0.3922785818576813, -0.8877919912338257, 0.9956647753715515, 0.02729075402021408, -0.19967547059059143, 0.15920071303844452, 0.5249800682067871, 0.8083320260047913, 0.08814161270856857, -0.9256705641746521, 2.4898605346679688, 1.467515468597412, -1.3624299764633179, -1.3057209253311157, -0.20925623178482056, -0.8614528179168701, -0.32740238308906555, 0.32714906334877014, -1.3447805643081665, -1.937378168106079, -0.03224722295999527, 1.1701102256774902, 1.9262158870697021, -0.25931429862976074, -0.20487822592258453, -0.882744550704956, 0.5282428860664368, -2.2211248874664307, 0.04632798954844475, -1.1960076093673706, 0.07155235856771469, -0.4357113540172577, -0.31130820512771606, 0.8251004219055176, -0.3143370449542999, -1.9314305782318115, 0.8853402137756348, 0.1196894571185112, 0.7646545767784119, -1.0333008766174316, 1.008899211883545, -0.5426416397094727, 0.17333579063415527, 1.4037220478057861, -0.8800221681594849, 1.1340153217315674, -0.2527797520160675, -1.12294340133667, 1.0359400510787964, -1.5485891103744507, -0.9782608151435852, -0.35702699422836304], [-1.4248026609420776, 0.8509915471076965, 0.008776495233178139, -0.21823200583457947, 0.27227166295051575, -0.8561646938323975, -0.37911173701286316, 0.10151702910661697, -0.5423242449760437, -0.23240022361278534, -0.37593966722488403, -0.8481584787368774, 1.0760746002197266, -0.911899745464325, 0.30769503116607666, -0.3707684576511383, 2.177668809890747, 1.4167953729629517, 0.8042976260185242, -0.715838611125946, 0.0718177929520607, -0.43221646547317505, 1.3659555912017822, -0.8172567486763, 0.13838344812393188, -0.018582075834274292, 0.9541423320770264, -0.09716720134019852, -0.8176053762435913, 0.05860123410820961, 0.5924437046051025, -0.594867467880249, 0.4556591212749481, 0.5791016221046448, -0.9695199131965637, 0.3730010390281677, 0.26310908794403076, -0.462434321641922, 1.977400541305542, 0.6138098239898682, -1.1413750648498535, 2.3436219692230225, 0.33885088562965393, 0.18885914981365204, -1.5607550144195557, 0.2559639811515808, 0.7356144189834595, -0.43211662769317627, -0.17187319695949554, -0.1432117074728012, 0.1948288232088089, 0.4266048073768616, -0.8644993901252747, 0.01402316428720951, -1.709370493888855, -0.5890878438949585, -1.1270768642425537, 1.051938772201538, -0.06168777123093605, 0.5858501195907593, -0.6316985487937927, -2.4291815757751465, 0.5375459790229797, 0.08376314491033554, 0.014734488911926746, 1.7328563928604126, -0.17567546665668488, 1.938209891319275, 1.1283427476882935, 0.8977721929550171, -0.16723117232322693, 2.2287096977233887, -0.05462082475423813, 0.28921717405319214, -0.17565181851387024, 1.6827335357666016, -1.1426581144332886, -0.7126038074493408, -0.6882485747337341, 0.929101288318634, 1.0987722873687744, -0.8783242106437683, -0.701732337474823, -1.7277077436447144, -1.4758713245391846, -1.1215413808822632, -0.8974891901016235, -0.7253721952438354, -1.8382809162139893, -0.4113820195198059, 1.0671330690383911, 1.2297941446304321, -0.47605249285697937, 1.284151315689087, -2.5436689853668213, 1.296079397201538, 0.09154154360294342, 0.3017161786556244, 0.7909855246543884, -0.8905208110809326, -1.0894490480422974, -0.3574215769767761, 0.6211180686950684, 2.050330877304077, 0.019058551639318466, -0.017961282283067703, -0.766535758972168, -1.2584675550460815, 2.6557743549346924, 1.5071427822113037, -0.5432371497154236, 0.40193048119544983, -0.754754364490509, -0.06624399870634079, -1.0179765224456787, 0.6196652054786682, -0.3099120855331421, 1.1556686162948608, -1.3234020471572876, -0.1885712742805481, -0.9036867618560791, -0.744412362575531, -0.014044987969100475, 0.48545947670936584, 0.25370755791664124, 0.8222559094429016, -0.5923845171928406, 0.7475654482841492, 0.6183808445930481], [-0.48691582679748535, -0.987765371799469, -0.2328273206949234, 0.11696046590805054, 1.1937850713729858, 0.8206198215484619, -1.22908353805542, 0.9211210012435913, 0.6353518962860107, 1.8477612733840942, -0.8354676365852356, -1.1945700645446777, -0.580271303653717, 0.49842485785484314, 0.46655791997909546, -0.12011512368917465, -0.09372194856405258, 0.9835608601570129, -0.734376072883606, -0.7163257598876953, 0.1085103303194046, 0.10410555452108383, 0.28570348024368286, 0.590397834777832, 0.6213883757591248, -1.5636471509933472, 1.0847722291946411, -1.1858574151992798, -0.8917189836502075, -0.39096590876579285, -0.5283329486846924, -0.16798846423625946, 0.5909718871116638, -0.2957908809185028, 1.085746169090271, 0.7374980449676514, 0.02435188554227352, -1.1657317876815796, 1.2541110515594482, 1.0345407724380493, 1.401267647743225, 0.4300917088985443, 0.8042895793914795, -0.9478439688682556, -0.045026302337646484, -0.8586829900741577, 1.103376865386963, 0.4296215772628784, -0.2378332018852234, 0.24953454732894897, 1.590047001838684, 0.747684121131897, 0.093475840985775, -1.0765992403030396, 0.6905962824821472, -1.3662221431732178, 2.0637943744659424, 0.36547422409057617, 0.37597179412841797, 0.11975519359111786, 1.088651418685913, 0.18057586252689362, -0.43716108798980713, 0.10103766620159149, 0.40802180767059326, 0.12045127153396606, 0.25327736139297485, -0.7628944516181946, 0.15920351445674896, 0.7760503888130188, -0.3143410086631775, -1.9699496030807495, -1.3598148822784424, -1.571640133857727, -0.27086272835731506, -0.7964390516281128, 0.44414010643959045, 0.060429271310567856, -0.34466853737831116, -0.8976742029190063, 2.6446194648742676, 0.10866601765155792, -0.903143584728241, 0.5774965286254883, -0.5951113104820251, 0.224721297621727, 0.27587100863456726, -0.16621768474578857, 1.0778131484985352, 0.8911792635917664, -0.5478907227516174, 0.05099084973335266, -0.1940077245235443, 0.6936092376708984, -0.5054236650466919, 0.8522738814353943, 0.2774244248867035, 0.20329250395298004, 1.1755146980285645, 0.08196444809436798, -1.0331653356552124, -2.15720534324646, -1.0375431776046753, -1.060157299041748, -0.23965752124786377, 1.5376191139221191, -1.6143168210983276, -0.18528327345848083, 0.8472373485565186, -2.02738356590271, 0.3274933099746704, 2.0951836109161377, -1.9001597166061401, 0.3161522448062897, 0.04101595655083656, 0.27979281544685364, -0.17549628019332886, -0.9379501938819885, 0.33247530460357666, 0.8185158371925354, 0.19744238257408142, 0.5688356757164001, 0.24857456982135773, -0.24048380553722382, 0.45443615317344666, -1.2641894817352295, -2.77732515335083, -1.3787695169448853, -0.003573404159396887], [-1.7935211658477783, -1.2715022563934326, 0.5819461941719055, -0.08855197578668594, 0.5598600506782532, 0.4791555106639862, -0.6884840726852417, -0.11999478936195374, -1.0898385047912598, 0.3189813494682312, -0.16714335978031158, -0.8203098773956299, -0.8582176566123962, -0.32841670513153076, -0.7798829078674316, -1.0487478971481323, -0.8144400715827942, -0.2532360255718231, -0.5451037287712097, -1.3786295652389526, 0.6723145842552185, -1.7407993078231812, -1.2005184888839722, 2.3421313762664795, 1.43879234790802, 0.9992759227752686, 0.3912305533885956, 1.9071778059005737, 0.4699108600616455, -0.037847794592380524, -0.705872118473053, 0.14055606722831726, 0.1165362074971199, -0.9084551334381104, 0.31320664286613464, -0.4074101746082306, -0.3977656960487366, -0.41997087001800537, -1.221612811088562, -0.7378467917442322, -0.15903350710868835, 0.4564535319805145, -0.09029590338468552, 1.1357929706573486, -0.698312520980835, -0.6808675527572632, 1.170594573020935, -2.9815447330474854, 2.568910598754883, 1.0361684560775757, -0.3793124556541443, -1.1324403285980225, -0.6501171588897705, -1.9300510883331299, -0.41837289929389954, -0.8479429483413696, -0.06697633862495422, 0.29974666237831116, 0.7324085235595703, -0.2122846096754074, 1.014966368675232, 0.5241106748580933, -0.7095235586166382, 0.606053352355957, -0.9277552962303162, -1.1667710542678833, -1.764561414718628, -0.9960943460464478, 1.8257087469100952, 0.24564124643802643, 0.16375195980072021, 0.9951929450035095, -0.15860973298549652, 0.2521258592605591, -1.0530428886413574, 0.590112030506134, 0.41570234298706055, -0.09024645388126373, -0.32621440291404724, 0.24902497231960297, 0.6117905974388123, 1.147266149520874, -0.8709887862205505, 1.6122373342514038, 0.28246361017227173, -0.30250799655914307, -1.9853506088256836, -0.5299602746963501, -0.8940349221229553, 0.46275147795677185, -0.0687161535024643, 1.7344902753829956, -2.1083760261535645, 0.6481289267539978, -2.1537461280822754, -0.08669955283403397, 0.255373477935791, -1.0128589868545532, -1.4244755506515503, 1.211737036705017, 0.6531832814216614, -0.9181159734725952, 0.10979781299829483, 1.4900354146957397, 1.7659562826156616, -0.5761207342147827, 0.9423831105232239, 1.6683701276779175, -0.2216050773859024, -1.0910961627960205, 2.0403659343719482, -0.25543129444122314, 0.2664976716041565, 1.0741022825241089, 0.5447589755058289, 1.2615108489990234, -0.2776840031147003, 0.9939383864402771, -2.161372661590576, 0.1582196205854416, 1.9876104593276978, 1.3323912620544434, -0.40571630001068115, -0.04322604089975357, 0.9183154106140137, -0.9970479011535645, -1.5025687217712402, 1.2502422332763672, -0.38069647550582886], [-0.29424917697906494, -0.5080618262290955, 0.13685880601406097, -1.1344342231750488, 1.5075278282165527, -1.173085331916809, 0.6166003942489624, 0.052828580141067505, -0.24288977682590485, 0.4340566098690033, 0.27005723118782043, 0.2726607024669647, -1.0270746946334839, -0.823491632938385, 0.023126181215047836, 2.201129674911499, 1.3812987804412842, -0.09836587309837341, 0.6128876805305481, 0.24651788175106049, -0.39485669136047363, 0.31304311752319336, -0.8393393754959106, -1.2221133708953857, -0.17987969517707825, 2.2890148162841797, -0.7174309492111206, -0.8587486147880554, 1.13655424118042, -0.41907304525375366, 0.17712703347206116, -0.5544041991233826, -1.3521904945373535, 0.5106312036514282, -1.5317620038986206, 0.21850484609603882, -0.8202760815620422, 0.5650213956832886, 0.4951690137386322, -0.9005306363105774, -1.3676514625549316, 0.16373547911643982, 1.1151736974716187, 0.7259264588356018, -0.6646900177001953, -0.9675970673561096, -1.2035943269729614, 0.11424664407968521, -0.02673560008406639, -0.6958593726158142, -1.1873576641082764, -0.05184813216328621, -1.5636101961135864, -0.34424081444740295, 1.461034893989563, 0.9123559594154358, -0.7586588859558105, 1.0077002048492432, 0.05987677723169327, 0.19770118594169617, 0.17421020567417145, -2.2058351039886475, 0.8701781034469604, -0.43884310126304626, -0.4042554199695587, 1.0264122486114502, -0.5201483368873596, -0.07565995305776596, -0.20775403082370758, 1.2159979343414307, -0.457662433385849, -0.0598599873483181, 1.4971154928207397, 0.6002264618873596, 0.5148242115974426, -1.9202406406402588, 1.1394312381744385, -0.8994157910346985, 1.1558856964111328, 0.7196034789085388, 1.1133151054382324, -0.5935041904449463, -0.21093663573265076, -1.7941858768463135, -0.6927470564842224, 1.1259517669677734, 0.23381417989730835, -0.6061199903488159, -1.367806077003479, 1.3863756656646729, 0.5884157419204712, 1.4454705715179443, -2.4369242191314697, 1.3923064470291138, -0.16632135212421417, -0.7401943802833557, 0.27772825956344604, 1.252297282218933, -0.7559365034103394, -1.3769688606262207, -0.41423317790031433, -0.8344199657440186, -1.0274758338928223, 0.8341391682624817, 1.5100443363189697, -0.5397251844406128, -1.4949274063110352, 0.05731702968478203, 1.4416158199310303, -0.8004826903343201, -0.3792162239551544, 0.2345002144575119, 1.2928540706634521, -1.2379801273345947, 1.7957831621170044, -0.7878322005271912, 0.35353755950927734, -0.040388502180576324, -0.28224244713783264, -1.1197031736373901, 0.7360246181488037, 1.365058183670044, 0.23772282898426056, -0.32735759019851685, -0.4837508499622345, 0.4889087378978729, -0.241971954703331, 0.7565592527389526, 0.25564542412757874], [0.6352705955505371, -1.3776479959487915, -0.07531032711267471, 0.1531798392534256, -0.9791445732116699, 2.442094087600708, -1.0541565418243408, 1.4299815893173218, -0.7177548408508301, -0.24487550556659698, -0.22202809154987335, 1.320312261581421, -0.3858574330806732, -0.8782997131347656, 0.3227647542953491, -1.2735933065414429, -0.3936024308204651, -1.0658679008483887, 1.6413207054138184, -0.11530089378356934, -1.3800429105758667, -1.1921392679214478, -0.6830088496208191, 0.7969055771827698, 0.6135981678962708, -0.31033334136009216, -0.5434761047363281, 0.35083892941474915, 0.48260706663131714, 0.18685999512672424, -1.2931162118911743, -0.24191340804100037, 1.6253035068511963, -1.589135766029358, -0.3716481029987335, -1.0916088819503784, -0.6637682318687439, 1.0042989253997803, -1.059592843055725, -0.4839456379413605, 0.2752339243888855, 0.2560442090034485, -1.5733777284622192, -0.7065937519073486, -1.5905182361602783, -0.6261810064315796, 0.42966046929359436, 1.1595637798309326, 0.6374257206916809, -0.08894925564527512, -0.9881591200828552, -1.351047396659851, 2.588120222091675, 1.0557745695114136, -0.4214630126953125, -0.32303667068481445, -1.0819027423858643, 1.8989834785461426, -1.277234435081482, 0.34991294145584106, -1.6969200372695923, 0.6541860103607178, -0.5510455965995789, 0.9988049864768982, 0.9106831550598145, -0.0029958244413137436, -0.1383594274520874, -0.5216531753540039, -0.49927467107772827, 0.2535369098186493, -0.8745023012161255, 0.1068740040063858, -1.5736600160598755, -0.5307506918907166, -0.8640233874320984, 0.2899233400821686, 0.15412317216396332, 0.049429867416620255, 0.06834236532449722, -0.7125695943832397, 0.20333875715732574, 1.9424850940704346, -0.004899872932583094, -0.9981118440628052, 0.33599653840065, -0.037761032581329346, -1.040994644165039, -1.589892029762268, 0.6808807849884033, 0.37974584102630615, -1.063418984413147, 0.0515216700732708, -1.7328181266784668, -1.0379109382629395, 0.18605497479438782, -0.8501592874526978, -0.20798853039741516, 0.8475669026374817, 0.3424519896507263, 0.9277383089065552, 1.1253457069396973, 0.5391955375671387, 0.5067205429077148, -0.41375383734703064, -0.008290493860840797, 1.6417263746261597, -1.3383291959762573, 0.16172459721565247, 0.1067599356174469, 0.013238180428743362, 0.04210396111011505, 1.0495411157608032, 1.0597760677337646, -0.34131908416748047, 0.4948800206184387, -1.473906397819519, 0.7162398099899292, 1.2379556894302368, -0.40202078223228455, 0.5575850009918213, -0.9494296908378601, 0.07204311341047287, -0.012300114147365093, 1.399546504020691, -0.24886032938957214, -0.9066950082778931, -1.579038143157959, 0.4293835759162903, -0.02565949782729149], [-0.9317768812179565, -1.080671787261963, -2.117682695388794, -0.22384202480316162, 2.844728708267212, 1.1542061567306519, 0.4687473773956299, 1.5343745946884155, 2.3354649543762207, 1.0903143882751465, -0.7249043583869934, -0.07148505747318268, 1.9437172412872314, 0.18928837776184082, -0.8335272073745728, -0.6251920461654663, 0.21999366581439972, -0.38568925857543945, 0.746522843837738, 0.3608590066432953, -0.8582555055618286, 1.6553555727005005, -1.1468020677566528, -1.2958658933639526, 0.2378917634487152, 0.7320972084999084, 0.6836790442466736, 0.25953707098960876, -0.03929588198661804, 1.5333327054977417, -0.004775624722242355, 0.2808825671672821, 0.4944166839122772, -0.4236041009426117, -0.41311490535736084, -0.7818354368209839, -0.7675620317459106, -0.019731596112251282, -1.324084997177124, -0.9237498044967651, 2.3116378784179688, -0.3558906316757202, 1.5811420679092407, 0.36468052864074707, 0.14587600529193878, 1.0881035327911377, 0.4139501750469208, 0.16679607331752777, -0.19262386858463287, -1.2187325954437256, 1.1416324377059937, -0.313037246465683, -1.082291841506958, 0.27439066767692566, -2.008457899093628, 0.004229111131280661, -1.8121243715286255, -0.20288823544979095, 1.0707212686538696, 0.8211907744407654, 0.18046484887599945, -0.49696001410484314, 0.6672447323799133, -2.211195230484009, -0.5458387136459351, -2.8426003456115723, 0.484860897064209, -0.6666283011436462, -1.4067726135253906, 1.4470990896224976, -0.2827443480491638, 0.07481466233730316, -0.2379956990480423, 0.261019229888916, 1.265562891960144, 0.5788870453834534, 0.29538694024086, 0.1892576366662979, 0.9966093301773071, -0.9201064705848694, -0.9498359560966492, -0.6162261366844177, -0.05462384223937988, 0.6644642353057861, -0.29606568813323975, -1.2718883752822876, 0.1355358064174652, -0.3059990108013153, 0.17840372025966644, 0.08137284964323044, -1.2580156326293945, -1.6648207902908325, 0.2599309980869293, 1.5088149309158325, -1.0557057857513428, 0.2062412053346634, -1.6216270923614502, -0.576754093170166, 0.09235994517803192, -1.6183825731277466, 0.15067969262599945, 0.7353773713111877, -0.7238474488258362, -0.579400360584259, 2.083944082260132, -1.0066664218902588, -0.15096315741539001, -2.3184752464294434, 0.45044007897377014, 0.6642292141914368, 0.0014676508726552129, -0.661244809627533, 0.2446490079164505, -1.323860764503479, -0.10028016567230225, 0.2646355926990509, -0.3812023103237152, -0.8185311555862427, 1.0611299276351929, -0.5037190318107605, -0.1643282026052475, -0.5003461241722107, 0.19546252489089966, -0.54185551404953, -1.0985546112060547, 1.512281060218811, -1.4512112140655518, -0.33577680587768555, 0.7364062070846558], [-2.3081681728363037, 0.012883120216429234, -0.5037849545478821, 0.04671856388449669, -0.2024381309747696, 0.341889351606369, 0.622616708278656, -0.38439473509788513, -1.430343508720398, -0.07701512426137924, -1.8110076189041138, 0.4238255023956299, -0.644699215888977, 1.3176026344299316, 0.320892333984375, 1.052851676940918, -0.009351056069135666, -0.31532010436058044, 2.168534755706787, 0.5739040374755859, -0.18761105835437775, 1.4058681726455688, 0.1748165786266327, -1.1720374822616577, -3.2342207431793213, 0.045326050370931625, 0.17317776381969452, -0.9405020475387573, -1.5234005451202393, -0.2682030200958252, 1.3863918781280518, 0.967908501625061, -0.21287624537944794, -1.501175045967102, 1.279352068901062, -1.5906866788864136, 1.8623144626617432, -0.5061826705932617, -0.7849346995353699, 0.4443306624889374, 1.0067590475082397, 0.16003383696079254, 0.5715544819831848, -3.7164947986602783, 1.1433840990066528, -1.1465846300125122, -0.5796847343444824, -0.637538731098175, -1.5084022283554077, 0.11166663467884064, -0.7210760116577148, 1.6011872291564941, -0.8804256916046143, -1.2254122495651245, -0.1946060061454773, 0.6380293369293213, 0.7119306325912476, 0.5343586206436157, 1.1405665874481201, -0.04589628800749779, -0.8177821040153503, -0.756982147693634, 1.1303375959396362, -0.18038247525691986, 1.2898838520050049, 1.196425199508667, -1.0305702686309814, 0.5023347735404968, 0.6981528997421265, -0.23783135414123535, -0.34785303473472595, -0.9573171734809875, 0.3682015538215637, 0.062158700078725815, -0.18753616511821747, -2.0274085998535156, 0.7752372622489929, -0.13889022171497345, -1.7225673198699951, -0.255533903837204, -0.20006896555423737, -0.05206020176410675, -0.6361474394798279, 1.3080565929412842, -1.3607242107391357, -1.2296533584594727, -1.0246840715408325, -0.16220197081565857, -1.0292433500289917, -0.0827227458357811, 0.609687864780426, 0.4602822959423065, -0.1316181868314743, -0.0938350185751915, -0.331133633852005, 0.30781689286231995, -0.3653741180896759, -0.42611265182495117, 0.7505085468292236, -0.8146877288818359, 1.7143785953521729, 0.6629913449287415, -1.1035449504852295, 0.35755276679992676, 1.1383540630340576, 0.2988269329071045, -1.4944788217544556, -0.806058943271637, 0.1730993539094925, 2.0141079425811768, 0.934184730052948, -0.22818860411643982, 0.3356679379940033, 0.3712623119354248, 0.46455323696136475, 0.7694776654243469, -0.522540271282196, 0.36422815918922424, -1.270356297492981, 0.8166244029998779, 0.3525647521018982, 2.501821517944336, 0.07892826199531555, 1.9149996042251587, -1.019271731376648, 0.8953527212142944, -0.6899836659431458, 1.94572913646698, -1.029698371887207], [0.10702578723430634, 1.3077852725982666, -0.7579461336135864, 0.011079599149525166, 0.21406465768814087, -0.5225628614425659, 1.7308998107910156, 0.6878193020820618, 1.020893931388855, 0.45073452591896057, -0.5037986040115356, -0.17733056843280792, 0.2683178782463074, -0.5778665542602539, -0.2722725570201874, -0.08533842861652374, -0.15208634734153748, -0.6399171948432922, -0.5062351226806641, 1.6205923557281494, 0.5184827446937561, 0.47132933139801025, 1.103807806968689, -0.14564356207847595, -0.08742675185203552, 0.2508101761341095, -0.5120425224304199, 0.194830983877182, 1.6672853231430054, 0.8139841556549072, 1.3575066328048706, 0.5650462508201599, -0.7614821791648865, -1.2091145515441895, -0.38398876786231995, -0.7165860533714294, 0.2587891221046448, -2.2472636699676514, -1.61872398853302, 0.5496982336044312, 0.2246515303850174, 0.8956860303878784, -0.11878018081188202, -0.17627492547035217, 1.1096339225769043, 1.5318408012390137, -0.21470555663108826, -0.7278554439544678, -0.37554436922073364, -0.9219051003456116, -0.7491496205329895, 0.19668982923030853, 0.9261561632156372, 0.802141547203064, 0.045088231563568115, -0.10444190353155136, -0.7521362900733948, 1.0177394151687622, -1.1210126876831055, -1.2515865564346313, 0.5646634101867676, -0.07938415557146072, -0.006226693745702505, 0.3355102837085724, 0.9465639591217041, -0.9524723291397095, 1.8331595659255981, -0.37899595499038696, 2.0436148643493652, 0.8202943205833435, 0.47107672691345215, -0.5189685821533203, -1.0739003419876099, -0.24383088946342468, 0.18527019023895264, 0.660223662853241, -2.085984468460083, 0.009208857081830502, 0.8551011681556702, -0.8844233751296997, 2.185579299926758, 0.11448683589696884, -0.833901584148407, 0.7804902195930481, -0.09412562847137451, 1.2889617681503296, 0.07610858231782913, 0.6762698888778687, -0.18295462429523468, -2.35336971282959, 1.3812990188598633, 1.1369435787200928, 1.6498867273330688, 0.7925716042518616, -0.6299079060554504, -2.1904804706573486, -0.056766629219055176, 0.9301324486732483, -1.345384955406189, 1.7951219081878662, -1.3223292827606201, 1.0323115587234497, 0.18582013249397278, 0.7422532439231873, -0.9429410099983215, 1.0745971202850342, 0.8988066911697388, -1.6046514511108398, 0.26708731055259705, -0.5372858047485352, 1.2194030284881592, 0.3738277554512024, 3.631542682647705, 0.14815597236156464, 0.020045237615704536, -0.1198945865035057, 0.3896976411342621, 0.6068946123123169, -0.05192401632666588, -0.024209776893258095, -0.8515188097953796, 0.9420915842056274, -1.4129540920257568, 0.12987163662910461, 0.43988800048828125, 1.4713867902755737, -0.9929512739181519, -0.6809694766998291, -0.20776940882205963], [0.5252575874328613, 0.9251543283462524, -0.4355638027191162, -1.2341315746307373, -0.47005900740623474, 1.0843631029129028, -0.6350309252738953, 1.3825973272323608, 0.4287967085838318, -0.49345284700393677, -1.50485360622406, -0.6692371964454651, -0.3971695601940155, -0.9754855632781982, 0.36876174807548523, -2.282148599624634, 0.48852115869522095, -0.7945883274078369, 0.8432832956314087, -1.225692868232727, -0.959842324256897, -0.3146737813949585, -1.3679107427597046, 0.5232541561126709, -0.14259164035320282, 0.2228584736585617, 0.3779344856739044, -0.8026120066642761, 0.0006001395522616804, -0.5836874842643738, 0.2772236466407776, -0.9182416200637817, 1.1448005437850952, -0.047495413571596146, -0.0006244938122108579, 0.9304443001747131, -1.3287746906280518, 1.0206031799316406, 2.007852077484131, 1.178030252456665, -1.07760488986969, -1.5796412229537964, 1.9733645915985107, 1.3646759986877441, 0.3940083086490631, -1.1794251203536987, 1.7351298332214355, -0.5870566964149475, 0.49839866161346436, 0.13238288462162018, 1.2475132942199707, 1.2194443941116333, 0.6685323715209961, -0.7506290674209595, 0.72539222240448, -0.09801637381315231, -0.9007607102394104, -1.5674777030944824, -2.3062756061553955, 2.04862380027771, 1.6718807220458984, 0.19778972864151, -0.4992762804031372, -0.26317283511161804, 0.6401418447494507, 0.03238436579704285, -0.7842133045196533, 1.1569414138793945, 0.14058595895767212, 0.9141615033149719, 1.0008994340896606, 0.8566690683364868, 0.19382809102535248, -0.40988889336586, -1.934001088142395, 1.681801676750183, -1.1019481420516968, 0.7681271433830261, 0.251127153635025, 1.0770467519760132, 1.06121826171875, 0.6085882186889648, 0.07927948236465454, 0.9757295846939087, -0.578299343585968, -0.3237939178943634, 0.07532186061143875, 1.1013940572738647, -0.7763192653656006, 0.7791661024093628, 1.1971709728240967, 0.033952899277210236, 0.5702991485595703, 0.3191130459308624, -0.3718774616718292, 0.6965876221656799, -0.4447679817676544, 0.30036717653274536, -0.8188754320144653, 2.480192184448242, -0.5902416706085205, 0.9012196063995361, 0.2165626585483551, -0.6638545989990234, -2.043020009994507, -0.234772726893425, 0.9498751163482666, 0.7899759411811829, -0.708692193031311, 0.27080032229423523, 0.7451309561729431, -0.2713303864002228, -0.3406282663345337, -0.5734891891479492, 1.28118097782135, -2.7066807746887207, -0.3851381540298462, 1.0201674699783325, 1.3530688285827637, 1.9621152877807617, 0.30860576033592224, -0.8690586686134338, 0.5205011963844299, -1.0121724605560303, 0.7312312722206116, -1.8481643199920654, 0.7513850927352905, -1.4268912076950073, 0.19660227000713348], [0.3067665696144104, -1.7032527923583984, 1.1203852891921997, 0.028590494766831398, 0.6302925944328308, 1.3311101198196411, -1.886377215385437, -0.34865108132362366, 1.2370314598083496, -1.3981389999389648, 1.5017811059951782, -0.3157309293746948, -0.6330574750900269, 0.2444557249546051, 0.02384413406252861, 0.7807843685150146, -1.4156900644302368, 0.8505377173423767, 0.8860270380973816, 0.3325763940811157, 0.859452486038208, -1.681475043296814, 0.5244653820991516, 1.9191254377365112, 0.3212023079395294, -0.8642043471336365, 0.22696711122989655, 0.6364373564720154, 1.4241669178009033, -1.7597525119781494, 1.9336512088775635, 0.11420382559299469, -0.22633932530879974, -1.5895638465881348, -0.42501315474510193, -0.5205470323562622, 0.45293399691581726, -1.042534351348877, 0.8397852778434753, 1.2170183658599854, -0.26847216486930847, -0.40615537762641907, -0.3058960735797882, -0.1282411813735962, -0.43146568536758423, 0.32023149728775024, -1.0378813743591309, 0.7299581170082092, 0.7590051293373108, -1.901663064956665, -0.32990458607673645, 1.158245325088501, -0.6154539585113525, 1.6211804151535034, 0.8295490741729736, -0.6715743541717529, -1.0823346376419067, -1.1396563053131104, -0.9636937975883484, 0.219288170337677, 0.9023869633674622, -0.5673748254776001, -1.0667848587036133, 0.7294658422470093, -0.6976789236068726, 1.353710651397705, -1.1012448072433472, -3.3302090167999268, 0.6444849967956543, 0.2768293023109436, 0.013554970733821392, -1.0702409744262695, 0.11182916164398193, -1.0732760429382324, -0.40248623490333557, 0.12424913048744202, -2.301330327987671, 1.1841821670532227, -0.34636276960372925, 1.0553123950958252, -0.6301581263542175, 1.4127414226531982, -0.4928687810897827, -1.9199199676513672, 0.4938768744468689, 0.6317768096923828, 2.4419660568237305, -0.7149783968925476, -0.688156008720398, 0.369284063577652, -0.407351016998291, -1.3969273567199707, 2.58380389213562, 0.10906346887350082, 2.389934778213501, 0.4014695882797241, 1.5774896144866943, 1.6974281072616577, -0.01608368009328842, -0.9051380157470703, 1.6253561973571777, -1.0100481510162354, 1.637847900390625, 0.18207146227359772, -1.122928261756897, -1.3073867559432983, -1.5822488069534302, 1.198932409286499, -1.4027249813079834, 0.14735643565654755, 0.02368742786347866, 0.26994407176971436, 0.48749595880508423, 0.5059734582901001, 0.11683733761310577, -1.545645833015442, 0.009751946665346622, -0.3076465427875519, -0.2269778847694397, -1.9156051874160767, -0.7937068343162537, 2.23390793800354, 1.3284754753112793, 0.6652172207832336, 0.7099822759628296, 1.044611930847168, 1.5282742977142334, 0.6008554697036743, -1.6605380773544312], [-1.303615927696228, -0.9884997010231018, 0.7223883271217346, 0.1319708228111267, -0.16433914005756378, -0.9555140733718872, 0.37756481766700745, -0.4076107144355774, 0.989491879940033, 0.13681960105895996, -2.0482475757598877, 0.022022444754838943, 0.48802080750465393, 0.565081775188446, 0.48868274688720703, 0.19075587391853333, 0.2828153967857361, 0.12767325341701508, -0.034163594245910645, -0.014574674889445305, 0.6568805575370789, -0.7886418700218201, -0.6496303677558899, 0.4588490128517151, 0.21304909884929657, 0.8088285326957703, -1.0042392015457153, -1.689034342765808, -0.38961026072502136, -0.6922289729118347, -1.0814849138259888, -1.2624284029006958, 0.14332009851932526, 0.34161514043807983, 0.06390655040740967, -0.35724499821662903, 0.3652738630771637, 1.6732556819915771, -2.404163360595703, 1.6449588537216187, -0.09479463845491409, -1.8267618417739868, -1.2372419834136963, -0.1599653959274292, 2.4718945026397705, 1.726277232170105, -0.7635136246681213, 0.6720471978187561, 0.5751839280128479, -0.1062449961900711, -0.6108764410018921, 1.0294737815856934, 0.630293607711792, 0.05669272691011429, 0.21719706058502197, -1.299458384513855, 0.5245192050933838, 0.667602002620697, 0.12030917406082153, 0.8474597930908203, 1.2071579694747925, 0.42300963401794434, 1.3391979932785034, -1.0165584087371826, -0.4263856112957001, 1.7936146259307861, 0.8334343433380127, -0.5401422381401062, -0.3335554599761963, -0.9618299007415771, 0.4554970860481262, 0.11355870217084885, -0.4081757366657257, -1.186029076576233, -0.6863114833831787, 1.154139757156372, 1.0685309171676636, -1.9894347190856934, -0.17716169357299805, 0.9449983239173889, -0.0020968536846339703, 1.3931933641433716, -0.36187154054641724, -0.616244375705719, -1.4779345989227295, -1.7060869932174683, 0.20744478702545166, -1.5976405143737793, -0.4879574179649353, 1.181822657585144, -0.2660963833332062, -0.2726162374019623, -0.32354259490966797, -0.3303760886192322, 0.9777210354804993, 1.6584762334823608, 0.30320948362350464, 0.2582377791404724, 0.07295776158571243, 0.3326188921928406, -1.2564352750778198, 0.01612919010221958, 0.9722152948379517, -1.2515650987625122, 1.3088887929916382, 1.1114592552185059, -1.5144357681274414, 1.451157569885254, -0.006515780929476023, 0.11954344809055328, -1.2221291065216064, 0.10555935651063919, -0.44573739171028137, 0.26432475447654724, -0.7766408324241638, -0.6663490533828735, 0.23089441657066345, 0.574754536151886, -1.4134883880615234, -0.2946290969848633, -0.7719156742095947, -0.5951191782951355, 0.48384764790534973, -0.4179587960243225, -0.8606961369514465, -0.8842970132827759, 0.9989064931869507, -1.6282931566238403, 1.2596361637115479], [-0.4974943995475769, 0.7361776828765869, 1.1373136043548584, 0.3541260063648224, -0.7676268219947815, 0.9468409419059753, -0.694284975528717, 0.6938800811767578, -0.10099858790636063, -0.9833215475082397, 0.026643335819244385, 0.6527661681175232, -0.38340696692466736, -0.053183138370513916, 0.42813098430633545, 0.4529912769794464, 1.3808817863464355, 1.1968661546707153, -0.25186148285865784, 0.9943087697029114, -0.3456084728240967, 0.01655731536448002, -0.5951951146125793, -0.3371317684650421, -1.1093692779541016, 0.5527220964431763, -0.9493592977523804, -1.6467195749282837, 0.33525657653808594, -0.701905369758606, -2.502676248550415, -1.4703298807144165, -0.5160349011421204, 0.1606285572052002, -1.0317275524139404, 0.4745699465274811, -1.0388829708099365, 0.2514380216598511, 0.12613292038440704, -0.2999721169471741, -0.533842921257019, -0.3129115700721741, -1.488417625427246, 1.6394034624099731, -2.335841417312622, 1.425813913345337, 0.277955025434494, 0.6203379034996033, -0.44813117384910583, -0.10250913351774216, -0.24849732220172882, -0.21348276734352112, 1.6276782751083374, -0.386934369802475, -1.1404651403427124, 0.743862509727478, -0.4081813395023346, -0.7106401920318604, 0.29609793424606323, -1.365187644958496, -0.160818412899971, 0.5573091506958008, 0.1131349429488182, 0.03075779415667057, 0.09759094566106796, 0.10950086265802383, 0.7837091088294983, 0.4669745862483978, -0.000944470870308578, 0.27962398529052734, 0.7098215222358704, 1.2440016269683838, 0.01582985557615757, 0.02230123057961464, 0.6698023080825806, 0.5256971716880798, -1.4316068887710571, -1.1595873832702637, -0.793979823589325, 0.5687438249588013, -0.5591207146644592, -0.880616307258606, -0.01400988269597292, -0.8079816102981567, -0.03070046752691269, -0.8408889174461365, 2.7758100032806396, 0.49950671195983887, 0.7993773221969604, 1.5133346319198608, 1.2754285335540771, 0.22553132474422455, -1.1164156198501587, -1.2680569887161255, -1.6886788606643677, 0.5885216593742371, 0.45865073800086975, -0.3918415904045105, -0.4394870400428772, 0.7440474629402161, -0.41459423303604126, -0.7973687052726746, -0.9031362533569336, -0.16190260648727417, -0.15615078806877136, -1.2890946865081787, -0.3377652168273926, -0.23234295845031738, 1.361520528793335, 0.424301415681839, -0.8550073504447937, 0.3776555061340332, -1.310886263847351, 0.4705711901187897, -0.2239658385515213, -0.3107580244541168, 0.3576981723308563, -0.9986522197723389, 0.09944146871566772, -1.0768115520477295, -0.3888018727302551, 0.33460891246795654, -0.3644872009754181, 1.323996901512146, 1.064176082611084, 0.23037974536418915, 0.9875978827476501, -0.6464566588401794, 0.4840937852859497], [0.21579259634017944, -0.30909693241119385, 0.4198058545589447, 0.35090380907058716, -0.1294144243001938, -0.44751814007759094, -1.892595648765564, -1.369884967803955, -0.6713542938232422, -1.0587875843048096, 0.22031816840171814, -1.3530570268630981, -0.5284339189529419, -0.02955811470746994, 1.319720983505249, 0.6199407577514648, -0.18230921030044556, -0.3564654290676117, -1.596387267112732, 0.1235755980014801, -1.1454637050628662, 0.39328017830848694, -0.45491376519203186, -0.9554929137229919, -0.4994642436504364, -1.1127924919128418, 1.1568872928619385, 1.0959595441818237, -1.2019258737564087, -0.1944141983985901, 2.133024215698242, 0.14202171564102173, -0.7198017239570618, -0.25782492756843567, 1.755974292755127, -2.223299503326416, 0.41776421666145325, -0.04516258090734482, -2.238649606704712, -1.5329245328903198, 0.05850803852081299, -0.9414694905281067, -0.6135889291763306, 0.18374480307102203, 0.11365853995084763, -0.24463976919651031, 0.9243372082710266, -0.051592908799648285, -0.6478099226951599, 2.1830251216888428, -1.5993666648864746, 0.6725690364837646, 0.32220596075057983, -0.2845352590084076, 0.6863204836845398, 1.3526787757873535, 0.7574039101600647, 1.1916966438293457, -0.1345125138759613, 0.08011021465063095, -1.847991943359375, 0.11933132261037827, 0.32922518253326416, 1.5085186958312988, 0.26028111577033997, 0.05221601948142052, -1.3334945440292358, -0.4854638874530792, 0.4694423973560333, 0.11530941724777222, -0.3786899447441101, -1.279937982559204, 1.217545509338379, -0.43232133984565735, -0.6455774903297424, -0.7840332984924316, 0.37404492497444153, 0.1369129866361618, 0.8961700797080994, -0.40997007489204407, 0.04616469889879227, -0.8466964364051819, 0.9613065123558044, -0.007838538847863674, 0.5529187321662903, -0.06849777698516846, -0.4786514341831207, 2.528944492340088, 0.1750325709581375, 1.0902447700500488, 0.1389070302248001, 0.7472917437553406, -1.3543484210968018, 1.1685669422149658, -1.6128389835357666, -1.2426080703735352, -2.504054546356201, -0.15830889344215393, 1.798744559288025, -1.615637183189392, -0.6939185857772827, 0.35033440589904785, 0.5526024103164673, -0.7351910471916199, 0.6429333090782166, 0.02756296657025814, 0.582619309425354, 0.29447227716445923, 0.10888078063726425, -1.0297199487686157, 0.11037193983793259, -0.04327472671866417, 1.408759593963623, -1.0541573762893677, -0.20240069925785065, -0.6542283296585083, 1.3620233535766602, 0.4253535568714142, 1.2785879373550415, 1.0912227630615234, -1.1025950908660889, 0.6647608280181885, -1.1047190427780151, -0.30965456366539, 0.28053024411201477, 0.21031928062438965, -1.4543033838272095, 2.2026867866516113, -1.7466861009597778], [-0.3282873332500458, -0.4562353491783142, 0.06188022717833519, -0.6382988691329956, -1.1602360010147095, -1.0616114139556885, -0.9958534240722656, 0.5336953997612, 0.882334291934967, -0.3606575131416321, 1.0888508558273315, -0.992682158946991, -0.7420362234115601, -1.2683321237564087, -0.660624623298645, -1.4861798286437988, 1.360909342765808, -2.304996967315674, 1.7619389295578003, 0.08322175592184067, 1.3995511531829834, 0.848433256149292, 0.0142471669241786, 2.5204832553863525, 1.0503848791122437, -0.6538543105125427, 0.4711671471595764, -0.7788442373275757, -1.2048746347427368, -1.1297563314437866, 0.5978613495826721, 0.5947096347808838, 0.23108039796352386, 0.33081504702568054, -0.3673941195011139, -0.8203526139259338, 0.6004668474197388, 0.007641839794814587, -0.8682357668876648, -0.020013809204101562, -1.4530729055404663, 1.047442078590393, -0.182774618268013, -2.0951483249664307, 0.6620711088180542, -2.0667145252227783, 0.0670238584280014, 0.035505522042512894, -0.6445781588554382, 0.36172792315483093, 0.021301036700606346, -1.2540987730026245, 1.6116341352462769, -0.33286765217781067, -1.5738184452056885, 0.9800541400909424, 0.3437163233757019, 0.5653718113899231, 1.328526496887207, -2.59185791015625, -0.014970238320529461, -0.5840098857879639, 0.7887276411056519, -0.6396760940551758, -0.002093668794259429, -0.32369667291641235, -0.814786434173584, 1.1762480735778809, -1.3856717348098755, -0.09003520756959915, -0.336322158575058, 1.497219204902649, -0.8567858934402466, 0.40325719118118286, 0.5054161548614502, 0.5117320418357849, 1.0801647901535034, 0.5217345356941223, 1.0042760372161865, -0.3811442255973816, 0.27506569027900696, -1.1970714330673218, 0.9173020720481873, 0.05946122109889984, 1.1269030570983887, -0.6278720498085022, -0.12900321185588837, -0.7737975120544434, 2.3262369632720947, -0.3089880347251892, 1.1636766195297241, -0.0496881902217865, 0.20862209796905518, 2.1920852661132812, 0.8152133822441101, -0.013909258879721165, 1.9597166776657104, 0.050135646015405655, -0.621556282043457, 0.9727385640144348, -1.240921974182129, -0.44059404730796814, -0.264042466878891, 0.08336915075778961, -0.535750687122345, 0.8704396486282349, 0.3482738435268402, -0.23130910098552704, -1.510979413986206, -1.489067554473877, 1.5114680528640747, 0.6439923048019409, -1.488777756690979, 0.21562591195106506, -0.09753129631280899, -0.618523895740509, -1.406449794769287, 0.07336974889039993, 2.5258562564849854, 1.0017939805984497, 0.38302844762802124, 2.2292122840881348, 0.2805038094520569, 0.6404775977134705, -0.860873281955719, 0.3339337110519409, 0.7462414503097534, -0.2145659476518631, -0.21116934716701508], [-1.2405214309692383, -0.9026316404342651, -0.9986424446105957, 1.233687400817871, -0.49802547693252563, -0.9093162417411804, -1.1949058771133423, -0.21196123957633972, -0.675771951675415, -0.2021801471710205, 0.4178869426250458, 0.9884447455406189, 0.32126426696777344, -0.6392783522605896, -0.30625003576278687, 0.09181351959705353, -0.024017244577407837, -0.06247469037771225, 0.3058258295059204, -2.8264877796173096, -1.481907606124878, 1.7192392349243164, -1.7410811185836792, -0.12402797490358353, -0.020465940237045288, 0.1281740367412567, 0.20997169613838196, 2.3911657333374023, -0.5397493839263916, 0.136099174618721, -1.7869219779968262, -0.5223961472511292, 0.6558400988578796, -0.7596714496612549, 0.2811354398727417, -0.7453373074531555, 2.191938877105713, 1.6829043626785278, 0.5654112100601196, 0.18928520381450653, 2.2177324295043945, 1.8278748989105225, 1.0687757730484009, 2.550135850906372, -0.6339966654777527, -0.802539050579071, 0.2368861883878708, -1.207055687904358, 0.6469345688819885, -0.6683098077774048, 0.04786817356944084, 1.8582302331924438, -0.057535476982593536, 0.5442067980766296, 0.7868504524230957, -0.014484485611319542, -0.13710637390613556, 2.5920207500457764, 1.2149807214736938, -1.0352535247802734, -0.03188035264611244, -1.1226806640625, 0.5426160097122192, -0.2247738242149353, -0.3280642330646515, -0.3816283941268921, -0.38367390632629395, 0.6866174340248108, -0.53437340259552, 0.662013590335846, 0.39719974994659424, 1.69923996925354, -0.4207409918308258, -0.9173160195350647, -0.5449362397193909, -2.1298766136169434, 0.7814649939537048, 1.5046825408935547, 0.14585494995117188, -1.7015377283096313, -0.47318196296691895, -2.4681026935577393, 0.1856040507555008, -0.24834507703781128, 0.3717629909515381, 2.2058427333831787, 1.110551357269287, -1.6543503999710083, -0.626776933670044, -1.6332063674926758, 2.902850389480591, -1.5320227146148682, -0.33452773094177246, -0.4237290024757385, 0.842502772808075, -0.29566967487335205, -2.4743921756744385, -0.7919259667396545, -0.6825559735298157, -0.046099189668893814, 0.0810643881559372, 0.39259883761405945, -0.31511473655700684, -1.3518530130386353, -0.08355256170034409, 0.20419476926326752, 0.4208700656890869, 0.4168458878993988, -1.554154396057129, 0.6240582466125488, 0.44202104210853577, 0.8039547204971313, 0.9290558099746704, -1.1625890731811523, -0.4111303985118866, 0.20816658437252045, -0.5954205989837646, 1.404440999031067, 1.4367703199386597, 0.8899672627449036, 1.5758119821548462, 0.11332941055297852, 1.6896131038665771, -0.7988077402114868, -1.2244676351547241, -0.0436822846531868, -0.34664949774742126, 0.1127910166978836, 0.36261752247810364], [1.0569382905960083, -0.45966607332229614, 2.149787187576294, -1.3897124528884888, -2.0028860569000244, -1.4344562292099, -1.4449981451034546, 0.9495638608932495, -0.20205944776535034, 1.6460518836975098, -0.058129437267780304, 1.7352622747421265, 0.6719605326652527, -0.539789617061615, 1.161468505859375, -0.4531320035457611, -0.44188350439071655, 0.05520850047469139, 0.277548223733902, -0.7795626521110535, -1.0078232288360596, 0.05727744102478027, -0.18508504331111908, -0.43130382895469666, -1.4619849920272827, -1.0918598175048828, 0.4248955547809601, 0.7888831496238708, 0.5327253937721252, -1.288791537284851, -0.055081646889448166, 0.5990745425224304, 1.2279011011123657, 0.6302021145820618, 0.613591730594635, -0.8854857683181763, -1.240763783454895, -0.5585916638374329, -0.36943185329437256, -0.6285139322280884, 0.06713485717773438, -0.465067595243454, 0.3854197561740875, 0.28313732147216797, 0.14734940230846405, -0.657272219657898, 0.36935561895370483, -1.437449336051941, 0.6073744893074036, 1.7715219259262085, -1.4111279249191284, -0.039301179349422455, 0.33591166138648987, -0.8016931414604187, -0.0013177968794479966, -1.9201751947402954, 0.05079016461968422, 1.2077252864837646, -0.0769856870174408, -0.2078896462917328, 0.8986838459968567, 0.2120347023010254, 0.19603820145130157, 0.1707058846950531, 0.8144703507423401, -0.057258959859609604, -1.2636353969573975, -0.0932646319270134, 0.6032941937446594, -0.3874876797199249, 0.7118648290634155, 0.16723498702049255, -0.6079380512237549, -0.36465156078338623, 0.6278825998306274, 0.7475154995918274, 0.667543351650238, 0.5521916151046753, 1.1561566591262817, 0.4463948905467987, 0.8781868815422058, 1.2566137313842773, -0.12832975387573242, 0.2878551483154297, -1.7834967374801636, 1.0714216232299805, 0.35636618733406067, 0.7398243546485901, 0.016290273517370224, 0.4780963957309723, 0.49684983491897583, -0.7993307709693909, -1.5244466066360474, 0.6228011250495911, 1.3173072338104248, -0.6465595364570618, 0.5374544858932495, 0.41516318917274475, 0.15278999507427216, -0.4345494210720062, -0.7791991829872131, -0.7464932799339294, 0.3333779573440552, -0.9009034037590027, 1.275221586227417, -0.23184099793434143, -1.2059102058410645, -1.2269070148468018, -0.2612917125225067, 1.4207435846328735, -0.038189422339200974, 0.5992123484611511, -1.8514975309371948, 0.1252632886171341, 0.35376107692718506, 0.31290900707244873, 0.6696509122848511, -0.2095532864332199, 0.6140018105506897, 0.9954020380973816, 0.7469278573989868, -0.49891677498817444, -2.1395177841186523, -3.464597463607788, -0.9649639129638672, -0.8860963582992554, -0.04375069960951805, -0.44336479902267456, -0.0559944212436676], [1.3818845748901367, 1.2478086948394775, -0.7884103059768677, -1.3120689392089844, -0.05149003863334656, -1.1088998317718506, -0.610184907913208, -1.0753079652786255, 0.1142188012599945, 0.15752777457237244, 0.46135252714157104, 0.5883797407150269, -0.6880455613136292, -0.11200626939535141, 0.15642546117305756, 1.0716781616210938, -1.5375187397003174, -0.19627706706523895, -1.1135741472244263, -1.043512225151062, 0.8294637203216553, -0.3340439200401306, -1.1681169271469116, -0.32594889402389526, 0.6531679630279541, 0.3559306561946869, -1.0841760635375977, -0.7305101156234741, -0.2517189681529999, 0.14733470976352692, -0.596215009689331, 1.230575680732727, -0.5742442011833191, 0.15538360178470612, -0.8175986409187317, -1.1519311666488647, 1.8950791358947754, 0.2358168363571167, -1.1919177770614624, 1.5805801153182983, 1.0520801544189453, -0.9460538029670715, 0.005961500108242035, -0.8776047229766846, -2.173455238342285, -0.2414071261882782, 0.8656368851661682, 2.3394651412963867, -1.711950421333313, 0.7894574999809265, 0.7864459156990051, -0.8020962476730347, 0.62070232629776, 0.5636479258537292, -1.0591987371444702, 1.6664998531341553, 0.828270435333252, 0.612445056438446, -0.07682070136070251, -1.1423389911651611, 0.6593983769416809, -0.36472228169441223, -0.6060836315155029, 0.3406286835670471, 0.7784042358398438, 0.7684748768806458, 0.01912328042089939, -0.4905759394168854, -1.3524606227874756, 0.41599294543266296, 0.19385355710983276, 0.6709264516830444, 0.46983328461647034, -0.0934889018535614, 0.08734845370054245, 0.4674878418445587, 0.7198514938354492, 1.261273741722107, -0.7828813791275024, -0.024289242923259735, 0.20002129673957825, 0.3196140229701996, -0.4267929196357727, 1.4755115509033203, 1.765371322631836, -1.1064534187316895, 0.29514580965042114, -2.1857504844665527, -0.5550557971000671, 0.40384551882743835, -0.5470036268234253, -0.1541537195444107, 0.4989149868488312, -0.3437047302722931, 0.40665364265441895, 2.054074287414551, -1.0089964866638184, -0.32845792174339294, -0.9602943062782288, 1.394981026649475, -0.9042337536811829, -1.2926884889602661, 0.4264812171459198, 0.45713579654693604, 0.6662392616271973, -0.8300012350082397, -0.32522547245025635, 0.32363519072532654, -0.4451967775821686, -0.36211514472961426, 0.8057666420936584, -0.15373508632183075, 0.1819671094417572, 0.2498205155134201, 0.43670985102653503, 0.7263786196708679, -0.33606621623039246, 1.8794307708740234, -0.03712254390120506, 0.5624586343765259, 0.7491207718849182, -0.2992265522480011, -0.6401079893112183, 0.11635420471429825, 1.04084312915802, 1.2133629322052002, -0.799697756767273, 0.7122036814689636, -0.18290744721889496], [-0.3111869692802429, 1.0153721570968628, -0.05112265795469284, -0.6978104114532471, 0.3469111919403076, 0.7129043936729431, -1.9242169857025146, -0.3635873794555664, -1.1863903999328613, -2.235051393508911, 0.14241145551204681, -1.572431206703186, -0.058082230389118195, -0.9768528938293457, 0.8471277356147766, 2.5415737628936768, -0.032084256410598755, -1.0963314771652222, 1.6875654458999634, 0.412957102060318, 0.20691388845443726, -0.09626937657594681, 0.12334977835416794, -0.7700263857841492, 0.68161541223526, 0.12763789296150208, -0.7139686346054077, 1.3868858814239502, -0.5400708317756653, -0.2835024297237396, -0.015597734600305557, 1.9988822937011719, -0.4831276834011078, 1.245826005935669, -0.5971633195877075, 0.9299083948135376, 0.7008898258209229, 0.6919136047363281, 0.8454036712646484, -0.2814493179321289, -1.9473977088928223, 0.7179741263389587, 1.0427995920181274, -0.1526748389005661, -0.21470153331756592, 0.8555407524108887, 1.093989372253418, -0.3738129138946533, -1.6810953617095947, -0.3020005226135254, -0.04544825851917267, 2.0096726417541504, -1.475690245628357, -0.8653269410133362, -2.4478647708892822, 1.0245074033737183, 0.4643693268299103, 0.11163810640573502, 2.2799198627471924, 0.3171559274196625, 0.04500861465930939, -0.6464423537254333, -0.22034883499145508, 1.4626882076263428, 0.8059422969818115, 1.261446475982666, 1.1562296152114868, -0.747542142868042, 0.11967037618160248, -0.8070098757743835, -0.38040754199028015, 1.2660839557647705, 0.6290077567100525, 0.029691100120544434, -1.576839804649353, -0.3466883599758148, -1.6762301921844482, 0.5423843264579773, -0.33257320523262024, -1.2365506887435913, 0.49585843086242676, 0.7352389097213745, 0.9593514204025269, 0.9844129085540771, -0.8431588411331177, -1.3816605806350708, 0.024224068969488144, 1.5892436504364014, 0.7490193843841553, 0.9365949034690857, 1.8151496648788452, -1.4601397514343262, 0.6384686827659607, 1.133012056350708, -0.6910357475280762, 1.5420856475830078, 0.2313825637102127, 0.32418227195739746, -0.5562552213668823, -0.23756122589111328, -0.39915722608566284, 1.3588364124298096, 0.03631618618965149, 1.2503814697265625, -0.04904197156429291, -0.7241693735122681, 0.7534181475639343, -0.1319105327129364, -1.2454181909561157, -0.7848787307739258, -0.24796871840953827, 0.34553536772727966, 0.4739522337913513, -0.8915281295776367, 0.04069575294852257, -0.66805499792099, 0.013977471739053726, -0.11352955549955368, 0.05175544321537018, -0.9589623212814331, 0.4571443796157837, -1.5955288410186768, -0.41017886996269226, -0.3529369533061981, 0.18147628009319305, -0.7935441732406616, -2.3732430934906006, 1.3844021558761597, 0.18555204570293427], [-0.03725043311715126, -0.8051584362983704, -0.44634076952934265, 0.6753605604171753, -0.2712729573249817, 0.30434170365333557, -0.25887903571128845, 1.509783387184143, -0.249294713139534, -0.7409725189208984, 0.8828075528144836, -0.4774059057235718, -0.41254839301109314, -0.4143442213535309, -1.4902844429016113, -0.6298224925994873, 1.7281595468521118, 0.05770603194832802, 0.023743445053696632, 1.7236881256103516, 0.6848906874656677, -1.8584314584732056, -0.05526043847203255, -0.41634783148765564, -0.7774621844291687, 0.8169544339179993, 0.48305630683898926, 1.0827131271362305, 0.6051626801490784, -0.969032883644104, 0.4457230865955353, 0.6972713470458984, -1.277878761291504, -0.7190431356430054, 1.1896895170211792, -0.5055864453315735, -0.06498038023710251, -0.8494648337364197, 0.6724338531494141, -0.1917545348405838, -0.3806378245353699, -0.24833159148693085, 0.4186601936817169, -1.364870548248291, -0.7136390209197998, -1.8389848470687866, -0.7856996059417725, 0.7666078209877014, 0.3213590085506439, -1.924034595489502, 0.5416324138641357, -0.5058769583702087, 0.5640009641647339, -0.14809203147888184, -0.4117491543292999, -0.1491338461637497, -0.0950164720416069, -0.23865287005901337, -0.09294027090072632, -0.33104610443115234, -0.7056325078010559, 0.4980570375919342, 0.28526872396469116, -0.5120453238487244, -1.0512847900390625, -0.6345157027244568, -0.26811540126800537, -1.1976243257522583, -0.5103437900543213, -0.3780430555343628, 1.4178704023361206, -0.2597309350967407, -0.5209389328956604, -0.17515021562576294, -0.11619015783071518, 0.003553739283233881, 0.6928029656410217, -0.21026505529880524, 1.1911884546279907, -0.0337241068482399, 1.0732715129852295, 0.1447092443704605, 0.6414573788642883, 0.2438480406999588, 0.006802596151828766, 1.0822488069534302, -0.0798136442899704, -0.8560397028923035, -0.6981814503669739, 0.9316052198410034, 1.5719608068466187, -0.3478796184062958, 1.342699408531189, 0.5520757436752319, 0.3416225016117096, 2.680145263671875, -0.4213586747646332, -0.07921145856380463, 1.8249253034591675, 0.9657503962516785, 0.24096770584583282, -0.43815866112709045, 1.226121187210083, 0.36428335309028625, -1.9716922044754028, -0.30002978444099426, 0.02732612006366253, 1.7630754709243774, -0.003751324722543359, 0.14187848567962646, -2.3088016510009766, -0.8979125618934631, -0.4333270490169525, 0.6107568144798279, 0.5812115669250488, -0.7887017726898193, -0.4415404796600342, 0.8672065138816833, -2.177643060684204, 0.1171388104557991, -1.6885594129562378, 0.7609531879425049, -2.135329008102417, 0.3608602285385132, 0.5044082403182983, -0.8915241360664368, 0.7653928399085999, -2.4210169315338135, -1.7051461935043335], [-0.7275018692016602, -0.6681228280067444, 0.8511103391647339, 0.42227572202682495, -0.9041209816932678, 0.36803722381591797, 0.19142891466617584, -1.9467982053756714, 0.5885530114173889, 0.8788644671440125, -0.8961341381072998, 1.1412770748138428, -1.0064460039138794, -1.157863974571228, 0.038255274295806885, 0.6010907292366028, -0.18954475224018097, -0.3309294581413269, 0.2013826072216034, 0.9711331129074097, -0.5400062799453735, 1.0414149761199951, 1.3141465187072754, 0.7631288766860962, -0.3923386335372925, 0.12534236907958984, -0.6370123028755188, 0.7024983763694763, -1.4798301458358765, -0.579047441482544, 0.6401692628860474, -2.7655155658721924, 0.7047885060310364, -0.744454026222229, -3.774837017059326, -0.34164875745773315, -0.5877689719200134, 0.5077043175697327, -0.5931643843650818, -0.4369266629219055, -1.471901535987854, 0.917439877986908, -0.6871286630630493, -1.1545212268829346, -0.6972384452819824, -1.5879011154174805, -0.7416682243347168, 0.3469983637332916, -1.1705189943313599, 0.2332201451063156, -0.03983559086918831, -0.160124272108078, -1.1588330268859863, -0.261446088552475, -0.27598774433135986, 0.6232095956802368, 0.06590302288532257, -0.5379676222801208, -1.716680645942688, 0.009533639065921307, 1.7985806465148926, -0.8722853660583496, 0.7048619985580444, -1.1146703958511353, 0.04825830087065697, 0.08331563323736191, -1.2577413320541382, -0.5250570178031921, 0.18108323216438293, -0.27031370997428894, 0.9997050166130066, -0.16347746551036835, -0.34710973501205444, 0.8350530862808228, -0.3149527311325073, 0.49364015460014343, -0.06057290360331535, 0.9636412262916565, -1.5411536693572998, 1.1538691520690918, -1.1282873153686523, -0.8537241220474243, -0.6270991563796997, 0.21448461711406708, 1.787062644958496, -0.9590715765953064, -1.2549198865890503, -1.1546553373336792, 0.2736871540546417, 0.2904728055000305, 1.3276963233947754, 0.3261933922767639, -0.43976566195487976, 0.6082359552383423, 1.2488420009613037, -0.8065826296806335, 0.984557569026947, -0.48215246200561523, -0.06473281234502792, -0.40196681022644043, -0.252888560295105, 0.7048214077949524, 0.30144551396369934, 0.26120397448539734, -0.4322291910648346, -0.9888599514961243, -0.5200543999671936, 0.41318777203559875, -2.138923168182373, -1.0806219577789307, -0.11990127712488174, -1.35739266872406, 1.0508923530578613, 1.6509184837341309, -0.4928375482559204, -1.7307803630828857, -0.7892243266105652, -0.25610917806625366, 0.7690672278404236, 0.9440969228744507, 0.609224796295166, -1.3962057828903198, -1.9174050092697144, 0.21654976904392242, 2.0326173305511475, 0.539698600769043, -0.9538339376449585, -0.47202372550964355, 0.30904555320739746], [-1.1110739707946777, -1.9886400699615479, 1.237330436706543, 0.8048009276390076, -1.4924390316009521, -0.3876478374004364, -1.2819219827651978, 1.5511168241500854, 0.6096506118774414, -2.748107433319092, 0.16070669889450073, -0.25770318508148193, -2.12086820602417, -0.2710506319999695, -1.080864429473877, -0.6183936595916748, -0.6813691258430481, 1.939488410949707, 0.5849130153656006, 1.0498862266540527, -0.7166547775268555, 0.8326146006584167, -0.9635204672813416, -0.4098866581916809, -0.8080538511276245, 0.2527202069759369, -0.37903496623039246, -1.5703219175338745, 0.45117783546447754, 0.9180184006690979, -0.9391998052597046, 2.117412805557251, 0.967840313911438, 0.8183212876319885, 0.5301083922386169, 0.9720124006271362, 2.219857692718506, -0.2522571086883545, -1.2319270372390747, 0.9293309450149536, -0.536780059337616, 0.3594593107700348, -2.197634220123291, 1.1895052194595337, 1.8357990980148315, 0.4351872205734253, 0.3608623743057251, 2.235409736633301, 0.67044997215271, -0.6722913384437561, -0.5294797420501709, -0.4790891110897064, -0.364743173122406, -1.9868234395980835, 0.9558333158493042, 1.3442574739456177, 1.0388963222503662, -1.7254502773284912, -0.26420778036117554, 1.1446921825408936, 1.14542555809021, -0.9557127952575684, 0.470839262008667, 0.08368462324142456, -0.06460176408290863, -1.275821566581726, -1.4177892208099365, -0.7998809814453125, 1.4893105030059814, -0.44940638542175293, 0.7915375232696533, 0.8275030255317688, 0.2680474519729614, -0.4450843632221222, 1.1525590419769287, -0.6995762586593628, -0.5597912669181824, 0.3322753608226776, -0.3187505602836609, 0.5298389196395874, -0.684621274471283, -1.6629911661148071, -0.1366174966096878, 0.31561601161956787, 0.06978106498718262, -0.1458386480808258, -1.1478321552276611, -2.6124823093414307, -0.04560696706175804, -0.5041491985321045, -0.08416715264320374, -1.7040457725524902, -0.25431978702545166, 0.06929504871368408, 0.4885826110839844, 2.0750486850738525, -0.2705823481082916, -0.6983604431152344, 0.43317410349845886, -0.8976645469665527, 1.5056511163711548, 1.533238410949707, -0.028775285929441452, -1.3194645643234253, -0.8220648169517517, 0.5728832483291626, -0.4780273139476776, -0.435104638338089, -2.4061083793640137, -0.3986802399158478, -1.3750511407852173, 0.3963873088359833, -0.13009542226791382, 0.10412224382162094, 1.0696779489517212, 0.024510007351636887, -1.0973176956176758, 0.6529585719108582, -0.174359992146492, -0.21916308999061584, -0.1128271222114563, -1.7337093353271484, 0.504124641418457, 0.06259767711162567, 0.1297248750925064, -0.059376224875450134, 0.9365047216415405, 0.4980486333370209, -0.978187620639801], [1.066830039024353, -1.7914955615997314, -1.6325786113739014, 1.564369559288025, -0.43839383125305176, -0.918944776058197, -0.5641765594482422, -1.3796597719192505, 1.726750373840332, -0.010438780300319195, -0.5487359762191772, -0.2281240075826645, -0.030498893931508064, 0.9345471262931824, -0.28188472986221313, 1.9101338386535645, -0.09395971149206161, -0.4672224521636963, -1.5466727018356323, -0.6693848371505737, 0.5736750364303589, -0.9757494926452637, -0.8497495055198669, 0.7359106540679932, -0.07490532100200653, 0.37227165699005127, -0.26474809646606445, 0.3034211993217468, -1.1134651899337769, 0.5216492414474487, -0.809171736240387, -0.33866873383522034, 0.7987568974494934, 0.5640533566474915, -0.4218069911003113, -1.226924180984497, -0.405360609292984, -2.0981948375701904, 0.9524086713790894, 0.7307517528533936, 0.4373622238636017, -1.004412293434143, -0.999988317489624, -1.6342233419418335, -0.5258855223655701, 0.3865596055984497, 0.6085698008537292, 1.689245581626892, 0.34410908818244934, 0.6408107876777649, -0.370318740606308, 1.131896734237671, -0.2900101840496063, -1.1249440908432007, -1.1577637195587158, 0.9039694666862488, -1.4031875133514404, -1.0121150016784668, 0.46195197105407715, 0.7479794025421143, 1.5775387287139893, -1.2565500736236572, 0.024938788264989853, -1.2667840719223022, -0.702786386013031, 0.12435322999954224, -1.2601873874664307, 1.9976229667663574, -0.8443830609321594, 0.06737501174211502, 0.44606855511665344, 0.19167043268680573, -1.213423490524292, -0.18958058953285217, -0.3332076370716095, 0.49614831805229187, 1.709997534751892, 1.2245473861694336, 0.08854120969772339, -2.0654659271240234, -0.09975307434797287, -2.682443857192993, -0.8822256326675415, -1.3511072397232056, -0.5008409023284912, 1.2293630838394165, -1.3583788871765137, 0.926009476184845, -0.3038007318973541, -0.8880168199539185, -0.16273675858974457, 0.3737207353115082, -0.7021105289459229, -0.1919390708208084, 0.21458080410957336, -0.47678446769714355, 2.3336403369903564, -1.3946624994277954, -1.4854810237884521, -0.25291287899017334, -0.1792858988046646, -0.1104218065738678, -2.294740676879883, -1.4746763706207275, 1.7738759517669678, 1.195359230041504, -0.3246498703956604, 0.8812674283981323, -1.017518401145935, -0.3560405969619751, 0.6859111189842224, -0.11442749202251434, -1.151960015296936, 1.799132227897644, -1.1657105684280396, 0.17416103184223175, 0.40063145756721497, -0.8408738374710083, 1.3761450052261353, -0.5385553240776062, -1.806075096130371, -0.7241752743721008, -0.571101188659668, -0.0533825159072876, 1.799705982208252, -1.5226812362670898, -1.7215187549591064, 0.11703462898731232, 0.9342247247695923], [-0.8442840576171875, 3.0213165283203125, 0.314114511013031, 0.5188947916030884, -0.2816217839717865, -0.4789305329322815, -0.25046032667160034, 1.2138640880584717, -1.6644647121429443, -0.18554094433784485, -0.5275722146034241, -0.9545840620994568, -0.08611937612295151, 2.11665415763855, 0.31944701075553894, 0.7812657952308655, 0.07982182502746582, -2.2002336978912354, -0.9674835801124573, 0.5671267509460449, 0.7280760407447815, 0.58514803647995, 2.1269936561584473, -1.5338640213012695, -1.0945671796798706, -0.8390969038009644, -0.47277671098709106, 2.018364667892456, -1.5500365495681763, 1.6341583728790283, 2.2900381088256836, 1.3229330778121948, -0.18860435485839844, -0.15934130549430847, 0.046582967042922974, 0.16327427327632904, -1.5619791746139526, -0.7450119256973267, 0.8624923825263977, -1.9273700714111328, 0.5941828489303589, -1.3931314945220947, 1.171128749847412, -0.689499020576477, -2.104741334915161, -1.0097936391830444, -0.30593734979629517, -0.49410101771354675, 0.07321973890066147, 0.1635875105857849, 0.73918217420578, -0.16141916811466217, 0.24936184287071228, 2.08675217628479, 0.7633101940155029, 1.918798804283142, -0.9897288084030151, 0.5063073039054871, 0.661509096622467, -1.1604751348495483, -0.17522886395454407, 0.5898398756980896, 1.4596564769744873, 0.14826363325119019, 0.8013227581977844, -0.9798199534416199, 1.0094387531280518, 0.09204339236021042, 1.310913324356079, -0.7510308623313904, -0.7950149178504944, -1.1443889141082764, -1.3137762546539307, 1.2111793756484985, -0.0424010306596756, -0.11876586824655533, 0.35635650157928467, 1.5090904235839844, 1.319601058959961, 0.1462455838918686, -0.19165942072868347, 0.5421265363693237, -1.3221598863601685, -0.9524480104446411, 0.7611054182052612, 0.20335721969604492, -1.0600258111953735, 1.2403801679611206, 0.8334910869598389, -0.32137835025787354, 0.48992159962654114, 1.4266397953033447, 1.1622817516326904, 0.38707560300827026, 2.1605608463287354, 2.114816188812256, -0.7937814593315125, 2.0401315689086914, -1.033610224723816, 0.6647851467132568, -1.513731598854065, -0.6167275309562683, 0.13367010653018951, 0.5016196370124817, 0.9164749979972839, 0.5636071562767029, -2.401212692260742, -0.7629414200782776, -0.26354363560676575, -0.37807703018188477, 0.5783726572990417, 0.649089515209198, 1.1050071716308594, 0.7709274291992188, 2.121473789215088, -0.40336665511131287, -0.4450366497039795, 0.20157407224178314, -0.45822107791900635, 0.5456743836402893, -0.5498728156089783, 0.8355077505111694, 0.10462689399719238, 1.408393144607544, 0.4167381525039673, -1.3229907751083374, -1.0066519975662231, -0.18104906380176544, -0.40674838423728943], [0.1691773533821106, 0.9071664810180664, -0.055134303867816925, -0.7031993269920349, -0.7365577220916748, 0.3886354863643646, 1.2405850887298584, -1.7819191217422485, -0.7266040444374084, -1.158811092376709, -0.5504639148712158, -0.4111205041408539, 0.21032510697841644, -0.15198826789855957, -0.4891130328178406, -0.6030763387680054, 1.1205440759658813, 1.2596089839935303, 1.3006682395935059, 1.015699863433838, -1.009545922279358, 1.1918543577194214, 1.0805739164352417, 1.1382126808166504, 1.1893349885940552, 0.9824026823043823, 0.8089364171028137, -0.03973878175020218, -0.09828956425189972, -0.16714686155319214, -0.41706520318984985, -0.8682869076728821, 1.7214710712432861, 0.8204243183135986, 0.8972724676132202, 0.4357766807079315, -0.3725280463695526, 0.13661065697669983, 2.214658260345459, -0.2169848531484604, 1.1016267538070679, -0.37093886733055115, 0.5277449488639832, 0.1750756800174713, -0.35509997606277466, 0.0436369813978672, -1.5678224563598633, -0.02356983907520771, -0.5859222412109375, 2.030717134475708, -0.2908819019794464, 0.9057351350784302, -0.07235772162675858, -0.01990961842238903, -0.5426115393638611, -2.1129770278930664, 0.5062345862388611, 0.5246273279190063, 1.1808758974075317, -0.17781341075897217, 0.06942511349916458, 0.18793417513370514, 2.5142385959625244, -0.41122967004776, -0.732833743095398, 0.8107680082321167, 0.4509460926055908, 1.884603500366211, -2.2246081829071045, -0.8717063069343567, 0.07197306305170059, -2.0903286933898926, 0.632888674736023, 1.875440001487732, 2.392392158508301, 1.6620843410491943, -0.09039080142974854, -1.206385850906372, -1.023322343826294, 0.034055475145578384, -0.9034214019775391, -0.16742929816246033, -0.4832760691642761, 1.1283100843429565, -0.3881007432937622, -1.2625925540924072, 0.18872025609016418, -0.3659629821777344, 1.1014723777770996, 2.1410253047943115, -1.4358898401260376, -0.5107832551002502, 0.6126012206077576, -0.8934215903282166, 1.0496296882629395, 1.4839203357696533, -0.9312317371368408, 0.3244197964668274, 1.8476109504699707, 0.03800003230571747, -1.873740315437317, 1.494631290435791, 0.3354807496070862, 1.6487834453582764, 1.7106406688690186, 1.280910611152649, -0.29253655672073364, -0.4829390048980713, -1.9259012937545776, -1.2052415609359741, 0.5942248106002808, 0.11411871016025543, 0.5221134424209595, 1.5892002582550049, -0.21713663637638092, 1.1588562726974487, -0.740746796131134, 0.07066196948289871, 0.35628828406333923, 0.11576872318983078, 0.31643426418304443, 0.08844726532697678, -0.6980217695236206, -0.27344489097595215, -0.1890161782503128, -1.2509814500808716, -0.17583389580249786, -0.7268538475036621, 0.053112465888261795], [0.7108624577522278, 0.9976536631584167, -0.5293620228767395, 0.19225788116455078, -0.1675364077091217, -1.4273685216903687, 1.4517630338668823, 0.9409995079040527, -2.720227003097534, 0.18269005417823792, -1.6622989177703857, -1.3446511030197144, -0.9264923334121704, -1.492919683456421, -0.1587659865617752, -0.3142640292644501, 1.8302271366119385, -0.7629590630531311, -0.45581743121147156, -1.5631831884384155, -0.9219803810119629, 0.6713469624519348, 1.1726242303848267, -2.055633306503296, -2.059490442276001, -0.06860212236642838, 0.30761563777923584, 0.6811361908912659, -0.4499467611312866, 0.42362508177757263, -0.5122442245483398, 2.663378953933716, -0.6241466403007507, 0.0591588169336319, 0.20986993610858917, 0.35520708560943604, 0.823671281337738, 0.8450362682342529, 0.10766152292490005, 0.705220103263855, 0.28445085883140564, -1.2310211658477783, -1.6747537851333618, -0.6392741799354553, -2.868041515350342, -0.08755294978618622, -0.020725851878523827, -0.10003280639648438, -1.4206562042236328, 0.35265466570854187, 0.7538644671440125, -0.03873371332883835, -0.6698572039604187, 1.4035446643829346, -1.5880825519561768, -1.258089542388916, 0.43932899832725525, -1.2570163011550903, -0.4796534776687622, 0.3769984841346741, -1.1991993188858032, 0.3840367794036865, 1.3651503324508667, 0.39422842860221863, 0.767177402973175, 0.6545279622077942, 1.106692910194397, 0.43404585123062134, -0.39149045944213867, -0.9233399629592896, -0.48897650837898254, -1.8744651079177856, -0.3556056320667267, -0.11824249476194382, -0.9417059421539307, 0.23996375501155853, -1.458742618560791, 0.2629881501197815, 1.3224674463272095, -0.062288567423820496, -0.5752649307250977, -0.7893886566162109, 0.08426336199045181, 0.608752429485321, -0.5253291726112366, 1.5003068447113037, -1.1631672382354736, 1.1774224042892456, 0.02050900273025036, -0.02437550202012062, -1.3507076501846313, -1.0288273096084595, -1.149835228919983, -0.7058950662612915, 0.863434910774231, 0.4445626437664032, -0.322305828332901, 0.5878142714500427, -0.11682911217212677, -0.9043114185333252, 0.7712721824645996, 1.095348596572876, 0.4980238378047943, -0.7205922603607178, -0.7052983641624451, -0.11911895871162415, 0.6134496331214905, 0.5894266963005066, -0.6542915105819702, 0.8507513403892517, -0.7353422045707703, -0.5544491410255432, -0.9605207443237305, 0.3902342617511749, 0.498102605342865, -0.9113160967826843, -0.7611357569694519, 0.8368699550628662, -0.5483266711235046, -0.8031200170516968, 1.0677088499069214, 0.34803831577301025, 0.707048237323761, -1.1865087747573853, 1.344308853149414, 0.6408123970031738, -0.4085451662540436, -0.42182305455207825, 1.6646660566329956], [-0.5276699662208557, 0.09983434528112411, 0.0287027508020401, 0.1269713193178177, 0.6471580266952515, 0.32024282217025757, 0.8821784257888794, 1.500206470489502, -1.3282153606414795, 1.2376797199249268, 1.7524486780166626, 1.0727626085281372, -0.3564007580280304, 0.5806772708892822, 0.7131861448287964, -0.7161291241645813, 0.3367828130722046, 0.46752414107322693, -0.2915988862514496, -1.1733988523483276, -0.5572056770324707, -1.0378682613372803, -0.6550294756889343, 0.7588396072387695, 1.685192584991455, 0.1007707491517067, 1.412936806678772, 1.2414085865020752, -0.5277193188667297, 0.8569273948669434, -0.8394805788993835, 0.09271997213363647, -2.6174333095550537, 0.14592503011226654, 1.22492253780365, 0.4516182839870453, -0.2129175364971161, -0.6471149325370789, 0.4097972810268402, -0.15412414073944092, -0.08974432945251465, -1.1751480102539062, -0.7730081081390381, -0.5483640432357788, 1.132148265838623, -0.12623950839042664, -0.2543022930622101, 0.7807502150535583, 0.47151657938957214, 0.3329016864299774, -1.3446637392044067, -2.521474599838257, -0.7631335854530334, -2.2520644664764404, -0.0346098430454731, -0.7993295192718506, -0.7519598603248596, 1.414044976234436, -1.554686188697815, 0.5142483711242676, -0.4082815647125244, 0.3629671633243561, -0.013677090406417847, 1.5235822200775146, 0.14233070611953735, -0.08537153154611588, -0.5660116672515869, 0.6228371858596802, 0.03753422573208809, -1.4300304651260376, 0.22393548488616943, -0.14744839072227478, 1.713337779045105, -1.4025557041168213, 0.05935463681817055, -1.3854464292526245, -0.3138728439807892, 1.4231444597244263, 1.392171025276184, -2.0635101795196533, 0.3802299499511719, 1.4300572872161865, 1.5490387678146362, 2.3761401176452637, -1.9671998023986816, 0.6571056246757507, 0.9638313055038452, 0.9288336038589478, -0.5242865085601807, 0.3011495769023895, 1.5114355087280273, 0.30432984232902527, -0.4866918921470642, -2.2992396354675293, -0.07704325020313263, -1.6708101034164429, -1.907715082168579, 1.2253576517105103, -2.1514599323272705, -0.9588603973388672, -1.069977045059204, -1.6135516166687012, -0.4181138277053833, 0.03550081327557564, -0.8728442192077637, -0.30564841628074646, 1.6228832006454468, 0.7743055820465088, 0.9988997578620911, 0.6200560331344604, -1.073547124862671, -0.1654481440782547, 0.28450480103492737, 0.11989381164312363, 0.026164257898926735, 0.0492069274187088, 1.2529932260513306, 0.44616857171058655, 0.8558357357978821, -0.3780834674835205, 1.2682843208312988, 2.3260436058044434, -0.728216826915741, 0.04586714133620262, -1.0521197319030762, 2.3089301586151123, 1.3629344701766968, -0.4791087806224823, 0.4564068913459778], [1.1652413606643677, 0.6033939123153687, -0.2664487361907959, -0.12381967157125473, -0.4909406900405884, 0.5549346804618835, 0.4945047199726105, 0.6255446672439575, -0.41853246092796326, -1.8111093044281006, 1.1176570653915405, 1.2888400554656982, 0.7718713879585266, -0.3116879463195801, 2.3052008152008057, -1.3133602142333984, -0.056356754153966904, -0.33687272667884827, -0.3842955529689789, 0.6504144668579102, -0.14896729588508606, 0.7074254751205444, 0.21685980260372162, 1.8357408046722412, 0.04073987528681755, 0.10457704216241837, 0.9809137582778931, 1.22907555103302, 0.21500694751739502, -0.2167568951845169, -1.280127763748169, -0.20613746345043182, -0.4680754840373993, 0.4591166377067566, -0.1532781422138214, 0.24058294296264648, 1.3681843280792236, -1.1192965507507324, 0.25485801696777344, 0.8802148699760437, 0.20872119069099426, -1.0383937358856201, -0.9515467882156372, -0.34140118956565857, -0.5078074932098389, 0.2014906108379364, -0.5287234783172607, 0.8671627044677734, 0.944678544998169, 0.0507052019238472, 1.0834325551986694, 1.9185926914215088, -0.5162297487258911, -0.08665372431278229, -0.7968679666519165, -1.5455164909362793, 0.6226556301116943, 0.23211240768432617, 0.6461375951766968, -0.21875062584877014, 0.607639729976654, 0.782890796661377, 0.8643587827682495, 1.2978990077972412, -0.9467629194259644, 0.3917889893054962, 0.9926548004150391, -0.5210222601890564, 0.5080229043960571, -0.20452632009983063, -1.3892285823822021, 0.8239898085594177, -0.3130401074886322, -1.2327022552490234, 0.4944664537906647, -0.6504815220832825, -0.14975838363170624, 0.09872002899646759, 0.5339347720146179, -0.47044041752815247, 0.9824144244194031, 1.3141627311706543, -0.1795981079339981, 2.0537993907928467, -0.5634801983833313, -0.18781104683876038, 0.4011368751525879, -2.8056201934814453, -0.8105306625366211, 1.5100504159927368, 0.05625050514936447, 1.2330858707427979, -0.6548508405685425, 0.03108268231153488, 1.1636497974395752, 1.2768614292144775, 0.6014235019683838, -1.5436336994171143, 0.4413738548755646, 0.4817126989364624, 1.3290023803710938, 1.4982264041900635, -3.8803439140319824, 0.26444709300994873, 0.6570512056350708, -1.0080755949020386, -0.5386478900909424, 1.1570637226104736, 0.06460192054510117, 0.13701818883419037, 0.20421405136585236, -0.7330750226974487, -0.43646612763404846, 1.3848947286605835, -2.536527395248413, -0.13629235327243805, -0.7962936162948608, 1.603244662284851, -0.6130182147026062, -0.8363078832626343, -1.1516743898391724, 0.34271958470344543, 0.3535919785499573, -0.470675528049469, 0.6453379988670349, 0.03261561691761017, -0.24695159494876862, 1.2904636859893799, 1.0927537679672241], [0.4042190611362457, 1.076350212097168, -0.4168349504470825, -1.3961113691329956, 0.011288237757980824, -0.8405776619911194, -0.4822080731391907, 1.428349494934082, -1.793162226676941, -2.253770589828491, 0.055846087634563446, -0.2950715720653534, 0.5863623023033142, 0.6939144730567932, 2.2210605144500732, -0.277749240398407, -0.8174831867218018, 0.5006451606750488, 1.3233838081359863, 0.24548229575157166, -0.31429046392440796, -1.6598789691925049, 0.4854409098625183, -1.886641263961792, -0.4683115780353546, 1.0803864002227783, 0.16298478841781616, -0.9339563846588135, -0.7708939909934998, -0.7769344449043274, -0.3576608896255493, 0.5201813578605652, -1.2526463270187378, -0.2795811891555786, -0.8628249168395996, -0.6825432181358337, 0.5780444145202637, -2.946340560913086, 0.005400700028985739, 0.5081165432929993, -0.15358252823352814, 1.6327844858169556, 0.28987592458724976, -0.6429981589317322, -0.36917802691459656, 1.4813312292099, -1.7020862102508545, 0.2815859615802765, -1.332757830619812, 0.9033781886100769, 1.4989421367645264, 1.0770431756973267, -0.2750975489616394, -1.3161780834197998, 1.0087924003601074, 0.1773015856742859, 0.22901509702205658, 0.16595754027366638, -0.8782759308815002, -0.47835877537727356, 0.11526918411254883, 1.509701132774353, 1.0859349966049194, 1.3232985734939575, -0.6565787196159363, 0.5346770882606506, -0.28890177607536316, 0.24695061147212982, 1.065373182296753, 0.1555582880973816, 1.6654613018035889, 0.2078080177307129, 0.6167568564414978, 0.8134568929672241, -1.551109790802002, -0.027300510555505753, 0.6891576051712036, -0.6762546896934509, 0.9480147957801819, 0.4819076955318451, 0.03031274303793907, 0.20049843192100525, 0.8089264631271362, -0.31964313983917236, 0.9377871155738831, -0.10693621635437012, 1.2600152492523193, -0.611865222454071, -1.1183220148086548, -1.6059987545013428, 0.4621482789516449, 0.9330151677131653, -1.4887334108352661, -1.3192434310913086, 0.217489093542099, -0.2502090036869049, -0.03354443237185478, -0.15753573179244995, -0.6877924203872681, -0.3718945384025574, 0.5211791396141052, -0.2636777460575104, -0.8232438564300537, 0.5955420732498169, 0.8989790081977844, -0.9683755040168762, 0.11195216327905655, 1.3862435817718506, 0.19925865530967712, -0.2887325882911682, -0.06244247406721115, -0.9638824462890625, -0.6061149835586548, -0.12288440018892288, 0.6161046028137207, 0.3143530786037445, -0.05906899645924568, 1.6455000638961792, 1.4863471984863281, 1.0906908512115479, 0.3648184537887573, -0.7083373665809631, 0.7787752747535706, -0.9681137204170227, 0.21995662152767181, 1.712138295173645, 1.5587157011032104, -0.92369544506073, -0.4741794466972351], [0.06376181542873383, -0.9095898270606995, -0.14852412045001984, 0.13637132942676544, -0.13284388184547424, -0.8114088177680969, 0.594910204410553, 1.3562968969345093, 0.7017173767089844, 0.7235361337661743, 1.1393526792526245, -0.5232356190681458, 0.9370411038398743, 1.0282251834869385, -0.23337723314762115, -0.952810525894165, 1.0976020097732544, 0.3012891113758087, -0.5264775156974792, -0.11828607320785522, -0.12503889203071594, 0.5853099226951599, -0.629395604133606, -0.09646888822317123, -0.472434401512146, 1.4611161947250366, -0.7304904460906982, 1.579603910446167, 0.5019536018371582, 0.05931571125984192, -1.227441668510437, 1.8935484886169434, -1.7059000730514526, -0.6678597331047058, -0.8613607883453369, 0.6030076146125793, 0.5920365452766418, 1.2913833856582642, -0.9534645080566406, -0.11438585072755814, 0.3576653003692627, -0.14794421195983887, 0.15320177376270294, -0.3468282222747803, 0.6840204000473022, -0.835741400718689, 2.0383050441741943, 0.6576390266418457, -0.8723894357681274, -0.4765032231807709, 0.6676008701324463, 0.19136925041675568, -0.6568009853363037, -1.0456231832504272, -0.23551255464553833, 0.07352850586175919, 1.0989655256271362, -0.08512996137142181, -1.4710193872451782, 0.4292673170566559, 0.8614289164543152, -0.23347380757331848, -0.1314857453107834, -1.1720607280731201, -2.4582955837249756, -1.7132331132888794, 1.1459373235702515, 1.4370390176773071, -1.7906306982040405, -1.2953778505325317, 0.9552661776542664, 2.446546792984009, -1.0489979982376099, -0.4296344220638275, -1.3608158826828003, -1.1167815923690796, 0.07595868408679962, 0.13589949905872345, -0.6361685395240784, 0.7793189883232117, -0.6952205896377563, 0.6585094332695007, -0.8719797730445862, -0.0857977420091629, 0.4614611864089966, 0.7219070196151733, -1.2475481033325195, 0.7742723226547241, -0.7737137079238892, 0.3174456059932709, 0.1436137557029724, 1.2490522861480713, 0.6011307835578918, 0.5000977516174316, -0.3180435001850128, 0.826805055141449, 0.8552733063697815, -1.0170258283615112, 1.4733983278274536, -1.1093426942825317, 0.9877565503120422, -0.19271253049373627, 0.3181847333908081, 0.3395715653896332, 0.1266428679227829, -0.09889582544565201, -0.06763789057731628, 0.14798177778720856, -0.3983403742313385, 1.2110848426818848, 1.9571304321289062, 1.0774179697036743, -0.04179280251264572, 0.5839924216270447, 1.2809913158416748, 1.367942452430725, 0.7734108567237854, 0.988234281539917, 0.9646007418632507, 1.550730586051941, -0.30843475461006165, -0.7434877157211304, 1.967457890510559, 1.1684974431991577, -0.29802167415618896, 0.022270632907748222, -0.5234882831573486, -0.17292630672454834, 0.3842342793941498], [2.0229716300964355, 0.07245182245969772, 0.8503187894821167, -0.8306334614753723, 0.4200892448425293, 0.5825695991516113, 1.2923043966293335, 2.0415728092193604, -0.6964444518089294, 1.5935249328613281, -0.01721562072634697, 0.9578870534896851, -0.9111579060554504, 1.6554912328720093, -1.4888083934783936, -0.4509606957435608, 0.8318261504173279, 0.09363839775323868, -0.9963971376419067, 1.411928415298462, -1.743073582649231, 1.476600170135498, 0.6271266341209412, 0.23024578392505646, 0.07202625274658203, 0.31497740745544434, -1.0410650968551636, -0.42635151743888855, 0.20033963024616241, 0.004725382663309574, -1.6839776039123535, 0.17112474143505096, 1.410269021987915, -1.7794981002807617, -0.9053765535354614, -1.702431082725525, 0.19243235886096954, 0.9955462217330933, -0.362737774848938, -1.130850076675415, 0.3477655053138733, 0.3315387964248657, 0.1564815789461136, 0.7785719633102417, 0.41670578718185425, -2.8634448051452637, -0.7785547971725464, 0.5559653639793396, -1.535591959953308, -0.35988545417785645, 0.22553102672100067, -0.05460929498076439, -1.294508934020996, -0.05172468349337578, 0.36516112089157104, 0.6282687783241272, -0.9235994219779968, 0.7406928539276123, 0.01068858616054058, 0.8467193245887756, 0.8201746940612793, 0.43194347620010376, -1.0362777709960938, -0.4622390866279602, 0.3858684301376343, 0.20689484477043152, 0.2394033968448639, -0.14435546100139618, 1.0343091487884521, -0.20030584931373596, 0.5830790996551514, 0.9105151295661926, 0.13753756880760193, -0.08790959417819977, 0.9705656170845032, 0.5847688913345337, -1.3667924404144287, -1.0238378047943115, 0.35122570395469666, 0.17000870406627655, -0.038996338844299316, 0.04526340961456299, -1.456354022026062, 2.950289249420166, 0.12408164143562317, -2.515530824661255, 0.2810046970844269, -1.315438985824585, 0.9708237051963806, -0.39649489521980286, 0.8133566975593567, -0.2390144020318985, 0.5345419645309448, 0.19864504039287567, 1.179699182510376, 2.329665422439575, 0.1493963748216629, -0.6140065789222717, -1.0590747594833374, -0.2875036895275116, -1.0436915159225464, -0.9366571307182312, -0.4124850034713745, 0.6780087947845459, 0.1862514466047287, 0.3283361494541168, -0.21559491753578186, 0.32616904377937317, 1.243608832359314, 2.3049662113189697, 1.151811957359314, 1.9321856498718262, 1.2059524059295654, 1.1556529998779297, -0.4579859673976898, -0.24999138712882996, -1.0232691764831543, 0.036633651703596115, -0.06886501610279083, -0.6731245517730713, -0.7307538986206055, -1.0264053344726562, 0.6541537046432495, 0.03025393933057785, 0.4147981107234955, -1.5033506155014038, 0.16120995581150055, 2.0514156818389893, 1.0991358757019043], [0.102253757417202, -1.1985150575637817, -0.899061918258667, -0.9149914979934692, -0.5495854616165161, 0.2629983425140381, -2.7239928245544434, 0.8141711950302124, -1.830004334449768, 0.07786662131547928, 0.5215775966644287, -1.479556679725647, 1.2071583271026611, -0.44511258602142334, -0.6286179423332214, 0.015175454318523407, 0.764999508857727, 0.9882655143737793, -0.3416324555873871, 0.05579540506005287, 1.4299495220184326, 0.4682452380657196, 0.04656343534588814, -0.25568076968193054, -1.4330087900161743, 0.3523159325122833, 0.06871547549962997, -0.3706616461277008, 0.33494096994400024, 2.3181185722351074, 0.20285704731941223, 0.8500573039054871, 0.47796276211738586, 0.585299551486969, -1.4416844844818115, 1.921698808670044, 1.0611202716827393, 0.728495717048645, 1.0925134420394897, 0.7090063095092773, -1.1981616020202637, 1.0689738988876343, 0.002096428070217371, 0.4103642702102661, -0.034519463777542114, -0.27675408124923706, 0.4486715793609619, -0.12375394254922867, 0.425887793302536, 0.06765881180763245, 0.5221219658851624, -0.584888756275177, -0.7693502902984619, 1.4875836372375488, 1.8936165571212769, -1.1210837364196777, 0.6246052384376526, 0.6917221546173096, 0.0509658083319664, 0.3794155716896057, 1.0851631164550781, -0.7238733172416687, -0.3220808207988739, 0.15678554773330688, 0.39103829860687256, -1.3842134475708008, -0.6798442602157593, -0.07106366008520126, -1.380926489830017, -1.450882911682129, -0.18016493320465088, 0.36781585216522217, -0.9958232641220093, 0.6836647987365723, 0.20870108902454376, -0.9286221265792847, 0.02913454920053482, -1.9429599046707153, -0.13056819140911102, -0.253799170255661, -1.2757705450057983, 1.0478347539901733, -0.23823021352291107, -0.6363402605056763, 0.3706299364566803, 0.1503702849149704, -0.9864977598190308, -0.14223317801952362, -0.09766259044408798, -0.8353021144866943, -0.6148558855056763, -0.42709454894065857, -1.4671202898025513, -0.5083499550819397, 0.6244895458221436, 1.2424019575119019, 0.340196430683136, 0.7593311071395874, -1.1220862865447998, -0.9401341676712036, 0.11570683121681213, 1.686213493347168, -0.5595389604568481, -0.14319683611392975, -0.46024003624916077, 0.10230421274900436, -1.1474608182907104, 1.1661313772201538, -1.0703558921813965, 1.2212977409362793, -1.0869579315185547, 0.1531071811914444, -0.5423768162727356, 0.05435832217335701, -1.3938205242156982, -1.8632375001907349, 0.42901501059532166, -0.7798323631286621, -0.4333253800868988, -1.6961771249771118, 2.237703323364258, 1.3617390394210815, 0.8003882169723511, -1.065663456916809, 1.253908634185791, 0.9644869565963745, -1.5159364938735962, 0.5763549208641052, -0.8609379529953003], [0.08892372250556946, 0.7610030770301819, 0.37850603461265564, 0.41267117857933044, 1.137831449508667, -0.05127320811152458, 0.4148988127708435, 0.6534122824668884, -1.342782735824585, 0.3747332692146301, -0.2570677101612091, -0.5312448740005493, 0.47916924953460693, -0.3292439877986908, 0.8188101649284363, 1.701305627822876, 0.945345938205719, 0.7515896558761597, 0.07666496187448502, 1.8677326440811157, -0.4561808407306671, 1.2154630422592163, 0.3286972641944885, 0.2305690199136734, 0.14505267143249512, 1.623249888420105, -1.5364049673080444, -0.1417822539806366, -0.3450687527656555, -0.6508766412734985, 0.025366339832544327, 1.6327155828475952, 1.6077569723129272, 0.841335654258728, -0.2745816111564636, -0.18277963995933533, -1.5411072969436646, -0.21270208060741425, -0.271775484085083, 0.5776485204696655, -1.597513198852539, 0.695176899433136, -0.9818440079689026, -1.0478343963623047, -1.234214425086975, -1.5159753561019897, -0.11617785692214966, -0.15883652865886688, 1.490447759628296, -0.7040799856185913, -1.0860602855682373, -0.3343724012374878, 0.07606853544712067, -0.8987879753112793, 0.5174955725669861, 1.0519894361495972, 0.36676931381225586, 1.6420373916625977, -1.3940091133117676, 0.5929926633834839, 1.5613906383514404, -0.11112522333860397, 0.20934300124645233, 1.3485857248306274, -0.4574762284755707, -1.4154434204101562, -0.33468690514564514, -0.049663081765174866, 0.20227627456188202, 1.7638605833053589, -0.2992381155490875, -0.6978517770767212, -0.6474250555038452, 1.1439505815505981, 0.4357881546020508, -0.6405029892921448, 1.4461321830749512, 0.5711514949798584, -0.23075473308563232, -0.031043794006109238, 1.016100287437439, -0.10727035999298096, 0.21452556550502777, -1.7458295822143555, -0.5092535018920898, 0.9553031325340271, 0.7748355269432068, 0.23509123921394348, -0.19674807786941528, 0.5135830044746399, -0.970740020275116, -0.8542484045028687, 0.22381629049777985, 1.044697880744934, 1.9763625860214233, -1.1230765581130981, -0.5886406302452087, 1.794439673423767, -0.8127735257148743, -1.048996925354004, -0.8416576385498047, -0.07347677648067474, -0.05153190344572067, -0.349189430475235, -0.908507227897644, 1.2847812175750732, -0.995554506778717, 0.8597585558891296, -0.5155906081199646, -0.15801554918289185, 1.8162332773208618, 0.5390715003013611, -1.4597386121749878, 0.22865834832191467, -0.5019949674606323, -0.9697408080101013, 0.34110918641090393, -0.8231715559959412, 0.15174248814582825, 0.23683048784732819, 0.9765881896018982, -0.012109086848795414, -2.6097939014434814, 0.5941655039787292, -1.8592917919158936, -1.6431466341018677, -0.7527539730072021, -0.2899930775165558, -0.7025552988052368], [0.966899037361145, -0.9048842191696167, 0.5910460352897644, 0.7637057304382324, -1.528668761253357, -1.4924583435058594, -0.7722710967063904, 1.1991745233535767, -1.1036648750305176, 0.9543946385383606, -1.0352153778076172, 2.09832501411438, -0.011857351288199425, -1.4756513833999634, -1.4032244682312012, -0.36693671345710754, -1.6941308975219727, 0.10902157425880432, -1.9893176555633545, -1.8262995481491089, 0.9041123390197754, 0.6871153116226196, -1.6307692527770996, -0.6683416962623596, -0.7885913848876953, 1.0566400289535522, 0.18946011364459991, -0.25644388794898987, 0.4001789093017578, 0.512415885925293, -0.2489299774169922, 1.4438822269439697, 0.5854406952857971, -0.507151186466217, 0.07974661886692047, 1.0596837997436523, -0.44867369532585144, -0.0781024917960167, -0.7442174553871155, -2.5664188861846924, 0.11931204795837402, -0.2248631864786148, -1.050183892250061, -0.07855191081762314, -1.1168959140777588, 1.3345856666564941, 0.33669406175613403, 0.053706564009189606, -0.14083363115787506, 0.12364634871482849, -1.5953094959259033, -1.57760488986969, 0.45005476474761963, 2.348334312438965, 1.5621604919433594, -1.2182551622390747, -0.40885910391807556, 1.1161020994186401, 0.05086319148540497, -1.3836390972137451, -0.5500795245170593, -0.3611007630825043, -0.1905885487794876, -0.7691949605941772, -0.5267069339752197, 0.19135208427906036, 0.3965683579444885, -0.5138328075408936, -0.8706218004226685, 1.061131238937378, 0.3227074146270752, -0.1337074190378189, -0.39044782519340515, -0.4241879880428314, 1.2753735780715942, 0.968586802482605, -0.05938918888568878, 0.8013782501220703, 0.9564907550811768, 0.875264048576355, 0.8926495313644409, 0.476046085357666, 0.10990432649850845, 0.36071330308914185, -1.2805421352386475, -0.14706648886203766, 0.29681074619293213, -1.2217820882797241, -0.1332419067621231, 0.7137998342514038, 0.04793885350227356, 0.327572762966156, -0.9801076054573059, 0.4769172966480255, -0.6385697722434998, 0.1567523330450058, 1.86532461643219, 1.021358609199524, -0.10010673105716705, 1.0252995491027832, -1.3099393844604492, 1.2943624258041382, -1.0020675659179688, 0.5723239183425903, 0.7380656599998474, -0.64287269115448, -0.5636497139930725, -0.05399152263998985, -0.3171547055244446, -0.25687870383262634, -1.1368739604949951, 0.3294171690940857, 1.0453999042510986, -0.03528746962547302, 1.3871409893035889, -1.171606421470642, -0.2575610876083374, -0.27797046303749084, -0.15829962491989136, -0.9139727354049683, 2.280285596847534, -0.19320406019687653, -0.39748454093933105, -1.7381318807601929, 0.07357332855463028, -0.41803914308547974, 1.7218302488327026, -0.005971466191112995, -0.09292401373386383], [-0.3555561304092407, 0.23502816259860992, -1.4682190418243408, 0.9721198081970215, 0.4573718011379242, -0.7543861269950867, -1.350782036781311, 0.2935703992843628, 1.6052459478378296, 1.2730789184570312, 0.7287412285804749, -0.5052490830421448, -1.5276206731796265, -0.9093650579452515, -0.046486638486385345, 2.2967731952667236, 0.3125402331352234, 0.7464969754219055, -0.026319073513150215, -0.044254474341869354, 3.0814931392669678, 0.5470595359802246, -0.5467632412910461, 0.8949988484382629, -1.0227078199386597, 0.5070580244064331, 0.37154024839401245, 0.10438258200883865, -0.2826060652732849, -0.5145378708839417, -0.24182535707950592, 0.9419149160385132, -1.9111279249191284, -1.2665551900863647, 0.6058618426322937, 0.49684977531433105, 1.090224266052246, -0.21004773676395416, 0.09945471584796906, -1.3059453964233398, 0.21431474387645721, -0.5058107376098633, -1.019430160522461, -0.40556490421295166, 0.35001039505004883, -0.6317084431648254, 1.6436971426010132, -0.5195393562316895, 0.31517884135246277, -0.7713267803192139, -1.6655691862106323, -1.4298704862594604, 1.430134654045105, 0.47382089495658875, -0.3390614688396454, 0.15436847507953644, 0.6186942458152771, -0.5246306657791138, 0.27586376667022705, -1.632264494895935, -1.3817873001098633, 1.3303205966949463, 0.10000398755073547, 0.9765546321868896, -0.054741375148296356, -0.3328230679035187, -0.9588810801506042, 0.6252805590629578, -0.08481620997190475, -0.06976280361413956, 0.10970151424407959, 0.0008156675030477345, 1.436173915863037, -1.3156397342681885, -0.08459553867578506, -0.6664388179779053, 0.3433859348297119, 0.25268271565437317, -0.6395220160484314, -0.6284084320068359, 0.6436998248100281, 0.0702139288187027, -0.4771931767463684, -0.3420199751853943, 1.502561330795288, -0.5927038192749023, 0.08933066576719284, 1.733933448791504, -1.5093340873718262, -0.12407561391592026, -0.34692734479904175, 1.173388957977295, 1.0064120292663574, 0.12692250311374664, 0.9361528754234314, 0.06294403225183487, 1.0018129348754883, -2.0165727138519287, 1.5867514610290527, -1.934187889099121, -0.07362031936645508, 0.6268072128295898, 0.8407585620880127, 0.05087059363722801, -0.5375252366065979, 0.5063223838806152, 0.08353357762098312, 0.7634115815162659, -0.06267361342906952, -0.3102927505970001, 0.5774983167648315, 0.14622996747493744, 0.25365978479385376, 2.2023017406463623, 1.7208107709884644, 0.48354795575141907, -0.47562533617019653, -0.9061199426651001, -0.17711377143859863, 1.7584176063537598, -2.106370449066162, -0.8535999059677124, 1.2268555164337158, 0.9875445365905762, -1.5452643632888794, 0.16493940353393555, 0.8865687251091003, -0.5699775218963623, 0.7113336324691772], [0.03762200474739075, 1.8877729177474976, 0.18386730551719666, 0.33437997102737427, 0.8672705292701721, 1.0785855054855347, -0.19055084884166718, -0.7033205032348633, -1.2972784042358398, 1.88083016872406, -1.3360812664031982, -2.7968215942382812, -0.5543056726455688, -0.39760980010032654, -0.5809249877929688, -1.0733506679534912, -0.6633411049842834, -1.4273338317871094, -0.8119732141494751, -0.46464818716049194, -0.22433659434318542, -0.36330562829971313, -1.0674822330474854, -0.5379469394683838, -0.519114077091217, 0.21906021237373352, -0.41778555512428284, -1.682166337966919, 0.2967768907546997, -0.8165166974067688, -0.29304391145706177, -0.5280224084854126, 2.386136770248413, 1.0161232948303223, 0.32088106870651245, -0.6026549339294434, -1.3873783349990845, 2.4232256412506104, 0.7947744131088257, 2.0697181224823, -1.026442289352417, 1.0031559467315674, -0.411593496799469, 1.1424890756607056, -0.2950794994831085, 0.4839504659175873, -0.2933434844017029, 1.8269174098968506, -1.1042308807373047, -0.9221580028533936, -0.966022789478302, -1.2169142961502075, 1.730898141860962, 1.211937427520752, -2.125148296356201, -1.010874629020691, 0.13698936998844147, -0.7404515147209167, -1.6387953758239746, 2.4945905208587646, 1.7020529508590698, 1.9719735383987427, -1.3120182752609253, 2.2586419582366943, 0.3213258981704712, 0.9034937024116516, 0.2593748867511749, -0.500617504119873, 1.4970940351486206, -0.7734931111335754, -0.8156744241714478, 0.11035218089818954, -0.8422473669052124, -1.095362901687622, 0.2586415708065033, -0.4725929796695709, -1.2421650886535645, -0.2562313675880432, -0.8823593854904175, -0.24797701835632324, 0.1390005350112915, 0.1154053807258606, -0.5098248720169067, -0.6020923852920532, -1.3948023319244385, 1.350762963294983, -0.4155964255332947, -0.04625128582119942, 0.7161723971366882, 0.8822006583213806, -2.4931278228759766, 1.1569440364837646, 1.0666186809539795, -0.15483352541923523, 0.4577961564064026, 0.7201015949249268, -0.4786801040172577, -1.420255422592163, 0.42269372940063477, -0.6578934192657471, -0.026242995634675026, 1.3182910680770874, -1.3987007141113281, -0.3637966513633728, 2.6390109062194824, -1.1724262237548828, 1.9226348400115967, -0.8246371150016785, -1.33207368850708, -0.8838138580322266, 0.7354546189308167, -0.008908689953386784, -0.4152888357639313, 0.2884322702884674, -0.6199923157691956, -0.05833171680569649, 0.6781570315361023, 0.5866313576698303, 0.8298989534378052, 0.4825828969478607, 0.6944928169250488, -0.19138559699058533, 0.6192692518234253, -0.32988137006759644, 0.6577258110046387, -0.3031119406223297, -0.17446111142635345, -0.1802283525466919, -0.01169342827051878], [0.9353656768798828, 0.6266087889671326, -1.283976435661316, -0.6742213368415833, 0.5515915751457214, 0.7955926060676575, 1.507359504699707, 0.7832887172698975, 2.227682590484619, 0.3675919771194458, 0.44602227210998535, 0.7241290807723999, 0.40952059626579285, 1.8170453310012817, -1.1650899648666382, -1.8819838762283325, 0.7774367928504944, 1.974940299987793, 2.502427816390991, -0.599435567855835, 1.2832821607589722, -1.3945484161376953, -0.002085904125124216, -0.5400702953338623, 1.3144879341125488, -1.2051664590835571, -0.47003620862960815, -0.5133379697799683, 1.2356833219528198, -0.2712624669075012, -0.41309577226638794, -0.19227829575538635, -0.24332597851753235, 0.9687135815620422, 1.094494104385376, -0.5283772945404053, -0.3233417272567749, 0.612879753112793, 1.0559436082839966, 0.6526922583580017, -1.1517735719680786, 1.244242548942566, 1.5921331644058228, 1.0911651849746704, 0.0699676051735878, -0.03005124442279339, 0.7871878743171692, -0.7763415575027466, -0.6388821005821228, -0.5312426686286926, 0.5458645224571228, 1.1615194082260132, 0.8368450403213501, 1.2013386487960815, -0.034981369972229004, -0.6184248924255371, -0.55411696434021, 1.1607372760772705, 0.19649071991443634, -1.572903037071228, 0.861242949962616, 1.7028896808624268, -0.47436800599098206, -0.5660607814788818, -1.045261263847351, 2.3430891036987305, 0.015093703754246235, 1.575278401374817, -0.39125388860702515, -0.4435097277164459, 0.6326650381088257, 1.1141999959945679, 0.22022593021392822, -0.2867729365825653, 0.079012930393219, -0.8106356263160706, -0.10527760535478592, -0.8147522211074829, -0.21536697447299957, 0.5265777111053467, -1.4911373853683472, 1.2657688856124878, -1.616329550743103, -1.4584957361221313, 0.9261998534202576, 1.6787474155426025, -0.4856395721435547, 0.6898516416549683, 0.18159855902194977, 0.9149657487869263, 2.7635529041290283, 1.1308621168136597, 0.2888336777687073, -0.8514600992202759, 0.22712275385856628, -1.4253435134887695, 0.568735659122467, -0.24778993427753448, -0.0849849283695221, -1.9170371294021606, -0.6459775567054749, 2.104785203933716, 0.7526625990867615, 0.24699287116527557, -1.8972009420394897, -0.7401039600372314, -1.6875640153884888, -1.2158023118972778, 0.5115833878517151, -1.1090575456619263, 1.6468998193740845, -0.7778374552726746, 0.2895366847515106, -1.3728736639022827, 0.5223673582077026, -1.3761597871780396, -1.7147542238235474, -0.4726012647151947, -0.8204740285873413, 0.9912477731704712, 1.013741135597229, -0.9700267910957336, -0.014000438153743744, -1.326968789100647, -1.746368408203125, -0.5656173229217529, 0.5935261845588684, 1.6590380668640137, -0.3521512746810913], [2.5325913429260254, -0.3724987506866455, -0.3785039484500885, -0.8246501684188843, 0.3728313148021698, -1.8892992734909058, 1.4864963293075562, 0.3029768466949463, -0.9333328604698181, -0.09712006151676178, -1.8508344888687134, 0.15475547313690186, 0.7355511784553528, -1.901623010635376, -1.0558634996414185, -0.30524200201034546, -0.8553050756454468, -0.0015708350110799074, 0.8049077987670898, -1.1737264394760132, -0.23703646659851074, -2.6000771522521973, -0.5080479383468628, -0.09880374372005463, 0.9717052578926086, 0.9873268604278564, 0.09792399406433105, 1.0953395366668701, 1.643484115600586, -2.5295627117156982, 1.0326402187347412, 1.1627733707427979, -1.6356440782546997, -0.5373950600624084, -0.5883591771125793, -0.16575129330158234, -1.9743906259536743, 1.1864069700241089, 1.3190827369689941, 1.00200355052948, -0.49852168560028076, 0.13388432562351227, 0.8950919508934021, 0.7650332450866699, -1.2044285535812378, 2.091071128845215, -0.3174646198749542, -0.26181846857070923, -1.712375521659851, -1.287003993988037, 0.7865085005760193, -0.20795737206935883, 0.2955780327320099, -0.24912995100021362, 0.06189689040184021, -0.2119937241077423, 1.2050704956054688, 0.3568635880947113, -1.3676296472549438, 0.7444932460784912, -0.0744936540722847, 0.3480722904205322, -0.19226063787937164, -1.315313696861267, -0.7378204464912415, -1.1225061416625977, -0.05164458975195885, -1.7884804010391235, 0.3737107515335083, 0.9850391149520874, 0.6735047698020935, -0.30495667457580566, -1.7199606895446777, -0.3173372149467468, 1.4815833568572998, 1.3180735111236572, -0.9671390652656555, 0.09515709429979324, -1.2868047952651978, -0.6531010866165161, -0.07703649997711182, 1.4038561582565308, 0.5719133615493774, 0.4933936595916748, 0.2981385886669159, 1.3588851690292358, -0.38388240337371826, -1.2876174449920654, -0.4708530008792877, 0.22681643068790436, -1.223130464553833, 0.8333106637001038, 0.4079097509384155, 0.7584691047668457, 0.10294656455516815, 0.6713059544563293, -2.3405227661132812, -1.2540125846862793, 0.022186215966939926, -0.6205066442489624, -0.2147969752550125, -0.8325830101966858, 0.3221265971660614, -0.40527474880218506, 0.09164594858884811, 0.2639507055282593, 0.005197546444833279, -0.3902120888233185, 0.28611430525779724, 1.716953158378601, -0.42997756600379944, 0.042385540902614594, -1.1492396593093872, 1.1980538368225098, -0.5118740200996399, -1.303643822669983, -0.007483355235308409, -0.17801529169082642, -0.5715528130531311, 0.6480851769447327, 0.6738648414611816, -1.3250131607055664, -0.9376981854438782, 1.1894636154174805, -0.9256313443183899, -0.5441394448280334, 0.3389589786529541, 0.5795406103134155, -0.0876518115401268], [0.5449743270874023, -0.9495327472686768, -0.03970402106642723, -1.0277132987976074, 0.4871516227722168, -0.6669215559959412, -0.9718447923660278, -0.30494073033332825, -0.2930683195590973, 0.7310714721679688, -0.30194804072380066, -0.5685621500015259, -0.7508508563041687, 0.5072959661483765, -0.6940532922744751, -0.39075329899787903, -0.7727354764938354, 1.3125431537628174, -0.9957155585289001, 1.7601468563079834, 0.6766120195388794, 1.678022027015686, 0.17304450273513794, 0.7219814658164978, 0.5335906744003296, 0.4964525103569031, 0.7756804823875427, -0.4208000600337982, -1.7434160709381104, -0.20857085287570953, 0.6113133430480957, 0.5273534655570984, 0.3016875386238098, 0.04815433546900749, -1.3126941919326782, -0.9632745981216431, -0.5785763263702393, 0.7368495464324951, -0.4359849989414215, -0.876052737236023, -1.1973873376846313, 2.368046522140503, 1.5332087278366089, -0.20768266916275024, 0.4918428063392639, -0.19511845707893372, -0.3096040189266205, 1.1482361555099487, 0.25747987627983093, -1.7757456302642822, -2.302711248397827, -0.9554678201675415, -0.9912465810775757, 0.572735071182251, -0.8148297071456909, 1.2112414836883545, 0.8476125001907349, 0.6580097675323486, 0.6004517674446106, 1.2857348918914795, -1.4554505348205566, 1.96180260181427, 0.19624678790569305, -0.38232389092445374, 0.5036942362785339, -0.032161250710487366, -0.8417893052101135, -1.2392055988311768, 2.645305633544922, -1.5680540800094604, -0.5432270765304565, -1.101777195930481, -0.5497279763221741, 0.665358304977417, -0.07593950629234314, 0.4712739586830139, -1.7645244598388672, 0.08315777033567429, 0.023408876731991768, -0.1331796497106552, -1.0935962200164795, 1.5475380420684814, 0.5624733567237854, -0.19750574231147766, 0.3634883463382721, 1.3879780769348145, 0.783035159111023, -0.5713260173797607, -1.0030136108398438, -1.8218168020248413, 2.012458562850952, 1.0007880926132202, 0.9118486046791077, -0.307994544506073, -0.5592302680015564, 1.5573426485061646, -0.05976079776883125, 1.827373743057251, -0.31178590655326843, -0.036679718643426895, -0.35114359855651855, 0.8533861041069031, 0.748397171497345, -1.350584864616394, -1.0598480701446533, 0.00950135849416256, 0.710675835609436, -0.1540907621383667, -0.8469409942626953, 0.8346443176269531, 0.7109033465385437, -1.2929688692092896, 0.1164570152759552, -0.10913023352622986, 0.5602532029151917, 0.49810338020324707, 0.29469552636146545, 1.6085388660430908, -0.16546796262264252, 1.7417086362838745, -0.25685325264930725, 0.9296154379844666, 0.9665951132774353, -1.4118379354476929, 1.7177119255065918, 0.6660676598548889, 0.8850982189178467, -0.8085920810699463, -0.10595622658729553], [0.17589914798736572, -0.03598055988550186, 0.032919131219387054, -0.35127773880958557, 1.7516162395477295, 2.974679946899414, 0.17718824744224548, 1.5606064796447754, 0.10235484689474106, 1.334106206893921, -0.8205710649490356, -0.7805529236793518, -0.3613225817680359, 0.273097425699234, 2.750380754470825, -0.8499380350112915, -1.6168797016143799, -0.14375387132167816, -1.705607295036316, 0.6373144388198853, -0.7970653176307678, 0.38391727209091187, -1.6234276294708252, 0.702510416507721, 1.8954230546951294, -0.9651145339012146, -1.0878087282180786, 0.003677202155813575, 0.27880796790122986, 0.4720136523246765, 0.15217912197113037, -2.699134349822998, -0.7330218553543091, -0.9334583282470703, -0.6822761297225952, 0.04209928214550018, -1.2254436016082764, 1.2208805084228516, 0.3920140862464905, 0.5357933044433594, 1.010296106338501, 0.23761171102523804, 0.21513567864894867, -0.17313732206821442, -1.1003519296646118, 0.24203039705753326, -0.3805316090583801, -0.0652860701084137, -0.007610551547259092, -0.26164257526397705, -1.9099783897399902, -0.4689885973930359, 0.40802615880966187, -0.32522591948509216, 0.08567962795495987, 0.02404613234102726, -1.4941730499267578, 0.2940266728401184, -0.14653131365776062, 1.5471224784851074, -1.444849967956543, -0.6342130899429321, 2.6550099849700928, -1.829559087753296, 0.11448368430137634, -0.009638817980885506, -0.8139426112174988, -1.3600411415100098, 1.0382142066955566, 0.2009550929069519, -1.5121163129806519, -1.730653166770935, 0.15249432623386383, 0.42963212728500366, -0.10774961113929749, -0.6476246118545532, -0.4021148383617401, 0.5698516964912415, -1.2352510690689087, -1.4095146656036377, -1.2617636919021606, 1.6675678491592407, -1.8228641748428345, -0.6458102464675903, -0.09334947168827057, 0.3531416058540344, 0.06552738696336746, 0.3540385067462921, -0.687458336353302, -1.2716140747070312, 0.14415156841278076, -0.4536772668361664, 1.7635481357574463, -0.12240888178348541, 0.8814692497253418, -1.8835530281066895, 0.9785851240158081, 0.2456820160150528, -0.6238363981246948, -0.8391998410224915, 1.8449957370758057, 0.4763011336326599, 1.5258562564849854, 0.15756329894065857, 0.6006565690040588, 1.279717206954956, -0.523563027381897, -0.5936278700828552, -0.7110056281089783, -0.43300122022628784, 0.8292034864425659, -1.8891416788101196, -1.3333394527435303, -0.5103525519371033, -1.2743098735809326, 0.9990583062171936, 2.1572751998901367, 0.3056054711341858, -0.30374959111213684, -0.14686496555805206, -0.29535531997680664, -1.163955569267273, 0.3793906569480896, 0.5828681588172913, 0.1328943371772766, -0.20473147928714752, -0.11032150685787201, 0.9757627248764038, -0.9803062081336975], [-1.8156647682189941, -1.2735230922698975, -0.018009617924690247, 1.069319725036621, -0.06777683645486832, 0.3763146996498108, -0.9883388876914978, -2.133202075958252, 1.0751657485961914, 0.18608711659908295, -0.5235931277275085, 0.0029110892210155725, 0.6665001511573792, -1.1068150997161865, -0.36950168013572693, -1.0365206003189087, -0.7526538968086243, -0.4161580204963684, 1.0472511053085327, -1.4398963451385498, 0.6596196889877319, -1.0885990858078003, -0.8717203140258789, -0.02826215885579586, 1.6921427249908447, 0.34739768505096436, -1.1380995512008667, -0.07322586327791214, -0.4454473853111267, -0.8992502689361572, 0.09980535507202148, 1.418843150138855, -0.7792423367500305, 0.5639393925666809, 1.0786614418029785, 0.13225261867046356, 0.21031299233436584, -0.06316659599542618, -0.898100733757019, 0.637543797492981, 1.4135639667510986, 0.7274724841117859, -1.2862399816513062, 2.113767385482788, 1.3553954362869263, 0.43905219435691833, 0.8797684907913208, -1.3294777870178223, 0.652763843536377, 1.3286696672439575, 0.2669849097728729, -1.9251070022583008, 0.06524238735437393, 0.9160311222076416, -2.4096267223358154, 1.0573701858520508, 1.7285878658294678, 0.1253623515367508, -0.46827003359794617, 0.9812053442001343, -0.22407756745815277, -1.54782235622406, -1.0276271104812622, -1.0957614183425903, 2.8235886096954346, 0.9215741753578186, -0.18082386255264282, 0.0796872079372406, 0.3506075143814087, -0.7925055623054504, -1.106997013092041, -0.3244701623916626, 2.8508083820343018, -0.3993610739707947, 0.13638794422149658, 0.45314326882362366, 0.9989773631095886, 0.01676058955490589, 0.8573302626609802, -0.19053436815738678, 0.8963943719863892, 1.2136098146438599, -0.498471736907959, -1.269857406616211, -0.6120201349258423, -1.7713515758514404, -0.253580242395401, 0.2217501550912857, 0.8852760195732117, 1.4637120962142944, -0.1884509027004242, 0.6142297983169556, -0.6115681529045105, -0.1594657003879547, 1.8706551790237427, -1.6428382396697998, 0.5330085754394531, -1.011299729347229, -0.2205648571252823, -0.3289315700531006, -0.06262905150651932, 0.202167347073555, -0.5634535551071167, 1.775877594947815, -0.8457486629486084, 0.02438821643590927, -2.277665376663208, 3.160857915878296, 0.6778595447540283, 0.461569607257843, 0.6836614608764648, -1.2629166841506958, 1.2498464584350586, 0.003374324645847082, 0.5270655751228333, 1.200959324836731, -1.759472131729126, 0.2812214195728302, 0.7190532088279724, -0.4554313123226166, 0.3424767851829529, 0.11523943394422531, 0.5491004586219788, 0.5240080952644348, -0.7795592546463013, 1.1516836881637573, 0.03207088261842728, 1.3139714002609253, -1.1556826829910278], [0.7356219291687012, 0.15688124299049377, -1.6297175884246826, -0.17122969031333923, 0.35051795840263367, 0.13917246460914612, -0.5648099780082703, -0.5890234112739563, -1.679660677909851, -0.1632806956768036, -0.1625320017337799, -0.3953145146369934, 0.9278257489204407, -0.9465562701225281, 0.7594994902610779, -0.6121955513954163, -0.20899878442287445, -0.4112314283847809, -0.440553218126297, 1.3415859937667847, 0.12783178687095642, 0.7372785210609436, 0.24272781610488892, -0.5861782431602478, 0.16553384065628052, 2.079679012298584, -0.4652886390686035, 0.8003387451171875, 1.469691276550293, 0.43396633863449097, 0.9704681634902954, 1.4855003356933594, -1.4563171863555908, -0.31783902645111084, 0.2995590269565582, 0.600447416305542, -0.03175796568393707, -0.8844608068466187, -0.18066692352294922, 1.622594952583313, -0.691318154335022, -3.2938475608825684, -0.2657003402709961, -0.45366033911705017, -1.3566633462905884, 0.7462831139564514, 0.18518708646297455, -0.4649364948272705, 0.757163405418396, -0.5061363577842712, -1.2956948280334473, 0.9158488512039185, -1.052988886833191, 0.5075129866600037, -2.5248374938964844, -0.5634827017784119, -0.7172868847846985, -0.9091963768005371, -0.7072266340255737, 0.3592473566532135, -0.05274631828069687, 0.8250839710235596, -0.5201519727706909, -0.5878812670707703, -0.6676701307296753, -1.3396083116531372, -0.10397924482822418, -1.3733681440353394, -1.1518135070800781, 1.2863138914108276, -0.8742038011550903, -0.24691152572631836, -0.7834427356719971, -0.4944579601287842, -0.35250556468963623, -0.07635592669248581, -2.420525312423706, 0.7470909953117371, -0.24524416029453278, -0.6212278604507446, 0.3471047878265381, -0.8296783566474915, 1.6730005741119385, -0.016658427193760872, 0.06863410025835037, 0.4965471923351288, 1.2345298528671265, 0.33391237258911133, -1.1094123125076294, -0.3117012679576874, 0.24235078692436218, -0.7074012160301208, 1.517587661743164, -0.9762471914291382, -1.4212958812713623, 0.8959648013114929, 0.8886229991912842, 0.0037739204708486795, -1.0665948390960693, -0.006806318182498217, 0.09634792059659958, -0.6844362616539001, -0.1808231770992279, 2.6957356929779053, -0.8304543495178223, 1.0799682140350342, 0.7386403679847717, 1.018937587738037, 1.0981847047805786, 1.8573124408721924, 1.3413289785385132, -0.11524783074855804, -1.6847776174545288, 1.4867419004440308, -0.09081415086984634, 0.6355814933776855, 0.3067665100097656, -1.0213748216629028, -1.7518819570541382, -0.5002428889274597, -0.8830949068069458, 0.8738105893135071, -0.5219109058380127, 0.19523906707763672, -1.9123417139053345, 1.3747124671936035, 1.2652325630187988, -0.7918958067893982, 0.007311714347451925], [1.4200375080108643, -0.7177659273147583, -0.2661706805229187, 0.7718288898468018, -0.10825015604496002, -1.0422002077102661, 0.9513791799545288, 0.3046957552433014, -0.025051917880773544, 0.7545768022537231, 0.7565490007400513, 0.5770399570465088, 1.1113536357879639, 0.10516131669282913, -0.08118179440498352, -0.49114790558815, 0.5995954871177673, -0.2825603783130646, -1.5898758172988892, -0.0776982456445694, 0.8549550771713257, -0.6870419979095459, 2.135496139526367, -1.1527378559112549, 0.16986314952373505, -0.2623068392276764, 0.8655475378036499, 0.16220222413539886, 0.05890687182545662, -0.17302767932415009, -2.693251609802246, 1.0822324752807617, -0.9223008155822754, -0.13700003921985626, 0.10210356116294861, 1.5648478269577026, 0.941486120223999, 0.7999287843704224, 0.49956586956977844, -0.33503270149230957, 0.3435833156108856, -0.7721120119094849, 0.39240071177482605, 0.23326021432876587, -0.8177658915519714, 0.7979458570480347, -1.1496317386627197, 0.5753328800201416, 0.4517587721347809, 0.7444772124290466, -1.1255974769592285, 0.7008245587348938, -0.254537969827652, -0.6014237999916077, -1.262413740158081, -0.6260043978691101, -0.04996609687805176, 0.8410036563873291, 1.4136500358581543, 2.156911849975586, -0.4985281229019165, 0.5936238169670105, -2.649358034133911, -0.9434083700180054, -0.8484999537467957, 0.21671807765960693, -0.17150919139385223, 0.31933826208114624, 0.23052673041820526, 1.6696741580963135, 0.542643129825592, 1.4810680150985718, -0.9055107235908508, 0.7724339962005615, 0.678831934928894, 1.1761201620101929, 0.48574233055114746, -2.254009962081909, 0.9020362496376038, 0.09089217334985733, 1.1461766958236694, 0.015428123064339161, -0.8637937903404236, -0.5837372541427612, 0.09015389531850815, 1.3049758672714233, -0.20174966752529144, -0.12681680917739868, 0.6102889180183411, 0.33055806159973145, 0.05178413912653923, -0.17224867641925812, -0.2796814441680908, 0.47530344128608704, 0.06943152099847794, -0.8557327389717102, -0.29620930552482605, -1.2098500728607178, 0.30525776743888855, -0.3622812330722809, 0.08828484266996384, 0.5760626792907715, 0.08838855475187302, 0.6572095155715942, 1.0746711492538452, 0.05635758489370346, -0.28211185336112976, -1.331061601638794, 0.5784289836883545, -2.323059320449829, -0.3592798411846161, 0.9231814742088318, 1.1047165393829346, 1.2598934173583984, 0.7701596021652222, -0.6673070192337036, 0.916817307472229, 1.7065826654434204, 1.5051360130310059, 0.16761192679405212, -0.6894502639770508, 0.8216135501861572, -2.1386449337005615, 1.2147750854492188, 1.2597060203552246, 0.9182215332984924, 1.944963812828064, -1.2253774404525757, 0.05276668816804886], [0.614012598991394, -0.6396634578704834, -0.8515976667404175, -0.34219589829444885, 2.1811318397521973, 1.6627326011657715, 0.14533212780952454, -0.3941584527492523, 0.015075993724167347, 1.019712209701538, 0.2422482818365097, -0.5358811020851135, -0.06744404882192612, -0.4931034445762634, -0.7170670032501221, 0.6844469904899597, 0.8907179832458496, 1.7206692695617676, -0.19192297756671906, 0.6773335337638855, 0.9917754530906677, 0.3531109690666199, -2.1537113189697266, -1.0394394397735596, -0.06054319068789482, -0.5548139810562134, -0.222502663731575, 1.56098473072052, -1.6496316194534302, 0.6820234060287476, -1.0373485088348389, 1.1424001455307007, -0.10118414461612701, -0.19190211594104767, 0.9805055856704712, 1.309966802597046, -0.15664735436439514, 1.4148846864700317, 0.8007898330688477, -0.9341486692428589, -0.30876800417900085, 0.46487167477607727, -0.468263179063797, -1.2867017984390259, 0.981135368347168, -0.7776170372962952, 0.7913840413093567, 2.369112253189087, -1.4936832189559937, 0.7890508770942688, 0.05230823904275894, 0.9979785680770874, 0.11392142623662949, -0.9170845150947571, 0.7695122957229614, -0.20288754999637604, 0.026537055149674416, 0.33331820368766785, -0.6092270016670227, -0.09173153340816498, 1.0095930099487305, -0.5459222793579102, 1.2545915842056274, -0.6688591241836548, 1.1652413606643677, 0.8149017095565796, 0.5338998436927795, 1.1454390287399292, 1.0556621551513672, 1.4939066171646118, -1.2962323427200317, 0.1752500981092453, -1.7822693586349487, 1.4676779508590698, -0.9862384796142578, -0.20798031985759735, -2.953460693359375, 0.5301694869995117, 1.6415692567825317, 0.06259489059448242, 0.9322300553321838, -1.1268987655639648, -1.3605010509490967, 0.9279292225837708, 0.07410220056772232, -2.172948122024536, 1.044470191001892, -1.0763036012649536, 1.2532373666763306, 0.5859858989715576, 0.05444330349564552, 0.0008570519275963306, -0.5977309346199036, -1.2183645963668823, 1.6362369060516357, -1.042313814163208, 1.0899882316589355, 2.3079116344451904, -1.368914008140564, 2.407395124435425, -0.4094964265823364, 0.5263994336128235, 0.003124939277768135, 0.3078211843967438, -1.0230072736740112, 1.252485752105713, 0.6759178042411804, -0.5058197379112244, 1.2564798593521118, 0.5925430059432983, -1.2393118143081665, 2.3420281410217285, -1.576216220855713, 0.9011479020118713, -0.7779574990272522, -0.9330691695213318, 0.19774165749549866, -1.8206197023391724, 0.0371989943087101, 0.5730702877044678, 0.04773550480604172, -1.7660642862319946, -1.1011791229248047, 1.2367509603500366, -0.8630951046943665, 0.7434735894203186, -0.9754675030708313, -0.12696991860866547, 0.4499639570713043], [1.5130807161331177, 0.9821863174438477, 0.8285572528839111, -2.1343655586242676, -0.026714099571108818, 1.0470541715621948, -0.7605027556419373, -0.2581126093864441, 2.984337568283081, 0.3656952679157257, -1.1490111351013184, -0.30350106954574585, 1.0225186347961426, 0.1911730319261551, -1.0372204780578613, -0.35063716769218445, 1.152658224105835, -0.22839339077472687, 0.9764410257339478, 0.7315473556518555, 2.3532772064208984, 0.672290027141571, 0.59177565574646, -0.7388762831687927, -2.0719423294067383, -0.7029164433479309, 0.807942271232605, -0.790286123752594, -1.2057710886001587, -1.041040301322937, 0.49993693828582764, 1.0807828903198242, 0.806391179561615, -0.5131959915161133, -1.5674744844436646, 0.9431024789810181, 0.49953585863113403, 0.9073590040206909, 1.931775689125061, -1.2771143913269043, -1.6481562852859497, -0.9547117352485657, 0.3144051134586334, 1.6389415264129639, 0.42470139265060425, 0.4474533796310425, 0.42520102858543396, -0.4111775755882263, -0.7867443561553955, -1.361678123474121, -1.5101227760314941, 0.4496769905090332, -1.246355414390564, -0.7568045854568481, 0.0380924716591835, -0.43083834648132324, 1.0601463317871094, -0.10745108127593994, -2.118068218231201, -1.1501734256744385, 1.0591436624526978, -1.7251676321029663, -1.4886888265609741, -0.7343840003013611, -0.20260004699230194, 0.014736946672201157, 0.21971958875656128, 0.3700554370880127, 1.6420323848724365, -0.971424400806427, 0.8919030427932739, 2.062371253967285, -0.5695094466209412, 1.375352382659912, 0.20034272968769073, -0.8566266298294067, -0.10602685064077377, 0.3195396363735199, -0.20885466039180756, -0.5471184253692627, 1.198825716972351, 0.735986053943634, 0.23515750467777252, -0.2566089332103729, 0.3798932731151581, -0.014606107026338577, -0.2789897322654724, -0.02764228731393814, 1.1645716428756714, 2.280658483505249, 0.2116064876317978, -0.5705863833427429, -0.32806897163391113, -1.0441561937332153, 0.34478312730789185, -2.1230010986328125, 0.6333340406417847, 1.8026973009109497, 0.9559361934661865, 1.047807216644287, -0.9031446576118469, 0.2659119963645935, -1.8940718173980713, -0.21244679391384125, -0.17852288484573364, 1.1093698740005493, -0.9039934873580933, -0.8131825923919678, -0.24152357876300812, 0.32509034872055054, 0.37027910351753235, -2.587366819381714, 0.8445191979408264, 0.1683971881866455, 1.1914126873016357, 1.1922417879104614, -0.549251139163971, -0.8042445778846741, -0.6298373341560364, 1.0153642892837524, 0.9133482575416565, 1.0837733745574951, 2.0369608402252197, 1.5204697847366333, -0.47999465465545654, 0.4960932731628418, -0.2331240475177765, -1.5105341672897339, -1.312036156654358], [0.540978193283081, 0.13381168246269226, 1.5973472595214844, 0.223566934466362, 0.25809454917907715, 0.25251132249832153, -0.671232283115387, 0.06779412925243378, 0.4102838933467865, -0.5956431031227112, 0.5245351195335388, -0.6714357733726501, -0.726513683795929, 1.0607136487960815, 0.39658913016319275, -0.2062707543373108, 0.7297112941741943, 0.13285157084465027, 0.29839277267456055, 0.3112337291240692, 0.2540140151977539, -2.165043592453003, -1.1317890882492065, -2.1756398677825928, -0.46057838201522827, 1.0136605501174927, 0.12045910209417343, -1.1734650135040283, -0.34096410870552063, 1.8900173902511597, -1.3005656003952026, -0.6349881291389465, -1.1396158933639526, -2.4830381870269775, 1.5478087663650513, 0.20048809051513672, 0.15162569284439087, 0.7609192132949829, 1.3355910778045654, 0.8376967310905457, -0.9211726784706116, 1.0880850553512573, 0.40714389085769653, -1.0578949451446533, -1.0970592498779297, 0.29008835554122925, 0.0814051404595375, 1.0913971662521362, 0.22374427318572998, 0.40636637806892395, -1.1689680814743042, -0.5844159722328186, -2.4357175827026367, 0.6988297700881958, 1.2507996559143066, 0.44766220450401306, -1.0710992813110352, -0.7469949126243591, -0.2155916690826416, -0.7811669707298279, 0.8981370329856873, -0.7145059704780579, 0.9728514552116394, -0.6490374207496643, 0.4204142689704895, 1.2970030307769775, -1.8059196472167969, -0.29515206813812256, 1.043491244316101, -0.24317796528339386, 0.11531476676464081, 2.9496467113494873, -1.3428512811660767, -0.720264196395874, -0.559374213218689, -0.3005464971065521, -0.1315697878599167, 0.6012269854545593, -0.325471967458725, -1.5919617414474487, -1.3147860765457153, 0.9628446698188782, 0.8890320658683777, -0.3932156562805176, 0.5426210761070251, 0.7803038954734802, -0.596584677696228, -1.185121774673462, 0.6101247668266296, 0.7458451390266418, -0.43349531292915344, -0.11798915266990662, -0.5109918713569641, 0.7985543012619019, -0.9784038662910461, -2.244025468826294, 0.5644127130508423, -0.8595519065856934, -1.3489044904708862, -0.8989087343215942, -0.29911261796951294, -1.178123116493225, -0.1072663739323616, 0.40200746059417725, 0.39983049035072327, -0.5171993374824524, -1.7787859439849854, -0.7139459252357483, -0.6139886379241943, 1.3237872123718262, 1.7604295015335083, -0.8025047183036804, -1.1447242498397827, 0.3709920048713684, 0.3989699184894562, 0.4313752055168152, 0.16511602699756622, -0.00506444601342082, 0.8901758790016174, -0.39352723956108093, 0.9829135537147522, -0.6068491339683533, 0.2890269458293915, -1.1949660778045654, 0.48709774017333984, -1.4416190385818481, -0.04261273518204689, -1.7308461666107178, -0.4906618893146515], [-0.5492942333221436, -0.765211284160614, 0.2580128312110901, -0.4336520731449127, 1.053648591041565, -0.3189832866191864, -0.06283169239759445, -1.2804583311080933, -0.10495707392692566, 0.13350409269332886, 0.9385289549827576, -0.10210900008678436, -1.1349447965621948, 0.6823554039001465, -0.5389693975448608, 0.16868337988853455, 0.6292023658752441, 1.5814716815948486, 0.10072011500597, 1.0673469305038452, 0.8274363279342651, 0.0025375750847160816, 0.11414648592472076, 1.5840407609939575, 0.12092167884111404, 1.413751244544983, 0.4082096219062805, -1.2018213272094727, -1.1633267402648926, 1.4938774108886719, -1.255557656288147, 1.0245811939239502, -0.4591764807701111, 0.45952993631362915, -0.38836848735809326, 0.059814780950546265, 0.12986022233963013, 0.45461905002593994, 0.1591869294643402, -0.3642674684524536, -0.7038986086845398, 0.8724936246871948, 0.8862826228141785, 1.1167618036270142, -1.7977557182312012, -0.628201961517334, -0.6673792004585266, 0.7698186635971069, -0.6985378861427307, -0.20258168876171112, 0.3715512156486511, -0.7104856967926025, -2.4098258018493652, 0.04844420775771141, -1.2454084157943726, 0.5716179609298706, -0.5931693315505981, -1.5319490432739258, -1.1167761087417603, -0.07957156747579575, 0.4083574116230011, -0.5173589587211609, 0.6502968072891235, 0.3935551047325134, 0.30284854769706726, 1.7192350625991821, -0.14393825829029083, 2.6931309700012207, 0.004953890573233366, 0.36023518443107605, -0.6178809404373169, -0.27530908584594727, 1.1570526361465454, -1.1993274688720703, 0.41337841749191284, -1.335842251777649, 1.20963454246521, -0.972388744354248, 2.6227450370788574, 1.5132253170013428, -1.1170016527175903, 1.6802561283111572, 1.094174861907959, 0.7895681262016296, -2.3336071968078613, -0.889664888381958, 0.40036267042160034, 1.2951549291610718, 1.5713051557540894, 1.1967949867248535, 0.8251677751541138, 1.0825003385543823, 0.2861766219139099, -0.10588104277849197, 0.7440089583396912, -0.8834743499755859, 2.0162205696105957, -0.04617027938365936, 1.3492003679275513, -0.19510139524936676, -1.1997182369232178, -0.5947611927986145, -1.429549217224121, 0.23138859868049622, -0.548422634601593, 1.6111838817596436, -0.530945897102356, 0.4379245936870575, 0.13870267570018768, -0.016281170770525932, -0.8979021310806274, 0.40651366114616394, 0.37205755710601807, -0.4630127251148224, -1.9906286001205444, -0.7267228960990906, 0.8918982744216919, 0.8623816967010498, 0.8319688439369202, -1.5904147624969482, -0.19635869562625885, 0.7577619552612305, -0.06464541703462601, -0.709037721157074, -1.2327549457550049, -0.5041688084602356, 1.611340045928955, -0.36453869938850403, -1.7751669883728027], [1.232667088508606, 0.08867522329092026, 0.8065162897109985, -0.7475076913833618, 0.06324666738510132, 1.193333625793457, 0.933027982711792, -1.0615559816360474, -0.9644178748130798, 1.3626821041107178, -0.8638888597488403, 0.3750998377799988, -0.9221263527870178, 0.76186203956604, 0.4114183783531189, 0.25925031304359436, 0.700842022895813, 1.641002893447876, 0.8612419366836548, -0.5095828771591187, -0.7756502032279968, 1.771862506866455, 0.1230645626783371, -0.6515455842018127, 0.7679401636123657, 0.6346707940101624, 0.7216892838478088, -0.3477177023887634, -0.8797504305839539, 0.08462987840175629, -0.638583242893219, 0.8966104388237, -0.09875969588756561, 0.7796710729598999, 0.304749459028244, -0.5696779489517212, 1.442163109779358, -0.029480719938874245, -2.12847900390625, 0.816181480884552, 0.1772516369819641, -0.9832653403282166, -0.659185528755188, -0.44293707609176636, 0.10556110739707947, 1.3714263439178467, -0.585480809211731, -0.24450446665287018, -1.4645991325378418, 0.1065458208322525, 1.5191657543182373, -1.7947615385055542, -0.711799681186676, 0.024851728230714798, 0.3400768041610718, -0.43875283002853394, -0.03915850445628166, 0.593408465385437, -0.4231470227241516, 1.4556381702423096, 0.5670603513717651, 1.0146695375442505, -0.5480126738548279, 0.06253155320882797, -1.6549525260925293, 1.2747122049331665, -0.13978438079357147, -0.6514549255371094, 0.08368539810180664, 0.24384406208992004, 0.03318139165639877, -0.6092321872711182, 0.3077455461025238, 0.739328145980835, -1.9528582096099854, -0.5115681290626526, 0.17412340641021729, -0.7502489686012268, 0.322809636592865, 1.405004858970642, 0.5695666074752808, -0.7945177555084229, 0.8583787083625793, -1.5215424299240112, -0.5834128856658936, 0.2327401041984558, 0.25388652086257935, 0.6062030792236328, -0.9557181596755981, -1.197317123413086, -0.8151462078094482, -1.2459287643432617, 0.1784900426864624, -0.08280409127473831, 0.5914431214332581, 0.9748042821884155, 0.2940189242362976, -0.38827934861183167, 0.6266558170318604, -1.4416097402572632, 0.02307724952697754, 0.9798981547355652, -0.7697333693504333, -1.2054413557052612, -0.18546746671199799, -1.237483024597168, -0.413002073764801, -0.5701771378517151, 1.4001796245574951, 1.273268461227417, -0.4524484872817993, -0.520314633846283, -0.00995991937816143, 0.2185460478067398, 0.03054617904126644, -1.705211877822876, 1.0100328922271729, 2.001147508621216, -0.666682243347168, -1.0442744493484497, -0.6167160272598267, 0.15362201631069183, -2.198211908340454, 0.3653886020183563, -0.3068890869617462, 0.9615050554275513, 0.7739705443382263, 0.3704833388328552, 1.7997068166732788], [-0.15006260573863983, -0.10469992458820343, 0.014571720734238625, 0.41722020506858826, -1.2249852418899536, -0.07914191484451294, 1.032476782798767, -1.1366078853607178, 1.1271709203720093, 0.8907825946807861, 0.029719870537519455, -1.6016077995300293, -0.9506555199623108, -0.22431151568889618, 0.02072000317275524, 0.528681218624115, -1.8105885982513428, -0.7735930681228638, 0.6759052276611328, -1.0083767175674438, 0.27649977803230286, 2.2557122707366943, 1.3180807828903198, -0.5886743068695068, 1.063690423965454, 1.3479472398757935, -1.4006211757659912, -0.7266141176223755, 1.3758434057235718, 0.1689082682132721, 0.5228126049041748, 1.8785276412963867, -0.2126990109682083, -0.5880589485168457, 0.5863455533981323, -0.9294155836105347, 1.377256155014038, 0.35828131437301636, 1.0218701362609863, 0.37419310212135315, 0.7288907170295715, -0.7920613884925842, -0.42228519916534424, 0.7034847140312195, -2.526578187942505, -0.06265410035848618, -1.0870290994644165, 0.6343870162963867, 0.3769521713256836, 0.004265307914465666, 1.2031270265579224, -0.18892186880111694, 1.2318381071090698, 0.1711013913154602, 0.6638755202293396, -0.8717844486236572, -1.1095235347747803, 1.1390725374221802, -1.1508922576904297, -0.6988435983657837, -0.4929380416870117, 0.3234368562698364, 0.13767093420028687, -0.10980986058712006, 1.3400801420211792, -0.30599114298820496, -0.3095792829990387, 1.2642611265182495, -1.3929719924926758, -2.3111119270324707, -0.8159513473510742, -0.650912344455719, -1.330320119857788, -0.2194727212190628, 0.8569570183753967, -0.18276499211788177, 1.1115516424179077, -0.3032726049423218, -0.23382757604122162, -0.5434042811393738, 1.337868094444275, 0.6544851064682007, 1.963765263557434, 1.590034008026123, -0.030780188739299774, -3.1811389923095703, 0.13632206618785858, -0.12212823331356049, -1.613879680633545, -0.04993373528122902, -0.6736921072006226, -1.1820342540740967, -0.5740767121315002, 0.7402786612510681, 0.18190200626850128, -0.6419410109519958, -1.2263673543930054, -0.8409934043884277, 1.1179338693618774, 0.775594174861908, 0.18567204475402832, 0.15298457443714142, 0.021049559116363525, -0.9415424466133118, -0.08912355452775955, 1.0409355163574219, 0.04169397056102753, -0.12231062352657318, -0.10825452953577042, 0.07600650191307068, 1.5117965936660767, 1.866005539894104, -0.37176600098609924, -0.7141087055206299, -0.6941960453987122, 0.822821319103241, -1.2010328769683838, 0.5348122715950012, 0.20820669829845428, 0.14762207865715027, 0.08888798952102661, 1.3622430562973022, -0.41209590435028076, -0.37456321716308594, -0.5463051795959473, -0.7583755850791931, -1.1084078550338745, 1.5281208753585815, -1.271085262298584], [1.1651034355163574, 0.2946138083934784, 0.35872742533683777, 0.3561236560344696, -0.05447070673108101, -1.063830852508545, 0.7176627516746521, -0.33924543857574463, 0.016293227672576904, -0.49397867918014526, -1.5883489847183228, 1.3835676908493042, -0.6917256116867065, -1.5718764066696167, -0.1334533989429474, -0.1462346762418747, 0.7899128198623657, -0.6014314293861389, 0.3257991373538971, -0.07956479489803314, -0.6927057504653931, -0.5107699036598206, 0.3940109312534332, -0.495297372341156, 1.7497591972351074, -2.09157657623291, 0.2737577259540558, -0.10649463534355164, -0.011762331239879131, 0.17673909664154053, 1.7923389673233032, 1.5017585754394531, -2.042391538619995, -0.8747506737709045, 0.18269658088684082, 0.02758222259581089, -1.2015268802642822, 1.087634563446045, 0.47376999258995056, 1.0807955265045166, 0.49493446946144104, 0.10024760663509369, 0.8687337040901184, -0.31975242495536804, -0.04513809084892273, 0.6517582535743713, -0.11407481878995895, -0.29418134689331055, 0.1977483183145523, 0.2812972068786621, 1.1540677547454834, 0.37129339575767517, 0.09148062765598297, -1.133257508277893, 1.2167518138885498, 0.6291599273681641, -1.5341094732284546, -0.5368807315826416, -0.7596274614334106, -1.0159658193588257, 0.1973082721233368, 1.3025251626968384, -1.4778742790222168, -0.12434212118387222, -0.06525100022554398, -0.9661723375320435, -0.18760652840137482, -0.630610466003418, -0.7984625101089478, 0.4637134075164795, 0.31396934390068054, 0.2485083043575287, -0.486812561750412, 1.122664213180542, 0.5319707989692688, 0.8415094614028931, -0.23648881912231445, -0.3522668480873108, 1.893853783607483, 0.8239256143569946, 0.3333330452442169, -0.3429304361343384, -0.6447904706001282, 0.16545157134532928, 0.24450883269309998, 0.13649335503578186, -0.5066540241241455, 1.2659419775009155, -0.3633009195327759, 2.1799967288970947, -0.07024624943733215, 0.40417104959487915, 0.32714536786079407, 0.15969212353229523, 0.026245316490530968, 1.2511368989944458, -0.6686711311340332, -0.7286248207092285, 0.025620711967349052, 0.74261075258255, -0.14588725566864014, 1.6778008937835693, 0.46332618594169617, -0.2516493797302246, -0.3142055869102478, 0.9861682653427124, -0.5438154339790344, -0.5669218897819519, -0.19366154074668884, 0.05261610075831413, -1.3803414106369019, -0.721359133720398, -0.018748806789517403, -1.5265533924102783, 0.3366655111312866, 0.5213319659233093, 0.5758013725280762, -0.9837777614593506, -0.5078607201576233, -1.008162021636963, 0.3195864260196686, -1.3346388339996338, -0.7314608097076416, -1.0007894039154053, -1.2101441621780396, -0.37541690468788147, 0.03862148895859718, 0.7501338124275208, -1.2567837238311768], [-1.9398301839828491, -0.35736069083213806, -0.1811048835515976, -1.3141858577728271, -0.33920273184776306, 0.5446532368659973, 1.3189119100570679, 0.3413383364677429, 0.28377005457878113, -0.17403016984462738, 1.1494425535202026, 0.23816698789596558, -2.760885000228882, -0.7149136662483215, 0.7998623251914978, 0.3795640468597412, 0.9057947397232056, -1.4208205938339233, -0.5532306432723999, 1.3627182245254517, -0.617667555809021, 0.6345503926277161, -0.34157875180244446, 0.8234770894050598, -0.730121910572052, -0.2877025306224823, -1.8449679613113403, 1.7224891185760498, 0.1834610551595688, 1.6041288375854492, -0.056104015558958054, 0.4761633276939392, 0.7003164887428284, -0.10973764955997467, 1.4065831899642944, -2.2086687088012695, 1.0590795278549194, -1.7780213356018066, 2.068631172180176, -1.089780569076538, 0.736682116985321, -1.540901780128479, -1.7892218828201294, -1.5910388231277466, 0.9733843207359314, 0.37048667669296265, -0.5199819207191467, 0.5172148942947388, 0.39388415217399597, 0.1974126547574997, -0.9236857891082764, 1.38792085647583, 1.0009911060333252, 0.7740437388420105, 1.9904155731201172, 0.0008290609694086015, -0.557813286781311, -0.9820079207420349, -1.930274486541748, 0.1505480706691742, 0.19666901230812073, 1.2978250980377197, 0.8919029831886292, -1.1867464780807495, 2.4339094161987305, -1.306735634803772, 1.771324872970581, -0.4639662802219391, -0.5957921147346497, -0.3585027754306793, 0.6437640190124512, -0.4901842772960663, -0.7688380479812622, 1.090555191040039, 1.2436484098434448, -0.1634192019701004, 0.14212235808372498, 0.4550933539867401, -0.48132285475730896, 1.0555126667022705, 1.4851734638214111, 2.153913974761963, -0.1415889710187912, 0.4260515570640564, 1.4233441352844238, 0.4668324291706085, -0.6074680685997009, -0.9999100565910339, 0.018503375351428986, -0.11363304406404495, -0.27331194281578064, 0.03929043188691139, -1.2020645141601562, 0.2543903887271881, 0.4934422969818115, 0.6101918816566467, -0.7518972158432007, 1.5499714612960815, -0.8063855171203613, -0.11604747921228409, -1.4745694398880005, -0.7457653284072876, 2.402712106704712, -0.16400046646595, 1.3995157480239868, -0.8435419797897339, -0.6446092128753662, 1.1671656370162964, 1.8886494636535645, 0.9798389077186584, 1.0917388200759888, -0.263449490070343, -0.6151946783065796, -1.16566002368927, 0.024030517786741257, 1.1628321409225464, -0.25070345401763916, -1.4042967557907104, -0.8108155727386475, 2.0735483169555664, 0.5204072594642639, -0.6743901371955872, -0.8628914952278137, 0.6868634819984436, 0.7548297047615051, 0.570355236530304, -0.645343005657196, -0.8413078784942627, 0.43944674730300903], [-0.6501535177230835, 1.9710184335708618, 1.0675477981567383, -0.7158581018447876, 0.7436865568161011, 1.0129977464675903, -2.3760147094726562, 1.1836313009262085, 0.4052410423755646, 0.3394007086753845, -0.4595832824707031, 1.7663766145706177, -0.14697414636611938, 0.057894106954336166, -2.425482749938965, -1.2050367593765259, 0.7709948420524597, -0.11854034662246704, 0.024582428857684135, 0.025063201785087585, -0.5003606677055359, 0.42141854763031006, 0.595039427280426, 0.49116265773773193, -0.7705700397491455, -0.8179842233657837, 0.867836594581604, 0.10570614039897919, -1.097368836402893, 0.7600003480911255, -0.11914084851741791, 0.18551574647426605, -0.2166864275932312, -0.6081132888793945, -0.5663226246833801, 1.6133875846862793, -1.7449594736099243, 1.7537065744400024, 0.5976815819740295, -0.1623421013355255, 0.643022358417511, -0.19280973076820374, -1.6848939657211304, -0.5850293040275574, 0.3171546459197998, 0.48659589886665344, 0.23712648451328278, -0.37745508551597595, 1.7383532524108887, 1.5403361320495605, 1.2800770998001099, -1.3792812824249268, 0.3818168342113495, 0.12159240245819092, -0.7528824806213379, 0.049534544348716736, -0.7453964352607727, -0.07439948618412018, 0.25385063886642456, 0.4684825837612152, -0.5323209166526794, -0.08990225940942764, -0.35691484808921814, -0.726571798324585, -0.5951818227767944, -0.2082960605621338, 0.21189762651920319, 0.25557100772857666, -0.1275070160627365, -0.4960804581642151, -0.08401526510715485, -1.7628942728042603, 0.6034850478172302, -0.3678830862045288, 0.9144246578216553, 2.181912660598755, 0.5990851521492004, -1.4174960851669312, -1.1285529136657715, -1.091611385345459, 0.0823400467634201, 0.49609166383743286, 0.9898947477340698, 0.6044209599494934, -0.4315451383590698, 0.5810334086418152, 0.16688896715641022, 0.05655932426452637, -0.43867969512939453, -0.43714407086372375, 0.5417654514312744, -0.5778928995132446, -0.19333577156066895, 0.7940780520439148, 0.6556611657142639, 0.33123335242271423, -2.900648593902588, -0.46297645568847656, 0.021153254434466362, -0.2538372278213501, 2.0537047386169434, -0.7520950436592102, 1.8058717250823975, -0.0428922213613987, -1.7230464220046997, -0.7972810864448547, -0.6562687754631042, 1.6518497467041016, -0.5769490003585815, 0.4057190716266632, -1.4095375537872314, 1.6379685401916504, 1.6769694089889526, 0.0008356984471902251, 0.5235536694526672, 1.325086236000061, -0.6279183030128479, -1.213523507118225, 0.1170659139752388, -1.6054936647415161, 0.9426470994949341, -0.8344227075576782, -0.8159271478652954, 0.8621758818626404, -0.665931224822998, 1.162262201309204, 1.204059362411499, -1.4339828491210938, 0.578399658203125], [-0.8255048394203186, 0.34133797883987427, 1.0698925256729126, 0.35976442694664, 1.6360915899276733, -1.229520559310913, 2.0680854320526123, 0.07669954746961594, 0.5650129318237305, -0.05500108376145363, -0.8831767439842224, 1.4255772829055786, 0.4361687898635864, -1.7829934358596802, -0.030915087088942528, 0.11183516681194305, -0.6063678860664368, 2.098449945449829, 1.3022722005844116, 1.4376972913742065, -1.2446931600570679, -1.1854640245437622, 1.5151530504226685, -0.37519556283950806, 1.4167778491973877, 0.29525503516197205, -0.11464796960353851, -0.9856706261634827, 1.3417028188705444, -1.7030428647994995, -0.6619179844856262, -1.3350999355316162, 0.027681566774845123, 0.9339585900306702, 0.11815125495195389, 0.6562569737434387, 0.49103784561157227, -0.06097034737467766, 0.1396058201789856, 0.9751600623130798, 0.7811985015869141, -0.7493563890457153, -1.5437055826187134, 0.6177093982696533, -1.683303713798523, 0.8501806259155273, 1.9141680002212524, 0.9725381731987, 0.7033853530883789, -1.4243414402008057, 1.119829535484314, 0.22965575754642487, 0.8097941279411316, -0.8752123713493347, -1.4895944595336914, 0.6974077820777893, -0.11167620867490768, 1.1170365810394287, -0.8913877606391907, 0.8850370049476624, -0.6180033683776855, -1.1410856246948242, -0.4663248062133789, -1.203229308128357, 0.7487501502037048, 1.129866361618042, 1.9728798866271973, -0.9743365049362183, 0.29799726605415344, 1.2050307989120483, 1.5998940467834473, -0.9184765219688416, -1.6824514865875244, -2.1424386501312256, -0.21024200320243835, 0.059489600360393524, 1.1635782718658447, 0.6287567615509033, 0.4482996463775635, -0.19350677728652954, -1.2154953479766846, -0.9990653991699219, 0.2259402573108673, -1.0541640520095825, -1.572056531906128, -0.895038366317749, -0.33741143345832825, 0.6376449465751648, -0.7319839596748352, 0.14326457679271698, -1.7423388957977295, -0.6008055210113525, -0.17069977521896362, 0.2648152709007263, 1.698960542678833, 0.35195621848106384, -0.5659870505332947, 1.169405460357666, -1.9744365215301514, -0.025894895195961, 0.21041622757911682, -0.8342192769050598, 0.08287785202264786, -0.7458099722862244, 1.1574422121047974, -0.8616588115692139, 1.227015495300293, -0.8488053679466248, 0.35136085748672485, 0.9072398543357849, -1.350833773612976, -0.1140061765909195, 1.0299991369247437, 1.3019304275512695, 1.0435601472854614, -0.10403811186552048, 0.22613902390003204, -1.4578019380569458, 1.3568987846374512, 1.9588220119476318, -0.030128393322229385, 0.7881115078926086, 0.9775038361549377, 2.51623272895813, -0.13728965818881989, 0.3181000053882599, -0.7663644552230835, 0.3665304183959961, 1.694901704788208], [0.33961838483810425, -1.5369610786437988, 0.802529513835907, 0.32756930589675903, 0.16049738228321075, -1.4988305568695068, 0.46296191215515137, -1.934258222579956, -0.7363091111183167, 0.7164981365203857, 0.48095372319221497, -1.1772114038467407, -0.37155881524086, 0.18894916772842407, -0.023275058716535568, -0.6346948742866516, 0.45497626066207886, 1.6997199058532715, -1.448421835899353, 0.7450039386749268, 1.461167573928833, -1.0712285041809082, -1.614743709564209, 1.2746213674545288, 0.4525620937347412, 1.0647577047348022, 0.004190691281110048, -1.7373676300048828, 0.33407482504844666, -0.44166994094848633, 1.4980229139328003, 0.875070333480835, -0.38737019896507263, -2.099142074584961, 1.7596946954727173, 0.35198816657066345, 0.1572447568178177, 1.1156444549560547, 0.39881768822669983, -0.08055024594068527, 0.5565733313560486, -0.9439409375190735, 0.654036819934845, 0.8938233852386475, 0.17010098695755005, -1.3483295440673828, -1.0903711318969727, -1.9249731302261353, 0.5532875657081604, 0.043137822300195694, -0.4516270160675049, 0.2281973958015442, -0.5515570640563965, -2.30840802192688, -0.2655755579471588, 1.4706416130065918, 0.24254345893859863, -0.8055766224861145, 0.017082691192626953, -0.48129063844680786, -1.7506400346755981, 0.29210159182548523, -1.8501510620117188, 1.3184409141540527, -2.4785056114196777, 0.3222227692604065, -0.2725095748901367, -0.11673197895288467, 0.10183512419462204, 1.1262872219085693, -1.0982060432434082, -0.2391606718301773, -1.2833969593048096, -0.49481865763664246, -0.6952462792396545, -0.2998206615447998, 0.5732907056808472, -0.25863462686538696, -0.07437556982040405, -0.15245164930820465, 1.3466167449951172, -0.02238626219332218, -0.7170344591140747, 0.3582857847213745, 0.4907292425632477, -0.28423362970352173, -2.1026053428649902, -1.1636561155319214, -1.8623461723327637, -0.631801187992096, -0.573668897151947, -0.4301810562610626, 1.3086622953414917, -0.793147087097168, -1.7738429307937622, 2.34334397315979, 2.4405665397644043, -2.8150150775909424, 1.7972768545150757, -0.2369617521762848, -1.7317780256271362, -0.5959022045135498, -0.29320454597473145, -1.62459397315979, -0.3033161461353302, 0.992965817451477, 0.34880056977272034, -0.512185275554657, -0.1083049401640892, 0.7201262712478638, -0.4226560890674591, 1.7456105947494507, 0.7930485606193542, 0.5246050953865051, 1.9323140382766724, -1.1602946519851685, 0.6526082754135132, -0.17444375157356262, -0.9708929061889648, -1.0647615194320679, -0.9155651330947876, 0.5112155675888062, -1.257659673690796, 0.7527045607566833, -0.842841625213623, 0.4193146228790283, -0.567082941532135, -1.137993574142456, 1.7755628824234009], [0.17727360129356384, -0.7974846363067627, 0.8730154633522034, 0.07203307747840881, -0.08030129224061966, -0.9811187386512756, 1.557782769203186, -1.3163459300994873, -0.2816266119480133, -0.8224919438362122, -1.8066126108169556, 0.31312206387519836, -1.7385720014572144, 0.9806961417198181, -0.39085105061531067, -0.3500010371208191, -0.540701687335968, -0.9555382132530212, -0.9420557618141174, 0.8983500599861145, 0.6792788505554199, 1.8701175451278687, 1.0282366275787354, 0.37819012999534607, -0.688126802444458, 0.44761109352111816, -1.6078828573226929, 0.5669790506362915, 1.2734562158584595, 2.440310478210449, -1.474039912223816, 0.5501141548156738, -0.26580074429512024, 0.7696675658226013, -0.8035020232200623, 0.514125406742096, -1.218563199043274, 0.7425814867019653, -0.726387083530426, 0.6671534180641174, 0.0928991436958313, -0.06764818727970123, 0.28129470348358154, 0.028980322182178497, -0.8566951155662537, -0.07443635910749435, 1.5214221477508545, 1.4879502058029175, -1.0733360052108765, -0.9925938844680786, -0.1504281908273697, -1.013840675354004, -0.8256940841674805, 0.5504922866821289, 0.9151598811149597, 0.3051319718360901, 1.5233557224273682, -1.2546250820159912, 1.8981571197509766, 1.0014184713363647, -0.06304852664470673, -0.676897406578064, 0.7462512850761414, -1.027719497680664, -0.9737367033958435, 0.833581805229187, 0.6077550649642944, -0.7748034000396729, -1.8753576278686523, 1.8512226343154907, 0.4310838282108307, 2.897392988204956, -0.16040964424610138, 0.1728108823299408, 0.11749342828989029, 0.268638551235199, -1.8668893575668335, 0.490168035030365, 0.28141212463378906, -1.7148771286010742, 0.5197968482971191, -0.40455362200737, -1.4500921964645386, 0.1603921502828598, -0.33624792098999023, 0.32815253734588623, 0.2288741022348404, -1.4464097023010254, -0.7310254573822021, 0.22467248141765594, -0.3844992220401764, 0.4412255585193634, -2.1292951107025146, -0.2038610279560089, -0.9311242699623108, -0.02980823442339897, 0.36708003282546997, 0.456167072057724, 1.4837652444839478, -0.8111872673034668, -2.7662370204925537, 0.04775041341781616, -0.04158082976937294, 0.7142997980117798, 0.28301629424095154, 0.6167677640914917, 0.7241945266723633, -0.7149196267127991, 0.27780529856681824, -0.9256536364555359, 0.4272632598876953, -0.7710046172142029, 0.13590028882026672, 0.22256462275981903, -1.2101982831954956, -1.5432103872299194, 0.7886554002761841, 0.6304530501365662, 0.9415053725242615, 0.7897236943244934, 0.3482901453971863, -0.7544540166854858, -0.05383002758026123, 0.44230005145072937, -0.4363146126270294, 0.8837490677833557, -1.2171673774719238, -0.6673000454902649, 0.6144134402275085], [-1.2046867609024048, 1.303700566291809, -0.5625704526901245, -1.3935259580612183, -0.37301719188690186, 0.7978127598762512, 0.31438469886779785, 0.07593371719121933, -0.045557182282209396, -0.5020626783370972, 0.7575926184654236, -1.260616421699524, 0.8741215467453003, -0.7232193946838379, 0.5619639754295349, 2.1647815704345703, -0.16544488072395325, 0.6588849425315857, 0.335563063621521, 0.26660284399986267, 0.02318909764289856, -0.14222566783428192, 0.21735160052776337, 0.1946234107017517, 1.069800853729248, 2.1568171977996826, 0.6272371411323547, -0.7466357946395874, -0.851713240146637, 0.8877784609794617, 1.0436196327209473, 3.1379008293151855, 0.4550931751728058, -0.009970362298190594, 0.06820816546678543, -1.1336870193481445, 0.20838621258735657, -0.8921838998794556, 1.3544734716415405, 0.5216749310493469, -0.6790781617164612, 0.5632612109184265, 0.6965624094009399, 1.3380708694458008, 0.4882955849170685, -0.7858940362930298, -1.3027256727218628, 0.4498100280761719, -0.2506047785282135, -0.5374283790588379, -1.0753881931304932, 0.4033072888851166, -0.31052538752555847, -0.704859733581543, 0.050007350742816925, 0.3817892372608185, -0.33989474177360535, -0.5835385918617249, -1.2464467287063599, -0.7189969420433044, -0.27541932463645935, 0.41317740082740784, 0.9060773253440857, -0.4340374171733856, -1.2151621580123901, 0.9973726272583008, 0.09938184171915054, 1.39039945602417, 0.312106192111969, -0.6085138916969299, 0.025215625762939453, 0.2917906939983368, 0.6274585723876953, 0.7854342460632324, 0.9598442912101746, 0.3269076645374298, -0.7109829187393188, -1.555640697479248, 0.5602396726608276, -0.6334277987480164, -0.6557595133781433, -2.083373546600342, 0.24510495364665985, 0.1859232634305954, 0.9866975545883179, -0.6566701531410217, 1.7906196117401123, 0.0012210470158606768, -1.0085123777389526, 0.510928750038147, 0.5736755728721619, -1.8606805801391602, -0.5526708960533142, 0.718533456325531, -1.5130881071090698, -0.12097589671611786, 1.209872841835022, -0.04368986189365387, -0.8185274600982666, 0.3328540325164795, 0.7342578768730164, 1.2039320468902588, 0.879278838634491, 1.724238395690918, 0.138894185423851, 0.36857473850250244, -1.4690577983856201, -0.9397346377372742, 1.0756160020828247, -0.32947856187820435, -2.08798885345459, -0.7938075661659241, 0.10276425629854202, 0.442046582698822, -0.31497663259506226, 0.9043321013450623, -1.2874332666397095, 0.6344400644302368, -1.1865527629852295, 0.722341001033783, 2.0590598583221436, -0.672188937664032, 1.7212601900100708, -1.4432950019836426, -0.01651587523519993, -0.635153591632843, 0.20596003532409668, 0.030011825263500214, 0.4844094216823578], [0.8342146873474121, -0.12250760942697525, -0.8488922715187073, -0.5798164010047913, -1.636039137840271, 1.3116589784622192, -1.6183465719223022, 0.3011254072189331, -0.4742630422115326, 1.0118565559387207, -1.2394393682479858, -0.05136016756296158, -0.6706964373588562, 0.7402201890945435, 0.7790194153785706, -0.29242315888404846, -0.20772458612918854, -0.6310392618179321, -1.2209197282791138, -0.6922367215156555, 0.2767806053161621, -1.0860960483551025, 0.2280154526233673, -0.6464081406593323, 0.30168595910072327, 1.3995805978775024, -0.8841412663459778, -0.46927064657211304, 1.1213889122009277, 1.0443477630615234, -1.9593652486801147, -0.19986091554164886, 0.07075376808643341, 0.04764586314558983, 1.3824348449707031, 1.201219916343689, 0.9756210446357727, 0.9946750998497009, -0.8618550300598145, 1.5399867296218872, -0.2637051045894623, -1.8907761573791504, 0.3559432923793793, 0.691878080368042, -0.21880221366882324, -0.2496679574251175, 1.14419424533844, -2.3867430686950684, 0.06871041655540466, -0.1787445843219757, 1.4948524236679077, -1.213250994682312, -0.1921742707490921, -1.7813774347305298, -2.4976539611816406, -0.28511467576026917, -1.0886123180389404, -1.159000039100647, 1.2404013872146606, -2.2859859466552734, -0.7528620958328247, -0.9339544177055359, 0.5980991721153259, -0.06261570751667023, -0.16939818859100342, 1.6020344495773315, -0.2943762540817261, 0.15312694013118744, -1.0191118717193604, 1.4841865301132202, -0.21692173182964325, -1.3979568481445312, -0.5705472826957703, -0.09146731346845627, 0.17404426634311676, -0.6098057627677917, 2.0156710147857666, 0.011020619422197342, -0.5856367349624634, -1.4004172086715698, -0.894260048866272, 0.5224026441574097, -0.04254525899887085, -1.9285091161727905, 0.9203736186027527, -0.6433911323547363, 1.7212698459625244, 0.8027918934822083, 0.0040552932769060135, 0.291881799697876, 0.6410219669342041, 1.1616019010543823, 0.8138058185577393, 0.3965678811073303, -0.07355772703886032, 2.2922019958496094, 0.6213621497154236, -0.43042540550231934, -0.919670045375824, -0.12704962491989136, 2.2869668006896973, -0.15138457715511322, 0.37061789631843567, 0.2102544605731964, -2.3733458518981934, 0.3584131896495819, 1.2021880149841309, 1.0397965908050537, -0.2431361973285675, 1.2571725845336914, 1.3353556394577026, 1.8124390840530396, -0.9447221755981445, 0.9157294631004333, 1.6061838865280151, 0.45868828892707825, 2.379537343978882, 0.1221080794930458, -0.3113504946231842, 0.03579191491007805, -1.0554358959197998, 0.998371422290802, 1.5215513706207275, -0.20798902213573456, 0.21859464049339294, 0.71424400806427, -2.0085043907165527, -1.0037609338760376, -0.17935502529144287], [-0.2536853551864624, -0.2491910755634308, 0.2518021762371063, -2.240285634994507, -0.21134109795093536, 1.8253774642944336, 0.7524197101593018, 0.7382407188415527, 1.9541771411895752, -2.020890712738037, -1.5709666013717651, -0.6327381134033203, 0.7682916522026062, 0.4704391360282898, 1.585888147354126, 1.800573706626892, 0.09958285838365555, 0.5343618392944336, 0.5060216784477234, 0.48166099190711975, -0.6011850237846375, -0.15634165704250336, 0.9169077277183533, -0.9531683921813965, -0.34291595220565796, 2.5165748596191406, 0.9909045100212097, 0.5224893689155579, 0.5030342936515808, -1.6815848350524902, -1.11907958984375, 0.9775156378746033, 0.08984469622373581, 0.3097976744174957, 0.6383609175682068, 0.27906864881515503, 0.9810847043991089, -0.006814645603299141, -1.6388951539993286, 0.4735298752784729, 1.2841246128082275, 0.30677077174186707, -0.7647498250007629, -0.8792058229446411, -0.012902958318591118, -0.5975237488746643, -1.6703832149505615, 0.8132692575454712, -0.31467998027801514, 0.6768918633460999, -0.11240258067846298, 1.1725938320159912, -0.3462144434452057, -0.8515497446060181, 1.6617833375930786, -1.346500277519226, 1.170986533164978, -1.4432978630065918, -0.5607457756996155, -0.5356017351150513, 0.32624518871307373, -0.1996208131313324, -0.022630413994193077, 0.7071352601051331, 0.3975256085395813, -0.17670385539531708, -0.5744155049324036, 0.013352965004742146, 0.42750537395477295, -0.17345474660396576, 0.6907137632369995, 0.920933723449707, -1.189536690711975, -1.0746511220932007, 1.6072165966033936, -1.0120867490768433, -1.186148762702942, 0.4338529407978058, 1.2289283275604248, 1.8217796087265015, 1.1579666137695312, 0.00792519561946392, -1.1978825330734253, -1.1773147583007812, 0.5896194577217102, 2.1561007499694824, -0.8083184361457825, -0.24143381416797638, 1.2872092723846436, -2.012166976928711, 0.24139682948589325, 1.188180685043335, -0.6143377423286438, -0.7597612142562866, 0.7655880451202393, -1.1598305702209473, 1.0212260484695435, 0.5158328413963318, -0.3493829369544983, 2.3050425052642822, -0.42579659819602966, -1.1108275651931763, 0.5183660984039307, -0.020517893135547638, 0.9384264945983887, 0.7391674518585205, -1.9980533123016357, 0.10795123130083084, -0.12023477256298065, 0.17682436108589172, 0.6713252067565918, 1.193385362625122, -1.2349255084991455, 0.7599030137062073, -0.6993517875671387, -0.2636289894580841, 1.6407701969146729, -0.04945754632353783, 0.10673500597476959, 1.0403205156326294, 0.22744770348072052, 1.131148338317871, 1.06047785282135, -0.20276343822479248, -0.31632766127586365, -2.1114463806152344, -0.47013387084007263, 0.3540612459182739, -1.5755128860473633], [-1.0459460020065308, 0.4149828255176544, 0.22703632712364197, 1.4510003328323364, 0.5248303413391113, -1.3796623945236206, 0.168410524725914, -1.169165015220642, -0.29006320238113403, 0.5175068974494934, -1.730893611907959, 1.0485512018203735, -1.0816679000854492, 0.5457390546798706, -0.2792885899543762, -0.6464244723320007, -0.881455659866333, -0.19202260673046112, -0.9334202408790588, 1.5213819742202759, 2.346204996109009, 1.948914647102356, 0.3270646333694458, 1.469309687614441, -0.37375253438949585, 0.16772237420082092, -0.5389243364334106, 1.091812252998352, 1.4570372104644775, 0.13227583467960358, 0.9868831634521484, -1.1134694814682007, -0.5404835939407349, -1.2354940176010132, 0.5918610095977783, 1.5867695808410645, -0.023513514548540115, -0.8147138953208923, 0.27817046642303467, 0.4049908518791199, 0.7939021587371826, 1.7323731184005737, -0.110514335334301, 0.24234513938426971, -0.22648608684539795, -2.2099592685699463, 0.3618203103542328, 1.3201416730880737, -1.3320462703704834, -0.35214102268218994, 1.2772672176361084, 2.4984171390533447, -0.7383975386619568, -2.4966564178466797, -0.4492129981517792, -0.2882249653339386, -0.9368404150009155, -0.9011956453323364, 0.4365271031856537, 1.189551591873169, -1.587279200553894, -0.7395166754722595, -0.003795949276536703, -0.3096902370452881, -0.6513327360153198, 1.0040936470031738, 0.5251955986022949, 1.8801652193069458, 0.07770764827728271, -0.323095440864563, 0.969188392162323, 1.0528310537338257, -1.0874947309494019, -0.6534367203712463, 0.4398249089717865, 0.5979170203208923, -0.022779151797294617, -0.13275910913944244, 0.017036905512213707, -0.5242453813552856, 0.8929156064987183, 2.235752582550049, -1.1822420358657837, -0.041382208466529846, 0.8442642688751221, -0.8316025137901306, -0.6440796852111816, 0.16283190250396729, 1.6521023511886597, -0.26386526226997375, 1.238631010055542, 3.0244617462158203, 2.799116849899292, -0.49120014905929565, -1.3081430196762085, -2.5184836387634277, -0.20279601216316223, 1.406035304069519, -0.3409402370452881, -1.42897367477417, -0.8676183223724365, 0.8380752205848694, 1.342787742614746, -1.4458860158920288, -1.4095401763916016, 0.15892867743968964, 0.05221033841371536, 0.5280448198318481, -1.0568708181381226, 0.5741760730743408, 0.16040930151939392, -0.23145504295825958, 1.2965459823608398, -0.270448237657547, 0.45591580867767334, -0.5523518919944763, -1.0143533945083618, 1.1471812725067139, -2.1079978942871094, 0.6812587976455688, -0.6601383090019226, 0.9531204700469971, -0.866102933883667, -0.19688862562179565, -0.22185391187667847, 1.1021718978881836, -0.5921440124511719, 0.1260741949081421, 0.6301616430282593], [0.7117307782173157, 0.34709325432777405, -1.383217215538025, -1.9593441486358643, -0.04904552921652794, -1.6874637603759766, -0.7309828400611877, 0.09287282824516296, 0.7875412106513977, -0.871063768863678, 0.5927766561508179, 0.019157888367772102, -1.4841828346252441, 0.49718567728996277, 1.843368649482727, -0.4020138680934906, 1.3673851490020752, -1.4579414129257202, -0.867709219455719, 0.27390122413635254, -0.3229692578315735, -0.2142975926399231, -0.4256584942340851, 1.452133059501648, -1.1144039630889893, 0.9857773780822754, -0.10831399261951447, 0.7603274583816528, 2.163666009902954, 0.13532119989395142, -0.07793153077363968, 0.08770659565925598, -0.47380873560905457, -0.6013703942298889, -1.3440251350402832, -1.8341017961502075, -3.019071340560913, 0.034280452877283096, -0.7660841345787048, 0.8551142811775208, 1.495548963546753, 1.6851524114608765, 0.2250286042690277, 0.7369768619537354, 2.023991823196411, 0.06962857395410538, -0.11686830222606659, -1.1068649291992188, 1.0570456981658936, -0.5040079355239868, 1.013870120048523, 0.0008894254569895566, -0.6097541451454163, 0.9593330025672913, 2.0346179008483887, 0.22755838930606842, -1.6513372659683228, 3.0781567096710205, 0.9322742819786072, -0.2294117957353592, 0.836165189743042, 0.25278300046920776, -1.501941204071045, 0.33545875549316406, 0.10765647888183594, -0.6087637543678284, 1.1508454084396362, 0.059742797166109085, 0.12093750387430191, 1.6842283010482788, 0.568230152130127, 0.5078943967819214, 0.5418099761009216, -0.5642590522766113, -0.660834550857544, 0.624828577041626, 0.382232666015625, 0.7711731791496277, -2.3940820693969727, -0.21360869705677032, 0.349527508020401, 0.07785919308662415, 0.36862465739250183, -2.6636123657226562, 0.36952799558639526, 0.8426536917686462, 0.7223027944564819, 0.7330034971237183, 0.04601571336388588, -0.6774137616157532, -1.127417802810669, 1.3849515914916992, 1.7035645246505737, -0.4001586437225342, 0.9167816638946533, -0.6725449562072754, 0.0025987508706748486, -1.321743369102478, 0.8833102583885193, 1.4055604934692383, -1.4495854377746582, 0.7116076350212097, 1.374640941619873, 0.305477112531662, 1.6291307210922241, -1.1636215448379517, -0.15090245008468628, -1.7274317741394043, 0.30810436606407166, -0.3385842442512512, -2.0399327278137207, -0.6872280240058899, 0.8361632227897644, -0.41061538457870483, -0.32037562131881714, 0.574867844581604, 0.5587500333786011, 0.1089184507727623, -0.940090000629425, 0.7736429572105408, -0.44898003339767456, -1.6265791654586792, 0.26771804690361023, 1.5761743783950806, 0.6667407155036926, 0.3703157901763916, -0.5759732723236084, -0.3200458884239197, -0.44888776540756226], [-2.2818424701690674, -1.700685977935791, 0.30221429467201233, -0.9238507747650146, -0.6716028451919556, -0.43805018067359924, 0.2178480625152588, -2.4245104789733887, 0.6070351004600525, -0.4513739347457886, 0.1201082170009613, 2.147214412689209, -0.5814965963363647, -0.06911846250295639, -2.381152868270874, -0.32287466526031494, -0.7699427604675293, 0.38046780228614807, 0.0539090633392334, 0.8594542741775513, 0.7182927131652832, 0.8129525184631348, 0.5186164379119873, 0.2503724694252014, -1.6197272539138794, -0.9520279765129089, -0.7393022775650024, 1.932492971420288, 1.2712409496307373, -1.0600428581237793, 0.9640640616416931, -0.5634731650352478, -1.457580804824829, 0.006819492671638727, -1.0357558727264404, 0.2198452204465866, -0.8955414295196533, -0.48935896158218384, 0.022259389981627464, -1.5726114511489868, 0.638106107711792, -1.0224889516830444, -0.2934334874153137, -1.7360162734985352, -1.3748406171798706, -0.718370795249939, -0.2076486200094223, -1.0483989715576172, -0.9856054782867432, -0.456041157245636, 0.21723826229572296, -0.245387464761734, -0.008916596882045269, 1.2385224103927612, -2.3567473888397217, -0.004909045062959194, 0.5431312918663025, -0.5257965326309204, -0.8616572022438049, 1.2859270572662354, 1.6655842065811157, 0.16694580018520355, -2.253556489944458, 1.1362414360046387, 1.204811692237854, -1.989896297454834, 0.9182308316230774, 0.8630332350730896, -2.135152816772461, -1.475976586341858, -0.7619195580482483, 0.8218115568161011, -0.39414694905281067, 0.014299333095550537, -1.0858685970306396, 0.5133460760116577, 1.0553795099258423, 0.38986125588417053, 0.47332534193992615, -1.3640577793121338, -1.1083866357803345, -0.37070927023887634, 0.6955403089523315, 0.5779702067375183, 1.6831485033035278, 0.7386878728866577, -0.9596279859542847, -1.1033847332000732, -1.7757560014724731, -1.4240038394927979, 1.7900221347808838, 0.22482480108737946, -0.468696653842926, -0.02402029186487198, -0.07385122030973434, -0.8446165919303894, -1.362294316291809, 1.3705101013183594, -0.7261791825294495, -0.6445415019989014, -1.4330261945724487, 0.19729012250900269, -0.6862354874610901, -0.9870824217796326, -1.446609377861023, 1.5420726537704468, -1.0190918445587158, 0.6170975565910339, -0.2959720492362976, -0.23297584056854248, 0.0764697939157486, -0.8690871000289917, -1.4167110919952393, 0.21210309863090515, 0.4785601794719696, 1.3838376998901367, 1.5495071411132812, 0.22488966584205627, -0.13797861337661743, -1.7410979270935059, -0.6274294257164001, 0.7215778231620789, -0.10873042792081833, 0.5148885846138, -1.8932219743728638, 0.880250096321106, 0.11829434335231781, -1.3815768957138062, -0.028507519513368607], [0.974623441696167, 1.1242258548736572, 0.1249513104557991, -0.11305885016918182, -0.968238115310669, 0.18144889175891876, 1.1069858074188232, 0.5463314056396484, -1.0672491788864136, 0.4727681279182434, -0.17031125724315643, -0.019098548218607903, -1.7771821022033691, 0.06178026273846626, -0.2346131056547165, -0.9041517972946167, -1.5565754175186157, -1.0447659492492676, 1.5252734422683716, 0.17076627910137177, 0.6873756051063538, -1.1055141687393188, 0.46117889881134033, 0.9790468811988831, -0.3439292013645172, 0.37605464458465576, -1.2315635681152344, -0.51146000623703, 0.29568153619766235, 0.17598354816436768, -0.6413837671279907, 0.10480117052793503, -0.9011104106903076, -0.8669751286506653, 0.7700328230857849, 1.0147593021392822, 1.346128225326538, 1.3449763059616089, 0.37427031993865967, 0.2857833504676819, 0.6501098871231079, 0.20807066559791565, -0.2988594174385071, -0.6765520572662354, 1.6875454187393188, 0.6860663890838623, -0.16190111637115479, 0.6591624617576599, -1.2795705795288086, 1.5398412942886353, 1.1358444690704346, 0.6068981289863586, -2.5778298377990723, 2.3935389518737793, -0.6336579322814941, 1.1815017461776733, -0.5640174746513367, -2.0043814182281494, -1.5358937978744507, -0.6193434000015259, -0.6496019959449768, 0.7320255041122437, -0.38963180780410767, 1.9179593324661255, -1.107921838760376, -0.9025595188140869, -0.33207935094833374, 0.37617769837379456, 0.5815194249153137, -1.1141881942749023, 0.03646497800946236, 0.026117928326129913, 0.8593300580978394, 0.3027796447277069, 0.4192037880420685, -0.6626824140548706, 1.4538402557373047, 0.017520729452371597, 0.5424454808235168, -0.6387573480606079, -2.689835548400879, 0.733898937702179, -0.210499569773674, 1.6855034828186035, -0.8945308327674866, -0.6180378794670105, 0.6421697735786438, -1.437197208404541, -1.6772217750549316, -0.5059545636177063, 0.5133237838745117, 0.7071349024772644, -0.11714663356542587, 1.7776042222976685, 0.7722575068473816, -0.7079719305038452, -0.019160103052854538, 0.2152886837720871, 0.28236818313598633, 0.35557717084884644, 3.157021999359131, -0.0943901389837265, 1.8550864458084106, 0.7460645437240601, -1.5651121139526367, -0.4199505150318146, 0.12540335953235626, -0.8357417583465576, -1.4257466793060303, -0.710902214050293, -0.9660596251487732, 0.5479187369346619, -0.15048688650131226, 0.9355776309967041, 0.015468579716980457, -1.7471721172332764, 0.35812488198280334, 2.286808967590332, -0.4668378531932831, -0.2474098801612854, 0.15663792192935944, 0.44139716029167175, 0.3657741844654083, -2.1135940551757812, -1.3843845129013062, 0.09549589455127716, 0.939028799533844, -1.100615382194519, -0.4358938932418823], [-0.7435250282287598, 0.8950061202049255, -0.7764968872070312, -0.06899378448724747, 0.2147616147994995, -0.24291646480560303, 0.6420902013778687, -0.5715790390968323, -0.3667161464691162, -0.36635488271713257, 0.6765778660774231, 0.0783621221780777, -0.14510685205459595, -0.3134515881538391, 0.1627057045698166, -0.42337357997894287, 0.5735461711883545, 1.2512661218643188, -1.6530288457870483, -0.31955981254577637, 3.0052266120910645, 0.16953985393047333, 1.4323729276657104, -0.30347010493278503, -0.4117184281349182, -1.8901578187942505, -0.8696262240409851, 1.2659052610397339, 1.327467441558838, -0.5981906056404114, -2.241988182067871, -0.8542352318763733, -0.48766160011291504, -1.1059015989303589, 0.7297345399856567, -0.47328388690948486, -0.41085731983184814, -0.7557620406150818, -0.7352669835090637, 0.5223509073257446, -0.8758254647254944, -0.6235187649726868, 0.2242904156446457, -2.736974000930786, 1.675998568534851, -1.6624008417129517, 2.6306190490722656, -0.3605572283267975, 0.4056068956851959, -0.6411389708518982, -0.5816040635108948, 0.8699203133583069, -0.5361490249633789, 0.2480422854423523, 0.8509182333946228, -0.3255196809768677, -1.144526720046997, -1.6051702499389648, 0.5022647380828857, -1.1654362678527832, 1.879348635673523, 0.06415163725614548, -0.15294106304645538, 1.372695803642273, 0.06478328257799149, 1.2874054908752441, -1.7547322511672974, 1.4192707538604736, -0.3253638446331024, 0.5285077095031738, -0.0031046706717461348, 0.35817480087280273, 0.07913710176944733, 0.5296863317489624, -1.160725712776184, -0.6438692212104797, 0.7354339957237244, -0.6965686082839966, 0.2088034451007843, -0.7053918838500977, -0.8474633693695068, -1.0218167304992676, -1.1465293169021606, 1.0089002847671509, -0.786457359790802, -0.9254509210586548, -1.7749110460281372, -0.5686843395233154, 0.4256625771522522, 0.06599600613117218, 0.11452777683734894, 1.101746916770935, -1.3242677450180054, -0.11150728911161423, 0.7121172547340393, 0.33441925048828125, 0.3308580219745636, -1.3175890445709229, -1.4808928966522217, -0.567751407623291, 0.6168566346168518, 1.878016471862793, 0.917881965637207, 1.0241774320602417, 0.7737210988998413, 1.7450811862945557, 1.8357149362564087, 0.9774581789970398, 0.7657365202903748, 0.1917196810245514, 0.6589443683624268, 1.1660659313201904, -1.8119996786117554, 0.149737149477005, -2.067061424255371, -0.11917730420827866, -0.9243262410163879, 0.0643475353717804, 1.1065412759780884, -0.6844411492347717, 0.5632027983665466, 0.844663679599762, -0.5936855673789978, -1.2672210931777954, 0.8913871645927429, 2.1016945838928223, -0.6281466484069824, 0.32094231247901917, 2.6915087699890137], [0.4520241618156433, 0.305117130279541, -0.6077831387519836, -0.48652806878089905, 0.6241134405136108, -0.23669154942035675, 0.15612323582172394, -0.8249478340148926, -0.6588864326477051, 1.1010146141052246, -0.8557401299476624, 1.2523428201675415, 0.4361964464187622, 1.1460673809051514, 0.6132333278656006, 1.307790756225586, -0.6248276829719543, -0.2178628295660019, -1.2290273904800415, 0.7706283926963806, -0.4762699604034424, 0.3100312352180481, -0.676857054233551, 0.9222334027290344, -0.8493802547454834, 0.5780021548271179, -0.1064370721578598, -0.24929364025592804, -0.791503369808197, -0.28807246685028076, 0.12801358103752136, 1.3929144144058228, -1.328436255455017, -0.4415861964225769, 0.15000616014003754, -0.2044302076101303, 0.4822719991207123, 0.9530662298202515, -1.7482719421386719, 1.21699857711792, -2.2541916370391846, -0.05091668292880058, -0.6019090414047241, 0.2241937220096588, -0.8636733293533325, -0.927868664264679, 0.020216461271047592, 1.7272306680679321, 1.583495855331421, 0.3795546293258667, -1.0901424884796143, -0.2931838929653168, -1.9296406507492065, -1.0867031812667847, -1.2574986219406128, 0.731647253036499, -0.6655154228210449, -0.6792460083961487, 1.306535243988037, 0.6583134531974792, 0.5663878321647644, 0.8642302751541138, -0.5213684439659119, -0.6286777853965759, -0.10486558824777603, 0.6405760049819946, 0.6437442898750305, -0.02874888852238655, 0.9084888696670532, -0.6033614277839661, 0.1038735881447792, 0.7261459827423096, -0.8970803618431091, 0.4744041860103607, 0.0814150869846344, 0.490648478269577, -0.6031540632247925, -0.12935440242290497, 1.1359831094741821, -1.810748815536499, -0.29808974266052246, -0.05621396377682686, 1.3965266942977905, 0.5963099598884583, -1.5518279075622559, 0.08216622471809387, 0.1439451277256012, 1.6775845289230347, 0.18470659852027893, -0.07844963669776917, -2.245661735534668, 0.46475499868392944, -0.30721694231033325, 2.5681982040405273, 1.0530270338058472, -0.9672324061393738, 2.4575018882751465, -1.1632565259933472, -0.0611678846180439, -0.08716658502817154, -1.8843092918395996, 1.2227784395217896, -1.7466926574707031, 1.4312078952789307, -0.24141071736812592, -2.248821258544922, 1.7726898193359375, -0.2891271710395813, 0.7092724442481995, -1.1789133548736572, 0.5325644016265869, -0.7774280905723572, 0.03267022594809532, 1.8236100673675537, 2.039219617843628, 0.4223874509334564, -0.5220897197723389, -0.8811919093132019, 0.9525629878044128, -0.26687878370285034, -0.369950532913208, 0.7472519278526306, 0.45997506380081177, 0.448236882686615, 0.11038023233413696, -0.4120689630508423, 0.481792151927948, -1.2708078622817993, -0.39080238342285156], [0.933280348777771, -2.0084891319274902, 1.2532627582550049, 0.8232840299606323, -1.3297899961471558, -1.1472649574279785, -0.6721388697624207, -1.3977128267288208, -1.5390108823776245, -1.7709245681762695, -0.45190805196762085, 1.1551624536514282, -0.534162163734436, 0.45468541979789734, 1.1987452507019043, 0.9165911674499512, 0.19265128672122955, -0.23707062005996704, -0.17546851933002472, 0.9796726107597351, 0.450708270072937, -0.06356233358383179, 0.17633460462093353, -1.6594401597976685, -2.245049476623535, -1.52411949634552, -0.2730367183685303, -0.08298054337501526, -0.47224336862564087, -0.6137093901634216, -0.35926854610443115, -1.1702419519424438, -0.6889742612838745, 0.2963259816169739, -0.519465982913971, 0.7714249491691589, -0.5883215665817261, -0.32722803950309753, -0.39570993185043335, -0.43034133315086365, -0.45488789677619934, 0.18198448419570923, -1.0486963987350464, 1.6920440196990967, 1.0567433834075928, 0.05993799865245819, -0.2761884927749634, -0.47513583302497864, -1.467864990234375, 0.11060995608568192, 0.16795577108860016, -0.47748038172721863, 0.6132699847221375, 0.15831062197685242, 0.02628541924059391, 0.7677027583122253, -0.01960637979209423, -0.47192439436912537, 0.743805468082428, -1.028383731842041, -0.2675928771495819, 0.8900824785232544, -0.7949427962303162, -0.5581073760986328, -0.637610137462616, -0.3547314703464508, -0.30331751704216003, 1.1371656656265259, 0.9814428687095642, 0.17483209073543549, -0.47359198331832886, -0.7396326661109924, -0.6776471734046936, -0.3192989230155945, 0.11383431404829025, -1.6623241901397705, -0.21463026106357574, -1.5536268949508667, 0.7856020927429199, 0.6206287145614624, -0.7256124019622803, -0.12395029515028, -1.9748659133911133, -1.299772024154663, 0.5518907308578491, -0.9038575291633606, 0.021313508972525597, -0.9417768120765686, 0.06636696308851242, 1.1016857624053955, 0.6496747732162476, -0.7159314751625061, 0.8464682102203369, -2.1275601387023926, -0.25559020042419434, 2.436951160430908, 1.8594334125518799, -0.39904260635375977, 1.921220302581787, 0.778806746006012, 0.34840044379234314, 0.25124213099479675, -1.8925608396530151, 1.0014169216156006, -0.7366756200790405, -1.5194942951202393, -0.369698166847229, 0.8638307452201843, 2.286076784133911, 0.6651830673217773, 0.2897509038448334, -0.932558536529541, -0.5409477353096008, -1.3881748914718628, -0.26565128564834595, 2.210240602493286, -0.9156851172447205, -0.12960204482078552, -0.8314476609230042, 1.4817007780075073, -0.2811420261859894, 1.2624647617340088, -1.0803463459014893, 0.6812101006507874, -1.1896635293960571, 0.6903480887413025, 2.243589162826538, 0.05505475401878357, -0.13015341758728027], [-1.7783496379852295, -1.9696513414382935, -0.45383334159851074, -1.4011772871017456, 1.767844796180725, 0.49453604221343994, -1.1973432302474976, -0.23549488186836243, -0.8988034725189209, -1.2513713836669922, 0.9367795586585999, 0.3098263740539551, -1.3240779638290405, -0.9547296166419983, -1.6656168699264526, -0.2772771120071411, -0.3128369152545929, 1.9073046445846558, -0.8398210406303406, -1.1183102130889893, -0.4877431094646454, -0.3848676383495331, -1.0697951316833496, -0.9417210817337036, -0.02171327732503414, -0.4316326081752777, -0.9696723222732544, -0.7271670699119568, 0.41876426339149475, 1.5720126628875732, 1.3352198600769043, 0.36936742067337036, -0.21105796098709106, 0.5824199914932251, -0.9633444547653198, 0.18817949295043945, -1.3638510704040527, 0.08324840664863586, -1.2511425018310547, -1.0576707124710083, -0.32798677682876587, -0.9699193835258484, -0.9182952046394348, -0.3921590745449066, -1.6880306005477905, -1.129428505897522, -2.4910402297973633, -1.045699954032898, -0.9498558640480042, -0.7743989825248718, -0.12952613830566406, 1.1066352128982544, -1.4623334407806396, -0.72733473777771, -0.07214998453855515, -1.4167357683181763, -2.1751461029052734, -1.0254197120666504, -0.8554087281227112, 0.4211525619029999, 0.3574150800704956, 0.8287881016731262, 0.1405908316373825, -1.360143780708313, -1.806302785873413, -1.3375803232192993, 1.2129995822906494, 1.090072751045227, 1.1682666540145874, 2.092350482940674, -1.2785651683807373, 0.1622096300125122, 2.276062250137329, 0.8191326260566711, 1.562071681022644, 0.9427140355110168, 0.1710684895515442, -0.7108839750289917, 0.2791050970554352, -0.46514955163002014, -0.941824197769165, -0.5220664143562317, -1.3502910137176514, -0.4421945810317993, -0.20633885264396667, -0.47822585701942444, -0.12079214304685593, 0.0670747309923172, -0.14694611728191376, 0.06601503491401672, 2.8773322105407715, -0.9177343845367432, -0.5774737596511841, 0.7204259634017944, 0.5973387360572815, 1.4316447973251343, -1.2315747737884521, 0.41937702894210815, -0.12590186297893524, -0.3283655643463135, -1.5445634126663208, 0.3656274080276489, 0.32698434591293335, -1.4012202024459839, 0.42774713039398193, -0.5392448306083679, 0.2820165753364563, 0.08701223134994507, -0.59935462474823, 0.6975123286247253, -1.4416134357452393, 0.05387319624423981, 1.7605977058410645, -1.7456161975860596, 0.8063362836837769, 0.288499653339386, 1.1387665271759033, -0.32097873091697693, 2.2530767917633057, -1.8859128952026367, 0.3310811221599579, 0.7103149890899658, -2.01725435256958, -1.1573946475982666, -0.43950504064559937, 0.8186255097389221, 2.6079869270324707, 0.8118948936462402, -0.47164788842201233], [0.24683810770511627, -2.600606918334961, -1.5680960416793823, -0.41246363520622253, 0.6425274610519409, -0.08982731401920319, -0.08603423833847046, -0.230416402220726, 1.8670496940612793, 0.24934278428554535, 0.431101530790329, 1.098223328590393, -1.6237937211990356, 1.6135836839675903, -0.5185282230377197, 0.4857088625431061, -0.5567746162414551, 1.6997548341751099, 1.9272964000701904, 1.7967054843902588, 1.261518955230713, 0.6686777472496033, 0.37200114130973816, 0.450756311416626, -0.18988308310508728, -0.16262738406658173, 2.7753095626831055, 0.6591176986694336, 1.0932101011276245, 0.42923063039779663, 1.705287218093872, 1.093335747718811, -0.7230983376502991, 1.856209397315979, 0.32695817947387695, -2.9952139854431152, 0.4769231975078583, 0.6164166927337646, 0.7672669887542725, 1.188502550125122, -0.5476663112640381, 0.4436159133911133, 0.22475597262382507, 0.16136139631271362, 1.9336062669754028, -0.9473216533660889, 1.51906156539917, -0.45749273896217346, 1.5079487562179565, 0.3135088086128235, -1.0103000402450562, -0.6298106908798218, -0.3860827684402466, -0.2657654881477356, 0.8835498094558716, 0.7662997245788574, 1.2635769844055176, 0.18253906071186066, -0.0714360699057579, -0.5833940505981445, -0.7817228436470032, 0.4758825898170471, -0.8168778419494629, -0.9224976301193237, -0.6021175980567932, 0.5387541651725769, 1.0078699588775635, 0.3034563362598419, 0.42788317799568176, -1.1013336181640625, -0.4018106460571289, 0.45548319816589355, 0.5275823473930359, -0.9131312370300293, 0.8142396807670593, 0.3988667130470276, 0.5967451930046082, 0.11887318640947342, -1.00651216506958, 0.3461056351661682, 1.618215799331665, 0.23943330347537994, 1.6689223051071167, -0.4232903718948364, -0.3712529242038727, -1.489562749862671, 1.214448094367981, -0.008925172500312328, 1.3279387950897217, -0.6713364124298096, -1.4478408098220825, 0.8694267272949219, -0.8193867206573486, 2.302729368209839, 0.13910046219825745, -1.576880693435669, 0.6805267930030823, 2.1781373023986816, -1.14182448387146, 1.2707409858703613, -0.16490361094474792, 0.22115254402160645, 0.1866709142923355, 0.10898391157388687, -1.3220924139022827, 0.8372688889503479, -0.585288941860199, 0.11627545207738876, -1.7460132837295532, -1.7531898021697998, 1.6641108989715576, 1.0115108489990234, 0.7028087973594666, 0.7531153559684753, -1.5737783908843994, 0.18988549709320068, 1.8380666971206665, 0.8206533789634705, -0.9460757970809937, -0.41638612747192383, 1.2914085388183594, 1.280757188796997, -0.010075709782540798, -1.7670953273773193, 1.051140308380127, 0.4389190971851349, -0.9482583999633789, 1.383446455001831, 1.2534222602844238], [0.47629302740097046, 1.2867611646652222, 0.7273281216621399, 0.7345000505447388, 0.19291138648986816, -0.09386236220598221, 0.8948737978935242, 0.7072538137435913, 1.1937382221221924, -2.9525656700134277, 0.36263102293014526, -2.0591611862182617, -0.18261383473873138, 0.04048793017864227, -0.8337305188179016, -0.4588016867637634, -1.8693829774856567, -0.09641560912132263, -0.47500327229499817, 0.087906613945961, -0.7830678224563599, 1.3659723997116089, -1.2869149446487427, 2.4675981998443604, 0.10513250529766083, -0.31427907943725586, 0.33849775791168213, 1.5008463859558105, 1.1463061571121216, -0.12830160558223724, 0.65237957239151, -0.5780650973320007, -1.1755605936050415, -0.05949103832244873, -0.7631549835205078, -1.0403413772583008, -0.6374601721763611, 0.18540462851524353, -1.6074135303497314, -0.7365431189537048, -0.7490853071212769, -0.6138860583305359, -0.06898719817399979, 1.0057471990585327, -0.8145198822021484, 0.712020754814148, -0.31811705231666565, 0.9253513813018799, -0.49672046303749084, -1.7956163883209229, -0.5107833743095398, -2.1687064170837402, 0.2951487898826599, -0.7757321000099182, -1.0917855501174927, 2.200899600982666, 1.0914387702941895, -0.6182214021682739, -0.07179724425077438, -0.36691197752952576, 0.8588973879814148, 1.1111900806427002, 1.1944022178649902, 1.323094367980957, -0.5659797191619873, 0.5124015808105469, -1.5055795907974243, -1.3612216711044312, 1.1932750940322876, -0.22386868298053741, -1.2118147611618042, 0.373273104429245, -2.224336624145508, 0.26331308484077454, -0.7446613311767578, 1.2524038553237915, 0.2240407019853592, 1.3851375579833984, -0.2586933970451355, 0.09080293774604797, -0.8796459436416626, 0.27760130167007446, -3.149807929992676, 0.6985540986061096, -0.521628201007843, -0.09262847900390625, -0.7681265473365784, -0.2669938802719116, -2.507091760635376, 0.7701168060302734, -0.8372827172279358, 0.8023662567138672, -0.905652642250061, -0.46210405230522156, -0.13579638302326202, 1.2644792795181274, 0.024288559332489967, -0.3028668165206909, -0.22431440651416779, -0.12041013687849045, 0.28285473585128784, 0.08480411022901535, -0.23971661925315857, 0.11120587587356567, 0.50248122215271, -0.7891645431518555, 0.19127406179904938, 1.081882357597351, -0.5109966993331909, 0.7036781311035156, 0.9896562099456787, 0.06447922438383102, -1.9578198194503784, 0.4024280607700348, 0.8980332016944885, -0.00706190662458539, 0.8237939476966858, 0.924579918384552, 1.1004644632339478, -0.25159674882888794, 1.0244855880737305, -0.33767417073249817, -0.19098559021949768, 0.281033456325531, 0.24061432480812073, 0.16947808861732483, 1.7264304161071777, 1.1850240230560303, 0.15225239098072052], [1.0949068069458008, 0.023156099021434784, -0.8643968105316162, 0.15241114795207977, -2.847886085510254, -1.053165316581726, -0.33628687262535095, 0.34958958625793457, -2.197127342224121, 0.660934329032898, 1.0151795148849487, 0.9727737307548523, -0.26204627752304077, 0.43386244773864746, -0.701559841632843, 0.5349887609481812, -0.9777250289916992, 0.02505587600171566, -1.2780938148498535, 1.4206528663635254, 0.6988677978515625, 0.9405140280723572, 0.2492608278989792, -1.7931668758392334, 0.574675440788269, -1.8368642330169678, 0.8813326954841614, 0.8745288848876953, 0.22897495329380035, -0.07337646931409836, -0.5835686922073364, 0.7985122799873352, -1.1287716627120972, 0.43066442012786865, -0.3767870366573334, -0.929320216178894, -0.5248026847839355, 1.4459452629089355, 2.636211633682251, -0.7381763458251953, 0.17531688511371613, -1.3755708932876587, 0.620211124420166, 0.49180278182029724, 0.004289789590984583, -0.07144080102443695, 1.343992829322815, 0.8164803981781006, 0.16235966980457306, -0.5065979361534119, 0.06555047631263733, 0.21844130754470825, -1.3203431367874146, 2.25657320022583, 0.5272454619407654, -1.5716028213500977, 0.5786588788032532, 0.18747153878211975, -1.2030764818191528, -0.8790837526321411, -0.36445263028144836, 0.0009429987403564155, 1.4971202611923218, 0.7211856842041016, 0.2009110450744629, 0.23415303230285645, -0.4505299925804138, -0.46662265062332153, 0.14757350087165833, 1.3344639539718628, -0.35772186517715454, 1.1034213304519653, -0.44551336765289307, 0.10723461955785751, 0.4360310137271881, 1.6935133934020996, 0.21797260642051697, 2.2806806564331055, 0.8520830869674683, 0.20556959509849548, -0.025474248453974724, 0.304663747549057, 0.7565461993217468, 1.391860842704773, 0.5043674111366272, 0.00803801417350769, -0.14587049186229706, -0.525000810623169, -1.4907578229904175, -1.0210810899734497, -0.5205575823783875, 0.5915374159812927, -1.4077485799789429, -0.7550772428512573, -0.9769766926765442, -0.957869291305542, -0.04316053166985512, 1.2569979429244995, 0.83760005235672, 0.2225833684206009, 1.2991938591003418, -1.4873085021972656, 0.15166324377059937, -0.4380161464214325, -0.6176486015319824, 0.6458414793014526, -1.888951301574707, -0.013713520020246506, 1.2147126197814941, 0.7592272162437439, 0.5539976954460144, 1.4449282884597778, 2.045935869216919, 0.47173967957496643, 0.4135299324989319, 0.7686730623245239, -0.29081806540489197, 0.3457392156124115, 2.0548791885375977, 0.35069745779037476, -0.5283406376838684, -0.30911362171173096, 0.1750074177980423, -1.6779643297195435, -3.6334598064422607, -1.4341157674789429, -1.9017058610916138, 1.8181136846542358, -0.059548258781433105], [-0.22649094462394714, -0.6385089159011841, 0.17118589580059052, 0.9667553901672363, -0.16154955327510834, 0.8216953277587891, 1.2229129076004028, 1.6024432182312012, -2.1372218132019043, 0.3463531732559204, -1.291630506515503, -0.6650466322898865, 0.9954418540000916, 1.17183518409729, -0.11322156339883804, -0.14608757197856903, -0.12355336546897888, -0.29704171419143677, 0.9703826308250427, -1.0698094367980957, -0.00899601262062788, 1.459805965423584, 1.4309850931167603, 0.19483385980129242, 1.005846381187439, 1.1937150955200195, -0.6027931571006775, 0.28521594405174255, -1.668156623840332, -0.07874296605587006, 3.0077977180480957, -0.35983026027679443, -0.5792994499206543, 0.015224503353238106, 0.010705657303333282, 0.24264781177043915, 0.7586325407028198, 0.24396905303001404, -0.554488480091095, 0.257861852645874, 0.19267213344573975, 1.0486674308776855, 1.0188018083572388, -1.0114595890045166, 1.541330337524414, 0.06341878324747086, -1.5846391916275024, 0.7430551648139954, -0.243840292096138, -0.13731494545936584, 1.5446100234985352, -0.8561819791793823, 1.060496211051941, -0.6177667379379272, -0.2360815405845642, 0.5120046138763428, 0.3880380690097809, 0.08527947962284088, -0.8586889505386353, -1.5997042655944824, 1.561692476272583, 0.3635477125644684, -2.429579019546509, -1.2132774591445923, 0.9114647507667542, 0.7820141911506653, -0.5631855726242065, -0.6246475577354431, -0.20460698008537292, -0.1180238202214241, 0.9908183217048645, -0.04153953120112419, 0.3769472539424896, -1.0930224657058716, 0.15513399243354797, 0.6230801939964294, -0.4369785189628601, 0.5644370913505554, 1.8657951354980469, 1.0737695693969727, 0.700507402420044, 1.0287171602249146, -0.7185155153274536, 0.5594324469566345, -2.1603810787200928, 0.2458699643611908, 0.7591617107391357, 0.20442993938922882, 0.567833423614502, 0.6972290277481079, -1.32557213306427, 0.5587899088859558, -0.05149111896753311, -0.4284229278564453, 0.33346787095069885, 2.791667938232422, 2.7184219360351562, -2.296137571334839, 0.06970805674791336, 0.2646681070327759, 2.55367374420166, -0.0495096854865551, -1.103894829750061, -0.511951744556427, 0.6126906871795654, 1.5376391410827637, 0.9994819164276123, -1.352787733078003, 1.8391327857971191, 0.627088725566864, 0.6581363081932068, 0.12584249675273895, 0.4836627244949341, 0.936132550239563, -1.5497591495513916, -0.5756442546844482, -2.6052095890045166, -0.38798388838768005, -0.27764400839805603, 1.408830165863037, 0.7052892446517944, 1.337829828262329, -1.1867860555648804, -1.4937710762023926, -1.455924153327942, -0.18649204075336456, -0.37710243463516235, -1.0138747692108154, 0.3415229320526123], [-0.24095062911510468, 0.40431615710258484, 1.6730201244354248, -0.6249104142189026, 1.0942898988723755, 0.45494022965431213, 0.5301874279975891, 0.6838681697845459, -1.0309010744094849, 0.8570570945739746, -0.19407863914966583, 0.4705392122268677, -0.7286840081214905, 0.36246755719184875, -0.4106827974319458, -2.2933456897735596, -1.7906535863876343, -0.4174746572971344, -0.4430740773677826, -0.9812002778053284, -0.5041544437408447, -1.3668609857559204, -0.004860214423388243, -0.4979057312011719, 0.9264500141143799, 0.11906957626342773, 0.5749214887619019, 1.473405122756958, 0.8884958028793335, -0.22332730889320374, -0.3584041893482208, 0.15801146626472473, -1.063122034072876, 0.2665832042694092, -1.2947899103164673, -0.569389820098877, 1.2456355094909668, -1.4597158432006836, 2.2088820934295654, 0.043472323566675186, 1.444924235343933, 0.3199370503425598, 1.1649894714355469, -0.7105526328086853, 0.842078685760498, -0.7935124635696411, 0.3965695798397064, 0.5252662301063538, -1.0246566534042358, 0.21456654369831085, -0.11989255994558334, 1.1793256998062134, -0.20229336619377136, 0.47465652227401733, -0.5248251557350159, -0.16335636377334595, -0.6852090358734131, -0.11456383019685745, -2.5806026458740234, 2.1374454498291016, 0.05401281267404556, 0.7709886431694031, 1.2443389892578125, -1.739227533340454, -0.012055211700499058, -0.3030886948108673, -0.5903302431106567, -2.2088022232055664, 1.0296493768692017, 1.3748139142990112, 0.683914303779602, 0.9099581241607666, -0.6046475768089294, -1.2454291582107544, -0.46125301718711853, 0.8784274458885193, 0.8510798215866089, -0.8474478125572205, 0.7458804249763489, 0.03392379730939865, -0.390914648771286, -0.43244534730911255, 0.32900121808052063, -0.722495973110199, 0.43463486433029175, -1.0026476383209229, 0.3269883692264557, -1.7773809432983398, 1.8241318464279175, 0.5081456303596497, -1.7091608047485352, 1.0067330598831177, 0.29442405700683594, 1.4307968616485596, -1.087927222251892, 1.0812458992004395, 0.20788007974624634, 0.5013841986656189, -0.28431499004364014, 0.19322894513607025, 0.9122414588928223, 0.5118977427482605, 0.26168105006217957, 0.5388917326927185, -1.5330195426940918, 0.653243362903595, 0.4458773732185364, 0.25688081979751587, 2.308615207672119, 0.02687772363424301, 0.8922858834266663, 0.20572441816329956, 0.9700378775596619, 1.0129199028015137, -0.17837995290756226, -0.8042185306549072, 1.0517412424087524, -0.34403905272483826, 1.3435972929000854, 0.2008400708436966, -0.561904788017273, -0.9200596809387207, -0.4949360191822052, -0.1998717188835144, 1.3959784507751465, 1.6738004684448242, -0.20301386713981628, -0.8856374621391296, -0.9961808919906616], [1.623989224433899, -0.7918728590011597, -1.034210205078125, -1.7404149770736694, 1.439046025276184, 0.17988532781600952, 0.9358510971069336, 1.5024560689926147, -0.1237218901515007, -0.08110760152339935, 0.949597179889679, -0.5393401384353638, 0.5589359998703003, 0.2846775949001312, 1.8620378971099854, -0.12367765605449677, -0.13531093299388885, 0.852325439453125, -0.2757880389690399, -0.3984553813934326, -0.624674916267395, -0.147799551486969, 0.6316230893135071, -0.5294761657714844, -0.4858640432357788, -0.24145065248012543, 0.7438456416130066, -1.0734777450561523, -0.545272707939148, 0.5566720962524414, -1.6018271446228027, 2.0683016777038574, -0.30126720666885376, 0.9670191407203674, 0.6277600526809692, 0.6630822420120239, -0.16507408022880554, 0.02189592458307743, 0.013423647731542587, -1.1164278984069824, 0.25209224224090576, 1.1940617561340332, -1.0433051586151123, -0.9518224596977234, -0.23051419854164124, -0.19512052834033966, -0.888937771320343, 0.18582752346992493, -0.06643465906381607, 0.3510367274284363, -2.3391966819763184, 0.03795870020985603, 0.26292771100997925, -0.09285254776477814, 0.6199145317077637, -0.32724401354789734, -1.5426442623138428, 0.3148617446422577, -0.4194600284099579, 0.4259820282459259, 0.46166279911994934, -0.8490161895751953, 0.8112069368362427, 0.34528911113739014, 0.41723668575286865, -0.7549141049385071, 2.5073862075805664, 0.3773597180843353, 0.042185261845588684, -0.9283319115638733, 0.5191946625709534, -1.8574291467666626, -0.11123978346586227, 0.5138718485832214, -1.3051445484161377, 1.0146502256393433, -0.03222207352519035, 0.04665357992053032, -1.7095258235931396, -1.0782089233398438, -0.498149037361145, -0.517133891582489, 0.25526681542396545, 1.0905438661575317, 0.5823569893836975, 0.14267289638519287, 0.3049849271774292, -0.1314607560634613, -0.0036335771437734365, -0.6276629567146301, -0.2391708791255951, -1.2783585786819458, -0.02014843188226223, 0.045547693967819214, -0.7965694069862366, 0.7865912318229675, 1.3108385801315308, 2.0704381465911865, 1.2358753681182861, -1.1709926128387451, 0.7334098219871521, 0.634390115737915, 0.7840888500213623, 0.20556950569152832, -0.23842284083366394, 0.24516119062900543, -0.4147900938987732, 1.6245946884155273, -2.2655117511749268, -1.849656105041504, 0.8504448533058167, -0.06253810226917267, -0.3543621003627777, 1.0005502700805664, -0.5607900619506836, 0.704838216304779, -2.6662399768829346, -1.036515712738037, -1.0543138980865479, 1.0873793363571167, -0.09863229840993881, -1.5046628713607788, -0.12721648812294006, -1.1137974262237549, 1.1678134202957153, 0.5832611322402954, -0.8688007593154907, -0.20391298830509186, 0.3126469850540161], [0.14996345341205597, 0.9275119304656982, 0.1855892390012741, 1.09848952293396, 1.6216588020324707, -0.13862714171409607, -0.06478003412485123, -1.1626479625701904, -1.7325934171676636, 0.5175166130065918, 0.12675034999847412, -0.33029016852378845, -1.3717674016952515, -0.3013768196105957, 0.579757809638977, 0.9536778926849365, -0.3865172266960144, 0.4289717972278595, -0.4270254969596863, 0.14043112099170685, -0.6956719756126404, 0.33176136016845703, -0.6469332575798035, 0.23367415368556976, -0.4641723334789276, 0.5859979391098022, 0.5472356081008911, 1.312273383140564, -0.8345668315887451, 0.0006849605124443769, 0.14767664670944214, 0.02494213357567787, -1.4330685138702393, 0.03372635319828987, 0.9338260293006897, -0.37168580293655396, -0.3458350598812103, 0.1296466737985611, -0.4394616484642029, 1.2812857627868652, -2.189756393432617, -2.0438971519470215, 0.8130585551261902, 0.14536680281162262, -0.35763248801231384, -0.9930686950683594, 0.35580411553382874, -0.7594894766807556, 1.108458161354065, -2.0575060844421387, 0.9519975185394287, -0.7543302774429321, 0.2197820395231247, 0.4847702085971832, -0.5526193380355835, 1.034130573272705, -1.278969645500183, -0.18557366728782654, -0.18860818445682526, 1.063143253326416, -0.3871572017669678, 0.5096706748008728, 2.1298882961273193, -0.661106288433075, 0.007181897759437561, 1.018725872039795, 0.6541244387626648, -1.048007607460022, -0.5420363545417786, -0.03908269479870796, -0.8897174000740051, 0.38752806186676025, -1.082892656326294, 2.4390361309051514, 0.7826061844825745, -0.9466238617897034, 1.115862250328064, -0.9940718412399292, 1.2523815631866455, 0.39341363310813904, -0.5262925028800964, -0.6679288148880005, 0.37047067284584045, -0.07641703635454178, 0.08605404198169708, -0.3176478445529938, -0.9755584597587585, 0.7034244537353516, -1.688422441482544, -1.139796495437622, -1.4356075525283813, 0.07931043207645416, 0.23150339722633362, -0.008639154955744743, 0.9716728329658508, -0.8116124272346497, 0.6067586541175842, -0.12337612360715866, -1.9653191566467285, -1.2536417245864868, 1.401415467262268, -0.7731337547302246, -1.386921763420105, 0.1916656196117401, 0.7093375325202942, 0.11653915792703629, 0.8477469682693481, -0.5500853061676025, 0.3051740825176239, 0.3955243229866028, -0.20650379359722137, -1.970440149307251, 0.7051176428794861, 0.5981199741363525, -1.0482064485549927, -1.1732139587402344, 0.7584806680679321, 0.4095909297466278, 1.4142616987228394, 0.299599826335907, -0.9070678949356079, -0.3504718840122223, -0.15567371249198914, 0.6678299903869629, -0.8778943419456482, -0.254170298576355, 0.6452191472053528, 1.3883169889450073, -0.1150752529501915], [-0.46118074655532837, -0.5914133191108704, 0.7208014726638794, -0.4568609297275543, 0.6995897889137268, 0.3403898775577545, 0.8254042863845825, -0.5346460342407227, -0.8949901461601257, -0.4754992425441742, -0.14400377869606018, -0.7840744853019714, 0.14116591215133667, -0.1838759481906891, -1.0360184907913208, 1.148908019065857, 1.4639251232147217, -0.9224182963371277, 0.949373722076416, -1.1726319789886475, 0.058022018522024155, -0.09147174656391144, 0.6642184853553772, 1.1450954675674438, -3.7446134090423584, 0.4291163682937622, 1.4369107484817505, 0.381797730922699, 0.2020210325717926, 0.6897129416465759, -0.04984039440751076, 0.769723117351532, -0.9825748205184937, -1.7759345769882202, -0.9887127876281738, 0.5545942783355713, -0.5422426462173462, 1.400710105895996, 1.7595248222351074, 1.3479435443878174, 0.7777456045150757, 1.782204031944275, -0.4827749729156494, -0.859123706817627, 1.4418065547943115, 0.055609021335840225, 1.8969085216522217, -0.3430686593055725, 1.1256084442138672, 1.400842308998108, -0.24567320942878723, -0.45169055461883545, -1.8405048847198486, -0.5531349182128906, -2.1716907024383545, 1.5479698181152344, -0.9873434901237488, 0.4609200954437256, -0.24040141701698303, -1.1604019403457642, -1.063734531402588, -1.8464164733886719, 0.6960298418998718, 1.494025468826294, -1.94999361038208, 0.3587031066417694, 1.5659083127975464, 0.044729433953762054, 0.9048275947570801, 0.43381965160369873, 0.4980330467224121, -0.04129861667752266, -1.511064887046814, -0.9955440163612366, -0.1873527318239212, 0.6341363787651062, -0.3724985420703888, 0.2281818985939026, -1.3232340812683105, -0.17839765548706055, -0.6627539396286011, -0.9270561337471008, 0.3303905725479126, 1.4342808723449707, -0.17186978459358215, 0.4299413859844208, -0.23771218955516815, 0.7636116147041321, -0.34589073061943054, -1.8406000137329102, 0.34187081456184387, -1.3404840230941772, 0.8552523255348206, -0.6041574478149414, -1.4060006141662598, -0.1788027435541153, -0.18764165043830872, 0.8865499496459961, -1.6899219751358032, -1.2351324558258057, 0.012555690482258797, 1.200671911239624, 2.108921527862549, -0.029557909816503525, -0.8490342497825623, 0.4132113456726074, 0.9637670516967773, 0.7933357357978821, -1.2977789640426636, -2.2696173191070557, 0.6205601692199707, -1.0317726135253906, -1.8724088668823242, 0.8414226770401001, -0.2252524048089981, 0.8564600348472595, 0.32327884435653687, 0.5992834568023682, -0.7929044961929321, -0.8648971319198608, -1.4466543197631836, 0.11051470041275024, 0.12054707109928131, 0.2048334926366806, 0.9167137742042542, -0.6712697744369507, -1.0098648071289062, 1.132032871246338, -1.0634779930114746], [0.12705427408218384, -0.49249598383903503, -0.8751499652862549, -0.0586385577917099, -0.6142103672027588, -0.2059193104505539, -0.2711465060710907, 0.9891841411590576, -0.003677942557260394, -1.2095550298690796, 1.7187139987945557, -0.5722303986549377, 0.058426979929208755, 0.00045511152711696923, 0.5187809467315674, -0.21874669194221497, 0.39234498143196106, 0.3307914733886719, -0.7107395529747009, 0.2878299355506897, 1.156376600265503, 1.2709486484527588, 0.7562398910522461, 0.42773833870887756, -1.4079874753952026, 0.5610144138336182, 1.4703150987625122, -0.5730182528495789, -0.4508951008319855, 1.401384949684143, 0.7963890433311462, 1.4133973121643066, 0.8581989407539368, -1.0287326574325562, 2.19919490814209, -2.488541603088379, -1.1332156658172607, 0.0911867693066597, -0.24073266983032227, -0.005217730067670345, -0.0312546044588089, 0.4557742774486542, 0.03512436896562576, 1.2449960708618164, -0.4175489544868469, 1.9717578887939453, 0.40146008133888245, 1.7522200345993042, 0.6128618121147156, 0.8772917985916138, -0.42797282338142395, -1.5802137851715088, -0.025231637060642242, -1.5367320775985718, -1.0487357378005981, -0.5527778267860413, 0.16681596636772156, 1.0229225158691406, -0.38142290711402893, 0.3946229815483093, 0.2738761901855469, -1.608095645904541, -1.6731762886047363, -0.6946266889572144, 1.1787774562835693, 1.1749879121780396, -0.8224799036979675, -0.642107367515564, 1.0300242900848389, 1.0953209400177002, 1.6040269136428833, -0.5528243184089661, 0.4144171178340912, -0.7439063191413879, 0.48062899708747864, -0.3655991852283478, 0.1599927842617035, -0.31305813789367676, -0.6235166788101196, -0.6872990727424622, 0.8327961564064026, 0.6295868754386902, 0.053082291036844254, -1.074684739112854, 1.1599589586257935, 0.7795859575271606, -0.6677811741828918, -0.23858217895030975, 0.4786164164543152, -1.086506962776184, -0.24310564994812012, 0.36841943860054016, -1.6082302331924438, -0.6201096177101135, 0.17120201885700226, 0.750292181968689, 0.7031528949737549, -0.44615501165390015, 2.4608821868896484, 1.734649658203125, 0.7696369290351868, 0.8902778029441833, 0.9499141573905945, 1.9294127225875854, -0.3007589876651764, -0.9371259212493896, -0.46730872988700867, 0.8674542307853699, -0.8612123727798462, -1.0395927429199219, 1.243556261062622, -0.9603643417358398, 2.529419422149658, 0.43168553709983826, 0.5569326281547546, 0.07434891164302826, 0.700798749923706, 0.960847020149231, -0.5054075717926025, -1.3602025508880615, -1.184224009513855, -1.6948671340942383, 0.3483688235282898, -0.5872897505760193, 0.13068480789661407, -1.0112706422805786, -0.6682683229446411, 0.47014468908309937, 0.23673909902572632], [0.366562157869339, -0.08019943535327911, 0.22758546471595764, 0.7492707967758179, -0.48811033368110657, 0.41774246096611023, -0.8968619108200073, 0.7345403432846069, 0.9472322463989258, -0.9858115911483765, -1.0840691328048706, -0.2898033857345581, 0.6863920092582703, -0.18168292939662933, 1.8188456296920776, -0.420235812664032, 0.008094624616205692, -0.9215279817581177, -0.4826676845550537, 0.5333428382873535, -1.189354419708252, 1.2924268245697021, -1.781339168548584, -0.8260995149612427, 2.1336216926574707, 0.4689693748950958, -1.0552197694778442, 0.638172447681427, 0.08153904229402542, -0.5423620939254761, -0.8404948711395264, -0.34243592619895935, -0.09328897297382355, 0.16446606814861298, 0.7858168482780457, -1.3655719757080078, -2.427269697189331, -2.1462619304656982, 0.8342164158821106, 0.9172438383102417, 0.4047926664352417, 0.27753788232803345, -0.9759085178375244, -0.46209248900413513, 0.002505134791135788, 0.4244447350502014, -1.189541220664978, 0.3702729046344757, 0.21111319959163666, -0.4226948916912079, -0.2812323272228241, 0.3871060609817505, -2.331023931503296, -0.8082464337348938, 1.3446179628372192, -1.7303380966186523, -0.07552460581064224, 2.5082616806030273, -0.944586992263794, 0.7103075385093689, -1.3884354829788208, 0.7388327121734619, -1.4563062191009521, -0.8555425405502319, -2.270480155944824, 0.00162878364790231, -1.6871533393859863, 1.1338675022125244, 1.1968811750411987, 1.165035605430603, -0.013602777384221554, 2.0773212909698486, -0.28911691904067993, 0.09499367326498032, -1.271482229232788, -1.1674731969833374, 0.7921586036682129, 0.4790792763233185, -0.6666908860206604, -1.0103367567062378, -1.0897754430770874, -1.1498316526412964, -0.014669517055153847, -0.4529842138290405, 0.42873871326446533, -0.278342068195343, -0.3781125545501709, 0.9103573560714722, 0.178750678896904, -0.38869303464889526, -0.40256163477897644, -0.33590060472488403, -0.4470036029815674, 0.2265542596578598, -1.354779839515686, 0.24664144217967987, 0.05259079858660698, -2.014338970184326, 0.9326931834220886, -0.26002347469329834, -1.650223970413208, 1.377097249031067, 0.49100741744041443, -1.1628116369247437, -1.524169683456421, 1.3242051601409912, -0.3731727600097656, -0.37440550327301025, -0.940792441368103, -0.3576163947582245, 0.09527023136615753, -0.95902019739151, 1.2242878675460815, 0.22109393775463104, 0.7331984043121338, 1.0064425468444824, -0.46884703636169434, -0.7051159143447876, 0.032338205724954605, -0.8441417217254639, -0.5720226168632507, -0.5240381360054016, 1.6338146924972534, 1.9273958206176758, 0.7537263631820679, -0.16565322875976562, 0.2540518641471863, 0.7649278044700623, -1.263839602470398], [0.4832415282726288, -0.6961634755134583, -0.3820053040981293, -0.030624262988567352, 1.2069154977798462, 1.0056629180908203, 1.2650259733200073, -0.9224246144294739, 0.4319659173488617, -0.8654508590698242, 0.2711538076400757, -0.322775274515152, -0.005080241709947586, 2.1897802352905273, 0.037952274084091187, 1.049615502357483, -0.09280107915401459, 0.7448899745941162, 0.8138467669487, -0.5803437232971191, 0.10684102028608322, 0.8507633805274963, 0.21812860667705536, -0.4760368764400482, 0.010169148445129395, -0.689304530620575, -0.04076210409402847, -2.2280335426330566, 1.4120842218399048, -1.265682578086853, -0.8017415404319763, -0.046189311891794205, 1.8328797817230225, 0.38717523217201233, -0.5574457049369812, -0.6388930678367615, 0.01860615983605385, -0.40895581245422363, -0.6746824979782104, -0.15150021016597748, -0.015834558755159378, -1.1422178745269775, 0.7931987047195435, 0.3290505111217499, 0.25649574398994446, 1.5363874435424805, 0.1653219759464264, 0.41809532046318054, -1.4597339630126953, -0.8669247627258301, 1.2026222944259644, 0.10471337288618088, -0.967379629611969, 0.0172229316085577, 0.3621020019054413, 0.46435031294822693, 0.6868762373924255, -1.2701585292816162, -1.5474056005477905, 0.673041045665741, -1.3783684968948364, -0.5855038166046143, 0.16339804232120514, 0.33477890491485596, 0.8119097352027893, -1.3249728679656982, -0.5325713157653809, -2.017969846725464, 1.3953619003295898, -2.028407573699951, -2.6176211833953857, 0.4290585219860077, -0.5672919154167175, 0.5370380282402039, -1.2639015913009644, 1.035828948020935, 1.1087123155593872, -0.7476129531860352, 0.8145858645439148, -1.6876415014266968, 0.20374731719493866, 0.46706485748291016, -0.7200939655303955, -0.46180540323257446, -0.4739750027656555, -0.3193809390068054, -0.9629461169242859, 0.10568983852863312, 0.9453916549682617, -0.0237190630286932, -1.3499962091445923, -2.967630386352539, 0.24261420965194702, -1.1493226289749146, -0.6358808279037476, 0.09511596709489822, -0.44254130125045776, 0.7928054928779602, -0.8210185170173645, 1.0252097845077515, 0.18578371405601501, -0.7465576529502869, -1.1647660732269287, 1.904815673828125, -1.1741880178451538, -1.2966190576553345, -0.952278733253479, -2.3536388874053955, 1.2601890563964844, 0.7888153195381165, 2.1208577156066895, -0.9369167685508728, -1.411773920059204, 1.4284719228744507, -0.6578059196472168, -0.5552436709403992, -0.5209341645240784, -0.4535546600818634, -1.1064801216125488, -1.50205397605896, -0.6616935133934021, -1.0942530632019043, -0.20902524888515472, -1.4244253635406494, 0.09280191361904144, 0.8774260878562927, -0.3785383701324463, 1.5229699611663818, 0.9965219497680664], [-0.6635192632675171, 1.236862063407898, -0.8411908149719238, -0.2745593190193176, 0.2581441402435303, 0.035076722502708435, 0.8431623578071594, 0.8126509189605713, -0.21265387535095215, 2.38580060005188, -1.566820740699768, -0.404922217130661, 1.015151858329773, 0.009217034094035625, 1.4246658086776733, 1.8761893510818481, -1.0052603483200073, 1.533528208732605, -0.11966672539710999, 0.7202998995780945, 1.5343778133392334, -0.7140321135520935, -0.04625316709280014, -0.6725534200668335, -0.007310833781957626, -0.3545340597629547, -1.9581140279769897, -0.45843809843063354, 0.21346910297870636, -0.2159399390220642, -1.9319881200790405, -0.36694613099098206, -0.4330992102622986, 0.44039463996887207, 0.2904147803783417, -0.9844195246696472, -1.0901784896850586, 0.4567563235759735, -0.45959287881851196, -0.4268598258495331, 1.055704951286316, -0.3582738935947418, -1.015641689300537, 0.6981102824211121, 0.4755357503890991, -1.018815040588379, -2.621882438659668, 0.9812641143798828, -0.15006385743618011, 0.1889519840478897, -1.2696843147277832, 0.1773548424243927, -0.09237585961818695, 1.1776893138885498, -0.7116618752479553, 0.8928682804107666, -1.520316243171692, -0.5412299633026123, 0.8329463601112366, -1.975168228149414, 0.39060840010643005, 0.8667813539505005, -0.09034660458564758, -1.0442053079605103, 0.47848188877105713, 1.0883831977844238, 1.2908180952072144, -0.890570342540741, -0.4767106771469116, -1.2032524347305298, -0.9428329467773438, -0.12395378202199936, 1.4304674863815308, 2.1244235038757324, 1.5057768821716309, 0.06340072304010391, -0.9305769801139832, 1.4509837627410889, -0.5806809067726135, 0.20870712399482727, -0.2598899006843567, -0.03373675048351288, -0.07896352559328079, -0.9188905954360962, -0.3345879912376404, 0.33933037519454956, -0.1019197404384613, -0.6463593244552612, -1.5705300569534302, -0.06887269020080566, 0.5339414477348328, -0.20931307971477509, 1.9913228750228882, 1.2384963035583496, -0.42008480429649353, -2.031416177749634, -0.38299834728240967, 0.38856297731399536, -0.18805289268493652, -1.1483829021453857, -0.1532513052225113, 0.9989052414894104, 0.06916232407093048, -0.9494743943214417, -0.7935019731521606, -0.2414310723543167, 2.2278294563293457, 0.07529216259717941, 0.4661272466182709, 0.22465258836746216, -0.7425294518470764, -0.33917391300201416, 1.7698582410812378, -0.14724791049957275, 1.4282724857330322, -0.06804535537958145, -0.32804110646247864, -0.9238487482070923, -0.6028834581375122, -0.8304555416107178, -1.4437172412872314, 1.4356310367584229, 0.6918200254440308, -0.774559497833252, -0.3301376700401306, -0.008261110633611679, -0.5703316330909729, -0.043588828295469284, -0.20855605602264404], [1.3702905178070068, 1.028260350227356, -0.2303953468799591, -1.3899937868118286, 0.1562185287475586, -0.02367100678384304, 1.1502225399017334, 1.2202203273773193, 1.5249075889587402, 1.7972201108932495, -0.24502936005592346, -0.06435677409172058, 0.353094220161438, 0.612516462802887, 0.4431157112121582, -0.036573659628629684, -0.548187792301178, -0.7927144765853882, 0.11859751492738724, -1.621343731880188, 0.26624274253845215, 1.106596827507019, -0.57282555103302, 0.35897794365882874, 0.06085425987839699, -0.7872908711433411, -1.224650263786316, -1.0352987051010132, 2.294110059738159, -0.21461594104766846, 0.7422360181808472, 1.0885810852050781, 1.1559034585952759, -1.0329021215438843, -0.11665412038564682, 0.042087819427251816, -0.44730672240257263, 0.3604480028152466, -2.3164308071136475, -0.13948899507522583, 0.5760394930839539, -0.8365070223808289, -0.7093096375465393, -0.6972231268882751, 1.4194693565368652, 1.369820475578308, 0.6569773554801941, 2.0748660564422607, 0.054949801415205, -1.1598803997039795, 0.27835333347320557, 0.37082216143608093, -0.07399264723062515, -0.322481632232666, 0.6472289562225342, -0.05825445428490639, -0.25609830021858215, 0.1717793494462967, 0.5873255133628845, 0.7643111944198608, 0.9919555187225342, 1.5088906288146973, 1.2316060066223145, -0.8214552998542786, 1.7655638456344604, 1.4187363386154175, 1.8887202739715576, -1.1039204597473145, -1.3148068189620972, 0.4717690944671631, 0.8659073710441589, 0.08412203192710876, 0.5823202133178711, -0.0790790393948555, 1.811123013496399, 1.1728625297546387, -1.2809598445892334, 0.23011110723018646, 0.9729685187339783, -0.11616286635398865, -0.0010550523875281215, 1.618762731552124, -2.1958155632019043, 1.069088101387024, 0.41318413615226746, -0.4268048107624054, -0.6176092028617859, -0.0730016902089119, -0.8546069860458374, 1.6464298963546753, -0.09569023549556732, -0.9973726868629456, 0.9201525449752808, -0.18717947602272034, 0.7012072205543518, 0.028011631220579147, -0.045102037489414215, 0.6692296862602234, -0.2701455056667328, 0.6322422027587891, -0.5540966391563416, -0.7104721665382385, 1.1045771837234497, -0.447710245847702, -0.2600332200527191, 0.07216618955135345, 0.9425522089004517, 0.32615646719932556, 1.0686166286468506, 0.7305359840393066, 0.5795297622680664, -0.5890613794326782, 1.2742605209350586, -0.003375991014763713, -0.12578535079956055, -1.7817683219909668, 1.0300925970077515, 0.6890312433242798, -0.5734022855758667, 0.09854963421821594, -0.1384548395872116, 1.1644866466522217, -2.3649654388427734, -0.7480977773666382, -0.3731774389743805, -0.6975746750831604, 0.5795028209686279, -0.5183210968971252, 0.5464445948600769], [0.5473522543907166, 1.9663323163986206, -0.4792460799217224, 1.0846681594848633, 0.614974319934845, 1.2069505453109741, -1.0339759588241577, -0.03547334298491478, 0.48251423239707947, 0.8077718019485474, -0.790541410446167, -1.1060776710510254, -1.959646463394165, 1.052986979484558, -1.3009607791900635, 0.5705701112747192, 0.018813859671354294, 0.030506813898682594, 0.2895822823047638, -1.902464509010315, -1.5527173280715942, 0.70493483543396, 1.0209602117538452, 0.5005061626434326, -0.6840775012969971, 1.0586249828338623, -0.887607991695404, -1.1779333353042603, -0.21406550705432892, 1.6453676223754883, -1.2134255170822144, 0.01113779004663229, -1.5026836395263672, -0.1096649020910263, 2.969606637954712, -0.15300117433071136, -0.1639121025800705, -0.044034503400325775, -1.6550372838974, -0.08636701852083206, -1.6739221811294556, -0.710442841053009, -0.5026198625564575, -0.944862425327301, 0.09478125721216202, -0.003709215670824051, 0.6203453540802002, 0.09410545229911804, -1.0077266693115234, 0.33294397592544556, 0.6809015274047852, 1.3761827945709229, 0.3305521011352539, 0.6437973380088806, -0.024814940989017487, -0.8833434581756592, -0.6546682119369507, -0.5045843124389648, 0.3375220000743866, 0.29004642367362976, -0.15649084746837616, 1.668607234954834, 0.5938305854797363, 1.7955043315887451, -1.5859607458114624, -0.49068257212638855, -0.9941060543060303, -0.43568137288093567, -0.661281406879425, 0.4534761309623718, 1.2660554647445679, -0.586747944355011, -0.3886791467666626, 1.0955246686935425, 1.2542036771774292, 0.43384140729904175, 0.054048750549554825, 0.08424842357635498, -1.585319995880127, 1.0952554941177368, 1.129865050315857, -1.2179957628250122, -0.7462409734725952, 1.1886643171310425, -0.9396907091140747, -1.9040166139602661, 0.16399605572223663, -0.8219438195228577, 0.6031255125999451, 1.273537039756775, 0.6102283596992493, -2.6570932865142822, 0.9442577362060547, 0.13387978076934814, -0.5398464202880859, -1.5828020572662354, 0.009378704242408276, 0.5258176326751709, 0.07235008478164673, -0.5180596709251404, 1.0361891984939575, 0.3261173367500305, -0.36501675844192505, -0.2513963282108307, -0.9553337097167969, -0.5443084836006165, -0.16848118603229523, -0.8760848641395569, 0.11075999587774277, 0.4136430025100708, -0.7454113960266113, 1.141281008720398, 1.7178717851638794, 1.1500346660614014, 1.0699377059936523, -0.48825734853744507, -0.4398585855960846, 0.002066107001155615, -0.6000484228134155, 0.8969202041625977, -0.40055209398269653, 1.396204948425293, 0.11007527261972427, -1.5618611574172974, 0.17879128456115723, 1.5744274854660034, -2.3584578037261963, 0.3903728723526001, 1.9256139993667603], [0.35359784960746765, -0.06413013488054276, 0.5142050385475159, 0.19995816051959991, -0.9412130117416382, -0.14204740524291992, 2.143352746963501, 0.4547501504421234, -0.5831025838851929, 3.3339664936065674, -1.7099231481552124, -0.2150939255952835, 0.929057240486145, -0.030858896672725677, -1.1284133195877075, 1.7363616228103638, 0.5779786109924316, 0.9127057194709778, 0.6145148277282715, 0.2677866220474243, 0.37267225980758667, 0.38217177987098694, -0.8290042281150818, -0.4817163646221161, -0.057235654443502426, 1.1339589357376099, 0.10640282183885574, -0.5348524451255798, -0.9846135377883911, 0.10843244940042496, 0.5190314054489136, 0.38160866498947144, 0.3166263699531555, 1.0678335428237915, 1.3695049285888672, -0.9081994295120239, -0.3400673270225525, 1.2513282299041748, 0.9969272613525391, 0.10746559500694275, -0.9439995884895325, 0.24061304330825806, -1.7133675813674927, 0.018916994333267212, -0.744086742401123, 0.19937095046043396, 1.1551156044006348, 2.0443685054779053, -0.8028965592384338, 0.773669958114624, 1.1288987398147583, 0.21159470081329346, 0.22265732288360596, -0.5391178131103516, 1.7961660623550415, -1.1659305095672607, 1.1960431337356567, 1.3283590078353882, 0.5490078330039978, -1.9239380359649658, -0.1492672562599182, 0.6902068257331848, -1.3833763599395752, 1.5751429796218872, 1.4306533336639404, -1.92070472240448, -0.9397248029708862, 1.4468928575515747, -2.190018892288208, -0.7084583044052124, -0.05824622884392738, -0.7408512234687805, -1.827978253364563, -0.057131312787532806, -0.614224374294281, 1.0864484310150146, 0.7590101957321167, 2.6856045722961426, -1.39841890335083, 0.3163102865219116, 0.20281964540481567, 0.9060778021812439, 1.7639458179473877, 1.1022531986236572, -0.15509524941444397, 0.2245495766401291, 0.5363785028457642, 0.11957961320877075, 0.8541680574417114, 0.14343321323394775, -0.005250158254057169, 1.1487085819244385, -0.10143667459487915, 0.5619336366653442, 0.26366761326789856, -0.11771895736455917, -0.8045096397399902, 1.017156720161438, 1.3110066652297974, -0.6703271269798279, -0.5902953743934631, -0.21786069869995117, 0.05879189446568489, -0.266173779964447, 0.018044983968138695, 1.7281962633132935, -0.14260411262512207, -0.8746699690818787, -0.35649222135543823, 0.08688240498304367, -0.4549960494041443, 0.34442076086997986, -0.2172023355960846, -0.2195245772600174, 1.060640573501587, -0.7278232574462891, 0.49226245284080505, -0.8796621561050415, -1.1073949337005615, 0.6058177351951599, 0.9451357126235962, 2.195683479309082, -2.2089648246765137, -0.11193094402551651, -2.4570565223693848, -1.0453048944473267, 0.3045516908168793, -0.06773743778467178, -0.22432532906532288], [-0.19408181309700012, -0.030665751546621323, 0.6457253098487854, -1.866312026977539, -0.42627596855163574, 0.33654505014419556, -1.746402382850647, -2.2434535026550293, 1.243809461593628, -0.4237360954284668, -0.4784266948699951, -0.006122506223618984, 1.0473448038101196, 0.6889289617538452, 0.7938726544380188, 0.05702760070562363, -1.8703402280807495, 0.2729037404060364, 0.8756930828094482, -2.2509772777557373, 0.7800769805908203, 0.34286314249038696, -0.8273852467536926, -0.8544912934303284, 1.8629906177520752, 0.029616838321089745, 0.3362203538417816, -0.94589763879776, 1.7309861183166504, -0.38162925839424133, -1.172264575958252, 1.6016803979873657, 2.0037736892700195, -0.5890583992004395, 0.4794389605522156, 0.04294360429048538, -2.4490792751312256, 1.0736078023910522, -1.1227483749389648, 1.2844536304473877, -0.5930584073066711, 1.0550668239593506, -0.06834884732961655, -0.22030802071094513, -1.6052072048187256, -0.011083816178143024, -1.1595444679260254, -0.4703773856163025, -0.4517103433609009, 0.6676685214042664, -1.2589551210403442, -0.9283813834190369, 0.8355871438980103, 0.07389192283153534, -1.2139050960540771, 1.2851696014404297, -0.9277237057685852, 0.3936607539653778, -0.16355906426906586, -0.33181875944137573, 2.5093750953674316, -0.5060160160064697, 0.246371790766716, -0.7314462661743164, 0.9425830841064453, -0.7686522603034973, -0.7819586992263794, -0.41303175687789917, 0.7858718037605286, 0.12384571135044098, 1.2241967916488647, 0.42250216007232666, 0.34897682070732117, -0.14284038543701172, 0.967523992061615, -0.3059484362602234, 1.7253681421279907, 1.010720133781433, 0.5643994808197021, 1.2468364238739014, 1.5906721353530884, 2.7980141639709473, -0.6659937500953674, -0.21867983043193817, -0.1882261335849762, 0.35390955209732056, 0.3262072503566742, -0.4122900068759918, -0.09006807953119278, -0.5385300517082214, -0.2894987463951111, 0.08422830700874329, -0.04701169580221176, 0.06829294562339783, 0.4522815942764282, -0.27475836873054504, 1.97616708278656, -0.43830907344818115, 1.6019281148910522, -0.3913296163082123, 0.32818812131881714, 1.3556456565856934, 0.9711225032806396, -0.3925304114818573, -1.4763091802597046, 0.4056667387485504, 1.7132683992385864, 1.6750298738479614, -0.7618004679679871, 0.819922685623169, -0.10388768464326859, -0.6240283846855164, -0.5451416969299316, -0.797615647315979, 0.5803521275520325, 2.205509901046753, 0.6552900671958923, -0.12567178905010223, -0.7323170304298401, -0.26138734817504883, -0.5723158717155457, 0.9178187251091003, 1.3434088230133057, 0.8153384327888489, -1.2680954933166504, -0.5722309350967407, 1.2529569864273071, -0.21488305926322937, 1.0652515888214111], [-2.103532314300537, -0.5955738425254822, -0.602259635925293, 1.4432426691055298, 0.816185474395752, -1.3495525121688843, 0.41142749786376953, -0.6631438732147217, -0.7035983800888062, -0.3739241361618042, -1.039905309677124, 0.07792221009731293, -0.6244478821754456, -0.4815123379230499, 0.7937874794006348, 0.9118925333023071, -0.12300904840230942, -1.849610447883606, 0.47144615650177, -0.3501763641834259, -0.15878251194953918, -0.21254666149616241, -1.1683626174926758, 1.0011589527130127, 0.5834253430366516, -0.8876779079437256, 1.9371299743652344, -0.19098283350467682, -0.10123427212238312, 1.0308400392532349, -0.6201019883155823, 1.3500276803970337, -0.3460276424884796, 1.2198150157928467, 0.18238131701946259, 0.6744694113731384, -1.1956149339675903, 1.2685362100601196, 0.37529614567756653, 0.6163548827171326, -0.5923548936843872, 0.376049280166626, -0.1536911278963089, -0.46207401156425476, -2.6398255825042725, 2.1269009113311768, 0.8982978463172913, 0.371611088514328, 0.647367000579834, -0.5170543193817139, -0.9872570633888245, -2.224268674850464, -1.536994218826294, -0.44548720121383667, -0.8147080540657043, -0.15710416436195374, -0.687559187412262, -0.6352004408836365, -0.013986285775899887, -0.6916654706001282, 0.5213775038719177, 0.07065793126821518, 0.5398398041725159, -1.3855962753295898, 0.9692621827125549, 0.3253377377986908, 0.2229066640138626, 0.4102170169353485, 0.5470174551010132, -0.11024858057498932, -1.7139418125152588, -0.4613397419452667, -1.0938630104064941, 0.6602073907852173, -0.24496524035930634, -1.200063705444336, 1.394026517868042, 0.9242134690284729, -0.7875998616218567, -1.1669440269470215, -0.20921951532363892, -0.21266591548919678, -1.8534334897994995, 0.31830570101737976, 0.14199455082416534, -0.8349159955978394, 0.39280763268470764, -0.5578393340110779, 1.6341865062713623, -0.22489969432353973, -1.2384378910064697, 1.46219801902771, 0.2812923789024353, 0.12510477006435394, -0.926149845123291, -0.15810564160346985, 0.6184902787208557, -0.13250403106212616, 0.11511258780956268, 1.4064499139785767, -0.5089710354804993, -0.9226512908935547, -0.2688279449939728, 0.10906928777694702, -1.5528514385223389, 0.7541031837463379, 0.33043426275253296, -1.1662124395370483, -1.5445784330368042, 0.7300094366073608, -1.5281405448913574, -0.0011511705815792084, -0.47723349928855896, 1.1923383474349976, -0.22119759023189545, -1.2211536169052124, -0.5290026068687439, -0.4723041355609894, -2.7470860481262207, -0.8932012319564819, -0.7159668803215027, 1.2971528768539429, 0.2787773907184601, 1.3804020881652832, 0.5251637101173401, 1.8453736305236816, -0.059135641902685165, -1.214333415031433, 0.5666408538818359], [0.6991548538208008, -0.28379079699516296, -0.7342783212661743, 0.16738368570804596, 0.004540689289569855, 1.5234668254852295, -1.5853708982467651, -0.4321815073490143, -2.064727544784546, -1.1517170667648315, -0.7031598091125488, -2.513056993484497, 1.9483801126480103, 0.9964225888252258, -1.7077288627624512, 2.172334909439087, -0.03439800441265106, -1.02366304397583, -0.19121290743350983, 1.0353219509124756, -0.5549236536026001, 0.9459004998207092, -0.34179893136024475, -0.8477578163146973, -1.004919171333313, 2.8046207427978516, -0.34671443700790405, 0.7192190289497375, -0.5864535570144653, 0.26310309767723083, 0.21679063141345978, -1.0722051858901978, -2.3131861686706543, -1.2885210514068604, -0.8806509971618652, -0.25410422682762146, 0.33715832233428955, 1.0813281536102295, -1.0284159183502197, 0.6900147795677185, -0.8732522130012512, -1.1836206912994385, -1.4425135850906372, -0.7069350481033325, 1.1367303133010864, 1.0036877393722534, 0.7198110818862915, 1.3511991500854492, 1.0334203243255615, -1.490366816520691, 2.2374367713928223, -1.7634048461914062, -0.4785100817680359, 0.5315008759498596, -0.19353757798671722, -0.4149572551250458, -1.1463903188705444, 0.10964455455541611, -0.6741049289703369, -1.7165387868881226, 0.16108660399913788, 2.2381393909454346, 0.1899535357952118, 0.468777060508728, 0.11990740895271301, -1.3971607685089111, -0.11641240119934082, 1.0758121013641357, 0.2517925500869751, 0.5395962595939636, 0.4998958706855774, 0.7086502313613892, -1.0105700492858887, 1.2434452772140503, -1.867904782295227, -0.9264236688613892, -0.3411017656326294, -0.37193986773490906, -0.4820171296596527, -0.3646131753921509, 0.29502907395362854, -0.08942504972219467, -0.43440237641334534, 1.922736644744873, -0.04815647751092911, 0.005153126083314419, 1.6554319858551025, 0.2127763032913208, 1.908840298652649, 0.4420976936817169, 0.5717675089836121, 1.774355411529541, -0.18300923705101013, -0.12403973937034607, -1.707334280014038, -1.2231439352035522, 0.6023095846176147, 0.8504396677017212, -0.2645237445831299, -0.9889424443244934, 0.23497526347637177, -0.6627922058105469, 0.110401950776577, 0.7791633605957031, -3.093731164932251, -0.14386813342571259, 0.8244344592094421, 0.823409914970398, 0.030463121831417084, -0.26929110288619995, -1.1563647985458374, 0.7218518853187561, 0.9986259937286377, -1.824293613433838, -0.8297214508056641, -1.6965978145599365, 0.516758143901825, 0.20633898675441742, 1.0990852117538452, 1.6606703996658325, 1.073015570640564, -0.30020684003829956, 0.7199561595916748, -0.33300724625587463, 0.11602728068828583, 0.06522101908922195, 1.4638005495071411, -1.0686051845550537, -1.8009803295135498], [0.8858076930046082, -0.03368302434682846, -0.7212803959846497, -2.390063524246216, -1.0288056135177612, 0.4557289183139801, -0.9519188404083252, 0.3469212055206299, 0.036267369985580444, -0.439984530210495, 1.4131762981414795, 1.2120635509490967, -1.7424190044403076, -0.006186590995639563, -0.3799552023410797, 0.49597346782684326, -0.6074115037918091, -0.05947651341557503, 2.4071879386901855, -0.4082730710506439, 2.1933183670043945, -0.24272598326206207, -0.13527780771255493, -2.1716818809509277, 1.5973988771438599, -1.0885478258132935, -0.22536680102348328, 0.27055203914642334, -0.16851070523262024, -1.7698554992675781, -0.15717196464538574, -1.1932555437088013, 1.6327195167541504, 0.6452133059501648, -0.2776683270931244, 0.7940403819084167, -1.0973107814788818, 0.4751693904399872, -0.9441095590591431, -0.8356684446334839, -0.015374971553683281, 1.9222501516342163, -0.5490376949310303, 0.5213119387626648, -0.3570603132247925, -1.1084330081939697, 0.7976651787757874, -0.04937246814370155, 0.25758177042007446, 0.623877227306366, 1.0436478853225708, 1.4374500513076782, -1.2001149654388428, 0.9725489616394043, -0.0848417803645134, 0.012777229771018028, -1.013400912284851, -0.09405043721199036, 1.5745097398757935, 2.250694513320923, 0.10668353736400604, -1.5151265859603882, 2.3465752601623535, -0.2378574162721634, -0.06016521155834198, 0.22071413695812225, 1.2188301086425781, -1.4914920330047607, -1.4634814262390137, 0.5114967226982117, 0.052062854170799255, -0.8250405788421631, 0.46172547340393066, 1.2282224893569946, -1.6299986839294434, -0.05019806697964668, 0.20230261981487274, -0.3135818839073181, 1.0454663038253784, -0.1877167522907257, -1.4633737802505493, 0.3157615661621094, -0.2545040249824524, -1.1110631227493286, 0.9885722994804382, -1.092313528060913, 0.3772391080856323, 0.6237177848815918, -0.7989214658737183, 0.5471773743629456, -0.7396494150161743, -1.0177592039108276, -1.2058175802230835, -0.2604287564754486, 1.4103634357452393, -0.5586969256401062, 0.6398344039916992, -0.46361207962036133, -0.9210854172706604, -0.5455380082130432, 0.8305137157440186, 0.8093968033790588, 0.8673056364059448, 0.01107763685286045, -0.056414566934108734, 1.7752125263214111, -0.13663853704929352, -0.3664870262145996, 0.7774827480316162, -1.3802433013916016, -0.19598940014839172, -0.6785953044891357, 0.04601951688528061, 0.43136563897132874, 0.6589648127555847, 1.180134654045105, 2.020348072052002, -1.265712857246399, 0.08551958948373795, 0.1175905391573906, 1.7302182912826538, -0.0625147819519043, 0.030103446915745735, -0.49432864785194397, 0.11266856640577316, -0.9609998464584351, -0.7228514552116394, 0.34400296211242676, -0.3565681278705597], [1.3881474733352661, -2.2492423057556152, -0.3033541440963745, -0.3144420087337494, -0.46798089146614075, -1.449286937713623, 0.2466164380311966, 2.5918774604797363, 0.21712343394756317, -1.2717878818511963, 0.7355281710624695, -0.984541654586792, -2.2885282039642334, -1.2497711181640625, 1.0490617752075195, 0.08257520943880081, 0.9687014222145081, -0.23630443215370178, 0.36253824830055237, 0.5706140995025635, 1.0884819030761719, 1.509016752243042, 0.8628420233726501, -1.1072615385055542, 0.2670360207557678, -0.7266091704368591, 0.4737755358219147, -0.07112514972686768, -0.5008669495582581, 1.9276260137557983, 0.6473499536514282, 1.9043275117874146, 1.1673269271850586, 0.19867199659347534, 1.4500786066055298, 1.4473954439163208, 0.4496993124485016, 1.2473810911178589, 0.7894855737686157, -0.05041154846549034, 0.7989582419395447, 0.023284731432795525, -1.4105464220046997, -0.13846468925476074, 0.2152959257364273, 1.22647225856781, 0.6599912047386169, -0.2071739286184311, 0.26973196864128113, 1.7773584127426147, -0.2659097909927368, 0.2805996239185333, -0.2543478310108185, 1.8362746238708496, -1.6170096397399902, 0.9349793791770935, 0.3026392161846161, -0.7126348614692688, -0.39799565076828003, 0.13650302588939667, -2.653660774230957, 0.012715351767838001, 0.2364673912525177, -0.15797889232635498, 0.8275864124298096, 0.9717467427253723, 1.1557739973068237, -0.8825343251228333, 0.1266777664422989, 0.04625820741057396, -0.1397114396095276, 3.152657985687256, -1.0800392627716064, 0.6518753170967102, 0.5079289674758911, -1.5183459520339966, -0.8758118748664856, 0.9442657232284546, 0.16092927753925323, 0.19782444834709167, 1.7208623886108398, -0.10236816108226776, 0.38672366738319397, -0.9551366567611694, -0.9215812683105469, 1.3343425989151, -1.272753119468689, -0.34775951504707336, -0.5879808068275452, -0.40406832098960876, -1.4990589618682861, 0.29071149230003357, 0.6233401298522949, -0.15320444107055664, 1.4230694770812988, -0.7981469631195068, 1.788129210472107, 1.1441311836242676, -0.9781795144081116, 0.4866633713245392, 0.3802092373371124, 0.6458396911621094, 0.9138619303703308, -0.550250768661499, 2.3146748542785645, -1.2158557176589966, -0.2926827371120453, 0.15705594420433044, -0.6437261700630188, 2.218827962875366, 0.3453293740749359, -1.3109548091888428, -0.701242208480835, -0.7458028197288513, -0.7329195737838745, -0.028107639402151108, -0.966636598110199, -1.6593191623687744, -0.022694505751132965, -1.3587268590927124, 0.08757936209440231, 0.9001367688179016, -0.4452660381793976, -0.7842441201210022, -1.1977653503417969, 1.4016931056976318, 0.6244969964027405, -0.9330227375030518, 0.3256569504737854], [0.8300420641899109, 0.6842508912086487, 0.2969290018081665, 0.5337316989898682, -0.28067660331726074, 0.9728485345840454, -0.31507688760757446, 0.9737939834594727, 1.129289150238037, -1.2673345804214478, 0.7179847955703735, -0.4157284200191498, -1.609659194946289, 0.27484166622161865, 0.2880040407180786, 0.12152903527021408, 0.8394742012023926, 1.0937833786010742, 0.5702213644981384, 0.7952098250389099, 1.650771975517273, -1.0216840505599976, 0.003925814293324947, 0.5296636819839478, 0.9073559045791626, 0.03806298226118088, -1.2801411151885986, -2.125788688659668, 0.8680648803710938, 0.27155810594558716, 0.08467987924814224, -1.2080607414245605, 1.1753770112991333, -0.7426419854164124, 0.2314286082983017, 0.6105173230171204, -0.03196284547448158, 0.5898731350898743, 0.968202531337738, 0.637890100479126, 1.2882000207901, -0.44969627261161804, 1.1828587055206299, 0.532091498374939, -1.2421942949295044, 0.1103956550359726, 0.4174780547618866, 1.2312543392181396, -0.09928210824728012, 0.6913731098175049, -1.2699832916259766, -2.0760676860809326, 0.487307608127594, 1.8107280731201172, -0.9146527051925659, 0.9175692796707153, -0.45791035890579224, -1.1065784692764282, 0.3070557415485382, 1.0990957021713257, -0.3612250089645386, -0.19434292614459991, -0.9617924690246582, -0.9707690477371216, -0.47963210940361023, 1.3903316259384155, -0.7588052153587341, -1.7828316688537598, 0.9374716281890869, 0.3815688490867615, -1.070928931236267, -0.20898039638996124, -1.7595298290252686, -2.9608566761016846, 1.643457293510437, 0.2295137345790863, -0.02280132658779621, 0.9068831205368042, 0.08919983357191086, -1.3204617500305176, -0.004167674575001001, -1.003588080406189, 0.17761345207691193, -0.1267644166946411, 0.3408019542694092, -0.7919931411743164, 0.45966193079948425, 0.06449694186449051, 0.12937329709529877, -1.1206247806549072, 0.03794926777482033, -0.36996421217918396, 0.3507491648197174, 1.3237237930297852, 0.27402016520500183, -1.082509160041809, 1.4926215410232544, 0.20621831715106964, -2.127441644668579, 0.20186002552509308, 1.0526517629623413, 0.4301646649837494, -0.11273877322673798, -0.7347549796104431, -0.88874751329422, -0.16433852910995483, 0.5375427007675171, 0.06428204476833344, 0.02745315618813038, 0.33809781074523926, 0.12695883214473724, 0.873799741268158, 0.9014066457748413, -0.4982675313949585, 0.5570123195648193, 1.4907630681991577, 0.058732952922582626, 0.5396016836166382, -0.08205284923315048, -1.2575814723968506, -1.0001821517944336, 0.5951234102249146, 0.32947853207588196, -0.6325675845146179, -0.24411147832870483, -0.9039333462715149, -0.09953827410936356, 0.6919782161712646, 1.5600415468215942], [0.8550546169281006, 0.5890506505966187, -0.43977710604667664, 3.1351606845855713, -1.6862492561340332, 0.5871453881263733, 0.07443417608737946, 0.8196148872375488, -0.34823983907699585, 1.3034210205078125, -0.876876175403595, 1.2725403308868408, -0.9155412316322327, -1.8075571060180664, -0.6791567802429199, 0.33560481667518616, 0.7722713947296143, 1.7603508234024048, 0.3172591030597687, -2.145055055618286, -1.0165737867355347, -1.197893738746643, 0.1837637722492218, 0.25573599338531494, 1.5664396286010742, 0.8771710991859436, -0.321094274520874, 0.339471697807312, 1.3052325248718262, 0.24047677218914032, -1.0564414262771606, 0.8274739980697632, 0.518181562423706, -1.6614677906036377, 1.4993277788162231, -0.2728143036365509, 0.06387591361999512, -0.007396059110760689, -1.1429938077926636, 0.45033806562423706, -2.2279276847839355, 0.4380980432033539, 1.7285020351409912, -1.3595713376998901, -1.1279194355010986, -0.6503540277481079, -0.1073472648859024, -0.8676661849021912, -0.35564154386520386, -0.8248112797737122, -0.843970000743866, 1.7881500720977783, 0.15207725763320923, -0.6295583844184875, 0.8225139379501343, -1.1789259910583496, -0.9172733426094055, -2.1576058864593506, -1.2107892036437988, -0.4038568437099457, -0.7129805684089661, 0.2544457018375397, 0.1480122059583664, -1.3194524049758911, -0.3461799621582031, 0.7636985182762146, -1.3894219398498535, 0.2911643981933594, 0.46411454677581787, -0.13437305390834808, 0.5650326013565063, 0.8060588836669922, -0.3752405345439911, -1.2613227367401123, -0.29419806599617004, -1.0700340270996094, -0.27231088280677795, 0.6844437718391418, -0.0783391147851944, 0.5051770806312561, 0.46906182169914246, -0.7800599336624146, -0.4566671848297119, 0.7902206182479858, 0.7882135510444641, -0.20487453043460846, 1.5609973669052124, -0.9811490774154663, -0.7359095215797424, 0.3062020242214203, 0.6473702788352966, 0.21221062541007996, -0.8037537932395935, 1.0144572257995605, -1.6679649353027344, -0.7028382420539856, -1.4738925695419312, -0.03565555438399315, -1.192807674407959, -0.49778491258621216, -0.251880407333374, 0.1486244946718216, -0.8004360198974609, 0.2657882869243622, 0.21967698633670807, -0.5362674593925476, -1.1050447225570679, -1.0825858116149902, -0.16920636594295502, 1.1666122674942017, 0.3840250074863434, 0.27230849862098694, 0.6493848562240601, 1.418432593345642, -0.7753416299819946, -1.407720923423767, -0.10430759191513062, -0.18031109869480133, 0.3113109767436981, -0.43193674087524414, 0.18766933679580688, -1.7841166257858276, -0.946089506149292, 1.3397473096847534, -0.2605534791946411, 0.009483139961957932, 0.3723885416984558, 0.304656058549881, 0.1385868489742279], [0.2716297507286072, 1.2233126163482666, -0.7761817574501038, 0.47242483496665955, 0.2774272859096527, 0.7664641737937927, -0.888181209564209, 0.25461938977241516, -0.31467747688293457, -0.6593398451805115, 0.06331872195005417, -0.7372456789016724, 1.2945445775985718, -0.6873822212219238, -0.3138323128223419, -0.16202658414840698, -1.281734824180603, 0.2163994014263153, 3.5149357318878174, -0.9048039317131042, 0.41489481925964355, 0.9958117008209229, 1.2634475231170654, -0.8232813477516174, 0.10210888087749481, -0.7716028690338135, 0.19946298003196716, -1.0305657386779785, -1.0100796222686768, -0.9345240592956543, -0.2176845818758011, 0.02459380403161049, 0.5000433325767517, -0.10303652286529541, 0.29599788784980774, -0.6455045938491821, -0.28371700644493103, 1.432634949684143, 1.532434105873108, 0.110554538667202, -0.6036761403083801, 0.033349476754665375, -1.4161484241485596, -1.3764374256134033, -0.6254370212554932, -0.753516435623169, 2.2179150581359863, 0.6581947207450867, 1.5299960374832153, -0.15260164439678192, 0.8154825568199158, 1.244047999382019, -1.6244686841964722, -0.1959109753370285, -1.2146645784378052, 1.2675526142120361, 0.8528957366943359, -1.3590714931488037, 0.4569288492202759, -0.9571444392204285, -0.8221288323402405, 0.4811282455921173, -0.11916659027338028, 0.35166051983833313, -2.1499886512756348, 0.8785998821258545, -0.6643872261047363, -2.3061442375183105, -0.9212168455123901, -1.145099401473999, -0.31865596771240234, -0.6937828063964844, 0.35440459847450256, -0.27944496273994446, 0.28451764583587646, -0.3841107487678528, 1.2804663181304932, 2.7452664375305176, -1.9001548290252686, 0.40355631709098816, 0.9001665115356445, 1.3960546255111694, 0.28963157534599304, 1.5658540725708008, -0.0721585676074028, 0.5938553810119629, -1.4407200813293457, -0.6016653180122375, -1.4104890823364258, -0.4731277525424957, 1.097090721130371, 1.330318808555603, 0.315547376871109, -0.26528048515319824, 0.2628198266029358, 0.1582220047712326, 1.9304789304733276, -0.6658198833465576, -0.9006404876708984, -0.06596037745475769, 0.4634645879268646, -1.6387125253677368, 1.2744747400283813, 0.6613867878913879, -0.6481544971466064, 0.5261668562889099, 0.0907863900065422, -0.40724313259124756, -0.603269636631012, -1.720884084701538, -0.021847646683454514, 2.1261098384857178, -0.3260950744152069, 0.18016096949577332, 0.024280499666929245, 0.0665104016661644, 1.6788424253463745, -1.2239229679107666, 0.3668462932109833, -0.719484806060791, -0.0795881599187851, -0.31528809666633606, -0.1780727207660675, 0.13311852514743805, -2.6227335929870605, 0.17695701122283936, -0.5568738579750061, -0.10701698064804077, 0.5627117156982422], [-1.5711947679519653, 0.743995189666748, 0.8161346912384033, 1.2855448722839355, -0.45147451758384705, -0.8173270225524902, -1.6290603876113892, 1.165906548500061, 1.2430771589279175, 1.0701286792755127, 0.8808340430259705, 1.51286780834198, -0.7148876190185547, -0.4659411907196045, 0.12183865159749985, 1.3618940114974976, -1.4021700620651245, -1.5914674997329712, -1.8818175792694092, 0.19031427800655365, 0.19994841516017914, 1.2726560831069946, -0.54689621925354, -1.2991589307785034, -1.2432912588119507, 1.1365848779678345, -1.04635751247406, 1.8812044858932495, 0.8235534429550171, -0.0815022885799408, 0.8872605562210083, 0.29290252923965454, 0.555869460105896, -0.26003384590148926, -0.43567392230033875, 0.9714620113372803, -0.7254856824874878, 0.46885713934898376, -1.6927940845489502, -2.0596351623535156, 0.9314554929733276, -0.989905595779419, -0.25357362627983093, 0.8742856979370117, 0.1809338927268982, -1.633934736251831, -0.34473419189453125, 0.08113618940114975, 1.7203909158706665, 0.10471425205469131, 1.13629949092865, 1.1634851694107056, -1.6464685201644897, 0.6278976798057556, 0.1351177841424942, 1.3678652048110962, 0.11712411046028137, 1.3804504871368408, -0.9714697003364563, 2.3069980144500732, -0.6029054522514343, 1.4562194347381592, 0.05879699066281319, 0.1496770828962326, 0.010254963301122189, -0.034225042909383774, -0.1891537457704544, 0.9854241609573364, 0.9389008283615112, -0.6841319799423218, 0.2609196603298187, 0.6182752847671509, -0.06086789816617966, 1.9208006858825684, -1.0039836168289185, -0.6910088062286377, 0.15766149759292603, -0.08294432610273361, 2.227940797805786, -0.1764300912618637, 0.15464307367801666, -0.9765032529830933, 1.2461107969284058, -1.0889800786972046, 0.2005143016576767, 0.8664675354957581, 1.124961018562317, 1.9058136940002441, -0.8837948441505432, -0.1869828850030899, -0.3306539058685303, 0.5174055695533752, 0.5219469666481018, 0.6685953140258789, -0.41077595949172974, 0.32707691192626953, 0.633970320224762, 1.6996253728866577, 0.8084194660186768, 0.758510410785675, -0.32319942116737366, -0.01153613906353712, -1.5922772884368896, -1.4136470556259155, 0.1793735921382904, 0.33238375186920166, -0.11449553072452545, 0.43042051792144775, 0.9606587290763855, 0.7156172394752502, -0.5619062185287476, 0.23880654573440552, 1.4571735858917236, 0.622667133808136, 0.07608763873577118, 0.9517440795898438, 0.3694630265235901, -0.6186866164207458, -0.4015137255191803, -1.5767256021499634, 0.46299439668655396, 2.4750144481658936, -1.5130259990692139, 1.5826953649520874, 1.305528998374939, 0.585570216178894, 0.41966649889945984, -1.4896715879440308, -0.7684034109115601], [-0.19585108757019043, -0.7330387830734253, 0.34122830629348755, -0.31974172592163086, 0.21064218878746033, -0.3527502715587616, 0.34082505106925964, 1.6246458292007446, -0.8607433438301086, -0.6392279863357544, 1.2166082859039307, 0.3534178137779236, 0.5899088978767395, -0.7127346396446228, -0.9227874279022217, 0.09911858290433884, -0.45288345217704773, -0.06981970369815826, 1.2051875591278076, 0.4172494113445282, 1.3190792798995972, -1.0473966598510742, 0.9324154257774353, -1.483288049697876, -0.016506250947713852, -0.32981815934181213, -1.5732457637786865, 1.8619344234466553, 0.2982194125652313, -0.28551048040390015, 0.6060448884963989, -1.053054690361023, -1.2463101148605347, 0.508268415927887, -0.8830894827842712, 1.812021255493164, 0.12739385664463043, 0.26823127269744873, 1.0599676370620728, -0.8959174156188965, -0.8114051818847656, -0.8169019222259521, 0.37607917189598083, 1.1648553609848022, -0.6732370257377625, 0.6500494480133057, -2.4043757915496826, 1.6352672576904297, -0.13749679923057556, -1.0090769529342651, -0.6784197688102722, 0.5004822611808777, -0.21124787628650665, 0.11238235235214233, 0.5110343098640442, -0.05052760988473892, -0.4439370632171631, 0.11813630908727646, -1.1406612396240234, -0.6390866041183472, -0.5307532548904419, 0.7602454423904419, 0.5530518889427185, 0.19783751666545868, 0.9328671097755432, -0.6677892804145813, -0.23933200538158417, 1.6479413509368896, 0.7383241057395935, 0.2952381372451782, -0.45982998609542847, 1.4302716255187988, -1.0491719245910645, 1.480259895324707, -0.10699209570884705, 1.3772327899932861, -0.7892885804176331, -0.40694722533226013, 1.4091697931289673, 1.478112816810608, -1.8070729970932007, -0.010573466308414936, 1.1491169929504395, 0.0337720550596714, -0.2950463593006134, 0.3747667968273163, -0.4142395257949829, 0.7262340188026428, -0.7709468007087708, 0.04330861568450928, 1.0222314596176147, -0.9634951949119568, -0.06918145716190338, 0.7442722916603088, -0.7810248136520386, -1.5464534759521484, -0.409878134727478, 1.4441884756088257, 2.831977367401123, -0.1688740849494934, -0.04674158990383148, -1.6373499631881714, -1.3374946117401123, 1.3331319093704224, 0.056363895535469055, 1.9167979955673218, -2.5860543251037598, 0.06901971250772476, 0.17009615898132324, 0.7462742328643799, 1.8772835731506348, -0.1679336279630661, 0.1523256003856659, 0.6971297860145569, -0.20185035467147827, -1.3080909252166748, -1.272187352180481, 0.33815887570381165, 0.23166368901729584, 0.1947866678237915, 1.4271748065948486, -0.6109450459480286, -0.5436265468597412, -0.393206924200058, -1.3900326490402222, -1.2416341304779053, -1.2742539644241333, -0.5508140921592712, -0.07884960621595383], [0.0725666806101799, 0.06282859295606613, -0.16752755641937256, -0.41254228353500366, 1.7070297002792358, 0.08116383850574493, 0.9473256468772888, 0.5552123785018921, 0.3809897005558014, 0.4219828248023987, -2.351006269454956, -0.6230184435844421, -0.4600692689418793, 0.4223908483982086, 1.7637417316436768, 1.3791553974151611, -1.4443353414535522, -0.04537883400917053, 0.7336980700492859, -0.09548962116241455, -1.128098964691162, -0.937766969203949, -0.4845646917819977, 0.3552758991718292, 0.8068736791610718, -1.0355525016784668, 0.34224188327789307, -0.42624029517173767, -0.009392603300511837, 0.15098212659358978, -0.748618483543396, 0.14758966863155365, -0.6447230577468872, 0.6264823079109192, -1.868303894996643, -0.28696921467781067, 0.033715032041072845, 0.9613808393478394, -0.17750301957130432, -1.794296145439148, 1.55934476852417, 0.47665438055992126, 0.3756638467311859, 0.8186126947402954, -0.652136504650116, 1.40699303150177, 0.17662465572357178, -0.5794284343719482, 1.313984751701355, -0.6111947894096375, 0.5316454172134399, 0.3604395389556885, 0.07288065552711487, 0.5469849705696106, -2.18365478515625, -0.03560957312583923, -0.520002007484436, 0.9952799081802368, 0.3236078917980194, 0.4122524857521057, -0.5239234566688538, 0.9270821809768677, -0.2512461245059967, -0.6279321312904358, 0.4969598054885864, 1.4740538597106934, -0.04139747470617294, -2.1958045959472656, -0.13535098731517792, -0.986725389957428, -0.19873853027820587, -1.95623779296875, -0.25643599033355713, 0.9052631855010986, -0.2506260275840759, -1.4978222846984863, -0.10740771144628525, -0.3754132091999054, 0.04624239355325699, -0.11151567846536636, -0.7820906043052673, 1.5044225454330444, 0.05058401823043823, -1.2319920063018799, -1.0107672214508057, 0.3762166500091553, -1.188076138496399, 0.13649095594882965, -0.011350931599736214, -1.9169106483459473, 0.6141102910041809, 0.2093234807252884, -1.0321965217590332, 1.0142987966537476, 0.934414267539978, 0.9555151462554932, -0.8786733746528625, 0.29796528816223145, 0.6430904269218445, -0.8006851077079773, -0.15876390039920807, 1.251626968383789, 0.5935644507408142, 0.6695719361305237, -2.200291633605957, -1.4579753875732422, -1.4847737550735474, 0.21533715724945068, 0.2559308111667633, -0.4150373339653015, -0.5667353272438049, 0.4248654842376709, 1.2017900943756104, -0.7521779537200928, 1.0078744888305664, 0.916797935962677, -0.07403568923473358, -0.9927800297737122, 1.0518560409545898, 1.4788987636566162, 0.1212245374917984, 0.38573572039604187, 0.3488110601902008, 1.4563401937484741, 1.1173256635665894, 0.4970806837081909, -1.5667611360549927, -0.09938545525074005, -0.24560481309890747], [0.7906537652015686, 1.2737481594085693, -1.7498835325241089, -0.4374459683895111, -1.4405572414398193, -0.8173355460166931, 0.27719706296920776, 0.21108682453632355, 1.862716794013977, 0.7905627489089966, 0.945785641670227, -0.24652670323848724, -0.06601016968488693, -0.7862585783004761, 0.7853152751922607, 0.5988736748695374, 0.0259084552526474, -0.22427703440189362, 1.073014259338379, -0.2823316156864166, -0.3329043388366699, 0.2982923090457916, -0.9661557078361511, 0.32905369997024536, -0.7059609293937683, -0.5375235676765442, -0.3892536759376526, 0.604973316192627, -0.1732264906167984, -0.09453320503234863, -1.1491508483886719, 0.43692266941070557, 1.2136532068252563, 0.3242865800857544, 0.1848803460597992, -0.4063907861709595, 0.26775115728378296, 0.3464977443218231, -0.7686241865158081, -0.13194304704666138, 1.2314547300338745, 0.2586349844932556, -1.314795970916748, 0.3411639332771301, 0.6133219003677368, -0.17845867574214935, 0.32403719425201416, 0.9403926730155945, -0.05262989178299904, -0.15536236763000488, 0.5158106088638306, 0.6441543102264404, -0.49730345606803894, 0.0012520303716883063, -0.3545958995819092, 1.2135188579559326, 0.6578002572059631, 1.4681248664855957, 1.1634423732757568, -0.6542187333106995, 0.6102277040481567, 0.12801943719387054, -0.12485065311193466, -1.0343011617660522, 0.4750819802284241, 0.11033237725496292, -0.1778404414653778, -0.05238736420869827, 1.050377368927002, -0.10549657791852951, 1.8929452896118164, -0.3945956826210022, 0.12393171340227127, 1.049425482749939, 1.1055762767791748, -1.5009102821350098, -0.7025138735771179, 0.46120765805244446, 1.5263274908065796, 0.12523353099822998, -0.39241933822631836, -1.2213404178619385, -0.8105063438415527, 1.6787348985671997, 1.400092363357544, -0.3413589894771576, -0.5124952793121338, 0.1989625096321106, -0.9140633940696716, 1.383828043937683, -0.05810845270752907, -0.6038778424263, 2.0311877727508545, 0.1196720078587532, -0.3536435067653656, -1.4297239780426025, 1.759366750717163, 0.9422786831855774, -0.07894385606050491, -0.18682773411273956, 0.18544034659862518, 0.4716763496398926, -1.9758824110031128, -1.1127203702926636, -0.49765822291374207, -0.09980073571205139, -0.22822923958301544, 1.8327850103378296, 1.2608026266098022, -0.36722761392593384, -1.160419225692749, -0.16102352738380432, 0.9063811898231506, -0.3471762239933014, 0.6659975051879883, 0.16030165553092957, 0.24273544549942017, 1.8544445037841797, -0.4199083149433136, 0.6568657159805298, -0.5268088579177856, -0.5490524172782898, -1.5685923099517822, -0.8941031098365784, 0.24534234404563904, -0.16114529967308044, -0.37789466977119446, -0.32267263531684875, -1.3479877710342407], [-0.9867594242095947, 0.195736363530159, 0.18995130062103271, 0.37943530082702637, -1.1121050119400024, -0.5832967758178711, 0.7529516816139221, 0.6880109906196594, -0.054578233510255814, -0.38337042927742004, -1.6565591096878052, -0.05897083505988121, 0.697320282459259, 0.21540723741054535, -0.8687312006950378, 0.0401686429977417, -0.2739064395427704, 0.11877410113811493, -0.7336268424987793, -0.432607501745224, -0.253223180770874, 0.08284647017717361, -1.018670916557312, 0.23046855628490448, 0.3400455117225647, 0.5269380211830139, 0.22085055708885193, -0.5896891355514526, -0.9856950044631958, 0.8850710391998291, 0.613685131072998, 0.4232958257198334, -0.0444379486143589, 1.2989636659622192, 0.3592076301574707, -0.3969929814338684, 1.2756158113479614, 1.7636274099349976, 0.45796552300453186, -0.2732985019683838, 1.1444131135940552, 1.2627053260803223, -0.4092087149620056, 0.2915005385875702, 0.42068684101104736, 0.8316006660461426, 0.00997113436460495, 0.46590811014175415, 1.5787993669509888, 0.6132416129112244, -0.21483391523361206, -0.43086087703704834, -0.2940794825553894, -1.5276999473571777, 0.6664252281188965, 0.4809097647666931, 0.8184240460395813, -1.5388853549957275, 0.5380815863609314, 1.4989473819732666, -1.0799764394760132, 1.0927730798721313, -1.4427376985549927, 0.7083536982536316, -0.4816189110279083, -0.15446868538856506, -0.4667871296405792, -1.4638690948486328, 0.18881948292255402, -2.711458683013916, 0.9052128195762634, -0.7740795016288757, 0.5956640839576721, 0.14065998792648315, 0.17242112755775452, 1.2841358184814453, -0.3007766008377075, 0.8537333607673645, -2.5275752544403076, 0.29857751727104187, 0.7390581369400024, 2.21474552154541, -0.6194227933883667, -0.05216319113969803, 0.6650874614715576, -0.23947329819202423, 0.8459378480911255, 0.22491605579853058, -1.1262741088867188, 1.2436443567276, 0.13876117765903473, -0.9956924915313721, 1.9315263032913208, 1.2073596715927124, 1.0925461053848267, 0.9566850662231445, -1.6019198894500732, 0.10176057368516922, -0.5858685970306396, 0.8374010920524597, -1.8726918697357178, -1.5280112028121948, -0.41004759073257446, 0.9890066981315613, -0.46865445375442505, -0.843407154083252, -0.8243093490600586, 1.4439421892166138, 0.424263596534729, 0.5878286957740784, -0.8801142573356628, 0.3214259445667267, 2.7974045276641846, -0.41621750593185425, 0.5746393799781799, -1.0574283599853516, -1.488262414932251, -0.65317702293396, 1.5126192569732666, 0.20004913210868835, -0.4000225067138672, -0.3458843231201172, -1.9710569381713867, -0.8325256109237671, -1.3390942811965942, -0.18767113983631134, -1.2134360074996948, -2.037860155105591, 0.5017076134681702], [0.163602814078331, 2.197679281234741, 1.6287568807601929, 0.7504439949989319, -0.32432088255882263, -0.8051344156265259, -0.7569185495376587, 1.4038041830062866, 0.45119810104370117, -0.5604419112205505, 0.13324329257011414, 0.07262367010116577, -0.25351008772850037, 1.5847232341766357, -0.08146922290325165, 0.11043722927570343, -0.6688559651374817, 0.23097015917301178, -0.5467212796211243, 1.0202373266220093, -0.24219244718551636, 0.061934277415275574, 1.1441611051559448, -1.7540562152862549, 0.3280332386493683, 0.06129969283938408, 0.8313331604003906, -0.6271601915359497, -1.127489686012268, 2.1330864429473877, 1.364329218864441, 0.4597584307193756, -1.2495901584625244, -1.3682827949523926, -0.698497474193573, 0.3907908797264099, -0.2295454442501068, 0.026785781607031822, -0.5874891877174377, 0.386014461517334, 0.2434339076280594, 0.3744359016418457, -1.2242411375045776, -0.032547567039728165, -1.0592448711395264, -0.48184603452682495, -0.6522833108901978, -1.7619746923446655, -0.46645140647888184, 0.36427217721939087, -0.8059353828430176, 1.793483853340149, -0.9129515886306763, 0.19323782622814178, -0.5214194059371948, 0.18257245421409607, 1.8234919309616089, -0.42348161339759827, 0.12029629945755005, -0.46942007541656494, 0.003457733429968357, -0.6271186470985413, -0.6547035574913025, -1.4511125087738037, 1.6904562711715698, 0.9072338342666626, 0.11725214123725891, 1.048899531364441, 0.05406765267252922, 1.97940194606781, 0.5960766673088074, -0.5541979074478149, 0.41913479566574097, 0.2147790491580963, -0.8458454012870789, 0.43636655807495117, -1.0615718364715576, 0.8559554815292358, -0.19913250207901, 0.08155220746994019, -0.835475504398346, -0.8356342911720276, -1.987318754196167, 2.489441156387329, -0.7359066009521484, 1.626700520515442, 0.31611132621765137, 1.1487500667572021, 2.0264549255371094, -0.47761455178260803, 0.18082565069198608, 0.9175122976303101, 0.08339197933673859, 0.26234087347984314, -0.3139698803424835, -1.3912619352340698, -0.477029412984848, -0.3396837115287781, -0.5803331732749939, 1.5079892873764038, -0.8268332481384277, 0.5623661279678345, 0.7130002379417419, 1.3233871459960938, 1.093366026878357, -0.881127655506134, -1.380759835243225, 0.9870696663856506, 0.1552632451057434, -0.6954771280288696, 0.10416360944509506, -0.5551901459693909, 0.6569609642028809, -1.0530625581741333, 0.24535495042800903, -1.0970629453659058, -0.8653073906898499, 1.8385608196258545, -0.6546951532363892, -1.6865328550338745, -0.7168216109275818, 0.6421195268630981, -0.01906139962375164, 1.9007251262664795, -0.3570914566516876, 0.14771215617656708, 0.22479978203773499, -1.2775859832763672, 0.58022141456604], [-1.1415183544158936, 1.523948311805725, -0.29411929845809937, 1.1107486486434937, 1.8571043014526367, 0.6281967759132385, -1.9582061767578125, 1.557888388633728, -0.27816101908683777, -1.07463800907135, 0.5418527126312256, -2.14989972114563, -1.3811839818954468, 0.6901036500930786, 0.7369924187660217, 0.7662786245346069, -1.3804975748062134, 1.4828557968139648, 0.4637993276119232, -0.018125196918845177, -1.0106171369552612, 0.8662211298942566, 2.084240198135376, 0.2553254961967468, -0.7170024514198303, -0.824798047542572, -0.19473646581172943, -0.09458638727664948, 1.1822744607925415, 0.4190349876880646, 1.9141614437103271, -0.0796089842915535, 0.1195644959807396, -0.897635817527771, -1.637794852256775, -0.8464164137840271, 0.22595307230949402, -1.1754677295684814, -0.42664551734924316, -2.1303038597106934, 0.16059055924415588, 0.4514258801937103, -0.1416696459054947, -0.14081883430480957, -0.3004187345504761, -1.1219924688339233, -0.3392086923122406, -0.6382415890693665, -0.18365265429019928, -0.3653690814971924, 0.9748409986495972, 0.20926706492900848, -1.3260713815689087, -1.275488018989563, 1.9143613576889038, 0.35953596234321594, -0.6214420199394226, 0.2778186798095703, 0.7718857526779175, 0.1871645450592041, -0.6143541932106018, -1.574438214302063, 1.363816738128662, 0.35660383105278015, -0.952720046043396, -0.2898824214935303, -2.1104393005371094, -1.070649266242981, 0.8575767874717712, -0.31804800033569336, -0.724126398563385, -0.8375428318977356, -0.3794873356819153, -1.044602870941162, -0.15970557928085327, -1.088943600654602, -1.5243459939956665, 0.8809885382652283, -2.498795509338379, -0.7979861497879028, 0.18632681667804718, 0.3961852192878723, 2.122288703918457, 1.3991634845733643, 1.0903781652450562, 0.1798667013645172, 0.013334458693861961, 2.3071014881134033, -1.2646477222442627, 0.2150932103395462, -0.9847553372383118, 1.1211910247802734, 1.6775832176208496, 1.4495736360549927, 1.918752908706665, -0.3916446268558502, -0.4139391779899597, 0.6621778011322021, -1.9910578727722168, -0.5789421796798706, 0.5813933610916138, 0.5444495677947998, 0.2656371593475342, 0.5240355730056763, -0.04638461768627167, 0.9308794140815735, -1.1204807758331299, -1.7650150060653687, -0.24309833347797394, -0.45140424370765686, 0.6435295343399048, 0.32189440727233887, 0.850961446762085, -1.016756534576416, 0.5716352462768555, -0.687466561794281, 0.578892707824707, -0.34664663672447205, 0.6842008829116821, 0.6640178561210632, 1.8959678411483765, -0.7982612252235413, 0.8521348237991333, 1.4624111652374268, 1.0032106637954712, -0.5936776995658875, -0.18288059532642365, 0.29636695981025696, 0.5747039914131165], [-0.9080765247344971, -0.2848733961582184, 2.0972139835357666, 0.06529252976179123, 0.4684910774230957, 0.19749154150485992, -2.1692521572113037, 0.3470119833946228, 0.3816513121128082, -2.148545265197754, -0.08326853066682816, -0.1657266616821289, 0.07952350378036499, -0.5275029540061951, -0.06795442849397659, 0.9976799488067627, 0.200186088681221, 0.3087483048439026, 1.151078701019287, -0.002780171111226082, 0.8599927425384521, -1.986212134361267, -2.413238763809204, 2.1071181297302246, -0.8563467264175415, 1.1782974004745483, -0.6217694282531738, -0.42137399315834045, -1.5552335977554321, -0.18790203332901, 0.3880927562713623, 0.3215719759464264, -0.7137478590011597, 1.1263679265975952, -1.9393796920776367, 0.08634352684020996, 1.5780233144760132, 0.971686840057373, 1.4856853485107422, 0.021651597693562508, 0.17779409885406494, -0.43077874183654785, -0.03024033084511757, 0.6464524865150452, -1.2177950143814087, 0.4959298074245453, 0.09047335386276245, 1.2780590057373047, -1.675801157951355, 1.2752050161361694, -1.3143954277038574, 1.655747413635254, 1.2155681848526, 0.5107325315475464, -1.5461989641189575, -0.18122535943984985, -1.4799232482910156, -0.39759373664855957, 0.03778291493654251, 0.6657658815383911, -0.9400455355644226, 0.3768397867679596, 0.414668470621109, 1.2781572341918945, -0.9824174046516418, 1.8347668647766113, -0.16622672975063324, 0.5459375977516174, 0.29866740107536316, 2.0260026454925537, -2.5059916973114014, 0.6315637826919556, -0.8956763744354248, -1.1135152578353882, -0.5775948762893677, 0.010137854143977165, 2.2508151531219482, 0.005437632091343403, -0.3273615837097168, -0.6505677700042725, -0.1813935786485672, 0.9478659629821777, -0.5694589614868164, 0.5379777550697327, 0.714921236038208, 0.8758922815322876, 0.29182618856430054, -1.978074550628662, 1.524137258529663, 0.7283650040626526, -1.4898651838302612, 0.277060866355896, 0.608855128288269, -0.16045312583446503, 0.40553075075149536, -0.6958062052726746, -1.7371875047683716, 0.09925632923841476, 0.585866391658783, 0.741202175617218, -0.00556695694103837, -1.145689606666565, -0.9925617575645447, -1.4472273588180542, -1.9842180013656616, 0.8060688972473145, 0.6832995414733887, 0.5609686374664307, 0.5458913445472717, 2.4982988834381104, 1.3756654262542725, -1.4199962615966797, -1.0946050882339478, -0.6449815630912781, -0.5363671183586121, 0.5094457268714905, -0.8223046660423279, -2.127760171890259, 0.41170910000801086, -0.7049499750137329, 0.16538433730602264, -1.2697726488113403, -0.45009785890579224, -1.4369397163391113, -0.31548601388931274, -0.22048449516296387, -1.7759947776794434, 0.5900397896766663, -1.27420175075531], [-0.143984854221344, -2.0911614894866943, -0.3832451403141022, 0.06362147629261017, -0.26315733790397644, 0.5032777786254883, -1.5787878036499023, -2.7902309894561768, 0.2537168562412262, -1.3447824716567993, -0.027900926768779755, 0.09221035242080688, 0.6656510829925537, 0.07824033498764038, -0.7772516012191772, 2.0759692192077637, -0.6423522233963013, 1.174729585647583, -1.2436167001724243, 1.5300061702728271, 0.22423939406871796, 0.9920766353607178, 1.7165888547897339, -0.01782839000225067, -0.7717378735542297, 0.36340370774269104, -0.4900186359882355, 0.6175525188446045, 0.7491384744644165, -0.3444988429546356, 0.6945911049842834, 0.9309771656990051, -1.4133638143539429, -1.4663892984390259, 0.07291419059038162, 1.6378206014633179, 0.14585857093334198, -0.790847897529602, -0.6015112400054932, 0.2347504198551178, 0.787779688835144, 0.8471248149871826, -0.44336703419685364, 1.127097249031067, -0.3293851613998413, 0.6501113176345825, -0.8294128775596619, 1.4406960010528564, -0.05813632532954216, -0.02290065959095955, 0.4323035776615143, 2.235123872756958, 2.5836760997772217, 0.30943870544433594, 0.07807181030511856, 0.7866570949554443, 0.6369770169258118, 1.5671066045761108, -0.8714388012886047, 0.07150926440954208, 0.7314449548721313, 0.21845541894435883, 1.3177396059036255, -1.1448265314102173, -1.0355987548828125, 2.350783109664917, 1.4936480522155762, 1.1267861127853394, -1.3674663305282593, 0.9062545895576477, 0.8255194425582886, -0.01041390374302864, -0.35281476378440857, 0.6683778166770935, -0.9787046313285828, -0.3479318618774414, 1.1444411277770996, -1.0585740804672241, 1.6178003549575806, 0.8005136251449585, -1.0289157629013062, -1.2521251440048218, -0.1734921932220459, 0.6074020862579346, -0.5246406197547913, -0.29160186648368835, 1.1198800802230835, -0.5275145173072815, 1.191072940826416, -1.2236849069595337, 0.049506571143865585, 1.0293563604354858, 1.2089380025863647, 1.1138561964035034, 1.2529799938201904, 0.308576375246048, 0.017387254163622856, 0.8331431746482849, 0.30894309282302856, -0.18488851189613342, 0.45640695095062256, 1.9404547214508057, 0.008719807490706444, -0.43705883622169495, -0.6458527445793152, 0.09099023789167404, 0.2298794388771057, 0.18693262338638306, -0.32049277424812317, -1.8280353546142578, -0.7951912879943848, 0.7241896390914917, -0.16880007088184357, -0.9882547855377197, 0.3528849184513092, 0.2329203486442566, 0.05197341740131378, 1.4366285800933838, -0.042992137372493744, 0.15310120582580566, -0.05679185688495636, -0.48282018303871155, 0.15584298968315125, 0.9788533449172974, 0.5859277844429016, -0.3288601040840149, 0.3360940217971802, -0.06801153719425201, -0.3246777057647705], [0.08944347500801086, 0.5753938555717468, -0.6932170391082764, -0.8668204545974731, -0.7355222105979919, 0.3535579741001129, -0.4606993794441223, 1.0693929195404053, -0.38760700821876526, 1.2621634006500244, 0.7434160709381104, 0.14024876058101654, 0.537818968296051, -0.3953332006931305, -0.856640100479126, -0.7306971549987793, -0.10487563908100128, -0.5064538717269897, -0.3075419068336487, -0.6889916658401489, -0.7551037669181824, 0.9185724854469299, -1.197799563407898, -0.7369831204414368, 1.5076048374176025, -0.6711194515228271, -1.4850248098373413, -1.1804388761520386, 0.8398088812828064, 0.3910321593284607, -1.8237686157226562, 0.816261887550354, -0.8588548898696899, 0.7676202058792114, -0.27023643255233765, -0.9760844707489014, -1.0092765092849731, 0.5119087100028992, 0.38003209233283997, -1.608803153038025, -1.1370792388916016, -0.6477429270744324, -1.5218809843063354, -0.12125424295663834, -0.6574934124946594, 1.8222447633743286, -0.2598572373390198, -0.4402487874031067, -0.19555167853832245, -0.6173553466796875, -0.7610650658607483, 1.0974241495132446, 2.5773184299468994, 0.6932599544525146, -0.5631111860275269, -1.0891878604888916, -0.644184410572052, -0.058548860251903534, -0.16348396241664886, 1.2171260118484497, 0.5344249606132507, -1.1637341976165771, -2.079317569732666, 0.02152872458100319, 0.9327236413955688, 0.9071547985076904, 2.6067051887512207, -0.2849087119102478, -0.6426589488983154, -0.1072327047586441, -0.43049323558807373, 0.6535459756851196, -0.19743803143501282, -0.8054689764976501, -1.5551068782806396, -1.284432291984558, -0.4768674969673157, -0.31902414560317993, -0.5206073522567749, -0.40507811307907104, 0.6843238472938538, -0.2544142007827759, -0.3779551684856415, 0.09555748850107193, 1.3962478637695312, 1.902470350265503, -0.6771578788757324, -0.09706183522939682, -0.171498641371727, 0.6608167886734009, -1.1916446685791016, 0.4083872437477112, 0.09172343462705612, 0.5775774121284485, 1.4792530536651611, -0.22079631686210632, 0.4477776288986206, 0.476929634809494, -0.06280487030744553, 1.3255329132080078, -0.7622666358947754, -0.4636465907096863, -1.9532678127288818, -0.14986319839954376, -0.4876224994659424, -0.5029659271240234, 0.2218700349330902, 1.5496760606765747, 1.302289366722107, -0.041452113538980484, 0.48608866333961487, -0.5773926973342896, 0.10416991263628006, -0.5612823367118835, 0.753642737865448, 1.146735429763794, 0.5000571012496948, -0.25412923097610474, -0.6300790309906006, 1.9435545206069946, 0.9885419011116028, -0.35075342655181885, 0.9079983830451965, 0.5334203243255615, -0.35581594705581665, 0.37930935621261597, -0.6229333281517029, -1.2903048992156982, 1.653045892715454], [-0.0051267086528241634, 0.15547895431518555, 1.525682806968689, -0.0899282768368721, -0.24818481504917145, 0.6767789721488953, -0.70550936460495, -0.6200113892555237, 1.7279272079467773, -0.6262339949607849, -0.1651606261730194, -0.4826233983039856, -0.5096399784088135, 0.8017074465751648, -0.07253467291593552, -1.0517581701278687, -0.06487995386123657, 0.644537627696991, 0.5518956184387207, -0.10885844379663467, -1.8067443370819092, 1.3103183507919312, -0.3585728108882904, 0.038765884935855865, -0.7521439790725708, -0.1774994134902954, -0.3434845209121704, 1.9880084991455078, 0.6308068633079529, 0.3490780293941498, 0.7797366380691528, -0.15648722648620605, -0.03076070547103882, 1.6618473529815674, 1.7403175830841064, 0.11369844526052475, 0.4512685239315033, 0.3313796818256378, 0.7719442248344421, -1.0113332271575928, -1.6717860698699951, 1.3654000759124756, 0.23041702806949615, 0.6112182140350342, -0.4534193277359009, 0.3348871171474457, 0.5791817903518677, 0.6240136623382568, -1.3981369733810425, 0.9304167032241821, 1.6449213027954102, -1.1845847368240356, 0.40320315957069397, -0.7334792613983154, 1.0125494003295898, -0.7283808588981628, -1.7457550764083862, 0.18530426919460297, -1.7772670984268188, 1.778505802154541, 1.3329660892486572, 0.23763138055801392, 1.1798408031463623, 1.243896484375, 1.101331353187561, -0.6382851004600525, -0.5482402443885803, 1.146931529045105, 0.3535621762275696, 0.7683227062225342, -1.2135934829711914, 1.1876033544540405, -0.6897438764572144, -0.0029340593609958887, -0.4360142946243286, -0.03786648437380791, 0.5240541696548462, 0.7136273980140686, -0.6304528713226318, -0.41138625144958496, 1.1750776767730713, 0.8985065817832947, -0.12354125827550888, 0.6377325057983398, -1.3066468238830566, -0.8386196494102478, -1.0573699474334717, -1.1979862451553345, 1.2674509286880493, -0.23597760498523712, -1.8596296310424805, -0.11137598752975464, -0.20139914751052856, 0.9223389625549316, -0.8246333003044128, -0.29228004813194275, 1.3768998384475708, -1.4720797538757324, 0.9571841359138489, -0.8040066361427307, -0.07626482099294662, -1.2720564603805542, -0.6501628756523132, 0.7194744944572449, 0.7238793969154358, 0.005193017423152924, -0.05358857288956642, -1.4813019037246704, -1.4336694478988647, 0.8450934886932373, 1.845164179801941, -0.3700513541698456, -1.2452690601348877, -1.3608688116073608, 0.0042075361125171185, 0.45912501215934753, -0.709278404712677, -0.16733022034168243, -1.0461513996124268, 0.036650948226451874, -1.4733723402023315, -0.02732391655445099, 1.3818250894546509, 1.6906218528747559, 0.10572554171085358, 1.36882746219635, 2.3947131633758545, 0.5224969387054443, 1.196875810623169], [-0.0727628841996193, -0.49076735973358154, -1.4810549020767212, -0.8829478025436401, 0.49814993143081665, -0.9229763746261597, 2.4712250232696533, 0.17350414395332336, 0.4154419004917145, -1.141774296760559, -0.286271870136261, -0.24663971364498138, -0.9151623845100403, -0.4021851420402527, 1.3207192420959473, -1.0653254985809326, 1.122431993484497, 1.3191591501235962, -0.36875417828559875, 1.0507757663726807, -0.945292592048645, -0.4240279197692871, -1.1693553924560547, -0.9338277578353882, 0.30352821946144104, -1.000954270362854, -0.972511351108551, 1.1755175590515137, -0.9153019189834595, 1.0132098197937012, 0.5566372871398926, 0.3823234438896179, 0.8371524214744568, 0.8830670118331909, -0.07494432479143143, -1.3814468383789062, 0.6991958618164062, -0.015073034912347794, 0.6400038599967957, -1.0508058071136475, -0.832050085067749, -0.5229560136795044, 1.1171423196792603, -2.269644021987915, -0.12618514895439148, -0.3708754777908325, -0.2768802046775818, 0.39109963178634644, -0.09220230579376221, 0.7223753929138184, 0.5101478695869446, 2.1137449741363525, -0.3909538686275482, 1.1691184043884277, -0.49670448899269104, -1.993769884109497, 0.5027812719345093, 1.5512889623641968, 0.7374045252799988, 0.7236851453781128, -0.2750095725059509, -0.6321938633918762, 0.32864001393318176, -0.4475918114185333, -0.8344823122024536, 0.3610515892505646, 0.8548954725265503, -2.6454436779022217, 1.2898961305618286, 1.147274374961853, -1.6309442520141602, -0.9655170440673828, -0.29386281967163086, 0.9825774431228638, 2.7524023056030273, -1.0969420671463013, -0.6445547342300415, -0.7575685381889343, 0.6324257850646973, 0.38778623938560486, 1.306681752204895, 3.0484492778778076, -0.39935746788978577, -0.3610866963863373, -1.9447932243347168, -1.0869574546813965, -0.5547481775283813, 0.7349704504013062, -1.5831780433654785, -0.28590482473373413, 0.6524969935417175, -0.10928186029195786, -0.883739709854126, -0.6795335412025452, 1.6239769458770752, 0.18036946654319763, -0.07946640253067017, -1.3086532354354858, -0.21808862686157227, -0.7694011926651001, -0.5019686818122864, 0.49827292561531067, 0.8581075668334961, 1.5077919960021973, -0.332640677690506, 0.45455166697502136, -0.7530922293663025, -0.0435473807156086, -1.5373183488845825, 1.2293123006820679, 0.32390403747558594, 0.856113851070404, -1.1103564500808716, 0.7808703184127808, -2.929443120956421, -1.1035879850387573, -1.3779027462005615, 1.0628865957260132, -1.4534083604812622, -1.037548542022705, 0.6643230319023132, 0.15081864595413208, -0.677055835723877, 0.6541839838027954, 0.02365107834339142, 0.005856543313711882, 1.8017274141311646, 0.6060880422592163, 0.49511727690696716], [0.8924525380134583, 0.6243401765823364, 1.9198652505874634, 0.7487871646881104, -1.3662631511688232, -1.8687195777893066, 0.8004530668258667, -0.01908220164477825, -1.3982340097427368, -0.1156538799405098, -0.4330631494522095, 0.6540292501449585, 0.4848012626171112, 0.8180812001228333, -0.6346123814582825, -0.5770740509033203, 1.4700781106948853, 0.5561146140098572, -0.6456409692764282, 0.3113364577293396, 0.5784863233566284, 0.4709565043449402, -0.3517410457134247, -0.504043459892273, -0.5711097717285156, 0.9159010648727417, -0.8797684907913208, -0.9960664510726929, -0.1077672690153122, 0.9914265871047974, -0.872012734413147, -0.2887246310710907, -1.6476489305496216, 2.639606237411499, 0.945698082447052, 0.2579105794429779, -1.7687382698059082, 0.4969078600406647, -0.26762211322784424, -0.5200062990188599, -1.5866976976394653, 1.2577041387557983, -0.9782884120941162, 0.04035840183496475, -0.846061646938324, -0.5264167785644531, 0.2528802752494812, 0.5052539110183716, -2.892110586166382, -0.163767009973526, 0.8743693828582764, -0.6281812787055969, -1.0862727165222168, -0.23797471821308136, 2.1601431369781494, 1.7258195877075195, -0.9720120429992676, -0.49074000120162964, -0.08431072533130646, 0.24473078548908234, -0.22940173745155334, -0.12321929633617401, -1.298363447189331, -1.1634169816970825, -0.21025000512599945, 1.0742624998092651, -1.1094343662261963, -0.5080670118331909, 0.06967224180698395, -2.3072993755340576, 0.5672098994255066, -0.317812442779541, 0.20861750841140747, -0.7775018811225891, 0.6190198063850403, 1.6300908327102661, -0.32974520325660706, 0.586897075176239, -1.618139386177063, 0.09520230442285538, 0.2396005094051361, -0.14887182414531708, -1.1365230083465576, 0.41294801235198975, 0.31966713070869446, 0.8613552451133728, -0.15248849987983704, 0.43547216057777405, -1.2298959493637085, -1.382006287574768, -1.1262387037277222, -0.2734065651893616, -2.0765304565429688, -1.1821211576461792, -0.3710964322090149, -0.44917798042297363, -0.5201278924942017, -1.124666452407837, -1.1999226808547974, 0.09399136900901794, -0.9905502200126648, 1.9844590425491333, 0.7723872065544128, 0.32925111055374146, 0.443152517080307, 0.10558386147022247, 0.8855388164520264, -0.23156221210956573, 1.5211037397384644, 1.7293827533721924, 0.8129276037216187, 0.8117441534996033, -0.841082751750946, 2.1751654148101807, 0.673507809638977, -0.4935489594936371, -0.5769636631011963, 0.14111833274364471, -0.5748636722564697, 0.6245649456977844, 0.3341452181339264, 2.2302558422088623, 0.691168487071991, 0.2247694581747055, 0.7670043110847473, -0.13268359005451202, 1.9087538719177246, 0.8845617771148682, 0.5189447999000549], [0.8030382990837097, 0.637294590473175, 0.040915507823228836, 0.9672986268997192, -0.029721885919570923, 0.22153253853321075, -1.5703641176223755, -0.037229739129543304, 1.2401201725006104, 0.22693324089050293, -0.4454171657562256, 0.6098628044128418, 0.06595693528652191, -0.01888156495988369, -0.4026772975921631, -0.5892152190208435, 0.17416076362133026, -0.5212299823760986, 0.49913284182548523, 1.3960096836090088, 0.9954147338867188, 0.7598921656608582, -0.005730772856622934, 0.5336472988128662, 0.1392962634563446, -0.5533417463302612, 3.412322998046875, 0.6314075589179993, -1.5230348110198975, -1.676800012588501, 1.2357202768325806, -2.467233419418335, -2.132385730743408, 0.43829360604286194, 0.3770642578601837, -0.382782906293869, -2.788309335708618, 0.5585280656814575, 1.3669742345809937, -0.3834076225757599, 1.6948808431625366, -0.6489464044570923, 1.8376128673553467, 1.6478191614151, -0.6796533465385437, -0.3046985864639282, -0.2533957064151764, 0.5197795629501343, -0.5199142694473267, 0.38027724623680115, -0.9889496564865112, 2.1642322540283203, 0.5832242369651794, -0.21479195356369019, -0.677699863910675, 1.5205258131027222, 0.06443191319704056, -1.6995165348052979, 0.4748554229736328, -0.13536015152931213, 3.2151265144348145, -0.18473994731903076, 1.0732371807098389, 0.9852129817008972, 2.007864475250244, 0.1912393718957901, -1.6868517398834229, 1.3478180170059204, -0.4331458806991577, 1.0164357423782349, -1.6066254377365112, -0.8686290979385376, -0.9594502449035645, 0.0334218367934227, -0.04023231193423271, -1.6834805011749268, 1.6713006496429443, 0.37971699237823486, 0.659494936466217, 0.06685876101255417, 0.3603663444519043, -1.7848215103149414, 0.3326650559902191, 0.0152808902785182, 1.109999179840088, 1.1245818138122559, 0.2575176954269409, 0.8663414120674133, 2.8718855381011963, -0.2814072072505951, -0.3061535656452179, -2.348647117614746, -1.6367602348327637, 0.13974259793758392, -0.5944269895553589, 0.6713876128196716, -0.85930335521698, 0.525314211845398, 1.0749658346176147, 0.8472476005554199, -0.9014675617218018, -1.440640926361084, 0.10839241743087769, 0.6500116586685181, 0.14794157445430756, 1.3511731624603271, 0.7401860952377319, 0.010289872996509075, -0.545577883720398, 1.9443317651748657, -0.9451045393943787, 0.26984497904777527, 1.593874454498291, 0.444568008184433, 1.0594422817230225, -1.3337466716766357, 0.9908202886581421, -1.183424949645996, 0.38343703746795654, 0.43081915378570557, 0.2641603648662567, 1.7422631978988647, -0.9815413951873779, -0.2915314733982086, 0.14504151046276093, 0.6289293169975281, -0.7690431475639343, 1.1795694828033447, 1.2977054119110107], [-0.48309290409088135, 1.1745303869247437, 0.06491325795650482, 1.4274420738220215, 1.0257455110549927, 0.33935546875, 0.26975947618484497, 0.1131914034485817, 1.5305120944976807, -0.29341092705726624, -1.2191896438598633, -1.3569883108139038, 0.6762083768844604, 1.9232654571533203, -1.1813949346542358, -0.7763836979866028, -0.9283964037895203, 1.627182960510254, 1.7999446392059326, 0.07260225713253021, -0.5935921669006348, 0.4564432203769684, 0.7942737936973572, 1.4794235229492188, 0.6990066766738892, 1.078526496887207, 0.42272496223449707, -0.8460071682929993, -1.1355952024459839, 1.3005470037460327, -0.1804133802652359, -1.682850956916809, -0.04585615172982216, -0.26899489760398865, -2.60718035697937, 1.2453757524490356, 0.6404597163200378, 0.5581213235855103, -0.7360796928405762, -1.15519118309021, -0.05769176408648491, -0.8745189905166626, 0.6761085987091064, -0.5616137981414795, -2.046251058578491, -1.9261294603347778, -1.1409969329833984, -1.1597518920898438, -1.8611118793487549, 0.21192002296447754, -1.1459585428237915, -1.4025236368179321, -0.10633346438407898, -1.9824090003967285, -0.40473517775535583, -0.24797044694423676, -0.4233474135398865, -1.0387593507766724, -0.4524427652359009, 0.6450257897377014, -1.5371114015579224, -0.8815262317657471, -0.02507457509636879, -2.3057732582092285, -0.27301302552223206, -0.07863801717758179, -0.12273672223091125, 0.14466726779937744, 0.4518711566925049, -0.5676578283309937, -0.6652082204818726, 0.17233331501483917, -0.765937089920044, -0.6183724403381348, 1.2383108139038086, 0.44403326511383057, -0.10932379961013794, -0.1402418315410614, -0.7870786190032959, 0.5365449786186218, 0.19005507230758667, -0.03173549473285675, -0.2496488094329834, 1.023812174797058, -1.917378306388855, -0.2730545699596405, -0.37001150846481323, -1.7964494228363037, 0.37464356422424316, 1.1781425476074219, 0.06925264000892639, 0.5234500765800476, -1.4089727401733398, 0.06744354218244553, -0.10666870325803757, 0.1314634382724762, -0.25024205446243286, 1.247031807899475, -0.524281919002533, -0.02362162061035633, -0.47705307602882385, -0.19700859487056732, -0.046907663345336914, -1.9265992641448975, -1.008355736732483, -0.633525013923645, -0.27743983268737793, -0.3929266333580017, 0.6206369400024414, 1.5916799306869507, -0.13448329269886017, 0.2992352247238159, -0.21660327911376953, 0.9969505071640015, -0.3690616488456726, -1.7488452196121216, -0.03852057084441185, -0.2902483344078064, 0.2318969964981079, 0.24084633588790894, 1.0139496326446533, -1.1347808837890625, -0.18751630187034607, 0.199937105178833, 1.5739622116088867, -1.9335808753967285, 2.0833230018615723, -1.595051646232605, -0.6228786110877991], [-0.2812314033508301, 0.06240471079945564, 0.6302732229232788, 0.8586499094963074, -0.4726375341415405, 0.3094034194946289, 0.975161075592041, -0.7243145704269409, 0.542481005191803, -0.3031250238418579, 0.8979403376579285, -0.7917922735214233, 1.0475339889526367, 0.09149036556482315, -0.5803592205047607, 1.8912220001220703, -0.35432687401771545, 0.050503190606832504, -0.9012495875358582, 1.1881346702575684, -0.24524733424186707, -0.03772277384996414, -0.4745728075504303, 0.2040538787841797, 2.211883544921875, -0.9503212571144104, -0.41671112179756165, 0.10989052057266235, 0.02476658672094345, 0.4061126410961151, 1.2932438850402832, -0.35871556401252747, -0.5648384690284729, -0.7473688125610352, -2.481485366821289, -1.654440999031067, 0.8562354445457458, -0.48549872636795044, -2.696577548980713, -0.8704061508178711, -2.2209904193878174, -1.5297770500183105, -0.3628930151462555, 0.8153924942016602, -0.182512566447258, 0.18564128875732422, -0.4877702295780182, 0.7056341767311096, -0.3019641041755676, -0.02334827370941639, -0.8991623520851135, -0.8941110968589783, 1.5143064260482788, 1.2717645168304443, -0.6264027953147888, -1.0876115560531616, -0.4474385976791382, -0.17514845728874207, -0.7500084042549133, -0.6990624070167542, -0.7364481091499329, -0.5014680624008179, 1.3290222883224487, -0.8459162712097168, 2.422245502471924, -0.7757985591888428, 0.6977435350418091, -2.4734435081481934, -0.20422770082950592, -0.8119995594024658, 0.48967447876930237, -2.2084946632385254, -0.8193679451942444, -1.377895474433899, -0.20715688169002533, -0.05031207203865051, 1.7443499565124512, -0.23601576685905457, -0.239643856883049, 1.3588911294937134, -0.3775397837162018, -0.8653362989425659, 1.8683655261993408, 0.057524293661117554, 0.7246277332305908, 0.20325757563114166, -0.23011429607868195, -0.2903496026992798, -2.888228416442871, 0.8379160761833191, 2.5553958415985107, -0.818419337272644, -2.0461232662200928, -1.0038050413131714, -2.5244224071502686, -1.3638120889663696, 0.09356042742729187, -0.07026489078998566, -0.7805718779563904, -0.013838939368724823, 0.19720369577407837, 0.2307378649711609, 0.08264710754156113, -1.488786220550537, -0.5225890874862671, 0.5532066226005554, -1.3214079141616821, -1.6546322107315063, -0.053569432348012924, -1.2337572574615479, 0.13200512528419495, 0.419455885887146, -0.25419315695762634, -1.033069372177124, -0.9704927802085876, -1.4820048809051514, -0.732363760471344, -1.6109431982040405, -1.8774062395095825, 0.8654677867889404, 1.3718540668487549, 2.1056721210479736, 0.21176205575466156, -1.6275420188903809, -0.5438897013664246, -0.16306239366531372, 0.6671257615089417, -0.9580323696136475, 0.06393472850322723], [1.12855863571167, -0.1881650984287262, 1.2474440336227417, 1.6809319257736206, -0.1899944394826889, -1.1208535432815552, -0.8024752736091614, -0.7958197593688965, 0.10990580916404724, -0.6854075789451599, -0.13797836005687714, 1.2087535858154297, -0.8632256984710693, 1.1523566246032715, 0.11747652292251587, 0.24959298968315125, 1.420742392539978, 0.12130241841077805, -0.8696621060371399, 0.6970687508583069, -0.14823326468467712, -0.5121622681617737, -0.5069870352745056, -0.536441445350647, 0.43615439534187317, 0.24813438951969147, -0.6239498257637024, 0.9886100888252258, -0.05596204102039337, 0.4677765965461731, -0.30807945132255554, -0.47419995069503784, -1.3806389570236206, 1.9175008535385132, -0.4205484390258789, -2.104464054107666, -1.7107759714126587, -1.7901957035064697, -1.5583280324935913, 1.0867252349853516, 0.7497402429580688, -0.49140024185180664, 0.7411947250366211, -0.8530896306037903, -0.6069187521934509, -1.308760643005371, -1.5027880668640137, -0.054377831518650055, -0.002352025592699647, 0.97257399559021, -2.0236849784851074, -1.0996477603912354, -0.08844327181577682, 1.107987642288208, -1.4015611410140991, 0.3900561034679413, 0.21837221086025238, -0.4479636251926422, 0.20157869160175323, 1.7031605243682861, 0.54038006067276, 1.5225212574005127, -0.46519050002098083, 0.45503005385398865, 1.0175827741622925, -1.0364130735397339, -0.536264181137085, 0.15560656785964966, -2.021346092224121, -0.10157205164432526, -1.7554694414138794, -0.4581156373023987, -1.208999752998352, -0.12156152725219727, -1.0415247678756714, 0.3974262773990631, -0.5148374438285828, 1.4412288665771484, 0.5151081681251526, 0.07141882926225662, 1.0529165267944336, 0.5419532656669617, -0.05254794657230377, 0.6470497846603394, 1.3079047203063965, 2.8191192150115967, 0.9750874042510986, -0.0603216290473938, -0.7444947957992554, 1.9858031272888184, 0.248690664768219, -0.9360961318016052, 0.44622448086738586, -1.5530990362167358, -1.2833836078643799, 0.1986561417579651, -0.5365905165672302, 1.0817307233810425, -1.0746304988861084, 0.7678295969963074, -0.1389714479446411, -1.1404918432235718, -0.2875908613204956, 0.041427671909332275, 1.3069111108779907, 0.7032698392868042, 0.6614518761634827, 1.4579211473464966, -1.6684415340423584, 0.7324792742729187, 1.547170639038086, -0.8805618286132812, 1.4989650249481201, 0.4613870680332184, 2.7636878490448, -0.19704416394233704, 1.7759594917297363, 0.07520606368780136, 0.05258186161518097, -1.314637541770935, -0.4070144593715668, -0.6588212847709656, 1.8278169631958008, -0.8220590949058533, 0.6930129528045654, 0.6001426577568054, 0.340402752161026, 0.6211104989051819, -1.7578214406967163], [0.4461258053779602, 0.677943766117096, -0.7174592614173889, -0.5683398246765137, -0.7832933068275452, 1.249066710472107, 1.1810351610183716, 0.8414376974105835, 0.025964224711060524, 0.7593615055084229, -0.07831304520368576, -0.09091166406869888, 0.5649100542068481, 0.42081519961357117, 0.43760842084884644, -1.2564024925231934, -1.0573716163635254, -1.7334870100021362, 0.5304415225982666, -0.535079836845398, -1.139190435409546, -0.406909316778183, -1.4521689414978027, -0.5625009536743164, 0.042491160333156586, 1.5426857471466064, -0.009445116855204105, -0.9483201503753662, -0.7997651696205139, -1.1702276468276978, 0.6061390042304993, -0.5686218738555908, -0.3304031491279602, 0.9195559620857239, -0.18648523092269897, 0.49360111355781555, -0.4910697937011719, -0.7503634691238403, -1.9602004289627075, -0.6873022317886353, -0.07321900874376297, 0.9004685282707214, 1.2222896814346313, -0.07011332362890244, -0.5060203075408936, -1.647599220275879, 0.35855141282081604, -0.46570611000061035, 0.44246822595596313, -2.6095387935638428, 0.8578484654426575, 0.8556726574897766, 0.2380232810974121, -1.2605671882629395, 0.8846561908721924, -0.1173899918794632, 0.0493059866130352, 0.43398988246917725, 0.681686520576477, 1.213546633720398, -0.23788584768772125, 1.5371448993682861, 1.7359281778335571, -0.3119681477546692, 1.3558791875839233, -0.7533524036407471, -0.2683056890964508, -0.2488217055797577, -0.5789239406585693, 0.8461543917655945, 0.6240894198417664, -1.1191155910491943, 0.7182601690292358, -1.150654911994934, -0.5495727062225342, -1.0246386528015137, 0.16699030995368958, -0.1521836668252945, 0.39957401156425476, -0.3682207763195038, -0.3828515112400055, 1.3316854238510132, 1.0377578735351562, 0.7679067850112915, -0.7949533462524414, -0.36337852478027344, 0.5405636429786682, 0.5970262289047241, 1.3042373657226562, -0.22602935135364532, -0.02199070155620575, -0.23406223952770233, 2.0134551525115967, 0.5193821787834167, 0.29576030373573303, 0.38429296016693115, -0.9070443511009216, 0.788175106048584, 1.8451974391937256, 1.2165331840515137, -1.9756228923797607, -0.44576317071914673, -0.5934905409812927, -0.2646326422691345, 0.4128017723560333, 0.2526395618915558, -1.7148182392120361, 0.1110965758562088, 0.052315663546323776, 0.08682537078857422, -0.2763230800628662, -1.0763161182403564, -0.2374909669160843, -0.6865044832229614, -0.0075330124236643314, -1.1065062284469604, -2.7089078426361084, 0.035749685019254684, 1.3195841312408447, 1.0404363870620728, -0.5704705119132996, -0.9529207944869995, 0.5389531850814819, 0.30510473251342773, -1.5594793558120728, -1.2400950193405151, -1.4730029106140137, -2.580803155899048, 0.285547137260437], [-1.9167596101760864, 0.2509077787399292, -0.9176156520843506, 1.32933509349823, 0.020080290734767914, -0.39237841963768005, -0.474575936794281, -0.3062400221824646, 0.396246075630188, -2.0393764972686768, -1.0378692150115967, -1.2019962072372437, -1.6224937438964844, -1.3711802959442139, 0.22058376669883728, -1.3270165920257568, 0.3289817273616791, 0.8200766444206238, 0.20632696151733398, -0.22154568135738373, -1.3300470113754272, 1.3152655363082886, 1.8209493160247803, 0.8181900382041931, -0.6910096406936646, 2.392608880996704, -0.28783461451530457, 1.2158544063568115, 0.18793389201164246, -2.15069842338562, -2.004227638244629, 1.2235164642333984, -0.6174492239952087, -0.6066682934761047, 0.3209548592567444, 0.22334805130958557, -0.09160979092121124, 0.7133147120475769, -0.02660931460559368, 1.4419573545455933, -0.6193098425865173, -0.43920761346817017, 0.6356286406517029, -1.4541618824005127, -0.8531230092048645, 1.3081709146499634, -1.3902792930603027, 1.7884360551834106, -1.2971781492233276, 0.3643868863582611, -0.8593935966491699, -0.034973617643117905, 0.1318979561328888, 1.4710369110107422, -0.7286452054977417, 0.1460934430360794, -1.5135806798934937, -1.210896611213684, -0.5276371836662292, -0.08343306183815002, -1.8996726274490356, -1.0988634824752808, -1.4018605947494507, -0.61017245054245, 0.5450245141983032, -0.03954059258103371, -0.7082785964012146, -1.581729769706726, 0.5344970226287842, -0.1116296574473381, 0.4324755370616913, 1.1962320804595947, 0.572199821472168, -0.9417445659637451, -0.8978327512741089, 0.594683825969696, -0.5682120323181152, -1.4264942407608032, -0.39645880460739136, 0.6362014412879944, -0.852353036403656, -0.94761061668396, -1.7453242540359497, 1.8981858491897583, -0.5985891222953796, 2.3621647357940674, 0.8411741256713867, -0.04505812004208565, 0.5168521404266357, 0.9879971742630005, -0.20127269625663757, -0.40064582228660583, -0.6654082536697388, -0.4651678204536438, -0.23866461217403412, -0.49333688616752625, 0.010381482541561127, 0.12791922688484192, 1.1368869543075562, 0.07440853118896484, 0.03893493860960007, -0.4590927064418793, -0.6562826037406921, 1.4184479713439941, -0.9964144825935364, -0.9271767139434814, -1.1971269845962524, 2.6246776580810547, -0.6063463091850281, 1.2154453992843628, 0.987289309501648, -0.038162052631378174, 0.4528508186340332, -0.6279903054237366, 1.9492133855819702, -0.20536088943481445, -0.19634586572647095, 0.9978604316711426, 0.054968614131212234, 0.6770999431610107, 0.6065787672996521, 1.7017626762390137, -0.8966132402420044, -1.9402787685394287, -0.2340136170387268, -0.8902260661125183, 0.43978214263916016, -0.15675106644630432, 0.29419398307800293], [-0.8152798414230347, -0.18001888692378998, 1.669756531715393, -0.4038983881473541, 1.975109577178955, 0.17267358303070068, 0.08199212700128555, 0.3806877136230469, 1.0921528339385986, 0.10045886784791946, -1.666624903678894, -0.6161323189735413, -1.1129319667816162, 0.5675702691078186, 0.9806373715400696, 0.868592381477356, 1.136358380317688, 0.9320082664489746, 1.4532485008239746, 2.097573757171631, 0.31787025928497314, -1.1960633993148804, -0.09660676121711731, -0.28925180435180664, -0.13727355003356934, -1.4972352981567383, 3.135474681854248, -0.33418020606040955, 0.4292599558830261, 0.005631957668811083, 0.44697824120521545, 1.4520679712295532, -0.47617414593696594, -0.4803790748119354, -0.05890479311347008, -0.5404712557792664, 0.3381032645702362, 0.26866331696510315, 0.548007607460022, -2.182462692260742, -0.9757747650146484, 1.6113356351852417, -2.4873759746551514, -0.14579910039901733, 1.4370993375778198, 2.576707363128662, -0.3073945641517639, -0.796744167804718, -1.2339165210723877, 1.0335031747817993, -0.5638830065727234, -0.08288312703371048, 0.16224493086338043, 1.193565845489502, 0.9918450117111206, -0.030211329460144043, -0.8128607273101807, 0.12796269357204437, 0.26913630962371826, -0.7237963676452637, 0.8291680812835693, -1.1336356401443481, -0.810796320438385, 1.1876620054244995, 0.44797301292419434, 1.1529278755187988, 1.498766303062439, -0.3310246467590332, -1.731605887413025, -1.122672438621521, -0.11664091795682907, 0.7242974638938904, 1.5040143728256226, -0.5182897448539734, 1.8513152599334717, 0.9643693566322327, -0.6428946256637573, -0.4770900011062622, -0.36265096068382263, -0.7112469673156738, 0.35294362902641296, -0.7222821712493896, -2.2260844707489014, 1.2471765279769897, 1.5511914491653442, -0.6078689694404602, -0.13527901470661163, 1.0253432989120483, -1.5210269689559937, 1.2421019077301025, -1.9417920112609863, -0.12644106149673462, -0.6356754302978516, -0.5664774775505066, -0.13754023611545563, 0.856838583946228, -0.1471260040998459, -1.502314805984497, 0.1934470683336258, -2.22466778755188, 0.04019561782479286, -1.2516956329345703, -1.518140196800232, 0.662562370300293, 0.6424762606620789, 1.2934550046920776, 1.7943772077560425, 0.27896979451179504, 0.7506529092788696, 0.7434555888175964, -0.9734843373298645, 0.8586455583572388, 0.8448993563652039, -0.250394731760025, -0.4672089219093323, 0.3550616204738617, 1.5467634201049805, -0.25520601868629456, -0.6396774649620056, 0.11932898312807083, -1.470095157623291, 0.548301637172699, -0.15914888679981232, 0.8704934120178223, 1.223163366317749, -0.7188509702682495, 0.6469876170158386, 1.462976098060608, 0.43320536613464355], [0.7134335041046143, 0.061143048107624054, 1.2203184366226196, -1.9414887428283691, -0.8493009805679321, -0.39098626375198364, -0.6556690335273743, -0.2672517001628876, 0.4872317314147949, -0.13207851350307465, 0.9648348689079285, 1.6039698123931885, 0.2196960598230362, 0.5877959728240967, 1.014514684677124, 1.5400022268295288, -1.7966945171356201, 0.3077772259712219, -0.04934648796916008, 0.14536505937576294, 0.30756908655166626, -0.20409271121025085, -1.0472757816314697, -0.4384656846523285, 1.4779200553894043, -2.1655478477478027, 0.5974975228309631, 0.27311453223228455, -0.17759932577610016, -0.389045774936676, -0.14984402060508728, 1.7473433017730713, 0.3813137710094452, -2.066103935241699, 0.39962485432624817, -0.13001906871795654, 0.663878321647644, 0.4560118615627289, -0.6430913209915161, 0.6080371737480164, -0.6728997230529785, 1.0586473941802979, -1.3231463432312012, -0.24754051864147186, -0.47843632102012634, 1.1134312152862549, -1.6154100894927979, 0.02017126977443695, 0.2834847867488861, 1.563866138458252, -0.10370396822690964, 0.1034085601568222, -0.9862749576568604, 1.377170205116272, -0.6463066935539246, -1.1315083503723145, -0.06616872549057007, 0.9072861075401306, 0.010385507717728615, 2.5460574626922607, -0.825329601764679, -0.36081361770629883, -2.491636276245117, -0.20372295379638672, 2.7548351287841797, 0.11339665204286575, -0.22134315967559814, -1.3328560590744019, 0.21128374338150024, -0.8330779075622559, -0.024477943778038025, 0.5088428854942322, 0.5316651463508606, 0.5075982809066772, 1.0697708129882812, 0.8298496603965759, 0.018047474324703217, -0.5156896114349365, 2.0896494388580322, 0.07961809635162354, -2.604971408843994, -0.03855283930897713, 0.24174199998378754, -0.11660518497228622, -0.453462153673172, 0.3405388593673706, 0.21859019994735718, 0.00473712058737874, -1.1441929340362549, -1.9166754484176636, 0.5970982909202576, -0.6168221831321716, 1.1109141111373901, 1.6163133382797241, 1.2972551584243774, 1.2474453449249268, -0.20989763736724854, 0.2284931242465973, -0.848343551158905, -0.42994579672813416, 1.0430246591567993, 1.3577147722244263, 0.08336638659238815, -1.2688454389572144, -1.6841645240783691, -0.07291994243860245, -2.103351593017578, 0.7286115288734436, -0.4627436399459839, 0.6683077812194824, -0.5059899091720581, -0.942389726638794, -1.136376976966858, -1.8382930755615234, -0.502220094203949, 1.2240523099899292, 0.8536454439163208, 0.07196284830570221, 0.6973828077316284, -1.267126441001892, -0.04377039521932602, 0.36491283774375916, -0.04831986874341965, -1.1973000764846802, -2.171355724334717, 0.9327746033668518, 0.900753378868103, -1.5447514057159424, -0.9285274744033813], [1.3615831136703491, -1.1199290752410889, 1.033707618713379, -1.6645863056182861, -1.6143556833267212, -0.2049127221107483, -1.6035926342010498, -1.3683987855911255, 0.6370235681533813, 1.0257248878479004, -0.5948983430862427, 1.3756859302520752, 0.6590972542762756, -0.6758023500442505, 1.2891299724578857, -1.099906325340271, -0.09926556795835495, -0.09021701663732529, 1.1666468381881714, -0.6889433860778809, 0.17410950362682343, 1.324736475944519, -0.40925133228302, -3.0270156860351562, -0.13015463948249817, 0.4984820485115051, 0.13299578428268433, 1.2524495124816895, -0.0424647182226181, 0.326337605714798, -0.18309801816940308, 0.5507838129997253, -0.9655012488365173, 0.3035225570201874, -0.6612186431884766, 0.2599918246269226, 1.1087572574615479, 0.29373887181282043, 0.6201287508010864, 0.9725703001022339, -0.39357244968414307, -0.602229654788971, 0.3480239510536194, 0.11030928790569305, 0.8507478833198547, -0.15153877437114716, -0.16728773713111877, -0.7262941002845764, -1.8002957105636597, 1.3557310104370117, -1.0787227153778076, -0.6934729814529419, -0.39422011375427246, 0.8731639385223389, 0.920056164264679, -0.4265254735946655, 0.054292041808366776, -0.088007353246212, 0.6326249241828918, 1.0127508640289307, -0.1773195117712021, 1.3405406475067139, 0.18832947313785553, -2.346712589263916, -0.5906272530555725, 1.424376130104065, -0.8151037693023682, 1.2959696054458618, -0.28382548689842224, -0.09773508459329605, 0.10103552788496017, 1.1554926633834839, -1.0001212358474731, -0.5860519409179688, -0.8080379366874695, 1.200679898262024, 1.2856566905975342, 0.5077708959579468, -0.5596976280212402, 0.4165779650211334, 0.12755686044692993, -0.31126290559768677, -1.6092609167099, -0.5488811135292053, 3.0127315521240234, -0.37906184792518616, 0.05739649385213852, -1.4333609342575073, 0.858216404914856, 0.4379047453403473, 0.04073842614889145, -0.02315601147711277, 0.7812639474868774, -0.2434825450181961, 0.2021878957748413, -0.48928719758987427, 0.5996403694152832, -0.1230008453130722, 0.3314465582370758, 0.7659922242164612, 0.7562530040740967, -0.8785033822059631, -0.0739896222949028, -0.8078808188438416, -0.4445268213748932, 0.22481849789619446, 0.6244556307792664, 0.7047654390335083, 1.2908287048339844, -2.0521225929260254, 0.9185320734977722, 0.480904221534729, 0.8898711800575256, -2.452044725418091, 0.46599358320236206, 0.8745819926261902, -0.2472507804632187, 0.8052858710289001, 0.3977266252040863, -1.2771509885787964, -1.2986279726028442, -0.25143057107925415, -0.6748451590538025, 0.828141987323761, -0.28767526149749756, 1.967362642288208, 0.20913533866405487, 0.8134394884109497, 0.5758193135261536], [1.3860105276107788, 0.1844324767589569, 0.010222313925623894, -0.15636175870895386, 0.9110763669013977, -1.6683663129806519, -0.23619620501995087, -2.4047579765319824, -1.2359334230422974, 1.0388067960739136, -1.114774465560913, 1.773430585861206, 1.6065819263458252, -0.945914626121521, -0.07927295565605164, -0.41772133111953735, 0.9344390630722046, -0.4315902292728424, 0.49562868475914, 1.3269004821777344, -0.8391616344451904, -0.5982945561408997, 0.48865142464637756, -1.1043763160705566, -1.106754183769226, 0.5746470093727112, 0.9695590734481812, 0.8071123361587524, -0.10325302183628082, -0.8650885820388794, 1.4517357349395752, 1.247423529624939, 0.010401555337011814, -0.8488172888755798, 1.4678537845611572, 1.405654788017273, -1.0952612161636353, -1.5386855602264404, 0.8888816833496094, -0.9497119188308716, 1.1294814348220825, -1.1751315593719482, -0.174979105591774, 1.0064733028411865, 0.25901108980178833, -0.7773271203041077, 0.49764835834503174, 1.0723092555999756, 0.16060781478881836, 0.6453773975372314, -0.00921204686164856, 0.27208930253982544, -0.5198934078216553, -0.3387780487537384, 1.0014010667800903, -0.2894901633262634, -0.9772706031799316, 1.6866809129714966, -1.751160740852356, -0.4092218577861786, 0.45094674825668335, 0.7016779780387878, -0.011923802085220814, -0.5586057901382446, -1.0838680267333984, 1.6732269525527954, -0.2450244277715683, 0.051804136484861374, 0.2302800714969635, 1.3229960203170776, -1.4035847187042236, 1.4712188243865967, -0.37366947531700134, -0.5922831296920776, 0.6842817664146423, 1.0678305625915527, -0.09132563322782516, -2.0302250385284424, -0.24178902804851532, -0.38027074933052063, 0.4605977535247803, 0.1558854877948761, -1.041271448135376, 0.2979326546192169, 0.6663862466812134, -0.34473302960395813, -0.6669664978981018, 0.4320199191570282, 0.6529459953308105, 1.0726279020309448, -0.05798739194869995, -0.4628453850746155, -0.9090664982795715, -0.3121168315410614, 0.5324299335479736, -0.8787997961044312, 1.4077969789505005, -2.094954490661621, 1.5562585592269897, 0.11611705273389816, 0.898552656173706, 0.9654667973518372, 1.0084962844848633, -0.9194117188453674, 0.20624034106731415, -0.8153789639472961, -1.0762333869934082, -0.5635102391242981, -0.13754847645759583, 0.707970380783081, -0.7980023622512817, -0.11022818833589554, -1.1688811779022217, 0.8557345867156982, 1.7631614208221436, 1.1211307048797607, -0.37922579050064087, -0.9279149174690247, 0.6126341819763184, -0.1327141672372818, 0.7081159353256226, 0.9067872762680054, 0.3360078036785126, -0.7251459956169128, 1.4033839702606201, -0.1960054636001587, -0.911108672618866, -0.07912778854370117, 0.23253114521503448], [0.25514861941337585, 0.17044776678085327, 0.10119453072547913, -0.5737990140914917, -0.1367836743593216, -0.6787676811218262, -2.17814564704895, -0.8631240725517273, -0.6319145560264587, 0.22394976019859314, -2.9684832096099854, -1.634596824645996, 0.5201259255409241, 1.1602779626846313, -0.2216789573431015, 0.5961626172065735, -2.005263328552246, 1.249456763267517, -0.21573330461978912, -0.34490883350372314, -0.30389419198036194, -0.9562604427337646, 1.5968786478042603, -0.41113096475601196, 1.1327826976776123, -1.2974478006362915, -1.7431271076202393, 1.4607971906661987, 1.4500913619995117, 3.1165146827697754, -0.34228241443634033, 1.261523962020874, -0.9386976957321167, -1.5390983819961548, 1.1569825410842896, -1.5543354749679565, -0.4113099277019501, 0.6268763542175293, -1.3146826028823853, 0.37929853796958923, 1.2205255031585693, 0.8426806330680847, 0.9498815536499023, -0.5388121008872986, -0.8523131608963013, -0.033020298928022385, 0.5205696225166321, 0.8750773072242737, -0.04224473610520363, 0.07210026681423187, -0.008425397798418999, 0.769390344619751, -1.023020625114441, 1.771121859550476, -0.39974266290664673, -0.16836635768413544, 0.31491148471832275, -0.502873420715332, -1.0017995834350586, -1.1087191104888916, -0.2829550802707672, 0.35827091336250305, 0.125401571393013, 0.31132110953330994, -0.7817594408988953, 1.7655181884765625, 0.7610661387443542, -0.9547627568244934, -0.5865691900253296, -1.9396388530731201, 0.6065358519554138, 0.02903520129621029, -0.4397643804550171, -0.6378520131111145, 0.12723088264465332, 1.4711145162582397, -0.18265460431575775, 0.40660256147384644, 1.0256223678588867, -1.4555848836898804, -0.792241096496582, 0.8238676190376282, 0.2893548905849457, -0.3127717971801758, 1.218087911605835, -1.7927945852279663, 0.6281299591064453, 1.0028152465820312, -0.2027241289615631, 0.7462605834007263, -0.9059908986091614, -0.8272076845169067, 0.9674509763717651, -0.5210262537002563, 0.7466660141944885, -0.09630710631608963, -0.30906248092651367, 0.6155260801315308, -0.9797509908676147, -0.09566988050937653, 2.307676076889038, 0.34473857283592224, -0.4313482940196991, 0.5410828590393066, -1.211238145828247, -0.22644726932048798, 0.2518114447593689, 0.09493949264287949, -0.2249322384595871, 1.36111581325531, 0.9122700095176697, 0.5438348054885864, 1.9175971746444702, 1.0061026811599731, -1.0455963611602783, 0.1152799129486084, 1.0688185691833496, -1.2619593143463135, 0.21726395189762115, -0.3668034076690674, -0.9410749077796936, -0.6014700531959534, 1.4853975772857666, -0.43619370460510254, -0.07995732873678207, 0.49373340606689453, 0.675947904586792, 1.4395523071289062, -0.2469688504934311], [0.4008273780345917, -2.0885469913482666, 0.5091711282730103, -0.8977623581886292, -0.04934520274400711, 1.9750553369522095, 0.8587389588356018, -1.436569094657898, 2.3316543102264404, -0.6811844706535339, -2.983248710632324, 1.5486050844192505, -1.0773714780807495, 0.10694219917058945, 1.2906898260116577, -0.2554287612438202, 0.08381635695695877, 0.31435853242874146, -0.725631833076477, 1.1066317558288574, -0.8588284254074097, -0.7750759124755859, -1.4398515224456787, 0.35001760721206665, 0.5803651809692383, -0.5786051154136658, 0.7965907454490662, -1.2261626720428467, 0.4835662543773651, 0.11213304847478867, -0.280716210603714, 0.19802796840667725, 0.4204665422439575, -1.0061087608337402, 1.5719162225723267, -0.4318527281284332, -0.17232736945152283, -0.44350549578666687, 1.9735140800476074, 0.38470491766929626, 0.15778848528862, -1.4565231800079346, 1.1789005994796753, 0.3171026408672333, 1.1420960426330566, -0.7834144234657288, -0.15450558066368103, 0.4592462182044983, 1.021973967552185, -0.11944057792425156, 1.54896080493927, 0.38401952385902405, -0.1901751458644867, 0.2496170997619629, -1.4671145677566528, 0.35055509209632874, -0.4688623547554016, 2.154649257659912, -0.6584009528160095, 1.0189749002456665, 0.06583786755800247, -0.28672993183135986, -0.5018900632858276, -1.5097523927688599, 1.177324891090393, -1.0324673652648926, 0.5358865857124329, 0.5394015908241272, -0.4114520847797394, -0.7149786949157715, -0.6437004208564758, -0.1887107640504837, 0.348573237657547, -0.24414187669754028, -1.6906499862670898, 1.4671564102172852, -0.37478479743003845, 0.5913991928100586, 1.882362961769104, 1.8877789974212646, 0.8248677849769592, 0.19514790177345276, 0.840865433216095, -1.1807202100753784, -0.7842257618904114, 0.6888271570205688, -0.46629834175109863, 1.2898318767547607, -2.5346903800964355, -0.2668774127960205, -0.703131914138794, 1.3563878536224365, 0.3351808190345764, -1.555245041847229, -0.6617432832717896, -1.0520323514938354, -0.10555993020534515, 0.3504934310913086, -0.8254520297050476, 0.011627710424363613, 0.3962326645851135, 1.2469905614852905, 0.08551257848739624, -0.8554909229278564, -0.6888853311538696, -1.538100004196167, 0.26423215866088867, -0.44063010811805725, -2.4673171043395996, 0.5245868563652039, -1.1722172498703003, 0.10296634584665298, 1.1422733068466187, 0.14137333631515503, -0.6994560360908508, 0.5555973052978516, 0.450774610042572, -1.2347886562347412, -0.9345315098762512, 1.026794672012329, -0.25102221965789795, 0.23650123178958893, 0.6464868187904358, -0.8642058372497559, -1.4580031633377075, -0.03023896925151348, -1.6570379734039307, -1.1995536088943481, -0.05800913646817207], [0.3672511875629425, -1.1734943389892578, -1.502902626991272, -0.7410166263580322, 0.1425086259841919, -0.5148541927337646, -0.4398588240146637, 0.9026622772216797, -0.14468570053577423, 1.2059886455535889, -0.2990718483924866, 0.8648941516876221, -0.6282671093940735, 0.2992655038833618, 0.825447678565979, 0.25240206718444824, 0.8857927322387695, -1.850849986076355, 1.654977560043335, 0.9625139832496643, 1.6250673532485962, -0.3728291690349579, -1.4100326299667358, -1.9971058368682861, 1.315475583076477, 0.5561105012893677, 1.3376981019973755, -0.7497628331184387, 1.1863514184951782, -2.78535532951355, -0.3014572858810425, 0.00856062863022089, -0.46219873428344727, 1.6940778493881226, -0.8672274351119995, 0.6495921015739441, 0.3745107352733612, 0.38999930024147034, 0.7687538862228394, 0.5403393507003784, -2.082709789276123, -0.3662899434566498, -0.7535738945007324, -1.4923620223999023, 0.41849979758262634, 0.1135978251695633, -1.625043272972107, -1.6770390272140503, 0.08176909387111664, -0.16798828542232513, 0.6026978492736816, 1.5317375659942627, -0.4899960458278656, 0.6440023183822632, -1.3018227815628052, 0.7757853865623474, 0.0015853674849495292, 0.015709295868873596, 0.7757829427719116, 0.9065666198730469, -0.06353338062763214, 0.8844383955001831, 1.0995005369186401, 1.7624037265777588, -0.25202304124832153, 0.6355693936347961, 0.3341350853443146, 0.11746270209550858, 0.17615275084972382, -0.12858183681964874, 0.9727535843849182, 1.2361905574798584, -1.6731253862380981, -0.10865002125501633, -1.250462293624878, 0.25216302275657654, 0.49726423621177673, 0.34454330801963806, -0.2757807970046997, 0.9507684111595154, -0.46995478868484497, 0.3609117269515991, -0.6057013869285583, -0.2338479459285736, 0.5596202611923218, 2.3809635639190674, -0.23267529904842377, 2.079016923904419, -0.52092045545578, -0.6408753395080566, -0.801362931728363, 0.2449072152376175, 1.1385506391525269, 0.4563697576522827, -0.7828618884086609, 0.4663892686367035, 0.4061313271522522, -2.432568073272705, 1.2650209665298462, 0.5299156904220581, -1.8257628679275513, 0.02756602317094803, -1.143623948097229, -1.6984769105911255, 0.4813545346260071, 0.5836440324783325, 0.31329524517059326, 0.3303788900375366, 0.012895755469799042, 1.655803918838501, -0.21834160387516022, 1.8506661653518677, -0.7054117918014526, -0.3559249937534332, -0.2524321973323822, -0.010370567440986633, -0.621889591217041, 1.5127296447753906, -1.3507264852523804, 0.05957210063934326, 0.9933019876480103, 0.1425374448299408, -0.3909260332584381, -0.6810892224311829, -1.5863624811172485, 0.4586353302001953, -1.959909439086914, 0.16088160872459412, 0.3643461763858795], [-0.11939439177513123, -0.7223906517028809, -0.36892902851104736, 1.6838256120681763, -0.09388822317123413, -0.5276099443435669, -2.1298458576202393, -1.2023954391479492, -0.25963136553764343, 2.7605066299438477, -0.29634055495262146, 1.7809700965881348, -0.27311357855796814, -0.08510739356279373, 0.18012085556983948, 0.16162432730197906, 0.0016813986003398895, -1.0653928518295288, 0.978484034538269, 0.7099534273147583, -0.902463436126709, 2.0876965522766113, -0.21969564259052277, 2.7520623207092285, 0.020075786858797073, -0.24091528356075287, -0.9240050911903381, 0.7344648241996765, 0.2975042462348938, 0.6073158383369446, -0.877453625202179, 0.03902926295995712, -2.064509868621826, 1.0229567289352417, 0.40935227274894714, 0.4545665383338928, -1.1496193408966064, 1.2438443899154663, 0.528417706489563, -0.1845559924840927, -0.3954050540924072, -0.15652713179588318, 0.4107542634010315, 0.237637460231781, -0.7136470675468445, -0.30786991119384766, -3.329934597015381, 0.33120349049568176, 0.836754560470581, -1.2921844720840454, 0.017487771809101105, -1.0600024461746216, 0.18964490294456482, -0.9144482016563416, -2.6523008346557617, -1.0701260566711426, -0.6636717915534973, 0.5395785570144653, -0.4181581139564514, 0.43621864914894104, -1.16364324092865, 0.8732222318649292, 0.9798990488052368, -0.7558924555778503, -0.0980324074625969, -1.2013031244277954, 0.20640595257282257, -0.6389592289924622, -0.5357640385627747, 0.32272815704345703, 0.0841962993144989, 2.5201244354248047, -0.7102171778678894, 1.6821322441101074, 0.4844663143157959, -0.35331666469573975, 0.36521679162979126, 0.5030824542045593, 0.615374743938446, 0.5377601981163025, 0.8236538171768188, -0.5205832123756409, -0.41448482871055603, 0.2292836606502533, -1.1442687511444092, -0.5754207372665405, 1.5258796215057373, -1.2902753353118896, 0.7875006198883057, -0.23132409155368805, -1.080134630203247, -0.03609219938516617, 0.5762777328491211, -0.21838843822479248, 0.5154537558555603, 0.8137997984886169, -0.8043526411056519, -1.124890685081482, 0.7532512545585632, 0.8581812381744385, -1.3699947595596313, -0.21024081110954285, 0.16278010606765747, -0.5116545557975769, -2.0609521865844727, -2.0637919902801514, -1.8846076726913452, -0.08691272139549255, -0.26029884815216064, 1.4858038425445557, 0.47163262963294983, -1.3130978345870972, 0.7406775951385498, -1.0062040090560913, -0.350772887468338, -1.6821763515472412, 0.6034152507781982, 0.4098877012729645, -0.3610900044441223, 0.8837617635726929, -0.6987608075141907, 0.5121161341667175, -0.7554413676261902, -0.31225958466529846, 1.278001308441162, 2.032848596572876, 0.9009173512458801, 0.9598220586776733, 1.4646366834640503], [-1.8200966119766235, -2.6646366119384766, -0.8584544658660889, -1.6136854887008667, 0.05718567222356796, -0.9929656982421875, -0.2668808400630951, 0.23864826560020447, -0.906923770904541, 2.2690951824188232, 1.3647764921188354, 0.3184463679790497, -0.9423210024833679, -0.7365907430648804, 0.4539756178855896, -0.9172887206077576, -1.768265962600708, 1.8386337757110596, -1.3423888683319092, 0.8998279571533203, -0.8209051489830017, 0.45101532340049744, 0.27380555868148804, -0.1143656000494957, 1.2762024402618408, 0.16695307195186615, -0.5902113318443298, -1.2029919624328613, 0.22191163897514343, 1.2164396047592163, 0.2678188681602478, 0.3888123631477356, -2.023172378540039, -0.44719263911247253, -0.26584044098854065, -0.3212425708770752, -1.7761887311935425, -0.8529167771339417, 1.21136474609375, 0.9472886919975281, 0.715546727180481, -0.9895362257957458, -0.41345953941345215, -1.3725528717041016, 0.48423686623573303, -0.1944960355758667, -0.39253881573677063, 0.10725414007902145, 1.6702920198440552, 0.3749102056026459, -0.025067202746868134, 1.017651915550232, 0.4661259055137634, -1.3523025512695312, -0.07934154570102692, -0.5811122059822083, -0.48008885979652405, 0.24572961032390594, -0.9243094325065613, 0.13367611169815063, -0.5814199447631836, 0.50551837682724, -0.6426802277565002, 0.06023857370018959, -0.43833038210868835, -1.1234527826309204, 1.8018581867218018, 0.12137772142887115, 1.4293749332427979, -0.5113835334777832, -0.9590681195259094, 0.20976799726486206, 0.8942921161651611, -0.3481862246990204, 0.20256800949573517, -1.406062126159668, -1.9909017086029053, 1.230623483657837, 0.5119500756263733, 0.46266019344329834, -0.067902110517025, -0.7607346177101135, -0.27321910858154297, 1.3383159637451172, -0.9566455483436584, -0.13758373260498047, 0.1946249008178711, 0.1589076966047287, -1.2439463138580322, -1.7906678915023804, 0.2039872258901596, 0.2861802577972412, -1.01032555103302, -0.8125572204589844, -0.4940875172615051, 0.39789819717407227, -0.039730533957481384, 0.9955241680145264, -0.3969990909099579, 1.611830472946167, 0.7999884486198425, 1.9351251125335693, -0.30432048439979553, -1.1210896968841553, 0.2973530888557434, -1.3153353929519653, -0.5271336436271667, 0.13203269243240356, -0.06518841534852982, 1.1194078922271729, 0.23243896663188934, -0.5031498074531555, -0.07352946698665619, -0.7711200714111328, -1.0599216222763062, -0.21521443128585815, 2.4204845428466797, 0.19540852308273315, 1.451480507850647, -0.026380710303783417, 2.265972852706909, -0.8014782667160034, -1.422947883605957, -0.37587839365005493, -0.35805743932724, -0.15014658868312836, -2.0212526321411133, -1.3484020233154297, 0.07246953248977661], [1.8984993696212769, -0.10583572089672089, 1.8423378467559814, -1.5157701969146729, -0.6910849809646606, -0.07347168773412704, -1.2984412908554077, 0.7425394058227539, -0.01109092216938734, -0.10631763190031052, 0.17544743418693542, -0.7372642159461975, -0.494976282119751, 1.2320610284805298, -1.4687165021896362, -0.2981959879398346, -0.9134692549705505, 0.6797367930412292, 2.6991896629333496, -0.2298891693353653, -1.586974859237671, -1.9156428575515747, 1.6778403520584106, 0.2371101677417755, -1.4441932439804077, 0.4231525659561157, 0.1007431373000145, -0.43231400847435, 0.5040602684020996, 0.1103229820728302, 0.8095663189888, -1.206026554107666, -0.6442307233810425, -0.1184515580534935, 0.6719818115234375, 0.17424769699573517, 0.5574238896369934, 0.5819047689437866, 0.6895705461502075, -0.5363139510154724, -1.1301919221878052, 0.6127398610115051, 0.6751965880393982, 1.0280283689498901, -0.5474085211753845, 0.9651769399642944, -0.6665302515029907, -0.5460938215255737, -0.22004665434360504, -1.6209685802459717, 1.1520432233810425, 0.21597427129745483, 1.0793260335922241, -0.7544600367546082, -1.6577845811843872, -0.02442028746008873, -0.35100099444389343, -1.4357420206069946, 0.22467713057994843, -0.50522780418396, 0.7981961965560913, 2.1853034496307373, 0.5873039960861206, 1.8486753702163696, 0.3067780137062073, -0.7424899339675903, 2.4801714420318604, -1.8288147449493408, -0.011679059825837612, -1.6400359869003296, 0.3036780059337616, -0.4675540626049042, -0.32283347845077515, 1.78804612159729, 0.9306976199150085, -0.09609304368495941, 1.060191035270691, 0.02661030925810337, 1.2852073907852173, 0.05009637400507927, -1.3184539079666138, 1.5323185920715332, -1.7892895936965942, 1.9883522987365723, 0.6335413455963135, 1.2051033973693848, -0.6319432258605957, 0.8096346855163574, 0.9494534730911255, 0.813045859336853, 1.7500864267349243, 2.4305830001831055, -1.2239346504211426, -0.5888094902038574, 0.5798595547676086, -0.9285895228385925, 0.26036208868026733, 0.26477277278900146, -0.5022697448730469, 0.5016851425170898, -0.19595234096050262, -0.4247382581233978, 0.5193807482719421, -1.15668785572052, 1.5227670669555664, -1.1020302772521973, -1.7810136079788208, 0.44301578402519226, -0.30930623412132263, 0.11308334022760391, 0.4452781677246094, -1.6320624351501465, 0.9329339265823364, 0.3397805690765381, -1.6275514364242554, -0.11847802251577377, 0.318432092666626, 0.21574324369430542, -0.1357199102640152, 0.2833358347415924, 1.6212252378463745, 0.5183300971984863, 0.7256130576133728, -0.4219730794429779, 0.8481295704841614, 1.2472320795059204, -2.0234313011169434, 0.8278083205223083, 0.3804491460323334], [0.24183420836925507, 1.709446907043457, -0.26333168148994446, -0.13328106701374054, 0.007894898764789104, 0.782866358757019, -0.05591397359967232, 1.198213815689087, 1.755066990852356, -1.8961365222930908, -0.6990485191345215, 0.21893297135829926, -0.16292087733745575, -0.370072603225708, -0.8605579733848572, 0.45294439792633057, -0.7352786064147949, -0.7175891995429993, -0.5639755129814148, -0.8647801876068115, 1.9880446195602417, 0.015057063661515713, -0.44103115797042847, -0.8359602093696594, -0.6302350163459778, -1.514035701751709, -0.43412765860557556, 0.6873478293418884, 1.2353651523590088, 0.40690821409225464, 0.5143594145774841, 1.5112898349761963, -0.8855644464492798, -1.3123623132705688, -1.513372540473938, -0.07170810550451279, -2.0201308727264404, -0.09112142026424408, -1.4551724195480347, -2.2417359352111816, -0.768688440322876, -0.19066877663135529, -0.20808593928813934, -0.6970095634460449, 2.3173892498016357, -2.0447092056274414, 1.448923945426941, 1.499932885169983, -0.18911971151828766, -0.21734365820884705, -1.2057054042816162, -0.16070200502872467, -1.1407833099365234, -1.6844263076782227, -0.33139678835868835, 0.49875345826148987, -1.3565603494644165, -0.7362479567527771, 1.5898663997650146, 0.8772609233856201, 0.1527635008096695, -0.621332049369812, -1.2147048711776733, 0.979529082775116, 0.997518002986908, 0.4311043322086334, -2.1225175857543945, -1.5708152055740356, -0.3364308774471283, -0.21762089431285858, -2.8136539459228516, 2.611743450164795, -0.28765803575515747, 1.5450515747070312, -1.0843195915222168, -0.7913192510604858, -1.037967562675476, 2.101957082748413, -0.0359933078289032, -0.7875048518180847, -1.3479695320129395, -0.5837056040763855, -0.6403899192810059, 0.9329144954681396, 1.246324062347412, 0.07403574138879776, 0.4896630644798279, 0.11147850006818771, -0.0991429015994072, -0.1682145893573761, -0.7287795543670654, -0.8847004175186157, -0.7802018523216248, -0.09233377128839493, 1.29428231716156, -2.0942254066467285, -1.8066867589950562, 1.35806143283844, 0.07025511562824249, 1.1111339330673218, -0.24610520899295807, -0.0269990973174572, 0.4872001111507416, -0.30217933654785156, -1.052883267402649, -0.948093831539154, -0.32201263308525085, -0.24985525012016296, -0.6526399850845337, -0.211545929312706, 0.12288619577884674, 1.2199190855026245, 0.03357040882110596, 2.3484487533569336, 0.47524917125701904, 0.9386592507362366, -0.9134680032730103, 1.132901668548584, -0.4199713468551636, -0.13287512958049774, 0.25043779611587524, -0.1875249743461609, 0.2699281871318817, -0.9971299767494202, 1.5510505437850952, 0.16253051161766052, -0.5642470717430115, -1.3644696474075317, -0.5041658878326416], [0.4560246467590332, 0.8037371039390564, 0.4159802794456482, 0.4876338243484497, -0.3050846755504608, 0.5436122417449951, 0.18814557790756226, -0.25050443410873413, -1.0346364974975586, 0.519713819026947, -0.3446311354637146, 0.19608692824840546, 0.9485682845115662, 0.3081382215023041, -1.2391343116760254, 0.3735049068927765, 0.6580925583839417, -1.0461499691009521, -0.767336368560791, -0.2567346394062042, -1.9801533222198486, 0.5531896948814392, -1.825047254562378, -0.5613871812820435, 0.04011350870132446, -1.0923045873641968, 0.962218165397644, -1.2770464420318604, 0.5296229124069214, 0.5823798775672913, -0.4376956522464752, 0.160890132188797, -1.1909412145614624, 0.18698789179325104, 0.7705943584442139, -0.21947205066680908, -0.34835851192474365, -0.33540067076683044, 0.6835051774978638, 1.899875283241272, 0.6951955556869507, -1.2743659019470215, 0.5588324666023254, -0.1253969669342041, -0.2706226110458374, -0.2116299420595169, 1.9403660297393799, 0.7377123832702637, 0.33051106333732605, -2.4680261611938477, 0.17505069077014923, 3.5980887413024902, -0.317899614572525, 1.3889492750167847, -0.04792433977127075, -0.6670160293579102, -0.7339650988578796, -0.34386399388313293, 0.6842964291572571, -0.16180266439914703, -0.7677661776542664, -0.43211349844932556, 0.4721754193305969, 0.11128628998994827, 0.9292129874229431, -2.5765039920806885, 0.42431530356407166, 0.8199819326400757, -1.7455620765686035, 0.22637996077537537, -1.4576034545898438, -0.35667386651039124, 1.5136451721191406, -1.3597439527511597, -0.4769260585308075, -0.2255384624004364, -0.21905842423439026, -0.8764071464538574, -0.7276251316070557, -0.8894269466400146, 0.10327848047018051, -1.2449642419815063, -1.1069434881210327, -0.28129252791404724, -0.2642674148082733, 0.31824663281440735, -0.1299677938222885, 0.5843276977539062, -0.4711443781852722, 0.053151387721300125, 0.2465868443250656, 1.4745148420333862, 0.967793881893158, -0.33868178725242615, 0.2921046316623688, 1.417027473449707, 2.231735944747925, 0.7983196973800659, -0.4748325049877167, -1.4593486785888672, 0.929883599281311, 0.12603609263896942, 0.9916628003120422, -0.7539033889770508, -1.2668445110321045, 2.1570990085601807, 0.5474861860275269, 1.2165879011154175, 1.3886040449142456, -0.7807779908180237, -1.2971056699752808, 2.349832057952881, 0.07121339440345764, -0.4873688817024231, 0.8773956894874573, -0.15427683293819427, 0.869286298751831, -1.3568748235702515, -0.766414225101471, -0.9147903323173523, 0.007242595311254263, 0.33014968037605286, 1.954192042350769, 0.34926798939704895, -0.6950333118438721, -0.3173817992210388, -1.399021029472351, 0.4648496210575104, -2.689361810684204], [0.1184411346912384, -0.2098665088415146, -1.1602345705032349, 1.7002439498901367, 1.3413033485412598, 1.0683094263076782, 0.3593289852142334, -0.3940901756286621, 0.7413591742515564, 1.5791213512420654, -0.22245743870735168, -0.3568740487098694, -0.6669397950172424, 0.4151451587677002, 0.24781066179275513, 1.2993849515914917, -0.24831056594848633, 0.7968095541000366, 0.40723854303359985, 0.35728004574775696, -0.07729050517082214, 0.36398646235466003, -0.010257367976009846, -0.7392521500587463, 0.04020516574382782, -0.4517532289028168, 0.7219600677490234, -2.6093616485595703, -0.5901082754135132, -3.5131301879882812, -1.7778207063674927, 0.3047839403152466, 0.8890816569328308, 0.4759276509284973, 0.6668940186500549, -0.6220253705978394, -0.2893303334712982, 0.5566108822822571, -1.3188636302947998, 0.6538876891136169, 1.249040126800537, 1.2009416818618774, -1.6694371700286865, 1.6153812408447266, -0.48576340079307556, -0.9352617263793945, -1.332429051399231, 2.1899125576019287, 0.5794151425361633, -0.1794273406267166, -0.24673809111118317, 0.4255908727645874, -0.8544000387191772, -0.9258037209510803, -0.7427617907524109, 1.2196438312530518, 0.8935486078262329, 0.9798244833946228, -0.4832558035850525, -2.3726298809051514, 1.2363872528076172, 0.589398980140686, 0.17650280892848969, -2.0734217166900635, -0.51070636510849, 1.2356239557266235, 0.1213877946138382, 1.368921160697937, -2.6001651287078857, 0.11840907484292984, 0.08069948107004166, 0.9770556688308716, 0.524158775806427, 0.9049779176712036, -0.6246598958969116, 1.174476146697998, -0.6121209859848022, 1.5426830053329468, -0.9006900191307068, -0.16270944476127625, 0.3285158574581146, 1.6551748514175415, -0.042609430849552155, -0.07374882698059082, -0.9443787336349487, 0.4728647768497467, -1.4101814031600952, 1.2976489067077637, 1.907512903213501, 0.8206061720848083, -0.32069894671440125, 1.561448335647583, -1.908756136894226, 0.5580630302429199, 2.139981985092163, 0.5956124663352966, 1.2866562604904175, 0.9414873719215393, -0.7596084475517273, 0.31704261898994446, 1.4009841680526733, 0.49604368209838867, 0.1616441160440445, 0.6153326034545898, 0.8061869740486145, -0.2939840853214264, 0.8230425715446472, 0.17866908013820648, 1.421766996383667, 0.8457701802253723, -0.32697582244873047, 0.5625836253166199, 1.3187205791473389, 0.1050732359290123, 0.44574642181396484, -0.06984204053878784, -0.02231488563120365, -0.12223232537508011, 3.8823819160461426, 0.7444287538528442, -0.7518746256828308, 0.2494438737630844, 0.41307470202445984, 0.17939512431621552, -0.36942270398139954, 0.047573402523994446, -1.1549640893936157, 1.5659116506576538, 0.8783767819404602], [0.2549976706504822, 0.5894156694412231, 0.8508865237236023, -1.0426169633865356, 0.6412622928619385, 1.3570127487182617, 1.8732264041900635, -0.029828602448105812, -0.07002081722021103, -0.7427650094032288, -1.267757534980774, -1.5208243131637573, -0.6245612502098083, -0.08008863031864166, 0.36511462926864624, -0.8218296766281128, 0.5128856897354126, 1.4513912200927734, -0.33834487199783325, -1.3353997468948364, 0.3810459077358246, 1.22310471534729, 0.21507366001605988, -0.7479676604270935, -0.2982315719127655, 0.394980788230896, 1.2832081317901611, 1.94101881980896, 0.12723639607429504, -0.41933953762054443, 0.8949232697486877, -0.3942498564720154, -1.6078156232833862, 0.8759599924087524, 0.2966804504394531, 1.3842631578445435, -1.2682287693023682, -0.19316162168979645, -0.4069056808948517, 0.293227881193161, -0.7123501300811768, -0.15780866146087646, -0.09947626292705536, 1.5770641565322876, 0.8475159406661987, 0.3261207640171051, 0.7063154578208923, -1.3853999376296997, -0.12435346841812134, -0.4583224952220917, -0.24471071362495422, 1.0132254362106323, 0.2442770004272461, 1.5501940250396729, -1.1184234619140625, 0.9455010294914246, -1.0471891164779663, -0.18397028744220734, -0.012593329884111881, 0.37601569294929504, -0.05335502326488495, -0.6694012880325317, 1.26968252658844, 0.07989564538002014, -0.5029886960983276, -1.6502536535263062, -1.5782673358917236, 0.5692882537841797, 1.8188804388046265, -1.0322363376617432, 0.467780202627182, 0.4895225465297699, 0.322674959897995, -0.20268335938453674, -0.6469070911407471, -0.9505289793014526, -0.0799255445599556, 1.484934687614441, 0.35028257966041565, -1.0176371335983276, 0.9366435408592224, 0.13028974831104279, -1.5854547023773193, 1.3669313192367554, -0.6790842413902283, 0.971310555934906, 0.5600669980049133, -0.5546725392341614, -1.2738670110702515, 0.1982584148645401, 1.0491405725479126, -0.12158528715372086, 0.5182834267616272, 0.2882945239543915, 0.9441176056861877, 0.8653625845909119, -1.5876352787017822, 0.4098646342754364, 0.08591143041849136, -0.31054094433784485, -0.4065953493118286, 1.1751281023025513, 0.8843955993652344, -1.8628727197647095, 1.097062587738037, 0.7951822280883789, 0.9636629223823547, 0.48839080333709717, 0.06945580989122391, 0.38911113142967224, 0.7508013844490051, -0.6013249754905701, 0.4275994598865509, 0.6983977556228638, -1.8050899505615234, 1.423120379447937, 0.10155913233757019, -1.040292739868164, 0.4253629446029663, 0.11130687594413757, 1.1681008338928223, -0.5405066013336182, 0.262807160615921, 1.8489457368850708, -1.0470856428146362, -1.0006967782974243, 0.9915727972984314, 0.8139358758926392, 0.13263621926307678], [-1.3902300596237183, 0.06777270138263702, 0.011981186456978321, 0.9652225971221924, -1.735080361366272, -1.1467169523239136, -0.6655015349388123, -0.8706530928611755, -0.31192347407341003, -0.9559776782989502, 0.5689857602119446, -0.8956931233406067, -0.3347882032394409, -1.612356185913086, -1.4928659200668335, 0.5537089705467224, -1.7648285627365112, -2.443469285964966, 0.04670170694589615, -0.20495158433914185, -1.2842118740081787, 0.3487260639667511, 0.43488574028015137, 0.49489128589630127, -1.7306022644042969, 0.9104779958724976, 0.5935184955596924, 0.9782549738883972, 0.7286401391029358, -1.3676444292068481, 0.9000117182731628, -0.5451129078865051, -0.04386742040514946, -0.08831393718719482, 1.4810727834701538, -0.08504817634820938, 1.5018706321716309, -1.2519986629486084, -0.7747951745986938, 0.7800257205963135, 0.21726983785629272, -0.9958431124687195, -0.27416470646858215, -0.15018808841705322, -1.817332148551941, 0.29750338196754456, 0.5414911508560181, -1.7603416442871094, 2.88071870803833, -0.4110974967479706, -0.029926864430308342, -0.37173697352409363, 0.3849460780620575, 0.44033199548721313, -1.267491102218628, 1.3955214023590088, 0.8984637260437012, -0.07532623410224915, -0.7876846194267273, 1.1742078065872192, 0.6359046101570129, 0.5664098262786865, 0.7564716935157776, -0.6404662728309631, 0.3636959195137024, 1.5532900094985962, -0.8107843399047852, 0.8898198008537292, 1.3981437683105469, -0.07900362461805344, -1.0324116945266724, -0.7717924118041992, -1.117849349975586, 0.9528340101242065, -0.8000874519348145, -1.811069369316101, -2.0589394569396973, 0.8312755227088928, -0.8379934430122375, -0.36188945174217224, -0.038344480097293854, -0.565334141254425, -0.024754134938120842, -0.19571925699710846, 0.1250244528055191, 0.21530258655548096, 0.9414700269699097, -0.7896000146865845, -0.699837863445282, -0.4974362254142761, -1.444467306137085, 0.16695109009742737, -1.013159990310669, -2.958047866821289, 1.2426323890686035, 0.25545963644981384, -0.3134143352508545, 1.735709309577942, 1.2297698259353638, -1.3154208660125732, 0.35181111097335815, -0.11752442270517349, 1.8706810474395752, 0.1266593635082245, -0.0859619677066803, 0.8049495816230774, 0.84002286195755, -0.42432135343551636, -0.3643827438354492, -0.5929282307624817, 0.09487591683864594, 1.3532750606536865, 0.12008261680603027, -1.1036330461502075, -1.459100604057312, 1.4412118196487427, 0.8466435074806213, -0.6314159631729126, 0.12802763283252716, -0.6439220309257507, -0.35152941942214966, 0.6002408862113953, -0.06474611163139343, 0.731638491153717, -1.4404677152633667, 0.32431089878082275, 1.7720950841903687, -1.3210598230361938, 0.411905974149704], [-0.6967405676841736, 0.8962940573692322, -1.476496696472168, -0.5456655025482178, 0.02772693708539009, -0.06230204179883003, -0.8157554268836975, -0.391418993473053, -0.045865580439567566, -1.381781816482544, 0.1914381980895996, 1.4501742124557495, 1.2486368417739868, -2.1252613067626953, 1.0336391925811768, -0.3588789403438568, 0.23394101858139038, 1.4490514993667603, 1.0487533807754517, -1.3266825675964355, -1.887665867805481, 0.09385614842176437, 1.5746275186538696, 0.41312363743782043, -0.5887835025787354, 0.5099378228187561, -1.9645646810531616, 0.3438933193683624, 1.426198959350586, -0.7601165175437927, -0.04820014163851738, 1.9478777647018433, 0.9423381090164185, 2.1057016849517822, 1.7158600091934204, 0.01981077529489994, -0.4032595157623291, -0.7068572044372559, 0.2615030109882355, 0.1623273342847824, 0.49993351101875305, -2.516122341156006, 0.5177262425422668, -1.5472941398620605, -0.5014045834541321, 0.053324583917856216, -0.6715473532676697, 0.8230935335159302, 0.014122530817985535, 0.4479118585586548, 1.4114341735839844, 0.8757059574127197, 0.9395026564598083, -0.969691276550293, 0.8873912692070007, -0.5344688296318054, -1.0253173112869263, 0.8595142960548401, 1.3376017808914185, -0.6276251077651978, 1.2358684539794922, 0.26128482818603516, 1.0949921607971191, -2.2220311164855957, -0.04963882267475128, 2.000365734100342, 0.39157915115356445, -0.841947615146637, -1.2267769575119019, -1.3501453399658203, 0.19522786140441895, 0.2779916226863861, -0.9939514398574829, 0.5066477656364441, 0.44235146045684814, -1.4024028778076172, 0.6692890524864197, -0.5241488218307495, 0.3088377118110657, 0.19751328229904175, -2.3248162269592285, 1.5506616830825806, -0.5086758732795715, -0.30144789814949036, -0.11469845473766327, -0.3089877963066101, -0.07581152021884918, -0.36683186888694763, -0.2942136228084564, 1.3722652196884155, -0.12489969283342361, -0.22912320494651794, 0.26708081364631653, 1.6870418787002563, 1.6676982641220093, -1.2135623693466187, 2.0111899375915527, 0.7582579255104065, 0.3687591254711151, -0.18022345006465912, -0.9372879862785339, -2.7697200775146484, -0.04088514670729637, -1.1925511360168457, -1.670844554901123, -0.5754674077033997, -1.040605902671814, 2.026663303375244, 1.526151418685913, -0.2551932632923126, 0.7475329041481018, -2.7039437294006348, 0.11804395169019699, -0.48547056317329407, 1.5356113910675049, -0.4667811393737793, 0.840377688407898, -1.989179015159607, 0.5765647292137146, 0.527934730052948, 0.2627289891242981, 0.7859322428703308, 0.7786713242530823, -2.116586685180664, 0.5776376724243164, 1.3865158557891846, -1.5547219514846802, -1.6185903549194336, -0.8581802248954773], [0.43634650111198425, -1.5971099138259888, 0.4312073290348053, -1.0953845977783203, 0.697910487651825, 0.12179546803236008, 0.6268453598022461, -0.5242606997489929, 1.089682936668396, 0.6494948267936707, 1.0786654949188232, 0.7027983069419861, 0.3229396641254425, 1.3424603939056396, 0.21434983611106873, -1.5266213417053223, 2.4810879230499268, 0.7906723022460938, 0.6764108538627625, 0.5300446152687073, 0.26872310042381287, -1.7757587432861328, 0.7866472601890564, -2.0868752002716064, -0.49638500809669495, -1.008773684501648, 0.15609654784202576, 1.7212258577346802, -0.3737046718597412, -0.028676362708210945, 1.463148832321167, 0.12729033827781677, 0.18879638612270355, -0.7978670001029968, 1.3499088287353516, -0.9918221831321716, 0.4861917495727539, 1.1088141202926636, 0.9484824538230896, 0.056257154792547226, -1.4686903953552246, 0.12205654382705688, -0.19351071119308472, 1.6836568117141724, -0.8187925219535828, -0.8312331438064575, -0.5892162322998047, -1.4166468381881714, 1.0012474060058594, 0.8444949388504028, -0.9453615546226501, -0.7224071025848389, -0.7744907736778259, 1.256961464881897, 1.2482967376708984, -0.6321476101875305, 1.024303674697876, 0.2556590735912323, 0.8210952877998352, -0.6098308563232422, -0.2557358741760254, 0.5511848330497742, 0.3278137445449829, 0.8826776742935181, 0.8803407549858093, 1.4750200510025024, -2.431180238723755, 0.6572400331497192, -2.077610492706299, 0.9050962328910828, -0.8634989261627197, -0.18101787567138672, -1.8183890581130981, -1.016559362411499, 0.49681735038757324, 0.7564117312431335, 0.02117445319890976, 0.1414737105369568, 0.17355898022651672, -0.14687061309814453, -2.5960264205932617, 0.33584925532341003, -1.5286391973495483, 0.20121726393699646, -1.6725435256958008, 0.9190541505813599, 1.7260677814483643, 0.35337406396865845, -0.14413751661777496, -1.13343346118927, -0.5060622096061707, -1.2278393507003784, 0.5498548150062561, 1.7306653261184692, 0.9013009667396545, 1.5625531673431396, -0.06451322883367538, -0.581816554069519, -1.3443926572799683, 1.1587952375411987, 0.2801915109157562, 0.9933237433433533, -0.37112653255462646, -0.45221295952796936, 0.9527197480201721, -0.9362223148345947, 0.7668352723121643, 2.2567615509033203, -1.6156612634658813, -1.236031174659729, 0.22325006127357483, 0.7795529365539551, 0.9443275928497314, -0.5841273069381714, -2.022141456604004, -1.0226274728775024, -0.5727053880691528, 0.8237428069114685, -1.6065385341644287, -0.40731850266456604, 0.44464346766471863, 0.3747027516365051, 0.040871601551771164, -0.8359214663505554, -0.38980185985565186, -0.39783409237861633, -0.1420689970254898, 1.2817482948303223, 0.38027939200401306], [-0.4562765061855316, 0.7576059103012085, -0.1947847157716751, -1.1887727975845337, -0.2725558578968048, 0.15245380997657776, -1.2806451320648193, -0.29109659790992737, 0.2920691668987274, -0.278159499168396, -0.008702786639332771, 0.1614075005054474, 0.09232732653617859, -0.4812917411327362, 0.7334539294242859, 2.416623592376709, 0.5294589400291443, 0.051464106887578964, -0.46809571981430054, -1.0431674718856812, 0.3096561133861542, 0.7511572241783142, 1.673539638519287, -0.24679924547672272, -0.3064482808113098, 0.9318190217018127, -0.7567102909088135, -0.22210688889026642, 1.553382158279419, 0.8652137517929077, 0.9826975464820862, -0.15133458375930786, 1.5777835845947266, 1.2995569705963135, -1.3294122219085693, 0.0940052941441536, -2.6122541427612305, -0.7827861309051514, -1.0979262590408325, 0.7211530208587646, 0.13263456523418427, -0.6520498394966125, 0.07009503245353699, -1.6392159461975098, 0.7824039459228516, -0.5714749097824097, 0.8211462497711182, -0.38590458035469055, 0.3096095025539398, 1.2249946594238281, -1.0935711860656738, -1.9452532529830933, -1.002589464187622, -0.513944685459137, 0.12181510031223297, -0.015696410089731216, 1.953951120376587, 0.4798060953617096, 0.513617992401123, -1.9817769527435303, 0.7066200971603394, 0.3491877317428589, -0.953583836555481, 1.3765168190002441, -0.37872618436813354, 0.47651320695877075, 0.30573680996894836, 0.6601710319519043, 1.0804307460784912, 2.092935800552368, 0.805720329284668, 0.8269765377044678, -1.8831318616867065, -0.05848889797925949, -0.23931388556957245, 1.4535614252090454, -0.6013205647468567, 0.08005792647600174, 1.7932735681533813, 0.33696532249450684, -0.4111844301223755, 0.7908623814582825, -0.5612262487411499, 0.8903190493583679, -0.5447764992713928, -1.5684728622436523, 0.29119443893432617, -0.44083279371261597, -2.01513934135437, 0.4094774127006531, 1.0093706846237183, -0.0869559571146965, 0.16109736263751984, 0.29742753505706787, -0.20943300426006317, 1.0748285055160522, -0.1344311535358429, 1.0374400615692139, 0.283761203289032, -0.9398715496063232, -0.8771815896034241, -0.037930216640233994, -0.16426680982112885, 0.5302025079727173, 0.40275001525878906, -0.39876601099967957, 0.023571213707327843, -1.8153173923492432, -0.38277649879455566, 0.6677815914154053, 0.28434574604034424, -0.750446617603302, 0.4182339012622833, -0.8307071328163147, -1.8486696481704712, 1.0774136781692505, -0.45561525225639343, 1.3075264692306519, -1.0802487134933472, -0.40364140272140503, -0.6218187808990479, -2.605531692504883, 0.5937426090240479, 0.08845513314008713, -0.28932663798332214, 1.2018873691558838, -0.02872159332036972, 0.8061671257019043, 1.1665884256362915], [0.3736574947834015, -1.595044493675232, -0.8328023552894592, -0.8883401155471802, 0.8704268932342529, 0.415805459022522, 0.06684914976358414, -1.2835280895233154, -0.5656302571296692, 0.9841065406799316, 0.5498064756393433, -0.0825854018330574, -0.16786180436611176, -1.3990293741226196, 1.6664323806762695, -1.1178741455078125, 0.11181952804327011, -0.24650509655475616, -1.0894010066986084, -0.862280011177063, 0.8802452683448792, 0.41254493594169617, 0.8126262426376343, 0.5049600601196289, -1.2187973260879517, -0.10567590594291687, -0.3441035747528076, 0.4617792069911957, -0.08547413349151611, 1.1323026418685913, 1.458205223083496, -0.9742363691329956, 0.38519591093063354, 0.6435551047325134, 1.2350789308547974, 0.9725733399391174, -1.0456671714782715, 1.8771339654922485, -0.05349693074822426, -0.41509342193603516, -1.1595444679260254, -1.4664503335952759, 0.2736690044403076, 1.0929700136184692, -1.1044096946716309, -0.46568647027015686, -1.1047624349594116, -0.8850960731506348, -0.24149152636528015, -1.189244270324707, -0.7828841209411621, 0.9459741115570068, 0.8268529772758484, -0.35304948687553406, -0.717684268951416, -2.0618348121643066, -0.17915017902851105, 2.62044095993042, -0.04935126379132271, -0.33085793256759644, 1.4755735397338867, 0.45203283429145813, 1.1436392068862915, -0.10836393386125565, 1.530767798423767, -1.145182490348816, -0.9104273319244385, -1.4287922382354736, -0.8174005746841431, 1.6170204877853394, 0.5270518064498901, 0.050607144832611084, -0.41401177644729614, 1.0847585201263428, -0.009814884513616562, -1.1105554103851318, 1.173701286315918, 2.511991024017334, -0.20197118818759918, 1.5105375051498413, -0.08107150346040726, -0.8339879512786865, 1.7995178699493408, 1.1277302503585815, 1.2257133722305298, -0.05259823426604271, -1.2178499698638916, 0.6834728121757507, 0.5399446487426758, -0.8530879616737366, -0.24126316606998444, -1.4014091491699219, 0.12823592126369476, 1.6661731004714966, -0.7334039211273193, 1.2328987121582031, -0.8063271045684814, -0.769426167011261, -0.5487818717956543, 1.4564393758773804, 0.3536425232887268, 0.2643420398235321, -0.5745338797569275, 0.8277385830879211, 0.5593955516815186, 1.2976614236831665, 0.314484179019928, -2.7269105911254883, 0.3466733396053314, -0.03659853711724281, 2.029404878616333, 0.4622216522693634, -0.3021909296512604, -0.8595300316810608, -0.14067237079143524, 0.2514733076095581, 1.0901384353637695, 0.2690332531929016, 0.7731490731239319, -2.519693374633789, 0.25454840064048767, -0.6219654083251953, -0.5138781070709229, -1.0636637210845947, -0.13224858045578003, -1.5634456872940063, 1.8041462898254395, -0.9163526892662048, 0.08894630521535873], [-0.8337328433990479, 0.5386926531791687, 1.056774616241455, 0.7723770141601562, -0.7157137393951416, 0.2122005820274353, 1.2354159355163574, -1.2117445468902588, 1.0587211847305298, 0.5003780722618103, 0.44971081614494324, -1.1372803449630737, -0.19021086394786835, -1.5840814113616943, 1.3584388494491577, 0.21113409101963043, -0.02013644576072693, -0.8063521385192871, 0.18858923017978668, -1.7457835674285889, -0.34155672788619995, 0.10311122238636017, 0.3534638583660126, 0.6604621410369873, 0.5747901797294617, -1.3514900207519531, 0.9087001085281372, 0.29024413228034973, 0.07714978605508804, -0.12739497423171997, 1.3671406507492065, 0.13848522305488586, 1.5903040170669556, -1.3511484861373901, -0.9577659964561462, -0.2943529486656189, -0.5816992521286011, 0.669244110584259, 0.30530014634132385, 0.9741339683532715, 1.1824649572372437, 0.7734258770942688, -0.7099445462226868, -0.8164284825325012, -0.5870298743247986, 0.6973175406455994, -0.3548963963985443, -0.8669620752334595, 2.201939582824707, 1.4122389554977417, -0.16012291610240936, 1.4853991270065308, -0.9591091275215149, 1.0076948404312134, -1.2078694105148315, 0.47486087679862976, 0.03705820441246033, -0.46689924597740173, 0.567129909992218, 0.39288368821144104, -0.4673783481121063, 0.7758203148841858, 0.929667592048645, -0.31712251901626587, 1.813740611076355, 0.49084413051605225, 1.9708231687545776, -0.49069666862487793, 0.37143638730049133, 0.9125798344612122, -0.700066328048706, -1.3775478601455688, 0.652825117111206, 0.7189234495162964, 0.9073200821876526, 0.6649233102798462, 0.27154988050460815, 0.024950366467237473, -2.260317325592041, -0.34712862968444824, -0.03601786121726036, -0.26803165674209595, 0.7091797590255737, -0.33051398396492004, 0.2549183666706085, 0.11248613893985748, -0.6178531646728516, 0.27024203538894653, 1.2694612741470337, -0.0665162205696106, 0.8255230784416199, -0.15637168288230896, -2.104240894317627, -0.22368639707565308, 0.006928931921720505, 0.46967583894729614, -0.9624319076538086, -0.022812215611338615, -0.170864999294281, -0.46542784571647644, -1.8132002353668213, 0.7699870467185974, 0.13582415878772736, 1.274431824684143, -0.46018850803375244, -2.1183412075042725, 0.0394328348338604, 2.1211493015289307, 0.5785706639289856, -0.8286261558532715, -0.27878478169441223, 1.2156578302383423, 1.8543825149536133, -0.9305368661880493, 0.03816800191998482, 1.470816731452942, -0.3342309296131134, 1.2128311395645142, 1.3556004762649536, -0.11411834508180618, -1.1824287176132202, -0.41290804743766785, -0.46004587411880493, 0.7770097255706787, -1.1120041608810425, -2.006589889526367, -0.3035427927970886, 1.964979648590088, 1.040279746055603], [-0.07672913372516632, 0.5942773818969727, -1.7344022989273071, -0.9778845906257629, 0.912764847278595, -0.4029487073421478, 1.043870210647583, 0.23177896440029144, -0.03222080320119858, 1.3430827856063843, -2.674628973007202, 0.49188750982284546, -1.7277984619140625, -0.009575864300131798, -0.33439722657203674, 1.148747205734253, 0.30149999260902405, 0.7534892559051514, -1.361890435218811, 0.7393835783004761, 0.20558176934719086, 1.2125974893569946, 0.6782251596450806, 1.4972164630889893, 0.6141319274902344, 0.3313116729259491, 0.4125608801841736, 0.20440638065338135, -0.3110330402851105, -0.9918054938316345, -0.17880791425704956, -1.4451481103897095, -1.0166224241256714, 0.12884871661663055, 0.8294795751571655, 0.8073201179504395, 0.6927583813667297, -0.589693009853363, 0.06798131763935089, -0.7236086130142212, 0.05773641914129257, 0.6644112467765808, -0.9735597968101501, -1.5464071035385132, 0.9290830492973328, 0.25235313177108765, 1.6251822710037231, 1.0661886930465698, 1.5947027206420898, 1.0738112926483154, 1.914605736732483, -0.260322630405426, 1.1814206838607788, -0.20254041254520416, -1.3277441263198853, -1.7104226350784302, 2.313551187515259, 0.14338044822216034, -0.4433465003967285, -0.010659720748662949, 0.9610663056373596, 1.2939428091049194, 0.3979298174381256, -1.5241494178771973, 1.7061291933059692, -2.246906280517578, 0.7260633111000061, -0.7855889797210693, 0.9979746341705322, -1.6147626638412476, -1.6548298597335815, 0.270774781703949, -0.40498030185699463, 0.1607656478881836, 1.3310091495513916, -0.682914137840271, -0.29856961965560913, 1.339021921157837, 1.1627289056777954, -0.5305768251419067, -0.3433525860309601, -0.6091397404670715, -0.7484747171401978, -1.2072629928588867, -0.41539549827575684, 0.2672869861125946, -0.040852900594472885, 0.5207584500312805, -0.19771891832351685, -1.1264114379882812, -1.4813346862792969, -1.3883981704711914, 0.4957068860530853, -2.59930682182312, 2.210343599319458, -1.6237757205963135, 0.022070473060011864, 0.14100511372089386, 1.5240600109100342, -0.28635480999946594, 0.048910290002822876, -0.31940463185310364, 0.8393459320068359, -0.6861556768417358, -1.1922849416732788, 0.3018950819969177, 1.0302001237869263, 0.3974556624889374, -0.36692383885383606, 1.2810406684875488, 1.442749261856079, 1.4493547677993774, 1.4498673677444458, -1.3662588596343994, 1.0517641305923462, 0.37199464440345764, -0.001880247495137155, -0.9923452734947205, 2.2631278038024902, -2.0743322372436523, -0.19124624133110046, -0.5765690803527832, -0.7594897747039795, 0.9184901118278503, -0.8598341941833496, -1.5525717735290527, -0.20102979242801666, -0.5354836583137512, -1.0783815383911133], [1.3725838661193848, 0.6530920267105103, 0.3523254692554474, 0.33307158946990967, -0.40121883153915405, -0.3365894854068756, 0.9137628674507141, 1.076627254486084, -1.4158706665039062, 1.0645508766174316, 0.5638284087181091, -0.6543381214141846, -0.00421015452593565, 0.40728017687797546, -1.0687679052352905, -0.18705353140830994, -1.2060414552688599, -0.22525326907634735, -0.25245401263237, 1.9231085777282715, -0.6465309858322144, 1.3771839141845703, -1.7220537662506104, 0.06570783257484436, 0.22894318401813507, 0.7905066609382629, 0.6787347197532654, 1.1403390169143677, 1.4213025569915771, 1.152182698249817, -1.7615160942077637, -0.4926508367061615, -0.4782075881958008, -0.7140145897865295, 0.7314043045043945, -0.5324926972389221, -0.6516577005386353, 0.5496492385864258, -1.3830772638320923, 0.12588052451610565, -0.44379881024360657, -1.085391879081726, -1.9018211364746094, 0.7352167367935181, 0.6086574196815491, -0.6852850914001465, -0.7783520817756653, -1.1546132564544678, -1.7466789484024048, 0.9416444301605225, -1.8719199895858765, 0.1795990914106369, -0.6546130776405334, 1.0541918277740479, -1.023113489151001, 0.6858783960342407, 0.6456394195556641, -0.7796962261199951, 1.9122555255889893, -0.4415469765663147, -1.2786836624145508, 0.7601119875907898, 0.012863396666944027, 0.4320422112941742, 1.6697014570236206, -0.5617649555206299, 1.5653815269470215, -0.8769564032554626, -0.4029041826725006, -1.3430808782577515, -1.176912546157837, -0.6114618182182312, 0.3508947789669037, -0.6484101414680481, -0.4231967329978943, 0.7039632797241211, 0.36240288615226746, -1.0051816701889038, 0.04885386303067207, 0.025161979719996452, -1.550703763961792, -0.8915213346481323, 0.16918706893920898, -1.6872525215148926, -0.2723325490951538, -1.2246628999710083, -0.09860453009605408, 0.23011590540409088, 0.08103521913290024, -0.7201865911483765, -0.27372244000434875, -0.3918251395225525, 0.6868831515312195, 0.02023431472480297, 1.350500464439392, 1.6961301565170288, 0.7977486252784729, 0.9357860684394836, -0.40917402505874634, 0.9803941249847412, -1.3007020950317383, -1.2093442678451538, -0.5626993179321289, -1.955509901046753, -1.054414987564087, 1.2776689529418945, 0.11698813736438751, 0.5471364855766296, -2.149196147918701, 0.07956213504076004, -0.6066550612449646, -1.225834608078003, -1.011128306388855, -1.8910205364227295, 0.7970304489135742, -0.6815446615219116, -1.1126527786254883, -0.3261359930038452, 1.639960527420044, -1.310322880744934, 0.2568618655204773, -0.4425541162490845, -1.0616276264190674, -0.1448918730020523, 1.4677504301071167, -0.5290592312812805, 0.799606442451477, 0.31242361664772034, -1.2500498294830322], [0.7088081240653992, 0.03033461421728134, -1.678476095199585, -0.29304543137550354, 0.8878034353256226, 1.1289067268371582, -0.8180450201034546, -1.0812195539474487, -1.8782888650894165, 2.0925025939941406, 1.7736822366714478, -0.5520609617233276, -0.15614554286003113, 1.2067183256149292, 0.18584194779396057, 0.7998303771018982, 0.0034874158445745707, 0.46542468667030334, 0.8898441791534424, 1.5273765325546265, 2.284350633621216, -0.1474856287240982, -0.8159394860267639, 0.33737391233444214, 0.03357146307826042, 1.2134991884231567, 1.3795701265335083, 0.3362559378147125, 0.4409349858760834, -0.6179161071777344, 1.0637929439544678, -1.041248083114624, -1.3564300537109375, -0.0798272117972374, -1.12788724899292, 1.0101231336593628, -0.6807755827903748, -0.48185837268829346, -0.3087189495563507, 0.2810269296169281, -1.4005202054977417, 0.6644355654716492, -1.5646361112594604, 1.2312067747116089, -0.46975278854370117, 0.7070720195770264, 0.13189102709293365, -1.5339806079864502, 0.928009033203125, 0.3476925790309906, -0.5603618621826172, -0.9224943518638611, 0.7408308982849121, 0.5525695085525513, 0.8870139122009277, -1.5500775575637817, 0.4811023771762848, 0.4395495653152466, 0.12914180755615234, -1.7575831413269043, 0.31392693519592285, -1.213179349899292, 0.0318838395178318, 1.0610836744308472, -0.8507699370384216, 0.18152117729187012, -0.011158996261656284, 0.026361260563135147, 0.7684547305107117, 0.6367378830909729, -0.3135667145252228, -0.8431367874145508, -0.6056586503982544, 1.056419849395752, -0.01906680129468441, 0.833435595035553, 0.8371724486351013, 0.7358568906784058, -0.6019536256790161, -1.5821197032928467, -1.6257878541946411, 1.2839248180389404, -0.6014710068702698, -0.015623088926076889, -2.2742600440979004, -0.21391567587852478, -1.6518219709396362, -1.5200098752975464, -0.724834144115448, 1.3879687786102295, 0.7344167232513428, 0.20036354660987854, 0.23679181933403015, 0.4650464951992035, -1.4001349210739136, -1.0120811462402344, -0.7860073447227478, -1.3318055868148804, -0.04878666624426842, -0.16907714307308197, 0.476608008146286, 0.009586821310222149, -0.6493340134620667, 0.3733956813812256, -1.4526891708374023, 0.3864286243915558, 1.4787094593048096, 0.6035712361335754, -0.31259357929229736, -0.9128498435020447, 0.5809990167617798, -0.14431093633174896, 0.07286564260721207, -0.08398930728435516, 1.2423137426376343, 0.27033719420433044, 0.6941158175468445, -1.4728161096572876, -0.7276508212089539, -0.1646994799375534, 0.2413914054632187, -1.6002898216247559, -0.8023269772529602, 0.23156948387622833, 0.46367302536964417, -0.15248531103134155, -1.2068517208099365, -1.1315994262695312, -1.254298448562622], [0.4606015980243683, -1.068885087966919, -0.38820675015449524, 0.7072682976722717, -0.7808671593666077, 1.5491654872894287, 1.0211020708084106, -1.0677614212036133, 1.7084101438522339, -1.8715734481811523, 1.2871246337890625, -1.1264232397079468, 0.3874734938144684, 1.7415930032730103, 0.9898667335510254, -0.2140171378850937, -0.3287546932697296, 0.7191950678825378, 0.7888763546943665, -0.44056543707847595, 0.3553983271121979, 1.2341344356536865, -0.8271122574806213, 0.6104399561882019, -0.395939439535141, -0.7933480739593506, -0.494865745306015, -0.4267917573451996, -0.8247467875480652, 0.3945086896419525, 0.3211664855480194, -0.6942529678344727, 0.827049195766449, 1.9656201601028442, -0.7582921981811523, -1.7416863441467285, 0.18544083833694458, 0.571042537689209, 0.185294046998024, -0.5858866572380066, 1.727262020111084, 1.735520362854004, 0.020057551562786102, 1.6079350709915161, 0.6722680926322937, 1.07760751247406, -0.847488284111023, -0.15011420845985413, -0.0859985277056694, -1.4071543216705322, 0.5258224010467529, -0.8428233861923218, -0.44880211353302, 0.04657309502363205, 0.6727096438407898, 1.479296326637268, 1.1331230401992798, -0.03766082972288132, 0.5264896750450134, 0.9171062707901001, -0.578020453453064, -0.5319163203239441, -1.8026959896087646, 0.6413505673408508, 0.4277026355266571, 0.3121911883354187, -0.3181535601615906, 0.9123411178588867, 0.23925207555294037, 0.9205893874168396, 0.6817187070846558, -1.193514108657837, -0.30208516120910645, -1.2198677062988281, -1.3181369304656982, 0.3651416599750519, -0.7028225660324097, -0.44912925362586975, 1.98703134059906, 0.13217376172542572, 0.9603898525238037, 1.5022062063217163, -0.10358095169067383, 0.34278544783592224, -0.8476818799972534, -0.45553070306777954, 0.3299340307712555, -0.34346187114715576, 1.9797123670578003, -0.5925573706626892, 0.08495063334703445, 1.2124959230422974, -0.7700057625770569, 0.1989966779947281, -0.49754494428634644, -1.3393478393554688, 1.2139240503311157, 1.3894387483596802, 1.3026988506317139, -0.8456342816352844, 1.2307746410369873, -0.1294964700937271, -0.6191513538360596, 1.4730991125106812, 1.6390860080718994, 1.1237335205078125, 0.7092322707176208, -1.4032341241836548, 0.8357028365135193, -0.31701186299324036, -0.259254515171051, 0.015542690642178059, 0.33147120475769043, 0.5689206719398499, 1.9321424961090088, -0.9656263589859009, -2.0951733589172363, -0.27408337593078613, -2.0386688709259033, -0.4514167606830597, -0.8068311810493469, 1.963074803352356, -1.056602120399475, -0.3755475580692291, 0.2991526126861572, -2.5851500034332275, -0.14943166077136993, -0.9661272764205933, -1.8187121152877808], [0.6986316442489624, -1.1151485443115234, -0.23610591888427734, -0.3838751018047333, -0.6345210075378418, 0.7017276883125305, 1.318299651145935, -1.845917820930481, 0.25863364338874817, -0.03536689281463623, 0.6986822485923767, 0.07207758724689484, 0.6066545248031616, 1.9708683490753174, -2.595590353012085, 1.1133962869644165, 0.05998850241303444, 1.0351148843765259, 0.14376042783260345, 0.6956772208213806, -0.3921743929386139, 0.3644130527973175, -1.051461100578308, -0.8271715044975281, -0.9449158906936646, -1.2721790075302124, 0.02326205000281334, 1.604231834411621, -0.6508315205574036, -1.3761298656463623, -0.3643363118171692, 0.3001485764980316, -0.04290173575282097, 0.2833387851715088, 0.01475108414888382, 0.050714340060949326, -1.639357566833496, 1.2837467193603516, 0.9180788993835449, -0.08482645452022552, 1.0458990335464478, -0.08359812945127487, -0.0995587632060051, -0.8257045149803162, -2.2611143589019775, -1.804002285003662, 1.256584644317627, -0.22183012962341309, 0.00880665984004736, 1.5489782094955444, -0.2535431385040283, -1.6638141870498657, -1.4371769428253174, -1.6455349922180176, 0.344694584608078, 0.34650200605392456, 0.8544943928718567, -1.8681442737579346, -1.2948366403579712, 2.179966926574707, -1.1368913650512695, -0.06453513354063034, 1.752940058708191, 0.27746498584747314, -1.7168407440185547, 0.4874040484428406, 0.5817130208015442, 0.2203604280948639, 0.10807353258132935, -0.44953447580337524, 0.8459218144416809, 1.3729963302612305, -0.13207431137561798, -0.4800053536891937, 1.2224558591842651, -0.4608236849308014, 1.455579161643982, 0.8815544843673706, -0.6933215260505676, 0.33231109380722046, -0.5267361402511597, 0.8689242601394653, -1.6376575231552124, -1.2903026342391968, 0.7962223291397095, 0.7364932298660278, 0.3614923357963562, 0.5284162759780884, -0.3199554681777954, -0.37239933013916016, 0.29850348830223083, -1.2982350587844849, 0.6733848452568054, -0.780931830406189, -0.13659906387329102, 2.0209360122680664, -1.4447333812713623, 0.7957912683486938, -0.9507646560668945, 1.8305420875549316, 0.301139771938324, -1.2429291009902954, -0.554604172706604, 0.7902919054031372, 0.5919004678726196, -0.176706001162529, -1.0164120197296143, 1.3719141483306885, 0.2528843283653259, -1.5975104570388794, 0.2882806658744812, -0.9677447080612183, 0.800349235534668, 1.5037606954574585, -1.5306955575942993, 0.5935413241386414, 0.25854983925819397, 0.21525123715400696, 0.129103884100914, -2.1784005165100098, 0.49278688430786133, 0.1794138252735138, 0.1787789762020111, -0.9526321291923523, 2.596961736679077, -0.49233105778694153, 1.277774691581726, 1.1708561182022095, 2.1338260173797607], [0.6482424736022949, 0.30512675642967224, -0.2829464077949524, 0.6282762289047241, 1.4959524869918823, -1.4080477952957153, -0.11799772083759308, 0.40296879410743713, 0.9407287836074829, 0.016841229051351547, -0.26509004831314087, -2.3022100925445557, -1.0920405387878418, -0.35895490646362305, -1.5823190212249756, 0.08537514507770538, 1.4279136657714844, 0.5837450623512268, -1.2783981561660767, -0.5471495389938354, -1.6595089435577393, -2.4374091625213623, 1.3590667247772217, -0.9460648894309998, -0.6386508941650391, -0.11660885810852051, 0.2618348300457001, -1.3690515756607056, 0.9344068765640259, -0.4014398753643036, -0.04874594137072563, 1.070302128791809, -0.5557400584220886, 0.3449516296386719, -0.5920065641403198, 0.49028661847114563, 1.164934515953064, -0.5983556509017944, -1.156772255897522, 0.386057585477829, 0.5747954845428467, 0.3769873082637787, 0.14245730638504028, -0.9362327456474304, 1.311848521232605, 0.25851356983184814, 0.042127083986997604, 0.006689152680337429, -0.8681451678276062, -0.05292427912354469, -0.9602092504501343, -0.2510524392127991, -0.46283087134361267, 0.6609460711479187, 0.9503871202468872, 1.8305540084838867, -1.3018649816513062, 1.6377673149108887, 0.8058770895004272, -0.7945023775100708, 0.7133194208145142, -1.4629464149475098, 0.9039969444274902, 0.37151721119880676, -1.2280393838882446, 0.7646690607070923, 1.8175281286239624, 0.46307405829429626, 0.263639897108078, 0.881567656993866, 1.9322134256362915, 0.11925376206636429, -1.1621919870376587, 1.1079261302947998, -0.606280505657196, 1.489804744720459, 1.5950671434402466, 0.6213380694389343, -1.601151943206787, -0.12360583990812302, -0.7544094920158386, 1.1001678705215454, -0.5349552035331726, 0.04745316505432129, -1.4153720140457153, 1.443967342376709, 0.9439847469329834, -0.41127586364746094, 1.0650156736373901, 0.21102745831012726, 0.37493252754211426, 0.0531500019133091, 1.845605492591858, 0.810165286064148, -0.1129973977804184, 1.0286905765533447, 0.4960000514984131, -0.4673134982585907, -1.1270948648452759, -1.9327304363250732, 1.2046540975570679, -0.5729491114616394, -0.04383319616317749, 0.5346111059188843, -1.527396559715271, 0.6848875284194946, 0.31958556175231934, 0.5603563785552979, 1.4822838306427002, -0.5127132534980774, -0.03609543293714523, 0.7779636383056641, -1.2574994564056396, -1.2136434316635132, 0.67857825756073, 0.31304290890693665, -0.6086776852607727, 0.38606953620910645, -0.026672733947634697, 0.837295413017273, 0.9572750329971313, 2.325730562210083, 0.42517709732055664, -0.42926329374313354, 1.9408490657806396, 0.7469056844711304, 1.6022918224334717, 0.18419203162193298, 0.16167135536670685], [-0.08460376411676407, 1.1310096979141235, 1.2381826639175415, -1.2371783256530762, -1.7699296474456787, -0.5502692461013794, 0.29040518403053284, -0.7934417128562927, -0.8735887408256531, 1.3256515264511108, -0.05668099597096443, -1.1088712215423584, 0.5803515911102295, 1.4952884912490845, 1.6096951961517334, 3.451219081878662, -2.7118959426879883, 1.4522019624710083, 0.21646682918071747, 0.9282006025314331, 0.4421616494655609, -0.04257815331220627, -1.0522634983062744, -0.724490225315094, 1.1179465055465698, -0.6618105173110962, 1.3601716756820679, -0.7529874444007874, 0.06831413507461548, 0.28521159291267395, -1.344983458518982, -0.7223063111305237, 1.087681770324707, 1.3423762321472168, -1.4573702812194824, 0.6113006472587585, 1.3668724298477173, -0.6455342769622803, -1.6419967412948608, 0.0976347029209137, -0.18475736677646637, -0.39258870482444763, 1.0576741695404053, -0.1099293902516365, -0.8392784595489502, -0.986440122127533, 0.017518529668450356, -0.04861440137028694, -0.7973929643630981, 1.7189651727676392, 0.6702203154563904, 1.5040154457092285, 0.748577892780304, 0.07492322474718094, 1.1595942974090576, 0.23597101867198944, -1.1682813167572021, -1.056606411933899, -1.0589534044265747, -0.1494690626859665, -0.975261390209198, -0.9431237578392029, -0.14460785686969757, -0.49997466802597046, 1.5570149421691895, -1.2157328128814697, 1.0232117176055908, -0.4365212023258209, -1.8965892791748047, -1.1416953802108765, 0.09290645271539688, -1.7111448049545288, -0.6517658829689026, 0.8029457330703735, -0.5800812840461731, 1.5177054405212402, 0.474955290555954, -0.6408084034919739, 1.808955430984497, 0.029655301943421364, -0.4136753976345062, -1.0067909955978394, 0.5221666693687439, 0.45572739839553833, 1.3859531879425049, -0.37398380041122437, -1.4458916187286377, 1.7172300815582275, 0.2712090015411377, -1.7007434368133545, 0.643598735332489, -0.8396598696708679, -1.3985588550567627, 1.4789773225784302, 0.4015142321586609, -0.9702249765396118, 1.4322607517242432, 2.0260257720947266, 0.4817773997783661, -1.2296242713928223, -0.1806279420852661, 0.4669632315635681, -1.027143120765686, 1.5641624927520752, 0.887424111366272, -0.45564642548561096, 0.5837634801864624, -0.08977551758289337, 0.8949049115180969, -0.1961960792541504, 1.1663703918457031, 1.3318097591400146, 1.6762536764144897, 0.674554705619812, -1.2936105728149414, -0.9076829552650452, 1.3826394081115723, -0.6939097046852112, 1.2762534618377686, 0.675488293170929, 0.2080857753753662, 1.8263890743255615, -0.3240036964416504, -0.16272197663784027, 0.04434249550104141, 1.374082326889038, 0.2239098697900772, 2.5887463092803955, -0.08434806764125824], [0.25269758701324463, -0.05759163945913315, 1.2659512758255005, 1.9090490341186523, 0.9633410573005676, -1.1949034929275513, -1.033862829208374, -1.2479358911514282, 1.5216559171676636, 0.10267705470323563, 0.9517393708229065, -0.4690154790878296, -0.6411203145980835, -0.26284295320510864, -1.6530542373657227, 1.6945416927337646, -0.017017384991049767, 0.7519652843475342, -0.24333977699279785, 0.643140435218811, 1.0901532173156738, 2.2451207637786865, 1.6065216064453125, -0.4906924068927765, 0.8204330205917358, -0.5518627762794495, 1.3032021522521973, 0.4104330837726593, 0.9482351541519165, 0.2916589081287384, 0.5377755165100098, 2.1146645545959473, -0.9048519134521484, -1.0890185832977295, 1.5983705520629883, 0.6520757675170898, 0.5848338007926941, 0.495150089263916, -1.2337104082107544, 0.8131290674209595, -0.8232209086418152, -1.2267601490020752, -1.1082557439804077, -0.7794093489646912, -1.3514413833618164, -1.5352307558059692, -0.29954326152801514, -1.1321941614151, 0.5021946430206299, 0.8202394247055054, 0.6432047486305237, 0.6178016662597656, 0.5937976241111755, -1.8093754053115845, 0.9811968803405762, 0.6288115382194519, -0.17083656787872314, 0.4199206829071045, 1.5035216808319092, -2.107288122177124, 1.3822053670883179, 1.6862425804138184, -1.2117382287979126, -0.43703094124794006, -1.4457675218582153, 1.0106899738311768, -0.07423904538154602, 0.7084028720855713, -0.6741839647293091, 0.7842682003974915, -0.8109336495399475, 1.0661380290985107, -1.9194202423095703, -0.8904000520706177, 0.19428841769695282, 0.01706872321665287, -2.834259033203125, -1.5249773263931274, 0.6299281716346741, -1.322200059890747, -0.07588889449834824, 1.8023359775543213, 2.0342977046966553, -1.5466675758361816, 0.41156792640686035, -1.0108859539031982, 0.6502962112426758, 0.47538742423057556, 1.3792790174484253, -2.3316500186920166, 1.97394859790802, 1.980029821395874, 3.293656587600708, -1.8467761278152466, -0.007487942464649677, 0.7780901193618774, -1.1620994806289673, -0.6551780700683594, -0.5944287776947021, -0.5685731768608093, 2.3028786182403564, -0.6106940507888794, -0.053209468722343445, 1.1805070638656616, -1.1308104991912842, -0.48593777418136597, -0.11231759190559387, 1.6602323055267334, -0.493141770362854, -0.03168940171599388, -0.3475046157836914, 0.10694850236177444, 1.4113259315490723, -0.30300652980804443, 1.152866005897522, 0.6042370796203613, -0.21579208970069885, 0.34541577100753784, 2.330274820327759, 0.5412917733192444, -0.4239133894443512, 0.06613679975271225, 0.7622788548469543, 0.2368641495704651, -0.8596137762069702, 0.7609438300132751, 0.7351399660110474, -0.10295580327510834, -0.6814543008804321], [-1.0918267965316772, -0.0050861649215221405, -2.022268295288086, 0.281480610370636, -0.6843570470809937, -0.48663508892059326, -0.9427112936973572, -0.3796643316745758, 0.6477594971656799, 1.5760605335235596, 1.1604814529418945, 0.24466224014759064, -1.7099497318267822, -0.3731859624385834, -0.5448780655860901, -0.49756667017936707, -2.639353036880493, -0.06982865184545517, 1.1764540672302246, -1.136884093284607, -0.331611692905426, -0.09596074372529984, 0.8750090003013611, -0.5840986371040344, -1.5791476964950562, -0.11976794898509979, 1.160392165184021, -0.042973946779966354, 0.7021848559379578, -1.6032990217208862, -0.3294009566307068, 0.296744704246521, 0.14046156406402588, -0.30680856108665466, 1.2420446872711182, -0.3862529695034027, -0.7339044213294983, 0.3048769533634186, -0.4063829779624939, 1.2477827072143555, -0.40310072898864746, 2.872161626815796, -0.029082132503390312, -0.6596318483352661, -0.4467538893222809, 1.1002787351608276, 0.020237600430846214, -1.096317172050476, 1.4343276023864746, -0.2334417700767517, 1.153130292892456, -1.547298789024353, -1.3284149169921875, 0.8807588815689087, -0.7610993385314941, 0.4771024286746979, 1.1412245035171509, -0.6069160103797913, 0.4765993654727936, -1.4548245668411255, 0.22172021865844727, 1.538865089416504, -2.1689491271972656, -0.473832368850708, 0.4864022731781006, 1.1234922409057617, 0.8217030763626099, -0.8991342782974243, 0.21225808560848236, 1.381922721862793, 1.036553978919983, -1.0136388540267944, -0.4810323715209961, 0.9905076026916504, 0.47841644287109375, -3.3831539154052734, -0.03847285732626915, -0.005650956649333239, -1.0318740606307983, -1.2051154375076294, 0.9446084499359131, 1.0439294576644897, -0.25527423620224, 0.5515286922454834, 0.4892686903476715, 0.8830145597457886, -1.3835389614105225, -1.8011728525161743, -1.9990508556365967, -0.38266947865486145, -0.4071938693523407, -0.3656594455242157, -0.5009889602661133, -0.5966204404830933, 0.4542969763278961, -1.0588524341583252, 0.312676340341568, -1.7488045692443848, 1.587632417678833, 1.3813116550445557, 0.7136244177818298, -0.27555668354034424, -1.4899266958236694, -0.4341661036014557, -0.5194551944732666, 0.415008008480072, 0.07284686714410782, 0.14587001502513885, 0.5545075535774231, 0.9725167751312256, 0.7085766196250916, 1.0232065916061401, -0.5837292075157166, 0.26276880502700806, 1.193332552909851, -0.518113911151886, 0.06255742162466049, 0.7896435856819153, -0.373351514339447, -0.7978593111038208, 0.4946437478065491, -0.4958952069282532, 0.5686472654342651, 0.9608299732208252, 0.03509528562426567, 0.7357211112976074, -0.3363126516342163, -0.1823141723871231, -1.342088222503662], [-1.8232609033584595, -0.7845866084098816, -0.8307166695594788, 0.8582856059074402, -0.1377095729112625, 0.9292237758636475, -0.7635030150413513, -0.9627525806427002, 1.3876148462295532, -1.912063479423523, -0.9041984677314758, -1.6609430313110352, -0.6469801664352417, 0.9250292778015137, -0.733574390411377, 0.9814301133155823, -0.3728937804698944, 0.043092094361782074, 0.6824633479118347, 1.6352356672286987, 0.009576216340065002, 0.16099673509597778, -0.4922234117984772, 0.9521547555923462, 0.6741953492164612, 0.46962976455688477, -0.07823751121759415, 0.727970540523529, 1.6727726459503174, 0.171162948012352, 0.8518493175506592, 1.0753440856933594, 0.6244544386863708, 0.31502947211265564, 1.4513179063796997, -0.9620091319084167, -2.7887866497039795, 0.2286127358675003, -0.692854642868042, -0.5093438625335693, 0.17686088383197784, 0.7214200496673584, -1.497074842453003, 0.8254732489585876, 2.6863927841186523, 0.40585842728614807, -0.05551624670624733, 1.0289362668991089, 0.11571483314037323, 0.24906155467033386, -0.9432580471038818, -0.893226146697998, -0.14355789124965668, -0.4232191741466522, 0.0841245949268341, -0.08401824533939362, -0.21932733058929443, 1.8419440984725952, 0.17747044563293457, 2.810269355773926, -0.1643645167350769, -1.8646953105926514, 0.15102380514144897, 0.3268008530139923, -0.46182096004486084, 0.6100356578826904, -1.1348158121109009, -0.09472192078828812, 1.193738579750061, 0.6937955021858215, 1.4871995449066162, 0.1299746334552765, 1.2371019124984741, -1.524970531463623, -0.6822037696838379, -0.22588427364826202, 1.7871192693710327, -0.6230378746986389, -2.9528818130493164, 0.8988045454025269, 1.4868825674057007, -0.13714434206485748, 0.3680804967880249, 0.7969666123390198, 0.7128788828849792, -0.6438658237457275, 0.8766670823097229, 0.08225316554307938, -0.15909413993358612, -1.8336105346679688, 0.4654134511947632, -0.4024609625339508, -0.7029053568840027, -0.93524569272995, 0.14493267238140106, -0.34100979566574097, 1.1398005485534668, 0.24107341468334198, -0.0023359693586826324, 1.1747462749481201, -0.2309601753950119, -0.3725424110889435, -0.7639160752296448, -0.5398378968238831, -1.2253639698028564, -1.4371191263198853, -0.40135928988456726, -0.6952014565467834, -0.3206060826778412, 0.10162685066461563, -0.293601930141449, 0.1701129972934723, 1.0758966207504272, 1.2370507717132568, -0.30733150243759155, 0.3750149607658386, 0.5310283899307251, 0.7059776186943054, -0.4642752707004547, 0.6599528789520264, -1.1317088603973389, 1.647079348564148, 0.37006300687789917, -0.008470714092254639, 0.2684129774570465, 1.1719430685043335, 0.1782008409500122, 0.6048173308372498, 1.1729168891906738], [-0.10308470577001572, -1.002208948135376, 1.2157586812973022, -0.5244684815406799, 0.19342437386512756, 0.5457190275192261, 0.6335519552230835, 0.04728389158844948, -0.37220498919487, -0.8318741321563721, -1.1870009899139404, -0.08488740772008896, -0.29334110021591187, 1.34351646900177, -0.040212396532297134, -0.07161756604909897, -0.8763033151626587, -1.1254146099090576, -1.0413833856582642, 1.4137587547302246, 1.211261510848999, -0.6256844997406006, -1.5971468687057495, 0.5353662371635437, 2.0763394832611084, -0.8175889253616333, -1.0342648029327393, -2.9656577110290527, -0.49653124809265137, -0.030129946768283844, -1.3500007390975952, 1.9473443031311035, 1.5461112260818481, -1.1292250156402588, -0.40897637605667114, -2.127563714981079, -0.07952158898115158, -1.3436427116394043, -0.028250351548194885, -0.06382384896278381, -0.04001859948039055, -0.32083413004875183, 0.5646227598190308, -0.3880053758621216, 2.0000247955322266, 0.33610838651657104, 0.09485504031181335, -0.13876454532146454, 1.3975547552108765, 0.7641093134880066, -0.43448570370674133, -0.14135834574699402, 0.6329583525657654, 0.13026019930839539, 0.3795351982116699, 0.8968639969825745, -1.6157479286193848, -0.7191780805587769, -1.0466550588607788, 3.261547803878784, 1.2895737886428833, -0.3289002776145935, 0.8733662962913513, -1.6960004568099976, -2.4215779304504395, 0.7009099125862122, -0.4170615077018738, 0.5956048965454102, 0.38924795389175415, -0.2311299741268158, -0.8075817823410034, 1.3944731950759888, -0.5760082602500916, 1.877820611000061, -1.0165106058120728, 0.05106387287378311, -0.7497945427894592, 1.0898189544677734, -1.5655081272125244, 0.4029482305049896, -0.6680339574813843, 0.21026581525802612, -1.6865493059158325, -2.2659411430358887, -0.9928619265556335, 1.079119086265564, 0.1522706001996994, 0.7023159265518188, -0.909616231918335, 0.4673647880554199, -0.8947338461875916, -0.7639808654785156, 1.7232084274291992, -3.2110302448272705, -0.4312673509120941, 1.2320609092712402, 1.8977214097976685, -0.016640324145555496, -0.7663285732269287, 0.9321870803833008, -1.0039986371994019, -0.3077739179134369, 1.4486736059188843, 0.27382692694664, -1.3149335384368896, 0.6281856894493103, 1.6599016189575195, 0.13510528206825256, 0.13747091591358185, 0.5269736051559448, -0.42876961827278137, 1.2822493314743042, 2.3120241165161133, 1.228660225868225, -1.2780225276947021, 0.6742580533027649, 1.769576907157898, 1.1323883533477783, -0.11930925399065018, 1.6944140195846558, -1.3165855407714844, 0.0430147722363472, -0.45769816637039185, -1.3034541606903076, -0.4072198271751404, 0.7331108450889587, -0.4036378860473633, 0.5848223567008972, 2.7005205154418945], [-0.3145330846309662, -0.3331355154514313, -0.47424933314323425, -1.461296558380127, -0.09194285422563553, -0.03391367942094803, -0.6672210693359375, 0.15271538496017456, 0.3270832300186157, -0.7879508137702942, 0.081845223903656, 0.5221394300460815, 0.007884982042014599, -0.46674084663391113, 0.653424859046936, 0.19869554042816162, -0.08892752975225449, -0.6319665908813477, -1.2087374925613403, 1.257097840309143, -1.377876877784729, -0.9263177514076233, 0.21431413292884827, -1.550428867340088, -0.8218488693237305, 0.22152408957481384, 0.8930690288543701, -1.4647843837738037, 0.6561605930328369, 0.5063964128494263, 1.1043354272842407, -1.9935578107833862, 0.10139667987823486, 1.6541106700897217, -0.874531090259552, -2.370234966278076, -0.4853825271129608, 0.7719060182571411, -0.35030415654182434, -0.7694655060768127, -1.782897710800171, 1.8584803342819214, 1.8363364934921265, -0.043695732951164246, -0.2598106563091278, 0.5914807915687561, 0.8182999491691589, -0.6725326180458069, -0.7105079293251038, 0.9449275135993958, 1.0262731313705444, 0.850646436214447, -1.7092845439910889, 2.2319388389587402, 1.6460736989974976, 1.1962145566940308, 1.1325536966323853, 1.1519787311553955, -0.027099670842289925, 1.8183730840682983, 0.06075276806950569, -0.242365300655365, -0.7519686818122864, 1.4609131813049316, 1.4542465209960938, 0.17551930248737335, -0.8635511994361877, -1.0987365245819092, -0.3762019872665405, 0.7631646394729614, 0.6014178395271301, -0.625072717666626, 1.214329481124878, 0.8510962724685669, -0.8565083742141724, 1.951732873916626, -0.08671437948942184, -0.7002200484275818, -1.0355217456817627, -0.7609148025512695, 1.3659666776657104, -0.632669985294342, 1.3738619089126587, -0.28187817335128784, -0.09060044586658478, -0.8781098127365112, 0.8294578194618225, 1.4490379095077515, 0.11256565153598785, -1.15546715259552, -2.8697142601013184, -0.6290994882583618, 0.2962099313735962, -0.7781526446342468, 0.4470333456993103, -0.014688340947031975, -1.5748238563537598, 2.5841784477233887, -0.3799091577529907, -0.21633954346179962, -1.113608956336975, -0.5373854637145996, 0.7408255934715271, -0.08704517781734467, -1.0664829015731812, 1.1664223670959473, -0.2873029410839081, 0.08435605466365814, -0.5382199287414551, 0.8866143822669983, 1.096234679222107, 0.39351385831832886, 0.6088929176330566, 0.8384103775024414, 1.2498846054077148, -1.452991247177124, -0.8575263023376465, -0.4207950532436371, 2.4056575298309326, -0.45025885105133057, -0.23286353051662445, -2.5283498764038086, 0.48304110765457153, 0.7472110986709595, -0.31270667910575867, -1.1245194673538208, -0.9110953211784363, -1.594667911529541, 0.10101328790187836], [2.093647003173828, -0.9726172685623169, -1.2246822118759155, -0.20786646008491516, -1.1843348741531372, 2.1981141567230225, 1.5194385051727295, 0.622463047504425, 0.7237368226051331, -0.10581451654434204, -0.13364209234714508, -1.998359203338623, 0.7397456765174866, 0.5728741884231567, -1.4164459705352783, -1.6626222133636475, 0.29241031408309937, -1.9136757850646973, -0.6193543672561646, 1.956680178642273, 0.41925692558288574, 0.4037627875804901, -1.6751309633255005, -1.3057982921600342, 0.9258629679679871, -0.4312954545021057, 0.5469765663146973, 0.9911239147186279, 0.4087998867034912, -0.27122604846954346, 0.18118363618850708, 1.7918667793273926, -0.40667271614074707, -0.3267693817615509, 0.7467674612998962, 0.2240987867116928, 2.799798011779785, 1.789802074432373, 0.4184102714061737, 1.4009394645690918, 0.4339962601661682, 1.0844483375549316, -0.20646460354328156, -0.7551583647727966, 0.2660086750984192, -0.7279008030891418, 0.3800188899040222, -1.689898133277893, 1.5914397239685059, -0.6233068108558655, 0.2530806064605713, -1.1740758419036865, -0.026204299181699753, 1.346669316291809, -0.7734180688858032, -0.8450281620025635, -0.6287947297096252, 1.2041795253753662, -2.1439270973205566, -0.7546966671943665, -0.17374512553215027, 0.9081057906150818, 1.831729769706726, -0.6303342580795288, -0.8647534847259521, -0.41156715154647827, 0.1759922355413437, -0.32009148597717285, 0.10130993276834488, 0.4704502820968628, 0.4022030830383301, -1.088289499282837, -1.6503478288650513, 1.339940071105957, 0.023688701912760735, -0.06413237005472183, -0.6290062069892883, -0.615742027759552, -0.5571358799934387, 0.9748799800872803, -0.9393928647041321, 0.530680775642395, 2.3446998596191406, 0.9361692070960999, 0.3768925964832306, -0.27882206439971924, -1.5788252353668213, 1.9255844354629517, -0.637939989566803, -0.8508032560348511, 0.04111219197511673, 0.8388439416885376, -0.5433443784713745, -0.9663248658180237, -0.6944525837898254, -0.8944939374923706, 2.0938308238983154, 1.5366281270980835, -1.2238715887069702, -1.7721768617630005, 0.17041368782520294, -0.21862278878688812, -0.8200011849403381, 0.5506971478462219, 0.4845789074897766, -0.04240282624959946, -1.378182053565979, -1.0681120157241821, 0.3013540804386139, 0.09023500233888626, -2.3569729328155518, 0.7677152752876282, 0.3204558789730072, -0.051950883120298386, 0.6813010573387146, -1.2458542585372925, -0.11036787927150726, -1.4404085874557495, 2.383458137512207, 0.8140743374824524, 0.12769772112369537, -0.958622395992279, 0.06559859961271286, -1.6647955179214478, -1.0959166288375854, -1.2757924795150757, 0.5193163156509399, -0.22505797445774078, 0.11146505922079086], [-0.1843509078025818, 0.8637915849685669, -1.272448182106018, -1.5154376029968262, 0.15626299381256104, 1.3741475343704224, -0.36366352438926697, 1.0967683792114258, 0.1413501352071762, -0.4575982391834259, -0.1087774932384491, 0.2753262221813202, -0.7136937379837036, 0.145733043551445, -0.07872407138347626, -0.026657337322831154, -0.4222954511642456, -0.5391283631324768, -1.1895686388015747, -0.8153429627418518, -0.08417977392673492, -1.0460454225540161, -0.6689233779907227, -0.8109287023544312, -0.5557610392570496, 1.6035170555114746, -0.3255174160003662, 0.7342983484268188, -0.3966936767101288, 1.1404021978378296, -0.8808180093765259, 0.24262958765029907, 1.2441319227218628, -0.3083476424217224, 0.9258604645729065, 1.3407005071640015, 0.6065719723701477, -1.310683250427246, 1.7315449714660645, -0.4038628041744232, -0.15539410710334778, -0.27300533652305603, -2.03428053855896, -0.037083759903907776, -0.1672387719154358, 0.9211874604225159, 0.5177052021026611, 0.32389262318611145, -0.8201411962509155, -1.2141393423080444, 1.3469611406326294, -0.1837005466222763, 0.2441922426223755, -1.4216076135635376, -1.8289729356765747, 0.04692168906331062, -0.5424916744232178, -0.7167356610298157, 0.3986397981643677, -1.9528307914733887, 0.39391252398490906, -0.7719331979751587, 2.404744863510132, -0.08292699605226517, 0.04063044488430023, 1.817076563835144, -0.8935511112213135, -0.04389314353466034, 1.4433391094207764, -0.4795158803462982, 0.4949812889099121, 0.8170971870422363, 0.7194380164146423, 2.618605375289917, 1.252177119255066, 0.8186059594154358, 1.44071364402771, 0.3432430624961853, -0.9747795462608337, 2.0665042400360107, 0.7893439531326294, -1.506231665611267, 2.416929006576538, 0.2893305718898773, -0.4743175804615021, 1.9373345375061035, -0.6270588040351868, 1.5028718709945679, -0.5350137948989868, -0.49139249324798584, -0.7959508895874023, 1.1793265342712402, 0.8080813884735107, 0.4580179452896118, -1.726738452911377, 0.8068094849586487, 0.31263452768325806, -1.9380978345870972, 0.8217372894287109, -2.079822063446045, -0.2679409384727478, -0.8236963748931885, -1.1614954471588135, 0.16713540256023407, -2.4459688663482666, -1.8151570558547974, -1.759440541267395, -2.00498628616333, -0.4951106011867523, 0.9161947965621948, -1.4223499298095703, 0.43142300844192505, -1.3596638441085815, -0.6320237517356873, -0.7856238484382629, 0.5518448352813721, 1.5442239046096802, 0.2936137020587921, -0.2637615203857422, -0.46099379658699036, -0.171027272939682, 1.0719934701919556, 0.3905908465385437, -0.5364379286766052, 0.4928554892539978, 2.337095260620117, 0.6131531000137329, -0.38645243644714355, -0.686058759689331], [-0.11041438579559326, -0.22685623168945312, 0.2286500632762909, -0.27022913098335266, 0.8761275410652161, -2.2963011264801025, 0.6868706345558167, -0.6147231459617615, -1.0797151327133179, -0.19041317701339722, 0.6817558407783508, -1.6695454120635986, 1.3850175142288208, 0.9695132970809937, 1.268804907798767, 0.29055255651474, 1.9783838987350464, 1.3908154964447021, 0.5593053102493286, 0.8809680938720703, -0.2901356518268585, -1.7651491165161133, 1.4569014310836792, -0.22239097952842712, 0.39726579189300537, 2.3169093132019043, 1.4732475280761719, -0.8198559284210205, -1.4716291427612305, -1.3919075727462769, 1.3683685064315796, -0.13823701441287994, -0.6476603746414185, 1.2680284976959229, 1.921892523765564, -1.0744576454162598, -0.5667126178741455, 1.2182323932647705, 0.47280439734458923, 0.7868859171867371, 0.4357183873653412, -1.0586094856262207, -1.2165199518203735, -0.3335129916667938, -0.7312445640563965, 0.34044551849365234, -0.31435027718544006, -1.724043369293213, 0.41707974672317505, -1.36745023727417, -1.411653757095337, -0.6728468537330627, 0.9572142362594604, -0.5805150866508484, -0.06809797883033752, -1.5676963329315186, 1.619567632675171, 0.9594978094100952, -1.9250129461288452, -0.7751423716545105, -0.7405645251274109, -0.9503692388534546, -1.1270455121994019, -0.5061657428741455, -0.9713798761367798, 0.9294542074203491, -1.6086785793304443, 0.18613992631435394, 0.14539557695388794, 0.157157763838768, 0.7496222853660583, -1.1047015190124512, 1.6768378019332886, -1.9463032484054565, -1.261492371559143, 0.41399985551834106, 0.7730019688606262, 0.21845808625221252, -0.12677356600761414, 1.7466570138931274, 0.7225795388221741, -0.19177380204200745, -0.4202706515789032, -1.2925677299499512, -0.9982260465621948, -0.06021595373749733, 0.7408756017684937, -2.019548177719116, -1.0384914875030518, 0.5595971345901489, -0.4727512001991272, 0.6441329121589661, 0.09445031732320786, 1.483426809310913, 0.30155637860298157, 0.5080137252807617, 0.022129245102405548, 0.16025759279727936, 0.3804550766944885, 0.37028244137763977, -0.950598418712616, -1.606305718421936, -0.873717188835144, -2.4092936515808105, -2.2492311000823975, -0.4855620861053467, -0.4616399109363556, 0.30613982677459717, -0.7517403364181519, -1.2339962720870972, 0.049486227333545685, 0.01787881925702095, 0.5425251126289368, -0.2984895408153534, 0.19633929431438446, -2.763396978378296, -0.5249190330505371, -0.8611527681350708, -1.6446815729141235, -0.33552688360214233, 0.4420865476131439, 1.3252760171890259, -0.01842048577964306, 0.008431979455053806, -1.356166124343872, -0.3128109872341156, 0.09107528626918793, -0.7390685677528381, -0.41340598464012146], [0.7145456075668335, 2.751631021499634, -0.4180198609828949, -0.7264653444290161, 0.8944275975227356, -2.7602107524871826, -0.5770910382270813, -0.9270393252372742, 0.4021894931793213, 0.5452916622161865, -0.4232184886932373, 2.0026934146881104, -1.6034181118011475, 0.23158718645572662, -0.7185317277908325, 1.5979584455490112, 0.8062967658042908, -0.4617275893688202, 0.9312958717346191, -0.2600962221622467, -0.9591044187545776, -0.44672513008117676, -1.0313345193862915, -0.03835321590304375, 0.6240666508674622, 0.10097572207450867, -0.20532472431659698, 1.064745306968689, 0.9488253593444824, 0.49999919533729553, -1.5378837585449219, 0.9714398384094238, -0.4595460593700409, 0.5191044211387634, 1.6356956958770752, -2.0626473426818848, 0.32655805349349976, 0.2195374220609665, 2.022982597351074, 1.0050469636917114, 0.009462378919124603, -0.12677179276943207, 1.6435725688934326, -0.28103530406951904, -0.3041498363018036, 1.138494610786438, 0.8303148150444031, -0.39530426263809204, 0.1926492303609848, -1.1894880533218384, -0.2627947926521301, 0.6624149084091187, -0.20675890147686005, -0.654362142086029, 0.5109685063362122, 0.18403089046478271, 1.0153491497039795, 1.7577388286590576, 0.5720275640487671, 1.4079465866088867, -0.7164023518562317, 1.056330680847168, -0.518477737903595, 1.0750418901443481, -0.05559617653489113, 0.4232320189476013, -0.7885005474090576, 1.064409613609314, 0.9718422889709473, -0.384627103805542, 0.12139527499675751, 0.10899236053228378, 0.677734375, 0.6576249003410339, 1.118914008140564, 0.7039390206336975, -0.2632376253604889, -0.25368908047676086, -0.8261491060256958, -0.0901370421051979, -0.15580783784389496, -0.3814465403556824, -1.9158743619918823, 1.4513338804244995, 0.20157700777053833, -0.09727934002876282, -0.5203086137771606, 0.08744252473115921, 0.9811511039733887, -0.22918887436389923, 0.05886685103178024, 1.1388477087020874, 0.15763378143310547, 0.7194008827209473, -0.11915861070156097, 0.019619500264525414, 0.11259637773036957, -0.7099932432174683, -0.8416558504104614, -1.6754733324050903, -0.20499630272388458, -0.2152196764945984, -1.0260614156723022, -0.31043317914009094, 1.3761224746704102, 0.43674436211586, 1.2643786668777466, -1.5936250686645508, -1.3063395023345947, -0.13061530888080597, 0.419132262468338, 0.07152459770441055, -0.1164160892367363, -1.2018446922302246, 1.3173038959503174, -0.9767601490020752, -1.0672515630722046, 0.4127740263938904, -0.5807350277900696, -0.7643827199935913, 0.5433545708656311, 0.2949831187725067, 0.08562309294939041, 0.6138334274291992, 0.5899229645729065, 0.3184826672077179, -0.31850698590278625, -0.9982480406761169, -0.30137407779693604], [-1.5220131874084473, 1.0251833200454712, -0.7809886336326599, -1.063865303993225, -1.110224723815918, -0.8943838477134705, -1.480668067932129, -0.445627897977829, -1.1599520444869995, 0.7343958020210266, 1.7375614643096924, -0.04552307352423668, -0.9442549347877502, 1.124230980873108, 2.1675565242767334, -0.015612385235726833, 0.03213304281234741, -0.7823716998100281, 0.7880926728248596, -0.8184467554092407, 0.07684357464313507, -0.8236311078071594, -0.6090429425239563, -1.6834622621536255, 0.3290227949619293, 1.572290301322937, -1.0130680799484253, 1.3205422163009644, 0.7364471554756165, 0.18402187526226044, 0.869178056716919, -0.05817326530814171, -1.2492434978485107, 0.37433546781539917, -0.2831293046474457, -0.05132213607430458, -0.08868756145238876, 2.774656057357788, -1.227783441543579, -1.8977524042129517, -1.1517611742019653, 0.17937445640563965, -0.19282382726669312, -0.9314494729042053, -0.3284471333026886, 1.0995469093322754, 0.46649298071861267, -0.1469317227602005, -0.8613942861557007, -0.6007798910140991, 0.6578798294067383, -2.058666944503784, 0.9167857766151428, -0.19260583817958832, -1.5793932676315308, -0.5309436321258545, -0.9481304287910461, 0.9104151725769043, -0.6685075163841248, -0.014367352239787579, 1.3809231519699097, 0.5653538703918457, -0.6935401558876038, -0.1257638931274414, -1.661539077758789, 0.9053780436515808, 0.23282279074192047, 0.8320778012275696, 1.2931305170059204, -0.00439945375546813, 0.7358080744743347, 0.012640171684324741, -0.04888899624347687, -0.20789368450641632, 0.799534022808075, 0.40526336431503296, -0.10004239529371262, 0.19350872933864594, -0.18343539535999298, 1.9257582426071167, 0.3435252904891968, -0.060711633414030075, -1.6929832696914673, -0.7059824466705322, 1.6317648887634277, -0.06833981722593307, -0.6457087397575378, 1.793045163154602, -2.0905497074127197, -0.5610650181770325, 2.4067270755767822, 1.1242316961288452, 0.8095503449440002, 0.009797187522053719, -2.266143798828125, -0.9231346845626831, 0.16418983042240143, -0.2974940240383148, -0.06908653676509857, -0.29505592584609985, 0.02357792668044567, -0.06043405458331108, 0.6094347238540649, 0.42469850182533264, -0.655632734298706, 2.1172678470611572, 1.3456071615219116, -1.6129566431045532, 0.3347925841808319, 0.675847589969635, -1.3559880256652832, -1.4950966835021973, -1.6677550077438354, 0.0474121980369091, -0.4839708209037781, 0.12014830112457275, 0.010438880883157253, -1.5709477663040161, -0.2601635456085205, -0.4245242476463318, 0.9150379300117493, -0.03300975635647774, -0.6170317530632019, 2.1875596046447754, 0.7659034729003906, 0.31539297103881836, -0.7764150500297546, -0.5514057874679565, -0.0012350952019914985], [0.22859987616539001, 1.2297070026397705, -0.997905969619751, -0.365448534488678, 0.2632192373275757, 1.0212501287460327, -0.13967782258987427, -1.2018239498138428, 0.037441227585077286, 0.821968138217926, -0.4334985017776489, 0.28318625688552856, 1.0315992832183838, 0.3944876790046692, 0.06973402202129364, 2.4929003715515137, -0.11829626560211182, 0.0892564058303833, -1.9422621726989746, -0.031140953302383423, -0.12483938038349152, 0.3594650626182556, 0.3779621422290802, -0.17369894683361053, -0.6560348272323608, -0.17878752946853638, -0.9006586074829102, -0.19403833150863647, 0.4403538405895233, 0.12438587099313736, -0.06558921188116074, 0.4855349361896515, -1.0046329498291016, 1.3006774187088013, -2.9540929794311523, 0.08279053121805191, -0.481171190738678, 0.3346157371997833, -3.032621383666992, -1.1225359439849854, -0.7591409087181091, -0.6548234224319458, 1.992221713066101, 1.3550496101379395, 0.8697072267532349, -0.2436945140361786, -1.127604603767395, -0.101005919277668, 0.927071213722229, 0.026250498369336128, 0.11591372638940811, -0.19440586864948273, -0.6790292263031006, -1.019683599472046, 0.5489597320556641, 0.625468373298645, -1.1502546072006226, 0.4676111340522766, -1.0803142786026, 0.5404601097106934, 0.38437509536743164, 0.0696641206741333, -0.024016045033931732, 0.6635503172874451, -0.910690188407898, 0.20276109874248505, 1.5043116807937622, -0.8399983048439026, 1.216455101966858, -1.2922825813293457, -0.8669148087501526, 2.541722536087036, 0.12193389236927032, -0.3326229155063629, 1.2981401681900024, -1.1197651624679565, 1.9348231554031372, -1.2744721174240112, 0.021709177643060684, -1.0551637411117554, 1.4393643140792847, -1.1643214225769043, -2.094158887863159, -2.0638694763183594, -1.787468671798706, 1.3544273376464844, -2.3042054176330566, 0.24037949740886688, -1.1569074392318726, 1.8896116018295288, -0.3070094883441925, 0.07140924781560898, 0.28875240683555603, -0.7602871656417847, 0.6249499917030334, 1.8986510038375854, -0.3205891251564026, 0.6593626141548157, 1.8636506795883179, -0.9629114866256714, -0.8422955274581909, -0.81844562292099, -0.8467037677764893, 1.5142250061035156, 0.6539367437362671, 0.49825751781463623, 0.6568759679794312, 2.07993745803833, 1.1208189725875854, 0.45002567768096924, -1.1961023807525635, 0.8916547298431396, -0.9174240827560425, 1.002174735069275, 0.8227382898330688, -2.0025293827056885, -1.0892144441604614, 0.8726275563240051, 0.9711553454399109, 0.29429885745048523, 0.7607980370521545, 0.6310222148895264, -0.7859494090080261, 0.04663349315524101, -1.159221887588501, -3.0347721576690674, 0.12464091926813126, -0.2125447392463684, -0.9241844415664673], [-0.9554770588874817, -0.8995618224143982, -1.1063215732574463, -0.27413907647132874, 0.5777005553245544, -0.7115727663040161, 0.08555573225021362, -1.5811121463775635, 3.098680257797241, 0.5300548672676086, 1.1744580268859863, 0.3669423460960388, 0.5167876482009888, -0.48530298471450806, 0.6028875112533569, -0.8392869234085083, -0.034726664423942566, 0.9905769228935242, 0.34511327743530273, 1.9610854387283325, -1.1479382514953613, 0.7486557960510254, -1.299940824508667, -0.15168651938438416, -0.38751617074012756, -1.0713591575622559, -1.2692581415176392, 0.6405695080757141, 0.08770895004272461, -0.5264659523963928, -0.46851107478141785, 0.09192115813493729, 1.177502155303955, -0.15770643949508667, 0.8099969029426575, -1.8301223516464233, -0.9879909157752991, -0.5030984282493591, 0.12220408767461777, -1.017642855644226, -0.9585297703742981, 0.11769559979438782, -0.5197615623474121, 1.2172218561172485, 1.191884160041809, -0.4728679656982422, -2.659740686416626, -0.1858750581741333, -0.5233399271965027, -1.279445767402649, -0.9005076289176941, 0.2851305902004242, 0.8644780516624451, 0.15639187395572662, -0.20057475566864014, -1.061123013496399, 0.9750924110412598, 0.7010513544082642, 1.4307668209075928, -0.11115372180938721, -0.10038797557353973, -0.27757054567337036, 0.5927671194076538, -0.5661332607269287, -0.7863613367080688, 0.8089579343795776, -2.3726415634155273, 1.4368274211883545, 1.7683180570602417, 0.05595988407731056, 0.7890520691871643, -1.1038079261779785, 0.36547529697418213, -2.399468421936035, 1.8102045059204102, 0.04975498095154762, 0.6082804799079895, -1.1678932905197144, 0.7106310725212097, -0.9547156691551208, 0.25340598821640015, -0.5130572319030762, -0.2016221433877945, -0.8909446597099304, -2.597674608230591, -1.000510573387146, -0.2943079173564911, -1.508890151977539, -1.7294358015060425, -0.07221168279647827, 0.4939233362674713, -0.4615761637687683, -0.3508276045322418, -0.5420233607292175, -0.42344456911087036, 1.555894136428833, -0.3339703381061554, -1.8720712661743164, 0.3759029805660248, 1.6388846635818481, -0.4640735387802124, 0.3186139166355133, -1.3130766153335571, -0.2179260402917862, -1.1302098035812378, 1.1113070249557495, -1.0488662719726562, 1.008360505104065, 1.5263694524765015, -0.9297723174095154, 0.9488818645477295, 0.3162791430950165, -1.2573614120483398, 1.4695416688919067, -0.6377869248390198, -0.9748538732528687, -0.32013967633247375, -0.21789029240608215, -1.429858922958374, 0.8320888876914978, -0.32736140489578247, 1.2167969942092896, 0.9824895262718201, -0.1050821915268898, 1.1946183443069458, -0.7861464023590088, 0.3203657269477844, 0.5631685256958008, 1.0903334617614746], [-0.5017210841178894, -0.19006863236427307, -0.045473530888557434, -0.47140687704086304, -0.07719147950410843, -1.0213016271591187, -0.710095226764679, -0.01556405983865261, 0.9116464257240295, 0.5368408560752869, 0.2674836218357086, -1.2234894037246704, -0.377232164144516, 1.1482319831848145, 1.4259033203125, 0.5903477668762207, -0.06380021572113037, 1.063495397567749, -0.5257477164268494, 0.43614327907562256, 0.002105857478454709, -0.5169911980628967, 0.9474139213562012, -1.4188734292984009, -1.607704520225525, -1.5596449375152588, -0.6303539872169495, 0.7287853956222534, -1.581697702407837, -1.0416452884674072, -0.6727651953697205, -0.2075466364622116, 0.1206686869263649, -0.7083767652511597, 1.014823317527771, 1.130860447883606, 0.47943371534347534, 0.10583634674549103, -0.1159137412905693, -0.2744368314743042, -0.8067339062690735, -0.823454737663269, -1.4055589437484741, 0.07835981249809265, 1.1720589399337769, 0.15108850598335266, -0.3199332356452942, -0.687229573726654, -0.303056925535202, -0.7733711004257202, 0.4894631505012512, 0.26534608006477356, 0.27074354887008667, -0.28125181794166565, -1.5704880952835083, 0.9860973358154297, -0.9799325466156006, 0.14468765258789062, 1.0571547746658325, -0.7221016883850098, -0.3284839987754822, -0.7662844657897949, 0.2874837815761566, 0.6171649098396301, -0.7616111040115356, 1.0251457691192627, 0.45301157236099243, -1.0607843399047852, -0.5711410045623779, 0.9385203123092651, -0.5626891255378723, 1.5757347345352173, -1.050372838973999, 0.8133971691131592, -1.2932816743850708, -0.5147343277931213, -0.08222303539514542, 0.7869554758071899, 0.9191560745239258, -0.8288344144821167, 0.2978798449039459, 0.8877599239349365, 0.4678993821144104, -1.6310527324676514, -1.2361652851104736, 0.23337790369987488, 1.0955041646957397, -0.6329476237297058, 0.3872520923614502, -0.9732306599617004, -0.37279972434043884, -0.3898061513900757, 0.9334189891815186, -0.7796048521995544, 1.4124945402145386, 1.3027416467666626, -0.9678981900215149, -0.9646683931350708, -1.570224642753601, -0.07111286371946335, -0.739074170589447, 0.3771072030067444, -0.2495431900024414, 0.07167220860719681, -0.11499883979558945, 2.3723204135894775, 1.2172168493270874, -1.5232775211334229, -0.41856420040130615, 0.12398108094930649, -1.0110423564910889, 0.9440439343452454, -0.31166836619377136, 0.42881664633750916, -0.9352816343307495, -0.7170282602310181, 0.722271203994751, -0.2455744743347168, -0.9320192933082581, -0.989052951335907, -1.357725977897644, -1.2697540521621704, 0.31621602177619934, -0.8888763785362244, -0.7357367277145386, -0.875333845615387, -0.34123125672340393, 1.1016287803649902, -0.5358482003211975], [-0.3161274790763855, 0.7396820783615112, -0.4999595582485199, -0.054842833429574966, -0.7177864909172058, 0.5262318849563599, 0.9512178301811218, -0.5215823650360107, -0.6259368658065796, -0.2063073366880417, -0.6035889387130737, 0.44236454367637634, -0.10253318399190903, -0.3395325541496277, -0.842696487903595, -0.2426941692829132, -0.706341028213501, 0.7358237504959106, -1.800277829170227, -0.3843276798725128, -1.2048946619033813, -0.6066863536834717, -0.6215915679931641, -0.511354386806488, 0.23175273835659027, -0.8799433708190918, 2.2684342861175537, 0.04267118498682976, 1.330692172050476, -0.718544602394104, -0.5270019173622131, 0.8353798389434814, 0.49555718898773193, 0.772432267665863, -0.12142464518547058, 0.48461368680000305, 0.6447334885597229, 0.09253524988889694, 0.9952062368392944, 1.6576112508773804, 3.35748553276062, 1.996708631515503, -1.6812669038772583, 1.621840476989746, 0.6977483630180359, -1.7574259042739868, 0.5720937252044678, 0.10208939760923386, 0.8886376619338989, -1.4172428846359253, 1.0135873556137085, -0.3868345618247986, 0.6573939323425293, -1.3147958517074585, -0.26010623574256897, -0.7408156991004944, 1.9477636814117432, 0.37048766016960144, 1.2586758136749268, 0.45310744643211365, 1.4164812564849854, -0.6523197293281555, 1.4717942476272583, -0.5424765944480896, -0.20241779088974, 0.18011844158172607, -1.3366118669509888, 0.5277170538902283, 0.8699535727500916, 0.34257951378822327, -1.0904337167739868, 1.6479928493499756, 0.06325975805521011, 0.6308127045631409, 0.8473806381225586, -0.12313280999660492, -0.9243225455284119, -0.6558442711830139, 0.02037958987057209, -0.272925466299057, 1.8851088285446167, -0.18944548070430756, 0.5841452479362488, 0.7002532482147217, -0.7432873249053955, -0.8821296691894531, 0.8231463432312012, 0.12080247700214386, -1.6030699014663696, -1.0956361293792725, 1.128314733505249, 0.4269048571586609, 0.8457894921302795, -0.5031631588935852, 0.2494714856147766, -0.2150057852268219, 2.099912405014038, -0.8655416965484619, 3.033665418624878, -0.22445644438266754, 0.46983909606933594, 2.3198659420013428, -0.36166736483573914, 1.0211583375930786, -0.812094509601593, -0.5740131139755249, -1.4294610023498535, -2.4705679416656494, 1.236138939857483, 1.1358588933944702, -0.2407597154378891, -2.159161329269409, 0.015650495886802673, 0.3340005576610565, -0.0505882166326046, 0.09898503869771957, -0.1526218205690384, 0.090194471180439, 1.5459542274475098, 0.04001244530081749, -0.43785959482192993, 0.2539553642272949, 1.8828976154327393, -0.32528576254844666, -0.36817866563796997, 0.10630388557910919, 0.4740650951862335, -0.0024177120067179203, 0.6511749029159546], [-1.4472743272781372, 0.4834856688976288, -1.0859166383743286, 2.088711977005005, 0.2423049360513687, -0.2226957529783249, 0.9164568185806274, -0.8653619289398193, -1.1523816585540771, 0.35748162865638733, -0.5402369499206543, -0.043056800961494446, -0.11645921319723129, -2.200258731842041, 0.05077741667628288, -2.6280875205993652, 0.32183516025543213, 1.7557014226913452, -1.50473952293396, -0.17272743582725525, -0.2978743314743042, -1.2553818225860596, -0.3809787631034851, -1.4234408140182495, -0.27366602420806885, 0.46371451020240784, -2.0437963008880615, -1.226155161857605, 0.5769988298416138, 2.4566142559051514, 1.9555741548538208, 1.2355177402496338, 1.4928069114685059, 1.3243499994277954, -0.32279524207115173, -0.7327404022216797, 1.0784271955490112, 1.3072381019592285, -0.7146910429000854, 0.7538497447967529, -0.6685571670532227, -0.7165791988372803, -1.3322887420654297, -0.19388270378112793, -0.31591954827308655, 0.9765601754188538, 0.1338323950767517, -0.07294716686010361, -0.06623999029397964, -0.787203311920166, -0.021980835124850273, 3.1402623653411865, 0.2991277575492859, 0.3474382758140564, 0.20456558465957642, 0.11882022023200989, -1.7692843675613403, 0.4123506546020508, -0.0024269954301416874, 0.146129310131073, 0.8488961458206177, -1.1251251697540283, -0.7017647624015808, -0.8815671801567078, 0.7770044803619385, -1.1861002445220947, -0.887497067451477, -1.5329540967941284, 0.25058111548423767, -1.096726417541504, 0.6437745690345764, 0.33026984333992004, -0.24208873510360718, 0.37675780057907104, -1.1579123735427856, 1.012975811958313, -1.08465576171875, 0.7591254115104675, -0.06337987631559372, -1.1175869703292847, -0.25052741169929504, 0.7080202102661133, 0.48367148637771606, 0.8509267568588257, -0.5378977656364441, -0.14340613782405853, -0.5666899681091309, 1.0545505285263062, 0.30517202615737915, 1.6708717346191406, 0.45923957228660583, -0.41472071409225464, 0.7016090750694275, 0.5613330006599426, -1.669077754020691, -0.7577886581420898, -0.6811754107475281, -0.579076886177063, 1.1942511796951294, 0.16641439497470856, -2.508685827255249, -1.0619513988494873, 0.3253382742404938, 0.6065714955329895, 0.5060490965843201, -0.32836753129959106, 1.9194380044937134, -0.30108723044395447, 0.2611043453216553, -1.72423255443573, 0.8867784738540649, -0.27503061294555664, 0.7487115859985352, 0.7109366059303284, -0.03185024484992027, -0.8951852917671204, 0.47036078572273254, 0.5020175576210022, -0.1587369590997696, -0.4167913496494293, 0.2849922180175781, 0.15386301279067993, 0.726155698299408, 1.594016194343567, -0.7126582860946655, -0.4550587832927704, 1.3658174276351929, 1.0949887037277222, -0.3843821883201599], [-1.2115229368209839, -1.2453625202178955, 1.026282548904419, 1.8545054197311401, 0.03677511215209961, -0.588156521320343, 0.6237474679946899, 0.1719278246164322, 0.7953853011131287, 1.4321023225784302, 0.7728421688079834, 1.1080549955368042, -1.0979732275009155, 0.09564808011054993, 1.178322196006775, -0.32543107867240906, -0.5774158835411072, -0.751316487789154, -1.5312070846557617, 0.03788186237215996, -0.9880427122116089, -0.8371156454086304, -0.008131730370223522, -0.5159088969230652, -0.43364590406417847, 1.161664366722107, -0.6293084621429443, -0.9247114658355713, -2.0890588760375977, 0.7898454666137695, -1.2211639881134033, -0.08239912986755371, -0.08125342428684235, -2.7452211380004883, -1.3013272285461426, -0.5585144758224487, -0.11882957071065903, -0.5156118869781494, -0.3611695468425751, 1.447164535522461, -1.93918776512146, -0.4232253134250641, -0.8547579050064087, 1.3047524690628052, -0.5610203742980957, 0.3947419822216034, 0.061318766325712204, -0.7263385653495789, 0.596183717250824, 0.2595815658569336, -1.0626404285430908, -0.941361665725708, -0.7748688459396362, 0.058388546109199524, -0.8720773458480835, 0.283402681350708, 0.9785683155059814, -0.16710662841796875, -0.0010727939661592245, 0.6097368597984314, 0.07539747655391693, 0.6355744004249573, 1.9694578647613525, 0.3482087254524231, -0.1319005936384201, 0.6092382073402405, 0.37020260095596313, -0.18822641670703888, -1.676794171333313, 1.4312078952789307, -0.9190962910652161, -1.0985395908355713, 0.6940023899078369, 2.572868585586548, -2.39766001701355, -0.002660249825567007, 0.33998310565948486, 0.9207334518432617, -1.0122112035751343, 1.5543506145477295, -1.0540329217910767, 0.18898369371891022, -0.06920371204614639, -0.7798018455505371, -0.12741008400917053, -0.398643434047699, -1.5131767988204956, -0.9452882409095764, -0.5570750832557678, 1.7828203439712524, 0.4125438928604126, -0.3985728323459625, 0.2596731185913086, 0.9905606508255005, -0.11096732318401337, 0.26894235610961914, -1.1438721418380737, 0.04400196671485901, 1.6930309534072876, 0.47986945509910583, 1.4447267055511475, 0.180924654006958, -1.0549958944320679, 1.6747387647628784, 0.49310484528541565, -0.6950627565383911, -1.6026114225387573, -0.1290491223335266, 1.6650084257125854, 0.4931953251361847, 0.05021148547530174, 0.1055607795715332, -0.010520493611693382, 0.45766010880470276, 0.2398122251033783, 0.06967686116695404, 1.1857130527496338, -1.788922667503357, 0.08931738883256912, -1.0110105276107788, -1.2204490900039673, -1.1385729312896729, -0.4154841899871826, -0.9904610514640808, 2.6550986766815186, -0.11981479823589325, 0.8580824732780457, -0.715957760810852, -0.6067206859588623], [-0.15533547103405, -2.586379289627075, 0.8781610727310181, -1.2968449592590332, -0.9074274897575378, 0.18388724327087402, 0.8564110398292542, 0.35506629943847656, -1.682529330253601, 1.4060101509094238, -0.04756133258342743, -1.515902042388916, -0.7916486263275146, 0.7269715666770935, -1.4271528720855713, -0.9498659372329712, -1.2191237211227417, 0.11512930691242218, -0.797778844833374, -0.5351106524467468, -1.0229227542877197, -0.40452077984809875, 0.1343362033367157, -0.5681655406951904, -0.14540396630764008, 0.5861288905143738, -1.412757158279419, 0.2668708562850952, -0.8915070295333862, -1.588533878326416, 0.5048530101776123, -0.918239951133728, 0.866220235824585, -0.4892871081829071, -0.9465153217315674, 0.08750247210264206, -0.8902678489685059, -0.51059490442276, 0.8519178032875061, -0.13430246710777283, 1.357893943786621, -0.6231656074523926, 0.927867591381073, 0.46499454975128174, -0.3379143476486206, -0.264864444732666, -1.353417992591858, -2.2030727863311768, 2.2095565795898438, -1.5322402715682983, -1.3699990510940552, -0.7817237973213196, -0.7741509079933167, 1.3194366693496704, 2.4996814727783203, -1.1692899465560913, -1.1913014650344849, 0.6214606165885925, 1.0011876821517944, -0.6709067821502686, 0.8355665802955627, 0.7513744235038757, -0.4638303518295288, -1.3938145637512207, 0.39898228645324707, -0.3913639783859253, -0.23514172434806824, 0.094474658370018, -0.7011468410491943, 1.1577614545822144, 0.8581956624984741, 0.3377053737640381, 0.31998497247695923, -1.1571356058120728, 0.016430871561169624, -0.047906722873449326, -0.20936396718025208, 0.27218300104141235, -0.12354281544685364, -1.5419760942459106, 0.00948859192430973, -0.6889874935150146, -0.7554416656494141, 0.29490694403648376, 2.021676540374756, -0.23332369327545166, 1.321311354637146, -0.20616310834884644, -0.1586381494998932, -0.03835068270564079, 0.21818435192108154, -0.22145172953605652, 0.3549162745475769, 0.4817439913749695, -1.3936265707015991, -0.18578919768333435, -0.9999355673789978, -1.719504714012146, -1.915547251701355, 0.23215016722679138, 0.8561460375785828, -1.6907286643981934, -0.6117034554481506, -0.559607207775116, -0.6495593786239624, -0.34139567613601685, -0.34330034255981445, 0.9350746870040894, 0.4799320697784424, -0.2207162231206894, 0.33895304799079895, -1.0456174612045288, -0.07259166985750198, -0.15456809103488922, -1.3373247385025024, -0.2959296703338623, 0.6775316596031189, -0.5623902678489685, 0.5961775183677673, -0.3470405042171478, -0.8824270963668823, -1.755910873413086, 0.5263662934303284, -1.3187185525894165, -0.09068700671195984, -1.2400188446044922, -2.03490948677063, 1.0912004709243774, 0.13547265529632568], [2.1979358196258545, 0.06813710927963257, 0.7427191138267517, 0.01920388452708721, -0.8286576867103577, 2.4945614337921143, -1.2341246604919434, 0.1371566355228424, -0.26664069294929504, 0.8352701663970947, -0.3152340352535248, 0.03838661313056946, 0.4072369337081909, -0.8229302763938904, 0.9815568923950195, 0.781701922416687, 1.5829631090164185, -0.6108863353729248, -0.5306929349899292, -0.2632245123386383, 1.8479987382888794, -1.0617727041244507, -0.46629926562309265, 0.12736882269382477, -0.4925574064254761, -0.37986305356025696, -0.9516907334327698, -0.498043030500412, 0.8913896679878235, -0.7593263983726501, 0.709839403629303, 2.3052971363067627, -1.3159983158111572, -1.0337034463882446, -0.98137366771698, 1.049464464187622, -0.7049821615219116, 0.7192811965942383, 1.0344135761260986, -0.42077112197875977, -0.07801409810781479, 0.09503867477178574, 0.12592017650604248, 0.41202300786972046, -0.8860852122306824, 0.8784827589988708, 0.2225857824087143, -1.5408552885055542, 0.2556140422821045, 0.6628101468086243, 0.4821105897426605, 0.35945767164230347, -0.7051843404769897, 0.4162644147872925, -1.1168802976608276, 1.3517651557922363, 0.7040925025939941, 0.09833187609910965, 0.0006414363742806017, 0.3831824064254761, -0.8942199349403381, -0.2217913269996643, -0.8309453725814819, 1.6073325872421265, -1.97600257396698, 0.8302613496780396, -1.1846177577972412, -0.32090622186660767, 1.8037413358688354, 0.8485336303710938, -0.4821799397468567, 1.2417303323745728, -0.6272062659263611, 0.16594138741493225, 2.4254932403564453, 0.9598787426948547, 0.7141740918159485, -0.016616715118288994, -0.9465005993843079, 0.20462708175182343, 1.435388445854187, 0.06101103499531746, -2.256878137588501, -1.2895170450210571, 0.7239308953285217, -0.35399529337882996, -0.03693455830216408, -0.43521836400032043, 0.5215210914611816, -0.4249386191368103, -1.1708195209503174, -1.5770248174667358, 1.7013760805130005, -1.2136130332946777, -2.573646068572998, -1.118972659111023, 0.9349437952041626, 0.08030837774276733, -1.6913574934005737, -0.7490159273147583, 0.6028826236724854, 0.05297959968447685, -0.7681584358215332, 0.9525054693222046, 0.23855914175510406, -1.1592237949371338, -1.8573758602142334, 3.4736742973327637, 2.168713092803955, -1.4992274045944214, -0.6158358454704285, -1.042169213294983, -1.1119025945663452, 0.04440135508775711, -1.7328636646270752, -0.758278489112854, 0.4928674101829529, 0.2553486227989197, -1.9062057733535767, -0.5767680406570435, 0.1437486857175827, 1.3060258626937866, -1.2648934125900269, -0.20934993028640747, 0.7213579416275024, 1.1046392917633057, 0.6705315113067627, 1.0210330486297607, 0.1575717329978943], [-0.44412684440612793, -0.4364287853240967, 2.0542526245117188, -0.11197298020124435, -1.1412622928619385, -0.6319026350975037, -0.24780243635177612, -0.4061860144138336, 0.3169165253639221, 0.03819317743182182, 1.1074602603912354, 0.4717774987220764, -1.0731022357940674, 0.8780415058135986, -1.6183494329452515, 1.4910787343978882, -0.9673294425010681, 0.5594037771224976, 0.5839101076126099, 1.4103901386260986, 0.5521277785301208, -0.752208411693573, 0.151958167552948, -1.0963636636734009, 0.479182630777359, 0.9711288809776306, 1.6299206018447876, -0.589898943901062, -1.7226924896240234, 1.648598313331604, 0.7087467908859253, 0.17465245723724365, 0.9827569127082825, 2.9103496074676514, -0.010621312074363232, -0.3755258023738861, 0.3568788766860962, 0.8987618088722229, 0.11475872248411179, -0.9113126993179321, -0.43017756938934326, 0.834814727306366, 0.04653489589691162, 1.628462314605713, -1.937999963760376, -1.6161409616470337, -0.2506211996078491, -1.0713924169540405, -0.12562055885791779, -0.16396594047546387, -0.20013168454170227, -0.6181080341339111, -0.4151579439640045, -1.5254571437835693, 0.20064474642276764, 1.2475247383117676, 0.426509290933609, -1.0664352178573608, 0.11602184176445007, -0.19743914902210236, 1.491979718208313, -0.9790366888046265, -1.7294557094573975, -1.0556044578552246, 0.6094166040420532, -0.8980754017829895, 1.9177889823913574, -0.657500147819519, -0.43577778339385986, 2.3315610885620117, 2.2953133583068848, 1.9903955459594727, -0.4256795048713684, 1.8137990236282349, -0.37605759501457214, 0.8496186137199402, -0.9652045965194702, -1.5274910926818848, 0.15756317973136902, -0.07156434655189514, -1.4202895164489746, -0.042396605014801025, -0.5902912616729736, 0.5776597857475281, -1.427046775817871, 0.14631716907024384, 0.6844159960746765, 1.2565572261810303, 0.6302059292793274, -1.0545525550842285, 0.31570112705230713, 1.2057557106018066, -1.0272146463394165, 0.23666667938232422, 0.05506899952888489, -0.4725683331489563, 1.7372708320617676, -0.5227459073066711, -1.149640440940857, -0.8756492733955383, 0.5204964876174927, -0.2982829213142395, -0.21460311114788055, 0.7642951011657715, -0.14403359591960907, 0.12242593616247177, 0.2353747934103012, 0.32150688767433167, -0.5800127387046814, 0.9021831154823303, -1.4322108030319214, -1.7668850421905518, -0.8771060705184937, 0.2927360534667969, -1.1711385250091553, 1.1404566764831543, -0.5104634761810303, -0.6422684192657471, 0.13493776321411133, 0.60318523645401, -0.4407995343208313, -1.7106313705444336, -0.5104001760482788, -0.7779374122619629, -0.7731731534004211, 0.5190803408622742, -0.48692286014556885, -0.02200404554605484, -0.06355317682027817], [-0.531583309173584, -0.4774772822856903, 0.8037915825843811, 3.135385274887085, -0.7408846020698547, 2.8425629138946533, 1.6602236032485962, 0.3651719391345978, -0.22551822662353516, 0.6531129479408264, -0.13907618820667267, -0.2609015107154846, -0.6316588521003723, 1.5450035333633423, 0.7696622014045715, 0.7299423813819885, -0.40610432624816895, -0.10782601684331894, -0.11434115469455719, -0.005544058512896299, 0.4614977240562439, 1.6412216424942017, 0.3512679636478424, -0.3294432461261749, 2.0953474044799805, -2.1011276245117188, -1.5139755010604858, -1.385426640510559, 0.2873809039592743, 0.7266499996185303, 0.5590178370475769, -0.23644320666790009, 0.8874129056930542, -0.18937411904335022, 0.404411256313324, 1.3066879510879517, 1.6429203748703003, 0.9193265438079834, -0.3045135736465454, -1.3189252614974976, -2.356670379638672, 0.8159993290901184, -0.8697645664215088, -1.4537831544876099, 0.32975658774375916, -0.06980972737073898, 1.446569561958313, -0.0864163264632225, -2.514967441558838, -1.0074880123138428, -2.224432945251465, -0.20611609518527985, -2.428596258163452, -1.8078173398971558, 1.5028244256973267, 0.34810084104537964, 0.4543587565422058, 2.476771354675293, -0.08570024371147156, -0.007000397890806198, -0.7198781371116638, -0.39022254943847656, 1.081209659576416, -0.8493766784667969, 0.36761239171028137, -1.4560564756393433, -0.987227737903595, 0.33856943249702454, 0.6886681318283081, -1.076765537261963, 0.5349966883659363, 0.7331952452659607, -0.010087760165333748, -0.8413572311401367, -1.1450209617614746, -0.3502749502658844, 0.6414679288864136, -0.27294304966926575, -0.1816103160381317, 2.1088528633117676, -0.7541558146476746, 1.14239501953125, 0.3511962592601776, -0.2496464103460312, 0.7858731150627136, 1.4392917156219482, -0.10182029008865356, 1.3978548049926758, 1.539311408996582, -1.5773160457611084, 0.6599277853965759, 0.0969223827123642, -0.669815182685852, 0.7040535807609558, -0.47808343172073364, 0.19311580061912537, 0.45426520705223083, 1.3382840156555176, -0.14199109375476837, -0.8223001956939697, -0.517699122428894, 0.10866107791662216, 0.7304677963256836, -1.8801554441452026, 0.686486542224884, -0.8359059691429138, 1.5891300439834595, 0.013240214437246323, -1.0430771112442017, 0.5828819870948792, -1.87056303024292, 1.1780225038528442, -0.760668933391571, 0.415096640586853, -2.810589551925659, 1.6042797565460205, 0.37738823890686035, 0.8491070866584778, 1.2338337898254395, 1.1042511463165283, -0.30221790075302124, 0.22357206046581268, -0.12192948162555695, 0.8359188437461853, 1.6368502378463745, -0.38503700494766235, 0.930515706539154, -0.9771811366081238, -1.3207346200942993], [2.6400694847106934, 0.08686970174312592, 0.5503348112106323, -0.13670401275157928, -0.18551556766033173, -0.2964519262313843, -0.8397470116615295, -0.13820019364356995, -0.8055352568626404, 0.7781009078025818, -0.05292021855711937, 0.7968729138374329, 0.4833040237426758, -0.9577822089195251, 0.4130103886127472, -0.2961917817592621, -0.010267887264490128, -0.3038908839225769, 0.5929922461509705, 0.8265866041183472, 0.0716925635933876, 0.8449031710624695, 0.11968144029378891, -1.3695759773254395, -0.2157202810049057, -0.05251400172710419, -0.2014608532190323, 0.8705278038978577, 1.1244415044784546, -1.5325508117675781, -0.25343212485313416, 0.5122227668762207, 0.16251277923583984, -2.388468027114868, 0.07425867021083832, -0.5904858708381653, 0.7875602841377258, 0.8143147230148315, 0.22299499809741974, -0.2501853108406067, -2.4945638179779053, -0.23972685635089874, 0.08707696944475174, -0.4432356655597687, -0.5586444735527039, -0.4238504469394684, -0.03888905048370361, 0.6603425741195679, 1.0593533515930176, 1.4375113248825073, -0.6855954527854919, 0.6384254097938538, 0.23655356466770172, -1.262642502784729, -0.9632292985916138, 0.8999588489532471, -0.6001505851745605, 0.04893020913004875, -0.3046872317790985, 2.1676931381225586, 0.929989218711853, -2.014467239379883, -1.3905109167099, -1.489201545715332, -2.437481164932251, -0.7529459595680237, 0.45962849259376526, -0.7296283841133118, -0.41659483313560486, -0.422939658164978, 1.5647119283676147, 1.4993246793746948, -0.15855592489242554, 0.9094394445419312, 0.25045472383499146, 0.9065518975257874, 0.4495091736316681, -0.8588225841522217, 0.7940772771835327, -0.09924149513244629, -0.11626178026199341, 0.868706226348877, -0.011322506703436375, 0.30223023891448975, 0.8030731678009033, 1.2296197414398193, 0.34207332134246826, -0.31893572211265564, 0.21377567946910858, -0.08612009137868881, 1.7378979921340942, -1.4721448421478271, 0.4132165014743805, 0.3908465802669525, 2.152040481567383, -2.4082672595977783, -2.1114397048950195, 0.9407792091369629, 1.636293888092041, 1.6210418939590454, 1.0159249305725098, 0.8079673051834106, -1.360353946685791, 0.04285334050655365, -0.04168746992945671, 0.6702631711959839, -0.7713077068328857, -0.1079547107219696, 0.6324005126953125, -0.05819350853562355, -0.9844111800193787, -1.532210350036621, 0.8720866441726685, -0.47194406390190125, 0.3395615220069885, 0.383826345205307, -0.12613867223262787, -1.173030972480774, 0.7519984245300293, -0.49530500173568726, -1.3913497924804688, -0.6669265031814575, -0.6983062624931335, -1.8489704132080078, -1.409865140914917, 0.7638988494873047, -0.5255347490310669, -0.0339149571955204, 0.41410231590270996], [0.42556723952293396, -0.21682240068912506, 0.4078937768936157, 0.44034305214881897, 1.395482063293457, 0.5455690026283264, -1.4028750658035278, -1.571372389793396, 1.5166369676589966, 1.0355168581008911, -0.9829628467559814, 0.42883163690567017, 1.2613619565963745, -0.9018344879150391, -0.5975812673568726, 1.210801601409912, 0.8195716142654419, -0.0831935703754425, -0.09505987912416458, 1.8301993608474731, 1.32352614402771, 0.33076968789100647, 0.27826306223869324, 0.6143184900283813, -0.8334980010986328, 0.013967974111437798, 0.1591406613588333, -2.3652501106262207, 1.0693210363388062, -0.638100266456604, -0.5084466934204102, 1.6125082969665527, -0.5411473512649536, 1.0556381940841675, -1.1493093967437744, -1.1615084409713745, -0.062467753887176514, -1.063079833984375, 1.2704429626464844, 1.1631128787994385, -0.16829079389572144, 0.5206173062324524, 0.5227521657943726, -1.4336192607879639, -0.42869833111763, 0.49730372428894043, 1.245245337486267, 0.8359268307685852, -0.14997990429401398, 1.0149245262145996, -0.9449734091758728, -0.5349551439285278, -0.7685982584953308, -0.29616406559944153, 0.7854287028312683, -1.0019892454147339, -0.38330885767936707, 0.16150633990764618, 1.9472378492355347, 0.972996711730957, -0.4717315137386322, -0.12869442999362946, 0.6205608248710632, 0.46763089299201965, -1.6266711950302124, -1.3339680433273315, -1.1799535751342773, -1.0003948211669922, -0.1586141735315323, 0.5977460741996765, -0.7001919746398926, 0.7711250185966492, -0.5831457376480103, 1.063510775566101, -0.24534209072589874, -1.3392189741134644, 1.3768857717514038, 0.02423957921564579, 0.16738227009773254, -0.43127861618995667, -0.6907987594604492, 2.080203056335449, -0.7818042039871216, -0.7094717025756836, 0.3456882834434509, -0.48306140303611755, 1.6561188697814941, -0.5073761343955994, -0.7438955903053284, 1.2104395627975464, 0.49065881967544556, -0.21896736323833466, -1.7326343059539795, 0.5116764903068542, -0.1692829728126526, 0.35007068514823914, -0.4240735173225403, -0.1192716434597969, 1.0481789112091064, -1.2165088653564453, -0.38813236355781555, -0.8866256475448608, -0.21407535672187805, -0.932370126247406, 0.47433409094810486, -1.097655177116394, -0.8806599974632263, -0.11196423321962357, -1.9364429712295532, -0.3277692198753357, 0.3405568301677704, 0.338845819234848, -0.5006113648414612, -0.255012184381485, 0.07300053536891937, -1.5795842409133911, 0.029388491064310074, -1.6469120979309082, 0.5183366537094116, -0.7217234373092651, 0.0697716549038887, -0.20206229388713837, -1.046474814414978, -0.22150050103664398, -0.10902304202318192, 0.8409233093261719, 0.6836003065109253, 0.6280531287193298, -1.3951361179351807], [-1.082185983657837, -0.9424102902412415, 1.8465754985809326, -0.4444774389266968, 0.45121052861213684, -1.3858380317687988, 0.3372955322265625, -1.5702369213104248, 0.4467965066432953, 0.765068531036377, -1.8437179327011108, -0.11490046977996826, 0.662154495716095, 0.127817302942276, 0.5148361921310425, 0.8783472776412964, 2.6475634574890137, -2.0614280700683594, -0.8515226244926453, 0.8019376397132874, 0.1703331172466278, -0.34682294726371765, 0.4765305817127228, -0.19196420907974243, -0.5719131827354431, -1.1242629289627075, -0.7595669031143188, 0.09426741302013397, 0.34591153264045715, 0.4080260097980499, 1.9602001905441284, 0.49120157957077026, -0.26155969500541687, 0.1189701110124588, 1.1779050827026367, 1.3792363405227661, -0.022413570433855057, -0.3217228949069977, -0.35017842054367065, -0.21336108446121216, -0.8817512392997742, -0.6644930243492126, -1.0036511421203613, -0.9560654163360596, 0.4938039779663086, -0.5203800797462463, -0.7455624341964722, -0.8353157043457031, -1.027769684791565, -0.4885888695716858, 1.2226859331130981, -1.1949001550674438, 0.2883649170398712, -0.3231350779533386, 0.764190137386322, -0.3785371780395508, -0.9510853290557861, -0.20480965077877045, -1.0062886476516724, 2.7098445892333984, -1.2245347499847412, 1.6757069826126099, 0.06345783919095993, 1.0490950345993042, 1.9024182558059692, 0.4538881182670593, 1.2025038003921509, 0.055321305990219116, 0.023103373125195503, 0.22011253237724304, 0.2604009807109833, 0.5733363032341003, -0.3774055242538452, 0.39728420972824097, 0.5928084850311279, -0.3295868933200836, -0.8157925605773926, -0.7812588214874268, -0.5488849878311157, -0.19273337721824646, -0.5153254270553589, 1.1778658628463745, -1.934749960899353, 0.31996890902519226, -1.9021434783935547, 1.2005658149719238, -1.0175275802612305, -2.0680065155029297, 0.8829193711280823, -1.2585084438323975, -0.26655417680740356, 2.749667167663574, 0.46479687094688416, -1.9560939073562622, -0.10864777117967606, 1.2253124713897705, 0.3436800241470337, -0.8337270617485046, -0.8454256653785706, 0.9735352396965027, -0.1374824494123459, -0.8265964984893799, -1.6776094436645508, -1.0582103729248047, 1.0620322227478027, -0.08945513516664505, 0.5983644723892212, 0.25132736563682556, -1.4987925291061401, -0.5904174447059631, 0.21276497840881348, -0.6075111031532288, -1.5507228374481201, -0.40944480895996094, 0.16543065011501312, 0.5981346964836121, 1.0596634149551392, -0.9942251443862915, 2.6940274238586426, 1.5307691097259521, 0.4280485510826111, -0.35751432180404663, -1.1489582061767578, 1.5923212766647339, -0.17715710401535034, -0.3341003954410553, 0.276529461145401, -0.1190885528922081, 0.8521397113800049], [-1.4513823986053467, 0.2497009038925171, -0.920458972454071, -1.7730416059494019, -1.2728554010391235, -0.3515629768371582, -1.2257384061813354, -1.4735547304153442, -0.06741984188556671, 0.6957469582557678, -0.043589700013399124, -1.301852822303772, -1.5743012428283691, -0.6946817636489868, -0.27490705251693726, 1.833650827407837, 0.14153185486793518, -0.5538040995597839, -0.49017050862312317, -0.029973164200782776, -0.4677000641822815, 0.5158976912498474, -0.10486330091953278, -1.081837773323059, 0.5660332441329956, 1.023927927017212, -1.5320367813110352, -0.11951007694005966, -1.226747751235962, 0.5534021258354187, 0.3528195023536682, -1.1865440607070923, -0.1122148260474205, 1.5736925601959229, -0.34586602449417114, 0.03810561075806618, 1.5665351152420044, 0.10851156711578369, 0.2348436415195465, 0.35746803879737854, 0.4656016230583191, 1.044136643409729, 0.4982278347015381, 1.7718651294708252, -0.10457786172628403, 1.4475140571594238, -0.20703698694705963, -0.30567362904548645, 0.3659341633319855, 0.42439281940460205, -0.06933121383190155, 0.748343825340271, -2.5578083992004395, -0.3395844101905823, -0.6343938708305359, 1.078297734260559, -0.6241123676300049, 1.5380303859710693, 0.30659544467926025, -0.5682671070098877, 0.3671874701976776, 0.7597838044166565, -0.3084092438220978, -0.41146525740623474, -1.5308353900909424, 1.0075372457504272, 1.4928035736083984, -0.2431807816028595, -0.6670693159103394, -0.3751647472381592, -0.3283058702945709, -1.1495124101638794, -0.3766493499279022, -0.11828401684761047, 0.3469151556491852, 0.5701514482498169, -1.1072465181350708, 1.2704535722732544, -0.215517058968544, 0.21159818768501282, 0.37479907274246216, -0.6825218200683594, -1.3840802907943726, 0.37100887298583984, 0.4967581033706665, 0.621813178062439, -1.0035932064056396, 0.09426186978816986, 0.500356912612915, 0.9287603497505188, -0.8749464154243469, 0.9594405889511108, 1.4927489757537842, 1.5686854124069214, -0.33061689138412476, 1.8930739164352417, 1.8110723495483398, -2.1757466793060303, 1.1185483932495117, 0.31517696380615234, 1.4584044218063354, -0.45238196849823, 0.1854092925786972, 0.3252931237220764, 0.485625296831131, 0.6801703572273254, -1.1758437156677246, 0.03633376210927963, 1.9250966310501099, -0.29220059514045715, -0.24011695384979248, -0.15951445698738098, 0.9030270576477051, 0.13081063330173492, 0.4372028410434723, -0.7204146385192871, 0.7344948053359985, -0.7710742354393005, -0.5809352397918701, -2.102060317993164, -0.37578126788139343, -0.003995758946985006, 1.050445556640625, 0.381769597530365, 0.28756487369537354, 0.16030776500701904, -0.6083842515945435, -0.2878769338130951, 0.3772057890892029], [0.13024123013019562, 0.2214379757642746, -0.0937722846865654, 0.3727109432220459, -0.31276875734329224, 1.1723592281341553, 1.1116547584533691, -0.8364938497543335, -0.3349667191505432, 0.6190505027770996, -0.5372425317764282, -0.008188381791114807, -0.8593245148658752, -1.5585014820098877, -1.4380416870117188, 0.26540258526802063, -1.457406759262085, 0.7114904522895813, -2.2994775772094727, -0.024193493649363518, 1.4797097444534302, 1.1317238807678223, 1.1993032693862915, 1.2298506498336792, 0.170144185423851, -0.36346951127052307, -0.37291276454925537, -1.1556676626205444, -1.1086719036102295, -0.054945070296525955, 0.09173353016376495, 0.7432278990745544, 0.8552875518798828, 0.8789969682693481, 0.6722119450569153, 0.6469495892524719, -0.6289210319519043, 0.8970537781715393, 1.5751054286956787, -0.04159106686711311, -1.5339590311050415, -0.7579535841941833, 0.7571209073066711, -0.9073001146316528, -0.8211480975151062, -0.80800461769104, -0.7827896475791931, 0.046547695994377136, -1.6920483112335205, -0.9728895425796509, 1.0484589338302612, -0.6054553985595703, 1.0978500843048096, 1.0615050792694092, 0.43482688069343567, 1.4515856504440308, 0.47616517543792725, -1.2611290216445923, 0.2560805678367615, -1.2059993743896484, -1.078096628189087, 0.5097986459732056, 0.08925464004278183, 0.4951629340648651, 1.0108168125152588, -0.14919821918010712, -0.5364022850990295, -0.07475554943084717, 0.7272352576255798, -0.9079831838607788, -0.14846408367156982, -0.4351539611816406, -0.515165388584137, -1.6098016500473022, -1.594773769378662, 0.24347719550132751, -0.08193033933639526, -0.8693153858184814, 0.0008082194835878909, 0.1377619206905365, -0.09927903860807419, 1.0993390083312988, 0.22235502302646637, -0.8417127132415771, 0.039561450481414795, 0.8224295973777771, 3.0880203247070312, 1.429854393005371, 0.5088611841201782, 1.774746060371399, 0.3859064280986786, -0.09041935205459595, 1.2827273607254028, 0.3376478850841522, -0.9241564273834229, 0.23296497762203217, 0.24520477652549744, 0.061562102288007736, 0.901470959186554, 0.2991344630718231, 0.2855031192302704, -1.232529878616333, -1.2048250436782837, 0.3322185277938843, 0.0634683147072792, 0.3741268813610077, -1.0510478019714355, -0.41499754786491394, 0.17719803750514984, 1.7774707078933716, 1.0109021663665771, -0.1311795562505722, 0.6422218680381775, -0.1300056278705597, -2.502316474914551, 1.7983384132385254, 1.4569566249847412, -1.4024922847747803, -0.6634358167648315, 0.39025402069091797, -0.07593832165002823, -2.734096050262451, 0.23460133373737335, -1.369058609008789, -0.29106369614601135, 1.821354866027832, -0.12470121681690216, 0.6597537398338318, 0.4247174561023712], [0.3386407792568207, 0.6206523180007935, -0.058715272694826126, 0.9707252979278564, 0.408929705619812, 1.2468907833099365, -0.5295174717903137, -1.1679869890213013, 0.14826051890850067, 0.798479437828064, 0.589519739151001, -1.9686752557754517, -0.36025914549827576, 0.36316242814064026, 0.5965238809585571, 0.14794737100601196, -0.1813092678785324, -0.33858758211135864, -0.4568483531475067, 0.07713901251554489, 1.1547726392745972, 1.617035150527954, 0.5085892081260681, 0.6204137802124023, -0.06645951420068741, 0.6229416131973267, 0.022860607132315636, -0.429521769285202, 0.864518940448761, 0.17749537527561188, 0.680908739566803, 1.8728152513504028, -0.2959223687648773, -1.0613592863082886, 0.8412598371505737, 0.8093846440315247, -0.7257974743843079, -0.7567146420478821, 0.13831493258476257, -2.1202266216278076, 2.8406972885131836, 3.0791592597961426, 1.7029038667678833, 3.028607130050659, -0.11583741754293442, -1.0051860809326172, 1.1146345138549805, 0.36356863379478455, 0.9418780207633972, 0.15503841638565063, -1.740230679512024, -0.8816609382629395, 0.02818240039050579, 1.9164173603057861, -0.23269353806972504, 1.0175155401229858, -0.34191277623176575, 0.05470401793718338, 0.3838275372982025, -0.6877990961074829, 1.4362194538116455, -1.4590535163879395, 0.010060339234769344, 0.016764217987656593, -0.5112671852111816, 1.3028560876846313, -0.9207126498222351, 0.5689854025840759, -0.23098896443843842, 1.0213872194290161, 1.5427788496017456, -0.8689486384391785, 1.4873732328414917, -0.6681566834449768, 0.7844390273094177, 0.2629009783267975, 1.5269949436187744, -0.4831404685974121, 1.0402872562408447, 1.5499860048294067, 0.9840519428253174, 0.5333396196365356, 0.8937007784843445, -0.4064965844154358, -0.7970117330551147, -1.0973681211471558, 0.2210274189710617, -1.719569206237793, 0.36820417642593384, 0.1912405639886856, 0.22436246275901794, 2.078996181488037, -0.30287423729896545, -2.252065420150757, 0.44027942419052124, -1.1343227624893188, 1.6269993782043457, -0.9737667441368103, 0.32555219531059265, 0.19524653255939484, -0.2474697083234787, -0.3957276940345764, -0.5687183737754822, -0.4879782199859619, -0.39330241084098816, 0.08081898093223572, 0.42685094475746155, 1.1972600221633911, 0.32249346375465393, 0.8798894882202148, 0.5729688405990601, -1.5172425508499146, -0.9227912425994873, 0.20657067000865936, 2.062809467315674, -0.5381560325622559, 0.5470231771469116, 1.4504982233047485, -0.8040534853935242, -0.17295871675014496, 0.6231315732002258, 0.6754077672958374, 1.583362340927124, 0.3100724518299103, 0.6761415600776672, 0.9826430678367615, -1.9494305849075317, -0.011343037709593773, 0.2409941852092743], [0.4195762276649475, 0.8699900507926941, 2.658154249191284, -0.07983264327049255, -0.30497804284095764, -0.8535293340682983, -1.4304367303848267, -0.7292875647544861, 1.1120935678482056, 1.0013545751571655, 0.5400606989860535, -0.8870027661323547, 2.910963535308838, -1.6913427114486694, 0.9733292460441589, 1.6778055429458618, -0.20372287929058075, 0.056843988597393036, -0.7243137359619141, -0.4820384383201599, 1.1257916688919067, -1.1598458290100098, -0.7057885527610779, 1.6958229541778564, -1.4810343980789185, 0.3924352824687958, -0.494821697473526, -0.11123143136501312, 0.6090344786643982, -0.5544682145118713, 1.0418224334716797, -1.6164042949676514, -0.3781339228153229, -0.5607647895812988, 0.34323641657829285, 0.3092939853668213, -0.1261579394340515, 0.09994050115346909, -0.32916584610939026, -1.0953574180603027, -0.40601933002471924, 2.162041664123535, -0.3440168797969818, 1.068656325340271, 0.3030245006084442, -2.061197280883789, -1.001226544380188, -0.23900140821933746, -1.2471179962158203, -1.5991342067718506, 0.8453868627548218, -0.8915512561798096, 0.4625498652458191, -1.6810685396194458, 0.3636924624443054, -1.4231845140457153, 1.3694735765457153, 0.07649961858987808, -0.5437061786651611, -0.42184653878211975, 0.33913934230804443, -0.15856219828128815, 0.18304936587810516, -1.4998961687088013, 0.15578331053256989, -0.0660368949174881, -1.0852503776550293, -1.6669131517410278, -0.5031542181968689, -1.8181475400924683, 1.3481519222259521, -0.6199080944061279, 0.7085896730422974, -0.38711076974868774, 1.2654222249984741, -0.2622660994529724, -0.36794590950012207, -1.660345435142517, -0.2562556862831116, -0.020865850150585175, 1.4174203872680664, 0.023102687671780586, 1.1392440795898438, 0.7323544025421143, -1.7944427728652954, -1.624971628189087, -0.9307794570922852, -0.6753560304641724, -0.010033874772489071, 0.5620063543319702, 0.17115619778633118, -0.011013883166015148, 0.890239953994751, -1.241140604019165, 0.4723910391330719, 0.5293659567832947, -1.0573511123657227, -1.0195072889328003, -0.12497799843549728, 1.5380553007125854, -0.5900677442550659, 0.4608364999294281, -0.16648617386817932, 0.5045823454856873, 0.5410748720169067, -0.9112576842308044, -0.8196690082550049, 1.2799603939056396, -1.0370699167251587, -1.8413898944854736, 0.4954739511013031, 0.46924343705177307, -0.3986911177635193, 1.1857964992523193, 0.2056194543838501, 0.6873734593391418, 0.1496206372976303, -1.9806994199752808, -0.6228984594345093, -0.7631124258041382, 0.3107185661792755, -0.8008681535720825, 1.1866897344589233, 0.8165276646614075, -0.2997782826423645, 1.327257513999939, 1.0860071182250977, -0.00875762663781643, -0.14131943881511688], [0.08612851798534393, -0.0660187155008316, 0.17217062413692474, -1.0446288585662842, 0.379339337348938, -0.4055364429950714, -0.6885181665420532, -0.39792197942733765, -1.1069260835647583, -0.22034572064876556, 1.3471399545669556, 2.092592716217041, 0.29495155811309814, 0.4737745523452759, 0.05220898985862732, -0.6086199879646301, 0.24227353930473328, -0.4395233690738678, -0.21624092757701874, -0.15861444175243378, -0.6987847685813904, -1.0740920305252075, 1.353285551071167, -1.2178393602371216, -1.1083475351333618, 0.09590359032154083, 0.7713590860366821, 1.1909064054489136, -1.2341283559799194, -0.55690598487854, -0.2310420274734497, 1.7115566730499268, -1.0805541276931763, -0.7793843150138855, 0.5174764394760132, 0.450539767742157, -1.311550498008728, 0.8306469917297363, -0.5685123801231384, 1.675872564315796, 0.720531165599823, -0.8891907930374146, 2.581122875213623, -0.23428820073604584, 1.3219150304794312, 0.642065167427063, 0.11951281130313873, -0.5162562727928162, -0.29739028215408325, -0.8363083004951477, -1.873029112815857, 0.6845415234565735, -1.0491816997528076, 0.5873647332191467, 0.3207891285419464, 1.3933185338974, -0.6944830417633057, -0.5810542106628418, 0.9381988644599915, 0.9043364524841309, -0.19595642387866974, 1.556557297706604, -0.6312585473060608, -0.24446885287761688, 0.6063053607940674, 0.8184888958930969, -1.4158427715301514, 1.1364809274673462, 0.6684852838516235, -0.5518431067466736, 0.917161762714386, -1.6340640783309937, -0.5958693623542786, -2.3267202377319336, 0.034263428300619125, 2.977081537246704, 0.37158218026161194, 0.057152580469846725, 1.1853150129318237, 1.195356845855713, 1.090969443321228, -0.22421424090862274, -0.39140740036964417, -0.6547753214836121, 0.9381406307220459, -0.270965039730072, -0.4558662474155426, -1.3288333415985107, -0.16527298092842102, -0.0844205692410469, 0.10247350484132767, 0.20972637832164764, -0.1913907676935196, -1.34744393825531, -1.0444904565811157, -0.32875263690948486, 1.816916823387146, -0.46391189098358154, -0.20066426694393158, -1.0175273418426514, -0.8318603038787842, 0.9049504995346069, -0.4906167685985565, 0.6160129904747009, -0.3476428687572479, -1.805044412612915, -0.42771318554878235, 0.28950780630111694, 1.1760436296463013, -0.3475407361984253, 0.18806350231170654, 0.7579610347747803, 0.5560163259506226, 0.044611506164073944, -0.34300267696380615, 1.4335148334503174, 1.4214470386505127, -1.528651237487793, 0.30420005321502686, 0.8001698851585388, 0.2324688583612442, -1.1262704133987427, -0.7306023836135864, 1.337990641593933, 0.3591479957103729, 1.2720717191696167, 2.0914199352264404, 0.10950751602649689, -0.21993613243103027], [-1.0059736967086792, -0.4907093346118927, -1.359067678451538, -0.9189611673355103, 0.21365103125572205, -0.7962686419487, -0.6713839173316956, -0.10828053951263428, 2.020327568054199, 0.3361361622810364, 0.9609488248825073, 0.14279748499393463, 1.6075643301010132, 1.0239542722702026, -0.9415973424911499, -0.3719121813774109, -1.0842037200927734, 0.4207741320133209, 0.5036935806274414, 0.5906027555465698, 1.4568016529083252, 1.2725635766983032, 0.5347744226455688, 0.8564718961715698, -1.1532517671585083, 0.024402063339948654, 0.7115313410758972, 0.0021299233194440603, -0.547038733959198, -0.18414293229579926, 0.9538838863372803, -0.5655473470687866, 1.9228012561798096, -2.7665019035339355, 0.5113009214401245, -0.20676358044147491, 2.021082878112793, -1.3334648609161377, 0.0011445315321907401, 0.7901878356933594, 0.6716938614845276, -2.4324965476989746, -1.0319265127182007, -0.5635491013526917, 0.2914562523365021, 1.6822044849395752, 0.29969522356987, -0.048303790390491486, 1.3494881391525269, 1.3564244508743286, -1.0365663766860962, 0.40194863080978394, -0.774810791015625, 0.36840009689331055, 0.22670863568782806, -0.5120795965194702, -0.6083456873893738, 0.8123752474784851, 0.06439702957868576, -0.4220152795314789, -0.7566977739334106, 0.11174828559160233, -1.1937150955200195, 0.2859603464603424, -1.1764897108078003, 0.33255523443222046, -0.5186304450035095, 0.9148897528648376, -0.3356446921825409, 0.2322305589914322, -0.765609622001648, -0.8139595985412598, -1.230191707611084, 0.4722817838191986, 0.4819960296154022, 0.7156386375427246, -0.5277159810066223, 0.5046498775482178, -0.3879300653934479, -1.2741737365722656, -0.8614715337753296, -1.3174264430999756, 0.562891960144043, 3.0137367248535156, -0.2636747658252716, -0.07031898200511932, -1.179701805114746, -0.5636202692985535, 0.7782630324363708, 1.7858495712280273, 1.4494422674179077, -0.4299931228160858, 1.4699751138687134, -0.2811359763145447, 1.3669124841690063, 1.77047598361969, 1.2883447408676147, 1.5444895029067993, 1.0582655668258667, -0.6288110017776489, -0.5744020342826843, 1.0387834310531616, 2.1361587047576904, 1.2765841484069824, -0.49841517210006714, 0.5356175303459167, -0.49588093161582947, 0.26300427317619324, 0.36478278040885925, -0.26341989636421204, 2.1006147861480713, 0.2182343304157257, 0.9475046396255493, 0.8151319026947021, -1.9071800708770752, -0.43241336941719055, -2.811859130859375, 0.3939533829689026, -0.2260877788066864, 1.7834277153015137, -0.13281318545341492, -0.748336911201477, -0.3172479569911957, 1.2142086029052734, -1.1532210111618042, 1.3064053058624268, -0.13359318673610687, -1.3591623306274414, -0.6783481240272522], [1.8341996669769287, -0.4348960816860199, 2.3080668449401855, -0.2680932283401489, 1.2803250551223755, 0.1347353607416153, 0.7566079497337341, 0.1863432079553604, -1.1666940450668335, 0.3263237774372101, -2.544740915298462, 2.393984079360962, -0.13085179030895233, -2.1335368156433105, -0.2168755978345871, -0.6668843030929565, -2.826967477798462, -0.10383426398038864, -1.3521422147750854, -0.3614959120750427, -0.8713887929916382, -1.0233293771743774, 1.7540836334228516, 1.1225714683532715, 0.20333977043628693, -0.9399980902671814, 0.6793298125267029, 0.22145996987819672, 0.04634057730436325, -1.3801525831222534, 0.9414182901382446, 0.15922066569328308, 0.5918103456497192, 1.2532854080200195, -0.7030812501907349, 1.297441840171814, -0.3958395719528198, 1.4573478698730469, -1.18672513961792, -0.7836394906044006, -0.2186104655265808, 0.686857283115387, -0.3001025915145874, 0.15177905559539795, -0.11708034574985504, 0.4080701172351837, 1.567137360572815, -0.04316721111536026, 0.6489824056625366, -0.6515305638313293, 1.8956300020217896, 0.6122718453407288, 0.11541148275136948, -0.019788134843111038, -0.026609912514686584, -0.2470349818468094, 1.3613176345825195, 1.0629175901412964, 0.5267775654792786, -2.0393948554992676, 0.16644351184368134, -0.4819854497909546, -0.01464875042438507, -0.10875531286001205, -0.04596921056509018, 0.774034857749939, -0.024372214451432228, 0.6255005598068237, -0.8191691040992737, -1.7493306398391724, -0.5871255993843079, 0.40985217690467834, -1.0627347230911255, 1.0363467931747437, -2.2206497192382812, -0.1565505862236023, -0.31455543637275696, -0.2239094227552414, -1.1247888803482056, -0.3095751702785492, -0.6624932885169983, 1.0945013761520386, -0.3140333890914917, 0.8890457153320312, -1.5899757146835327, -0.005983331706374884, -1.087107539176941, 0.49581146240234375, -0.12161651253700256, 1.3231291770935059, 0.8368452191352844, -0.5600073933601379, 0.2302112728357315, 0.5202339887619019, -2.032231569290161, -0.5009204149246216, -0.7348042726516724, -1.213236927986145, -0.3770286440849304, -0.7351582050323486, 0.5364938974380493, 1.1959083080291748, -1.5146123170852661, 0.1734868437051773, -0.9426406025886536, 0.365867555141449, 0.23790299892425537, 0.6854790449142456, 0.8183815479278564, 2.166893482208252, 0.5418881177902222, -1.307413935661316, 0.7941843271255493, -0.2446695864200592, -0.5991734862327576, 1.7646386623382568, -0.391679972410202, 0.5893142223358154, -0.4691702723503113, -1.6066443920135498, 0.6500149369239807, -0.25807181000709534, 0.8815120458602905, -1.328895092010498, 0.39406320452690125, -0.5407630801200867, -2.2886617183685303, -0.18886367976665497, 0.41773679852485657], [-0.10038813203573227, -0.8697022199630737, 0.8292377591133118, -0.37514176964759827, 0.5879306793212891, -0.9157698750495911, -0.09268245846033096, -0.46855628490448, 0.026998396962881088, 0.3625175952911377, -0.9918429255485535, -0.5233910083770752, 0.6121231317520142, -0.3715366721153259, -1.0093132257461548, -0.19662916660308838, -0.5647273063659668, -0.7755943536758423, -0.04699232056736946, -0.24212433397769928, 1.1984410285949707, 2.515307664871216, 1.23459792137146, -0.1899014264345169, -1.6071608066558838, 0.08965516090393066, 0.2551443874835968, 0.09102848917245865, 1.0724090337753296, 1.332623839378357, 0.2703050971031189, 0.53902667760849, -0.9713485240936279, 0.21032880246639252, -0.3311653733253479, -1.3444267511367798, 1.0591988563537598, -0.013180476613342762, 1.4897000789642334, 0.41465502977371216, 1.0815523862838745, 0.5601242780685425, 1.3918951749801636, 0.16317932307720184, 0.373887300491333, 0.33431848883628845, 0.6899492740631104, 0.11182024329900742, 0.010915877297520638, 0.3955265283584595, -0.21606455743312836, 0.2638387382030487, 0.5724585056304932, 0.3882671594619751, -0.17407548427581787, -0.06498897075653076, -0.14102715253829956, 0.07483715564012527, 1.4352260828018188, 0.38183289766311646, 1.7126834392547607, 1.6160438060760498, 0.4825815260410309, -0.19001083076000214, 0.2669825851917267, 1.0153080224990845, -0.011467348784208298, 0.7416834235191345, -0.031031440943479538, 0.4668278992176056, 1.4157567024230957, -0.09235498309135437, -0.3949041962623596, 0.9517754912376404, -1.0003143548965454, 0.6671680808067322, -0.06989621371030807, -0.3726598918437958, -0.020090775564312935, -1.4881480932235718, 3.003873825073242, -1.935504674911499, 0.7733691930770874, 0.5889822244644165, 0.025373773649334908, -0.4624718129634857, 0.39881831407546997, -0.27947255969047546, 0.300279438495636, 0.24658629298210144, -0.26444175839424133, -0.31035706400871277, -0.275312602519989, -1.81745183467865, 0.6975458860397339, -1.2010728120803833, -2.4796576499938965, -0.012715620920062065, -0.7391995191574097, -1.0023964643478394, 0.36889544129371643, 0.18440867960453033, 0.7303456664085388, -1.046975016593933, 1.0732176303863525, 0.8379722237586975, -0.3080962598323822, 0.5271978974342346, -0.4791887104511261, 0.7511031031608582, 0.22370649874210358, -0.7599749565124512, -0.6813662052154541, -2.1497039794921875, -0.4373365044593811, -1.213220477104187, 0.5542660355567932, 0.48015549778938293, 1.042712926864624, 0.4553627371788025, 0.8220974802970886, -2.213242769241333, 0.9482403993606567, -0.6109245419502258, -0.6646504998207092, 2.2147774696350098, 1.264627456665039, -0.3072268068790436, 1.6657778024673462], [-0.057248685508966446, 1.5363829135894775, 1.1232481002807617, 1.6491690874099731, -1.737268328666687, -0.23493188619613647, 0.6367076635360718, 0.17288756370544434, 0.23391728103160858, 1.0494444370269775, 0.4707576334476471, 0.15969552099704742, 0.016569653525948524, 0.48713165521621704, -0.5147531628608704, 1.0698163509368896, -1.7010284662246704, -0.7926300168037415, -0.32339271903038025, -1.6703201532363892, 1.2091881036758423, 0.40412187576293945, -0.669747531414032, 0.18648536503314972, 0.7716001272201538, 1.7273658514022827, 0.4734134376049042, 0.5074659585952759, -0.3046829104423523, -0.8988263010978699, 1.3321175575256348, -0.736276388168335, -0.7067839503288269, -0.7900526523590088, 1.3534469604492188, -1.945641040802002, 0.17255984246730804, -0.6960881948471069, -0.7162862420082092, -2.87650990486145, 1.6336971521377563, -1.0016217231750488, -1.2291936874389648, 1.6271568536758423, -1.0386550426483154, 2.239832878112793, -0.5882629752159119, 0.5806494355201721, 0.35985222458839417, 1.2421061992645264, -1.0195339918136597, -1.1398131847381592, -0.4259364902973175, -0.9203904867172241, 0.2006167769432068, 1.0832825899124146, -0.32182326912879944, 0.8958982229232788, -0.7961729168891907, -0.2404419332742691, 1.177743911743164, 0.202143132686615, -0.8695741891860962, 2.152204990386963, -0.6850991249084473, 0.20600607991218567, -1.8904601335525513, -0.5186929106712341, 1.3326318264007568, -1.5790599584579468, -0.14976336061954498, 0.7283941507339478, -0.5450518727302551, 1.68537175655365, 1.2204272747039795, 0.3022729456424713, 1.476060390472412, -0.5035554766654968, -0.46727049350738525, -1.77052903175354, 0.30505090951919556, 1.056758999824524, 0.2730908989906311, -0.19029855728149414, -0.5297142267227173, 0.33891937136650085, -0.8496343493461609, -0.7648908495903015, 2.525590658187866, 1.4890196323394775, 0.20916418731212616, 0.6378818154335022, -1.2393803596496582, -0.3794684410095215, -2.0677742958068848, -0.06986460834741592, 1.353898525238037, 0.20066289603710175, -0.619723379611969, 1.1824365854263306, 0.9258504509925842, 2.3076462745666504, 1.3133662939071655, 0.6109983325004578, 0.05298375338315964, 1.9080308675765991, 0.151121586561203, -1.8090112209320068, 0.31407490372657776, 0.08910898864269257, -0.7198317050933838, -0.37642380595207214, 0.3704436719417572, -0.922522246837616, 0.8002020716667175, -1.107863187789917, -1.690722107887268, 1.4725568294525146, -0.5731745958328247, 1.7665882110595703, -0.06466304510831833, 1.4105595350265503, 0.40717336535453796, -0.339489221572876, 2.2276580333709717, -0.10924958437681198, 1.8644832372665405, -0.03519435226917267, 1.648932933807373], [-0.7897641062736511, -1.0802321434020996, -0.43360424041748047, -0.11628839373588562, -1.0465443134307861, 1.6529721021652222, 0.46353447437286377, -0.7602996826171875, 0.25445422530174255, 0.5297901034355164, 0.8519318699836731, 0.587901771068573, -1.1816145181655884, 2.0129573345184326, -1.303537368774414, 0.733165442943573, 0.7475775480270386, -2.0185964107513428, -0.3564411699771881, 0.9139187932014465, -0.4437251687049866, 0.7840985655784607, 0.8981145620346069, -0.34724608063697815, 1.4684998989105225, -0.18317480385303497, 1.344042420387268, 0.4900417625904083, 0.2299443632364273, -1.9369103908538818, -2.087160587310791, -1.9368864297866821, -0.5010274052619934, 0.3716413974761963, -1.771375298500061, -0.7793377637863159, 1.243706464767456, -0.7630922794342041, 0.645692765712738, 0.10008446872234344, -1.2386751174926758, 1.0994120836257935, -0.2181793600320816, -1.019474983215332, 1.189415693283081, 0.21426787972450256, -0.8733689785003662, -1.1450605392456055, -1.3549365997314453, -0.0431022122502327, 1.0127569437026978, 1.178452730178833, -0.8722936511039734, 0.8520422577857971, -0.8748921155929565, -0.6986097097396851, -0.34066063165664673, 0.561873197555542, -0.5313334465026855, -0.7972322106361389, -0.10037456452846527, 1.265681505203247, 0.5351012945175171, -0.9826987385749817, -0.5038838386535645, 0.0552787110209465, -0.882122278213501, 1.27120840549469, 0.9227281212806702, -0.011946640908718109, -0.17412149906158447, -0.8198930621147156, 0.19687478244304657, -0.6880378127098083, -0.34457719326019287, -0.5118717551231384, 0.7123326659202576, 0.12379519641399384, -1.2063175439834595, -0.6852505207061768, -0.04252270236611366, 0.8103055357933044, 1.1163289546966553, 0.22421345114707947, 0.7198963761329651, -2.800123691558838, 0.022005610167980194, -2.4154372215270996, -1.5286263227462769, -0.600554347038269, -1.045161485671997, -0.1312437206506729, 0.68455570936203, -0.44903135299682617, -1.2549445629119873, 1.1091874837875366, 0.1091204285621643, 0.4097629487514496, 0.7205561399459839, 0.781777560710907, 0.6576374173164368, 1.18958580493927, -1.0898466110229492, -0.4110681116580963, -0.19907744228839874, -0.6208105683326721, -1.1240671873092651, -0.27703216671943665, -0.6420148015022278, -0.5375215411186218, 1.0819337368011475, 1.000404715538025, -1.656562328338623, -0.19730086624622345, -0.06760269403457642, 0.24720092117786407, 0.11017656326293945, -0.7779510021209717, -0.24262070655822754, 2.059525489807129, -0.734607458114624, -0.3864900767803192, -1.0309710502624512, -0.4521825611591339, 0.5284727215766907, -2.0376811027526855, 0.5107617378234863, 0.38972002267837524, -0.407839834690094], [0.42706313729286194, 0.8751109838485718, -0.39772382378578186, 0.6559043526649475, -0.0645935982465744, -0.31733593344688416, -1.155380368232727, -0.701569139957428, 1.153641700744629, 0.5650240182876587, -0.34090477228164673, 0.919921875, 2.572859525680542, 1.0916661024093628, 0.86434006690979, 0.25657421350479126, 1.0971776247024536, -0.5520998239517212, -0.07136674970388412, -0.6034749150276184, -0.38753002882003784, 1.0179518461227417, 1.42716383934021, 0.6086219549179077, 1.3183177709579468, -0.7381532192230225, 0.7695586681365967, -0.21389587223529816, -1.5918368101119995, 0.44610464572906494, -2.4496641159057617, 0.41434744000434875, -0.7931171655654907, -0.1478750854730606, -0.663470447063446, -1.583631992340088, 0.19002488255500793, -0.16346973180770874, 1.247316837310791, -2.2568483352661133, -0.2346305549144745, -0.8492531180381775, 0.7582367062568665, 1.4294742345809937, -0.03489099442958832, -2.169005870819092, -0.25485873222351074, -1.6389906406402588, -0.12889905273914337, -1.2382535934448242, 1.4123626947402954, 1.7839412689208984, -0.49754348397254944, 0.36466750502586365, -1.2608951330184937, 2.63250732421875, -0.4795300364494324, 2.0193405151367188, 0.3613525629043579, 1.8150111436843872, -0.5735407471656799, 1.0985865592956543, -1.5664743185043335, 1.2252726554870605, 0.9478500485420227, -1.221532940864563, 0.15276828408241272, 0.2933918237686157, 0.7632102966308594, -0.040152888745069504, 0.6713050603866577, -0.9358380436897278, -0.018587257713079453, 0.28557270765304565, -1.2656264305114746, 1.066825032234192, -0.4225837290287018, 0.7336817383766174, 0.6977660655975342, -0.04069003462791443, 0.8464000225067139, -1.703338861465454, 1.8631649017333984, -1.5962679386138916, 0.3320225179195404, -1.3182162046432495, -0.4085395038127899, -0.32413622736930847, 1.2992762327194214, -0.3690459728240967, -0.14785133302211761, 0.02804778516292572, 0.6946106553077698, -0.9747853875160217, -0.7831906676292419, -0.797178328037262, 0.25423678755760193, 0.17226222157478333, 2.131070852279663, -0.010570853017270565, 0.178408682346344, -0.36948075890541077, 0.9126443862915039, -0.7233282327651978, 1.004827618598938, 0.05488597974181175, 1.2393230199813843, 1.1131335496902466, -0.44756293296813965, -0.3550542891025543, 0.26424309611320496, 1.0638340711593628, -1.0059411525726318, -0.9302452802658081, 0.6841268539428711, -2.4007861614227295, -1.2175253629684448, 1.211190104484558, 0.1555507928133011, 0.8284635543823242, 0.705592155456543, -0.7551760077476501, -0.33842775225639343, -0.04900773987174034, -0.2745228111743927, 0.28589656949043274, 1.1726610660552979, -0.9739620685577393, -0.5481630563735962], [0.2147074192762375, -0.7354692220687866, -0.8578446507453918, 0.06593595445156097, -1.1058590412139893, 0.48958057165145874, -0.879436731338501, 1.83548903465271, 2.9040591716766357, -0.3726460635662079, -0.9632466435432434, 0.17720061540603638, 1.8910012245178223, -1.5764331817626953, 1.0897883176803589, 0.0120288310572505, -0.43069198727607727, -0.2718546986579895, -2.4030351638793945, 1.0559206008911133, 0.7023879289627075, 1.0143980979919434, -0.8679433465003967, 1.7877496480941772, 0.3668568730354309, 0.5339342951774597, 0.18232925236225128, 1.4168791770935059, -0.9574148654937744, -0.832865834236145, 1.134941577911377, -0.06459123641252518, 0.004824731964617968, -1.310412883758545, -1.1106209754943848, 1.263656735420227, 0.01194382831454277, 0.3838614523410797, 0.8279445171356201, 0.8644120097160339, -0.425141841173172, 2.512988805770874, 0.24722808599472046, -1.3724932670593262, 1.7781485319137573, 0.6951730251312256, -0.20919859409332275, 0.28097808361053467, -1.0286600589752197, 0.8028885722160339, 0.24379320442676544, -0.9928825497627258, 0.4147898554801941, 0.1438242346048355, -0.003346616169437766, -0.3276178538799286, -0.8817983269691467, 0.9179471731185913, -0.5137395262718201, 1.455130696296692, -0.4371718764305115, 0.004382604733109474, 0.5300882458686829, 2.991755962371826, -0.5505619645118713, 0.6499606966972351, -1.373595952987671, 0.017806729301810265, -0.18790441751480103, 0.22868360579013824, -0.9103224873542786, 0.046245139092206955, 0.44379377365112305, -0.8623515367507935, 0.18010376393795013, 0.25427117943763733, 0.045950304716825485, 0.25101587176322937, 0.5447965860366821, -0.573650598526001, -0.9532123804092407, -0.7374459505081177, 0.43890056014060974, 0.3019302189350128, -0.6468841433525085, -0.5988072752952576, 0.22398339211940765, 0.8643182516098022, -0.7224527597427368, 0.5318337678909302, 1.0583422183990479, 0.8215528130531311, -0.5277717709541321, -0.72043377161026, -1.739628553390503, -1.640552282333374, 2.130159616470337, 0.3404465615749359, -1.0487428903579712, 1.241036057472229, -0.6601078510284424, -0.7609973549842834, -0.35274532437324524, 0.3597968518733978, 0.39782533049583435, -0.10632665455341339, 0.8565207123756409, 1.9893180131912231, -2.0723719596862793, -0.1996731460094452, 0.5372782945632935, -1.0402127504348755, 0.21761320531368256, 0.8138562440872192, -0.019049935042858124, 1.6525713205337524, -1.3028135299682617, 1.0801985263824463, 0.8421000242233276, 1.2868107557296753, -0.9070578813552856, -0.27122801542282104, -0.46811291575431824, 0.3212156295776367, 0.3149549067020416, -0.8614141941070557, -0.37033867835998535, -0.48526787757873535, -0.12389454245567322], [-1.1824538707733154, 0.2951519191265106, -0.09649576246738434, 1.194075107574463, 0.04874749481678009, 0.6617109775543213, -0.15034814178943634, 1.0939297676086426, 1.1378196477890015, -0.3323991298675537, 1.2945080995559692, 1.613450288772583, -0.349764347076416, 1.145485520362854, 1.8976507186889648, 0.6132795214653015, 1.124260425567627, 0.3219095766544342, -0.21052362024784088, 1.3016685247421265, -0.2715173065662384, 1.5642743110656738, -0.3424834907054901, -0.6451728343963623, 0.5551286339759827, -0.2986973226070404, -0.9994370937347412, -0.7870882153511047, 0.4858546853065491, 0.3517538607120514, 0.40721604228019714, -0.6444966793060303, -0.7598617672920227, 1.2252377271652222, 1.000996708869934, -1.556601643562317, 0.148609921336174, 1.0444345474243164, -0.6487054824829102, -0.476277232170105, 1.2260054349899292, 0.24505718052387238, 1.6721985340118408, -0.8555727005004883, 0.6394900679588318, -1.4982315301895142, -0.7681615352630615, 1.104707956314087, -2.4569175243377686, 1.2762483358383179, -0.3847409188747406, 0.8965805768966675, 0.38941720128059387, 1.3170520067214966, 1.3621338605880737, 0.6329379081726074, 0.846265971660614, 0.5149214863777161, 0.07271438837051392, 0.7616966962814331, -0.4344538748264313, -1.0140923261642456, 0.28401869535446167, -0.7304298877716064, 0.22120699286460876, -0.2943086326122284, -0.3288072645664215, 0.013505322858691216, -0.36302217841148376, 0.33137696981430054, -0.28316009044647217, 0.2513957619667053, 0.6150739789009094, -0.022555045783519745, 2.1150546073913574, 1.7011700868606567, 1.1427206993103027, 0.28265345096588135, 0.9292506575584412, -0.2697061598300934, 0.20570656657218933, 1.0420212745666504, 0.5167417526245117, -0.5558986663818359, 2.3112564086914062, 0.7280042767524719, -0.3350948393344879, 1.1905046701431274, -1.9819591045379639, -0.8000746965408325, 0.4478090703487396, 0.6543250679969788, 0.05194888636469841, -0.38374820351600647, -0.20272083580493927, -0.3882140815258026, 0.16262301802635193, 2.3762314319610596, -1.5498242378234863, 1.3062348365783691, -0.4298883378505707, -0.8980224132537842, -0.17810791730880737, -0.5038550496101379, 1.0595316886901855, 0.6871575713157654, 2.38843035697937, -0.2820992171764374, -0.47628897428512573, -0.27650541067123413, 0.799451470375061, 0.280514657497406, 0.22552506625652313, 0.31279194355010986, -2.1147706508636475, 0.4652855098247528, -0.006494017783552408, -0.0886460468173027, -0.5884304642677307, -1.4824163913726807, 1.353001356124878, 1.4358885288238525, 0.2171023190021515, -0.14250972867012024, -0.8187874555587769, -0.20126253366470337, 2.855360507965088, 0.2650838792324066, -0.964509904384613], [3.076032876968384, -0.3454359471797943, 1.4337033033370972, -0.9280519485473633, 0.08164563030004501, 0.909298300743103, 0.451063334941864, -1.1262240409851074, 0.519037663936615, -0.2719082236289978, -0.6282755732536316, -0.1847015619277954, -0.4953104555606842, 1.509757399559021, 0.4753413200378418, 0.3200106620788574, 0.5264191627502441, 0.8788012266159058, -0.9513016939163208, -0.1581825166940689, 0.22652123868465424, -1.3888819217681885, 0.5812544822692871, -0.2645370662212372, 0.4076610803604126, 1.1030278205871582, -0.5355431437492371, 0.25540676712989807, -0.7566423416137695, -0.6648052930831909, 0.9563599824905396, -0.05358736217021942, 0.8968787789344788, 0.1318558156490326, 1.5294069051742554, -1.4518117904663086, -0.23222048580646515, 1.4451830387115479, -1.6034773588180542, 0.1596360057592392, -1.093326210975647, -1.679843544960022, -0.4700351655483246, 0.972135603427887, -1.7479127645492554, -0.13286149501800537, -1.9632333517074585, 0.3504694402217865, 0.16209599375724792, -0.384067565202713, -0.7908885478973389, -0.2774967551231384, 0.25162845849990845, 1.0426799058914185, -0.5674569606781006, 1.074031949043274, 0.28483378887176514, -0.058995675295591354, -0.9383604526519775, 1.354731559753418, 1.284409523010254, 0.3501843214035034, 0.723253071308136, 0.4026384949684143, 0.7040375471115112, -0.6039159297943115, 0.317555695772171, -0.992193341255188, -1.5083427429199219, -0.539904773235321, -0.7419880628585815, -0.09707728028297424, 0.5031687021255493, 0.1857689619064331, -0.40634986758232117, -0.4902752935886383, -0.5539007186889648, -0.9367712736129761, 0.9009928703308105, -0.738215982913971, 2.9129021167755127, 0.2765524685382843, -1.390140414237976, 0.06923390179872513, 2.1182878017425537, -1.1590816974639893, 0.17306022346019745, -0.043349772691726685, 0.4599105417728424, 0.3038167953491211, 1.9118342399597168, -0.9076297283172607, -1.7743600606918335, -1.1921013593673706, -0.28152790665626526, 1.46576988697052, 0.9198941588401794, -0.3560779094696045, -1.0679861307144165, -0.9379239678382874, 0.16524241864681244, -1.377885103225708, 1.5687435865402222, -1.3520129919052124, 0.4344760775566101, 1.0942038297653198, 2.598299980163574, 2.513291835784912, 0.5113791227340698, -2.374612808227539, 0.8386577367782593, -0.26773399114608765, -0.9608923196792603, -0.523054301738739, -1.1450170278549194, 2.101879358291626, -1.1850087642669678, -1.9928028583526611, -0.5917890667915344, -0.7070689797401428, -0.24834713339805603, 0.22151994705200195, -1.5399514436721802, 1.7513872385025024, -2.4808509349823, 0.9839553236961365, 1.7670073509216309, 0.6079627275466919, -0.665350079536438], [-1.150599479675293, -0.2729872465133667, -0.5679342150688171, -0.7216972708702087, -0.18991802632808685, -1.9750211238861084, -1.2966785430908203, -0.044306568801403046, -0.428849458694458, 0.7149755358695984, 0.11924149096012115, 1.047033667564392, -0.9212889075279236, -0.7865849137306213, -1.2095530033111572, 1.268532156944275, -0.2407497763633728, -1.049379587173462, 0.4653657078742981, 0.3848903179168701, -0.16833744943141937, -0.23652683198451996, 0.8630415797233582, -1.2245521545410156, 0.13928334414958954, 0.3179446756839752, 0.6809260845184326, -0.2063513845205307, 0.004041378386318684, -1.282555341720581, 0.09604198485612869, 0.024183420464396477, -1.5677740573883057, 0.1769041121006012, -0.6354857087135315, 0.720413863658905, 0.21860840916633606, 0.6962919235229492, 1.0056853294372559, -0.20526237785816193, -0.5925263166427612, 0.596469521522522, 1.4961607456207275, -0.33750417828559875, -0.5649206042289734, -0.5514343976974487, 0.2537744641304016, 1.3369693756103516, 1.4262866973876953, 0.18440884351730347, -0.03531952202320099, 0.13175828754901886, -0.4379294514656067, -1.346010446548462, 1.0850178003311157, -0.9460023641586304, -0.8930643796920776, 0.2625928223133087, 0.5741276741027832, 1.3890882730484009, 0.86834716796875, 0.8602345585823059, -0.46906372904777527, 1.159005045890808, -0.9638922214508057, 1.6479661464691162, 0.7356235384941101, -0.3334917426109314, -0.4325600862503052, 0.04935632646083832, -0.7315624356269836, -0.09806250035762787, -1.9758172035217285, 0.04740076884627342, 1.9517322778701782, 1.4354289770126343, 0.056323010474443436, 0.5140171647071838, 0.019533835351467133, -0.37380003929138184, -0.4228496551513672, 0.4165033996105194, -1.7136391401290894, -0.13755397498607635, -1.2025270462036133, 0.21926631033420563, 1.2800973653793335, 0.41844895482063293, -0.04765155166387558, 0.5109588503837585, 0.1503603458404541, -0.8060871958732605, 0.49856996536254883, -1.3278403282165527, 1.1910477876663208, -0.08457053452730179, 0.8508905172348022, -0.2533406913280487, -0.4887721836566925, -0.16850346326828003, 1.9019132852554321, -0.6779632568359375, 0.6381135582923889, -1.5058367252349854, 0.16986660659313202, 0.3273369371891022, -0.5041916966438293, -0.37968000769615173, 1.2710274457931519, -0.12907466292381287, 2.003563404083252, -0.11581946164369583, 0.19565722346305847, -1.3951128721237183, -0.6851815581321716, -1.1538655757904053, -0.7370181679725647, 1.6084951162338257, -0.6326133608818054, 1.3490040302276611, -0.6878612041473389, 0.5460529923439026, 1.4391236305236816, 0.9809336066246033, -0.1488315761089325, -0.9802032113075256, 0.4148838222026825, 0.380494087934494, 0.22215476632118225], [-0.9130300879478455, 1.3950575590133667, 0.028742294758558273, -0.1745840609073639, -0.7920523881912231, 2.2951831817626953, 1.8692330121994019, -0.05792386829853058, -0.7267305254936218, 1.1770473718643188, -0.49992650747299194, 0.5781390070915222, -1.943590760231018, 0.981702446937561, 0.4532153606414795, -1.748616099357605, 1.1587227582931519, 1.7762022018432617, -2.1982977390289307, 2.178182601928711, 0.4437550902366638, -0.2911163568496704, 0.8719648122787476, 0.19756439328193665, 0.6319276690483093, 1.1516081094741821, -1.2749930620193481, 0.6787444353103638, 0.06525063514709473, 1.1330265998840332, -2.135457754135132, -1.498486876487732, 0.591687798500061, 0.08612848073244095, -0.8099386692047119, -1.773681640625, -1.3165137767791748, 0.9811765551567078, -0.46926137804985046, -0.6831347942352295, -0.792140543460846, 1.559350848197937, -1.186018705368042, 0.11015792936086655, 0.4525764584541321, -0.3815930187702179, 0.6507478952407837, -1.3762047290802002, 0.6597053408622742, 1.0461686849594116, 1.2428686618804932, 1.0253204107284546, 0.2476671189069748, -0.42714741826057434, 0.8464611172676086, 0.7794649600982666, 0.9400680065155029, -1.8651552200317383, -0.10631322860717773, -0.572563111782074, 0.9887130260467529, -0.5187403559684753, 0.10652775317430496, -0.9009851217269897, 0.07105277478694916, -0.44133052229881287, -1.145198941230774, 0.21060600876808167, 0.1979803442955017, 0.25393369793891907, 0.3184616267681122, 0.8191659450531006, -0.1687660664319992, 0.8452788591384888, 0.3256760537624359, 0.6723438501358032, -0.6718891859054565, 0.760751485824585, -1.2835205793380737, -0.22685669362545013, -0.34076836705207825, 0.40725448727607727, -0.39682674407958984, 0.5494717955589294, 0.6670727729797363, -2.044159173965454, 0.846829891204834, -1.9860303401947021, 1.6459381580352783, 0.1563367396593094, 0.17932672798633575, -1.0750428438186646, -0.983942985534668, -0.6747239232063293, -1.164520025253296, 1.1852130889892578, -1.2592158317565918, 1.2589565515518188, -0.9076547026634216, -0.02449594810605049, -0.9872149229049683, -1.7482473850250244, -0.7023333311080933, -1.1472378969192505, 0.8762900233268738, -0.8672803044319153, -0.554122805595398, 0.6297304034233093, -0.6967030167579651, -1.5749480724334717, 0.2572340965270996, 1.4619252681732178, -0.8664150238037109, -0.18267010152339935, 0.2638307213783264, 0.03630302846431732, 0.1392381489276886, 0.3862365782260895, -0.3206075429916382, -0.8250583410263062, 0.051426324993371964, -1.6814223527908325, -0.3758304715156555, 0.8694697618484497, 2.0474467277526855, -0.6463538408279419, 0.28147000074386597, -1.9447286128997803, -0.6858855485916138], [-1.488808274269104, -1.7283353805541992, 1.512380599975586, -0.44615885615348816, -0.9435609579086304, -0.6767255067825317, 0.00907087605446577, 0.45327630639076233, -1.7074410915374756, -1.2832088470458984, 1.1443212032318115, 0.18448947370052338, 0.4455392062664032, 0.14651907980442047, -1.2085816860198975, 0.11039368063211441, 0.40418004989624023, -0.3481154441833496, -0.35194188356399536, -1.5050902366638184, 0.563143253326416, -0.629907488822937, -0.3561270236968994, 1.7751144170761108, -0.2611778676509857, -1.1656241416931152, -0.945663332939148, 1.2887969017028809, -1.2204270362854004, 1.6350237131118774, 1.035399317741394, -1.6481908559799194, -0.36993977427482605, -0.3239973485469818, 0.03063107840716839, 0.019710788503289223, 1.7284445762634277, 1.9012371301651, -0.7837475538253784, -0.5773579478263855, 0.5433146953582764, -1.0517650842666626, 0.6293184161186218, 0.7023221254348755, 1.3263683319091797, -1.1180938482284546, -0.6267284154891968, 0.7506478428840637, -1.7755701541900635, -0.08638528734445572, -1.408880591392517, 0.059993304312229156, 1.2947938442230225, 0.6295691132545471, 0.02063537761569023, 0.424127459526062, 0.3575012981891632, 0.4458475708961487, -0.03798183798789978, -1.3452186584472656, 0.5469778180122375, -0.9368213415145874, -0.4101312756538391, -1.6070268154144287, 0.39551976323127747, 0.9224032163619995, -0.8730909824371338, 1.177852749824524, -0.07541245967149734, -0.593069851398468, 1.7936118841171265, -0.5531368255615234, 1.1292706727981567, 2.47163462638855, -1.0367913246154785, -0.8705191612243652, -0.6776748299598694, -0.9626508355140686, -1.581356167793274, -0.4178299009799957, -0.11800320446491241, 0.525725245475769, -0.41546690464019775, 0.7360829710960388, 0.8395509719848633, -0.655048668384552, 0.8502141237258911, -0.5053684711456299, 0.21898500621318817, 0.1705099493265152, 1.1900081634521484, -0.11499199271202087, -0.6017228364944458, 0.3746795952320099, -0.30058541893959045, 0.17330080270767212, -0.14519736170768738, 2.4061594009399414, -2.123450994491577, -1.0170780420303345, 0.43753644824028015, 1.062938928604126, -0.8173525333404541, 0.12624193727970123, -0.9971169829368591, -0.9730075597763062, 0.08113428950309753, 1.5645756721496582, -1.1148487329483032, 1.0953360795974731, 2.4634525775909424, -0.23485593497753143, 2.806417226791382, -0.3425329029560089, -0.36481910943984985, 0.33991971611976624, -0.9705999493598938, -0.6261224746704102, 2.1225717067718506, -0.2156398743391037, 0.28542524576187134, -1.950847864151001, 0.3774160146713257, 0.6187947988510132, 1.159809947013855, 0.4201544225215912, 0.6887982487678528, 0.850163459777832, -0.059719305485486984], [-1.0194569826126099, -0.44756820797920227, -1.1640046834945679, -1.7051653861999512, 0.451326847076416, -1.3161611557006836, 0.12755146622657776, 0.21099747717380524, -0.35965391993522644, 0.46348902583122253, -0.8676039576530457, 0.7629746198654175, -1.1557797193527222, 0.19953970611095428, 0.41125863790512085, -0.25459328293800354, -0.06716496497392654, 0.914244532585144, 1.1717067956924438, 0.5099301338195801, -0.08774473518133163, 0.635073184967041, 0.48460373282432556, -0.6111847162246704, 1.1365967988967896, -0.06841987371444702, 0.40613889694213867, -1.3822215795516968, 0.02725307084619999, -1.7731854915618896, 0.013447805307805538, 2.466336488723755, -0.6687746047973633, -0.5792813301086426, 0.5540494322776794, 0.8369525074958801, -0.03559557721018791, 0.19167259335517883, 1.3005077838897705, 1.387302041053772, 0.200156569480896, -0.2067399024963379, -1.2433674335479736, -0.548937201499939, -0.874652624130249, 0.09648877382278442, -0.6414220929145813, 1.8261973857879639, -0.16523420810699463, 0.8937543630599976, -1.107174038887024, -0.5704103112220764, 1.1878662109375, 0.6279297471046448, -0.37558263540267944, 0.16769669950008392, -0.23039883375167847, -1.718845248222351, -0.9666048884391785, 0.06554903835058212, 0.16418036818504333, 1.3987966775894165, -1.5249013900756836, -2.112194538116455, 0.3139248192310333, 1.5287206172943115, -0.9741987586021423, 1.740735411643982, 0.948870837688446, -1.84908127784729, 1.0126290321350098, 0.5859008431434631, 0.17087504267692566, -0.8219172358512878, 0.483550488948822, 0.25283369421958923, 0.23283857107162476, 0.6032536625862122, -0.5684040784835815, -0.5318822264671326, -0.05699532851576805, 0.8115057945251465, -1.4587229490280151, 0.9140204191207886, 1.0405828952789307, 0.6886128783226013, 1.3355220556259155, 0.6244824528694153, 0.31729137897491455, -1.8289798498153687, -0.859451174736023, 0.3909894526004791, 0.23671703040599823, 1.0413191318511963, 1.7111918926239014, -0.5040437579154968, 0.07567350566387177, 0.7838744521141052, 0.5223466753959656, 0.1716393083333969, 0.5042808055877686, 1.2934792041778564, -0.6580238342285156, 0.6456993818283081, -1.0203794240951538, 1.306604266166687, -0.7289530634880066, 1.1953864097595215, 0.5024755001068115, 0.8198830485343933, -0.40860456228256226, 0.9513176083564758, -0.8907628059387207, 0.2097342163324356, -0.47975701093673706, -1.2145090103149414, 0.3226536512374878, 1.6731328964233398, -0.6287743449211121, 1.309854507446289, -0.7081340551376343, -0.6547849178314209, -0.8632742166519165, -0.3045378029346466, 1.898711919784546, -0.2758151590824127, -1.8303433656692505, 0.1524246484041214, 0.30604517459869385], [-0.2697645127773285, 0.45320722460746765, 1.0040339231491089, 2.1346662044525146, -1.9304039478302002, 1.1997580528259277, -0.014445682987570763, -0.9028486609458923, 0.833373486995697, -0.3887293040752411, -0.10298715531826019, 0.7197887897491455, 0.3802558183670044, 0.23252902925014496, -1.2042808532714844, -1.2918264865875244, -1.083182692527771, 0.38264918327331543, -1.424895167350769, -2.1992204189300537, 1.2943488359451294, 0.8785251379013062, 0.22890931367874146, -0.25434738397598267, -1.402021884918213, 0.3152604401111603, 0.6835721731185913, -0.578637421131134, -0.7245742678642273, -0.237202987074852, -1.8895559310913086, 0.6657531261444092, -0.36244824528694153, -0.230735644698143, -0.58730548620224, 1.0105348825454712, -0.9678010940551758, -1.2914648056030273, -0.43040215969085693, -0.298602432012558, 0.058504119515419006, -1.4901838302612305, -0.9328372478485107, -1.5583871603012085, -0.6428341269493103, -1.1201796531677246, 1.943026065826416, -0.9415303468704224, 0.9965524077415466, -1.2513175010681152, 0.07573436200618744, 0.380779504776001, 2.1876282691955566, 0.22355246543884277, -0.38920891284942627, 0.9549628496170044, 0.26977309584617615, 0.957068681716919, -0.20541395246982574, 0.1234140694141388, 0.47589558362960815, -2.338901996612549, 0.34238141775131226, 0.3043445944786072, -0.678612470626831, -1.1032564640045166, -1.0867154598236084, 1.38814115524292, 0.09313533455133438, 2.6606853008270264, -1.3486361503601074, -0.12982618808746338, 2.0920979976654053, 0.2094581127166748, 0.16762195527553558, 0.48997223377227783, -0.7834435701370239, 0.8211939930915833, 2.1434805393218994, 1.0805383920669556, 1.0763651132583618, -0.9413725733757019, 0.05035247281193733, -1.3598097562789917, 0.8899336457252502, 0.8003975749015808, -0.9467690587043762, 1.0619988441467285, -1.9441275596618652, -0.06067957356572151, -0.5981471538543701, -2.1852896213531494, -0.6235620379447937, -1.3016438484191895, -0.02705821581184864, 1.4039872884750366, -0.09728551656007767, -1.7714065313339233, -1.8363652229309082, 1.075299859046936, -2.0382113456726074, -0.02238631807267666, -2.830075263977051, -0.12792521715164185, 0.9867191910743713, 0.852414071559906, -0.31118860840797424, -0.7583965063095093, -0.5086609125137329, 0.4010055661201477, -2.14363956451416, 1.3614695072174072, -0.4796100854873657, 0.7229726910591125, 1.0813829898834229, 0.014217931777238846, 0.9223311543464661, -0.41684040427207947, -0.8127056360244751, -2.577147960662842, 1.196306586265564, 1.475508451461792, -0.20298470556735992, 0.3249438405036926, 0.015106693841516972, -1.3501495122909546, -0.4759921431541443, 1.4867433309555054, -0.3192613422870636], [-0.14055049419403076, 1.4063724279403687, 1.1712331771850586, -0.7130381464958191, 0.07133873552083969, -0.9279630184173584, -1.2733619213104248, -0.022601012140512466, -0.7307978868484497, 0.2704174816608429, 2.45980167388916, -0.4260541796684265, -0.1984746903181076, 1.8157910108566284, 0.27429649233818054, -0.5516989827156067, -0.10582433640956879, -1.8651951551437378, -1.2077611684799194, 0.1497867852449417, 1.093835711479187, -1.509703278541565, 0.5721272826194763, 1.1216102838516235, -0.6616151928901672, 0.8076181411743164, -0.5449206233024597, 0.27073752880096436, -1.351304292678833, 0.5667961239814758, -1.0066347122192383, 0.3534868657588959, -2.2800965309143066, 1.5785918235778809, -0.5737346410751343, 0.0019764346070587635, 0.7566932439804077, 0.5980051755905151, -1.0288466215133667, 1.0143766403198242, -0.003971558529883623, -0.11831878125667572, 0.19906310737133026, 0.13654446601867676, -1.2324227094650269, 0.254483163356781, 0.35209906101226807, -0.5026833415031433, 0.43966439366340637, 0.3668839633464813, 1.0193209648132324, -0.6015897393226624, -0.05745700001716614, -0.2633625864982605, 1.236722707748413, -1.2384997606277466, 0.4096483886241913, -0.632869303226471, -0.923728883266449, 0.7393493056297302, 1.0446226596832275, -1.3363398313522339, 0.3423696756362915, -1.5017987489700317, 0.8963031768798828, -0.7431761622428894, -0.9173208475112915, -1.247945785522461, 1.6699854135513306, -0.13952989876270294, -1.0545545816421509, 0.2906649112701416, 0.9371974468231201, 0.32698264718055725, -0.15106718242168427, -0.7977921962738037, -0.9568593502044678, -1.2643623352050781, -1.291744589805603, -1.2366048097610474, 0.22231332957744598, 1.2549723386764526, 0.5831976532936096, 1.5504716634750366, 0.6260085701942444, 1.3396098613739014, 0.6298999786376953, -0.8685173392295837, -0.03576476871967316, -0.6872342824935913, -0.16742655634880066, -1.2385882139205933, -0.6298626661300659, 0.5828390717506409, -0.2750737965106964, -0.346004456281662, -0.27998432517051697, 0.9405031800270081, 1.4621052742004395, 0.3465575873851776, 0.12244202941656113, -1.1134676933288574, -0.8749775290489197, 0.6797304749488831, -1.103585958480835, 1.3095802068710327, -2.633244276046753, 2.4778056144714355, -1.456526517868042, 0.45857295393943787, -0.4876266121864319, -0.32291820645332336, 0.006419372279196978, 0.10508477687835693, 0.3902531862258911, 0.6305083632469177, 0.9328892230987549, 0.46614348888397217, 1.7365981340408325, 0.37766700983047485, 1.1597517728805542, 1.002162218093872, 0.12487374246120453, -0.20463766157627106, 0.07543952763080597, -0.23699653148651123, 0.507897138595581, -0.050217438489198685, -0.26596853137016296], [-0.08161824196577072, -0.5022469758987427, -0.28962185978889465, 0.0736255794763565, -0.46142104268074036, -2.078209400177002, -0.3832590878009796, 0.617335855960846, -0.49208536744117737, 0.28478574752807617, -2.1549668312072754, 0.5959409475326538, -1.2545549869537354, -1.1363883018493652, -0.9604493975639343, -0.8980557918548584, -0.5074400305747986, -1.0143916606903076, 0.6977760791778564, 0.2625468671321869, -0.6443901658058167, 1.145384430885315, 0.0902768075466156, 0.36280888319015503, -0.36937206983566284, 0.21780849993228912, -1.445688009262085, 0.05920162424445152, -2.840552806854248, -2.0878918170928955, -0.07301665842533112, -0.39596524834632874, -1.5064308643341064, 1.445931077003479, -0.4981865882873535, -1.0596585273742676, 0.12930873036384583, -2.0874903202056885, -0.2790408432483673, 0.13526122272014618, -1.1241832971572876, -0.6783657670021057, -0.6574351191520691, 0.512408435344696, -1.6714714765548706, 1.9899028539657593, -0.8237890601158142, 0.356919527053833, -0.47634702920913696, 0.13007070124149323, -0.17898648977279663, 0.7347634434700012, 1.0407036542892456, -1.3882863521575928, -0.12874886393547058, -0.766589343547821, -0.0685315728187561, 0.005728685297071934, -0.08629786223173141, 1.4279088973999023, 0.07883458584547043, -2.0402731895446777, 2.4331958293914795, -0.7737361192703247, 1.122359037399292, 0.8293167948722839, -0.5374642610549927, -0.024835333228111267, 0.5906685590744019, -0.5703120827674866, -1.6601001024246216, 0.2479272037744522, 0.976783812046051, 0.4741772711277008, -0.40783053636550903, 1.2185362577438354, 1.3411598205566406, -0.24485662579536438, 0.761443555355072, -0.466251403093338, -0.29288381338119507, 0.09574081003665924, -0.1466045379638672, -0.36504149436950684, -0.016595344990491867, -2.173879384994507, 1.4631670713424683, -0.5470467805862427, 1.5244992971420288, -1.7470637559890747, -1.002036452293396, 2.0024771690368652, 1.7748807668685913, 1.1055766344070435, -0.6742321252822876, 2.0169968605041504, -0.06159740686416626, -1.0124857425689697, 0.6610520482063293, -0.6271305084228516, 0.8016244769096375, -0.9562603831291199, 0.7363207936286926, -0.7587368488311768, 0.6445820331573486, -0.688277006149292, -0.956188976764679, -1.4834997653961182, 0.003734371392056346, 1.0879839658737183, -1.4851750135421753, 1.6488826274871826, -0.8703692555427551, -0.5153782963752747, 1.3813097476959229, 2.1300010681152344, -0.13825000822544098, -1.660836935043335, -1.9420640468597412, 0.16991275548934937, -1.5828406810760498, 0.8028287291526794, 0.33644339442253113, -1.937638282775879, -0.6915631890296936, 1.2700467109680176, 0.26969894766807556, 0.5504578351974487, -0.9172877073287964], [-2.06705904006958, -0.4585006535053253, 0.9652600288391113, -0.026060013100504875, -1.8847272396087646, 0.5199240446090698, -0.7612444758415222, 0.6431142091751099, 2.308511734008789, 0.4065123498439789, 0.6300260424613953, -1.2493207454681396, -1.0585887432098389, 1.8250408172607422, -0.4447439908981323, -0.4692928194999695, -0.485505074262619, -0.568253755569458, 0.09209717065095901, 0.4954448938369751, 2.333444356918335, -1.5618979930877686, -0.5184916257858276, -0.1782945841550827, 0.5335140824317932, -0.9179888963699341, 0.8960441946983337, 0.38047096133232117, 0.6922703385353088, -0.5852062106132507, 0.24598421156406403, 1.037630319595337, 1.4055378437042236, 2.014080762863159, -0.23531660437583923, 0.7240497469902039, -0.6871212124824524, 1.0861188173294067, -0.16383369266986847, 2.252713441848755, 0.23117853701114655, -0.24461272358894348, 0.1767842173576355, -1.1212774515151978, -0.026340316981077194, 0.3662048876285553, -0.6621220111846924, -0.21539026498794556, -0.2399234026670456, -0.4643251895904541, 0.12142335623502731, 0.3516327440738678, -0.16605567932128906, 0.5336071252822876, 1.1669223308563232, 1.0581244230270386, -0.9723525047302246, -1.3077571392059326, -0.46702685952186584, 0.3635549545288086, -1.664988398551941, -1.8155558109283447, -0.08072759956121445, 1.661412000656128, -0.24081382155418396, 1.9014158248901367, -0.2503548264503479, -0.8480180501937866, 0.21063295006752014, -1.5990201234817505, 2.733670473098755, 0.31238168478012085, 0.6564863324165344, 1.3812499046325684, 1.8911370038986206, 2.879053831100464, -0.9827361106872559, 0.1737450361251831, -1.3473258018493652, -0.8794460296630859, 0.6274945139884949, -0.3365943431854248, 0.08138932287693024, 2.4703876972198486, -0.5133209824562073, 0.181359201669693, 0.7852135896682739, -1.4538226127624512, 0.8543715476989746, -0.5691964626312256, 1.9221135377883911, 0.39255809783935547, 0.4301784336566925, 1.475271463394165, -1.121468186378479, 1.4766790866851807, -0.848669707775116, -0.31927767395973206, -0.3701268136501312, -0.6247746348381042, -0.9294497966766357, -0.5424573421478271, 0.8489115834236145, 0.5971486568450928, 0.30819061398506165, 1.3534820079803467, 0.9327312111854553, -1.266494870185852, 1.6614956855773926, 0.887187659740448, 1.2863744497299194, 1.6580461263656616, 0.025325993075966835, -0.13244685530662537, 1.1384159326553345, -0.6803928017616272, -0.8226228952407837, 0.5658047199249268, 0.9883983135223389, -0.33036622405052185, 1.2686283588409424, 2.0362515449523926, -0.22424644231796265, 0.11324480175971985, 0.17538811266422272, 0.0685788094997406, -0.358110249042511, 0.7100111246109009, 0.17982357740402222], [0.19470557570457458, -0.5710819363594055, -0.5559757351875305, -1.4217981100082397, 2.075441598892212, 0.18848930299282074, 1.2631587982177734, -0.1357998549938202, -0.8316722512245178, 0.5615833401679993, 0.1968691200017929, -0.0973031222820282, 2.7953169345855713, 0.7548894882202148, 2.5012519359588623, -0.35932326316833496, -0.1223059892654419, -0.574303388595581, -1.0403157472610474, 0.8256908059120178, -0.04597765579819679, -0.5013310313224792, -1.605217456817627, -1.3163591623306274, -2.1854779720306396, -0.6672503352165222, 0.06680438667535782, 1.537871241569519, -0.3633117973804474, -0.429376482963562, 0.3026707172393799, -0.05564592778682709, 0.9537584781646729, -0.12186933308839798, 1.2368245124816895, -1.1919937133789062, 1.8044036626815796, -0.1403941512107849, 0.2115541696548462, -0.8289529085159302, 0.6752281785011292, 1.0840822458267212, -0.07543006539344788, -0.8244917392730713, -0.7635040283203125, 0.6704867482185364, 0.5201924443244934, 0.20077306032180786, -1.5159908533096313, -1.8809425830841064, -0.06488198041915894, 0.20737622678279877, 0.4259379208087921, -0.8879079818725586, 0.44210389256477356, -2.2351105213165283, 0.2578917443752289, -0.9586237668991089, -0.43557775020599365, -0.9797737002372742, -0.5690801739692688, -0.2134411334991455, -0.5795422196388245, -0.0061987945809960365, -1.4015114307403564, 0.5758118033409119, -0.9372693300247192, 0.9794092774391174, 1.60750412940979, -0.02550964616239071, -0.5877071022987366, 0.25793758034706116, 1.621030569076538, -0.896929144859314, -0.13870011270046234, -0.021434491500258446, -1.7412545680999756, 1.6065864562988281, -1.0416598320007324, -1.5895737409591675, -0.17832587659358978, -0.8316117525100708, -2.5175299644470215, -0.7108846306800842, 0.18637914955615997, -1.6046841144561768, 0.7209693789482117, 0.21890880167484283, 1.4361884593963623, 0.610765278339386, -0.011327732354402542, -0.6686064004898071, -0.4921037554740906, -0.9648157358169556, 0.16912730038166046, -0.7183804512023926, -0.7175836563110352, -0.7639737129211426, 0.4829728901386261, 0.12683095037937164, 0.9558063745498657, -0.33842429518699646, -1.3912341594696045, 0.08171103894710541, 1.8867299556732178, -1.7153903245925903, 1.008607268333435, 0.9276157021522522, -0.032077040523290634, -0.5967908501625061, 0.9964312314987183, 0.060478705912828445, 1.5928558111190796, 0.8146061301231384, -0.024279093369841576, 0.12929368019104004, 1.1205166578292847, 0.8655529022216797, 0.2811160385608673, 0.24410490691661835, -0.2041589766740799, -0.26276567578315735, 1.2146246433258057, -0.3639313876628876, 1.0283520221710205, 0.15097349882125854, -0.14092279970645905, -2.776738166809082, 0.7653605341911316], [0.5567466020584106, -0.05336596816778183, -0.751784086227417, -0.323241651058197, -0.20254099369049072, 0.6596952676773071, 0.0907355397939682, 1.7785942554473877, 0.20666109025478363, -0.2381603717803955, 0.5485717058181763, -0.7172849178314209, 0.2124219536781311, -1.0314674377441406, 1.3595614433288574, -0.5399749279022217, 1.7757775783538818, 0.06456707417964935, -0.1788841038942337, 0.09776211529970169, 0.316776841878891, -0.6137533783912659, 0.41631942987442017, 1.4788460731506348, 0.5819206237792969, 0.9268889427185059, 0.5026338696479797, 0.03112093359231949, -0.9783459901809692, -0.4569317698478699, 0.9970375895500183, -0.5592749714851379, -0.4527844488620758, 1.8341082334518433, -1.712754487991333, -0.16933363676071167, 0.7260911464691162, 0.9848803877830505, 0.2648058235645294, -1.659736156463623, 1.1905485391616821, 1.879632830619812, 0.24026429653167725, 0.48571717739105225, 1.4819656610488892, 1.6437979936599731, 1.0238198041915894, 0.39003702998161316, 0.8240317106246948, 0.9774449467658997, 0.5344455242156982, -0.7473651766777039, 0.5785343647003174, 1.4589073657989502, -0.5848453044891357, -1.0099903345108032, -0.6112067103385925, 0.2645792067050934, 0.5363103151321411, 1.9687215089797974, 0.4831494092941284, -0.07915645092725754, 0.26785004138946533, -1.0277564525604248, -0.7381649017333984, -0.16682516038417816, -0.3317435085773468, 1.668233036994934, -1.2575600147247314, -0.33324146270751953, -1.5902196168899536, 0.15308411419391632, 1.4422084093093872, 0.756511390209198, -0.202768936753273, -0.8901422023773193, -0.6677047610282898, -1.1042633056640625, -1.6581236124038696, -1.6525959968566895, -0.8341204524040222, -0.1971641331911087, 0.027940427884459496, 0.7166643142700195, 1.0662246942520142, -1.1618536710739136, -1.609839916229248, -0.2034081369638443, 1.9056092500686646, 0.7751495838165283, -0.6335590481758118, 0.2890275716781616, -0.8937699794769287, 0.26507091522216797, -0.8618192076683044, 1.624432921409607, 0.12462355196475983, 1.518301010131836, -3.386660575866699, 0.11844546347856522, -0.8840875625610352, 0.9270584583282471, 0.5669687390327454, -2.14192271232605, -1.099692940711975, 0.69040447473526, 1.0784846544265747, 2.1422483921051025, 1.008187174797058, -0.2590261399745941, -0.8820721507072449, -0.8063187599182129, 1.5838282108306885, 1.7639117240905762, -0.5402142405509949, 0.49895811080932617, -0.9163537621498108, 0.4485900104045868, -0.02768000215291977, -0.871961236000061, 0.5395984649658203, -0.5500877499580383, 0.6178049445152283, 0.4035325348377228, 0.26792410016059875, 1.9131247997283936, -0.770675539970398, -0.6153110861778259, -0.7690427899360657], [2.0470809936523438, 0.4371522068977356, 0.46179819107055664, 2.454671621322632, -0.8633428812026978, 2.3470382690429688, -0.8712955117225647, 0.5988479852676392, 0.47350072860717773, -0.6571587324142456, 1.8337126970291138, -1.8072413206100464, 0.3069777190685272, 0.302272230386734, -0.6294649839401245, -0.9361771941184998, 1.4577386379241943, 0.8384159207344055, -0.12262972444295883, 0.910749077796936, -0.17131900787353516, 1.0405852794647217, -2.4989147186279297, 0.5083638429641724, -0.13576370477676392, -0.3477180302143097, -1.367108702659607, -0.2753911316394806, -0.8559269309043884, -1.1467410326004028, 0.07478702068328857, -0.6141954064369202, -0.5093018412590027, 0.774300754070282, -0.7633339762687683, -0.7467218637466431, -0.6242040395736694, -1.467276930809021, 1.469118356704712, 1.0336639881134033, -0.10824984312057495, 0.15871769189834595, 0.7505186796188354, 1.3826558589935303, -1.9268724918365479, -0.13974636793136597, -0.526384711265564, -0.4088008999824524, 1.9872817993164062, 0.12266530096530914, -0.009974353015422821, 1.5658515691757202, -1.5088046789169312, -0.11972890794277191, -0.8000005483627319, -0.7804257869720459, 0.6425175070762634, -1.1404865980148315, 0.9059994220733643, -0.30033087730407715, -0.1923375129699707, -0.5161884427070618, 0.46228697896003723, 0.17949214577674866, -0.24875591695308685, 2.5572094917297363, 0.7547373175621033, -1.5715901851654053, -0.23315799236297607, -0.1802901178598404, -0.10662031918764114, 1.8836897611618042, -0.0723860040307045, -1.4073748588562012, 0.5477071404457092, -0.30292725563049316, -0.3410230875015259, 0.28687405586242676, -1.8490263223648071, 0.08364832401275635, -0.65635085105896, -2.0763185024261475, 1.9113670587539673, 0.5809938907623291, 1.8973913192749023, -0.5222069621086121, 1.2993096113204956, -0.7534372806549072, -0.6195554137229919, 1.3678299188613892, 1.3490797281265259, -0.5497179627418518, 0.7291825413703918, -0.14818406105041504, -0.6471852660179138, -0.41973450779914856, 1.0033174753189087, -0.0737806111574173, -1.4888628721237183, 1.5199544429779053, -1.177665114402771, -0.9511215090751648, -0.43245869874954224, 0.23308566212654114, 2.276474952697754, -1.273809552192688, -1.7439115047454834, 0.4838968813419342, -0.624682605266571, 1.115216851234436, 1.1043790578842163, -1.0243229866027832, 1.0682563781738281, -0.41293707489967346, 0.8496031761169434, 0.5436369180679321, -0.11457285284996033, 0.034586101770401, -1.3140308856964111, -0.059985823929309845, 0.49700891971588135, 0.3468048870563507, -0.47767481207847595, 0.5596030354499817, 0.6863014698028564, 1.3143839836120605, 0.22097262740135193, -2.3304996490478516, 0.3295391798019409], [0.7022501826286316, -0.8148438334465027, -0.05109810456633568, 0.3556489050388336, -1.1741161346435547, -0.5627554655075073, -1.0345697402954102, -1.0401006937026978, 0.7243108153343201, -1.1640959978103638, -0.34646525979042053, -1.3502113819122314, -0.11708083748817444, -1.2968969345092773, -0.7237790822982788, -0.1121460422873497, -0.28790172934532166, 0.3647840917110443, 0.5111064910888672, 0.40959450602531433, -0.9099876880645752, -1.1190110445022583, -1.6943529844284058, -0.06366276741027832, -0.008102101273834705, -0.3143737316131592, -1.6448383331298828, 0.9802484512329102, -1.1980969905853271, 0.3085167706012726, 1.1737040281295776, -0.108664371073246, -0.1141946017742157, -1.0598851442337036, 0.4688000977039337, -0.9413049221038818, 1.1373298168182373, -0.3292817771434784, -2.542018175125122, -2.303621768951416, -0.17413395643234253, 0.058169856667518616, 1.7023316621780396, -1.6316710710525513, 0.09252401441335678, -1.240907073020935, 0.4540632367134094, -0.5620633959770203, 1.1048587560653687, -0.5702490210533142, 0.32882964611053467, -2.355376958847046, 1.5623222589492798, 0.4883345067501068, 0.413377046585083, 1.5344637632369995, 0.7691792845726013, 0.12213000655174255, 0.9955268502235413, -2.141998767852783, 0.8132410049438477, 0.5039931535720825, -0.4540491998195648, 0.20297248661518097, -1.212275505065918, -1.5016505718231201, 0.7685104608535767, 1.1086974143981934, 1.214354157447815, -1.0480241775512695, 0.2814329266548157, 1.411340355873108, -0.17942824959754944, 0.5237402319908142, -1.657289743423462, -0.47071582078933716, -0.4217492341995239, -1.4360225200653076, 1.0095598697662354, -0.6601817011833191, 1.0379321575164795, -0.42727723717689514, 0.6979770064353943, 0.9638356566429138, -0.25378480553627014, 1.0094759464263916, 0.7381492853164673, -1.27940833568573, -0.34795787930488586, -1.245980143547058, 0.8131861090660095, -0.04055438190698624, 0.14850014448165894, 0.4773029386997223, -0.5666380524635315, 0.7012282013893127, 0.7330151200294495, -0.7257716655731201, 0.5886959433555603, 1.2532384395599365, -0.5142130255699158, 0.7032322287559509, -0.5666249990463257, -0.8253818154335022, -0.5782676935195923, 1.5225061178207397, 0.06614450365304947, -0.13442781567573547, -0.21967901289463043, -0.6901255249977112, -0.0165729820728302, -0.7465271949768066, 1.148816704750061, -0.021861430257558823, -1.0531607866287231, -0.5325936675071716, 1.9404919147491455, 1.3135716915130615, -0.3804902136325836, 2.20095157623291, -0.42275547981262207, -0.7166019082069397, -1.3695693016052246, -0.34720730781555176, 0.08940740674734116, -0.5224481821060181, -0.19298489391803741, -0.7613924741744995, -0.05237913876771927], [-0.20571939647197723, -0.3322577178478241, -1.1574227809906006, -1.880165696144104, -0.43431365489959717, -0.9775063395500183, -0.696351945400238, 0.21754589676856995, 0.8003600239753723, 0.6042231321334839, 0.18720735609531403, -1.9778285026550293, -0.5927554965019226, 0.7531523704528809, 0.2943122386932373, 1.1961966753005981, -1.5679601430892944, 1.7946784496307373, 2.666198492050171, 0.9515500068664551, 1.128659725189209, -0.2039192169904709, 0.8672545552253723, 0.7520029544830322, -0.17651613056659698, 0.6590728163719177, 2.1225385665893555, -1.285455584526062, -0.4378252327442169, 0.029796797782182693, -1.0472981929779053, 0.9012909531593323, 2.2083637714385986, -0.8097928166389465, 0.41986554861068726, 0.29513683915138245, -0.6798810958862305, 0.6165971159934998, 1.2413556575775146, -0.16915521025657654, 0.3157885670661926, 0.38639169931411743, 1.7186996936798096, 0.15382064878940582, -0.3310091495513916, 1.3443374633789062, 0.6844743490219116, 1.2049773931503296, -0.8748146295547485, -1.225944995880127, -1.0511213541030884, -1.410713791847229, 1.0419414043426514, -0.5360977053642273, 0.6479855179786682, -1.3529542684555054, -0.13421860337257385, -0.5831640958786011, -0.11272601783275604, 0.023095056414604187, -0.4369129538536072, -0.9367256760597229, 1.2809251546859741, 0.6369820833206177, 0.16909486055374146, -1.312847375869751, 0.39972344040870667, 2.6234543323516846, -0.37871381640434265, -3.1944379806518555, 0.026978764683008194, -0.1362629234790802, -0.708038866519928, -0.47751715779304504, 1.609108328819275, 0.2690390944480896, -0.8304246068000793, -0.7301558256149292, -0.4919016659259796, -1.884790301322937, 0.07003316283226013, -0.23354002833366394, -1.0156993865966797, 0.05999060347676277, -0.802734911441803, 1.297511339187622, 0.8292170763015747, 0.8291106820106506, -0.8372963070869446, -0.5838879346847534, -0.7135761976242065, -0.28388160467147827, -0.2870491147041321, -0.41412413120269775, -2.3099589347839355, -0.1533862054347992, 1.0789257287979126, -0.0019380450248718262, 1.7650136947631836, -1.5467580556869507, 0.1592983901500702, 2.6636931896209717, 1.2921667098999023, 1.4779194593429565, -0.6996446847915649, 1.1654391288757324, -1.194024920463562, 0.8495949506759644, 2.536482810974121, -0.2362360954284668, -0.7306554913520813, 1.7538541555404663, 0.8231235146522522, 0.06906242668628693, 0.7021329402923584, 0.36106646060943604, -1.0998831987380981, 1.0620217323303223, -0.6324107050895691, 1.1923333406448364, 1.134429693222046, 0.3033181130886078, -1.4572036266326904, 0.31725022196769714, 1.9258075952529907, -0.9147500395774841, 0.40569135546684265, -0.9608315825462341, 0.7021209597587585]], "dtype": "float32"}], "e": {"__nd__": [[29.32322883605957, 22.71409797668457, -22.86578369140625, -25.778043746948242, -2.2562596797943115, -8.49807357788086, -3.6109166145324707, -18.67813491821289, -29.1375732421875, -3.754627227783203, -17.212913513183594, 5.922724723815918, -6.367324352264404, -14.676885604858398, 2.4502077102661133, -2.966979503631592, -26.291555404663086, 6.566356658935547, 26.026960372924805, -5.275064468383789, -10.520699501037598, -23.368511199951172, -2.642930030822754, -10.004608154296875, 1.753381609916687, 15.243980407714844, -18.65373992919922, 16.249492645263672, -0.11592888832092285, -20.193912506103516, 12.26325798034668, -9.316747665405273, -2.2049386501312256, -7.168173313140869, -24.77429962158203, -18.421297073364258, -16.802976608276367, -17.89183807373047, 18.5550537109375, 7.563370704650879, 18.148710250854492, -46.69463348388672, 1.79979407787323, -2.1390435695648193, -30.245634078979492, 2.7099342346191406, -5.817131042480469, 1.564713954925537, 24.06546401977539, 3.038411855697632, -14.525649070739746, -0.9381246566772461, 11.67431926727295, -23.77558708190918, -0.0018369118915870786, -22.716066360473633, -9.26258659362793, -8.633278846740723, 4.575325965881348, -19.213071823120117, -5.736498832702637, 14.13325023651123, 3.5087361335754395, 5.1591339111328125, -11.30507755279541, 28.696935653686523, 31.122028350830078, -27.382997512817383, -27.33552360534668, 0.3638364374637604, -8.51330280303955, -4.961719036102295, -19.076196670532227, -1.9542564153671265, 17.700380325317383, -3.711826801300049, -6.984144687652588, 15.085407257080078, -4.067283630371094, 10.731855392456055, -15.973570823669434, -34.26688003540039, -21.579496383666992, -20.16098976135254, 2.3227949142456055, -5.784183979034424, -3.0557003021240234, -2.1920433044433594, -14.40543270111084, 17.987401962280273, -18.014184951782227, 1.3146040439605713, 9.673152923583984, 3.0760324001312256, -3.5808353424072266, -8.597541809082031, 11.440954208374023, -13.840280532836914, -17.133073806762695, 4.798508167266846, 22.881378173828125, -7.592387676239014, 5.160278797149658, 24.509565353393555, -23.71219825744629, -4.600236892700195, -30.45927619934082, -1.589038372039795, 38.98027038574219, -25.961713790893555, 3.718717336654663, -33.066226959228516, 20.429716110229492, -11.38686752319336, 3.1451938152313232, -1.25603187084198, 6.579138278961182, 2.5738348960876465, -36.54801940917969, 15.177044868469238, 4.285765647888184, 1.1041189432144165, 7.514466285705566, -1.398368000984192, 25.414140701293945, -14.643532752990723, -25.209163665771484, -2.8737757205963135, 7.71034574508667], [3.1901636123657227, -11.043519020080566, -16.33682632446289, -3.552591323852539, -4.924628734588623, -24.15861701965332, 0.707034707069397, -9.259925842285156, 21.771604537963867, 8.324371337890625, -23.30245590209961, -7.438471794128418, 20.329357147216797, 5.876759052276611, -22.961864471435547, -3.307807207107544, 30.068397521972656, -11.140281677246094, -9.24634838104248, 23.87607192993164, -37.93838119506836, -9.367467880249023, 25.99667739868164, -3.16422963142395, -21.477184295654297, -2.8482556343078613, 33.286773681640625, 13.854095458984375, 15.8051118850708, -7.452298164367676, 14.519675254821777, 23.786937713623047, 9.12572193145752, 0.10089092701673508, 15.574736595153809, -4.000762462615967, -16.635360717773438, 19.34461784362793, 23.23357391357422, 36.60984802246094, 9.201093673706055, -21.503873825073242, -12.181682586669922, 20.059432983398438, 2.8067286014556885, 6.709716796875, 7.6089863777160645, 31.673757553100586, 15.866880416870117, -11.524293899536133, -18.113567352294922, -13.468791961669922, 15.639494895935059, -20.399646759033203, -24.374622344970703, -14.893186569213867, 15.167613983154297, 10.001538276672363, -2.700371742248535, -22.514131546020508, 32.73548126220703, 20.990825653076172, -1.8211561441421509, 13.173885345458984, 4.595575332641602, -24.08472442626953, 4.620697975158691, 27.33773422241211, 16.360286712646484, -5.346404552459717, 28.609025955200195, -5.775990962982178, -0.5636582374572754, 6.5822882652282715, -10.25091552734375, -5.849692344665527, 31.441959381103516, -24.379688262939453, -9.826313972473145, -3.339331865310669, 27.215476989746094, 0.2866802215576172, 12.467812538146973, 0.138236403465271, 3.105048179626465, -0.186599999666214, -14.290026664733887, 11.987570762634277, -15.906973838806152, 5.562873840332031, -1.1719740629196167, -1.7833600044250488, -0.41685938835144043, 10.532746315002441, -9.226449966430664, 16.180625915527344, -7.0278143882751465, 27.101463317871094, 43.671241760253906, -20.199630737304688, -34.138832092285156, -13.3587064743042, 35.574710845947266, 7.720948219299316, -12.971362113952637, 3.562765598297119, 5.866113662719727, -18.77485466003418, 7.431524276733398, -5.802239418029785, -20.95418930053711, 6.621201038360596, 1.9765962362289429, 7.738985061645508, 7.862344741821289, -0.05559033527970314, 10.859618186950684, -7.144420146942139, 28.44904899597168, 5.33066463470459, 24.592235565185547, 23.231647491455078, 17.277612686157227, 11.044537544250488, -16.607175827026367, -16.218793869018555, 7.192185878753662, -10.543710708618164, 6.24539041519165], [-16.072080612182617, 5.843602180480957, -1.425350546836853, 2.1000373363494873, 25.098342895507812, 25.7768497467041, 25.16859245300293, 2.9750053882598877, 0.72281414270401, 14.72101879119873, -11.476563453674316, 7.398632049560547, 0.6620150804519653, 2.6912498474121094, 14.097529411315918, -8.891288757324219, 12.894326210021973, -8.030835151672363, -20.62254524230957, 8.747711181640625, 19.99428367614746, -21.95767593383789, -1.2872929573059082, 19.73528289794922, 19.502029418945312, -4.714038372039795, -18.639190673828125, -17.0090389251709, -0.6697273850440979, -7.943517684936523, 21.34768295288086, -19.474674224853516, -2.261213541030884, -20.41739273071289, -0.564943253993988, -12.63145637512207, -6.387346267700195, -4.291213512420654, -29.977556228637695, -12.021163940429688, 19.97883415222168, -44.15447235107422, 7.391282558441162, -33.14059829711914, -7.953167915344238, 10.394948959350586, -14.065746307373047, 8.660621643066406, -10.47490406036377, -11.543397903442383, -37.75327682495117, -22.007326126098633, 5.126275539398193, -3.4287147521972656, -9.639421463012695, -15.732978820800781, -16.55208969116211, 23.537567138671875, -9.67609977722168, 28.644176483154297, 5.057775974273682, 4.721397876739502, 11.73206901550293, -11.77370548248291, 6.617649078369141, 6.543510437011719, -15.074498176574707, -5.6079936027526855, 35.041168212890625, -15.13803482055664, -3.6641244888305664, -40.43968200683594, -9.318318367004395, -33.85517883300781, 0.9354996085166931, 27.365018844604492, 12.309794425964355, 28.49067497253418, 36.426292419433594, -11.073009490966797, -12.5642728805542, 21.568233489990234, -16.09095573425293, 6.873448848724365, 9.896656036376953, 8.59273624420166, 6.3883795738220215, 7.700728416442871, -0.5927956700325012, -11.512383460998535, -2.2214696407318115, -22.851083755493164, -12.139458656311035, 10.748103141784668, -1.5810209512710571, -11.082427024841309, -19.336177825927734, 21.093265533447266, -45.92304611206055, 3.3481645584106445, -16.780019760131836, -14.433313369750977, 8.365888595581055, -31.157047271728516, -12.005642890930176, 1.7061415910720825, 0.17514865100383759, 19.03921890258789, 10.897115707397461, -6.017353534698486, 16.577190399169922, -15.727343559265137, 1.203406810760498, 7.884919166564941, -15.162694931030273, -16.864797592163086, 21.40315055847168, -12.418620109558105, 7.924149036407471, 10.624324798583984, 5.781286239624023, 16.75891876220703, 13.709197998046875, 10.427314758300781, 1.8874880075454712, -16.564477920532227, 8.812525749206543, -3.0118839740753174, 15.265162467956543], [-15.689971923828125, -15.540889739990234, 16.375404357910156, 0.7690592408180237, -12.56256103515625, 11.126391410827637, 5.902049541473389, -2.531039237976074, -11.293206214904785, 16.422809600830078, 2.1943483352661133, -1.2561099529266357, 7.987639427185059, -5.688293933868408, 21.000492095947266, -0.08626887947320938, 13.06911849975586, 20.235626220703125, -0.6002253890037537, 5.625063419342041, -8.46790599822998, -4.112863063812256, 4.581770420074463, 38.014583587646484, -5.33604621887207, -18.520526885986328, -20.896034240722656, 1.6780389547348022, -7.861227512359619, -13.324292182922363, -47.929500579833984, -39.731815338134766, -6.607842445373535, -6.627510070800781, -17.04358673095703, -13.021005630493164, 16.401710510253906, 8.714932441711426, -3.8595352172851562, 15.564443588256836, 11.499288558959961, 15.670492172241211, -14.448532104492188, 2.3895223140716553, 12.844472885131836, -11.228875160217285, 0.37738168239593506, -8.3503999710083, 8.230466842651367, 8.747857093811035, 22.195220947265625, 9.116249084472656, 18.157590866088867, 3.34074330329895, 31.201570510864258, 26.721208572387695, 4.085640907287598, -0.38261082768440247, -17.324703216552734, 15.187572479248047, 4.27190637588501, -21.502962112426758, 5.1184563636779785, -19.733551025390625, 1.10441255569458, -38.32616424560547, 15.933871269226074, -6.024904727935791, -16.65309715270996, -10.435315132141113, -2.464454174041748, 14.125961303710938, -7.7940826416015625, 9.462477684020996, -11.296039581298828, -14.991487503051758, 4.876461029052734, -2.642305850982666, -11.602910041809082, 8.205296516418457, 15.574576377868652, -7.594625949859619, -15.797501564025879, -35.513675689697266, 36.22285842895508, -14.70136547088623, 0.7233061194419861, -33.28275680541992, -5.430080890655518, -4.155716419219971, 22.38326072692871, -33.98158645629883, -21.950754165649414, -21.043216705322266, -29.84280014038086, 12.385268211364746, 28.446361541748047, 19.408390045166016, -15.006677627563477, 23.474136352539062, 1.7939180135726929, -2.132061243057251, 13.937542915344238, -5.06511926651001, 5.7278337478637695, -4.935209274291992, 15.1026029586792, 4.183348178863525, 21.242841720581055, -6.5521955490112305, 1.6567120552062988, 16.660045623779297, -7.461217403411865, 10.813674926757812, -12.62589168548584, -8.50069808959961, 12.281989097595215, -2.430527925491333, -4.775338649749756, -7.241945266723633, -8.839641571044922, 7.237733364105225, -26.90291976928711, 25.916275024414062, 24.013769149780273, -16.342235565185547, 11.73218822479248, -3.365367889404297, 18.436206817626953], [-41.14568328857422, -19.376218795776367, -5.245682239532471, 20.116146087646484, 23.440359115600586, -3.770404815673828, 0.7353429794311523, -4.626506805419922, 23.56885528564453, 11.813684463500977, 7.781103610992432, 2.549583911895752, 25.16543197631836, -29.425750732421875, -28.417755126953125, -8.09653091430664, -0.2953997850418091, 10.556504249572754, -6.346883773803711, -12.45666790008545, 21.114992141723633, 8.017016410827637, -0.3663969039916992, -5.888549327850342, 7.26025915145874, 0.8376518487930298, -26.606517791748047, -0.3043232262134552, -3.027846336364746, 2.5525171756744385, 24.54911231994629, -8.733858108520508, -12.62948226928711, 2.5826058387756348, -8.171351432800293, -37.435482025146484, -25.914806365966797, 12.377167701721191, 0.5178601145744324, -24.19129180908203, -11.73986530303955, 31.52054786682129, 0.9031084179878235, -13.577120780944824, 2.9056687355041504, -17.166458129882812, -1.8600151538848877, -10.135111808776855, 19.423309326171875, -6.499163627624512, -12.489510536193848, 24.179384231567383, -18.90053367614746, 12.333419799804688, 5.749064922332764, 10.209084510803223, 24.88414764404297, 6.081624984741211, 15.522117614746094, -21.715723037719727, 5.017510414123535, -1.5264233350753784, -32.89568328857422, -10.339442253112793, -13.162625312805176, -15.150473594665527, -13.28823471069336, 6.188286781311035, -6.785267353057861, 9.037790298461914, 8.66299819946289, 7.949220180511475, -4.256028652191162, 5.186071872711182, 26.056821823120117, -13.812993049621582, -33.349308013916016, 28.544034957885742, 1.8707538843154907, -3.0454325675964355, 8.569928169250488, 12.402148246765137, 9.858887672424316, 1.203540325164795, 32.64139938354492, 1.5602186918258667, 14.098026275634766, -4.620050430297852, 9.553386688232422, -0.47243812680244446, 0.3184836804866791, -0.7178354263305664, -12.063436508178711, 27.76936149597168, -14.219047546386719, 0.14416790008544922, -11.871198654174805, 4.380549430847168, -13.090510368347168, 12.700180053710938, -5.171195983886719, 8.325287818908691, 10.077308654785156, -5.712687969207764, -37.09677505493164, 4.205341339111328, 26.13010025024414, -14.57844352722168, -8.384940147399902, 21.891393661499023, 14.690607070922852, 1.6417362689971924, 1.553902268409729, 5.304122447967529, -19.44343376159668, 1.6945666074752808, -8.93942642211914, -1.4031903743743896, 6.784534454345703, -39.19477462768555, 17.857473373413086, 6.141849040985107, -7.828449726104736, 8.167118072509766, -17.10999298095703, 11.200629234313965, 2.4491002559661865, 2.109121322631836, -0.8116300106048584]], "dtype": "float32"}}, {"a": [{"__nd__": [[-0.27645379304885864, -0.7689549326896667, 0.0414571613073349, -0.49288031458854675, 0.409775972366333, 0.7577093839645386, 0.7189847230911255, 2.4413673877716064, 0.8960291147232056, -0.9684327244758606, 1.2713515758514404, 0.2495608925819397, -0.6553853750228882, 0.9937117695808411, -0.35712355375289917, 1.1640278100967407, -1.225097417831421, -0.4045209288597107, 0.23546814918518066, -2.114354133605957, -0.694335401058197, 0.3057935833930969, 2.812478542327881, 0.009671643376350403, 0.5704991221427917, -1.4061251878738403, -0.4330752193927765, 2.194916248321533, 0.1857612282037735, -0.28531894087791443, 1.333789348602295, 1.7006926536560059, -1.6121479272842407, -1.2806782722473145, 0.7527258396148682, -1.1418025493621826, -0.31820744276046753, -1.3178280591964722, -0.6881751418113708, 1.3233619928359985, 0.4727805554866791, -0.8915767669677734, -0.7135109305381775, -0.2834744155406952, 1.1731775999069214, 0.6829369068145752, 0.6225023865699768, 2.2628700733184814, 0.10823100805282593, -0.06757351756095886, 0.4720732569694519, -1.3333169221878052, -1.1535654067993164, 1.5274064540863037, 0.44085803627967834, 0.5078003406524658, 1.3570122718811035, -1.202681303024292, -0.9991503357887268, 0.31972891092300415, -1.390241265296936, -0.3041171133518219, 0.3407200872898102, -0.252860963344574, -0.25217488408088684, -0.24261528253555298, 0.132660910487175, 0.9065446257591248, -3.1389272212982178, -1.510612964630127, 0.7160780429840088, -0.38632655143737793, -0.5035258531570435, -0.19147063791751862, -1.3041259050369263, 0.15732920169830322, -0.8196645379066467, -1.4018574953079224, 1.9958219528198242, 0.024507589638233185, -0.45565563440322876, 0.29949748516082764, 0.026568396016955376, 2.5100409984588623, -0.08625509589910507, -1.2136290073394775, -1.232940912246704, -2.67739200592041, 0.23493950068950653, -0.25056135654449463, 2.6099283695220947, -1.0616297721862793, -0.09072037786245346, -1.230929970741272, -0.5852723121643066, -0.9641890525817871], [0.7481147646903992, 0.3296716809272766, -0.4403088390827179, -0.4361031651496887, -3.007012367248535, -1.0975955724716187, 0.23198355734348297, 1.514647126197815, -0.8145979046821594, 1.415996789932251, 1.314540982246399, 0.3722347319126129, 0.8526737093925476, -0.49633434414863586, -0.9446033239364624, 1.4333397150039673, -0.04323214292526245, 1.1731683015823364, -0.6691028475761414, 1.4245485067367554, 0.2708745300769806, -0.031423911452293396, 0.5202351212501526, -1.1463433504104614, -0.18996864557266235, 0.43874499201774597, 0.31403377652168274, -0.19499340653419495, -0.3862135112285614, -0.17943990230560303, -0.46083542704582214, -0.4817076027393341, -0.7442362904548645, 0.6349596977233887, -0.7115988731384277, 0.26559171080589294, -0.2600364685058594, 0.6818313598632812, 1.1172925233840942, -0.4193984866142273, 0.2485491782426834, 0.1378214806318283, -1.7759504318237305, -0.9508237838745117, -1.507559895515442, -1.494184136390686, -0.3051341474056244, 1.1392829418182373, -2.1653473377227783, -0.4997321665287018, 0.0981748178601265, 0.43598315119743347, -0.03111395426094532, 1.6505857706069946, 0.48868080973625183, 1.6903825998306274, -1.2877321243286133, -0.6279018521308899, -1.0339614152908325, 1.7483001947402954, -1.357266902923584, 0.6099369525909424, -0.07464831322431564, 1.975293517112732, -2.092756748199463, 0.36325645446777344, 0.11759039759635925, 0.7759968042373657, -0.35409829020500183, -0.731533408164978, -1.0910052061080933, -0.4772122800350189, 0.7614571452140808, -0.89470374584198, 0.4252760410308838, 0.8166554570198059, -1.1715881824493408, 0.0045894114300608635, 0.8786965012550354, 1.0139964818954468, -0.19923216104507446, 0.1984672099351883, -1.1435883045196533, 0.36779728531837463, 0.40905964374542236, 2.7887327671051025, 0.7062831521034241, -0.3523607552051544, -0.044773854315280914, -1.0137910842895508, -0.11415617913007736, -0.19012349843978882, 0.7166991233825684, 1.1218953132629395, -0.02443341352045536, -0.5296716690063477], [-1.1707737445831299, 0.497458815574646, 0.13175828754901886, 0.9355738162994385, 0.12189851701259613, -1.7422337532043457, -0.5119315385818481, 0.3544447422027588, 1.4346803426742554, 0.4200025498867035, 0.5538464784622192, 0.7239682078361511, 0.9011513590812683, -1.142655372619629, 1.3386074304580688, 1.7224528789520264, -0.4312147796154022, 0.38463273644447327, -1.8804048299789429, -1.7657749652862549, 0.6017884612083435, -0.9121437668800354, 0.2696084976196289, 0.026887929067015648, 0.8552315831184387, 0.39981475472450256, 1.1865220069885254, -1.5795867443084717, 0.6597587466239929, -1.587040901184082, 0.6491071581840515, -0.14248354732990265, 2.927276372909546, 0.9062876105308533, -0.5482953190803528, -0.30641821026802063, -2.972719192504883, -1.3837867975234985, 0.6681807041168213, 0.39044836163520813, 0.6840630769729614, -1.5180628299713135, -1.0900720357894897, 0.4946855306625366, -0.6896681785583496, -0.9493265151977539, -1.1645125150680542, -1.660355806350708, -0.6811245679855347, -0.7220731377601624, -0.5370319485664368, 0.09488781541585922, 0.4868025481700897, -0.9292860627174377, 2.411810874938965, 1.192340612411499, -0.40779080986976624, -0.21295730769634247, -1.3025083541870117, -0.43694987893104553, 0.6068029999732971, -1.8407413959503174, -0.08364751189947128, 0.4749610126018524, 0.709701418876648, 0.2655651271343231, -0.4454095661640167, -0.10689660161733627, 0.06641386449337006, -0.1877366602420807, 0.4311883747577667, 1.7552940845489502, 1.3795894384384155, -0.07373154908418655, 0.0610990971326828, 1.0365257263183594, -0.039110034704208374, -1.1846705675125122, 1.0926018953323364, 1.2696338891983032, 1.4308332204818726, -1.6721839904785156, 0.39021825790405273, 0.061935510486364365, -0.2637607753276825, 0.013743337243795395, -1.0454131364822388, 0.538963794708252, -1.61459481716156, 2.9006619453430176, -0.5494241118431091, 0.8157771229743958, -0.7609995007514954, 0.6017890572547913, 0.07605662196874619, -0.18280690908432007], [0.1488669067621231, 1.3875857591629028, -0.4414026141166687, -0.07944782078266144, -0.29179847240448, -0.5095446109771729, -0.01437341421842575, 1.1583361625671387, 1.557712197303772, 0.26632893085479736, 1.0068614482879639, 0.46477237343788147, -1.1867854595184326, 0.4807727634906769, 1.71930992603302, 1.363021731376648, 0.10415900498628616, 0.4911886155605316, -0.2354421764612198, 1.2196742296218872, -0.31437215209007263, -0.5914247632026672, -1.3853856325149536, -0.15123596787452698, 0.4714415371417999, 0.08209194988012314, 0.17321205139160156, -0.5907170176506042, 0.8230938911437988, -0.22659888863563538, -0.36406564712524414, -0.927179753780365, 0.9581193327903748, 0.33135727047920227, -0.6116594076156616, 0.4768117368221283, -1.0588085651397705, 0.6513408422470093, -1.5432785749435425, 1.6552543640136719, -0.8543110489845276, -0.6617674231529236, -0.947309672832489, 0.22238142788410187, 0.8714784383773804, -0.709219217300415, -0.20356474816799164, -0.15959449112415314, -1.5150361061096191, -2.260197877883911, -2.3960421085357666, 0.5845546722412109, -1.5259177684783936, 0.11796706169843674, 0.26960697770118713, 0.8922653198242188, -0.8987663984298706, 1.040453314781189, -0.5799788236618042, -1.0359933376312256, 0.5919873714447021, -0.370236337184906, 0.014768478460609913, -0.020681997761130333, 1.8613839149475098, -0.3807951807975769, 1.2307668924331665, -0.38149189949035645, 0.21691440045833588, 1.4776191711425781, -0.3138079345226288, 0.9385027289390564, 0.6251732110977173, 0.8869732618331909, -2.0517783164978027, 0.8006747961044312, 1.1776288747787476, -0.3210274279117584, 0.018333200365304947, -0.20058861374855042, 0.6355490684509277, 1.2864960432052612, -0.1923695057630539, 0.6758694648742676, -2.3532257080078125, 2.661315679550171, 1.6380828619003296, 0.27278101444244385, -0.7898394465446472, -1.6560533046722412, 2.107886552810669, -1.3447636365890503, -0.46150821447372437, 1.2838295698165894, -2.7459404468536377, -2.3682897090911865], [-0.18068517744541168, -2.418142080307007, 0.5542165637016296, -0.849918782711029, -0.3696237802505493, 1.8506059646606445, -0.7342088222503662, 0.6310199499130249, 0.4122227430343628, 0.15901996195316315, 0.2528240978717804, -0.5170620083808899, 0.6904293298721313, -0.0561424158513546, -0.9540041089057922, -1.0450726747512817, 0.2649707794189453, 0.3777289390563965, 0.37995821237564087, -0.03599568456411362, -1.7353860139846802, -0.42220795154571533, 0.6766464710235596, -0.4990929365158081, 0.16592665016651154, -0.35382217168807983, 0.6718882322311401, 0.23173275589942932, 0.6297698020935059, -0.30480268597602844, -0.2155199944972992, 0.1559848189353943, 0.178805872797966, 1.4937033653259277, 0.6142382025718689, -0.31262362003326416, 0.09289443492889404, 0.5731073617935181, 0.0411786250770092, 1.3187508583068848, 0.03932661935687065, -0.03208640217781067, 1.3835148811340332, -2.7386913299560547, 0.9665548205375671, 2.328411817550659, 0.5941098928451538, -1.6433758735656738, 0.011863704770803452, -0.8190159797668457, 1.2287626266479492, 0.07931694388389587, 0.055304475128650665, -0.5112574100494385, -0.2733440101146698, 0.16588154435157776, -1.1433005332946777, -1.0490381717681885, -0.18429245054721832, 0.04334739223122597, 0.8499979376792908, 0.2819455862045288, 0.5117638111114502, 0.34986329078674316, 1.4274775981903076, 2.3547422885894775, -0.47993698716163635, 0.1278509497642517, -0.9686018824577332, -0.36347007751464844, 1.0857582092285156, 0.605820894241333, -0.6217265129089355, 0.06782805919647217, -2.5077037811279297, -1.0791569948196411, 0.735181987285614, -0.4620768427848816, -0.06210196390748024, 0.40270981192588806, 0.7259552478790283, -0.6754822134971619, 0.5164997577667236, -0.5840467810630798, -0.00449663819745183, 0.2434156984090805, -0.7659499049186707, -0.39167240262031555, 0.6292448043823242, -1.0721367597579956, -1.8925524950027466, -1.919933557510376, 1.0930919647216797, 0.31313076615333557, -1.2025924921035767, 0.5860520601272583], [0.4108226001262665, 0.15801800787448883, 1.688189148902893, -0.5631324648857117, -1.2569360733032227, 0.7064513564109802, -0.5762571096420288, 1.4065800905227661, 0.3334614336490631, -0.33365142345428467, -0.5812724232673645, -0.3830476999282837, 0.30740052461624146, 0.8335064649581909, 0.00475058751180768, -1.4975736141204834, -0.2279197722673416, 0.49022239446640015, -0.5844995379447937, -0.9324083924293518, -0.3006954789161682, -0.9593833088874817, 0.5570695400238037, -1.6321682929992676, 0.605808675289154, -1.9862213134765625, 0.7178143858909607, -1.619688630104065, 0.35614824295043945, -0.5275775194168091, 0.6755306124687195, -0.22212858498096466, 1.3583674430847168, -0.4587927460670471, 1.124226689338684, 1.6113518476486206, -0.2466128021478653, 0.020717978477478027, 0.6020125150680542, 2.343613862991333, 0.5831491947174072, -0.012156425043940544, 1.5522840023040771, -1.2867166996002197, -0.45436015725135803, -0.8951147198677063, -0.7265791893005371, 1.3385809659957886, -0.6860175728797913, -0.1779160350561142, -1.2948323488235474, -0.11108186095952988, -0.10734444111585617, -1.8329864740371704, -0.10556164383888245, 1.3636821508407593, 0.1537049114704132, 0.8788217902183533, -1.255010724067688, 0.2045520842075348, 0.7346373200416565, -1.1296210289001465, -0.5516979098320007, -1.567376732826233, 0.6366881132125854, -0.40546080470085144, -0.3358743488788605, -0.41244015097618103, -2.3910858631134033, 0.30519333481788635, 0.4038330018520355, -0.6591552495956421, 2.1371734142303467, 0.871143639087677, 0.09128990769386292, -0.6802337765693665, 0.01720578223466873, -2.361349582672119, -0.9490720629692078, 0.2094707489013672, 0.6966251730918884, -1.2581814527511597, -0.87346351146698, 1.0447098016738892, -0.41130056977272034, 0.4108692407608032, 0.5480772256851196, 1.966154932975769, 0.6589744687080383, -0.8150604963302612, 1.2669821977615356, 1.046799659729004, -0.4939875900745392, 0.06714294105768204, 0.7876707911491394, 1.1188167333602905], [0.21832863986492157, -0.6945196986198425, 1.2072584629058838, -0.7713683843612671, -0.9187902212142944, -1.372784972190857, -0.04524221271276474, 0.5863187313079834, -1.05612313747406, -0.3872241973876953, 0.46865949034690857, -0.1771385818719864, 0.21493729948997498, -0.5840147137641907, -1.0142422914505005, -0.9225651025772095, -1.6484336853027344, -0.9431276917457581, 0.0560426265001297, -0.328369677066803, -3.286555767059326, -1.2113096714019775, -0.8864470720291138, -0.8436408638954163, 0.477193683385849, 0.5614238381385803, -0.36969414353370667, 0.4082396924495697, 2.1846139430999756, 1.7484328746795654, -0.577116847038269, 1.2160929441452026, 0.34677472710609436, -2.120149612426758, -0.29034921526908875, 0.8668252229690552, -0.6416826844215393, 0.7847292423248291, -2.249685525894165, -0.4959360659122467, -0.43185487389564514, -0.19182154536247253, -0.7651196122169495, -0.7548660635948181, 0.9376471638679504, -0.32954269647598267, -0.6117268204689026, -0.6594452857971191, 0.01749165914952755, -0.16122260689735413, -1.1372624635696411, -1.7214113473892212, 0.01839778572320938, -0.5250157117843628, -1.1477299928665161, -0.4596555829048157, -0.48884740471839905, 1.5360907316207886, -0.7509170174598694, -0.4515184462070465, -1.7443668842315674, -0.9342021346092224, -0.5805354714393616, 0.3444576859474182, 0.003967731259763241, 0.151511549949646, -2.574047088623047, 0.16387389600276947, -1.904375433921814, 0.1293102651834488, 0.5001022219657898, 1.892374873161316, 0.41019803285598755, -0.720788300037384, 0.1358448714017868, -0.8564618825912476, -0.5160406827926636, -2.353440046310425, -0.6531738042831421, 1.0049463510513306, 1.701687216758728, -0.5596168637275696, -1.0428959131240845, 1.1293048858642578, -0.7023178935050964, -1.0242191553115845, -1.1358237266540527, 2.486865997314453, -0.07986710220575333, -0.7619029879570007, 0.45381587743759155, 1.3542969226837158, -1.4418482780456543, 0.49106064438819885, -0.9380824565887451, 0.6710947155952454], [-1.8482528924942017, 1.0631446838378906, -0.459200918674469, 0.04184122011065483, -0.09742239862680435, -0.7728593945503235, 0.73299241065979, 1.0232402086257935, -0.5287643074989319, 0.45015525817871094, -0.7383673191070557, 0.3555351793766022, 1.115859031677246, -0.12423761188983917, -0.06988602131605148, 0.4285835921764374, -2.029092788696289, -1.5949772596359253, 0.029081257060170174, 0.3926602303981781, 0.15418687462806702, -1.0218794345855713, -0.15345622599124908, -0.26490187644958496, -0.5837549567222595, -2.068284511566162, 0.9136496782302856, -1.0549969673156738, -1.1652402877807617, 0.42929208278656006, -0.863098680973053, -0.2651337683200836, -0.3485746383666992, -2.363896369934082, -0.29993870854377747, -1.2984619140625, -0.5906247496604919, 1.285041093826294, -2.445531129837036, -0.5723777413368225, -0.038963235914707184, -0.3952183425426483, -0.8183613419532776, -1.4358255863189697, 0.6161515712738037, -0.6025994420051575, -0.16763627529144287, -2.0349161624908447, 1.0304861068725586, 0.042412154376506805, 1.1179500818252563, -0.7264935374259949, -0.7262828946113586, 1.5158623456954956, 0.7472640872001648, 0.7012292146682739, -0.07093080133199692, -0.3015036880970001, -0.759614884853363, -0.12615783512592316, 0.3479064404964447, 0.39511778950691223, 0.4924944043159485, -1.3019416332244873, -0.9459325671195984, -0.9414806365966797, -0.2343616932630539, 0.4828103482723236, -0.8636000752449036, 0.46385785937309265, 0.09154260158538818, 2.1413164138793945, 0.18580546975135803, 0.3503434360027313, -0.639268159866333, -1.2640204429626465, 0.9469173550605774, 0.08808065205812454, -0.19559814035892487, 0.5878182649612427, 0.033081959933042526, 0.5424982309341431, 0.674892783164978, 0.7406521439552307, 1.2716277837753296, 0.2726024091243744, 0.7179449796676636, 0.27316272258758545, 0.90658038854599, 0.9230347275733948, -1.2820823192596436, 1.2189208269119263, 2.019665002822876, -0.8138769865036011, 0.39569661021232605, -1.0788615942001343], [-0.4068543314933777, -0.7516087293624878, 0.8311585783958435, -0.1703687161207199, -0.7577875852584839, -0.3254649341106415, -0.37978100776672363, 0.8049561977386475, -2.2298247814178467, 0.5093698501586914, -0.5444512367248535, -0.31516408920288086, 1.62064790725708, 0.08380278199911118, 0.5329112410545349, -2.5342631340026855, 1.189959168434143, -0.5730280876159668, 0.368677020072937, -2.3445210456848145, -1.270348310470581, 0.6997841000556946, -1.1449167728424072, 0.6343403458595276, 0.3517487049102783, -1.2488399744033813, -1.2354422807693481, 0.3748120665550232, -1.4751697778701782, -0.7547057867050171, 0.6067257523536682, 0.41153448820114136, 1.7402276992797852, 0.5432680249214172, -0.004501526243984699, 1.9153960943222046, 0.07758026570081711, -0.3673504590988159, -1.544999361038208, 1.0123088359832764, 0.08725189417600632, -1.1034224033355713, -1.2177948951721191, -0.07341734319925308, 0.9790316224098206, -0.09674125164747238, -1.6219953298568726, 0.19730478525161743, -0.5602473020553589, -0.5235547423362732, -0.3855536878108978, -1.3928927183151245, 0.4589788615703583, -2.306816577911377, -0.20366494357585907, -1.8713093996047974, -0.8345761895179749, -0.920279860496521, -0.5633164048194885, -0.005128561519086361, -0.056175749748945236, 0.8030299544334412, 0.7227885723114014, 1.8622344732284546, 0.039168741554021835, 0.960352897644043, -0.2088400274515152, 1.5888607501983643, 1.2911549806594849, -0.3022792935371399, -0.8531040549278259, 0.36850517988204956, 0.3857954442501068, 1.603928565979004, 0.7792921662330627, -0.8624860644340515, -0.05063853785395622, -0.2566746175289154, 0.45492643117904663, 0.07808136194944382, 1.8329085111618042, 1.6669518947601318, -0.2871577739715576, -0.17130924761295319, 0.1226416751742363, -0.06594084203243256, -1.8347408771514893, -0.1739111840724945, -0.15194161236286163, 0.012305299751460552, 0.4801565706729889, 0.9581499099731445, -2.013852596282959, 0.2809615135192871, 0.5405066609382629, -1.0357041358947754], [-0.6740044355392456, 0.4488801956176758, 0.4319309890270233, -0.1766384094953537, -0.4407304525375366, -2.1256768703460693, 0.6274816989898682, 0.411162793636322, 0.9241509437561035, -0.6585179567337036, 0.13086169958114624, 0.10923223942518234, -0.18648934364318848, -0.36643773317337036, -2.495614528656006, -0.6093129515647888, -0.1532949060201645, 0.6156550645828247, -0.15286260843276978, -0.181029811501503, 1.2808010578155518, -0.031206389889121056, -0.3550422787666321, 0.016299717128276825, -0.50529944896698, -0.7486199140548706, -0.5245221853256226, -0.9136952757835388, -0.7296411991119385, 0.6731123328208923, -0.6695553064346313, 1.9548685550689697, 1.3878811597824097, 0.2665621340274811, 0.25019556283950806, 0.3724477291107178, 1.2799447774887085, 0.04629998281598091, -0.3333689272403717, 1.844721794128418, -0.6460506916046143, 1.8941417932510376, -1.0331467390060425, -0.6967341303825378, -1.335434913635254, 1.681182861328125, 1.4228129386901855, 0.8167903423309326, 0.15331952273845673, 1.67823326587677, -0.4184912443161011, -0.7936837673187256, 0.6203522682189941, 0.328872412443161, 0.7391726970672607, -0.4925108551979065, 1.3110134601593018, -0.19647404551506042, -0.5338186025619507, -0.5136200189590454, 0.9709736108779907, 1.124755859375, 0.9555904269218445, 0.8118875622749329, -0.30818474292755127, -0.6551848649978638, -0.20614729821681976, -0.8816419839859009, -1.0918495655059814, -0.07203111052513123, 0.21193185448646545, -0.7936634421348572, 0.27891698479652405, 0.774847686290741, 0.9093397855758667, -0.5958817005157471, 0.5982213020324707, 0.8354969620704651, 0.5893101692199707, -0.43218570947647095, 2.753746747970581, 0.3671974539756775, -0.3861980140209198, 1.3288074731826782, -0.5551825761795044, 1.3779637813568115, -0.9180988073348999, 1.7162424325942993, 0.10296966880559921, -0.9745306968688965, -0.4742854833602905, -0.1996409296989441, -1.5759254693984985, -0.12090606987476349, -0.690902590751648, -1.1209903955459595], [-1.3039382696151733, 2.5564613342285156, -0.6662469506263733, 0.5700178742408752, -0.3122805655002594, 0.49322500824928284, 0.852222740650177, -0.5945652723312378, 1.0072555541992188, 0.3385697901248932, -1.4525913000106812, -2.955590009689331, 0.5659007430076599, -0.5460560917854309, 1.3979274034500122, -0.9025808572769165, -0.34361374378204346, 0.09714676439762115, 0.8632901310920715, -0.5293352603912354, -0.6536536812782288, 0.6586030125617981, 1.102678656578064, 0.018772495910525322, 1.6096971035003662, 2.366945505142212, -0.15210413932800293, 0.11315250396728516, 0.10498572140932083, 0.363538920879364, -0.033352818340063095, -0.7595855593681335, 1.1622722148895264, -1.8851033449172974, 0.9238016605377197, 0.24226044118404388, -0.4292983114719391, -0.14552198350429535, -0.22877198457717896, 0.4454345703125, -0.34060245752334595, 0.09970035403966904, 0.9002090692520142, -0.40011152625083923, 1.1134973764419556, -1.7984719276428223, 1.1579034328460693, -0.4738818109035492, 0.8878548741340637, 1.2606099843978882, 1.299629807472229, 0.28102099895477295, -0.41690051555633545, -0.22170482575893402, -1.898881196975708, 0.1999867558479309, -1.1443372964859009, 1.0205063819885254, -0.6526796817779541, 1.0361589193344116, 1.0368632078170776, -1.3781691789627075, 0.010946547612547874, 0.35987603664398193, -1.0577481985092163, -1.8393585681915283, -0.6391292214393616, 0.4599801301956177, -1.5164830684661865, -0.9570599794387817, -0.5067881345748901, 1.3054230213165283, -0.0047010439448058605, 1.138850212097168, -0.22635246813297272, 1.7832273244857788, -0.6328508853912354, -1.575466275215149, 0.6642293334007263, 1.2293858528137207, -0.9549384713172913, -0.143446683883667, 0.7901439070701599, 0.23881162703037262, -1.1933848857879639, -1.1178605556488037, 0.4081766605377197, -0.0713234692811966, 1.2880853414535522, 1.2058409452438354, 0.8293612003326416, -1.5891501903533936, -0.36428266763687134, 0.3053019642829895, 0.886002242565155, -0.686901330947876], [-0.6590227484703064, 0.5611188411712646, 0.10161955654621124, 0.18518009781837463, 0.9623098969459534, 0.4564954936504364, 2.057990550994873, -1.2391868829727173, -0.46653059124946594, -1.180609107017517, -0.8098660111427307, 0.38384056091308594, -0.18119965493679047, -0.8776398301124573, 0.6104524731636047, 0.055202044546604156, 0.20468471944332123, 0.6366235017776489, -1.2982664108276367, -2.0141139030456543, 1.390817642211914, -0.41861531138420105, 1.3575170040130615, -0.5275751352310181, -2.3161063194274902, -0.9422464966773987, -0.2768186926841736, 2.0947277545928955, -0.02081216871738434, -0.48998039960861206, 0.615131676197052, -1.16947340965271, -0.4886116683483124, -0.9544655680656433, -1.1437299251556396, -1.1750166416168213, 0.26900142431259155, 0.2851836085319519, 0.48228201270103455, 1.517268419265747, 0.46719786524772644, -0.3236735761165619, -0.733400285243988, -0.7278527021408081, 0.6617677211761475, -1.3377876281738281, 0.942865788936615, 0.11088649928569794, -0.4912562966346741, -0.03687596321105957, 0.5225628614425659, -0.37896469235420227, -0.5875477194786072, -0.5569964647293091, 1.0076767206192017, 0.5816490054130554, 0.7980709075927734, 0.9320793747901917, -2.1257481575012207, 1.6285102367401123, -0.5973432660102844, -0.8639882802963257, 1.6337977647781372, 2.2743349075317383, -1.6430764198303223, -1.0118857622146606, 1.4095534086227417, 0.018554596230387688, 0.0002928582835011184, 1.020811676979065, -0.12643322348594666, 1.7679086923599243, -1.3957781791687012, 1.1078027486801147, 0.013888172805309296, 0.31082335114479065, 0.6273050904273987, 0.4753476679325104, 0.5714151263237, -0.3888026177883148, -0.2420712560415268, 0.2944316864013672, 1.9311717748641968, 1.9049609899520874, 0.24350689351558685, 2.4198262691497803, -0.7692108750343323, -0.605573296546936, 0.2851685583591461, -0.28916001319885254, 0.8071573376655579, 0.78233402967453, -0.1512317657470703, -1.870089054107666, 0.6132981777191162, -1.1396448612213135], [0.2734498679637909, 0.40216052532196045, -0.25760123133659363, 0.3743344843387604, 0.637567400932312, -0.30743974447250366, -0.571424126625061, -0.04231516271829605, -0.3792320489883423, 0.11744367331266403, 0.2211356908082962, -1.510091781616211, -0.9544606804847717, 0.5765870809555054, 1.6042848825454712, 0.6030030250549316, -1.4366881847381592, -0.14441201090812683, 0.2610595226287842, -0.7585487961769104, -0.8334758281707764, -0.5205111503601074, 0.7369444370269775, 0.48483097553253174, -0.4748370051383972, 1.4349156618118286, -0.36825674772262573, -1.6259901523590088, -0.3909232020378113, -0.028603265061974525, -0.7931092381477356, -0.6858691573143005, -0.21502366662025452, 0.028740456327795982, -1.9501913785934448, 0.2451600432395935, -0.8712493777275085, 0.7768879532814026, 1.542744755744934, 0.6179870367050171, 1.6715312004089355, 0.32909920811653137, -0.4076724052429199, 0.39658159017562866, 0.09819269925355911, -0.8644938468933105, -0.19761396944522858, -0.02384556457400322, 2.3928847312927246, -0.14614006876945496, 0.7826143503189087, -0.08109825104475021, 0.6949977278709412, 2.3845818042755127, 0.3833928108215332, 1.3341206312179565, 1.6587499380111694, 0.19044864177703857, -0.13922366499900818, 1.1351770162582397, -0.4180268347263336, 0.8354033827781677, 0.6934441924095154, -0.5427375435829163, 0.24025408923625946, 0.16222605109214783, -0.5145395994186401, -0.9089929461479187, 0.1715899556875229, -0.31094253063201904, -0.5638979077339172, 0.0393572673201561, -0.3616621494293213, 1.1280652284622192, -0.7346336245536804, 0.927932858467102, -1.5732414722442627, -0.024430524557828903, 0.2287205010652542, -1.3988418579101562, -0.45446357131004333, -0.434876948595047, -0.810184895992279, 0.05773314833641052, 0.6888547539710999, -0.4962466061115265, 1.7216907739639282, 0.16016322374343872, -0.2860731780529022, 0.66422438621521, -1.4226672649383545, 0.6287292838096619, 0.08537040650844574, -0.91424959897995, -1.6472156047821045, -0.18109673261642456], [0.6880481243133545, 0.2100573182106018, 1.3767822980880737, -0.5782783627510071, 0.21582883596420288, -0.7017017006874084, 0.05751883238554001, 0.4133877456188202, 1.1769485473632812, 0.6467233300209045, 0.6348134875297546, -1.2479939460754395, -1.7779500484466553, 1.1592389345169067, 1.0678011178970337, 1.4338481426239014, -0.2748855948448181, 2.340796709060669, -0.19727864861488342, -0.6839185357093811, 0.1292978823184967, 0.8552232384681702, -0.359192818403244, -1.5617481470108032, -0.040054235607385635, 0.24089740216732025, -1.0505671501159668, 0.6802206635475159, 0.060013677924871445, 1.1908655166625977, 0.27767184376716614, -0.3043728768825531, -0.49269193410873413, -1.1154621839523315, 0.26693397760391235, -0.8956335186958313, -0.41374826431274414, -0.5571449398994446, -0.4374653995037079, 0.037525489926338196, 1.4081439971923828, -1.3899534940719604, -0.7046347260475159, 1.1649155616760254, -0.18032829463481903, 0.9543678164482117, -1.8343534469604492, -0.5891697406768799, -0.03997372090816498, -1.4476535320281982, 2.2887392044067383, 1.2242149114608765, -1.3532099723815918, -0.29765209555625916, -2.3864552974700928, 0.16026295721530914, 1.2472351789474487, -0.45321205258369446, 1.0385531187057495, 0.37665003538131714, 0.28535768389701843, 0.1789456307888031, -0.3498774766921997, -0.40159159898757935, -0.4123463034629822, -0.41262486577033997, 0.24825476109981537, -1.0670195817947388, 0.5716844201087952, 0.12617838382720947, -0.40516331791877747, 1.3740936517715454, -0.0752822607755661, -0.7638674378395081, -0.15955737233161926, -1.0342320203781128, -0.31416505575180054, 0.9826768040657043, 1.2261605262756348, 0.46219852566719055, 0.49182814359664917, 0.31408560276031494, 1.518263816833496, 2.4418044090270996, -0.5557176470756531, 1.4430123567581177, -0.506392240524292, 0.955554723739624, 0.294481486082077, -0.9209710359573364, 1.6628199815750122, -0.30334869027137756, 0.5758237838745117, 2.264295816421509, 0.2687550187110901, 0.6608520150184631], [-0.5975197553634644, 1.6203774213790894, 1.036271572113037, -0.05744960904121399, 0.8127167224884033, -0.43342798948287964, 0.024352598935365677, 0.2254824936389923, -1.300865888595581, 0.4688790738582611, 0.950425922870636, 0.23820136487483978, 0.07363928109407425, -1.4404375553131104, 1.1854918003082275, -0.3944622576236725, -0.2739732265472412, 0.7152099609375, -1.1875053644180298, -0.11070261895656586, -0.17469969391822815, 0.05307293310761452, -1.2923157215118408, -0.8433922529220581, 0.29867470264434814, 0.9412180185317993, -0.11025442183017731, 0.680888295173645, 0.7690718770027161, 0.13871708512306213, -0.21557044982910156, -0.7996760010719299, -0.4417455792427063, 0.887837827205658, -0.5658425688743591, 0.504900336265564, 1.578142523765564, -0.18745966255664825, -1.0249215364456177, -1.348490595817566, -0.9945880770683289, -0.4660996198654175, 0.8519219160079956, -0.9708896279335022, -0.11856624484062195, -1.0085240602493286, -0.11521628499031067, -2.0230798721313477, 0.22600802779197693, -0.45888784527778625, 0.2964825928211212, -0.06620296090841293, 1.2233937978744507, 1.8400847911834717, -0.06423939764499664, -0.1630953997373581, 0.2323806881904602, -0.11913822591304779, -0.2157791554927826, -0.9301851391792297, 0.5074504017829895, -0.5687242150306702, 1.1124353408813477, -0.016146603971719742, -0.4769642949104309, 0.4549461603164673, -2.3603322505950928, -1.601874589920044, 0.26999402046203613, -1.0543203353881836, -0.8952094912528992, -0.31746625900268555, 0.510918915271759, -0.42725783586502075, -1.0007050037384033, -2.1046018600463867, -0.41678759455680847, 0.6680504679679871, -1.2487175464630127, -0.6412573456764221, -1.5362932682037354, 0.4794062674045563, 0.7798454761505127, 0.18018028140068054, 1.8067874908447266, 0.8694618344306946, -1.205100655555725, -1.5104093551635742, 0.8751910924911499, 2.1072170734405518, -0.7104512453079224, -1.411624550819397, -0.0959727093577385, -1.3525782823562622, -2.0046300888061523, 0.37301063537597656], [-1.3205764293670654, 0.20394226908683777, -0.2519756257534027, 0.1186985895037651, 2.6325342655181885, -0.10224272310733795, 0.45780134201049805, 1.068542242050171, 1.064026117324829, -1.0237159729003906, 1.3136482238769531, 0.4487760663032532, -0.7437891364097595, 1.0491777658462524, -1.4448111057281494, -0.45173829793930054, -0.14202003180980682, 0.1829172819852829, 0.5641699433326721, -0.41004136204719543, -0.6113080978393555, -1.9809281826019287, 1.3119707107543945, 0.5463519096374512, -1.3115566968917847, 0.9226634502410889, 0.947433590888977, -1.737826943397522, -0.7350765466690063, 0.9810465574264526, -0.763786792755127, -0.32956188917160034, 1.4587117433547974, -0.9882137179374695, -0.9474596381187439, -0.4451904296875, 1.3690638542175293, 0.1285199224948883, -2.6620287895202637, -1.065667748451233, 0.7020594477653503, -1.1899919509887695, 0.4969276487827301, 0.5067465305328369, -0.8669160604476929, -0.7362019419670105, -1.0535366535186768, 0.5807414054870605, 1.795230507850647, -0.7809423208236694, 0.05939413234591484, -1.0084680318832397, 1.2244206666946411, 1.281410813331604, 2.0746498107910156, -0.2548905313014984, 0.9190056920051575, 0.26306161284446716, 0.2512686252593994, 0.5189594626426697, -0.7051061391830444, 0.564348042011261, -0.908517062664032, 1.5489517450332642, 0.512791633605957, 0.915321409702301, -1.5833563804626465, -0.5883464813232422, -1.326911449432373, -0.6166518926620483, -0.6803349852561951, -0.661377489566803, -0.4145030379295349, -0.616101086139679, -1.0697910785675049, 0.7332568168640137, -0.2332136482000351, -1.9867630004882812, -1.6602298021316528, 0.9399586319923401, 0.22823576629161835, 0.47396641969680786, -0.4448068141937256, -1.704668641090393, 0.7096846699714661, 0.8250577449798584, -0.2318834513425827, 1.6355586051940918, -0.5371367335319519, -0.5553110837936401, 0.4960288405418396, 1.604606032371521, 0.8231845498085022, -0.43015390634536743, -0.027256740257143974, 0.17444518208503723], [-1.300257921218872, 0.7845257520675659, -0.9126979112625122, 0.8594087958335876, -1.1310136318206787, 0.7327052354812622, 0.3247471749782562, 0.2344178408384323, 0.17637188732624054, 0.6673658490180969, -1.5583299398422241, 1.5019325017929077, 0.25665199756622314, -0.011619245633482933, -0.3628641664981842, 1.9729464054107666, -0.4030420482158661, 0.4600435793399811, -0.18585628271102905, -1.0563507080078125, 1.6382941007614136, -2.6694486141204834, 1.4151495695114136, -0.595334529876709, 0.6104205250740051, -0.9702388644218445, -0.4569377303123474, -1.0428706407546997, -0.2215392142534256, 1.4268081188201904, -0.10749562829732895, -1.1186506748199463, 0.016937987878918648, -0.9445995092391968, 0.8435233235359192, 0.2924235463142395, -0.9104034304618835, 0.7695252895355225, 0.7702707052230835, 0.2812855541706085, 0.08079338818788528, 0.26521435379981995, -0.8489736318588257, -0.6096097826957703, 0.3792983889579773, -0.535067081451416, 1.1793056726455688, 0.9292442798614502, -1.0621745586395264, -0.438414067029953, -0.4377315640449524, -0.0933079645037651, -1.2696186304092407, 0.16569621860980988, 0.4321155548095703, -0.06203273683786392, -0.9453105330467224, -0.5294575691223145, 0.9514954090118408, -0.43518170714378357, 0.690314531326294, -0.7595282196998596, 0.7620150446891785, 2.040865182876587, 0.010801948606967926, 0.6526072025299072, -0.114983469247818, 1.7744123935699463, 0.18140311539173126, -1.2005056142807007, 0.7340356707572937, 0.7031705975532532, 0.20728036761283875, -0.33747878670692444, -0.7212390303611755, 1.4718244075775146, -1.9648370742797852, 0.647759199142456, -0.07874830067157745, 1.5948923826217651, -0.2399609237909317, 1.021246075630188, 0.33652129769325256, 1.0500677824020386, 1.0251383781433105, 2.908494472503662, -2.203562021255493, -1.3295994997024536, -1.767793893814087, -0.39696022868156433, -0.16114753484725952, 1.4328199625015259, -1.079055666923523, 0.36535266041755676, -0.24516580998897552, 0.28032588958740234], [0.3507102131843567, -0.3971508741378784, -0.7719241976737976, -0.088943712413311, -1.1912686824798584, 0.5711201429367065, 0.25236502289772034, -0.04898883402347565, -0.11612094193696976, 1.2804702520370483, -1.0712445974349976, 0.00899540726095438, -1.2058923244476318, 0.8887360095977783, -0.3220941126346588, 1.684586763381958, 0.7688735127449036, 0.6721461415290833, 1.3530458211898804, -0.8527154922485352, 0.3889405429363251, 0.9657424688339233, -0.06433514505624771, -0.7548753619194031, -0.8376419544219971, 0.2193356454372406, 0.9995116591453552, -0.6021875739097595, 1.017642855644226, -0.6450423002243042, 0.16171292960643768, 3.3736422061920166, -0.9968662858009338, 0.34348541498184204, -1.7335219383239746, -0.020309656858444214, -1.0041472911834717, -1.1443902254104614, 0.5663360357284546, 0.285048246383667, -2.59688663482666, 0.21036548912525177, -0.11877920478582382, -1.6462018489837646, 2.03810715675354, 1.2198125123977661, 0.3028339743614197, -0.6380642652511597, 0.6043571829795837, -0.2350829690694809, -0.9577810764312744, 1.4523367881774902, -0.9048483371734619, 0.3989526033401489, -1.040285587310791, 0.5590953826904297, -1.318355679512024, 0.9791898131370544, -0.2743717432022095, 0.9833743572235107, -0.621310830116272, 0.23199626803398132, 0.9377530813217163, -0.4207976758480072, -0.7686824202537537, 1.3803393840789795, 0.5215739011764526, 0.49125543236732483, 1.443385124206543, -1.2816435098648071, -1.0613714456558228, -0.7605021595954895, -0.21189071238040924, -1.7939296960830688, 1.2919145822525024, -0.8046998381614685, 0.8818362355232239, 0.6130934357643127, -0.9768697023391724, -1.3040034770965576, -1.48017156124115, -0.32288283109664917, 1.9770939350128174, -0.9085142016410828, 0.24720855057239532, -0.35826900601387024, 0.8707070350646973, 1.365958571434021, -0.41896966099739075, 0.8261814117431641, 0.4319702684879303, -0.13849341869354248, -0.5361097455024719, -0.05448663979768753, 1.817436695098877, -0.7747280597686768], [-0.6377712488174438, -0.10361050069332123, -0.7680198550224304, 0.9424458146095276, -0.11257556080818176, 0.9148154258728027, -0.3320070207118988, -0.28669095039367676, 2.1896519660949707, 0.4748496115207672, -1.3731744289398193, -0.2233646661043167, 1.5068224668502808, -1.9495067596435547, 0.8594824075698853, 0.6486930847167969, -1.94642972946167, 0.5133237242698669, -0.18859852850437164, 0.5359101891517639, 0.7035180926322937, 0.9175084233283997, -1.155808687210083, 1.065863847732544, 0.9670650959014893, 1.2769776582717896, -0.1354006975889206, 0.19987013936042786, -0.13999326527118683, 1.1417573690414429, -0.301193505525589, 1.492620587348938, -0.17617911100387573, 2.0899415016174316, -0.5348164439201355, -0.9953441619873047, -0.05884455516934395, -0.2944449484348297, -1.3139119148254395, -0.3395618498325348, 2.129746437072754, 2.095473051071167, -0.46065255999565125, -1.5028448104858398, 1.0954530239105225, 1.4761593341827393, -0.559928297996521, -1.1913516521453857, 0.34671324491500854, 0.5070645809173584, 0.3165777623653412, 0.10865932703018188, 0.8136459589004517, -0.3543456196784973, -1.5161936283111572, -0.01768915355205536, -1.4635573625564575, 0.11196818947792053, -0.29608842730522156, -0.21706703305244446, 1.011982798576355, 0.4754776060581207, -0.32808926701545715, 0.636496901512146, -0.3162352442741394, -0.7343893647193909, 0.45452287793159485, 0.10571040958166122, -2.3696930408477783, -0.38091835379600525, 0.13967853784561157, -1.5261801481246948, 1.1720942258834839, -1.2869973182678223, 0.8516026139259338, -1.0807228088378906, -1.3252547979354858, -1.0133925676345825, -0.03432002291083336, -1.4270858764648438, 1.7297240495681763, -0.6777264475822449, 0.7835481762886047, 0.5171149969100952, 0.015241600573062897, 0.48155131936073303, 0.8922607898712158, 0.6413559317588806, 1.197447419166565, 0.7313713431358337, -2.074061632156372, -0.6406228542327881, 0.6840655207633972, 0.6046324372291565, 1.9805070161819458, -1.7276856899261475], [-0.8078844547271729, 0.45082390308380127, -0.33087974786758423, -0.49249905347824097, 0.9540289044380188, -1.1228574514389038, 0.9050134420394897, 0.8058010339736938, -0.6465059518814087, 0.781900942325592, -1.2908142805099487, 0.6059102416038513, -1.5236077308654785, 1.3255338668823242, 0.7992982864379883, 1.5331745147705078, -0.28880149126052856, 1.195992350578308, 0.3885781466960907, -0.15455874800682068, 0.2494548112154007, -0.3332739770412445, -1.2742178440093994, -0.3137848377227783, -0.5946672558784485, 0.16727478802204132, -0.9993545413017273, -1.1750845909118652, 1.0473477840423584, 2.482276201248169, 0.7410476207733154, 1.6779299974441528, -2.0772218704223633, 1.116518497467041, 0.6807515621185303, 0.8518733382225037, -1.156259298324585, -0.3016059994697571, -0.036414846777915955, 1.2376371622085571, -0.7471173405647278, -2.6238813400268555, -0.08842344582080841, -0.3152175843715668, 0.28115108609199524, 0.1161302775144577, 0.13011345267295837, -0.9627794027328491, 0.7519188523292542, -0.41588127613067627, 0.6743442416191101, 0.19780950248241425, -0.5397305488586426, -1.3526071310043335, 1.0302734375, 1.3899284601211548, -0.24470064043998718, -0.03884357586503029, 0.5017064213752747, 0.8522018194198608, 0.2950669229030609, 1.1518380641937256, -0.025225123390555382, 0.3745293915271759, 0.46885642409324646, 0.4401194155216217, -0.4344576895236969, -1.825671672821045, -1.7673078775405884, 0.08945625275373459, 0.6367353796958923, -0.41316935420036316, -0.6501088738441467, -1.4869465827941895, 0.32406923174858093, -0.2832217216491699, -0.7556067705154419, 0.0750090479850769, 0.860317051410675, 0.36170634627342224, -0.6753795742988586, -0.1560618281364441, 1.3555930852890015, 1.4680782556533813, 1.0161229372024536, 1.1578233242034912, 0.2122134119272232, -0.3593944311141968, 1.0367921590805054, -0.34404510259628296, -0.2043628990650177, 0.21207746863365173, 0.46974438428878784, 0.7072058320045471, 0.012773199938237667, -0.3003244698047638], [1.0256930589675903, 1.2949376106262207, 1.286942958831787, 0.018920475617051125, -1.4043294191360474, -1.1270830631256104, -0.8161957263946533, 1.4603837728500366, 0.5502263903617859, -1.8323789834976196, -0.5719563961029053, -1.5177552700042725, -0.11482266336679459, -0.6060507893562317, -0.8313082456588745, 0.28625547885894775, -0.5928505063056946, -0.7576858997344971, 0.3532276451587677, -0.43613889813423157, 1.3593177795410156, 0.3607422113418579, 0.6085226535797119, -0.6215137243270874, -0.6064409017562866, 0.39575985074043274, 2.61273455619812, -1.0171115398406982, 0.24311526119709015, -0.5093207359313965, -1.322141170501709, -0.6915075778961182, -0.4572872221469879, -1.0489097833633423, 0.7901689410209656, 0.06993638724088669, -1.4816049337387085, -0.4041818678379059, 1.0713870525360107, 0.9520067572593689, -0.047671567648649216, 0.39087510108947754, -0.5425618290901184, 0.19384074211120605, 0.4335525631904602, -1.8067890405654907, -1.1117804050445557, -0.5327199697494507, 1.9101669788360596, 0.9178078174591064, -1.0255178213119507, 0.3268800675868988, 0.8253982663154602, -1.183795690536499, 1.033386468887329, -0.5171437859535217, -0.33993515372276306, 1.8581252098083496, 0.25846871733665466, -0.4389859437942505, -0.023431502282619476, 1.2903112173080444, -1.6587227582931519, 0.23045411705970764, -1.0512818098068237, 1.4287899732589722, 0.23093155026435852, 1.688469648361206, -2.4109156131744385, -0.9911978840827942, 1.7232943773269653, -1.2981206178665161, -0.17637485265731812, -0.8237600326538086, -0.03701438754796982, -0.30323541164398193, 1.0910112857818604, -1.439220666885376, 0.6838465332984924, 2.0490317344665527, 0.3588336408138275, -1.0882072448730469, -0.9127660989761353, 0.9815889596939087, -0.5829737782478333, 1.2328908443450928, 0.48577141761779785, 1.5748772621154785, 0.5866304039955139, -1.0293552875518799, 1.0999853610992432, -0.5684170722961426, -0.44178855419158936, -2.175802707672119, -0.600926399230957, -0.3374224901199341], [-0.8355675339698792, -1.08591628074646, -0.563302755355835, -1.200291395187378, 0.11009657382965088, -0.22442907094955444, 0.07791376858949661, -0.3174171447753906, 2.135545492172241, 0.27703380584716797, -0.8740128874778748, 0.05641757324337959, 0.9543026685714722, 0.06357940286397934, 0.1186247318983078, -0.7924112677574158, -1.0081007480621338, 0.04158145934343338, 0.7859076261520386, -2.2561306953430176, 0.4890182912349701, -0.41863542795181274, -0.42831212282180786, -1.537959337234497, 0.9812547564506531, 2.5677425861358643, -0.7610471248626709, -1.033475637435913, 0.668587863445282, 0.3666875660419464, -0.2756645083427429, 0.20447446405887604, 0.3157520592212677, 0.6708195209503174, -2.2115025520324707, -1.488308310508728, -1.404342770576477, 1.3783624172210693, -0.9470731019973755, 0.21750932931900024, -1.9501261711120605, 0.3656863570213318, -1.2511177062988281, 1.7846029996871948, 1.1227483749389648, -2.2056291103363037, -0.48707541823387146, -0.3150698244571686, -0.24102924764156342, -1.04261314868927, -0.6218979358673096, 0.8011103272438049, -0.5538397431373596, 0.7923490405082703, -0.5893660187721252, -0.21489709615707397, 0.746598482131958, -0.9037303924560547, -0.21147015690803528, 1.020825982093811, 0.8121325969696045, -1.5697354078292847, -0.21385936439037323, -1.3557589054107666, -0.32976993918418884, 0.47002550959587097, -0.6979178190231323, 0.40564072132110596, 1.1465259790420532, 0.9117615818977356, 0.1444956213235855, -0.6204324960708618, 2.6450860500335693, 0.4949730634689331, -0.2642308175563812, -0.7282016277313232, 0.37745627760887146, 0.3317471742630005, -0.1455206423997879, 0.8363842368125916, -0.06452322751283646, 0.7590558528900146, -0.4817417562007904, 0.8758049011230469, 0.7648690938949585, -1.0729668140411377, -0.7775963544845581, 0.4318181276321411, 0.24506492912769318, 0.2810235321521759, -0.015417800284922123, -0.5132706165313721, 0.5606220364570618, 0.4019564092159271, -1.353158950805664, 1.0197103023529053], [-0.30766695737838745, -0.5730839371681213, 0.6949666142463684, 0.5011774897575378, -1.0055524110794067, -0.11654257774353027, -0.4664148688316345, 0.6056992411613464, 0.6343197226524353, -0.37459227442741394, 0.7714011073112488, 0.05087682977318764, -0.09784951061010361, -1.1004329919815063, -1.0752159357070923, 2.122262477874756, 1.6015723943710327, -0.1605835258960724, -1.117024540901184, 2.2763710021972656, -0.15787972509860992, 0.5667867064476013, 1.8294169902801514, -0.18947328627109528, -1.0381759405136108, -0.03484829142689705, -0.3767430782318115, 0.09156741946935654, 0.8265634179115295, 0.93764328956604, -0.3361246883869171, -0.9201922416687012, -0.541633665561676, -1.4701683521270752, -1.6160385608673096, -0.21092721819877625, 0.63847416639328, 0.2121637612581253, -1.102635383605957, 1.5696035623550415, -0.18597647547721863, -1.6059383153915405, 0.9877372980117798, 0.5113069415092468, 0.1582699865102768, -0.3454461395740509, -1.0602946281433105, 1.2582160234451294, -0.7348848581314087, -0.929850697517395, -0.5132951736450195, -0.0149611197412014, -0.29120656847953796, 0.20711646974086761, 0.1450323760509491, -0.33900952339172363, 0.9806947112083435, 2.001399040222168, -1.0735224485397339, -0.9868996143341064, 0.043917905539274216, -0.38412734866142273, 0.8139581680297852, 0.673551082611084, -2.028942584991455, 0.12843367457389832, -0.12375066429376602, 0.12052707374095917, 1.313072681427002, 1.6880173683166504, -0.586905300617218, 0.04417373239994049, -0.7807914614677429, 1.3485225439071655, 1.9461320638656616, -0.7338578104972839, 0.7427827715873718, -1.5868794918060303, -0.3853379786014557, 0.7937823534011841, 0.8693625330924988, 0.08693189173936844, -0.8608099222183228, 0.9511933326721191, 0.9902405738830566, 0.019115805625915527, 0.8985913991928101, 2.6650404930114746, 0.48943790793418884, -2.34065842628479, 0.12039628624916077, -0.8185892105102539, 0.9541524052619934, -0.9946386218070984, -0.6127339601516724, -0.8837277889251709], [0.1515234410762787, -1.9834918975830078, 0.0268104188144207, -0.7922961711883545, 1.248477816581726, 1.2985080480575562, -1.3346272706985474, 0.7187955975532532, -1.824532389640808, -0.9306024312973022, -0.8213937878608704, 0.310750812292099, 0.23049868643283844, -1.9437601566314697, -0.3197402358055115, -0.3996427655220032, -0.36012837290763855, 0.4731622338294983, 1.0130261182785034, -0.3634704649448395, 1.2104730606079102, 0.31658831238746643, 0.39481988549232483, 1.8404654264450073, 0.3126690983772278, -0.24001175165176392, 0.759906530380249, -0.2754581868648529, 1.20854914188385, 0.008882617577910423, 0.6515363454818726, -1.0539186000823975, 1.6391711235046387, 1.121234655380249, 0.36778584122657776, 0.801287055015564, 0.9693621397018433, -0.5422893762588501, -0.5826667547225952, 1.1541485786437988, -1.5523887872695923, 0.7629779577255249, -1.2595049142837524, 0.3809066116809845, 0.2596333622932434, 0.838901162147522, -1.0354377031326294, 0.7092112302780151, 0.8586148619651794, -0.5178106427192688, -0.6289967894554138, -1.8615161180496216, -1.6200315952301025, 1.4343931674957275, 0.4550178647041321, 0.15468759834766388, 0.42391714453697205, -1.0769565105438232, -0.038692403584718704, 0.688303530216217, -1.1109404563903809, -1.6965324878692627, 0.7049961686134338, -3.1161928176879883, 0.5079644918441772, -0.4733069837093353, 0.46004411578178406, 0.5729214549064636, 1.1286755800247192, -0.1368618607521057, 0.7720344662666321, -0.9428108930587769, 0.5763412714004517, 0.7488738894462585, -0.7863014340400696, -0.2865928113460541, 1.397135853767395, 1.3137720823287964, -0.051769550889730453, -0.15436606109142303, 0.327549546957016, -0.5850092768669128, -0.15784229338169098, -2.561306953430176, 0.13345186412334442, -0.5234253406524658, -1.0169845819473267, 0.6937723159790039, 0.5312039256095886, -0.5057564973831177, 0.4728216528892517, -2.1174139976501465, -1.2481358051300049, -0.21084623038768768, 0.769340991973877, -0.505155622959137], [0.30840644240379333, -0.48947596549987793, -0.36231571435928345, -1.3099031448364258, -0.13548484444618225, 0.3315483033657074, -1.4011578559875488, -0.8435072898864746, 1.5993691682815552, 1.482879877090454, 0.4977439343929291, 0.4098590016365051, 1.6896998882293701, -0.5397481322288513, -0.27388960123062134, 1.2720801830291748, -1.110023856163025, -0.46726155281066895, -0.6757269501686096, 0.8781399726867676, -1.2642709016799927, -2.5145909786224365, -0.1132517158985138, -0.30457961559295654, 1.7840938568115234, -1.7733385562896729, 0.24553564190864563, -0.36519837379455566, -0.21452994644641876, -0.07787469029426575, -2.21213960647583, 0.924462080001831, 0.14207732677459717, 0.3284724950790405, 1.954168438911438, 0.25960999727249146, 0.14587707817554474, -0.283292680978775, -0.49047616124153137, -0.10110928118228912, 2.3922784328460693, 1.029727578163147, -0.34307968616485596, 0.05409625545144081, 0.6972975730895996, -0.8802570104598999, 0.7099044322967529, 0.833352267742157, 0.48305124044418335, 0.42640984058380127, -0.31512969732284546, -0.5539513826370239, -0.14334149658679962, 0.1059965193271637, -1.0170592069625854, 0.3040933310985565, -1.8610221147537231, -1.392080307006836, 0.5355748534202576, -0.15625321865081787, -0.2190999835729599, -1.6349380016326904, 0.4667615592479706, 0.7183573246002197, 0.07822181284427643, 0.550480842590332, 1.447922706604004, -0.36359110474586487, 1.422532081604004, 0.5959833860397339, 0.02355744130909443, -0.6786921620368958, 1.3282312154769897, 1.4242733716964722, -0.007295573595911264, -1.6280125379562378, -1.9841605424880981, 0.5340377688407898, -0.21851903200149536, 0.5058881640434265, 1.4873930215835571, 1.3809785842895508, -0.21894796192646027, 0.4837521016597748, 1.5354512929916382, -1.1590726375579834, -0.3317851424217224, 0.1109270453453064, 0.9369214773178101, 0.31199911236763, -0.2183956503868103, -0.45688486099243164, 0.4212951064109802, -2.064242124557495, 0.7205185294151306, -0.0032423939555883408], [1.6268253326416016, -0.8527037501335144, 0.3376457989215851, 0.03839258849620819, 0.03779969736933708, -1.425466537475586, 1.3062591552734375, 1.7169396877288818, 0.4521375298500061, -0.08103445172309875, -0.26324743032455444, -0.34539467096328735, 0.8289289474487305, -0.8132039904594421, 1.5330572128295898, -1.4144924879074097, 0.525644063949585, -0.6708831191062927, 0.7298203110694885, -0.9761868715286255, 1.364479422569275, 0.18009983003139496, 0.011152820661664009, -1.3999351263046265, 0.3584797978401184, -0.9885466694831848, 1.8307627439498901, 0.42732563614845276, -0.3642592132091522, -0.16557073593139648, -0.7642084956169128, 1.5454314947128296, -0.5833730101585388, -0.8333228826522827, 0.6480708718299866, 0.10363319516181946, 0.14743246138095856, -0.3321082890033722, -0.6214237213134766, -0.23632977902889252, 0.8339160680770874, 0.9763370752334595, -1.231473684310913, 0.34469908475875854, 1.234682321548462, -1.990761160850525, 1.2036163806915283, -1.414948582649231, -0.5920241475105286, 0.9700114130973816, -0.492807537317276, -1.1340056657791138, -1.8766491413116455, -1.0410748720169067, -0.41107985377311707, 0.9333302974700928, -1.694331407546997, 0.30445948243141174, -1.574007511138916, -0.9911476373672485, 0.4535093307495117, 2.597167491912842, 0.7807289958000183, 0.42437320947647095, 0.2927616238594055, 0.7780507206916809, -0.029350636526942253, 1.658751130104065, 0.22553935647010803, 0.6150373816490173, 0.15255409479141235, 2.2816317081451416, -0.08687835931777954, 1.381474494934082, -0.8292962908744812, 0.08069644123315811, -0.9117377400398254, -0.22986845672130585, -1.1673957109451294, 0.2801378071308136, -0.29322555661201477, -1.3355053663253784, -0.6330151557922363, -0.4102117121219635, 0.5997422337532043, -1.7284908294677734, 0.34267154335975647, 0.18462663888931274, -1.6203043460845947, 0.027456866577267647, 0.8725202083587646, -0.9552473425865173, -1.0569231510162354, 0.2355281412601471, 1.7971789836883545, 0.6298382878303528], [-0.16445162892341614, 1.0632051229476929, -0.1986686736345291, 0.2552734315395355, -0.773216724395752, -0.1513361930847168, 1.7165688276290894, 0.7051886320114136, 0.8909255266189575, -0.5828596353530884, 0.06910236179828644, -0.31543540954589844, 0.4195699095726013, -0.22779394686222076, 0.2310103178024292, 1.505611538887024, 0.9896470308303833, 0.9400491714477539, 0.7019729614257812, -0.28158965706825256, 0.8589743971824646, -1.3226935863494873, 1.2962838411331177, -1.9569145441055298, -1.4295364618301392, 0.48061737418174744, -2.1926729679107666, 0.7682467103004456, -0.482203871011734, -1.6992071866989136, 0.009988270699977875, -0.45775070786476135, 1.880775809288025, 2.068584680557251, 0.9205150008201599, -0.2851722538471222, -0.7351759076118469, 0.9451128840446472, -2.2044200897216797, 0.3701053559780121, -0.5804276466369629, 0.678666353225708, 0.29535743594169617, 0.49629440903663635, 0.15123462677001953, 1.6475002765655518, -0.03208357095718384, -0.5337479710578918, -0.4334927201271057, 0.6189074516296387, 0.20642070472240448, -0.8965212106704712, 1.115809679031372, 0.690220296382904, 0.311379075050354, 0.8722724318504333, -1.2119057178497314, 1.3068044185638428, -0.43744248151779175, -0.9993689656257629, -0.1549765020608902, -0.03536311909556389, 0.9946379065513611, 0.5221927165985107, -1.0956496000289917, 1.655408501625061, 0.5273615717887878, 0.43184810876846313, 2.4374866485595703, -1.4574040174484253, 1.2808208465576172, -1.0583754777908325, 0.3292851150035858, 1.8304154872894287, 0.5461680889129639, 1.0415247678756714, -1.574798345565796, -0.6297318935394287, 0.35055649280548096, -0.36720848083496094, -1.2271063327789307, -1.629199743270874, 0.01271548680961132, -0.36449986696243286, 1.1222058534622192, -0.31529873609542847, -0.41396617889404297, -0.25602835416793823, -0.5174579620361328, -0.023762820288538933, -1.5083038806915283, 1.2416359186172485, 2.0298471450805664, 0.6511242985725403, -1.1620557308197021, -0.40087446570396423], [2.0110509395599365, 1.4201388359069824, -1.6197713613510132, -0.9687070250511169, -1.0337351560592651, -0.6209895610809326, -2.379234790802002, -0.694966733455658, -0.35868602991104126, -0.18780387938022614, 1.0804489850997925, -0.4806232750415802, 0.4845546782016754, -0.22780458629131317, 0.15793070197105408, 1.4195637702941895, 0.531349778175354, 1.4298031330108643, 1.4976502656936646, -1.5715620517730713, 0.6075117588043213, -0.6695396900177002, -0.39189809560775757, -0.07937853038311005, -0.7143595814704895, -0.32050690054893494, 0.6052637100219727, -0.6713210344314575, -0.9384207129478455, 0.05764508247375488, 0.02611844427883625, 0.287249892950058, -1.7418936491012573, -1.4047136306762695, 0.1289007067680359, -0.5421108603477478, 1.2335234880447388, 0.7710421681404114, 0.705253005027771, -0.12214644253253937, 0.2980203330516815, -1.143763542175293, 2.0653514862060547, -0.6171481609344482, -0.7374371886253357, -0.8631919026374817, 0.6798743605613708, 0.22642165422439575, -0.6335486769676208, 0.7788912653923035, 0.8689845204353333, -0.2901484966278076, -1.2432823181152344, 1.707105278968811, -0.11443651467561722, 0.6031686663627625, -0.754473090171814, 0.37286606431007385, 0.4518313705921173, 0.41572752594947815, -0.4546394646167755, 0.6786322593688965, 0.8041600584983826, -1.8221129179000854, -0.1527942270040512, 0.7933712601661682, -0.10925988852977753, 0.10120020061731339, 0.4114053249359131, 1.4447133541107178, 0.6304750442504883, 1.4881101846694946, -1.144303798675537, -0.4617573022842407, 0.3796917498111725, -0.04738864302635193, -2.249605417251587, 0.679425060749054, -0.011510813608765602, -0.6301316022872925, -0.31869035959243774, 1.922715663909912, 0.2506696879863739, 0.8124352693557739, -0.8188692927360535, -0.9210197329521179, 0.6402769088745117, -0.36029762029647827, 1.531695008277893, 0.9830757975578308, -0.4783700406551361, -2.0652477741241455, 0.7986588478088379, 0.34568849205970764, 0.24945461750030518, -0.8940386772155762], [1.1976176500320435, -1.3064181804656982, 0.6889997124671936, -1.6152396202087402, -1.1260019540786743, -1.0142018795013428, -0.05438602715730667, 0.46831172704696655, -1.8541353940963745, -1.1490991115570068, 0.7065098881721497, 0.23697951436042786, -1.1111294031143188, -0.1705024540424347, 0.52901691198349, -0.9697592258453369, -0.215650737285614, -0.5797534584999084, -1.907335638999939, -0.7129312753677368, -0.95298171043396, -1.7117918729782104, -0.5781236886978149, -0.2642848491668701, 0.5692077875137329, 0.6948862075805664, 0.21467003226280212, -0.8947482109069824, -1.724065899848938, -0.4645750820636749, 0.05926542356610298, 1.2542498111724854, 1.268803596496582, 0.32174769043922424, -0.310920387506485, -2.072632312774658, -0.1424446403980255, 1.5108563899993896, -0.6250771284103394, -0.777316153049469, 1.7709835767745972, 0.5258530378341675, 0.3754217326641083, -1.5798029899597168, -0.5910061001777649, -1.4431936740875244, 0.016816454008221626, -0.2300412356853485, -1.4378799200057983, -0.41893935203552246, 2.096297025680542, -1.3248672485351562, 0.11468395590782166, 1.0524696111679077, 0.15723532438278198, 1.4158645868301392, -0.41715988516807556, -0.7161832451820374, -0.21804320812225342, -1.0975691080093384, 0.07009603828191757, -0.45060935616493225, -0.866158127784729, -0.24354620277881622, 0.48065370321273804, -0.12422815710306168, 0.15556706488132477, 2.0202009677886963, -0.7153375148773193, 0.9076148867607117, -0.05742771923542023, 1.7830226421356201, 0.3956379294395447, -0.12283055484294891, 0.358101487159729, 1.2866617441177368, 0.7951298952102661, 0.8057108521461487, -0.1824037730693817, 0.054167490452528, 0.4347740709781647, 0.3403259515762329, 1.8675137758255005, 0.590404748916626, -0.36281275749206543, -0.6501592397689819, 0.3790282607078552, 0.49088749289512634, 0.7736636400222778, 1.2447268962860107, -0.09720047563314438, -0.23050114512443542, -1.304567813873291, 1.236885666847229, 0.8774576187133789, 0.31053686141967773], [-0.1183004081249237, -0.7172607183456421, 1.0247786045074463, -0.42537176609039307, -0.5721125602722168, 1.338106393814087, 1.0320425033569336, 0.38992422819137573, -0.6266863942146301, -0.9515830278396606, -0.3453102707862854, 0.2711073160171509, -1.0585308074951172, -0.3280301094055176, 1.1069551706314087, -0.5977935791015625, -1.8838671445846558, 0.8293490409851074, -1.3063480854034424, 1.538260817527771, -0.6084195375442505, 0.22776883840560913, -0.07315269857645035, 1.909087896347046, 0.35955631732940674, 2.143648862838745, 0.7835718393325806, 0.6705076098442078, 1.2991399765014648, 1.4272490739822388, 0.39280933141708374, -1.3790401220321655, 0.4026428461074829, 0.2800097167491913, -0.4548868238925934, 0.6807738542556763, -1.1165133714675903, -1.5191646814346313, -0.3385871350765228, 0.17332619428634644, -1.267855167388916, 0.25990527868270874, -0.5434861183166504, 1.9502711296081543, 0.20412828028202057, -0.7131545543670654, -1.8531091213226318, 1.741900086402893, -0.4551107883453369, -2.3501083850860596, 0.05307527258992195, -0.9589919447898865, 1.599860429763794, -1.4761459827423096, 0.4334554970264435, 0.5716349482536316, 0.274933397769928, -0.15499038994312286, 2.355020523071289, -0.0044685848988592625, -0.032005827873945236, -1.7069000005722046, 0.57709801197052, 0.05188751220703125, -1.8958970308303833, 0.3225240707397461, 1.1783125400543213, 1.9095468521118164, -0.6226494312286377, 0.15547679364681244, -1.5920525789260864, -0.24445313215255737, 0.24412238597869873, 0.8090324401855469, -0.40728142857551575, 0.6383886337280273, -0.33830544352531433, -0.3370669484138489, -0.10167964547872543, 0.1525530070066452, -0.6644095182418823, -0.6665639281272888, -1.1089353561401367, 1.67669677734375, 0.13551197946071625, 0.7509885430335999, 1.7567132711410522, -0.608174204826355, 1.0011060237884521, -0.29542410373687744, 1.2422212362289429, -2.7371599674224854, 3.6574244499206543, 0.6483721137046814, -0.15119560062885284, -0.31453946232795715], [1.363362193107605, -0.6439046263694763, 0.33536094427108765, 0.48357200622558594, -1.6333192586898804, -1.6453783512115479, -0.01248274277895689, 0.2888539433479309, -0.4491156339645386, 0.5849986672401428, -0.6794925928115845, 1.014174461364746, -0.33271950483322144, -0.20672118663787842, 0.05241881683468819, 1.6067748069763184, -0.9550799131393433, -0.9390065670013428, -0.6525055766105652, -0.11166931688785553, -1.443583369255066, 0.4289364814758301, -1.070929765701294, 1.210142731666565, 0.6937893033027649, 0.027663836255669594, -0.9366579651832581, -0.07654564827680588, -0.7865653038024902, -1.534604549407959, -1.4672880172729492, 0.12248099595308304, -0.9817853569984436, 0.11732408404350281, 1.7406331300735474, 0.5931602716445923, 0.5142067670822144, -1.0395946502685547, -0.4539594352245331, 0.8876620531082153, 0.10942599922418594, 1.5399430990219116, 0.824499785900116, -0.786247730255127, 0.7534468770027161, -0.734893262386322, -1.9573272466659546, -0.49554145336151123, 0.6807666420936584, -2.353259801864624, 1.2246265411376953, 0.6668790578842163, 1.3726942539215088, -0.7616097331047058, 1.0291987657546997, 1.4658703804016113, 0.7635471224784851, 0.6641731858253479, 1.0936356782913208, 0.24192892014980316, -1.122984528541565, 1.826715350151062, -1.1872029304504395, -0.7222378849983215, -0.18675071001052856, -1.8218610286712646, 0.26956579089164734, -0.06852822750806808, 0.012641729786992073, 0.7989009618759155, -0.9276425242424011, 0.4460887014865875, 0.4352628290653229, 0.049248989671468735, 0.04405504837632179, 2.589028835296631, 0.5853076577186584, -0.8749461770057678, 0.66807621717453, 0.18537190556526184, -0.6483697295188904, 2.1688873767852783, 0.5359789729118347, 0.36830395460128784, 0.10944041609764099, -1.4722412824630737, -0.10513866692781448, 1.9906182289123535, -1.4220813512802124, -1.0225439071655273, -0.07350976020097733, 0.17597614228725433, -0.03880874440073967, -0.8991009593009949, -0.5684354901313782, 0.5786511301994324], [0.027401257306337357, 1.8418749570846558, 0.6258484721183777, 0.6158586144447327, 2.687063217163086, -0.9516943693161011, 0.10523886978626251, 0.5479486584663391, -1.0974419116973877, 1.2690577507019043, -0.7887439131736755, -0.10962025821208954, 0.14934135973453522, 0.4749271273612976, -0.21331529319286346, 0.7421759366989136, 0.9371967315673828, 0.9780390858650208, -0.593484103679657, -0.9703197479248047, -0.3856794238090515, 0.31266993284225464, 1.6150248050689697, 0.10938306897878647, -0.30058610439300537, -0.8466513156890869, -1.1007682085037231, -1.4615446329116821, 1.196771502494812, -0.4289844334125519, -0.5732369422912598, 0.10631343722343445, -0.9977779984474182, -1.3419910669326782, 1.0809743404388428, -0.697238564491272, -0.7997045516967773, 0.659474790096283, -0.1710796356201172, 0.08502788096666336, 0.2388746738433838, -1.0640314817428589, -0.2920129597187042, 0.5122357606887817, 0.6051570773124695, -1.3459746837615967, 0.0914505198597908, 0.3597513735294342, 1.055066704750061, 0.17519812285900116, -0.015319766476750374, 1.308260440826416, 1.449210524559021, 1.798624038696289, 0.22133634984493256, 2.1390202045440674, -1.0921467542648315, -2.1021714210510254, 1.4047462940216064, -0.17282643914222717, -0.3093029856681824, -1.6193010807037354, 0.13308002054691315, -0.009159340523183346, -1.5055105686187744, -1.7885414361953735, 1.2105751037597656, -1.5625832080841064, -1.202913522720337, -0.8137950897216797, -0.3261741101741791, 0.43353086709976196, -0.2895025312900543, 0.5453012585639954, -0.05434449762105942, 0.5212944149971008, -0.09512107819318771, -0.48500046133995056, 1.5016701221466064, -0.5587921142578125, 0.3908122479915619, -0.7771214246749878, 0.1338299959897995, 0.5101791620254517, 0.6541996002197266, -0.7173023223876953, -0.004735871683806181, -0.06585489958524704, -0.7936468124389648, -0.43763384222984314, -0.8255117535591125, 0.8851243853569031, 1.385108232498169, 0.9258548617362976, -0.6107727885246277, -1.511248230934143], [1.0427892208099365, -0.40952450037002563, 0.327635794878006, 0.7749161720275879, 0.682357907295227, -0.372107595205307, 0.31132423877716064, -0.46183493733406067, -1.1897412538528442, 0.1788203865289688, -1.802346110343933, -1.0213061571121216, 0.7850344181060791, -0.17484869062900543, 1.82068932056427, 0.37126854062080383, 0.3118562400341034, -0.12866699695587158, -0.6083112359046936, -0.9598595499992371, 0.9488660097122192, -0.47449856996536255, 0.9051176905632019, -1.0814602375030518, 1.3877090215682983, -0.2912808954715729, -0.5029577612876892, -0.0687846690416336, 0.0037260879762470722, 0.47607100009918213, 0.07172440737485886, 1.374330997467041, -0.16064289212226868, 0.429577112197876, -0.9609547853469849, 0.6558890342712402, -1.442653775215149, 0.17848846316337585, 2.0142805576324463, 0.2747608423233032, 1.6512227058410645, 0.013080479577183723, 0.6795979738235474, -0.014844721183180809, 1.5294299125671387, 0.42299923300743103, 0.15998852252960205, -0.5755173563957214, -0.8456655144691467, 1.9227237701416016, 1.8303028345108032, 0.26140764355659485, -0.2381904423236847, 0.09977816045284271, 0.6357635855674744, -0.13937126100063324, -0.33510035276412964, 1.688549518585205, 0.3079420328140259, 0.8266569972038269, -0.7405940890312195, -0.12767402827739716, -0.3815615773200989, 1.0697258710861206, 1.0217368602752686, -1.379204511642456, -1.0549360513687134, 0.5181499123573303, 1.0473395586013794, 1.0336970090866089, 0.7400256991386414, 1.4080111980438232, 0.10284098237752914, -0.27406907081604004, 0.22952179610729218, 1.489555835723877, -0.8802909851074219, -1.535321831703186, -0.21968014538288116, -0.6991849541664124, -0.2678537666797638, -1.2950135469436646, -0.43126291036605835, 0.6054632663726807, 0.7903560996055603, -0.01137707568705082, 0.974288284778595, -1.056486964225769, 0.5248193740844727, -1.1391547918319702, 0.9444839358329773, -1.0326082706451416, -0.25900572538375854, 2.3241779804229736, -0.8657986521720886, 0.7894679307937622], [1.822503685951233, 0.09847042709589005, -1.040015697479248, -1.088496208190918, -0.6591488122940063, 0.8120267391204834, -0.27076297998428345, 0.6698032021522522, 0.8576703667640686, -0.6995436549186707, 0.7213115096092224, -1.387874960899353, -0.12167306244373322, 1.854705572128296, -0.9741660356521606, 1.1965283155441284, 0.06937603652477264, 0.4799143970012665, -0.1467732936143875, 0.41994941234588623, 0.8860182762145996, 1.135056495666504, -0.40182673931121826, 1.2752225399017334, -2.6802515983581543, 0.24729523062705994, 0.8572676181793213, 0.18012486398220062, 0.22865012288093567, 0.660849392414093, -0.3216763436794281, 0.16543282568454742, 0.08706425130367279, 1.7144752740859985, 0.6615357995033264, 1.3718516826629639, 1.3142143487930298, -0.047157157212495804, -0.5887793898582458, -2.298208713531494, 0.4093249440193176, 0.6918691992759705, 0.9848615527153015, -0.666674792766571, -0.8498379588127136, -0.8710649609565735, 0.3919753134250641, 0.5011212825775146, -0.25134268403053284, 0.1713913381099701, -1.6335560083389282, -0.31863299012184143, -0.37605375051498413, 1.9933521747589111, 0.21854424476623535, -1.1066370010375977, 0.6162521839141846, 0.34389954805374146, 0.8244696259498596, 1.4601075649261475, 1.0811034440994263, -1.0543513298034668, 0.8844534754753113, -1.2387231588363647, 0.356210321187973, 0.5188164114952087, -0.1594744622707367, -1.328757882118225, -0.12977318465709686, 0.5258023142814636, 0.6540403962135315, 0.6750532984733582, 0.24921293556690216, 1.3815003633499146, -1.8792481422424316, 0.09378337860107422, -0.10189518332481384, 0.17578648030757904, -0.9138262271881104, -0.9699347615242004, 2.4778318405151367, -0.716010332107544, -0.4022330939769745, 0.861022412776947, 1.7983300685882568, -1.6705291271209717, -1.1814839839935303, 1.8784641027450562, -0.7412958741188049, -0.9236493706703186, 0.08365477621555328, -0.35484811663627625, 0.4564710557460785, -1.6706316471099854, -1.1311650276184082, 0.4385518431663513], [-1.183242917060852, 0.5126332640647888, -0.024630453437566757, -1.253042221069336, -1.7864255905151367, -0.059955839067697525, -0.6601395010948181, -0.20029836893081665, 0.790339469909668, 0.9215829968452454, 0.4453074634075165, 1.6613355875015259, -0.09643246978521347, 0.14849258959293365, 0.39948365092277527, -0.2016529142856598, -0.1709204614162445, -0.32561013102531433, -0.5466055274009705, -0.23029881715774536, 1.2839429378509521, -0.029540643095970154, 0.609072744846344, 0.8832514882087708, -0.3579384386539459, -0.5529582500457764, -1.0478945970535278, -0.16504305601119995, 1.365554928779602, 0.818784773349762, -1.371913194656372, -0.740095317363739, 0.3099650740623474, -0.243514746427536, -1.1292561292648315, 1.1402134895324707, -0.00830436497926712, -0.8470613360404968, 0.9337418675422668, -0.21159455180168152, 1.7178120613098145, -0.33506035804748535, -0.6710972189903259, 1.143343210220337, -0.7099607586860657, -0.4780098497867584, -0.00720512168481946, 0.0387817919254303, -1.0735468864440918, 0.6497076749801636, -0.8392930030822754, -1.1022659540176392, 0.05000026524066925, -1.2015312910079956, 0.20253612101078033, 0.009412289597094059, -0.9398729801177979, 0.12176890671253204, 0.3580700755119324, 0.3765079081058502, 0.3866453468799591, -0.3865818381309509, -1.568931221961975, -1.4616045951843262, -0.6624590754508972, -1.1654106378555298, -0.47328194975852966, 0.8793343305587769, -0.2868656516075134, -0.9884814023971558, 0.6076570749282837, 0.16473549604415894, -1.4123274087905884, 1.5227543115615845, -0.9621090888977051, 0.6047956347465515, -0.33505022525787354, 0.524092972278595, 1.771692156791687, 0.9314730167388916, 1.2619361877441406, 1.6746759414672852, 3.0738773345947266, -0.7535690665245056, 0.02800666354596615, 0.2072598785161972, 0.7683126926422119, 0.17598991096019745, 0.9508176445960999, 1.492290735244751, -1.2544105052947998, -0.5028657913208008, 0.05937584117054939, 0.5121344327926636, -0.9331271052360535, 1.1641243696212769], [0.7450013160705566, 0.20722205936908722, 1.5042821168899536, -0.9763807654380798, -0.1868949830532074, 0.46660566329956055, -1.2421131134033203, 1.5791423320770264, -0.13171391189098358, 2.7317230701446533, 0.005696030333638191, -0.3911929726600647, 0.5245782136917114, 0.641658365726471, -0.3950882852077484, 0.5944405198097229, -1.697632074356079, -0.01883811317384243, -1.0612685680389404, -0.0034007569774985313, 0.4738115668296814, -0.06313318014144897, 0.6524778604507446, 1.0089435577392578, -0.7887744903564453, -1.277687430381775, -0.9731656312942505, -0.21406877040863037, -0.7650689482688904, -0.584809422492981, -1.8515883684158325, 1.3741377592086792, -0.7477049231529236, 0.6269699931144714, -1.215653419494629, 0.2781292200088501, -0.2778030037879944, 0.7824799418449402, 0.14288072288036346, -2.0392038822174072, 1.0647633075714111, 0.6341555714607239, -1.5832186937332153, 0.5291005373001099, 0.8838075995445251, 0.21550211310386658, -0.6739152669906616, 0.1067853718996048, -1.1681054830551147, 1.330201268196106, 0.05011652037501335, -0.6711276769638062, -0.7529200911521912, 0.2824500799179077, 0.8102116584777832, 0.14907653629779816, -0.505909264087677, -1.0768622159957886, -0.1250026524066925, 2.43770170211792, -0.767932653427124, 0.5371733903884888, 0.33043980598449707, -1.5817110538482666, 1.063175916671753, -1.4283149242401123, 1.807637333869934, 2.460693120956421, -0.7007561922073364, -1.2453371286392212, 0.06521613150835037, 0.2938199043273926, 0.6373959183692932, -1.223839521408081, -1.5585278272628784, -0.6682947278022766, 1.4697649478912354, -1.6965891122817993, -0.47285398840904236, 0.8914652466773987, -0.2926500737667084, -1.3614078760147095, 1.7862176895141602, -1.069536566734314, -0.9834302663803101, 0.8661514520645142, -0.7480061054229736, -0.4621545076370239, -1.8454480171203613, 0.15085098147392273, -0.880993664264679, 0.03558063879609108, -0.8916346430778503, -0.2659118175506592, 0.5145889520645142, 2.893138885498047], [-0.8561116456985474, 1.16602623462677, 0.379313588142395, -0.7862539291381836, -0.575599193572998, -2.1598901748657227, -0.46001651883125305, -0.19338135421276093, -0.01618901640176773, -3.0949208736419678, 0.13955813646316528, 0.6600176692008972, -0.5008782148361206, -1.0779651403427124, -0.5272161364555359, -0.43966013193130493, -2.221050262451172, -0.8864442110061646, -0.06818519532680511, 1.211879849433899, 1.1283453702926636, -0.20426546037197113, -0.25991255044937134, -0.0634487047791481, -0.3659845292568207, -1.0229065418243408, -2.190840005874634, -0.9910244941711426, -1.8156613111495972, 0.48677995800971985, 1.4569692611694336, 0.08508632332086563, 0.6316469311714172, -0.7293657660484314, 1.15310537815094, 0.09485091269016266, 1.8398715257644653, 0.7333558201789856, -0.721123456954956, 0.9200454950332642, 0.6569443941116333, 0.3936628997325897, 1.4141552448272705, 1.0565025806427002, -0.7611480355262756, 2.4646759033203125, 1.024776577949524, -0.6357887387275696, -0.2082061618566513, 0.6755314469337463, 0.4606704115867615, -0.06902477890253067, -0.4690025746822357, 0.07243654876947403, -0.19716377556324005, -0.024328891187906265, 0.23511619865894318, -0.18957604467868805, 1.1221470832824707, 1.085694670677185, 0.4579283595085144, 0.9010053277015686, 0.46100515127182007, 0.7414994835853577, 0.6131788492202759, 0.33547618985176086, -1.7322314977645874, -1.4953986406326294, 0.016460498794913292, -1.092584490776062, -0.8169024586677551, -0.6689684987068176, -1.005570411682129, 1.3745850324630737, -0.07494068890810013, 0.8515835404396057, 2.0474483966827393, 1.4321094751358032, 0.013702156022191048, -0.044411152601242065, 0.671452522277832, -0.5561742782592773, 0.31958428025245667, -0.5750333070755005, -0.5870594382286072, 1.5451985597610474, -1.7834659814834595, -1.5362855195999146, -1.6329896450042725, 0.4050748348236084, -1.4147707223892212, -0.2252832055091858, 1.3303133249282837, 0.6974695324897766, -0.015414178371429443, 1.2882448434829712], [0.25262972712516785, 0.9022700786590576, 0.6493802070617676, 1.5245238542556763, 0.028394728899002075, 0.5345209240913391, 1.5261439085006714, -1.0272457599639893, -0.7646031975746155, 2.4120147228240967, 2.156670093536377, -0.6482051610946655, 0.21276219189167023, 1.9290893077850342, 1.837975025177002, -0.19008196890354156, 1.1425365209579468, -0.1635420322418213, 0.7015436887741089, 0.21441590785980225, 0.8272351622581482, 1.1736005544662476, -0.2809373736381531, -0.3677052855491638, -0.40126845240592957, 1.0007834434509277, -0.46222686767578125, 2.0212342739105225, -0.7315669655799866, 0.3588956892490387, 1.4926059246063232, -0.4347243309020996, -0.9351904988288879, 1.2253985404968262, -0.7913548946380615, -0.5598329305648804, 0.747654914855957, 1.9575555324554443, 1.6843432188034058, 0.47486618161201477, -1.036218523979187, -1.0410031080245972, -0.06763054430484772, -0.6263490915298462, -0.07960423082113266, 0.13327939808368683, 0.22168895602226257, -1.0143120288848877, -0.6168900728225708, -0.3455268144607544, -0.7933650016784668, -0.28785377740859985, -1.1358226537704468, -1.205992341041565, -2.4565296173095703, 0.8696383833885193, 0.7724398970603943, 0.10879664868116379, 0.8658796548843384, -0.39624762535095215, -1.3187021017074585, -0.1433776617050171, -2.4229891300201416, 0.8260900378227234, -0.16194313764572144, 0.2756265103816986, 0.4602651596069336, 0.9660786986351013, 1.993961215019226, -0.6980885863304138, 0.18880201876163483, 0.8273594379425049, -0.3954707384109497, 0.8483313322067261, -0.1949797421693802, 0.902664303779602, 1.0757635831832886, -0.2196681946516037, 0.9080333709716797, -1.5897598266601562, -1.803162932395935, 0.4018923342227936, 1.1319127082824707, 0.09525194019079208, 0.32144463062286377, 1.1393132209777832, 0.959634006023407, -0.7884404063224792, -0.35032758116722107, 0.341032475233078, -0.031056512147188187, 0.4264836311340332, -0.5533684492111206, 1.7436484098434448, 0.0225363876670599, 1.2113014459609985], [0.15432797372341156, 0.6288573741912842, 1.1056894063949585, -0.7062654495239258, -0.9782150983810425, -1.4106793403625488, -1.0303674936294556, 0.24178743362426758, -0.2502506375312805, 0.9419523477554321, -0.9824033379554749, -1.8577073812484741, -0.3161049783229828, 2.1412975788116455, -1.2111917734146118, -0.26027143001556396, 1.562419056892395, 0.5973618030548096, -0.5773703455924988, -0.48784592747688293, 0.9214051961898804, -0.37636658549308777, -0.7973289489746094, 0.6639603972434998, 0.4841005802154541, -0.8320978283882141, -1.510093331336975, 0.9157442450523376, -0.18334823846817017, 0.7385509014129639, -0.2960165739059448, -1.3678547143936157, 1.4362947940826416, -0.3704853057861328, 0.07964413613080978, -1.2229098081588745, -0.7331817746162415, 0.13932517170906067, 0.26949653029441833, 0.42855027318000793, 0.570448637008667, -0.13758544623851776, 1.5799546241760254, -0.27700310945510864, 0.8445754647254944, 0.9561752080917358, 0.02862105704843998, 0.7632156610488892, 1.2909578084945679, -0.43084102869033813, -0.9516544342041016, -0.5443721413612366, 1.9290764331817627, 0.18831245601177216, -1.296948790550232, 1.1576236486434937, 0.27696657180786133, -1.6436009407043457, 0.9381912350654602, -0.49376416206359863, 1.2505301237106323, -1.7445075511932373, -0.3945911228656769, -0.6066080331802368, 1.0239249467849731, -0.19918310642242432, -0.9760648012161255, -0.0069454689510166645, 2.1146163940429688, -0.28052109479904175, -0.6635047793388367, -1.2942395210266113, -1.5011872053146362, 1.2340214252471924, 0.44814127683639526, 0.27049073576927185, 0.07724417001008987, -0.4905426800251007, 0.6952194571495056, -0.16752611100673676, 1.0530644655227661, 0.9097135663032532, 1.3702037334442139, 2.927424192428589, 0.047106336802244186, -0.9942152500152588, -0.8890672922134399, 0.8852133750915527, 0.13527698814868927, -0.7495540380477905, 0.5628268122673035, 1.6513646841049194, -0.48233798146247864, -0.5685124397277832, -1.109360694885254, -1.6128803491592407], [-0.2080903947353363, -0.8760420680046082, -0.21234455704689026, -0.14337900280952454, -0.5938985347747803, -0.6677464842796326, -1.261664867401123, 0.5816309452056885, -0.6039487719535828, -0.04236046224832535, 0.6303580403327942, 0.12554991245269775, 0.5305953025817871, -0.1845126748085022, 0.03156999498605728, 0.2775547206401825, 0.6527629494667053, 1.568601131439209, 0.897179126739502, 0.11449837684631348, -0.5226694941520691, -0.20547831058502197, -0.19094617664813995, 1.2661370038986206, -0.361271470785141, -0.7700503468513489, 1.0074063539505005, -0.11928858608007431, -1.1417707204818726, 1.52059805393219, -1.318002462387085, -0.5699360370635986, 0.8791555166244507, 0.1266539990901947, -0.41810280084609985, 0.03573547676205635, 1.1777384281158447, -1.5463602542877197, -0.4949052631855011, 0.48835358023643494, -1.5129425525665283, -0.24151989817619324, -0.20924484729766846, -2.5702879428863525, -0.44432273507118225, 1.232236385345459, -0.6299657821655273, 1.2288159132003784, 0.8099771738052368, 0.009265251457691193, -1.2315771579742432, 0.7202138900756836, 0.9972492456436157, -1.7308984994888306, -1.197352647781372, -1.8493757247924805, 0.021365126594901085, 0.35971447825431824, -0.5236366391181946, 0.1600007563829422, 0.002815835177898407, -0.0916290208697319, -0.6632997989654541, 0.5830433368682861, -1.9135761260986328, -0.005478087347000837, -2.129222869873047, -0.16338782012462616, 0.07311635464429855, -1.4920036792755127, 0.49134117364883423, -2.2431905269622803, -0.19064973294734955, 0.44837072491645813, -0.2414238154888153, 1.245637059211731, -1.920073390007019, -0.8425567150115967, 0.8585477471351624, -0.6478559970855713, 0.4895841181278229, -0.29709941148757935, -1.532804250717163, -0.6523788571357727, 0.4678506851196289, -0.4175451099872589, 0.266968697309494, -1.5356974601745605, -0.6046345233917236, 0.27748700976371765, 0.15637433528900146, 0.7740821838378906, 0.06658631563186646, 0.8251424431800842, 0.2223232537508011, 1.1579891443252563], [-0.5465373992919922, -1.1997038125991821, 0.8618616461753845, -1.7298787832260132, 0.6131479144096375, -0.4243265986442566, 0.7282776236534119, -0.4972103536128998, 0.5617464184761047, 0.6556074619293213, 0.5433747172355652, 0.6908998489379883, 0.6243394017219543, 0.05894226208329201, 0.18875202536582947, 1.6718111038208008, 0.6372691988945007, -0.01241292618215084, -1.3912668228149414, -0.49969053268432617, -0.615855872631073, 1.1728053092956543, 0.9638893604278564, -1.6424554586410522, -0.7165005207061768, 0.5536217093467712, -0.8698341846466064, -0.29974889755249023, -1.049996256828308, 1.0905400514602661, -0.34662726521492004, 0.7769109010696411, -0.7974370718002319, -0.9974796175956726, 0.26724281907081604, -1.5906846523284912, -1.0566197633743286, 0.7815611362457275, 0.7878323197364807, 1.265244483947754, -1.2425538301467896, 0.7804882526397705, -0.0923633873462677, -0.5184722542762756, 1.5737318992614746, 0.3494696021080017, -0.8790567517280579, -2.6911826133728027, -0.9550518989562988, -0.4966713786125183, 0.8085750341415405, 0.1761704534292221, 0.526176393032074, 0.49198928475379944, -0.42399609088897705, -0.3032277524471283, 0.2970329821109772, 0.5912445783615112, 1.2154899835586548, -0.3892767131328583, -0.9030241370201111, 0.48074871301651, 1.0060573816299438, 1.03061842918396, -1.524389386177063, 0.6410843133926392, 2.5789053440093994, 1.066707968711853, -0.594042181968689, 1.0941296815872192, -1.3501520156860352, 0.3821650445461273, 0.5170339941978455, 0.9038780927658081, -0.9828141927719116, 0.6282352209091187, 1.4159436225891113, -0.1077670082449913, 0.6336014270782471, -0.5434275269508362, -0.8708539605140686, 1.4955147504806519, -1.353588342666626, -1.7227013111114502, -0.4622267186641693, -0.005043641198426485, -0.9933047890663147, 1.130411148071289, 0.3361581563949585, -0.49320393800735474, 1.2275720834732056, -0.39237818121910095, 1.289028525352478, 0.13587349653244019, 1.516565203666687, -0.7043078541755676], [-1.6862233877182007, -0.40936627984046936, -1.0998175144195557, 1.5468961000442505, 1.291633129119873, 0.4331318438053131, 0.10963940620422363, -0.7835298180580139, -0.4777171015739441, 1.4941511154174805, 1.2716408967971802, 0.5292940735816956, 0.5556660890579224, -0.10036469995975494, -0.3254997730255127, 0.06150706112384796, 1.318422794342041, 0.08814190328121185, -1.5137923955917358, 0.8430293798446655, 0.12484428286552429, -0.6072061061859131, -1.8998006582260132, -0.5228363275527954, -0.25161996483802795, 0.011931388638913631, 0.2863873541355133, 0.6435623168945312, -0.25726979970932007, 1.4261903762817383, 0.6831636428833008, -1.7465927600860596, -0.1193387359380722, -1.4831078052520752, 0.9904510378837585, -0.9754953980445862, -0.27688106894493103, -0.1439875215291977, -0.7782521843910217, 1.5407040119171143, 0.9538930058479309, -1.814716100692749, 1.1501562595367432, -0.2686152160167694, -0.5373428463935852, 0.06684664636850357, -0.7428574562072754, -0.02400163933634758, -0.3364720642566681, -0.815862774848938, -0.6815658211708069, 0.030368678271770477, 0.04210128262639046, 2.170553684234619, 0.6585233807563782, 1.4288625717163086, -1.3442823886871338, -2.4727530479431152, -0.11882049590349197, 1.2542603015899658, -0.6073117852210999, -1.4554238319396973, -0.32604098320007324, -1.7271020412445068, 0.2757536470890045, 0.09714934229850769, -0.4229030907154083, 1.1093374490737915, -1.4634751081466675, -1.3127042055130005, -0.4441673159599304, 0.13810698688030243, 0.7023894786834717, -0.11635658890008926, 0.7514029145240784, 1.6909364461898804, 1.4403655529022217, -0.9797104001045227, -0.6434388756752014, -1.2006886005401611, -0.9204639792442322, -0.8199509382247925, 0.5337682366371155, 0.34627285599708557, 1.2535851001739502, -0.8034911751747131, 0.9498511552810669, -1.0771607160568237, -1.9328944683074951, 0.6163479089736938, 0.23179945349693298, -0.8248424530029297, -0.6455473303794861, 0.05228279531002045, 0.17682653665542603, 0.12035125494003296], [-1.9015482664108276, 0.7476085424423218, -1.8797082901000977, -0.5553980469703674, 0.6110749244689941, -0.20799103379249573, -1.4679794311523438, 1.105180025100708, -0.29163068532943726, 1.0520374774932861, -0.17579954862594604, 0.3401035964488983, 0.3575417995452881, 0.1337955743074417, 0.08396870642900467, -0.7034528851509094, 0.19626227021217346, -0.9998793601989746, -1.6631288528442383, -1.789724588394165, 0.4649684429168701, 0.6201236844062805, -0.2988496422767639, 0.1929176151752472, 1.1365960836410522, 0.39979615807533264, 0.9224334359169006, -1.1854020357131958, 1.2684556245803833, 0.16714216768741608, -0.11460544914007187, 0.7648698687553406, -0.5429694652557373, -0.36342957615852356, 1.2669419050216675, 0.10498417913913727, -1.5997631549835205, 1.1021158695220947, 0.6248224377632141, 0.044900938868522644, -0.006370824296027422, 0.3965848684310913, 0.15782374143600464, 1.504812240600586, -0.3145081102848053, 1.2919232845306396, -0.2916320264339447, 0.8542010188102722, 0.7915408611297607, -0.3169621527194977, -1.3385976552963257, 0.6836017966270447, 0.7227305173873901, -1.0677934885025024, -0.4276646375656128, -0.45234477519989014, -0.8890335559844971, -0.9283852577209473, 0.44175127148628235, -0.6164823770523071, 0.170111283659935, -1.2027312517166138, -0.25806212425231934, -1.4437557458877563, 0.002067569177597761, 0.12611514329910278, -0.9696038365364075, 0.5617602467536926, 1.1338329315185547, 0.24510642886161804, 0.5500304102897644, -0.07616544514894485, -0.7582924365997314, 0.5782760381698608, 1.6519639492034912, 1.1396899223327637, -2.5925920009613037, 0.07301869988441467, 0.6851287484169006, -0.372647225856781, 0.6641924381256104, -1.9634069204330444, -0.898385763168335, -1.322205901145935, 1.253754734992981, -0.41869083046913147, -0.5154820680618286, -1.7376481294631958, 1.0020254850387573, 0.2450438141822815, -1.183905005455017, -0.6413030624389648, 0.24156971275806427, 0.10226472467184067, -0.5147882103919983, 0.5265023112297058], [-0.5711967349052429, 0.6923551559448242, 0.19928477704524994, 1.5504213571548462, 0.5941425561904907, -1.5285931825637817, 1.3717480897903442, 0.5570328831672668, -1.453650951385498, 0.6832805275917053, 0.08871778845787048, 0.1656050980091095, 0.3753806948661804, -1.6172319650650024, -1.906883955001831, 1.1082345247268677, 0.5304774641990662, -0.9677384495735168, 0.00045628921361640096, -0.03109235316514969, 0.09277236461639404, 0.004818089306354523, -0.7621297240257263, -1.8591991662979126, 1.421599268913269, -0.13261988759040833, -0.058436449617147446, -1.8596744537353516, 0.927143931388855, 1.636650800704956, 0.07401274889707565, 0.09960329532623291, 0.6902318596839905, 1.2693856954574585, -0.463979572057724, -1.3389281034469604, -0.3050934970378876, -0.6499108672142029, 1.9872934818267822, 1.958385944366455, 1.103515625, 0.5324453711509705, -0.9927724599838257, -1.8298981189727783, 0.5610978603363037, -0.4361169636249542, -1.6652425527572632, 0.30877891182899475, 0.47127479314804077, 1.4281647205352783, 0.8139241337776184, 0.776843786239624, 0.46534132957458496, 1.3322700262069702, 0.18050384521484375, 0.5711528658866882, -0.5153987407684326, 1.9332174062728882, 2.2051150798797607, -1.2728371620178223, -1.3829227685928345, -0.18269075453281403, 0.46048858761787415, 0.08122117072343826, -0.7768843770027161, 0.48781782388687134, 0.30809879302978516, -0.29121971130371094, 0.14113552868366241, -1.0070605278015137, -1.8703101873397827, 0.12138993293046951, -0.04604120925068855, 2.7072744369506836, 1.635679006576538, 1.3990554809570312, 1.2691622972488403, -0.5250532627105713, 0.28712567687034607, -0.8850846886634827, -0.22876586019992828, 0.39547017216682434, -0.02125720866024494, -0.32284125685691833, 0.9302071928977966, -0.23699851334095, -1.0556236505508423, 0.8487009406089783, -1.036655068397522, -0.8542546629905701, -0.16275298595428467, -3.035057544708252, 0.1732269674539566, -0.43214017152786255, 1.6819442510604858, -0.6010223031044006], [-0.15619587898254395, 0.4641769528388977, -0.2359800636768341, -0.8708043098449707, -1.1843030452728271, 0.6548794507980347, -1.364518642425537, 0.030443591997027397, 2.502622604370117, -0.9009549021720886, 0.12428754568099976, -0.28033146262168884, 0.15213488042354584, -1.161933422088623, -0.49142828583717346, -0.04744469374418259, -1.9908103942871094, -1.1585677862167358, -0.35455846786499023, 1.0198930501937866, -1.7415599822998047, 0.2074321061372757, 0.03943505883216858, 1.1492187976837158, 0.8141114115715027, -0.7313324213027954, 0.49064213037490845, -0.3084922730922699, 0.8700127601623535, -1.4067356586456299, -0.4244282841682434, -1.625293493270874, 2.1792666912078857, -0.7211312055587769, 0.02233009599149227, 0.9867944121360779, -0.8495775461196899, -0.2771163880825043, -1.0393284559249878, 0.058961693197488785, -1.1418997049331665, -1.0133603811264038, 0.16763614118099213, 1.4877249002456665, -0.5342152714729309, -0.4184635579586029, -0.6761996746063232, -1.2688082456588745, 0.38417813181877136, 1.7898892164230347, 0.8126135468482971, -0.4340764284133911, 1.8944648504257202, 0.34633928537368774, 0.2497292459011078, -1.6699105501174927, 0.008956455625593662, -0.923117458820343, 0.5362956523895264, -0.09319406747817993, 0.15766283869743347, -0.1485683172941208, 0.45424845814704895, 0.6559627056121826, 0.6216548085212708, 0.32081541419029236, 1.0705934762954712, 0.534254252910614, -0.540748119354248, 0.09083667397499084, 0.9391335248947144, -0.27220237255096436, 1.3696246147155762, 0.5082218050956726, -1.8485132455825806, 0.32606935501098633, -0.23526300489902496, 1.467274785041809, 0.10363222658634186, 0.1915150135755539, 1.2611844539642334, -0.31293174624443054, -0.6282135844230652, 0.4242587387561798, -0.8086279630661011, 0.1418192833662033, -0.3391610383987427, 0.7796165347099304, 1.0960218906402588, -2.1510508060455322, 0.7620535492897034, 0.31371521949768066, 0.07706917077302933, 1.221700668334961, -1.7124567031860352, 0.30300360918045044], [0.5051832795143127, 2.6868557929992676, 0.3031651973724365, 0.49060559272766113, 0.7106375098228455, -1.0300700664520264, 0.30581134557724, -0.5006943941116333, -0.4299826920032501, 1.2714606523513794, -0.6987303495407104, 0.6691247224807739, 0.34749531745910645, -0.30644482374191284, 0.11833801865577698, 1.3759832382202148, 0.2755172550678253, 0.6423378586769104, 0.20115777850151062, -0.023163586854934692, 0.010544652119278908, 1.2389850616455078, -0.4972309470176697, -1.767238736152649, 0.20919828116893768, -0.05535748600959778, 1.199881672859192, -0.17858768999576569, -0.6251852512359619, 0.44308531284332275, -1.56867253780365, 0.38213595747947693, 0.5541625618934631, 0.5858153700828552, 0.7268169522285461, 1.066206932067871, 0.14203962683677673, 0.9253861308097839, -1.1283044815063477, -0.11417413502931595, 1.379150629043579, -1.2344584465026855, 0.3487069606781006, -1.8909856081008911, 0.8613865971565247, -0.5153557658195496, 0.3662099242210388, -0.9856457710266113, -0.0746726244688034, 0.677767276763916, -0.0549531951546669, -0.11177290230989456, 0.7699940204620361, 0.28901657462120056, 0.9557759761810303, -0.3269752860069275, -0.8180637955665588, 0.012683494947850704, 0.22045114636421204, 1.5640405416488647, 0.0748763307929039, -0.027112653478980064, 0.8224777579307556, -0.8027258515357971, -0.687099814414978, 0.8119064569473267, -1.0494999885559082, -0.7906268239021301, 0.2647409737110138, -0.5250672101974487, -0.3193493187427521, 0.359253853559494, -2.0127599239349365, 0.5264812111854553, -0.47939905524253845, 1.9066569805145264, -1.360439419746399, -1.7926855087280273, 0.49648794531822205, -1.2387709617614746, -1.6120902299880981, 0.233201265335083, -0.7083035111427307, -0.9298617243766785, 0.33657681941986084, 0.3167400062084198, -0.27080947160720825, -0.8406462073326111, -0.6531254053115845, -2.065335750579834, -0.8252846598625183, 1.1394848823547363, -0.21194016933441162, -1.0751960277557373, -0.12702801823616028, 0.9787966012954712], [-0.060846950858831406, -0.5208706259727478, -0.6985414624214172, -0.4502391219139099, 1.8854068517684937, -0.66106778383255, -0.9818449020385742, -0.14456866681575775, -0.652761697769165, 0.11960779130458832, 2.6417434215545654, 1.444040060043335, 1.4057552814483643, 0.03786376491189003, 0.5102258920669556, 0.7135829329490662, 0.03172630816698074, -0.6890191435813904, -0.06655996292829514, 0.517049252986908, 0.7774527668952942, -0.05628642812371254, 1.7983839511871338, 0.6005738973617554, -0.6155405044555664, -1.6982964277267456, -0.3181231915950775, -0.2322978377342224, 1.1282532215118408, 0.2636656165122986, -0.626903235912323, 0.07796573638916016, 0.1245468333363533, 0.6565819382667542, -1.8171989917755127, -0.0970437154173851, -0.33461663126945496, -0.0841502845287323, 0.3646438717842102, 1.009437084197998, 1.3715839385986328, 0.27221205830574036, -0.19362179934978485, -0.4449457824230194, -1.2312313318252563, -0.9737905859947205, -0.561553955078125, 2.504101037979126, 0.13516677916049957, 0.7535040974617004, 1.505192518234253, 0.29635289311408997, 0.30821993947029114, -1.0138739347457886, -0.5800654888153076, 0.7586048245429993, 0.15632791817188263, -0.8764079213142395, -0.6406256556510925, -0.07231373339891434, -0.0759393498301506, -0.49214762449264526, -0.7674832344055176, -0.12949880957603455, -0.09697073698043823, 0.5199797749519348, 1.772896409034729, -1.4507825374603271, 0.20874151587486267, 1.093356966972351, 0.42906320095062256, -0.3675801455974579, 0.5944908857345581, 0.7181338667869568, 0.4456838369369507, -0.18571053445339203, -0.609885573387146, -0.4851982295513153, -1.5662914514541626, 0.4144960939884186, 0.42895200848579407, -1.481607437133789, 0.5312960743904114, -0.6096064448356628, 0.16691909730434418, -1.0227915048599243, -0.8517916202545166, 0.26802778244018555, 3.088454008102417, 0.01720902882516384, -2.3267476558685303, -1.0012729167938232, -1.040913462638855, -0.17078959941864014, 1.4906644821166992, 1.081485629081726], [-0.6947423219680786, -0.6279402375221252, -0.04175132140517235, 0.3480145037174225, 0.34692347049713135, 1.3278076648712158, -0.8488497734069824, 1.840105414390564, -0.10289062559604645, -0.9555309414863586, -1.2727324962615967, 0.11867856234312057, 0.23920372128486633, -1.0426621437072754, 0.3135835528373718, -1.2060643434524536, 0.5171387791633606, 0.6757261157035828, 0.5687574744224548, 0.14693590998649597, -1.6094465255737305, 0.8536789417266846, -1.6666070222854614, -0.9091378450393677, -0.13395345211029053, -0.048107318580150604, 1.001668095588684, -0.4053100049495697, 1.2560254335403442, -0.989626407623291, 2.540912389755249, -0.22872702777385712, -0.623113214969635, 0.19829051196575165, -1.7230114936828613, 0.32495948672294617, 0.46927013993263245, 0.1489851176738739, -2.6697659492492676, -0.9693732261657715, -2.299283027648926, -2.319319009780884, -0.5355775952339172, -0.8622367978096008, -1.0358775854110718, -1.5511037111282349, 0.6488873958587646, 0.6102595329284668, 0.48255714774131775, 1.0880496501922607, 0.873862624168396, 0.5201375484466553, 1.1219316720962524, -0.38514241576194763, 1.1317635774612427, -0.11438227444887161, -1.0871403217315674, 3.2550971508026123, 1.441323161125183, -2.967885971069336, -0.3566681146621704, -1.0684332847595215, -2.25313663482666, 1.0893343687057495, 0.37295734882354736, -1.658331274986267, -0.17195461690425873, -0.24014395475387573, -0.9153695106506348, -0.9215587377548218, 0.055981095880270004, -1.6830981969833374, 1.1942903995513916, -0.6397770047187805, -0.9341436624526978, -2.1718780994415283, 1.7421293258666992, -0.7635326385498047, -0.15367506444454193, 0.030167898163199425, 1.2502799034118652, 0.3226752281188965, 0.3159734606742859, -1.5857369899749756, 1.6232857704162598, 0.006734118796885014, 0.963931143283844, 1.3847839832305908, 0.08840204030275345, 0.0869620069861412, 0.5925556421279907, -0.7490553855895996, -0.33931097388267517, 0.4494091272354126, -1.342191219329834, 0.3965773284435272], [-1.5155813694000244, 1.2804303169250488, -0.62520831823349, 1.1127028465270996, 0.45248183608055115, 2.245412826538086, -0.03876984491944313, 0.35824304819107056, 0.5014881491661072, -1.0551679134368896, -0.49524956941604614, -0.7200785875320435, -1.9667892456054688, -0.47830793261528015, 0.13842448592185974, -0.9236229658126831, -0.3159613311290741, -0.21431505680084229, -0.44037890434265137, 0.9267865419387817, -0.5603099465370178, -0.29631155729293823, -0.6156218647956848, 1.3392776250839233, -0.27781277894973755, 0.24507422745227814, -0.25872474908828735, -0.7748779058456421, -0.10860490053892136, 0.22705593705177307, 1.2143021821975708, 0.45621687173843384, -0.5091211199760437, -0.5657997727394104, -0.23520825803279877, 0.6275771260261536, 0.5543985962867737, 0.47395360469818115, -1.816325068473816, 0.7265979647636414, 0.8948836922645569, -0.035163458436727524, -0.09536315500736237, -1.601845622062683, -1.9708629846572876, -0.6961967945098877, 0.71429443359375, -0.057873792946338654, -0.6858285665512085, 0.3090076148509979, -0.28113433718681335, -0.44793665409088135, -0.5377241373062134, -0.3340108096599579, -0.07533086091279984, -1.6904196739196777, -1.2832448482513428, 0.7154251337051392, 0.2873404622077942, 0.44213542342185974, -0.5431607961654663, -0.00755418511107564, -1.5044543743133545, -0.5441421270370483, 1.5157612562179565, -0.0784572884440422, -0.5002793073654175, 0.831731379032135, -0.502223789691925, -0.081351138651371, 0.444524347782135, 0.9826257824897766, 1.1299583911895752, -0.3453736901283264, -1.2978591918945312, -1.9396573305130005, -0.402876615524292, -1.9159963130950928, 1.4089579582214355, 1.1879230737686157, 0.8608014583587646, -0.773772120475769, 1.010133981704712, 0.20483475923538208, 0.6531877517700195, -2.0047152042388916, 1.3274078369140625, 3.6342787742614746, -0.8184220790863037, -0.17556500434875488, 0.2892512381076813, 0.0643642395734787, -1.2302067279815674, -1.88426673412323, 0.1408928781747818, 0.8562701344490051], [0.3213271498680115, -0.7376441359519958, -0.11633294820785522, -0.5683603286743164, -0.003983796574175358, -0.7180922031402588, -1.1912503242492676, -1.2585053443908691, -0.4784710109233856, -1.9742813110351562, -0.2039216160774231, 0.2771511971950531, -0.3123330771923065, 0.5614310503005981, -0.44485238194465637, 0.6998878717422485, -0.6402091383934021, -2.605769395828247, 0.19661618769168854, 0.7102735638618469, 0.5592712163925171, 1.5587836503982544, 0.35267651081085205, -0.31847503781318665, 1.6184568405151367, -1.0993090867996216, 1.834003210067749, 2.4823570251464844, 0.11371131241321564, -0.23702076077461243, -0.29383546113967896, 1.8696391582489014, -0.7468910217285156, 0.10682833939790726, -0.914836049079895, -0.6053770780563354, -0.6867009401321411, -0.8760083913803101, -0.6744338274002075, -1.024914026260376, 0.6126855611801147, -0.05353982001543045, 1.879536509513855, 1.2946499586105347, -0.6442975997924805, -0.5720974206924438, 0.9352565407752991, 0.3664095103740692, 0.2199764847755432, 0.3728553354740143, -0.6714422702789307, -0.27761152386665344, 0.9322806596755981, -1.0449572801589966, 2.691298246383667, -0.4609224498271942, 0.5768371820449829, 0.23455314338207245, -0.7801492214202881, -0.8750551342964172, -1.8784445524215698, 1.2005889415740967, -1.5860226154327393, -0.17613579332828522, 0.1303882598876953, 1.1309001445770264, -0.9430338740348816, -0.3719431459903717, -0.10714641213417053, -1.2494405508041382, 0.600721001625061, 0.1843441128730774, 1.177736759185791, 0.486604243516922, -0.1303323209285736, -2.153432846069336, -0.28507232666015625, -0.2561345398426056, -1.0489705801010132, -1.9922966957092285, 0.14601260423660278, -1.4179030656814575, 0.8073252439498901, -0.1911155730485916, 0.3349516987800598, 0.4133475720882416, 1.1492053270339966, -0.20967647433280945, 1.5823869705200195, -0.056430477648973465, -0.2775849401950836, -0.22565780580043793, -0.7451566457748413, -0.4105020761489868, -1.8922940492630005, 1.6432031393051147], [-0.0010642202105373144, 0.15931221842765808, -0.7264564037322998, -0.7259505987167358, 0.034098587930202484, 0.12411755323410034, -0.8171349763870239, 0.43590980768203735, 0.3149905502796173, 0.523395836353302, -0.9026055335998535, -0.9749341607093811, -0.22747176885604858, -0.1367979794740677, -0.4811500608921051, 1.8070073127746582, -0.9690523743629456, -0.19200986623764038, -1.299174189567566, -0.7748032212257385, -0.5628799796104431, -2.059030055999756, -0.278408020734787, -1.6187233924865723, 0.3009362518787384, 0.13546161353588104, 1.6351802349090576, 0.5107043385505676, 0.20297138392925262, 0.10627385228872299, -1.205275535583496, 1.7032970190048218, 0.5837926268577576, -1.2641547918319702, 1.3323969841003418, -1.3448998928070068, 0.08801145851612091, -0.41836345195770264, -0.44636139273643494, -0.3740406334400177, 0.8332401514053345, 1.2098037004470825, 0.5495949983596802, 1.8890116214752197, 0.5706313252449036, 0.061992283910512924, 0.33492785692214966, -0.010535018518567085, -0.8724800944328308, 0.9065678715705872, -0.8864333629608154, 0.9452752470970154, -1.0498920679092407, 0.9718378186225891, -0.8283172845840454, -0.21581436693668365, 0.12610842287540436, -1.0288219451904297, -1.3076163530349731, 0.39198970794677734, -0.20606769621372223, 1.0386189222335815, -1.4685070514678955, 1.3489270210266113, -0.6795754432678223, -0.21484030783176422, -0.42467761039733887, -0.6243720650672913, 0.9450116157531738, 1.5819377899169922, -0.2286939173936844, 2.022892475128174, 0.38731539249420166, -1.3103971481323242, 0.4915465712547302, -0.8897485136985779, 0.8884421586990356, 0.6490806937217712, 1.426261305809021, -0.6168513298034668, 2.0029454231262207, 0.6084570288658142, -0.439707487821579, 0.28113454580307007, -1.127846360206604, 1.2880934476852417, -0.7137386202812195, -0.4669637680053711, 0.38431525230407715, -1.3361406326293945, 1.0604547262191772, -1.0542564392089844, 0.7901161313056946, 0.0883219838142395, 1.0438481569290161, 0.3272430896759033], [0.6875025033950806, 1.6062707901000977, 0.5923487544059753, -0.3367133140563965, 1.0843522548675537, -1.0376774072647095, 0.4235621988773346, 0.7708294987678528, 1.4173493385314941, -0.1555054634809494, 0.5309040546417236, 0.9300834536552429, -0.6559028029441833, -0.7875769734382629, 1.088511347770691, 0.2810148596763611, -0.038370877504348755, -0.1496196836233139, 0.025393353775143623, 0.9145622849464417, -1.3267204761505127, 1.0383191108703613, 1.0370500087738037, -0.9121322631835938, -0.14745588600635529, 0.45145198702812195, 0.6408969163894653, 0.09257439523935318, -1.7609035968780518, 0.31443411111831665, -0.2605924904346466, 0.2996743321418762, 0.6835620999336243, -0.3556253910064697, 0.5526611804962158, -0.94036865234375, 1.4853122234344482, -2.555614709854126, 1.5586923360824585, 0.406617134809494, 0.05762224644422531, 0.258562296628952, -0.197383850812912, 1.3353207111358643, 0.8759346008300781, 1.364279866218567, -0.3202230930328369, -0.25277262926101685, -0.7274122834205627, 0.3471261262893677, -0.09415458142757416, 0.6212268471717834, 0.13442294299602509, -0.2876521050930023, -1.1030281782150269, -0.04145864024758339, 0.39887797832489014, 1.3929862976074219, -0.11643223464488983, -1.7052263021469116, 0.2052411437034607, -0.937957763671875, -2.1599223613739014, -1.4628552198410034, 1.3576890230178833, 1.5493724346160889, 0.08147229999303818, -0.11176992207765579, 0.6359351873397827, 2.1040565967559814, -0.07962500303983688, -0.07396210730075836, -0.28478625416755676, -1.0517351627349854, -0.45628830790519714, 0.9652644991874695, -0.7865308523178101, 1.058678388595581, -0.4156092405319214, -1.6976360082626343, -0.6845848560333252, -0.045283496379852295, -0.5176560878753662, 0.7248769998550415, -0.7395486235618591, -1.392043948173523, -0.32660722732543945, 0.1270066797733307, 0.6957141757011414, 0.03574345260858536, -1.2785584926605225, 0.9270275235176086, -0.11776070296764374, -0.8647100925445557, 0.05737387388944626, 1.0222482681274414], [-1.0925554037094116, -1.4600725173950195, -0.13210420310497284, 0.26426106691360474, 0.5031127333641052, 0.5203367471694946, -0.12095669656991959, 1.370270848274231, -1.0844184160232544, -1.0430262088775635, -0.578002393245697, -0.060688622295856476, 0.1565302312374115, -0.3082597255706787, 0.2812071144580841, -1.4294501543045044, 0.057308126240968704, 1.5413768291473389, 1.210957407951355, -0.5925107598304749, 0.05212375894188881, -0.17083382606506348, -0.4463721215724945, 0.1745920330286026, 0.8603841662406921, 0.9601648449897766, -0.3440109193325043, -0.29254356026649475, 0.11396408081054688, 1.296971082687378, -0.574040412902832, 1.496513843536377, 0.6767178773880005, -0.20101220905780792, -1.4204022884368896, 0.5197323560714722, 0.05441465601325035, 1.5128310918807983, -1.0668994188308716, 0.5252480506896973, -0.23105329275131226, 0.0010241649579256773, 2.520843267440796, 0.061589401215314865, -0.45055481791496277, 0.22953158617019653, -1.4826514720916748, -0.8395271897315979, 0.34490731358528137, -2.9409432411193848, 0.49624472856521606, 1.2647390365600586, -0.02932245470583439, 1.1594709157943726, -0.3491816222667694, -0.40506312251091003, 1.2696542739868164, 1.3225173950195312, 1.0502301454544067, 0.20961083471775055, -0.5257024765014648, -0.5999802350997925, 0.5378449559211731, -0.30034899711608887, -0.61937016248703, -0.08408793807029724, 2.5408170223236084, -0.5713719725608826, 0.3308975100517273, 0.6470081210136414, -0.28997889161109924, 1.190608263015747, -2.1177406311035156, 3.5516748428344727, 2.0257568359375, 1.7258296012878418, -0.5158749222755432, -1.4004851579666138, -1.217960000038147, 0.4547102749347687, 1.4407062530517578, -0.6775190234184265, 0.7512898445129395, -0.23039312660694122, 0.9228838682174683, -0.38089683651924133, 0.8952319622039795, 0.9481669068336487, -0.36060819029808044, 2.5496811866760254, 0.6549065709114075, -0.032678622752428055, -1.7945935726165771, -0.055102717131376266, 0.49645745754241943, -0.3536655306816101], [-0.1889982372522354, -1.0362367630004883, 0.970110297203064, 0.9178336262702942, -1.5344913005828857, -1.6122493743896484, -0.31400659680366516, 0.9075809717178345, 0.8466393351554871, -0.18360395729541779, 0.3302921950817108, -0.27880406379699707, -0.6478710770606995, 0.19678367674350739, -0.3545355200767517, 0.3477447032928467, 0.5358160734176636, -0.8848810791969299, -0.12520916759967804, -0.8039068579673767, -0.7146314382553101, 0.05509711429476738, 2.3944075107574463, 0.742414116859436, -0.9764314889907837, 1.1696733236312866, 1.0041431188583374, -0.2427625060081482, 1.8938835859298706, 3.250840902328491, -0.42310062050819397, -0.38502830266952515, -1.2845652103424072, 0.2257567197084427, 0.5873547792434692, -0.5468723177909851, 0.8602091670036316, -1.0384621620178223, -0.30738890171051025, -1.5036492347717285, 1.1707710027694702, -2.0466089248657227, -0.5897127985954285, -1.6120017766952515, -1.4043164253234863, 1.7408230304718018, 0.3539796769618988, 0.3298633396625519, 0.378921777009964, 1.0219898223876953, -0.4388011693954468, -1.4917038679122925, 0.555753231048584, 1.48476243019104, 0.9020293951034546, 0.8789302706718445, -1.1464375257492065, 0.8509598970413208, 0.18873484432697296, 0.39244142174720764, -0.3065147399902344, 0.3161886930465698, -0.6261727213859558, -0.1752934455871582, -1.5931415557861328, 0.4470716416835785, 0.3932454288005829, -0.279000848531723, 0.2595117390155792, -1.155875563621521, -0.15074579417705536, 2.8290820121765137, -0.2221411019563675, -0.6597668528556824, -0.8610016703605652, -0.7630043029785156, 1.1927659511566162, -0.26311540603637695, 1.0967570543289185, 1.7002016305923462, -0.3428519070148468, 0.9658199548721313, -1.7269459962844849, -1.1576061248779297, 1.2895375490188599, 1.1652028560638428, -1.5591399669647217, -0.27726516127586365, -0.8089382648468018, -0.25871631503105164, 2.011390447616577, -0.09933432191610336, -0.17223766446113586, 0.6674942970275879, 0.6236920356750488, 1.705362319946289], [0.6882907152175903, 0.37305840849876404, 1.0291935205459595, 0.8901113867759705, 0.09246683865785599, -0.9704704880714417, 1.178097128868103, -0.5335512161254883, 0.3012559115886688, 0.4461226165294647, 0.057118970900774, -1.7973397970199585, 0.28226637840270996, -0.14873623847961426, -0.5351457595825195, -0.8128842115402222, -0.20387573540210724, 1.3273552656173706, 1.9770060777664185, 1.463983178138733, 0.17592382431030273, -1.1307874917984009, 0.5531448721885681, -0.12099073082208633, 0.47725895047187805, -1.3966819047927856, -1.9787100553512573, -0.3893629014492035, 0.504315972328186, 0.48607417941093445, -2.7570695877075195, -0.24540528655052185, 1.3794304132461548, -2.359656810760498, -0.6655018329620361, -0.18029546737670898, 1.0916166305541992, -0.608201801776886, 0.5416746139526367, 1.05613112449646, -1.256394863128662, 0.5920157432556152, -1.2683488130569458, -0.2470862865447998, -2.124500274658203, -0.32434573769569397, -0.513150691986084, -2.715789556503296, -1.3611117601394653, -0.39679884910583496, 1.5589615106582642, 0.774626612663269, -0.539772093296051, -1.035452127456665, 0.4118051826953888, 0.40043750405311584, 1.3278652429580688, -0.3610573709011078, -1.1829824447631836, 0.7710829377174377, 1.4526164531707764, -1.2217425107955933, 2.5932350158691406, -0.019495069980621338, -1.0445256233215332, -0.8124526143074036, -0.09087730944156647, -0.6020406484603882, -1.5951781272888184, -1.3816908597946167, -0.5080049633979797, 1.7406835556030273, 0.041213348507881165, -1.5831108093261719, 0.18005383014678955, 1.5351521968841553, 0.5657293796539307, 0.18769635260105133, -0.13903450965881348, 1.3849918842315674, 1.063576102256775, -0.27024176716804504, 0.5524671077728271, -1.4427627325057983, 0.33189767599105835, 0.08835894614458084, -0.061232149600982666, 0.09427738934755325, 1.04110848903656, -0.3706130087375641, 0.13620132207870483, 1.1452579498291016, -1.8741819858551025, -0.6519526839256287, -1.6666626930236816, -1.0594309568405151], [1.9598112106323242, 0.7080434560775757, 1.1891505718231201, 1.1705291271209717, -0.37687575817108154, 1.4901288747787476, 0.8320131301879883, -2.8008620738983154, -0.7239766120910645, 0.019141066819429398, -0.7023844122886658, 0.43667542934417725, 0.4873761832714081, 0.09224635362625122, -0.0005728742689825594, 0.39101725816726685, -0.7102605104446411, -1.0891008377075195, -0.640031635761261, -0.2869631350040436, 0.5968611836433411, 1.023918867111206, -0.5091974139213562, 0.964061439037323, -0.08280736207962036, 0.01611972413957119, -0.0485706552863121, -0.5968847274780273, -0.06044844537973404, 0.036710724234580994, 0.8313217759132385, 1.915536880493164, -0.484579861164093, -1.6531697511672974, -0.887858510017395, -2.139312505722046, -0.2180011123418808, 0.12838372588157654, -0.4372081160545349, 0.4721768796443939, 1.7546992301940918, -0.1836411952972412, 0.6556496620178223, -1.181762456893921, 0.35259583592414856, -0.46445223689079285, -0.5950542092323303, 1.4184296131134033, 0.40508732199668884, -0.41992950439453125, -0.05485333129763603, 0.861966073513031, -0.39959877729415894, -0.7369527220726013, -0.9602757096290588, 0.7313408255577087, -0.3461480438709259, 1.1408804655075073, -0.24015597999095917, -0.5411460995674133, -0.8441082835197449, -0.8259849548339844, -1.779144525527954, 0.24510866403579712, -0.6431500911712646, 0.6454991698265076, -0.20610752701759338, -0.4911425709724426, 0.9967066049575806, -0.2718712091445923, 0.2832677662372589, 0.5465396642684937, -1.524719476699829, 1.0475279092788696, -1.6792430877685547, 1.0906590223312378, -0.7280406951904297, 0.34027087688446045, 0.29888176918029785, -0.5456480979919434, -2.3220386505126953, 0.516387939453125, 0.3714490234851837, -0.23888802528381348, -0.5201720595359802, -0.5410462617874146, 2.0972654819488525, 0.8618383407592773, -0.06509772688150406, 0.0892842710018158, -1.4349586963653564, 0.6008090972900391, -1.533639907836914, 1.287261724472046, 0.8848022818565369, -1.3665016889572144], [-1.533191204071045, 0.47206082940101624, 0.8878636956214905, -0.5294948816299438, 1.4448440074920654, 1.6166309118270874, -1.5771994590759277, 0.0856047198176384, 0.1644546538591385, -0.235702246427536, 0.9420602321624756, 0.22455117106437683, -0.35982179641723633, 0.8692910075187683, 1.0457828044891357, -0.2392355352640152, -1.4116533994674683, 1.860353946685791, 0.3293973207473755, -1.034232497215271, 0.6914278864860535, 0.8115106225013733, -1.420850157737732, 2.4707229137420654, -0.21613536775112152, -1.091919183731079, 0.7655707001686096, 1.2036241292953491, -1.7470612525939941, -1.1345371007919312, 0.24993790686130524, -0.29431524872779846, -0.11246991157531738, -0.9272220730781555, -0.5349639058113098, -0.9355034232139587, 0.09409861266613007, -0.9267642498016357, 1.0233575105667114, 1.7761954069137573, -0.688590407371521, 1.7707045078277588, 0.18099911510944366, -0.5572174191474915, 1.523533582687378, -0.0401310995221138, -0.05867309868335724, 1.9192978143692017, -0.2972121238708496, 0.05621141567826271, -1.1632440090179443, -0.35618892312049866, -2.202068328857422, -0.7184309959411621, -0.04419361427426338, -0.583869218826294, -0.4671138823032379, -1.6569303274154663, -0.779497504234314, 0.12604641914367676, 1.3741191625595093, 1.1778455972671509, -0.13921356201171875, 0.1967753767967224, -1.1239351034164429, -1.2759425640106201, -1.1362255811691284, 1.6742215156555176, -0.42337167263031006, -3.0172083377838135, 0.043353646993637085, -0.9509457945823669, 0.6205078959465027, -1.0024598836898804, 1.0207031965255737, 0.04902646690607071, -0.8355078101158142, -0.4935056269168854, 1.009887933731079, 0.977872908115387, 0.19829919934272766, -0.053183600306510925, 2.168166399002075, -0.28213945031166077, -0.8766719698905945, 0.7117265462875366, -0.2831759452819824, -1.0952671766281128, -0.5813995599746704, 1.0512819290161133, -1.1436378955841064, -0.15114332735538483, 1.8943263292312622, -1.1269620656967163, 0.8325222134590149, 0.6605420112609863], [0.02610745280981064, 0.016763200983405113, -1.2103309631347656, 0.7964891195297241, 1.1408884525299072, -0.9084625840187073, -0.5539252758026123, 0.6446928977966309, 0.15133942663669586, -0.4116850793361664, 0.37894386053085327, 0.2566261887550354, -1.5047448873519897, 1.762662410736084, 0.30439823865890503, 1.2383817434310913, -1.7586904764175415, 1.8237062692642212, 0.3437284231185913, 1.195945382118225, 0.2395123690366745, -0.8051555156707764, 0.3547093868255615, -1.0227323770523071, 0.21640919148921967, -0.3302343189716339, -0.061119481921195984, 1.9200141429901123, -0.6588571667671204, -1.0701977014541626, -0.07402992248535156, -0.6885637640953064, -0.4854869246482849, -0.016728971153497696, -1.9583940505981445, 0.3873807489871979, 0.255130410194397, -1.3897638320922852, -0.7312731146812439, 1.6773663759231567, -0.06325141340494156, -0.2960452735424042, 0.45015984773635864, -0.013102279044687748, 1.0171325206756592, 0.7098400592803955, -0.6802484393119812, 0.7023897767066956, 0.3593452572822571, -1.6457645893096924, -0.01232729572802782, -0.12534207105636597, -0.34382468461990356, -2.4792497158050537, 1.800580382347107, 0.5644165277481079, -0.4815966784954071, 0.1075177788734436, 0.12096388638019562, -0.09877713769674301, -0.10291627794504166, -1.2626577615737915, 3.155128002166748, 1.9319871664047241, 2.469569683074951, 1.2182445526123047, -1.2734289169311523, 1.0747812986373901, -0.0009922755416482687, -0.07806774973869324, 0.149480402469635, 0.7895827889442444, -1.2775989770889282, -1.6144431829452515, 0.5805611610412598, 1.4896016120910645, -0.1693659871816635, -1.1992835998535156, -0.908791184425354, -1.7105079889297485, -0.7529899477958679, -1.5361422300338745, 1.0641573667526245, 0.21449430286884308, -0.9613202810287476, 1.1555529832839966, -0.22159895300865173, -3.0439465045928955, 1.431301474571228, 1.1326453685760498, 2.4829540252685547, -0.32634401321411133, 0.150232195854187, 0.8860871195793152, -0.06889645755290985, 0.48960381746292114], [0.5498005151748657, -0.6022252440452576, -0.09035033732652664, 0.5880242586135864, -1.2718896865844727, 0.06966841220855713, -1.5861567258834839, -0.5015843510627747, 0.3723418712615967, 0.2728882133960724, 0.6566246747970581, 0.5986600518226624, 0.6956967711448669, 0.2521486282348633, 1.202344298362732, 0.006923465058207512, 0.41945263743400574, 0.7921784520149231, 0.34134441614151, 0.44503360986709595, 0.10704159736633301, 0.0549473762512207, -0.6548188924789429, -1.657125473022461, -0.6174361109733582, 0.4037744700908661, -1.8752506971359253, -2.296938896179199, 0.4244038462638855, 1.3446147441864014, 0.5515428781509399, -0.3283722698688507, 1.7817330360412598, -0.4697168469429016, -0.171635240316391, -0.650499701499939, -0.6814447641372681, -0.5944355130195618, 0.9776827096939087, -1.0958119630813599, -0.27823129296302795, -0.3757339417934418, 0.6553841829299927, -0.12745927274227142, -0.282529354095459, 0.5790281295776367, -0.44204962253570557, -2.2064034938812256, 1.0993907451629639, -0.2788209915161133, -0.23641668260097504, -0.2560800313949585, -0.11027475446462631, 0.4602755606174469, -0.7305753231048584, -1.6302167177200317, 0.4759191572666168, -0.44711580872535706, -0.5160526037216187, 0.6670427918434143, -2.2066736221313477, 0.5599199533462524, -0.42162075638771057, -0.12856410443782806, -0.608036458492279, 1.6683504581451416, 1.0407061576843262, -1.2595231533050537, -0.9678248167037964, 0.48650315403938293, 2.8038017749786377, -1.1796526908874512, 0.13554900884628296, 0.28717729449272156, -0.7906397581100464, -0.2533465027809143, -1.2211276292800903, 2.295386791229248, -0.39516136050224304, 0.4329536557197571, 1.2828930616378784, -0.7509334683418274, 0.22282785177230835, 0.571972668170929, -1.359739065170288, 0.9752518534660339, -0.32786327600479126, -0.037915267050266266, 0.8574856519699097, 0.576607882976532, -1.9515479803085327, -0.7222138047218323, -0.7063129544258118, -0.8249700665473938, -0.31107276678085327, -1.750848650932312], [0.037630852311849594, 0.484231173992157, -0.6258285045623779, -0.14921046793460846, 1.0220065116882324, -0.8178368806838989, 0.7014876008033752, 0.6316587924957275, -0.8642702102661133, -1.2532615661621094, -0.09022179245948792, 2.130565881729126, -0.807908833026886, 0.35493844747543335, -0.7865716814994812, -0.35535892844200134, -0.4576856195926666, -0.0007278291741386056, -0.030988972634077072, -0.30941784381866455, 0.03052780218422413, 1.367542028427124, 0.7325889468193054, 0.6585139036178589, 0.5501542687416077, -1.6365132331848145, 0.05720163881778717, 0.7160124778747559, -0.881231427192688, -0.9001328945159912, -0.4397338032722473, 1.424919843673706, -1.6966309547424316, -1.690281629562378, -1.3304777145385742, -0.5386550426483154, -0.48358842730522156, -0.5950033664703369, -0.632331907749176, -0.9273660778999329, -0.8448405861854553, 1.8531438112258911, 0.6866325736045837, 0.8182551264762878, 0.5974019169807434, -1.3744693994522095, -0.7319909930229187, -0.29536670446395874, -0.36860764026641846, 0.27819398045539856, -0.4077765941619873, 1.6595537662506104, -0.3667312264442444, -0.6723060011863708, -0.39916735887527466, -0.5183998346328735, 0.0530194528400898, 0.32605481147766113, -2.238908052444458, 2.1216609477996826, -1.1011558771133423, -2.423474073410034, 0.21402433514595032, -0.2920610010623932, 0.660913348197937, -1.0480103492736816, 1.0422000885009766, -1.5357927083969116, -0.5644207000732422, -0.17051143944263458, -1.0861804485321045, 1.072704553604126, 0.6179593801498413, 0.2658344507217407, 0.9237396717071533, -0.33266133069992065, -0.33809974789619446, 0.09147386252880096, 0.35526132583618164, 1.3136845827102661, -1.1155178546905518, 0.6626761555671692, -0.47279244661331177, -0.8388611078262329, 1.2195215225219727, -0.5029804706573486, -0.26556047797203064, -0.6209532022476196, -0.37932878732681274, -0.8158462643623352, 0.26415374875068665, 1.7299457788467407, -2.000690221786499, 0.12703202664852142, -0.9165821075439453, 1.9373129606246948], [-1.02805495262146, 0.00214770738966763, 0.6243345737457275, 0.0415397584438324, 0.7067608833312988, -0.4560406506061554, 0.3170171082019806, 0.2869047224521637, -1.3129281997680664, -1.952575445175171, 0.0655117928981781, 0.3112657070159912, 1.2059004306793213, -1.1938586235046387, 2.415534734725952, -0.32427090406417847, 0.9504192471504211, 0.6855718493461609, 1.8480921983718872, 0.695819079875946, -0.012563493102788925, -0.3194132149219513, -0.030575856566429138, 0.906424343585968, -2.013221502304077, -0.7280277609825134, 1.19332754611969, 0.01783853769302368, -0.45912879705429077, -0.8418806195259094, -0.22432966530323029, 0.7122549414634705, -0.12814852595329285, -0.43949180841445923, 2.5795538425445557, -0.16387338936328888, 0.18404634296894073, 0.8884126543998718, -0.7836039066314697, -0.5620460510253906, 0.7667551636695862, -0.6629223227500916, -0.13643383979797363, -0.11770640313625336, -0.7096049785614014, -0.4616483747959137, 0.03854724392294884, -0.7109730243682861, 0.25913184881210327, 1.0050517320632935, 0.3710896968841553, -1.0426021814346313, -1.8094624280929565, 0.05310244858264923, 1.9940755367279053, 0.9243289828300476, -0.5168721079826355, 0.14061956107616425, 1.4910612106323242, -0.13534927368164062, -1.444733738899231, 0.6498380303382874, 0.7183987498283386, 2.8657610416412354, 0.862321138381958, -0.7735516428947449, -0.7984888553619385, -0.5189745426177979, 0.15596795082092285, 1.2790178060531616, -0.9342981576919556, -1.8277760744094849, -0.4037335216999054, -0.4477449655532837, 1.141933798789978, -0.02452453039586544, -0.3024759888648987, -0.10331299155950546, 0.8456394672393799, -0.7158005833625793, -0.1301974654197693, 0.2655896246433258, 0.09066904336214066, -0.3511626422405243, -0.21443967521190643, 0.3770495355129242, -0.8928378820419312, 1.8491578102111816, 0.5395416617393494, -0.96806401014328, -0.11557811498641968, -2.2043402194976807, 0.2247496098279953, -1.2243138551712036, -1.1310851573944092, 0.14697547256946564], [-0.1813599318265915, -1.505586862564087, 0.1485244780778885, -0.8778761625289917, 1.5481551885604858, 0.11019236594438553, -0.8683099746704102, 0.3190440535545349, 0.3554689288139343, -0.1830611228942871, -0.34677204489707947, -0.5724239349365234, 0.787411630153656, 0.9984375834465027, 0.1492200493812561, 1.197838306427002, 1.172309398651123, 0.4223523736000061, -0.29227885603904724, -1.1278619766235352, 0.3772279620170593, 0.06582632660865784, 0.6795807480812073, 0.07602623105049133, -1.5376851558685303, -1.1895228624343872, 1.857283115386963, 0.881818950176239, 1.4454162120819092, -0.6055451035499573, -0.599966824054718, -1.164109230041504, -0.08169499039649963, -0.18052923679351807, 1.52060067653656, -0.7055811285972595, -1.1957945823669434, 0.611372172832489, -1.6979490518569946, 0.23402796685695648, 0.2627042531967163, -0.0016564984107390046, 0.32620498538017273, 0.4367236793041229, 0.4509105980396271, 0.44172796607017517, 1.5731276273727417, 0.06084803491830826, -0.2690187692642212, 1.1796563863754272, -0.061907339841127396, 1.596592903137207, -0.05842959135770798, 0.755535900592804, -0.8185960650444031, 0.6302377581596375, 0.346065878868103, 0.036140650510787964, 0.26341941952705383, 0.14882762730121613, 0.4868672490119934, 0.6297963857650757, -0.14120377600193024, -1.0088131427764893, 1.1607853174209595, -0.5685067772865295, 0.9934708476066589, -0.9753064513206482, 0.21950408816337585, 0.449133962392807, -0.9040523171424866, -0.5493910908699036, 0.27063125371932983, -0.6193239688873291, 0.6867014169692993, -1.6454824209213257, 0.4586719274520874, -1.6314483880996704, 0.01004298310726881, 0.4939936101436615, -0.19682934880256653, 0.6011071801185608, 0.5228930711746216, 1.3263617753982544, 1.1350215673446655, 0.2175407111644745, 1.1889969110488892, -0.6224366426467896, -0.6557298302650452, -2.150677442550659, -0.07217197120189667, -0.9326290488243103, -0.5981243252754211, 0.5154805779457092, 0.42468056082725525, -0.9418259263038635], [-1.072517991065979, -2.2012696266174316, 0.7363234758377075, -0.12757426500320435, 0.9109974503517151, -0.1927342414855957, 0.5085883736610413, 1.2856314182281494, -0.07291355729103088, -0.8409004807472229, 0.3909424841403961, 0.15333901345729828, -1.6863007545471191, -0.2686994969844818, -0.09145090728998184, -0.5977205038070679, -1.8021929264068604, 0.33120784163475037, -0.2010420262813568, -0.03793719783425331, -0.8330771923065186, 0.3401210308074951, 0.2710520923137665, 1.548866868019104, -0.9794071316719055, -0.9893776178359985, 1.1925193071365356, 1.0976802110671997, -0.28815126419067383, 1.1870317459106445, -1.4303566217422485, -0.9847787022590637, 0.12349741905927658, 2.455455780029297, -1.9215060472488403, 1.0768985748291016, 0.9275817275047302, -0.4650452435016632, -0.2596109211444855, 0.36027175188064575, 1.0566203594207764, 1.2077146768569946, -1.0920823812484741, 0.8595306277275085, 0.5547151565551758, 2.0539252758026123, -1.0579748153686523, 0.5508511066436768, -0.06281118094921112, -0.1762140840291977, 0.11279290169477463, -1.0618212223052979, 0.30615246295928955, 0.4930442273616791, 2.9081263542175293, 0.8988246917724609, -0.14628584682941437, -1.804368495941162, 0.389598548412323, -0.9117223620414734, 1.0823607444763184, 0.07204481214284897, 1.5196274518966675, 1.0997753143310547, 0.0878388062119484, 0.9330781698226929, 0.06307495385408401, 1.280548095703125, -0.3946443498134613, 1.023637294769287, -0.33074918389320374, -0.7725344896316528, -0.9564524292945862, 0.20957989990711212, -1.1296226978302002, 0.3794920742511749, 0.6264047622680664, -0.726265549659729, -0.11440476030111313, 0.9068830609321594, -2.6073880195617676, -0.1368764191865921, 0.38977518677711487, 0.7990684509277344, 0.7626919746398926, -0.450698584318161, -0.9246488213539124, 0.8285495042800903, 1.0965549945831299, 0.3855237066745758, -0.5294979810714722, -0.6231870651245117, -0.511512041091919, -0.17229963839054108, -0.8944901823997498, -0.5146782994270325], [-0.3316926956176758, 0.10189057141542435, 0.8001817464828491, -0.42864128947257996, 0.7177925109863281, 2.358081817626953, 0.24156402051448822, 1.5371066331863403, -1.391611099243164, -0.2538241446018219, -0.7348254323005676, -0.24115808308124542, -0.5009720325469971, 1.540804147720337, 0.5781787037849426, -2.771848678588867, -0.19764269888401031, 0.826805591583252, -0.7099612951278687, 0.28648000955581665, 0.04486097767949104, -0.7529522776603699, 1.586755394935608, 0.9457917213439941, -1.7726587057113647, -2.035601854324341, -0.2770738899707794, -0.4515179991722107, -0.7597249150276184, 0.10118643194437027, 0.30527353286743164, -0.8844150304794312, 0.1709885597229004, 0.10265818238258362, -0.6515074372291565, -0.2028087079524994, -1.8604379892349243, -0.18675069510936737, 1.2832056283950806, 0.878779411315918, 1.0840355157852173, 0.11806187778711319, -1.6606560945510864, 0.12626494467258453, -2.189392328262329, 0.6695953011512756, 0.47530874609947205, -0.17311200499534607, -0.4916004538536072, 0.07121676206588745, 0.029995551332831383, -0.7677318453788757, 0.04734150320291519, 0.5635189414024353, -0.3761166036128998, 1.3051129579544067, 2.035371780395508, -0.7325586676597595, -0.5818691253662109, 0.9164446592330933, 1.1986525058746338, -0.8729230761528015, -0.5728959441184998, -0.573684573173523, -1.7248015403747559, -1.3927561044692993, 0.6090384721755981, -1.9580261707305908, 0.03649603947997093, -1.7806557416915894, 0.18092454969882965, -0.061763517558574677, -0.24253807961940765, -1.3846124410629272, -0.4351580739021301, 0.6869863867759705, -1.8623785972595215, -0.8430503606796265, 0.14695985615253448, 0.3854405879974365, -0.6517239809036255, 0.47567734122276306, -1.693323016166687, 1.75727117061615, 1.7504969835281372, 0.5697077512741089, 1.155483603477478, -1.6206028461456299, 0.3802025020122528, -0.5630570650100708, -1.50613534450531, -0.09791287034749985, 0.8174574971199036, -0.7779402136802673, -0.20435957610607147, 0.1770697981119156]], "dtype": "float32"}, {"__nd__": [[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]], "dtype": "float32"}], "e": {"__nd__": [[-0.27645379304885864, 0.7481147646903992, -1.1707737445831299, 0.1488669067621231, -0.18068517744541168, 0.4108226001262665, 0.21832863986492157, -1.8482528924942017, -0.4068543314933777, -0.6740044355392456, -1.3039382696151733, -0.6590227484703064, 0.2734498679637909, 0.6880481243133545, -0.5975197553634644, -1.3205764293670654, -1.300257921218872, 0.3507102131843567, -0.6377712488174438, -0.8078844547271729, 1.0256930589675903, -0.8355675339698792, -0.30766695737838745, 0.1515234410762787, 0.30840644240379333, 1.6268253326416016, -0.16445162892341614, 2.0110509395599365, 1.1976176500320435, -0.1183004081249237, 1.363362193107605, 0.027401257306337357, 1.0427892208099365, 1.822503685951233, -1.183242917060852, 0.7450013160705566, -0.8561116456985474, 0.25262972712516785, 0.15432797372341156, -0.2080903947353363, -0.5465373992919922, -1.6862233877182007, -1.9015482664108276, -0.5711967349052429, -0.15619587898254395, 0.5051832795143127, -0.060846950858831406, -0.6947423219680786, -1.5155813694000244, 0.3213271498680115, -0.0010642202105373144, 0.6875025033950806, -1.0925554037094116, -0.1889982372522354, 0.6882907152175903, 1.9598112106323242, -1.533191204071045, 0.02610745280981064, 0.5498005151748657, 0.037630852311849594, -1.02805495262146, -0.1813599318265915, -1.072517991065979, -0.3316926956176758], [-0.7689549326896667, 0.3296716809272766, 0.497458815574646, 1.3875857591629028, -2.418142080307007, 0.15801800787448883, -0.6945196986198425, 1.0631446838378906, -0.7516087293624878, 0.4488801956176758, 2.5564613342285156, 0.5611188411712646, 0.40216052532196045, 0.2100573182106018, 1.6203774213790894, 0.20394226908683777, 0.7845257520675659, -0.3971508741378784, -0.10361050069332123, 0.45082390308380127, 1.2949376106262207, -1.08591628074646, -0.5730839371681213, -1.9834918975830078, -0.48947596549987793, -0.8527037501335144, 1.0632051229476929, 1.4201388359069824, -1.3064181804656982, -0.7172607183456421, -0.6439046263694763, 1.8418749570846558, -0.40952450037002563, 0.09847042709589005, 0.5126332640647888, 0.20722205936908722, 1.16602623462677, 0.9022700786590576, 0.6288573741912842, -0.8760420680046082, -1.1997038125991821, -0.40936627984046936, 0.7476085424423218, 0.6923551559448242, 0.4641769528388977, 2.6868557929992676, -0.5208706259727478, -0.6279402375221252, 1.2804303169250488, -0.7376441359519958, 0.15931221842765808, 1.6062707901000977, -1.4600725173950195, -1.0362367630004883, 0.37305840849876404, 0.7080434560775757, 0.47206082940101624, 0.016763200983405113, -0.6022252440452576, 0.484231173992157, 0.00214770738966763, -1.505586862564087, -2.2012696266174316, 0.10189057141542435], [0.0414571613073349, -0.4403088390827179, 0.13175828754901886, -0.4414026141166687, 0.5542165637016296, 1.688189148902893, 1.2072584629058838, -0.459200918674469, 0.8311585783958435, 0.4319309890270233, -0.6662469506263733, 0.10161955654621124, -0.25760123133659363, 1.3767822980880737, 1.036271572113037, -0.2519756257534027, -0.9126979112625122, -0.7719241976737976, -0.7680198550224304, -0.33087974786758423, 1.286942958831787, -0.563302755355835, 0.6949666142463684, 0.0268104188144207, -0.36231571435928345, 0.3376457989215851, -0.1986686736345291, -1.6197713613510132, 0.6889997124671936, 1.0247786045074463, 0.33536094427108765, 0.6258484721183777, 0.327635794878006, -1.040015697479248, -0.024630453437566757, 1.5042821168899536, 0.379313588142395, 0.6493802070617676, 1.1056894063949585, -0.21234455704689026, 0.8618616461753845, -1.0998175144195557, -1.8797082901000977, 0.19928477704524994, -0.2359800636768341, 0.3031651973724365, -0.6985414624214172, -0.04175132140517235, -0.62520831823349, -0.11633294820785522, -0.7264564037322998, 0.5923487544059753, -0.13210420310497284, 0.970110297203064, 1.0291935205459595, 1.1891505718231201, 0.8878636956214905, -1.2103309631347656, -0.09035033732652664, -0.6258285045623779, 0.6243345737457275, 0.1485244780778885, 0.7363234758377075, 0.8001817464828491], [-0.49288031458854675, -0.4361031651496887, 0.9355738162994385, -0.07944782078266144, -0.849918782711029, -0.5631324648857117, -0.7713683843612671, 0.04184122011065483, -0.1703687161207199, -0.1766384094953537, 0.5700178742408752, 0.18518009781837463, 0.3743344843387604, -0.5782783627510071, -0.05744960904121399, 0.1186985895037651, 0.8594087958335876, -0.088943712413311, 0.9424458146095276, -0.49249905347824097, 0.018920475617051125, -1.200291395187378, 0.5011774897575378, -0.7922961711883545, -1.3099031448364258, 0.03839258849620819, 0.2552734315395355, -0.9687070250511169, -1.6152396202087402, -0.42537176609039307, 0.48357200622558594, 0.6158586144447327, 0.7749161720275879, -1.088496208190918, -1.253042221069336, -0.9763807654380798, -0.7862539291381836, 1.5245238542556763, -0.7062654495239258, -0.14337900280952454, -1.7298787832260132, 1.5468961000442505, -0.5553980469703674, 1.5504213571548462, -0.8708043098449707, 0.49060559272766113, -0.4502391219139099, 0.3480145037174225, 1.1127028465270996, -0.5683603286743164, -0.7259505987167358, -0.3367133140563965, 0.26426106691360474, 0.9178336262702942, 0.8901113867759705, 1.1705291271209717, -0.5294948816299438, 0.7964891195297241, 0.5880242586135864, -0.14921046793460846, 0.0415397584438324, -0.8778761625289917, -0.12757426500320435, -0.42864128947257996], [0.409775972366333, -3.007012367248535, 0.12189851701259613, -0.29179847240448, -0.3696237802505493, -1.2569360733032227, -0.9187902212142944, -0.09742239862680435, -0.7577875852584839, -0.4407304525375366, -0.3122805655002594, 0.9623098969459534, 0.637567400932312, 0.21582883596420288, 0.8127167224884033, 2.6325342655181885, -1.1310136318206787, -1.1912686824798584, -0.11257556080818176, 0.9540289044380188, -1.4043294191360474, 0.11009657382965088, -1.0055524110794067, 1.248477816581726, -0.13548484444618225, 0.03779969736933708, -0.773216724395752, -1.0337351560592651, -1.1260019540786743, -0.5721125602722168, -1.6333192586898804, 2.687063217163086, 0.682357907295227, -0.6591488122940063, -1.7864255905151367, -0.1868949830532074, -0.575599193572998, 0.028394728899002075, -0.9782150983810425, -0.5938985347747803, 0.6131479144096375, 1.291633129119873, 0.6110749244689941, 0.5941425561904907, -1.1843030452728271, 0.7106375098228455, 1.8854068517684937, 0.34692347049713135, 0.45248183608055115, -0.003983796574175358, 0.034098587930202484, 1.0843522548675537, 0.5031127333641052, -1.5344913005828857, 0.09246683865785599, -0.37687575817108154, 1.4448440074920654, 1.1408884525299072, -1.2718896865844727, 1.0220065116882324, 0.7067608833312988, 1.5481551885604858, 0.9109974503517151, 0.7177925109863281], [0.7577093839645386, -1.0975955724716187, -1.7422337532043457, -0.5095446109771729, 1.8506059646606445, 0.7064513564109802, -1.372784972190857, -0.7728593945503235, -0.3254649341106415, -2.1256768703460693, 0.49322500824928284, 0.4564954936504364, -0.30743974447250366, -0.7017017006874084, -0.43342798948287964, -0.10224272310733795, 0.7327052354812622, 0.5711201429367065, 0.9148154258728027, -1.1228574514389038, -1.1270830631256104, -0.22442907094955444, -0.11654257774353027, 1.2985080480575562, 0.3315483033657074, -1.425466537475586, -0.1513361930847168, -0.6209895610809326, -1.0142018795013428, 1.338106393814087, -1.6453783512115479, -0.9516943693161011, -0.372107595205307, 0.8120267391204834, -0.059955839067697525, 0.46660566329956055, -2.1598901748657227, 0.5345209240913391, -1.4106793403625488, -0.6677464842796326, -0.4243265986442566, 0.4331318438053131, -0.20799103379249573, -1.5285931825637817, 0.6548794507980347, -1.0300700664520264, -0.66106778383255, 1.3278076648712158, 2.245412826538086, -0.7180922031402588, 0.12411755323410034, -1.0376774072647095, 0.5203367471694946, -1.6122493743896484, -0.9704704880714417, 1.4901288747787476, 1.6166309118270874, -0.9084625840187073, 0.06966841220855713, -0.8178368806838989, -0.4560406506061554, 0.11019236594438553, -0.1927342414855957, 2.358081817626953], [0.7189847230911255, 0.23198355734348297, -0.5119315385818481, -0.01437341421842575, -0.7342088222503662, -0.5762571096420288, -0.04524221271276474, 0.73299241065979, -0.37978100776672363, 0.6274816989898682, 0.852222740650177, 2.057990550994873, -0.571424126625061, 0.05751883238554001, 0.024352598935365677, 0.45780134201049805, 0.3247471749782562, 0.25236502289772034, -0.3320070207118988, 0.9050134420394897, -0.8161957263946533, 0.07791376858949661, -0.4664148688316345, -1.3346272706985474, -1.4011578559875488, 1.3062591552734375, 1.7165688276290894, -2.379234790802002, -0.05438602715730667, 1.0320425033569336, -0.01248274277895689, 0.10523886978626251, 0.31132423877716064, -0.27076297998428345, -0.6601395010948181, -1.2421131134033203, -0.46001651883125305, 1.5261439085006714, -1.0303674936294556, -1.261664867401123, 0.7282776236534119, 0.10963940620422363, -1.4679794311523438, 1.3717480897903442, -1.364518642425537, 0.30581134557724, -0.9818449020385742, -0.8488497734069824, -0.03876984491944313, -1.1912503242492676, -0.8171349763870239, 0.4235621988773346, -0.12095669656991959, -0.31400659680366516, 1.178097128868103, 0.8320131301879883, -1.5771994590759277, -0.5539252758026123, -1.5861567258834839, 0.7014876008033752, 0.3170171082019806, -0.8683099746704102, 0.5085883736610413, 0.24156402051448822], [2.4413673877716064, 1.514647126197815, 0.3544447422027588, 1.1583361625671387, 0.6310199499130249, 1.4065800905227661, 0.5863187313079834, 1.0232402086257935, 0.8049561977386475, 0.411162793636322, -0.5945652723312378, -1.2391868829727173, -0.04231516271829605, 0.4133877456188202, 0.2254824936389923, 1.068542242050171, 0.2344178408384323, -0.04898883402347565, -0.28669095039367676, 0.8058010339736938, 1.4603837728500366, -0.3174171447753906, 0.6056992411613464, 0.7187955975532532, -0.8435072898864746, 1.7169396877288818, 0.7051886320114136, -0.694966733455658, 0.46831172704696655, 0.38992422819137573, 0.2888539433479309, 0.5479486584663391, -0.46183493733406067, 0.6698032021522522, -0.20029836893081665, 1.5791423320770264, -0.19338135421276093, -1.0272457599639893, 0.24178743362426758, 0.5816309452056885, -0.4972103536128998, -0.7835298180580139, 1.105180025100708, 0.5570328831672668, 0.030443591997027397, -0.5006943941116333, -0.14456866681575775, 1.840105414390564, 0.35824304819107056, -1.2585053443908691, 0.43590980768203735, 0.7708294987678528, 1.370270848274231, 0.9075809717178345, -0.5335512161254883, -2.8008620738983154, 0.0856047198176384, 0.6446928977966309, -0.5015843510627747, 0.6316587924957275, 0.2869047224521637, 0.3190440535545349, 1.2856314182281494, 1.5371066331863403], [0.8960291147232056, -0.8145979046821594, 1.4346803426742554, 1.557712197303772, 0.4122227430343628, 0.3334614336490631, -1.05612313747406, -0.5287643074989319, -2.2298247814178467, 0.9241509437561035, 1.0072555541992188, -0.46653059124946594, -0.3792320489883423, 1.1769485473632812, -1.300865888595581, 1.064026117324829, 0.17637188732624054, -0.11612094193696976, 2.1896519660949707, -0.6465059518814087, 0.5502263903617859, 2.135545492172241, 0.6343197226524353, -1.824532389640808, 1.5993691682815552, 0.4521375298500061, 0.8909255266189575, -0.35868602991104126, -1.8541353940963745, -0.6266863942146301, -0.4491156339645386, -1.0974419116973877, -1.1897412538528442, 0.8576703667640686, 0.790339469909668, -0.13171391189098358, -0.01618901640176773, -0.7646031975746155, -0.2502506375312805, -0.6039487719535828, 0.5617464184761047, -0.4777171015739441, -0.29163068532943726, -1.453650951385498, 2.502622604370117, -0.4299826920032501, -0.652761697769165, -0.10289062559604645, 0.5014881491661072, -0.4784710109233856, 0.3149905502796173, 1.4173493385314941, -1.0844184160232544, 0.8466393351554871, 0.3012559115886688, -0.7239766120910645, 0.1644546538591385, 0.15133942663669586, 0.3723418712615967, -0.8642702102661133, -1.3129281997680664, 0.3554689288139343, -0.07291355729103088, -1.391611099243164], [-0.9684327244758606, 1.415996789932251, 0.4200025498867035, 0.26632893085479736, 0.15901996195316315, -0.33365142345428467, -0.3872241973876953, 0.45015525817871094, 0.5093698501586914, -0.6585179567337036, 0.3385697901248932, -1.180609107017517, 0.11744367331266403, 0.6467233300209045, 0.4688790738582611, -1.0237159729003906, 0.6673658490180969, 1.2804702520370483, 0.4748496115207672, 0.781900942325592, -1.8323789834976196, 0.27703380584716797, -0.37459227442741394, -0.9306024312973022, 1.482879877090454, -0.08103445172309875, -0.5828596353530884, -0.18780387938022614, -1.1490991115570068, -0.9515830278396606, 0.5849986672401428, 1.2690577507019043, 0.1788203865289688, -0.6995436549186707, 0.9215829968452454, 2.7317230701446533, -3.0949208736419678, 2.4120147228240967, 0.9419523477554321, -0.04236046224832535, 0.6556074619293213, 1.4941511154174805, 1.0520374774932861, 0.6832805275917053, -0.9009549021720886, 1.2714606523513794, 0.11960779130458832, -0.9555309414863586, -1.0551679134368896, -1.9742813110351562, 0.523395836353302, -0.1555054634809494, -1.0430262088775635, -0.18360395729541779, 0.4461226165294647, 0.019141066819429398, -0.235702246427536, -0.4116850793361664, 0.2728882133960724, -1.2532615661621094, -1.952575445175171, -0.1830611228942871, -0.8409004807472229, -0.2538241446018219], [1.2713515758514404, 1.314540982246399, 0.5538464784622192, 1.0068614482879639, 0.2528240978717804, -0.5812724232673645, 0.46865949034690857, -0.7383673191070557, -0.5444512367248535, 0.13086169958114624, -1.4525913000106812, -0.8098660111427307, 0.2211356908082962, 0.6348134875297546, 0.950425922870636, 1.3136482238769531, -1.5583299398422241, -1.0712445974349976, -1.3731744289398193, -1.2908142805099487, -0.5719563961029053, -0.8740128874778748, 0.7714011073112488, -0.8213937878608704, 0.4977439343929291, -0.26324743032455444, 0.06910236179828644, 1.0804489850997925, 0.7065098881721497, -0.3453102707862854, -0.6794925928115845, -0.7887439131736755, -1.802346110343933, 0.7213115096092224, 0.4453074634075165, 0.005696030333638191, 0.13955813646316528, 2.156670093536377, -0.9824033379554749, 0.6303580403327942, 0.5433747172355652, 1.2716408967971802, -0.17579954862594604, 0.08871778845787048, 0.12428754568099976, -0.6987303495407104, 2.6417434215545654, -1.2727324962615967, -0.49524956941604614, -0.2039216160774231, -0.9026055335998535, 0.5309040546417236, -0.578002393245697, 0.3302921950817108, 0.057118970900774, -0.7023844122886658, 0.9420602321624756, 0.37894386053085327, 0.6566246747970581, -0.09022179245948792, 0.0655117928981781, -0.34677204489707947, 0.3909424841403961, -0.7348254323005676], [0.2495608925819397, 0.3722347319126129, 0.7239682078361511, 0.46477237343788147, -0.5170620083808899, -0.3830476999282837, -0.1771385818719864, 0.3555351793766022, -0.31516408920288086, 0.10923223942518234, -2.955590009689331, 0.38384056091308594, -1.510091781616211, -1.2479939460754395, 0.23820136487483978, 0.4487760663032532, 1.5019325017929077, 0.00899540726095438, -0.2233646661043167, 0.6059102416038513, -1.5177552700042725, 0.05641757324337959, 0.05087682977318764, 0.310750812292099, 0.4098590016365051, -0.34539467096328735, -0.31543540954589844, -0.4806232750415802, 0.23697951436042786, 0.2711073160171509, 1.014174461364746, -0.10962025821208954, -1.0213061571121216, -1.387874960899353, 1.6613355875015259, -0.3911929726600647, 0.6600176692008972, -0.6482051610946655, -1.8577073812484741, 0.12554991245269775, 0.6908998489379883, 0.5292940735816956, 0.3401035964488983, 0.1656050980091095, -0.28033146262168884, 0.6691247224807739, 1.444040060043335, 0.11867856234312057, -0.7200785875320435, 0.2771511971950531, -0.9749341607093811, 0.9300834536552429, -0.060688622295856476, -0.27880406379699707, -1.7973397970199585, 0.43667542934417725, 0.22455117106437683, 0.2566261887550354, 0.5986600518226624, 2.130565881729126, 0.3112657070159912, -0.5724239349365234, 0.15333901345729828, -0.24115808308124542], [-0.6553853750228882, 0.8526737093925476, 0.9011513590812683, -1.1867854595184326, 0.6904293298721313, 0.30740052461624146, 0.21493729948997498, 1.115859031677246, 1.62064790725708, -0.18648934364318848, 0.5659007430076599, -0.18119965493679047, -0.9544606804847717, -1.7779500484466553, 0.07363928109407425, -0.7437891364097595, 0.25665199756622314, -1.2058923244476318, 1.5068224668502808, -1.5236077308654785, -0.11482266336679459, 0.9543026685714722, -0.09784951061010361, 0.23049868643283844, 1.6896998882293701, 0.8289289474487305, 0.4195699095726013, 0.4845546782016754, -1.1111294031143188, -1.0585308074951172, -0.33271950483322144, 0.14934135973453522, 0.7850344181060791, -0.12167306244373322, -0.09643246978521347, 0.5245782136917114, -0.5008782148361206, 0.21276219189167023, -0.3161049783229828, 0.5305953025817871, 0.6243394017219543, 0.5556660890579224, 0.3575417995452881, 0.3753806948661804, 0.15213488042354584, 0.34749531745910645, 1.4057552814483643, 0.23920372128486633, -1.9667892456054688, -0.3123330771923065, -0.22747176885604858, -0.6559028029441833, 0.1565302312374115, -0.6478710770606995, 0.28226637840270996, 0.4873761832714081, -0.35982179641723633, -1.5047448873519897, 0.6956967711448669, -0.807908833026886, 1.2059004306793213, 0.787411630153656, -1.6863007545471191, -0.5009720325469971], [0.9937117695808411, -0.49633434414863586, -1.142655372619629, 0.4807727634906769, -0.0561424158513546, 0.8335064649581909, -0.5840147137641907, -0.12423761188983917, 0.08380278199911118, -0.36643773317337036, -0.5460560917854309, -0.8776398301124573, 0.5765870809555054, 1.1592389345169067, -1.4404375553131104, 1.0491777658462524, -0.011619245633482933, 0.8887360095977783, -1.9495067596435547, 1.3255338668823242, -0.6060507893562317, 0.06357940286397934, -1.1004329919815063, -1.9437601566314697, -0.5397481322288513, -0.8132039904594421, -0.22779394686222076, -0.22780458629131317, -0.1705024540424347, -0.3280301094055176, -0.20672118663787842, 0.4749271273612976, -0.17484869062900543, 1.854705572128296, 0.14849258959293365, 0.641658365726471, -1.0779651403427124, 1.9290893077850342, 2.1412975788116455, -0.1845126748085022, 0.05894226208329201, -0.10036469995975494, 0.1337955743074417, -1.6172319650650024, -1.161933422088623, -0.30644482374191284, 0.03786376491189003, -1.0426621437072754, -0.47830793261528015, 0.5614310503005981, -0.1367979794740677, -0.7875769734382629, -0.3082597255706787, 0.19678367674350739, -0.14873623847961426, 0.09224635362625122, 0.8692910075187683, 1.762662410736084, 0.2521486282348633, 0.35493844747543335, -1.1938586235046387, 0.9984375834465027, -0.2686994969844818, 1.540804147720337], [-0.35712355375289917, -0.9446033239364624, 1.3386074304580688, 1.71930992603302, -0.9540041089057922, 0.00475058751180768, -1.0142422914505005, -0.06988602131605148, 0.5329112410545349, -2.495614528656006, 1.3979274034500122, 0.6104524731636047, 1.6042848825454712, 1.0678011178970337, 1.1854918003082275, -1.4448111057281494, -0.3628641664981842, -0.3220941126346588, 0.8594824075698853, 0.7992982864379883, -0.8313082456588745, 0.1186247318983078, -1.0752159357070923, -0.3197402358055115, -0.27388960123062134, 1.5330572128295898, 0.2310103178024292, 0.15793070197105408, 0.52901691198349, 1.1069551706314087, 0.05241881683468819, -0.21331529319286346, 1.82068932056427, -0.9741660356521606, 0.39948365092277527, -0.3950882852077484, -0.5272161364555359, 1.837975025177002, -1.2111917734146118, 0.03156999498605728, 0.18875202536582947, -0.3254997730255127, 0.08396870642900467, -1.906883955001831, -0.49142828583717346, 0.11833801865577698, 0.5102258920669556, 0.3135835528373718, 0.13842448592185974, -0.44485238194465637, -0.4811500608921051, 1.088511347770691, 0.2812071144580841, -0.3545355200767517, -0.5351457595825195, -0.0005728742689825594, 1.0457828044891357, 0.30439823865890503, 1.202344298362732, -0.7865716814994812, 2.415534734725952, 0.1492200493812561, -0.09145090728998184, 0.5781787037849426], [1.1640278100967407, 1.4333397150039673, 1.7224528789520264, 1.363021731376648, -1.0450726747512817, -1.4975736141204834, -0.9225651025772095, 0.4285835921764374, -2.5342631340026855, -0.6093129515647888, -0.9025808572769165, 0.055202044546604156, 0.6030030250549316, 1.4338481426239014, -0.3944622576236725, -0.45173829793930054, 1.9729464054107666, 1.684586763381958, 0.6486930847167969, 1.5331745147705078, 0.28625547885894775, -0.7924112677574158, 2.122262477874756, -0.3996427655220032, 1.2720801830291748, -1.4144924879074097, 1.505611538887024, 1.4195637702941895, -0.9697592258453369, -0.5977935791015625, 1.6067748069763184, 0.7421759366989136, 0.37126854062080383, 1.1965283155441284, -0.2016529142856598, 0.5944405198097229, -0.43966013193130493, -0.19008196890354156, -0.26027143001556396, 0.2775547206401825, 1.6718111038208008, 0.06150706112384796, -0.7034528851509094, 1.1082345247268677, -0.04744469374418259, 1.3759832382202148, 0.7135829329490662, -1.2060643434524536, -0.9236229658126831, 0.6998878717422485, 1.8070073127746582, 0.2810148596763611, -1.4294501543045044, 0.3477447032928467, -0.8128842115402222, 0.39101725816726685, -0.2392355352640152, 1.2383817434310913, 0.006923465058207512, -0.35535892844200134, -0.32427090406417847, 1.197838306427002, -0.5977205038070679, -2.771848678588867], [-1.225097417831421, -0.04323214292526245, -0.4312147796154022, 0.10415900498628616, 0.2649707794189453, -0.2279197722673416, -1.6484336853027344, -2.029092788696289, 1.189959168434143, -0.1532949060201645, -0.34361374378204346, 0.20468471944332123, -1.4366881847381592, -0.2748855948448181, -0.2739732265472412, -0.14202003180980682, -0.4030420482158661, 0.7688735127449036, -1.94642972946167, -0.28880149126052856, -0.5928505063056946, -1.0081007480621338, 1.6015723943710327, -0.36012837290763855, -1.110023856163025, 0.525644063949585, 0.9896470308303833, 0.531349778175354, -0.215650737285614, -1.8838671445846558, -0.9550799131393433, 0.9371967315673828, 0.3118562400341034, 0.06937603652477264, -0.1709204614162445, -1.697632074356079, -2.221050262451172, 1.1425365209579468, 1.562419056892395, 0.6527629494667053, 0.6372691988945007, 1.318422794342041, 0.19626227021217346, 0.5304774641990662, -1.9908103942871094, 0.2755172550678253, 0.03172630816698074, 0.5171387791633606, -0.3159613311290741, -0.6402091383934021, -0.9690523743629456, -0.038370877504348755, 0.057308126240968704, 0.5358160734176636, -0.20387573540210724, -0.7102605104446411, -1.4116533994674683, -1.7586904764175415, 0.41945263743400574, -0.4576856195926666, 0.9504192471504211, 1.172309398651123, -1.8021929264068604, -0.19764269888401031], [-0.4045209288597107, 1.1731683015823364, 0.38463273644447327, 0.4911886155605316, 0.3777289390563965, 0.49022239446640015, -0.9431276917457581, -1.5949772596359253, -0.5730280876159668, 0.6156550645828247, 0.09714676439762115, 0.6366235017776489, -0.14441201090812683, 2.340796709060669, 0.7152099609375, 0.1829172819852829, 0.4600435793399811, 0.6721461415290833, 0.5133237242698669, 1.195992350578308, -0.7576858997344971, 0.04158145934343338, -0.1605835258960724, 0.4731622338294983, -0.46726155281066895, -0.6708831191062927, 0.9400491714477539, 1.4298031330108643, -0.5797534584999084, 0.8293490409851074, -0.9390065670013428, 0.9780390858650208, -0.12866699695587158, 0.4799143970012665, -0.32561013102531433, -0.01883811317384243, -0.8864442110061646, -0.1635420322418213, 0.5973618030548096, 1.568601131439209, -0.01241292618215084, 0.08814190328121185, -0.9998793601989746, -0.9677384495735168, -1.1585677862167358, 0.6423378586769104, -0.6890191435813904, 0.6757261157035828, -0.21431505680084229, -2.605769395828247, -0.19200986623764038, -0.1496196836233139, 1.5413768291473389, -0.8848810791969299, 1.3273552656173706, -1.0891008377075195, 1.860353946685791, 1.8237062692642212, 0.7921784520149231, -0.0007278291741386056, 0.6855718493461609, 0.4223523736000061, 0.33120784163475037, 0.826805591583252], [0.23546814918518066, -0.6691028475761414, -1.8804048299789429, -0.2354421764612198, 0.37995821237564087, -0.5844995379447937, 0.0560426265001297, 0.029081257060170174, 0.368677020072937, -0.15286260843276978, 0.8632901310920715, -1.2982664108276367, 0.2610595226287842, -0.19727864861488342, -1.1875053644180298, 0.5641699433326721, -0.18585628271102905, 1.3530458211898804, -0.18859852850437164, 0.3885781466960907, 0.3532276451587677, 0.7859076261520386, -1.117024540901184, 1.0130261182785034, -0.6757269501686096, 0.7298203110694885, 0.7019729614257812, 1.4976502656936646, -1.907335638999939, -1.3063480854034424, -0.6525055766105652, -0.593484103679657, -0.6083112359046936, -0.1467732936143875, -0.5466055274009705, -1.0612685680389404, -0.06818519532680511, 0.7015436887741089, -0.5773703455924988, 0.897179126739502, -1.3912668228149414, -1.5137923955917358, -1.6631288528442383, 0.00045628921361640096, -0.35455846786499023, 0.20115777850151062, -0.06655996292829514, 0.5687574744224548, -0.44037890434265137, 0.19661618769168854, -1.299174189567566, 0.025393353775143623, 1.210957407951355, -0.12520916759967804, 1.9770060777664185, -0.640031635761261, 0.3293973207473755, 0.3437284231185913, 0.34134441614151, -0.030988972634077072, 1.8480921983718872, -0.29227885603904724, -0.2010420262813568, -0.7099612951278687], [-2.114354133605957, 1.4245485067367554, -1.7657749652862549, 1.2196742296218872, -0.03599568456411362, -0.9324083924293518, -0.328369677066803, 0.3926602303981781, -2.3445210456848145, -0.181029811501503, -0.5293352603912354, -2.0141139030456543, -0.7585487961769104, -0.6839185357093811, -0.11070261895656586, -0.41004136204719543, -1.0563507080078125, -0.8527154922485352, 0.5359101891517639, -0.15455874800682068, -0.43613889813423157, -2.2561306953430176, 2.2763710021972656, -0.3634704649448395, 0.8781399726867676, -0.9761868715286255, -0.28158965706825256, -1.5715620517730713, -0.7129312753677368, 1.538260817527771, -0.11166931688785553, -0.9703197479248047, -0.9598595499992371, 0.41994941234588623, -0.23029881715774536, -0.0034007569774985313, 1.211879849433899, 0.21441590785980225, -0.48784592747688293, 0.11449837684631348, -0.49969053268432617, 0.8430293798446655, -1.789724588394165, -0.03109235316514969, 1.0198930501937866, -0.023163586854934692, 0.517049252986908, 0.14693590998649597, 0.9267865419387817, 0.7102735638618469, -0.7748032212257385, 0.9145622849464417, -0.5925107598304749, -0.8039068579673767, 1.463983178138733, -0.2869631350040436, -1.034232497215271, 1.195945382118225, 0.44503360986709595, -0.30941784381866455, 0.695819079875946, -1.1278619766235352, -0.03793719783425331, 0.28648000955581665], [-0.694335401058197, 0.2708745300769806, 0.6017884612083435, -0.31437215209007263, -1.7353860139846802, -0.3006954789161682, -3.286555767059326, 0.15418687462806702, -1.270348310470581, 1.2808010578155518, -0.6536536812782288, 1.390817642211914, -0.8334758281707764, 0.1292978823184967, -0.17469969391822815, -0.6113080978393555, 1.6382941007614136, 0.3889405429363251, 0.7035180926322937, 0.2494548112154007, 1.3593177795410156, 0.4890182912349701, -0.15787972509860992, 1.2104730606079102, -1.2642709016799927, 1.364479422569275, 0.8589743971824646, 0.6075117588043213, -0.95298171043396, -0.6084195375442505, -1.443583369255066, -0.3856794238090515, 0.9488660097122192, 0.8860182762145996, 1.2839429378509521, 0.4738115668296814, 1.1283453702926636, 0.8272351622581482, 0.9214051961898804, -0.5226694941520691, -0.615855872631073, 0.12484428286552429, 0.4649684429168701, 0.09277236461639404, -1.7415599822998047, 0.010544652119278908, 0.7774527668952942, -1.6094465255737305, -0.5603099465370178, 0.5592712163925171, -0.5628799796104431, -1.3267204761505127, 0.05212375894188881, -0.7146314382553101, 0.17592382431030273, 0.5968611836433411, 0.6914278864860535, 0.2395123690366745, 0.10704159736633301, 0.03052780218422413, -0.012563493102788925, 0.3772279620170593, -0.8330771923065186, 0.04486097767949104], [0.3057935833930969, -0.031423911452293396, -0.9121437668800354, -0.5914247632026672, -0.42220795154571533, -0.9593833088874817, -1.2113096714019775, -1.0218794345855713, 0.6997841000556946, -0.031206389889121056, 0.6586030125617981, -0.41861531138420105, -0.5205111503601074, 0.8552232384681702, 0.05307293310761452, -1.9809281826019287, -2.6694486141204834, 0.9657424688339233, 0.9175084233283997, -0.3332739770412445, 0.3607422113418579, -0.41863542795181274, 0.5667867064476013, 0.31658831238746643, -2.5145909786224365, 0.18009983003139496, -1.3226935863494873, -0.6695396900177002, -1.7117918729782104, 0.22776883840560913, 0.4289364814758301, 0.31266993284225464, -0.47449856996536255, 1.135056495666504, -0.029540643095970154, -0.06313318014144897, -0.20426546037197113, 1.1736005544662476, -0.37636658549308777, -0.20547831058502197, 1.1728053092956543, -0.6072061061859131, 0.6201236844062805, 0.004818089306354523, 0.2074321061372757, 1.2389850616455078, -0.05628642812371254, 0.8536789417266846, -0.29631155729293823, 1.5587836503982544, -2.059030055999756, 1.0383191108703613, -0.17083382606506348, 0.05509711429476738, -1.1307874917984009, 1.023918867111206, 0.8115106225013733, -0.8051555156707764, 0.0549473762512207, 1.367542028427124, -0.3194132149219513, 0.06582632660865784, 0.3401210308074951, -0.7529522776603699], [2.812478542327881, 0.5202351212501526, 0.2696084976196289, -1.3853856325149536, 0.6766464710235596, 0.5570695400238037, -0.8864470720291138, -0.15345622599124908, -1.1449167728424072, -0.3550422787666321, 1.102678656578064, 1.3575170040130615, 0.7369444370269775, -0.359192818403244, -1.2923157215118408, 1.3119707107543945, 1.4151495695114136, -0.06433514505624771, -1.155808687210083, -1.2742178440093994, 0.6085226535797119, -0.42831212282180786, 1.8294169902801514, 0.39481988549232483, -0.1132517158985138, 0.011152820661664009, 1.2962838411331177, -0.39189809560775757, -0.5781236886978149, -0.07315269857645035, -1.070929765701294, 1.6150248050689697, 0.9051176905632019, -0.40182673931121826, 0.609072744846344, 0.6524778604507446, -0.25991255044937134, -0.2809373736381531, -0.7973289489746094, -0.19094617664813995, 0.9638893604278564, -1.8998006582260132, -0.2988496422767639, -0.7621297240257263, 0.03943505883216858, -0.4972309470176697, 1.7983839511871338, -1.6666070222854614, -0.6156218647956848, 0.35267651081085205, -0.278408020734787, 1.0370500087738037, -0.4463721215724945, 2.3944075107574463, 0.5531448721885681, -0.5091974139213562, -1.420850157737732, 0.3547093868255615, -0.6548188924789429, 0.7325889468193054, -0.030575856566429138, 0.6795807480812073, 0.2710520923137665, 1.586755394935608], [0.009671643376350403, -1.1463433504104614, 0.026887929067015648, -0.15123596787452698, -0.4990929365158081, -1.6321682929992676, -0.8436408638954163, -0.26490187644958496, 0.6343403458595276, 0.016299717128276825, 0.018772495910525322, -0.5275751352310181, 0.48483097553253174, -1.5617481470108032, -0.8433922529220581, 0.5463519096374512, -0.595334529876709, -0.7548753619194031, 1.065863847732544, -0.3137848377227783, -0.6215137243270874, -1.537959337234497, -0.18947328627109528, 1.8404654264450073, -0.30457961559295654, -1.3999351263046265, -1.9569145441055298, -0.07937853038311005, -0.2642848491668701, 1.909087896347046, 1.210142731666565, 0.10938306897878647, -1.0814602375030518, 1.2752225399017334, 0.8832514882087708, 1.0089435577392578, -0.0634487047791481, -0.3677052855491638, 0.6639603972434998, 1.2661370038986206, -1.6424554586410522, -0.5228363275527954, 0.1929176151752472, -1.8591991662979126, 1.1492187976837158, -1.767238736152649, 0.6005738973617554, -0.9091378450393677, 1.3392776250839233, -0.31847503781318665, -1.6187233924865723, -0.9121322631835938, 0.1745920330286026, 0.742414116859436, -0.12099073082208633, 0.964061439037323, 2.4707229137420654, -1.0227323770523071, -1.657125473022461, 0.6585139036178589, 0.906424343585968, 0.07602623105049133, 1.548866868019104, 0.9457917213439941], [0.5704991221427917, -0.18996864557266235, 0.8552315831184387, 0.4714415371417999, 0.16592665016651154, 0.605808675289154, 0.477193683385849, -0.5837549567222595, 0.3517487049102783, -0.50529944896698, 1.6096971035003662, -2.3161063194274902, -0.4748370051383972, -0.040054235607385635, 0.29867470264434814, -1.3115566968917847, 0.6104205250740051, -0.8376419544219971, 0.9670650959014893, -0.5946672558784485, -0.6064409017562866, 0.9812547564506531, -1.0381759405136108, 0.3126690983772278, 1.7840938568115234, 0.3584797978401184, -1.4295364618301392, -0.7143595814704895, 0.5692077875137329, 0.35955631732940674, 0.6937893033027649, -0.30058610439300537, 1.3877090215682983, -2.6802515983581543, -0.3579384386539459, -0.7887744903564453, -0.3659845292568207, -0.40126845240592957, 0.4841005802154541, -0.361271470785141, -0.7165005207061768, -0.25161996483802795, 1.1365960836410522, 1.421599268913269, 0.8141114115715027, 0.20919828116893768, -0.6155405044555664, -0.13395345211029053, -0.27781277894973755, 1.6184568405151367, 0.3009362518787384, -0.14745588600635529, 0.8603841662406921, -0.9764314889907837, 0.47725895047187805, -0.08280736207962036, -0.21613536775112152, 0.21640919148921967, -0.6174361109733582, 0.5501542687416077, -2.013221502304077, -1.5376851558685303, -0.9794071316719055, -1.7726587057113647], [-1.4061251878738403, 0.43874499201774597, 0.39981475472450256, 0.08209194988012314, -0.35382217168807983, -1.9862213134765625, 0.5614238381385803, -2.068284511566162, -1.2488399744033813, -0.7486199140548706, 2.366945505142212, -0.9422464966773987, 1.4349156618118286, 0.24089740216732025, 0.9412180185317993, 0.9226634502410889, -0.9702388644218445, 0.2193356454372406, 1.2769776582717896, 0.16727478802204132, 0.39575985074043274, 2.5677425861358643, -0.03484829142689705, -0.24001175165176392, -1.7733385562896729, -0.9885466694831848, 0.48061737418174744, -0.32050690054893494, 0.6948862075805664, 2.143648862838745, 0.027663836255669594, -0.8466513156890869, -0.2912808954715729, 0.24729523062705994, -0.5529582500457764, -1.277687430381775, -1.0229065418243408, 1.0007834434509277, -0.8320978283882141, -0.7700503468513489, 0.5536217093467712, 0.011931388638913631, 0.39979615807533264, -0.13261988759040833, -0.7313324213027954, -0.05535748600959778, -1.6982964277267456, -0.048107318580150604, 0.24507422745227814, -1.0993090867996216, 0.13546161353588104, 0.45145198702812195, 0.9601648449897766, 1.1696733236312866, -1.3966819047927856, 0.01611972413957119, -1.091919183731079, -0.3302343189716339, 0.4037744700908661, -1.6365132331848145, -0.7280277609825134, -1.1895228624343872, -0.9893776178359985, -2.035601854324341], [-0.4330752193927765, 0.31403377652168274, 1.1865220069885254, 0.17321205139160156, 0.6718882322311401, 0.7178143858909607, -0.36969414353370667, 0.9136496782302856, -1.2354422807693481, -0.5245221853256226, -0.15210413932800293, -0.2768186926841736, -0.36825674772262573, -1.0505671501159668, -0.11025442183017731, 0.947433590888977, -0.4569377303123474, 0.9995116591453552, -0.1354006975889206, -0.9993545413017273, 2.61273455619812, -0.7610471248626709, -0.3767430782318115, 0.759906530380249, 0.24553564190864563, 1.8307627439498901, -2.1926729679107666, 0.6052637100219727, 0.21467003226280212, 0.7835718393325806, -0.9366579651832581, -1.1007682085037231, -0.5029577612876892, 0.8572676181793213, -1.0478945970535278, -0.9731656312942505, -2.190840005874634, -0.46222686767578125, -1.510093331336975, 1.0074063539505005, -0.8698341846466064, 0.2863873541355133, 0.9224334359169006, -0.058436449617147446, 0.49064213037490845, 1.199881672859192, -0.3181231915950775, 1.001668095588684, -0.25872474908828735, 1.834003210067749, 1.6351802349090576, 0.6408969163894653, -0.3440109193325043, 1.0041431188583374, -1.9787100553512573, -0.0485706552863121, 0.7655707001686096, -0.061119481921195984, -1.8752506971359253, 0.05720163881778717, 1.19332754611969, 1.857283115386963, 1.1925193071365356, -0.2770738899707794], [2.194916248321533, -0.19499340653419495, -1.5795867443084717, -0.5907170176506042, 0.23173275589942932, -1.619688630104065, 0.4082396924495697, -1.0549969673156738, 0.3748120665550232, -0.9136952757835388, 0.11315250396728516, 2.0947277545928955, -1.6259901523590088, 0.6802206635475159, 0.680888295173645, -1.737826943397522, -1.0428706407546997, -0.6021875739097595, 0.19987013936042786, -1.1750845909118652, -1.0171115398406982, -1.033475637435913, 0.09156741946935654, -0.2754581868648529, -0.36519837379455566, 0.42732563614845276, 0.7682467103004456, -0.6713210344314575, -0.8947482109069824, 0.6705076098442078, -0.07654564827680588, -1.4615446329116821, -0.0687846690416336, 0.18012486398220062, -0.16504305601119995, -0.21406877040863037, -0.9910244941711426, 2.0212342739105225, 0.9157442450523376, -0.11928858608007431, -0.29974889755249023, 0.6435623168945312, -1.1854020357131958, -1.8596744537353516, -0.3084922730922699, -0.17858768999576569, -0.2322978377342224, -0.4053100049495697, -0.7748779058456421, 2.4823570251464844, 0.5107043385505676, 0.09257439523935318, -0.29254356026649475, -0.2427625060081482, -0.3893629014492035, -0.5968847274780273, 1.2036241292953491, 1.9200141429901123, -2.296938896179199, 0.7160124778747559, 0.01783853769302368, 0.881818950176239, 1.0976802110671997, -0.4515179991722107], [0.1857612282037735, -0.3862135112285614, 0.6597587466239929, 0.8230938911437988, 0.6297698020935059, 0.35614824295043945, 2.1846139430999756, -1.1652402877807617, -1.4751697778701782, -0.7296411991119385, 0.10498572140932083, -0.02081216871738434, -0.3909232020378113, 0.060013677924871445, 0.7690718770027161, -0.7350765466690063, -0.2215392142534256, 1.017642855644226, -0.13999326527118683, 1.0473477840423584, 0.24311526119709015, 0.668587863445282, 0.8265634179115295, 1.20854914188385, -0.21452994644641876, -0.3642592132091522, -0.482203871011734, -0.9384207129478455, -1.724065899848938, 1.2991399765014648, -0.7865653038024902, 1.196771502494812, 0.0037260879762470722, 0.22865012288093567, 1.365554928779602, -0.7650689482688904, -1.8156613111495972, -0.7315669655799866, -0.18334823846817017, -1.1417707204818726, -1.049996256828308, -0.25726979970932007, 1.2684556245803833, 0.927143931388855, 0.8700127601623535, -0.6251852512359619, 1.1282532215118408, 1.2560254335403442, -0.10860490053892136, 0.11371131241321564, 0.20297138392925262, -1.7609035968780518, 0.11396408081054688, 1.8938835859298706, 0.504315972328186, -0.06044844537973404, -1.7470612525939941, -0.6588571667671204, 0.4244038462638855, -0.881231427192688, -0.45912879705429077, 1.4454162120819092, -0.28815126419067383, -0.7597249150276184], [-0.28531894087791443, -0.17943990230560303, -1.587040901184082, -0.22659888863563538, -0.30480268597602844, -0.5275775194168091, 1.7484328746795654, 0.42929208278656006, -0.7547057867050171, 0.6731123328208923, 0.363538920879364, -0.48998039960861206, -0.028603265061974525, 1.1908655166625977, 0.13871708512306213, 0.9810465574264526, 1.4268081188201904, -0.6450423002243042, 1.1417573690414429, 2.482276201248169, -0.5093207359313965, 0.3666875660419464, 0.93764328956604, 0.008882617577910423, -0.07787469029426575, -0.16557073593139648, -1.6992071866989136, 0.05764508247375488, -0.4645750820636749, 1.4272490739822388, -1.534604549407959, -0.4289844334125519, 0.47607100009918213, 0.660849392414093, 0.818784773349762, -0.584809422492981, 0.48677995800971985, 0.3588956892490387, 0.7385509014129639, 1.52059805393219, 1.0905400514602661, 1.4261903762817383, 0.16714216768741608, 1.636650800704956, -1.4067356586456299, 0.44308531284332275, 0.2636656165122986, -0.989626407623291, 0.22705593705177307, -0.23702076077461243, 0.10627385228872299, 0.31443411111831665, 1.296971082687378, 3.250840902328491, 0.48607417941093445, 0.036710724234580994, -1.1345371007919312, -1.0701977014541626, 1.3446147441864014, -0.9001328945159912, -0.8418806195259094, -0.6055451035499573, 1.1870317459106445, 0.10118643194437027], [1.333789348602295, -0.46083542704582214, 0.6491071581840515, -0.36406564712524414, -0.2155199944972992, 0.6755306124687195, -0.577116847038269, -0.863098680973053, 0.6067257523536682, -0.6695553064346313, -0.033352818340063095, 0.615131676197052, -0.7931092381477356, 0.27767184376716614, -0.21557044982910156, -0.763786792755127, -0.10749562829732895, 0.16171292960643768, -0.301193505525589, 0.7410476207733154, -1.322141170501709, -0.2756645083427429, -0.3361246883869171, 0.6515363454818726, -2.21213960647583, -0.7642084956169128, 0.009988270699977875, 0.02611844427883625, 0.05926542356610298, 0.39280933141708374, -1.4672880172729492, -0.5732369422912598, 0.07172440737485886, -0.3216763436794281, -1.371913194656372, -1.8515883684158325, 1.4569692611694336, 1.4926059246063232, -0.2960165739059448, -1.318002462387085, -0.34662726521492004, 0.6831636428833008, -0.11460544914007187, 0.07401274889707565, -0.4244282841682434, -1.56867253780365, -0.626903235912323, 2.540912389755249, 1.2143021821975708, -0.29383546113967896, -1.205275535583496, -0.2605924904346466, -0.574040412902832, -0.42310062050819397, -2.7570695877075195, 0.8313217759132385, 0.24993790686130524, -0.07402992248535156, 0.5515428781509399, -0.4397338032722473, -0.22432966530323029, -0.599966824054718, -1.4303566217422485, 0.30527353286743164], [1.7006926536560059, -0.4817076027393341, -0.14248354732990265, -0.927179753780365, 0.1559848189353943, -0.22212858498096466, 1.2160929441452026, -0.2651337683200836, 0.41153448820114136, 1.9548685550689697, -0.7595855593681335, -1.16947340965271, -0.6858691573143005, -0.3043728768825531, -0.7996760010719299, -0.32956188917160034, -1.1186506748199463, 3.3736422061920166, 1.492620587348938, 1.6779299974441528, -0.6915075778961182, 0.20447446405887604, -0.9201922416687012, -1.0539186000823975, 0.924462080001831, 1.5454314947128296, -0.45775070786476135, 0.287249892950058, 1.2542498111724854, -1.3790401220321655, 0.12248099595308304, 0.10631343722343445, 1.374330997467041, 0.16543282568454742, -0.740095317363739, 1.3741377592086792, 0.08508632332086563, -0.4347243309020996, -1.3678547143936157, -0.5699360370635986, 0.7769109010696411, -1.7465927600860596, 0.7648698687553406, 0.09960329532623291, -1.625293493270874, 0.38213595747947693, 0.07796573638916016, -0.22872702777385712, 0.45621687173843384, 1.8696391582489014, 1.7032970190048218, 0.2996743321418762, 1.496513843536377, -0.38502830266952515, -0.24540528655052185, 1.915536880493164, -0.29431524872779846, -0.6885637640953064, -0.3283722698688507, 1.424919843673706, 0.7122549414634705, -1.164109230041504, -0.9847787022590637, -0.8844150304794312], [-1.6121479272842407, -0.7442362904548645, 2.927276372909546, 0.9581193327903748, 0.178805872797966, 1.3583674430847168, 0.34677472710609436, -0.3485746383666992, 1.7402276992797852, 1.3878811597824097, 1.1622722148895264, -0.4886116683483124, -0.21502366662025452, -0.49269193410873413, -0.4417455792427063, 1.4587117433547974, 0.016937987878918648, -0.9968662858009338, -0.17617911100387573, -2.0772218704223633, -0.4572872221469879, 0.3157520592212677, -0.541633665561676, 1.6391711235046387, 0.14207732677459717, -0.5833730101585388, 1.880775809288025, -1.7418936491012573, 1.268803596496582, 0.4026428461074829, -0.9817853569984436, -0.9977779984474182, -0.16064289212226868, 0.08706425130367279, 0.3099650740623474, -0.7477049231529236, 0.6316469311714172, -0.9351904988288879, 1.4362947940826416, 0.8791555166244507, -0.7974370718002319, -0.1193387359380722, -0.5429694652557373, 0.6902318596839905, 2.1792666912078857, 0.5541625618934631, 0.1245468333363533, -0.623113214969635, -0.5091211199760437, -0.7468910217285156, 0.5837926268577576, 0.6835620999336243, 0.6767178773880005, -1.2845652103424072, 1.3794304132461548, -0.484579861164093, -0.11246991157531738, -0.4854869246482849, 1.7817330360412598, -1.6966309547424316, -0.12814852595329285, -0.08169499039649963, 0.12349741905927658, 0.1709885597229004], [-1.2806782722473145, 0.6349596977233887, 0.9062876105308533, 0.33135727047920227, 1.4937033653259277, -0.4587927460670471, -2.120149612426758, -2.363896369934082, 0.5432680249214172, 0.2665621340274811, -1.8851033449172974, -0.9544655680656433, 0.028740456327795982, -1.1154621839523315, 0.887837827205658, -0.9882137179374695, -0.9445995092391968, 0.34348541498184204, 2.0899415016174316, 1.116518497467041, -1.0489097833633423, 0.6708195209503174, -1.4701683521270752, 1.121234655380249, 0.3284724950790405, -0.8333228826522827, 2.068584680557251, -1.4047136306762695, 0.32174769043922424, 0.2800097167491913, 0.11732408404350281, -1.3419910669326782, 0.429577112197876, 1.7144752740859985, -0.243514746427536, 0.6269699931144714, -0.7293657660484314, 1.2253985404968262, -0.3704853057861328, 0.1266539990901947, -0.9974796175956726, -1.4831078052520752, -0.36342957615852356, 1.2693856954574585, -0.7211312055587769, 0.5858153700828552, 0.6565819382667542, 0.19829051196575165, -0.5657997727394104, 0.10682833939790726, -1.2641547918319702, -0.3556253910064697, -0.20101220905780792, 0.2257567197084427, -2.359656810760498, -1.6531697511672974, -0.9272220730781555, -0.016728971153497696, -0.4697168469429016, -1.690281629562378, -0.43949180841445923, -0.18052923679351807, 2.455455780029297, 0.10265818238258362], [0.7527258396148682, -0.7115988731384277, -0.5482953190803528, -0.6116594076156616, 0.6142382025718689, 1.124226689338684, -0.29034921526908875, -0.29993870854377747, -0.004501526243984699, 0.25019556283950806, 0.9238016605377197, -1.1437299251556396, -1.9501913785934448, 0.26693397760391235, -0.5658425688743591, -0.9474596381187439, 0.8435233235359192, -1.7335219383239746, -0.5348164439201355, 0.6807515621185303, 0.7901689410209656, -2.2115025520324707, -1.6160385608673096, 0.36778584122657776, 1.954168438911438, 0.6480708718299866, 0.9205150008201599, 0.1289007067680359, -0.310920387506485, -0.4548868238925934, 1.7406331300735474, 1.0809743404388428, -0.9609547853469849, 0.6615357995033264, -1.1292561292648315, -1.215653419494629, 1.15310537815094, -0.7913548946380615, 0.07964413613080978, -0.41810280084609985, 0.26724281907081604, 0.9904510378837585, 1.2669419050216675, -0.463979572057724, 0.02233009599149227, 0.7268169522285461, -1.8171989917755127, -1.7230114936828613, -0.23520825803279877, -0.914836049079895, 1.3323969841003418, 0.5526611804962158, -1.4204022884368896, 0.5873547792434692, -0.6655018329620361, -0.887858510017395, -0.5349639058113098, -1.9583940505981445, -0.171635240316391, -1.3304777145385742, 2.5795538425445557, 1.52060067653656, -1.9215060472488403, -0.6515074372291565], [-1.1418025493621826, 0.26559171080589294, -0.30641821026802063, 0.4768117368221283, -0.31262362003326416, 1.6113518476486206, 0.8668252229690552, -1.2984619140625, 1.9153960943222046, 0.3724477291107178, 0.24226044118404388, -1.1750166416168213, 0.2451600432395935, -0.8956335186958313, 0.504900336265564, -0.4451904296875, 0.2924235463142395, -0.020309656858444214, -0.9953441619873047, 0.8518733382225037, 0.06993638724088669, -1.488308310508728, -0.21092721819877625, 0.801287055015564, 0.25960999727249146, 0.10363319516181946, -0.2851722538471222, -0.5421108603477478, -2.072632312774658, 0.6807738542556763, 0.5931602716445923, -0.697238564491272, 0.6558890342712402, 1.3718516826629639, 1.1402134895324707, 0.2781292200088501, 0.09485091269016266, -0.5598329305648804, -1.2229098081588745, 0.03573547676205635, -1.5906846523284912, -0.9754953980445862, 0.10498417913913727, -1.3389281034469604, 0.9867944121360779, 1.066206932067871, -0.0970437154173851, 0.32495948672294617, 0.6275771260261536, -0.6053770780563354, -1.3448998928070068, -0.94036865234375, 0.5197323560714722, -0.5468723177909851, -0.18029546737670898, -2.139312505722046, -0.9355034232139587, 0.3873807489871979, -0.650499701499939, -0.5386550426483154, -0.16387338936328888, -0.7055811285972595, 1.0768985748291016, -0.2028087079524994], [-0.31820744276046753, -0.2600364685058594, -2.972719192504883, -1.0588085651397705, 0.09289443492889404, -0.2466128021478653, -0.6416826844215393, -0.5906247496604919, 0.07758026570081711, 1.2799447774887085, -0.4292983114719391, 0.26900142431259155, -0.8712493777275085, -0.41374826431274414, 1.578142523765564, 1.3690638542175293, -0.9104034304618835, -1.0041472911834717, -0.05884455516934395, -1.156259298324585, -1.4816049337387085, -1.404342770576477, 0.63847416639328, 0.9693621397018433, 0.14587707817554474, 0.14743246138095856, -0.7351759076118469, 1.2335234880447388, -0.1424446403980255, -1.1165133714675903, 0.5142067670822144, -0.7997045516967773, -1.442653775215149, 1.3142143487930298, -0.00830436497926712, -0.2778030037879944, 1.8398715257644653, 0.747654914855957, -0.7331817746162415, 1.1777384281158447, -1.0566197633743286, -0.27688106894493103, -1.5997631549835205, -0.3050934970378876, -0.8495775461196899, 0.14203962683677673, -0.33461663126945496, 0.46927013993263245, 0.5543985962867737, -0.6867009401321411, 0.08801145851612091, 1.4853122234344482, 0.05441465601325035, 0.8602091670036316, 1.0916166305541992, -0.2180011123418808, 0.09409861266613007, 0.255130410194397, -0.6814447641372681, -0.48358842730522156, 0.18404634296894073, -1.1957945823669434, 0.9275817275047302, -1.8604379892349243], [-1.3178280591964722, 0.6818313598632812, -1.3837867975234985, 0.6513408422470093, 0.5731073617935181, 0.020717978477478027, 0.7847292423248291, 1.285041093826294, -0.3673504590988159, 0.04629998281598091, -0.14552198350429535, 0.2851836085319519, 0.7768879532814026, -0.5571449398994446, -0.18745966255664825, 0.1285199224948883, 0.7695252895355225, -1.1443902254104614, -0.2944449484348297, -0.3016059994697571, -0.4041818678379059, 1.3783624172210693, 0.2121637612581253, -0.5422893762588501, -0.283292680978775, -0.3321082890033722, 0.9451128840446472, 0.7710421681404114, 1.5108563899993896, -1.5191646814346313, -1.0395946502685547, 0.659474790096283, 0.17848846316337585, -0.047157157212495804, -0.8470613360404968, 0.7824799418449402, 0.7333558201789856, 1.9575555324554443, 0.13932517170906067, -1.5463602542877197, 0.7815611362457275, -0.1439875215291977, 1.1021158695220947, -0.6499108672142029, -0.2771163880825043, 0.9253861308097839, -0.0841502845287323, 0.1489851176738739, 0.47395360469818115, -0.8760083913803101, -0.41836345195770264, -2.555614709854126, 1.5128310918807983, -1.0384621620178223, -0.608201801776886, 0.12838372588157654, -0.9267642498016357, -1.3897638320922852, -0.5944355130195618, -0.5950033664703369, 0.8884126543998718, 0.611372172832489, -0.4650452435016632, -0.18675069510936737], [-0.6881751418113708, 1.1172925233840942, 0.6681807041168213, -1.5432785749435425, 0.0411786250770092, 0.6020125150680542, -2.249685525894165, -2.445531129837036, -1.544999361038208, -0.3333689272403717, -0.22877198457717896, 0.48228201270103455, 1.542744755744934, -0.4374653995037079, -1.0249215364456177, -2.6620287895202637, 0.7702707052230835, 0.5663360357284546, -1.3139119148254395, -0.036414846777915955, 1.0713870525360107, -0.9470731019973755, -1.102635383605957, -0.5826667547225952, -0.49047616124153137, -0.6214237213134766, -2.2044200897216797, 0.705253005027771, -0.6250771284103394, -0.3385871350765228, -0.4539594352245331, -0.1710796356201172, 2.0142805576324463, -0.5887793898582458, 0.9337418675422668, 0.14288072288036346, -0.721123456954956, 1.6843432188034058, 0.26949653029441833, -0.4949052631855011, 0.7878323197364807, -0.7782521843910217, 0.6248224377632141, 1.9872934818267822, -1.0393284559249878, -1.1283044815063477, 0.3646438717842102, -2.6697659492492676, -1.816325068473816, -0.6744338274002075, -0.44636139273643494, 1.5586923360824585, -1.0668994188308716, -0.30738890171051025, 0.5416746139526367, -0.4372081160545349, 1.0233575105667114, -0.7312731146812439, 0.9776827096939087, -0.632331907749176, -0.7836039066314697, -1.6979490518569946, -0.2596109211444855, 1.2832056283950806], [1.3233619928359985, -0.4193984866142273, 0.39044836163520813, 1.6552543640136719, 1.3187508583068848, 2.343613862991333, -0.4959360659122467, -0.5723777413368225, 1.0123088359832764, 1.844721794128418, 0.4454345703125, 1.517268419265747, 0.6179870367050171, 0.037525489926338196, -1.348490595817566, -1.065667748451233, 0.2812855541706085, 0.285048246383667, -0.3395618498325348, 1.2376371622085571, 0.9520067572593689, 0.21750932931900024, 1.5696035623550415, 1.1541485786437988, -0.10110928118228912, -0.23632977902889252, 0.3701053559780121, -0.12214644253253937, -0.777316153049469, 0.17332619428634644, 0.8876620531082153, 0.08502788096666336, 0.2747608423233032, -2.298208713531494, -0.21159455180168152, -2.0392038822174072, 0.9200454950332642, 0.47486618161201477, 0.42855027318000793, 0.48835358023643494, 1.265244483947754, 1.5407040119171143, 0.044900938868522644, 1.958385944366455, 0.058961693197488785, -0.11417413502931595, 1.009437084197998, -0.9693732261657715, 0.7265979647636414, -1.024914026260376, -0.3740406334400177, 0.406617134809494, 0.5252480506896973, -1.5036492347717285, 1.05613112449646, 0.4721768796443939, 1.7761954069137573, 1.6773663759231567, -1.0958119630813599, -0.9273660778999329, -0.5620460510253906, 0.23402796685695648, 0.36027175188064575, 0.878779411315918], [0.4727805554866791, 0.2485491782426834, 0.6840630769729614, -0.8543110489845276, 0.03932661935687065, 0.5831491947174072, -0.43185487389564514, -0.038963235914707184, 0.08725189417600632, -0.6460506916046143, -0.34060245752334595, 0.46719786524772644, 1.6715312004089355, 1.4081439971923828, -0.9945880770683289, 0.7020594477653503, 0.08079338818788528, -2.59688663482666, 2.129746437072754, -0.7471173405647278, -0.047671567648649216, -1.9501261711120605, -0.18597647547721863, -1.5523887872695923, 2.3922784328460693, 0.8339160680770874, -0.5804276466369629, 0.2980203330516815, 1.7709835767745972, -1.267855167388916, 0.10942599922418594, 0.2388746738433838, 1.6512227058410645, 0.4093249440193176, 1.7178120613098145, 1.0647633075714111, 0.6569443941116333, -1.036218523979187, 0.570448637008667, -1.5129425525665283, -1.2425538301467896, 0.9538930058479309, -0.006370824296027422, 1.103515625, -1.1418997049331665, 1.379150629043579, 1.3715839385986328, -2.299283027648926, 0.8948836922645569, 0.6126855611801147, 0.8332401514053345, 0.05762224644422531, -0.23105329275131226, 1.1707710027694702, -1.256394863128662, 1.7546992301940918, -0.688590407371521, -0.06325141340494156, -0.27823129296302795, -0.8448405861854553, 0.7667551636695862, 0.2627042531967163, 1.0566203594207764, 1.0840355157852173], [-0.8915767669677734, 0.1378214806318283, -1.5180628299713135, -0.6617674231529236, -0.03208640217781067, -0.012156425043940544, -0.19182154536247253, -0.3952183425426483, -1.1034224033355713, 1.8941417932510376, 0.09970035403966904, -0.3236735761165619, 0.32909920811653137, -1.3899534940719604, -0.4660996198654175, -1.1899919509887695, 0.26521435379981995, 0.21036548912525177, 2.095473051071167, -2.6238813400268555, 0.39087510108947754, 0.3656863570213318, -1.6059383153915405, 0.7629779577255249, 1.029727578163147, 0.9763370752334595, 0.678666353225708, -1.143763542175293, 0.5258530378341675, 0.25990527868270874, 1.5399430990219116, -1.0640314817428589, 0.013080479577183723, 0.6918691992759705, -0.33506035804748535, 0.6341555714607239, 0.3936628997325897, -1.0410031080245972, -0.13758544623851776, -0.24151989817619324, 0.7804882526397705, -1.814716100692749, 0.3965848684310913, 0.5324453711509705, -1.0133603811264038, -1.2344584465026855, 0.27221205830574036, -2.319319009780884, -0.035163458436727524, -0.05353982001543045, 1.2098037004470825, 0.258562296628952, 0.0010241649579256773, -2.0466089248657227, 0.5920157432556152, -0.1836411952972412, 1.7707045078277588, -0.2960452735424042, -0.3757339417934418, 1.8531438112258911, -0.6629223227500916, -0.0016564984107390046, 1.2077146768569946, 0.11806187778711319], [-0.7135109305381775, -1.7759504318237305, -1.0900720357894897, -0.947309672832489, 1.3835148811340332, 1.5522840023040771, -0.7651196122169495, -0.8183613419532776, -1.2177948951721191, -1.0331467390060425, 0.9002090692520142, -0.733400285243988, -0.4076724052429199, -0.7046347260475159, 0.8519219160079956, 0.4969276487827301, -0.8489736318588257, -0.11877920478582382, -0.46065255999565125, -0.08842344582080841, -0.5425618290901184, -1.2511177062988281, 0.9877372980117798, -1.2595049142837524, -0.34307968616485596, -1.231473684310913, 0.29535743594169617, 2.0653514862060547, 0.3754217326641083, -0.5434861183166504, 0.824499785900116, -0.2920129597187042, 0.6795979738235474, 0.9848615527153015, -0.6710972189903259, -1.5832186937332153, 1.4141552448272705, -0.06763054430484772, 1.5799546241760254, -0.20924484729766846, -0.0923633873462677, 1.1501562595367432, 0.15782374143600464, -0.9927724599838257, 0.16763614118099213, 0.3487069606781006, -0.19362179934978485, -0.5355775952339172, -0.09536315500736237, 1.879536509513855, 0.5495949983596802, -0.197383850812912, 2.520843267440796, -0.5897127985954285, -1.2683488130569458, 0.6556496620178223, 0.18099911510944366, 0.45015984773635864, 0.6553841829299927, 0.6866325736045837, -0.13643383979797363, 0.32620498538017273, -1.0920823812484741, -1.6606560945510864], [-0.2834744155406952, -0.9508237838745117, 0.4946855306625366, 0.22238142788410187, -2.7386913299560547, -1.2867166996002197, -0.7548660635948181, -1.4358255863189697, -0.07341734319925308, -0.6967341303825378, -0.40011152625083923, -0.7278527021408081, 0.39658159017562866, 1.1649155616760254, -0.9708896279335022, 0.5067465305328369, -0.6096097826957703, -1.6462018489837646, -1.5028448104858398, -0.3152175843715668, 0.19384074211120605, 1.7846029996871948, 0.5113069415092468, 0.3809066116809845, 0.05409625545144081, 0.34469908475875854, 0.49629440903663635, -0.6171481609344482, -1.5798029899597168, 1.9502711296081543, -0.786247730255127, 0.5122357606887817, -0.014844721183180809, -0.666674792766571, 1.143343210220337, 0.5291005373001099, 1.0565025806427002, -0.6263490915298462, -0.27700310945510864, -2.5702879428863525, -0.5184722542762756, -0.2686152160167694, 1.504812240600586, -1.8298981189727783, 1.4877249002456665, -1.8909856081008911, -0.4449457824230194, -0.8622367978096008, -1.601845622062683, 1.2946499586105347, 1.8890116214752197, 1.3353207111358643, 0.061589401215314865, -1.6120017766952515, -0.2470862865447998, -1.181762456893921, -0.5572174191474915, -0.013102279044687748, -0.12745927274227142, 0.8182551264762878, -0.11770640313625336, 0.4367236793041229, 0.8595306277275085, 0.12626494467258453], [1.1731775999069214, -1.507559895515442, -0.6896681785583496, 0.8714784383773804, 0.9665548205375671, -0.45436015725135803, 0.9376471638679504, 0.6161515712738037, 0.9790316224098206, -1.335434913635254, 1.1134973764419556, 0.6617677211761475, 0.09819269925355911, -0.18032829463481903, -0.11856624484062195, -0.8669160604476929, 0.3792983889579773, 2.03810715675354, 1.0954530239105225, 0.28115108609199524, 0.4335525631904602, 1.1227483749389648, 0.1582699865102768, 0.2596333622932434, 0.6972975730895996, 1.234682321548462, 0.15123462677001953, -0.7374371886253357, -0.5910061001777649, 0.20412828028202057, 0.7534468770027161, 0.6051570773124695, 1.5294299125671387, -0.8498379588127136, -0.7099607586860657, 0.8838075995445251, -0.7611480355262756, -0.07960423082113266, 0.8445754647254944, -0.44432273507118225, 1.5737318992614746, -0.5373428463935852, -0.3145081102848053, 0.5610978603363037, -0.5342152714729309, 0.8613865971565247, -1.2312313318252563, -1.0358775854110718, -1.9708629846572876, -0.6442975997924805, 0.5706313252449036, 0.8759346008300781, -0.45055481791496277, -1.4043164253234863, -2.124500274658203, 0.35259583592414856, 1.523533582687378, 1.0171325206756592, -0.282529354095459, 0.5974019169807434, -0.7096049785614014, 0.4509105980396271, 0.5547151565551758, -2.189392328262329], [0.6829369068145752, -1.494184136390686, -0.9493265151977539, -0.709219217300415, 2.328411817550659, -0.8951147198677063, -0.32954269647598267, -0.6025994420051575, -0.09674125164747238, 1.681182861328125, -1.7984719276428223, -1.3377876281738281, -0.8644938468933105, 0.9543678164482117, -1.0085240602493286, -0.7362019419670105, -0.535067081451416, 1.2198125123977661, 1.4761593341827393, 0.1161302775144577, -1.8067890405654907, -2.2056291103363037, -0.3454461395740509, 0.838901162147522, -0.8802570104598999, -1.990761160850525, 1.6475002765655518, -0.8631919026374817, -1.4431936740875244, -0.7131545543670654, -0.734893262386322, -1.3459746837615967, 0.42299923300743103, -0.8710649609565735, -0.4780098497867584, 0.21550211310386658, 2.4646759033203125, 0.13327939808368683, 0.9561752080917358, 1.232236385345459, 0.3494696021080017, 0.06684664636850357, 1.2919232845306396, -0.4361169636249542, -0.4184635579586029, -0.5153557658195496, -0.9737905859947205, -1.5511037111282349, -0.6961967945098877, -0.5720974206924438, 0.061992283910512924, 1.364279866218567, 0.22953158617019653, 1.7408230304718018, -0.32434573769569397, -0.46445223689079285, -0.0401310995221138, 0.7098400592803955, 0.5790281295776367, -1.3744693994522095, -0.4616483747959137, 0.44172796607017517, 2.0539252758026123, 0.6695953011512756], [0.6225023865699768, -0.3051341474056244, -1.1645125150680542, -0.20356474816799164, 0.5941098928451538, -0.7265791893005371, -0.6117268204689026, -0.16763627529144287, -1.6219953298568726, 1.4228129386901855, 1.1579034328460693, 0.942865788936615, -0.19761396944522858, -1.8343534469604492, -0.11521628499031067, -1.0535366535186768, 1.1793056726455688, 0.3028339743614197, -0.559928297996521, 0.13011345267295837, -1.1117804050445557, -0.48707541823387146, -1.0602946281433105, -1.0354377031326294, 0.7099044322967529, 1.2036163806915283, -0.03208357095718384, 0.6798743605613708, 0.016816454008221626, -1.8531091213226318, -1.9573272466659546, 0.0914505198597908, 0.15998852252960205, 0.3919753134250641, -0.00720512168481946, -0.6739152669906616, 1.024776577949524, 0.22168895602226257, 0.02862105704843998, -0.6299657821655273, -0.8790567517280579, -0.7428574562072754, -0.2916320264339447, -1.6652425527572632, -0.6761996746063232, 0.3662099242210388, -0.561553955078125, 0.6488873958587646, 0.71429443359375, 0.9352565407752991, 0.33492785692214966, -0.3202230930328369, -1.4826514720916748, 0.3539796769618988, -0.513150691986084, -0.5950542092323303, -0.05867309868335724, -0.6802484393119812, -0.44204962253570557, -0.7319909930229187, 0.03854724392294884, 1.5731276273727417, -1.0579748153686523, 0.47530874609947205], [2.2628700733184814, 1.1392829418182373, -1.660355806350708, -0.15959449112415314, -1.6433758735656738, 1.3385809659957886, -0.6594452857971191, -2.0349161624908447, 0.19730478525161743, 0.8167903423309326, -0.4738818109035492, 0.11088649928569794, -0.02384556457400322, -0.5891697406768799, -2.0230798721313477, 0.5807414054870605, 0.9292442798614502, -0.6380642652511597, -1.1913516521453857, -0.9627794027328491, -0.5327199697494507, -0.3150698244571686, 1.2582160234451294, 0.7092112302780151, 0.833352267742157, -1.414948582649231, -0.5337479710578918, 0.22642165422439575, -0.2300412356853485, 1.741900086402893, -0.49554145336151123, 0.3597513735294342, -0.5755173563957214, 0.5011212825775146, 0.0387817919254303, 0.1067853718996048, -0.6357887387275696, -1.0143120288848877, 0.7632156610488892, 1.2288159132003784, -2.6911826133728027, -0.02400163933634758, 0.8542010188102722, 0.30877891182899475, -1.2688082456588745, -0.9856457710266113, 2.504101037979126, 0.6102595329284668, -0.057873792946338654, 0.3664095103740692, -0.010535018518567085, -0.25277262926101685, -0.8395271897315979, 0.3298633396625519, -2.715789556503296, 1.4184296131134033, 1.9192978143692017, 0.7023897767066956, -2.2064034938812256, -0.29536670446395874, -0.7109730243682861, 0.06084803491830826, 0.5508511066436768, -0.17311200499534607], [0.10823100805282593, -2.1653473377227783, -0.6811245679855347, -1.5150361061096191, 0.011863704770803452, -0.6860175728797913, 0.01749165914952755, 1.0304861068725586, -0.5602473020553589, 0.15331952273845673, 0.8878548741340637, -0.4912562966346741, 2.3928847312927246, -0.03997372090816498, 0.22600802779197693, 1.795230507850647, -1.0621745586395264, 0.6043571829795837, 0.34671324491500854, 0.7519188523292542, 1.9101669788360596, -0.24102924764156342, -0.7348848581314087, 0.8586148619651794, 0.48305124044418335, -0.5920241475105286, -0.4334927201271057, -0.6335486769676208, -1.4378799200057983, -0.4551107883453369, 0.6807666420936584, 1.055066704750061, -0.8456655144691467, -0.25134268403053284, -1.0735468864440918, -1.1681054830551147, -0.2082061618566513, -0.6168900728225708, 1.2909578084945679, 0.8099771738052368, -0.9550518989562988, -0.3364720642566681, 0.7915408611297607, 0.47127479314804077, 0.38417813181877136, -0.0746726244688034, 0.13516677916049957, 0.48255714774131775, -0.6858285665512085, 0.2199764847755432, -0.8724800944328308, -0.7274122834205627, 0.34490731358528137, 0.378921777009964, -1.3611117601394653, 0.40508732199668884, -0.2972121238708496, 0.3593452572822571, 1.0993907451629639, -0.36860764026641846, 0.25913184881210327, -0.2690187692642212, -0.06281118094921112, -0.4916004538536072], [-0.06757351756095886, -0.4997321665287018, -0.7220731377601624, -2.260197877883911, -0.8190159797668457, -0.1779160350561142, -0.16122260689735413, 0.042412154376506805, -0.5235547423362732, 1.67823326587677, 1.2606099843978882, -0.03687596321105957, -0.14614006876945496, -1.4476535320281982, -0.45888784527778625, -0.7809423208236694, -0.438414067029953, -0.2350829690694809, 0.5070645809173584, -0.41588127613067627, 0.9178078174591064, -1.04261314868927, -0.929850697517395, -0.5178106427192688, 0.42640984058380127, 0.9700114130973816, 0.6189074516296387, 0.7788912653923035, -0.41893935203552246, -2.3501083850860596, -2.353259801864624, 0.17519812285900116, 1.9227237701416016, 0.1713913381099701, 0.6497076749801636, 1.330201268196106, 0.6755314469337463, -0.3455268144607544, -0.43084102869033813, 0.009265251457691193, -0.4966713786125183, -0.815862774848938, -0.3169621527194977, 1.4281647205352783, 1.7898892164230347, 0.677767276763916, 0.7535040974617004, 1.0880496501922607, 0.3090076148509979, 0.3728553354740143, 0.9065678715705872, 0.3471261262893677, -2.9409432411193848, 1.0219898223876953, -0.39679884910583496, -0.41992950439453125, 0.05621141567826271, -1.6457645893096924, -0.2788209915161133, 0.27819398045539856, 1.0050517320632935, 1.1796563863754272, -0.1762140840291977, 0.07121676206588745], [0.4720732569694519, 0.0981748178601265, -0.5370319485664368, -2.3960421085357666, 1.2287626266479492, -1.2948323488235474, -1.1372624635696411, 1.1179500818252563, -0.3855536878108978, -0.4184912443161011, 1.299629807472229, 0.5225628614425659, 0.7826143503189087, 2.2887392044067383, 0.2964825928211212, 0.05939413234591484, -0.4377315640449524, -0.9577810764312744, 0.3165777623653412, 0.6743442416191101, -1.0255178213119507, -0.6218979358673096, -0.5132951736450195, -0.6289967894554138, -0.31512969732284546, -0.492807537317276, 0.20642070472240448, 0.8689845204353333, 2.096297025680542, 0.05307527258992195, 1.2246265411376953, -0.015319766476750374, 1.8303028345108032, -1.6335560083389282, -0.8392930030822754, 0.05011652037501335, 0.4606704115867615, -0.7933650016784668, -0.9516544342041016, -1.2315771579742432, 0.8085750341415405, -0.6815658211708069, -1.3385976552963257, 0.8139241337776184, 0.8126135468482971, -0.0549531951546669, 1.505192518234253, 0.873862624168396, -0.28113433718681335, -0.6714422702789307, -0.8864333629608154, -0.09415458142757416, 0.49624472856521606, -0.4388011693954468, 1.5589615106582642, -0.05485333129763603, -1.1632440090179443, -0.01232729572802782, -0.23641668260097504, -0.4077765941619873, 0.3710896968841553, -0.061907339841127396, 0.11279290169477463, 0.029995551332831383], [-1.3333169221878052, 0.43598315119743347, 0.09488781541585922, 0.5845546722412109, 0.07931694388389587, -0.11108186095952988, -1.7214113473892212, -0.7264935374259949, -1.3928927183151245, -0.7936837673187256, 0.28102099895477295, -0.37896469235420227, -0.08109825104475021, 1.2242149114608765, -0.06620296090841293, -1.0084680318832397, -0.0933079645037651, 1.4523367881774902, 0.10865932703018188, 0.19780950248241425, 0.3268800675868988, 0.8011103272438049, -0.0149611197412014, -1.8615161180496216, -0.5539513826370239, -1.1340056657791138, -0.8965212106704712, -0.2901484966278076, -1.3248672485351562, -0.9589919447898865, 0.6668790578842163, 1.308260440826416, 0.26140764355659485, -0.31863299012184143, -1.1022659540176392, -0.6711276769638062, -0.06902477890253067, -0.28785377740859985, -0.5443721413612366, 0.7202138900756836, 0.1761704534292221, 0.030368678271770477, 0.6836017966270447, 0.776843786239624, -0.4340764284133911, -0.11177290230989456, 0.29635289311408997, 0.5201375484466553, -0.44793665409088135, -0.27761152386665344, 0.9452752470970154, 0.6212268471717834, 1.2647390365600586, -1.4917038679122925, 0.774626612663269, 0.861966073513031, -0.35618892312049866, -0.12534207105636597, -0.2560800313949585, 1.6595537662506104, -1.0426021814346313, 1.596592903137207, -1.0618212223052979, -0.7677318453788757], [-1.1535654067993164, -0.03111395426094532, 0.4868025481700897, -1.5259177684783936, 0.055304475128650665, -0.10734444111585617, 0.01839778572320938, -0.7262828946113586, 0.4589788615703583, 0.6203522682189941, -0.41690051555633545, -0.5875477194786072, 0.6949977278709412, -1.3532099723815918, 1.2233937978744507, 1.2244206666946411, -1.2696186304092407, -0.9048483371734619, 0.8136459589004517, -0.5397305488586426, 0.8253982663154602, -0.5538397431373596, -0.29120656847953796, -1.6200315952301025, -0.14334149658679962, -1.8766491413116455, 1.115809679031372, -1.2432823181152344, 0.11468395590782166, 1.599860429763794, 1.3726942539215088, 1.449210524559021, -0.2381904423236847, -0.37605375051498413, 0.05000026524066925, -0.7529200911521912, -0.4690025746822357, -1.1358226537704468, 1.9290764331817627, 0.9972492456436157, 0.526176393032074, 0.04210128262639046, 0.7227305173873901, 0.46534132957458496, 1.8944648504257202, 0.7699940204620361, 0.30821993947029114, 1.1219316720962524, -0.5377241373062134, 0.9322806596755981, -1.0498920679092407, 0.13442294299602509, -0.02932245470583439, 0.555753231048584, -0.539772093296051, -0.39959877729415894, -2.202068328857422, -0.34382468461990356, -0.11027475446462631, -0.3667312264442444, -1.8094624280929565, -0.05842959135770798, 0.30615246295928955, 0.04734150320291519], [1.5274064540863037, 1.6505857706069946, -0.9292860627174377, 0.11796706169843674, -0.5112574100494385, -1.8329864740371704, -0.5250157117843628, 1.5158623456954956, -2.306816577911377, 0.328872412443161, -0.22170482575893402, -0.5569964647293091, 2.3845818042755127, -0.29765209555625916, 1.8400847911834717, 1.281410813331604, 0.16569621860980988, 0.3989526033401489, -0.3543456196784973, -1.3526071310043335, -1.183795690536499, 0.7923490405082703, 0.20711646974086761, 1.4343931674957275, 0.1059965193271637, -1.0410748720169067, 0.690220296382904, 1.707105278968811, 1.0524696111679077, -1.4761459827423096, -0.7616097331047058, 1.798624038696289, 0.09977816045284271, 1.9933521747589111, -1.2015312910079956, 0.2824500799179077, 0.07243654876947403, -1.205992341041565, 0.18831245601177216, -1.7308984994888306, 0.49198928475379944, 2.170553684234619, -1.0677934885025024, 1.3322700262069702, 0.34633928537368774, 0.28901657462120056, -1.0138739347457886, -0.38514241576194763, -0.3340108096599579, -1.0449572801589966, 0.9718378186225891, -0.2876521050930023, 1.1594709157943726, 1.48476243019104, -1.035452127456665, -0.7369527220726013, -0.7184309959411621, -2.4792497158050537, 0.4602755606174469, -0.6723060011863708, 0.05310244858264923, 0.755535900592804, 0.4930442273616791, 0.5635189414024353], [0.44085803627967834, 0.48868080973625183, 2.411810874938965, 0.26960697770118713, -0.2733440101146698, -0.10556164383888245, -1.1477299928665161, 0.7472640872001648, -0.20366494357585907, 0.7391726970672607, -1.898881196975708, 1.0076767206192017, 0.3833928108215332, -2.3864552974700928, -0.06423939764499664, 2.0746498107910156, 0.4321155548095703, -1.040285587310791, -1.5161936283111572, 1.0302734375, 1.033386468887329, -0.5893660187721252, 0.1450323760509491, 0.4550178647041321, -1.0170592069625854, -0.41107985377311707, 0.311379075050354, -0.11443651467561722, 0.15723532438278198, 0.4334554970264435, 1.0291987657546997, 0.22133634984493256, 0.6357635855674744, 0.21854424476623535, 0.20253612101078033, 0.8102116584777832, -0.19716377556324005, -2.4565296173095703, -1.296948790550232, -1.197352647781372, -0.42399609088897705, 0.6585233807563782, -0.4276646375656128, 0.18050384521484375, 0.2497292459011078, 0.9557759761810303, -0.5800654888153076, 1.1317635774612427, -0.07533086091279984, 2.691298246383667, -0.8283172845840454, -1.1030281782150269, -0.3491816222667694, 0.9020293951034546, 0.4118051826953888, -0.9602757096290588, -0.04419361427426338, 1.800580382347107, -0.7305753231048584, -0.39916735887527466, 1.9940755367279053, -0.8185960650444031, 2.9081263542175293, -0.3761166036128998], [0.5078003406524658, 1.6903825998306274, 1.192340612411499, 0.8922653198242188, 0.16588154435157776, 1.3636821508407593, -0.4596555829048157, 0.7012292146682739, -1.8713093996047974, -0.4925108551979065, 0.1999867558479309, 0.5816490054130554, 1.3341206312179565, 0.16026295721530914, -0.1630953997373581, -0.2548905313014984, -0.06203273683786392, 0.5590953826904297, -0.01768915355205536, 1.3899284601211548, -0.5171437859535217, -0.21489709615707397, -0.33900952339172363, 0.15468759834766388, 0.3040933310985565, 0.9333302974700928, 0.8722724318504333, 0.6031686663627625, 1.4158645868301392, 0.5716349482536316, 1.4658703804016113, 2.1390202045440674, -0.13937126100063324, -1.1066370010375977, 0.009412289597094059, 0.14907653629779816, -0.024328891187906265, 0.8696383833885193, 1.1576236486434937, -1.8493757247924805, -0.3032277524471283, 1.4288625717163086, -0.45234477519989014, 0.5711528658866882, -1.6699105501174927, -0.3269752860069275, 0.7586048245429993, -0.11438227444887161, -1.6904196739196777, -0.4609224498271942, -0.21581436693668365, -0.04145864024758339, -0.40506312251091003, 0.8789302706718445, 0.40043750405311584, 0.7313408255577087, -0.583869218826294, 0.5644165277481079, -1.6302167177200317, -0.5183998346328735, 0.9243289828300476, 0.6302377581596375, 0.8988246917724609, 1.3051129579544067], [1.3570122718811035, -1.2877321243286133, -0.40779080986976624, -0.8987663984298706, -1.1433005332946777, 0.1537049114704132, -0.48884740471839905, -0.07093080133199692, -0.8345761895179749, 1.3110134601593018, -1.1443372964859009, 0.7980709075927734, 1.6587499380111694, 1.2472351789474487, 0.2323806881904602, 0.9190056920051575, -0.9453105330467224, -1.318355679512024, -1.4635573625564575, -0.24470064043998718, -0.33993515372276306, 0.746598482131958, 0.9806947112083435, 0.42391714453697205, -1.8610221147537231, -1.694331407546997, -1.2119057178497314, -0.754473090171814, -0.41715988516807556, 0.274933397769928, 0.7635471224784851, -1.0921467542648315, -0.33510035276412964, 0.6162521839141846, -0.9398729801177979, -0.505909264087677, 0.23511619865894318, 0.7724398970603943, 0.27696657180786133, 0.021365126594901085, 0.2970329821109772, -1.3442823886871338, -0.8890335559844971, -0.5153987407684326, 0.008956455625593662, -0.8180637955665588, 0.15632791817188263, -1.0871403217315674, -1.2832448482513428, 0.5768371820449829, 0.12610842287540436, 0.39887797832489014, 1.2696542739868164, -1.1464375257492065, 1.3278652429580688, -0.3461480438709259, -0.4671138823032379, -0.4815966784954071, 0.4759191572666168, 0.0530194528400898, -0.5168721079826355, 0.346065878868103, -0.14628584682941437, 2.035371780395508], [-1.202681303024292, -0.6279018521308899, -0.21295730769634247, 1.040453314781189, -1.0490381717681885, 0.8788217902183533, 1.5360907316207886, -0.3015036880970001, -0.920279860496521, -0.19647404551506042, 1.0205063819885254, 0.9320793747901917, 0.19044864177703857, -0.45321205258369446, -0.11913822591304779, 0.26306161284446716, -0.5294575691223145, 0.9791898131370544, 0.11196818947792053, -0.03884357586503029, 1.8581252098083496, -0.9037303924560547, 2.001399040222168, -1.0769565105438232, -1.392080307006836, 0.30445948243141174, 1.3068044185638428, 0.37286606431007385, -0.7161832451820374, -0.15499038994312286, 0.6641731858253479, -2.1021714210510254, 1.688549518585205, 0.34389954805374146, 0.12176890671253204, -1.0768622159957886, -0.18957604467868805, 0.10879664868116379, -1.6436009407043457, 0.35971447825431824, 0.5912445783615112, -2.4727530479431152, -0.9283852577209473, 1.9332174062728882, -0.923117458820343, 0.012683494947850704, -0.8764079213142395, 3.2550971508026123, 0.7154251337051392, 0.23455314338207245, -1.0288219451904297, 1.3929862976074219, 1.3225173950195312, 0.8509598970413208, -0.3610573709011078, 1.1408804655075073, -1.6569303274154663, 0.1075177788734436, -0.44711580872535706, 0.32605481147766113, 0.14061956107616425, 0.036140650510787964, -1.804368495941162, -0.7325586676597595], [-0.9991503357887268, -1.0339614152908325, -1.3025083541870117, -0.5799788236618042, -0.18429245054721832, -1.255010724067688, -0.7509170174598694, -0.759614884853363, -0.5633164048194885, -0.5338186025619507, -0.6526796817779541, -2.1257481575012207, -0.13922366499900818, 1.0385531187057495, -0.2157791554927826, 0.2512686252593994, 0.9514954090118408, -0.2743717432022095, -0.29608842730522156, 0.5017064213752747, 0.25846871733665466, -0.21147015690803528, -1.0735224485397339, -0.038692403584718704, 0.5355748534202576, -1.574007511138916, -0.43744248151779175, 0.4518313705921173, -0.21804320812225342, 2.355020523071289, 1.0936356782913208, 1.4047462940216064, 0.3079420328140259, 0.8244696259498596, 0.3580700755119324, -0.1250026524066925, 1.1221470832824707, 0.8658796548843384, 0.9381912350654602, -0.5236366391181946, 1.2154899835586548, -0.11882049590349197, 0.44175127148628235, 2.2051150798797607, 0.5362956523895264, 0.22045114636421204, -0.6406256556510925, 1.441323161125183, 0.2873404622077942, -0.7801492214202881, -1.3076163530349731, -0.11643223464488983, 1.0502301454544067, 0.18873484432697296, -1.1829824447631836, -0.24015597999095917, -0.779497504234314, 0.12096388638019562, -0.5160526037216187, -2.238908052444458, 1.4910612106323242, 0.26341941952705383, 0.389598548412323, -0.5818691253662109], [0.31972891092300415, 1.7483001947402954, -0.43694987893104553, -1.0359933376312256, 0.04334739223122597, 0.2045520842075348, -0.4515184462070465, -0.12615783512592316, -0.005128561519086361, -0.5136200189590454, 1.0361589193344116, 1.6285102367401123, 1.1351770162582397, 0.37665003538131714, -0.9301851391792297, 0.5189594626426697, -0.43518170714378357, 0.9833743572235107, -0.21706703305244446, 0.8522018194198608, -0.4389859437942505, 1.020825982093811, -0.9868996143341064, 0.688303530216217, -0.15625321865081787, -0.9911476373672485, -0.9993689656257629, 0.41572752594947815, -1.0975691080093384, -0.0044685848988592625, 0.24192892014980316, -0.17282643914222717, 0.8266569972038269, 1.4601075649261475, 0.3765079081058502, 2.43770170211792, 1.085694670677185, -0.39624762535095215, -0.49376416206359863, 0.1600007563829422, -0.3892767131328583, 1.2542603015899658, -0.6164823770523071, -1.2728371620178223, -0.09319406747817993, 1.5640405416488647, -0.07231373339891434, -2.967885971069336, 0.44213542342185974, -0.8750551342964172, 0.39198970794677734, -1.7052263021469116, 0.20961083471775055, 0.39244142174720764, 0.7710829377174377, -0.5411460995674133, 0.12604641914367676, -0.09877713769674301, 0.6670427918434143, 2.1216609477996826, -0.13534927368164062, 0.14882762730121613, -0.9117223620414734, 0.9164446592330933], [-1.390241265296936, -1.357266902923584, 0.6068029999732971, 0.5919873714447021, 0.8499979376792908, 0.7346373200416565, -1.7443668842315674, 0.3479064404964447, -0.056175749748945236, 0.9709736108779907, 1.0368632078170776, -0.5973432660102844, -0.4180268347263336, 0.28535768389701843, 0.5074504017829895, -0.7051061391830444, 0.690314531326294, -0.621310830116272, 1.011982798576355, 0.2950669229030609, -0.023431502282619476, 0.8121325969696045, 0.043917905539274216, -1.1109404563903809, -0.2190999835729599, 0.4535093307495117, -0.1549765020608902, -0.4546394646167755, 0.07009603828191757, -0.032005827873945236, -1.122984528541565, -0.3093029856681824, -0.7405940890312195, 1.0811034440994263, 0.3866453468799591, -0.767932653427124, 0.4579283595085144, -1.3187021017074585, 1.2505301237106323, 0.002815835177898407, -0.9030241370201111, -0.6073117852210999, 0.170111283659935, -1.3829227685928345, 0.15766283869743347, 0.0748763307929039, -0.0759393498301506, -0.3566681146621704, -0.5431607961654663, -1.8784445524215698, -0.20606769621372223, 0.2052411437034607, -0.5257024765014648, -0.3065147399902344, 1.4526164531707764, -0.8441082835197449, 1.3741191625595093, -0.10291627794504166, -2.2066736221313477, -1.1011558771133423, -1.444733738899231, 0.4868672490119934, 1.0823607444763184, 1.1986525058746338], [-0.3041171133518219, 0.6099369525909424, -1.8407413959503174, -0.370236337184906, 0.2819455862045288, -1.1296210289001465, -0.9342021346092224, 0.39511778950691223, 0.8030299544334412, 1.124755859375, -1.3781691789627075, -0.8639882802963257, 0.8354033827781677, 0.1789456307888031, -0.5687242150306702, 0.564348042011261, -0.7595282196998596, 0.23199626803398132, 0.4754776060581207, 1.1518380641937256, 1.2903112173080444, -1.5697354078292847, -0.38412734866142273, -1.6965324878692627, -1.6349380016326904, 2.597167491912842, -0.03536311909556389, 0.6786322593688965, -0.45060935616493225, -1.7069000005722046, 1.826715350151062, -1.6193010807037354, -0.12767402827739716, -1.0543513298034668, -0.3865818381309509, 0.5371733903884888, 0.9010053277015686, -0.1433776617050171, -1.7445075511932373, -0.0916290208697319, 0.48074871301651, -1.4554238319396973, -1.2027312517166138, -0.18269075453281403, -0.1485683172941208, -0.027112653478980064, -0.49214762449264526, -1.0684332847595215, -0.00755418511107564, 1.2005889415740967, 1.0386189222335815, -0.937957763671875, -0.5999802350997925, 0.3161886930465698, -1.2217425107955933, -0.8259849548339844, 1.1778455972671509, -1.2626577615737915, 0.5599199533462524, -2.423474073410034, 0.6498380303382874, 0.6297963857650757, 0.07204481214284897, -0.8729230761528015], [0.3407200872898102, -0.07464831322431564, -0.08364751189947128, 0.014768478460609913, 0.5117638111114502, -0.5516979098320007, -0.5805354714393616, 0.4924944043159485, 0.7227885723114014, 0.9555904269218445, 0.010946547612547874, 1.6337977647781372, 0.6934441924095154, -0.3498774766921997, 1.1124353408813477, -0.908517062664032, 0.7620150446891785, 0.9377530813217163, -0.32808926701545715, -0.025225123390555382, -1.6587227582931519, -0.21385936439037323, 0.8139581680297852, 0.7049961686134338, 0.4667615592479706, 0.7807289958000183, 0.9946379065513611, 0.8041600584983826, -0.866158127784729, 0.57709801197052, -1.1872029304504395, 0.13308002054691315, -0.3815615773200989, 0.8844534754753113, -1.568931221961975, 0.33043980598449707, 0.46100515127182007, -2.4229891300201416, -0.3945911228656769, -0.6632997989654541, 1.0060573816299438, -0.32604098320007324, -0.25806212425231934, 0.46048858761787415, 0.45424845814704895, 0.8224777579307556, -0.7674832344055176, -2.25313663482666, -1.5044543743133545, -1.5860226154327393, -1.4685070514678955, -2.1599223613739014, 0.5378449559211731, -0.6261727213859558, 2.5932350158691406, -1.779144525527954, -0.13921356201171875, 3.155128002166748, -0.42162075638771057, 0.21402433514595032, 0.7183987498283386, -0.14120377600193024, 1.5196274518966675, -0.5728959441184998], [-0.252860963344574, 1.975293517112732, 0.4749610126018524, -0.020681997761130333, 0.34986329078674316, -1.567376732826233, 0.3444576859474182, -1.3019416332244873, 1.8622344732284546, 0.8118875622749329, 0.35987603664398193, 2.2743349075317383, -0.5427375435829163, -0.40159159898757935, -0.016146603971719742, 1.5489517450332642, 2.040865182876587, -0.4207976758480072, 0.636496901512146, 0.3745293915271759, 0.23045411705970764, -1.3557589054107666, 0.673551082611084, -3.1161928176879883, 0.7183573246002197, 0.42437320947647095, 0.5221927165985107, -1.8221129179000854, -0.24354620277881622, 0.05188751220703125, -0.7222378849983215, -0.009159340523183346, 1.0697258710861206, -1.2387231588363647, -1.4616045951843262, -1.5817110538482666, 0.7414994835853577, 0.8260900378227234, -0.6066080331802368, 0.5830433368682861, 1.03061842918396, -1.7271020412445068, -1.4437557458877563, 0.08122117072343826, 0.6559627056121826, -0.8027258515357971, -0.12949880957603455, 1.0893343687057495, -0.5441421270370483, -0.17613579332828522, 1.3489270210266113, -1.4628552198410034, -0.30034899711608887, -0.1752934455871582, -0.019495069980621338, 0.24510866403579712, 0.1967753767967224, 1.9319871664047241, -0.12856410443782806, -0.2920610010623932, 2.8657610416412354, -1.0088131427764893, 1.0997753143310547, -0.573684573173523], [-0.25217488408088684, -2.092756748199463, 0.709701418876648, 1.8613839149475098, 1.4274775981903076, 0.6366881132125854, 0.003967731259763241, -0.9459325671195984, 0.039168741554021835, -0.30818474292755127, -1.0577481985092163, -1.6430764198303223, 0.24025408923625946, -0.4123463034629822, -0.4769642949104309, 0.512791633605957, 0.010801948606967926, -0.7686824202537537, -0.3162352442741394, 0.46885642409324646, -1.0512818098068237, -0.32976993918418884, -2.028942584991455, 0.5079644918441772, 0.07822181284427643, 0.2927616238594055, -1.0956496000289917, -0.1527942270040512, 0.48065370321273804, -1.8958970308303833, -0.18675071001052856, -1.5055105686187744, 1.0217368602752686, 0.356210321187973, -0.6624590754508972, 1.063175916671753, 0.6131788492202759, -0.16194313764572144, 1.0239249467849731, -1.9135761260986328, -1.524389386177063, 0.2757536470890045, 0.002067569177597761, -0.7768843770027161, 0.6216548085212708, -0.687099814414978, -0.09697073698043823, 0.37295734882354736, 1.5157612562179565, 0.1303882598876953, -0.6795754432678223, 1.3576890230178833, -0.61937016248703, -1.5931415557861328, -1.0445256233215332, -0.6431500911712646, -1.1239351034164429, 2.469569683074951, -0.608036458492279, 0.660913348197937, 0.862321138381958, 1.1607853174209595, 0.0878388062119484, -1.7248015403747559], [-0.24261528253555298, 0.36325645446777344, 0.2655651271343231, -0.3807951807975769, 2.3547422885894775, -0.40546080470085144, 0.151511549949646, -0.9414806365966797, 0.960352897644043, -0.6551848649978638, -1.8393585681915283, -1.0118857622146606, 0.16222605109214783, -0.41262486577033997, 0.4549461603164673, 0.915321409702301, 0.6526072025299072, 1.3803393840789795, -0.7343893647193909, 0.4401194155216217, 1.4287899732589722, 0.47002550959587097, 0.12843367457389832, -0.4733069837093353, 0.550480842590332, 0.7780507206916809, 1.655408501625061, 0.7933712601661682, -0.12422815710306168, 0.3225240707397461, -1.8218610286712646, -1.7885414361953735, -1.379204511642456, 0.5188164114952087, -1.1654106378555298, -1.4283149242401123, 0.33547618985176086, 0.2756265103816986, -0.19918310642242432, -0.005478087347000837, 0.6410843133926392, 0.09714934229850769, 0.12611514329910278, 0.48781782388687134, 0.32081541419029236, 0.8119064569473267, 0.5199797749519348, -1.658331274986267, -0.0784572884440422, 1.1309001445770264, -0.21484030783176422, 1.5493724346160889, -0.08408793807029724, 0.4470716416835785, -0.8124526143074036, 0.6454991698265076, -1.2759425640106201, 1.2182445526123047, 1.6683504581451416, -1.0480103492736816, -0.7735516428947449, -0.5685067772865295, 0.9330781698226929, -1.3927561044692993], [0.132660910487175, 0.11759039759635925, -0.4454095661640167, 1.2307668924331665, -0.47993698716163635, -0.3358743488788605, -2.574047088623047, -0.2343616932630539, -0.2088400274515152, -0.20614729821681976, -0.6391292214393616, 1.4095534086227417, -0.5145395994186401, 0.24825476109981537, -2.3603322505950928, -1.5833563804626465, -0.114983469247818, 0.5215739011764526, 0.45452287793159485, -0.4344576895236969, 0.23093155026435852, -0.6979178190231323, -0.12375066429376602, 0.46004411578178406, 1.447922706604004, -0.029350636526942253, 0.5273615717887878, -0.10925988852977753, 0.15556706488132477, 1.1783125400543213, 0.26956579089164734, 1.2105751037597656, -1.0549360513687134, -0.1594744622707367, -0.47328194975852966, 1.807637333869934, -1.7322314977645874, 0.4602651596069336, -0.9760648012161255, -2.129222869873047, 2.5789053440093994, -0.4229030907154083, -0.9696038365364075, 0.30809879302978516, 1.0705934762954712, -1.0494999885559082, 1.772896409034729, -0.17195461690425873, -0.5002793073654175, -0.9430338740348816, -0.42467761039733887, 0.08147229999303818, 2.5408170223236084, 0.3932454288005829, -0.09087730944156647, -0.20610752701759338, -1.1362255811691284, -1.2734289169311523, 1.0407061576843262, 1.0422000885009766, -0.7984888553619385, 0.9934708476066589, 0.06307495385408401, 0.6090384721755981], [0.9065446257591248, 0.7759968042373657, -0.10689660161733627, -0.38149189949035645, 0.1278509497642517, -0.41244015097618103, 0.16387389600276947, 0.4828103482723236, 1.5888607501983643, -0.8816419839859009, 0.4599801301956177, 0.018554596230387688, -0.9089929461479187, -1.0670195817947388, -1.601874589920044, -0.5883464813232422, 1.7744123935699463, 0.49125543236732483, 0.10571040958166122, -1.825671672821045, 1.688469648361206, 0.40564072132110596, 0.12052707374095917, 0.5729214549064636, -0.36359110474586487, 1.658751130104065, 0.43184810876846313, 0.10120020061731339, 2.0202009677886963, 1.9095468521118164, -0.06852822750806808, -1.5625832080841064, 0.5181499123573303, -1.328757882118225, 0.8793343305587769, 2.460693120956421, -1.4953986406326294, 0.9660786986351013, -0.0069454689510166645, -0.16338782012462616, 1.066707968711853, 1.1093374490737915, 0.5617602467536926, -0.29121971130371094, 0.534254252910614, -0.7906268239021301, -1.4507825374603271, -0.24014395475387573, 0.831731379032135, -0.3719431459903717, -0.6243720650672913, -0.11176992207765579, -0.5713719725608826, -0.279000848531723, -0.6020406484603882, -0.4911425709724426, 1.6742215156555176, 1.0747812986373901, -1.2595231533050537, -1.5357927083969116, -0.5189745426177979, -0.9753064513206482, 1.280548095703125, -1.9580261707305908], [-3.1389272212982178, -0.35409829020500183, 0.06641386449337006, 0.21691440045833588, -0.9686018824577332, -2.3910858631134033, -1.904375433921814, -0.8636000752449036, 1.2911549806594849, -1.0918495655059814, -1.5164830684661865, 0.0002928582835011184, 0.1715899556875229, 0.5716844201087952, 0.26999402046203613, -1.326911449432373, 0.18140311539173126, 1.443385124206543, -2.3696930408477783, -1.7673078775405884, -2.4109156131744385, 1.1465259790420532, 1.313072681427002, 1.1286755800247192, 1.422532081604004, 0.22553935647010803, 2.4374866485595703, 0.4114053249359131, -0.7153375148773193, -0.6226494312286377, 0.012641729786992073, -1.202913522720337, 1.0473395586013794, -0.12977318465709686, -0.2868656516075134, -0.7007561922073364, 0.016460498794913292, 1.993961215019226, 2.1146163940429688, 0.07311635464429855, -0.594042181968689, -1.4634751081466675, 1.1338329315185547, 0.14113552868366241, -0.540748119354248, 0.2647409737110138, 0.20874151587486267, -0.9153695106506348, -0.502223789691925, -0.10714641213417053, 0.9450116157531738, 0.6359351873397827, 0.3308975100517273, 0.2595117390155792, -1.5951781272888184, 0.9967066049575806, -0.42337167263031006, -0.0009922755416482687, -0.9678248167037964, -0.5644207000732422, 0.15596795082092285, 0.21950408816337585, -0.3946443498134613, 0.03649603947997093], [-1.510612964630127, -0.731533408164978, -0.1877366602420807, 1.4776191711425781, -0.36347007751464844, 0.30519333481788635, 0.1293102651834488, 0.46385785937309265, -0.3022792935371399, -0.07203111052513123, -0.9570599794387817, 1.020811676979065, -0.31094253063201904, 0.12617838382720947, -1.0543203353881836, -0.6166518926620483, -1.2005056142807007, -1.2816435098648071, -0.38091835379600525, 0.08945625275373459, -0.9911978840827942, 0.9117615818977356, 1.6880173683166504, -0.1368618607521057, 0.5959833860397339, 0.6150373816490173, -1.4574040174484253, 1.4447133541107178, 0.9076148867607117, 0.15547679364681244, 0.7989009618759155, -0.8137950897216797, 1.0336970090866089, 0.5258023142814636, -0.9884814023971558, -1.2453371286392212, -1.092584490776062, -0.6980885863304138, -0.28052109479904175, -1.4920036792755127, 1.0941296815872192, -1.3127042055130005, 0.24510642886161804, -1.0070605278015137, 0.09083667397499084, -0.5250672101974487, 1.093356966972351, -0.9215587377548218, -0.081351138651371, -1.2494405508041382, 1.5819377899169922, 2.1040565967559814, 0.6470081210136414, -1.155875563621521, -1.3816908597946167, -0.2718712091445923, -3.0172083377838135, -0.07806774973869324, 0.48650315403938293, -0.17051143944263458, 1.2790178060531616, 0.449133962392807, 1.023637294769287, -1.7806557416915894], [0.7160780429840088, -1.0910052061080933, 0.4311883747577667, -0.3138079345226288, 1.0857582092285156, 0.4038330018520355, 0.5001022219657898, 0.09154260158538818, -0.8531040549278259, 0.21193185448646545, -0.5067881345748901, -0.12643322348594666, -0.5638979077339172, -0.40516331791877747, -0.8952094912528992, -0.6803349852561951, 0.7340356707572937, -1.0613714456558228, 0.13967853784561157, 0.6367353796958923, 1.7232943773269653, 0.1444956213235855, -0.586905300617218, 0.7720344662666321, 0.02355744130909443, 0.15255409479141235, 1.2808208465576172, 0.6304750442504883, -0.05742771923542023, -1.5920525789260864, -0.9276425242424011, -0.3261741101741791, 0.7400256991386414, 0.6540403962135315, 0.6076570749282837, 0.06521613150835037, -0.8169024586677551, 0.18880201876163483, -0.6635047793388367, 0.49134117364883423, -1.3501520156860352, -0.4441673159599304, 0.5500304102897644, -1.8703101873397827, 0.9391335248947144, -0.3193493187427521, 0.42906320095062256, 0.055981095880270004, 0.444524347782135, 0.600721001625061, -0.2286939173936844, -0.07962500303983688, -0.28997889161109924, -0.15074579417705536, -0.5080049633979797, 0.2832677662372589, 0.043353646993637085, 0.149480402469635, 2.8038017749786377, -1.0861804485321045, -0.9342981576919556, -0.9040523171424866, -0.33074918389320374, 0.18092454969882965], [-0.38632655143737793, -0.4772122800350189, 1.7552940845489502, 0.9385027289390564, 0.605820894241333, -0.6591552495956421, 1.892374873161316, 2.1413164138793945, 0.36850517988204956, -0.7936634421348572, 1.3054230213165283, 1.7679086923599243, 0.0393572673201561, 1.3740936517715454, -0.31746625900268555, -0.661377489566803, 0.7031705975532532, -0.7605021595954895, -1.5261801481246948, -0.41316935420036316, -1.2981206178665161, -0.6204324960708618, 0.04417373239994049, -0.9428108930587769, -0.6786921620368958, 2.2816317081451416, -1.0583754777908325, 1.4881101846694946, 1.7830226421356201, -0.24445313215255737, 0.4460887014865875, 0.43353086709976196, 1.4080111980438232, 0.6750532984733582, 0.16473549604415894, 0.2938199043273926, -0.6689684987068176, 0.8273594379425049, -1.2942395210266113, -2.2431905269622803, 0.3821650445461273, 0.13810698688030243, -0.07616544514894485, 0.12138993293046951, -0.27220237255096436, 0.359253853559494, -0.3675801455974579, -1.6830981969833374, 0.9826257824897766, 0.1843441128730774, 2.022892475128174, -0.07396210730075836, 1.190608263015747, 2.8290820121765137, 1.7406835556030273, 0.5465396642684937, -0.9509457945823669, 0.7895827889442444, -1.1796526908874512, 1.072704553604126, -1.8277760744094849, -0.5493910908699036, -0.7725344896316528, -0.061763517558574677], [-0.5035258531570435, 0.7614571452140808, 1.3795894384384155, 0.6251732110977173, -0.6217265129089355, 2.1371734142303467, 0.41019803285598755, 0.18580546975135803, 0.3857954442501068, 0.27891698479652405, -0.0047010439448058605, -1.3957781791687012, -0.3616621494293213, -0.0752822607755661, 0.510918915271759, -0.4145030379295349, 0.20728036761283875, -0.21189071238040924, 1.1720942258834839, -0.6501088738441467, -0.17637485265731812, 2.6450860500335693, -0.7807914614677429, 0.5763412714004517, 1.3282312154769897, -0.08687835931777954, 0.3292851150035858, -1.144303798675537, 0.3956379294395447, 0.24412238597869873, 0.4352628290653229, -0.2895025312900543, 0.10284098237752914, 0.24921293556690216, -1.4123274087905884, 0.6373959183692932, -1.005570411682129, -0.3954707384109497, -1.5011872053146362, -0.19064973294734955, 0.5170339941978455, 0.7023894786834717, -0.7582924365997314, -0.04604120925068855, 1.3696246147155762, -2.0127599239349365, 0.5944908857345581, 1.1942903995513916, 1.1299583911895752, 1.177736759185791, 0.38731539249420166, -0.28478625416755676, -2.1177406311035156, -0.2221411019563675, 0.041213348507881165, -1.524719476699829, 0.6205078959465027, -1.2775989770889282, 0.13554900884628296, 0.6179593801498413, -0.4037335216999054, 0.27063125371932983, -0.9564524292945862, -0.24253807961940765], [-0.19147063791751862, -0.89470374584198, -0.07373154908418655, 0.8869732618331909, 0.06782805919647217, 0.871143639087677, -0.720788300037384, 0.3503434360027313, 1.603928565979004, 0.774847686290741, 1.138850212097168, 1.1078027486801147, 1.1280652284622192, -0.7638674378395081, -0.42725783586502075, -0.616101086139679, -0.33747878670692444, -1.7939296960830688, -1.2869973182678223, -1.4869465827941895, -0.8237600326538086, 0.4949730634689331, 1.3485225439071655, 0.7488738894462585, 1.4242733716964722, 1.381474494934082, 1.8304154872894287, -0.4617573022842407, -0.12283055484294891, 0.8090324401855469, 0.049248989671468735, 0.5453012585639954, -0.27406907081604004, 1.3815003633499146, 1.5227543115615845, -1.223839521408081, 1.3745850324630737, 0.8483313322067261, 1.2340214252471924, 0.44837072491645813, 0.9038780927658081, -0.11635658890008926, 0.5782760381698608, 2.7072744369506836, 0.5082218050956726, 0.5264812111854553, 0.7181338667869568, -0.6397770047187805, -0.3453736901283264, 0.486604243516922, -1.3103971481323242, -1.0517351627349854, 3.5516748428344727, -0.6597668528556824, -1.5831108093261719, 1.0475279092788696, -1.0024598836898804, -1.6144431829452515, 0.28717729449272156, 0.2658344507217407, -0.4477449655532837, -0.6193239688873291, 0.20957989990711212, -1.3846124410629272], [-1.3041259050369263, 0.4252760410308838, 0.0610990971326828, -2.0517783164978027, -2.5077037811279297, 0.09128990769386292, 0.1358448714017868, -0.639268159866333, 0.7792921662330627, 0.9093397855758667, -0.22635246813297272, 0.013888172805309296, -0.7346336245536804, -0.15955737233161926, -1.0007050037384033, -1.0697910785675049, -0.7212390303611755, 1.2919145822525024, 0.8516026139259338, 0.32406923174858093, -0.03701438754796982, -0.2642308175563812, 1.9461320638656616, -0.7863014340400696, -0.007295573595911264, -0.8292962908744812, 0.5461680889129639, 0.3796917498111725, 0.358101487159729, -0.40728142857551575, 0.04405504837632179, -0.05434449762105942, 0.22952179610729218, -1.8792481422424316, -0.9621090888977051, -1.5585278272628784, -0.07494068890810013, -0.1949797421693802, 0.44814127683639526, -0.2414238154888153, -0.9828141927719116, 0.7514029145240784, 1.6519639492034912, 1.635679006576538, -1.8485132455825806, -0.47939905524253845, 0.4456838369369507, -0.9341436624526978, -1.2978591918945312, -0.1303323209285736, 0.4915465712547302, -0.45628830790519714, 2.0257568359375, -0.8610016703605652, 0.18005383014678955, -1.6792430877685547, 1.0207031965255737, 0.5805611610412598, -0.7906397581100464, 0.9237396717071533, 1.141933798789978, 0.6867014169692993, -1.1296226978302002, -0.4351580739021301], [0.15732920169830322, 0.8166554570198059, 1.0365257263183594, 0.8006747961044312, -1.0791569948196411, -0.6802337765693665, -0.8564618825912476, -1.2640204429626465, -0.8624860644340515, -0.5958817005157471, 1.7832273244857788, 0.31082335114479065, 0.927932858467102, -1.0342320203781128, -2.1046018600463867, 0.7332568168640137, 1.4718244075775146, -0.8046998381614685, -1.0807228088378906, -0.2832217216491699, -0.30323541164398193, -0.7282016277313232, -0.7338578104972839, -0.2865928113460541, -1.6280125379562378, 0.08069644123315811, 1.0415247678756714, -0.04738864302635193, 1.2866617441177368, 0.6383886337280273, 2.589028835296631, 0.5212944149971008, 1.489555835723877, 0.09378337860107422, 0.6047956347465515, -0.6682947278022766, 0.8515835404396057, 0.902664303779602, 0.27049073576927185, 1.245637059211731, 0.6282352209091187, 1.6909364461898804, 1.1396899223327637, 1.3990554809570312, 0.32606935501098633, 1.9066569805145264, -0.18571053445339203, -2.1718780994415283, -1.9396573305130005, -2.153432846069336, -0.8897485136985779, 0.9652644991874695, 1.7258296012878418, -0.7630043029785156, 1.5351521968841553, 1.0906590223312378, 0.04902646690607071, 1.4896016120910645, -0.2533465027809143, -0.33266133069992065, -0.02452453039586544, -1.6454824209213257, 0.3794920742511749, 0.6869863867759705], [-0.8196645379066467, -1.1715881824493408, -0.039110034704208374, 1.1776288747787476, 0.735181987285614, 0.01720578223466873, -0.5160406827926636, 0.9469173550605774, -0.05063853785395622, 0.5982213020324707, -0.6328508853912354, 0.6273050904273987, -1.5732414722442627, -0.31416505575180054, -0.41678759455680847, -0.2332136482000351, -1.9648370742797852, 0.8818362355232239, -1.3252547979354858, -0.7556067705154419, 1.0910112857818604, 0.37745627760887146, 0.7427827715873718, 1.397135853767395, -1.9841605424880981, -0.9117377400398254, -1.574798345565796, -2.249605417251587, 0.7951298952102661, -0.33830544352531433, 0.5853076577186584, -0.09512107819318771, -0.8802909851074219, -0.10189518332481384, -0.33505022525787354, 1.4697649478912354, 2.0474483966827393, 1.0757635831832886, 0.07724417001008987, -1.920073390007019, 1.4159436225891113, 1.4403655529022217, -2.5925920009613037, 1.2691622972488403, -0.23526300489902496, -1.360439419746399, -0.609885573387146, 1.7421293258666992, -0.402876615524292, -0.28507232666015625, 0.8884421586990356, -0.7865308523178101, -0.5158749222755432, 1.1927659511566162, 0.5657293796539307, -0.7280406951904297, -0.8355078101158142, -0.1693659871816635, -1.2211276292800903, -0.33809974789619446, -0.3024759888648987, 0.4586719274520874, 0.6264047622680664, -1.8623785972595215], [-1.4018574953079224, 0.0045894114300608635, -1.1846705675125122, -0.3210274279117584, -0.4620768427848816, -2.361349582672119, -2.353440046310425, 0.08808065205812454, -0.2566746175289154, 0.8354969620704651, -1.575466275215149, 0.4753476679325104, -0.024430524557828903, 0.9826768040657043, 0.6680504679679871, -1.9867630004882812, 0.647759199142456, 0.6130934357643127, -1.0133925676345825, 0.0750090479850769, -1.439220666885376, 0.3317471742630005, -1.5868794918060303, 1.3137720823287964, 0.5340377688407898, -0.22986845672130585, -0.6297318935394287, 0.679425060749054, 0.8057108521461487, -0.3370669484138489, -0.8749461770057678, -0.48500046133995056, -1.535321831703186, 0.17578648030757904, 0.524092972278595, -1.6965891122817993, 1.4321094751358032, -0.2196681946516037, -0.4905426800251007, -0.8425567150115967, -0.1077670082449913, -0.9797104001045227, 0.07301869988441467, -0.5250532627105713, 1.467274785041809, -1.7926855087280273, -0.4851982295513153, -0.7635326385498047, -1.9159963130950928, -0.2561345398426056, 0.6490806937217712, 1.058678388595581, -1.4004851579666138, -0.26311540603637695, 0.18769635260105133, 0.34027087688446045, -0.4935056269168854, -1.1992835998535156, 2.295386791229248, 0.09147386252880096, -0.10331299155950546, -1.6314483880996704, -0.726265549659729, -0.8430503606796265], [1.9958219528198242, 0.8786965012550354, 1.0926018953323364, 0.018333200365304947, -0.06210196390748024, -0.9490720629692078, -0.6531738042831421, -0.19559814035892487, 0.45492643117904663, 0.5893101692199707, 0.6642293334007263, 0.5714151263237, 0.2287205010652542, 1.2261605262756348, -1.2487175464630127, -1.6602298021316528, -0.07874830067157745, -0.9768697023391724, -0.03432002291083336, 0.860317051410675, 0.6838465332984924, -0.1455206423997879, -0.3853379786014557, -0.051769550889730453, -0.21851903200149536, -1.1673957109451294, 0.35055649280548096, -0.011510813608765602, -0.1824037730693817, -0.10167964547872543, 0.66807621717453, 1.5016701221466064, -0.21968014538288116, -0.9138262271881104, 1.771692156791687, -0.47285398840904236, 0.013702156022191048, 0.9080333709716797, 0.6952194571495056, 0.8585477471351624, 0.6336014270782471, -0.6434388756752014, 0.6851287484169006, 0.28712567687034607, 0.10363222658634186, 0.49648794531822205, -1.5662914514541626, -0.15367506444454193, 1.4089579582214355, -1.0489705801010132, 1.426261305809021, -0.4156092405319214, -1.217960000038147, 1.0967570543289185, -0.13903450965881348, 0.29888176918029785, 1.009887933731079, -0.908791184425354, -0.39516136050224304, 0.35526132583618164, 0.8456394672393799, 0.01004298310726881, -0.11440476030111313, 0.14695985615253448], [0.024507589638233185, 1.0139964818954468, 1.2696338891983032, -0.20058861374855042, 0.40270981192588806, 0.2094707489013672, 1.0049463510513306, 0.5878182649612427, 0.07808136194944382, -0.43218570947647095, 1.2293858528137207, -0.3888026177883148, -1.3988418579101562, 0.46219852566719055, -0.6412573456764221, 0.9399586319923401, 1.5948923826217651, -1.3040034770965576, -1.4270858764648438, 0.36170634627342224, 2.0490317344665527, 0.8363842368125916, 0.7937823534011841, -0.15436606109142303, 0.5058881640434265, 0.2801378071308136, -0.36720848083496094, -0.6301316022872925, 0.054167490452528, 0.1525530070066452, 0.18537190556526184, -0.5587921142578125, -0.6991849541664124, -0.9699347615242004, 0.9314730167388916, 0.8914652466773987, -0.044411152601242065, -1.5897598266601562, -0.16752611100673676, -0.6478559970855713, -0.5434275269508362, -1.2006886005401611, -0.372647225856781, -0.8850846886634827, 0.1915150135755539, -1.2387709617614746, 0.4144960939884186, 0.030167898163199425, 1.1879230737686157, -1.9922966957092285, -0.6168513298034668, -1.6976360082626343, 0.4547102749347687, 1.7002016305923462, 1.3849918842315674, -0.5456480979919434, 0.977872908115387, -1.7105079889297485, 0.4329536557197571, 1.3136845827102661, -0.7158005833625793, 0.4939936101436615, 0.9068830609321594, 0.3854405879974365], [-0.45565563440322876, -0.19923216104507446, 1.4308332204818726, 0.6355490684509277, 0.7259552478790283, 0.6966251730918884, 1.701687216758728, 0.033081959933042526, 1.8329085111618042, 2.753746747970581, -0.9549384713172913, -0.2420712560415268, -0.45446357131004333, 0.49182814359664917, -1.5362932682037354, 0.22823576629161835, -0.2399609237909317, -1.48017156124115, 1.7297240495681763, -0.6753795742988586, 0.3588336408138275, -0.06452322751283646, 0.8693625330924988, 0.327549546957016, 1.4873930215835571, -0.29322555661201477, -1.2271063327789307, -0.31869035959243774, 0.4347740709781647, -0.6644095182418823, -0.6483697295188904, 0.3908122479915619, -0.2678537666797638, 2.4778318405151367, 1.2619361877441406, -0.2926500737667084, 0.671452522277832, -1.803162932395935, 1.0530644655227661, 0.4895841181278229, -0.8708539605140686, -0.9204639792442322, 0.6641924381256104, -0.22876586019992828, 1.2611844539642334, -1.6120902299880981, 0.42895200848579407, 1.2502799034118652, 0.8608014583587646, 0.14601260423660278, 2.0029454231262207, -0.6845848560333252, 1.4407062530517578, -0.3428519070148468, 1.063576102256775, -2.3220386505126953, 0.19829919934272766, -0.7529899477958679, 1.2828930616378784, -1.1155178546905518, -0.1301974654197693, -0.19682934880256653, -2.6073880195617676, -0.6517239809036255], [0.29949748516082764, 0.1984672099351883, -1.6721839904785156, 1.2864960432052612, -0.6754822134971619, -1.2581814527511597, -0.5596168637275696, 0.5424982309341431, 1.6669518947601318, 0.3671974539756775, -0.143446683883667, 0.2944316864013672, -0.434876948595047, 0.31408560276031494, 0.4794062674045563, 0.47396641969680786, 1.021246075630188, -0.32288283109664917, -0.6777264475822449, -0.1560618281364441, -1.0882072448730469, 0.7590558528900146, 0.08693189173936844, -0.5850092768669128, 1.3809785842895508, -1.3355053663253784, -1.629199743270874, 1.922715663909912, 0.3403259515762329, -0.6665639281272888, 2.1688873767852783, -0.7771214246749878, -1.2950135469436646, -0.716010332107544, 1.6746759414672852, -1.3614078760147095, -0.5561742782592773, 0.4018923342227936, 0.9097135663032532, -0.29709941148757935, 1.4955147504806519, -0.8199509382247925, -1.9634069204330444, 0.39547017216682434, -0.31293174624443054, 0.233201265335083, -1.481607437133789, 0.3226752281188965, -0.773772120475769, -1.4179030656814575, 0.6084570288658142, -0.045283496379852295, -0.6775190234184265, 0.9658199548721313, -0.27024176716804504, 0.516387939453125, -0.053183600306510925, -1.5361422300338745, -0.7509334683418274, 0.6626761555671692, 0.2655896246433258, 0.6011071801185608, -0.1368764191865921, 0.47567734122276306], [0.026568396016955376, -1.1435883045196533, 0.39021825790405273, -0.1923695057630539, 0.5164997577667236, -0.87346351146698, -1.0428959131240845, 0.674892783164978, -0.2871577739715576, -0.3861980140209198, 0.7901439070701599, 1.9311717748641968, -0.810184895992279, 1.518263816833496, 0.7798454761505127, -0.4448068141937256, 0.33652129769325256, 1.9770939350128174, 0.7835481762886047, 1.3555930852890015, -0.9127660989761353, -0.4817417562007904, -0.8608099222183228, -0.15784229338169098, -0.21894796192646027, -0.6330151557922363, 0.01271548680961132, 0.2506696879863739, 1.8675137758255005, -1.1089353561401367, 0.5359789729118347, 0.1338299959897995, -0.43126291036605835, -0.4022330939769745, 3.0738773345947266, 1.7862176895141602, 0.31958428025245667, 1.1319127082824707, 1.3702037334442139, -1.532804250717163, -1.353588342666626, 0.5337682366371155, -0.898385763168335, -0.02125720866024494, -0.6282135844230652, -0.7083035111427307, 0.5312960743904114, 0.3159734606742859, 1.010133981704712, 0.8073252439498901, -0.439707487821579, -0.5176560878753662, 0.7512898445129395, -1.7269459962844849, 0.5524671077728271, 0.3714490234851837, 2.168166399002075, 1.0641573667526245, 0.22282785177230835, -0.47279244661331177, 0.09066904336214066, 0.5228930711746216, 0.38977518677711487, -1.693323016166687], [2.5100409984588623, 0.36779728531837463, 0.061935510486364365, 0.6758694648742676, -0.5840467810630798, 1.0447098016738892, 1.1293048858642578, 0.7406521439552307, -0.17130924761295319, 1.3288074731826782, 0.23881162703037262, 1.9049609899520874, 0.05773314833641052, 2.4418044090270996, 0.18018028140068054, -1.704668641090393, 1.0500677824020386, -0.9085142016410828, 0.5171149969100952, 1.4680782556533813, 0.9815889596939087, 0.8758049011230469, 0.9511933326721191, -2.561306953430176, 0.4837521016597748, -0.4102117121219635, -0.36449986696243286, 0.8124352693557739, 0.590404748916626, 1.67669677734375, 0.36830395460128784, 0.5101791620254517, 0.6054632663726807, 0.861022412776947, -0.7535690665245056, -1.069536566734314, -0.5750333070755005, 0.09525194019079208, 2.927424192428589, -0.6523788571357727, -1.7227013111114502, 0.34627285599708557, -1.322205901145935, -0.32284125685691833, 0.4242587387561798, -0.9298617243766785, -0.6096064448356628, -1.5857369899749756, 0.20483475923538208, -0.1911155730485916, 0.28113454580307007, 0.7248769998550415, -0.23039312660694122, -1.1576061248779297, -1.4427627325057983, -0.23888802528381348, -0.28213945031166077, 0.21449430286884308, 0.571972668170929, -0.8388611078262329, -0.3511626422405243, 1.3263617753982544, 0.7990684509277344, 1.75727117061615], [-0.08625509589910507, 0.40905964374542236, -0.2637607753276825, -2.3532257080078125, -0.00449663819745183, -0.41130056977272034, -0.7023178935050964, 1.2716277837753296, 0.1226416751742363, -0.5551825761795044, -1.1933848857879639, 0.24350689351558685, 0.6888547539710999, -0.5557176470756531, 1.8067874908447266, 0.7096846699714661, 1.0251383781433105, 0.24720855057239532, 0.015241600573062897, 1.0161229372024536, -0.5829737782478333, 0.7648690938949585, 0.9902405738830566, 0.13345186412334442, 1.5354512929916382, 0.5997422337532043, 1.1222058534622192, -0.8188692927360535, -0.36281275749206543, 0.13551197946071625, 0.10944041609764099, 0.6541996002197266, 0.7903560996055603, 1.7983300685882568, 0.02800666354596615, -0.9834302663803101, -0.5870594382286072, 0.32144463062286377, 0.047106336802244186, 0.4678506851196289, -0.4622267186641693, 1.2535851001739502, 1.253754734992981, 0.9302071928977966, -0.8086279630661011, 0.33657681941986084, 0.16691909730434418, 1.6232857704162598, 0.6531877517700195, 0.3349516987800598, -1.127846360206604, -0.7395486235618591, 0.9228838682174683, 1.2895375490188599, 0.33189767599105835, -0.5201720595359802, -0.8766719698905945, -0.9613202810287476, -1.359739065170288, 1.2195215225219727, -0.21443967521190643, 1.1350215673446655, 0.7626919746398926, 1.7504969835281372], [-1.2136290073394775, 2.7887327671051025, 0.013743337243795395, 2.661315679550171, 0.2434156984090805, 0.4108692407608032, -1.0242191553115845, 0.2726024091243744, -0.06594084203243256, 1.3779637813568115, -1.1178605556488037, 2.4198262691497803, -0.4962466061115265, 1.4430123567581177, 0.8694618344306946, 0.8250577449798584, 2.908494472503662, -0.35826900601387024, 0.48155131936073303, 1.1578233242034912, 1.2328908443450928, -1.0729668140411377, 0.019115805625915527, -0.5234253406524658, -1.1590726375579834, -1.7284908294677734, -0.31529873609542847, -0.9210197329521179, -0.6501592397689819, 0.7509885430335999, -1.4722412824630737, -0.7173023223876953, -0.01137707568705082, -1.6705291271209717, 0.2072598785161972, 0.8661514520645142, 1.5451985597610474, 1.1393132209777832, -0.9942152500152588, -0.4175451099872589, -0.005043641198426485, -0.8034911751747131, -0.41869083046913147, -0.23699851334095, 0.1418192833662033, 0.3167400062084198, -1.0227915048599243, 0.006734118796885014, -2.0047152042388916, 0.4133475720882416, 1.2880934476852417, -1.392043948173523, -0.38089683651924133, 1.1652028560638428, 0.08835894614458084, -0.5410462617874146, 0.7117265462875366, 1.1555529832839966, 0.9752518534660339, -0.5029804706573486, 0.3770495355129242, 0.2175407111644745, -0.450698584318161, 0.5697077512741089], [-1.232940912246704, 0.7062831521034241, -1.0454131364822388, 1.6380828619003296, -0.7659499049186707, 0.5480772256851196, -1.1358237266540527, 0.7179449796676636, -1.8347408771514893, -0.9180988073348999, 0.4081766605377197, -0.7692108750343323, 1.7216907739639282, -0.506392240524292, -1.205100655555725, -0.2318834513425827, -2.203562021255493, 0.8707070350646973, 0.8922607898712158, 0.2122134119272232, 0.48577141761779785, -0.7775963544845581, 0.8985913991928101, -1.0169845819473267, -0.3317851424217224, 0.34267154335975647, -0.41396617889404297, 0.6402769088745117, 0.3790282607078552, 1.7567132711410522, -0.10513866692781448, -0.004735871683806181, 0.974288284778595, -1.1814839839935303, 0.7683126926422119, -0.7480061054229736, -1.7834659814834595, 0.959634006023407, -0.8890672922134399, 0.266968697309494, -0.9933047890663147, 0.9498511552810669, -0.5154820680618286, -1.0556236505508423, -0.3391610383987427, -0.27080947160720825, -0.8517916202545166, 0.963931143283844, 1.3274078369140625, 1.1492053270339966, -0.7137386202812195, -0.32660722732543945, 0.8952319622039795, -1.5591399669647217, -0.061232149600982666, 2.0972654819488525, -0.2831759452819824, -0.22159895300865173, -0.32786327600479126, -0.26556047797203064, -0.8928378820419312, 1.1889969110488892, -0.9246488213539124, 1.155483603477478], [-2.67739200592041, -0.3523607552051544, 0.538963794708252, 0.27278101444244385, -0.39167240262031555, 1.966154932975769, 2.486865997314453, 0.27316272258758545, -0.1739111840724945, 1.7162424325942993, -0.0713234692811966, -0.605573296546936, 0.16016322374343872, 0.955554723739624, -1.5104093551635742, 1.6355586051940918, -1.3295994997024536, 1.365958571434021, 0.6413559317588806, -0.3593944311141968, 1.5748772621154785, 0.4318181276321411, 2.6650404930114746, 0.6937723159790039, 0.1109270453453064, 0.18462663888931274, -0.25602835416793823, -0.36029762029647827, 0.49088749289512634, -0.608174204826355, 1.9906182289123535, -0.06585489958524704, -1.056486964225769, 1.8784641027450562, 0.17598991096019745, -0.4621545076370239, -1.5362855195999146, -0.7884404063224792, 0.8852133750915527, -1.5356974601745605, 1.130411148071289, -1.0771607160568237, -1.7376481294631958, 0.8487009406089783, 0.7796165347099304, -0.8406462073326111, 0.26802778244018555, 1.3847839832305908, 3.6342787742614746, -0.20967647433280945, -0.4669637680053711, 0.1270066797733307, 0.9481669068336487, -0.27726516127586365, 0.09427738934755325, 0.8618383407592773, -1.0952671766281128, -3.0439465045928955, -0.037915267050266266, -0.6209532022476196, 1.8491578102111816, -0.6224366426467896, 0.8285495042800903, -1.6206028461456299], [0.23493950068950653, -0.044773854315280914, -1.61459481716156, -0.7898394465446472, 0.6292448043823242, 0.6589744687080383, -0.07986710220575333, 0.90658038854599, -0.15194161236286163, 0.10296966880559921, 1.2880853414535522, 0.2851685583591461, -0.2860731780529022, 0.294481486082077, 0.8751910924911499, -0.5371367335319519, -1.767793893814087, -0.41896966099739075, 1.197447419166565, 1.0367921590805054, 0.5866304039955139, 0.24506492912769318, 0.48943790793418884, 0.5312039256095886, 0.9369214773178101, -1.6203043460845947, -0.5174579620361328, 1.531695008277893, 0.7736636400222778, 1.0011060237884521, -1.4220813512802124, -0.7936468124389648, 0.5248193740844727, -0.7412958741188049, 0.9508176445960999, -1.8454480171203613, -1.6329896450042725, -0.35032758116722107, 0.13527698814868927, -0.6046345233917236, 0.3361581563949585, -1.9328944683074951, 1.0020254850387573, -1.036655068397522, 1.0960218906402588, -0.6531254053115845, 3.088454008102417, 0.08840204030275345, -0.8184220790863037, 1.5823869705200195, 0.38431525230407715, 0.6957141757011414, -0.36060819029808044, -0.8089382648468018, 1.04110848903656, -0.06509772688150406, -0.5813995599746704, 1.431301474571228, 0.8574856519699097, -0.37932878732681274, 0.5395416617393494, -0.6557298302650452, 1.0965549945831299, 0.3802025020122528], [-0.25056135654449463, -1.0137910842895508, 2.9006619453430176, -1.6560533046722412, -1.0721367597579956, -0.8150604963302612, -0.7619029879570007, 0.9230347275733948, 0.012305299751460552, -0.9745306968688965, 1.2058409452438354, -0.28916001319885254, 0.66422438621521, -0.9209710359573364, 2.1072170734405518, -0.5553110837936401, -0.39696022868156433, 0.8261814117431641, 0.7313713431358337, -0.34404510259628296, -1.0293552875518799, 0.2810235321521759, -2.34065842628479, -0.5057564973831177, 0.31199911236763, 0.027456866577267647, -0.023762820288538933, 0.9830757975578308, 1.2447268962860107, -0.29542410373687744, -1.0225439071655273, -0.43763384222984314, -1.1391547918319702, -0.9236493706703186, 1.492290735244751, 0.15085098147392273, 0.4050748348236084, 0.341032475233078, -0.7495540380477905, 0.27748700976371765, -0.49320393800735474, 0.6163479089736938, 0.2450438141822815, -0.8542546629905701, -2.1510508060455322, -2.065335750579834, 0.01720902882516384, 0.0869620069861412, -0.17556500434875488, -0.056430477648973465, -1.3361406326293945, 0.03574345260858536, 2.5496811866760254, -0.25871631503105164, -0.3706130087375641, 0.0892842710018158, 1.0512819290161133, 1.1326453685760498, 0.576607882976532, -0.8158462643623352, -0.96806401014328, -2.150677442550659, 0.3855237066745758, -0.5630570650100708], [2.6099283695220947, -0.11415617913007736, -0.5494241118431091, 2.107886552810669, -1.8925524950027466, 1.2669821977615356, 0.45381587743759155, -1.2820823192596436, 0.4801565706729889, -0.4742854833602905, 0.8293612003326416, 0.8071573376655579, -1.4226672649383545, 1.6628199815750122, -0.7104512453079224, 0.4960288405418396, -0.16114753484725952, 0.4319702684879303, -2.074061632156372, -0.2043628990650177, 1.0999853610992432, -0.015417800284922123, 0.12039628624916077, 0.4728216528892517, -0.2183956503868103, 0.8725202083587646, -1.5083038806915283, -0.4783700406551361, -0.09720047563314438, 1.2422212362289429, -0.07350976020097733, -0.8255117535591125, 0.9444839358329773, 0.08365477621555328, -1.2544105052947998, -0.880993664264679, -1.4147707223892212, -0.031056512147188187, 0.5628268122673035, 0.15637433528900146, 1.2275720834732056, 0.23179945349693298, -1.183905005455017, -0.16275298595428467, 0.7620535492897034, -0.8252846598625183, -2.3267476558685303, 0.5925556421279907, 0.2892512381076813, -0.2775849401950836, 1.0604547262191772, -1.2785584926605225, 0.6549065709114075, 2.011390447616577, 0.13620132207870483, -1.4349586963653564, -1.1436378955841064, 2.4829540252685547, -1.9515479803085327, 0.26415374875068665, -0.11557811498641968, -0.07217197120189667, -0.5294979810714722, -1.50613534450531], [-1.0616297721862793, -0.19012349843978882, 0.8157771229743958, -1.3447636365890503, -1.919933557510376, 1.046799659729004, 1.3542969226837158, 1.2189208269119263, 0.9581499099731445, -0.1996409296989441, -1.5891501903533936, 0.78233402967453, 0.6287292838096619, -0.30334869027137756, -1.411624550819397, 1.604606032371521, 1.4328199625015259, -0.13849341869354248, -0.6406228542327881, 0.21207746863365173, -0.5684170722961426, -0.5132706165313721, -0.8185892105102539, -2.1174139976501465, -0.45688486099243164, -0.9552473425865173, 1.2416359186172485, -2.0652477741241455, -0.23050114512443542, -2.7371599674224854, 0.17597614228725433, 0.8851243853569031, -1.0326082706451416, -0.35484811663627625, -0.5028657913208008, 0.03558063879609108, -0.2252832055091858, 0.4264836311340332, 1.6513646841049194, 0.7740821838378906, -0.39237818121910095, -0.8248424530029297, -0.6413030624389648, -3.035057544708252, 0.31371521949768066, 1.1394848823547363, -1.0012729167938232, -0.7490553855895996, 0.0643642395734787, -0.22565780580043793, -1.0542564392089844, 0.9270275235176086, -0.032678622752428055, -0.09933432191610336, 1.1452579498291016, 0.6008090972900391, -0.15114332735538483, -0.32634401321411133, -0.7222138047218323, 1.7299457788467407, -2.2043402194976807, -0.9326290488243103, -0.6231870651245117, -0.09791287034749985], [-0.09072037786245346, 0.7166991233825684, -0.7609995007514954, -0.46150821447372437, 1.0930919647216797, -0.4939875900745392, -1.4418482780456543, 2.019665002822876, -2.013852596282959, -1.5759254693984985, -0.36428266763687134, -0.1512317657470703, 0.08537040650844574, 0.5758237838745117, -0.0959727093577385, 0.8231845498085022, -1.079055666923523, -0.5361097455024719, 0.6840655207633972, 0.46974438428878784, -0.44178855419158936, 0.5606220364570618, 0.9541524052619934, -1.2481358051300049, 0.4212951064109802, -1.0569231510162354, 2.0298471450805664, 0.7986588478088379, -1.304567813873291, 3.6574244499206543, -0.03880874440073967, 1.385108232498169, -0.25900572538375854, 0.4564710557460785, 0.05937584117054939, -0.8916346430778503, 1.3303133249282837, -0.5533684492111206, -0.48233798146247864, 0.06658631563186646, 1.289028525352478, -0.6455473303794861, 0.24156971275806427, 0.1732269674539566, 0.07706917077302933, -0.21194016933441162, -1.040913462638855, -0.33931097388267517, -1.2302067279815674, -0.7451566457748413, 0.7901161313056946, -0.11776070296764374, -1.7945935726165771, -0.17223766446113586, -1.8741819858551025, -1.533639907836914, 1.8943263292312622, 0.150232195854187, -0.7063129544258118, -2.000690221786499, 0.2247496098279953, -0.5981243252754211, -0.511512041091919, 0.8174574971199036], [-1.230929970741272, 1.1218953132629395, 0.6017890572547913, 1.2838295698165894, 0.31313076615333557, 0.06714294105768204, 0.49106064438819885, -0.8138769865036011, 0.2809615135192871, -0.12090606987476349, 0.3053019642829895, -1.870089054107666, -0.91424959897995, 2.264295816421509, -1.3525782823562622, -0.43015390634536743, 0.36535266041755676, -0.05448663979768753, 0.6046324372291565, 0.7072058320045471, -2.175802707672119, 0.4019564092159271, -0.9946386218070984, -0.21084623038768768, -2.064242124557495, 0.2355281412601471, 0.6511242985725403, 0.34568849205970764, 1.236885666847229, 0.6483721137046814, -0.8991009593009949, 0.9258548617362976, 2.3241779804229736, -1.6706316471099854, 0.5121344327926636, -0.2659118175506592, 0.6974695324897766, 1.7436484098434448, -0.5685124397277832, 0.8251424431800842, 0.13587349653244019, 0.05228279531002045, 0.10226472467184067, -0.43214017152786255, 1.221700668334961, -1.0751960277557373, -0.17078959941864014, 0.4494091272354126, -1.88426673412323, -0.4105020761489868, 0.0883219838142395, -0.8647100925445557, -0.055102717131376266, 0.6674942970275879, -0.6519526839256287, 1.287261724472046, -1.1269620656967163, 0.8860871195793152, -0.8249700665473938, 0.12703202664852142, -1.2243138551712036, 0.5154805779457092, -0.17229963839054108, -0.7779402136802673], [-0.5852723121643066, -0.02443341352045536, 0.07605662196874619, -2.7459404468536377, -1.2025924921035767, 0.7876707911491394, -0.9380824565887451, 0.39569661021232605, 0.5405066609382629, -0.690902590751648, 0.886002242565155, 0.6132981777191162, -1.6472156047821045, 0.2687550187110901, -2.0046300888061523, -0.027256740257143974, -0.24516580998897552, 1.817436695098877, 1.9805070161819458, 0.012773199938237667, -0.600926399230957, -1.353158950805664, -0.6127339601516724, 0.769340991973877, 0.7205185294151306, 1.7971789836883545, -1.1620557308197021, 0.24945461750030518, 0.8774576187133789, -0.15119560062885284, -0.5684354901313782, -0.6107727885246277, -0.8657986521720886, -1.1311650276184082, -0.9331271052360535, 0.5145889520645142, -0.015414178371429443, 0.0225363876670599, -1.109360694885254, 0.2223232537508011, 1.516565203666687, 0.17682653665542603, -0.5147882103919983, 1.6819442510604858, -1.7124567031860352, -0.12702801823616028, 1.4906644821166992, -1.342191219329834, 0.1408928781747818, -1.8922940492630005, 1.0438481569290161, 0.05737387388944626, 0.49645745754241943, 0.6236920356750488, -1.6666626930236816, 0.8848022818565369, 0.8325222134590149, -0.06889645755290985, -0.31107276678085327, -0.9165821075439453, -1.1310851573944092, 0.42468056082725525, -0.8944901823997498, -0.20435957610607147], [-0.9641890525817871, -0.5296716690063477, -0.18280690908432007, -2.3682897090911865, 0.5860520601272583, 1.1188167333602905, 0.6710947155952454, -1.0788615942001343, -1.0357041358947754, -1.1209903955459595, -0.686901330947876, -1.1396448612213135, -0.18109673261642456, 0.6608520150184631, 0.37301063537597656, 0.17444518208503723, 0.28032588958740234, -0.7747280597686768, -1.7276856899261475, -0.3003244698047638, -0.3374224901199341, 1.0197103023529053, -0.8837277889251709, -0.505155622959137, -0.0032423939555883408, 0.6298382878303528, -0.40087446570396423, -0.8940386772155762, 0.31053686141967773, -0.31453946232795715, 0.5786511301994324, -1.511248230934143, 0.7894679307937622, 0.4385518431663513, 1.1641243696212769, 2.893138885498047, 1.2882448434829712, 1.2113014459609985, -1.6128803491592407, 1.1579891443252563, -0.7043078541755676, 0.12035125494003296, 0.5265023112297058, -0.6010223031044006, 0.30300360918045044, 0.9787966012954712, 1.081485629081726, 0.3965773284435272, 0.8562701344490051, 1.6432031393051147, 0.3272430896759033, 1.0222482681274414, -0.3536655306816101, 1.705362319946289, -1.0594309568405151, -1.3665016889572144, 0.6605420112609863, 0.48960381746292114, -1.750848650932312, 1.9373129606246948, 0.14697547256946564, -0.9418259263038635, -0.5146782994270325, 0.1770697981119156]], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(matmul_at_b, json.loads(_TESTS), 'triton', *[0.0001, 0.0001])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")